# Phase-0: Setup & harmonization

# Step-0.1:Grid & CRS

In [ ]:
# ===============================
# Phase 0.1 — Grid & CRS setup
# ===============================
import os, json, numpy as np
import rasterio
from rasterio.crs import CRS
from fiona.transform import transform_geom

# Optional imports (for shapefile I/O)
try:
    import fiona
    HAVE_FIONA = True
except Exception as e:
    HAVE_FIONA = False
    print("[WARN] Fiona not available; will skip shapefile reprojection/copy. Error:", e)

# -------------------- USER PATHS --------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
DEM_IN = os.path.join(BASE, "NTD_DEM_Clipped.tif")
SHP_IN = os.path.join(BASE, "Study_Area.shp")
OUT_DIR = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS")
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------- SETTINGS ----------------------
TARGET_EPSG = 32645  # WGS 84 / UTM Zone 45N
TARGET_CRS = CRS.from_epsg(TARGET_EPSG)
STD_NODATA_INT16 = -32768  # sensible NoData for int16 DEMs
ANCHOR_DEM_OUT = os.path.join(OUT_DIR, "DEM_anchor_32645.tif")
SPEC_JSON = os.path.join(OUT_DIR, "project_spec.json")
SHP_REPROJECTED = os.path.join(OUT_DIR, "Study_Area_32645.shp")

# Predefine to avoid NameError later
shp_out_path = None

# -------------------- READ DEM ----------------------
with rasterio.open(DEM_IN) as src:
    crs = src.crs
    epsg = crs.to_epsg() if crs else None
    transform = src.transform
    bounds = src.bounds
    res_x = transform.a
    res_y = -transform.e
    width, height = src.width, src.height
    dtype = src.dtypes[0]
    nodata = src.nodata

    print("== DEM INFO ==")
    print("Path:", DEM_IN)
    print("CRS:", crs)
    print("EPSG:", epsg)
    print("Size (rows, cols):", height, width)
    print("Resolution (x, y):", res_x, res_y)
    print("Bounds (left, bottom, right, top):", (bounds.left, bounds.bottom, bounds.right, bounds.top))
    print("dtype:", dtype, "NoData:", nodata)

    # --- Sanity checks ---
    if epsg is None:
        raise RuntimeError("DEM has no EPSG code. Assign WGS 84 / UTM 45N (EPSG:32645) to this raster and rerun.")

    if epsg != TARGET_EPSG:
        raise RuntimeError(f"DEM EPSG is {epsg}, expected {TARGET_EPSG}. Reproject the DEM first, then rerun Phase 0.1.")

    # Standardize NoData
    nodata_to_write = STD_NODATA_INT16 if nodata is None else nodata

    # Copy DEM into the reports folder as the anchor grid with standardized NoData & compression
    profile = src.profile.copy()
    profile.update(
        nodata=nodata_to_write,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED"
    )

    data = src.read(1)
    # If float sneaks in, map NaNs to NoData and cast back
    if np.issubdtype(data.dtype, np.floating):
        data = np.where(np.isfinite(data), data, nodata_to_write).astype(src.dtypes[0])

    with rasterio.open(ANCHOR_DEM_OUT, "w", **profile) as dst:
        dst.write(data, 1)

print(f"[OK] Anchor DEM written: {ANCHOR_DEM_OUT}")

# -------------------- REPROJECT/COPY SHAPEFILE ----------------------
if HAVE_FIONA and os.path.exists(SHP_IN):
    with fiona.open(SHP_IN, "r") as shp_src:
        # Fiona CRS is a dict (may be None)
        shp_crs_dict = shp_src.crs
        src_crs_obj = CRS(shp_crs_dict) if shp_crs_dict else None

        print("== SHAPEFILE INFO ==")
        print("Path:", SHP_IN)
        print("CRS (dict):", shp_crs_dict)
        print("CRS (obj):", src_crs_obj)

        # Determine if reprojection is needed
        if src_crs_obj is None:
            print("[WARN] Shapefile has no CRS; assuming EPSG:4326 for reprojection.")
            src_crs_dict = CRS.from_epsg(4326).to_dict()
            needs_reproj = True
        else:
            src_crs_dict = src_crs_obj.to_dict()
            needs_reproj = (src_crs_obj != TARGET_CRS)

        if needs_reproj:
            dst_crs_dict = TARGET_CRS.to_dict()
            out_schema = shp_src.schema.copy()
            # Write reprojected shapefile
            with fiona.open(
                SHP_REPROJECTED, "w",
                driver="ESRI Shapefile",
                crs=dst_crs_dict,
                schema=out_schema
            ) as shp_dst:
                for feat in shp_src:
                    geom_in = feat.get("geometry")
                    if geom_in is None:
                        continue
                    geom_out = transform_geom(src_crs_dict, dst_crs_dict, geom_in)
                    shp_dst.write({"geometry": geom_out, "properties": feat["properties"]})
            shp_out_path = SHP_REPROJECTED
            print(f"[OK] Study_Area reprojected to EPSG:{TARGET_EPSG}: {shp_out_path}")
        else:
            # Already in target CRS — copy all sidecar files to OUT_DIR
            base_noext = os.path.splitext(SHP_IN)[0]
            for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
                srcp = base_noext + ext
                if os.path.exists(srcp):
                    dstp = os.path.join(OUT_DIR, os.path.basename(srcp))
                    if srcp != dstp:
                        with open(srcp, "rb") as fi, open(dstp, "wb") as fo:
                            fo.write(fi.read())
            shp_out_path = os.path.join(OUT_DIR, os.path.basename(SHP_IN))
            print("[INFO] Shapefile already in target CRS; copied to:", shp_out_path)
else:
    print("[WARN] Skipped shapefile reprojection/copy (missing Fiona or file).")

# -------------------- WRITE PROJECT SPEC JSON ----------------------
# Guard: make sure shp_out_path exists (or set to None explicitly)
if shp_out_path is not None and (not os.path.exists(shp_out_path)):
    print("[WARN] shp_out_path was set but file not found on disk; setting to None.")
    shp_out_path = None

with rasterio.open(ANCHOR_DEM_OUT) as anchor:
    transform = anchor.transform
    bounds = anchor.bounds
    width, height = anchor.width, anchor.height
    res_x = transform.a
    res_y = -transform.e
    dtype = anchor.dtypes[0]
    nodata = anchor.nodata

spec = {
    "anchor_dem_path": ANCHOR_DEM_OUT,
    "original_dem_path": DEM_IN,
    "study_shapefile_input": SHP_IN,
    "study_shapefile_32645": shp_out_path,
    "crs_epsg": TARGET_EPSG,
    "crs_wkt": TARGET_CRS.to_wkt(),
    "width_px": width,
    "height_px": height,
    "resolution_x_m": float(res_x),
    "resolution_y_m": float(res_y),
    "transform": [transform.a, transform.b, transform.c, transform.d, transform.e, transform.f],
    "bounds_left": float(bounds.left),
    "bounds_bottom": float(bounds.bottom),
    "bounds_right": float(bounds.right),
    "bounds_top": float(bounds.top),
    "dtype": dtype,
    "nodata": int(nodata) if nodata is not None else None,
    "notes": {
        "coherence_threshold": 0.3,
        "all_future_rasters_must_match_this_grid": True
    }
}

with open(SPEC_JSON, "w") as f:
    json.dump(spec, f, indent=2)

print(f"[OK] Project spec written: {SPEC_JSON}")
print("[DONE] Phase 0.1 complete — proceed to Phase 0.2 (Study mask) using the shapefile at:",
      shp_out_path if shp_out_path else "Shapefile not prepared")


# Step-0.2: Masks

In [ ]:
# ===============================
# Phase 0.2 — Study Mask
# ===============================
import os, json, numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.crs import CRS
try:
    import fiona
    from fiona.transform import transform_geom
    HAVE_FIONA = True
except Exception as e:
    HAVE_FIONA = False
    print("[ERROR] Fiona not available:", e)

# -------------------- PATHS --------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
PHASE01_DIR = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS")
ANCHOR_DEM = os.path.join(PHASE01_DIR, "DEM_anchor_32645.tif")
PROJECT_SPEC = os.path.join(PHASE01_DIR, "project_spec.json")
STUDY_SHP = os.path.join(BASE, "Study_Area.shp")

OUT_DIR = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
os.makedirs(OUT_DIR, exist_ok=True)
MASK_TIF = os.path.join(OUT_DIR, "study_mask.tif")
STATS_JSON = os.path.join(OUT_DIR, "study_mask_stats.json")

# -------------------- CHECKS --------------------
if not os.path.exists(ANCHOR_DEM):
    raise FileNotFoundError(f"Anchor DEM not found: {ANCHOR_DEM} (run Phase 0.1 first)")
if not HAVE_FIONA:
    raise RuntimeError("Fiona is required for reading shapefiles. Install with: pip install fiona")

if not os.path.exists(STUDY_SHP):
    raise FileNotFoundError(f"Study_Area.shp not found: {STUDY_SHP}")

# -------------------- READ ANCHOR GRID --------------------
with rasterio.open(ANCHOR_DEM) as ref:
    tgt_crs = ref.crs
    transform = ref.transform
    width, height = ref.width, ref.height
    res_x = transform.a
    res_y = -transform.e
    profile = ref.profile

print("== ANCHOR GRID ==")
print("CRS:", tgt_crs, "| size:", height, "x", width, "| res (m):", res_x, res_y)

# -------------------- LOAD & (IF NEEDED) REPROJECT POLYGONS --------------------
shapes = []
with fiona.open(STUDY_SHP, "r") as src:
    src_crs = src.crs
    if src_crs is None:
        print("[WARN] Shapefile has no CRS; assuming EPSG:4326")
        src_crs = CRS.from_epsg(4326).to_dict()
    dst_crs = tgt_crs.to_dict()

    for feat in src:
        geom = feat.get("geometry")
        if geom is None:
            continue
        # Reproject on-the-fly if CRS differs
        if CRS(src_crs) != tgt_crs:
            geom = transform_geom(src_crs, dst_crs, geom)
        shapes.append((geom, 1))

if not shapes:
    raise RuntimeError("No valid geometries found in Study_Area.shp")

# -------------------- RASTERIZE TO MASK --------------------
mask = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,                   # outside polygon = 0
    default_value=1,          # inside polygon = 1
    dtype="uint8",
    all_touched=True          # include pixels touched by boundary
)

# -------------------- SAVE MASK --------------------
out_profile = profile.copy()
out_profile.update(
    dtype="uint8",
    count=1,
    nodata=0,        # 0=outside; if you prefer explicit nodata, set to 255 and fill outside with 0
    compress="LZW",
    tiled=True,
    bigtiff="IF_NEEDED"
)

with rasterio.open(MASK_TIF, "w", **out_profile) as dst:
    dst.write(mask, 1)

print(f"[OK] Study mask written: {MASK_TIF}")

# -------------------- BASIC STATS --------------------
px_area_m2 = abs(transform.a * transform.e)  # note e is negative
inside_px = int(mask.sum())
area_km2 = inside_px * px_area_m2 / 1e6

stats = {
    "mask_path": MASK_TIF,
    "anchor_dem": ANCHOR_DEM,
    "crs_epsg": tgt_crs.to_epsg(),
    "width_px": width,
    "height_px": height,
    "resolution_x_m": float(res_x),
    "resolution_y_m": float(res_y),
    "inside_pixels": inside_px,
    "pixel_area_m2": float(px_area_m2),
    "area_km2": float(area_km2),
    "all_touched": True
}

with open(STATS_JSON, "w") as f:
    json.dump(stats, f, indent=2)

print(f"[OK] Study mask stats written: {STATS_JSON}")
print("[DONE] Phase 0.2 — Study Mask complete.")


# Step-0.3: Quality Masks

In [ ]:
# ============================================
# Step 0.3 — Quality Masks 
# ============================================

import os, json
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import rowcol
from scipy.io import loadmat

# Distance transform 
try:
    from scipy.ndimage import binary_dilation, distance_transform_edt
    HAVE_SCIPY_ND = True
except Exception:
    HAVE_SCIPY_ND = False

# Optional reprojection of lon/lat -> anchor CRS
try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

# -------------------- PATHS --------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

PHASE01_DIR = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS")
PHASE02_DIR = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
OUT_DIR     = os.path.join(BASE, "Results", "Step-0.3-Quality_Masks")
os.makedirs(OUT_DIR, exist_ok=True)

ANCHOR_DEM  = os.path.join(PHASE01_DIR, "DEM_anchor_32645.tif")
STUDY_MASK  = os.path.join(PHASE02_DIR, "study_mask.tif")

PS_DIR      = os.path.join(BASE, "NTD_PS_InSAR_data")
MAT_COH_1   = os.path.join(PS_DIR, "coherence_clipped.mat")
MAT_COORD_1 = os.path.join(PS_DIR, "coordinates_clipped.mat")

# -------------------- OUTPUTS --------------------
COH_GRID_TIF          = os.path.join(OUT_DIR, "coherence_max.tif")
PS_COUNT_TIF          = os.path.join(OUT_DIR, "ps_count.tif")

COH_MASK_STRICT_TIF   = os.path.join(OUT_DIR, "coherence_mask_strict.tif")
COH_MASK_LOOSE_TIF    = os.path.join(OUT_DIR, "coherence_mask_loose.tif")
VALID_MASK_STRICT_TIF = os.path.join(OUT_DIR, "valid_mask_strict.tif")
VALID_MASK_LOOSE_TIF  = os.path.join(OUT_DIR, "valid_mask_loose.tif")

COH_MASK_LEGACY_TIF   = os.path.join(OUT_DIR, "coherence_mask.tif")   # legacy alias = loose
VALID_MASK_LEGACY_TIF = os.path.join(OUT_DIR, "valid_mask.tif")       # legacy alias = loose

CONTEXT_MASK_TIF      = os.path.join(OUT_DIR, "context_mask.tif")

# Option B outputs
DYNAMIC_MASK_TIF      = os.path.join(OUT_DIR, "dynamic_mask.tif")
PS_DIST_M_TIF         = os.path.join(OUT_DIR, "ps_distance_m.tif")    # QA (optional)

STATS_JSON            = os.path.join(OUT_DIR, "quality_mask_stats.json")

# -------------------- CONFIG --------------------
COH_THR_STRICT = 0.10
COH_THR_LOOSE  = 0.05

# Mask dilation used ONLY for coherence/valid masks (not required for dynamic option B)
DILATE_PIXELS_FOR_COH = 3

# Valid mask should require PS presence
PSC_MIN_FOR_VALID = 1

# coherence nodata
COH_NODATA = -9999.0

# -------------------- OPTION B SETTINGS --------------------
# Choose ONE mode:

# Mode 1 (fixed): set a distance threshold in meters
USE_AUTO_TUNE = True
D_MAX_M_FIXED = 250.0

# Mode 2 (auto-tune): pick smallest D_MAX_M (from candidates) that reaches target coverage
TARGET_DYNAMIC_OVER_STUDY = 0.60  # 0.60 = 60% of study area
D_MAX_CANDIDATES_M = [50, 75, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000, 1500, 2000]

# If you also want a tiny initial dilation of PS presence before distance transform (optional)
PS_SEED_DILATE_PIXELS = 0  # 0 recommended; set 1-2 only if PS rasterization is too sparse

def pp(msg):
    from datetime import datetime
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

# -------------------- HELPERS --------------------
def _read_anchor():
    if not os.path.exists(ANCHOR_DEM):
        raise FileNotFoundError(f"Anchor DEM not found: {ANCHOR_DEM}")
    with rasterio.open(ANCHOR_DEM) as ref:
        anchor = {
            "transform": ref.transform,
            "crs": ref.crs,
            "bounds": ref.bounds,
            "width": ref.width,
            "height": ref.height,
            "profile": ref.profile.copy(),
            "res_x": ref.transform.a,
            "res_y": -ref.transform.e,
        }
    return anchor

def _snap_mask_to_anchor(mask_path, anchor):
    if not os.path.exists(mask_path):
        raise FileNotFoundError(f"Study mask not found: {mask_path}")

    with rasterio.open(mask_path) as src:
        src_arr = src.read(1, masked=True)
        src_data = np.where(src_arr.mask, 0, src_arr.data).astype("float32")

        dst = np.zeros((anchor["height"], anchor["width"]), dtype="float32")
        reproject(
            source=src_data,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            resampling=Resampling.nearest,
            src_nodata=0,
            dst_nodata=0
        )
    return (dst > 0.5).astype(np.uint8)

def _load_ps_data_from_mat(mat_coh_path, mat_coord_path):
    if not os.path.exists(mat_coh_path):
        pp(f"[ERROR] Coherence file not found: {mat_coh_path}")
        return None
    if not os.path.exists(mat_coord_path):
        pp(f"[ERROR] Coordinates file not found: {mat_coord_path}")
        return None

    mc = loadmat(mat_coh_path, squeeze_me=False)
    if "coh" not in mc:
        pp(f"[ERROR] 'coh' not found in {mat_coh_path} keys={list(mc.keys())}")
        return None
    coh = np.squeeze(mc["coh"]).astype(np.float32)

    mcoord = loadmat(mat_coord_path, squeeze_me=False)
    if "lon" not in mcoord or "lat" not in mcoord:
        pp(f"[ERROR] 'lon'/'lat' not found in {mat_coord_path} keys={list(mcoord.keys())}")
        return None
    lon = np.squeeze(mcoord["lon"]).astype(np.float64)
    lat = np.squeeze(mcoord["lat"]).astype(np.float64)

    n = min(len(coh), len(lon), len(lat))
    if n == 0:
        return None
    if n != len(coh) or n != len(lon) or n != len(lat):
        pp(f"[WARN] Length mismatch. Truncating to {n}. coh={len(coh)} lon={len(lon)} lat={len(lat)}")
        coh, lon, lat = coh[:n], lon[:n], lat[:n]

    good = np.isfinite(coh) & np.isfinite(lon) & np.isfinite(lat)
    coh, lon, lat = coh[good], lon[good], lat[good]
    if coh.size == 0:
        return None

    pp(f"[LOAD] PS points (finite): {coh.size:,}")
    return coh.astype(np.float32), lon.astype(np.float64), lat.astype(np.float64)

def _reproject_and_clip(coh, lon, lat, anchor):
    if not HAVE_PYPROJ:
        raise RuntimeError("pyproj not installed. Install pyproj to reproject lon/lat to anchor CRS.")
    tf = Transformer.from_crs(4326, anchor["crs"], always_xy=True)
    x, y = tf.transform(lon, lat)

    b = anchor["bounds"]
    inside = (x >= b.left) & (x <= b.right) & (y >= b.bottom) & (y <= b.top)
    if not inside.any():
        pp("[WARN] No PS points inside anchor bounds after reprojection.")
        return np.array([], dtype=np.float32), np.array([], dtype=np.float64), np.array([], dtype=np.float64)

    pp(f"[REPROJ] {int(inside.sum()):,}/{len(x):,} inside anchor bounds")
    return coh[inside].astype(np.float32), x[inside].astype(np.float64), y[inside].astype(np.float64)

def _vectorized_rasterize(coh, x, y, anchor):
    H, W = anchor["height"], anchor["width"]
    tfm = anchor["transform"]

    if x.size == 0:
        return np.full((H, W), np.nan, dtype=np.float32), np.zeros((H, W), dtype=np.uint16)

    rows, cols = rowcol(tfm, x, y, op=np.floor)
    rows = np.asarray(rows, dtype=np.int64)
    cols = np.asarray(cols, dtype=np.int64)

    ok = (rows >= 0) & (rows < H) & (cols >= 0) & (cols < W) & np.isfinite(coh)
    rows, cols, coh = rows[ok], cols[ok], coh[ok]
    if coh.size == 0:
        return np.full((H, W), np.nan, dtype=np.float32), np.zeros((H, W), dtype=np.uint16)

    ps_count = np.zeros((H, W), dtype=np.uint16)
    np.add.at(ps_count, (rows, cols), 1)

    coh_max = np.full((H, W), -np.inf, dtype=np.float32)
    np.maximum.at(coh_max, (rows, cols), coh.astype(np.float32))
    coh_max = np.where(np.isfinite(coh_max) & (coh_max > -np.inf), coh_max, np.nan).astype(np.float32)

    return coh_max, ps_count

def _dilate_u8(mask_u8, iters):
    if iters <= 0:
        return mask_u8
    if not HAVE_SCIPY_ND:
        pp("[WARN] scipy.ndimage not available; skipping dilation.")
        return mask_u8
    st = np.ones((3, 3), dtype=bool)
    m = mask_u8.astype(bool)
    for _ in range(iters):
        m = binary_dilation(m, structure=st)
    return m.astype(np.uint8)

def _save_u8(path, arr_u8, anchor, nodata=0):
    prof = anchor["profile"].copy()
    prof.update(driver="GTiff", dtype="uint8", count=1, nodata=nodata,
                compress="LZW", tiled=True, BIGTIFF="IF_SAFER",
                transform=anchor["transform"], crs=anchor["crs"],
                width=anchor["width"], height=anchor["height"])
    with rasterio.open(path, "w", **prof) as dst:
        dst.write(arr_u8.astype(np.uint8), 1)

def _save_u16(path, arr_u16, anchor, nodata=0):
    prof = anchor["profile"].copy()
    prof.update(driver="GTiff", dtype="uint16", count=1, nodata=nodata,
                compress="LZW", tiled=True, BIGTIFF="IF_SAFER",
                transform=anchor["transform"], crs=anchor["crs"],
                width=anchor["width"], height=anchor["height"])
    with rasterio.open(path, "w", **prof) as dst:
        dst.write(arr_u16.astype(np.uint16), 1)

def _save_f32(path, arr_f32, anchor, nodata):
    prof = anchor["profile"].copy()
    prof.update(driver="GTiff", dtype="float32", count=1, nodata=float(nodata),
                compress="LZW", tiled=True, BIGTIFF="IF_SAFER",
                transform=anchor["transform"], crs=anchor["crs"],
                width=anchor["width"], height=anchor["height"])
    out = np.where(np.isfinite(arr_f32), arr_f32.astype(np.float32), np.float32(nodata))
    with rasterio.open(path, "w", **prof) as dst:
        dst.write(out, 1)

def _build_dynamic_mask_optionB(ps_present_u8, study_u8, px_m, out_dist_tif=True):
    """
    dynamic = study AND (distance_to_nearest_PS <= D_MAX_M)
    distance computed in meters on anchor grid
    """
    if not HAVE_SCIPY_ND:
        raise RuntimeError("scipy.ndimage (distance_transform_edt) not available. Install scipy.")

    ps_seed = ps_present_u8.copy()

    if PS_SEED_DILATE_PIXELS > 0:
        ps_seed = _dilate_u8(ps_seed, PS_SEED_DILATE_PIXELS)
        pp(f"[PS] Seed dilated by {PS_SEED_DILATE_PIXELS} iterations before distance transform")

    ps_bool = (ps_seed == 1)

    if not ps_bool.any():
        dist_m = np.full(ps_present_u8.shape, np.inf, dtype=np.float32)
        dynamic = np.zeros_like(study_u8, dtype=np.uint8)
        chosen = None
        return dynamic, dist_m, chosen

    # distance_transform_edt gives distance to nearest FALSE if you pass boolean array directly.
    # We need distance to nearest PS(True), so compute distance on inverted mask:
    dist_px = distance_transform_edt(~ps_bool).astype(np.float32)
    dist_m = dist_px * float(px_m)

    # Save distance raster for QA (masked outside study to reduce confusion)
    if out_dist_tif:
        dist_m_out = np.where(study_u8 == 1, dist_m, np.nan).astype(np.float32)
        _save_f32(PS_DIST_M_TIF, dist_m_out, anchor, nodata=COH_NODATA)
        pp(f"[OUT] ps_distance_m.tif → {PS_DIST_M_TIF}")

    # Choose D_MAX_M
    if USE_AUTO_TUNE:
        target = float(TARGET_DYNAMIC_OVER_STUDY)
        study_px = int(study_u8.sum())
        if study_px == 0:
            chosen = None
            dynamic = np.zeros_like(study_u8, dtype=np.uint8)
            return dynamic, dist_m, chosen

        chosen = None
        for D in D_MAX_CANDIDATES_M:
            dm = ((study_u8 == 1) & (dist_m <= float(D))).astype(np.uint8)
            frac = dm.sum() / study_px
            pp(f"[TUNE] D_MAX_M={int(D):4d} m → dynamic/study={frac*100:5.1f}%")
            if frac >= target:
                chosen = float(D)
                dynamic = dm
                pp(f"[SELECT] D_MAX_M={int(D)} m achieves target {target*100:.1f}%")
                break
        if chosen is None:
            chosen = float(D_MAX_CANDIDATES_M[-1])
            dynamic = ((study_u8 == 1) & (dist_m <= chosen)).astype(np.uint8)
            frac = dynamic.sum() / study_px
            pp(f"[SELECT] Max candidate used D_MAX_M={int(chosen)} m → {frac*100:.1f}% (target not reached)")
    else:
        chosen = float(D_MAX_M_FIXED)
        dynamic = ((study_u8 == 1) & (dist_m <= chosen)).astype(np.uint8)
        pp(f"[SELECT] Fixed D_MAX_M={int(chosen)} m")

    return dynamic, dist_m, chosen

def main():
    pp("=== Step 0.3 — Quality Masks ===")

    global anchor  # used in _build_dynamic_mask_optionB for distance raster saving
    anchor = _read_anchor()

    study = _snap_mask_to_anchor(STUDY_MASK, anchor)  # uint8 0/1
    study_px = int(study.sum())
    pp(f"[ANCHOR] CRS={anchor['crs']} size={anchor['height']}x{anchor['width']} res=({anchor['res_x']:.3f},{anchor['res_y']:.3f}) m")
    pp(f"[STUDY]  Coverage: {study_px:,}/{study.size:,} ({study_px/study.size*100:.2f}%)")

    # Context mask
    context_mask = (study == 1).astype(np.uint8)
    _save_u8(CONTEXT_MASK_TIF, context_mask, anchor)
    pp(f"[OUT] context_mask.tif → {int(context_mask.sum()):,} px")

    # Load PS
    ps = _load_ps_data_from_mat(MAT_COH_1, MAT_COORD_1)
    if ps is None:
        pp("[ERROR] Failed to load PS. Writing empty outputs.")
        coh_max = np.full((anchor["height"], anchor["width"]), np.nan, dtype=np.float32)
        ps_count = np.zeros((anchor["height"], anchor["width"]), dtype=np.uint16)
        dynamic_mask = np.zeros_like(context_mask, dtype=np.uint8)

        _save_f32(COH_GRID_TIF, coh_max, anchor, nodata=COH_NODATA)
        _save_u16(PS_COUNT_TIF, ps_count, anchor)
        _save_u8(DYNAMIC_MASK_TIF, dynamic_mask, anchor)
    else:
        coh, lon, lat = ps
        coh, x, y = _reproject_and_clip(coh, lon, lat, anchor)

        coh_max, ps_count = _vectorized_rasterize(coh, x, y, anchor)
        _save_f32(COH_GRID_TIF, coh_max, anchor, nodata=COH_NODATA)
        _save_u16(PS_COUNT_TIF, ps_count, anchor)
        pp(f"[OUT] coherence_max.tif → {COH_GRID_TIF}")
        pp(f"[OUT] ps_count.tif      → {PS_COUNT_TIF}")

        # Coherence masks (optional dilation)
        coh_filled = np.where(np.isfinite(coh_max), coh_max, -np.inf).astype(np.float32)
        coh_mask_strict = (coh_filled >= float(COH_THR_STRICT)).astype(np.uint8)
        coh_mask_loose  = (coh_filled >= float(COH_THR_LOOSE)).astype(np.uint8)

        if DILATE_PIXELS_FOR_COH > 0:
            coh_mask_strict = _dilate_u8(coh_mask_strict, DILATE_PIXELS_FOR_COH)
            coh_mask_loose  = _dilate_u8(coh_mask_loose,  DILATE_PIXELS_FOR_COH)
            pp(f"[DILATE] Applied {DILATE_PIXELS_FOR_COH} iterations to coherence masks")

        # Valid masks must require PS presence (PSC_MIN_FOR_VALID)
        psc_valid = (ps_count >= int(PSC_MIN_FOR_VALID)).astype(np.uint8)
        valid_strict = ((study == 1) & (coh_mask_strict == 1) & (psc_valid == 1)).astype(np.uint8)
        valid_loose  = ((study == 1) & (coh_mask_loose  == 1) & (psc_valid == 1)).astype(np.uint8)

        _save_u8(COH_MASK_STRICT_TIF, coh_mask_strict, anchor)
        _save_u8(COH_MASK_LOOSE_TIF,  coh_mask_loose,  anchor)
        _save_u8(VALID_MASK_STRICT_TIF, valid_strict, anchor)
        _save_u8(VALID_MASK_LOOSE_TIF,  valid_loose,  anchor)

        # Legacy copies (loose)
        _save_u8(COH_MASK_LEGACY_TIF,  coh_mask_loose, anchor)
        _save_u8(VALID_MASK_LEGACY_TIF, valid_loose,  anchor)

        # Option B dynamic mask (distance-to-nearest-PS)
        ps_present = (ps_count >= 1).astype(np.uint8)
        px_m = float(anchor["res_x"])  # 12.5
        dynamic_mask, dist_m, chosen_D = _build_dynamic_mask_optionB(ps_present, study, px_m, out_dist_tif=True)

        _save_u8(DYNAMIC_MASK_TIF, dynamic_mask, anchor)
        pp(f"[OUT] dynamic_mask.tif → {int(dynamic_mask.sum()):,} px ({dynamic_mask.sum()/dynamic_mask.size*100:.2f}%)")

    # Stats
    px_area_m2 = abs(anchor["res_x"] * anchor["res_y"])
    stats = {
        "generated_at_utc": __import__("datetime").datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "anchor": {
            "crs": str(anchor["crs"]),
            "width": int(anchor["width"]),
            "height": int(anchor["height"]),
            "res_x_m": float(anchor["res_x"]),
            "res_y_m": float(anchor["res_y"]),
        },
        "config": {
            "coh_thr_strict": float(COH_THR_STRICT),
            "coh_thr_loose": float(COH_THR_LOOSE),
            "dilate_pixels_for_coh": int(DILATE_PIXELS_FOR_COH),
            "psc_min_for_valid": int(PSC_MIN_FOR_VALID),
            "coh_nodata": float(COH_NODATA),
            "optionB": {
                "use_auto_tune": bool(USE_AUTO_TUNE),
                "target_dynamic_over_study": float(TARGET_DYNAMIC_OVER_STUDY),
                "d_max_m_fixed": float(D_MAX_M_FIXED),
                "d_max_candidates_m": [float(x) for x in D_MAX_CANDIDATES_M],
                "ps_seed_dilate_pixels": int(PS_SEED_DILATE_PIXELS),
                "ps_distance_raster": PS_DIST_M_TIF,
            }
        },
        "coverage_px": {
            "study": int(study.sum()),
        },
        "area_km2": {
            "study": float(study.sum() * (px_area_m2 / 1e6)),
        },
        "paths": {
            "coh_grid": COH_GRID_TIF,
            "ps_count": PS_COUNT_TIF,
            "coh_mask_strict": COH_MASK_STRICT_TIF,
            "coh_mask_loose": COH_MASK_LOOSE_TIF,
            "valid_mask_strict": VALID_MASK_STRICT_TIF,
            "valid_mask_loose": VALID_MASK_LOOSE_TIF,
            "coh_mask_legacy": COH_MASK_LEGACY_TIF,
            "valid_mask_legacy": VALID_MASK_LEGACY_TIF,
            "context_mask": CONTEXT_MASK_TIF,
            "dynamic_mask": DYNAMIC_MASK_TIF,
            "ps_distance_m": PS_DIST_M_TIF
        }
    }

    try:
        with rasterio.open(DYNAMIC_MASK_TIF) as dm:
            d = dm.read(1).astype(np.uint8)
        stats["coverage_px"]["dynamic"] = int(d.sum())
        stats["area_km2"]["dynamic"] = float(d.sum() * (px_area_m2 / 1e6))
        if int(study.sum()) > 0:
            stats["coverage_px"]["dynamic_over_study_pct"] = float(d.sum() / int(study.sum()) * 100.0)
    except Exception:
        pass

    with open(STATS_JSON, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2)

    pp("=== SUMMARY ===")
    pp(f"Study px   : {int(study.sum()):,}")
    if "dynamic" in stats["coverage_px"]:
        pp(f"Dynamic px : {int(stats['coverage_px']['dynamic']):,} ({stats['coverage_px'].get('dynamic_over_study_pct', 0.0):.1f}% of study)")
    pp(f"[META] {STATS_JSON}")
    pp("=== Step 0.3 Complete ===")

if __name__ == "__main__":
    main()


# Step-0.4: Make Blocks

In [ ]:
# =========================================================
# Step 0.4 — Make Blocks 
# =========================================================
import os, json
from datetime import datetime
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# ---------------- Paths ----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

ANCHOR_TIF     = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
STUDY_MASK_TIF = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

OUT_DIR        = os.path.join(BASE, "Results", "Step-0.4-Blocks")
os.makedirs(OUT_DIR, exist_ok=True)
BLOCKS_TIF     = os.path.join(OUT_DIR, "blocks_id_32645_12p5m_int16.tif")
META_JSON      = os.path.join(OUT_DIR, "blocks_metadata.json")
PNG_QUICK      = os.path.join(OUT_DIR, "blocks_quicklook.png")

# ---------------- Settings ----------------
TILE_SIZE_M    = 1000.0        # tile edge length in meters (1 km)
NODATA_INT16   = -32768        # nodata for int16 output

pp = lambda m: print(f"[{datetime.now():%H:%M:%S}] {m}", flush=True)

# ---------------- Helpers ----------------
def read_anchor(path):
    with rasterio.open(path) as ds:
        return {
            "tf": ds.transform, "crs": ds.crs,
            "w": ds.width, "h": ds.height,
            "prof": ds.profile,
            "epsg": int(ds.crs.to_epsg()) if ds.crs else None
        }

def read_mask_bool(path, anchor):
    with rasterio.open(path) as ds:
        same = (ds.crs==anchor["crs"] and ds.transform==anchor["tf"]
                and ds.width==anchor["w"] and ds.height==anchor["h"])
        if same:
            a = ds.read(1)
        else:
            a = np.zeros((anchor["h"], anchor["w"]), dtype="float64")
            reproject(ds.read(1).astype("float64"), a,
                      src_transform=ds.transform, src_crs=ds.crs,
                      dst_transform=anchor["tf"], dst_crs=anchor["crs"],
                      resampling=Resampling.nearest,
                      src_nodata=0, dst_nodata=0)
    return (a > 0)

def write_blocks(path, arr, anchor):
    prof = anchor["prof"].copy()
    prof.update(
        driver="GTiff", count=1, dtype="int16", nodata=NODATA_INT16,
        compress="LZW", tiled=True,
        transform=anchor["tf"], crs=anchor["crs"],
        width=anchor["w"], height=anchor["h"]
    )
    tmp = path + ".part"
    with rasterio.open(tmp, "w", **prof) as dst:
        dst.write(arr.astype("int16"), 1)
    if os.path.exists(path):
        try: os.remove(path)
        except: pass
    os.replace(tmp, path)

def save_quicklook(path_png, arr):
    try:
        import matplotlib.pyplot as plt
        A = arr.copy().astype(float)
        A[A == NODATA_INT16] = np.nan
        plt.figure(figsize=(7, 9), dpi=140)
        plt.imshow(A, interpolation="nearest")
        plt.title("Block IDs (1 km tiles)"); plt.axis("off")
        plt.colorbar(shrink=0.8, pad=0.02, label="Block ID")
        plt.tight_layout(); plt.savefig(path_png, bbox_inches="tight"); plt.close()
    except Exception as e:
        print(f"[WARN] Quicklook failed: {e}")

# ---------------- Main ----------------
def main():
    pp("=== Step 0.4 — Make Blocks (1 km × 1 km) ===")
    anchor = read_anchor(ANCHOR_TIF)
    tf = anchor["tf"]
    px_x, px_y = abs(tf.a), abs(tf.e)  # pixel size (m)

    if not (np.isfinite(px_x) and np.isfinite(px_y) and px_x > 0 and px_y > 0):
        raise SystemExit("[GRID] Invalid pixel size from anchor transform.")

    pp(f"[ANCHOR] EPSG:{anchor['epsg']} grid={anchor['w']}×{anchor['h']}  res≈{px_x:.4f} m × {px_y:.4f} m")

    study_mask = read_mask_bool(STUDY_MASK_TIF, anchor)
    h, w = study_mask.shape

    # Pixels per 1 km
    tile_px_x = max(1, int(round(TILE_SIZE_M / px_x)))
    tile_px_y = max(1, int(round(TILE_SIZE_M / px_y)))
    pp(f"[TILES] 1 km ≈ {tile_px_x} px (x) × {tile_px_y} px (y)")

    # Build block ID raster (IDs only inside study_mask)
    blocks = np.full((h, w), NODATA_INT16, dtype="int32")
    bid = 0
    for r0 in range(0, h, tile_px_y):
        r1 = min(h, r0 + tile_px_y)
        for c0 in range(0, w, tile_px_x):
            c1 = min(w, c0 + tile_px_x)
            tile_mask = study_mask[r0:r1, c0:c1]
            if tile_mask.any():
                bid += 1
                blocks[r0:r1, c0:c1][tile_mask] = bid

    write_blocks(BLOCKS_TIF, blocks, anchor)
    pp(f"[OUT] Blocks raster → {BLOCKS_TIF}")

    meta = {
        "generated_at_utc": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "anchor_epsg": anchor["epsg"],
        "pixel_size_m": {"x": px_x, "y": px_y},
        "tile_size_m": TILE_SIZE_M,
        "tile_size_px": {"x": tile_px_x, "y": tile_px_y},
        "grid": {"width": w, "height": h},
        "n_blocks": int(blocks[blocks != NODATA_INT16].max()) if (blocks != NODATA_INT16).any() else 0,
        "inputs": {"anchor": ANCHOR_TIF, "study_mask": STUDY_MASK_TIF},
        "output": BLOCKS_TIF,
        "nodata": NODATA_INT16
    }
    with open(META_JSON, "w") as f:
        json.dump(meta, f, indent=2)
    pp(f"[META] → {META_JSON}")

    save_quicklook(PNG_QUICK, blocks)
    pp(f"[PNG]  → {PNG_QUICK}")
    pp("=== Done (Step 0.4) ===")

if __name__ == "__main__":
    main()


# Step-0.5: SlopeUnit

In [ ]:
# ============================================================
# Step-0.5 — Slope Units 
# ============================================================
import os
import sys
from datetime import datetime
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from scipy import ndimage as ndi

# -------------------- WHITEBOX PY3.8 PATCH --------------------
if sys.version_info < (3, 9):
    import importlib.resources as _ilr
    import importlib_resources as _ilrb
    if not hasattr(_ilr, "files"):
        _ilr.files = _ilrb.files

import whitebox.whitebox_tools as wbtmod
# Bypass wrapper check
wbtmod.sys.version_info = (3, 9, 0)

from whitebox import WhiteboxTools

# -------------------- PATHS --------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

DEM_IN = os.path.join(BASE, "NTD_DEM_Clipped.tif")
ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
STUDY_MASK_TIF = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

OUT_DIR = os.path.join(BASE, "Results", "Step-0.5-SlopeUnits")
os.makedirs(OUT_DIR, exist_ok=True)

# Intermediate rasters
DEM_ALIGN  = os.path.join(OUT_DIR, "dem_aligned_float32.tif")
DEM_FILLED = os.path.join(OUT_DIR, "dem_filled.tif")
D8_PTR     = os.path.join(OUT_DIR, "d8_pointer.tif")
FAC        = os.path.join(OUT_DIR, "flow_accum_cells.tif")
STREAMS    = os.path.join(OUT_DIR, "streams.tif")
SUBBASINS  = os.path.join(OUT_DIR, "subbasins_raw.tif")

# Final
SLOPE_UNIT_TIF = os.path.join(OUT_DIR, "slope_unit_id_32645_12p5m_int32.tif")

# -------------------- SETTINGS --------------------
COMPRESS = "lzw"
NODATA_FLOAT = -9999.0
NODATA_OUT = 0

# Auto-tuning thresholds (cells). Try in order until enough units.
STREAM_THRESHOLDS_TO_TRY = [5000, 2000, 1000, 500, 200, 100, 50, 25, 10]

# Remove tiny units after segmentation (pixels on anchor grid)
MIN_UNIT_PIXELS = 200

# Require at least this many final units
MIN_UNITS_REQUIRED = 10

# -------------------- UTILS --------------------
def pp(msg): print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

def assert_exists(p, lab):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing {lab}: {p}")

def read_anchor(anchor_path):
    with rasterio.open(anchor_path) as ds:
        prof = ds.profile.copy()
        return prof, ds.crs, ds.transform, ds.width, ds.height

def read_mask_bool(mask_path, ref_crs, ref_tf, W, H):
    with rasterio.open(mask_path) as ds:
        arr = ds.read(1)
        same = (ds.crs == ref_crs and ds.transform == ref_tf and ds.width == W and ds.height == H)
        if same:
            return (arr == 1)
        out = np.zeros((H, W), dtype=np.uint8)
        reproject(
            source=arr.astype(np.uint8),
            destination=out,
            src_transform=ds.transform,
            src_crs=ds.crs,
            dst_transform=ref_tf,
            dst_crs=ref_crs,
            resampling=Resampling.nearest,
            src_nodata=0,
            dst_nodata=0,
        )
        return (out == 1)

def align_dem_to_anchor(dem_in, dem_out, ref_crs, ref_tf, W, H, study_mask):
    with rasterio.open(dem_in) as ds:
        src = ds.read(1).astype(np.float32)
        src_nd = ds.nodata
        if src_nd is not None:
            src = np.where(src == src_nd, np.nan, src)

        dst = np.full((H, W), np.nan, dtype=np.float32)

        reproject(
            source=np.where(np.isfinite(src), src, NODATA_FLOAT).astype(np.float32),
            destination=dst,
            src_transform=ds.transform,
            src_crs=ds.crs,
            dst_transform=ref_tf,
            dst_crs=ref_crs,
            resampling=Resampling.bilinear,
            src_nodata=NODATA_FLOAT,
            dst_nodata=np.nan,
        )

    dst = np.where(study_mask, dst, np.nan)

    prof = {
        "driver": "GTiff",
        "dtype": "float32",
        "count": 1,
        "width": W,
        "height": H,
        "crs": ref_crs,
        "transform": ref_tf,
        "nodata": NODATA_FLOAT,
        "compress": COMPRESS,
        "tiled": True,
    }
    tmp = dem_out + ".part"
    with rasterio.open(tmp, "w", **prof) as outds:
        outds.write(np.where(np.isfinite(dst), dst, NODATA_FLOAT).astype(np.float32), 1)
    if os.path.exists(dem_out):
        os.remove(dem_out)
    os.replace(tmp, dem_out)

def write_int32(path, arr, ref_profile, nodata=0):
    prof = ref_profile.copy()
    prof.update(driver="GTiff", dtype="int32", count=1, nodata=nodata, compress=COMPRESS, tiled=True)
    tmp = path + ".part"
    with rasterio.open(tmp, "w", **prof) as ds:
        ds.write(arr.astype(np.int32), 1)
    if os.path.exists(path):
        os.remove(path)
    os.replace(tmp, path)

def relabel_connected_components(id_map, min_pixels):
    """
    Split each ID into connected components (8-neigh),
    drop components < min_pixels, relabel to 1..N contiguous.
    """
    out = id_map.copy().astype(np.int64)
    out[out < 0] = 0
    structure = np.ones((3, 3), dtype=np.uint8)

    new = np.zeros_like(out, dtype=np.int32)
    next_id = 1

    ids = np.unique(out)
    ids = ids[ids != 0]

    for uid in ids.tolist():
        m = (out == uid)
        lab, nlab = ndi.label(m, structure=structure)
        if nlab == 0:
            continue
        sizes = np.bincount(lab.ravel())
        for c in range(1, nlab + 1):
            if sizes[c] >= int(min_pixels):
                new[lab == c] = next_id
                next_id += 1

    return new, (next_id - 1)

def count_stream_pixels(streams_path):
    with rasterio.open(streams_path) as ds:
        s = ds.read(1)
    return int((s > 0).sum())

# -------------------- MAIN --------------------
def main():
    pp("Step-0.5 starting (Slope Units)")

    assert_exists(DEM_IN, "NTD_DEM_Clipped.tif")
    assert_exists(ANCHOR_TIF, "DEM_anchor_32645.tif")
    assert_exists(STUDY_MASK_TIF, "study_mask.tif")

    ref_profile, ref_crs, ref_tf, W, H = read_anchor(ANCHOR_TIF)
    study = read_mask_bool(STUDY_MASK_TIF, ref_crs, ref_tf, W, H)
    if int(study.sum()) == 0:
        raise SystemExit("study_mask has 0 pixels on anchor grid.")

    pp("Aligning DEM to anchor grid...")
    align_dem_to_anchor(DEM_IN, DEM_ALIGN, ref_crs, ref_tf, W, H, study)

    # WhiteboxTools workspace
    wbt = WhiteboxTools()
    wbt.set_working_dir(OUT_DIR)
    wbt.verbose = False

    # Ensure binary exists (downloads if missing)
    try:
        wbt.help()
    except Exception as e:
        pp(f"WhiteboxTools binary check warning: {e}")

    pp("Filling depressions...")
    wbt.fill_depressions(DEM_ALIGN, DEM_FILLED)

    pp("Computing D8 pointer...")
    wbt.d8_pointer(DEM_FILLED, D8_PTR)

    pp("Computing flow accumulation (cells)...")
    wbt.d8_flow_accumulation(DEM_FILLED, FAC, out_type="cells")

    best_units = None
    best_n = 0
    best_thr = None

    for thr in STREAM_THRESHOLDS_TO_TRY:
        pp(f"Extracting streams (threshold={thr} cells)...")
        wbt.extract_streams(FAC, STREAMS, threshold=int(thr))

        spx = count_stream_pixels(STREAMS)
        pp(f"stream_pixels={spx}")
        if spx == 0:
            pp("No streams. Lowering threshold.")
            continue

        pp("Delineating subbasins (streams split watersheds)...")
        # Critical: subbasins uses stream raster to split basins
        wbt.subbasins(D8_PTR, STREAMS, SUBBASINS)

        with rasterio.open(SUBBASINS) as ds:
            b = ds.read(1)
            b_nd = ds.nodata
            if b_nd is not None:
                b = np.where(b == b_nd, 0, b)
            b = b.astype(np.int32)

        b = np.where(study, b, 0).astype(np.int32)
        b[b < 0] = 0

        pp("Removing tiny units + relabeling...")
        slope_units, n_units = relabel_connected_components(b, MIN_UNIT_PIXELS)

        pp(f"threshold={thr} -> units={n_units}")
        if n_units > best_n:
            best_n = n_units
            best_units = slope_units
            best_thr = thr

        if n_units >= MIN_UNITS_REQUIRED:
            pp("Enough units produced. Stopping auto-tune.")
            break

    if best_units is None or best_n < 2:
        raise SystemExit(
            "Failed to create slope units (>1). "
            "Actions: set MIN_UNIT_PIXELS=50, add smaller thresholds (5,2,1), "
            "and verify DEM has valid elevations inside study_mask."
        )

    pp(f"Best result: units={best_n} at stream_threshold={best_thr} cells")
    write_int32(SLOPE_UNIT_TIF, best_units, ref_profile, nodata=NODATA_OUT)

    pp("Step-0.5 complete")
    pp(f"Output: {SLOPE_UNIT_TIF}")

if __name__ == "__main__":
    main()


# Phase-1: DEM derivatives

# Step-1.1: Slope & Aspect

In [ ]:
# ===============================
# Phase 1.1 — Slope & Aspect
# ===============================
import os, json, numpy as np
import rasterio
import matplotlib.pyplot as plt

# -------- PATHS --------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
PHASE01 = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS")
PHASE02 = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
OUTDIR  = os.path.join(BASE, "Results", "Step-1.1-Slope_Aspect")
os.makedirs(OUTDIR, exist_ok=True)

ANCHOR_DEM = os.path.join(PHASE01, "DEM_anchor_32645.tif")
STUDY_MASK = os.path.join(PHASE02, "study_mask.tif")

SLOPE_TIF  = os.path.join(OUTDIR, "slope_deg.tif")
ASPECT_TIF = os.path.join(OUTDIR, "aspect_deg.tif")
STATS_JSON = os.path.join(OUTDIR, "slope_aspect_stats.json")
PLOT_PNG   = os.path.join(OUTDIR, "slope_aspect_hist.png")

# -------- LOAD DEM & MASK --------
if not os.path.exists(ANCHOR_DEM):
    raise FileNotFoundError(f"Anchor DEM not found: {ANCHOR_DEM}. Run Phase 0.1 first.")
if not os.path.exists(STUDY_MASK):
    raise FileNotFoundError(f"Study mask not found: {STUDY_MASK}. Run Phase 0.2 first.")

with rasterio.open(ANCHOR_DEM) as src:
    dem = src.read(1).astype("float32")
    dem_nodata = src.nodata
    transform  = src.transform
    crs        = src.crs
    prof       = src.profile
    res_x      = transform.a           # meters (UTM)
    res_y      = -transform.e          # meters (positive)

# convert NoData to NaN 
if dem_nodata is not None:
    dem = np.where(dem == dem_nodata, np.nan, dem)

with rasterio.open(STUDY_MASK) as sm:
    study_mask = sm.read(1).astype(np.uint8)

# -------- SLOPE & ASPECT (planar, UTM) --------
# gradients: dy along rows, dx along cols; spacing in meters
dy, dx = np.gradient(dem, res_y, res_x)

# slope in radians → degrees
slope_rad = np.arctan(np.hypot(dx, dy))
slope_deg = np.degrees(slope_rad).astype("float32")

# aspect: 0° = North, 90° = East, clockwise; -1 for flat (<1°)
# use atan2(dx, -dy) to rotate frame: y (north) up, x (east) right
aspect_rad = np.arctan2(dx, -dy)
aspect_deg = np.degrees(aspect_rad)

# Handle NaN values before modulo operation to avoid RuntimeWarning
valid_mask = np.isfinite(aspect_deg)
aspect_deg[valid_mask] = (aspect_deg[valid_mask] + 360.0) % 360.0
aspect_deg = aspect_deg.astype("float32")

# set flat pixels' aspect to -1
aspect_deg[slope_deg < 1.0] = -1.0

# -------- APPLY STUDY MASK (outside → NoData) --------
NODATA_FLOAT = -9999.0
slope_out  = slope_deg.copy()
aspect_out = aspect_deg.copy()

slope_out[study_mask == 0]  = NODATA_FLOAT
aspect_out[study_mask == 0] = NODATA_FLOAT

# also keep internal NaNs as NoData in output
slope_out[~np.isfinite(slope_out)]   = NODATA_FLOAT
aspect_out[~np.isfinite(aspect_out)] = NODATA_FLOAT

# -------- SAVE TIFS --------
out_prof = prof.copy()
out_prof.update(dtype="float32", count=1, nodata=NODATA_FLOAT,
                compress="LZW", tiled=True, bigtiff="IF_NEEDED")

with rasterio.open(SLOPE_TIF, "w", **out_prof) as dst:
    dst.write(slope_out, 1)

with rasterio.open(ASPECT_TIF, "w", **out_prof) as dst:
    dst.write(aspect_out, 1)

print(f"[OK] slope_deg.tif  → {SLOPE_TIF}")
print(f"[OK] aspect_deg.tif → {ASPECT_TIF}")

# -------- BASIC STATS  --------
inside = (study_mask == 1)
s_valid = np.isfinite(slope_deg) & inside
a_valid = np.isfinite(aspect_deg) & inside & (aspect_deg >= 0)  # exclude -1 flats

stats = {
    "crs_epsg": crs.to_epsg(),
    "resolution_m": {"x": float(res_x), "y": float(res_y)},
    "slope_deg": {
        "count": int(s_valid.sum()),
        "min": float(np.nanmin(slope_deg[s_valid])) if s_valid.any() else None,
        "max": float(np.nanmax(slope_deg[s_valid])) if s_valid.any() else None,
        "mean": float(np.nanmean(slope_deg[s_valid])) if s_valid.any() else None,
        "p25": float(np.nanpercentile(slope_deg[s_valid], 25)) if s_valid.any() else None,
        "p50": float(np.nanpercentile(slope_deg[s_valid], 50)) if s_valid.any() else None,
        "p75": float(np.nanpercentile(slope_deg[s_valid], 75)) if s_valid.any() else None,
        "p95": float(np.nanpercentile(slope_deg[s_valid], 95)) if s_valid.any() else None
    },
    "aspect_deg_nonflat": {
        "count": int(a_valid.sum()),
        "min": float(np.nanmin(aspect_deg[a_valid])) if a_valid.any() else None,
        "max": float(np.nanmax(aspect_deg[a_valid])) if a_valid.any() else None,
        "mean": float(np.nanmean(aspect_deg[a_valid])) if a_valid.any() else None
    },
    "flat_pixels_count": int((inside & (slope_deg < 1.0)).sum())
}
with open(STATS_JSON, "w") as f:
    json.dump(stats, f, indent=2)
print(f"[OK] stats → {STATS_JSON}")

# -------- QUICK HIST PLOT (optional) --------
try:
    fig = plt.figure(figsize=(10,4))
    ax1 = plt.subplot(1,2,1)
    ax1.hist(slope_deg[s_valid].ravel(), bins=60)
    ax1.set_title("Slope"); ax1.set_xlabel("deg"); ax1.set_ylabel("count")

    ax2 = plt.subplot(1,2,2)
    if a_valid.any():
        ax2.hist(aspect_deg[a_valid].ravel(), bins=72, range=(0,360))
    ax2.set_title("Aspect"); ax2.set_xlabel("deg (Non-flat)"); ax2.set_ylabel("count")

    plt.tight_layout()
    plt.savefig(PLOT_PNG, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] hist plot → {PLOT_PNG}")
except Exception as e:
    print("[WARN] Could not make hist plot:", e)

print("[DONE] Phase 1.1 — Slope & Aspect")

# Visualization: Slope & Aspect

In [ ]:
# ========================================================
# Step-1.1 — Visualization for Slope & Aspect 
# ========================================================
import os
import time
import stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.patches import Polygon, Rectangle, Patch
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds
from scipy import ndimage

# -------------------- PATHS -----------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
OUTDIR    = os.path.join(BASE, "Results", "Step-1.1-Slope_Aspect")
PHASE02   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")

SLOPE_TIF = os.path.join(OUTDIR, "slope_deg.tif")
ASPECT_TIF= os.path.join(OUTDIR, "aspect_deg.tif")
STUDY_MASK= os.path.join(PHASE02, "study_mask.tif")

# Optional: shapefile boundary 
STUDY_SHP = os.path.join(BASE, "Study_Area.shp")
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass

# -------------------- STYLE -----------------------
FIGSIZE               = (2, 2)
CAX_SIZE              = "5%"
CAX_PAD               = 0.01
TICK_FONTSIZE         = 3
CB_LABEL_FONTSIZE     = 6
CB_TICK_FONTSIZE      = 5
BOUNDARY_LINEWIDTH    = 0.5
BOUNDARY_COLOR        = 'red'
MASK_CONTOUR_COLOR    = 'red'
MASK_CONTOUR_WIDTH    = 0.8

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS & MINUTE TICKS
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    ax.add_patch(Polygon([apex, left_base, tail], closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))

    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))

    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))

    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))

    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader 
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feature in shp:
                    if feature['geometry'] is not None:
                        geometries.append(feature['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def plot_shapefile_on_map(ax, shapefile_data, line_color=BOUNDARY_COLOR, line_width=BOUNDARY_LINEWIDTH):
    if shapefile_data is None: 
        return
    try:
        if HAVE_GEOPANDAS and hasattr(shapefile_data, 'plot'):
            shapefile_data.boundary.plot(ax=ax, color=line_color, linewidth=line_width, zorder=12)
        elif isinstance(shapefile_data, dict) and 'geometries' in shapefile_data:
            for geom in shapefile_data['geometries']:
                gt = geom['type']
                if gt == 'Polygon':
                    coords = geom['coordinates'][0]
                    x = [pt[0] for pt in coords]; y = [pt[1] for pt in coords]
                    ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
                elif gt == 'MultiPolygon':
                    for polygon in geom['coordinates']:
                        coords = polygon[0]
                        x = [pt[0] for pt in coords]; y = [pt[1] for pt in coords]
                        ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
                elif gt in ['LineString', 'MultiLineString']:
                    if gt == 'LineString':
                        coords = geom['coordinates']
                        x = [pt[0] for pt in coords]; y = [pt[1] for pt in coords]
                        ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
                    else:
                        for line in geom['coordinates']:
                            x = [pt[0] for pt in line]; y = [pt[1] for pt in line]
                            ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
    except Exception:
        pass

# =========================================================
# LOAD DATA
# =========================================================
print("Loading slope, aspect, and mask data...")
with rasterio.open(SLOPE_TIF) as src:
    slope = src.read(1); slope_crs = src.crs; slope_nodata = src.nodata
    slope_bounds = src.bounds

with rasterio.open(ASPECT_TIF) as src:
    aspect = src.read(1); aspect_crs = src.crs; aspect_nodata = src.nodata

with rasterio.open(STUDY_MASK) as src:
    mask = src.read(1)

# Geographic extent
west, south, east, north = transform_bounds(slope_crs, "EPSG:4326", *slope_bounds)
extent = [west, east, south, north]

# Mask & nodata handling
slope_plot = np.where(slope == slope_nodata, np.nan, slope)
aspect_plot = np.where(aspect == aspect_nodata, np.nan, aspect)
if mask.shape == slope_plot.shape:
    slope_plot  = np.where(mask == 1, slope_plot,  np.nan)
    aspect_plot = np.where(mask == 1, aspect_plot, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != data shape {slope_plot.shape}")

# Optional shapefile boundary
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# =========================================================
# SLOPE MAP
# =========================================================
print("Creating slope map...")
fig1, ax1 = plt.subplots(1, 1, figsize=FIGSIZE)

# Robust upper limit for nice contrast
if np.isfinite(slope_plot).any():
    slope_vmax = float(np.nanpercentile(slope_plot, 98))
else:
    slope_vmax = 60.0

im1 = ax1.imshow(
    slope_plot, cmap='plasma', extent=extent, origin='upper',
    vmin=0, vmax=slope_vmax, zorder=1
)

# Boundary
plot_shapefile_on_map(ax1, boundary, line_color=BOUNDARY_COLOR, line_width=BOUNDARY_LINEWIDTH)
try:
    if mask.shape == slope_plot.shape:
        ax1.contour(mask, levels=[0.5], colors=MASK_CONTOUR_COLOR, linewidths=MASK_CONTOUR_WIDTH,
                    extent=extent, origin='upper', zorder=10)
except Exception as e:
    print(f"[WARN] Could not draw mask boundary on slope: {e}")

# North arrow & scale bar
add_north_arrow_tri(ax1, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax1, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south+north)/2.0,
                      segments=4, fontsize=4)

# DMS ticks with minute grid
ax1.set_xlim(west, east); ax1.set_ylim(south, north)
set_minute_ticks(ax1, west, east, south, north, dx_min=5, dy_min=10)
ax1.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax1.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax1.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax1.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax1.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Colorbar 
div1 = make_axes_locatable(ax1)
cax1 = div1.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb1  = plt.colorbar(im1, cax=cax1)
cb1.set_label('Slope (°)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb1.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

plt.tight_layout(pad=0.2)
out_png = os.path.join(OUTDIR, "Slope Map.png")
out_pdf = os.path.join(OUTDIR, "Slope Map.pdf")
safe_savefig(fig1, out_png, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig1, out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =========================================================
# ASPECT MAP
# =========================================================
print("Creating aspect map...")
fig2, ax2 = plt.subplots(1, 1, figsize=FIGSIZE)

# Aspect: -1 marks flat in your pipeline; mask to NaN for display
aspect_masked = np.where(aspect_plot == -1, np.nan, aspect_plot)
flat_areas    = np.where(aspect_plot == -1, 1.0, np.nan)  

# Use a cyclic colormap; hsv is fine, twilight is also good. Keeping hsv for consistency.
im2 = ax2.imshow(
    aspect_masked, cmap='hsv', extent=extent, origin='upper',
    vmin=0, vmax=360, zorder=1
)

# Gray overlay for flats 
if np.isfinite(flat_areas).any():
    ax2.imshow(flat_areas, cmap='gray', extent=extent, origin='upper',
               alpha=0.7, vmin=0, vmax=1, zorder=2)

# Boundary overlays
plot_shapefile_on_map(ax2, boundary, line_color=BOUNDARY_COLOR, line_width=BOUNDARY_LINEWIDTH)
try:
    if mask.shape == aspect_plot.shape:
        ax2.contour(mask, levels=[0.5], colors=MASK_CONTOUR_COLOR, linewidths=MASK_CONTOUR_WIDTH,
                    extent=extent, origin='upper', zorder=10)
except Exception as e:
    print(f"[WARN] Could not draw mask boundary on aspect: {e}")

# North arrow & scale bar
add_north_arrow_tri(ax2, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax2, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south+north)/2.0,
                      segments=4, fontsize=4)

# DMS ticks with minute grid
ax2.set_xlim(west, east); ax2.set_ylim(south, north)
set_minute_ticks(ax2, west, east, south, north, dx_min=5, dy_min=10)
ax2.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax2.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax2.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax2.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax2.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Colorbar 
div2 = make_axes_locatable(ax2)
cax2 = div2.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb2  = plt.colorbar(im2, cax=cax2)
cb2.set_label('Aspect (°)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb2.ax.tick_params(labelsize=CB_TICK_FONTSIZE)
cb2.set_ticks([0, 90, 180, 270, 360])
cb2.set_ticklabels(['N (0°)', 'E (90°)', 'S (180°)', 'W (270°)', 'N (360°)'])
plt.tight_layout(pad=0.2)
out_png = os.path.join(OUTDIR, "Aspect Map.png")
out_pdf = os.path.join(OUTDIR, "Aspect Map.pdf")
safe_savefig(fig2, out_png, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig2, out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\n[DONE] Slope & Aspect maps rendered and saved ")


In [ ]:
# ========================================================
# Step-1.1 — Visualization for Slope & Aspect 
# ========================================================
import os
import time
import stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle, Patch
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds

# -------------------- PATHS -----------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
OUTDIR    = os.path.join(BASE, "Results", "Step-1.1-Slope_Aspect")
PHASE02   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")

SLOPE_TIF  = os.path.join(OUTDIR, "slope_deg.tif")
ASPECT_TIF = os.path.join(OUTDIR, "aspect_deg.tif")
STUDY_MASK = os.path.join(PHASE02, "study_mask.tif")

# Optional shapefile boundary 
STUDY_SHP  = os.path.join(BASE, "Study_Area.shp")
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass

# -------------------- STYLE -----------------------
FIGSIZE            = (2, 2)
CAX_SIZE           = "5%"
CAX_PAD            = 0.01
TICK_FONTSIZE      = 3
CB_LABEL_FONTSIZE  = 6
CB_TICK_FONTSIZE   = 5
BOUNDARY_LINEWIDTH = 0.3
BOUNDARY_COLOR     = 'red'
MASK_CONTOUR_COLOR = 'red'
MASK_CONTOUR_WIDTH = 0.3

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS & MINUTE TICKS
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    ax.add_patch(Polygon([apex, left_base, tail], closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))

    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))

    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))

    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))

    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    # Try geopandas
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    # Try fiona 
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2):
    # 1) Try shapefile first
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0])
                        ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates'])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line)
                                ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    # 2) Fallback: mask contour
    if mask_array is not None:
        try:
            arr = mask_array
            if erode_iters and erode_iters > 0:
                from scipy.ndimage import binary_erosion
                arr = binary_erosion(arr.astype(bool), iterations=erode_iters).astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass

    return False

# =========================================================
# LOAD DATA
# =========================================================
print("Loading slope, aspect, and mask data...")
with rasterio.open(SLOPE_TIF) as src:
    slope = src.read(1); slope_crs = src.crs; slope_nodata = src.nodata
    slope_bounds = src.bounds

with rasterio.open(ASPECT_TIF) as src:
    aspect = src.read(1); aspect_crs = src.crs; aspect_nodata = src.nodata

with rasterio.open(STUDY_MASK) as src:
    mask = src.read(1)

# Geographic extent
west, south, east, north = transform_bounds(slope_crs, "EPSG:4326", *slope_bounds)
extent = [west, east, south, north]

# Mask & nodata handling
slope_plot  = np.where(slope  == slope_nodata,  np.nan, slope)
aspect_plot = np.where(aspect == aspect_nodata, np.nan, aspect)
if mask.shape == slope_plot.shape:
    slope_plot  = np.where(mask == 1, slope_plot,  np.nan)
    aspect_plot = np.where(mask == 1, aspect_plot, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != data shape {slope_plot.shape}")

# Optional shapefile boundary 
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# =========================================================
# SLOPE MAP
# =========================================================
print("Creating slope map...")
fig1, ax1 = plt.subplots(1, 1, figsize=FIGSIZE)

# Robust limit for contrast
slope_vmax = float(np.nanpercentile(slope_plot, 98)) if np.isfinite(slope_plot).any() else 60.0

im1 = ax1.imshow(
    slope_plot, cmap='plasma', extent=extent, origin='upper',
    vmin=0, vmax=slope_vmax, zorder=1
)

# Single boundary 
_ = draw_study_boundary(ax1, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2)

# North arrow & scale bar
add_north_arrow_tri(ax1, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax1, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south+north)/2.0,
                      segments=4, fontsize=4)

# DMS ticks with minute grid
ax1.set_xlim(west, east); ax1.set_ylim(south, north)
set_minute_ticks(ax1, west, east, south, north, dx_min=5, dy_min=10)
ax1.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax1.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax1.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax1.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax1.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Colorbar 
div1 = make_axes_locatable(ax1)
cax1 = div1.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb1  = plt.colorbar(im1, cax=cax1)
cb1.set_label('Slope (°)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb1.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

plt.tight_layout(pad=0.2)
slope_png = os.path.join(OUTDIR, "Slope Map.png")
slope_pdf = os.path.join(OUTDIR, "Slope Map.pdf")
safe_savefig(fig1, slope_png, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig1, slope_pdf, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =========================================================
# ASPECT MAP
# =========================================================
print("Creating aspect map...")
fig2, ax2 = plt.subplots(1, 1, figsize=FIGSIZE)

# -1 marks flat (per your pipeline); show flats in gray overlay
aspect_masked = np.where(aspect_plot == -1, np.nan, aspect_plot)
flat_areas    = np.where(aspect_plot == -1, 1.0, np.nan)  # optional overlay

im2 = ax2.imshow(
    aspect_masked, cmap='hsv', extent=extent, origin='upper',
    vmin=0, vmax=360, zorder=1
)

# Gray overlay for flats 
if np.isfinite(flat_areas).any():
    ax2.imshow(flat_areas, cmap='gray', extent=extent, origin='upper',
               alpha=0.7, vmin=0, vmax=1, zorder=2)

# Single boundary 
_ = draw_study_boundary(ax2, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2)

# North arrow & scale bar
add_north_arrow_tri(ax2, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax2, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south+north)/2.0,
                      segments=4, fontsize=4)

# DMS ticks with minute grid
ax2.set_xlim(west, east); ax2.set_ylim(south, north)
set_minute_ticks(ax2, west, east, south, north, dx_min=5, dy_min=10)
ax2.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax2.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax2.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax2.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax2.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Colorbar 
div2 = make_axes_locatable(ax2)
cax2 = div2.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb2  = plt.colorbar(im2, cax=cax2)
cb2.set_label('Aspect (°)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb2.ax.tick_params(labelsize=CB_TICK_FONTSIZE)
cb2.set_ticks([0, 90, 180, 270, 360])
cb2.set_ticklabels(['N (0°)', 'E (90°)', 'S (180°)', 'W (270°)', 'N (360°)'])

plt.tight_layout(pad=0.2)
aspect_png = os.path.join(OUTDIR, "Aspect Map.png")
aspect_pdf = os.path.join(OUTDIR, "Aspect Map.pdf")
safe_savefig(fig2, aspect_png, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig2, aspect_pdf, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\n[DONE] Slope & Aspect maps ")


In [ ]:
# ========================================================
# Step-1.1 — Visualization for Slope & Aspect 
# ========================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from rasterio.warp import transform_bounds

# -------------------- PATHS -----------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
OUTDIR    = os.path.join(BASE, "Results", "Step-1.1-Slope_Aspect")
PHASE02   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")

SLOPE_TIF  = os.path.join(OUTDIR, "slope_deg.tif")
ASPECT_TIF = os.path.join(OUTDIR, "aspect_deg.tif")
STUDY_MASK = os.path.join(PHASE02, "study_mask.tif")

# Optional shapefile boundary
STUDY_SHP  = os.path.join(BASE, "Study_Area.shp")
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass

# -------------------- PLOT STYLE -----------------------
FIGSIZE            = (1, 1)
SAVE_DPI           = 600
CAX_SIZE           = "8%"
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 3.0
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_LINEWIDTH = 0.4
BOUNDARY_COLOR     = 'black'
MASK_CONTOUR_COLOR = 'black'
MASK_CONTOUR_WIDTH = 0.4
LW_AXIS            = 0.3
XLABEL_PAD         = 0.5
YLABEL_PAD         = 0.5
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True  # EPS is fine; we write white bg

# =========================================================
# BACKGROUND MODE — FORCE WHITE (no transparency → no black)
# =========================================================
BG_MODE = "white"  # keep "white" to avoid any black from viewers

if BG_MODE == "white":
    mpl.rcParams['savefig.transparent'] = False
    mpl.rcParams['figure.facecolor'] = 'white'
    mpl.rcParams['axes.facecolor'] = 'white'
else:
    mpl.rcParams['savefig.transparent'] = True
    mpl.rcParams['figure.facecolor'] = 'none'
    mpl.rcParams['axes.facecolor'] = 'none'

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass
        leg = ax.get_legend()
        if leg is not None and leg.get_frame() is not None:
            fr = leg.get_frame()
            fr.set_facecolor('white'); fr.set_alpha(1.0)

def force_axes_bg(ax, fig=None):
    """Ensure figure/axes & colorbars match BG_MODE."""
    if fig is None: fig = ax.figure
    if BG_MODE == "white":
        fig.patch.set_facecolor('white'); fig.patch.set_alpha(1.0)
        ax.set_facecolor('white');        ax.patch.set_alpha(1.0)
        for child in fig.axes:
            try:
                child.set_facecolor('white'); child.patch.set_alpha(1.0)
            except Exception:
                pass
    else:
        fig.patch.set_facecolor('none'); fig.patch.set_alpha(0.0)
        ax.set_facecolor('none');        ax.patch.set_alpha(0.0)
        for child in fig.axes:
            try:
                child.set_facecolor('none'); child.patch.set_alpha(0.0)
            except Exception:
                pass

def save_all(fig, base_no_ext):
    if BG_MODE == "white":
        kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
        safe_savefig(fig, base_no_ext + ".png", **kw)
        if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
        if SAVE_SVG:
            kw_svg = dict(kw); kw_svg['format'] = 'svg'
            safe_savefig(fig, base_no_ext + ".svg", **kw_svg)
        if SAVE_EPS:
            scrub_alpha_for_eps(fig)
            kw_eps = dict(kw); kw_eps['format'] = 'eps'
            safe_savefig(fig, base_no_ext + ".eps", **kw_eps)
    else:
        kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='none', edgecolor='none', transparent=True)
        safe_savefig(fig, base_no_ext + ".png", **kw)
        if SAVE_PDF:  # print-friendly PDF still white
            kw_pdf = dict(kw); kw_pdf['facecolor'] = 'white'; kw_pdf['edgecolor'] = 'white'; kw_pdf['transparent'] = False
            safe_savefig(fig, base_no_ext + ".pdf", **kw_pdf)
        if SAVE_SVG:
            kw_svg = dict(kw); kw_svg['format'] = 'svg'
            safe_savefig(fig, base_no_ext + ".svg", **kw_svg)

# =========================================================
# DMS FORMATTERS & MINUTE TICKS
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg)))
    m_float = (abs(decimal_deg) - d) * 60
    m = int(np.floor(m_float))
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'N' if decimal_deg >= 0 else 'S'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'E' if decimal_deg >= 0 else 'W'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR (no pad rectangles)
# =========================================================
def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72; y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    # both halves on white canvas are fine:
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1, 2, 5, 10):
        if m*base >= target: return m*base
    return 10*base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd

    total_len_m = nice_len_125(width_m/5.0)
    seg_len_m   = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m   * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.016 * height_data

    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black' if i == 0 else 'none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2):
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0])
                        ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates'])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line)
                                ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass
    if mask_array is not None:
        try:
            arr = mask_array
            try:
                from scipy.ndimage import binary_erosion
                if erode_iters > 0:
                    arr = binary_erosion(arr.astype(bool), iterations=erode_iters).astype(float)
            except Exception:
                arr = (arr > 0).astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# LOAD DATA
# =========================================================
print("Loading slope, aspect, and mask data...")
with rasterio.open(SLOPE_TIF) as src:
    slope = src.read(1); slope_crs = src.crs; slope_nodata = src.nodata
    slope_bounds = src.bounds

with rasterio.open(ASPECT_TIF) as src:
    aspect = src.read(1); aspect_crs = src.crs; aspect_nodata = src.nodata

with rasterio.open(STUDY_MASK) as src:
    mask = src.read(1)

# Geographic extent (WGS84)
west, south, east, north = transform_bounds(slope_crs, "EPSG:4326", *slope_bounds)
extent = [west, east, south, north]

# Mask & nodata handling — set NoData/masked to WHITE (opaque)
slope_plot  = np.where(slope  == slope_nodata,  np.nan, slope)
aspect_plot = np.where(aspect == aspect_nodata, np.nan, aspect)
if mask.shape == slope_plot.shape:
    slope_plot  = np.where(mask == 1, slope_plot,  np.nan)
    aspect_plot = np.where(mask == 1, aspect_plot, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != data shape {slope_plot.shape}")

boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# =========================================================
# SLOPE MAP
# =========================================================
print("Creating slope map...")
fig1, ax1 = plt.subplots(1, 1, figsize=FIGSIZE)
for sp in ax1.spines.values(): sp.set_linewidth(LW_AXIS)

slope_vmax = float(np.nanpercentile(slope_plot, 98)) if np.isfinite(slope_plot).any() else 60.0

cmap_slope = plt.get_cmap('plasma').copy()
# WHITE for NaNs when BG_MODE=white; transparent only if you switch to "transparent"
cmap_slope.set_bad((1, 1, 1, 1) if BG_MODE == "white" else (1, 1, 1, 0))

im1 = ax1.imshow(slope_plot, cmap=cmap_slope, extent=extent, origin='upper',
                 vmin=0, vmax=slope_vmax, zorder=1)

_ = draw_study_boundary(ax1, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2)

ax1.set_xlim(west, east); ax1.set_ylim(south, north)
set_minute_ticks(ax1, west, east, south, north, dx_min=5, dy_min=10)
ax1.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax1.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax1.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
ax1.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
for lbl in ax1.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
#ax1.set_xlabel("Longitude", fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=XLABEL_PAD)
#ax1.set_ylabel("Latitude",  fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=YLABEL_PAD)

add_north_arrow_tri(ax1)
add_checker_scale_bar(ax1, west, east, south, north, crs_type='wgs84',
                      mid_lat_deg=(south+north)/2.0, segments=4)

cax1 = inset_axes(ax1, width=CAX_SIZE, height="100%", loc="lower left",
                  bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax1.transAxes, borderpad=0)
cb1  = plt.colorbar(im1, cax=cax1)
cb1.set_label('Slope (°)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
cb1.ax.set_facecolor('white' if BG_MODE == "white" else 'none')
cb1.ax.patch.set_alpha(1.0 if BG_MODE == "white" else 0.0)
cb1.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
if hasattr(cb1, "outline") and cb1.outline is not None: cb1.outline.set_linewidth(0.3)

force_axes_bg(ax1, fig1)
plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)
save_all(fig1, os.path.join(OUTDIR, "Slope Map"))
plt.show()

# =========================================================
# ASPECT MAP
# =========================================================
print("Creating aspect map...")
fig2, ax2 = plt.subplots(1, 1, figsize=FIGSIZE)
for sp in ax2.spines.values(): sp.set_linewidth(LW_AXIS)

aspect_masked = np.where(aspect_plot == -1, np.nan, aspect_plot)
flat_areas    = np.where(aspect_plot == -1, 1.0, np.nan)

cmap_aspect = plt.get_cmap('hsv').copy()
cmap_aspect.set_bad((1, 1, 1, 1) if BG_MODE == "white" else (1, 1, 1, 0))

im2 = ax2.imshow(aspect_masked, cmap=cmap_aspect, extent=extent, origin='upper',
                 vmin=0, vmax=360, zorder=1)

if np.isfinite(flat_areas).any():
    ax2.imshow(flat_areas, cmap='gray', extent=extent, origin='upper',
               alpha=1.0, vmin=0, vmax=1, zorder=2)

_ = draw_study_boundary(ax2, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2)

ax2.set_xlim(west, east); ax2.set_ylim(south, north)
set_minute_ticks(ax2, west, east, south, north, dx_min=5, dy_min=10)
ax2.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax2.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax2.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
ax2.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
for lbl in ax2.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
#ax2.set_xlabel("Longitude", fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=XLABEL_PAD)
#ax2.set_ylabel("Latitude",  fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=YLABEL_PAD)

add_north_arrow_tri(ax2)
add_checker_scale_bar(ax2, west, east, south, north, crs_type='wgs84',
                      mid_lat_deg=(south+north)/2.0, segments=4)

cax2 = inset_axes(ax2, width=CAX_SIZE, height="100%", loc="lower left",
                  bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax2.transAxes, borderpad=0)
cb2  = plt.colorbar(im2, cax=cax2)
cb2.set_label('Aspect (°)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
cb2.ax.set_facecolor('white' if BG_MODE == "white" else 'none')
cb2.ax.patch.set_alpha(1.0 if BG_MODE == "white" else 0.0)
cb2.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
cb2.set_ticks([0, 90, 180, 270, 360])
cb2.set_ticklabels(['N (0°)', 'E (90°)', 'S (180°)', 'W (270°)', 'N (360°)'])
if hasattr(cb2, "outline") and cb2.outline is not None: cb2.outline.set_linewidth(0.3)

force_axes_bg(ax2, fig2)
plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)
save_all(fig2, os.path.join(OUTDIR, "Aspect Map"))
plt.show()

print("\n[DONE] Slope & Aspect maps →", OUTDIR)


# Step-1.2: Northness / Eastness

In [ ]:
# =================================
# Phase 1.2 — Northness & Eastness
# =================================
import os, json, numpy as np
import rasterio
import matplotlib.pyplot as plt

# -------- PATHS --------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
PHASE11 = os.path.join(BASE, "Results", "Step-1.1-Slope_Aspect")
PHASE02 = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
OUTDIR  = os.path.join(BASE, "Results", "Step-1.2-Northness_Eastness")
os.makedirs(OUTDIR, exist_ok=True)

ASPECT_TIF = os.path.join(PHASE11, "aspect_deg.tif")
STUDY_MASK = os.path.join(PHASE02, "study_mask.tif")

NORTH_TIF  = os.path.join(OUTDIR, "northness.tif")
EAST_TIF   = os.path.join(OUTDIR, "eastness.tif")
STATS_JSON = os.path.join(OUTDIR, "north_east_stats.json")
PLOT_PNG   = os.path.join(OUTDIR, "north_east_hist.png")

# -------- LOAD ASPECT & MASK --------
if not os.path.exists(ASPECT_TIF):
    raise FileNotFoundError(f"Aspect raster not found: {ASPECT_TIF}. Run Phase 1.1 first.")
if not os.path.exists(STUDY_MASK):
    raise FileNotFoundError(f"Study mask not found: {STUDY_MASK}. Run Phase 0.2 first.")

with rasterio.open(ASPECT_TIF) as src:
    aspect = src.read(1).astype("float32")
    prof   = src.profile
    crs    = src.crs
    nodata_in = src.nodata

with rasterio.open(STUDY_MASK) as sm:
    study_mask = sm.read(1).astype(np.uint8)

# Convert NoData -> NaN for math
if nodata_in is not None:
    aspect = np.where(aspect == nodata_in, np.nan, aspect)

# -------- NORTHNESS & EASTNESS --------
# Convention from Phase 1.1: aspect=0° (North), 90° (East), clockwise; -1 marks flats
# Set flats to 0 (directionless) AFTER computing cos/sin by masking them explicitly
is_flat = (aspect == -1)

# radians where valid (exclude NaNs and -1)
aspect_rad = np.deg2rad(aspect)
northness = np.cos(aspect_rad).astype("float32")
eastness  = np.sin(aspect_rad).astype("float32")

# For flat pixels (aspect=-1), set to 0 (directionless)
northness[is_flat] = 0.0
eastness[is_flat]  = 0.0

# Outside study area -> NoData; keep NaNs as NoData
NODATA_FLOAT = -9999.0
north_out = northness.copy()
east_out  = eastness.copy()

north_out[study_mask == 0] = NODATA_FLOAT
east_out [study_mask == 0] = NODATA_FLOAT

north_out[~np.isfinite(north_out)] = NODATA_FLOAT
east_out[~np.isfinite(east_out)]   = NODATA_FLOAT

# -------- SAVE TIFs --------
out_prof = prof.copy()
out_prof.update(dtype="float32", count=1, nodata=NODATA_FLOAT,
                compress="LZW", tiled=True, bigtiff="IF_NEEDED")

with rasterio.open(NORTH_TIF, "w", **out_prof) as dst:
    dst.write(north_out, 1)
with rasterio.open(EAST_TIF, "w", **out_prof) as dst:
    dst.write(east_out, 1)

print(f"[OK] northness.tif → {NORTH_TIF}")
print(f"[OK] eastness.tif  → {EAST_TIF}")

# -------- STATS (inside study area) --------
inside = (study_mask == 1)
v_n = np.isfinite(northness) & inside
v_e = np.isfinite(eastness)  & inside

stats = {
    "crs_epsg": crs.to_epsg(),
    "northness": {
        "count": int(v_n.sum()),
        "min": float(np.nanmin(northness[v_n])) if v_n.any() else None,
        "max": float(np.nanmax(northness[v_n])) if v_n.any() else None,
        "mean": float(np.nanmean(northness[v_n])) if v_n.any() else None
    },
    "eastness": {
        "count": int(v_e.sum()),
        "min": float(np.nanmin(eastness[v_e])) if v_e.any() else None,
        "max": float(np.nanmax(eastness[v_e])) if v_e.any() else None,
        "mean": float(np.nanmean(eastness[v_e])) if v_e.any() else None
    },
    "notes": {
        "definition": "northness=cos(aspect_rad), eastness=sin(aspect_rad); aspect 0°=N, 90°=E; flats (-1) set to 0",
        "range_expected": "[-1,1]"
    }
}
with open(STATS_JSON, "w") as f:
    json.dump(stats, f, indent=2)
print(f"[OK] stats → {STATS_JSON}")

# -------- QUICK HIST (optional) --------
try:
    fig = plt.figure(figsize=(10,4))
    ax1 = plt.subplot(1,2,1)
    ax1.hist(northness[v_n].ravel(), bins=60, range=(-1,1))
    ax1.set_title("Northness"); ax1.set_xlabel("value"); ax1.set_ylabel("count")
    ax2 = plt.subplot(1,2,2)
    ax2.hist(eastness[v_e].ravel(), bins=60, range=(-1,1))
    ax2.set_title("Eastness"); ax2.set_xlabel("value"); ax2.set_ylabel("count")
    plt.tight_layout()
    plt.savefig(PLOT_PNG, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] hist → {PLOT_PNG}")
except Exception as e:
    print("[WARN] Could not make hist plot:", e)

print("[DONE] Phase 1.2 — Northness & Eastness")


# Visualization: Northness & Eastness

In [ ]:
# ========================================================
# Visualization for Northness & Eastness  
# ========================================================
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle, Patch
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds
from pyproj import Transformer
from scipy import ndimage

# =========================================================
# PATHS
# =========================================================
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP02 = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK = os.path.join(STEP02, "study_mask.tif")

OUTDIR = os.path.join(BASE, "Results", "Step-1.2-Northness_Eastness")
NORTH_TIF = os.path.join(OUTDIR, "northness.tif")
EAST_TIF  = os.path.join(OUTDIR, "eastness.tif")

# =========================================================
# DMS FORMATTERS
# =========================================================
def decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

# =========================================================
# NORTH ARROW & SCALE BAR
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N'):
    x0 = pad
    y0 = 1.0 - pad - size
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * 0.10, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=8, fontweight='bold',
            color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    import numpy as np
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    import numpy as np
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    import numpy as np
    width_data = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m = nice_len_125(width_m / 5.0)
    seg_len_m   = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# LOAD
# =========================================================
print("Loading northness, eastness, and mask data...")
with rasterio.open(NORTH_TIF) as src:
    north_data   = src.read(1)
    north_crs    = src.crs
    north_nodata = src.nodata
    north_bounds = src.bounds

with rasterio.open(EAST_TIF) as src:
    east_data    = src.read(1)
    east_nodata  = src.nodata

with rasterio.open(STUDY_MASK) as src:
    mask_data    = src.read(1)

# =========================================================
# PREP
# =========================================================
transformer = Transformer.from_crs(north_crs, "EPSG:4326", always_xy=True)
west, south, east, north = transform_bounds(north_crs, "EPSG:4326", *north_bounds)

north_plot = np.where(north_data == north_nodata, np.nan, north_data)
east_plot  = np.where(east_data  == east_nodata,  np.nan, east_data)

if mask_data.shape == north_data.shape:
    north_plot = np.where(mask_data == 1, north_plot, np.nan)
    east_plot  = np.where(mask_data == 1, east_plot,  np.nan)
else:
    print(f"Warning: Mask shape {mask_data.shape} != raster shape {north_data.shape}")

geographic_extent = [west, east, south, north]

# =========================================================
# NORTHNESS MAP 
# =========================================================
print("Creating northness map...")
fig1, ax1 = plt.subplots(1, 1, figsize=(2, 2))

north_im = ax1.imshow(
    north_plot, cmap='RdBu_r',
    extent=geographic_extent, origin='upper',
    vmin=-1, vmax=1
)

# Study boundary
try:
    if mask_data.shape == north_data.shape:
        ax1.contour(mask_data, levels=[0.5], colors='red', linewidths=0.8,
                    extent=geographic_extent, origin='upper', zorder=10)
        print("[OK] Boundary contour drawn on northness map")
except Exception as e:
    print(f"Warning: Could not draw boundary on northness map: {e}")

# North arrow + scale bar
add_north_arrow_tri(ax1, size=0.09, pad=0.07)
add_checker_scale_bar(ax1, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                      segments=4, fontsize=4)

# Axes formatting (match slope/aspect style)
ax1.set_xlim(west, east)
ax1.set_ylim(south, north)
ax1.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax1.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax1.tick_params(axis='x', which='major', labelsize=3, rotation=45)
ax1.tick_params(axis='y', which='major', labelsize=3, rotation=90)
for lbl in ax1.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')
    
# axis labels omitted to keep the compact style
# ax1.set_xlabel('Longitude', fontsize=6, fontweight='bold')
# ax1.set_ylabel('Latitude',  fontsize=6, fontweight='bold')

# Colorbar
div1 = make_axes_locatable(ax1)
cax1 = div1.append_axes("right", size="5%", pad=0.06)
cbar1 = plt.colorbar(north_im, cax=cax1)
cbar1.set_label('Northness', fontsize=6, fontweight='bold')
cbar1.ax.tick_params(labelsize=5)
cbar1.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar1.set_ticklabels(['South\n(-1)', '-0.5', 'E/W\n(0)', '0.5', 'North\n(1)'])

plt.tight_layout()
north_out = os.path.join(OUTDIR, "Northness_Map.png")
plt.savefig(north_out, dpi=300, bbox_inches='tight', facecolor='white')
print(f"[OK] Northness map saved: {north_out}")
plt.show()

# =========================================================
# EASTNESS MAP 
# =========================================================
print("Creating eastness map...")
fig2, ax2 = plt.subplots(1, 1, figsize=(2, 2))

east_im = ax2.imshow(
    east_plot, cmap='RdYlGn',
    extent=geographic_extent, origin='upper',
    vmin=-1, vmax=1
)

try:
    if mask_data.shape == east_data.shape:
        ax2.contour(mask_data, levels=[0.5], colors='red', linewidths=0.8,
                    extent=geographic_extent, origin='upper', zorder=10)
        print("[OK] Boundary contour drawn on eastness map")
except Exception as e:
    print(f"Warning: Could not draw boundary on eastness map: {e}")

add_north_arrow_tri(ax2, size=0.09, pad=0.07)
add_checker_scale_bar(ax2, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                      segments=4, fontsize=4)

ax2.set_xlim(west, east)
ax2.set_ylim(south, north)
ax2.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax2.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax2.tick_params(axis='x', which='major', labelsize=3, rotation=45)
ax2.tick_params(axis='y', which='major', labelsize=3, rotation=90)
for lbl in ax2.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')
    
# ax2.set_xlabel('Longitude', fontsize=6, fontweight='bold')
# ax2.set_ylabel('Latitude',  fontsize=6, fontweight='bold')

div2 = make_axes_locatable(ax2)
cax2 = div2.append_axes("right", size="5%", pad=0.06)
cbar2 = plt.colorbar(east_im, cax=cax2)
cbar2.set_label('Eastness', fontsize=6, fontweight='bold')
cbar2.ax.tick_params(labelsize=5)
cbar2.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar2.set_ticklabels(['West\n(-1)', '-0.5', 'N/S\n(0)', '0.5', 'East\n(1)'])

plt.tight_layout()
east_out = os.path.join(OUTDIR, "Eastness_Map.png")
plt.savefig(east_out, dpi=300, bbox_inches='tight', facecolor='white')
print(f"[OK] Eastness map saved: {east_out}")
plt.show()

# =========================================================
# SUMMARY (optional)
# =========================================================
north_valid = north_plot[~np.isnan(north_plot)]
east_valid  = east_plot[~np.isnan(east_plot)]

print("\n" + "="*50)
print("NORTHNESS AND EASTNESS SUMMARY")
print("="*50)
print(f"CRS: {north_crs}")
print(f"BBox (WGS84): W={format_dms_lon(west)}, E={format_dms_lon(east)}, "
      f"S={format_dms_lat(south)}, N={format_dms_lat(north)}")
print(f"Northness: min={np.nanmin(north_valid):.3f}, max={np.nanmax(north_valid):.3f}, "
      f"mean={np.nanmean(north_valid):.3f}, std={np.nanstd(north_valid):.3f}")
print(f"Eastness:  min={np.nanmin(east_valid):.3f}, max={np.nanmax(east_valid):.3f}, "
      f"mean={np.nanmean(east_valid):.3f}, std={np.nanstd(east_valid):.3f}")
print("="*50, "\nVisualization complete!")


In [ ]:
# ========================================================
# Step-1.2 — Visualization for Northness & Eastness 
# ========================================================
import os, time, stat, numpy as np, rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from rasterio.warp import transform_bounds
from matplotlib import colors as mcolors

# ---------------- Optional Shapefile ----------------
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass

# ---------------- PATHS ----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP02      = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK  = os.path.join(STEP02, "study_mask.tif")
STUDY_SHP   = os.path.join(BASE, "Study_Area.shp")

OUTDIR      = os.path.join(BASE, "Results", "Step-1.2-Northness_Eastness")
NORTH_TIF   = os.path.join(OUTDIR, "northness.tif")
EAST_TIF    = os.path.join(OUTDIR, "eastness.tif")

# ---------------- PLOTTING STYLE  ----------------
FIGSIZE            = (1, 1)          
SAVE_DPI           = 600
CAX_SIZE           = "8%"             
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 3.0
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_LINEWIDTH = 0.4
BOUNDARY_COLOR     = 'black'
MASK_CONTOUR_COLOR = 'black'
MASK_CONTOUR_WIDTH = 0.4
LW_AXIS            = 0.3
XLABEL_PAD         = 0.5
YLABEL_PAD         = 0.5
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True   

# ---------------- White, opaque canvas  ----------------
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# ---------------- Safe save helpers  ----------------
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            os.chmod(path, stat.S_IWRITE)
    except Exception: pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def force_axes_bg_opaque(ax, fig=None):
    if fig is None: fig = ax.figure
    fig.patch.set_facecolor('white'); fig.patch.set_alpha(1.0)
    ax.set_facecolor('white'); ax.patch.set_alpha(1.0)
    for child in fig.axes:
        try:
            child.set_facecolor('white'); child.patch.set_alpha(1.0)
        except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# ---------------- DMS formatters & Minute ticks ----------------
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg)))
    m_float = (abs(decimal_deg) - d) * 60
    m = int(np.floor(m_float))
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'N' if decimal_deg >= 0 else 'S'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'E' if decimal_deg >= 0 else 'W'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min / 60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# ---------------- North arrow & checkerboard scale bar  ----------------
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72; y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# ---------------- Shapefile loader + boundary drawer  ----------------
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path): return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}": gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception: pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception: pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2):
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0])
                        ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception: pass
    if mask_array is not None:
        try:
            from scipy.ndimage import binary_erosion
            arr = (mask_array > 0).astype(float)
            if erode_iters > 0:
                arr = binary_erosion(arr.astype(bool), iterations=erode_iters).astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception: pass
    return False

# ---------------- LOAD RASTERS ----------------
print("Loading northness, eastness, and mask data...")
with rasterio.open(NORTH_TIF) as src:
    north = src.read(1); north_crs = src.crs; north_nodata = src.nodata
    north_bounds = src.bounds
with rasterio.open(EAST_TIF) as src:
    east = src.read(1); east_nodata = src.nodata
with rasterio.open(STUDY_MASK) as src:
    mask = src.read(1)

# Extent in WGS84
west, south, east_lon, north_lat = transform_bounds(north_crs, "EPSG:4326", *north_bounds)
extent = [west, east_lon, south, north_lat]

# Mask & NoData
north_plot = np.where(north == north_nodata, np.nan, north)
east_plot  = np.where(east  == east_nodata,  np.nan, east)
if mask.shape == north_plot.shape:
    north_plot = np.where(mask == 1, north_plot, np.nan)
    east_plot  = np.where(mask == 1, east_plot,  np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != data shape {north_plot.shape}")

# Optional boundary
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# Symmetric diverging normalization
norm_01 = mcolors.TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)

# =========================================================
# NORTHNESS MAP
# =========================================================
print("Creating Northness map...")
fig1, ax1 = plt.subplots(1, 1, figsize=FIGSIZE)
for sp in ax1.spines.values(): sp.set_linewidth(LW_AXIS)

# Cmap with white for NaNs 
cmap_n = plt.get_cmap('RdBu_r').copy()
cmap_n.set_bad((1,1,1,1))

im1 = ax1.imshow(north_plot, cmap=cmap_n, norm=norm_01,
                 extent=extent, origin='upper', zorder=1)

_ = draw_study_boundary(ax1, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

# DMS ticks + labels
ax1.set_xlim(west, east_lon); ax1.set_ylim(south, north_lat)
set_minute_ticks(ax1, west, east_lon, south, north_lat, dx_min=5, dy_min=10)
ax1.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax1.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax1.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
ax1.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
for lbl in ax1.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
#ax1.set_xlabel("Longitude", fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=XLABEL_PAD)
#ax1.set_ylabel("Latitude",  fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=YLABEL_PAD)

# Cartography
add_north_arrow_tri(ax1)
add_checker_scale_bar(ax1, west, east_lon, south, north_lat, crs_type='wgs84', segments=4)

# Right inset colorbar
cax1 = inset_axes(ax1, width=CAX_SIZE, height="100%", loc="lower left",
                  bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax1.transAxes, borderpad=0)
cb1  = plt.colorbar(im1, cax=cax1)
cb1.set_label('Northness', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
cb1.ax.set_facecolor('white'); cb1.ax.patch.set_alpha(1.0)
cb1.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
cb1.set_ticks([-1, -0.5, 0, 0.5, 1])
cb1.set_ticklabels(['South\n(-1)', '-0.5', 'E/W\n(0)', '0.5', 'North\n(1)'])
if hasattr(cb1, "outline") and cb1.outline is not None: cb1.outline.set_linewidth(0.3)

force_axes_bg_opaque(ax1, fig1)
plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)
save_all(fig1, os.path.join(OUTDIR, "Northness Map"))
plt.show()

# =========================================================
# EASTNESS MAP
# =========================================================
print("Creating Eastness map...")
fig2, ax2 = plt.subplots(1, 1, figsize=FIGSIZE)
for sp in ax2.spines.values(): sp.set_linewidth(LW_AXIS)

cmap_e = plt.get_cmap('PiYG').copy()
cmap_e.set_bad((1,1,1,1))

im2 = ax2.imshow(east_plot, cmap=cmap_e, norm=norm_01,
                 extent=extent, origin='upper', zorder=1)

_ = draw_study_boundary(ax2, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

ax2.set_xlim(west, east_lon); ax2.set_ylim(south, north_lat)
set_minute_ticks(ax2, west, east_lon, south, north_lat, dx_min=5, dy_min=10)
ax2.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax2.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax2.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
ax2.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
for lbl in ax2.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
#ax2.set_xlabel("Longitude", fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=XLABEL_PAD)
#ax2.set_ylabel("Latitude",  fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=YLABEL_PAD)

add_north_arrow_tri(ax2)
add_checker_scale_bar(ax2, west, east_lon, south, north_lat, crs_type='wgs84', segments=4)

cax2 = inset_axes(ax2, width=CAX_SIZE, height="100%", loc="lower left",
                  bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax2.transAxes, borderpad=0)
cb2  = plt.colorbar(im2, cax=cax2)
cb2.set_label('Eastness', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
cb2.ax.set_facecolor('white'); cb2.ax.patch.set_alpha(1.0)
cb2.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
cb2.set_ticks([-1, -0.5, 0, 0.5, 1])
cb2.set_ticklabels(['West\n(-1)', '-0.5', 'N/S\n(0)', '0.5', 'East\n(1)'])
if hasattr(cb2, "outline") and cb2.outline is not None: cb2.outline.set_linewidth(0.3)

force_axes_bg_opaque(ax2, fig2)
plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)
save_all(fig2, os.path.join(OUTDIR, "Eastness Map"))
plt.show()

print("\n[DONE] Northness & Eastness maps →", OUTDIR)


# Step-1.3: Roughness

In [ ]:
# ===============================
# Phase 1.3 — Roughness (TRI)
# ===============================
import os, json, numpy as np
import rasterio
from scipy.ndimage import generic_filter
import matplotlib.pyplot as plt

# -------- PATHS --------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
PHASE01 = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS")
PHASE02 = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
OUTDIR  = os.path.join(BASE, "Results", "Step-1.3-Roughness")
os.makedirs(OUTDIR, exist_ok=True)

ANCHOR_DEM = os.path.join(PHASE01, "DEM_anchor_32645.tif")
STUDY_MASK = os.path.join(PHASE02, "study_mask.tif")

TRI_STD_TIF   = os.path.join(OUTDIR, "roughness_tri_std.tif")
TRI_RANGE_TIF = os.path.join(OUTDIR, "roughness_tri_range.tif")
ROUGH_TIF     = os.path.join(OUTDIR, "roughness.tif")  # main product (alias of TRI_STD)
STATS_JSON    = os.path.join(OUTDIR, "roughness_stats.json")
PLOT_PNG      = os.path.join(OUTDIR, "roughness_hist.png")

# -------- CONFIG --------
KERNEL = 3         # window size (odd): 3, 5, 7 ...
MIN_VALID = 3      # minimum valid pixels in window to compute a value
NODATA = -9999.0   # output nodata (float32)

# -------- LOAD DEM & MASK --------
if not os.path.exists(ANCHOR_DEM):
    raise FileNotFoundError(f"Anchor DEM not found: {ANCHOR_DEM} (run Phase 0.1).")
if not os.path.exists(STUDY_MASK):
    raise FileNotFoundError(f"Study mask not found: {STUDY_MASK} (run Phase 0.2).")

with rasterio.open(ANCHOR_DEM) as src:
    dem = src.read(1).astype("float32")
    dem_nodata = src.nodata
    prof = src.profile
    crs  = src.crs

with rasterio.open(STUDY_MASK) as sm:
    study_mask = sm.read(1).astype(np.uint8)

# Convert NoData to NaN for filtering
if dem_nodata is not None:
    dem = np.where(dem == dem_nodata, np.nan, dem)

# -------- HELPERS (NaN-aware filters) --------
def _tri_std(win):
    w = win[np.isfinite(win)]
    if w.size >= MIN_VALID:
        return np.float32(np.std(w))
    return np.float32(np.nan)

def _tri_range(win):
    w = win[np.isfinite(win)]
    if w.size >= MIN_VALID:
        return np.float32(np.nanmax(w) - np.nanmin(w))
    return np.float32(np.nan)

# -------- COMPUTE ROUGHNESS --------
print(f"Computing TRI_STD with {KERNEL}x{KERNEL} window...")
tri_std = generic_filter(
    dem, function=_tri_std, size=KERNEL,
    mode="constant", cval=np.nan
).astype("float32")

print(f"Computing TRI_RANGE with {KERNEL}x{KERNEL} window...")
tri_range = generic_filter(
    dem, function=_tri_range, size=KERNEL,
    mode="constant", cval=np.nan
).astype("float32")

# -------- APPLY STUDY MASK & NODATA --------
def mask_out(arr):
    out = arr.copy()
    out[study_mask == 0] = np.nan
    out[~np.isfinite(out)] = NODATA
    return out

tri_std_out   = mask_out(tri_std)
tri_range_out = mask_out(tri_range)

# -------- SAVE TIFs --------
out_prof = prof.copy()
out_prof.update(dtype="float32", count=1, nodata=NODATA,
                compress="LZW", tiled=True, bigtiff="IF_NEEDED")

with rasterio.open(TRI_STD_TIF, "w", **out_prof) as dst:
    dst.write(tri_std_out, 1)
with rasterio.open(TRI_RANGE_TIF, "w", **out_prof) as dst:
    dst.write(tri_range_out, 1)

# Main product alias
with rasterio.open(ROUGH_TIF, "w", **out_prof) as dst:
    dst.write(tri_std_out, 1)

print(f"[OK] roughness_tri_std.tif   → {TRI_STD_TIF}")
print(f"[OK] roughness_tri_range.tif → {TRI_RANGE_TIF}")
print(f"[OK] roughness.tif (alias of TRI_STD) → {ROUGH_TIF}")

# -------- STATS (inside study area) --------
inside = (study_mask == 1)
v_std   = np.isfinite(tri_std)   & inside
v_range = np.isfinite(tri_range) & inside

stats = {
    "crs_epsg": crs.to_epsg(),
    "window_size": KERNEL,
    "metrics": {
        "TRI_STD": {
            "count": int(v_std.sum()),
            "min": float(np.nanmin(tri_std[v_std])) if v_std.any() else None,
            "max": float(np.nanmax(tri_std[v_std])) if v_std.any() else None,
            "mean": float(np.nanmean(tri_std[v_std])) if v_std.any() else None,
            "p25": float(np.nanpercentile(tri_std[v_std], 25)) if v_std.any() else None,
            "p50": float(np.nanpercentile(tri_std[v_std], 50)) if v_std.any() else None,
            "p75": float(np.nanpercentile(tri_std[v_std], 75)) if v_std.any() else None,
            "p95": float(np.nanpercentile(tri_std[v_std], 95)) if v_std.any() else None
        },
        "TRI_RANGE": {
            "count": int(v_range.sum()),
            "min": float(np.nanmin(tri_range[v_range])) if v_range.any() else None,
            "max": float(np.nanmax(tri_range[v_range])) if v_range.any() else None,
            "mean": float(np.nanmean(tri_range[v_range])) if v_range.any() else None,
            "p25": float(np.nanpercentile(tri_range[v_range], 25)) if v_range.any() else None,
            "p50": float(np.nanpercentile(tri_range[v_range], 50)) if v_range.any() else None,
            "p75": float(np.nanpercentile(tri_range[v_range], 75)) if v_range.any() else None,
            "p95": float(np.nanpercentile(tri_range[v_range], 95)) if v_range.any() else None
        }
    },
    "notes": {
        "TRI_STD": "Std. dev. of elevation in kxk window (meters).",
        "TRI_RANGE": "Max-minus-min elevation in kxk window (meters).",
        "roughness_main": "roughness.tif is TRI_STD."
    }
}
with open(STATS_JSON, "w") as f:
    json.dump(stats, f, indent=2)
print(f"[OK] stats → {STATS_JSON}")

# -------- OPTIONAL HIST PLOT --------
try:
    fig = plt.figure(figsize=(10,4))
    ax1 = plt.subplot(1,2,1)
    ax1.hist(tri_std[v_std].ravel(), bins=60)
    ax1.set_title(f"TRI_STD (k={KERNEL})")
    ax1.set_xlabel("meters"); ax1.set_ylabel("count")

    ax2 = plt.subplot(1,2,2)
    ax2.hist(tri_range[v_range].ravel(), bins=60)
    ax2.set_title(f"TRI_RANGE (k={KERNEL})")
    ax2.set_xlabel("meters"); ax2.set_ylabel("count")

    plt.tight_layout()
    plt.savefig(PLOT_PNG, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] hist → {PLOT_PNG}")
except Exception as e:
    print("[WARN] Could not make hist plot:", e)

print("[DONE] Phase 1.3 — Roughness")


# Visualization: Roughness

In [ ]:
# ========================================================
# Step-1.3 — Visualization for Terrain Roughness Index (TRI)
# ========================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds
from matplotlib import colors as mcolors

# Optional shapefile 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass

# =========================================================
# PATHS
# =========================================================
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP_MASK   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK  = os.path.join(STEP_MASK, "study_mask.tif")
STUDY_SHP   = os.path.join(BASE, "Study_Area.shp")

OUTDIR      = os.path.join(BASE, "Results", "Step-1.3-Roughness")
TRI_STD_TIF   = os.path.join(OUTDIR, "roughness_tri_std.tif")
TRI_RANGE_TIF = os.path.join(OUTDIR, "roughness_tri_range.tif")

TRI_STD_PNG   = os.path.join(OUTDIR, "TRI_Standard_Deviation_Map.png")
TRI_STD_PDF   = os.path.join(OUTDIR, "TRI_Standard_Deviation_Map.pdf")
TRI_RNG_PNG   = os.path.join(OUTDIR, "TRI_Range_Map.png")
TRI_RNG_PDF   = os.path.join(OUTDIR, "TRI_Range_Map.pdf")

# =========================================================
# STYLE
# =========================================================
FIGSIZE           = (2, 2)
CAX_SIZE          = "5%"
CAX_PAD           = 0.01
TICK_FONTSIZE     = 3
CB_LABEL_FONTSIZE = 6
CB_TICK_FONTSIZE  = 5
BOUNDARY_COLOR    = 'red'   
BOUNDARY_LW       = 0.3
MASK_COLOR        = 'red'     
MASK_LW           = 0.3

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS & MINUTE TICKS
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + unified boundary drawer 
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw exactly ONE boundary: shapefile if available, else mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0])
                        ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates'])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line)
                                ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            from scipy.ndimage import binary_erosion
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# LOAD RASTERS & MASK
# =========================================================
print("Loading TRI rasters and mask...")
with rasterio.open(TRI_STD_TIF) as src:
    tri_std        = src.read(1)
    tri_std_crs    = src.crs
    tri_std_nodata = src.nodata
    tri_std_bounds = src.bounds

with rasterio.open(TRI_RANGE_TIF) as src:
    tri_rng        = src.read(1)
    tri_rng_nodata = src.nodata

with rasterio.open(STUDY_MASK) as src:
    mask           = src.read(1)

# Extent in WGS84
west, south, east, north = transform_bounds(tri_std_crs, "EPSG:4326", *tri_std_bounds)
extent = [west, east, south, north]

# Prepare arrays (apply nodata + mask)
tri_std_plot = np.where(tri_std == tri_std_nodata, np.nan, tri_std)
tri_rng_plot = np.where(tri_rng == tri_rng_nodata, np.nan, tri_rng)

if mask.shape == tri_std.shape:
    tri_std_plot = np.where(mask == 1, tri_std_plot, np.nan)
    tri_rng_plot = np.where(mask == 1, tri_rng_plot, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != TRI shape {tri_std.shape}; mask overlay may be skipped.")

# Robust display limits (98th percentile)
std_valid = tri_std_plot[~np.isnan(tri_std_plot)]
rng_valid = tri_rng_plot[~np.isnan(tri_rng_plot)]
vmax_std  = np.nanpercentile(std_valid, 98) if std_valid.size else 1.0
vmax_rng  = np.nanpercentile(rng_valid, 98) if rng_valid.size else 1.0

# Optional shapefile boundary
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# =========================================================
# TRI Standard Deviation
# =========================================================
print("Creating TRI Standard Deviation map...")
fig1, ax1 = plt.subplots(1, 1, figsize=FIGSIZE)
im1 = ax1.imshow(tri_std_plot, cmap='viridis', extent=extent, origin='upper',
                 vmin=0, vmax=vmax_std, zorder=1)

# Single boundary
_ = draw_study_boundary(ax1, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

# North arrow & scale bar
add_north_arrow_tri(ax1, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax1, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south+north)/2.0,
                      segments=4, fontsize=4)

# Axes: DMS ticks
ax1.set_xlim(west, east); ax1.set_ylim(south, north)
set_minute_ticks(ax1, west, east, south, north, dx_min=5, dy_min=10)
ax1.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax1.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax1.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax1.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax1.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Colorbar
div1 = make_axes_locatable(ax1)
cax1 = div1.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb1  = plt.colorbar(im1, cax=cax1)
cb1.set_label('TRI Std Dev (m)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb1.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

plt.tight_layout(pad=0.2)
safe_savefig(fig1, TRI_STD_PNG, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig1, TRI_STD_PDF, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =========================================================
# TRI Range
# =========================================================
print("Creating TRI Range map...")
fig2, ax2 = plt.subplots(1, 1, figsize=FIGSIZE)
im2 = ax2.imshow(tri_rng_plot, cmap='magma', extent=extent, origin='upper',
                 vmin=0, vmax=vmax_rng, zorder=1)
im2.set_clim(0, 100)

# Single boundary
_ = draw_study_boundary(ax2, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

# North arrow & scale bar
add_north_arrow_tri(ax2, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax2, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south+north)/2.0,
                      segments=4, fontsize=4)

# Axes: DMS ticks
ax2.set_xlim(west, east); ax2.set_ylim(south, north)
set_minute_ticks(ax2, west, east, south, north, dx_min=5, dy_min=10)
ax2.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax2.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax2.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax2.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax2.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Colorbar
div2 = make_axes_locatable(ax2)
cax2 = div2.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb2  = plt.colorbar(im2, cax=cax2)
cb2.set_label('TRI Range (m)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb2.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

plt.tight_layout(pad=0.2)
safe_savefig(fig2, TRI_RNG_PNG, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig2, TRI_RNG_PDF, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\n[DONE] TRI StdDev & Range maps ")


In [ ]:
# ========================================================
# Step-1.3 — Visualization for Terrain Roughness Index (TRI)
# ========================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from rasterio.warp import transform_bounds

# ---------------- Optional shapefile ----------------
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass

# ---------------- PATHS ----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP_MASK   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK  = os.path.join(STEP_MASK, "study_mask.tif")
STUDY_SHP   = os.path.join(BASE, "Study_Area.shp")

OUTDIR        = os.path.join(BASE, "Results", "Step-1.3-Roughness")
TRI_STD_TIF   = os.path.join(OUTDIR, "roughness_tri_std.tif")
TRI_RANGE_TIF = os.path.join(OUTDIR, "roughness_tri_range.tif")

# ---------------- PLOT STYLE ----------------
FIGSIZE            = (1, 1)
SAVE_DPI           = 600
CAX_SIZE           = "8%"   
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 3.0
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_COLOR     = 'black'
BOUNDARY_LINEWIDTH = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_CONTOUR_WIDTH = 0.4
LW_AXIS            = 0.3
XLABEL_PAD         = 0.5
YLABEL_PAD         = 0.5
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True  

# lock TRI Range upper limit
TRI_RANGE_FIXED_MAX = None  # e.g., 100.0 to force 0–100

# ---------------- White, opaque canvas (no black) ----------------
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# ---------------- Safe save helpers ----------------
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            os.chmod(path, stat.S_IWRITE)
    except Exception: pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs); print(f"[OUTPUT] Saved: {path}"); return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs); print(f"[OUTPUT] Saved: {alt}"); return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def force_axes_bg_opaque(ax, fig=None):
    if fig is None: fig = ax.figure
    fig.patch.set_facecolor('white'); fig.patch.set_alpha(1.0)
    ax.set_facecolor('white'); ax.patch.set_alpha(1.0)
    for child in fig.axes:
        try:
            child.set_facecolor('white'); child.patch.set_alpha(1.0)
        except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# ---------------- DMS formatters & minute ticks ----------------
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg)))
    m_float = (abs(decimal_deg) - d) * 60
    m = int(np.floor(m_float))
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'N' if decimal_deg >= 0 else 'S'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'E' if decimal_deg >= 0 else 'W'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min / 60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# ---------------- North arrow & checkerboard scale bar ----------------
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72; y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# ---------------- Shapefile loader + boundary drawer ----------------
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path): return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}": gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception: pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception: pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2):
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0])
                        ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception: pass
    if mask_array is not None:
        try:
            from scipy.ndimage import binary_erosion
            arr = (mask_array > 0).astype(float)
            if erode_iters > 0:
                arr = binary_erosion(arr.astype(bool), iterations=erode_iters).astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception: pass
    return False

# ---------------- LOAD DATA ----------------
print("Loading TRI rasters and mask...")
with rasterio.open(TRI_STD_TIF) as src:
    tri_std        = src.read(1)
    tri_std_crs    = src.crs
    tri_std_nodata = src.nodata
    tri_std_bounds = src.bounds

with rasterio.open(TRI_RANGE_TIF) as src:
    tri_rng        = src.read(1)
    tri_rng_nodata = src.nodata

with rasterio.open(STUDY_MASK) as src:
    mask = src.read(1)

# Extent in WGS84
west, south, east, north = transform_bounds(tri_std_crs, "EPSG:4326", *tri_std_bounds)
extent = [west, east, south, north]

# Mask & NoData
tri_std_plot = np.where(tri_std == tri_std_nodata, np.nan, tri_std)
tri_rng_plot = np.where(tri_rng == tri_rng_nodata, np.nan, tri_rng)
if mask.shape == tri_std_plot.shape:
    tri_std_plot = np.where(mask == 1, tri_std_plot, np.nan)
    tri_rng_plot = np.where(mask == 1, tri_rng_plot, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != TRI shape {tri_std_plot.shape}")

# Robust display limits (98th percentile)
std_valid = tri_std_plot[np.isfinite(tri_std_plot)]
rng_valid = tri_rng_plot[np.isfinite(tri_rng_plot)]
vmax_std  = float(np.percentile(std_valid, 98)) if std_valid.size else 1.0
if TRI_RANGE_FIXED_MAX is None:
    vmax_rng = float(np.percentile(rng_valid, 98)) if rng_valid.size else 1.0
else:
    vmax_rng = float(TRI_RANGE_FIXED_MAX)

# =========================================================
# TRI Standard Deviation (viridis)
# =========================================================
print("Creating TRI Standard Deviation map...")
fig1, ax1 = plt.subplots(1, 1, figsize=FIGSIZE)
for sp in ax1.spines.values(): sp.set_linewidth(LW_AXIS)

cmap_std = plt.get_cmap('viridis').copy()
cmap_std.set_bad((1,1,1,1))  # NaNs render as white

im1 = ax1.imshow(tri_std_plot, cmap=cmap_std, extent=extent, origin='upper',
                 vmin=0, vmax=vmax_std, zorder=1)

_ = draw_study_boundary(ax1, extent, load_shapefile_boundary(STUDY_SHP, 4326), mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

# DMS ticks + labels
ax1.set_xlim(west, east); ax1.set_ylim(south, north)
set_minute_ticks(ax1, west, east, south, north, dx_min=5, dy_min=10)
ax1.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax1.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax1.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
ax1.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
for lbl in ax1.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
ax1.set_xlabel("Longitude", fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=XLABEL_PAD)
ax1.set_ylabel("Latitude",  fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=YLABEL_PAD)

# Cartography
add_north_arrow_tri(ax1)
add_checker_scale_bar(ax1, west, east, south, north, crs_type='wgs84', segments=4)

# Right inset colorbar
cax1 = inset_axes(ax1, width=CAX_SIZE, height="100%", loc="lower left",
                  bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax1.transAxes, borderpad=0)
cb1  = plt.colorbar(im1, cax=cax1)
cb1.set_label('TRI Std Dev (m)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
cb1.ax.set_facecolor('white'); cb1.ax.patch.set_alpha(1.0)
cb1.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
if hasattr(cb1, "outline") and cb1.outline is not None: cb1.outline.set_linewidth(0.3)

force_axes_bg_opaque(ax1, fig1)
plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)
save_all(fig1, os.path.join(OUTDIR, "TRI_Standard_Deviation_Map"))
plt.show()

# =========================================================
# TRI Range 
# =========================================================
print("Creating TRI Range map...")
fig2, ax2 = plt.subplots(1, 1, figsize=FIGSIZE)
for sp in ax2.spines.values(): sp.set_linewidth(LW_AXIS)

cmap_rng = plt.get_cmap('magma').copy()
cmap_rng.set_bad((1,1,1,1))  

im2 = ax2.imshow(tri_rng_plot, cmap=cmap_rng, extent=extent, origin='upper',
                 vmin=0, vmax=vmax_rng, zorder=1)

_ = draw_study_boundary(ax2, extent, load_shapefile_boundary(STUDY_SHP, 4326), mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

ax2.set_xlim(west, east); ax2.set_ylim(south, north)
set_minute_ticks(ax2, west, east, south, north, dx_min=5, dy_min=10)
ax2.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax2.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax2.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45) 
ax2.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
for lbl in ax2.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
ax2.set_xlabel("Longitude", fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=XLABEL_PAD)
ax2.set_ylabel("Latitude",  fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=YLABEL_PAD)

add_north_arrow_tri(ax2)
add_checker_scale_bar(ax2, west, east, south, north, crs_type='wgs84', segments=4)

cax2 = inset_axes(ax2, width=CAX_SIZE, height="100%", loc="lower left",
                  bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax2.transAxes, borderpad=0)
cb2  = plt.colorbar(im2, cax=cax2)
cb2.set_label('TRI Range (m)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
cb2.ax.set_facecolor('white'); cb2.ax.patch.set_alpha(1.0)
cb2.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
if hasattr(cb2, "outline") and cb2.outline is not None: cb2.outline.set_linewidth(0.3)

force_axes_bg_opaque(ax2, fig2)
plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)
save_all(fig2, os.path.join(OUTDIR, "TRI_Range_Map"))
plt.show()

print("\n[DONE] TRI StdDev & Range maps →", OUTDIR)


# Step-1.3: Roughness QA add-on

In [ ]:
# ===============================
# Phase 1.3 — Roughness QA add-on
# ===============================
import os, json, numpy as np
import rasterio
import matplotlib.pyplot as plt

# -------- PATHS --------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
R_DIR = os.path.join(BASE, "Results", "Step-1.3-Roughness")
S_DIR = os.path.join(BASE, "Results", "Step-1.1-Slope_Aspect")
M_DIR = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
QA_DIR = os.path.join(R_DIR, "QA")
os.makedirs(QA_DIR, exist_ok=True)

TRI_STD_TIF   = os.path.join(R_DIR, "roughness_tri_std.tif")
TRI_RANGE_TIF = os.path.join(R_DIR, "roughness_tri_range.tif")
SLOPE_TIF     = os.path.join(S_DIR, "slope_deg.tif")
STUDY_MASK_TIF= os.path.join(M_DIR, "study_mask.tif")

P99_STD_MASK_TIF   = os.path.join(QA_DIR, "tri_std_p99_mask.tif")
P99_RANGE_MASK_TIF = os.path.join(QA_DIR, "tri_range_p99_mask.tif")
SUMMARY_JSON       = os.path.join(QA_DIR, "roughness_p99_summary.json")
MAP_PNG            = os.path.join(QA_DIR, "roughness_p99_QA_map.png")

# -------- LOAD RASTERS --------
with rasterio.open(TRI_STD_TIF) as r1:
    tri_std = r1.read(1).astype("float32")
    prof = r1.profile
    transform = r1.transform
    crs = r1.crs
    nodata_r = r1.nodata

with rasterio.open(TRI_RANGE_TIF) as r2:
    tri_rng = r2.read(1).astype("float32")

with rasterio.open(SLOPE_TIF) as rs:
    slope = rs.read(1).astype("float32")
    slope_nodata = rs.nodata

with rasterio.open(STUDY_MASK_TIF) as rm:
    study_mask = rm.read(1).astype(np.uint8)

# NoData -> NaN for math
if nodata_r is not None:
    tri_std = np.where(tri_std == nodata_r, np.nan, tri_std)
    tri_rng = np.where(tri_rng == nodata_r, np.nan, tri_rng)
if slope_nodata is not None:
    slope = np.where(slope == slope_nodata, np.nan, slope)

inside = (study_mask == 1)

# -------- P99 THRESHOLDS (inside study area) --------
valid_std   = np.isfinite(tri_std) & inside
valid_range = np.isfinite(tri_rng) & inside
valid_slope = np.isfinite(slope)   & inside

if not valid_std.any() or not valid_range.any():
    raise RuntimeError("No valid roughness pixels inside study area for P99 computation.")

p99_std   = float(np.nanpercentile(tri_std[valid_std],   99))
p99_range = float(np.nanpercentile(tri_rng[valid_range], 99))

# Binary masks (inside only)
m_std   = (tri_std >= p99_std)   & inside
m_range = (tri_rng >= p99_range) & inside

# -------- SLOPE COMPARISON --------
def slope_stats(mask):
    arr = slope[mask & valid_slope]
    if arr.size == 0:
        return {"count": 0, "min": None, "max": None, "mean": None, "p50": None, "p90": None}
    return {
        "count": int(arr.size),
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
        "mean": float(np.nanmean(arr)),
        "p50": float(np.nanpercentile(arr, 50)),
        "p90": float(np.nanpercentile(arr, 90))
    }

stats_std_in   = slope_stats(m_std)
stats_std_out  = slope_stats(inside & (~m_std))
stats_rng_in   = slope_stats(m_range)
stats_rng_out  = slope_stats(inside & (~m_range))

# -------- SAVE MASKS --------
mask_prof = prof.copy()
mask_prof.update(dtype="uint8", nodata=0, count=1, compress="LZW", tiled=True, bigtiff="IF_NEEDED")

with rasterio.open(P99_STD_MASK_TIF, "w", **mask_prof) as dst:
    dst.write(m_std.astype(np.uint8), 1)
with rasterio.open(P99_RANGE_MASK_TIF, "w", **mask_prof) as dst:
    dst.write(m_range.astype(np.uint8), 1)

print(f"[OK] tri_std_p99_mask.tif   → {P99_STD_MASK_TIF}")
print(f"[OK] tri_range_p99_mask.tif → {P99_RANGE_MASK_TIF}")

# -------- SUMMARY JSON --------
summary = {
    "crs_epsg": crs.to_epsg(),
    "p99_thresholds": {"TRI_STD_m": p99_std, "TRI_RANGE_m": p99_range},
    "counts": {
        "pixels_inside": int(inside.sum()),
        "p99_tri_std": int(m_std.sum()),
        "p99_tri_range": int(m_range.sum())
    },
    "slope_deg": {
        "inside_p99_tri_std": stats_std_in,
        "outside_p99_tri_std": stats_std_out,
        "inside_p99_tri_range": stats_rng_in,
        "outside_p99_tri_range": stats_rng_out
    }
}
with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)
print(f"[OK] summary → {SUMMARY_JSON}")

# -------- QUICK MAP (3-panel) --------
try:
    fig = plt.figure(figsize=(14, 8))

    # Panel 1: Slope (deg)
    ax1 = plt.subplot(2,2,1)
    im1 = ax1.imshow(np.where(inside, slope, np.nan), cmap="viridis", vmin=0, vmax=np.nanpercentile(slope[valid_slope], 99))
    ax1.set_title("Slope (deg)")
    ax1.axis("off")
    cb1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
    # overlay p99 outlines
    ax1.contour(m_std, levels=[0.5], colors="r", linewidths=0.8)    # red = TRI_STD P99
    ax1.contour(m_range, levels=[0.5], colors="orange", linewidths=0.8)  # orange = TRI_RANGE P99

    # Panel 2: TRI_STD with mask
    ax2 = plt.subplot(2,2,2)
    im2 = ax2.imshow(np.where(inside, tri_std, np.nan), cmap="magma",
                     vmin=0, vmax=np.nanpercentile(tri_std[valid_std], 99))
    ax2.set_title(f"TRI_STD (m), P99={p99_std:.2f} (red)")
    ax2.axis("off")
    cb2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
    ax2.contour(m_std, levels=[0.5], colors="r", linewidths=0.9)

    # Panel 3: TRI_RANGE with mask
    ax3 = plt.subplot(2,2,3)
    im3 = ax3.imshow(np.where(inside, tri_rng, np.nan), cmap="inferno",
                     vmin=0, vmax=np.nanpercentile(tri_rng[valid_range], 99))
    ax3.set_title(f"TRI_RANGE (m), P99={p99_range:.2f} (orange)")
    ax3.axis("off")
    cb3 = plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
    ax3.contour(m_range, levels=[0.5], colors="orange", linewidths=0.9)

    # Legend panel
    ax4 = plt.subplot(2,2,4)
    ax4.axis("off")
    txt = (
        f"P99 thresholds\n"
        f"  • TRI_STD  : {p99_std:.2f} m\n"
        f"  • TRI_RANGE: {p99_range:.2f} m\n\n"
        f"Slope (deg) stats on P99 areas\n"
        f"  • TRI_STD  P99 → n={stats_std_in['count']}, "
        f"mean={stats_std_in['mean']:.1f}, p90={stats_std_in['p90']:.1f}\n"
        f"  • TRI_RANGE P99 → n={stats_rng_in['count']}, "
        f"mean={stats_rng_in['mean']:.1f}, p90={stats_rng_in['p90']:.1f}\n"
    )
    ax4.text(0.02, 0.98, txt, va="top", ha="left", fontsize=10)

    plt.tight_layout()
    plt.savefig(MAP_PNG, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] QA map → {MAP_PNG}")
except Exception as e:
    print("[WARN] Could not create QA map:", e)

print("[DONE] Roughness QA add-on")


# Step-1.4–1.6: Hydro-conditioning → Flow Acc → TWI

In [ ]:
# =======================================================
# Phases 1.4–1.6  (Hydro-conditioning → Flow Acc → TWI)
# =======================================================
import os, json, heapq, numpy as np
import rasterio

# ---------- PATHS ----------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
PH0_1 = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS")
PH0_2 = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")

OUT14 = os.path.join(BASE, "Results", "Step-1.4-HydroConditioning")
OUT15 = os.path.join(BASE, "Results", "Step-1.5-FlowAccumulation")
OUT16 = os.path.join(BASE, "Results", "Step-1.6-TWI")
for d in (OUT14, OUT15, OUT16): os.makedirs(d, exist_ok=True)

ANCHOR_DEM = os.path.join(PH0_1, "DEM_anchor_32645.tif")
STUDY_MASK = os.path.join(PH0_2, "study_mask.tif")

# 1.4 outputs
DEM_FILLED_TIF = os.path.join(OUT14, "dem_filled.tif")
FLOWDIR_TIF    = os.path.join(OUT14, "flowdir_d8.tif")
HYDRO_JSON     = os.path.join(OUT14, "hydroconditioning_summary.json")

# 1.5 outputs
ACC_CELLS_TIF  = os.path.join(OUT15, "flowacc_cells.tif")     # upstream cells (excl. self)
ACC_AREA_TIF   = os.path.join(OUT15, "flowacc_area_m2.tif")   # upstream area m² (incl. self)
SCA_TIF        = os.path.join(OUT15, "sca_m.tif")             # specific catchment area (m)
FLOWACC_JSON   = os.path.join(OUT15, "flowacc_summary.json")

# 1.6 outputs
TWI_TIF        = os.path.join(OUT16, "twi.tif")
TWI_JSON       = os.path.join(OUT16, "twi_summary.json")

# ---------- CONFIG ----------
NODATA_FLOAT    = -9999.0
EPS_TIE         = 1e-3     # tiny elevation tie-breaker to make a strictly decreasing surface
EPS_SCA         = 1e-6     # stabilizer for ln(a)
EPS_SLOPE_RAD   = np.deg2rad(0.1)  # floor for tan(slope)
# 8-neighborhood offsets and D8 codes (Arc-style)
OFFS = [ (0,1,1), (-1,1,2), (-1,0,4), (-1,-1,8), (0,-1,16), (1,-1,32), (1,0,64), (1,1,128) ]
# distance to neighbors (orthogonal=dx, diagonal=sqrt(2)*dx); dx=dy on your UTM grid
# we'll compute dx from raster transform below

# ---------- LOAD GRID & MASK ----------
if not os.path.exists(ANCHOR_DEM):  raise FileNotFoundError(ANCHOR_DEM)
if not os.path.exists(STUDY_MASK):  raise FileNotFoundError(STUDY_MASK)

with rasterio.open(ANCHOR_DEM) as src:
    dem = src.read(1).astype("float64")
    dem_nodata = src.nodata
    prof = src.profile
    transform = src.transform
    crs = src.crs
    H, W = src.height, src.width
    dx = transform.a
    dy = -transform.e  # positive
    bounds = src.bounds

with rasterio.open(STUDY_MASK) as sm:
    study_mask = sm.read(1).astype(np.uint8)

valid = np.isfinite(dem)
if dem_nodata is not None:
    valid &= (dem != dem_nodata)
valid &= (study_mask == 1)

# Convert NoData to NaN for math everywhere
dem = np.where(valid, dem, np.nan)

# =======================================================
# Phase 1.4 — Priority-flood depression filling
# =======================================================
def priority_flood_fill(dem_np, valid_mask):
    """Barnes-style priority flood. NaNs/outside domain are ignored."""
    H, W = dem_np.shape
    filled = dem_np.copy()
    visited = np.zeros((H, W), dtype=bool)
    pq = []  # (elev, r, c)

    # seed: all valid border cells (domain edge)
    # also consider interior "edge" cells touching invalid as seeds
    def push_seed(r, c):
        visited[r, c] = True
        heapq.heappush(pq, (filled[r, c], r, c))

    for r in range(H):
        for c in (0, W-1):
            if valid_mask[r, c] and not visited[r, c]:
                push_seed(r, c)
    for c in range(W):
        for r in (0, H-1):
            if valid_mask[r, c] and not visited[r, c]:
                push_seed(r, c)
    # also queue valid cells adjacent to any invalid neighbor
    for r in range(1, H-1):
        for c in range(1, W-1):
            if valid_mask[r, c] and not visited[r, c]:
                if not (valid_mask[r-1, c-1] and valid_mask[r-1, c] and valid_mask[r-1, c+1] and
                        valid_mask[r,   c-1] and                             valid_mask[r,   c+1] and
                        valid_mask[r+1, c-1] and valid_mask[r+1, c] and valid_mask[r+1, c+1]):
                    push_seed(r, c)

    # flood
    while pq:
        z, r, c = heapq.heappop(pq)
        for dr, dc, _code in OFFS:
            rr, cc = r + dr, c + dc
            if 0 <= rr < H and 0 <= cc < W and valid_mask[rr, cc] and not visited[rr, cc]:
                visited[rr, cc] = True
                zn = filled[rr, cc]
                if zn < z:
                    filled[rr, cc] = z  # raise pit
                    heapq.heappush(pq, (z, rr, cc))
                else:
                    heapq.heappush(pq, (zn, rr, cc))
    return filled

dem_filled = priority_flood_fill(dem, valid)

# Make an "effective" strictly monotone surface by adding a tiny tie-breaker
# This prevents plateaus from creating cycles in D8 routing.
rr = (np.arange(H)[:, None] / max(H-1, 1))
cc = (np.arange(W)[None, :] / max(W-1, 1))
eff_dem = dem_filled + EPS_TIE * (rr + cc)

# Save filled DEM
prof_f = prof.copy()
prof_f.update(dtype="float32", nodata=NODATA_FLOAT, compress="LZW", tiled=True, bigtiff="IF_NEEDED")
to_write = np.where(valid, dem_filled, np.nan).astype("float32")
to_write[~np.isfinite(to_write)] = NODATA_FLOAT
with rasterio.open(DEM_FILLED_TIF, "w", **prof_f) as dst:
    dst.write(to_write, 1)

# =======================================================
# Phase 1.5 — D8 flow direction + accumulation
# =======================================================
diag = np.sqrt(2.0) * dx  # dx==dy on UTM grid here
dists = np.array([dx, diag, dx, diag, dx, diag, dx, diag], dtype="float64")

# receiver arrays
recv_r = np.full((H, W), -1, dtype=np.int32)
recv_c = np.full((H, W), -1, dtype=np.int32)
flowdir = np.zeros((H, W), dtype=np.int16)

# choose receiver with greatest positive drop/distance on effective DEM (strictly decreasing)
for r in range(H):
    for c in range(W):
        if not valid[r, c]:
            continue
        z0 = eff_dem[r, c]
        best_slope = -np.inf
        best = (-1, -1, 0)  # rr,cc,code
        for (k, (dr, dc, code)) in enumerate(OFFS):
            rrn, ccn = r + dr, c + dc
            if 0 <= rrn < H and 0 <= ccn < W and valid[rrn, ccn]:
                drop = z0 - eff_dem[rrn, ccn]
                slope = drop / dists[k]
                if slope > best_slope:
                    best_slope = slope
                    best = (rrn, ccn, code)
        if best_slope > 0:
            rrn, ccn, code = best
            recv_r[r, c] = rrn
            recv_c[r, c] = ccn
            flowdir[r, c] = code
        else:
            # local minima at domain edge: leave as 0 (sink/outlet)
            flowdir[r, c] = 0

# Save flowdir
prof_fd = prof.copy()
prof_fd.update(dtype="int16", nodata=0, compress="LZW", tiled=True, bigtiff="IF_NEEDED")
with rasterio.open(FLOWDIR_TIF, "w", **prof_fd) as dst:
    arr = np.where(valid, flowdir, 0)
    dst.write(arr, 1)

# Flow accumulation (topologically safe: sort by effective elevation DESC)
order = np.argsort(eff_dem[valid], kind="mergesort")  # ascending
# we need DESC so donors (higher) push to receivers (lower)
valid_rc = np.column_stack(np.nonzero(valid))
order = valid_rc[order[::-1]]  # array of (r,c) in descending eff_dem

acc_inclusive = np.zeros((H, W), dtype="float64")
acc_inclusive[valid] = 1.0  # each cell contributes itself

for r, c in order:
    rrn, ccn = recv_r[r, c], recv_c[r, c]
    if rrn >= 0:
        acc_inclusive[rrn, ccn] += acc_inclusive[r, c]

# By convention in earlier steps, "cells" excluded self, so subtract 1 here:
acc_cells = np.where(valid, acc_inclusive - 1.0, np.nan).astype("float32")
cell_area = float(dx * dy)
acc_area_m2 = np.where(valid, acc_inclusive * cell_area, np.nan).astype("float32")
sca_m = np.where(valid, acc_inclusive * cell_area / dx, np.nan).astype("float32")

def write_float(out_path, arr):
    prof_o = prof.copy()
    prof_o.update(dtype="float32", nodata=NODATA_FLOAT, compress="LZW", tiled=True, bigtiff="IF_NEEDED")
    out = arr.copy()
    out[~np.isfinite(out)] = NODATA_FLOAT
    with rasterio.open(out_path, "w", **prof_o) as dst:
        dst.write(out, 1)

write_float(ACC_CELLS_TIF, acc_cells)
write_float(ACC_AREA_TIF, acc_area_m2)
write_float(SCA_TIF, sca_m)

# =======================================================
# Phase 1.6 — TWI from filled DEM slope + SCA
# =======================================================
# planar slope from dem_filled
# spacing in meters
dy_grad, dx_grad = np.gradient(dem_filled, dy, dx)
slope_rad = np.arctan(np.hypot(dx_grad, dy_grad)).astype("float32")

tan_term = np.tan(np.maximum(slope_rad, EPS_SLOPE_RAD)).astype("float32")
twi = np.log((sca_m + EPS_SCA) / tan_term).astype("float32")

prof_twi = prof.copy()
prof_twi.update(dtype="float32", nodata=NODATA_FLOAT, compress="LZW", tiled=True, bigtiff="IF_NEEDED")
twi_out = np.where(valid, twi, np.nan)
twi_out[~np.isfinite(twi_out)] = NODATA_FLOAT
with rasterio.open(TWI_TIF, "w", **prof_twi) as dst:
    dst.write(twi_out, 1)

# -------------------- Summaries --------------------
inside = (study_mask == 1)
def stats_nan(a, mask):
    v = np.isfinite(a) & mask
    if not np.any(v): return {"count": 0}
    return {
        "count": int(v.sum()),
        "min": float(np.nanmin(a[v])),
        "max": float(np.nanmax(a[v])),
        "mean": float(np.nanmean(a[v])),
        "p95": float(np.nanpercentile(a[v], 95))
    }

hydro_summary = {
    "crs_epsg": crs.to_epsg(),
    "resolution_m": {"x": float(dx), "y": float(dy)},
    "dem_filled": stats_nan(dem_filled, inside),
    "notes": {"filling": "priority-flood; NaNs/outside ignored"}
}
with open(HYDRO_JSON, "w") as f:
    json.dump(hydro_summary, f, indent=2)

flowacc_summary = {
    "cell_area_m2": cell_area,
    "acc_cells_excl_self": stats_nan(acc_cells, inside),
    "acc_area_m2_incl_self": stats_nan(acc_area_m2, inside),
    "sca_m": stats_nan(sca_m, inside)
}
with open(FLOWACC_JSON, "w") as f:
    json.dump(flowacc_summary, f, indent=2)

twi_summary = {
    "formula": "TWI = ln( SCA / tan(slope) )",
    "stabilizers": {"EPS_SCA": EPS_SCA, "EPS_SLOPE_RAD_deg": float(np.rad2deg(EPS_SLOPE_RAD))},
    "twi": stats_nan(twi, inside)
}
with open(TWI_JSON, "w") as f:
    json.dump(twi_summary, f, indent=2)

print(f"[OK] 1.4 dem_filled.tif   → {DEM_FILLED_TIF}")
print(f"[OK] 1.4 flowdir_d8.tif   → {FLOWDIR_TIF}")
print(f"[OK] 1.5 flowacc_cells.tif→ {ACC_CELLS_TIF}")
print(f"[OK] 1.5 flowacc_area_m2.tif→ {ACC_AREA_TIF}")
print(f"[OK] 1.5 sca_m.tif        → {SCA_TIF}")
print(f"[OK] 1.6 twi.tif          → {TWI_TIF}")
print("[DONE] Phases 1.4–1.6 (no richdem)")


# Visualization: Filled DEM Map

In [ ]:
# ============================================================
# Step-1.4 — Visualization for Hydro-Conditioned DEM 
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds
from matplotlib import colors as mcolors

# Optional shapefile readers 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass

# ------------------------- PATHS ---------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

# Inputs
STEP_MASK_DIR   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK_TIF  = os.path.join(STEP_MASK_DIR, "study_mask.tif")
STUDY_SHP       = os.path.join(BASE, "Study_Area.shp")

# Outputs
OUTPUT          = os.path.join(BASE, "Results", "Step-1.4-HydroConditioning")
DEM_FILLED_TIF  = os.path.join(OUTPUT, "dem_filled.tif")
DEM_PNG         = os.path.join(OUTPUT, "Hydro-Conditioned_DEM_Map.png")
DEM_PDF         = os.path.join(OUTPUT, "Hydro-Conditioned_DEM_Map.pdf")

# ------------------------- CONSTANTS / STYLE ----------------
NODATA              = -9999.0

FIGSIZE             = (2, 2)    
CAX_SIZE            = "5%"      
CAX_PAD             = 0.01      
TICK_FONTSIZE       = 3
CB_LABEL_FONTSIZE   = 6
CB_TICK_FONTSIZE    = 5

BOUNDARY_COLOR      = 'red'     
BOUNDARY_LW         = 0.3
MASK_COLOR          = 'red'     
MASK_LW             = 0.3

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS & MINUTE TICKS 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + unified boundary drawer 
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw exactly ONE boundary: shapefile if available, else mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0])
                        ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates'])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line)
                                ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            from scipy.ndimage import binary_erosion
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# LOAD DATA
# =========================================================
print("Loading Hydro-Conditioned DEM and mask...")
with rasterio.open(DEM_FILLED_TIF) as src:
    dem          = src.read(1)
    dem_crs      = src.crs
    dem_nodata   = src.nodata if src.nodata is not None else NODATA
    dem_bounds   = src.bounds

with rasterio.open(STUDY_MASK_TIF) as src:
    mask = src.read(1)

# Prepare DEM 
dem_plot = np.where(dem == dem_nodata, np.nan, dem)
if mask.shape == dem_plot.shape:
    dem_plot = np.where(mask == 1, dem_plot, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != DEM shape {dem_plot.shape}; mask overlay may be skipped.")

# Geographic extent (WGS84)
west, south, east, north = transform_bounds(dem_crs, "EPSG:4326", *dem_bounds)
extent = [west, east, south, north]

# Robust stretch for elevation (1–99%)
valid = dem_plot[~np.isnan(dem_plot)]
vmin = np.nanpercentile(valid, 1) if valid.size else np.nanmin(dem_plot)
vmax = np.nanpercentile(valid, 99) if valid.size else np.nanmax(dem_plot)

# Optional shapefile boundary
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# =========================================================
# CREATE MAP 
# =========================================================
print("Creating Hydro-Conditioned DEM map...")
fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)

# Base image
im = ax.imshow(dem_plot, cmap="terrain",
               extent=extent, origin='upper',
               vmin=vmin, vmax=vmax, zorder=1)

# Single boundary
_ = draw_study_boundary(ax, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

# North arrow & scale bar 
add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south + north) / 2.0,
                      segments=4, fontsize=4)

# Axes: DMS ticks + tiny labels 
ax.set_xlim(west, east); ax.set_ylim(south, north)
set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Colorbar
div = make_axes_locatable(ax)
cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb  = plt.colorbar(im, cax=cax)
cb.set_label('Elevation (m, MSL)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

# Stats box 
stats_txt = (
    f"Min: {np.nanmin(valid):.1f} m\n"
    f"Max: {np.nanmax(valid):.1f} m\n"
    f"Mean: {np.nanmean(valid):.1f} m\n"
    f"Median: {np.nanmedian(valid):.1f} m"
)
ax.text(0.65, 0.22, stats_txt,
        transform=ax.transAxes,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
        fontsize=3)

plt.tight_layout(pad=0.2)
safe_savefig(fig, DEM_PNG, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig, DEM_PDF, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =========================================================
# SUMMARY — Console 
# =========================================================
print("\n" + "="*60)
print("HYDRO-CONDITIONED DEM SUMMARY")
print("="*60)
print(f"\nCOORDINATE SYSTEM:")
print(f"  • CRS: {dem_crs}")
print(f"  • Geographic bounds (WGS84):")
print(f"    - West:  {format_dms_lon(west)}")
print(f"    - East:  {format_dms_lon(east)}")
print(f"    - South: {format_dms_lat(south)}")
print(f"    - North: {format_dms_lat(north)}")

print(f"\nELEVATION STATISTICS:")
print(f"  • Range: {np.nanmin(valid):.2f} – {np.nanmax(valid):.2f} m")
print(f"  • Mean ± Std: {np.nanmean(valid):.2f} ± {np.nanstd(valid):.2f} m")
print(f"  • Median: {np.nanmedian(valid):.2f} m")
print(f"  • Stretch (1–99%): {vmin:.2f} – {vmax:.2f} m")

print("\nINTERPRETATION:")
print("  • Hydro-conditioning removes spurious pits/sinks to ensure realistic drainage.")
print("  • This DEM underpins flow direction, flow accumulation, TWI/SPI, curvature, etc.")
print("="*60)


# Visualization: Flow Directions

In [ ]:
# ============================================================
# Step-1.4 — Visualization for D8 Flow Direction 
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds
import matplotlib.colors as mcolors

# Optional shapefile readers 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass

# =========================================================
# PATHS
# =========================================================
BASE = os.environ.get("LSM_DATA_DIR", "./data")

STEP_MASK_DIR = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK_TIF = os.path.join(STEP_MASK_DIR, "study_mask.tif")
STUDY_SHP      = os.path.join(BASE, "Study_Area.shp")

OUTDIR       = os.path.join(BASE, "Results", "Step-1.4-HydroConditioning")
FLOWDIR_TIF  = os.path.join(OUTDIR, "flowdir_d8.tif")
FLOWDIR_PNG  = os.path.join(OUTDIR, "Flow_Direction_Map.png")
FLOWDIR_PDF  = os.path.join(OUTDIR, "Flow_Direction_Map.pdf")

# =========================================================
# STYLE 
# =========================================================
FIGSIZE             = (2, 2)
CAX_SIZE            = "5%"
CAX_PAD             = 0.01
TICK_FONTSIZE       = 3
CB_LABEL_FONTSIZE   = 6
CB_TICK_FONTSIZE    = 5
BOUNDARY_COLOR      = 'red'
BOUNDARY_LW         = 0.3
MASK_COLOR          = 'red'
MASK_LW             = 0.3

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS & MINUTE TICKS
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + Unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw ONE boundary: shapefile if available, else eroded mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates']); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            from scipy.ndimage import binary_erosion
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# LOAD DATA
# =========================================================
print("Loading Flow Direction (D8) and mask data...")
with rasterio.open(FLOWDIR_TIF) as src:
    flowdir_raw  = src.read(1)
    flow_crs     = src.crs
    flow_bounds  = src.bounds
    flow_nodata  = src.nodata

with rasterio.open(STUDY_MASK_TIF) as src:
    mask = src.read(1)

# Extent (WGS84)
west, south, east, north = transform_bounds(flow_crs, "EPSG:4326", *flow_bounds)
extent = [west, east, south, north]

# Mask and nodata
flowdir = np.where(flowdir_raw == flow_nodata, np.nan, flowdir_raw)
if mask.shape == flowdir.shape:
    flowdir = np.where(mask == 1, flowdir, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != flowdir shape {flowdir.shape}; mask overlay may be skipped.")

# =========================================================
# D8 CATEGORICAL MAPPING 
# =========================================================
# D8: 1=E, 2=NE, 4=N, 8=NW, 16=W, 32=SW, 64=S, 128=SE, 0=Sink
code_to_idx = {0:0, 1:1, 2:2, 4:3, 8:4, 16:5, 32:6, 64:7, 128:8}
idx_labels_short = ["Sink","E","NE","N","NW","W","SW","S","SE"]

mapped = np.full(flowdir.shape, np.nan, dtype=float)
for code, idx in code_to_idx.items():
    mapped[flowdir == code] = idx

# Compact categorical palette 
flow_colors = ['#808080',  # Sink
               '#e41a1c',  # E
               '#ff7f00',  # NE
               '#ffff33',  # N
               '#4daf4a',  # NW
               '#377eb8',  # W
               '#984ea3',  # SW
               '#a65628',  # S
               '#f781bf']  # SE
flow_cmap = mcolors.ListedColormap(flow_colors)
norm = mcolors.BoundaryNorm(np.arange(-0.5, 9.5, 1), flow_cmap.N)

# =========================================================
# OPTIONAL SHAPE BOUNDARY
# =========================================================
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# =========================================================
# CREATE MAP 
# =========================================================
print("Creating D8 Flow Direction map...")
fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)

im = ax.imshow(mapped, cmap=flow_cmap, norm=norm,
               extent=extent, origin='upper', zorder=1)

# Single boundary 
_ = draw_study_boundary(ax, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

# North arrow & scale bar 
add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south + north) / 2.0,
                      segments=4, fontsize=4)

# Axes: WGS84 DMS ticks + tiny labels
ax.set_xlim(west, east); ax.set_ylim(south, north)
set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Colorbar 
div = make_axes_locatable(ax)
cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb  = plt.colorbar(im, cax=cax, ticks=np.arange(0, 9))
cb.set_label('Flow Direction (D8)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)
cb.ax.set_yticklabels(idx_labels_short)

# Compact stats box 
valid = ~np.isnan(mapped)
vals  = mapped[valid].astype(int)
total = vals.size
counts = {i: int(np.sum(vals == i)) for i in range(9)}
pct = lambda i: (counts[i] / total * 100.0) if total > 0 else 0.0
stats_txt = (
    f"Pixels: {total:,}\n"
    f"Cardinal: {counts[1]+counts[3]+counts[5]+counts[7]:,} ({(counts[1]+counts[3]+counts[5]+counts[7])/total*100:.1f}%)\n"
    f"Diagonals: {counts[2]+counts[4]+counts[6]+counts[8]:,} ({(counts[2]+counts[4]+counts[6]+counts[8])/total*100:.1f}%)\n"
    f"Sinks: {counts[0]:,} ({pct(0):.1f}%)"
)
ax.text(0.05, 0.72, stats_txt,
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
        fontsize=3)

plt.tight_layout(pad=0.2)
safe_savefig(fig, FLOWDIR_PNG, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig, FLOWDIR_PDF, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =========================================================
# SUMMARY — Console
# =========================================================
print("\n" + "="*60)
print("D8 FLOW DIRECTION SUMMARY")
print("="*60)
print(f"\nCOORDINATE SYSTEM:")
print(f"  • CRS: {flow_crs}")
print(f"  • Geographic bounds (WGS84):")
print(f"    - West:  {format_dms_lon(west)}")
print(f"    - East:  {format_dms_lon(east)}")
print(f"    - South: {format_dms_lat(south)}")
print(f"    - North: {format_dms_lat(north)}")

print(f"\nPIXEL COUNTS BY DIRECTION:")
order_labels = ["Sink","E","NE","N","NW","W","SW","S","SE"]
for i, lab in enumerate(order_labels):
    print(f"  • {lab}: {counts[i]:,} ({pct(i):.1f}%)")

if total > 0:
    freq = [(i, counts[i]) for i in range(1,9)]
    freq.sort(key=lambda x: x[1], reverse=True)
    print("\nDOMINANT DIRECTIONS:")
    for rank, (i, c) in enumerate(freq[:3], 1):
        print(f"  {rank}. {order_labels[i]} — {c:,} ({c/total*100:.1f}%)")

print("\nNOTES:")
print("  • D8 uses the single-neighbor steepest descent; codes are powers of 2.")
print("  • Sinks may reflect depressions or true outlets depending on conditioning.")
print("="*60)


# Visualization: Specific Catchment Area (SCA) - Flow Accumulation

In [ ]:
# ============================================================
# Step-1.5 — Visualization for Specific Catchment Area 
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds
from matplotlib import colors as mcolors

# Optional shapefile readers 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion
    HAVE_SCIPY = True
except Exception:
    pass

# ------------------------- PATHS -------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

# Inputs
STEP_MASK_DIR = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK_TIF = os.path.join(STEP_MASK_DIR, "study_mask.tif")
STUDY_SHP      = os.path.join(BASE, "Study_Area.shp")

# Outputs
OUTDIR   = os.path.join(BASE, "Results", "Step-1.5-FlowAccumulation")
SCA_TIF  = os.path.join(OUTDIR, "sca_m.tif")
SCA_PNG  = os.path.join(OUTDIR, "SCA_Map.png")
SCA_PDF  = os.path.join(OUTDIR, "SCA_Map.pdf")

# ------------------------- STYLE -------------------------
FIGSIZE           = (2, 2)
CAX_SIZE          = "5%"
CAX_PAD           = 0.01
TICK_FONTSIZE     = 3
CB_LABEL_FONTSIZE = 6
CB_TICK_FONTSIZE  = 5
BOUNDARY_COLOR    = 'red'
BOUNDARY_LW       = 0.3
MASK_COLOR        = 'red'
MASK_LW           = 0.3

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS & MINUTE TICKS 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + Unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw ONE boundary: shapefile if available, else eroded mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates']); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float) if HAVE_SCIPY \
                  else mask_array.astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# LOAD DATA
# =========================================================
print("Loading SCA and mask data...")
with rasterio.open(SCA_TIF) as src:
    sca_raw   = src.read(1)
    sca_crs   = src.crs
    sca_bounds = src.bounds
    sca_nodata = src.nodata

with rasterio.open(STUDY_MASK_TIF) as src:
    mask = src.read(1)

# Extent in WGS84
west, south, east, north = transform_bounds(sca_crs, "EPSG:4326", *sca_bounds)
extent = [west, east, south, north]

# Mask + nodata
sca = np.where(sca_raw == sca_nodata, np.nan, sca_raw)
if mask.shape == sca.shape:
    sca = np.where(mask == 1, sca, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != SCA shape {sca.shape}; mask overlay may be skipped.")

# Log10 scaling with small epsilon
eps = 1e-6
sca_log = np.where(np.isnan(sca), np.nan, np.log10(sca + eps))
valid = sca_log[~np.isnan(sca_log)]

# Robust stretch (2–98%)
vmin = np.nanpercentile(valid, 2) if valid.size else np.nanmin(sca_log)
vmax = np.nanpercentile(valid, 98) if valid.size else np.nanmax(sca_log)

# Optional shapefile boundary
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# =========================================================
# CREATE MAP
# =========================================================
print("Creating SCA map...")
fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)

im = ax.imshow(sca_log, cmap='Blues',
               extent=extent, origin='upper',
               vmin=vmin, vmax=vmax, zorder=1)

# Single boundary 
_ = draw_study_boundary(ax, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

# North arrow & scale bar
add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                      segments=4, fontsize=4)

# Axes: DMS ticks + tiny labels
ax.set_xlim(west, east); ax.set_ylim(south, north)
set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Colorbar 
div = make_axes_locatable(ax)
cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb  = plt.colorbar(im, cax=cax)
cb.set_label('log10(SCA) [m]', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

# Stats box 
sca_lin_valid = np.where(~np.isnan(sca), sca, np.nan)
sca_lin_valid = sca_lin_valid[~np.isnan(sca_lin_valid)]
stats_txt = (
    f"Min: {np.nanmin(sca_lin_valid):.2f} m\n"
    f"Max: {np.nanmax(sca_lin_valid):.0f} m\n"
    f"Mean: {np.nanmean(sca_lin_valid):.1f} m\n"
    f"Median: {np.nanmedian(sca_lin_valid):.1f} m"
)
ax.text(0.68, 0.22, stats_txt,
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
        fontsize=3)

plt.tight_layout(pad=0.2)
safe_savefig(fig, SCA_PNG, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig, SCA_PDF, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =========================================================
# SUMMARY — Console
# =========================================================
print("\n" + "="*60)
print("SPECIFIC CATCHMENT AREA (SCA) SUMMARY")
print("="*60)
print(f"\nCOORDINATE SYSTEM:")
print(f"  • CRS: {sca_crs}")
print(f"  • Geographic bounds (WGS84):")
print(f"    - West:  {format_dms_lon(west)}")
print(f"    - East:  {format_dms_lon(east)}")
print(f"    - South: {format_dms_lat(south)}")
print(f"    - North: {format_dms_lat(north)}")

if sca_lin_valid.size:
    print(f"\nSCA STATISTICS:")
    print(f"  • Range: {np.nanmin(sca_lin_valid):.3f} – {np.nanmax(sca_lin_valid):.3f} m")
    print(f"  • Mean ± Std: {np.nanmean(sca_lin_valid):.3f} ± {np.nanstd(sca_lin_valid):.3f} m")
    print(f"  • Median: {np.nanmedian(sca_lin_valid):.3f} m")
    print(f"  • 95th percentile: {np.nanpercentile(sca_lin_valid, 95):.3f} m")
    print(f"  • Valid pixels: {sca_lin_valid.size:,}")

    p25 = np.nanpercentile(sca_lin_valid, 25)
    p75 = np.nanpercentile(sca_lin_valid, 75)
    p90 = np.nanpercentile(sca_lin_valid, 90)
    p95 = np.nanpercentile(sca_lin_valid, 95)
    ridge_areas   = np.sum(sca_lin_valid <= p25)
    hillslopes    = np.sum((sca_lin_valid > p25) & (sca_lin_valid <= p75))
    valley_bottom = np.sum((sca_lin_valid > p75) & (sca_lin_valid <= p95))
    main_channels = np.sum(sca_lin_valid > p95)

    print(f"\nTOPOGRAPHIC POSITION (by SCA percentiles):")
    print(f"  • Ridges (0–25%): {ridge_areas:,} ({ridge_areas/len(sca_lin_valid)*100:.1f}%) — SCA ≤ {p25:.2f} m")
    print(f"  • Hillslopes (25–75%): {hillslopes:,} ({hillslopes/len(sca_lin_valid)*100:.1f}%) — {p25:.2f}–{p75:.2f} m")
    print(f"  • Valleys (75–95%): {valley_bottom:,} ({valley_bottom/len(sca_lin_valid)*100:.1f}%) — {p75:.2f}–{p95:.2f} m")
    print(f"  • Main channels (95–100%): {main_channels:,} ({main_channels/len(sca_lin_valid)*100:.1f}%) — > {p95:.2f} m")

print("\nNOTES:")
print("  • SCA = upstream contributing area per unit contour width (m).")
print("  • log10 scaling improves contrast across orders of magnitude.")
print("  • High SCA highlights channels and potential saturation pathways.")
print("="*60)


# Visualization: Topographic Wetness Index (TWI)

In [ ]:
# ============================================================
# Step-1.6 — Visualization for Topographic Wetness Index (TWI)
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds
from matplotlib import colors as mcolors

# Optional deps 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion
    HAVE_SCIPY = True
except Exception:
    pass

# ------------------------- PATHS ---------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STUDY_MASK_TIF = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP      = os.path.join(BASE, "Study_Area.shp")

OUTPUT = os.path.join(BASE, "Results", "Step-1.6-TWI")
TWI_TIF = os.path.join(OUTPUT, "twi.tif")
OUT_PNG = os.path.join(OUTPUT, "TWI_Map.png")
OUT_PDF = os.path.join(OUTPUT, "TWI_Map.pdf")

NODATA = -9999.0

# ------------------------- STYLE ---------------------
FIGSIZE           = (2, 2)     
CAX_SIZE          = "5%"       
CAX_PAD           = 0.01       
TICK_FONTSIZE     = 3
CB_LABEL_FONTSIZE = 6
CB_TICK_FONTSIZE  = 5
BOUNDARY_COLOR    = 'red'
BOUNDARY_LW       = 0.3
MASK_COLOR        = 'red'
MASK_LW           = 0.3

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + Unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw exactly ONE boundary: shapefile if available, else eroded mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates']); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float) if HAVE_SCIPY \
                  else mask_array.astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# COMPACT TWI PLOT 
# =========================================================
if __name__ == "__main__":
    # Load rasters
    print("Loading TWI and mask...")
    with rasterio.open(TWI_TIF) as src:
        twi = src.read(1)
        crs = src.crs
        bounds = src.bounds
        nodata = src.nodata if src.nodata is not None else NODATA
    with rasterio.open(STUDY_MASK_TIF) as src:
        mask = src.read(1)

    # Masking
    arr = np.where(twi == nodata, np.nan, twi)
    if mask.shape == arr.shape:
        arr = np.where(mask == 1, arr, np.nan)
    else:
        print(f"[WARN] Mask shape {mask.shape} != TWI shape {arr.shape}; mask overlay may be skipped.")

    # Extent (WGS84)
    west, south, east, north = transform_bounds(crs, "EPSG:4326", *bounds)
    extent = [west, east, south, north]

    # Robust stretch 2–98%
    valid = arr[~np.isnan(arr)]
    if valid.size == 0:
        raise RuntimeError("No valid TWI pixels to plot.")
    vmin = np.nanpercentile(valid, 2)
    vmax = np.nanpercentile(valid, 98)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = np.nanmin(valid), np.nanmax(valid)

    # Optional shapefile boundary
    boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

    # Figure 
    print("Creating TWI map...")
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    im = ax.imshow(arr, cmap="GnBu", extent=extent, origin='upper',
                   vmin=vmin, vmax=vmax, zorder=1)

    # Single boundary 
    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                            mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

    # North arrow & checker scale bar
    add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
    add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                          segments=4, fontsize=4)

    # Axes: DMS ticks + tiny labels
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45); lbl.set_ha('right')

    # Colorbar 
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label('TWI', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

    # Compact stats box 
    #stats_txt = (
        #f"Min: {np.nanmin(valid):.2f}\n"
        #f"Max: {np.nanmax(valid):.2f}\n"
        #f"Mean: {np.nanmean(valid):.2f}\n"
        #f"Median: {np.nanmedian(valid):.2f}"
    #)
    #ax.text(0.72, 0.22, stats_txt,
            #transform=ax.transAxes, va='top',
            #bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            #fontsize=3)

    # Save 
    plt.tight_layout(pad=0.2)
    safe_savefig(fig, OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
    safe_savefig(fig, OUT_PDF, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    # Console summary
    print("\n" + "="*60)
    print("TOPOGRAPHIC WETNESS INDEX (TWI) SUMMARY")
    print("="*60)
    print(f"CRS: {crs}")
    print(f"BBox (WGS84): W={format_dms_lon(west)}, E={format_dms_lon(east)}, "
          f"S={format_dms_lat(south)}, N={format_dms_lat(north)}")
    print(f"Range: {np.nanmin(valid):.3f} – {np.nanmax(valid):.3f}")
    print(f"Mean ± Std: {np.nanmean(valid):.3f} ± {np.nanstd(valid):.3f}")
    print(f"Median: {np.nanmedian(valid):.3f}")
    print("="*60)


# Updated Code for Visualization for TWI

In [ ]:
# ============================================================
# Step-1.6 — Visualization for Topographic Wetness Index (TWI)
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from rasterio.warp import transform_bounds

# ---------------- Optional shapefile ----------------
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass

# ---------------- PATHS ----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP_MASK   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK  = os.path.join(STEP_MASK, "study_mask.tif")
STUDY_SHP   = os.path.join(BASE, "Study_Area.shp")

OUTDIR      = os.path.join(BASE, "Results", "Step-1.6-TWI")
TWI_TIF     = os.path.join(OUTDIR, "twi.tif")

# ---------------- PLOT STYLE  ----------------
FIGSIZE            = (1, 1)
SAVE_DPI           = 600
CAX_SIZE           = "8%"     
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 3.0
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_COLOR     = 'black'
BOUNDARY_LINEWIDTH = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_CONTOUR_WIDTH = 0.4
LW_AXIS            = 0.3
XLABEL_PAD         = 0.5
YLABEL_PAD         = 0.5
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True

# Opaque white canvas
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# ---------------- Safe save helpers  ----------------
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            os.chmod(path, stat.S_IWRITE)
    except Exception: pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs); print(f"[OUTPUT] Saved: {path}"); return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs); print(f"[OUTPUT] Saved: {alt}"); return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def force_axes_bg_opaque(ax, fig=None):
    if fig is None: fig = ax.figure
    fig.patch.set_facecolor('white'); fig.patch.set_alpha(1.0)
    ax.set_facecolor('white'); ax.patch.set_alpha(1.0)
    for child in fig.axes:
        try:
            child.set_facecolor('white'); child.patch.set_alpha(1.0)
        except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# ---------------- DMS formatters & minute ticks  ----------------
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg)))
    m_float = (abs(decimal_deg) - d) * 60
    m = int(np.floor(m_float))
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'N' if decimal_deg >= 0 else 'S'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'E' if decimal_deg >= 0 else 'W'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min / 60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# ---------------- North arrow & checkerboard scale bar  ----------------
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72; y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)

# ---------------- Shapefile loader + boundary drawer  ----------------
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path): return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}": gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception: pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception: pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2):
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0])
                        ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception: pass
    # fallback to mask contour if needed
    try:
        from scipy.ndimage import binary_erosion as _be
        arr = (_be(mask_array.astype(bool), iterations=erode_iters).astype(float)
               if (mask_array is not None) else None)
    except Exception:
        arr = (mask_array.astype(float) if mask_array is not None else None)
    if arr is not None:
        ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                   extent=extent, origin='upper', zorder=11)
        return True
    return False

# =========================================================
# TWI MAP 
# =========================================================
if __name__ == "__main__":
    os.makedirs(OUTDIR, exist_ok=True)

    # Load rasters
    print("Loading TWI and mask...")
    with rasterio.open(TWI_TIF) as src:
        twi         = src.read(1)
        twi_crs     = src.crs
        twi_bounds  = src.bounds
        twi_nodata  = src.nodata

    with rasterio.open(STUDY_MASK) as src:
        mask = src.read(1)

    # Extent in WGS84 
    west, south, east, north = transform_bounds(twi_crs, "EPSG:4326", *twi_bounds)
    extent = [west, east, south, north]

    # Mask & NoData 
    arr = np.where(twi == twi_nodata, np.nan, twi)
    if mask.shape == arr.shape:
        arr = np.where(mask == 1, arr, np.nan)
    else:
        print(f"[WARN] Mask shape {mask.shape} != TWI shape {arr.shape}")

    # Robust display limits 
    valid = arr[np.isfinite(arr)]
    if valid.size == 0:
        raise RuntimeError("No valid TWI pixels to plot.")
    vmin = float(np.percentile(valid, 2))
    vmax = float(np.percentile(valid, 98))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
        vmin, vmax = float(np.nanmin(valid)), float(np.nanmax(valid))

    # Boundary object
    boundary = load_shapefile_boundary(STUDY_SHP, 4326)

    # Create map  
    print("Creating TWI map...")
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)

    cmap = plt.get_cmap('GnBu').copy()   
    cmap.set_bad((1,1,1,1))
    im = ax.imshow(arr, cmap=cmap, extent=extent, origin='upper',
                   vmin=vmin, vmax=vmax, zorder=1)

    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                            mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

    # Axes 
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
    for lbl in ax.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
    #ax.set_xlabel("Longitude", fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=XLABEL_PAD)
    #ax.set_ylabel("Latitude",  fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=YLABEL_PAD)

    # Cartography 
    add_north_arrow_tri(ax)
    add_checker_scale_bar(ax, west, east, south, north, crs_type='wgs84', segments=4)

    # Right inset colorbar 
    cax = inset_axes(ax, width=CAX_SIZE, height="100%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax.transAxes, borderpad=0)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label('TWI', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
    cb.ax.set_facecolor('white'); cb.ax.patch.set_alpha(1.0)
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
    if hasattr(cb, "outline") and cb.outline is not None: cb.outline.set_linewidth(0.3)

    force_axes_bg_opaque(ax, fig)
    plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)

    # Save all plot
    save_all(fig, os.path.join(OUTDIR, "TWI_Map"))
    plt.show()

    print("\n[DONE] TWI map →", OUTDIR)


# Step-1.7: Profile curvature and Plan curvature

In [ ]:
# ============================================================
# Step-1.7 — Profile curvature and Plan curvature
# ============================================================
import os, json, warnings
from typing import Tuple, Optional, Dict
import numpy as np
import rasterio
from rasterio.transform import Affine

#--------------------PATH-----------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
#Input Path
DEM_FILLED_TIF  = os.path.join(BASE, r"Results\Step-1.4-HydroConditioning", "dem_filled.tif")
STUDY_MASK_TIF  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")

#Output Path
OUT_DIR         = os.path.join(BASE, r"Results\Step-1.7-Curvatures"); os.makedirs(OUT_DIR, exist_ok=True)
PROF_CURV_TIF   = os.path.join(OUT_DIR, "profile_curv_32645_12p5m_float32.tif")
PLAN_CURV_TIF   = os.path.join(OUT_DIR, "plan_curv_32645_12p5m_float32.tif")
CURV_STATS_JSON = os.path.join(OUT_DIR, "curvature_stats.json")

# ----------------SETTINGS--------------------------
NODATA = -9999.0
EPS    = 1e-8

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

try:
    from scipy.ndimage import convolve
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
    warnings.warn("scipy.ndimage not found; using slower numpy fallback for convolution.")

def _read_mask(path: str) -> np.ndarray:
    with rasterio.open(path) as ds:
        m = ds.read(1, masked=False)
        if ds.nodata is not None:
            m = np.where(m == ds.nodata, 0, m)
        return (m != 0).astype(np.uint8)

def _save_float32(out_path: str, arr: np.ndarray, ref_path: str, nodata: float = NODATA):
    arr = arr.astype(np.float32, copy=False)
    with rasterio.open(ref_path) as ref:
        meta = ref.meta.copy()
    meta.update(dtype="float32", nodata=nodata, compress="LZW")
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(arr, 1)

def _hist_png(values: np.ndarray, out_png: str, title: str):
    if not HAVE_MPL:
        return
    v = values[np.isfinite(values)]
    v = v[(v != NODATA)]
    if v.size == 0:
        return
    plt.figure(figsize=(6,4))
    plt.hist(v, bins=80)
    plt.title(title)
    plt.xlabel("value")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def _stats(values: np.ndarray, mask: Optional[np.ndarray] = None) -> Dict:
    if mask is not None:
        values = np.where(mask, values, np.nan)
    v = values[np.isfinite(values)]
    v = v[(v != NODATA)]
    if v.size == 0:
        return {"count": 0}
    q = np.nanpercentile(v, [0,1,5,25,50,75,95,99,100])
    return {
        "count": int(v.size),
        "min": float(np.nanmin(v)),
        "p01": float(q[1]),
        "p05": float(q[2]),
        "p25": float(q[3]),
        "p50": float(q[4]),
        "p75": float(q[5]),
        "p95": float(q[6]),
        "p99": float(q[7]),
        "max": float(np.nanmax(v)),
        "mean": float(np.nanmean(v)),
        "std": float(np.nanstd(v)),
    }

def _convolve(img: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    if HAVE_SCIPY:
        return convolve(img, kernel, mode="nearest")
    pad_y, pad_x = kernel.shape[0]//2, kernel.shape[1]//2
    padded = np.pad(img, ((pad_y,pad_y),(pad_x,pad_x)), mode="edge")
    out = np.empty_like(img, dtype=np.float64)
    kh, kw = kernel.shape
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            patch = padded[i:i+kh, j:j+kw]
            out[i, j] = np.sum(patch * kernel)
    return out

def compute_curvatures(dem_filled_path: str, study_mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    with rasterio.open(dem_filled_path) as ds:
        dem = ds.read(1).astype(np.float64)
        tr = ds.transform
        res_x = tr.a
        res_y = -tr.e if tr.e < 0 else tr.e
        if ds.nodata is not None:
            dem = np.where(dem == ds.nodata, np.nan, dem)

    kx = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]], dtype=np.float64) / (8.0 * res_x)
    ky = np.array([[ 1,  2,  1],
                   [ 0,  0,  0],
                   [-1, -2, -1]], dtype=np.float64) / (8.0 * res_y)

    p = _convolve(dem, kx)
    q = _convolve(dem, ky)

    kxx = np.array([[0, 0, 0],
                    [1,-2, 1],
                    [0, 0, 0]], dtype=np.float64) / (res_x**2)
    kyy = np.array([[0, 1, 0],
                    [0,-2, 0],
                    [0, 1, 0]], dtype=np.float64) / (res_y**2)
    kxy = np.array([[ 1, 0, -1],
                    [ 0, 0,  0],
                    [-1, 0,  1]], dtype=np.float64) / (4.0 * res_x * res_y)

    r = _convolve(dem, kxx)
    t = _convolve(dem, kyy)
    s = _convolve(dem, kxy)

    g2 = p*p + q*q
    denom_plan = np.power(g2, 1.5) + EPS
    denom_prof = (g2 * np.sqrt(1.0 + g2)) + EPS

    plan_curv    = ( r*(q*q) - 2.0*s*p*q + t*(p*p) ) / denom_plan
    profile_curv = ( r*(p*p) + 2.0*s*p*q + t*(q*q) ) / denom_prof

    for arr in (plan_curv, profile_curv):
        arr[~np.isfinite(arr)] = 0.0
        if study_mask is not None:
            arr[study_mask == 0] = NODATA

    return profile_curv.astype(np.float32), plan_curv.astype(np.float32)

mask = _read_mask(STUDY_MASK_TIF)
prof, plan = compute_curvatures(DEM_FILLED_TIF, mask)
_save_float32(PROF_CURV_TIF, prof, DEM_FILLED_TIF, NODATA)
_save_float32(PLAN_CURV_TIF, plan, DEM_FILLED_TIF, NODATA)

cstats = {
    "profile_curvature": _stats(prof, mask),
    "plan_curvature":    _stats(plan, mask),
}
with open(CURV_STATS_JSON, "w") as f:
    json.dump(cstats, f, indent=2)

if HAVE_MPL:
    _hist_png(prof, os.path.join(OUT_DIR, "profile_curv_hist.png"), "Profile curvature")
    _hist_png(plan, os.path.join(OUT_DIR, "plan_curv_hist.png"),    "Plan curvature")

print("Step-1.7 completed.")

# Step-1.7 — Visualization for Profile Curvature 

In [ ]:
# ============================================================
# Step-1.7 — Profile Curvature Visualization 
# ============================================================
import os
import time
import stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib import colors as mcolors  

# Optional Warp
try:
    from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds
    _HAVE_WARP = True
except Exception:
    _HAVE_WARP = False

# Optional shapefile readers
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass

# -------------------- PATHS -----------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STUDY_MASK_TIF  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP       = os.path.join(BASE, "Study_Area.shp")   
OUT_DIR         = os.path.join(BASE, r"Results\Step-1.7-Curvatures")
PROF_CURV_TIF   = os.path.join(OUT_DIR, "profile_curv_32645_12p5m_float32.tif")
OUT_PNG         = os.path.join(OUT_DIR, "Profile Curvature Map.png")
OUT_PDF         = os.path.join(OUT_DIR, "Profile Curvature Map.pdf")

# -------------------- STYLE -----------------------
FIGSIZE          = (2, 2)
CAX_SIZE         = "5%"
CAX_PAD          = 0.01
COLORBAR_LABEL   = 'Curvature (1/m)'
TITLE            = "Profile Curvature"
TICK_FONTSIZE    = 3
LABEL_FONTSIZE   = 5
TITLE_FONTSIZE   = 6
BOUNDARY_LINEWIDTH = 0.5
BOUNDARY_COLOR     = 'red'
CURV_CMAP_MODE   = 'vivid'

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS & MINUTE TICKS
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR  
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))

    ax.text(apex[0], y1 + size * float(label_upshift), label, transform=ax.transAxes,
            ha='center', va='bottom', fontsize=8, fontweight='bold',
            color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))

    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))

    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))

    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Utility
# =========================================================
def load_profile_curvature_geo(tif_path, out_dir):
    geo_tif = os.path.join(out_dir, "profile_curv_geo.tif")
    os.makedirs(out_dir, exist_ok=True)

    with rasterio.open(tif_path) as src:
        src_crs = src.crs
        nodata  = src.nodata

        if _HAVE_WARP and src_crs and not src_crs.is_geographic:
            transform, width, height = calculate_default_transform(
                src_crs, "EPSG:4326", src.width, src.height, *src.bounds
            )
            kwargs = src.meta.copy()
            kwargs.update({'crs': "EPSG:4326", 'transform': transform, 'width': width, 'height': height})
            with rasterio.open(geo_tif, 'w', **kwargs) as dst:
                reproject(
                    source=rasterio.band(src, 1),
                    destination=rasterio.band(dst, 1),
                    src_transform=src.transform,
                    src_crs=src_crs,
                    dst_transform=transform,
                    dst_crs="EPSG:4326",
                    resampling=Resampling.bilinear
                )
            with rasterio.open(geo_tif) as ds:
                arr = ds.read(1)
                b = ds.bounds
                extent = [b.left, b.right, b.bottom, b.top]
                nodata = ds.nodata
            return arr, extent, nodata
        else:
            arr = src.read(1)
            try:
                if _HAVE_WARP:
                    b = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
                    extent = [b[0], b[2], b[1], b[3]]
                else:
                    b = src.bounds
                    extent = [b.left, b.right, b.bottom, b.top]
            except Exception:
                b = src.bounds
                extent = [b.left, b.right, b.bottom, b.top]
            return arr, extent, nodata

# =========================================================
# Shapefile loader + plotting
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        print(f"[WARN] Shapefile not found: {shapefile_path}")
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception as e:
            print(f"[WARN] Geopandas failed: {e}")
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feature in shp:
                    if feature['geometry'] is not None:
                        geometries.append(feature['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception as e:
            print(f"[WARN] Fiona failed: {e}")
    return None

def plot_shapefile_on_map(ax, shapefile_data, line_color=BOUNDARY_COLOR, line_width=BOUNDARY_LINEWIDTH):
    if shapefile_data is None:
        return
    try:
        if HAVE_GEOPANDAS and hasattr(shapefile_data, 'plot'):
            shapefile_data.boundary.plot(ax=ax, color=line_color, linewidth=line_width, zorder=12)
        elif isinstance(shapefile_data, dict) and 'geometries' in shapefile_data:
            for geom in shapefile_data['geometries']:
                gt = geom['type']
                if gt == 'Polygon':
                    coords = geom['coordinates'][0]
                    x = [pt[0] for pt in coords]; y = [pt[1] for pt in coords]
                    ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
                elif gt == 'MultiPolygon':
                    for polygon in geom['coordinates']:
                        coords = polygon[0]
                        x = [pt[0] for pt in coords]; y = [pt[1] for pt in coords]
                        ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
                elif gt in ['LineString', 'MultiLineString']:
                    if gt == 'LineString':
                        coords = geom['coordinates']
                        x = [pt[0] for pt in coords]; y = [pt[1] for pt in coords]
                        ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
                    else:
                        for line in geom['coordinates']:
                            x = [pt[0] for pt in line]; y = [pt[1] for pt in line]
                            ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
    except Exception as e:
        print(f"[WARN] Boundary plot error: {e}")

# =========================================================
# Curvature colormap builder 
# =========================================================
def build_curvature_cmap(mode='vivid'):
    mode = (mode or '').lower()
    if mode == 'rdblu':
        return plt.get_cmap('RdBu_r'), 'RdBu_r'
    if mode == 'coolwarm':
        return plt.get_cmap('coolwarm'), 'coolwarm'
    # Default Vivid
    CURV_COLORS = [
        "#0B2E8C", "#2D62D0", "#78A7FF", "#FFFFFF",
        "#FFB36B", "#FF6A2E", "#990D0D"
    ]
    return mcolors.LinearSegmentedColormap.from_list("CurvVivid", CURV_COLORS, N=256), 'CurvVivid'

# =========================================================
# PLOT
# =========================================================
if __name__ == "__main__":
    # Load curvature 
    arr, extent, nodata = load_profile_curvature_geo(PROF_CURV_TIF, OUT_DIR)
    west, east, south, north = extent

    # Study mask 
    mask = None
    try:
        with rasterio.open(STUDY_MASK_TIF) as ms:
            mask = ms.read(1)
            if mask.shape != arr.shape:
                mask = None
    except Exception as e:
        print(f"[WARN] Could not read mask: {e}")

    # Boundary shapefile 
    boundary_gdf_or_geom = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

    # Clean nodata
    arr = np.where(arr == nodata, np.nan, arr)

    # Robust symmetric limits around 0 
    valid = arr[~np.isnan(arr)]
    if valid.size == 0:
        raise RuntimeError("No valid curvature values to plot.")
    lim = max(np.nanpercentile(np.abs(valid), 98), 1e-8)
    vmin, vmax = -lim, +lim

    # Colormap + zero-centered normalization
    cmap, cmap_name = build_curvature_cmap(CURV_CMAP_MODE)
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    # Figure
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    im = ax.imshow(arr, cmap=cmap, norm=norm, extent=extent, origin='upper', zorder=1)

    # Boundary shapefile 
    plot_shapefile_on_map(ax, boundary_gdf_or_geom, line_color=BOUNDARY_COLOR, line_width=BOUNDARY_LINEWIDTH)

    # Optional mask boundary 
    try:
        if mask is not None:
            ax.contour(mask, levels=[0.5], colors='red', linewidths=0.4,
                       extent=extent, origin='upper', zorder=10)
    except Exception as e:
        print(f"[WARN] Mask boundary warn: {e}")

    # North arrow & checker scale bar
    add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
    add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                          segments=4, fontsize=4)

    # Axes
    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45); lbl.set_ha('right')

    # Colorbar
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label(COLORBAR_LABEL, fontsize=6, fontweight='bold')
    cb.ax.tick_params(labelsize=5)
    plt.tight_layout(pad=0.2)

    # Save PNG & PDF
    safe_savefig(fig, OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
    safe_savefig(fig, OUT_PDF, dpi=300, bbox_inches='tight', facecolor='white')

    print(f"[OK] Profile Curvature saved:\n  PNG: {OUT_PNG}\n  PDF: {OUT_PDF}\n  Colormap: {cmap_name}")
    plt.show()

In [ ]:
# ============================================================
# Step-1.7 — Profile Curvature Visualization 
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib import colors as mcolors
from rasterio.warp import transform_bounds

# Optional shapefile 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass

# PATHS 
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP_MASK      = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK_TIF = os.path.join(STEP_MASK, "study_mask.tif")
STUDY_SHP      = os.path.join(BASE, "Study_Area.shp")

OUT_DIR        = os.path.join(BASE, r"Results\Step-1.7-Curvatures")
PROF_CURV_TIF  = os.path.join(OUT_DIR, "profile_curv_32645_12p5m_float32.tif")

# PLOT STYLE  
FIGSIZE            = (1, 1)
SAVE_DPI           = 600
CAX_SIZE           = "8%"
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 2.5
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_COLOR     = 'black'
BOUNDARY_LINEWIDTH = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_CONTOUR_WIDTH = 0.4
LW_AXIS            = 0.3
XLABEL_PAD         = 0.5
YLABEL_PAD         = 0.5
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True

# White, opaque canvas
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

#  Safe save helpers  
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            os.chmod(path, stat.S_IWRITE)
    except Exception: pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs); print(f"[OUTPUT] Saved: {path}"); return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs); print(f"[OUTPUT] Saved: {alt}"); return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def force_axes_bg_opaque(ax, fig=None):
    if fig is None: fig = ax.figure
    fig.patch.set_facecolor('white'); fig.patch.set_alpha(1.0)
    ax.set_facecolor('white'); ax.patch.set_alpha(1.0)
    for child in fig.axes:
        try:
            child.set_facecolor('white'); child.patch.set_alpha(1.0)
        except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# DMS formatters & minute ticks 
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg)))
    m_float = (abs(decimal_deg) - d) * 60
    m = int(np.floor(m_float))
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'N' if decimal_deg >= 0 else 'S'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'E' if decimal_deg >= 0 else 'W'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min / 60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# North arrow & checkerboard scale bar 
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72; y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)

# Shapefile loader + boundary drawer 
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path): return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}": gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception: pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception: pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2):
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0])
                        ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception: pass
    # Mask Fallback
    try:
        from scipy.ndimage import binary_erosion as _be
        arr = _be(mask_array.astype(bool), iterations=erode_iters).astype(float)
    except Exception:
        arr = (mask_array.astype(float) if mask_array is not None else None)
    if arr is not None:
        ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                   extent=extent, origin='upper', zorder=11)
        return True
    return False

# Curvature colormap  
def build_curvature_cmap(mode="vivid"):
    mode = (mode or "").lower()

    if mode == "rdblu":      
        return plt.get_cmap("RdBu_r")
    if mode == "coolwarm":  
        return plt.get_cmap("coolwarm")
    if mode == "puor":       
        return mcolors.LinearSegmentedColormap.from_list(
            "PuOr_r",
            ["#2D004B", "#7F3B08"][::-1]  
        )
    if mode == "brbg":      
        return plt.get_cmap("BrBG_r")

    if mode == "blkor":      
        colors = ["#0B2E8C", "#5F8DFF", "#D4E4FF", "#FFFFFF", "#FFE0CC", "#FF8F3D", "#993D00"]
        return mcolors.LinearSegmentedColormap.from_list("BlueOrange", colors, N=256)

    if mode == "tealrose":   
        colors = ["#005F73", "#0A9396", "#94D2BD", "#FFFFFF", "#F4ACB7", "#E56B6F", "#6D2E46"]
        return mcolors.LinearSegmentedColormap.from_list("TealRose", colors, N=256)

    if mode == "vivid":      
        colors = ["#0B2E8C", "#2D62D0", "#78A7FF", "#FFFFFF", "#FFB36B", "#FF6A2E", "#990D0D"]
        return mcolors.LinearSegmentedColormap.from_list("CurvVivid", colors, N=256)

    # Fallback
    return plt.get_cmap("RdBu_r")

# =========================================================
# PLOT 
# =========================================================
if __name__ == "__main__":
    os.makedirs(OUT_DIR, exist_ok=True)

    # Load curvature raster + mask
    with rasterio.open(PROF_CURV_TIF) as src:
        arr         = src.read(1)
        curv_crs    = src.crs
        curv_bounds = src.bounds
        nodata      = src.nodata

    with rasterio.open(STUDY_MASK_TIF) as ms:
        mask = ms.read(1)

    # Extent WGS84
    west, south, east, north = transform_bounds(curv_crs, "EPSG:4326", *curv_bounds)
    extent = [west, east, south, north]

    # NoData + mask
    arr = np.where(arr == nodata, np.nan, arr)
    if mask.shape == arr.shape:
        arr = np.where(mask == 1, arr, np.nan)
    else:
        print(f"[WARN] Mask shape {mask.shape} != curvature shape {arr.shape}")

    # Symmetric limits around 0
    valid = arr[np.isfinite(arr)]
    if valid.size == 0:
        raise RuntimeError("No valid curvature pixels to plot.")
    lim = float(np.percentile(np.abs(valid), 98))
    if not np.isfinite(lim) or lim <= 0:
        lim = float(np.nanmax(np.abs(valid)))
    vmin, vmax = -lim, +lim
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    # Boundary
    boundary = load_shapefile_boundary(STUDY_SHP, 4326)

    # Draw map 
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)

    cmap = build_curvature_cmap()
    cmap.set_bad((1,1,1,1))
    im = ax.imshow(arr, cmap=cmap, norm=norm, extent=extent, origin='upper', zorder=1)

    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                            mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

    # Axes 
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
    for lbl in ax.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
    #ax.set_xlabel("Longitude", fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=XLABEL_PAD)
    #ax.set_ylabel("Latitude",  fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=YLABEL_PAD)

    # Cartography
    add_north_arrow_tri(ax)
    add_checker_scale_bar(ax, west, east, south, north, crs_type='wgs84', segments=4)

    # Right inset colorbar 
    cax = inset_axes(ax, width=CAX_SIZE, height="100%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax.transAxes, borderpad=0)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label('Profile Curvature (1/m)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
    cb.ax.set_facecolor('white'); cb.ax.patch.set_alpha(1.0)
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
    if hasattr(cb, "outline") and cb.outline is not None: cb.outline.set_linewidth(0.3)

    force_axes_bg_opaque(ax, fig)
    plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)

    # Exports 
    save_all(fig, os.path.join(OUT_DIR, "Profile_Curvature_Map"))
    plt.show()

    print("\n[DONE] Profile Curvature map →", OUT_DIR)


# Step-1.7: Visualization for Plan Curvature 

In [ ]:
# ============================================================
# Step-1.7 — Plan Curvature Visualization 
# ============================================================
import os
import time
import stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib import colors as mcolors  # for TwoSlopeNorm & custom cmap

# Optional Warp 
try:
    from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds
    _HAVE_WARP = True
except Exception:
    _HAVE_WARP = False

# Optional shapefile readers
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass

# -------------------- PATHS -----------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STUDY_MASK_TIF  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP       = os.path.join(BASE, "Study_Area.shp")   
OUT_DIR         = os.path.join(BASE, r"Results\Step-1.7-Curvatures")
PLAN_CURV_TIF   = os.path.join(OUT_DIR, "plan_curv_32645_12p5m_float32.tif")
OUT_PNG         = os.path.join(OUT_DIR, "Plan Curvature Map.png")
OUT_PDF         = os.path.join(OUT_DIR, "Plan Curvature Map.pdf")

# -------------------- STYLE -----------------------
FIGSIZE            = (2, 2)
CAX_SIZE           = "5%"
CAX_PAD            = 0.01
COLORBAR_LABEL     = 'Curvature (1/m)'
TITLE              = "Plan Curvature"
TICK_FONTSIZE      = 3
LABEL_FONTSIZE     = 5
TITLE_FONTSIZE     = 6
BOUNDARY_LINEWIDTH = 0.5
BOUNDARY_COLOR     = 'red'
CURV_CMAP_MODE   = 'vivid'

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS & MINUTE TICKS
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR  
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    # Arrow base position (lowered to avoid overlap with label)
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))

    ax.text(apex[0], y1 + size * float(label_upshift), label, transform=ax.transAxes,
            ha='center', va='bottom', fontsize=8, fontweight='bold',
            color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))

    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))

    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))

    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Utility 
# =========================================================
def load_plan_curvature_geo(tif_path, out_dir):
    geo_tif = os.path.join(out_dir, "plan_curv_geo.tif")
    os.makedirs(out_dir, exist_ok=True)

    with rasterio.open(tif_path) as src:
        src_crs = src.crs
        nodata  = src.nodata

        if _HAVE_WARP and src_crs and not src_crs.is_geographic:
            transform, width, height = calculate_default_transform(
                src_crs, "EPSG:4326", src.width, src.height, *src.bounds
            )
            kwargs = src.meta.copy()
            kwargs.update({'crs': "EPSG:4326", 'transform': transform, 'width': width, 'height': height})
            with rasterio.open(geo_tif, 'w', **kwargs) as dst:
                reproject(
                    source=rasterio.band(src, 1),
                    destination=rasterio.band(dst, 1),
                    src_transform=src.transform,
                    src_crs=src_crs,
                    dst_transform=transform,
                    dst_crs="EPSG:4326",
                    resampling=Resampling.bilinear
                )
            with rasterio.open(geo_tif) as ds:
                arr = ds.read(1)
                b = ds.bounds
                extent = [b.left, b.right, b.bottom, b.top]
                nodata = ds.nodata
            return arr, extent, nodata
        else:
            # Already geographic 
            arr = src.read(1)
            try:
                if _HAVE_WARP:
                    b = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
                    extent = [b[0], b[2], b[1], b[3]]
                else:
                    b = src.bounds
                    extent = [b.left, b.right, b.bottom, b.top]
            except Exception:
                b = src.bounds
                extent = [b.left, b.right, b.bottom, b.top]
            return arr, extent, nodata

# =========================================================
# Shapefile loader + plotting
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        print(f"[WARN] Shapefile not found: {shapefile_path}")
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception as e:
            print(f"[WARN] Geopandas failed: {e}")
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feature in shp:
                    if feature['geometry'] is not None:
                        geometries.append(feature['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception as e:
            print(f"[WARN] Fiona failed: {e}")
    return None

def plot_shapefile_on_map(ax, shapefile_data, line_color=BOUNDARY_COLOR, line_width=BOUNDARY_LINEWIDTH):
    if shapefile_data is None:
        return
    try:
        if HAVE_GEOPANDAS and hasattr(shapefile_data, 'plot'):
            shapefile_data.boundary.plot(ax=ax, color=line_color, linewidth=line_width, zorder=12)
        elif isinstance(shapefile_data, dict) and 'geometries' in shapefile_data:
            for geom in shapefile_data['geometries']:
                gt = geom['type']
                if gt == 'Polygon':
                    coords = geom['coordinates'][0]
                    x = [pt[0] for pt in coords]; y = [pt[1] for pt in coords]
                    ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
                elif gt == 'MultiPolygon':
                    for polygon in geom['coordinates']:
                        coords = polygon[0]
                        x = [pt[0] for pt in coords]; y = [pt[1] for pt in coords]
                        ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
                elif gt in ['LineString', 'MultiLineString']:
                    if gt == 'LineString':
                        coords = geom['coordinates']
                        x = [pt[0] for pt in coords]; y = [pt[1] for pt in coords]
                        ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
                    else:
                        for line in geom['coordinates']:
                            x = [pt[0] for pt in line]; y = [pt[1] for pt in line]
                            ax.plot(x, y, color=line_color, linewidth=line_width, zorder=12)
    except Exception as e:
        print(f"[WARN] Boundary plot error: {e}")

# =========================================================
# Curvature colormap builder 
# =========================================================
def build_curvature_cmap(mode='vivid'):
    mode = (mode or '').lower()
    if mode == 'rdblu':
        return plt.get_cmap('RdBu_r'), 'RdBu_r'
    if mode == 'coolwarm':
        return plt.get_cmap('coolwarm'), 'coolwarm'
    # Default vivid
    CURV_COLORS = [
        "#0B2E8C", "#2D62D0", "#78A7FF", "#FFFFFF",
        "#FFB36B", "#FF6A2E", "#990D0D"
    ]
    return mcolors.LinearSegmentedColormap.from_list("CurvVivid", CURV_COLORS, N=256), 'CurvVivid'

# =========================================================
# PLOT
# =========================================================
if __name__ == "__main__":
    # Load curvature 
    arr, extent, nodata = load_plan_curvature_geo(PLAN_CURV_TIF, OUT_DIR)
    west, east, south, north = extent

    # Study boundary shapefile 
    boundary_gdf_or_geom = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

    # Apply study mask boundary 
    mask = None
    try:
        with rasterio.open(STUDY_MASK_TIF) as ms:
            mask = ms.read(1)
            if mask.shape != arr.shape:
                mask = None
    except Exception as e:
        print(f"[WARN] Could not read mask: {e}")

    # Clean nodata
    arr = np.where(arr == nodata, np.nan, arr)

    # Symmetric robust limits around zero (98th percentile of |values|)
    valid = arr[~np.isnan(arr)]
    if valid.size == 0:
        raise RuntimeError("No valid curvature values to plot.")
    lim = max(np.nanpercentile(np.abs(valid), 98), 1e-8)
    vmin, vmax = -lim, +lim

    # High-contrast colormap + zero-centered normalization
    cmap, cmap_name = build_curvature_cmap(CURV_CMAP_MODE)
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    # Figure 
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    im = ax.imshow(arr, cmap=cmap, norm=norm, extent=extent, origin='upper', zorder=1)
    
    # Boundary shapefile 
    plot_shapefile_on_map(ax, boundary_gdf_or_geom, line_color=BOUNDARY_COLOR, line_width=BOUNDARY_LINEWIDTH)

    # Optional mask boundary 
    try:
        if mask is not None:
            ax.contour(mask, levels=[0.5], colors='red', linewidths=0.4,
                       extent=extent, origin='upper', zorder=10)
    except Exception as e:
        print(f"[WARN] Boundary draw warning: {e}")

    # North arrow & checker scale bar 
    add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
    add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                          segments=4, fontsize=4)

    # Axes limits & minute-gridded DMS ticks
    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45); lbl.set_ha('right')

    # Title (optional)
    # ax.set_title(f"{TITLE}  [{cmap_name}]", fontsize=TITLE_FONTSIZE, fontweight='bold', pad=2)

    # Side colorbar 
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label(COLORBAR_LABEL, fontsize=6, fontweight='bold')
    cb.ax.tick_params(labelsize=5)

    plt.tight_layout(pad=0.2)

    # Save PNG & PDF  
    safe_savefig(fig, OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
    safe_savefig(fig, OUT_PDF, dpi=300, bbox_inches='tight', facecolor='white')

    print(f"[OK] Plan Curvature saved:\n  PNG: {OUT_PNG}\n  PDF: {OUT_PDF}\n  Colormap: {cmap_name}")
    plt.show()

In [ ]:
# ============================================================
# Step-1.7 — Plan Curvature Visualization 
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib import colors as mcolors
from rasterio.warp import transform_bounds

# Optional Shapefile 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass

# PATHS 
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP_MASK      = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK_TIF = os.path.join(STEP_MASK, "study_mask.tif")
STUDY_SHP      = os.path.join(BASE, "Study_Area.shp")

OUT_DIR        = os.path.join(BASE, r"Results\Step-1.7-Curvatures")
PLAN_CURV_TIF  = os.path.join(OUT_DIR, "plan_curv_32645_12p5m_float32.tif")

# PLOT STYLE 
FIGSIZE            = (1, 1)
SAVE_DPI           = 600
CAX_SIZE           = "8%"
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 2.5
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_COLOR     = 'black'
BOUNDARY_LINEWIDTH = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_CONTOUR_WIDTH = 0.4
LW_AXIS            = 0.3
XLABEL_PAD         = 0.5
YLABEL_PAD         = 0.5
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True

# White, opaque canvas
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# Safe save helpers 
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            os.chmod(path, stat.S_IWRITE)
    except Exception: pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs); print(f"[OUTPUT] Saved: {path}"); return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs); print(f"[OUTPUT] Saved: {alt}"); return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def force_axes_bg_opaque(ax, fig=None):
    if fig is None: fig = ax.figure
    fig.patch.set_facecolor('white'); fig.patch.set_alpha(1.0)
    ax.set_facecolor('white'); ax.patch.set_alpha(1.0)
    for child in fig.axes:
        try:
            child.set_facecolor('white'); child.patch.set_alpha(1.0)
        except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# DMS formatters & minute ticks 
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg)))
    m_float = (abs(decimal_deg) - d) * 60
    m = int(np.floor(m_float))
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'N' if decimal_deg >= 0 else 'S'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'E' if decimal_deg >= 0 else 'W'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min / 60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# North arrow & checkerboard scale bar 
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72; y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)

# Shapefile loader + Boundary drawer 
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path): return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}": gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception: pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception: pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2):
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0])
                        ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception: pass
    # Mask Fallback
    try:
        from scipy.ndimage import binary_erosion as _be
        arr = _be(mask_array.astype(bool), iterations=erode_iters).astype(float)
    except Exception:
        arr = (mask_array.astype(float) if mask_array is not None else None)
    if arr is not None:
        ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                   extent=extent, origin='upper', zorder=11)
        return True
    return False

# Curvature colormap  
def build_curvature_cmap(mode="vivid"):
    mode = (mode or "").lower()
    if mode == "rdblu":     return plt.get_cmap("RdBu_r")
    if mode == "coolwarm":  return plt.get_cmap("coolwarm")
    if mode == "brbg":      return plt.get_cmap("BrBG_r")
    if mode == "tealrose":  return mcolors.LinearSegmentedColormap.from_list(
                                "TealRose", ["#005F73","#0A9396","#94D2BD","#FFFFFF","#F4ACB7","#E56B6F","#6D2E46"], N=256)
    if mode == "blkor":     return mcolors.LinearSegmentedColormap.from_list(
                                "BlueOrange", ["#0B2E8C","#5F8DFF","#D4E4FF","#FFFFFF","#FFE0CC","#FF8F3D","#993D00"], N=256)
    # Default Vivid
    return mcolors.LinearSegmentedColormap.from_list(
        "CurvVivid", ["#0B2E8C","#2D62D0","#78A7FF","#FFFFFF","#FFB36B","#FF6A2E","#990D0D"], N=256
    )

# =========================================================
# PLOT 
# =========================================================
if __name__ == "__main__":
    os.makedirs(OUT_DIR, exist_ok=True)

    # Load raster + mask
    with rasterio.open(PLAN_CURV_TIF) as src:
        arr        = src.read(1)
        curv_crs   = src.crs
        curv_bounds= src.bounds
        nodata     = src.nodata
    with rasterio.open(STUDY_MASK_TIF) as ms:
        mask = ms.read(1)

    # Extent in WGS84
    west, south, east, north = transform_bounds(curv_crs, "EPSG:4326", *curv_bounds)
    extent = [west, east, south, north]

    # NoData + mask
    arr = np.where(arr == nodata, np.nan, arr)
    if mask.shape == arr.shape:
        arr = np.where(mask == 1, arr, np.nan)
    else:
        print(f"[WARN] Mask shape {mask.shape} != curvature shape {arr.shape}")

    # Symmetric robust limits
    valid = arr[np.isfinite(arr)]
    if valid.size == 0:
        raise RuntimeError("No valid curvature pixels to plot.")
    lim = float(np.percentile(np.abs(valid), 98))
    if not np.isfinite(lim) or lim <= 0:
        lim = float(np.nanmax(np.abs(valid)))
    vmin, vmax = -lim, +lim
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    # Boundary
    boundary = load_shapefile_boundary(STUDY_SHP, 4326)

    # Draw map  
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)

    cmap = build_curvature_cmap("vivid")  
    cmap.set_bad((1,1,1,1))
    im = ax.imshow(arr, cmap=cmap, norm=norm, extent=extent, origin='upper', zorder=1)

    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                            mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

    # Axes 
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
    for lbl in ax.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
    # ax.set_xlabel("Longitude", fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=XLABEL_PAD)
    # ax.set_ylabel("Latitude",  fontsize=LABEL_FONTSIZE, fontweight='bold', labelpad=YLABEL_PAD)

    # Cartography
    add_north_arrow_tri(ax)
    add_checker_scale_bar(ax, west, east, south, north, crs_type='wgs84', segments=4)

    # Right inset colorbar 
    cax = inset_axes(ax, width=CAX_SIZE, height="100%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax.transAxes, borderpad=0)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label('Plan Curvature (1/m)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
    cb.ax.set_facecolor('white'); cb.ax.patch.set_alpha(1.0)
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
    if hasattr(cb, "outline") and cb.outline is not None: cb.outline.set_linewidth(0.3)

    force_axes_bg_opaque(ax, fig)
    plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)

    # Exports
    save_all(fig, os.path.join(OUT_DIR, "Plan_Curvature_Map"))
    plt.show()

    print("\n[DONE] Plan Curvature map →", OUT_DIR)


# Step-1.8: TPI (Topographic Position Index)

In [ ]:
# ============================================================
# Step-1.8 — TPI (Topographic Position Index)
# ============================================================
import os, json, warnings
from typing import Tuple, Optional, Dict
import numpy as np
import rasterio

#-------------------------PATH---------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
# Input Path
DEM_FILLED_TIF  = os.path.join(BASE, r"Results\Step-1.4-HydroConditioning", "dem_filled.tif")
STUDY_MASK_TIF  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")

# Output Path
OUT_DIR         = os.path.join(BASE, r"Results\Step-1.8-TPI"); os.makedirs(OUT_DIR, exist_ok=True)
TPI_S_SMALL_TIF = os.path.join(OUT_DIR, "tpi_s5_32645_12p5m_float32.tif")
TPI_S_LARGE_TIF = os.path.join(OUT_DIR, "tpi_s25_32645_12p5m_float32.tif")
TPI_STATS_JSON  = os.path.join(OUT_DIR, "tpi_stats.json")

#---------------------SETTINGS-----------------------
NODATA = -9999.0

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

try:
    from scipy.ndimage import uniform_filter
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
    warnings.warn("scipy.ndimage not found; using numpy fallback for mean filter.")

def _read_mask(path: str) -> np.ndarray:
    with rasterio.open(path) as ds:
        m = ds.read(1, masked=False)
        if ds.nodata is not None:
            m = np.where(m == ds.nodata, 0, m)
        return (m != 0).astype(np.uint8)

def _save_float32(out_path: str, arr: np.ndarray, ref_path: str, nodata: float = NODATA):
    arr = arr.astype(np.float32, copy=False)
    with rasterio.open(ref_path) as ref:
        meta = ref.meta.copy()
    meta.update(dtype="float32", nodata=nodata, compress="LZW")
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(arr, 1)

def _hist_png(values: np.ndarray, out_png: str, title: str):
    if not HAVE_MPL:
        return
    v = values[np.isfinite(values)]
    v = v[(v != NODATA)]
    if v.size == 0:
        return
    plt.figure(figsize=(6,4))
    plt.hist(v, bins=80)
    plt.title(title)
    plt.xlabel("value")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def _stats(values: np.ndarray, mask: Optional[np.ndarray] = None) -> Dict:
    if mask is not None:
        values = np.where(mask, values, np.nan)
    v = values[np.isfinite(values)]
    v = v[(v != NODATA)]
    if v.size == 0:
        return {"count": 0}
    q = np.nanpercentile(v, [0,1,5,25,50,75,95,99,100])
    return {
        "count": int(v.size),
        "min": float(np.nanmin(v)),
        "p01": float(q[1]),
        "p05": float(q[2]),
        "p25": float(q[3]),
        "p50": float(q[4]),
        "p75": float(q[5]),
        "p95": float(q[6]),
        "p99": float(q[7]),
        "max": float(np.nanmax(v)),
        "mean": float(np.nanmean(v)),
        "std": float(np.nanstd(v)),
    }

def _uniform_mean(img: np.ndarray, size: int) -> np.ndarray:
    if HAVE_SCIPY:
        return uniform_filter(img, size=size, mode="nearest")
    k = np.ones((size, size), dtype=np.float64) / (size*size)
    pad = size//2
    padded = np.pad(img, ((pad,pad),(pad,pad)), mode="edge")
    out = np.empty_like(img, dtype=np.float64)
    h, w = img.shape
    for i in range(h):
        for j in range(w):
            out[i, j] = np.sum(padded[i:i+size, j:j+size] * k)
    return out

def compute_tpi(dem_filled_path: str, study_mask: np.ndarray,
                small_win: int = 5, large_win: int = 25) -> Tuple[np.ndarray, np.ndarray]:
    with rasterio.open(dem_filled_path) as ds:
        dem = ds.read(1).astype(np.float64)
        if ds.nodata is not None:
            dem = np.where(dem == ds.nodata, np.nan, dem)

    nan_mask = ~np.isfinite(dem)
    if np.any(nan_mask):
        dem[nan_mask] = np.nanmedian(dem)

    mean_small = _uniform_mean(dem, size=small_win)
    mean_large = _uniform_mean(dem, size=large_win)

    tpi_small = dem - mean_small
    tpi_large = dem - mean_large

    if study_mask is not None:
        tpi_small[study_mask == 0] = NODATA
        tpi_large[study_mask == 0] = NODATA

    return tpi_small.astype(np.float32), tpi_large.astype(np.float32)

mask = _read_mask(STUDY_MASK_TIF)
tpi_s, tpi_l = compute_tpi(DEM_FILLED_TIF, mask, small_win=5, large_win=25)
_save_float32(TPI_S_SMALL_TIF, tpi_s, DEM_FILLED_TIF, NODATA)
_save_float32(TPI_S_LARGE_TIF, tpi_l, DEM_FILLED_TIF, NODATA)

tstats = {
    "tpi_small_5":  _stats(tpi_s, mask),
    "tpi_large_25": _stats(tpi_l, mask),
}
with open(TPI_STATS_JSON, "w") as f:
    json.dump(tstats, f, indent=2)

if HAVE_MPL:
    _hist_png(tpi_s, os.path.join(OUT_DIR, "tpi_s5_hist.png"),  "TPI small (5)")
    _hist_png(tpi_l, os.path.join(OUT_DIR, "tpi_s25_hist.png"), "TPI large (25)")

print("Step-1.8 completed.")


# Visualization Code for TPI (Topographic Position Index)

In [ ]:
# ============================================================
# Step-1.8 — Visualization for TPI 
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds

# Optional deps 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion
    HAVE_SCIPY = True
except Exception:
    pass

# ------------------------- PATHS ---------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STUDY_MASK_TIF   = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP        = os.path.join(BASE, "Study_Area.shp")

OUT_DIR          = os.path.join(BASE, r"Results\Step-1.8-TPI")
TPI_S_SMALL_TIF  = os.path.join(OUT_DIR, "tpi_s5_32645_12p5m_float32.tif")
TPI_S_LARGE_TIF  = os.path.join(OUT_DIR, "tpi_s25_32645_12p5m_float32.tif")
OUT_SMALL_PNG    = os.path.join(OUT_DIR, "TPI_Small_Map.png")
OUT_SMALL_PDF    = os.path.join(OUT_DIR, "TPI_Small_Map.pdf")
OUT_LARGE_PNG    = os.path.join(OUT_DIR, "TPI_Large_Map.png")
OUT_LARGE_PDF    = os.path.join(OUT_DIR, "TPI_Large_Map.pdf")

# ------------------------- STYLE ---------------------
FIGSIZE           = (2, 2)
CAX_SIZE          = "5%"
CAX_PAD           = 0.01
TICK_FONTSIZE     = 3
CB_LABEL_FONTSIZE = 6
CB_TICK_FONTSIZE  = 5
BOUNDARY_COLOR    = 'red'
BOUNDARY_LW       = 0.3
MASK_COLOR        = 'red'
MASK_LW           = 0.3

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + Unified Boundary Drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw exactly ONE boundary: shapefile if available, else eroded mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates']); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float) if HAVE_SCIPY \
                  else mask_array.astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# Reusable plotter 
# =========================================================
def plot_tpi_compact(tpi_tif: str, mask_tif: str, out_png: str, out_pdf: str):
    # Load rasters
    with rasterio.open(tpi_tif) as src:
        tpi    = src.read(1)
        crs    = src.crs
        bounds = src.bounds
        nodata = src.nodata
    with rasterio.open(mask_tif) as src:
        mask = src.read(1)

    # Apply mask and nodata
    arr = np.where(tpi == nodata, np.nan, tpi)
    if mask.shape == arr.shape:
        arr = np.where(mask == 1, arr, np.nan)
    else:
        print(f"[WARN] Mask shape {mask.shape} != raster shape {arr.shape}; mask overlay may be skipped.")

    # Extent in WGS84
    west, south, east, north = transform_bounds(crs, "EPSG:4326", *bounds)
    extent = [west, east, south, north]

    # Robust symmetric stretch around 0 
    valid = arr[~np.isnan(arr)]
    if valid.size == 0:
        raise RuntimeError("No valid TPI pixels to plot.")
    lim = max(np.nanpercentile(np.abs(valid), 98), 1e-8)
    vmin, vmax = -lim, +lim

    # Optional shapefile boundary
    boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

    # Figure 
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    im = ax.imshow(arr, cmap='PiYG', extent=extent, origin='upper',
                   vmin=vmin, vmax=vmax, zorder=1)

    # Single boundary 
    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                            mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

    # North arrow & checker scale bar
    add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
    add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                          segments=4, fontsize=4)

    # Axes: DMS minute ticks + tiny fonts
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45); lbl.set_ha('right')

    # Colorbar 
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label('TPI (m)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

    # Tiny stats box
    #stats_txt = (
        #f"|TPI| 98%: {lim:.2f} m\n"
        #f"Min/Max: {np.nanmin(valid):.2f}/{np.nanmax(valid):.2f} m\n"
        #f"Mean: {np.nanmean(valid):.2f} m\n"
        #f"Median: {np.nanmedian(valid):.2f} m"
    #)
    #ax.text(0.70, 0.25, stats_txt,
            #transform=ax.transAxes, va='top',
            #bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            #fontsize=3)

    # Save 
    plt.tight_layout(pad=0.2)
    safe_savefig(fig, out_png, dpi=300, bbox_inches='tight', facecolor='white')
    safe_savefig(fig, out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    # Console summary
    print("\n" + "="*60)
    print(f"TPI SUMMARY — {os.path.basename(tpi_tif)}")
    print("="*60)
    print(f"CRS: {crs}")
    print(f"BBox (WGS84): W={format_dms_lon(west)}, E={format_dms_lon(east)}, "
          f"S={format_dms_lat(south)}, N={format_dms_lat(north)}")
    print(f"Range: {np.nanmin(valid):.3f} – {np.nanmax(valid):.3f} m")
    print(f"Mean ± Std: {np.nanmean(valid):.3f} ± {np.nanstd(valid):.3f} m")
    print(f"Median: {np.nanmedian(valid):.3f} m")
    print("="*60)

# =========================================================
# RUN BOTH MAPS 
# =========================================================
if __name__ == "__main__":
    plot_tpi_compact(
        tpi_tif=TPI_S_SMALL_TIF,
        mask_tif=STUDY_MASK_TIF,
        out_png=OUT_SMALL_PNG,
        out_pdf=OUT_SMALL_PDF
    )
    plot_tpi_compact(
        tpi_tif=TPI_S_LARGE_TIF,
        mask_tif=STUDY_MASK_TIF,
        out_png=OUT_LARGE_PNG,
        out_pdf=OUT_LARGE_PDF
    )


# Updated Code for TPI Visualization

In [ ]:
# ============================================================
# Step-1.8 — TPI Visualization 
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib import colors as mcolors
from rasterio.warp import transform_bounds

# Optional Shapefile 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass

# ---------------------------- PATHS -------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STUDY_MASK_TIF   = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP        = os.path.join(BASE, "Study_Area.shp")

OUT_DIR          = os.path.join(BASE, r"Results\Step-1.8-TPI")
TPI_S_SMALL_TIF  = os.path.join(OUT_DIR, "tpi_s5_32645_12p5m_float32.tif")
TPI_S_LARGE_TIF  = os.path.join(OUT_DIR, "tpi_s25_32645_12p5m_float32.tif")

# -------------------------- PLOT STYLE ----------------------
FIGSIZE            = (1, 1)
SAVE_DPI           = 600
CAX_SIZE           = "8%"     
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 2.5
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_COLOR     = 'black'
BOUNDARY_LINEWIDTH = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_CONTOUR_WIDTH = 0.4
LW_AXIS            = 0.3
XLABEL_PAD         = 0.5
YLABEL_PAD         = 0.5
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True

# Opaque, white exports
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try:
                    im.set_alpha(1.0)
                except Exception:
                    pass

def force_axes_bg_opaque(ax, fig=None):
    if fig is None:
        fig = ax.figure
    fig.patch.set_facecolor('white'); fig.patch.set_alpha(1.0)
    ax.set_facecolor('white'); ax.patch.set_alpha(1.0)
    for child in fig.axes:
        try:
            child.set_facecolor('white'); child.patch.set_alpha(1.0)
        except Exception:
            pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF:
        safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG:
        safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# =========================================================
# DMS formatters & minute ticks
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg)))
    m_float = (abs(decimal_deg) - d) * 60
    m = int(np.floor(m_float))
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'N' if decimal_deg >= 0 else 'S'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    hemi = 'E' if decimal_deg >= 0 else 'W'
    return f"{d}°{m:02d}′{s:02.0f}″{hemi}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min / 60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# North arrow & checkerboard scale bar
# =========================================================
def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1, 2, 5, 10):
        if m*base >= target:
            return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72; y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)

# =========================================================
# Shapefile loader + boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path): return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2):
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0])
                        ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0])
                            ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass
    # Mask fallback
    try:
        from scipy.ndimage import binary_erosion as _be
        arr = _be(mask_array.astype(bool), iterations=erode_iters).astype(float)
    except Exception:
        arr = (mask_array.astype(float) if mask_array is not None else None)
    if arr is not None:
        ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                   extent=extent, origin='upper', zorder=11)
        return True
    return False

# =========================================================
# Dark, high-contrast diverging colormaps
# =========================================================
def build_tpi_cmap(mode="deep"):
    mode = (mode or "").lower()

    if mode == "rdblu":    return plt.get_cmap("RdBu_r")
    if mode == "coolwarm": return plt.get_cmap("coolwarm")
    if mode == "brbg":     return plt.get_cmap("BrBG_r")
    if mode == "blkor":
        return mcolors.LinearSegmentedColormap.from_list(
            "BlueOrange_Dark",
            ["#0A2A6B", "#2F61A8", "#A8C4EF", "#EDEBE6", "#F5B087", "#D26022", "#6A2B00"],
            N=256
        )

    if mode == "fireice":
        # Deep navy → teal → pale sand → rust → maroon 
        return mcolors.LinearSegmentedColormap.from_list(
            "FireIce_Dark",
            ["#06172A", "#06304F", "#0B6D8C", "#E8D9A8", "#B33C24", "#6A0F0F"],
            N=256
        )

    if mode == "dense_brbg":
        # BrBG-inspired but darker ends and compressed light center
        return mcolors.LinearSegmentedColormap.from_list(
            "BrBG_DenseDark",
            ["#08306B", "#2C7FB8", "#A6BDDB", "#EFEDE8", "#D9A779", "#B2622B", "#7F3B08"],
            N=256
        )

    # Default: deep teal → jade → warm sand → oxblood 
    return mcolors.LinearSegmentedColormap.from_list(
        "Deep_Diverge",
        ["#041F1F", "#0A6262", "#5CB8A6", "#E9E4D8", "#B86A5B", "#6E1F1A"],
        N=256
    )

# =========================================================
# Core plotter 
# =========================================================
def plot_tpi_compact(tpi_tif: str, out_basename: str, cmap_mode="deep", pctl=95):
    # Load rasters
    with rasterio.open(tpi_tif) as src:
        tpi    = src.read(1)
        crs    = src.crs
        bounds = src.bounds
        nodata = src.nodata
    with rasterio.open(STUDY_MASK_TIF) as ms:
        mask = ms.read(1)

    # Apply mask & nodata
    arr = np.where(tpi == nodata, np.nan, tpi)
    if mask.shape == arr.shape:
        arr = np.where(mask == 1, arr, np.nan)
    else:
        print(f"[WARN] Mask shape {mask.shape} != raster shape {arr.shape}")

    # Extent in WGS84
    west, south, east, north = transform_bounds(crs, "EPSG:4326", *bounds)
    extent = [west, east, south, north]

    # Symmetric robust limits 
    valid = arr[np.isfinite(arr)]
    if valid.size == 0:
        raise RuntimeError("No valid TPI pixels to plot.")
    lim = float(np.percentile(np.abs(valid), pctl))
    if not np.isfinite(lim) or lim <= 0:
        lim = float(np.nanmax(np.abs(valid)))
    vmin, vmax = -lim, +lim
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    # Boundary
    boundary = load_shapefile_boundary(STUDY_SHP, 4326)

    # Draw map
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    for sp in ax.spines.values():
        sp.set_linewidth(LW_AXIS)

    cmap = build_tpi_cmap(cmap_mode)
    cmap.set_bad((1, 1, 1, 1))
    im = ax.imshow(arr, cmap=cmap, norm=norm, extent=extent, origin='upper', zorder=1)

    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                            mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

    # Axes
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(XROT_DEG); lbl.set_ha('right')

    # Cartography
    add_north_arrow_tri(ax)
    add_checker_scale_bar(ax, west, east, south, north, crs_type='wgs84', segments=4)

    # Right inset colorbar 
    cax = inset_axes(ax, width=CAX_SIZE, height="100%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax.transAxes, borderpad=0)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label('TPI (m)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
    cb.ax.set_facecolor('white'); cb.ax.patch.set_alpha(1.0)
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
    if hasattr(cb, "outline") and cb.outline is not None:
        cb.outline.set_linewidth(0.3)

    force_axes_bg_opaque(ax, fig)
    plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)

    # Exports
    base = os.path.join(OUT_DIR, out_basename)
    save_all(fig, base)
    plt.show()

    # Console summary
    print("\n" + "="*60)
    print(f"TPI SUMMARY — {os.path.basename(tpi_tif)}")
    print("="*60)
    print(f"BBox (WGS84): W={format_dms_lon(west)}  E={format_dms_lon(east)}  "
          f"S={format_dms_lat(south)}  N={format_dms_lat(north)}")
    print(f"|TPI| {pctl}%: {lim:.3f} m  |  Range: {np.nanmin(valid):.3f} … {np.nanmax(valid):.3f} m")
    print(f"Mean ± Std: {np.nanmean(valid):.3f} ± {np.nanstd(valid):.3f} m  |  Median: {np.nanmedian(valid):.3f} m")
    print("="*60)

# =========================================================
# Run both maps
# =========================================================
if __name__ == "__main__":
    os.makedirs(OUT_DIR, exist_ok=True)

    # Small-window TPI 
    plot_tpi_compact(
        tpi_tif=TPI_S_SMALL_TIF,
        out_basename="TPI_Small_Map",
        cmap_mode="deep",   
        pctl=95
    )

    # Large-window TPI 
    plot_tpi_compact(
        tpi_tif=TPI_S_LARGE_TIF,
        out_basename="TPI_Large_Map",
        cmap_mode="deep",
        pctl=95
    )


# Step-1.9: SPI (Stream Power Index)

In [ ]:
# ============================================================
# Step-1.9 — SPI (Stream Power Index)
# ============================================================
import os, json
from typing import Tuple, Optional, Dict
import numpy as np
import rasterio

#-------------------PATH-------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
# Input Path
SCA_TIF         = os.path.join(BASE, r"Results\Step-1.5-FlowAccumulation", "sca_m.tif")
SLOPE_DEG_TIF   = os.path.join(BASE, r"Results\Step-1.1-Slope_Aspect", "slope_deg.tif")
STUDY_MASK_TIF  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")

# Output Path
OUT_DIR         = os.path.join(BASE, r"Results\Step-1.9-SPI"); os.makedirs(OUT_DIR, exist_ok=True)
SPI_RAW_TIF     = os.path.join(OUT_DIR, "spi_raw_32645_12p5m_float32.tif")
SPI_LN_TIF      = os.path.join(OUT_DIR, "spi_ln_32645_12p5m_float32.tif")
SPI_STATS_JSON  = os.path.join(OUT_DIR, "spi_stats.json")

#-----------------------SETTINGS------------------
NODATA = -9999.0
EPS    = 1e-8

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

def _read_mask(path: str) -> np.ndarray:
    with rasterio.open(path) as ds:
        m = ds.read(1, masked=False)
        if ds.nodata is not None:
            m = np.where(m == ds.nodata, 0, m)
        return (m != 0).astype(np.uint8)

def _save_float32(out_path: str, arr: np.ndarray, ref_path: str, nodata: float = NODATA):
    arr = arr.astype(np.float32, copy=False)
    with rasterio.open(ref_path) as ref:
        meta = ref.meta.copy()
    meta.update(dtype="float32", nodata=nodata, compress="LZW")
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(arr, 1)

def _hist_png(values: np.ndarray, out_png: str, title: str):
    if not HAVE_MPL:
        return
    v = values[np.isfinite(values)]
    v = v[(v != NODATA)]
    if v.size == 0:
        return
    plt.figure(figsize=(6,4))
    plt.hist(v, bins=80)
    plt.title(title)
    plt.xlabel("value")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def _stats(values: np.ndarray, mask: Optional[np.ndarray] = None) -> Dict:
    if mask is not None:
        values = np.where(mask, values, np.nan)
    v = values[np.isfinite(values)]
    v = v[(v != NODATA)]
    if v.size == 0:
        return {"count": 0}
    q = np.nanpercentile(v, [0,1,5,25,50,75,95,99,100])
    return {
        "count": int(v.size),
        "min": float(np.nanmin(v)),
        "p01": float(q[1]),
        "p05": float(q[2]),
        "p25": float(q[3]),
        "p50": float(q[4]),
        "p75": float(q[5]),
        "p95": float(q[6]),
        "p99": float(q[7]),
        "max": float(np.nanmax(v)),
        "mean": float(np.nanmean(v)),
        "std": float(np.nanstd(v)),
    }

def compute_spi(sca_path: str, slope_deg_path: str, study_mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray, str]:
    with rasterio.open(sca_path) as ds_sca:
        sca = ds_sca.read(1).astype(np.float64)
        if ds_sca.nodata is not None:
            sca = np.where(sca == ds_sca.nodata, np.nan, sca)
        ref_path = slope_deg_path

    with rasterio.open(slope_deg_path) as ds_slope:
        slope_deg = ds_slope.read(1).astype(np.float64)
        if ds_slope.nodata is not None:
            slope_deg = np.where(slope_deg == ds_slope.nodata, np.nan, slope_deg)

    slope_rad = np.deg2rad(slope_deg)
    tanb = np.tan(slope_rad)

    spi_raw = sca * tanb
    spi_ln  = np.log(spi_raw + EPS)

    for arr in (spi_raw, spi_ln):
        arr[~np.isfinite(arr)] = np.nan
        if study_mask is not None:
            arr[study_mask == 0] = np.nan

    spi_raw = np.where(np.isnan(spi_raw), NODATA, spi_raw)
    spi_ln  = np.where(np.isnan(spi_ln),  NODATA, spi_ln)

    return spi_raw.astype(np.float32), spi_ln.astype(np.float32), ref_path

mask = _read_mask(STUDY_MASK_TIF)
spi_raw, spi_ln, ref_for_spi = compute_spi(SCA_TIF, SLOPE_DEG_TIF, mask)
_save_float32(SPI_RAW_TIF, spi_raw, ref_for_spi, NODATA)
_save_float32(SPI_LN_TIF,  spi_ln,  ref_for_spi, NODATA)

sstats = {
    "spi_raw": _stats(spi_raw, mask),
    "spi_ln":  _stats(spi_ln,  mask),
    "note": "SPI_ln = ln(SCA * tan(slope_rad) + eps). Prefer spi_ln for stacking to stabilize range."
}
with open(SPI_STATS_JSON, "w") as f:
    json.dump(sstats, f, indent=2)

if HAVE_MPL:
    _hist_png(spi_raw, os.path.join(OUT_DIR, "spi_raw_hist.png"), "SPI raw")
    _hist_png(spi_ln,  os.path.join(OUT_DIR, "spi_ln_hist.png"),  "SPI ln")

print("Step-1.9 completed.")

# Visualization Code for SPI (Stream Power Index)

In [ ]:
# ============================================================
# Step-1.9 — Visualization for SPI 
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds

# Optional deps (prefer shapefile boundary; mask fallback)
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion
    HAVE_SCIPY = True
except Exception:
    pass

# ------------------- PATHS -------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STUDY_MASK_TIF = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP      = os.path.join(BASE, "Study_Area.shp")

OUT_DIR     = os.path.join(BASE, r"Results\Step-1.9-SPI")
SPI_RAW_TIF = os.path.join(OUT_DIR, "spi_raw_32645_12p5m_float32.tif")
SPI_LN_TIF  = os.path.join(OUT_DIR, "spi_ln_32645_12p5m_float32.tif")

OUT_RAW_PNG = os.path.join(OUT_DIR, "SPI_Raw_Map.png")
OUT_RAW_PDF = os.path.join(OUT_DIR, "SPI_Raw_Map.pdf")
OUT_LN_PNG  = os.path.join(OUT_DIR, "SPI_Ln_Map.png")
OUT_LN_PDF  = os.path.join(OUT_DIR, "SPI_Ln_Map.pdf")

# ------------------- SETTINGS / STYLE ----------------------
NODATA = -9999.0
EPS    = 1e-8

FIGSIZE           = (2, 2)
CAX_SIZE          = "5%"
CAX_PAD           = 0.01
TICK_FONTSIZE     = 3
CB_LABEL_FONTSIZE = 6
CB_TICK_FONTSIZE  = 5
BOUNDARY_COLOR    = 'red'
BOUNDARY_LW       = 0.3
MASK_COLOR        = 'red'
MASK_LW           = 0.3

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS + 5′/10′ ticks (Step-1.3 style)
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km)) ) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw exactly ONE boundary: shapefile if available, else eroded mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates']); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float) if HAVE_SCIPY \
                  else mask_array.astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# CORE PLOTTER (compact Step-1.3 style, NO STATS BOX)
# =========================================================
def plot_spi_compact(spi_tif: str, mask_tif: str, out_png: str, out_pdf: str, mode: str):
    with rasterio.open(spi_tif) as src:
        data   = src.read(1)
        crs    = src.crs
        bounds = src.bounds
        nodata = src.nodata if src.nodata is not None else NODATA
    with rasterio.open(mask_tif) as src:
        mask = src.read(1)

    # Raw & masked
    raw = np.where(data == nodata, np.nan, data)
    if mask.shape == raw.shape:
        raw = np.where(mask == 1, raw, np.nan)
    else:
        print(f"[WARN] Mask shape {mask.shape} != raster shape {raw.shape}; mask overlay may be skipped.")

    # Extent (WGS84)
    west, south, east, north = transform_bounds(crs, "EPSG:4326", *bounds)
    extent = [west, east, south, north]

    # Visualization transform + labels
    if mode == "raw":
        vis = np.where(np.isnan(raw), np.nan, np.log10(raw + EPS))
        cmap = "YlGnBu"
        cbar_label = "log10(SPI)"
    elif mode == "ln":
        vis = raw.copy()
        cmap = "YlOrBr"
        cbar_label = "SPI (ln)"
    else:
        raise ValueError("mode must be 'raw' or 'ln'")

    # Robust 2–98% stretch on the displayed values
    valid_vis = vis[~np.isnan(vis)]
    if valid_vis.size == 0:
        raise RuntimeError("No valid SPI pixels to plot.")
    vmin = np.nanpercentile(valid_vis, 2)
    vmax = np.nanpercentile(valid_vis, 98)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = np.nanmin(valid_vis), np.nanmax(valid_vis)

    # Optional shapefile boundary
    boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

    # ---- Figure (exact Step-1.3 grammar) ----
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    im = ax.imshow(vis, cmap=cmap, extent=extent, origin='upper',
                   vmin=vmin, vmax=vmax, zorder=1)

    # Single boundary (shp preferred, else mask)
    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                            mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

    # North arrow & checker scale bar
    add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
    add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                          segments=4, fontsize=4)

    # Axes: DMS ticks + tiny fonts
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45); lbl.set_ha('right')

    # Colorbar (right, 5% width, 0.01 pad)
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label(cbar_label, fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

    # Save (PNG + PDF)
    plt.tight_layout(pad=0.2)
    safe_savefig(fig, out_png, dpi=300, bbox_inches='tight', facecolor='white')
    safe_savefig(fig, out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    # Console summary 
    raw_valid = raw[~np.isnan(raw)]
    print("\n" + "="*60)
    print(f"{('Stream Power Index (raw→log10 display)' if mode=='raw' else 'Stream Power Index (ln)')} SUMMARY")
    print("="*60)
    print(f"CRS: {crs}")
    print(f"BBox (WGS84): W={format_dms_lon(west)}, E={format_dms_lon(east)}, "
          f"S={format_dms_lat(south)}, N={format_dms_lat(north)}")
    print(f"Raw SPI  Min/Max: {np.nanmin(raw_valid):.6f} — {np.nanmax(raw_valid):.6f}")
    print(f"Raw SPI  Mean ± Std: {np.nanmean(raw_valid):.6f} ± {np.nanstd(raw_valid):.6f}")
    print(f"Raw SPI  Median: {np.nanmedian(raw_valid):.6f}")
    print(f"Display stretch ({'log10' if mode=='raw' else 'ln'}): {vmin:.3f} → {vmax:.3f}")
    print("="*60)

# =========================================================
# RUN BOTH MAPS 
# =========================================================
if __name__ == "__main__":
    # SPI raw visualized as log10
    plot_spi_compact(
        spi_tif=SPI_RAW_TIF,
        mask_tif=STUDY_MASK_TIF,
        out_png=OUT_RAW_PNG,
        out_pdf=OUT_RAW_PDF,
        mode="raw"
    )
    # SPI ln visualized directly
    plot_spi_compact(
        spi_tif=SPI_LN_TIF,
        mask_tif=STUDY_MASK_TIF,
        out_png=OUT_LN_PNG,
        out_pdf=OUT_LN_PDF,
        mode="ln"
    )


# Updated Code for SPI Visualization

In [ ]:
# ============================================================
# Step-1.9 — SPI Visualization 
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from rasterio.warp import transform_bounds

# ------------- Optional Shapefile -------------
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion; HAVE_SCIPY = True
except Exception:
    pass

# ----------------------------- PATHS ------------------------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STUDY_MASK_TIF = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP      = os.path.join(BASE, "Study_Area.shp")

OUT_DIR     = os.path.join(BASE, r"Results\Step-1.9-SPI")
SPI_RAW_TIF = os.path.join(OUT_DIR, "spi_raw_32645_12p5m_float32.tif")   
SPI_LN_TIF  = os.path.join(OUT_DIR, "spi_ln_32645_12p5m_float32.tif")    

# Outputs
OUT_RAW_BASENAME = os.path.join(OUT_DIR, "SPI_Raw_Map")  
OUT_LN_BASENAME  = os.path.join(OUT_DIR, "SPI_Ln_Map")

# ----------------------------- STYLE -----------------------------------------
FIGSIZE            = (1, 1)     
SAVE_DPI           = 600
CAX_SIZE           = "8%"       
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 2.5
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_COLOR     = 'black'
BOUNDARY_LINEWIDTH = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_CONTOUR_WIDTH = 0.4
LW_AXIS            = 0.3
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True

# Canvas: opaque white 
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# ----------------------------- Helpers: save/IO --------------------------------
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs); print(f"[OUTPUT] Saved: {path}"); return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs); print(f"[OUTPUT] Saved: {alt}"); return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# ----------------------------- DMS ticks --------------------------------------
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg))); m_float = (abs(decimal_deg)-d)*60
    m = int(np.floor(m_float)); s = (m_float - m)*60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min/60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# ------------------- North arrow & checkerboard scale bar ---------------------
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size*0.72; y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size*0.09)
    ax.add_patch(Polygon([apex, left_base, tail], closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north)/2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0/mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i*seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    end_km = total_len_m/1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)

# ------------------- Shapefile loader + Unified Boundary ----------------------
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path): return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}": gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2):
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass
    # Mask fallback
    try:
        arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float)
    except Exception:
        arr = (mask_array.astype(float) if mask_array is not None else None)
    if arr is not None:
        ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                   extent=extent, origin='upper', zorder=11)
        return True
    return False

# ------------------- Colormap choice  -----------------------
def build_spi_cmap(mode="inferno"):
    mode = (mode or "").lower()
    if mode in {"inferno","magma","plasma","cividis","viridis"}:
        return plt.get_cmap(mode)
    if mode == "terrain_dark":
        # Custom dark terrain-like ramp
        from matplotlib import colors as mcolors
        return mcolors.LinearSegmentedColormap.from_list(
            "terrain_dark",
            ["#0D1B2A","#1B263B","#274C77","#4C956C","#B5C99A","#F2E8CF","#E4C666","#C06E52"],
            N=256
        )
    return plt.get_cmap("inferno")

# ----------------------------- Core plotter -----------------------------------
EPS = 1e-8  # for log display safety

def plot_spi_compact(spi_tif: str, out_basename: str,
                     display="log10",  
                     cmap_mode="inferno",
                     p_lo=2, p_hi=98):
    # Load rasters
    with rasterio.open(spi_tif) as src:
        data   = src.read(1)
        crs    = src.crs
        bounds = src.bounds
        nodata = src.nodata
    with rasterio.open(STUDY_MASK_TIF) as ms:
        mask = ms.read(1)

    # Mask & nodata
    raw = np.where(data == nodata, np.nan, data)
    if mask.shape == raw.shape:
        raw = np.where(mask == 1, raw, np.nan)
    else:
        print(f"[WARN] Mask shape {mask.shape} != raster shape {raw.shape}")

    # Extent (WGS84)
    west, south, east, north = transform_bounds(crs, "EPSG:4326", *bounds)
    extent = [west, east, south, north]

    # Display transform
    if display == "log10":
        # guard against negatives/zeros
        vis = np.where(np.isfinite(raw) & (raw > 0), np.log10(raw + EPS), np.nan)
        cbar_label = "log10(SPI)"
    elif display == "linear":
        vis = raw.copy()
        cbar_label = "SPI (ln)"
    else:
        raise ValueError("display must be 'log10' or 'linear'")

    # Robust stretch
    valid = vis[np.isfinite(vis)]
    if valid.size == 0:
        raise RuntimeError("No valid SPI pixels to plot.")
    vmin = np.nanpercentile(valid, p_lo)
    vmax = np.nanpercentile(valid, p_hi)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = np.nanmin(valid), np.nanmax(valid)

    # Boundary
    boundary = load_shapefile_boundary(STUDY_SHP, 4326)

    # Figure 
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)

    cmap = build_spi_cmap(cmap_mode)
    cmap.set_bad((1,1,1,1))
    im = ax.imshow(vis, cmap=cmap, extent=extent, origin='upper',
                   vmin=vmin, vmax=vmax, zorder=1)

    # One boundary only
    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                            mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

    # Cartography
    add_north_arrow_tri(ax)
    add_checker_scale_bar(ax, west, east, south, north, crs_type='wgs84', segments=4)

    # Axes 
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS,
                   pad=TICK_PAD, length=1.3, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS,
                   pad=TICK_PAD, length=1.3, rotation=90)
    for lbl in ax.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')

    # Right inset colorbar 
    cax = inset_axes(ax, width=CAX_SIZE, height="100%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.0, 1.0, 1.0),
                     bbox_transform=ax.transAxes, borderpad=0)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label(cbar_label, fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
    if hasattr(cb, "outline") and cb.outline is not None: cb.outline.set_linewidth(0.3)

    # Layout & export
    plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)
    save_all(fig, out_basename)
    plt.show()

# RUN 
if __name__ == "__main__":
    os.makedirs(OUT_DIR, exist_ok=True)

    # SPI raw 
    plot_spi_compact(
        spi_tif=SPI_RAW_TIF,
        out_basename=OUT_RAW_BASENAME,
        display="log10",
        cmap_mode="magma",   # options: 'inferno','magma','plasma','cividis','viridis','terrain_dark'
        p_lo=2, p_hi=98
    )

    # SPI ln 
    plot_spi_compact(
        spi_tif=SPI_LN_TIF,
        out_basename=OUT_LN_BASENAME,
        display="linear",
        cmap_mode="magma",
        p_lo=2, p_hi=98
    )


# Step-1.10: Relief Amplitude

In [ ]:
# ====================================
# Step-1.10: Relief Amplitude
# ====================================
import os, json, warnings
from typing import Optional, Dict
import numpy as np
import rasterio
from scipy.ndimage import maximum_filter, minimum_filter

# ------------------------- PATH -------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
# Input Paths
DEM_FILLED_TIF  = os.path.join(BASE, r"Results\Step-1.4-HydroConditioning", "dem_filled.tif")
STUDY_MASK_TIF  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")

# Output Paths
OUT_DIR         = os.path.join(BASE, r"Results\Step-1.10-ReliefAmplitude")
os.makedirs(OUT_DIR, exist_ok=True)
RELIEF_AMP_TIF  = os.path.join(OUT_DIR, "relief_amplitude_32645_12p5m_float32.tif")
RELIEF_AMP_STATS_JSON = os.path.join(OUT_DIR, "relief_amplitude_stats.json")
RELIEF_AMP_HIST_PNG   = os.path.join(OUT_DIR, "relief_amplitude_hist.png")

# --------------------- SETTINGS --------------------------
NODATA = -9999.0
WINDOW_SIZE = 5  # window size for Relief Amplitude calculation (e.g., 5x5)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

# ------------------ HELPER FUNCTIONS -------------------

def _read_mask(path: str) -> np.ndarray:
    """Read the study mask and return as a binary array (1 for valid, 0 for invalid)."""
    with rasterio.open(path) as ds:
        m = ds.read(1, masked=False)
        if ds.nodata is not None:
            m = np.where(m == ds.nodata, 0, m)
        return (m != 0).astype(np.uint8)

def _save_float32(out_path: str, arr: np.ndarray, ref_path: str, nodata: float = NODATA):
    """Save the array as a float32 GeoTIFF."""
    arr = arr.astype(np.float32, copy=False)
    with rasterio.open(ref_path) as ref:
        meta = ref.meta.copy()
    meta.update(dtype="float32", nodata=nodata, compress="LZW")
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(arr, 1)

def _hist_png(values: np.ndarray, out_png: str, title: str):
    """Generate a histogram and save it as PNG."""
    if not HAVE_MPL:
        return
    v = values[np.isfinite(values)]
    v = v[(v != NODATA)]
    if v.size == 0:
        return
    plt.figure(figsize=(6,4))
    plt.hist(v, bins=80)
    plt.title(title)
    plt.xlabel("value")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def _stats(values: np.ndarray, mask: Optional[np.ndarray] = None) -> Dict:
    """Calculate basic statistics (min, max, mean, std, percentiles) for an array."""
    if mask is not None:
        values = np.where(mask, values, np.nan)
    v = values[np.isfinite(values)]
    v = v[(v != NODATA)]
    if v.size == 0:
        return {"count": 0}
    q = np.nanpercentile(v, [0,1,5,25,50,75,95,99,100])
    return {
        "count": int(v.size),
        "min": float(np.nanmin(v)),
        "p01": float(q[1]),
        "p05": float(q[2]),
        "p25": float(q[3]),
        "p50": float(q[4]),
        "p75": float(q[5]),
        "p95": float(q[6]),
        "p99": float(q[7]),
        "max": float(np.nanmax(v)),
        "mean": float(np.nanmean(v)),
        "std": float(np.nanstd(v)),
    }

def compute_relief_amplitude(dem_filled_path: str, study_mask: np.ndarray) -> np.ndarray:
    """Compute Relief Amplitude (RA) as max - min elevation in a sliding window."""
    with rasterio.open(dem_filled_path) as ds:
        dem = ds.read(1).astype(np.float64)
        if ds.nodata is not None:
            dem = np.where(dem == ds.nodata, np.nan, dem)
    
    # Apply maximum and minimum filters to compute Relief Amplitude
    relief_amp = maximum_filter(dem, size=WINDOW_SIZE) - minimum_filter(dem, size=WINDOW_SIZE)
    
    # Apply the study mask
    if study_mask is not None:
        relief_amp[study_mask == 0] = NODATA
    
    return relief_amp.astype(np.float32)

# -------------------- MAIN PROCESS ------------------------

# Read study mask
mask = _read_mask(STUDY_MASK_TIF)

# Compute Relief Amplitude
relief_amplitude = compute_relief_amplitude(DEM_FILLED_TIF, mask)

# Save Relief Amplitude to GeoTIFF
_save_float32(RELIEF_AMP_TIF, relief_amplitude, DEM_FILLED_TIF, NODATA)

# Calculate statistics for Relief Amplitude
relief_amp_stats = _stats(relief_amplitude, mask)

# Save statistics as JSON
with open(RELIEF_AMP_STATS_JSON, "w") as f:
    json.dump(relief_amp_stats, f, indent=2)

# Generate histogram plot
if HAVE_MPL:
    _hist_png(relief_amplitude, RELIEF_AMP_HIST_PNG, "Relief Amplitude (RA)")

# Final print statement
print("Step-1.10 completed.")


# Step-1.11: Distance from River

In [ ]:
# ============================================================
# Step-1.11 — Distance from River 
# ============================================================

import os, json
import numpy as np
import rasterio
from rasterio.features import shapes, rasterize
from shapely.geometry import shape
from shapely.ops import unary_union
from shapely.prepared import prep

# ---- Fiona for shapefile read ----
import fiona

# ---- Required for distance transform ----
from scipy.ndimage import distance_transform_edt

# ---- Optional CRS transform ----
try:
    from pyproj import Transformer
    from shapely.ops import transform as shp_transform
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

# ---- Optional plot ----
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

# ---------------- PATHS ----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

RIVER_SHP      = os.path.join(BASE, "Nepal_Rivers.shp")
STUDY_MASK_TIF = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
REF_TIF        = os.path.join(BASE, r"Results\Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")

OUT_DIR        = os.path.join(BASE, r"Results\Step-1.11-DistanceFromRiver")
os.makedirs(OUT_DIR, exist_ok=True)

RIVER_RAS_TIF  = os.path.join(OUT_DIR, "river_mask_32645_12p5m_byte.tif")
DIST_TIF       = os.path.join(OUT_DIR, "distance_from_river_m_32645_12p5m_float32.tif")
STATS_JSON     = os.path.join(OUT_DIR, "distance_from_river_stats.json")
HIST_PNG       = os.path.join(OUT_DIR, "distance_from_river_hist.png")

# ---------------- SETTINGS ----------------
NODATA_FLOAT = -9999.0
BURN_RIVER   = 1

# ---------------- GUARDS ----------------
if not os.path.exists(RIVER_SHP):
    raise FileNotFoundError(RIVER_SHP)
if not os.path.exists(STUDY_MASK_TIF):
    raise FileNotFoundError(STUDY_MASK_TIF)
if not os.path.exists(REF_TIF):
    raise FileNotFoundError(REF_TIF)

# ---------------- HELPERS ----------------
def read_mask(mask_path):
    with rasterio.open(mask_path) as ds:
        m = ds.read(1)
        if ds.nodata is not None:
            m = np.where(m == ds.nodata, 0, m)
        return (m != 0).astype(np.uint8), ds.crs, ds.transform, (ds.height, ds.width)

def read_ref(ref_path):
    with rasterio.open(ref_path) as ds:
        prof = ds.profile.copy()
        tr = ds.transform
        crs = ds.crs
        dx = float(tr.a)
        dy = float(-tr.e) if tr.e < 0 else float(tr.e)
        return prof, crs, tr, (ds.height, ds.width), dx, dy

def mask_to_polygon(study_mask, transform):
    geoms = []
    for geom, val in shapes(study_mask.astype(np.uint8), mask=(study_mask == 1), transform=transform):
        if val == 1:
            geoms.append(shape(geom))
    if not geoms:
        raise RuntimeError("No polygons from study_mask.")
    return unary_union(geoms)

def save_byte(out_path, arr_u8, ref_profile):
    prof = ref_profile.copy()
    prof.update(dtype="uint8", count=1, nodata=0, compress="LZW", tiled=True, bigtiff="IF_NEEDED")
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(arr_u8.astype(np.uint8), 1)

def save_float(out_path, arr_f32, ref_profile):
    prof = ref_profile.copy()
    prof.update(dtype="float32", count=1, nodata=NODATA_FLOAT, compress="LZW", tiled=True, bigtiff="IF_NEEDED")
    out = arr_f32.astype(np.float32, copy=False)
    out[~np.isfinite(out)] = NODATA_FLOAT
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(out, 1)

def stats(arr, inside_mask):
    v = arr[(inside_mask == 1) & np.isfinite(arr) & (arr != NODATA_FLOAT)]
    if v.size == 0:
        return {"count": 0}
    q = np.nanpercentile(v, [0, 1, 5, 25, 50, 75, 95, 99, 100])
    return {
        "count": int(v.size),
        "min": float(np.nanmin(v)),
        "p01": float(q[1]),
        "p05": float(q[2]),
        "p25": float(q[3]),
        "p50": float(q[4]),
        "p75": float(q[5]),
        "p95": float(q[6]),
        "p99": float(q[7]),
        "max": float(np.nanmax(v)),
        "mean": float(np.nanmean(v)),
        "std": float(np.nanstd(v)),
    }

# ---------------- LOAD GRID ----------------
study_mask, mask_crs, mask_tr, mask_shape = read_mask(STUDY_MASK_TIF)
ref_prof, ref_crs, ref_tr, ref_shape, dx, dy = read_ref(REF_TIF)

if mask_shape != ref_shape or mask_tr != ref_tr or mask_crs != ref_crs:
    raise RuntimeError("Study mask grid/CRS/transform must match the reference raster grid.")

study_poly = mask_to_polygon(study_mask, ref_tr)
study_poly_prep = prep(study_poly)

# ---------------- READ + REPROJECT + CLIP RIVERS ----------------
with fiona.open(RIVER_SHP) as src:
    river_crs_wkt = src.crs_wkt  # most reliable when present
    river_crs = river_crs_wkt if river_crs_wkt else src.crs

    need_reproj = False
    transformer = None

    # If CRS info missing, you must define it externally
    if river_crs is None:
        raise RuntimeError("Nepal_Rivers.shp has no CRS. Define CRS (.prj) before running.")

    # Decide reprojection
    # - If fiona provides WKT, compare via strings; else rely on equality check
    if river_crs_wkt and ref_crs and ref_crs.to_wkt():
        need_reproj = (river_crs_wkt != ref_crs.to_wkt())
    else:
        try:
            need_reproj = (src.crs != ref_crs)
        except Exception:
            need_reproj = False

    if need_reproj:
        if not HAVE_PYPROJ:
            raise RuntimeError("Rivers CRS != raster CRS, and pyproj is not available to reproject.")
        # Build transformer: from river CRS to raster CRS
        # Use WKT if available, otherwise proj dict
        src_crs_for_pyproj = river_crs_wkt if river_crs_wkt else src.crs
        transformer = Transformer.from_crs(src_crs_for_pyproj, ref_crs, always_xy=True)

    clipped_geoms = []
    for feat in src:
        if not feat or feat.get("geometry") is None:
            continue
        geom = shape(feat["geometry"])
        if geom.is_empty:
            continue

        if transformer is not None:
            geom = shp_transform(transformer.transform, geom)

        # fast reject by bbox via prepared polygon
        if not study_poly_prep.intersects(geom):
            continue

        g2 = geom.intersection(study_poly)
        if not g2.is_empty:
            clipped_geoms.append(g2)

if len(clipped_geoms) == 0:
    raise RuntimeError("No river geometries remain after clipping to the study area.")

# ---------------- RASTERIZE RIVERS (0/1) ----------------
river_mask = rasterize(
    ((g, BURN_RIVER) for g in clipped_geoms),
    out_shape=ref_shape,
    transform=ref_tr,
    fill=0,
    dtype="uint8",
    all_touched=False
)
river_mask[study_mask == 0] = 0
save_byte(RIVER_RAS_TIF, river_mask, ref_prof)

# ---------------- DISTANCE (meters) ----------------
# Make river pixels = 0, non-river = 1. EDT gives distance to nearest zero.
binary = np.ones(ref_shape, dtype=np.uint8)
binary[river_mask == 1] = 0

dist_m = distance_transform_edt(binary, sampling=(dy, dx)).astype(np.float32)
dist_m[study_mask == 0] = NODATA_FLOAT
save_float(DIST_TIF, dist_m, ref_prof)

# ---------------- STATS + OPTIONAL HIST ----------------
out_stats = {
    "crs_epsg": ref_crs.to_epsg() if ref_crs else None,
    "resolution_m": {"x": dx, "y": dy},
    "river_source": os.path.basename(RIVER_SHP),
    "river_mask": os.path.basename(RIVER_RAS_TIF),
    "distance_raster": os.path.basename(DIST_TIF),
    "distance_m": stats(dist_m, study_mask),
    "notes": {
        "clip": "Rivers are clipped to the study polygon derived from study_mask; outside features do not affect distances.",
        "distance": "Euclidean distance in meters using scipy.ndimage.distance_transform_edt with sampling=(dy,dx)."
    }
}
with open(STATS_JSON, "w") as f:
    json.dump(out_stats, f, indent=2)

if HAVE_MPL:
    v = dist_m[(study_mask == 1) & np.isfinite(dist_m) & (dist_m != NODATA_FLOAT)]
    if v.size > 0:
        plt.figure(figsize=(6,4))
        plt.hist(v, bins=80)
        plt.title("Distance from River (m)")
        plt.xlabel("meters")
        plt.ylabel("count")
        plt.tight_layout()
        plt.savefig(HIST_PNG, dpi=160)
        plt.close()

print(f"[OK] river mask   → {RIVER_RAS_TIF}")
print(f"[OK] distance tif → {DIST_TIF}")
print(f"[OK] stats json   → {STATS_JSON}")
if HAVE_MPL and os.path.exists(HIST_PNG):
    print(f"[OK] hist png    → {HIST_PNG}")
print("Step-1.11 completed.")


# Visualization of Distance from River

In [ ]:
# ============================================================
# Step-1.11 — Visualization for Distance From River
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds

# Optional deps 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion
    HAVE_SCIPY = True
except Exception:
    pass
try:
    from rasterio.warp import transform_geom
    HAVE_TRANSFORM_GEOM = True
except Exception:
    HAVE_TRANSFORM_GEOM = False

# ------------------- PATHS -------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STUDY_MASK_TIF = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP      = os.path.join(BASE, "Study_Area.shp")

OUT_DIR = os.path.join(BASE, r"Results\Step-1.11-DistanceFromRiver")

DIST_TIF  = os.path.join(OUT_DIR, "distance_from_river_m_32645_12p5m_float32.tif")
RIVER_MSK = os.path.join(OUT_DIR, "river_mask_32645_12p5m_byte.tif")

OUT_RAW_PNG = os.path.join(OUT_DIR, "DistanceFromRiver_Raw_Map.png")
OUT_RAW_PDF = os.path.join(OUT_DIR, "DistanceFromRiver_Raw_Map.pdf")
OUT_RAW_SVG = os.path.join(OUT_DIR, "DistanceFromRiver_Raw_Map.svg")
OUT_LOG_PNG = os.path.join(OUT_DIR, "DistanceFromRiver_Log_Map.png")
OUT_LOG_PDF = os.path.join(OUT_DIR, "DistanceFromRiver_Log_Map.pdf")
OUT_LOG_SVG = os.path.join(OUT_DIR, "DistanceFromRiver_Log_Map.svg")

# ------------------- SETTINGS / STYLE ----------------------
NODATA = -9999.0
EPS    = 1e-8

FIGSIZE           = (2, 2)
SAVE_DPI          = 600
CAX_SIZE          = "5%"
CAX_PAD           = 0.01
TICK_FONTSIZE     = 3
CB_LABEL_FONTSIZE = 6
CB_TICK_FONTSIZE  = 5
BOUNDARY_COLOR    = 'black'
BOUNDARY_LW       = 0.3
MASK_COLOR        = 'black'
MASK_LW           = 0.3

# Canvas: opaque white (following SPI style)
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    safe_savefig(fig, base_no_ext + ".pdf", **kw)
    safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})

# =========================================================
# DMS FORMATTERS + 5′/10′ ticks 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + Unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None

    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            epsg = None
            try:
                epsg = gdf.crs.to_epsg() if gdf.crs is not None else None
            except Exception:
                epsg = None
            if epsg != int(target_epsg):
                gdf = gdf.to_crs(epsg=target_epsg)
            return gdf
        except Exception:
            pass

    if HAVE_FIONA and HAVE_TRANSFORM_GEOM:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                src_crs = shp.crs_wkt or shp.crs
                for feat in shp:
                    geom = feat.get('geometry', None)
                    if geom is None:
                        continue
                    try:
                        g4326 = transform_geom(src_crs, f"EPSG:{target_epsg}", geom, precision=6)
                        geometries.append(g4326)
                    except Exception:
                        geometries.append(geom)
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass

    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw exactly ONE boundary: shapefile if available, else eroded mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates']); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float) if HAVE_SCIPY \
                  else mask_array.astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# CORE PLOTTER 
# =========================================================
def plot_distance_compact(dist_tif: str, study_mask_tif: str, out_basename: str, mode: str,
                          river_mask_tif: str = None, overlay_river: bool = True):
    # Read distance raster 
    with rasterio.open(dist_tif) as src:
        data   = src.read(1)
        crs    = src.crs
        bounds = src.bounds
        nodata = src.nodata if src.nodata is not None else NODATA
    if crs is None:
        raise RuntimeError(f"Raster CRS is None: {dist_tif}")

    # Read study mask 
    with rasterio.open(study_mask_tif) as src:
        mask = src.read(1)

    # Mask + nodata 
    raw = np.where(data == nodata, np.nan, data).astype(np.float32)
    if mask.shape == raw.shape:
        raw = np.where(mask == 1, raw, np.nan)
    else:
        print(f"[WARN] Study mask shape {mask.shape} != raster shape {raw.shape}; mask overlay may be skipped.")

    # Enforce non-negative distances for display 
    raw = np.where(raw >= 0, raw, np.nan)

    # Extent (WGS84)
    west, south, east, north = transform_bounds(crs, "EPSG:4326", *bounds)
    extent = [west, east, south, north]

    # Visualization transform + labels
    if mode == "raw":
        vis = raw.copy()
        cmap = "viridis"
        cmap = plt.get_cmap(cmap)      # CRITICAL: Convert string to colormap object
        cmap.set_bad((1,1,1,1))        # CRITICAL: Set NaN/bad values to WHITE instead of black
        cbar_label = "Distance to River (m)"
    elif mode == "log1p":
        vis = np.log1p(raw)  # stable for 0+
        cmap = "plasma"
        cmap = plt.get_cmap(cmap)      # CRITICAL: Convert string to colormap object
        cmap.set_bad((1,1,1,1))        # CRITICAL: Set NaN/bad values to WHITE instead of black
        cbar_label = "log1p(Distance to River) (m)"
    else:
        raise ValueError("mode must be 'raw' or 'log1p'")

    # Robust 2–98% stretch on displayed values
    valid_vis = vis[~np.isnan(vis)]
    if valid_vis.size == 0:
        raise RuntimeError("No valid distance pixels to plot.")
    vmin = np.nanpercentile(valid_vis, 2)
    vmax = np.nanpercentile(valid_vis, 98)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = np.nanmin(valid_vis), np.nanmax(valid_vis)

    # Optional shapefile boundary
    boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

    # Optional river overlay 
    river = None
    if river_mask_tif and os.path.exists(river_mask_tif):
        try:
            with rasterio.open(river_mask_tif) as src:
                river = src.read(1)
            if river.shape != raw.shape:
                print(f"[WARN] River mask shape {river.shape} != raster shape {raw.shape}; river overlay skipped.")
                river = None
        except Exception:
            river = None

    # Figure 
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    im = ax.imshow(vis, cmap=cmap, extent=extent, origin='upper',
                   vmin=vmin, vmax=vmax, zorder=1)

    # Single boundary 
    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                            mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

    # River overlay 
    if overlay_river and river is not None:
        try:
            ax.contour(river.astype(float), levels=[0.5], colors=['cyan'], linewidths=0.25,
                       extent=extent, origin='upper', zorder=10)
        except Exception:
            pass

    # North arrow & checker scale bar
    add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
    add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=(south + north) / 2.0,
                          segments=4, fontsize=4)

    # Axes: DMS ticks + tiny fonts
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45); lbl.set_ha('right')

    # Colorbar 
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label(cbar_label, fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

    # Save using SPI style approach
    plt.tight_layout(pad=0.2)
    save_all(fig, out_basename)
    plt.show()
    plt.close(fig)

    # Console summary
    raw_valid = raw[~np.isnan(raw)]
    print("\n" + "=" * 60)
    print(f"{('Distance From River (raw m)' if mode=='raw' else 'Distance From River (log1p)')} SUMMARY")
    print("=" * 60)
    print(f"CRS: {crs}")
    print(f"BBox (WGS84): W={format_dms_lon(west)}, E={format_dms_lon(east)}, "
          f"S={format_dms_lat(south)}, N={format_dms_lat(north)}")
    print(f"Raw Dist Min/Max (m): {np.nanmin(raw_valid):.3f} — {np.nanmax(raw_valid):.3f}")
    print(f"Raw Dist Mean ± Std (m): {np.nanmean(raw_valid):.3f} ± {np.nanstd(raw_valid):.3f}")
    print(f"Raw Dist Median (m): {np.nanmedian(raw_valid):.3f}")
    print(f"Display stretch ({mode}): {vmin:.3f} → {vmax:.3f}")
    if river is not None:
        rc = int(np.nansum((river == 1) & (mask == 1))) if (mask.shape == river.shape) else int(np.nansum(river == 1))
        print(f"River mask river-pixels (within study if aligned): {rc}")
    print("=" * 60)

# =========================================================
# RUN BOTH MAPS
# =========================================================
if __name__ == "__main__":
    # Raw distance in meters
    plot_distance_compact(
        dist_tif=DIST_TIF,
        study_mask_tif=STUDY_MASK_TIF,
        out_basename=OUT_RAW_PNG.replace('.png', ''),  # Remove extension for save_all function
        mode="raw",
        river_mask_tif=RIVER_MSK,
        overlay_river=True
    )

    # log1p(distance) 
    plot_distance_compact(
        dist_tif=DIST_TIF,
        study_mask_tif=STUDY_MASK_TIF,
        out_basename=OUT_LOG_PNG.replace('.png', ''),  # Remove extension for save_all function
        mode="log1p",
        river_mask_tif=RIVER_MSK,
        overlay_river=True
    )

# Step-1.12: Distance from Road 

In [ ]:
# ============================================================
# Step-1.12: Distance from Road 
# ============================================================

import os, json
import numpy as np
import rasterio
from rasterio.features import shapes, rasterize
from shapely.geometry import shape
from shapely.ops import unary_union
from shapely.prepared import prep

import fiona
from scipy.ndimage import distance_transform_edt

# ---- optional CRS transform ----
try:
    from pyproj import Transformer
    from shapely.ops import transform as shp_transform
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

# ---- optional plot ----
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

# ---------------- PATHS ----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

ROAD_SHP      = os.path.join(BASE, "Distance from Road_NTD.shp")
STUDY_MASK_TIF = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
REF_TIF        = os.path.join(BASE, r"Results\Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")

OUT_DIR        = os.path.join(BASE, r"Results\Step-1.12-DistanceFromRoad")
os.makedirs(OUT_DIR, exist_ok=True)

ROAD_RAS_TIF   = os.path.join(OUT_DIR, "road_mask_32645_12p5m_byte.tif")
DIST_TIF       = os.path.join(OUT_DIR, "distance_from_road_m_32645_12p5m_float32.tif")
STATS_JSON     = os.path.join(OUT_DIR, "distance_from_road_stats.json")
HIST_PNG       = os.path.join(OUT_DIR, "distance_from_road_hist.png")

# ---------------- SETTINGS ----------------
NODATA_FLOAT = -9999.0
BURN_ROAD    = 1

# ---------------- GUARDS ----------------
if not os.path.exists(ROAD_SHP):
    raise FileNotFoundError(ROAD_SHP)
if not os.path.exists(STUDY_MASK_TIF):
    raise FileNotFoundError(STUDY_MASK_TIF)
if not os.path.exists(REF_TIF):
    raise FileNotFoundError(REF_TIF)

# ---------------- HELPERS ----------------
def read_mask(mask_path):
    with rasterio.open(mask_path) as ds:
        m = ds.read(1)
        if ds.nodata is not None:
            m = np.where(m == ds.nodata, 0, m)
        return (m != 0).astype(np.uint8), ds.crs, ds.transform, (ds.height, ds.width)

def read_ref(ref_path):
    with rasterio.open(ref_path) as ds:
        prof = ds.profile.copy()
        tr = ds.transform
        crs = ds.crs
        dx = float(tr.a)
        dy = float(-tr.e) if tr.e < 0 else float(tr.e)
        return prof, crs, tr, (ds.height, ds.width), dx, dy

def mask_to_polygon(study_mask, transform):
    geoms = []
    for geom, val in shapes(study_mask.astype(np.uint8), mask=(study_mask == 1), transform=transform):
        if val == 1:
            geoms.append(shape(geom))
    if not geoms:
        raise RuntimeError("No polygons from study_mask.")
    return unary_union(geoms)

def save_byte(out_path, arr_u8, ref_profile):
    prof = ref_profile.copy()
    prof.update(dtype="uint8", count=1, nodata=0, compress="LZW", tiled=True, bigtiff="IF_NEEDED")
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(arr_u8.astype(np.uint8), 1)

def save_float(out_path, arr_f32, ref_profile):
    prof = ref_profile.copy()
    prof.update(dtype="float32", count=1, nodata=NODATA_FLOAT, compress="LZW", tiled=True, bigtiff="IF_NEEDED")
    out = arr_f32.astype(np.float32, copy=False)
    out[~np.isfinite(out)] = NODATA_FLOAT
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(out, 1)

def stats(arr, inside_mask):
    v = arr[(inside_mask == 1) & np.isfinite(arr) & (arr != NODATA_FLOAT)]
    if v.size == 0:
        return {"count": 0}
    q = np.nanpercentile(v, [0, 1, 5, 25, 50, 75, 95, 99, 100])
    return {
        "count": int(v.size),
        "min": float(np.nanmin(v)),
        "p01": float(q[1]),
        "p05": float(q[2]),
        "p25": float(q[3]),
        "p50": float(q[4]),
        "p75": float(q[5]),
        "p95": float(q[6]),
        "p99": float(q[7]),
        "max": float(np.nanmax(v)),
        "mean": float(np.nanmean(v)),
        "std": float(np.nanstd(v)),
    }

# ---------------- LOAD GRID ----------------
study_mask, mask_crs, mask_tr, mask_shape = read_mask(STUDY_MASK_TIF)
ref_prof, ref_crs, ref_tr, ref_shape, dx, dy = read_ref(REF_TIF)

if mask_shape != ref_shape or mask_tr != ref_tr or mask_crs != ref_crs:
    raise RuntimeError("Study mask grid/CRS/transform must match the reference raster grid.")

study_poly = mask_to_polygon(study_mask, ref_tr)
study_poly_prep = prep(study_poly)

# ---------------- READ + REPROJECT + CLIP ROADS ----------------
with fiona.open(ROAD_SHP) as src:
    road_crs_wkt = src.crs_wkt
    road_crs = road_crs_wkt if road_crs_wkt else src.crs

    if road_crs is None:
        raise RuntimeError("Road shapefile has no CRS (.prj missing). Define CRS before running.")

    transformer = None

    # determine if reprojection is needed
    need_reproj = False
    if road_crs_wkt and ref_crs and ref_crs.to_wkt():
        need_reproj = (road_crs_wkt != ref_crs.to_wkt())
    else:
        try:
            need_reproj = (src.crs != ref_crs)
        except Exception:
            need_reproj = False

    if need_reproj:
        if not HAVE_PYPROJ:
            raise RuntimeError("Road CRS != raster CRS, and pyproj is not available to reproject.")
        src_crs_for_pyproj = road_crs_wkt if road_crs_wkt else src.crs
        transformer = Transformer.from_crs(src_crs_for_pyproj, ref_crs, always_xy=True)

    clipped_geoms = []
    for feat in src:
        if not feat or feat.get("geometry") is None:
            continue
        geom = shape(feat["geometry"])
        if geom.is_empty:
            continue

        if transformer is not None:
            geom = shp_transform(transformer.transform, geom)

        if not study_poly_prep.intersects(geom):
            continue

        g2 = geom.intersection(study_poly)
        if not g2.is_empty:
            clipped_geoms.append(g2)

if len(clipped_geoms) == 0:
    raise RuntimeError("No road geometries remain after clipping to the study area.")

# ---------------- RASTERIZE ROADS (0/1) ----------------
road_mask = rasterize(
    ((g, BURN_ROAD) for g in clipped_geoms),
    out_shape=ref_shape,
    transform=ref_tr,
    fill=0,
    dtype="uint8",
    all_touched=False
)
road_mask[study_mask == 0] = 0
save_byte(ROAD_RAS_TIF, road_mask, ref_prof)

# ---------------- DISTANCE (meters) ----------------
binary = np.ones(ref_shape, dtype=np.uint8)
binary[road_mask == 1] = 0  # distance to nearest road pixel

dist_m = distance_transform_edt(binary, sampling=(dy, dx)).astype(np.float32)
dist_m[study_mask == 0] = NODATA_FLOAT
save_float(DIST_TIF, dist_m, ref_prof)

# ---------------- STATS + OPTIONAL HIST ----------------
out_stats = {
    "crs_epsg": ref_crs.to_epsg() if ref_crs else None,
    "resolution_m": {"x": dx, "y": dy},
    "road_source": os.path.basename(ROAD_SHP),
    "road_mask": os.path.basename(ROAD_RAS_TIF),
    "distance_raster": os.path.basename(DIST_TIF),
    "distance_m": stats(dist_m, study_mask),
    "notes": {
        "clip": "Roads clipped to study polygon from study_mask; outside features do not affect distances.",
        "distance": "Euclidean distance in meters using scipy.ndimage.distance_transform_edt with sampling=(dy,dx)."
    }
}
with open(STATS_JSON, "w") as f:
    json.dump(out_stats, f, indent=2)

if HAVE_MPL:
    v = dist_m[(study_mask == 1) & np.isfinite(dist_m) & (dist_m != NODATA_FLOAT)]
    if v.size > 0:
        plt.figure(figsize=(6,4))
        plt.hist(v, bins=80)
        plt.title("Distance from Road (m)")
        plt.xlabel("meters")
        plt.ylabel("count")
        plt.tight_layout()
        plt.savefig(HIST_PNG, dpi=160)
        plt.close()

print(f"[OK] road mask    → {ROAD_RAS_TIF}")
print(f"[OK] distance tif → {DIST_TIF}")
print(f"[OK] stats json   → {STATS_JSON}")
if HAVE_MPL and os.path.exists(HIST_PNG):
    print(f"[OK] hist png     → {HIST_PNG}")
print("Step-1.12 completed.")


# Visualization of Distance from Road

In [ ]:
# ============================================================
# Step-1.12 — Visualization for Distance From Road
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds

# Optional deps 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion
    HAVE_SCIPY = True
except Exception:
    pass
try:
    from rasterio.warp import transform_geom
    HAVE_TRANSFORM_GEOM = True
except Exception:
    HAVE_TRANSFORM_GEOM = False

# ------------------- PATHS -------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STUDY_MASK_TIF = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP      = os.path.join(BASE, "Study_Area.shp")

OUT_DIR = os.path.join(BASE, r"Results\Step-1.12-DistanceFromRoad")

DIST_TIF  = os.path.join(OUT_DIR, "distance_from_road_m_32645_12p5m_float32.tif")
ROAD_MSK = os.path.join(OUT_DIR, "road_mask_32645_12p5m_byte.tif")

OUT_RAW_PNG = os.path.join(OUT_DIR, "DistanceFromRoad_Raw_Map.png")
OUT_RAW_PDF = os.path.join(OUT_DIR, "DistanceFromRoad_Raw_Map.pdf")
OUT_RAW_SVG = os.path.join(OUT_DIR, "DistanceFromRoad_Raw_Map.svg")
OUT_LOG_PNG = os.path.join(OUT_DIR, "DistanceFromRoad_Log_Map.png")
OUT_LOG_PDF = os.path.join(OUT_DIR, "DistanceFromRoad_Log_Map.pdf")
OUT_LOG_SVG = os.path.join(OUT_DIR, "DistanceFromRoad_Log_Map.svg")

# ------------------- SETTINGS / STYLE ----------------------
NODATA = -9999.0
EPS    = 1e-8

FIGSIZE           = (2, 2)
SAVE_DPI          = 600
CAX_SIZE          = "5%"
CAX_PAD           = 0.01
TICK_FONTSIZE     = 3
CB_LABEL_FONTSIZE = 6
CB_TICK_FONTSIZE  = 5
BOUNDARY_COLOR    = 'black'
BOUNDARY_LW       = 0.3
MASK_COLOR        = 'black'
MASK_LW           = 0.3

# Canvas: opaque white (following SPI style)
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    safe_savefig(fig, base_no_ext + ".pdf", **kw)
    safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})

# =========================================================
# DMS FORMATTERS + 5′/10′ ticks 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + Unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None

    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            epsg = None
            try:
                epsg = gdf.crs.to_epsg() if gdf.crs is not None else None
            except Exception:
                epsg = None
            if epsg != int(target_epsg):
                gdf = gdf.to_crs(epsg=target_epsg)
            return gdf
        except Exception:
            pass

    if HAVE_FIONA and HAVE_TRANSFORM_GEOM:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                src_crs = shp.crs_wkt or shp.crs
                for feat in shp:
                    geom = feat.get('geometry', None)
                    if geom is None:
                        continue
                    try:
                        g4326 = transform_geom(src_crs, f"EPSG:{target_epsg}", geom, precision=6)
                        geometries.append(g4326)
                    except Exception:
                        geometries.append(geom)
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass

    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw exactly ONE boundary: shapefile if available, else eroded mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates']); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float) if HAVE_SCIPY \
                  else mask_array.astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# CORE PLOTTER 
# =========================================================
def plot_distance_compact(dist_tif: str, study_mask_tif: str, out_basename: str, mode: str,
                          road_mask_tif: str = None, overlay_road: bool = True):
    # Read distance raster 
    with rasterio.open(dist_tif) as src:
        data   = src.read(1)
        crs    = src.crs
        bounds = src.bounds
        nodata = src.nodata if src.nodata is not None else NODATA
    if crs is None:
        raise RuntimeError(f"Raster CRS is None: {dist_tif}")

    # Read study mask 
    with rasterio.open(study_mask_tif) as src:
        mask = src.read(1)

    # Mask + nodata 
    raw = np.where(data == nodata, np.nan, data).astype(np.float32)
    if mask.shape == raw.shape:
        raw = np.where(mask == 1, raw, np.nan)
    else:
        print(f"[WARN] Study mask shape {mask.shape} != raster shape {raw.shape}; mask overlay may be skipped.")

    # Enforce non-negative distances for display 
    raw = np.where(raw >= 0, raw, np.nan)

    # Extent (WGS84)
    west, south, east, north = transform_bounds(crs, "EPSG:4326", *bounds)
    extent = [west, east, south, north]

    # Visualization transform + labels
    if mode == "raw":
        vis = raw.copy()
        cmap = "Blues"
        cmap = plt.get_cmap(cmap)      # CRITICAL: Convert string to colormap object
        cmap.set_bad((1,1,1,1))        # CRITICAL: Set NaN/bad values to WHITE instead of black
        cbar_label = "Distance to Road (m)"
    elif mode == "log1p":
        vis = np.log1p(raw)  # stable for 0+
        cmap = "plasma"
        cmap = plt.get_cmap(cmap)      # CRITICAL: Convert string to colormap object
        cmap.set_bad((1,1,1,1))        # CRITICAL: Set NaN/bad values to WHITE instead of black
        cbar_label = "log1p(Distance to Road) (m)"
    else:
        raise ValueError("mode must be 'raw' or 'log1p'")

    # Robust 2–98% stretch on displayed values
    valid_vis = vis[~np.isnan(vis)]
    if valid_vis.size == 0:
        raise RuntimeError("No valid distance pixels to plot.")
    vmin = np.nanpercentile(valid_vis, 2)
    vmax = np.nanpercentile(valid_vis, 98)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = np.nanmin(valid_vis), np.nanmax(valid_vis)

    # Optional shapefile boundary
    boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

    # Optional road overlay 
    road = None
    if road_mask_tif and os.path.exists(road_mask_tif):
        try:
            with rasterio.open(road_mask_tif) as src:
                road = src.read(1)
            if road.shape != raw.shape:
                print(f"[WARN] Road mask shape {road.shape} != raster shape {raw.shape}; road overlay skipped.")
                road = None
        except Exception:
            road = None

    # Figure 
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    im = ax.imshow(vis, cmap=cmap, extent=extent, origin='upper',
                   vmin=vmin, vmax=vmax, zorder=1)

    # Single boundary 
    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                            mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

    # Road overlay 
    if overlay_road and road is not None:
        try:
            ax.contour(road.astype(float), levels=[0.5], colors=['purple'], linewidths=0.25,
                       extent=extent, origin='upper', zorder=10)
        except Exception:
            pass

    # North arrow & checker scale bar
    add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
    add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=(south + north) / 2.0,
                          segments=4, fontsize=4)

    # Axes: DMS ticks + tiny fonts
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45); lbl.set_ha('right')

    # Colorbar 
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label(cbar_label, fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

    # Save using SPI style approach
    plt.tight_layout(pad=0.2)
    save_all(fig, out_basename)
    plt.show()
    plt.close(fig)

    # Console summary
    raw_valid = raw[~np.isnan(raw)]
    print("\n" + "=" * 60)
    print(f"{('Distance From Road (raw m)' if mode=='raw' else 'Distance From Road (log1p)')} SUMMARY")
    print("=" * 60)
    print(f"CRS: {crs}")
    print(f"BBox (WGS84): W={format_dms_lon(west)}, E={format_dms_lon(east)}, "
          f"S={format_dms_lat(south)}, N={format_dms_lat(north)}")
    print(f"Raw Dist Min/Max (m): {np.nanmin(raw_valid):.3f} — {np.nanmax(raw_valid):.3f}")
    print(f"Raw Dist Mean ± Std (m): {np.nanmean(raw_valid):.3f} ± {np.nanstd(raw_valid):.3f}")
    print(f"Raw Dist Median (m): {np.nanmedian(raw_valid):.3f}")
    print(f"Display stretch ({mode}): {vmin:.3f} → {vmax:.3f}")
    if road is not None:
        rc = int(np.nansum((road == 1) & (mask == 1))) if (mask.shape == road.shape) else int(np.nansum(road == 1))
        print(f"Road mask road-pixels (within study if aligned): {rc}")
    print("=" * 60)

# =========================================================
# RUN BOTH MAPS
# =========================================================
if __name__ == "__main__":
    # Raw distance in meters
    plot_distance_compact(
        dist_tif=DIST_TIF,
        study_mask_tif=STUDY_MASK_TIF,
        out_basename=OUT_RAW_PNG.replace('.png', ''),  # Remove extension for save_all function
        mode="raw",
        road_mask_tif=ROAD_MSK,
        overlay_road=True
    )

    # log1p(distance) 
    plot_distance_compact(
        dist_tif=DIST_TIF,
        study_mask_tif=STUDY_MASK_TIF,
        out_basename=OUT_LOG_PNG.replace('.png', ''),  # Remove extension for save_all function
        mode="log1p",
        road_mask_tif=ROAD_MSK,
        overlay_road=True
    )

# Phase-2: GEE variables

# GEE Authorization

# Test Our Service Account Key

In [ ]:
# Public-release configuration: supply your own Earth Engine credentials.
# The original machine-specific service-account diagnostic was omitted.
# Do not commit JSON key files or other credentials to GitHub.
import os
print("Earth Engine configuration: set EE_PROJECT and optionally EE_SERVICE_ACCOUNT_KEY.")


# GEE Initialization

In [ ]:
# Google Earth Engine initialization (public configuration, no embedded credentials)
import os
import json
import ee

EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
EE_SERVICE_ACCOUNT_KEY = os.environ.get("EE_SERVICE_ACCOUNT_KEY", "").strip()
if not EE_PROJECT:
    raise RuntimeError("Set the EE_PROJECT environment variable to your Earth Engine-enabled Google Cloud project.")
if EE_SERVICE_ACCOUNT_KEY:
    # Keep the service-account JSON key OUTSIDE the Git repository.
    with open(EE_SERVICE_ACCOUNT_KEY, encoding="utf-8") as f:
        service_account_info = json.load(f)
    credentials = ee.ServiceAccountCredentials(
        email=service_account_info["client_email"],
        key_file=EE_SERVICE_ACCOUNT_KEY,
    )
    ee.Initialize(credentials=credentials, project=EE_PROJECT)
else:
    # Interactive OAuth alternative; users authenticate with their own account.
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine initialized with the configured project.")


# Step-2.1: NDVI

In [ ]:
# =========================================================
# Sentinel-2 NDVI (CSV + Raster)
# =========================================================
import os, io, json, time, zipfile, requests, struct, shutil
from datetime import datetime, timedelta
import ee
import numpy as np
import pandas as pd

# Optional raster stage (skip if libs missing)
try:
    import rasterio
    from rasterio.merge import merge
    from rasterio.warp import reproject, Resampling
    from rasterio.windows import Window
    from pyproj import Transformer
    _HAVE_RASTER = True
except Exception:
    _HAVE_RASTER = False

# ------------PATH------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
SHP_PATH  = os.path.join(BASE, "Study_Area.shp")
PRJ_PATH  = os.path.splitext(SHP_PATH)[0] + ".prj"

ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
MASK_TIF   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

OUT_DIR = os.path.join(BASE, "Results", "Step-2.1-NDVI")
os.makedirs(OUT_DIR, exist_ok=True)
CSV_OUT = os.path.join(OUT_DIR, "ndvi_12day_2020_2023.csv")
TIF_OUT = os.path.join(OUT_DIR, "ndvi_median_2020_2023_32645_12p5m_int16.tif")
TMP_DIR = os.path.join(OUT_DIR, "_tmp"); os.makedirs(TMP_DIR, exist_ok=True)

# Earth Engine auth
SERVICE_ACCOUNT_KEY = os.environ.get("EE_SERVICE_ACCOUNT_KEY", "")
PROJECT_ID = os.environ.get("EE_PROJECT", "")

# NDVI & export settings
DATE_START, DATE_END = "2019-01-09", "2022-07-16"
INTERVAL_DAYS = 12
CLOUDY_PCT = 60
REDUCE_SCALE = 30                     # m; for reduceRegion sampling (CSV)
MAX_PIXELS   = 1_000_000_000

DL_SCALE_M   = 60.0                   # *** monthly export resolution (m). Use 60 or 80 or 100 for stability
CONNECT_TIMEOUT = (30, 600)
SCALE_FACTOR = 10000                  # NDVI float → int16
NODATA = -32768

# Cleanup configuration
KEEP_TEMP_FILES = False  # Set to True for debugging purposes

def pp(msg): print(msg, flush=True)

# ----------------------------------
# GEE initialization (explicit project)
# ----------------------------------
def ee_auth():
    with open(SERVICE_ACCOUNT_KEY, "r") as f:
        key = json.load(f)
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    creds = ee.ServiceAccountCredentials(
        email=key["client_email"],
        key_data=json.dumps(key)
    )
    ee.Initialize(credentials=creds, project=PROJECT_ID)
    _ = ee.Number(1).getInfo()  # ping
    pp(f"[EE] Initialized with project: {PROJECT_ID}")

# -------------------------------------------------------
# Minimal shapefile reader + .prj CRS guess (no fiona)
# -------------------------------------------------------
def guess_epsg_from_prj(prj_path):
    try:
        txt = open(prj_path, "r", encoding="utf-8", errors="ignore").read()
    except Exception:
        return None
    t = txt.lower()
    if "wgs_1984_utm" in t or "wgs 1984 utm" in t:
        if "zone_45" in t or "zone 45" in t or "utm_45n" in t:
            return 32645
    if "gcs_wgs_1984" in t or "wgs 84" in t or "wgs_1984" in t:
        return 4326
    return None

def read_main_polygon_from_shp(shp_path):
    """
    Reads the largest polygon ring from a polygon/multipolygon .shp.
    Returns (coords, epsg_guess, bounds) where coords=[[x,y],...] in the native CRS.
    """
    with open(shp_path, "rb") as f:
        header = f.read(100)
        file_length = struct.unpack(">I", header[24:28])[0] * 2
        shape_type = struct.unpack("<I", header[32:36])[0]
        xmin, ymin, xmax, ymax = struct.unpack("<4d", header[36:68])

        polys = []
        f.seek(100)
        while f.tell() < file_length:
            rec_hdr = f.read(8)
            if len(rec_hdr) < 8: break
            content_len = struct.unpack(">I", rec_hdr[4:8])[0] * 2
            st = struct.unpack("<I", f.read(4))[0]
            if st in (5, 15):  # Polygon / PolygonZ
                _ = struct.unpack("<4d", f.read(32))
                num_parts = struct.unpack("<I", f.read(4))[0]
                num_points = struct.unpack("<I", f.read(4))[0]
                parts = [struct.unpack("<I", f.read(4))[0] for _ in range(num_parts)]
                pts = [struct.unpack("<2d", f.read(16)) for _ in range(num_points)]
                if st == 15:
                    remaining = content_len - (4+32+8 + num_parts*4 + num_points*16)
                    if remaining > 0: f.read(remaining)
                outer = []
                for i in range(num_parts):
                    a = parts[i]
                    b = parts[i+1] if i+1 < num_parts else num_points
                    ring = pts[a:b]
                    if len(ring) > len(outer): outer = ring
                if len(outer) >= 4:
                    # ensure closed
                    if outer[0] != outer[-1]:
                        outer = outer + [outer[0]]
                    polys.append(outer)
            else:
                remaining = content_len - 4
                if remaining > 0: f.read(remaining)

        if not polys:
            raise RuntimeError("No polygon geometry found in shapefile.")

        main_ring = max(polys, key=len)
        epsg = guess_epsg_from_prj(PRJ_PATH)
        return main_ring, epsg, (xmin, ymin, xmax, ymax)

# ----------------------------------
# Anchor grid from DEM_anchor_32645
# ----------------------------------
def load_anchor_spec(anchor_path):
    if not _HAVE_RASTER:
        return None
    with rasterio.open(anchor_path) as ds:
        tf = ds.transform
        w, h = ds.width, ds.height
        crs = ds.crs
        bounds = ds.bounds
    epsg = int(crs.to_epsg()) if crs and crs.to_epsg() else None
    return dict(tf=tf, width=w, height=h, epsg=epsg, bounds=(bounds.left,bounds.bottom,bounds.right,bounds.top))

# ----------------------------------
# S2 NDVI collection
# ----------------------------------
def mask_s2_qa60(img):
    qa = img.select("QA60")
    m = qa.bitwiseAnd(1<<10).eq(0).And(qa.bitwiseAnd(1<<11).eq(0))
    return img.updateMask(m)

def add_ndvi_with_proj(img):
    # Compute NDVI and give it the B8 projection (UTM 10 m), so it has a real CRS/scale
    nd = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    nd = nd.setDefaultProjection(img.select("B8").projection())
    return img.addBands(nd)

def build_s2_ndvi_ic(aoi):
    base = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterDate(DATE_START, DATE_END)
            .filterBounds(aoi))
    col = base.filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUDY_PCT)).map(mask_s2_qa60)
    col = col.map(add_ndvi_with_proj).select("NDVI")
    n = col.size().getInfo()
    if n == 0:
        pp("[INFO] Cloud-filtered collection empty; relaxing cloud mask.")
        col = base.map(add_ndvi_with_proj).select("NDVI")
        n = col.size().getInfo()
        if n == 0:
            raise SystemExit("No Sentinel-2 scenes for dates/region.")
        pp(f"[INFO] Using unmasked collection: {n} images.")
    else:
        pp(f"[INFO] Using cloud-filtered collection: {n} images.")
    return col

# ----------------------------------
# 12-day time series (CSV)
# ----------------------------------
def generate_intervals(start_str, end_str, step_days):
    start = datetime.strptime(start_str, "%Y-%m-%d")
    end   = datetime.strptime(end_str,   "%Y-%m-%d")
    out = []; cur = start; k = 0
    while cur < end:
        nxt = min(cur + timedelta(days=step_days), end)
        mid = (cur + (nxt-cur)/2).strftime("%Y-%m-%d")
        out.append((k, cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d"), mid))
        cur = nxt; k += 1
    return out

def extract_12day_timeseries(ic, aoi_geom):
    pp("[TS] Extracting 12-day NDVI time series …")
    intervals = generate_intervals(DATE_START, DATE_END, INTERVAL_DAYS)

    reducer = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.count(),
        sharedInputs=True,
        outputPrefix=''       # → NDVI_mean, NDVI_count
    )

    rows = []
    t0 = time.time()
    for k, d0, d1, mid in intervals:
        sub = ic.filterDate(d0, d1)
        n = sub.size().getInfo()
        if n == 0:
            continue
        comp = sub.median() if n > 1 else sub.first()
        stats = comp.reduceRegion(
            reducer=reducer,
            geometry=aoi_geom,
            scale=REDUCE_SCALE,
            bestEffort=True,
            tileScale=4,
            maxPixels=MAX_PIXELS
        ).getInfo()
        m = stats.get("NDVI_mean")
        c = stats.get("NDVI_count")
        if m is None:
            continue
        rows.append({
            "interval_id": k,
            "interval_start": d0,
            "interval_end": d1,
            "date": mid,
            "ndvi_mean": float(m),
            "pixel_count": int(c) if c is not None else None,
            "image_count": n
        })
        if (k+1) % 10 == 0:
            pp(f"  … {k+1}/{len(intervals)} (elapsed {time.time()-t0:.1f}s)")
    pp(f"[TS] Done. {len(rows)} intervals in {time.time()-t0:.1f}s")
    return rows

# ----------------------------------
# Monthly export helpers (memory-safe)
# ----------------------------------
def month_list(start_str, end_str):
    start = datetime.strptime(start_str, "%Y-%m-%d")
    end   = datetime.strptime(end_str,   "%Y-%m-%d")
    months = []
    cur = datetime(start.year, start.month, 1)
    while cur <= end:
        nxt = datetime(cur.year + (cur.month == 12), (cur.month % 12) + 1, 1)
        months.append((cur.strftime("%Y-%m-%d"),
                       (nxt - timedelta(days=1)).strftime("%Y-%m-%d"),
                       f"{cur.year}{cur.month:02d}"))
        cur = nxt
    return months

def _download_to_bytes(image, params, tag, max_retries=3):
    """Download with retries; only pass allowed params."""
    for attempt in range(max_retries):
        try:
            safe = {k: v for k, v in params.items()
                    if k in ("region", "scale", "crs", "crsTransform", "format", "maxPixels")}
            url = image.getDownloadURL(safe)
            dbg = {k: ('<geom>' if k == 'region' else v) for k, v in safe.items()}
            pp(f"[REQ] {tag} (try {attempt+1}/{max_retries}): {dbg}")
            r = requests.get(url, stream=True, timeout=CONNECT_TIMEOUT)
            if r.status_code != 200:
                msg = r.content.decode("utf-8", errors="ignore")
                if "memory" in msg.lower() and attempt < max_retries - 1:
                    # coarsen a bit and retry
                    if 'scale' in safe:
                        safe['scale'] = float(safe['scale']) * 1.5
                    continue
                raise RuntimeError(f"HTTP {r.status_code}: {msg[:300]} ...")
            head = r.raw.read(4)
            rest = r.raw.read()
            return head + rest
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(5)

def _bytes_to_tif(blob, out_path):
    tmp = out_path + ".part"
    if blob[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(blob)) as zf:
            tifs = [n for n in zf.namelist() if n.lower().endswith((".tif",".tiff"))]
            if not tifs: raise RuntimeError("ZIP contained no TIFF.")
            with zf.open(tifs[0]) as zin, open(tmp, "wb") as out:
                out.write(zin.read())
    else:
        with open(tmp, "wb") as out:
            out.write(blob)
    os.replace(tmp, out_path)

def _export_one_month(mo_ic, aoi_geom, tag, out_dir):
    """
    Export monthly median NDVI at DL_SCALE_M. Bounded & masked, with tiling fallback.
    Returns path to a single GeoTIFF or None.
    """
    ndvi = mo_ic.median().select('NDVI').clip(aoi_geom).selfMask()
    # Reproject to tame server cost (coarse)
    ndvi = ndvi.reproject(ee.Projection('EPSG:32645').atScale(DL_SCALE_M))
    ndvi_i16 = ndvi.multiply(SCALE_FACTOR).int16()

    # Simple region geometry = AOI bbox in EPSG:4326
    region_geo = ee.Geometry(aoi_geom).bounds(1).transform('EPSG:4326', 1).getInfo()

    params = {"region": region_geo, "scale": float(DL_SCALE_M), "format": "GEO_TIFF", "maxPixels": 1e13}

    # Single export
    single_tmp = os.path.join(out_dir, f"{tag}_single.tif")
    try:
        blob = _download_to_bytes(ndvi_i16, params, f"{tag}_single")
        _bytes_to_tif(blob, single_tmp)
        return single_tmp
    except Exception as e:
        pp(f"[MONTH] Single export failed ({tag}): {e}")

    # Tiled fallback
    bbox = ee.Geometry(region_geo).coordinates().getInfo()[0]
    xs = [p[0] for p in bbox]; ys = [p[1] for p in bbox]
    minlon, maxlon = min(xs), max(xs)
    minlat, maxlat = min(ys), max(ys)

    def lonlat_tiles(minlon, minlat, maxlon, maxlat, nx, ny):
        dx=(maxlon-minlon)/nx; dy=(maxlat-minlat)/ny
        for j in range(ny):
            for i in range(nx):
                x0=minlon+i*dx; x1=x0+dx
                y0=minlat+j*dy; y1=y0+dy
                yield [[x0,y0],[x1,y0],[x1,y1],[x0,y1],[x0,y0]]

    for splits in (2,3,4,5,6,8,10,12):
        pp(f"[MONTH] {tag} tiling {splits}×{splits} …")
        tile_paths = []; ok = 0; k = 0
        for ring in lonlat_tiles(minlon,minlat,maxlon,maxlat,splits,splits):
            k += 1
            tparams = dict(params); tparams["region"] = [ring]
            try:
                blob = _download_to_bytes(ndvi_i16, tparams, f"{tag}_tile_{splits}x{splits}_{k}")
                tile_tif = os.path.join(out_dir, f"{tag}_tile_{splits}_{k}.tif")
                _bytes_to_tif(blob, tile_tif)
                tile_paths.append(tile_tif); ok += 1
            except Exception as te:
                pp(f"   tile {k} failed: {te}")
        if ok == splits*splits and tile_paths:
            # mosaic to one monthly file
            srcs = [rasterio.open(p) for p in tile_paths]
            mosaic, mosaic_tf = merge(srcs, method="first"); mosaic_crs = srcs[0].crs
            for s in srcs: s.close()
            out_mo = os.path.join(out_dir, f"{tag}_mosaic.tif")
            profile = {"driver":"GTiff","height":mosaic.shape[1],"width":mosaic.shape[2],
                       "count":1,"dtype":"int16","crs":mosaic_crs,"transform":mosaic_tf,
                       "nodata":NODATA,"compress":"LZW","tiled":True}
            tmp = out_mo + ".part"
            with rasterio.open(tmp, "w", **profile) as ds:
                ds.write(mosaic, 1)
            os.replace(tmp, out_mo)
            return out_mo
    return None

def align_month_to_anchor(month_path, anchor_spec, mask_path, out_path):
    with rasterio.open(month_path) as src:
        src_arr = src.read(1); src_tf = src.transform; src_crs = src.crs
    dst_tf = anchor_spec["tf"]; dst_w = anchor_spec["width"]; dst_h = anchor_spec["height"]
    dst_crs = f"EPSG:{anchor_spec['epsg']}"
    dst = np.full((dst_h, dst_w), NODATA, dtype="int16")
    reproject(source=src_arr, destination=dst,
              src_transform=src_tf, src_crs=src_crs,
              dst_transform=dst_tf, dst_crs=dst_crs,
              resampling=Resampling.bilinear,
              src_nodata=NODATA, dst_nodata=NODATA)
    # mask
    if os.path.exists(mask_path):
        with rasterio.open(mask_path) as ms:
            m = ms.read(1)
            if (ms.transform != dst_tf) or (ms.width != dst_w) or (ms.height != dst_h) or (ms.crs != rasterio.crs.CRS.from_string(dst_crs)):
                mask_res = np.zeros((dst_h, dst_w), dtype="uint8")
                reproject(source=m, destination=mask_res,
                          src_transform=ms.transform, src_crs=ms.crs,
                          dst_transform=dst_tf, dst_crs=dst_crs,
                          resampling=Resampling.nearest,
                          src_nodata=0, dst_nodata=0)
                m = mask_res
            dst = np.where(m > 0, dst, NODATA)
    profile = {"driver":"GTiff","height":dst_h,"width":dst_w,"count":1,
               "dtype":"int16","crs":dst_crs,"transform":dst_tf,
               "nodata":NODATA,"compress":"LZW","tiled":True}
    tmp = out_path + ".part"
    with rasterio.open(tmp, "w", **profile) as ds:
        ds.write(dst, 1)
    os.replace(tmp, out_path)
    return out_path

def median_across_aligned(month_paths, out_tif, anchor_spec):
    """Per-pixel median across many aligned rasters using block windows (low RAM)."""
    if not month_paths:
        return False
    srcs = [rasterio.open(p) for p in month_paths]
    dst_tf = anchor_spec["tf"]; dst_w = anchor_spec["width"]; dst_h = anchor_spec["height"]
    dst_crs = f"EPSG:{anchor_spec['epsg']}"
    profile = {"driver":"GTiff","height":dst_h,"width":dst_w,"count":1,
               "dtype":"int16","crs":dst_crs,"transform":dst_tf,
               "nodata":NODATA,"compress":"LZW","tiled":True}
    tmp = out_tif + ".part"
    with rasterio.open(tmp, "w", **profile) as dst:
        for _, window in dst.block_windows(1):
            stack = []
            for s in srcs:
                a = s.read(1, window=window, boundless=True, fill_value=NODATA)
                a = np.where(a == NODATA, np.nan, a).astype("float32")
                stack.append(a)
            arr = np.stack(stack, axis=0)
            med = np.nanmedian(arr, axis=0)
            out = np.where(np.isnan(med), NODATA, med).astype("int16")
            dst.write(out, 1, window=window)
    os.replace(tmp, out_tif)
    for s in srcs: s.close()
    return True

# ----------------------------------
# Monthly → aligned → final median workflow
# ----------------------------------
def download_and_align_raster(ic, aoi_geom, anchor_spec, mask_path, out_tif):
    """
    Robust path: export each MONTH at coarse scale → align locally → median stack.
    """
    if not _HAVE_RASTER:
        pp("[RASTER] Skipped: rasterio/pyproj not installed.")
        return False

    pp("[RASTER] Monthly export path (robust to server memory limits)…")
    months = month_list(DATE_START, DATE_END)
    monthly_raw_dir = os.path.join(TMP_DIR, "m_raw"); os.makedirs(monthly_raw_dir, exist_ok=True)
    monthly_aligned_dir = os.path.join(TMP_DIR, "m_aligned"); os.makedirs(monthly_aligned_dir, exist_ok=True)

    aligned_paths = []
    for m_start, m_end, tag in months:
        mo_ic = ic.filterDate(m_start, (datetime.strptime(m_end,"%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d"))
        if mo_ic.size().getInfo() == 0:
            pp(f"[MONTH] {tag}: no images, skip.")
            continue
        pp(f"[MONTH] {tag}: exporting …")
        raw_path = _export_one_month(mo_ic, aoi_geom, f"ndvi_{tag}", monthly_raw_dir)
        if raw_path is None:
            pp(f"[MONTH] {tag}: export failed, skip.")
            continue
        aligned_path = os.path.join(monthly_aligned_dir, f"ndvi_{tag}_aligned.tif")
        align_month_to_anchor(raw_path, anchor_spec, mask_path, aligned_path)
        aligned_paths.append(aligned_path)

    if not aligned_paths:
        pp("[RASTER] No monthly rasters exported; final raster skipped.")
        return False

    pp(f"[RASTER] Median across {len(aligned_paths)} months …")
    ok = median_across_aligned(aligned_paths, out_tif, anchor_spec)
    if ok:
        pp(f"[RASTER] Final median NDVI → {out_tif}")
    return ok

# ----------------------------------
# Cleanup function
# ----------------------------------
def cleanup_temp_files():
    """Remove temporary directory and all its contents."""
    if os.path.exists(TMP_DIR):
        try:
            shutil.rmtree(TMP_DIR)
            pp(f"[CLEANUP] Removed temporary directory: {TMP_DIR}")
            return True
        except Exception as e:
            pp(f"[CLEANUP] Warning: Could not remove temp dir {TMP_DIR}: {e}")
            return False
    return True

# ----------------
# MAIN
# ----------------
def main():
    pp("=== MEMORY-SAFE Sentinel-2 NDVI (12-day CSV + 4-year median raster) ===")
    pp(f"[CONFIG] Monthly export scale: {DL_SCALE_M} m")
    pp(f"[CONFIG] Cleanup temporary files: {'No' if KEEP_TEMP_FILES else 'Yes'}")
    
    try:
        ee_auth()

        # AOI from shapefile
        try:
            ring_native, shp_epsg, shp_bounds = read_main_polygon_from_shp(SHP_PATH)
            if shp_epsg is None:
                pp("[AOI] Could not determine EPSG from .prj; will assume EPSG:32645 if anchor is available.")
        except Exception as e:
            raise SystemExit(f"[AOI] Failed to read shapefile: {e}")

        # Anchor grid (for raster snapping)
        anchor = None
        if _HAVE_RASTER and os.path.exists(ANCHOR_TIF):
            anchor = load_anchor_spec(ANCHOR_TIF)
            if anchor is None or anchor["epsg"] is None:
                raise SystemExit("[ANCHOR] Could not read anchor grid/EPSG.")
        else:
            raise SystemExit("[ANCHOR] Anchor grid not found; raster stage requires DEM_anchor_32645.tif.")

        # Build AOI geometry for EE using shapefile CRS (or anchor CRS if unknown)
        shp_proj_epsg = shp_epsg or (anchor["epsg"] if anchor else 4326)
        shp_proj = ee.Projection(f"EPSG:{shp_proj_epsg}")
        aoi_geom = ee.Geometry.Polygon([ring_native], proj=shp_proj, geodesic=False)

        # NDVI collection
        ic = build_s2_ndvi_ic(aoi_geom)

        # 12-day time series (CSV)
        rows = extract_12day_timeseries(ic, aoi_geom)
        if not rows:
            raise SystemExit("[TS] No NDVI samples extracted; check AOI/dates.")
        df = pd.DataFrame(rows).sort_values("date")
        df.to_csv(CSV_OUT, index=False)
        pp(f"[CSV] Wrote: {CSV_OUT} ({len(df)} rows)")

        # Raster export + align (monthly path)
        ok = download_and_align_raster(ic, aoi_geom, anchor, MASK_TIF, TIF_OUT)
        if ok:
            try:
                with rasterio.open(TIF_OUT) as ds:
                    a = ds.read(1)
                    m = a != ds.nodata
                    if m.any():
                        ndvi = a[m].astype("float32")/SCALE_FACTOR
                        pp(f"[QA] valid={m.sum()}  min/mean/max={float(ndvi.min()):.3f}/{float(ndvi.mean()):.3f}/{float(ndvi.max()):.3f}")
                    pp(f"[QA] CRS: {ds.crs}  shape: {ds.height} x {ds.width}")
            except Exception:
                pass
        else:
            pp("[NOTE] Raster stage failed; CSV time series is complete.")

    finally:
        # Always attempt cleanup, regardless of success or failure
        if not KEEP_TEMP_FILES:
            cleanup_temp_files()
        elif KEEP_TEMP_FILES:
            pp(f"[DEBUG] Keeping temporary files in: {TMP_DIR}")

    pp("=== Done ===")

if __name__ == "__main__":
    main()

# Visualization: NDVI

In [ ]:
# ============================================================
# Step-2.1 — Visualization for NDVI 
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds

# Optional deps 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion
    HAVE_SCIPY = True
except Exception:
    pass

# ------------------------- PATHS ---------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
OUT_NDVI   = os.path.join(BASE, "Results", "Step-2.1-NDVI")
PHASE02    = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK = os.path.join(PHASE02, "study_mask.tif")
STUDY_SHP  = os.path.join(BASE, "Study_Area.shp")

NDVI_TIF   = os.path.join(OUT_NDVI, "ndvi_median_2020_2023_32645_12p5m_int16.tif")
OUT_PNG    = os.path.join(OUT_NDVI, "NDVI Map.png")
OUT_PDF    = os.path.join(OUT_NDVI, "NDVI Map.pdf")

# ------------------------- SETTINGS / STYLE -----------------
SCALE_FACTOR       = 10000.0  
NODATA_FALLBACK    = -32768
NDVI_MIN, NDVI_MAX = -1.0, 1.0

FIGSIZE           = (2, 2)
CAX_SIZE          = "5%"
CAX_PAD           = 0.01
TICK_FONTSIZE     = 3
CB_LABEL_FONTSIZE = 6
CB_TICK_FONTSIZE  = 5
BOUNDARY_COLOR    = 'red'
BOUNDARY_LW       = 0.3
MASK_COLOR        = 'red'
MASK_LW           = 0.3

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw exactly ONE boundary: shapefile if available, else eroded mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates']); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float) if HAVE_SCIPY \
                  else mask_array.astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# LOAD & PREP DATA
# =========================================================
with rasterio.open(NDVI_TIF) as src:
    ndvi_raw    = src.read(1)
    ndvi_crs    = src.crs
    ndvi_bounds = src.bounds
    ndvi_nodata = src.nodata if src.nodata is not None else NODATA_FALLBACK

with rasterio.open(STUDY_MASK) as src:
    mask = src.read(1)

# Scale and mask
ndvi = np.where(ndvi_raw == ndvi_nodata, np.nan, ndvi_raw.astype(np.float32) / float(SCALE_FACTOR))
if mask.shape == ndvi.shape:
    ndvi = np.where(mask == 1, ndvi, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != NDVI shape {ndvi.shape}; mask overlay may be skipped.")

# Extent in WGS84
west, south, east, north = transform_bounds(ndvi_crs, "EPSG:4326", *ndvi_bounds)
extent = [west, east, south, north]

# Valid check
valid = ndvi[~np.isnan(ndvi)]
if valid.size == 0:
    raise RuntimeError("No valid NDVI pixels to plot.")

# Optional shapefile boundary
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# =========================================================
# CREATE NDVI MAP 
# =========================================================
fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)

# NDVI palette (bare→green); keep full −1…1 range
ndvi_colors = ['#8B4513', '#CD853F', '#DAA520', '#FFFF00', '#ADFF2F', '#32CD32', '#228B22', '#006400']
ndvi_cmap = mcolors.LinearSegmentedColormap.from_list('NDVI', ndvi_colors, N=256)

im = ax.imshow(ndvi, cmap=ndvi_cmap, extent=extent, origin='upper',
               vmin=NDVI_MIN, vmax=NDVI_MAX, zorder=1)

# Single boundary 
_ = draw_study_boundary(ax, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

# North arrow & checker scale bar
add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                      segments=4, fontsize=4)

# Axes
ax.set_xlim(west, east); ax.set_ylim(south, north)
set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Slim colorbar 
div = make_axes_locatable(ax)
cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb  = plt.colorbar(im, cax=cax)
cb.set_label('NDVI', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb.set_ticks([-1.0, -0.5, 0.0, 0.5, 1.0])
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

# Save 
plt.tight_layout(pad=0.2)
safe_savefig(fig, OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig, OUT_PDF, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =========================================================
# Console summary 
# =========================================================
print("\n" + "="*60)
print("NDVI — SUMMARY")
print("="*60)
print(f"CRS: {ndvi_crs}")
print(f"BBox (WGS84): W={format_dms_lon(west)}, E={format_dms_lon(east)}, "
      f"S={format_dms_lat(south)}, N={format_dms_lat(north)}")
print(f"Range: {np.nanmin(valid):.3f} – {np.nanmax(valid):.3f}")
print(f"Mean ± Std: {np.nanmean(valid):.3f} ± {np.nanstd(valid):.3f}")
print(f"Median: {np.nanmedian(valid):.3f}")
print("="*60)


# Updated Code for NDVI Visualization

In [ ]:
# ============================================================
# Step-2.1 — NDVI Visualization  
# ============================================================
import os, time, stat
import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from rasterio.warp import transform_bounds

# -------- Optional deps  -----
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion; HAVE_SCIPY = True
except Exception:
    pass

# ------------------------------ PATHS -----------------------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
OUT_NDVI   = os.path.join(BASE, "Results", "Step-2.1-NDVI")
PHASE02    = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK = os.path.join(PHASE02, "study_mask.tif")
STUDY_SHP  = os.path.join(BASE, "Study_Area.shp")

NDVI_TIF   = os.path.join(OUT_NDVI, "ndvi_median_2020_2023_32645_12p5m_int16.tif")
OUT_BASE   = os.path.join(OUT_NDVI, "NDVI_Map")

# ------------------------------ STYLE -----------------------------------------
SCALE_FACTOR       = 10000.0     # int16 → float NDVI
NODATA_FALLBACK    = -32768
NDVI_MIN, NDVI_MAX = -1.0, 1.0   # keep the canonical range

FIGSIZE            = (1, 1)      # tiny 1×1 in
SAVE_DPI           = 600
CAX_SIZE           = "8%"        # right colorbar width
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 2.5
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_COLOR     = 'black'
BOUNDARY_LINEWIDTH = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_CONTOUR_WIDTH = 0.4
LW_AXIS            = 0.3
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True

# Canvas: opaque white
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# ------------------------------ Save helpers ----------------------------------
def _ensure_writable(path):
    try:
        if os.path.exists(path): os.chmod(path, stat.S_IWRITE)
    except Exception: pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs); print(f"[OUTPUT] Saved: {path}"); return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs); print(f"[OUTPUT] Saved: {alt}"); return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# ------------------------------ DMS ticks -------------------------------------
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg))); m_float = (abs(decimal_deg)-d)*60
    m = int(np.floor(m_float)); s = (m_float - m)*60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min/60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# --------------- North arrow & checkerboard scale bar -------------------------
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size*0.72; y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size*0.09)
    ax.add_patch(Polygon([apex, left_base, tail], closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north)/2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0/mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i*seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    end_km = total_len_m/1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)

# -------------------------- NDVI colormap  ---------------------------
def build_ndvi_cmap(mode="balanced"):
    mode = (mode or "").lower()
    if mode == "ylgn":
        base = plt.get_cmap("YlGn", 256)(np.linspace(0, 1, 256))
        base = 0.90*base + 0.10*np.array([1,1,1,1])  # soften
        return mcolors.ListedColormap(base)

    # Color combination
    colors = [
        "#B8885E", "#D6AA77", "#E8CCA3", "#F5F1CC",
        "#D6F0BE", "#AEE5A9", "#79CF86", "#4DB36F", "#2E8B57"
    ]
    return mcolors.LinearSegmentedColormap.from_list("NDVI_Balanced", colors, N=256)

# --------------------------------- MAIN ---------------------------------------
if __name__ == "__main__":
    os.makedirs(OUT_NDVI, exist_ok=True)

    # Load NDVI + mask
    with rasterio.open(NDVI_TIF) as src:
        ndvi_raw    = src.read(1)
        ndvi_crs    = src.crs
        ndvi_bounds = src.bounds
        ndvi_nodata = src.nodata if src.nodata is not None else NODATA_FALLBACK
    with rasterio.open(STUDY_MASK) as src:
        mask = src.read(1)

    # Scale + mask
    ndvi = np.where(ndvi_raw == ndvi_nodata, np.nan, ndvi_raw.astype(np.float32) / float(SCALE_FACTOR))
    if mask.shape == ndvi.shape:
        ndvi = np.where(mask == 1, ndvi, np.nan)
    else:
        print(f"[WARN] Mask shape {mask.shape} != NDVI shape {ndvi.shape}")

    # Extent in WGS84
    west, south, east, north = transform_bounds(ndvi_crs, "EPSG:4326", *ndvi_bounds)
    extent = [west, east, south, north]

    # Validity check
    valid = ndvi[np.isfinite(ndvi)]
    if valid.size == 0:
        raise RuntimeError("No valid NDVI pixels to plot.")

    # Boundary
    boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

    # Figure
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
    for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)

    # Lighter palette 
    cmap = build_ndvi_cmap(mode="balanced")
    cmap.set_bad((1,1,1,1))  # masked → white
    im = ax.imshow(ndvi, cmap=cmap, extent=extent, origin='upper',
                   vmin=NDVI_MIN, vmax=NDVI_MAX, zorder=1)

    # One boundary only
    _ = draw_study_boundary(ax, extent, boundary, mask,
                            shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                            mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

    # Cartography
    add_north_arrow_tri(ax)
    add_checker_scale_bar(ax, west, east, south, north, crs_type='wgs84', segments=4)

    # Axes
    ax.set_xlim(west, east); ax.set_ylim(south, north)
    set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
    ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
    ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE,
                   width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE,
                   width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
    for lbl in ax.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')

    # Right inset colorbar 
    cax = inset_axes(ax, width=CAX_SIZE, height="100%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax.transAxes, borderpad=0)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label('NDVI', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
    cb.set_ticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
    if hasattr(cb, "outline") and cb.outline is not None: cb.outline.set_linewidth(0.3)

    # Layout & export
    plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)
    save_all(fig, OUT_BASE)
    plt.show()

    # Console summary
    print("\n" + "="*60)
    print("NDVI — SUMMARY")
    print("="*60)
    print(f"BBox (WGS84): W={format_dms_lon(west)}  E={format_dms_lon(east)}  "
          f"S={format_dms_lat(south)}  N={format_dms_lat(north)}")
    print(f"Range: {np.nanmin(valid):.3f} – {np.nanmax(valid):.3f}")
    print(f"Mean ± Std: {np.nanmean(valid):.3f} ± {np.nanstd(valid):.3f}")
    print(f"Median: {np.nanmedian(valid):.3f}")
    print("="*60)


# Step-2.2: Average Annual Precipitation (AAP)

In [ ]:
# =========================================================
# Step 2.2 — CHIRPS precipitation 
# =========================================================
import os, io, json, time, zipfile, requests, struct, shutil
from datetime import datetime, timedelta
import ee
import numpy as np
import pandas as pd

# Optional raster stage
try:
    import rasterio
    from rasterio.merge import merge
    from rasterio.warp import reproject, Resampling
    _HAVE_RASTER = True
except Exception:
    _HAVE_RASTER = False

# -------------PATH-----------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
SHP_PATH  = os.path.join(BASE, "Study_Area.shp")
PRJ_PATH  = os.path.splitext(SHP_PATH)[0] + ".prj"

ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
MASK_TIF   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

OUT_DIR = os.path.join(BASE, "Results", "Step-2.2-AAP")
os.makedirs(OUT_DIR, exist_ok=True)
CSV_OUT = os.path.join(OUT_DIR, "aap_daily_2020_2023.csv")
AAP_TIF = os.path.join(OUT_DIR, "aap_2020_2023_32645_12p5m_float32.tif")
TMP_DIR = os.path.join(OUT_DIR, "_tmp"); os.makedirs(TMP_DIR, exist_ok=True)

# GEE auth
SERVICE_ACCOUNT_KEY = os.environ.get("EE_SERVICE_ACCOUNT_KEY", "")
PROJECT_ID = os.environ.get("EE_PROJECT", "")

# CHIRPS + export settings
DATE_START, DATE_END = "2019-01-09", "2022-07-16"
CHIRPS_DATASET = "UCSB-CHG/CHIRPS/DAILY"
# IMPORTANT: Earth Engine 'scale' is in METERS. ~0.05° ≈ ~6 km.
CHIRPS_SCALE_M = 6000.0            # meters (safe for reductions & exports)
MAX_PIXELS     = 1e13
CONNECT_TIMEOUT = (30, 600)

# AAP raster output
AAP_NODATA = -9999.0

# Cleanup configuration
KEEP_TEMP_FILES = False  # Set to True for debugging purposes

def pp(msg): print(msg, flush=True)

# ----------------------------------
# GEE initialization (explicit project)
# ----------------------------------
def ee_auth():
    with open(SERVICE_ACCOUNT_KEY, "r") as f:
        key = json.load(f)
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    creds = ee.ServiceAccountCredentials(email=key["client_email"], key_data=json.dumps(key))
    ee.Initialize(credentials=creds, project=PROJECT_ID)
    _ = ee.Number(1).getInfo()  # ping
    pp(f"[EE] Initialized with project: {PROJECT_ID}")

# -------------------------------------------------------
# Minimal shapefile reader + .prj CRS sniff (no GDAL)
# -------------------------------------------------------
def guess_epsg_from_prj(prj_path):
    try:
        txt = open(prj_path, "r", encoding="utf-8", errors="ignore").read().lower()
    except Exception:
        return None
    if "wgs_1984_utm" in txt or "wgs 1984 utm" in txt:
        if "zone_45" in txt or "zone 45" in txt or "utm_45n" in txt:
            return 32645
    if "gcs_wgs_1984" in txt or "wgs 84" in txt or "wgs_1984" in txt:
        return 4326
    return None

def read_main_polygon_from_shp(shp_path):
    """
    Reads the largest polygon ring from a polygon/multipolygon .shp.
    Returns (coords, epsg_guess, bounds) where coords=[[x,y],...] in the native CRS.
    """
    with open(shp_path, "rb") as f:
        header = f.read(100)
        file_length = struct.unpack(">I", header[24:28])[0] * 2
        shape_type = struct.unpack("<I", header[32:36])[0]
        xmin, ymin, xmax, ymax = struct.unpack("<4d", header[36:68])

        polys = []
        f.seek(100)
        while f.tell() < file_length:
            rec_hdr = f.read(8)
            if len(rec_hdr) < 8: break
            content_len = struct.unpack(">I", rec_hdr[4:8])[0] * 2
            st = struct.unpack("<I", f.read(4))[0]
            if st in (5, 15):  # Polygon / PolygonZ
                _ = struct.unpack("<4d", f.read(32))
                num_parts = struct.unpack("<I", f.read(4))[0]
                num_points = struct.unpack("<I", f.read(4))[0]
                parts = [struct.unpack("<I", f.read(4))[0] for _ in range(num_parts)]
                pts = [struct.unpack("<2d", f.read(16)) for _ in range(num_points)]
                if st == 15:
                    remaining = content_len - (4+32+8 + num_parts*4 + num_points*16)
                    if remaining > 0: f.read(remaining)
                outer = []
                for i in range(num_parts):
                    a = parts[i]
                    b = parts[i+1] if i+1 < num_parts else num_points
                    ring = pts[a:b]
                    if len(ring) > len(outer): outer = ring
                if len(outer) >= 4:
                    if outer[0] != outer[-1]:
                        outer = outer + [outer[0]]
                    polys.append(outer)
            else:
                remaining = content_len - 4
                if remaining > 0: f.read(remaining)

        if not polys:
            raise RuntimeError("No polygon geometry found in shapefile.")

        main_ring = max(polys, key=len)
        epsg = guess_epsg_from_prj(PRJ_PATH)
        return main_ring, epsg, (xmin, ymin, xmax, ymax)

# ----------------------------------
# Anchor grid (target EPSG:32645, 12.5 m)
# ----------------------------------
def load_anchor_spec(anchor_path):
    if not _HAVE_RASTER:
        return None
    with rasterio.open(anchor_path) as ds:
        tf = ds.transform
        w, h = ds.width, ds.height
        crs = ds.crs
        bounds = ds.bounds
    epsg = int(crs.to_epsg()) if crs and crs.to_epsg() else None
    return dict(tf=tf, width=w, height=h, epsg=epsg,
                bounds=(bounds.left,bounds.bottom,bounds.right,bounds.top))

# ----------------------------------
# CHIRPS collection
# ----------------------------------
def build_chirps_ic(aoi):
    ic = (ee.ImageCollection(CHIRPS_DATASET)
          .filterDate(DATE_START, DATE_END)
          .filterBounds(aoi)
          .select("precipitation"))
    n = ic.size().getInfo()
    pp(f"[INFO] CHIRPS daily images in range: {n}")
    if n == 0:
        raise SystemExit("No CHIRPS images for dates/region.")
    return ic

# ----------------------------------
# Daily time series (FAST band-stack reduction, fallback-safe)
# ----------------------------------
def extract_daily_timeseries(ic, aoi_geom):
    pp("[TS] Extracting daily precipitation time series …")

    aoi_simpl = aoi_geom.simplify(200)  # speed up reductions safely

    def rename_by_date(img):
        # band name like P_YYYYMMDD (unique per day)
        return img.rename(ee.String('P_').cat(img.date().format('YYYYMMdd')))

    try:
        bandstack = ee.ImageCollection(ic.map(rename_by_date)).toBands()
        stats = bandstack.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi_simpl,
            scale=CHIRPS_SCALE_M,   # meters (≈6 km)
            bestEffort=True, tileScale=4, maxPixels=MAX_PIXELS
        )
        d = stats.getInfo() or {}
        rows = []
        for k, v in sorted(d.items()):   # k = 'P_YYYYMMDD'
            day = k.split('_')[-1]
            rows.append({
                "date": f"{day[:4]}-{day[4:6]}-{day[6:]}",
                "precip_mm": float(v) if v is not None else np.nan
            })
        pp(f"[TS] Done. {len(rows)} days.")
        return rows

    except Exception as e:
        # Fallback: per-image (still fine with proper meter scale)
        pp(f"[TS] Band-stack method failed ({e}). Falling back to per-image loop …")
        rows = []
        def img_to_feat(img):
            val = img.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=aoi_simpl,
                scale=CHIRPS_SCALE_M,
                bestEffort=True, tileScale=4, maxPixels=MAX_PIXELS
            ).get("precipitation")
            return ee.Feature(None, {
                "date": img.date().format("YYYY-MM-dd"),
                "precip_mm": val
            })
        fc = ee.FeatureCollection(ic.map(img_to_feat))
        dates  = fc.aggregate_array("date").getInfo()
        vals   = fc.aggregate_array("precip_mm").getInfo()
        for dte, v in zip(dates, vals):
            rows.append({"date": dte, "precip_mm": float(v) if v is not None else np.nan})
        rows.sort(key=lambda r: r["date"])
        pp(f"[TS] Done. {len(rows)} days (fallback).")
        return rows

# ----------------------------------
# Robust monthly export helpers (GeoTIFF @ ~6 km)
# ----------------------------------
def _download_to_bytes(image, params, tag, max_retries=3):
    allowed = ("region","scale","crs","crsTransform","format","maxPixels")
    safe = {k: v for k, v in params.items() if k in allowed}
    url = image.getDownloadURL(safe)
    dbg = {k: ('<geom>' if k=='region' else v) for k,v in safe.items()}
    for attempt in range(max_retries):
        pp(f"[REQ] {tag} (try {attempt+1}/{max_retries}): {dbg}")
        try:
            r = requests.get(url, stream=True, timeout=CONNECT_TIMEOUT)
            if r.status_code != 200:
                msg = r.content.decode("utf-8", errors="ignore")
                if "memory" in msg.lower() and attempt < max_retries - 1:
                    time.sleep(3); continue
                raise RuntimeError(f"HTTP {r.status_code}: {msg[:300]} …")
            head = r.raw.read(4); rest = r.raw.read()
            return head + rest
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(5)

def _bytes_to_tif(blob, out_path):
    tmp = out_path + ".part"
    if blob[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(blob)) as zf:
            tifs = [n for n in zf.namelist() if n.lower().endswith((".tif",".tiff"))]
            if not tifs: raise RuntimeError("ZIP contained no TIFF.")
            with zf.open(tifs[0]) as zin, open(tmp, "wb") as out: out.write(zin.read())
    else:
        with open(tmp, "wb") as out: out.write(blob)
    os.replace(tmp, out_path)

def export_monthly_totals(ic, aoi_geom, tmp_dir):
    """
    For each month in 2020–2023, export CHIRPS monthly total (mm).
    Returns list of file paths in chronological order.
    """
    pp("[RASTER] Monthly export path (robust to server memory limits)…")
    aoi4326 = aoi_geom.transform("EPSG:4326", 1)
    geom = aoi4326.getInfo()  # full GeoJSON geometry

    start = datetime.strptime(DATE_START, "%Y-%m-%d")
    end   = datetime.strptime(DATE_END,   "%Y-%m-%d")
    paths = []

    cur = datetime(start.year, start.month, 1)
    while cur <= end:
        y = cur.year
        m = cur.month
        m0 = datetime(y, m, 1)
        m1 = datetime(y + (1 if m==12 else 0), 1 if m==12 else m+1, 1)

        tag = f"chirps_{y}{m:02d}"
        pp(f"[MONTH] {y}{m:02d}: exporting …")

        monthly = (ic.filterDate(m0.strftime("%Y-%m-%d"), m1.strftime("%Y-%m-%d"))
                     .sum()                              # mm in the month
                     .select("precipitation")
                     .clip(aoi_geom)
                     .selfMask())                        # no zeros outside data

        params = {
            "region": geom,
            "crs": "EPSG:4326",          # keep WGS84
            "scale": CHIRPS_SCALE_M,     # meters ≈ 6 km
            "format": "GEO_TIFF",
            "maxPixels": MAX_PIXELS
        }

        out_tif = os.path.join(tmp_dir, f"{tag}.tif")
        blob = _download_to_bytes(monthly, params, f"{tag}_single")
        _bytes_to_tif(blob, out_tif)
        paths.append(out_tif)

        cur = m1  # next month

    return paths

# ----------------------------------
# Build AAP from monthly totals → warp to anchor → mask → save
# ----------------------------------
def build_aap_raster(month_tifs, anchor_spec, mask_path, out_tif):
    if not _HAVE_RASTER:
        pp("[RASTER] Skipped: rasterio not installed.")
        return False

    # group months -> years
    by_year = {}
    for p in month_tifs:
        # filename ...chirps_YYYYMM.tif
        base = os.path.basename(p)
        y = int(base.split("_")[1][:4])
        by_year.setdefault(y, []).append(p)

    years = sorted([y for y in by_year if 2020 <= y <= 2023])
    if not years:
        raise SystemExit("No monthly files to assemble.")

    annual_arrays = []
    src_crs = None; src_tf = None

    for y in years:
        files = sorted(by_year[y])
        pp(f"[RASTER] Summing {len(files)} months for {y} …")
        arr_sum = None
        for i, fp in enumerate(files):
            with rasterio.open(fp) as ds:
                A = ds.read(1).astype("float32")
                if src_crs is None:
                    src_crs = ds.crs
                    src_tf  = ds.transform
                if arr_sum is None:
                    arr_sum = A.copy()
                else:
                    if A.shape != arr_sum.shape or ds.transform != src_tf:
                        # rare: fallback mosaic to align grids
                        pp("  [WARN] month shape mismatch; merging …")
                        srcs = [rasterio.open(f) for f in files]
                        mosaic, m_tf = merge(srcs, method="first")
                        for s in srcs: s.close()
                        arr_sum = mosaic[0].astype("float32")
                        src_tf = m_tf
                        src_crs = ds.crs
                        break
                    arr_sum += A
        annual_arrays.append(arr_sum)

    # AAP = mean(annual_totals) over 2020–2023
    pp("[RASTER] Computing AAP (mean of annual totals 2020–2023) …")
    stack = np.stack(annual_arrays, axis=0)  # (ny, H, W)
    with np.errstate(invalid="ignore"):
        aap_native = np.nanmean(stack, axis=0)  # mm/year
    aap_native = np.where(np.isfinite(aap_native), aap_native, np.nan)

    # Warp to anchor CRS/grid (EPSG:32645, 12.5 m)
    dst_tf = anchor_spec["tf"]; dst_w=anchor_spec["width"]; dst_h=anchor_spec["height"]
    dst_crs = f"EPSG:{anchor_spec['epsg']}"

    dst = np.full((1, dst_h, dst_w), AAP_NODATA, dtype="float32")

    reproject(
        source=aap_native[np.newaxis, :, :],
        destination=dst,
        src_transform=src_tf, src_crs=src_crs,
        dst_transform=dst_tf, dst_crs=dst_crs,
        resampling=Resampling.bilinear,
        src_nodata=np.nan, dst_nodata=AAP_NODATA
    )

    # Apply study_mask if present
    if os.path.exists(mask_path):
        with rasterio.open(mask_path) as ms:
            m = ms.read(1)
            if (ms.transform != dst_tf) or (ms.width != dst_w) or (ms.height != dst_h) or (ms.crs != rasterio.crs.CRS.from_string(dst_crs)):
                pp("[MASK] Reprojecting study_mask to anchor grid …")
                mask_res = np.zeros((dst_h, dst_w), dtype="uint8")
                reproject(
                    source=m, destination=mask_res,
                    src_transform=ms.transform, src_crs=ms.crs,
                    dst_transform=dst_tf,      dst_crs=dst_crs,
                    resampling=Resampling.nearest,
                    src_nodata=0, dst_nodata=0
                )
                m = mask_res
            dst = np.where(m > 0, dst, AAP_NODATA)

    profile = {"driver":"GTiff","height":dst_h,"width":dst_w,"count":1,
               "dtype":"float32","crs":dst_crs,"transform":dst_tf,
               "nodata":AAP_NODATA,"compress":"LZW","tiled":True}
    tmp = out_tif + ".part"
    with rasterio.open(tmp, "w", **profile) as ds:
        ds.write(dst)
    os.replace(tmp, out_tif)
    pp(f"[RASTER] Final AAP → {out_tif}")
    return True

# ----------------------------------
# Cleanup function
# ----------------------------------
def cleanup_temp_files():
    """Remove temporary directory and all its contents."""
    if os.path.exists(TMP_DIR):
        try:
            shutil.rmtree(TMP_DIR)
            pp(f"[CLEANUP] Removed temporary directory: {TMP_DIR}")
            return True
        except Exception as e:
            pp(f"[CLEANUP] Warning: Could not remove temp dir {TMP_DIR}: {e}")
            return False
    return True

# ----------------
# MAIN
# ----------------
def main():
    pp("=== CHIRPS AAP (daily CSV + AAP raster) ===")
    pp(f"[CONFIG] scale={CHIRPS_SCALE_M} m, dates {DATE_START}..{DATE_END}")
    pp(f"[CONFIG] Cleanup temporary files: {'No' if KEEP_TEMP_FILES else 'Yes'}")
    
    try:
        ee_auth()

        # AOI from shapefile
        try:
            ring_native, shp_epsg, shp_bounds = read_main_polygon_from_shp(SHP_PATH)
            if shp_epsg is None:
                pp("[AOI] .prj not clear; assuming EPSG:32645 if anchor is present.")
        except Exception as e:
            raise SystemExit(f"[AOI] Failed to read shapefile: {e}")

        # Anchor grid
        if not _HAVE_RASTER or not os.path.exists(ANCHOR_TIF):
            raise SystemExit("[ANCHOR] Anchor grid is required to build final raster.")
        anchor = load_anchor_spec(ANCHOR_TIF)
        if anchor is None or anchor["epsg"] is None:
            raise SystemExit("[ANCHOR] Could not read anchor grid/EPSG.")

        # AOI geometry for EE (use shapefile EPSG or fall back to anchor EPSG)
        shp_proj_epsg = shp_epsg or anchor["epsg"]
        aoi_geom = ee.Geometry.Polygon([ring_native], proj=ee.Projection(f"EPSG:{shp_proj_epsg}"), geodesic=False)

        # 1) ImageCollection
        ic = build_chirps_ic(aoi_geom)

        # 2) Daily CSV (fast band-stack → single reduceRegion)
        rows = extract_daily_timeseries(ic, aoi_geom)
        pd.DataFrame(rows).to_csv(CSV_OUT, index=False)
        pp(f"[CSV] Wrote: {CSV_OUT} ({len(rows)} rows)")

        # 3) Monthly totals → Annual totals → AAP → warp to anchor
        month_tifs = export_monthly_totals(ic, aoi_geom, TMP_DIR)
        ok = build_aap_raster(month_tifs, anchor, MASK_TIF, AAP_TIF)

        # 4) QA
        if ok:
            try:
                with rasterio.open(AAP_TIF) as ds:
                    a = ds.read(1)
                    m = a != ds.nodata
                    if m.any():
                        vals = a[m]
                        pp(f"[QA] valid={int(m.sum())}  min/mean/max={float(vals.min()):.1f}/{float(vals.mean()):.1f}/{float(vals.max()):.1f} mm/yr")
                    pp(f"[QA] CRS: {ds.crs}  shape: {ds.height} x {ds.width}")
            except Exception:
                pass

    finally:
        # Always attempt cleanup, regardless of success or failure
        if not KEEP_TEMP_FILES:
            cleanup_temp_files()
        elif KEEP_TEMP_FILES:
            pp(f"[DEBUG] Keeping temporary files in: {TMP_DIR}")

    pp("=== Done ===")

if __name__ == "__main__":
    main()

# Visualization: AAP

In [ ]:
# ============================================================
# Step-2.2 — Visualization for AAP 
# ============================================================
import os, time, stat
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.warp import transform_bounds

# Optional deps 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion
    HAVE_SCIPY = True
except Exception:
    pass

# ------------------------- PATHS ---------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP_MASK   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK  = os.path.join(STEP_MASK, "study_mask.tif")
STUDY_SHP   = os.path.join(BASE, "Study_Area.shp")

OUT_AAP     = os.path.join(BASE, "Results", "Step-2.2-AAP")
AAP_TIF     = os.path.join(OUT_AAP, "aap_2020_2023_32645_12p5m_float32.tif")
DAILY_CSV   = os.path.join(OUT_AAP, "aap_daily_2020_2023.csv")

AAP_MAP_PNG = os.path.join(OUT_AAP, "AAP Map.png")
AAP_MAP_PDF = os.path.join(OUT_AAP, "AAP Map.pdf")
DAILY_CHART = os.path.join(OUT_AAP, "Daily Precipitation Timeseries.png")

AAP_NODATA  = -9999.0

# ------------------------- STYLE ---------------------
FIGSIZE           = (2, 2)
CAX_SIZE          = "5%"
CAX_PAD           = 0.01
TICK_FONTSIZE     = 3
CB_LABEL_FONTSIZE = 6
CB_TICK_FONTSIZE  = 5
BOUNDARY_COLOR    = 'red'
BOUNDARY_LW       = 0.3
MASK_COLOR        = 'red'
MASK_LW           = 0.3

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(deg_min, deg_max, step_min):
        step_deg = step_min / 60.0
        start = np.floor(deg_min * 60.0 / step_min) * step_deg
        end   = np.ceil (deg_max * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= deg_min) & (vals <= deg_max)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# NORTH ARROW & CHECKER SCALE BAR 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N',
                        arrow_downshift=0.06, label_upshift=0.18):
    x0 = pad
    y0 = 1.0 - pad - size - float(arrow_downshift)
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size * float(label_upshift), label,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target:
            return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data
        units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd

    total_len_m     = nice_len_125(width_m / 5.0)
    seg_len_m       = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    pad_x = total_len_units * 0.08
    pad_y = 0.035 * height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Shapefile loader + unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    """Draw exactly ONE boundary: shapefile if available, else eroded mask contour."""
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates']); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass

    if mask_array is not None:
        try:
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float) if HAVE_SCIPY \
                  else mask_array.astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# LOAD & PREP RASTER/MASK
# =========================================================
with rasterio.open(AAP_TIF) as src:
    aap         = src.read(1)
    aap_crs     = src.crs
    aap_bounds  = src.bounds
    aap_nodata  = src.nodata if src.nodata is not None else AAP_NODATA

with rasterio.open(STUDY_MASK) as src:
    mask = src.read(1)

aap = np.where(aap == aap_nodata, np.nan, aap.astype(np.float32))
if mask.shape == aap.shape:
    aap = np.where(mask == 1, aap, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != AAP shape {aap.shape}; mask overlay may be skipped.")

# Extent (WGS84) for DMS axes
west, south, east, north = transform_bounds(aap_crs, "EPSG:4326", *aap_bounds)
extent = [west, east, south, north]

valid = aap[~np.isnan(aap)]
if valid.size == 0:
    raise RuntimeError("No valid AAP pixels to plot.")

# Optional shapefile boundary
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# =========================================================
# AAP MAP — compact Step-1.3 style
# =========================================================
fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)

# Sequential precipitation colormap 
precip_colors = ['#f7fcf0', '#e0f3db', '#ccebc5', '#a8ddb5', '#7bccc4',
                 '#4eb3d3', '#2b8cbe', '#0868ac', '#084081']
precip_cmap = mcolors.LinearSegmentedColormap.from_list('precip', precip_colors, N=256)

# Robust 2–98% stretch
vmin = np.nanpercentile(valid, 2)
vmax = np.nanpercentile(valid, 98)
if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
    vmin, vmax = np.nanmin(valid), np.nanmax(valid)

im = ax.imshow(aap, cmap=precip_cmap, extent=extent, origin='upper',
               vmin=vmin, vmax=vmax, zorder=1)

# Single boundary 
_ = draw_study_boundary(ax, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_COLOR, mask_lw=MASK_LW, erode_iters=2)

# North arrow & scale bar
add_north_arrow_tri(ax, size=0.09, pad=0.07, arrow_downshift=0.06, label_upshift=0.18)
add_checker_scale_bar(ax, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                      segments=4, fontsize=4)

# Axes: DMS ticks + tiny fonts
ax.set_xlim(west, east); ax.set_ylim(south, north)
set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

# Slim right-side colorbar
div = make_axes_locatable(ax)
cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb  = plt.colorbar(im, cax=cax)
cb.set_label('AAP (mm/year)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

plt.tight_layout(pad=0.2)
safe_savefig(fig, AAP_MAP_PNG, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig, AAP_MAP_PDF, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =========================================================
# DAILY PRECIPITATION TIME SERIES 
# =========================================================
if os.path.exists(DAILY_CSV):
    df = pd.read_csv(DAILY_CSV)

    # Heuristic column detection
    cols_lower = {c: str(c).lower() for c in df.columns}
    date_col = next((c for c, v in cols_lower.items()
                     if any(k in v for k in ['date', 'time'])), df.columns[0])
    precip_col = next((c for c, v in cols_lower.items()
                       if any(k in v for k in ['precip', 'rain', 'mm', 'value'])),
                      df.columns[1] if len(df.columns) > 1 else df.columns[0])

    data = df[[date_col, precip_col]].dropna().copy()

    # Parse dates robustly
    parsed = None
    for fmt in ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d', '%d-%m-%Y', '%m-%d-%Y']:
        try:
            parsed = pd.to_datetime(data[date_col], format=fmt)
            break
        except Exception:
            continue
    if parsed is None:
        parsed = pd.to_datetime(data[date_col], errors='coerce')
    data[date_col] = parsed
    data[precip_col] = pd.to_numeric(data[precip_col], errors='coerce')
    data = data.dropna().sort_values(date_col)

    if len(data) > 0:
        fig2, ax2 = plt.subplots(1, 1, figsize=(6, 4))
        dates = data[date_col].values
        vals  = data[precip_col].values

        markerline, stemlines, baseline = ax2.stem(dates, vals, linefmt='C0', markerfmt='o', basefmt=' ')
        markerline.set_markerfacecolor('C0')
        markerline.set_markeredgecolor('C0')
        markerline.set_markersize(3.5)
        stemlines.set_color('C0'); stemlines.set_linewidth(1); stemlines.set_alpha(0.7)

        ax2.set_title('Daily Precipitation', fontsize=12, pad=10)
        ax2.set_xlabel('Time', fontsize=9); ax2.set_ylabel('Precipitation (mm)', fontsize=9)
        ymax = max(120, int(np.nanmax(vals) * 1.1)) if np.isfinite(np.nanmax(vals)) else 100
        ax2.set_ylim(0, ymax)

        ax2.xaxis.set_major_locator(mdates.YearLocator())
        ax2.xaxis.set_minor_locator(mdates.MonthLocator((1, 7)))
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m/%Y'))
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
        ax2.tick_params(axis='y', labelsize=8)
        ax2.grid(True, linestyle='-', alpha=0.3, color='gray'); ax2.set_axisbelow(True)

        plt.tight_layout()
        safe_savefig(fig2, DAILY_CHART, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
    else:
        print("[INFO] Daily precipitation CSV parsed but no valid rows after cleaning; skipped chart.")
else:
    print("[INFO] Daily precipitation CSV not found; skipped time series.")

# =========================================================
# Console summary (QA)
# =========================================================
print("\n" + "="*60)
print("AAP — SUMMARY")
print("="*60)
print(f"CRS: {aap_crs}")
print(f"BBox (WGS84): W={format_dms_lon(west)}, E={format_dms_lon(east)}, "
      f"S={format_dms_lat(south)}, N={format_dms_lat(north)}")
print(f"Range: {np.nanmin(valid):.2f} – {np.nanmax(valid):.2f} mm/yr")
print(f"Mean ± Std: {np.nanmean(valid):.2f} ± {np.nanstd(valid):.2f} mm/yr")
print(f"Median: {np.nanmedian(valid):.2f} mm/yr")
print(f"Display stretch (map): {vmin:.2f} → {vmax:.2f} mm/yr")
print("="*60)


# Updated Code for AAP Visualization

In [ ]:
# ============================================================
# Step-2.2 — AAP Visualization 
# ============================================================
import os, time, stat
import numpy as np
import pandas as pd
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from rasterio.warp import transform_bounds

# -------- Optional deps -----
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion; HAVE_SCIPY = True
except Exception:
    pass

# ------------------------- PATHS ---------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP_MASK   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
STUDY_MASK  = os.path.join(STEP_MASK, "study_mask.tif")
STUDY_SHP   = os.path.join(BASE, "Study_Area.shp")

OUT_AAP     = os.path.join(BASE, "Results", "Step-2.2-AAP")
AAP_TIF     = os.path.join(OUT_AAP, "aap_2020_2023_32645_12p5m_float32.tif")
DAILY_CSV   = os.path.join(OUT_AAP, "aap_daily_2020_2023.csv")

AAP_OUT_BASE = os.path.join(OUT_AAP, "AAP_Map")
DAILY_CHART  = os.path.join(OUT_AAP, "Daily_Precipitation_Timeseries.png")

AAP_NODATA  = -9999.0

# ------------------------- STYLE  ---------------------
FIGSIZE            = (1, 1)
SAVE_DPI           = 600
CAX_SIZE           = "8%"     
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 2.5
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_COLOR     = 'black'
BOUNDARY_LINEWIDTH = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_CONTOUR_WIDTH = 0.4
LW_AXIS            = 0.3
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True

# Opaque white canvas
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# =========================================================
# Save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path): os.chmod(path, stat.S_IWRITE)
    except Exception: pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs); print(f"[OUTPUT] Saved: {path}"); return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs); print(f"[OUTPUT] Saved: {alt}"); return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# =========================================================
# DMS formatters + 5′/10′ ticks
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg))); m_float = (abs(decimal_deg)-d)*60
    m = int(np.floor(m_float)); s = (m_float - m)*60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min/60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# North arrow & checker scale bar
# =========================================================
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size*0.72; y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size*0.09)
    ax.add_patch(Polygon([apex, left_base, tail], closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north)/2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0/mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i*seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    end_km = total_len_m/1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)

# =========================================================
# Shapefile loader + unified boundary drawer
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path): return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}": gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geometries.append(feat['geometry'])
            return {'geometries': geometries, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH,
                        erode_iters=2):
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass
    # Mask fallback
    try:
        arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float)
    except Exception:
        arr = (mask_array.astype(float) if mask_array is not None else None)
    if arr is not None:
        ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                   extent=extent, origin='upper', zorder=11)
        return True
    return False

# =========================================================
# Precipitation colormap 
# =========================================================
def build_precip_cmap(mode="bugn_soft"):
    mode = (mode or "").lower()
    if mode == "ylgnbu_soft":
        base = plt.get_cmap("YlGnBu", 256)(np.linspace(0, 1, 256))
        base = 0.92*base + 0.08*np.array([1,1,1,1])
        return mcolors.ListedColormap(base)
    # default
    base = plt.get_cmap("BuGn", 256)(np.linspace(0, 1, 256))
    base = 0.92*base + 0.08*np.array([1,1,1,1])
    return mcolors.ListedColormap(base)

# =========================================================
# LOAD & PREP RASTER/MASK
# =========================================================
os.makedirs(OUT_AAP, exist_ok=True)

with rasterio.open(AAP_TIF) as src:
    aap         = src.read(1)
    aap_crs     = src.crs
    aap_bounds  = src.bounds
    aap_nodata  = src.nodata if src.nodata is not None else AAP_NODATA

with rasterio.open(STUDY_MASK) as src:
    mask = src.read(1)

aap = np.where(aap == aap_nodata, np.nan, aap.astype(np.float32))
if mask.shape == aap.shape:
    aap = np.where(mask == 1, aap, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != AAP shape {aap.shape}")

# Extent (WGS84)
west, south, east, north = transform_bounds(aap_crs, "EPSG:4326", *aap_bounds)
extent = [west, east, south, north]

valid = aap[np.isfinite(aap)]
if valid.size == 0:
    raise RuntimeError("No valid AAP pixels to plot.")

# Boundary
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)

# =========================================================
# AAP MAP 
# =========================================================
fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)

# Robust stretch (2–98%)
vmin = float(np.nanpercentile(valid, 2))
vmax = float(np.nanpercentile(valid, 98))
if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
    vmin, vmax = float(np.nanmin(valid)), float(np.nanmax(valid))

cmap = build_precip_cmap(mode="ylgnbu_soft")
cmap.set_bad((1,1,1,1))
im = ax.imshow(aap, cmap=cmap, extent=extent, origin='upper',
               vmin=vmin, vmax=vmax, zorder=1)

# One boundary only
_ = draw_study_boundary(ax, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LINEWIDTH,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_CONTOUR_WIDTH, erode_iters=2)

# Cartography
add_north_arrow_tri(ax)
add_checker_scale_bar(ax, west, east, south, north, crs_type='wgs84', segments=4)

# Axes
ax.set_xlim(west, east); ax.set_ylim(south, north)
set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE,
               width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE,
               width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
for lbl in ax.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')

# Right inset colorbar 
cax = inset_axes(ax, width=CAX_SIZE, height="100%", loc="lower left",
                 bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax.transAxes, borderpad=0)
cb  = plt.colorbar(im, cax=cax)
cb.set_label('AAP (mm/yr)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
if hasattr(cb, "outline") and cb.outline is not None: cb.outline.set_linewidth(0.3)

plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)
save_all(fig, AAP_OUT_BASE)
plt.show()

# =========================================================
# DAILY PRECIPITATION TIME SERIES 
# =========================================================
if os.path.exists(DAILY_CSV):
    df = pd.read_csv(DAILY_CSV)

    # Heuristic column detection
    cols_lower = {c: str(c).lower() for c in df.columns}
    date_col = next((c for c, v in cols_lower.items()
                     if any(k in v for k in ['date', 'time'])), df.columns[0])
    precip_col = next((c for c, v in cols_lower.items()
                       if any(k in v for k in ['precip', 'rain', 'mm', 'value'])),
                      df.columns[1] if len(df.columns) > 1 else df.columns[0])

    data = df[[date_col, precip_col]].dropna().copy()

    # Parse dates robustly
    parsed = None
    for fmt in ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d', '%d-%m-%Y', '%m-%d-%Y']:
        try:
            parsed = pd.to_datetime(data[date_col], format=fmt)
            break
        except Exception:
            continue
    if parsed is None:
        parsed = pd.to_datetime(data[date_col], errors='coerce')
    data[date_col] = parsed
    data[precip_col] = pd.to_numeric(data[precip_col], errors='coerce')
    data = data.dropna().sort_values(date_col)

    if len(data) > 0:
        fig2, ax2 = plt.subplots(1, 1, figsize=(6, 4))
        dates = data[date_col].values
        vals  = data[precip_col].values

        markerline, stemlines, baseline = ax2.stem(dates, vals, linefmt='C0', markerfmt='o', basefmt=' ')
        markerline.set_markerfacecolor('C0')
        markerline.set_markeredgecolor('C0')
        markerline.set_markersize(3.5)
        stemlines.set_color('C0'); stemlines.set_linewidth(1); stemlines.set_alpha(0.7)

        ax2.set_title('Daily Precipitation', fontsize=12, pad=10)
        ax2.set_xlabel('Time', fontsize=9); ax2.set_ylabel('Precipitation (mm)', fontsize=9)
        ymax = max(120, int(np.nanmax(vals) * 1.1)) if np.isfinite(np.nanmax(vals)) else 100
        ax2.set_ylim(0, ymax)

        ax2.xaxis.set_major_locator(mdates.YearLocator())
        ax2.xaxis.set_minor_locator(mdates.MonthLocator((1, 7)))
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m/%Y'))
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
        ax2.tick_params(axis='y', labelsize=8)
        ax2.grid(True, linestyle='-', alpha=0.3, color='gray'); ax2.set_axisbelow(True)

        plt.tight_layout()
        safe_savefig(fig2, DAILY_CHART, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
    else:
        print("[INFO] Daily precipitation CSV parsed but no valid rows after cleaning; skipped chart.")
else:
    print("[INFO] Daily precipitation CSV not found; skipped time series.")

# =========================================================
# Console summary (QA)
# =========================================================
print("\n" + "="*60)
print("AAP — SUMMARY")
print("="*60)
print(f"BBox (WGS84): W={format_dms_lon(west)}  E={format_dms_lon(east)}  "
      f"S={format_dms_lat(south)}  N={format_dms_lat(north)}")
print(f"Range: {np.nanmin(valid):.2f} – {np.nanmax(valid):.2f} mm/yr")
print(f"Mean ± Std: {np.nanmean(valid):.2f} ± {np.nanstd(valid):.2f} mm/yr")
print(f"Median: {np.nanmedian(valid):.2f} mm/yr")
print(f"Display stretch: {vmin:.2f} → {vmax:.2f} mm/yr")
print("="*60)


# Step-2.3: Land Soil Moisture (Monsoon)

In [ ]:
# =========================================================
# Step 2.3 — ERA5-Land Soil Moisture 
# =========================================================
import os, io, json, time, zipfile, requests, struct, math, tempfile
import ee
import numpy as np
import pandas as pd
from datetime import datetime

# Optional raster stage
try:
    import rasterio
    from rasterio.warp import reproject, Resampling
    _HAVE_RASTER = True
except Exception:
    _HAVE_RASTER = False

# ---------------------PATH----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
# Input Path
SHP_PATH  = os.path.join(BASE, "Study_Area.shp")
PRJ_PATH  = os.path.splitext(SHP_PATH)[0] + ".prj"
ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
MASK_TIF   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

# Output Path
OUT_DIR = os.path.join(BASE, "Results", "Step-2.3-SM"); os.makedirs(OUT_DIR, exist_ok=True)
CSV_OUT = os.path.join(OUT_DIR, "sm_monsoon_monthly_2020_2023.csv")
SM_TIF  = os.path.join(OUT_DIR, "sm_monsoon_mean_2020_2023_32645_12p5m_float32.tif")

# GEE auth
SERVICE_ACCOUNT_KEY = os.environ.get("EE_SERVICE_ACCOUNT_KEY", "")
PROJECT_ID = os.environ.get("EE_PROJECT", "")

# ERA5-Land settings
DATASET = "ECMWF/ERA5_LAND/HOURLY"
BAND    = "volumetric_soil_water_layer_1"
MONSOON_MONTHS = [6, 7, 8, 9]
YEARS = [2020, 2021, 2022, 2023]

# Scales / limits
REDUCE_SCALE_M       = 25000       # 25 km reductions (CSV)
EXPORT_SCALE_M_BASE  = 10000       # desired download resolution (10 km) — will auto-bump if needed
MAX_PIXELS           = 1e12
CONNECT_TIMEOUT      = (30, 600)
NODATA               = -9999.0

# Performance settings
SIMPLIFY_TOL_DEG = 0.1
TILESCALE        = 16
PIX_MAX_SIDE     = 32700          # safety a bit below 32768

def pp(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

# ----------------------------------
# GEE initialization
# ----------------------------------
def ee_auth():
    try:
        with open(SERVICE_ACCOUNT_KEY, "r") as f:
            key = json.load(f)
        os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
        creds = ee.ServiceAccountCredentials(email=key["client_email"], key_data=json.dumps(key))
        ee.Initialize(credentials=creds, project=PROJECT_ID)
        _ = ee.Number(1).add(1).getInfo()
        pp("GEE initialized successfully")
        return True
    except Exception as e:
        pp(f"ERROR: GEE initialization failed: {e}")
        return False

# -------------------------------------------------------
# Minimal shapefile reader + .prj CRS sniff
# -------------------------------------------------------
def guess_epsg_from_prj(prj_path):
    try:
        txt = open(prj_path, "r", encoding="utf-8", errors="ignore").read().lower()
        if "wgs_1984_utm" in txt or "wgs 1984 utm" in txt:
            if "zone_45" in txt or "zone 45" in txt:
                return 32645
        if "gcs_wgs_1984" in txt or "wgs 84" in txt:
            return 4326
    except Exception:
        pass
    return None

def read_main_polygon_from_shp(shp_path):
    with open(shp_path, "rb") as f:
        header = f.read(100)
        file_length = struct.unpack(">I", header[24:28])[0] * 2
        xmin, ymin, xmax, ymax = struct.unpack("<4d", header[36:68])

        polys = []
        f.seek(100)
        while f.tell() < file_length:
            rec_hdr = f.read(8)
            if len(rec_hdr) < 8: break
            content_len = struct.unpack(">I", rec_hdr[4:8])[0] * 2
            st = struct.unpack("<I", f.read(4))[0]
            if st in (5, 15):  # Polygon / PolygonZ
                _ = struct.unpack("<4d", f.read(32))
                num_parts   = struct.unpack("<I", f.read(4))[0]
                num_points  = struct.unpack("<I", f.read(4))[0]
                parts       = [struct.unpack("<I", f.read(4))[0] for _ in range(num_parts)]
                pts         = [struct.unpack("<2d", f.read(16)) for _ in range(num_points)]
                if st == 15:
                    remaining = content_len - (4+32+8 + num_parts*4 + num_points*16)
                    if remaining > 0: f.read(remaining)
                outer = []
                for i in range(num_parts):
                    a = parts[i]
                    b = parts[i+1] if i+1 < num_parts else num_points
                    ring = pts[a:b]
                    if len(ring) > len(outer): outer = ring
                if len(outer) >= 4:
                    if outer[0] != outer[-1]:
                        outer = outer + [outer[0]]
                    polys.append(outer)
            else:
                remaining = content_len - 4
                if remaining > 0: f.read(remaining)

        if not polys:
            raise RuntimeError("No polygon found in shapefile")
        main_ring = max(polys, key=len)
        epsg = guess_epsg_from_prj(PRJ_PATH)
        return main_ring, epsg, (xmin, ymin, xmax, ymax)

# ----------------------------------
# Anchor grid loading
# ----------------------------------
def load_anchor_spec(anchor_path):
    if not _HAVE_RASTER:
        return None
    try:
        with rasterio.open(anchor_path) as ds:
            return {
                'tf': ds.transform,
                'width': ds.width,
                'height': ds.height,
                'epsg': int(ds.crs.to_epsg()) if ds.crs and ds.crs.to_epsg() else None
            }
    except Exception as e:
        pp(f"ERROR loading anchor: {e}")
        return None

# ----------------------------------
# ERA5-Land monthly collection
# ----------------------------------
def create_monthly_collection(aoi_geom):
    pp("Creating monthly ERA5-Land collection...")
    imgs = []
    for year in YEARS:
        for month in MONSOON_MONTHS:
            start_date = ee.Date.fromYMD(year, month, 1)
            end_date   = start_date.advance(1, 'month')
            monthly = (ee.ImageCollection(DATASET)
                       .filterDate(start_date, end_date)
                       .filterBounds(aoi_geom)
                       .select(BAND)
                       .mean()
                       .rename('sm')
                       .clip(aoi_geom)  # early clip
                       .set('year', year)
                       .set('month', month)
                       .set('date', start_date.format('YYYY-MM-01')))
            imgs.append(monthly)
    ic = ee.ImageCollection(imgs)
    pp(f"Created {ic.size().getInfo()} monthly images")
    return ic

# ----------------------------------
# Monthly CSV extraction (AOI mean)
# ----------------------------------
def extract_monthly_timeseries(monthly_ic, aoi_geom):
    pp("Extracting monthly time series...")
    aoi_simple = aoi_geom.transform('EPSG:4326', 1).simplify(SIMPLIFY_TOL_DEG)

    def reduce_monthly(img):
        red = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi_simple,
            scale=REDUCE_SCALE_M,
            crs='EPSG:4326',           # explicit CRS for stable reduction
            bestEffort=True,
            maxPixels=MAX_PIXELS,
            tileScale=TILESCALE
        )
        return ee.Feature(None, {
            'year': img.get('year'),
            'month': img.get('month'),
            'date': img.get('date'),
            'sm_m3m3': red.get('sm')
        })

    fc = monthly_ic.map(reduce_monthly)
    try:
        feats = fc.getInfo()['features']
        rows = []
        for ftr in feats:
            p = ftr['properties']
            v = p.get('sm_m3m3')
            rows.append({
                'year':  p['year'],
                'month': p['month'],
                'date':  p['date'],
                'sm_m3m3': float(v) if v is not None else np.nan
            })
        rows.sort(key=lambda x: (x['year'], x['month']))
        pp(f"Extracted {len(rows)} monthly values")
        return rows
    except Exception as e:
        pp(f"ERROR in batch extraction: {e}")
        # Fallback per-month (rarely needed)
        rows = []
        imgs = monthly_ic.toList(monthly_ic.size())
        n = monthly_ic.size().getInfo()
        for i in range(n):
            try:
                img = ee.Image(imgs.get(i))
                props = img.getInfo()['properties']
                red = img.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=aoi_simple,
                    scale=REDUCE_SCALE_M,
                    crs='EPSG:4326',
                    bestEffort=True,
                    maxPixels=MAX_PIXELS,
                    tileScale=TILESCALE
                ).getInfo()
                rows.append({
                    'year':  props['year'],
                    'month': props['month'],
                    'date':  props['date'],
                    'sm_m3m3': float(red.get('sm', np.nan))
                })
                pp(f"Processed {props['year']}-{props['month']:02d}")
            except Exception as e2:
                pp(f"ERROR processing month {i}: {e2}")
        rows.sort(key=lambda x: (x['year'], x['month']))
        return rows

# ----------------------------------
# Auto-scale helper to stay under 32k px/side
# ----------------------------------
def _estimate_safe_scale_m(aoi_geom_4326, base_scale_m=EXPORT_SCALE_M_BASE):
    """Return a scale in meters such that max(width/scale, height/scale) <= PIX_MAX_SIDE."""
    gj = aoi_geom_4326.getInfo()
    # Compute bbox in degrees
    def _bbox(coords):
        xs, ys = [], []
        def walk(c):
            if isinstance(c[0], (float, int)):
                xs.append(c[0]); ys.append(c[1])
            else:
                for cc in c: walk(cc)
        walk(coords)
        return min(xs), min(ys), max(xs), max(ys)

    if gj['type'] == 'Polygon':
        xmin, ymin, xmax, ymax = _bbox(gj['coordinates'])
    elif gj['type'] == 'MultiPolygon':
        xmin, ymin, xmax, ymax = _bbox([ring for poly in gj['coordinates'] for ring in poly])
    else:
        # fallback to bounding box from EE server
        b = ee.Geometry(aoi_geom_4326).bounds().coordinates().getInfo()[0]
        xs = [p[0] for p in b]; ys = [p[1] for p in b]
        xmin, ymin, xmax, ymax = min(xs), min(ys), max(xs), max(ys)

    width_deg  = max(1e-9, xmax - xmin)
    height_deg = max(1e-9, ymax - ymin)
    lat_mid = 0.5 * (ymin + ymax)

    # crude degrees->meters conversion
    m_per_deg_lat = 110540.0
    m_per_deg_lon = 111320.0 * math.cos(math.radians(lat_mid))
    width_m  = width_deg  * abs(m_per_deg_lon)
    height_m = height_deg * m_per_deg_lat

    max_span = max(width_m, height_m)
    safe_scale = max(base_scale_m, max_span / PIX_MAX_SIDE * 1.05)  # +5% cushion
    return float(safe_scale), (width_m, height_m)

# ----------------------------------
# Multi-year raster download (using system temp directory)
# ----------------------------------
def create_multiyear_raster_optimized(monthly_ic, aoi_geom, out_tif):
    pp("Creating multi-year monsoon mean raster...")
    img = monthly_ic.mean().rename('sm').clip(aoi_geom)

    aoi4326 = aoi_geom.transform('EPSG:4326', 1).simplify(SIMPLIFY_TOL_DEG)
    safe_scale_m, (wm, hm) = _estimate_safe_scale_m(aoi4326, EXPORT_SCALE_M_BASE)
    est_cols = max(1, int(round(wm / safe_scale_m)))
    est_rows = max(1, int(round(hm / safe_scale_m)))
    pp(f"AOI size ≈ {wm/1000:.1f} km × {hm/1000:.1f} km")
    pp(f"Download scale set to {safe_scale_m/1000:.2f} km → estimated grid {est_cols} × {est_rows} px")

    params = {
        'region': aoi4326.getInfo(),
        # NOTE: omit 'crs' to use image native projection; scale is in meters
        'scale': safe_scale_m,
        'format': 'GEO_TIFF',
        'maxPixels': MAX_PIXELS
    }

    try:
        pp("Downloading raster...")
        url = img.getDownloadURL(params)
        r = requests.get(url, stream=True, timeout=CONNECT_TIMEOUT)
        if r.status_code != 200:
            raise RuntimeError(f"Download failed: HTTP {r.status_code}")

        # Use system temporary file instead of custom temp directory
        with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as temp_file:
            temp_tif = temp_file.name
            
            blob = r.content
            if blob[:2] == b'PK':
                with zipfile.ZipFile(io.BytesIO(blob)) as zf:
                    tifs = [n for n in zf.namelist() if n.lower().endswith('.tif')]
                    if not tifs:
                        raise RuntimeError("No TIFF inside ZIP")
                    with zf.open(tifs[0]) as src:
                        temp_file.write(src.read())
            else:
                temp_file.write(blob)
                
        pp(f"Downloaded to temporary file: {temp_tif}")
        return temp_tif
    except Exception as e:
        pp(f"ERROR in raster creation: {e}")
        return None

# ----------------------------------
# Reproject to anchor & mask (with cleanup)
# ----------------------------------
def reproject_and_mask(src_tif, anchor_spec, mask_path, out_tif):
    if not _HAVE_RASTER or not src_tif:
        pp("Skipping raster reprojection")
        return False
    
    try:
        pp("Reprojecting to anchor grid...")
        with rasterio.open(src_tif) as src:
            src_data = src.read(1).astype('float32')
            src_transform = src.transform
            src_crs = src.crs

        dst_transform = anchor_spec['tf']
        dst_width  = anchor_spec['width']
        dst_height = anchor_spec['height']
        dst_crs    = f"EPSG:{anchor_spec['epsg']}"

        dst_data = np.full((dst_height, dst_width), NODATA, dtype='float32')
        src_clean = np.where(np.isfinite(src_data), src_data, NODATA)

        reproject(
            source=src_clean,
            destination=dst_data,
            src_transform=src_transform, src_crs=src_crs,
            dst_transform=dst_transform, dst_crs=dst_crs,
            resampling=Resampling.bilinear,
            src_nodata=NODATA, dst_nodata=NODATA
        )

        if os.path.exists(mask_path):
            pp("Applying study mask...")
            with rasterio.open(mask_path) as ms:
                m = ms.read(1)
                if (ms.transform != dst_transform or
                    ms.width    != dst_width or
                    ms.height   != dst_height or
                    str(ms.crs) != dst_crs):
                    mask_res = np.zeros((dst_height, dst_width), dtype='uint8')
                    reproject(
                        source=m, destination=mask_res,
                        src_transform=ms.transform, src_crs=ms.crs,
                        dst_transform=dst_transform, dst_crs=dst_crs,
                        resampling=Resampling.nearest,
                        src_nodata=0, dst_nodata=0
                    )
                    m = mask_res
                dst_data = np.where(m > 0, dst_data, NODATA)

        profile = {
            'driver': 'GTiff', 'height': dst_height, 'width': dst_width, 'count': 1,
            'dtype': 'float32', 'crs': dst_crs, 'transform': dst_transform,
            'nodata': NODATA, 'compress': 'LZW', 'tiled': True
        }
        tmp = out_tif + ".part"
        with rasterio.open(tmp, 'w', **profile) as ds:
            ds.write(dst_data, 1)
        os.replace(tmp, out_tif)

        valid = dst_data != NODATA
        if valid.any():
            vals = dst_data[valid]
            pp(f"Raster stats: min={vals.min():.4f}, mean={vals.mean():.4f}, max={vals.max():.4f}")
            pp(f"Valid pixels: {int(valid.sum())}")
        pp(f"Final raster saved: {out_tif}")
        
        # Clean up temporary file
        try:
            os.unlink(src_tif)
            pp("Temporary file cleaned up")
        except Exception as cleanup_error:
            pp(f"Warning: Could not cleanup temporary file {src_tif}: {cleanup_error}")
        
        return True
    except Exception as e:
        pp(f"ERROR in reprojection: {e}")
        # Try to cleanup temporary file even if processing failed
        try:
            os.unlink(src_tif)
        except:
            pass
        return False

# ----------------
# MAIN FUNCTION
# ----------------
def main():
    pp("=== ERA5-Land Soil Moisture Processing (NO TEMP FOLDER) ===")
    pp(f"Processing {len(YEARS)} years × {len(MONSOON_MONTHS)} months")
    pp(f"Reduction scale (CSV): {REDUCE_SCALE_M/1000:.0f} km")

    if not ee_auth():
        return

    try:
        ring_coords, shp_epsg, _ = read_main_polygon_from_shp(SHP_PATH)

        if not os.path.exists(ANCHOR_TIF):
            raise FileNotFoundError(f"Anchor file not found: {ANCHOR_TIF}")
        anchor_spec = load_anchor_spec(ANCHOR_TIF)
        if not anchor_spec or not anchor_spec.get('epsg'):
            raise RuntimeError("Could not load anchor specifications")

        proj_epsg = shp_epsg or anchor_spec['epsg']
        aoi_geom = ee.Geometry.Polygon([ring_coords],
                                       proj=ee.Projection(f"EPSG:{proj_epsg}"),
                                       geodesic=False)

        # STEP 1: Monthly time series
        pp("=== STEP 1: Monthly time series ===")
        monthly_ic   = create_monthly_collection(aoi_geom)
        monthly_rows = extract_monthly_timeseries(monthly_ic, aoi_geom)
        pd.DataFrame(monthly_rows).to_csv(CSV_OUT, index=False)
        pp(f"CSV saved: {CSV_OUT} ({len(monthly_rows)} records)")

        # STEP 2: Multi-year raster 
        pp("=== STEP 2: Multi-year raster download ===")
        temp_raster = create_multiyear_raster_optimized(monthly_ic, aoi_geom, SM_TIF)

        # STEP 3: Reproject & mask 
        if temp_raster:
            pp("=== STEP 3: Reproject & mask ===")
            ok = reproject_and_mask(temp_raster, anchor_spec, MASK_TIF, SM_TIF)
            if ok:
                pp("Processing completed successfully!")
            else:
                pp("Raster processing failed (CSV created).")
        else:
            pp("Raster creation failed (CSV created).")

    except Exception as e:
        pp(f"CRITICAL ERROR: {e}")
        import traceback; traceback.print_exc()

    pp("=== DONE ===")

if __name__ == "__main__":
    main()

# Visualization: Soil Moisture (Monsoon)

In [ ]:
# =========================================================
# Step 2.3 — Soil Moisture (ERA5-Land) 
# =========================================================
import os, time, stat
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Raster deps
import rasterio
from rasterio.warp import transform_bounds

# Optional deps for shapefile boundary 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion
    HAVE_SCIPY = True
except Exception:
    pass

# ------------------------- PATHS -------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
ALT_SM     = os.path.join(BASE, "Results", "Step-2.3-SoilMoisture")
MASK_DIR   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
MASK_TIF   = os.path.join(MASK_DIR, "study_mask.tif")
STUDY_SHP  = os.path.join(BASE, "Study_Area.shp")  

OUTPUT_DIR     = os.path.join(BASE, "Results", "Step-2.3-SM")
os.makedirs(OUTPUT_DIR, exist_ok=True)
SM_TIF_INTENDED = os.path.join(OUTPUT_DIR, "soil_moisture_monsoon_2020_2023_32645_12p5m_float32.tif")
CSV_INTENDED    = os.path.join(OUTPUT_DIR, "soil_moisture_monsoon_monthly_2020_2023.csv")

SM_MAP_PNG = os.path.join(OUTPUT_DIR, "Soil Moisture Map.png")
SM_MAP_PDF = os.path.join(OUTPUT_DIR, "Soil Moisture Map.pdf")
SM_TS_PNG  = os.path.join(OUTPUT_DIR, "Soil Moisture Monsoon Timeseries.png")

# ------------------------- SETTINGS ----------------------
SM_NODATA_FALLBACK = -9999.0
MONSOON_MONTHS     = [6, 7, 8, 9]
DEFAULT_YEARS      = [2020, 2021, 2022, 2023]

# Map style constants 
FIGSIZE           = (2, 2)
CAX_SIZE          = "5%"
CAX_PAD           = 0.01
TICK_FONTSIZE     = 3
CB_LABEL_FONTSIZE = 6
CB_TICK_FONTSIZE  = 5
BOUNDARY_COLOR    = 'red'
BOUNDARY_LW       = 0.3
MASK_LW           = 0.3
ERODE_ITERS       = 2  

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path):
            attrs = os.stat(path).st_mode
            if not (attrs & stat.S_IWRITE):
                os.chmod(path, stat.S_IWRITE)
    except Exception:
        pass

def _unique_path(path):
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    for i in range(1, 999):
        cand = f"{root}_{i}{ext}"
        if not os.path.exists(cand):
            return cand
    ts = time.strftime("%Y%m%d_%H%M%S")
    return f"{root}_{ts}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs)
        print(f"[OUTPUT] Saved: {path}")
        return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs)
        print(f"[OUTPUT] Saved: {alt}")
        return alt

# =========================================================
# DMS FORMATTERS 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(decimal_deg)
    m_float = abs(decimal_deg - d) * 60
    m = int(m_float)
    s = (m_float - m) * 60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(abs(decimal_deg))
    return f"{d}°{m:02d}'{s:04.1f}\"{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(lo, hi, step_min):
        step_deg = step_min / 60.0
        start = np.floor(lo * 60.0 / step_min) * step_deg
        end   = np.ceil (hi * 60.0 / step_min) * step_deg
        vals  = np.arange(start, end + step_deg * 0.5, step_deg)
        return vals[(vals >= lo) & (vals <= hi)]
    ax.set_xticks(build(west,  east,  dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# North arrow + checker scale bar 
# =========================================================
def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N'):
    x0 = pad; y0 = 1.0 - pad - size
    x1 = x0 + size * 0.72; y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(apex[0], y1 + size*0.10, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=8, fontweight='bold', color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    t = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(t))); base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= t: return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=4):
    width_data = east - west
    height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north)/2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd

    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data; bar_h = 0.018*height_data

    pad_x = total_len_units*0.08; pad_y = 0.035*height_data
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2*pad_x, bar_h + 2*pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i*seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i % 2 == 0 else 'white'),
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Unified boundary loader/drawer 
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path):
        return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}":
                gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            import fiona
            geoms = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geoms.append(feat['geometry'])
            return {'geometries': geoms, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=BOUNDARY_COLOR, mask_lw=MASK_LW,
                        erode_iters=ERODE_ITERS):
    # Try shapefile boundary first
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt in ['LineString', 'MultiLineString']:
                        if gt == 'LineString':
                            xs, ys = zip(*geom['coordinates']); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                        else:
                            for line in geom['coordinates']:
                                xs, ys = zip(*line); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass
    # Fallback to eroded mask contour
    if mask_array is not None:
        try:
            arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float) if HAVE_SCIPY \
                  else mask_array.astype(float)
            ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                       extent=extent, origin='upper', zorder=11)
            return True
        except Exception:
            pass
    return False

# =========================================================
# Robust file finders 
# =========================================================
def find_sm_tif(primary_path, folders_to_search):
    if primary_path and os.path.exists(primary_path):
        print(f"[OK] Using SM TIF: {primary_path}"); return primary_path
    print(f"[WARN] TIF not found at intended path:\n      {primary_path}")
    for folder in folders_to_search:
        if not os.path.isdir(folder): continue
        for fn in os.listdir(folder):
            f = fn.lower()
            if f.endswith(".tif") and any(k in f for k in ["soil_moisture", "sm_monsoon", "sm"]):
                p = os.path.join(folder, fn); print(f"[OK] Using: {p}"); return p
    print("[ERR] No SM TIF found."); return None

def find_sm_csv(primary_path, folders_to_search):
    if primary_path and os.path.exists(primary_path):
        print(f"[OK] Using CSV: {primary_path}"); return primary_path
    print(f"[WARN] CSV not found at intended path:\n      {primary_path}")
    for folder in folders_to_search:
        if not os.path.isdir(folder): continue
        for fn in os.listdir(folder):
            f = fn.lower()
            if f.endswith(".csv") and any(k in f for k in ["soil", "moist", "sm"]):
                p = os.path.join(folder, fn); print(f"[OK] Using: {p}"); return p
    print("[WARN] No CSV found."); return None

# =========================================================
# 1) Soil Moisture MAP 
# =========================================================
SM_TIF = find_sm_tif(SM_TIF_INTENDED, [OUTPUT_DIR, ALT_SM])
if SM_TIF is None:
    raise FileNotFoundError("Soil moisture TIF not found.")

with rasterio.open(SM_TIF) as src:
    sm_data   = src.read(1)
    sm_crs    = src.crs
    sm_bounds = src.bounds
    sm_nodata = src.nodata if src.nodata is not None else SM_NODATA_FALLBACK

with rasterio.open(MASK_TIF) as src:
    mask = src.read(1)

sm = np.where(sm_data == sm_nodata, np.nan, sm_data.astype(np.float32))
if mask.shape == sm.shape:
    sm = np.where(mask == 1, sm, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != SM shape {sm.shape}")

valid = sm[~np.isnan(sm)]
if valid.size == 0:
    raise RuntimeError("No valid Soil Moisture pixels to plot.")

west, south, east, north = transform_bounds(sm_crs, "EPSG:4326", *sm_bounds)
extent = [west, east, south, north]

# Sequential colormap 
sm_colors = ['#f7fcf0', '#e0f3db', '#ccebc5', '#a8ddb5', '#7bccc4',
             '#4eb3d3', '#2b8cbe', '#0868ac', '#084081']
sm_cmap = mcolors.LinearSegmentedColormap.from_list('sm_seq', sm_colors, N=256)

# Robust stretch, clipped to [0,1]
vmin = max(0.0, np.nanpercentile(valid, 2))
vmax = min(1.0, np.nanpercentile(valid, 98))
if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
    vmin, vmax = max(0.0, np.nanmin(valid)), min(1.0, np.nanmax(valid))

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
im = ax.imshow(sm, cmap=sm_cmap, extent=extent, origin='upper', vmin=vmin, vmax=vmax, zorder=1)

# Boundary 
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)
_ = draw_study_boundary(ax, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=BOUNDARY_COLOR, mask_lw=MASK_LW,
                        erode_iters=ERODE_ITERS)

# North arrow + scale bar 
add_north_arrow_tri(ax, size=0.09, pad=0.07)
add_checker_scale_bar(ax, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                      segments=4, fontsize=4)

# Axes 
ax.set_xlim(west, east); ax.set_ylim(south, north)
set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, rotation=45)
ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, rotation=90)
for lbl in ax.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')


# Slim right-side colorbar
div = make_axes_locatable(ax)
cax = div.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
cb  = plt.colorbar(im, cax=cax)
cb.set_label('Soil Moisture (m³/m³)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold')
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

plt.tight_layout(pad=0.2)
safe_savefig(fig, SM_MAP_PNG, dpi=300, bbox_inches='tight', facecolor='white')
safe_savefig(fig, SM_MAP_PDF, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =========================================================
# 2) Monsoon Time Series 
# =========================================================
CSV_PATH = find_sm_csv(CSV_INTENDED, [OUTPUT_DIR, ALT_SM])
if CSV_PATH is None:
    print("[INFO] Time series skipped (CSV not found).")
else:
    df = pd.read_csv(CSV_PATH)

    # Detect columns
    lower = {c: str(c).lower() for c in df.columns}
    year_col  = next((c for c,v in lower.items() if "year"  in v), df.columns[0])
    month_col = next((c for c,v in lower.items() if "month" in v), df.columns[1] if len(df.columns)>1 else df.columns[0])
    sm_col    = next((c for c,v in lower.items() if any(k in v for k in ["sm","soil","m3m3","moist"])), df.columns[-1])

    data = df[[year_col, month_col, sm_col]].copy()
    data.columns = ["year","month","sm_m3m3"]
    data = data.dropna()
    data["year"]    = pd.to_numeric(data["year"], errors="coerce")
    data["month"]   = pd.to_numeric(data["month"], errors="coerce")
    data["sm_m3m3"] = pd.to_numeric(data["sm_m3m3"], errors="coerce")
    data = data.dropna().sort_values(["year","month"])

    monsoon = data[data["month"].isin(MONSOON_MONTHS)]
    if monsoon.empty:
        monsoon = data.copy()
    years = sorted(monsoon["year"].dropna().unique().astype(int).tolist() or DEFAULT_YEARS)

    fig2, ax2 = plt.subplots(1, 1, figsize=(6, 4))
    palette = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']
    color_map = {y: palette[i % len(palette)] for i,y in enumerate(years)}
    for y in years:
        sub = monsoon[monsoon["year"]==y]
        if not sub.empty:
            ax2.plot(sub["month"], sub["sm_m3m3"], marker='o', linewidth=1.2, markersize=4,
                     label=str(y), color=color_map[y])

    title_suffix = " (Monsoon months)" if monsoon["month"].isin(MONSOON_MONTHS).all() else ""
    ax2.set_title(f"Soil Moisture Time Series 2020–2023{title_suffix}", fontsize=12, fontweight='bold', pad=10)
    ax2.set_xlabel("Month", fontsize=9); ax2.set_ylabel("Soil Moisture (m³/m³)", fontsize=9)

    if monsoon["month"].isin(MONSOON_MONTHS).all():
        ax2.set_xticks(MONSOON_MONTHS); ax2.set_xticklabels(['Jun','Jul','Aug','Sep'], fontsize=8)
    else:
        xt = sorted(monsoon["month"].unique()); ax2.set_xticks(xt); ax2.set_xticklabels([str(int(m)) for m in xt], fontsize=8)

    ax2.tick_params(axis='y', labelsize=8)
    ymin = max(0, np.nanmin(monsoon["sm_m3m3"])*0.95) if len(monsoon)>0 else 0
    ymax = min(1, np.nanmax(monsoon["sm_m3m3"])*1.08) if len(monsoon)>0 else 1
    ax2.set_ylim(ymin, ymax)
    ax2.grid(True, linestyle='-', alpha=0.3, color='gray'); ax2.set_axisbelow(True)
    ax2.legend(title="Year", loc="upper right", fontsize=8, title_fontsize=8, framealpha=0.9)

    plt.tight_layout()
    safe_savefig(fig2, SM_TS_PNG, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# =========================================================
# Console summary
# =========================================================
print("\n" + "="*70)
print("SOIL MOISTURE (ERA5-Land) — SUMMARY")
print("="*70)
print(f"  • SM TIF:  {SM_TIF}")
print(f"  • Map PNG: {SM_MAP_PNG}")
print(f"  • Map PDF: {SM_MAP_PDF}")
print(f"  • CSV:     {CSV_PATH if CSV_PATH else 'NOT FOUND'}")
print(f"  • TS PNG:  {SM_TS_PNG if os.path.exists(SM_TS_PNG) else 'Not saved'}")
print("="*70)


# Updated Code for Soil Moisture

In [ ]:
# =========================================================
# Step 2.3 — Soil Moisture (ERA5-Land) 
# =========================================================
import os, time, stat
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.dates as mdates

# Raster deps
import rasterio
from rasterio.warp import transform_bounds

# Optional deps for shapefile boundary 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion; HAVE_SCIPY = True
except Exception:
    pass

# ------------------------- PATHS -------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
ALT_SM     = os.path.join(BASE, "Results", "Step-2.3-SoilMoisture")
MASK_DIR   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
MASK_TIF   = os.path.join(MASK_DIR, "study_mask.tif")
STUDY_SHP  = os.path.join(BASE, "Study_Area.shp")  

OUTPUT_DIR     = os.path.join(BASE, "Results", "Step-2.3-SM")
os.makedirs(OUTPUT_DIR, exist_ok=True)
SM_TIF_INTENDED = os.path.join(OUTPUT_DIR, "soil_moisture_monsoon_2020_2023_32645_12p5m_float32.tif")
CSV_INTENDED    = os.path.join(OUTPUT_DIR, "soil_moisture_monsoon_monthly_2020_2023.csv")

SM_MAP_BASE = os.path.join(OUTPUT_DIR, "Soil_Moisture_Map")  
SM_TS_PNG   = os.path.join(OUTPUT_DIR, "Soil_Moisture_Monsoon_Timeseries.png")

# ------------------------- SETTINGS ----------------------
SM_NODATA_FALLBACK = -9999.0
MONSOON_MONTHS     = [6, 7, 8, 9]
DEFAULT_YEARS      = [2020, 2021, 2022, 2023]

# PLOT STYLE SETTINGS
FIGSIZE            = (1, 1)
SAVE_DPI           = 600
CAX_SIZE           = "8%"      
CAX_PAD            = 0.002
TICK_FONTSIZE      = 1.8
LABEL_FONTSIZE     = 2.0
CB_LABEL_FONTSIZE  = 2.5
CB_TICK_FONTSIZE   = 2.2
BOUNDARY_COLOR     = 'black'
BOUNDARY_LW        = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_LW            = 0.4
LW_AXIS            = 0.3
TICK_PAD           = 0.4
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True

# Opaque white canvas
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# =========================================================
# Safe save helpers
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path): os.chmod(path, stat.S_IWRITE)
    except Exception: pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs); print(f"[OUTPUT] Saved: {path}"); return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs); print(f"[OUTPUT] Saved: {alt}"); return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# =========================================================
# DMS FORMATTERS 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg))); m_float = (abs(decimal_deg)-d)*60
    m = int(np.floor(m_float)); s = (m_float - m)*60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min/60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# North arrow & checker scale bar
# =========================================================
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size*0.72; y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size*0.09)
    ax.add_patch(Polygon([apex, left_base, tail], closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north)/2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0/mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i*seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    end_km = total_len_m/1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)

# =========================================================
# Unified boundary loader/drawer 
# =========================================================
def load_shapefile_boundary(shapefile_path, target_epsg=4326):
    if not os.path.exists(shapefile_path): return None
    if HAVE_GEOPANDAS:
        try:
            gdf = gpd.read_file(shapefile_path)
            if str(gdf.crs) != f"EPSG:{target_epsg}": gdf = gdf.to_crs(f"EPSG:{target_epsg}")
            return gdf
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geoms = []
            with fiona.open(shapefile_path, 'r') as shp:
                for feat in shp:
                    if feat['geometry'] is not None:
                        geoms.append(feat['geometry'])
            return {'geometries': geoms, 'method': 'fiona'}
        except Exception:
            pass
    return None

def draw_study_boundary(ax, extent, boundary_gdf, mask_array,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_LW,
                        erode_iters=2):
    if boundary_gdf is not None:
        try:
            if HAVE_GEOPANDAS and hasattr(boundary_gdf, 'plot'):
                boundary_gdf.boundary.plot(ax=ax, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
            elif isinstance(boundary_gdf, dict) and 'geometries' in boundary_gdf:
                for geom in boundary_gdf['geometries']:
                    gt = geom['type']
                    if gt == 'Polygon':
                        xs, ys = zip(*geom['coordinates'][0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                    elif gt == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs, ys = zip(*poly[0]); ax.plot(xs, ys, color=shp_color, linewidth=shp_lw, zorder=12)
                return True
        except Exception:
            pass
    # Mask fallback
    try:
        arr = binary_erosion(mask_array.astype(bool), iterations=erode_iters).astype(float)
    except Exception:
        arr = (mask_array.astype(float) if mask_array is not None else None)
    if arr is not None:
        ax.contour(arr, levels=[0.5], colors=mask_color, linewidths=mask_lw,
                   extent=extent, origin='upper', zorder=11)
        return True
    return False

# =========================================================
# Robust file finders 
# =========================================================
def find_sm_tif(primary_path, folders_to_search):
    if primary_path and os.path.exists(primary_path):
        print(f"[OK] Using SM TIF: {primary_path}"); return primary_path
    print(f"[WARN] TIF not found at intended path:\n      {primary_path}")
    for folder in folders_to_search:
        if not os.path.isdir(folder): continue
        for fn in os.listdir(folder):
            f = fn.lower()
            if f.endswith(".tif") and any(k in f for k in ["soil_moisture", "sm_monsoon", "sm"]):
                p = os.path.join(folder, fn); print(f"[OK] Using: {p}"); return p
    print("[ERR] No SM TIF found."); return None

def find_sm_csv(primary_path, folders_to_search):
    if primary_path and os.path.exists(primary_path):
        print(f"[OK] Using CSV: {primary_path}"); return primary_path
    print(f"[WARN] CSV not found at intended path:\n      {primary_path}")
    for folder in folders_to_search:
        if not os.path.isdir(folder): continue
        for fn in os.listdir(folder):
            f = fn.lower()
            if f.endswith(".csv") and any(k in f for k in ["soil", "moist", "sm"]):
                p = os.path.join(folder, fn); print(f"[OK] Using: {p}"); return p
    print("[WARN] No CSV found."); return None

# =========================================================
# 1) Soil Moisture MAP 
# =========================================================
SM_TIF = find_sm_tif(SM_TIF_INTENDED, [OUTPUT_DIR, ALT_SM])
if SM_TIF is None:
    raise FileNotFoundError("Soil moisture TIF not found.")

with rasterio.open(SM_TIF) as src:
    sm_data   = src.read(1)
    sm_crs    = src.crs
    sm_bounds = src.bounds
    sm_nodata = src.nodata if src.nodata is not None else SM_NODATA_FALLBACK

with rasterio.open(MASK_TIF) as src:
    mask = src.read(1)

sm = np.where(sm_data == sm_nodata, np.nan, sm_data.astype(np.float32))
if mask.shape == sm.shape:
    sm = np.where(mask == 1, sm, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != SM shape {sm.shape}")

valid = sm[np.isfinite(sm)]
if valid.size == 0:
    raise RuntimeError("No valid Soil Moisture pixels to plot.")

west, south, east, north = transform_bounds(sm_crs, "EPSG:4326", *sm_bounds)
extent = [west, east, south, north]

# Soft sequential colormap 
def build_sm_cmap(mode="ylgnbu_soft"):
    mode = (mode or "").lower()

    def soften(cmap_name, amt=0.08, n=256):
        base = plt.get_cmap(cmap_name, n)(np.linspace(0, 1, n))
        # blend slightly toward white so dark tones don't overwhelm tiny plots
        base = (1-amt)*base + amt*np.array([1,1,1,1])
        return mcolors.ListedColormap(base)

    if mode in ("ylgnbu", "ylgnbu_soft"):  return soften("YlGnBu")
    if mode in ("bugn", "bugn_soft"):      return soften("BuGn")
    if mode in ("greens", "greens_soft"):  return soften("Greens")
    if mode in ("blues", "blues_soft"):    return soften("Blues")
    if mode in ("viridis", "viridis_soft"):return soften("viridis")
    if mode in ("cividis", "cividis_soft"):return soften("cividis")
    if mode in ("magma", "magma_soft"):    return soften("magma")  # higher contrast
    if mode in ("turq", "turquoise"):
        return mcolors.LinearSegmentedColormap.from_list(
            "turq_soft", ["#f7fcfd","#d0f0f6","#a6e3e9","#5ccbd6","#2a9db3","#17647c","#083a4a"], N=256
        )
    # default
    return soften("YlGnBu")

# Use it:
cmap = build_sm_cmap("blues_soft")  # try: "viridis_soft", "magma_soft", "cividis_soft", "blues_soft"
cmap.set_bad((1,1,1,1))

# Clipped to plausible SM range [0, 1]
vmin = max(0.0, float(np.nanpercentile(valid, 2)))
vmax = min(1.0, float(np.nanpercentile(valid, 98)))
if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
    vmin, vmax = max(0.0, float(np.nanmin(valid))), min(1.0, float(np.nanmax(valid)))

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)

im = ax.imshow(sm, cmap=cmap, extent=extent, origin='upper',
               vmin=vmin, vmax=vmax, zorder=1)

# Boundary 
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)
_ = draw_study_boundary(ax, extent, boundary, mask,
                        shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW,
                        mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_LW,
                        erode_iters=2)

# Cartography
add_north_arrow_tri(ax)
add_checker_scale_bar(ax, west, east, south, north,
                      crs_type='wgs84', mid_lat_deg=(south + north)/2.0,
                      segments=4, fontsize=LABEL_FONTSIZE)

# Axes 
ax.set_xlim(west, east); ax.set_ylim(south, north)
set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE,
               width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=45)
ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE,
               width=LW_AXIS, pad=TICK_PAD, length=1.3, rotation=90)
for lbl in ax.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')

# Right inset colorbar 
cax = inset_axes(ax, width=CAX_SIZE, height="100%", loc="lower left",
                 bbox_to_anchor=(1.02, 0.0, 1.0, 1.0), bbox_transform=ax.transAxes, borderpad=0)
cb  = plt.colorbar(im, cax=cax)
cb.set_label('Soil Moisture (m³/m³)', fontsize=CB_LABEL_FONTSIZE, fontweight='bold', labelpad=0.8)
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE, width=LW_AXIS, pad=0.4, length=1.3)
if hasattr(cb, "outline") and cb.outline is not None: cb.outline.set_linewidth(0.3)

plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)
save_all(fig, SM_MAP_BASE)
plt.show()

# =========================================================
# 2) Monsoon Time Series 
# =========================================================
def parse_int(v): 
    try: return int(v)
    except: return np.nan

CSV_PATH = find_sm_csv(CSV_INTENDED, [OUTPUT_DIR, ALT_SM])
if CSV_PATH is None:
    print("[INFO] Time series skipped (CSV not found).")
else:
    df = pd.read_csv(CSV_PATH)

    # Detect columns
    lower = {c: str(c).lower() for c in df.columns}
    year_col  = next((c for c,v in lower.items() if "year"  in v), df.columns[0])
    month_col = next((c for c,v in lower.items() if "month" in v), df.columns[1] if len(df.columns)>1 else df.columns[0])
    sm_col    = next((c for c,v in lower.items() if any(k in v for k in ["sm","soil","m3m3","moist"])), df.columns[-1])

    data = df[[year_col, month_col, sm_col]].copy()
    data.columns = ["year","month","sm_m3m3"]
    data["year"]    = data["year"].map(parse_int)
    data["month"]   = data["month"].map(parse_int)
    data["sm_m3m3"] = pd.to_numeric(data["sm_m3m3"], errors="coerce")
    data = data.dropna().sort_values(["year","month"])

    monsoon = data[data["month"].isin(MONSOON_MONTHS)]
    if monsoon.empty: monsoon = data.copy()

    years = sorted(monsoon["year"].dropna().unique().astype(int).tolist() or DEFAULT_YEARS)

    fig2, ax2 = plt.subplots(1, 1, figsize=(6, 4))
    palette = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']
    color_map = {y: palette[i % len(palette)] for i,y in enumerate(years)}
    for y in years:
        sub = monsoon[monsoon["year"]==y]
        if not sub.empty:
            ax2.plot(sub["month"], sub["sm_m3m3"], marker='o', linewidth=1.2, markersize=4,
                     label=str(y), color=color_map[y])

    title_suffix = " (Monsoon months)" if monsoon["month"].isin(MONSOON_MONTHS).all() else ""
    ax2.set_title(f"Soil Moisture Time Series 2020–2023{title_suffix}", fontsize=12, fontweight='bold', pad=10)
    ax2.set_xlabel("Month", fontsize=9); ax2.set_ylabel("Soil Moisture (m³/m³)", fontsize=9)

    if monsoon["month"].isin(MONSOON_MONTHS).all():
        ax2.set_xticks(MONSOON_MONTHS); ax2.set_xticklabels(['Jun','Jul','Aug','Sep'], fontsize=8)
    else:
        xt = sorted(monsoon["month"].unique()); ax2.set_xticks(xt); ax2.set_xticklabels([str(int(m)) for m in xt], fontsize=8)

    ax2.tick_params(axis='y', labelsize=8)
    ymin = max(0, np.nanmin(monsoon["sm_m3m3"])*0.95) if len(monsoon)>0 else 0
    ymax = min(1, np.nanmax(monsoon["sm_m3m3"])*1.08) if len(monsoon)>0 else 1
    ax2.set_ylim(ymin, ymax)
    ax2.grid(True, linestyle='-', alpha=0.3, color='gray'); ax2.set_axisbelow(True)
    ax2.legend(title="Year", loc="upper right", fontsize=8, title_fontsize=8, framealpha=0.9)

    plt.tight_layout()
    safe_savefig(fig2, SM_TS_PNG, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# =========================================================
# Console summary
# =========================================================
print("\n" + "="*70)
print("SOIL MOISTURE— SUMMARY")
print("="*70)
print(f"  • SM TIF:   {SM_TIF}")
print(f"  • Map base: {SM_MAP_BASE}.(png/pdf/svg/eps)")
print(f"  • CSV:      {CSV_PATH if CSV_PATH else 'NOT FOUND'}")
print(f"  • TS PNG:   {SM_TS_PNG if os.path.exists(SM_TS_PNG) else 'Not saved'}")
print(f"  • Display stretch: {vmin:.3f} → {vmax:.3f} m³/m³")
print("="*70)


# Step-2.4: Land Cover

In [ ]:
# =========================================================
# Step-2.4: Land Cover Extraction 
# =========================================================
import os, json, io
import ee
import numpy as np
import pandas as pd
from datetime import datetime
import struct
import requests
import tempfile
import zipfile

try:
    import rasterio
    from rasterio.warp import reproject, Resampling
    _HAVE_RASTER = True
except Exception:
    _HAVE_RASTER = False

# ---------------------PATH----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
SHP_PATH  = os.path.join(BASE, "Study_Area.shp")
PRJ_PATH  = os.path.splitext(SHP_PATH)[0] + ".prj"
ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
MASK_TIF   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

OUT_DIR = os.path.join(BASE, "Results", "Step-2.4-LandCover")
os.makedirs(OUT_DIR, exist_ok=True)

# KEY OUTPUTS
LC_RASTER_NATIVE = os.path.join(OUT_DIR, "ESA_WorldCover_Native_10m.tif")
LC_RASTER_FINAL = os.path.join(OUT_DIR, "ESA_WorldCover_Final_Reprojected.tif") 
LC_STATISTICS_CSV = os.path.join(OUT_DIR, "ESA_WorldCover_Statistics.csv")

# GEE auth
SERVICE_ACCOUNT_KEY = os.environ.get("EE_SERVICE_ACCOUNT_KEY", "")
PROJECT_ID = os.environ.get("EE_PROJECT", "")

# CRITICAL: Use NATIVE resolution - NO AGGREGATION!
NATIVE_SCALE = 10      # ESA WorldCover native resolution
MAX_PIXELS = 1e12
NODATA = -9999.0

ESA_LEGEND = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare/sparse vegetation", 70: "Snow and ice",
    80: "Permanent water bodies", 90: "Herbaceous wetland", 95: "Mangroves", 100: "Moss and lichen"
}

def pp(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

def ee_auth():
    try:
        with open(SERVICE_ACCOUNT_KEY, "r") as f:
            key = json.load(f)
        os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
        creds = ee.ServiceAccountCredentials(email=key["client_email"], key_data=json.dumps(key))
        ee.Initialize(credentials=creds, project=PROJECT_ID)
        _ = ee.Number(1).add(1).getInfo()
        pp("GEE initialized successfully")
        return True
    except Exception as e:
        pp(f"ERROR: GEE initialization failed: {e}")
        return False

def read_main_polygon_from_shp(shp_path):
    # Simplified version - same as before
    with open(shp_path, "rb") as f:
        header = f.read(100)
        file_length = struct.unpack(">I", header[24:28])[0] * 2
        polys = []
        f.seek(100)
        while f.tell() < file_length:
            rec_hdr = f.read(8)
            if len(rec_hdr) < 8: break
            content_len = struct.unpack(">I", rec_hdr[4:8])[0] * 2
            st = struct.unpack("<I", f.read(4))[0]
            if st in (5, 15):
                _ = struct.unpack("<4d", f.read(32))
                num_parts = struct.unpack("<I", f.read(4))[0]
                num_points = struct.unpack("<I", f.read(4))[0]
                parts = [struct.unpack("<I", f.read(4))[0] for _ in range(num_parts)]
                pts = [struct.unpack("<2d", f.read(16)) for _ in range(num_points)]
                if st == 15:
                    remaining = content_len - (4+32+8 + num_parts*4 + num_points*16)
                    if remaining > 0: f.read(remaining)
                outer = []
                for i in range(num_parts):
                    a = parts[i]; b = parts[i+1] if i+1 < num_parts else num_points
                    ring = pts[a:b]
                    if len(ring) > len(outer): outer = ring
                if len(outer) >= 4:
                    if outer[0] != outer[-1]: outer = outer + [outer[0]]
                    polys.append(outer)
            else:
                remaining = content_len - 4
                if remaining > 0: f.read(remaining)
        
        if not polys: raise RuntimeError("No polygon found")
        main_ring = max(polys, key=len)
        
        # Get EPSG
        epsg = None
        try:
            txt = open(os.path.splitext(shp_path)[0] + ".prj", "r").read().lower()
            if "wgs_1984_utm" in txt and "zone_45" in txt: epsg = 32645
            elif "wgs 84" in txt: epsg = 4326
        except: pass
        
        return main_ring, epsg

def get_esa_image(aoi_geom, year=2021):
    """Get ESA WorldCover image - simple and direct"""
    pp(f"Getting ESA WorldCover for {year}")
    
    dataset = "ESA/WorldCover/v200" if year == 2021 else "ESA/WorldCover/v100"
    
    collection = ee.ImageCollection(dataset).filterBounds(aoi_geom)
    if collection.size().getInfo() == 0:
        raise RuntimeError(f"No ESA WorldCover data found for {year}")
    
    img = collection.first().select("Map").clip(aoi_geom)
    pp(f"ESA image loaded for {year}")
    return img

def analyze_native_classes(img, aoi_geom):
    """Analyze classes at native resolution using histogram"""
    pp("Analyzing land cover classes at native 10m resolution...")
    
    try:
        histogram = img.reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(),
            geometry=aoi_geom.transform('EPSG:4326', 1),
            scale=NATIVE_SCALE,
            bestEffort=True,
            maxPixels=1e9,
            tileScale=1
        )
        
        hist_data = histogram.getInfo()
        
        if "Map" in hist_data and hist_data["Map"]:
            classes = hist_data["Map"]
            total_pixels = sum([float(count) for count in classes.values()])
            
            results = []
            pp(f"Analysis complete - {int(total_pixels):,} pixels analyzed")
            pp("Class breakdown:")
            
            for class_code, pixel_count in sorted(classes.items(), key=lambda x: float(x[1]), reverse=True):
                class_int = int(float(class_code))
                pixel_count_int = int(float(pixel_count))
                percentage = (float(pixel_count) / total_pixels) * 100
                area_ha = (pixel_count_int * NATIVE_SCALE * NATIVE_SCALE) / 10000
                class_name = ESA_LEGEND.get(class_int, f"Unknown_{class_int}")
                
                pp(f"  {class_int:3d}: {pixel_count_int:8,d} pixels ({percentage:5.1f}%) {area_ha:7.5f} ha - {class_name}")
                
                results.append({
                    'class_code': class_int,
                    'class_name': class_name,
                    'pixel_count': pixel_count_int,
                    'percentage': percentage,
                    'area_hectares': area_ha
                })
            
            # Save statistics
            df = pd.DataFrame(results)
            df.to_csv(LC_STATISTICS_CSV, index=False)
            pp(f"Statistics saved: {LC_STATISTICS_CSV}")
            
            return len(results)
            
        else:
            pp("WARNING: No histogram data returned")
            return 0
            
    except Exception as e:
        pp(f"ERROR in class analysis: {e}")
        return 0

def download_native_esa_raster(img, aoi_geom):
    """Download ESA raster at native resolution"""
    pp("Downloading ESA WorldCover at native 10m resolution...")
    
    try:
        aoi4326 = aoi_geom.transform('EPSG:4326', 1)
        
        # Estimate download size
        bounds = aoi_geom.bounds().getInfo()
        coords = bounds['coordinates'][0]
        width_deg = max([pt[0] for pt in coords]) - min([pt[0] for pt in coords])
        height_deg = max([pt[1] for pt in coords]) - min([pt[1] for pt in coords])
        
        # Rough estimate: 1 degree ≈ 111km ≈ 11,100 pixels at 10m
        est_pixels = int(width_deg * height_deg * 11100 * 11100)
        pp(f"Estimated download size: {est_pixels:,} pixels")
        
        # Adjust scale if area is too large
        scale = NATIVE_SCALE
        if est_pixels > 50_000_000:  # 50 million pixels
            scale = 20
            pp(f"Large area detected - using {scale}m resolution")
        
        params = {
            'region': aoi4326.getInfo(),
            'scale': scale,
            'format': 'GEO_TIFF',
            'maxPixels': MAX_PIXELS
        }
        
        pp("Requesting download...")
        url = img.getDownloadURL(params)
        
        pp("Downloading...")
        r = requests.get(url, stream=True, timeout=(30, 600))
        if r.status_code != 200:
            raise RuntimeError(f"Download failed: HTTP {r.status_code}")
        
        # Save data
        with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as temp_file:
            temp_tif = temp_file.name
            
            blob = r.content
            if blob[:2] == b'PK':
                with zipfile.ZipFile(io.BytesIO(blob)) as zf:
                    tif_files = [n for n in zf.namelist() if n.lower().endswith('.tif')]
                    if not tif_files: raise RuntimeError("No TIFF in ZIP")
                    with zf.open(tif_files[0]) as src:
                        temp_file.write(src.read())
            else:
                temp_file.write(blob)
        
        # Copy to output location
        import shutil
        shutil.copy2(temp_tif, LC_RASTER_NATIVE)
        pp(f"Native raster saved: {LC_RASTER_NATIVE}")
        
        # Analyze downloaded raster
        with rasterio.open(LC_RASTER_NATIVE) as src:
            data = src.read(1)
            unique_vals = np.unique(data[data != src.nodata])
            pp(f"Downloaded raster: {src.width}x{src.height}")
            pp(f"Classes in raster: {unique_vals}")
            
            # Quick stats
            for class_val in sorted(unique_vals):
                if class_val != src.nodata and class_val > 0:
                    count = np.sum(data == class_val)
                    percent = (count / data.size) * 100
                    class_name = ESA_LEGEND.get(int(class_val), f'Class_{int(class_val)}')
                    pp(f"  {int(class_val)}: {count:,} pixels ({percent:.1f}%) - {class_name}")
        
        # Clean up
        os.unlink(temp_tif)
        return True
        
    except Exception as e:
        pp(f"ERROR in raster download: {e}")
        import traceback
        traceback.print_exc()
        return False

def reproject_to_anchor(native_tif, anchor_tif, mask_tif, output_tif):
    """Reproject to match anchor grid"""
    if not _HAVE_RASTER:
        pp("Rasterio not available - skipping reprojection")
        return False
    
    try:
        pp("Reprojecting to match anchor grid...")
        
        # Load anchor specs
        with rasterio.open(anchor_tif) as anchor_src:
            anchor_profile = {
                'transform': anchor_src.transform,
                'width': anchor_src.width,
                'height': anchor_src.height,
                'crs': anchor_src.crs
            }
        
        # Load mask if exists
        mask_data = None
        if os.path.exists(mask_tif):
            with rasterio.open(mask_tif) as mask_src:
                mask_data = mask_src.read(1)
                pp(f"Loaded mask: {mask_src.width}x{mask_src.height}")
        
        # Reproject
        with rasterio.open(native_tif) as src:
            pp(f"Source: {src.width}x{src.height} {src.crs}")
            pp(f"Target: {anchor_profile['width']}x{anchor_profile['height']} {anchor_profile['crs']}")
            
            # Create output array
            output_data = np.full((anchor_profile['height'], anchor_profile['width']), 
                                NODATA, dtype=np.float32)
            
            # Reproject
            reproject(
                source=rasterio.band(src, 1),
                destination=output_data,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=anchor_profile['transform'],
                dst_crs=anchor_profile['crs'],
                resampling=Resampling.nearest,  # Nearest neighbor for categorical data
                dst_nodata=NODATA
            )
            
            # Apply mask
            if mask_data is not None and mask_data.shape == output_data.shape:
                output_data[mask_data == 0] = NODATA
                pp("Applied study area mask")
            
            # Save
            profile = {
                'driver': 'GTiff',
                'height': anchor_profile['height'],
                'width': anchor_profile['width'],
                'count': 1,
                'dtype': rasterio.float32,
                'crs': anchor_profile['crs'],
                'transform': anchor_profile['transform'],
                'nodata': NODATA,
                'compress': 'lzw'
            }
            
            with rasterio.open(output_tif, 'w', **profile) as dst:
                dst.write(output_data, 1)
            
            pp(f"Reprojected raster saved: {output_tif}")
            
            # Final stats
            unique_final = np.unique(output_data[output_data != NODATA])
            pp(f"Final classes: {unique_final}")
            
        return True
        
    except Exception as e:
        pp(f"ERROR in reprojection: {e}")
        return False

def main():
    """Main processing function - SIMPLE AND DIRECT"""
    pp("=== SIMPLE ESA WORLDCOVER EXTRACTION ===")
    
    if not ee_auth():
        return
    
    try:
        # Load study area
        ring_coords, shp_epsg = read_main_polygon_from_shp(SHP_PATH)
        proj_epsg = shp_epsg or 4326
        aoi_geom = ee.Geometry.Polygon([ring_coords], proj=ee.Projection(f"EPSG:{proj_epsg}"))
        pp(f"Study area loaded (EPSG:{proj_epsg})")
        
        # Get ESA WorldCover image
        esa_img = get_esa_image(aoi_geom, year=2021)
        
        # Analyze classes at native resolution
        num_classes = analyze_native_classes(esa_img, aoi_geom)
        pp(f"Classes detected: {num_classes}")
        
        if num_classes == 1:
            pp("  WARNING: Only 1 class detected!")
            pp("This could mean:")
            pp("  1. Your study area is genuinely homogeneous")
            pp("  2. The area is very small")
            pp("  3. Seasonal/temporal variation exists - try different dates")
        else:
            pp(f" SUCCESS: {num_classes} classes detected - good diversity!")
        
        # Download native resolution raster
        download_success = download_native_esa_raster(esa_img, aoi_geom)
        
        if not download_success:
            pp(" Download failed")
            return
        
        # Reproject to match anchor grid (optional)
        if os.path.exists(ANCHOR_TIF):
            pp("Reprojecting to match anchor grid...")
            reproject_success = reproject_to_anchor(LC_RASTER_NATIVE, ANCHOR_TIF, MASK_TIF, LC_RASTER_FINAL)
            if reproject_success:
                pp(f" Final output ready: {LC_RASTER_FINAL}")
            else:
                pp(f"  Reprojection failed, use native file: {LC_RASTER_NATIVE}")
        else:
            pp("No anchor file - using native projection")
            pp(f" Output ready: {LC_RASTER_NATIVE}")
        
        pp("\n=== RESULTS SUMMARY ===")
        pp(f"Classes detected: {num_classes}")
        pp(f"Statistics CSV: {LC_STATISTICS_CSV}")
        pp(f"Native raster: {LC_RASTER_NATIVE}")
        if os.path.exists(LC_RASTER_FINAL):
            pp(f"Reprojected raster: {LC_RASTER_FINAL}")
        
        if num_classes > 1:
            pp("  SUCCESS: Multiple land cover classes detected!")
            pp("   Your data should now show proper diversity.")
        else:
            pp("  Your study area may genuinely have low diversity.")
            pp("   Consider expanding the area or checking seasonal variations.")
            
    except Exception as e:
        pp(f"CRITICAL ERROR: {e}")
        import traceback
        traceback.print_exc()
    
    pp("=== PROCESSING COMPLETE ===")

# Main Execution
if __name__ == "__main__":
    main()

# Visualization of Land Cover

In [ ]:
# =========================================================
# Step 2.4 — Land Cover Visualization 
# =========================================================
import os, time, stat
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Polygon, Rectangle, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.dates as mdates

# Raster deps
import rasterio
from rasterio.warp import transform_bounds

# Optional deps for shapefile boundary 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion; HAVE_SCIPY = True
except Exception:
    pass

# ------------------------- PATHS -------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
ALT_LC     = os.path.join(BASE, "Results", "Step-2.4-LandCover")
MASK_DIR   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
MASK_TIF   = os.path.join(MASK_DIR, "study_mask.tif")
STUDY_SHP  = os.path.join(BASE, "Study_Area.shp")  

OUTPUT_DIR     = os.path.join(BASE, "Results", "Step-2.4-LandCover")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Updated paths based on successful diagnostic results
LC_TIF_INTENDED = os.path.join(OUTPUT_DIR, "ESA_WorldCover_Final_Reprojected.tif")
CSV_INTENDED    = os.path.join(OUTPUT_DIR, "landcover_detailed_diagnosis.csv")

LC_MAP_BASE = os.path.join(OUTPUT_DIR, "LandCover_Map")  
LC_TS_PNG   = os.path.join(OUTPUT_DIR, "LandCover_Stats.png")

# ------------------------- SETTINGS ----------------------
LC_NODATA_FALLBACK = -9999.0
DEFAULT_YEARS      = [2020, 2021]

# ESA WorldCover Legend
ESA_LEGEND = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare/sparse vegetation", 70: "Snow and ice",
    80: "Permanent water bodies", 90: "Herbaceous wetland", 95: "Mangroves", 100: "Moss and lichen"
}

# PLOT STYLE SETTINGS 
FIGSIZE            = (2, 2)           # Same tiny size as SM
SAVE_DPI           = 600
CAX_SIZE           = "8%"      
CAX_PAD            = 0.002
TICK_FONTSIZE      = 3.0
LABEL_FONTSIZE     = 3.0
CB_LABEL_FONTSIZE  = 1.8
CB_TICK_FONTSIZE   = 1.0
BOUNDARY_COLOR     = 'black'
BOUNDARY_LW        = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_LW            = 0.5
LW_AXIS            = 0.5
TICK_PAD           = 0.6
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True

# Opaque white canvas
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# =========================================================
# Safe save helpers - EXACT SAME AS SM CODE
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path): os.chmod(path, stat.S_IWRITE)
    except Exception: pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs); print(f"[OUTPUT] Saved: {path}"); return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs); print(f"[OUTPUT] Saved: {alt}"); return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# =========================================================
# DMS FORMATTERS 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg))); m_float = (abs(decimal_deg)-d)*60
    m = int(np.floor(m_float)); s = (m_float - m)*60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min/60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# North arrow & checker scale bar 
# =========================================================
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size*0.72; y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size*0.09)
    ax.add_patch(Polygon([apex, left_base, tail], closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north)/2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0/mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i*seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    end_km = total_len_m/1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)

# =========================================================
# ULTRA-COMPACT LEGEND WITH ABBREVIATIONS
# =========================================================
def add_inset_legend_box_compact(ax, unique_classes, esa_colors, west, east, south, north):
    """Ultra-compact legend with abbreviations for small maps"""
    
    # Abbreviations for common classes
    ABBREVIATIONS = {
        10: "Tree", 20: "Shrub", 30: "Grass", 40: "Crop",
        50: "Built", 60: "Bare", 70: "Snow", 80: "Water",
        90: "Wetland", 95: "Mangrv", 100: "Moss"
    }
    
    # Adjust the position - Move it down to avoid touching the plot
    legend_x = 0.15 # 15% from left (bottom-right corner)
    legend_y = 0.80  # Moved down slightly to avoid overlap with the plot
    
    num_classes = len(unique_classes)
    
    # Use two columns if more than 5 classes
    if num_classes <= 5:
        ncol = 1
        legend_width = 0.70  # Increased width for better space
        rows = num_classes
        legend_height = 0.06 + rows * 0.030  # Adjusted height
    else:
        ncol = 2
        rows = (num_classes + 1) // 2  # Ceil division
        legend_width = 0.50  # Slightly wider for two columns
        legend_height = 0.06 + rows * 0.030
    
    # Create compact box
    legend_box = FancyBboxPatch(
        (legend_x, legend_y), legend_width, legend_height,
        transform=ax.transAxes,
        boxstyle="round,pad=0.01,rounding_size=0.01",
        facecolor='white',
        edgecolor='#222222',
        linewidth=0.25,
        alpha=0.95,
        zorder=10
    )
    ax.add_patch(legend_box)
    
    # Add title
    ax.text(legend_x + legend_width/2, legend_y + legend_height - 0.015,
            'LC Classes', transform=ax.transAxes, ha='center', va='top',
            fontsize=CB_LABEL_FONTSIZE + 1, fontweight='bold', color='#222222', zorder=11)
    
    # Single column layout
    if ncol == 1:
        item_height = (legend_height - 0.04) / num_classes
        for i, class_val in enumerate(sorted(unique_classes)):
            y_pos = legend_y + legend_height - 0.035 - (i + 0.5) * item_height
            
            # Color box
            color_box = Rectangle(
                (legend_x + 0.02, y_pos - item_height/5), 
                0.016, item_height/2.5,
                transform=ax.transAxes,
                facecolor=esa_colors.get(class_val, '#cccccc'),
                edgecolor='black',
                linewidth=0.15,
                zorder=11
            )
            ax.add_patch(color_box)
            
            # Label with abbreviation
            abbr = ABBREVIATIONS.get(class_val, f"C{class_val}")
            ax.text(legend_x + 0.045, y_pos,
                    f"{abbr}",
                    transform=ax.transAxes, ha='left', va='center',
                    fontsize=LABEL_FONTSIZE + 0.5, zorder=11)
    
    # Two column layout for many classes
    else:
        item_height = (legend_height - 0.04) / rows
        for i, class_val in enumerate(sorted(unique_classes)):
            row = i // 2
            col = i % 2
            x_offset = legend_x + 0.025 + col * 0.20  # Increased width for better spacing
            y_pos = legend_y + legend_height - 0.045 - (row + 0.5) * item_height
            
            # Color box
            color_box = Rectangle(
                (x_offset, y_pos - item_height/5), 
                0.014, item_height/2.5,
                transform=ax.transAxes,
                facecolor=esa_colors.get(class_val, '#cccccc'),
                edgecolor='black',
                linewidth=0.15,
                zorder=11
            )
            ax.add_patch(color_box)
            
            # Label with abbreviation
            abbr = ABBREVIATIONS.get(class_val, f"C{class_val}")
            ax.text(x_offset + 0.025, y_pos, f"{abbr}", transform=ax.transAxes, ha='left', va='center',
                    fontsize=LABEL_FONTSIZE + 1.0, zorder=11)

def find_lc_csv(primary_path, folders_to_search):
    if primary_path and os.path.exists(primary_path):
        print(f"[OK] Using CSV: {primary_path}"); return primary_path
    print(f"[WARN] CSV not found at intended path:\n      {primary_path}")
    for folder in folders_to_search:
        if not os.path.isdir(folder): continue
        for fn in os.listdir(folder):
            f = fn.lower()
            if f.endswith(".csv") and any(k in f for k in ["landcover", "land", "diagnosis", "lc"]):
                p = os.path.join(folder, fn); print(f"[OK] Using: {p}"); return p
    print("[WARN] No CSV found."); return None


# =========================================================
# 1) Land Cover MAP 
# =========================================================
LC_TIF = find_lc_tif(LC_TIF_INTENDED, [OUTPUT_DIR, ALT_LC])
if LC_TIF is None:
    raise FileNotFoundError("Land Cover TIF not found.")

with rasterio.open(LC_TIF) as src:
    lc_data   = src.read(1)
    lc_crs    = src.crs
    lc_bounds = src.bounds
    lc_nodata = src.nodata if src.nodata is not None else LC_NODATA_FALLBACK

with rasterio.open(MASK_TIF) as src:
    mask = src.read(1)

lc = np.where(lc_data == lc_nodata, np.nan, lc_data.astype(np.float32))
if mask.shape == lc.shape:
    lc = np.where(mask == 1, lc, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != LC shape {lc.shape}")

valid = lc[np.isfinite(lc)]
if valid.size == 0:
    raise RuntimeError("No valid Land Cover pixels to plot.")

# Get unique classes and create colormap
unique_classes = np.unique(valid).astype(int)
print(f"[INFO] Classes found: {unique_classes}")

west, south, east, north = transform_bounds(lc_crs, "EPSG:4326", *lc_bounds)
extent = [west, east, south, north]

# Create discrete colormap for land cover classes
cmap, class_labels, esa_colors = build_lc_colormap(unique_classes)

# Map class values to colormap indices
lc_indexed = np.full_like(lc, np.nan)
for i, class_val in enumerate(sorted(unique_classes)):
    lc_indexed[lc == class_val] = i

# Use discrete bounds for categorical data
vmin = 0
vmax = len(unique_classes) - 1

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)

# Plot land cover using indexed values
im = ax.imshow(lc_indexed, cmap=cmap, extent=extent, origin='upper', vmin=vmin, vmax=vmax, zorder=1)

# Boundary 
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)
_ = draw_study_boundary(ax, extent, boundary, mask, shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW, mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_LW,
                        erode_iters=2)

# Cartography - keep original positions
add_north_arrow_tri(ax)
add_checker_scale_bar(ax, west, east, south, north, crs_type='wgs84', mid_lat_deg=(south + north)/2.0, segments=4, fontsize=LABEL_FONTSIZE)

# LEGEND with abbreviations 
add_inset_legend_box_compact(ax, unique_classes, esa_colors, west, east, south, north)

# Axes 
ax.set_xlim(west, east); ax.set_ylim(south, north)
set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.5, rotation=45, direction='out')
ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.5, rotation=90, direction='out')
for lbl in ax.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
plt.subplots_adjust(left=0.18, right=0.88, bottom=0.18, top=0.98)
save_all(fig, LC_MAP_BASE)
plt.show()

# =========================================================
# 2) Land Cover Statistics Chart
# =========================================================
CSV_PATH = find_lc_csv(CSV_INTENDED, [OUTPUT_DIR, ALT_LC])
if CSV_PATH is None:
    print("[INFO] Statistics chart skipped (CSV not found).")
else:
    df = pd.read_csv(CSV_PATH)

    # Calculate statistics from raster directly
    raster_stats = []
    total_pixels = len(valid)
    
    for class_val in sorted(unique_classes):
        count = np.sum(lc == class_val)
        percentage = (count / total_pixels) * 100
        area_ha = count * 100 / 10000  # 10m pixels to hectares
        
        raster_stats.append({
            'class_code': int(class_val),
            'class_name': ESA_LEGEND.get(int(class_val), f'Class_{int(class_val)}'),
            'percentage': percentage,
            'area_ha': area_ha
        })
    
    stats_df = pd.DataFrame(raster_stats).sort_values('percentage', ascending=False)

    # Create statistics chart 
    fig2, ax2 = plt.subplots(1, 1, figsize=(6, 4))
    
    # Color bars by class
    colors = [esa_colors.get(int(row['class_code']), '#gray') for _, row in stats_df.iterrows()]
    
    bars = ax2.bar(range(len(stats_df)), stats_df['percentage'], color=colors)
    
    ax2.set_title("Land Cover Composition", fontsize=12, fontweight='bold', pad=10)
    ax2.set_xlabel("Land Cover Classes", fontsize=9)
    ax2.set_ylabel("Percentage (%)", fontsize=9)
    
    # Set x-axis labels
    ax2.set_xticks(range(len(stats_df)))
    ax2.set_xticklabels([f"{int(row['class_code'])}" for _, row in stats_df.iterrows()], 
                        fontsize=8, rotation=45)
    
    ax2.tick_params(axis='y', labelsize=8)
    ax2.set_ylim(0, stats_df['percentage'].max() * 1.1)
    ax2.grid(True, linestyle='-', alpha=0.3, color='gray')
    ax2.set_axisbelow(True)
    
    # Add percentage labels on bars
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if height > 1:  # Only label bars > 1%
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    safe_savefig(fig2, LC_TS_PNG, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# =========================================================
# Console summary 
# =========================================================
print("\n" + "="*70)
print("LAND COVER — SUMMARY ")
print("="*70)
print(f"  • LC TIF:   {LC_TIF}")
print(f"  • Map base: {LC_MAP_BASE}.(png/pdf/svg)")
print(f"  • CSV:      {CSV_PATH if CSV_PATH else 'NOT FOUND'}")
print(f"  • Stats PNG: {LC_TS_PNG if os.path.exists(LC_TS_PNG) else 'Not saved'}")
print(f"  • Classes found: {len(unique_classes)} ({unique_classes})")
print(f"  • Legend: Ultra-compact with abbreviations, bottom-right position")
print("="*70)


In [ ]:
# =========================================================
# Step 2.4 — Land Cover Visualization 
# =========================================================
import os, time, stat
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Polygon, Rectangle, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.dates as mdates

# Raster deps
import rasterio
from rasterio.warp import transform_bounds

# Optional deps for shapefile boundary 
HAVE_GEOPANDAS = False
HAVE_FIONA = False
HAVE_SCIPY = False
try:
    import geopandas as gpd; HAVE_GEOPANDAS = True
except Exception:
    pass
try:
    import fiona; HAVE_FIONA = True
except Exception:
    pass
try:
    from scipy.ndimage import binary_erosion; HAVE_SCIPY = True
except Exception:
    pass

# ------------------------- PATHS -------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
ALT_LC     = os.path.join(BASE, "Results", "Step-2.4-LandCover")
MASK_DIR   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask")
MASK_TIF   = os.path.join(MASK_DIR, "study_mask.tif")
STUDY_SHP  = os.path.join(BASE, "Study_Area.shp")  

OUTPUT_DIR     = os.path.join(BASE, "Results", "Step-2.4-LandCover")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Updated paths based on successful diagnostic results
LC_TIF_INTENDED = os.path.join(OUTPUT_DIR, "ESA_WorldCover_Final_Reprojected.tif")
CSV_INTENDED    = os.path.join(OUTPUT_DIR, "landcover_detailed_diagnosis.csv")

LC_MAP_BASE = os.path.join(OUTPUT_DIR, "LandCover_Map")  
LC_TS_PNG   = os.path.join(OUTPUT_DIR, "LandCover_Stats.png")

# ------------------------- SETTINGS ----------------------
LC_NODATA_FALLBACK = -9999.0
DEFAULT_YEARS      = [2020, 2021]

# ESA WorldCover Legend
ESA_LEGEND = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare/sparse vegetation", 70: "Snow and ice",
    80: "Permanent water bodies", 90: "Herbaceous wetland", 95: "Mangroves", 100: "Moss and lichen"
}

# PLOT STYLE SETTINGS 
FIGSIZE            = (2, 2)           # Same tiny size as SM
SAVE_DPI           = 600
CAX_SIZE           = "8%"      
CAX_PAD            = 0.002
TICK_FONTSIZE      = 3.0
LABEL_FONTSIZE     = 3.0
CB_LABEL_FONTSIZE  = 1.8
CB_TICK_FONTSIZE   = 1.0
BOUNDARY_COLOR     = 'black'
BOUNDARY_LW        = 0.4
MASK_CONTOUR_COLOR = 'black'
MASK_LW            = 0.5
LW_AXIS            = 0.5
TICK_PAD           = 0.6
XROT_DEG           = 35

SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True

# Opaque white canvas
mpl.rcParams['savefig.transparent'] = False
mpl.rcParams['figure.facecolor']    = 'white'
mpl.rcParams['axes.facecolor']      = 'white'

# =========================================================
# Safe save helpers - EXACT SAME AS SM CODE
# =========================================================
def _ensure_writable(path):
    try:
        if os.path.exists(path): os.chmod(path, stat.S_IWRITE)
    except Exception: pass

def _unique_path(path):
    if not os.path.exists(path): return path
    root, ext = os.path.splitext(path)
    return f"{root}_{time.strftime('%Y%m%d_%H%M%S')}{ext}"

def safe_savefig(fig, path, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    _ensure_writable(path)
    try:
        fig.savefig(path, **kwargs); print(f"[OUTPUT] Saved: {path}"); return path
    except PermissionError:
        alt = _unique_path(path)
        print(f"[WARN] Permission denied for '{path}'. Saving as '{alt}'.")
        fig.savefig(alt, **kwargs); print(f"[OUTPUT] Saved: {alt}"); return alt

def scrub_alpha_for_eps(fig):
    from matplotlib.image import AxesImage
    for ax in fig.axes:
        for im in getattr(ax, "images", []):
            if isinstance(im, AxesImage):
                try: im.set_alpha(1.0)
                except Exception: pass

def save_all(fig, base_no_ext):
    kw = dict(dpi=SAVE_DPI, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
    safe_savefig(fig, base_no_ext + ".png", **kw)
    if SAVE_PDF: safe_savefig(fig, base_no_ext + ".pdf", **kw)
    if SAVE_SVG: safe_savefig(fig, base_no_ext + ".svg", **{**kw, "format": "svg"})
    if SAVE_EPS:
        scrub_alpha_for_eps(fig)
        safe_savefig(fig, base_no_ext + ".eps", **{**kw, "format": "eps"})

# =========================================================
# DMS FORMATTERS 
# =========================================================
def _decimal_to_dms(decimal_deg):
    d = int(np.floor(abs(decimal_deg))); m_float = (abs(decimal_deg)-d)*60
    m = int(np.floor(m_float)); s = (m_float - m)*60
    return d, m, s

def format_dms_lat(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'N' if decimal_deg >= 0 else 'S'}"

def format_dms_lon(decimal_deg, pos=None):
    d, m, s = _decimal_to_dms(decimal_deg)
    return f"{d}°{m:02d}′{s:02.0f}″{'E' if decimal_deg >= 0 else 'W'}"

def set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10):
    def build(a, b, step_min):
        step_deg = step_min/60.0
        start = np.floor(a*60.0/step_min)*step_deg
        end   = np.ceil (b*60.0/step_min)*step_deg
        vals  = np.arange(start, end + step_deg*0.5, step_deg)
        return vals[(vals >= a) & (vals <= b)]
    ax.set_xticks(build(west, east, dx_min))
    ax.set_yticks(build(south, north, dy_min))

# =========================================================
# North arrow & checker scale bar 
# =========================================================
def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))
def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1,2,5,10):
        if m*base >= target: return m*base
    return 10*base

def add_north_arrow_tri(ax, size=0.08, pad=0.06, label='N',
                        arrow_downshift=0.015, label_upshift=0.010):
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size*0.72; y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size*0.09)
    ax.add_patch(Polygon([apex, left_base, tail], closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(x0 + size*0.36, (1.0 - pad) + size*0.10 + label_upshift, label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, fontweight='bold', color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None,
                          segments=4, fontsize=LABEL_FONTSIZE):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None: mid_lat_deg = (south + north)/2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0/mpd
    total_len_m = nice_len_125(width_m/5.0); seg_len_m = total_len_m/float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5*width_data - 0.5*total_len_units
    y_base  = south + 0.06*height_data
    bar_h   = 0.016*height_data
    for i in range(segments):
        xs = x_start + i*seg_len_units
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData,
                               facecolor=('black' if i%2==0 else 'white'),
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.010*height_data, "km",
            ha='center', va='bottom', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010*height_data, "0",
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)
    end_km = total_len_m/1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010*height_data, end_label,
            ha='center', va='top', fontsize=LABEL_FONTSIZE, color='black', zorder=6)

# =========================================================
# ULTRA-COMPACT LEGEND WITH ABBREVIATIONS
# =========================================================
def add_inset_legend_box_compact(ax, unique_classes, esa_colors, west, east, south, north):
    """Ultra-compact legend with abbreviations for small maps"""
    
    # Abbreviations for common classes
    ABBREVIATIONS = {
        10: "Tree", 20: "Shrub", 30: "Grass", 40: "Crop",
        50: "Built", 60: "Bare", 70: "Snow", 80: "Water",
        90: "Wetland", 95: "Mangrv", 100: "Moss"
    }
    
    # Adjust the position - Move it down to avoid touching the plot
    legend_x = 0.15  # 15% from left (bottom-right corner)
    legend_y = 0.80  # Moved down slightly to avoid overlap with the plot
    
    num_classes = len(unique_classes)
    
    # Use two columns if more than 5 classes
    if num_classes <= 5:
        ncol = 1
        legend_width = 0.70  # Increased width for better space
        rows = num_classes
        legend_height = 0.06 + rows * 0.030  # Adjusted height
    else:
        ncol = 2
        rows = (num_classes + 1) // 2  # Ceil division
        legend_width = 0.50  # Slightly wider for two columns
        legend_height = 0.06 + rows * 0.030
    
    # Create compact box
    legend_box = FancyBboxPatch(
        (legend_x, legend_y), legend_width, legend_height,
        transform=ax.transAxes,
        boxstyle="round,pad=0.01,rounding_size=0.01",
        facecolor='white',
        edgecolor='#222222',
        linewidth=0.25,
        alpha=0.95,
        zorder=10
    )
    ax.add_patch(legend_box)
    
    # Add title
    ax.text(legend_x + legend_width/2, legend_y + legend_height - 0.015,
            'LC Classes', transform=ax.transAxes, ha='center', va='top',
            fontsize=CB_LABEL_FONTSIZE + 1, fontweight='bold', color='#222222', zorder=11)
    
    # Single column layout
    if ncol == 1:
        item_height = (legend_height - 0.04) / num_classes
        for i, class_val in enumerate(sorted(unique_classes)):
            y_pos = legend_y + legend_height - 0.035 - (i + 0.5) * item_height
            
            # Increase the size of the color box
            color_box = Rectangle(
                (legend_x + 0.02, y_pos - item_height/5), 
                0.020, item_height/2.5,  # Increased width (0.020) for larger color symbol
                transform=ax.transAxes,
                facecolor=esa_colors.get(class_val, '#cccccc'),
                edgecolor='black',
                linewidth=0.15,
                zorder=11
            )
            ax.add_patch(color_box)
            
            # Label with abbreviation
            abbr = ABBREVIATIONS.get(class_val, f"C{class_val}")
            ax.text(legend_x + 0.045, y_pos,
                    f"{abbr}",
                    transform=ax.transAxes, ha='left', va='center',
                    fontsize=LABEL_FONTSIZE + 0.5, zorder=11)
    
    # Two column layout for many classes
    else:
        item_height = (legend_height - 0.04) / rows
        for i, class_val in enumerate(sorted(unique_classes)):
            row = i // 2
            col = i % 2
            x_offset = legend_x + 0.025 + col * 0.20  # Increased width for better spacing
            y_pos = legend_y + legend_height - 0.045 - (row + 0.5) * item_height
            
            # Increase the size of the color box
            color_box = Rectangle(
                (x_offset, y_pos - item_height/5), 
                0.018, item_height/2.5,  # Increased width (0.018) for larger color symbol
                transform=ax.transAxes,
                facecolor=esa_colors.get(class_val, '#cccccc'),
                edgecolor='black',
                linewidth=0.15,
                zorder=11
            )
            ax.add_patch(color_box)
            
            # Label with abbreviation
            abbr = ABBREVIATIONS.get(class_val, f"C{class_val}")
            ax.text(x_offset + 0.025, y_pos, f"{abbr}", transform=ax.transAxes, ha='left', va='center',
                    fontsize=LABEL_FONTSIZE + 0.8, zorder=11)

def find_lc_csv(primary_path, folders_to_search):
    if primary_path and os.path.exists(primary_path):
        print(f"[OK] Using CSV: {primary_path}"); return primary_path
    print(f"[WARN] CSV not found at intended path:\n      {primary_path}")
    for folder in folders_to_search:
        if not os.path.isdir(folder): continue
        for fn in os.listdir(folder):
            f = fn.lower()
            if f.endswith(".csv") and any(k in f for k in ["landcover", "land", "diagnosis", "lc"]):
                p = os.path.join(folder, fn); print(f"[OK] Using: {p}"); return p
    print("[WARN] No CSV found."); return None

# =========================================================
# 1) Land Cover MAP 
# =========================================================
LC_TIF = find_lc_tif(LC_TIF_INTENDED, [OUTPUT_DIR, ALT_LC])
if LC_TIF is None:
    raise FileNotFoundError("Land Cover TIF not found.")

with rasterio.open(LC_TIF) as src:
    lc_data   = src.read(1)
    lc_crs    = src.crs
    lc_bounds = src.bounds
    lc_nodata = src.nodata if src.nodata is not None else LC_NODATA_FALLBACK

with rasterio.open(MASK_TIF) as src:
    mask = src.read(1)

lc = np.where(lc_data == lc_nodata, np.nan, lc_data.astype(np.float32))
if mask.shape == lc.shape:
    lc = np.where(mask == 1, lc, np.nan)
else:
    print(f"[WARN] Mask shape {mask.shape} != LC shape {lc.shape}")

valid = lc[np.isfinite(lc)]
if valid.size == 0:
    raise RuntimeError("No valid Land Cover pixels to plot.")

# Get unique classes and create colormap
unique_classes = np.unique(valid).astype(int)
print(f"[INFO] Classes found: {unique_classes}")

west, south, east, north = transform_bounds(lc_crs, "EPSG:4326", *lc_bounds)
extent = [west, east, south, north]

# Create discrete colormap for land cover classes
cmap, class_labels, esa_colors = build_lc_colormap(unique_classes)

# Map class values to colormap indices
lc_indexed = np.full_like(lc, np.nan)
for i, class_val in enumerate(sorted(unique_classes)):
    lc_indexed[lc == class_val] = i

# Use discrete bounds for categorical data
vmin = 0
vmax = len(unique_classes) - 1

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)

# Plot land cover using indexed values
im = ax.imshow(lc_indexed, cmap=cmap, extent=extent, origin='upper', vmin=vmin, vmax=vmax, zorder=1)

# Boundary 
boundary = load_shapefile_boundary(STUDY_SHP, target_epsg=4326)
_ = draw_study_boundary(ax, extent, boundary, mask, shp_color=BOUNDARY_COLOR, shp_lw=BOUNDARY_LW, mask_color=MASK_CONTOUR_COLOR, mask_lw=MASK_LW,
                        erode_iters=2)

# Cartography - keep original positions
add_north_arrow_tri(ax)
add_checker_scale_bar(ax, west, east, south, north, crs_type='wgs84', mid_lat_deg=(south + north)/2.0, segments=4, fontsize=LABEL_FONTSIZE)

# LEGEND with abbreviations 
add_inset_legend_box_compact(ax, unique_classes, esa_colors, west, east, south, north)

# Axes 
ax.set_xlim(west, east); ax.set_ylim(south, north)
set_minute_ticks(ax, west, east, south, north, dx_min=5, dy_min=10)
ax.xaxis.set_major_formatter(FuncFormatter(format_dms_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_dms_lat))
ax.tick_params(axis='x', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.5, rotation=45, direction='out')
ax.tick_params(axis='y', which='major', labelsize=TICK_FONTSIZE, width=LW_AXIS, pad=TICK_PAD, length=1.5, rotation=90, direction='out')
for lbl in ax.get_xticklabels(): lbl.set_rotation(XROT_DEG); lbl.set_ha('right')
plt.subplots_adjust(left=0.18, right=0.88, bottom=0.18, top=0.98)
save_all(fig, LC_MAP_BASE)
plt.show()

# =========================================================
# 2) Land Cover Statistics Chart
# =========================================================
CSV_PATH = find_lc_csv(CSV_INTENDED, [OUTPUT_DIR, ALT_LC])
if CSV_PATH is None:
    print("[INFO] Statistics chart skipped (CSV not found).")
else:
    df = pd.read_csv(CSV_PATH)

    # Calculate statistics from raster directly
    raster_stats = []
    total_pixels = len(valid)
    
    for class_val in sorted(unique_classes):
        count = np.sum(lc == class_val)
        percentage = (count / total_pixels) * 100
        area_ha = count * 100 / 10000  # 10m pixels to hectares
        
        raster_stats.append({
            'class_code': int(class_val),
            'class_name': ESA_LEGEND.get(int(class_val), f'Class_{int(class_val)}'),
            'percentage': percentage,
            'area_ha': area_ha
        })
    
    stats_df = pd.DataFrame(raster_stats).sort_values('percentage', ascending=False)

    # Create statistics chart 
    fig2, ax2 = plt.subplots(1, 1, figsize=(6, 4))
    
    # Color bars by class
    colors = [esa_colors.get(int(row['class_code']), '#gray') for _, row in stats_df.iterrows()]
    
    bars = ax2.bar(range(len(stats_df)), stats_df['percentage'], color=colors)
    
    ax2.set_title("Land Cover Composition", fontsize=12, fontweight='bold', pad=10)
    ax2.set_xlabel("Land Cover Classes", fontsize=9)
    ax2.set_ylabel("Percentage (%)", fontsize=9)
    
    # Set x-axis labels
    ax2.set_xticks(range(len(stats_df)))
    ax2.set_xticklabels([f"{int(row['class_code'])}" for _, row in stats_df.iterrows()], 
                        fontsize=8, rotation=45)
    
    ax2.tick_params(axis='y', labelsize=8)
    ax2.set_ylim(0, stats_df['percentage'].max() * 1.1)
    ax2.grid(True, linestyle='-', alpha=0.3, color='gray')
    ax2.set_axisbelow(True)
    
    # Add percentage labels on bars
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if height > 1:  # Only label bars > 1%
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    safe_savefig(fig2, LC_TS_PNG, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# =========================================================
# Console summary 
# =========================================================
print("\n" + "="*70)
print("LAND COVER — SUMMARY ")
print("="*70)
print(f"  • LC TIF:   {LC_TIF}")
print(f"  • Map base: {LC_MAP_BASE}.(png/pdf/svg)")
print(f"  • CSV:      {CSV_PATH if CSV_PATH else 'NOT FOUND'}")
print(f"  • Stats PNG: {LC_TS_PNG if os.path.exists(LC_TS_PNG) else 'Not saved'}")
print(f"  • Classes found: {len(unique_classes)} ({unique_classes})")
print(f"  • Legend: Ultra-compact with abbreviations, bottom-right position")
print("="*70)


# Phase-3: PS-InSAR products

# Step-3.1:Coherence

In [ ]:
# =========================================================
# Step 3.1 — PS-InSAR Coherence 
# =========================================================

import os
from datetime import datetime
import numpy as np
import pandas as pd

# -------------------------- Optional GIS --------------------------
try:
    import rasterio
    from rasterio.warp import transform as rio_transform, reproject, Resampling
    from rasterio.transform import rowcol
    _HAVE_RASTER = True
except Exception:
    _HAVE_RASTER = False

# -------------------------- MAT readers --------------------------
try:
    from scipy.io import loadmat as scipy_loadmat
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

try:
    import h5py
    _HAVE_H5PY = True
except Exception:
    _HAVE_H5PY = False

# -------------------------- PATHS --------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

MAT_DIR = os.path.join(BASE, "NTD_PS_InSAR_data")
ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
MASK_TIF   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

OUT_DIR = os.path.join(BASE, "Results", "Step-3.1-Coherence")
os.makedirs(OUT_DIR, exist_ok=True)

CSV_OUT = os.path.join(OUT_DIR, "ps_points_coherence.csv")
COH_TIF = os.path.join(OUT_DIR, "coherence_ps_mean_32645_12p5m_float32.tif")

FN_COH = "coherence_clipped.mat"
FN_VEL = "mean_v_clipped.mat"

NODATA = -9999.0
COH_PS_THRESHOLD = 0.70

# -------------------------- Utils --------------------------
def pp(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

def _flatten_h5(group, out, prefix=""):
    for k, v in group.items():
        name = f"{prefix}/{k}" if prefix else k
        if isinstance(v, h5py.Dataset):
            try:
                out[name] = np.array(v)
            except Exception:
                pass
        elif isinstance(v, h5py.Group):
            _flatten_h5(v, out, name)

def load_mat_any(fp):
    if _HAVE_SCIPY:
        try:
            D = scipy_loadmat(fp, squeeze_me=True, struct_as_record=False)
            out = {k: np.asarray(v) for k, v in D.items()
                   if not k.startswith("__") and isinstance(v, (np.ndarray, np.number))}
            if out:
                return out
        except Exception:
            pass
    if _HAVE_H5PY:
        with h5py.File(fp, "r") as f:
            out = {}
            _flatten_h5(f, out)
            return out
    raise RuntimeError(f"Cannot read MAT file: {fp}")

def inventory(D, title):
    pp(f"[MAT] {title}")
    for k, v in D.items():
        if isinstance(v, np.ndarray):
            pp(f"  {k}: shape={v.shape}, dtype={v.dtype}")

def _get_1d(D, keys):
    for key in keys:
        for k, v in D.items():
            lk = k.lower()
            if lk == key or lk.endswith(key) or key in lk:
                a = np.asarray(v)
                if a.ndim == 2 and 1 in a.shape:
                    a = a.reshape(-1)
                if a.ndim == 1:
                    return a.astype("float64")
    return None

# -------------------------- Build PS table --------------------------
def build_ps_table():
    pp("Reading coherence_clipped.mat …")
    coh_fp = os.path.join(MAT_DIR, FN_COH)
    vel_fp = os.path.join(MAT_DIR, FN_VEL)

    Dcoh = load_mat_any(coh_fp)
    inventory(Dcoh, "coherence_clipped.mat")

    lon = _get_1d(Dcoh, ["lon", "longitude"])
    lat = _get_1d(Dcoh, ["lat", "latitude"])
    coh = _get_1d(Dcoh, ["coh", "coherence", "gamma", "ps_coh"])

    if lon is None or lat is None or coh is None:
        raise RuntimeError("coherence_clipped.mat must contain lon, lat, coh vectors.")

    n = coh.size
    if lon.size != n or lat.size != n:
        raise RuntimeError(f"Internal length mismatch: lon={lon.size}, lat={lat.size}, coh={n}")

    vel = None
    if os.path.exists(vel_fp):
        Dvel = load_mat_any(vel_fp)
        vel = _get_1d(Dvel, ["mean_v", "velocity", "vel", "vel_mm_yr"])
        if vel is not None and vel.size != n:
            pp(f"[WARN] velocity size {vel.size} != {n}; ignored")
            vel = None

    df = pd.DataFrame({"lon": lon, "lat": lat, "coh": coh})
    if vel is not None:
        df["vel_mm_yr"] = vel

    df["coh"] = pd.to_numeric(df["coh"], errors="coerce")
    valid = df["coh"].notna()

    pp(f"PS count={len(df)}; valid coherence={valid.sum()} ({100*valid.mean():.1f}%)")
    if valid.any():
        qs = df.loc[valid, "coh"].quantile([0, .05, .25, .5, .75, .95, 1]).to_dict()
        pp("Coherence quantiles: " + ", ".join([f"{k:.2f}:{v:.3f}" for k, v in qs.items()]))
        pp(f"PS with coh ≥ {COH_PS_THRESHOLD:.2f}: {(df.loc[valid,'coh']>=COH_PS_THRESHOLD).mean()*100:.1f}%")

    df.to_csv(CSV_OUT, index=False)
    pp(f"[CSV] {CSV_OUT}")
    return df

# -------------------------- Rasterization --------------------------
def rasterize_to_anchor(df):
    if not _HAVE_RASTER:
        pp("[SKIP] rasterio not available")
        return

    # OPEN anchor and COPY everything needed while still open
    with rasterio.open(ANCHOR_TIF) as ad:
        dst_crs = ad.crs
        dst_tf  = ad.transform
        H, W    = ad.height, ad.width
        profile = ad.profile.copy()   # <-- critical fix (copy before closing)

    # lon/lat -> anchor CRS
    xs, ys = rio_transform("EPSG:4326", dst_crs, df["lon"].values, df["lat"].values)

    rows, cols = rowcol(dst_tf, xs, ys, op=int)
    rows = np.asarray(rows)
    cols = np.asarray(cols)

    inside = (rows >= 0) & (rows < H) & (cols >= 0) & (cols < W)
    rows, cols, coh = rows[inside], cols[inside], df["coh"].values[inside]

    good = np.isfinite(coh)
    rows, cols, coh = rows[good], cols[good], coh[good]

    sumA = np.zeros((H, W), dtype="float64")
    cntA = np.zeros((H, W), dtype="uint32")
    np.add.at(sumA, (rows, cols), coh)
    np.add.at(cntA, (rows, cols), 1)

    out = np.full((H, W), NODATA, dtype="float32")
    m = cntA > 0
    out[m] = (sumA[m] / cntA[m]).astype("float32")

    # Apply study mask (reproject if needed)
    if os.path.exists(MASK_TIF):
        with rasterio.open(MASK_TIF) as ms:
            M = ms.read(1)
            if (ms.crs != dst_crs) or (ms.transform != dst_tf) or (ms.width != W) or (ms.height != H):
                Mr = np.zeros((H, W), dtype="uint8")
                reproject(
                    M, Mr,
                    src_transform=ms.transform, src_crs=ms.crs,
                    dst_transform=dst_tf, dst_crs=dst_crs,
                    resampling=Resampling.nearest,
                    src_nodata=0, dst_nodata=0
                )
                M = Mr
        out = np.where(M > 0, out, NODATA)

    profile.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=NODATA,
        compress="LZW",
        tiled=True
    )

    tmp = COH_TIF + ".part"
    with rasterio.open(tmp, "w", **profile) as ds:
        ds.write(out, 1)
    os.replace(tmp, COH_TIF)

    vals = out[out != NODATA]
    if vals.size > 0:
        pp(f"[RASTER] valid pixels={vals.size}, min/mean/max={vals.min():.3f}/{vals.mean():.3f}/{vals.max():.3f}")
    pp(f"[RASTER] saved → {COH_TIF}")

# -------------------------- MAIN --------------------------
def main():
    pp("=== Step 3.1 — PS-InSAR Coherence ===")
    df = build_ps_table()
    if _HAVE_RASTER and os.path.exists(ANCHOR_TIF):
        rasterize_to_anchor(df)
    else:
        pp("[INFO] Raster skipped ")
    pp("=== DONE (Step 3.1) ===")

if __name__ == "__main__":
    main()


# Visualization: Coherence

In [ ]:
# =========================================================
# Step 3.1 — PS-InSAR Coherence Visualization
# =========================================================
#
# Main updates:
#   1. 2.5 × 5 inch map; 2.5 × 2.5 inch histogram and KDE.
#   2. Four longitude DMS labels.
#   3. Three latitude DMS labels (LinearLocator, rotated 90°).
#   4. Bottom-most latitude DMS label anchored above its tick so it
#      cannot collide with the first longitude label at the corner.
#   5. No "Latitude" / "Longitude" axis titles.
#   6. DMS font size 6.2, bold.
#   7. Same north arrow, scale bar, road, boundary, legend and colour bar.
#   8. PNG, PDF and SVG at 900 dpi.
#
# Scientific note:
#   Coherence is supplied as PS points in ps_points_coherence.csv.
#   The map therefore retains the original Step 3.1 k-nearest-neighbour
#   inverse-distance-weighted surface. No additional smoothing is performed
#   by imshow; interpolation="nearest" only controls raster display.
# =========================================================

from __future__ import annotations

import json
import os
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter, LinearLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False

try:
    from scipy.spatial import cKDTree
    from scipy.interpolate import griddata
    from scipy.stats import gaussian_kde
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    import geopandas as gpd
    from shapely.geometry import Point as ShapelyPoint
    from shapely.geometry import shape
    from shapely.ops import unary_union
    from shapely.prepared import prep
    HAVE_GPD = True
    HAVE_SHAPELY = True
except Exception:
    HAVE_GPD = False
    HAVE_SHAPELY = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

import rasterio
from rasterio.transform import rowcol
from rasterio.warp import transform_geom


# =========================================================
# PATHS
# =========================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

OUT_DIR = BASE / "Results" / "Step-3.1-Coherence"
CSV_INPUT = OUT_DIR / "ps_points_coherence.csv"

STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
STUDY_SHP = BASE / "Study_Area.shp"
ROAD_SHP = BASE / "NTD_Road.shp"

OUT_MAP_PNG = OUT_DIR / "Coherence Map.png"
OUT_MAP_PDF = OUT_DIR / "Coherence Map.pdf"
OUT_MAP_SVG = OUT_DIR / "Coherence Map.svg"

OUT_HIST_PNG = OUT_DIR / "Coherence Histogram.png"
OUT_HIST_PDF = OUT_DIR / "Coherence Histogram.pdf"
OUT_HIST_SVG = OUT_DIR / "Coherence Histogram.svg"

OUT_KDE_PNG = OUT_DIR / "Coherence Density (KDE).png"
OUT_KDE_PDF = OUT_DIR / "Coherence Density (KDE).pdf"
OUT_KDE_SVG = OUT_DIR / "Coherence Density (KDE).svg"

OUT_SUMMARY_JSON = OUT_DIR / "coherence_visualization_summary.json"

OUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# DISPLAY SETTINGS
# =========================================================
MAP_FIGSIZE = (2.5, 5)
HIST_FIGSIZE = (2.5, 2.5)
KDE_FIGSIZE = (2.5, 2.5)

FIG_DPI = 900
CAX_SIZE = "5%"
CAX_PAD = 0.01

COHERENCE_CMAP = plt.get_cmap("jet").copy()
COHERENCE_CMAP.set_bad(color="white", alpha=1.0)

COHERENCE_VMIN = 0.0
COHERENCE_VMAX = 1.0
COHERENCE_INFORMATION_THRESHOLD = 0.70

DISPLAY_BUFFER_FRACTION = 0.02
LONGITUDE_DMS_TICK_COUNT = 4
LATITUDE_DMS_TICK_COUNT = 3
DMS_TICK_FONTSIZE = 6.2

# Requested larger histogram text.
HIST_AXIS_LABEL_FONTSIZE = 7
HIST_TICK_FONTSIZE = 6

# KDE text sizes are also made readable.
KDE_AXIS_LABEL_FONTSIZE = 7
KDE_TICK_FONTSIZE = 6

# Existing Step 3.1 interpolation design.
GRID_RESOLUTION = 200
KNN_IDW_K = 32
IDW_POWER = 2.0
MAX_DISTANCE_DEG = 0.01

# Keep False to match the clean DV+ / SNDR map style.
# Set True only when PS point locations should be explicitly shown.
SHOW_PS_POINTS = False
PS_POINT_SIZE = 0.5
PS_POINT_ALPHA = 0.60

SHOW_FIGURES = True
SAVE_FIGURES = True

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = [
    "Times New Roman",
    "DejaVu Serif",
]
plt.rcParams["axes.unicode_minus"] = False


# =========================================================
# GENERAL HELPERS
# =========================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def find_column(
    columns: list[str],
    token: str,
) -> str | None:
    token_lower = token.lower()
    return next(
        (
            column
            for column in columns
            if token_lower in column.lower()
        ),
        None,
    )


# =========================================================
# DMS AND MAP DECORATIONS
# =========================================================
def dms_label(value: float, is_longitude: bool = True) -> str:
    absolute = abs(float(value))
    degrees = int(absolute)

    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_longitude
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def add_north_arrow_tri(
    axis,
    size: float = 0.09,
    pad: float = 0.09,
    label: str = "N",
    arrow_downshift: float = 0.025,
    label_upshift: float = 0.0,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (
        (1.0 - pad)
        + size * 0.10
        + label_upshift
    )

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base
        if candidate >= target_metres:
            return candidate

    return 10 * base


def add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    segments: int = 4,
    fontsize: float = 5.6,
    frame_lw: float = 0.4,
) -> None:
    width_deg = east - west
    height_deg = north - south
    mid_latitude = 0.5 * (south + north)

    metres_per_degree = metres_per_degree_lon(mid_latitude)
    width_metres = width_deg * metres_per_degree
    data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = total_length_metres * data_units_per_metre
    segment_length_units = segment_length_metres * data_units_per_metre

    x_start = (
        west
        + 0.5 * width_deg
        - 0.5 * total_length_units
    )
    y_base = south + 0.06 * height_deg
    bar_height = 0.018 * height_deg

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_deg

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (
                    x_start + index * segment_length_units,
                    y_base,
                ),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=(
                    "black" if index % 2 == 0 else "white"
                ),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_deg,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_deg,
        "0",
        ha="center",
        va="top",
        fontsize=fontsize,
        fontweight="bold",
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_deg,
        end_label,
        ha="center",
        va="top",
        fontsize=fontsize,
        fontweight="bold",
        zorder=15,
    )


# =========================================================
# BOUNDARY AND ROAD HELPERS
# =========================================================
def load_vector(
    path: Path,
    target_crs: str = "EPSG:4326",
):
    if not path.exists():
        log(f"[WARN] Vector file not found: {path}")
        return None

    if HAVE_GPD:
        try:
            vector = gpd.read_file(path)

            if vector.crs is None:
                raise ValueError(
                    f"Vector file has no CRS: {path}"
                )

            return vector.to_crs(target_crs)

        except Exception as exc:
            log(f"[WARN] GeoPandas loading failed for {path.name}: {exc}")

    if HAVE_FIONA:
        try:
            geometries = []

            with fiona.open(path, "r") as source:
                source_crs = source.crs

                if not source_crs:
                    raise ValueError(
                        f"Vector file has no CRS: {path}"
                    )

                for feature in source:
                    geometry = feature["geometry"]
                    if geometry is None:
                        continue

                    geometries.append(
                        transform_geom(
                            source_crs,
                            target_crs,
                            geometry,
                        )
                    )

            return {"geometries": geometries}

        except Exception as exc:
            log(f"[WARN] Fiona loading failed for {path.name}: {exc}")

    return None


def get_union_polygon(boundary_data):
    if boundary_data is None or not HAVE_SHAPELY:
        return None

    try:
        if HAVE_GPD and hasattr(boundary_data, "geometry"):
            return unary_union(
                list(boundary_data.geometry)
            )

        if isinstance(boundary_data, dict):
            geometries = [
                shape(geometry)
                for geometry in boundary_data["geometries"]
            ]
            return unary_union(geometries)

    except Exception as exc:
        log(f"[WARN] Boundary union failed: {exc}")

    return None


def plot_roads(
    axis,
    roads,
    colour: str = "red",
    linewidth: float = 0.7,
    alpha: float = 0.9,
    zorder: int = 850,
) -> None:
    if roads is None:
        return

    try:
        if HAVE_GPD and hasattr(roads, "plot"):
            roads.plot(
                ax=axis,
                color=colour,
                linewidth=linewidth,
                alpha=alpha,
                zorder=zorder,
            )
            return

        if isinstance(roads, dict):
            for geometry in roads["geometries"]:
                geometry_type = geometry["type"]

                if geometry_type == "LineString":
                    line_groups = [geometry["coordinates"]]
                elif geometry_type == "MultiLineString":
                    line_groups = geometry["coordinates"]
                else:
                    continue

                for line in line_groups:
                    axis.plot(
                        [point[0] for point in line],
                        [point[1] for point in line],
                        color=colour,
                        linewidth=linewidth,
                        alpha=alpha,
                        zorder=zorder,
                    )

    except Exception as exc:
        log(f"[WARN] Road plotting failed: {exc}")


def plot_boundary(
    axis,
    boundary_data,
    colour: str = "black",
    linewidth: float = 0.7,
    alpha: float = 1.0,
) -> None:
    if boundary_data is None:
        return

    try:
        if HAVE_GPD and hasattr(boundary_data, "boundary"):
            boundary_data.boundary.plot(
                ax=axis,
                color=colour,
                linewidth=linewidth,
                alpha=alpha,
                zorder=999,
            )
            return

        if isinstance(boundary_data, dict):
            for geometry in boundary_data["geometries"]:
                geometry_type = geometry["type"]

                if geometry_type == "Polygon":
                    polygon_groups = [geometry["coordinates"]]
                elif geometry_type == "MultiPolygon":
                    polygon_groups = geometry["coordinates"]
                else:
                    continue

                for polygon in polygon_groups:
                    exterior = polygon[0]
                    axis.plot(
                        [point[0] for point in exterior],
                        [point[1] for point in exterior],
                        color=colour,
                        linewidth=linewidth,
                        alpha=alpha,
                        zorder=999,
                    )

    except Exception as exc:
        log(f"[WARN] Boundary plotting failed: {exc}")


# =========================================================
# MASK AND POINT CLIPPING
# =========================================================
def clip_points_to_mask(
    dataframe: pd.DataFrame,
    longitude_column: str,
    latitude_column: str,
    mask_path: Path,
) -> pd.DataFrame:
    if not mask_path.exists():
        return dataframe

    with rasterio.open(mask_path) as dataset:
        longitudes = dataframe[longitude_column].to_numpy(
            dtype="float64"
        )
        latitudes = dataframe[latitude_column].to_numpy(
            dtype="float64"
        )

        if dataset.crs is None:
            raise ValueError(
                f"Study mask has no CRS: {mask_path}"
            )

        if str(dataset.crs).upper() != "EPSG:4326":
            if not HAVE_PYPROJ:
                raise RuntimeError(
                    "pyproj is required to transform PS coordinates "
                    "to the study-mask CRS."
                )

            x_values, y_values = Transformer.from_crs(
                "EPSG:4326",
                dataset.crs,
                always_xy=True,
            ).transform(longitudes, latitudes)
        else:
            x_values = longitudes
            y_values = latitudes

        rows, columns = rowcol(
            dataset.transform,
            x_values,
            y_values,
            op=np.floor,
        )
        rows = np.asarray(rows, dtype="int64")
        columns = np.asarray(columns, dtype="int64")

        inside_grid = (
            (rows >= 0)
            & (columns >= 0)
            & (rows < dataset.height)
            & (columns < dataset.width)
        )

        mask_values = np.zeros(rows.shape, dtype="float64")

        valid_indices = np.where(inside_grid)[0]
        if valid_indices.size:
            band = dataset.read(1)
            mask_values[valid_indices] = band[
                rows[valid_indices],
                columns[valid_indices],
            ].astype("float64")

        keep = (
            inside_grid
            & np.isfinite(mask_values)
            & (mask_values > 0.5)
        )

    return dataframe.loc[keep].copy()


def mask_grid_with_raster(
    surface: np.ndarray,
    longitude_grid: np.ndarray,
    latitude_grid: np.ndarray,
    mask_path: Path,
) -> np.ndarray:
    if not mask_path.exists():
        return surface

    with rasterio.open(mask_path) as dataset:
        if dataset.crs is None:
            raise ValueError(
                f"Study mask has no CRS: {mask_path}"
            )

        longitudes = longitude_grid.ravel()
        latitudes = latitude_grid.ravel()

        if str(dataset.crs).upper() != "EPSG:4326":
            if not HAVE_PYPROJ:
                raise RuntimeError(
                    "pyproj is required to transform the coherence grid "
                    "to the study-mask CRS."
                )

            x_values, y_values = Transformer.from_crs(
                "EPSG:4326",
                dataset.crs,
                always_xy=True,
            ).transform(longitudes, latitudes)
        else:
            x_values = longitudes
            y_values = latitudes

        rows, columns = rowcol(
            dataset.transform,
            x_values,
            y_values,
            op=np.floor,
        )
        rows = np.asarray(rows, dtype="int64")
        columns = np.asarray(columns, dtype="int64")

        inside_grid = (
            (rows >= 0)
            & (columns >= 0)
            & (rows < dataset.height)
            & (columns < dataset.width)
        )

        mask_values = np.zeros(rows.shape, dtype="float64")

        valid_indices = np.where(inside_grid)[0]
        if valid_indices.size:
            band = dataset.read(1)
            mask_values[valid_indices] = band[
                rows[valid_indices],
                columns[valid_indices],
            ].astype("float64")

        inside_mask = (
            inside_grid
            & np.isfinite(mask_values)
            & (mask_values > 0.5)
        )

    return np.where(
        inside_mask.reshape(surface.shape),
        surface,
        np.nan,
    )


def clip_points_to_polygon(
    dataframe: pd.DataFrame,
    longitude_column: str,
    latitude_column: str,
    polygon,
) -> pd.DataFrame:
    if polygon is None or not HAVE_SHAPELY:
        return dataframe

    prepared_polygon = prep(polygon)

    coordinates = zip(
        dataframe[longitude_column].to_numpy(dtype="float64"),
        dataframe[latitude_column].to_numpy(dtype="float64"),
    )

    keep = np.fromiter(
        (
            prepared_polygon.covers(
                ShapelyPoint(longitude, latitude)
            )
            for longitude, latitude in coordinates
        ),
        dtype=bool,
        count=len(dataframe),
    )

    return dataframe.loc[keep].copy()


def mask_grid_to_polygon(
    surface: np.ndarray,
    longitude_grid: np.ndarray,
    latitude_grid: np.ndarray,
    polygon,
) -> np.ndarray:
    if polygon is None or not HAVE_SHAPELY:
        return surface

    prepared_polygon = prep(polygon)

    keep = np.fromiter(
        (
            prepared_polygon.covers(
                ShapelyPoint(longitude, latitude)
            )
            for longitude, latitude in zip(
                longitude_grid.ravel(),
                latitude_grid.ravel(),
            )
        ),
        dtype=bool,
        count=longitude_grid.size,
    ).reshape(surface.shape)

    return np.where(keep, surface, np.nan)


# =========================================================
# KNN-IDW SURFACE
# =========================================================
def interpolate_knn_idw(
    point_coordinates: np.ndarray,
    values: np.ndarray,
    longitude_grid: np.ndarray,
    latitude_grid: np.ndarray,
    neighbours: int,
    power: float,
    maximum_distance: float | None,
) -> np.ndarray:
    grid_coordinates = np.column_stack(
        [
            longitude_grid.ravel(),
            latitude_grid.ravel(),
        ]
    )

    if not HAVE_SCIPY:
        raise RuntimeError(
            "SciPy is required for the Step 3.1 coherence surface."
        )

    if point_coordinates.shape[0] < 3:
        raise RuntimeError(
            "At least three PS points are required for interpolation."
        )

    tree = cKDTree(point_coordinates)

    neighbours = int(
        min(max(neighbours, 1), point_coordinates.shape[0])
    )

    distances, indices = tree.query(
        grid_coordinates,
        k=neighbours,
        workers=-1,
    )

    if neighbours == 1:
        distances = distances.reshape(-1, 1)
        indices = indices.reshape(-1, 1)

    valid_neighbours = np.isfinite(distances)

    if maximum_distance is not None:
        valid_neighbours &= distances <= maximum_distance

    weights = np.zeros_like(distances, dtype="float64")
    weights[valid_neighbours] = 1.0 / np.maximum(
        distances[valid_neighbours],
        1.0e-12,
    ) ** float(power)

    exact_match = valid_neighbours & (distances <= 1.0e-12)
    rows_with_exact_match = exact_match.any(axis=1)

    if rows_with_exact_match.any():
        weights[rows_with_exact_match, :] = 0.0
        first_match = np.argmax(
            exact_match[rows_with_exact_match, :],
            axis=1,
        )
        row_indices = np.where(rows_with_exact_match)[0]
        weights[row_indices, first_match] = 1.0

    numerator = np.sum(
        weights * values[indices],
        axis=1,
    )
    denominator = np.sum(weights, axis=1)

    predicted = np.full(
        grid_coordinates.shape[0],
        np.nan,
        dtype="float64",
    )

    valid_prediction = denominator > 0.0
    predicted[valid_prediction] = (
        numerator[valid_prediction]
        / denominator[valid_prediction]
    )

    return predicted.reshape(longitude_grid.shape)


# =========================================================
# STATISTICS
# =========================================================
def calculate_statistics(
    values: np.ndarray,
) -> dict[str, float | int]:
    values = values[np.isfinite(values)].astype("float64")

    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "max": float(np.max(values)),
        "above_information_threshold_fraction": float(
            np.mean(values >= COHERENCE_INFORMATION_THRESHOLD)
        ),
    }


# =========================================================
# MAP
# =========================================================
def create_coherence_map(
    dataframe: pd.DataFrame,
    longitude_column: str,
    latitude_column: str,
    coherence_column: str,
    surface: np.ndarray,
    west: float,
    east: float,
    south: float,
    north: float,
    boundary_data,
    roads,
) -> None:
    figure, axis = plt.subplots(
        1,
        1,
        figsize=MAP_FIGSIZE,
        facecolor="white",
    )
    axis.set_facecolor("white")

    image = axis.imshow(
        np.ma.masked_invalid(surface),
        cmap=COHERENCE_CMAP,
        extent=[west, east, south, north],
        vmin=COHERENCE_VMIN,
        vmax=COHERENCE_VMAX,
        origin="lower",
        interpolation="nearest",
        alpha=1.0,
        zorder=1,
    )

    if SHOW_PS_POINTS:
        axis.scatter(
            dataframe[longitude_column],
            dataframe[latitude_column],
            c=np.clip(
                dataframe[coherence_column].to_numpy(
                    dtype="float64"
                ),
                COHERENCE_VMIN,
                COHERENCE_VMAX,
            ),
            cmap=COHERENCE_CMAP,
            vmin=COHERENCE_VMIN,
            vmax=COHERENCE_VMAX,
            s=PS_POINT_SIZE,
            alpha=PS_POINT_ALPHA,
            edgecolors="none",
            zorder=2,
        )

    plot_roads(
        axis,
        roads,
        colour="red",
        linewidth=0.7,
        alpha=0.9,
    )
    plot_boundary(
        axis,
        boundary_data,
        colour="black",
        linewidth=0.7,
    )

    axis.set_xlim(west, east)
    axis.set_ylim(south, north)

    # No "Latitude" / "Longitude" axis titles by request.

    axis.xaxis.set_major_locator(
        LinearLocator(LONGITUDE_DMS_TICK_COUNT)
    )
    axis.xaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: dms_label(value, True)
        )
    )

    # Fixed latitude tick count so the rotated DMS labels
    # cannot overlap (previously the default locator placed
    # more ticks than the axis height could accommodate).
    axis.yaxis.set_major_locator(
        LinearLocator(LATITUDE_DMS_TICK_COUNT)
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: dms_label(value, False)
        )
    )

    axis.tick_params(
        axis="x",
        which="major",
        labelsize=DMS_TICK_FONTSIZE,
    )
    axis.tick_params(
        axis="y",
        which="major",
        labelsize=DMS_TICK_FONTSIZE,
    )

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")
        label.set_fontsize(DMS_TICK_FONTSIZE)
        label.set_fontweight("bold")

    y_tick_labels = axis.get_yticklabels()
    for label in y_tick_labels:
        label.set_fontsize(DMS_TICK_FONTSIZE)
        label.set_fontweight("bold")
        label.set_rotation(90)
        label.set_va("center")

    # Lift the bottom-most latitude label fully above its tick so it
    # clears the first (45°-rotated) longitude label at the corner.
    if y_tick_labels:
        y_tick_labels[0].set_va("bottom")
        log(
            "[ADJUST] Bottom latitude DMS label anchored above its "
            "tick (va='bottom') to avoid corner overlap with the "
            "first longitude label."
        )

    add_north_arrow_tri(
        axis,
        size=0.09,
        pad=0.09,
    )
    add_checker_scale_bar(
        axis,
        west,
        east,
        south,
        north,
        segments=4,
        fontsize=4,
    )

    legend_elements = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=1.0,
            label="Road",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=1.0,
            label="Study Area",
        ),
    ]

    legend = axis.legend(
        handles=legend_elements,
        loc="lower right",
        bbox_to_anchor=(0.98, 0.09),
        bbox_transform=axis.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=3.1,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0,
    )
    legend.get_frame().set_edgecolor("lightgray")
    legend.get_frame().set_linewidth(0.3)
    legend.get_frame().set_facecolor("white")
    for text_item in legend.get_texts():
        text_item.set_fontweight("bold")

    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes(
        "right",
        size=CAX_SIZE,
        pad=CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(labelsize=4.4, width=0.6, length=2)
    for tick_label in colorbar.ax.get_yticklabels():
        tick_label.set_fontweight("bold")
    colorbar.set_label(
        "Coherence (0–1)",
        fontsize=5.2,
        fontweight="bold",
    )
    colorbar.set_ticks(
        np.linspace(
            COHERENCE_VMIN,
            COHERENCE_VMAX,
            5,
        )
    )

    figure.tight_layout()

    if SAVE_FIGURES:
        figure.savefig(
            OUT_MAP_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )
        figure.savefig(
            OUT_MAP_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )
        figure.savefig(
            OUT_MAP_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )

        log(f"[MAP] {OUT_MAP_PNG}")
        log(f"[MAP] {OUT_MAP_PDF}")
        log(f"[MAP] {OUT_MAP_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# HISTOGRAM
# =========================================================
def create_coherence_histogram(
    values: np.ndarray,
    statistics: dict[str, float | int],
) -> None:
    figure, axis = plt.subplots(
        1,
        1,
        figsize=HIST_FIGSIZE,
        facecolor="white",
    )

    bins = np.linspace(0.0, 1.0, 21)
    counts, edges = np.histogram(values, bins=bins)
    centres = 0.5 * (edges[:-1] + edges[1:])
    colours = COHERENCE_CMAP(centres)

    axis.bar(
        centres,
        counts,
        width=(bins[1] - bins[0]) * 0.9,
        color=colours,
        edgecolor="black",
        linewidth=0.4,
        alpha=0.9,
    )

    axis.axvline(
        statistics["mean"],
        color="darkred",
        linestyle="--",
        linewidth=0.7,
        label=f"Mean: {statistics['mean']:.3f}",
    )
    axis.axvline(
        statistics["median"],
        color="orange",
        linestyle="--",
        linewidth=0.7,
        label=f"Median: {statistics['median']:.3f}",
    )
    axis.axvline(
        statistics["p95"],
        color="green",
        linestyle="--",
        linewidth=0.7,
        label=f"95th %ile: {statistics['p95']:.3f}",
    )

    axis.legend(
        fontsize=5,
        loc="upper left",
        framealpha=0.9,
    )

    # Requested larger text on both histogram axes.
    axis.set_xlabel(
        "Coherence Value",
        fontsize=HIST_AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )
    axis.set_ylabel(
        "Number of PS Points",
        fontsize=HIST_AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )

    axis.tick_params(
        axis="both",
        which="major",
        labelsize=HIST_TICK_FONTSIZE,
    )
    for tick_label in axis.get_xticklabels() + axis.get_yticklabels():
        tick_label.set_fontweight("bold")

    axis.set_xlim(0.0, 1.0)
    axis.grid(
        True,
        alpha=0.25,
        linestyle="-",
        linewidth=0.4,
        axis="y",
    )
    axis.set_axisbelow(True)

    figure.tight_layout(pad=1.2)

    if SAVE_FIGURES:
        figure.savefig(
            OUT_HIST_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )
        figure.savefig(
            OUT_HIST_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )
        figure.savefig(
            OUT_HIST_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )

        log(f"[HISTOGRAM] {OUT_HIST_PNG}")
        log(f"[HISTOGRAM] {OUT_HIST_PDF}")
        log(f"[HISTOGRAM] {OUT_HIST_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# KDE
# =========================================================
def create_coherence_kde(
    values: np.ndarray,
) -> None:
    figure, axis = plt.subplots(
        1,
        1,
        figsize=KDE_FIGSIZE,
        facecolor="white",
    )

    x_values = np.linspace(0.0, 1.0, 600)
    sample_count = int(values.size)
    bin_width = 1.0 / 20.0

    if HAVE_SCIPY and sample_count >= 5:
        kde = gaussian_kde(values, bw_method="scott")
        density = kde(x_values)
    else:
        histogram_density, histogram_edges = np.histogram(
            values,
            bins=100,
            range=(0.0, 1.0),
            density=True,
        )
        histogram_centres = 0.5 * (
            histogram_edges[:-1] + histogram_edges[1:]
        )
        density = np.interp(
            x_values,
            histogram_centres,
            histogram_density,
        )

    base_colour = COHERENCE_CMAP(0.65)

    axis.plot(
        x_values,
        density,
        linewidth=1.2,
        color=base_colour,
    )
    axis.fill_between(
        x_values,
        0.0,
        density,
        alpha=0.15,
        color=base_colour,
    )

    probability = float(
        np.mean(
            values >= COHERENCE_INFORMATION_THRESHOLD
        )
    )

    axis.text(
        0.05,
        0.08,
        (
            f"P(Coherence ≥ "
            f"{COHERENCE_INFORMATION_THRESHOLD:.2f}) = "
            f"{probability:.2f}"
        ),
        transform=axis.transAxes,
        fontsize=4.5,
        bbox={
            "boxstyle": "round,pad=0.25",
            "fc": "white",
            "ec": "0.7",
            "alpha": 0.9,
        },
    )

    axis.set_xlabel(
        "Coherence Value",
        fontsize=KDE_AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )
    axis.set_ylabel(
        "Density",
        fontsize=KDE_AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )
    axis.set_xlim(0.0, 1.0)
    axis.tick_params(
        axis="both",
        labelsize=KDE_TICK_FONTSIZE,
    )
    for tick_label in axis.get_xticklabels() + axis.get_yticklabels():
        tick_label.set_fontweight("bold")
    axis.grid(
        True,
        alpha=0.25,
        linestyle="-",
        linewidth=0.4,
        axis="y",
    )

    secondary_axis = axis.twinx()

    def density_to_counts(density_value):
        return (
            np.asarray(density_value)
            * sample_count
            * bin_width
        )

    minimum_density, maximum_density = axis.get_ylim()
    secondary_axis.set_ylim(
        density_to_counts(minimum_density),
        density_to_counts(maximum_density),
    )
    secondary_axis.set_ylabel(
        f"Approx. PS points per {bin_width:.02f} bin",
        fontsize=KDE_AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )
    secondary_axis.tick_params(
        axis="y",
        labelsize=KDE_TICK_FONTSIZE,
    )
    for tick_label in secondary_axis.get_yticklabels():
        tick_label.set_fontweight("bold")

    figure.tight_layout(pad=1.2)

    if SAVE_FIGURES:
        figure.savefig(
            OUT_KDE_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )
        figure.savefig(
            OUT_KDE_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )
        figure.savefig(
            OUT_KDE_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )

        log(f"[KDE] {OUT_KDE_PNG}")
        log(f"[KDE] {OUT_KDE_PDF}")
        log(f"[KDE] {OUT_KDE_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# MAIN
# =========================================================
def main() -> None:
    log("=== Step 3.1 coherence visualization ===")

    require_file(CSV_INPUT, "PS coherence CSV")

    dataframe = pd.read_csv(CSV_INPUT)

    latitude_column = find_column(
        list(dataframe.columns),
        "lat",
    )
    longitude_column = find_column(
        list(dataframe.columns),
        "lon",
    )
    coherence_column = find_column(
        list(dataframe.columns),
        "coh",
    )

    if not all(
        [
            latitude_column,
            longitude_column,
            coherence_column,
        ]
    ):
        raise ValueError(
            "The CSV must contain latitude, longitude and coherence columns."
        )

    dataframe = dataframe[
        [
            latitude_column,
            longitude_column,
            coherence_column,
        ]
    ].copy()

    for column in (
        latitude_column,
        longitude_column,
        coherence_column,
    ):
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        )

    dataframe = dataframe.dropna()

    coherence_values = dataframe[
        coherence_column
    ].to_numpy(dtype="float64")

    out_of_range = (
        (coherence_values < -1.0e-6)
        | (coherence_values > 1.0 + 1.0e-6)
    )

    if out_of_range.any():
        raise ValueError(
            f"{int(out_of_range.sum()):,} coherence values lie "
            "outside [0,1]."
        )

    dataframe[coherence_column] = np.clip(
        coherence_values,
        0.0,
        1.0,
    )

    dataframe = clip_points_to_mask(
        dataframe,
        longitude_column,
        latitude_column,
        STUDY_MASK_TIF,
    )

    boundary_data = load_vector(
        STUDY_SHP,
        target_crs="EPSG:4326",
    )
    roads = load_vector(
        ROAD_SHP,
        target_crs="EPSG:4326",
    )

    polygon = get_union_polygon(boundary_data)

    dataframe = clip_points_to_polygon(
        dataframe,
        longitude_column,
        latitude_column,
        polygon,
    )

    if dataframe.empty:
        raise RuntimeError(
            "No PS points remain after study-area clipping."
        )

    longitude_min = float(
        dataframe[longitude_column].min()
    )
    longitude_max = float(
        dataframe[longitude_column].max()
    )
    latitude_min = float(
        dataframe[latitude_column].min()
    )
    latitude_max = float(
        dataframe[latitude_column].max()
    )

    longitude_buffer = (
        longitude_max - longitude_min
    ) * DISPLAY_BUFFER_FRACTION
    latitude_buffer = (
        latitude_max - latitude_min
    ) * DISPLAY_BUFFER_FRACTION

    # Safeguard for extremely narrow coordinate ranges.
    longitude_buffer = max(longitude_buffer, 1.0e-5)
    latitude_buffer = max(latitude_buffer, 1.0e-5)

    west = longitude_min - longitude_buffer
    east = longitude_max + longitude_buffer
    south = latitude_min - latitude_buffer
    north = latitude_max + latitude_buffer

    longitude_grid_values = np.linspace(
        west,
        east,
        GRID_RESOLUTION,
    )
    latitude_grid_values = np.linspace(
        south,
        north,
        GRID_RESOLUTION,
    )
    longitude_grid, latitude_grid = np.meshgrid(
        longitude_grid_values,
        latitude_grid_values,
    )

    point_coordinates = np.column_stack(
        [
            dataframe[longitude_column].to_numpy(
                dtype="float64"
            ),
            dataframe[latitude_column].to_numpy(
                dtype="float64"
            ),
        ]
    )
    values = dataframe[coherence_column].to_numpy(
        dtype="float64"
    )

    surface = interpolate_knn_idw(
        point_coordinates=point_coordinates,
        values=values,
        longitude_grid=longitude_grid,
        latitude_grid=latitude_grid,
        neighbours=KNN_IDW_K,
        power=IDW_POWER,
        maximum_distance=MAX_DISTANCE_DEG,
    )

    surface = mask_grid_with_raster(
        surface,
        longitude_grid,
        latitude_grid,
        STUDY_MASK_TIF,
    )

    surface = mask_grid_to_polygon(
        surface,
        longitude_grid,
        latitude_grid,
        polygon,
    )

    statistics = calculate_statistics(values)

    log(
        f"[POINTS] n={statistics['n']:,} | "
        f"min={statistics['min']:.3f} | "
        f"median={statistics['median']:.3f} | "
        f"mean={statistics['mean']:.3f} | "
        f"p95={statistics['p95']:.3f} | "
        f"max={statistics['max']:.3f}"
    )
    log(
        f"[SURFACE] KNN-IDW k={KNN_IDW_K} | "
        f"power={IDW_POWER} | max distance={MAX_DISTANCE_DEG}°"
    )
    log(
        "[STYLE] 2.5×5 inch map | four longitude DMS labels | "
        "three latitude DMS labels (rotated 90°, bottom label "
        "lifted) | no axis titles | 900 dpi | PNG/PDF/SVG"
    )

    create_coherence_map(
        dataframe=dataframe,
        longitude_column=longitude_column,
        latitude_column=latitude_column,
        coherence_column=coherence_column,
        surface=surface,
        west=west,
        east=east,
        south=south,
        north=north,
        boundary_data=boundary_data,
        roads=roads,
    )

    create_coherence_histogram(
        values=values,
        statistics=statistics,
    )

    create_coherence_kde(values)

    summary = {
        "generated_at_utc": utc_now_iso(),
        "input": str(CSV_INPUT),
        "n_PS_points": int(values.size),
        "statistics": statistics,
        "map_surface": {
            "method": "k-nearest-neighbour inverse-distance weighting",
            "k": KNN_IDW_K,
            "power": IDW_POWER,
            "maximum_distance_degrees": MAX_DISTANCE_DEG,
            "grid_resolution": [
                GRID_RESOLUTION,
                GRID_RESOLUTION,
            ],
            "imshow_interpolation": "nearest",
            "PS_points_overlaid": SHOW_PS_POINTS,
        },
        "visual_style": {
            "map_figsize_inches": list(MAP_FIGSIZE),
            "histogram_figsize_inches": list(HIST_FIGSIZE),
            "kde_figsize_inches": list(KDE_FIGSIZE),
            "colormap": "jet",
            "axis_titles_shown": False,
            "longitude_DMS_tick_count": (
                LONGITUDE_DMS_TICK_COUNT
            ),
            "latitude_DMS_tick_count": (
                LATITUDE_DMS_TICK_COUNT
            ),
            "latitude_DMS_label_rotation_deg": 90,
            "bottom_latitude_label_va": "bottom",
            "DMS_tick_fontsize": DMS_TICK_FONTSIZE,
            "histogram_axis_label_fontsize": (
                HIST_AXIS_LABEL_FONTSIZE
            ),
            "histogram_tick_fontsize": (
                HIST_TICK_FONTSIZE
            ),
            "figure_dpi": FIG_DPI,
        },
        "outputs": {
            "map_png": str(OUT_MAP_PNG),
            "map_pdf": str(OUT_MAP_PDF),
            "map_svg": str(OUT_MAP_SVG),
            "histogram_png": str(OUT_HIST_PNG),
            "histogram_pdf": str(OUT_HIST_PDF),
            "histogram_svg": str(OUT_HIST_SVG),
            "kde_png": str(OUT_KDE_PNG),
            "kde_pdf": str(OUT_KDE_PDF),
            "kde_svg": str(OUT_KDE_SVG),
        },
    }

    with open(
        OUT_SUMMARY_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False,
        )

    log(f"[SUMMARY] {OUT_SUMMARY_JSON}")
    log("=== Step 3.1 coherence visualization completed ===")


def create_coherence_visualizations() -> None:
    try:
        main()
    except Exception as exc:
        print(
            f"[ERROR] Failed to create coherence visualizations: {exc}"
        )
        traceback.print_exc()
        raise


if __name__ == "__main__":
    create_coherence_visualizations()

# Step-3.2: LOS Velocity

In [ ]:
# =========================================================
# Step 3.2 — LOS velocity from StaMPS MAT (PS-InSAR)
# =========================================================
import os, struct, json
import numpy as np
import pandas as pd

# --- MAT readers (v7 & v7.3) ---
import scipy.io as sio
try:
    import h5py
    _HAVE_H5 = True
except Exception:
    _HAVE_H5 = False

# --- Rasterization / reprojection ---
try:
    import rasterio
    from rasterio.warp import transform as rio_transform
    from rasterio.warp import reproject, Resampling
    from pyproj import CRS, Transformer
    _HAVE_RASTER = True
except Exception:
    _HAVE_RASTER = False

from datetime import datetime

def pp(msg): print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

# -----------------------PATH-----------------------

BASE = os.environ.get("LSM_DATA_DIR", "./data")
IN_DIR  = os.path.join(BASE, "NTD_PS_InSAR_data")
OUT_DIR = os.path.join(BASE, "Results", "Step-3.2-LOS"); os.makedirs(OUT_DIR, exist_ok=True)

ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
MASK_TIF   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

# expected files
COORD_MAT = os.path.join(IN_DIR, "coordinates_clipped.mat")
VEL_MAT   = os.path.join(IN_DIR, "mean_v_clipped.mat")
COH_MAT   = os.path.join(IN_DIR, "coherence_clipped.mat")  

# outputs
CSV_OUT = os.path.join(OUT_DIR, "ps_points_los_velocity.csv")
VEL_TIF = os.path.join(OUT_DIR, "los_velocity_mean_mm_yr_32645_12p5m_float32.tif")
COUNT_TIF = os.path.join(OUT_DIR, "los_velocity_count_32645_12p5m_int16.tif")

# Parameters / thresholds
COH_MIN = None          # e.g., 0.3 to keep PS with coherence >= 0.3; None to disable
VEL_ABS_CLIP_MM_YR = 300.0  # clip points with |vel| > this (mm/yr); set None to disable
NODATA = -9999.0

# Coordinate hints
COORDS_DEFAULT_EPSG = 32645   # if coordinates look projected (not lon/lat)
ANCHOR_FORCE_EPSG   = 32645   # anchor grid CRS we expect

# =========================================================
# MAT loading & value extraction
# =========================================================
def load_mat_any(path):
    """
    Load MATLAB .mat from v7 (sio.loadmat) or v7.3 (h5py) into a plain dict of numpy arrays.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    try:
        md = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
        # remove meta keys
        return {k: v for k, v in md.items() if not k.startswith('__')}
    except Exception:
        if not _HAVE_H5:
            raise
        # v7.3 via h5py
        out = {}
        with h5py.File(path, 'r') as f:
            def to_np(obj):
                a = np.array(obj)
                # MATLAB stores column-major; flatten safely
                return np.array(a)
            def visit(name, obj):
                if isinstance(obj, h5py.Dataset):
                    key = name.split('/')[-1]
                    out[key] = to_np(obj)
            f.visititems(lambda name, obj: visit(name, obj))
        return out

def _find_two_col_array(d):
    # Look for typical keys
    pref = ["lonlat", "ll", "lon_lat", "coords", "coord", "xy", "pos", "posxy"]
    for k in list(d.keys()):
        a = np.asarray(d[k])
        if a.ndim == 2 and (a.shape[0] == 2 or a.shape[1] == 2):
            return a
    for k in pref:
        if k in d:
            a = np.asarray(d[k])
            if a.ndim == 2 and (a.shape[0] == 2 or a.shape[1] == 2):
                return a
    # None found
    return None

def _guess_coords(matdict):
    """
    Return (x, y, epsg_hint). If x,y look like lon/lat -> EPSG:4326, else COORDS_DEFAULT_EPSG.
    Accepts arrays shaped (N,2) or (2,N).
    """
    a = _find_two_col_array(matdict)
    if a is not None:
        if a.shape[0] == 2 and a.shape[1] != 2:
            x = np.asarray(a[0, :]).astype("float64").ravel()
            y = np.asarray(a[1, :]).astype("float64").ravel()
        else:
            x = np.asarray(a[:, 0]).astype("float64").ravel()
            y = np.asarray(a[:, 1]).astype("float64").ravel()
    else:
        # try lon + lat separately
        cand_lon = [k for k in matdict.keys() if k.lower() in ("lon","long","longitude","x","e","east","easting")]
        cand_lat = [k for k in matdict.keys() if k.lower() in ("lat","latitude","y","n","north","northing")]
        if cand_lon and cand_lat:
            lon = np.asarray(matdict[cand_lon[0]]).astype("float64").ravel()
            lat = np.asarray(matdict[cand_lat[0]]).astype("float64").ravel()
            x, y = lon, lat
        else:
            raise RuntimeError("Could not find coordinate arrays (lon/lat or x/y).")

    # heuristic: if within lon/lat ranges -> WGS84
    if np.all((x >= -180) & (x <= 180)) and np.all((y >= -90) & (y <= 90)):
        return x, y, 4326
    # looks projected
    return x, y, COORDS_DEFAULT_EPSG

def _guess_vector(matdict, names):
    """Find a 1-D vector by preferred names. Returns 1D float array."""
    # try exact keys first
    for k in names:
        if k in matdict:
            a = np.asarray(matdict[k]).astype("float64").ravel()
            if a.ndim == 1:
                return a
    # then "contains" search
    for k, v in matdict.items():
        lk = k.lower()
        if any(n in lk for n in names):
            a = np.asarray(v).astype("float64").ravel()
            if a.ndim == 1:
                return a
    # as last resort, first 1-D vector in dict
    for k, v in matdict.items():
        a = np.asarray(v).astype("float64").ravel()
        if a.ndim == 1 and a.size > 1:
            return a
    raise RuntimeError(f"Could not find a 1-D vector among names={names}")

# =========================================================
# Build PS table (x,y, vel, coh)
# =========================================================
def build_ps_table():
    pp("Reading MAT files …")
    Dcoord = load_mat_any(COORD_MAT)
    Dvel   = load_mat_any(VEL_MAT)
    Dcoh   = load_mat_any(COH_MAT) if os.path.exists(COH_MAT) else {}

    x, y, epsg_hint = _guess_coords(Dcoord)
    vel = _guess_vector(Dvel, names=["mean_v", "vel", "velocity", "v_los", "v"])
    coh = None
    if Dcoh:
        # typical names: coh_ps, coherence
        try:
            coh = _guess_vector(Dcoh, names=["coh_ps", "coherence", "gamma", "coh"])
        except Exception:
            coh = None

    # sanity length
    N = len(x)
    if len(vel) != N:
        raise RuntimeError(f"Length mismatch: coords={N} vs vel={len(vel)}")
    if coh is not None and len(coh) != N:
        pp(f"WARNING: coherence length {len(coh)} != coords {N}. Ignoring coherence.")
        coh = None

    df = pd.DataFrame({"x": x, "y": y, "vel_mm_yr": vel})
    if coh is not None:
        df["coh"] = coh

    # Basic cleaning
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    if "coh" in df.columns:
        df = df[df["coh"].between(0, 1) | df["coh"].isna()]
    if COH_MIN is not None and "coh" in df.columns:
        df = df[(df["coh"].isna()) | (df["coh"] >= COH_MIN)]
    if VEL_ABS_CLIP_MM_YR is not None:
        df = df[np.abs(df["vel_mm_yr"]) <= VEL_ABS_CLIP_MM_YR]

    df = df.dropna(subset=["x", "y", "vel_mm_yr"]).reset_index(drop=True)
    pp(f"Loaded {len(df)} PS points. Coord EPSG hint: {epsg_hint}")
    return df, int(epsg_hint)

# =========================================================
# Raster helpers
# =========================================================
def _load_anchor():
    if not _HAVE_RASTER:
        return None
    if not os.path.exists(ANCHOR_TIF):
        raise FileNotFoundError(ANCHOR_TIF)
    with rasterio.open(ANCHOR_TIF) as ds:
        return {
            "crs": ds.crs,
            "epsg": int(ds.crs.to_epsg()) if ds.crs and ds.crs.to_epsg() else None,
            "tf": ds.transform,
            "w": ds.width,
            "h": ds.height
        }

def _reproject_points(df, src_epsg, dst_epsg):
    if src_epsg == dst_epsg:
        return df["x"].values, df["y"].values
    transformer = Transformer.from_crs(CRS.from_epsg(src_epsg), CRS.from_epsg(dst_epsg), always_xy=True)
    X, Y = transformer.transform(df["x"].values, df["y"].values)
    return np.asarray(X), np.asarray(Y)

def rasterize_mean_velocity(df, anchor, out_mean, out_count, nodata=NODATA):
    """Bin points into anchor pixels: mean velocity + count."""
    tf = anchor["tf"]; w = anchor["w"]; h = anchor["h"]
    # map to pixel indices
    # col = (x - x0)/a ; row = (y - y0)/e  for affine tf
    A = np.array([[tf.a, tf.b], [tf.d, tf.e]])  # scale/skew
    b = np.array([tf.c, tf.f])
    Ai = np.linalg.inv(A)

    xy = np.vstack([df["X_anchor"].values, df["Y_anchor"].values]).T
    ij = (xy - b) @ Ai.T
    cols = np.floor(ij[:, 0]).astype(int)
    rows = np.floor(ij[:, 1]).astype(int)

    # keep in-bounds
    m = (cols >= 0) & (cols < w) & (rows >= 0) & (rows < h)
    if m.sum() == 0:
        raise RuntimeError("No points fell inside the anchor grid.")
    cols = cols[m]; rows = rows[m]
    vals = df["vel_mm_yr"].values[m]

    # accumulate sum & count
    sum_img = np.zeros((h, w), dtype="float64")
    cnt_img = np.zeros((h, w), dtype="int32")

    # vectorized bincount by ravel index
    idx = rows * w + cols
    sum_acc = np.bincount(idx, weights=vals, minlength=h*w).reshape(h, w)
    cnt_acc = np.bincount(idx, minlength=h*w).reshape(h, w)

    sum_img += sum_acc
    cnt_img += cnt_acc

    mean_img = np.full((h, w), nodata, dtype="float32")
    mpos = cnt_img > 0
    mean_img[mpos] = (sum_img[mpos] / cnt_img[mpos]).astype("float32")

    # write rasters
    profile = {
        "driver": "GTiff", "height": h, "width": w, "count": 1,
        "dtype": "float32", "crs": anchor["crs"], "transform": tf,
        "compress": "LZW", "tiled": True, "nodata": nodata
    }
    with rasterio.open(out_mean, "w", **profile) as ds:
        ds.write(mean_img, 1)

    prof_cnt = profile.copy()
    prof_cnt["dtype"] = "int16"; prof_cnt["nodata"] = 0
    with rasterio.open(out_count, "w", **prof_cnt) as ds:
        ds.write(np.clip(cnt_img, 0, np.iinfo(np.int16).max).astype("int16"), 1)

    # quick QA
    if mpos.any():
        v = mean_img[mpos]
        pp(f"[QA] mean raster: N={int(mpos.sum())}  min/mean/max = {v.min():.2f}/{v.mean():.2f}/{v.max():.2f} mm/yr")
    pp(f"[OUT] {out_mean}")
    pp(f"[OUT] {out_count}")

# =========================================================
# MAIN
# =========================================================
def main():
    pp("=== Step 3.2 — LOS velocity from StaMPS MAT ===")

    # 1) Read points (and filter)
    df, src_epsg = build_ps_table()

    # Save raw point CSV (source CRS columns)
    df_points = df.copy()
    df_points["src_epsg"] = src_epsg
    df_points.to_csv(CSV_OUT, index=False)
    pp(f"[CSV] {CSV_OUT}  (rows={len(df_points)})")

    # 2) Rasterize to anchor (if available)
    if not _HAVE_RASTER:
        pp("[RASTER] rasterio/pyproj missing — skipping rasterization.")
        pp("=== Done (Step 3.2) ===")
        return

    if not os.path.exists(ANCHOR_TIF):
        pp(f"[RASTER] Anchor not found: {ANCHOR_TIF} — skipping rasterization.")
        pp("=== Done (Step 3.2) ===")
        return

    anchor = _load_anchor()
    if anchor["epsg"] != ANCHOR_FORCE_EPSG:
        pp(f"[WARN] Anchor EPSG is {anchor['epsg']} (expected {ANCHOR_FORCE_EPSG}). Continuing.")

    # Reproject points to anchor CRS
    X, Y = _reproject_points(df, src_epsg, anchor["epsg"])
    df["X_anchor"] = X; df["Y_anchor"] = Y

    # Optional mask: drop points outside mask (fast bounding)
    if os.path.exists(MASK_TIF):
        with rasterio.open(MASK_TIF) as ms:
            # quick test: sample the mask at point pixels & keep >0
            tf = anchor["tf"]; w = anchor["w"]; h = anchor["h"]
            A = np.array([[tf.a, tf.b], [tf.d, tf.e]]); b = np.array([tf.c, tf.f])
            Ai = np.linalg.inv(A)
            xy = np.vstack([X, Y]).T
            ij = (xy - b) @ Ai.T
            cols = np.floor(ij[:, 0]).astype(int)
            rows = np.floor(ij[:, 1]).astype(int)
            m_in = (cols >= 0) & (cols < w) & (rows >= 0) & (rows < h)
            mask_band = ms.read(1)
            keep = np.zeros(len(df), dtype=bool)
            keep[m_in] = mask_band[rows[m_in], cols[m_in]] > 0
            kept = int(keep.sum())
            if kept > 0:
                df = df[keep].reset_index(drop=True)
                pp(f"[MASK] kept {kept} points inside study mask.")
            else:
                pp("[MASK] mask removed all points (unexpected). Proceeding without mask filtering.")

    # Rasterize mean LOS velocity + count
    rasterize_mean_velocity(df, anchor, VEL_TIF, COUNT_TIF, nodata=NODATA)
    pp("=== Done (Step 3.2) ===")

if __name__ == "__main__":
    main()


# Visualization: LOS Velocity

In [ ]:
# =========================================================
# PS-InSAR LOS Velocity Visualization  
# =========================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.patches import Polygon, Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
import traceback

# Optional libs 
try:
    from scipy.spatial import cKDTree
    from scipy.interpolate import griddata, RBFInterpolator
    from scipy.ndimage import binary_erosion
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
    print("[WARN] SciPy not available; falling back to simpler paths")

try:
    import geopandas as gpd
    from shapely.geometry import Point
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

# For polygon clip-paths 
try:
    from shapely.geometry import shape as shp_shape, Polygon as ShpPolygon, MultiPolygon as ShpMultiPolygon
    from shapely.ops import unary_union
    HAVE_SHAPELY = True
except Exception:
    HAVE_SHAPELY = False

from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch

import rasterio
from rasterio.warp import transform as rio_transform, transform_bounds, transform_geom
from rasterio.transform import rowcol
from rasterio import features as rio_features

# ---------------- USER PATHS ----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
MASK_TIF  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
CSV_PATH  = os.path.join(BASE, r"Results\Step-3.2-LOS\ps_points_los_velocity.csv")
SHAPEFILE = os.path.join(BASE, "Study_Area.shp")
ROAD_SHP  = os.path.join(BASE, "NTD_Road.shp")
OUT_DIR   = os.path.join(BASE, r"Results\Step-3.2-LOS")

# ---------------- Display & scaling options ----------------
GRID_RES             = 200
USE_RBF              = True
IDW_NEIGHBORS        = 64

# Fixed velocity range (mm/year)
VMIN_FIXED = -18.90
VMAX_FIXED =  18.90

# ---------------- Figure sizes  ----------------
MAP_FIGSIZE  = (2, 2)
HIST_FIGSIZE = (2, 2)
CAX_SIZE     = "5%"
CAX_PAD      = 0.01

# =========================================================
# Helpers: CRS, DMS, decorations
# =========================================================
def detect_crs(xvals, yvals):
    x_min, x_max = float(np.nanmin(xvals)), float(np.nanmax(xvals))
    y_min, y_max = float(np.nanmin(yvals)), float(np.nanmax(yvals))
    if (-180.0 <= x_min <= 180.0) and (-180.0 <= x_max <= 180.0) and (-90.0 <= y_min <= 90.0) and (-90.0 <= y_max <= 90.0):
        return 'wgs84'
    return 'utm'

def dms_label(dec, is_lon=True):
    a = abs(float(dec))
    d = int(a)
    rem_min = (a - d) * 60.0
    m = int(rem_min)
    s = int(round((rem_min - m) * 60.0))  # integer seconds

    # carry if rounding bumps to 60s or 60m
    if s == 60:
        s = 0
        m += 1
    if m == 60:
        m = 0
        d += 1

    hemi = ('E' if dec >= 0 else 'W') if is_lon else ('N' if dec >= 0 else 'S')
    return f"{d}°{m:02d}′{s:02d}″{hemi}"

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target: return m * base
    return 10 * base

def add_north_arrow_tri(ax, size=0.09, pad=0.09, label='N', arrow_downshift=0.025, label_upshift=0.0):
    """Triangle arrow moved slightly down, 'N' label kept at original (non-overlapping) position."""
    # label anchor (fixed near original spot so it doesn’t overlap arrow)
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10 + label_upshift

    # Arrow base shifted downward
    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size * 0.09)

    # Left white half
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    # Right black half
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    # Text 'N' at original position
    ax.text(label_anchor_x, label_anchor_y, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=8, fontweight='bold',
            color='black', zorder=7)

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None, segments=4, fontsize=6,
                          tile_lw=0.3, frame_lw=0.4):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd

    total_len_m = nice_len_125(width_m / 5.0); seg_len_m = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data
    pad_x   = total_len_units * 0.08; pad_y = 0.035 * height_data

    # soft white pad behind the bar
    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))

    # tiles (no edge lines -> fill only; cleaner)
    for i in range(segments):
        xs = x_start + i * seg_len_units
        face = 'black' if i % 2 == 0 else 'white'
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData, facecolor=face,
                               edgecolor='none', zorder=4))

    # thin outer frame
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=frame_lw, zorder=5))

    # labels
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.012 * height_data, "km",
            ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.012 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.012 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Boundary helpers + clipping 
# =========================================================
def draw_boundary_from_mask(ax, mask_tif_path, zorder=999, color="black", linewidth=0.7):
    if not os.path.exists(mask_tif_path):
        return False
    try:
        with rasterio.open(mask_tif_path) as ms:
            mask = ms.read(1)
            w, s, e, n = transform_bounds(ms.crs, "EPSG:4326", *ms.bounds)
        ax.contour(mask, levels=[0.5], colors=color, linewidths=linewidth,
                   extent=[w, e, s, n], origin='upper', zorder=zorder)
        print("[OK] Boundary drawn from mask")
        return True
    except Exception as ex:
        print(f"[WARN] Could not draw boundary from mask: {ex}")
        return False

def draw_boundary_from_shp(ax, shp_path, zorder=998, color="black", linewidth=0.7):
    if not (HAVE_GPD and os.path.exists(shp_path)): return False
    try:
        gdf = gpd.read_file(shp_path).to_crs(4326)
        gdf.boundary.plot(ax=ax, color=color, linewidth=linewidth, alpha=1.0, zorder=zorder)
        print("[OK] Boundary drawn from shapefile")
        return True
    except Exception as ex:
        print(f"[WARN] Could not draw boundary from shapefile: {ex}")
        return False

# ====== Highway helpers  ======
def load_roads(road_shp_path, target_crs_epsg=4326):
    if not os.path.exists(road_shp_path):
        print(f"[WARN] Road shapefile not found: {road_shp_path}")
        return None
    print(f"[INFO] Loading road shapefile: {road_shp_path}")
    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_shp_path)
            if roads.crs is None:
                print("[WARN] Road CRS is None; assuming EPSG:4326")
                roads.set_crs(epsg=4326, inplace=True)
            if str(roads.crs) != f"EPSG:{target_crs_epsg}":
                roads = roads.to_crs(epsg=target_crs_epsg)
            print(f"[OK] Roads loaded with {len(roads)} features; CRS={roads.crs}")
            return roads
        except Exception as e:
            print(f"[WARN] GeoPandas failed to load roads: {e}")
    if HAVE_FIONA:
        try:
            geoms = []
            with fiona.open(road_shp_path, 'r') as shp:
                src_crs = shp.crs
                for feat in shp:
                    geom = feat['geometry']
                    if geom is None:
                        continue
                    try:
                        geom = transform_geom(src_crs, f"EPSG:{target_crs_epsg}", geom)
                    except Exception as e:
                        print(f"[WARN] Road transform failed (keeping original): {e}")
                    geoms.append(geom)
            print(f"[OK] Roads loaded via Fiona with {len(geoms)} features")
            return {'geometries': geoms, 'method': 'fiona'}
        except Exception as e:
            print(f"[WARN] Fiona failed to load roads: {e}")
    print("[WARN] Unable to load roads shapefile with available backends")
    return None

def plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850):
    if roads is None:
        return
    try:
        if HAVE_GPD and hasattr(roads, 'plot'):
            try:
                roads.geometry = roads.geometry.apply(
                    lambda g: g if g is None or g.geom_type in ("LineString", "MultiLineString") else None
                )
            except Exception:
                pass
            roads.plot(ax=ax, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (GeoPandas)")
        elif isinstance(roads, dict) and 'geometries' in roads:
            for geom in roads['geometries']:
                if geom is None:
                    continue
                if geom['type'] == 'LineString':
                    xs = [p[0] for p in geom['coordinates']]
                    ys = [p[1] for p in geom['coordinates']]
                    ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
                elif geom['type'] == 'MultiLineString':
                    for line in geom['coordinates']:
                        xs = [p[0] for p in line]
                        ys = [p[1] for p in line]
                        ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (Fiona fallback)")
    except Exception as e:
        print(f"[WARN] Failed to plot roads: {e}")

# =========================================================
# Study polygon & clip-path helpers
# =========================================================
def _mask_to_polygons_4326(mask_tif_path):
    """Extract boundary polygon(s) from mask tif; return shapely MultiPolygon/Polygon in EPSG:4326."""
    if not HAVE_SHAPELY:
        print("[WARN] Shapely not available; cannot polygonize mask")
        return None
    with rasterio.open(mask_tif_path) as ds:
        data = ds.read(1)
        trans = ds.transform
        crs = ds.crs
        polys = []
        for geom, val in rio_features.shapes(data.astype('uint8'), mask=(data > 0), transform=trans):
            if val == 1:
                g = shp_shape(geom)
                # to EPSG:4326 if needed
                if crs and hasattr(crs, "to_string") and '4326' not in crs.to_string():
                    if HAVE_GPD:
                        g = gpd.GeoSeries([g], crs=crs).to_crs(4326).geometry.iloc[0]
                polys.append(g)
        if not polys:
            return None
        u = unary_union(polys)
        return u

def _shp_to_polygon_4326(shp_path):
    """Load boundary polygon from shapefile and ensure EPSG:4326."""
    if not HAVE_SHAPELY:
        print("[WARN] Shapely not available; cannot load polygon")
        return None
    if HAVE_GPD:
        g = gpd.read_file(shp_path)
        if g.empty:
            return None
        if g.crs is None:
            g.set_crs(epsg=4326, inplace=True)
        g = g.to_crs(4326)
        u = unary_union(g.geometry)
        return u
    else:
        # Fiona fallback to raw geometries 
        try:
            with fiona.open(shp_path) as src:
                crs = src.crs
                geoms = []
                for ft in src:
                    if ft['geometry'] is None: continue
                    geoms.append(shp_shape(ft['geometry']))
                if not geoms: return None
                u = unary_union(geoms)
                # Try to transform to 4326 if CRS present and not 4326
                if crs and crs.get('init', '').lower() != 'epsg:4326':
                    if HAVE_GPD:
                        u = gpd.GeoSeries([u], crs=crs).to_crs(4326).geometry.iloc[0]
                return u
        except Exception:
            return None

def get_study_polygon_4326(mask_tif_path, shp_path):
    """Prefer shapefile (if polygon); else fall back to mask raster polygonization."""
    poly = None
    if shp_path and os.path.exists(shp_path):
        try:
            poly = _shp_to_polygon_4326(shp_path)
        except Exception as e:
            print(f"[WARN] Failed to get polygon from shapefile: {e}")
    if (poly is None) and (mask_tif_path and os.path.exists(mask_tif_path)):
        try:
            poly = _mask_to_polygons_4326(mask_tif_path)
        except Exception as e:
            print(f"[WARN] Failed to polygonize mask: {e}")
    if poly is None:
        print("[WARN] Study polygon unavailable; clip-path will be skipped")
    return poly

def polygon_to_pathpatch(ax, poly, facecolor='none', edgecolor='none'):
    if not HAVE_SHAPELY:
        return None

    def _ring_to_vertices(ring):
        xs, ys = ring.coords.xy
        verts = list(zip(xs, ys))
        # Ensure closure
        if verts[0] != verts[-1]:
            verts.append(verts[0])
        codes = [MplPath.MOVETO] + [MplPath.LINETO]*(len(verts)-2) + [MplPath.CLOSEPOLY]
        return verts, codes

    if isinstance(poly, ShpPolygon):
        polys = [poly]
    else:
        polys = list(poly.geoms)  
    vertices = []; codes = []
    for p in polys:
        v, c = _ring_to_vertices(p.exterior); vertices += v; codes += c
        for hole in p.interiors:
            v, c = _ring_to_vertices(hole); vertices += v; codes += c
    path = MplPath(vertices, codes)
    patch = PathPatch(path, transform=ax.transData, facecolor=facecolor, edgecolor=edgecolor)
    return patch

def clip_roads_to_polygon(roads, poly_4326):
    if (roads is None) or (poly_4326 is None) or (not HAVE_SHAPELY):
        return roads
    try:
        if HAVE_GPD and hasattr(roads, 'geometry'):
            # Ensure CRS matches 4326
            if roads.crs is None:
                roads = roads.set_crs(epsg=4326)
            elif str(roads.crs) != "EPSG:4326":
                roads = roads.to_crs(4326)
            poly_gdf = gpd.GeoDataFrame(geometry=[poly_4326], crs=roads.crs)
            inter = gpd.overlay(roads, poly_gdf, how='intersection')
            return inter
        elif isinstance(roads, dict) and 'geometries' in roads:
            clipped = []
            for geom in roads['geometries']:
                try:
                    shp = shp_shape(geom)
                    inter = shp.intersection(poly_4326)
                    if not inter.is_empty and inter.geom_type in ("LineString", "MultiLineString"):
                        clipped.append(inter.__geo_interface__)
                except Exception:
                    pass
            return {'geometries': clipped, 'method': roads.get('method', 'fiona')}
    except Exception as e:
        print(f"[WARN] Road clipping failed, plotting original roads: {e}")
    return roads

# =========================================================
# Clipping to mask 
# =========================================================
def clip_points_to_mask_any_crs(df, x_col, y_col, mask_tif_path, crs_type):
    if not os.path.exists(mask_tif_path):
        print(f("[WARN] Mask TIFF missing: {mask_tif_path}"))
        return df
    try:
        with rasterio.open(mask_tif_path) as ms:
            mask = ms.read(1)
            mask_crs = ms.crs
            mask_transform = ms.transform
            mask_crs_str = mask_crs.to_string() if hasattr(mask_crs, 'to_string') else str(mask_crs)

            X = df[x_col].values; Y = df[y_col].values
            if crs_type == 'wgs84' and ('4326' not in mask_crs_str):
                xs, ys = rio_transform('EPSG:4326', mask_crs, X, Y)
                xs = np.array(xs); ys = np.array(ys)
            else:
                xs, ys = X, Y

            rows, cols = rowcol(mask_transform, xs, ys)
            rows = np.array(rows); cols = np.array(cols)

            inside = (rows >= 0) & (rows < mask.shape[0]) & (cols >= 0) & (cols < mask.shape[1])
            keep = np.zeros(len(df), dtype=bool)
            keep[inside] = (mask[rows[inside], cols[inside]] > 0)

            kept = df[keep].copy()
            removed = len(df) - len(kept)
            pct = (removed/len(df)*100) if len(df)>0 else 0.0
            print(f"[INFO] Mask clipping removed {removed} points ({pct:.1f}%)")
            if len(kept) == 0:
                print("[WARN] All points clipped by mask; returning original to avoid empty set")
                return df
            return kept
    except Exception as ex:
        print(f"[WARN] clip_points_to_mask_any_crs failed: {ex}")
        traceback.print_exc()
        return df

# =========================================================
# Interpolation 
# =========================================================
def interpolate_grid(x, y, v, west, east, south, north, grid_res=200):
    gx = np.linspace(west,  east,  grid_res)
    gy = np.linspace(south, north, grid_res)
    Xg, Yg = np.meshgrid(gx, gy)
    pts = np.column_stack([x, y])

    if HAVE_SCIPY:
        grid_pts = np.column_stack([Xg.ravel(), Yg.ravel()])
        try:
            if USE_RBF:
                rbf = RBFInterpolator(pts, v, kernel='thin_plate_spline', smoothing=0.1, epsilon=0.1)
                Vg = rbf(grid_pts).reshape(grid_res, grid_res)
            else:
                Vg = griddata(pts, v, grid_pts, method='cubic', fill_value=np.nan).reshape(grid_res, grid_res)
            tree = cKDTree(pts)
            distances, _ = tree.query(grid_pts, k=1)
            Vg[distances.reshape(grid_res, grid_res) > ((east - west) * 0.01)] = np.nan
            return Xg, Yg, Vg
        except Exception as e:
            print(f"[WARN] SciPy interpolation failed ({e}); falling back to IDW...")

    if HAVE_SCIPY:
        tree = cKDTree(pts)
        grid_pts = np.column_stack([Xg.ravel(), Yg.ravel()])
        k = min(IDW_NEIGHBORS, len(pts))
        dist, idx = tree.query(grid_pts, k=k)
        w = 1.0 / (dist + 1e-12)
        vv = v[idx] if k > 1 else v[idx][:, None]
        V_flat = np.sum(w * vv, axis=1) / np.sum(w, axis=1)
        Vg = V_flat.reshape(grid_res, grid_res)
        return Xg, Yg, Vg

    Vg = np.zeros((grid_res, grid_res), dtype=float)
    for i in range(grid_res):
        Xi = Xg[i]; Yi = Yg[i]
        for j in range(grid_res):
            d = np.hypot(pts[:, 0] - Xi[j], pts[:, 1] - Yi[j])
            w = 1.0 / (d + 1e-12)
            Vg[i, j] = np.sum(w * v) / np.sum(w)
    return Xg, Yg, Vg

# =========================================================
# Main
# =========================================================
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

    # Load & detect columns
    df = pd.read_csv(CSV_PATH)
    x_col = y_col = vel_col = None
    for col in df.columns:
        c = col.lower()
        if 'x_anchor' in c and x_col is None: x_col = col
        elif 'y_anchor' in c and y_col is None: y_col = col
        elif c in ('x', 'lon', 'longitude') and x_col is None: x_col = col
        elif c in ('y', 'lat', 'latitude') and y_col is None: y_col = col
        elif any(k in c for k in ['vel', 'velocity']) and vel_col is None: vel_col = col

    if x_col is None or y_col is None or vel_col is None:
        raise RuntimeError(f"Could not find x/y/velocity columns. Got: {list(df.columns)}")

    df = df.dropna(subset=[x_col, y_col, vel_col]).copy()
    x = df[x_col].to_numpy()
    y = df[y_col].to_numpy()
    v = df[vel_col].to_numpy()

    # CRS & clipping to study area  
    crs_type = detect_crs(x, y)
    if os.path.exists(MASK_TIF):
        df = clip_points_to_mask_any_crs(df, x_col, y_col, MASK_TIF, crs_type)
    else:
        print("[WARN] No mask file found; proceeding without mask clipping")

    # Refresh arrays after mask-based clipping
    x = df[x_col].to_numpy(); y = df[y_col].to_numpy(); v = df[vel_col].to_numpy()
    if len(df) == 0:
        raise RuntimeError("No PS points remain after clipping; cannot plot")

    # Extent  
    x_min, x_max = np.nanmin(x), np.nanmax(x)
    y_min, y_max = np.nanmin(y), np.nanmax(y)
    bx = (x_max - x_min) * 0.02 or 1.0
    by = (y_max - y_min) * 0.02 or 1.0
    west, east = x_min - bx, x_max + bx
    south, north = y_min - by, y_max + by
    extent = [west, east, south, north]

    # Interpolate background grid 
    Xg, Yg, Vgrid = interpolate_grid(x, y, v, west, east, south, north, grid_res=GRID_RES)

    # Aggressive boundary masking for grid & scatter 
    if os.path.exists(MASK_TIF):
        try:
            with rasterio.open(MASK_TIF) as ms:
                mask_data = ms.read(1)
                mask_crs = ms.crs
                mask_transform = ms.transform
                mask_crs_str = mask_crs.to_string() if hasattr(mask_crs, 'to_string') else str(mask_crs)

                if HAVE_SCIPY:
                    mask_eroded = binary_erosion(mask_data > 0, iterations=2).astype(np.uint8)
                else:
                    mask_eroded = (mask_data > 0).astype(np.uint8)

                # Mask the grid
                H, W = Vgrid.shape
                for i in range(H):
                    for j in range(W):
                        gx, gy = Xg[i, j], Yg[i, j]
                        if ('4326' not in mask_crs_str) and (crs_type == 'wgs84'):
                            mx, my = rio_transform('EPSG:4326', mask_crs, [gx], [gy])
                            mx, my = mx[0], my[0]
                        else:
                            mx, my = gx, gy
                        r, c = rowcol(mask_transform, mx, my)
                        keep = (0 <= r < mask_eroded.shape[0]) and (0 <= c < mask_eroded.shape[1]) and (mask_eroded[int(r), int(c)] > 0)
                        if not keep:
                            Vgrid[i, j] = np.nan

                # Filter scatter points
                keep_pts = []
                for xi, yi in zip(x, y):
                    if ('4326' not in mask_crs_str) and (crs_type == 'wgs84'):
                        mx, my = rio_transform('EPSG:4326', mask_crs, [xi], [yi])
                        mx, my = mx[0], my[0]
                    else:
                        mx, my = xi, yi
                    r, c = rowcol(mask_transform, mx, my)
                    inside = (0 <= r < mask_eroded.shape[0]) and (0 <= c < mask_eroded.shape[1]) and (mask_eroded[int(r), int(c)] > 0)
                    keep_pts.append(inside)
                keep_pts = np.array(keep_pts, dtype=bool)
                x, y, v = x[keep_pts], y[keep_pts], v[keep_pts]
                print(f"[OK] Removed {np.sum(~keep_pts)} PS points near/outside boundary after erosion")
        except Exception as ex:
            print(f"[WARN] Aggressive boundary masking failed: {ex}")
            traceback.print_exc()

    # Load study polygon (EPSG:4326) for strict clipping 
    study_poly = get_study_polygon_4326(MASK_TIF, SHAPEFILE) if HAVE_SHAPELY else None

    # Extra scatter filtering using polygon  
    if study_poly is not None and HAVE_SHAPELY:
        try:
            keep2 = []
            for xi, yi in zip(x, y):
                try:
                    keep2.append(study_poly.contains(Point(xi, yi)) or study_poly.touches(Point(xi, yi)))
                except Exception:
                    keep2.append(True)
            keep2 = np.array(keep2, dtype=bool)
            x, y, v = x[keep2], y[keep2], v[keep2]
            print(f"[OK] Strict polygon clip removed {np.sum(~keep2)} additional PS points")
        except Exception as e:
            print(f"[WARN] Strict scatter clip failed: {e}")

    # Color scaling  
    vmin, vmax = VMIN_FIXED, VMAX_FIXED
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    # Diverging colormap 
    vel_colors = ['#000080','#0000FF','#4169E1','#87CEEB','#E0F6FF',
                  '#FFFFFF','#FFE0E0','#FFB6C1','#FF4500','#DC143C','#8B0000']
    vel_cmap = mcolors.LinearSegmentedColormap.from_list('Velocity', vel_colors, N=256)

    # Load roads & clip them to polygon
    roads = load_roads(ROAD_SHP, target_crs_epsg=4326)
    roads = clip_roads_to_polygon(roads, study_poly)

    # ================= Velocity Map =================
    print("\n[INFO] Creating velocity map...")
    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE, facecolor='white')
    ax.set_facecolor('white')

    im = ax.imshow(Vgrid, cmap=vel_cmap, extent=extent, norm=norm,
                   origin='lower', interpolation='bilinear', alpha=1.0, zorder=1)
    sc = ax.scatter(x, y, c=v, s=0.6, cmap=vel_cmap, norm=norm, edgecolors='none', alpha=0.9, zorder=2)

    # Nothing drawn outside the polygon
    clip_patch = None
    if study_poly is not None and HAVE_SHAPELY:
        try:
            clip_patch = polygon_to_pathpatch(ax, study_poly)
            if clip_patch is not None:
                ax.add_patch(clip_patch)  # to register transforms
                im.set_clip_path(clip_patch)
                sc.set_clip_path(clip_patch)
        except Exception as e:
            print(f"[WARN] Failed to apply clip-path: {e}")

    # Highways (red) & Boundary (black)
    plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850)

    # Optionally clip line/polygon artists too 
    if clip_patch is not None:
        try:
            for art in ax.lines + ax.patches:
                if art is not clip_patch:
                    art.set_clip_path(clip_patch)
        except Exception:
            pass

    drew = draw_boundary_from_mask(ax, MASK_TIF, zorder=999, color="black", linewidth=0.7)
    if not drew:
        draw_boundary_from_shp(ax, SHAPEFILE, zorder=998, color="black", linewidth=0.7)

    # North arrow + Scale bar
    add_north_arrow_tri(ax, size=0.09, pad=0.09, arrow_downshift=0.025)
    add_checker_scale_bar(ax, west, east, south, north, crs_type='wgs84',
                          mid_lat_deg=(south + north) / 2.0, segments=4, fontsize=4)

    # Axis labels + DMS format
    if detect_crs([west, east], [south, north]) == 'wgs84':
        ax.set_xlabel("Longitude", fontsize=5, fontweight='bold')
        ax.set_ylabel("Latitude",  fontsize=5, fontweight='bold')
        ax.xaxis.set_major_formatter(FuncFormatter(lambda val, pos: dms_label(val, True)))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda val, pos: dms_label(val, False)))
        ax.xaxis.set_major_locator(MultipleLocator(0.1))
        ax.yaxis.set_major_locator(MultipleLocator(0.1))
        ax.tick_params(axis='x', which='major', labelsize=3)
        ax.tick_params(axis='y', which='major', labelsize=3)
        for lbl in ax.get_xticklabels():
            lbl.set_rotation(45); lbl.set_ha('right'); lbl.set_fontsize(3)
    else:
        ax.set_xlabel("Easting (m)", fontsize=5, fontweight='bold')
        ax.set_ylabel("Northing (m)", fontsize=5, fontweight='bold')
        ax.tick_params(axis='both', which='major', labelsize=3)

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='red', lw=1.0, label='Highway'),
        Line2D([0], [0], color='black', lw=1.0, label='Study Area')
    ]
    legend_box = ax.legend(
        handles=legend_elements,
        loc='center',
        bbox_to_anchor=(0.80, 0.15),
        bbox_transform=ax.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=3,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0
    )
    legend_box.get_frame().set_edgecolor('lightgray')
    legend_box.get_frame().set_linewidth(0.3)

    # Colorbar
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cbar = plt.colorbar(im, cax=cax)
    cbar.set_label("LOS Velocity (mm/year)", fontsize=4, fontweight='bold')
    cbar.set_ticks(np.linspace(vmin, vmax, 5))
    cbar.ax.tick_params(labelsize=3)

    # Limit view to polygon bounds 
    if study_poly is not None and HAVE_SHAPELY:
        minx, miny, maxx, maxy = study_poly.bounds
        ax.set_xlim(minx, maxx)
        ax.set_ylim(miny, maxy)

    # Save map 
    plt.tight_layout()
    map_png = os.path.join(OUT_DIR, "Velocity Map.png")
    map_pdf = os.path.join(OUT_DIR, "Velocity Map.pdf")
    plt.savefig(map_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.savefig(map_pdf, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"[OK] Saved → {map_png}")
    print(f"[OK] Saved → {map_pdf}")
    plt.show()

    # ================= Histogram  =================
    print("\n[INFO] Creating velocity histogram...")
    fig2, ax2 = plt.subplots(1, 1, figsize=HIST_FIGSIZE, facecolor='white')
    hist_vals = np.clip(v, vmin, vmax)
    bins = np.linspace(vmin, vmax, 21)
    counts, edges = np.histogram(hist_vals, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2.0

    ax2.bar(centers, counts, width=(bins[1]-bins[0])*0.9,
            color="#888888", edgecolor="black", linewidth=0.4, alpha=0.9)
    ax2.axvline(0,            color='k',      lw=0.8, label='Zero velocity')
    ax2.axvline(np.mean(v),   color='r',      ls='--', lw=0.7, label=f"Mean: {np.mean(v):.2f}")
    ax2.axvline(np.median(v), color='orange', ls='--', lw=0.7, label=f"Median: {np.median(v):.2f}")
    ax2.legend(fontsize=4, loc='upper right', framealpha=0.9)

    ax2.set_xlabel("LOS Velocity (mm/year)", fontsize=4, fontweight='bold')
    ax2.set_ylabel("Number of PS Points",     fontsize=4, fontweight='bold')
    ax2.tick_params(axis='both', labelsize=4)
    ax2.set_xlim(vmin, vmax)
    ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.4, axis='y')
    ax2.set_axisbelow(True)

    hist_png = os.path.join(OUT_DIR, "Velocity Histogram.png")
    hist_pdf = os.path.join(OUT_DIR, "Velocity Histogram.pdf")
    plt.tight_layout(pad=1.2)
    plt.savefig(hist_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.savefig(hist_pdf, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"[OK] Saved → {hist_png}")
    print(f"[OK] Saved → {hist_pdf}")
    plt.show()

if __name__ == "__main__":
    main()


# Step-3.3: Downslope unit vector (ŝ), downslope unit

In [ ]:
# =========================================================
# Step 3.3 — Downslope unit vector (ŝ), downslope unit
# =========================================================

import os, json
import numpy as np
from datetime import datetime
import rasterio
from rasterio.warp import reproject, Resampling

def pp(msg): print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

# ------------------- PATHS ----------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

# Inputs
ANCHOR_TIF  = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
MASK_TIF    = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")
STEP33_DIR  = os.path.join(BASE, "Results", "Step-1.1-Slope_Aspect")
SLOPE_SRC   = os.path.join(STEP33_DIR, "slope_deg.tif")
ASPECT_SRC  = os.path.join(STEP33_DIR, "aspect_deg.tif")

# Outputs
OUT_DIR     = os.path.join(BASE, "Results", "Step-3.3-Downslope unit vector (ŝ)")
os.makedirs(OUT_DIR, exist_ok=True)

SLOPE_OUT   = os.path.join(OUT_DIR, "slope_deg_32645_12p5m_float32.tif")
ASPECT_OUT  = os.path.join(OUT_DIR, "aspect_deg_from_north_cw_32645_12p5m_float32.tif")
SE_TIF      = os.path.join(OUT_DIR, "s_hat_east_32645_12p5m_float32.tif")
SN_TIF      = os.path.join(OUT_DIR, "s_hat_north_32645_12p5m_float32.tif")
SU_TIF      = os.path.join(OUT_DIR, "s_hat_up_32645_12p5m_float32.tif")
META_JSON   = os.path.join(OUT_DIR, "downslope_metadata.json")

# Behavior knobs
ASPECT_CONVENTION   = "north_cw"   # "north_cw" (GIS default) or "east_ccw" (math)
FLAT_SLOPE_DEG      = 1.0          # treat slope < 1° as flat (invalid aspect/ŝ)
NODATA              = -9999.0

# Optional: force exact |ŝ|=1 by renormalization (normally NOT needed)
RENORMALIZE_S_HAT   = False

# --------------------------
# HELPERS
# --------------------------
def read_anchor_spec(path):
    with rasterio.open(path) as ds:
        return {
            "tf": ds.transform, "crs": ds.crs,
            "width": ds.width, "height": ds.height,
            "profile": ds.profile
        }

def read_raster(path):
    with rasterio.open(path) as ds:
        arr = ds.read(1)
        return arr, ds.transform, ds.crs, ds.nodata

def maybe_reproject(arr, src_tf, src_crs, dst_spec, resampling=Resampling.bilinear, src_nodata=None):
    """Reproject only if src grid != dst grid."""
    same_grid = (src_crs == dst_spec["crs"]) and (src_tf == dst_spec["tf"]) \
                and (arr.shape == (dst_spec["height"], dst_spec["width"]))
    if same_grid:
        return arr.astype("float64")

    dst = np.full((dst_spec["height"], dst_spec["width"]), np.nan, dtype="float64")
    reproject(
        source=arr.astype("float64"),
        destination=dst,
        src_transform=src_tf, src_crs=src_crs,
        dst_transform=dst_spec["tf"], dst_crs=dst_spec["crs"],
        resampling=resampling,
        src_nodata=src_nodata, dst_nodata=np.nan,
    )
    return dst

def read_mask_to_anchor(mask_path, dst_spec):
    if not os.path.exists(mask_path):
        return None
    with rasterio.open(mask_path) as ms:
        if (ms.crs == dst_spec["crs"] and ms.transform == dst_spec["tf"]
                and ms.width == dst_spec["width"] and ms.height == dst_spec["height"]):
            m = ms.read(1).astype("float64")
        else:
            m = np.full((dst_spec["height"], dst_spec["width"]), np.nan, dtype="float64")
            reproject(
                source=ms.read(1).astype("float64"),
                destination=m,
                src_transform=ms.transform, src_crs=ms.crs,
                dst_transform=dst_spec["tf"], dst_crs=dst_spec["crs"],
                resampling=Resampling.nearest,
                src_nodata=0, dst_nodata=np.nan
            )
    return np.where(np.isfinite(m) & (m > 0), 1, 0).astype("uint8")

def write_tif(path, arr, dst_spec, nodata=NODATA):
    prof = dst_spec["profile"].copy()
    prof.update(driver="GTiff", dtype="float32", count=1, nodata=nodata,
                compress="LZW", tiled=True, transform=dst_spec["tf"],
                height=dst_spec["height"], width=dst_spec["width"],
                crs=dst_spec["crs"])
    tmp = path + ".part"
    with rasterio.open(tmp, "w", **prof) as ds:
        ds.write(arr.astype("float32"), 1)
    os.replace(tmp, path)

def convert_aspect_to_north_cw(aspect_deg_array, convention):
    """Return aspect in degrees-from-North clockwise, regardless of input convention."""
    if convention == "north_cw":
        return aspect_deg_array
    elif convention == "east_ccw":
        # from East CCW -> from North CW
        return (90.0 - aspect_deg_array) % 360.0
    else:
        raise ValueError(f"Unknown ASPECT_CONVENTION: {convention}")

# --------------------------
# MAIN
# --------------------------
def main():
    pp("=== Step 3.3 — Downslope unit vector (ŝ) ===")

    # 1) Anchor spec
    if not os.path.exists(ANCHOR_TIF):
        raise SystemExit(f"Missing anchor: {ANCHOR_TIF}")
    dst_spec = read_anchor_spec(ANCHOR_TIF)
    pp(f"Anchor: {dst_spec['width']}×{dst_spec['height']} @ {dst_spec['crs']}")

    # 2) Read slope/aspect
    if not os.path.exists(SLOPE_SRC) or not os.path.exists(ASPECT_SRC):
        raise SystemExit("Missing Step-1.1 inputs. Run Phase 1.1 first.")

    slope_arr, slope_tf, slope_crs, slope_nd = read_raster(SLOPE_SRC)
    aspect_arr, aspect_tf, aspect_crs, aspect_nd = read_raster(ASPECT_SRC)

    slope = maybe_reproject(slope_arr, slope_tf, slope_crs, dst_spec,
                            resampling=Resampling.bilinear, src_nodata=slope_nd)
    aspect_raw = maybe_reproject(aspect_arr, aspect_tf, aspect_crs, dst_spec,
                                 resampling=Resampling.bilinear, src_nodata=aspect_nd)

    # 3) Optional mask
    mask = read_mask_to_anchor(MASK_TIF, dst_spec)  # None if not present

    # 4) Validity mask
    slope_ok  = np.isfinite(slope) & (slope >= FLAT_SLOPE_DEG)
    aspect_ok = np.isfinite(aspect_raw) & (aspect_raw >= 0.0)  # common: -1 for flats
    valid = slope_ok & aspect_ok
    if mask is not None:
        valid &= (mask == 1)

    # 5) Aspect -> from North, CW
    aspect_nc = convert_aspect_to_north_cw(aspect_raw, ASPECT_CONVENTION)

    # 6) Compute ŝ (ENU)
    alpha = np.deg2rad(slope)       # slope
    phi   = np.deg2rad(aspect_nc)   # azimuth from North, CW

    s_e = np.sin(phi) * np.cos(alpha)   # east
    s_n = np.cos(phi) * np.cos(alpha)   # north
    s_u = -np.sin(alpha)                # up (negative = down)

    # Optional: enforce unit norm exactly
    if RENORMALIZE_S_HAT:
        nrm = np.sqrt(s_e*s_e + s_n*s_n + s_u*s_u)
        nrm = np.where(nrm > 0, nrm, 1.0)
        s_e = s_e / nrm
        s_n = s_n / nrm
        s_u = s_u / nrm

    # 7) Apply NoData to outputs
    def mask_nodata(a):
        out = a.astype("float32").copy()
        out[~valid] = NODATA
        return out

    slope_out  = mask_nodata(slope)
    aspect_out = mask_nodata(aspect_nc)
    se_out     = mask_nodata(s_e)
    sn_out     = mask_nodata(s_n)
    su_out     = mask_nodata(s_u)

    # 8) Save rasters
    pp("Writing outputs …")
    write_tif(SLOPE_OUT,  slope_out,  dst_spec)
    write_tif(ASPECT_OUT, aspect_out, dst_spec)
    write_tif(SE_TIF,     se_out,     dst_spec)
    write_tif(SN_TIF,     sn_out,     dst_spec)
    write_tif(SU_TIF,     su_out,     dst_spec)

    # 9) QA (FIXED): use original 'valid' boolean
    if np.any(valid):
        norm = np.sqrt(s_e[valid]**2 + s_n[valid]**2 + s_u[valid]**2)
        norm_mean = float(np.mean(norm))
        norm_std  = float(np.std(norm))
        norm_max_abs_dev = float(np.max(np.abs(norm - 1.0)))

        pp(f"|ŝ| mean={norm_mean:.6f}, std={norm_std:.6f} (should be ~1,~0)")
        pp(f"max | |ŝ|-1 | = {norm_max_abs_dev:.6e}")
        pp(f"Slope deg (valid): min={float(np.nanmin(slope[valid])):.2f}, "
           f"mean={float(np.nanmean(slope[valid])):.2f}, "
           f"max={float(np.nanmax(slope[valid])):.2f}")
        pp(f"Aspect deg (valid): min={float(np.nanmin(aspect_nc[valid])):.1f}, "
           f"max={float(np.nanmax(aspect_nc[valid])):.1f}")
    else:
        norm_mean = norm_std = norm_max_abs_dev = None
        pp("No valid pixels found for QA (check inputs/mask).")

    # 10) Metadata / provenance JSON
    def _stats(a, m):
        mm = m & np.isfinite(a)
        if not np.any(mm):
            return {"count": 0}
        return {
            "count": int(np.sum(mm)),
            "min": float(np.nanmin(a[mm])),
            "max": float(np.nanmax(a[mm])),
            "mean": float(np.nanmean(a[mm])),
            "p95": float(np.nanpercentile(a[mm], 95)),
        }

    meta = {
        "created_utc": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "grid": {
            "crs_epsg": (dst_spec["crs"].to_epsg() if dst_spec["crs"] else None),
            "transform": list(dst_spec["tf"]),
            "width": int(dst_spec["width"]),
            "height": int(dst_spec["height"]),
        },
        "inputs": {
            "slope_src": SLOPE_SRC,
            "aspect_src": ASPECT_SRC,
            "anchor_tif": ANCHOR_TIF,
            "mask_tif": MASK_TIF if os.path.exists(MASK_TIF) else None,
        },
        "assumptions": {
            "aspect_convention_in": ASPECT_CONVENTION,
            "aspect_convention_used": "north_cw",
            "flat_slope_deg_threshold": FLAT_SLOPE_DEG,
            "nodata_value": NODATA,
            "definition": "ŝ is unit downslope direction in ENU (east,north,up)",
            "renormalize_s_hat": bool(RENORMALIZE_S_HAT),
        },
        "qa": {
            "norm_mean": norm_mean,
            "norm_std": norm_std,
            "norm_max_abs_deviation": norm_max_abs_dev,
            "valid_pixel_count": int(np.sum(valid)),
            "s_hat_east":  _stats(s_e, valid),
            "s_hat_north": _stats(s_n, valid),
            "s_hat_up":    _stats(s_u, valid),
            "slope_deg":   _stats(slope, valid),
            "aspect_deg_from_north_cw": _stats(aspect_nc, valid),
        },
        "outputs": {
            "slope_deg_tif": SLOPE_OUT,
            "aspect_deg_from_north_cw_tif": ASPECT_OUT,
            "s_hat_east_tif": SE_TIF,
            "s_hat_north_tif": SN_TIF,
            "s_hat_up_tif": SU_TIF,
            "metadata_json": META_JSON,
        }
    }
    with open(META_JSON, "w") as f:
        json.dump(meta, f, indent=2)
    pp(f"[OK] wrote {META_JSON}")

    pp("=== Done (Step 3.3) ===")

if __name__ == "__main__":
    main()


# Visualization: Downslope unit vector (ŝ)

In [ ]:
# =========================================================
# Step 3.3 — ŝ (Downslope unit vector) visualization
# =========================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.patches import Polygon, Rectangle
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch
from mpl_toolkits.axes_grid1 import make_axes_locatable
import traceback

# -------- Optional libs --------
try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    from shapely.geometry import shape as shp_shape, Polygon as ShpPolygon, MultiPolygon as ShpMultiPolygon, Point as ShpPoint
    from shapely.ops import unary_union
    HAVE_SHAPELY = True
except Exception:
    HAVE_SHAPELY = False

import rasterio
from rasterio.warp import transform_bounds, transform_geom
from rasterio import features as rio_features

# ---------------- PATHS ----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
OUT_DIR  = os.path.join(BASE, "Results", "Step-3.3-Downslope unit vector (ŝ)")
os.makedirs(OUT_DIR, exist_ok=True)

# ŝ components 
SE_TIF = os.path.join(OUT_DIR, "s_hat_east_32645_12p5m_float32.tif")
SN_TIF = os.path.join(OUT_DIR, "s_hat_north_32645_12p5m_float32.tif")
SU_TIF = os.path.join(OUT_DIR, "s_hat_up_32645_12p5m_float32.tif")

# Boundary / roads 
MASK_TIF  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
SHAPEFILE = os.path.join(BASE, "Study_Area.shp")
ROAD_SHP  = os.path.join(BASE, "NTD_Road.shp")

# ---------------- Display knobs  ----------------
MAP_FIGSIZE  = (2, 2)
HIST_FIGSIZE = (2, 2)
CAX_SIZE     = "5%"
CAX_PAD      = 0.01
HIST_BINS    = 20

# Diverging colormap (−1 → +1) with white at 0
DIV_COLORS = [
    '#000080','#0000FF','#4169E1','#87CEEB','#E0F2FF',
    '#FFFFFF',
    '#FFE0E0','#FFB6C1','#FF6347','#DC143C','#8B0000'
]
DIV_CMAP_BASE = mcolors.LinearSegmentedColormap.from_list('div', DIV_COLORS, N=256)

# =========================================================
# Small helpers: DMS, arrow, scale
# =========================================================
def to_dms(val, is_lon=True):
    a = abs(val); d = int(a)
    m_float = (a - d) * 60.0; m = int(m_float); s = (m_float - m) * 60.0
    hemi = ('E' if val >= 0 else 'W') if is_lon else ('N' if val >= 0 else 'S')
    return f"{d}°{m:02d}'{s:04.1f}\"{hemi}"

def add_north_arrow_tri(ax, size=0.09, pad=0.09, label='N', arrow_downshift=0.015, label_upshift=0.0):
    # Prevent label overlap with arrow 
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10 + label_upshift

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size*0.09)

    ax.add_patch(Polygon([apex, left_base, tail], closed=True, transform=ax.transAxes,
                         facecolor='white', edgecolor='black', lw=1.0, zorder=10))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True, transform=ax.transAxes,
                         facecolor='black', edgecolor='black', lw=1.0, zorder=10))
    ax.text(label_anchor_x, label_anchor_y, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=9, fontweight='bold', color='black', zorder=11)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1, 2, 5, 10):
        if m*base >= target: return m*base
    return 10*base

def add_checker_scale_bar(ax, west, east, south, north, crs_is_geographic=False,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_is_geographic:
        mid_lat = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd
    else:
        width_m = width_data
        units_per_meter_x = 1.0
    total_len_m = nice_len_125(width_m / 5.0)
    seg_len_m   = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m   * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    ax.add_patch(Rectangle((x_start - total_len_units*0.08, y_base - 0.035*height_data),
                           total_len_units*1.16, bar_h + 0.07*height_data,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        color = 'black' if i % 2 == 0 else 'white'
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData, facecolor=color,
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.012 * height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.012 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.012 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Road + Boundary helpers 
# =========================================================
def load_roads(road_shp_path, target_crs):
    if not os.path.exists(road_shp_path):
        print(f"[WARN] Road shapefile not found: {road_shp_path}")
        return None
    target_crs_str = str(target_crs) if not isinstance(target_crs, str) else target_crs
    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_shp_path)
            if roads.crs is None:
                print("[WARN] Road CRS is None; assuming EPSG:4326")
                roads = roads.set_crs(epsg=4326)
            roads = roads.to_crs(target_crs_str)
            print(f"[OK] Roads (GeoPandas): {len(roads)} features; CRS={roads.crs}")
            return roads
        except Exception as e:
            print(f"[WARN] GeoPandas failed for roads: {e}")
    if HAVE_FIONA:
        try:
            geoms = []
            with fiona.open(road_shp_path, 'r') as shp:
                src_crs = shp.crs
                for feat in shp:
                    geom = feat['geometry']
                    if geom is None: continue
                    try:
                        geom_t = transform_geom(src_crs, target_crs_str, geom)
                    except Exception:
                        geom_t = geom
                    geoms.append(geom_t)
            print(f"[OK] Roads (Fiona): {len(geoms)} features")
            return {'geometries': geoms, 'method': 'fiona', 'crs': target_crs_str}
        except Exception as e:
            print(f"[WARN] Fiona failed for roads: {e}")
    return None

def plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850):
    if roads is None:
        return
    try:
        if HAVE_GPD and hasattr(roads, 'plot'):
            try:
                roads.geometry = roads.geometry.apply(
                    lambda g: g if (g is None or g.geom_type in ("LineString", "MultiLineString")) else None
                )
            except Exception:
                pass
            roads.plot(ax=ax, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (GeoPandas)")
        elif isinstance(roads, dict) and 'geometries' in roads:
            for geom in roads['geometries']:
                if geom is None: continue
                if geom['type'] == 'LineString':
                    xs = [p[0] for p in geom['coordinates']]
                    ys = [p[1] for p in geom['coordinates']]
                    ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
                elif geom['type'] == 'MultiLineString':
                    for line in geom['coordinates']:
                        xs = [p[0] for p in line]
                        ys = [p[1] for p in line]
                        ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (Fiona fallback)")
    except Exception as e:
        print(f"[WARN] Failed to plot roads: {e}")

def draw_boundary(ax, shp_path, target_crs, zorder=999, color="black", linewidth=0.7):
    """Draw study boundary (preferred from shapefile)."""
    if os.path.exists(shp_path):
        try:
            tcrs = str(target_crs) if not isinstance(target_crs, str) else target_crs
            if HAVE_GPD:
                gdf = gpd.read_file(shp_path).to_crs(tcrs)
                gdf.boundary.plot(ax=ax, color=color, linewidth=linewidth, alpha=1.0, zorder=zorder)
                print("[OK] Boundary drawn from shapefile (GeoPandas)")
                return True
            elif HAVE_FIONA:
                geoms = []
                with fiona.open(shp_path, 'r') as src:
                    src_crs = src.crs
                    for ft in src:
                        if ft['geometry'] is None: continue
                        geoms.append(transform_geom(src_crs, tcrs, ft['geometry']))
                for geom in geoms:
                    if geom['type'] == 'Polygon':
                        xs = [p[0] for p in geom['coordinates'][0]]
                        ys = [p[1] for p in geom['coordinates'][0]]
                        ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
                    elif geom['type'] == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs = [p[0] for p in poly[0]]
                            ys = [p[1] for p in poly[0]]
                            ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
                print("[OK] Boundary drawn from shapefile (Fiona)")
                return True
        except Exception as ex:
            print(f"[WARN] Boundary-from-shp failed: {ex}")
    # Fallback: mask if in same CRS
    if os.path.exists(MASK_TIF):
        try:
            with rasterio.open(MASK_TIF) as ms:
                if ms.crs == target_crs:
                    mask = ms.read(1); b = ms.bounds
                    extent = (b.left, b.right, b.bottom, b.top)
                    ax.contour(mask, levels=[0.5], colors=color, linewidths=linewidth,
                               extent=extent, origin='upper', zorder=zorder)
                    print("[OK] Boundary drawn from mask")
                    return True
        except Exception as ex:
            print(f"[WARN] Boundary-from-mask failed: {ex}")
    return False

# =========================================================
# Study polygon clip-path in target CRS 
# =========================================================
def get_study_polygon_in_crs(target_crs):
    """Return shapely (Multi)Polygon in target CRS if possible; else None."""
    if not HAVE_SHAPELY:
        print("[WARN] Shapely not available; strict clip disabled")
        return None
    tcrs = str(target_crs) if not isinstance(target_crs, str) else target_crs

    # Prefer shapefile
    if os.path.exists(SHAPEFILE):
        try:
            if HAVE_GPD:
                gdf = gpd.read_file(SHAPEFILE)
                if gdf.crs is None:
                    gdf = gdf.set_crs(epsg=4326)
                gdf = gdf.to_crs(tcrs)
                poly = unary_union(gdf.geometry)
                return poly
            elif HAVE_FIONA:
                geoms = []
                with fiona.open(SHAPEFILE, 'r') as src:
                    src_crs = src.crs
                    for ft in src:
                        if ft['geometry'] is None: continue
                        geoms.append(shp_shape(transform_geom(src_crs, tcrs, ft['geometry'])))
                if geoms:
                    return unary_union(geoms)
        except Exception as e:
            print(f"[WARN] Polygon from shapefile failed: {e}")

    # Fallback: polygonize mask and transform
    if os.path.exists(MASK_TIF):
        try:
            with rasterio.open(MASK_TIF) as ds:
                data = ds.read(1)
                trans = ds.transform
                crs = ds.crs
                polys = []
                for geom, val in rio_features.shapes(data.astype('uint8'), mask=(data > 0), transform=trans):
                    if val != 1: continue
                    g = shp_shape(geom)
                    # transform to target
                    if HAVE_GPD:
                        g = gpd.GeoSeries([g], crs=crs).to_crs(tcrs).geometry.iloc[0]
                    else:
                        # best effort if no geopandas
                        g = g  # remains in mask CRS; clip-path may misalign if CRS differs
                    polys.append(g)
                if polys:
                    return unary_union(polys)
        except Exception as e:
            print(f"[WARN] Polygon from mask failed: {e}")
    return None

def polygon_to_pathpatch(ax, poly):
    """Create a PathPatch in data coords from a shapely (Multi)Polygon, for clipping."""
    if not (HAVE_SHAPELY and (poly is not None)):
        return None

    def _ring_to_vertices(ring):
        xs, ys = ring.coords.xy
        verts = list(zip(xs, ys))
        if verts[0] != verts[-1]:
            verts.append(verts[0])
        codes = [MplPath.MOVETO] + [MplPath.LINETO]*(len(verts)-2) + [MplPath.CLOSEPOLY]
        return verts, codes

    if isinstance(poly, ShpPolygon):
        polys = [poly]
    else:
        polys = list(poly.geoms)

    vertices = []; codes = []
    for p in polys:
        v, c = _ring_to_vertices(p.exterior); vertices += v; codes += c
        for hole in p.interiors:
            v, c = _ring_to_vertices(hole); vertices += v; codes += c
    path = MplPath(vertices, codes)
    return PathPatch(path, transform=ax.transData, facecolor='none', edgecolor='none')

# =========================================================
# Core plotting function 
# =========================================================
def plot_component(tif_path, title, out_prefix, vmin=-1.0, vmax=1.0,
                   cmap_base=DIV_CMAP_BASE):
    if not os.path.exists(tif_path):
        print(f"[WARN] Missing raster: {tif_path}")
        return

    # --- read raster
    with rasterio.open(tif_path) as ds:
        arr = ds.read(1)
        nodata = ds.nodata
        b = ds.bounds
        raster_crs = ds.crs
        extent = (b.left, b.right, b.bottom, b.top)
        crs_is_geographic = bool(raster_crs and raster_crs.is_geographic)

    west, east, south, north = extent

    # Mask nodata/outside 
    bad = ~np.isfinite(arr)
    if nodata is not None:
        bad |= (arr == nodata)
    marr = np.ma.array(arr, mask=bad)
    cmap = cmap_base.copy()
    cmap.set_bad(alpha=0.0)

    # Axis tick formatters 
    xmid = (west + east) / 2.0
    ymid = (south + north) / 2.0
    to_ll = None
    if crs_is_geographic:
        def fmt_lon(x, pos): return to_dms(x, True)
        def fmt_lat(y, pos): return to_dms(y, False)
    else:
        if HAVE_PYPROJ and (raster_crs is not None):
            try:
                to_ll = Transformer.from_crs(raster_crs, "EPSG:4326", always_xy=True)
            except Exception:
                to_ll = None
        if to_ll is not None:
            def fmt_lon(x, pos):
                lon, _ = to_ll.transform(x, ymid)
                return to_dms(lon, True)
            def fmt_lat(y, pos):
                _, lat = to_ll.transform(xmid, y)
                return to_dms(lat, False)
        else:
            def fmt_lon(x, pos): return f"{x:.0f}"
            def fmt_lat(y, pos): return f"{y:.0f}"

    # Load overlays in raster CRS
    roads = load_roads(ROAD_SHP, raster_crs) if os.path.exists(ROAD_SHP) else None

    # Strict study polygon clip 
    study_poly = get_study_polygon_in_crs(raster_crs)

    # ================= Map =================
    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE, facecolor='white')
    ax.set_facecolor('white')

    im = ax.imshow(marr, extent=extent, origin='upper',
                   vmin=vmin, vmax=vmax, cmap=cmap, interpolation='none', zorder=1)

    # HARD clip-path so nothing paints outside AOI
    clip_patch = polygon_to_pathpatch(ax, study_poly)
    if clip_patch is not None:
        try:
            ax.add_patch(clip_patch)
            im.set_clip_path(clip_patch)
        except Exception as e:
            print(f"[WARN] Clip-path apply failed: {e}")

    # Overlays
    plot_roads_on_map(ax, roads, line_color='red', line_width=0.5, line_style='-', alpha=0.9, zorder=850)
    # Clip any line/polygon artists too (belt & suspenders)
    if clip_patch is not None:
        try:
            for art in ax.lines + ax.patches:
                if art is not clip_patch:
                    art.set_clip_path(clip_patch)
        except Exception:
            pass

    drew = draw_boundary(ax, SHAPEFILE, raster_crs, zorder=999, color="black", linewidth=0.7)

    # North arrow + Scale bar 
    add_north_arrow_tri(ax, size=0.09, pad=0.09, arrow_downshift=0.025)
    add_checker_scale_bar(ax, west, east, south, north,
                          crs_is_geographic=crs_is_geographic,
                          segments=4, fontsize=4)

    # Axis labels + DMS tick formatting 
    ax.set_xlabel("Longitude", fontsize=5, fontweight='bold')
    ax.set_ylabel("Latitude",  fontsize=5, fontweight='bold')
    if crs_is_geographic:
        ax.xaxis.set_major_locator(MultipleLocator(0.1))
        ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_lat))
    ax.tick_params(axis='x', which='major', labelsize=3)
    ax.tick_params(axis='y', which='major', labelsize=3)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45); lbl.set_ha('right'); lbl.set_fontsize(3)

    # Legend 
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='red', lw=1.0, label='Highway'),
        Line2D([0], [0], color='black', lw=1.0, label='Study Area')
    ]
    legend_box = ax.legend(
        handles=legend_elements, 
        loc='center',
        bbox_to_anchor=(0.78, 0.15), 
        bbox_transform=ax.transAxes,
        frameon=True, 
        framealpha=0.9, 
        fontsize=3, 
        labelspacing=0.3,
        borderpad=0.3, 
        handlelength=1.2, 
        handletextpad=0.4, 
        borderaxespad=0.0
    )
    legend_box.get_frame().set_edgecolor('lightgray')
    legend_box.get_frame().set_linewidth(0.3)

    # Colorbar 
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb = plt.colorbar(im, cax=cax)
    cb.ax.tick_params(labelsize=3)
    cb.set_label("Unit value (−1 to 1)", fontsize=4, fontweight='bold')

    if (study_poly is not None) and HAVE_SHAPELY:
        minx, miny, maxx, maxy = study_poly.bounds
        ax.set_xlim(minx, maxx)
        ax.set_ylim(miny, maxy)

    map_png = os.path.join(OUT_DIR, f"{out_prefix}_map.png")
    map_pdf = os.path.join(OUT_DIR, f"{out_prefix}_map.pdf")
    plt.tight_layout()
    plt.savefig(map_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(map_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"[OK] Saved: {map_png}")
    print(f"[OK] Saved: {map_pdf}")
    plt.show()

    # ================= Histogram =================
    vals = arr[~bad]
    vals = np.clip(vals, vmin, vmax)

    fig2, ax2 = plt.subplots(1, 1, figsize=HIST_FIGSIZE, facecolor='white')
    bins = np.linspace(vmin, vmax, HIST_BINS + 1)
    counts, edges = np.histogram(vals, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2.0
    ax2.bar(centers, counts, width=(bins[1]-bins[0])*0.9,
            color="#8fa6ff", edgecolor="black", linewidth=0.4, alpha=0.9)
    ax2.set_xlabel("Unit value (−1 to 1)", fontsize=4, fontweight='bold')
    ax2.set_ylabel("Pixel count",           fontsize=4, fontweight='bold')
    ax2.tick_params(axis='both', which='major', labelsize=4)
    ax2.set_xlim(vmin, vmax)
    ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.4, axis='y')
    ax2.set_axisbelow(True)

    hist_png = os.path.join(OUT_DIR, f"{out_prefix}_hist.png")
    hist_pdf = os.path.join(OUT_DIR, f"{out_prefix}_hist.pdf")
    plt.tight_layout(pad=1.2)
    plt.savefig(hist_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(hist_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"[OK] Saved: {hist_png}")
    print(f"[OK] Saved: {hist_pdf}")
    plt.show()

# =========================================================
# Run
# =========================================================
if __name__ == "__main__":
    try:
        plot_component(SE_TIF, "ŝ (east component)",  "s_hat_east",  vmin=-1.0, vmax=1.0)
        plot_component(SN_TIF, "ŝ (north component)", "s_hat_north", vmin=-1.0, vmax=1.0)
        plot_component(SU_TIF, "ŝ (up component)",    "s_hat_up",    vmin=-1.0, vmax=1.0)
    except Exception as e:
        print("[ERROR] Visualization failed:", e)
        traceback.print_exc()


# Step-3.4: LOS unit vector (l̂)

In [ ]:
# =========================================================
# Step 3.4 — LOS unit vector (l̂)
# =========================================================

import os, json, math, numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# -------- USER PATHS --------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
# Input Path
ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
STUDY_MASK = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

# Optional ŝ rasters for QA
STEP33  = os.path.join(BASE, "Results", "Step-3.3-Downslope unit vector (ŝ)")
S_EAST  = os.path.join(STEP33, "s_hat_east_32645_12p5m_float32.tif")
S_NORTH = os.path.join(STEP33, "s_hat_north_32645_12p5m_float32.tif")
S_UP    = os.path.join(STEP33, "s_hat_up_32645_12p5m_float32.tif")

# Output Path
OUTPUT_DIR  = os.path.join(BASE, "Results", "Step-3.4-LOS unit (l̂)")
os.makedirs(OUTPUT_DIR, exist_ok=True)

L_EAST_TIF  = os.path.join(OUTPUT_DIR, "l_hat_east_32645_12p5m_float32.tif")
L_NORTH_TIF = os.path.join(OUTPUT_DIR, "l_hat_north_32645_12p5m_float32.tif")
L_UP_TIF    = os.path.join(OUTPUT_DIR, "l_hat_up_32645_12p5m_float32.tif")
META_JSON   = os.path.join(OUTPUT_DIR, "los_geometry_metadata.json")

# -------- GEOMETRY OPTIONS --------
GEOM_MODE          = "A"          # "A" = from heading, "B" = direct look azimuth
TRACK              = "ASC"        # cosmetic; heading controls the math
HEADING_DEG        = 347.0        # deg from North, CW (verify!)
LOOK_AZIMUTH_DEG_B = 90.0         # used only if GEOM_MODE == "B"
INCIDENCE_DEG      = 38.0         # zenith angle (from vertical)

# If you want to try pulling values from StaMPS metadata:
TRY_READ_METADATA = True
CLIP_META_MAT = os.path.join(BASE, "INSAR_20210615", "NTD_PS_InSAR_data", "clipping_metadata.mat")

# Nodata & tolerances
NODATA = -9999.0
NORM_TOL = 1e-3

def pp(msg):
    print(msg, flush=True)

# ---------- Try to read heading/incidence from StaMPS MAT ----------
def _try_load_metadata():
    if not TRY_READ_METADATA or not os.path.exists(CLIP_META_MAT):
        return None
    try:
        try:
            import scipy.io as sio
            md = sio.loadmat(CLIP_META_MAT, squeeze_me=True, struct_as_record=False)
        except Exception:
            return None

        cand_inc  = ["inc", "incidence", "incidence_deg", "look_angle", "theta", "theta_deg"]
        cand_head = ["heading", "heading_deg", "track_heading", "azimuth_heading"]

        incidence = None
        heading   = None

        for k in cand_inc:
            if k in md and np.isscalar(md[k]):
                incidence = float(md[k]); break
        for k in cand_head:
            if k in md and np.isscalar(md[k]):
                heading = float(md[k]); break

        if incidence is None:
            for v in md.values():
                try:
                    if hasattr(v, "__dict__"):
                        for k in cand_inc:
                            if hasattr(v, k):
                                val = getattr(v, k)
                                if np.isscalar(val):
                                    incidence = float(val); break
                except Exception:
                    pass
                if incidence is not None:
                    break

        if heading is None:
            for v in md.values():
                try:
                    if hasattr(v, "__dict__"):
                        for k in cand_head:
                            if hasattr(v, k):
                                val = getattr(v, k)
                                if np.isscalar(val):
                                    heading = float(val); break
                except Exception:
                    pass
                if heading is not None:
                    break

        if incidence is None and heading is None:
            return None

        return {"incidence_deg": incidence, "heading_deg": heading}
    except Exception:
        return None

# ---------- Compute look azimuth & incidence ----------
def resolve_geometry():
    meta = _try_load_metadata()
    used = {}
    if meta is not None:
        if isinstance(meta.get("incidence_deg"), (int, float)):
            used["incidence_deg"] = float(meta["incidence_deg"])
        if isinstance(meta.get("heading_deg"), (int, float)):
            used["heading_deg"] = float(meta["heading_deg"])

    if GEOM_MODE == "A":
        heading = used.get("heading_deg", HEADING_DEG)
        look_az = (heading + 90.0) % 360.0  # right-looking radar assumption
    else:
        look_az = LOOK_AZIMUTH_DEG_B
        heading = used.get("heading_deg", HEADING_DEG)  # cosmetic only

    incidence = used.get("incidence_deg", INCIDENCE_DEG)

    used["track"]         = TRACK
    used["mode"]          = GEOM_MODE
    used["heading_deg"]   = float(heading)
    used["look_az_deg"]   = float(look_az)
    used["incidence_deg"] = float(incidence)
    return used

def main():
    pp("=== Step 3.4 — LOS unit vector (l̂) ===")

    # 0) Read anchor & mask
    if not os.path.exists(ANCHOR_TIF):
        raise FileNotFoundError(f"Anchor grid not found: {ANCHOR_TIF}")
    if not os.path.exists(STUDY_MASK):
        raise FileNotFoundError(f"Study mask not found: {STUDY_MASK}")

    with rasterio.open(ANCHOR_TIF) as src:
        prof = src.profile
        tf   = src.transform
        crs  = src.crs
        H, W = src.height, src.width

    with rasterio.open(STUDY_MASK) as ms:
        mask = ms.read(1).astype(np.uint8)
        if (ms.transform != tf) or (ms.width != W) or (ms.height != H) or (ms.crs != crs):
            pp("[MASK] Reprojecting study_mask to anchor grid …")
            mask_res = np.zeros((H, W), dtype="uint8")
            reproject(
                source=ms.read(1), destination=mask_res,
                src_transform=ms.transform, src_crs=ms.crs,
                dst_transform=tf, dst_crs=crs,
                resampling=Resampling.nearest,
                src_nodata=0, dst_nodata=0
            )
            mask = mask_res

    # 1) Resolve geometry (look azimuth & incidence)
    used = resolve_geometry()
    psi_deg = used["look_az_deg"]      # from North, clockwise
    th_deg  = used["incidence_deg"]    # zenith angle

    psi = math.radians(psi_deg)
    th  = math.radians(th_deg)

    # 2) Compute components (constant across scene)
    l_e = math.sin(psi) * math.sin(th)
    l_n = math.cos(psi) * math.sin(th)
    l_u = math.cos(th)

    # 3) Create arrays and apply mask
    L_e = np.full((H, W), l_e, dtype="float32")
    L_n = np.full((H, W), l_n, dtype="float32")
    L_u = np.full((H, W), l_u, dtype="float32")

    L_e[mask == 0] = NODATA
    L_n[mask == 0] = NODATA
    L_u[mask == 0] = NODATA

    # 4) Save rasters
    out_prof = prof.copy()
    out_prof.update(dtype="float32", count=1, nodata=NODATA,
                    compress="LZW", tiled=True, bigtiff="IF_NEEDED")

    with rasterio.open(L_EAST_TIF, "w", **out_prof) as dst:
        dst.write(L_e, 1)
    with rasterio.open(L_NORTH_TIF, "w", **out_prof) as dst:
        dst.write(L_n, 1)
    with rasterio.open(L_UP_TIF, "w", **out_prof) as dst:
        dst.write(L_u, 1)

    pp(f"[OK] l_hat_east  → {L_EAST_TIF}")
    pp(f"[OK] l_hat_north → {L_NORTH_TIF}")
    pp(f"[OK] l_hat_up    → {L_UP_TIF}")

    # 5) Metadata JSON
    used.update({
        "crs_epsg": crs.to_epsg(),
        "grid": {"width": W, "height": H, "transform": list(tf)},
        "los_definition": "unit vector from ground toward satellite in ENU; azimuth from north (cw), incidence from vertical"
    })
    with open(META_JSON, "w") as f:
        json.dump(used, f, indent=2)
    pp(f"[OK] metadata → {META_JSON}")

    # 6) QA: normalization (mask-aware)
    inside = (mask == 1)
    L_e_nan = np.where(L_e == NODATA, np.nan, L_e)
    L_n_nan = np.where(L_n == NODATA, np.nan, L_n)
    L_u_nan = np.where(L_u == NODATA, np.nan, L_u)
    mag = np.sqrt(L_e_nan**2 + L_n_nan**2 + L_u_nan**2)

    if np.any(inside):
        mag_ok = float(np.nanmax(np.abs(mag[inside] - 1.0)))
        pp(f"[QA] |l̂| deviation (max): {mag_ok:.2e} (<= {NORM_TOL} is good)")

    # Optional: dot(l̂, ŝ) if ŝ rasters exist
    if os.path.exists(S_EAST) and os.path.exists(S_NORTH) and os.path.exists(S_UP):
        with rasterio.open(S_EAST) as de:
            s_e = de.read(1).astype("float32")
            de_tf, de_crs, de_w, de_h = de.transform, de.crs, de.width, de.height
        with rasterio.open(S_NORTH) as dn:
            s_n = dn.read(1).astype("float32")
            dn_tf, dn_crs, dn_w, dn_h = dn.transform, dn.crs, dn.width, dn.height
        with rasterio.open(S_UP) as du:
            s_u = du.read(1).astype("float32")
            du_tf, du_crs, du_w, du_h = du.transform, du.crs, du.width, du.height

        def _ensure_match(A, src_tf, src_crs, src_w, src_h):
            if (src_tf != tf) or (src_w != W) or (src_h != H) or (src_crs != crs):
                B = np.full((H, W), NODATA, dtype="float32")
                reproject(
                    source=A, destination=B,
                    src_transform=src_tf, src_crs=src_crs,
                    dst_transform=tf, dst_crs=crs,
                    resampling=Resampling.bilinear,
                    src_nodata=NODATA, dst_nodata=NODATA
                )
                return B
            return A

        s_e = _ensure_match(s_e, de_tf, de_crs, de_w, de_h)
        s_n = _ensure_match(s_n, dn_tf, dn_crs, dn_w, dn_h)
        s_u = _ensure_match(s_u, du_tf, du_crs, du_w, du_h)

        dot = s_e * L_e + s_n * L_n + s_u * L_u

        # ---- FIX HERE: require ŝ pixels are valid too ----
        m = (inside &
             np.isfinite(dot) &
             (L_e != NODATA) &
             (s_e != NODATA) & (s_n != NODATA) & (s_u != NODATA))

        if m.any():
            pp(f"[QA] dot(l̂, ŝ): min={float(np.nanmin(dot[m])):.4f}  max={float(np.nanmax(dot[m])):.4f}")
        else:
            pp("[QA] dot(l̂, ŝ): no valid overlapping pixels (check masks/nodata).")

    pp("=== Done (Step 3.4) ===")

if __name__ == "__main__":
    main()


# Visualization: LOS unit vector (l̂)

In [ ]:
# =========================================================
# Step 3.4 — LOS unit vector (l̂) visualization
# =========================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter, MultipleLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
import traceback

# Optional libs for overlays
try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

# Required for DMS ticks when rasters are projected
try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

import rasterio
from rasterio.warp import transform_geom

# ---------------- PATHS ----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP34 = os.path.join(BASE, "Results", "Step-3.4-LOS unit (l̂)")
os.makedirs(STEP34, exist_ok=True)

L_EAST_TIF  = os.path.join(STEP34, "l_hat_east_32645_12p5m_float32.tif")
L_NORTH_TIF = os.path.join(STEP34, "l_hat_north_32645_12p5m_float32.tif")
L_UP_TIF    = os.path.join(STEP34, "l_hat_up_32645_12p5m_float32.tif")

# Boundary / Roads 
MASK_TIF  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
SHAPEFILE = os.path.join(BASE, "Study_Area.shp")
ROAD_SHP  = os.path.join(BASE, "NTD_Road.shp")

# ---------------- Display knobs ----------------
MAP_FIGSIZE  = (2, 2)
HIST_FIGSIZE = (2, 2)
CAX_SIZE     = "5%"
CAX_PAD      = 0.01
HIST_BINS    = 20
V_MIN, V_MAX = -1.0, 1.0

# Diverging colormap
DIV_COLORS = ['#000080','#0000FF','#4169E1','#87CEEB','#E0F2FF',
              '#FFFFFF','#FFE0E0','#FFB6C1','#FF6347','#DC143C','#8B0000']
DIV_CMAP_BASE = mcolors.LinearSegmentedColormap.from_list('div', DIV_COLORS, N=256)

# =========================================================
# Helpers
# =========================================================
def to_dms(val, is_lon=True):
    a = abs(val)
    d = int(a)
    m_float = (a - d) * 60.0
    m = int(m_float)
    s = (m_float - m) * 60.0
    hemi = ('E' if val >= 0 else 'W') if is_lon else ('N' if val >= 0 else 'S')
    return f"{d}°{m:02d}'{s:04.1f}\"{hemi}"

def add_north_arrow_tri(ax, size=0.09, pad=0.09, label='N'):
    x0 = pad
    y0 = 1.0 - pad - size
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size*0.09)
    ax.add_patch(Polygon([apex, left_base, tail], closed=True, transform=ax.transAxes,
                         facecolor='white', edgecolor='black', lw=1.0, zorder=10))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True, transform=ax.transAxes,
                         facecolor='black', edgecolor='black', lw=1.0, zorder=10))
    ax.text(apex[0], y1 + size*0.12, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=9, fontweight='bold', color='black', zorder=11)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target)))
    base = 10**exp
    for m in (1, 2, 5, 10):
        if m*base >= target:
            return m*base
    return 10*base

def add_checker_scale_bar(ax, west, east, south, north, crs_is_geographic=False,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_is_geographic:
        mid_lat = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd
    else:
        width_m = width_data
        units_per_meter_x = 1.0
    total_len_m = nice_len_125(width_m / 5.0)
    seg_len_m   = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m   * units_per_meter_x
    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data
    ax.add_patch(Rectangle((x_start - total_len_units*0.08, y_base - 0.035*height_data),
                           total_len_units*1.16, bar_h + 0.07*height_data,
                           transform=ax.transData, facecolor='white', edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        color = 'black' if i % 2 == 0 else 'white'
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData, facecolor=color,
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.012*height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.012*height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.012*height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Highway + boundary helpers
# =========================================================
def load_roads(road_shp_path, target_crs):
    """Load road shapefile and transform to target CRS."""
    if not os.path.exists(road_shp_path):
        print(f"[WARN] Road shapefile not found: {road_shp_path}")
        return None
    print(f"[INFO] Loading roads: {road_shp_path}")
    target_crs_str = str(target_crs) if not isinstance(target_crs, str) else target_crs

    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_shp_path)
            if roads.crs is None:
                print("[WARN] Road CRS is None; assuming EPSG:4326")
                roads = roads.set_crs(epsg=4326)
            roads = roads.to_crs(target_crs_str)
            print(f"[OK] Roads (GeoPandas): {len(roads)} features; CRS={roads.crs}")
            return roads
        except Exception as e:
            print(f"[WARN] GeoPandas failed for roads: {e}")

    if HAVE_FIONA:
        try:
            geoms = []
            with fiona.open(road_shp_path, 'r') as shp:
                src_crs = shp.crs
                for feat in shp:
                    geom = feat['geometry']
                    if geom is None: continue
                    try:
                        geom_t = transform_geom(src_crs, target_crs_str, geom)
                    except Exception as e:
                        print(f"[WARN] Road transform failed; keeping original: {e}")
                        geom_t = geom
                    geoms.append(geom_t)
            print(f"[OK] Roads (Fiona): {len(geoms)} features")
            return {'geometries': geoms, 'method': 'fiona', 'crs': target_crs_str}
        except Exception as e:
            print(f"[WARN] Fiona failed for roads: {e}")
    print("[WARN] Unable to load roads")
    return None

def plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850):
    if roads is None:
        return
    try:
        if HAVE_GPD and hasattr(roads, 'plot'):
            try:
                roads.geometry = roads.geometry.apply(
                    lambda g: g if g is None or g.geom_type in ("LineString", "MultiLineString") else None
                )
            except Exception:
                pass
            roads.plot(ax=ax, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (GeoPandas)")
        elif isinstance(roads, dict) and 'geometries' in roads:
            for geom in roads['geometries']:
                if geom is None: continue
                if geom['type'] == 'LineString':
                    xs = [p[0] for p in geom['coordinates']]
                    ys = [p[1] for p in geom['coordinates']]
                    ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
                elif geom['type'] == 'MultiLineString':
                    for line in geom['coordinates']:
                        xs = [p[0] for p in line]
                        ys = [p[1] for p in line]
                        ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (Fiona fallback)")
    except Exception as e:
        print(f"[WARN] Failed to plot roads: {e}")

def draw_boundary_from_mask(ax, mask_tif_path, raster_crs, zorder=999, color="black", linewidth=0.7):
    """Draw boundary using mask TIFF if mask CRS == raster CRS."""
    if not os.path.exists(mask_tif_path):
        return False
    try:
        with rasterio.open(mask_tif_path) as ms:
            if ms.crs != raster_crs:
                print("[INFO] Mask CRS differs from raster CRS; skip mask boundary")
                return False
            mask = ms.read(1)
            b = ms.bounds
            extent = (b.left, b.right, b.bottom, b.top)
        ax.contour(mask, levels=[0.5], colors=color, linewidths=linewidth,
                   extent=extent, origin='upper', zorder=zorder)
        print("[OK] Boundary drawn from mask")
        return True
    except Exception as ex:
        print(f"[WARN] Boundary-from-mask failed: {ex}")
        return False

def draw_boundary_from_shp(ax, shp_path, target_crs, zorder=998, color="black", linewidth=0.7):
    if not os.path.exists(shp_path):
        return False
    try:
        target_crs_str = str(target_crs) if not isinstance(target_crs, str) else target_crs
        if HAVE_GPD:
            gdf = gpd.read_file(shp_path)
            gdf = gdf.to_crs(target_crs_str)
            gdf.boundary.plot(ax=ax, color=color, linewidth=linewidth, alpha=1.0, zorder=zorder)
            print("[OK] Boundary drawn from shapefile (GeoPandas)")
            return True
        elif HAVE_FIONA:
            geoms = []
            with fiona.open(shp_path, 'r') as shp:
                src_crs = shp.crs
                for feat in shp:
                    geom = feat['geometry']
                    if geom is None: continue
                    geom_t = transform_geom(src_crs, target_crs_str, geom)
                    geoms.append(geom_t)
            for geom in geoms:
                if geom['type'] == 'Polygon':
                    xs = [p[0] for p in geom['coordinates'][0]]
                    ys = [p[1] for p in geom['coordinates'][0]]
                    ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
                elif geom['type'] == 'MultiPolygon':
                    for poly in geom['coordinates']:
                        xs = [p[0] for p in poly[0]]
                        ys = [p[1] for p in poly[0]]
                        ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
            print("[OK] Boundary drawn from shapefile (Fiona)")
            return True
    except Exception as ex:
        print(f"[WARN] Boundary-from-shp failed: {ex}")
    return False

# =========================================================
# Core plotting
# =========================================================
def plot_component(tif_path, title, out_prefix, vmin=V_MIN, vmax=V_MAX,
                   cmap_base=DIV_CMAP_BASE):
    if not os.path.exists(tif_path):
        print(f"[WARN] Missing raster: {tif_path}")
        return

    # Read raster
    with rasterio.open(tif_path) as ds:
        arr = ds.read(1)
        nodata = ds.nodata
        b = ds.bounds
        raster_crs = ds.crs
        extent = (b.left, b.right, b.bottom, b.top)
        crs_is_geographic = bool(raster_crs and raster_crs.is_geographic)

    # derive west/east/south/north for decorations
    west, east, south, north = extent

    # Mask nodata
    bad = ~np.isfinite(arr)
    if nodata is not None:
        bad |= (arr == nodata)
    marr = np.ma.array(arr, mask=bad)
    cmap = cmap_base.copy()
    cmap.set_bad(alpha=0.0)

    # DMS tick formatters
    xmid = (west + east) / 2.0
    ymid = (south + north) / 2.0

    if crs_is_geographic:
        def fmt_lon(x, pos): return to_dms(x, True)
        def fmt_lat(y, pos): return to_dms(y, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError("Raster is projected; DMS axis labels require pyproj. Please install 'pyproj'.")
        to_ll = Transformer.from_crs(raster_crs, "EPSG:4326", always_xy=True)
        def fmt_lon(x, pos):
            lon, lat = to_ll.transform(x, ymid)
            return to_dms(lon, True)
        def fmt_lat(y, pos):
            lon, lat = to_ll.transform(xmid, y)
            return to_dms(lat, False)

    # Overlays in raster CRS
    roads = load_roads(ROAD_SHP, raster_crs) if os.path.exists(ROAD_SHP) else None

    # ============ MAP ============
    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE, facecolor='white')
    ax.set_facecolor('white')

    im = ax.imshow(marr, extent=extent, origin='upper',
                   vmin=vmin, vmax=vmax, cmap=cmap, interpolation='none', zorder=1)

    # Highway and Boundary
    plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850)
    drew = draw_boundary_from_mask(ax, MASK_TIF, raster_crs, zorder=999, color="black", linewidth=0.7)
    if not drew:
        draw_boundary_from_shp(ax, SHAPEFILE, raster_crs, zorder=998, color="black", linewidth=0.7)

    # North arrow + Scale bar 
    add_north_arrow_tri(ax, size=0.09, pad=0.09)
    add_checker_scale_bar(ax, west, east, south, north,
                          crs_is_geographic=crs_is_geographic,
                          segments=4, fontsize=4)

    # Axis labels + DMS tick formatting
    if crs_is_geographic:
        ax.set_xlabel("Longitude", fontsize=5, fontweight='bold')
        ax.set_ylabel("Latitude",  fontsize=5, fontweight='bold')
        ax.xaxis.set_major_locator(MultipleLocator(0.1))  # 6 minutes
        # ax.yaxis.set_major_locator(MultipleLocator(0.1))
    else:
        ax.set_xlabel("Longitude", fontsize=5, fontweight='bold')
        ax.set_ylabel("Latitude", fontsize=5, fontweight='bold')

    ax.xaxis.set_major_formatter(FuncFormatter(fmt_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_lat))
    ax.tick_params(axis='x', which='major', labelsize=3)
    ax.tick_params(axis='y', which='major', labelsize=3)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45); lbl.set_ha('right'); lbl.set_fontsize(3)

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='red', lw=1.0, label='Highway'),
        Line2D([0], [0], color='black', lw=1.0, label='Study Area')
    ]
    legend_box = ax.legend(
        handles=legend_elements,
        loc='center',
        bbox_to_anchor=(0.82, 0.15),
        bbox_transform=ax.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=3,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0
    )
    legend_box.get_frame().set_edgecolor('lightgray')
    legend_box.get_frame().set_linewidth(0.3)

    # Side colorbar
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb = plt.colorbar(im, cax=cax)
    cb.ax.tick_params(labelsize=3)
    cb.set_label("Unit value (−1 to 1)", fontsize=4, fontweight='bold')
    map_path = os.path.join(STEP34, f"{out_prefix}_map.png")
    plt.tight_layout()
    plt.savefig(map_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"[OK] Saved: {map_path}")
    plt.show()

    # ============ HISTOGRAM ============
    vals = arr[~bad]
    vals = np.clip(vals, vmin, vmax)

    fig2, ax2 = plt.subplots(1, 1, figsize=HIST_FIGSIZE, facecolor='white')
    bins = np.linspace(vmin, vmax, HIST_BINS + 1)
    counts, edges = np.histogram(vals, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2.0
    ax2.bar(centers, counts, width=(bins[1]-bins[0])*0.9,
            color="#8fa6ff", edgecolor="black", linewidth=0.4, alpha=0.9)
    ax2.set_xlabel("Unit value (−1 to 1)", fontsize=4, fontweight='bold')
    ax2.set_ylabel("Pixel count",           fontsize=4, fontweight='bold')
    ax2.tick_params(axis='both', which='major', labelsize=4)
    ax2.set_xlim(vmin, vmax)
    ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.4, axis='y')
    ax2.set_axisbelow(True)

    hist_path = os.path.join(STEP34, f"{out_prefix}_hist.png")
    plt.tight_layout(pad=1.2)
    plt.savefig(hist_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"[OK] Saved: {hist_path}")
    plt.show()

# =========================================================
# Run all three components
# =========================================================
if __name__ == "__main__":
    try:
        plot_component(L_EAST_TIF,  "l̂ (east component)",  "l_hat_east")
        plot_component(L_NORTH_TIF, "l̂ (north component)", "l_hat_north")
        plot_component(L_UP_TIF,    "l̂ (up component)",    "l_hat_up")
    except Exception as e:
        print("[ERROR] Visualization failed:", e)
        traceback.print_exc()


In [ ]:
# =========================================================
# Step 3.4 — LOS unit vector (l̂) visualization
# =========================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.patches import Polygon, Rectangle
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch
from mpl_toolkits.axes_grid1 import make_axes_locatable
import traceback

# Optional libs 
try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    from shapely.geometry import shape as shp_shape, Polygon as ShpPolygon, MultiPolygon as ShpMultiPolygon
    from shapely.ops import unary_union
    HAVE_SHAPELY = True
except Exception:
    HAVE_SHAPELY = False

import rasterio
from rasterio.warp import transform_geom
from rasterio import features as rio_features

# ---------------- PATHS ----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")
STEP34 = os.path.join(BASE, "Results", "Step-3.4-LOS unit (l̂)")
os.makedirs(STEP34, exist_ok=True)

L_EAST_TIF  = os.path.join(STEP34, "l_hat_east_32645_12p5m_float32.tif")
L_NORTH_TIF = os.path.join(STEP34, "l_hat_north_32645_12p5m_float32.tif")
L_UP_TIF    = os.path.join(STEP34, "l_hat_up_32645_12p5m_float32.tif")

# Boundary / Roads 
MASK_TIF  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
SHAPEFILE = os.path.join(BASE, "Study_Area.shp")
ROAD_SHP  = os.path.join(BASE, "NTD_Road.shp")

# ---------------- Display knobs ----------------
MAP_FIGSIZE  = (2, 2)
HIST_FIGSIZE = (2, 2)
CAX_SIZE     = "5%"
CAX_PAD      = 0.01
HIST_BINS    = 20
V_MIN, V_MAX = -1.0, 1.0

# Diverging colormap (−1 → +1) with white at 0
DIV_COLORS = ['#000080','#0000FF','#4169E1','#87CEEB','#E0F2FF',
              '#FFFFFF','#FFE0E0','#FFB6C1','#FF6347','#DC143C','#8B0000']
DIV_CMAP_BASE = mcolors.LinearSegmentedColormap.from_list('div', DIV_COLORS, N=256)

# =========================================================
# Tiny-style helpers: DMS, north arrow, checker scale bar
# =========================================================
def to_dms(val, is_lon=True):
    a = abs(val); d = int(a)
    m_float = (a - d) * 60.0; m = int(m_float); s = (m_float - m) * 60.0
    hemi = ('E' if val >= 0 else 'W') if is_lon else ('N' if val >= 0 else 'S')
    return f"{d}°{m:02d}'{s:04.1f}\"{hemi}"

def add_north_arrow_tri(ax, size=0.09, pad=0.09, label='N', arrow_downshift=0.025, label_upshift=0.0):
    # Prevent label overlap with arrow (fixed position)
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10 + label_upshift

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size*0.09)

    ax.add_patch(Polygon([apex, left_base, tail], closed=True, transform=ax.transAxes,
                         facecolor='white', edgecolor='black', lw=1.0, zorder=10))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True, transform=ax.transAxes,
                         facecolor='black', edgecolor='black', lw=1.0, zorder=10))
    ax.text(label_anchor_x, label_anchor_y, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=9, fontweight='bold', color='black', zorder=11)

def meters_per_degree_lon(lat_deg): return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1, 2, 5, 10):
        if m*base >= target: return m*base
    return 10*base

def add_checker_scale_bar(ax, west, east, south, north, crs_is_geographic=False,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_is_geographic:
        mid_lat = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd
    else:
        width_m = width_data
        units_per_meter_x = 1.0
    total_len_m = nice_len_125(width_m / 5.0)
    seg_len_m   = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m   * units_per_meter_x
    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data

    ax.add_patch(Rectangle((x_start - total_len_units*0.08, y_base - 0.035*height_data),
                           total_len_units*1.16, bar_h + 0.07*height_data,
                           transform=ax.transData, facecolor='white', edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        color = 'black' if i % 2 == 0 else 'white'
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData, facecolor=color,
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.012*height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.012*height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.012*height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Roads + Boundary 
# =========================================================
def load_roads(road_shp_path, target_crs):
    if not os.path.exists(road_shp_path):
        print(f"[WARN] Road shapefile not found: {road_shp_path}")
        return None
    tcrs = str(target_crs) if not isinstance(target_crs, str) else target_crs
    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_shp_path)
            if roads.crs is None:
                print("[WARN] Road CRS is None; assuming EPSG:4326")
                roads = roads.set_crs(epsg=4326)
            roads = roads.to_crs(tcrs)
            print(f"[OK] Roads (GeoPandas): {len(roads)} features; CRS={roads.crs}")
            return roads
        except Exception as e:
            print(f"[WARN] GeoPandas failed for roads: {e}")
    if HAVE_FIONA:
        try:
            geoms = []
            with fiona.open(road_shp_path, 'r') as src:
                src_crs = src.crs
                for ft in src:
                    geom = ft['geometry']
                    if geom is None: continue
                    try:
                        geom_t = transform_geom(src_crs, tcrs, geom)
                    except Exception:
                        geom_t = geom
                    geoms.append(geom_t)
            print(f"[OK] Roads (Fiona): {len(geoms)} features")
            return {'geometries': geoms, 'method': 'fiona', 'crs': tcrs}
        except Exception as e:
            print(f"[WARN] Fiona failed for roads: {e}")
    return None

def plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850):
    if roads is None: return
    try:
        if HAVE_GPD and hasattr(roads, 'plot'):
            try:
                roads.geometry = roads.geometry.apply(
                    lambda g: g if (g is None or g.geom_type in ("LineString", "MultiLineString")) else None
                )
            except Exception:
                pass
            roads.plot(ax=ax, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (GeoPandas)")
        elif isinstance(roads, dict) and 'geometries' in roads:
            for geom in roads['geometries']:
                if geom is None: continue
                if geom['type'] == 'LineString':
                    xs = [p[0] for p in geom['coordinates']]
                    ys = [p[1] for p in geom['coordinates']]
                    ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
                elif geom['type'] == 'MultiLineString':
                    for line in geom['coordinates']:
                        xs = [p[0] for p in line]
                        ys = [p[1] for p in line]
                        ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (Fiona fallback)")
    except Exception as e:
        print(f"[WARN] Failed to plot roads: {e}")

def draw_boundary(ax, shp_path, target_crs, zorder=999, color="black", linewidth=0.7):
    if os.path.exists(shp_path):
        try:
            tcrs = str(target_crs) if not isinstance(target_crs, str) else target_crs
            if HAVE_GPD:
                gdf = gpd.read_file(shp_path).to_crs(tcrs)
                gdf.boundary.plot(ax=ax, color=color, linewidth=linewidth, alpha=1.0, zorder=zorder)
                print("[OK] Boundary drawn from shapefile (GeoPandas)")
                return True
            elif HAVE_FIONA:
                geoms = []
                with fiona.open(shp_path, 'r') as src:
                    src_crs = src.crs
                    for ft in src:
                        if ft['geometry'] is None: continue
                        geoms.append(transform_geom(src_crs, tcrs, ft['geometry']))
                for geom in geoms:
                    if geom['type'] == 'Polygon':
                        xs = [p[0] for p in geom['coordinates'][0]]
                        ys = [p[1] for p in geom['coordinates'][0]]
                        ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
                    elif geom['type'] == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs = [p[0] for p in poly[0]]
                            ys = [p[1] for p in poly[0]]
                            ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
                print("[OK] Boundary drawn from shapefile (Fiona)")
                return True
        except Exception as ex:
            print(f"[WARN] Boundary-from-shp failed: {ex}")
            
    # Fallback to mask only if CRS matches
    if os.path.exists(MASK_TIF):
        try:
            with rasterio.open(MASK_TIF) as ms:
                if ms.crs == target_crs:
                    mask = ms.read(1); b = ms.bounds
                    extent = (b.left, b.right, b.bottom, b.top)
                    ax.contour(mask, levels=[0.5], colors=color, linewidths=linewidth,
                               extent=extent, origin='upper', zorder=zorder)
                    print("[OK] Boundary drawn from mask")
                    return True
        except Exception as ex:
            print(f"[WARN] Boundary-from-mask failed: {ex}")
    return False

# =========================================================
# Study polygon clip-path in raster CRS
# =========================================================
def get_study_polygon_in_crs(target_crs):
    if not HAVE_SHAPELY:
        print("[WARN] Shapely not available; strict clip disabled")
        return None
    tcrs = str(target_crs) if not isinstance(target_crs, str) else target_crs

    # Prefer shapefile
    if os.path.exists(SHAPEFILE):
        try:
            if HAVE_GPD:
                gdf = gpd.read_file(SHAPEFILE)
                if gdf.crs is None: gdf = gdf.set_crs(epsg=4326)
                gdf = gdf.to_crs(tcrs)
                return unary_union(gdf.geometry)
            elif HAVE_FIONA:
                geoms = []
                with fiona.open(SHAPEFILE, 'r') as src:
                    src_crs = src.crs
                    for ft in src:
                        if ft['geometry'] is None: continue
                        geoms.append(shp_shape(transform_geom(src_crs, tcrs, ft['geometry'])))
                if geoms: return unary_union(geoms)
        except Exception as e:
            print(f"[WARN] Polygon from shapefile failed: {e}")

    # Fallback: polygonize mask and transform
    if os.path.exists(MASK_TIF):
        try:
            with rasterio.open(MASK_TIF) as ds:
                data = ds.read(1); trans = ds.transform; crs = ds.crs
                polys = []
                for geom, val in rio_features.shapes(data.astype('uint8'), mask=(data > 0), transform=trans):
                    if val != 1: continue
                    g = shp_shape(geom)
                    if HAVE_GPD:
                        g = gpd.GeoSeries([g], crs=crs).to_crs(tcrs).geometry.iloc[0]
                    polys.append(g)
                if polys: return unary_union(polys)
        except Exception as e:
            print(f"[WARN] Polygon from mask failed: {e}")
    return None

def polygon_to_pathpatch(ax, poly):
    if not (HAVE_SHAPELY and (poly is not None)): return None

    def _ring_to_vertices(ring):
        xs, ys = ring.coords.xy
        verts = list(zip(xs, ys))
        if verts[0] != verts[-1]:
            verts.append(verts[0])
        codes = [MplPath.MOVETO] + [MplPath.LINETO]*(len(verts)-2) + [MplPath.CLOSEPOLY]
        return verts, codes

    if isinstance(poly, ShpPolygon):
        polys = [poly]
    else:
        polys = list(poly.geoms)

    vertices = []; codes = []
    for p in polys:
        v, c = _ring_to_vertices(p.exterior); vertices += v; codes += c
        for hole in p.interiors:
            v, c = _ring_to_vertices(hole); vertices += v; codes += c
    return PathPatch(MplPath(vertices, codes), transform=ax.transData,
                     facecolor='none', edgecolor='none')

# =========================================================
# Core plotting
# =========================================================
def plot_component(tif_path, title, out_prefix, vmin=V_MIN, vmax=V_MAX,
                   cmap_base=DIV_CMAP_BASE):
    if not os.path.exists(tif_path):
        print(f"[WARN] Missing raster: {tif_path}")
        return

    # Read raster
    with rasterio.open(tif_path) as ds:
        arr = ds.read(1)
        nodata = ds.nodata
        b = ds.bounds
        raster_crs = ds.crs
        extent = (b.left, b.right, b.bottom, b.top)
        crs_is_geographic = bool(raster_crs and raster_crs.is_geographic)

    west, east, south, north = extent

    # Mask nodata
    bad = ~np.isfinite(arr)
    if nodata is not None:
        bad |= (arr == nodata)
    marr = np.ma.array(arr, mask=bad)
    cmap = cmap_base.copy()
    cmap.set_bad(alpha=0.0)

    # DMS tick formatters
    xmid = (west + east) / 2.0
    ymid = (south + north) / 2.0
    if crs_is_geographic:
        def fmt_lon(x, pos): return to_dms(x, True)
        def fmt_lat(y, pos): return to_dms(y, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError("Projected raster requires pyproj for DMS axis labels. Please install 'pyproj'.")
        to_ll = Transformer.from_crs(raster_crs, "EPSG:4326", always_xy=True)
        def fmt_lon(x, pos):
            lon, _ = to_ll.transform(x, ymid)
            return to_dms(lon, True)
        def fmt_lat(y, pos):
            _, lat = to_ll.transform(xmid, y)
            return to_dms(lat, False)

    # Overlays 
    roads = load_roads(ROAD_SHP, raster_crs) if os.path.exists(ROAD_SHP) else None

    # Strict study polygon clip 
    study_poly = get_study_polygon_in_crs(raster_crs)

    # ============ MAP ============
    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE, facecolor='white')
    ax.set_facecolor('white')

    im = ax.imshow(marr, extent=extent, origin='upper',
                   vmin=vmin, vmax=vmax, cmap=cmap, interpolation='none', zorder=1)

    # HARD clip-path so nothing paints outside AOI
    clip_patch = polygon_to_pathpatch(ax, study_poly)
    if clip_patch is not None:
        try:
            ax.add_patch(clip_patch)
            im.set_clip_path(clip_patch)
        except Exception as e:
            print(f"[WARN] Clip-path apply failed: {e}")

    # Highways & Boundary
    plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850)
    if clip_patch is not None:
        try:
            for art in ax.lines + ax.patches:
                if art is not clip_patch:
                    art.set_clip_path(clip_patch)
        except Exception:
            pass
    draw_boundary(ax, SHAPEFILE, raster_crs, zorder=999, color="black", linewidth=0.7)

    # North arrow + Scale bar 
    add_north_arrow_tri(ax, size=0.09, pad=0.09, arrow_downshift=0.025)
    add_checker_scale_bar(ax, west, east, south, north,
                          crs_is_geographic=crs_is_geographic,
                          segments=4, fontsize=4)

    # Axis labels + DMS tick formatting 
    ax.set_xlabel("Longitude", fontsize=5, fontweight='bold')
    ax.set_ylabel("Latitude",  fontsize=5, fontweight='bold')
    if crs_is_geographic:
        ax.xaxis.set_major_locator(MultipleLocator(0.1))
        ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_lat))
    ax.tick_params(axis='x', which='major', labelsize=3)
    ax.tick_params(axis='y', which='major', labelsize=3)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45); lbl.set_ha('right'); lbl.set_fontsize(3)

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='red', lw=1.0, label='Highway'),
        Line2D([0], [0], color='black', lw=1.0, label='Study Area')
    ]
    legend_box = ax.legend(
        handles=legend_elements,
        loc='center',
        bbox_to_anchor=(0.78, 0.15),
        bbox_transform=ax.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=2.5,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0
    )
    legend_box.get_frame().set_edgecolor('lightgray')
    legend_box.get_frame().set_linewidth(0.3)

    # Side colorbar 
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb = plt.colorbar(im, cax=cax)
    cb.ax.tick_params(labelsize=3)
    cb.set_label("Unit value (−1 to 1)", fontsize=4, fontweight='bold')

    # Optional: lock view to polygon bounds
    if (study_poly is not None) and HAVE_SHAPELY:
        minx, miny, maxx, maxy = study_poly.bounds
        ax.set_xlim(minx, maxx)
        ax.set_ylim(miny, maxy)

    map_png = os.path.join(STEP34, f"{out_prefix}_map.png")
    map_pdf = os.path.join(STEP34, f"{out_prefix}_map.pdf")
    plt.tight_layout()
    plt.savefig(map_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(map_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"[OK] Saved: {map_png}")
    print(f"[OK] Saved: {map_pdf}")
    plt.show()

    # ============ HISTOGRAM ============
    vals = arr[~bad]
    vals = np.clip(vals, vmin, vmax)

    fig2, ax2 = plt.subplots(1, 1, figsize=HIST_FIGSIZE, facecolor='white')
    bins = np.linspace(vmin, vmax, HIST_BINS + 1)
    counts, edges = np.histogram(vals, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2.0
    ax2.bar(centers, counts, width=(bins[1]-bins[0])*0.9,
            color="#8fa6ff", edgecolor="black", linewidth=0.4, alpha=0.9)
    ax2.set_xlabel("Unit value (−1 to 1)", fontsize=4, fontweight='bold')
    ax2.set_ylabel("Pixel count",           fontsize=4, fontweight='bold')
    ax2.tick_params(axis='both', which='major', labelsize=4)
    ax2.set_xlim(vmin, vmax)
    ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.4, axis='y')
    ax2.set_axisbelow(True)

    hist_png = os.path.join(STEP34, f"{out_prefix}_hist.png")
    hist_pdf = os.path.join(STEP34, f"{out_prefix}_hist.pdf")
    plt.tight_layout(pad=1.2)
    plt.savefig(hist_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(hist_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"[OK] Saved: {hist_png}")
    print(f"[OK] Saved: {hist_pdf}")
    plt.show()

# =========================================================
# Run all three components
# =========================================================
if __name__ == "__main__":
    try:
        plot_component(L_EAST_TIF,  "l̂ (east component)",  "l_hat_east")
        plot_component(L_NORTH_TIF, "l̂ (north component)", "l_hat_north")
        plot_component(L_UP_TIF,    "l̂ (up component)",    "l_hat_up")
    except Exception as e:
        print("[ERROR] Visualization failed:", e)
        traceback.print_exc()


# Step-3.5: Downslope Velocity (DV) from LOS and ŝ

In [ ]:
# =========================================================
# Step 3.5 — Downslope velocity (DV) 
# =========================================================

import os, json
from datetime import datetime
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling, transform_geom
from rasterio import features

pp = lambda m: print(f"[{datetime.now():%H:%M:%S}] {m}", flush=True)

# ----------------- PATHS ----------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

# Anchor & mask
ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
MASK_TIF   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP  = os.path.join(BASE, "Study_Area.shp")

# Inputs from Step-3.2 (LOS)
STEP32_DIR = os.path.join(BASE, "Results", "Step-3.2-LOS")
LOS_MEAN   = os.path.join(STEP32_DIR, "los_velocity_mean_mm_yr_32645_12p5m_float32.tif")
LOS_COUNT  = os.path.join(STEP32_DIR, "los_velocity_count_32645_12p5m_int16.tif")

# Inputs from Step-3.3 (ŝ components)
STEP33_DIR = os.path.join(BASE, "Results", "Step-3.3-Downslope unit vector (ŝ)")
S_EAST     = os.path.join(STEP33_DIR, "s_hat_east_32645_12p5m_float32.tif")
S_NORTH    = os.path.join(STEP33_DIR, "s_hat_north_32645_12p5m_float32.tif")
S_UP       = os.path.join(STEP33_DIR, "s_hat_up_32645_12p5m_float32.tif")

# Inputs from Step-3.4 (l̂ components)
STEP34_DIR = os.path.join(BASE, "Results", "Step-3.4-LOS unit (l̂)")
L_EAST     = os.path.join(STEP34_DIR, "l_hat_east_32645_12p5m_float32.tif")
L_NORTH    = os.path.join(STEP34_DIR, "l_hat_north_32645_12p5m_float32.tif")
L_UP_PREFS = [
    os.path.join(STEP34_DIR, "l_hat_up_32645_12p5m_float32.tif"),
    os.path.join(STEP34_DIR, "l_hat_up_32645_12p5m_float32.mat.tif"),
]

# Outputs
OUT_DIR   = os.path.join(BASE, "Results", "Step-3.5-DV"); os.makedirs(OUT_DIR, exist_ok=True)
DOT_TIF   = os.path.join(OUT_DIR, "los_dot_downslope_32645_12p5m_float32.tif")
DV_TIF    = os.path.join(OUT_DIR, "downslope_velocity_mm_yr_32645_12p5m_float32.tif")
MSK_TIF   = os.path.join(OUT_DIR, "dv_valid_mask_32645_12p5m_byte.tif")
LOWDOT_TIF = os.path.join(OUT_DIR, "dv_lowgeom_flag_32645_12p5m_byte.tif")  # QA flag
STATS_JSON = os.path.join(OUT_DIR, "dv_stats.json")

# ----------------- PARAMETERS -----------------
FLOAT_NODATA     = -9999.0
MIN_COUNT        = 1

# Geometry handling
DOT_MIN_ABS      = None   # HARD gate: keep pixels where |ŝ·l̂| ≥ 0.05 (None disables hard gate)
DOT_REG_EPS      = 0.05   # SOFT floor used only when DOT_MIN_ABS is None

RENORMALIZE_VECTORS = True
EPS = 1e-6

# ----------------- HELPERS -----------------
def read_anchor(path):
    if not os.path.exists(path):
        raise SystemExit(f"Missing anchor grid: {path}")
    with rasterio.open(path) as ds:
        return {
            "tf": ds.transform, "crs": ds.crs,
            "width": ds.width, "height": ds.height,
            "profile": ds.profile
        }

def write_float32(path, arr, dst_spec, nodata=FLOAT_NODATA):
    prof = dst_spec["profile"].copy()
    prof.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=nodata,
        compress="LZW",
        tiled=True,
        transform=dst_spec["tf"],
        height=dst_spec["height"],
        width=dst_spec["width"],
        crs=dst_spec["crs"],
        bigtiff="IF_SAFER",  # FIX: correct key
    )
    tmp = path + ".part"
    with rasterio.open(tmp, "w", **prof) as ds:
        out = np.where(np.isfinite(arr), arr, nodata).astype("float32")
        ds.write(out, 1)
    os.replace(tmp, path)

def write_byte(path, arr01, dst_spec):
    prof = dst_spec["profile"].copy()
    prof.update(
        driver="GTiff",
        dtype="uint8",
        count=1,
        nodata=0,
        compress="LZW",
        tiled=True,
        transform=dst_spec["tf"],
        height=dst_spec["height"],
        width=dst_spec["width"],
        crs=dst_spec["crs"],
        bigtiff="IF_SAFER",  # FIX: correct key
    )
    tmp = path + ".part"
    with rasterio.open(tmp, "w", **prof) as ds:
        ds.write(arr01.astype("uint8"), 1)
    os.replace(tmp, path)

def reproject_to_anchor(src_ds, dst_spec, resampling=Resampling.bilinear):
    same = (
        src_ds.crs == dst_spec["crs"] and
        src_ds.transform == dst_spec["tf"] and
        src_ds.width == dst_spec["width"] and
        src_ds.height == dst_spec["height"]
    )
    if same:
        arr = src_ds.read(1).astype("float64")
        nd  = src_ds.nodata
    else:
        arr = np.full((dst_spec["height"], dst_spec["width"]), np.nan, dtype="float64")
        reproject(
            source=src_ds.read(1).astype("float64"),
            destination=arr,
            src_transform=src_ds.transform,
            src_crs=src_ds.crs,
            dst_transform=dst_spec["tf"],
            dst_crs=dst_spec["crs"],
            resampling=resampling,
            src_nodata=src_ds.nodata,
            dst_nodata=np.nan,
        )
        nd = src_ds.nodata

    # map explicit nodata to NaN if metadata exists
    if nd is not None and np.isfinite(nd):
        arr = np.where(arr == nd, np.nan, arr)

    # defensive scrubbing (common sentinels / insane ranges)
    arr = np.where(
        (arr == -9999.0) | (arr == -32768.0) | (arr < -1e6) | (arr > 1e6),
        np.nan,
        arr
    )
    return arr

def read_to_anchor(path, dst_spec, resampling=Resampling.bilinear):
    """FIX: forwards 'resampling' properly."""
    if not os.path.exists(path):
        raise SystemExit(f"Missing input: {path}")
    with rasterio.open(path) as src:
        return reproject_to_anchor(src, dst_spec, resampling=resampling).astype("float32")

def make_mask_from_shapefile(shp_path, dst_spec):
    if not os.path.exists(shp_path):
        return None
    try:
        import fiona
    except Exception:
        pp("Fiona not available; cannot read shapefile for mask.")
        return None

    shapes = []
    with fiona.open(shp_path, "r") as src:
        src_crs = src.crs_wkt or src.crs
        for feat in src:
            geom = feat.get("geometry")
            if geom is None:
                continue
            try:
                geom_t = transform_geom(src_crs, dst_spec["crs"].to_string(), geom, precision=6)
            except Exception:
                geom_t = transform_geom(src_crs, dst_spec["crs"], geom, precision=6)
            shapes.append(geom_t)

    if not shapes:
        return None

    mask = features.rasterize(
        [(g, 1) for g in shapes],
        out_shape=(dst_spec["height"], dst_spec["width"]),
        transform=dst_spec["tf"],
        fill=0,
        all_touched=False,
        dtype="uint8"
    )
    return mask

def read_mask(dst_spec):
    if os.path.exists(MASK_TIF):
        with rasterio.open(MASK_TIF) as ms:
            same = (
                ms.crs == dst_spec["crs"] and
                ms.transform == dst_spec["tf"] and
                ms.width == dst_spec["width"] and
                ms.height == dst_spec["height"]
            )
            if same:
                m = ms.read(1)
            else:
                m = np.zeros((dst_spec["height"], dst_spec["width"]), dtype=np.float32)
                reproject(
                    source=ms.read(1).astype("float32"),
                    destination=m,
                    src_transform=ms.transform,
                    src_crs=ms.crs,
                    dst_transform=dst_spec["tf"],
                    dst_crs=dst_spec["crs"],
                    resampling=Resampling.nearest,
                    src_nodata=0,
                    dst_nodata=0
                )
        m = (m > 0).astype("uint8")
        pp("Using study_mask.tif for clipping.")
        return m

    pp("study_mask.tif not found — trying to rasterize Study_Area.shp …")
    m = make_mask_from_shapefile(STUDY_SHP, dst_spec)
    if m is not None:
        pp("Shapefile mask created successfully.")
    else:
        pp("Could not create mask from shapefile — outputs will NOT be clipped.")
    return m

def renormalize(u_e, u_n, u_u):
    mag = np.sqrt(u_e*u_e + u_n*u_n + u_u*u_u)
    good = np.isfinite(mag) & (mag > EPS)
    u_e2 = np.full_like(u_e, np.nan, dtype="float32")
    u_n2 = np.full_like(u_n, np.nan, dtype="float32")
    u_u2 = np.full_like(u_u, np.nan, dtype="float32")
    u_e2[good] = (u_e[good] / mag[good]).astype("float32")
    u_n2[good] = (u_n[good] / mag[good]).astype("float32")
    u_u2[good] = (u_u[good] / mag[good]).astype("float32")
    return u_e2, u_n2, u_u2

# ----------------- MAIN -----------------
def main():
    pp("=== Step 3.5 — Downslope velocity (DV) ===")
    dst = read_anchor(ANCHOR_TIF)
    pp(f"Anchor: {dst['width']}×{dst['height']} @ {dst['crs']}")

    # Mask
    msk = read_mask(dst)  # uint8 or None
    inside = (msk == 1) if msk is not None else None

    # Read LOS mean + count
    los_mean = read_to_anchor(LOS_MEAN, dst, resampling=Resampling.bilinear)

    count = None
    if os.path.exists(LOS_COUNT):
        # FIX EFFECTIVE: nearest is now truly used
        count = read_to_anchor(LOS_COUNT, dst, resampling=Resampling.nearest)

    # Read ŝ and l̂ components
    s_e = read_to_anchor(S_EAST,  dst, resampling=Resampling.bilinear)
    s_n = read_to_anchor(S_NORTH, dst, resampling=Resampling.bilinear)
    s_u = read_to_anchor(S_UP,    dst, resampling=Resampling.bilinear)

    l_up_path = next((p for p in L_UP_PREFS if os.path.exists(p)), None)
    if not l_up_path:
        raise SystemExit("Missing l̂ up component file (tried both names).")

    l_e = read_to_anchor(L_EAST,  dst, resampling=Resampling.bilinear)
    l_n = read_to_anchor(L_NORTH, dst, resampling=Resampling.bilinear)
    l_u = read_to_anchor(l_up_path, dst, resampling=Resampling.bilinear)

    # Optional re-normalization
    if RENORMALIZE_VECTORS:
        s_e, s_n, s_u = renormalize(s_e, s_n, s_u)
        l_e, l_n, l_u = renormalize(l_e, l_n, l_u)

    # dot = ŝ·l̂ (clamp for numerical drift)
    dot = (s_e * l_e) + (s_n * l_n) + (s_u * l_u)
    dot = np.where(np.isfinite(dot), dot, np.nan).astype("float32")
    dot = np.clip(dot, -1.0, 1.0)

    # ---------------- Build validity ----------------
    m_valid = np.ones(dot.shape, dtype=bool)
    if inside is not None:
        m_valid &= inside

    m_finite = np.isfinite(los_mean) & np.isfinite(dot)
    m_valid &= m_finite

    if count is not None:
        m_count = (np.isfinite(count) & (count >= float(MIN_COUNT)))
        m_valid &= m_count
    else:
        m_count = np.ones_like(m_valid, dtype=bool)

    # Geometry handling: HARD gate vs SOFT regularization
    if DOT_MIN_ABS is not None:
        m_geom = (np.abs(dot) >= float(DOT_MIN_ABS))
        m_valid &= m_geom
        denom = dot
        geom_used = f"|ŝ·l̂| ≥ {DOT_MIN_ABS}"
        lowdot_flag = (np.abs(dot) < float(DOT_MIN_ABS))
    else:
        denom = np.sign(dot) * np.maximum(np.abs(dot), float(DOT_REG_EPS))
        geom_used = f"regularized with |ŝ·l̂| floor = {DOT_REG_EPS}"
        m_geom = np.ones_like(m_valid, dtype=bool)
        lowdot_flag = (np.abs(dot) < float(DOT_REG_EPS))

    # DV
    dv = np.full_like(dot, np.nan, dtype="float32")
    good = m_valid & np.isfinite(denom)
    dv[good] = (los_mean[good] / denom[good]).astype("float32")

    # Diagnostic dot clipped to AOI only
    dot_clip = np.full_like(dot, np.nan, dtype="float32")
    if inside is not None:
        mm = inside & np.isfinite(dot)
        dot_clip[mm] = dot[mm]
    else:
        mm = np.isfinite(dot)
        dot_clip[mm] = dot[mm]

    # ---------------- Write outputs ----------------
    write_float32(DOT_TIF, dot_clip, dst, nodata=FLOAT_NODATA)
    write_float32(DV_TIF,  dv,       dst, nodata=FLOAT_NODATA)
    write_byte(MSK_TIF,    m_valid.astype("uint8"), dst)
    write_byte(LOWDOT_TIF, lowdot_flag.astype("uint8"), dst)

    # ---------------- Diagnostics ----------------
    total = dot.size
    pct_aoi = lambda x: 100.0 * x / max(total, 1)

    n_inside = int(inside.sum()) if inside is not None else total
    n_finite = int((m_finite & (inside if inside is not None else np.ones_like(m_finite, bool))).sum())
    n_count  = int((m_count  & m_finite & (inside if inside is not None else np.ones_like(m_finite, bool))).sum())
    n_geom   = int((m_geom   & m_count & m_finite & (inside if inside is not None else np.ones_like(m_finite, bool))).sum())
    n_valid  = int(m_valid.sum())

    pp("Coverage breakdown (relative to AOI):")
    pp(f"  Inside study mask : {n_inside:,} ({pct_aoi(n_inside):.2f}%)")
    pp(f"  Finite inputs     : {n_finite:,} ({pct_aoi(n_finite):.2f}%)")
    if count is not None:
        pp(f"  Count≥{MIN_COUNT:<3}           : {n_count:,} ({pct_aoi(n_count):.2f}%)")
    pp(f"  Geometry gate       : {geom_used}")
    pp(f"  |ŝ·l̂| pass         : {n_geom:,} ({pct_aoi(n_geom):.2f}%)")
    pp(f"  VALID DV pixels     : {n_valid:,} ({pct_aoi(n_valid):.3f}%)")

    if inside is not None:
        denom_study = int(inside.sum())
        pct_study = lambda x: 100.0 * x / max(denom_study, 1)
        los_finite_study = int((inside & np.isfinite(los_mean)).sum())
        los_geom_finite  = int((inside & np.isfinite(los_mean) & np.isfinite(dot)).sum())
        valid_in_study   = int((inside & m_valid).sum())
        pp("[CHECK] (relative to STUDY mask)")
        pp(f"  LOS finite in study           : {los_finite_study:,} ({pct_study(los_finite_study):.3f}%)")
        pp(f"  LOS∧geom finite in study      : {los_geom_finite:,} ({pct_study(los_geom_finite):.3f}%)")
        pp(f"  VALID DV pixels in study      : {valid_in_study:,} ({pct_study(valid_in_study):.3f}%)")

    # Stats JSON
    if n_valid > 0:
        vv = dv[m_valid].astype("float64")
        stats = {
            "valid_pixels": int(vv.size),
            "los_mean_mm_yr": {
                "min": float(np.nanmin(los_mean[m_valid])),
                "mean": float(np.nanmean(los_mean[m_valid])),
                "max": float(np.nanmax(los_mean[m_valid])),
            },
            "dot_shat_lhat": {
                "min": float(np.nanmin(dot[m_valid])),
                "mean": float(np.nanmean(dot[m_valid])),
                "max": float(np.nanmax(dot[m_valid])),
            },
            "dv_mm_yr": {
                "min": float(np.nanmin(vv)),
                "mean": float(np.nanmean(vv)),
                "p50": float(np.nanpercentile(vv, 50)),
                "p95": float(np.nanpercentile(vv, 95)),
                "max": float(np.nanmax(vv)),
            },
            "params": {
                "MIN_COUNT": int(MIN_COUNT),
                "geometry_mode": "hard" if DOT_MIN_ABS is not None else "soft",
                "DOT_MIN_ABS": DOT_MIN_ABS,
                "DOT_REG_EPS": float(DOT_REG_EPS) if DOT_MIN_ABS is None else None,
                "float_nodata": float(FLOAT_NODATA),
                "renormalize_vectors": bool(RENORMALIZE_VECTORS)
            }
        }
    else:
        stats = {"valid_pixels": 0, "note": "No valid pixels with current thresholds."}

    with open(STATS_JSON, "w") as f:
        json.dump(stats, f, indent=2)

    pp("Saved:")
    pp(f"  {DOT_TIF}")
    pp(f"  {DV_TIF}")
    pp(f"  {MSK_TIF}")
    pp(f"  {LOWDOT_TIF}  (low-geometry QA flag)")
    pp(f"  {STATS_JSON}")
    pp("=== Done (Step 3.5) ===")

if __name__ == "__main__":
    main()


# Visualization: Downslope Velocity (DV)

In [ ]:
# =========================================================
# Step 3.5 — Downslope Velocity (DV) Visualization
# =========================================================
import os, traceback
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.patches import Polygon, Rectangle, PathPatch
from matplotlib.path import Path as MplPath
from mpl_toolkits.axes_grid1 import make_axes_locatable

import rasterio
from rasterio.transform import rowcol
from rasterio.warp import transform as rio_transform, transform_geom
from rasterio import features as rio_features

# ---------- Optional libs ----------
try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    from scipy.ndimage import binary_erosion
    from scipy.spatial import cKDTree
    from scipy.interpolate import griddata, RBFInterpolator
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

try:
    from shapely.geometry import shape as shp_shape, Polygon as ShpPolygon, MultiPolygon as ShpMultiPolygon
    from shapely.ops import unary_union
    HAVE_SHAPELY = True
except Exception:
    HAVE_SHAPELY = False

# =========================================================
# PATHS
# =========================================================
BASE = os.environ.get("LSM_DATA_DIR", "./data")

STUDY_SHP   = os.path.join(BASE, "Study_Area.shp")
STUDY_MASK  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
ROAD_SHP    = os.path.join(BASE, "NTD_Road.shp")

OUTPUT_DIR  = os.path.join(BASE, "Results", "Step-3.5-DV")
os.makedirs(OUTPUT_DIR, exist_ok=True)

DV_TIF      = os.path.join(OUTPUT_DIR, "downslope_velocity_mm_yr_32645_12p5m_float32.tif")
MASK_TIF    = os.path.join(OUTPUT_DIR, "dv_valid_mask_32645_12p5m_byte.tif")

# =========================================================
# DISPLAY SETTINGS 
# =========================================================
MAP_FIGSIZE  = (2, 2)
HIST_FIGSIZE = (2, 2)
CAX_SIZE     = "5%"
CAX_PAD      = 0.01

# Color scaling
CONTRAST_PERCENTILE  = 98
FORCE_SYMMETRIC_ZERO = True
ABS_CAP_MM_PER_YR    = 30.0

# Interpolation settings 
GRID_RES                 = 200
USE_RBF                  = True
IDW_NEIGHBORS            = 64
MASK_EROSION_ITERATIONS  = 2

# Colormap 
DIV_COLORS = ['#000080','#0000FF','#4169E1','#87CEEB','#E0F6FF',
              '#FFFFFF','#FFE0E0','#FFB6C1','#FF4500','#DC143C','#8B0000']
DV_CMAP = mcolors.LinearSegmentedColormap.from_list('dv', DIV_COLORS, N=256)

# =========================================================
# Tiny-style helpers
# =========================================================
def to_dms(val, is_lon=True):
    a = abs(val); d = int(a)
    m_float = (a - d) * 60.0; m = int(m_float); s = (m_float - m) * 60.0
    hemi = ('E' if val >= 0 else 'W') if is_lon else ('N' if val >= 0 else 'S')
    return f"{d}°{m:02d}'{s:04.1f}\"{hemi}"

def add_north_arrow_tri(ax, size=0.09, pad=0.07, label='N', arrow_downshift=0.025, label_upshift=0.0):
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10 + label_upshift
    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=1.0, zorder=6))
    ax.text(label_anchor_x, label_anchor_y, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=8, fontweight='bold',
            color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1, 2, 5, 10):
        if m*base >= target: return m*base
    return 10*base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None, segments=4, fontsize=6,
                          tile_lw=0.3, frame_lw=0.4):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd

    total_len_m = nice_len_125(width_m / 5.0); seg_len_m = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x

    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data
    pad_x   = total_len_units * 0.08; pad_y = 0.035 * height_data

    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        face = 'black' if i % 2 == 0 else 'white'
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData, facecolor=face,
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=frame_lw, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.012 * height_data, "km",
            ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.012 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.012 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Interpolation 
# =========================================================
def interpolate_grid(x, y, v, west, east, south, north, grid_res=GRID_RES):
    gx = np.linspace(west,  east,  grid_res)
    gy = np.linspace(south, north, grid_res)
    Xg, Yg = np.meshgrid(gx, gy)
    pts = np.column_stack([x, y])

    if HAVE_SCIPY:
        grid_pts = np.column_stack([Xg.ravel(), Yg.ravel()])
        try:
            if USE_RBF:
                rbf = RBFInterpolator(pts, v, kernel='thin_plate_spline', smoothing=0.1, epsilon=0.1)
                Vg = rbf(grid_pts).reshape(grid_res, grid_res)
            else:
                Vg = griddata(pts, v, grid_pts, method='cubic', fill_value=np.nan).reshape(grid_res, grid_res)
            tree = cKDTree(pts)
            distances, _ = tree.query(grid_pts, k=1)
            Vg[distances.reshape(grid_res, grid_res) > ((east - west) * 0.01)] = np.nan
            return Xg, Yg, Vg
        except Exception as e:
            print(f"[WARN] SciPy interpolation failed ({e}); falling back to IDW...")

    if HAVE_SCIPY:
        tree = cKDTree(pts)
        grid_pts = np.column_stack([Xg.ravel(), Yg.ravel()])
        k = min(IDW_NEIGHBORS, len(pts))
        dist, idx = tree.query(grid_pts, k=k)
        w = 1.0 / (dist + 1e-12)
        vv = v[idx] if k > 1 else v[idx][:, None]
        V_flat = (w * vv).sum(axis=1) / w.sum(axis=1)
        Vg = V_flat.reshape(grid_res, grid_res)
        return Xg, Yg, Vg

    Vg = np.zeros((grid_res, grid_res), dtype=float)
    for i in range(grid_res):
        Xi = Xg[i]; Yi = Yg[i]
        for j in range(grid_res):
            d = np.hypot(pts[:, 0] - Xi[j], pts[:, 1] - Yi[j])
            w = 1.0 / (d + 1e-12)
            Vg[i, j] = (w * v).sum() / w.sum()
    return Xg, Yg, Vg

# =========================================================
# Highways + Boundary
# =========================================================
def load_roads(road_shp_path, target_crs):
    if not os.path.exists(road_shp_path):
        print(f"[WARN] Road shapefile not found: {road_shp_path}")
        return None
    target_crs_str = str(target_crs) if not isinstance(target_crs, str) else target_crs
    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_shp_path)
            if roads.crs is None:
                print("[WARN] Road CRS is None; assuming EPSG:4326")
                roads = roads.set_crs(epsg=4326)
            roads = roads.to_crs(target_crs_str)
            print(f"[OK] Roads (GeoPandas): {len(roads)} features; CRS={roads.crs}")
            return roads
        except Exception as e:
            print(f"[WARN] GeoPandas failed for roads: {e}")
    if HAVE_FIONA:
        try:
            geoms = []
            with fiona.open(road_shp_path, 'r') as shp:
                src_crs = shp.crs
                for feat in shp:
                    geom = feat['geometry']
                    if geom is None: continue
                    try:
                        geom_t = transform_geom(src_crs, target_crs_str, geom)
                    except Exception as e:
                        print(f"[WARN] Road transform failed; keeping original: {e}")
                        geom_t = geom
                    geoms.append(geom_t)
            print(f"[OK] Roads (Fiona): {len(geoms)} features")
            return {'geometries': geoms, 'method': 'fiona', 'crs': target_crs_str}
        except Exception as e:
            print(f"[WARN] Fiona failed for roads: {e}")
    print("[WARN] Unable to load roads")
    return None

def plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850):
    if roads is None: return
    try:
        if HAVE_GPD and hasattr(roads, 'plot'):
            try:
                roads.geometry = roads.geometry.apply(
                    lambda g: g if g is None or g.geom_type in ("LineString", "MultiLineString") else None
                )
            except Exception:
                pass
            roads.plot(ax=ax, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (GeoPandas)")
        elif isinstance(roads, dict) and 'geometries' in roads:
            for geom in roads['geometries']:
                if geom is None: continue
                if geom['type'] == 'LineString':
                    xs = [p[0] for p in geom['coordinates']]
                    ys = [p[1] for p in geom['coordinates']]
                    ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
                elif geom['type'] == 'MultiLineString':
                    for line in geom['coordinates']:
                        xs = [p[0] for p in line]
                        ys = [p[1] for p in line]
                        ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (Fiona fallback)")
    except Exception as e:
        print(f"[WARN] Failed to plot roads: {e}")

def draw_boundary(ax, shp_path, target_crs, zorder=999, color="black", linewidth=0.7):
    if os.path.exists(shp_path):
        try:
            target_crs_str = str(target_crs) if not isinstance(target_crs, str) else target_crs
            if HAVE_GPD:
                gdf = gpd.read_file(shp_path).to_crs(target_crs_str)
                gdf.boundary.plot(ax=ax, color=color, linewidth=linewidth, alpha=1.0, zorder=zorder)
                print("[OK] Boundary drawn from shapefile (GeoPandas)")
                return True
            elif HAVE_FIONA:
                geoms = []
                with fiona.open(shp_path, 'r') as src:
                    src_crs = src.crs
                    for ft in src:
                        if ft['geometry'] is None: continue
                        geoms.append(transform_geom(src_crs, target_crs_str, ft['geometry']))
                for geom in geoms:
                    if geom['type'] == 'Polygon':
                        xs = [p[0] for p in geom['coordinates'][0]]
                        ys = [p[1] for p in geom['coordinates'][0]]
                        ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
                    elif geom['type'] == 'MultiPolygon':
                        for poly in geom['coordinates']:
                            xs = [p[0] for p in poly[0]]
                            ys = [p[1] for p in poly[0]]
                            ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
                print("[OK] Boundary drawn from shapefile (Fiona)")
                return True
        except Exception as ex:
            print(f"[WARN] Boundary-from-shp failed: {ex}")
    if os.path.exists(STUDY_MASK):
        try:
            with rasterio.open(STUDY_MASK) as ms:
                if ms.crs == target_crs:
                    mask = ms.read(1); b = ms.bounds
                    extent = (b.left, b.right, b.bottom, b.top)
                    ax.contour(mask, levels=[0.5], colors=color, linewidths=linewidth,
                               extent=extent, origin='upper', zorder=zorder)
                    print("[OK] Boundary drawn from mask")
                    return True
        except Exception as ex:
            print(f"[WARN] Boundary-from-mask failed: {ex}")
    return False

# =========================================================
# Study polygon clip-path in raster CRS
# =========================================================
def get_study_polygon_in_crs(target_crs):
    if not HAVE_SHAPELY:
        print("[WARN] Shapely not available; strict clip disabled")
        return None
    tcrs = str(target_crs) if not isinstance(target_crs, str) else target_crs

    if os.path.exists(STUDY_SHP):
        try:
            if HAVE_GPD:
                gdf = gpd.read_file(STUDY_SHP)
                if gdf.crs is None: gdf = gdf.set_crs(epsg=4326)
                gdf = gdf.to_crs(tcrs)
                return unary_union(gdf.geometry)
            elif HAVE_FIONA:
                geoms = []
                with fiona.open(STUDY_SHP, 'r') as src:
                    src_crs = src.crs
                    for ft in src:
                        if ft['geometry'] is None: continue
                        geoms.append(shp_shape(transform_geom(src_crs, tcrs, ft['geometry'])))
                if geoms: return unary_union(geoms)
        except Exception as e:
            print(f"[WARN] Polygon from shapefile failed: {e}")

    if os.path.exists(STUDY_MASK):
        try:
            with rasterio.open(STUDY_MASK) as ds:
                data = ds.read(1); trans = ds.transform; crs = ds.crs
                polys = []
                for geom, val in rio_features.shapes(data.astype('uint8'), mask=(data > 0), transform=trans):
                    if val != 1: continue
                    g = shp_shape(geom)
                    if HAVE_GPD:
                        g = gpd.GeoSeries([g], crs=crs).to_crs(tcrs).geometry.iloc[0]
                    polys.append(g)
                if polys: return unary_union(polys)
        except Exception as e:
            print(f"[WARN] Polygon from mask failed: {e}")
    return None

def polygon_to_pathpatch(ax, poly):
    if not (HAVE_SHAPELY and (poly is not None)): return None

    def _ring_to_vertices(ring):
        xs, ys = ring.coords.xy
        verts = list(zip(xs, ys))
        if verts[0] != verts[-1]:
            verts.append(verts[0])
        codes = [MplPath.MOVETO] + [MplPath.LINETO]*(len(verts)-2) + [MplPath.CLOSEPOLY]
        return verts, codes

    if isinstance(poly, ShpPolygon):
        polys = [poly]
    else:
        polys = list(poly.geoms)

    vertices = []; codes = []
    for p in polys:
        v, c = _ring_to_vertices(p.exterior); vertices += v; codes += c
        for hole in p.interiors:
            v, c = _ring_to_vertices(hole); vertices += v; codes += c
    return PathPatch(MplPath(vertices, codes), transform=ax.transData,
                     facecolor='none', edgecolor='none')

# =========================================================
# Main DV Plotting 
# =========================================================
def plot_dv(dv_tif=DV_TIF, mask_tif=MASK_TIF, out_prefix="DV"):
    if not os.path.exists(dv_tif):
        raise FileNotFoundError(f"Missing DV raster: {dv_tif}")

    print("="*60)
    print("DV VISUALIZATION")
    print("="*60)

    with rasterio.open(dv_tif) as ds:
        dv = ds.read(1).astype(np.float32)
        nodata = ds.nodata
        b = ds.bounds
        raster_crs = ds.crs
        transform = ds.transform
        extent = (b.left, b.right, b.bottom, b.top)
        crs_is_geographic = bool(raster_crs and raster_crs.is_geographic)

    dv_clean = dv.copy()
    if nodata is not None:
        dv_clean[dv == nodata] = np.nan
    dv_clean[~np.isfinite(dv_clean)] = np.nan

    if mask_tif and os.path.exists(mask_tif):
        with rasterio.open(mask_tif) as ms:
            mask = ms.read(1).astype(bool)
            if mask.shape == dv_clean.shape:
                dv_clean[~mask] = np.nan
            else:
                print("[WARN] DV valid mask has different shape; skipping DV mask")

    rows, cols = np.where(np.isfinite(dv_clean))
    if rows.size == 0:
        raise RuntimeError("No valid DV values found.")
    xs = transform.c + (cols + 0.5) * transform.a + (rows + 0.5) * transform.b
    ys = transform.f + (cols + 0.5) * transform.d + (rows + 0.5) * transform.e
    v  = dv_clean[rows, cols]
    x, y = np.asarray(xs), np.asarray(ys)

    if os.path.exists(STUDY_MASK):
        try:
            with rasterio.open(STUDY_MASK) as ms:
                mask_data = ms.read(1)
                mask_crs = ms.crs
                mask_transform = ms.transform
                if HAVE_SCIPY:
                    mask_eroded = binary_erosion(mask_data > 0, iterations=MASK_EROSION_ITERATIONS).astype(np.uint8)
                else:
                    mask_eroded = (mask_data > 0).astype(np.uint8)

                if mask_crs != raster_crs:
                    mx, my = rio_transform(raster_crs, mask_crs, x, y)
                else:
                    mx, my = x, y

                keep_pts = []
                for xi, yi in zip(mx, my):
                    r, c = rowcol(mask_transform, xi, yi)
                    inside = (0 <= r < mask_eroded.shape[0]) and (0 <= c < mask_eroded.shape[1]) and (mask_eroded[int(r), int(c)] > 0)
                    keep_pts.append(inside)
                keep_pts = np.array(keep_pts, dtype=bool)
                x, y, v = x[keep_pts], y[keep_pts], v[keep_pts]
                print(f"[OK] Removed {np.sum(~keep_pts)} DV points near/outside boundary after erosion")
        except Exception as ex:
            print(f"[WARN] Boundary erosion failed: {ex}")

    if FORCE_SYMMETRIC_ZERO:
        vmax_data = float(np.percentile(np.abs(v), CONTRAST_PERCENTILE))
        vmax = float(min(ABS_CAP_MM_PER_YR, vmax_data))
        vmin = -vmax
    else:
        vmin = float(np.percentile(v, 100 - CONTRAST_PERCENTILE))
        vmax = float(np.percentile(v, CONTRAST_PERCENTILE))
        vmax = min(vmax, ABS_CAP_MM_PER_YR); vmin = max(vmin, -ABS_CAP_MM_PER_YR)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    print(f"[INFO] DV display range: [{vmin:.2f}, {vmax:.2f}] mm/yr")

    west, east = extent[0], extent[1]
    south, north = extent[2], extent[3]
    bx = (east - west) * 0.02 or 1.0
    by = (north - south) * 0.02 or 1.0
    west, east = west - bx, east + bx
    south, north = south - by, north + by
    extent_buf = [west, east, south, north]

    print("[INFO] Interpolating background grid...")
    Xg, Yg, Vgrid = interpolate_grid(x, y, v, west, east, south, north, grid_res=GRID_RES)

    if os.path.exists(STUDY_MASK):
        try:
            with rasterio.open(STUDY_MASK) as ms:
                mask_data = ms.read(1)
                mask_crs = ms.crs
                mask_transform = ms.transform
                if HAVE_SCIPY:
                    mask_eroded = binary_erosion(mask_data > 0, iterations=MASK_EROSION_ITERATIONS).astype(np.uint8)
                else:
                    mask_eroded = (mask_data > 0).astype(np.uint8)

                Hg, Wg = Vgrid.shape
                for i in range(Hg):
                    for j in range(Wg):
                        gx, gy = Xg[i, j], Yg[i, j]
                        if mask_crs != raster_crs:
                            mx, my = rio_transform(raster_crs, mask_crs, [gx], [gy])
                            mx, my = mx[0], my[0]
                        else:
                            mx, my = gx, gy
                        r, c = rowcol(mask_transform, mx, my)
                        keep = (0 <= r < mask_eroded.shape[0]) and (0 <= c < mask_eroded.shape[1]) and (mask_eroded[int(r), int(c)] > 0)
                        if not keep:
                            Vgrid[i, j] = np.nan
            print("[OK] Grid masked to boundary")
        except Exception as ex:
            print(f"[WARN] Grid boundary masking failed: {ex}")

    xmid = (extent[0] + extent[1]) / 2.0
    ymid = (extent[2] + extent[3]) / 2.0
    if crs_is_geographic:
        def fmt_lon(xv, pos): return to_dms(xv, True)
        def fmt_lat(yv, pos): return to_dms(yv, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError("Projected raster requires 'pyproj' for DMS axis labels.")
        to_ll = Transformer.from_crs(raster_crs, "EPSG:4326", always_xy=True)
        def fmt_lon(xv, pos):
            lon, _ = to_ll.transform(xv, ymid)
            return to_dms(lon, True)
        def fmt_lat(yv, pos):
            _, lat = to_ll.transform(xmid, yv)
            return to_dms(lat, False)

    roads = load_roads(ROAD_SHP, raster_crs) if os.path.exists(ROAD_SHP) else None
    study_poly = get_study_polygon_in_crs(raster_crs)
    clip_patch = None

    # ================= MAP =================
    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE, facecolor='white')
    ax.set_facecolor('white')

    im = ax.imshow(Vgrid, cmap=DV_CMAP, extent=extent_buf, norm=norm,
                   origin='lower', interpolation='bilinear', alpha=1.0, zorder=1)

    if study_poly is not None:
        clip_patch = polygon_to_pathpatch(ax, study_poly)
        if clip_patch is not None:
            try:
                ax.add_patch(clip_patch)
                im.set_clip_path(clip_patch)
            except Exception as e:
                print(f"[WARN] Clip-path apply failed: {e}")

    pts = ax.scatter(x, y, c=v, s=0.6, cmap=DV_CMAP, norm=norm,
                     edgecolors='none', alpha=0.9, zorder=2)
    if clip_patch is not None:
        try:
            pts.set_clip_path(clip_patch)
        except Exception:
            pass

    plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850)
    draw_boundary(ax, STUDY_SHP, raster_crs, zorder=999, color="black", linewidth=0.7)
    if clip_patch is not None:
        try:
            for art in ax.lines + ax.patches:
                if art is not clip_patch:
                    art.set_clip_path(clip_patch)
        except Exception:
            pass

    ax.set_xlabel("Longitude", fontsize=5, fontweight='bold')
    ax.set_ylabel("Latitude",  fontsize=5, fontweight='bold')
    if crs_is_geographic:
        ax.xaxis.set_major_locator(MultipleLocator(0.1))
        ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_lat))
    ax.tick_params(axis='x', which='major', labelsize=3)
    ax.tick_params(axis='y', which='major', labelsize=3)
    for t in ax.get_xticklabels():
        t.set_rotation(45); t.set_ha('right'); t.set_fontsize(3)

    add_north_arrow_tri(ax, size=0.09, pad=0.09)

    # >>> FIXED: call with crs_type / mid_lat_deg instead of crs_is_geographic
    crs_type = 'wgs84' if crs_is_geographic else 'utm'
    add_checker_scale_bar(ax, *extent_buf, crs_type=crs_type,
                          mid_lat_deg=ymid if crs_is_geographic else None,
                          segments=4, fontsize=4)

    from matplotlib.lines import Line2D
    legend_elems = [
        Line2D([0], [0], color='red',   lw=1.0, label='Highway'),
        Line2D([0], [0], color='black', lw=1.0, label='Study Area')
    ]
    legend_box = ax.legend(
        handles=legend_elems,
        loc='center',
        bbox_to_anchor=(0.82, 0.15),
        bbox_transform=ax.transAxes,
        frameon=True, framealpha=0.9, fontsize=2.5,
        labelspacing=0.3, borderpad=0.3, handlelength=1.2,
        handletextpad=0.4, borderaxespad=0.0
    )
    legend_box.get_frame().set_edgecolor('lightgray')
    legend_box.get_frame().set_linewidth(0.3)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb = plt.colorbar(im, cax=cax)
    cb.ax.tick_params(labelsize=3)
    cb.set_label("DV (mm/year)", fontsize=4, fontweight='bold')
    cb.set_ticks(np.linspace(vmin, vmax, 5))

    out_map_png = os.path.join(OUTPUT_DIR, f"{out_prefix} Map.png")
    out_map_pdf = os.path.join(OUTPUT_DIR, f"{out_prefix} Map.pdf")
    plt.tight_layout()
    plt.savefig(out_map_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(out_map_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"[OK] Saved map → {out_map_png}")
    print(f"[OK] Saved map → {out_map_pdf}")
    plt.show()

    # ================= HISTOGRAM =================
    vals_hist = np.clip(v, vmin, vmax)
    fig2, ax2 = plt.subplots(1, 1, figsize=HIST_FIGSIZE, facecolor='white')
    bins = np.linspace(vmin, vmax, 21)
    counts, edges = np.histogram(vals_hist, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2.0
    ax2.bar(centers, counts, width=(bins[1]-bins[0])*0.9,
            color="#8fa6ff", edgecolor="black", linewidth=0.4, alpha=0.9)
    ax2.axvline(0, color='k', lw=0.8, label='Zero velocity')
    ax2.axvline(np.mean(v),   color='r',      ls='--', lw=0.7, label=f"Mean: {np.mean(v):.2f}")
    ax2.axvline(np.median(v), color='orange', ls='--', lw=0.7, label=f"Median: {np.median(v):.2f}")
    ax2.legend(fontsize=3, loc='upper right', framealpha=0.9)
    ax2.set_xlabel("DV (mm/year)", fontsize=4, fontweight='bold')
    ax2.set_ylabel("Number of Pixels", fontsize=4, fontweight='bold')
    ax2.tick_params(axis='both', which='major', labelsize=4)
    ax2.set_xlim(vmin, vmax)
    ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.4, axis='y')
    ax2.set_axisbelow(True)

    out_hist_png = os.path.join(OUTPUT_DIR, f"{out_prefix} Histogram.png")
    out_hist_pdf = os.path.join(OUTPUT_DIR, f"{out_prefix} Histogram.pdf")
    plt.tight_layout(pad=1.2)
    plt.savefig(out_hist_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(out_hist_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"[OK] Saved histogram → {out_hist_png}")
    print(f"[OK] Saved histogram → {out_hist_pdf}")
    plt.show()

# ================= RUN =================
if __name__ == "__main__":
    try:
        plot_dv(out_prefix="DV")
    except Exception as e:
        print("[ERROR] DV visualization failed:", e)
        traceback.print_exc()


# Step-3.6: DV⁺ (downslope-only velocity)

In [ ]:
# =========================================================
# Step 3.6 — DV⁺ (downslope-only velocity) 
# =========================================================
import os, json
import numpy as np
from datetime import datetime
import rasterio
from rasterio.warp import reproject, Resampling

def pp(msg): 
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

# ------------------- PATHS -----------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

# Input
ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
STUDY_MASK = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

DV_DIR    = os.path.join(BASE, "Results", "Step-3.5-DV")
DV_TIF    = os.path.join(DV_DIR, "downslope_velocity_mm_yr_32645_12p5m_float32.tif")
DOT_TIF   = os.path.join(DV_DIR, "los_dot_downslope_32645_12p5m_float32.tif")

COUNT_TIF = os.path.join(BASE, "Results", "Step-3.2-LOS", "los_velocity_count_32645_12p5m_int16.tif")
COH_TIF   = os.path.join(BASE, "Results", "Step-3.1-Coherence", "coherence_ps_mean_32645_12p5m_float32.tif")

# Output
OUT_DIR = os.path.join(BASE, "Results", "Step-3.6-DV⁺"); os.makedirs(OUT_DIR, exist_ok=True)
DVPLUS_TIF        = os.path.join(OUT_DIR, "dv_plus_mm_yr_32645_12p5m_float32.tif")
DVPLUS_MASKED_TIF = os.path.join(OUT_DIR, "dv_plus_masked_mm_yr_32645_12p5m_float32.tif")
QA_MASK_TIF       = os.path.join(OUT_DIR, "dvplus_valid_mask_32645_12p5m_byte.tif")
META_JSON         = os.path.join(OUT_DIR, "dvplus_metadata.json")

# --------------------------
# PARAMETERS
# --------------------------
NODATA = -9999.0
MIN_COUNT = 1                 # minimum LOS point count per pixel (if COUNT_TIF exists)
MIN_COHERENCE = 0.25          # minimum coherence (if COH_TIF exists)
REQUIRE_POSITIVE_DOT = False  # optional extra geometry filter (only if DOT_TIF exists)
DV_POS_CAP_MM_YR = 150.0      # cap large positives before DV⁺ (winsorize); set None to disable

# --------------------------
# Helpers
# --------------------------
def read_anchor(path):
    with rasterio.open(path) as ds:
        return {
            "tf": ds.transform, "crs": ds.crs,
            "width": ds.width, "height": ds.height,
            "profile": ds.profile
        }

def read_raster(path):
    with rasterio.open(path) as ds:
        return ds.read(1), ds.transform, ds.crs, ds.nodata

def reproject_to_anchor(arr, src_tf, src_crs, dst_spec, resampling=Resampling.bilinear, src_nodata=None):
    same = (src_crs == dst_spec["crs"]) and (src_tf == dst_spec["tf"])
    if same and arr.shape == (dst_spec["height"], dst_spec["width"]):
        out = arr.astype("float64")
    else:
        out = np.full((dst_spec["height"], dst_spec["width"]), np.nan, dtype="float64")
        reproject(
            source=arr.astype("float64"),
            destination=out,
            src_transform=src_tf, src_crs=src_crs,
            dst_transform=dst_spec["tf"], dst_crs=dst_spec["crs"],
            resampling=resampling,
            src_nodata=src_nodata, dst_nodata=np.nan
        )
    # map explicit nodata to nan
    if src_nodata is not None and np.isfinite(src_nodata):
        out = np.where(out == float(src_nodata), np.nan, out)
    return out

def read_mask_as_byte(mask_path, dst_spec):
    with rasterio.open(mask_path) as ms:
        same = (
            ms.crs == dst_spec["crs"] and ms.transform == dst_spec["tf"] and
            ms.width == dst_spec["width"] and ms.height == dst_spec["height"]
        )
        if same:
            m = ms.read(1)
        else:
            m = np.zeros((dst_spec["height"], dst_spec["width"]), dtype="float64")
            reproject(
                source=ms.read(1).astype("float64"),
                destination=m,
                src_transform=ms.transform, src_crs=ms.crs,
                dst_transform=dst_spec["tf"], dst_crs=dst_spec["crs"],
                resampling=Resampling.nearest,
                src_nodata=0, dst_nodata=0
            )
        return (m > 0).astype("uint8")

def write_tif(path, arr, dst_spec, dtype="float32", nodata=NODATA, compress="LZW"):
    prof = dst_spec["profile"].copy()
    prof.update(
        driver="GTiff",
        count=1,
        dtype=dtype,
        nodata=nodata,
        transform=dst_spec["tf"],
        crs=dst_spec["crs"],
        height=dst_spec["height"],
        width=dst_spec["width"],
        tiled=True,
        compress=compress,
        bigtiff="IF_NEEDED"
    )
    tmp = path + ".part"
    with rasterio.open(tmp, "w", **prof) as ds:
        ds.write(arr.astype(dtype), 1)
    os.replace(tmp, path)

# --------------------------
# MAIN
# --------------------------
def main():
    pp("=== Step 3.6 — DV⁺ ===")

    # 0) Anchor / mask
    if not os.path.exists(ANCHOR_TIF): 
        raise SystemExit(f"Missing anchor: {ANCHOR_TIF}")
    if not os.path.exists(STUDY_MASK): 
        raise SystemExit(f"Missing study mask: {STUDY_MASK}")

    anchor = read_anchor(ANCHOR_TIF)
    study_mask = read_mask_as_byte(STUDY_MASK, anchor)

    # 1) Required DV raster
    if not os.path.exists(DV_TIF): 
        raise SystemExit(f"Missing DV raster: {DV_TIF}")

    dv_arr, dv_tf, dv_crs, dv_nd = read_raster(DV_TIF)
    dv = reproject_to_anchor(dv_arr, dv_tf, dv_crs, anchor,
                             resampling=Resampling.bilinear, src_nodata=dv_nd)

    # 2) Optional QA layers
    dot = None
    if os.path.exists(DOT_TIF):
        d_arr, d_tf, d_crs, d_nd = read_raster(DOT_TIF)
        dot = reproject_to_anchor(d_arr, d_tf, d_crs, anchor,
                                  resampling=Resampling.bilinear, src_nodata=d_nd)

    cnt = None
    if os.path.exists(COUNT_TIF):
        c_arr, c_tf, c_crs, c_nd = read_raster(COUNT_TIF)
        cnt = reproject_to_anchor(c_arr, c_tf, c_crs, anchor,
                                  resampling=Resampling.nearest, src_nodata=c_nd)

    coh = None
    if os.path.exists(COH_TIF):
        q_arr, q_tf, q_crs, q_nd = read_raster(COH_TIF)
        coh = reproject_to_anchor(q_arr, q_tf, q_crs, anchor,
                                  resampling=Resampling.bilinear, src_nodata=q_nd)

    # 3) Build QA validity mask
    valid = (study_mask == 1) & np.isfinite(dv)

    if cnt is not None:
        valid &= np.isfinite(cnt) & (cnt >= MIN_COUNT)

    if coh is not None:
        valid &= np.isfinite(coh) & (coh >= MIN_COHERENCE)

    if REQUIRE_POSITIVE_DOT and (dot is not None):
        valid &= np.isfinite(dot) & (dot > 0)

    # 4) DV⁺ = max(DV, 0) only where DV is finite
    dv_clip = dv.copy()

    # cap only finite values
    if DV_POS_CAP_MM_YR is not None:
        m = np.isfinite(dv_clip)
        dv_clip[m] = np.minimum(dv_clip[m], DV_POS_CAP_MM_YR)

    dv_plus = np.full_like(dv_clip, np.nan, dtype="float64")
    m = np.isfinite(dv_clip)
    dv_plus[m] = np.maximum(dv_clip[m], 0.0)

    # 5) Outputs
    # a) Raw DV⁺ (study area only; NaN -> NODATA)
    dvp_raw = np.full((anchor["height"], anchor["width"]), NODATA, dtype="float32")
    m_study = (study_mask == 1) & np.isfinite(dv_plus)
    dvp_raw[m_study] = dv_plus[m_study].astype("float32")
    write_tif(DVPLUS_TIF, dvp_raw, anchor, dtype="float32", nodata=NODATA)

    # b) QA-masked DV⁺ (valid only)
    dvp_masked = np.full((anchor["height"], anchor["width"]), NODATA, dtype="float32")
    m_ok = valid & np.isfinite(dv_plus)
    dvp_masked[m_ok] = dv_plus[m_ok].astype("float32")
    write_tif(DVPLUS_MASKED_TIF, dvp_masked, anchor, dtype="float32", nodata=NODATA)

    # c) Valid mask (byte)
    valid_byte = m_ok.astype("uint8")
    write_tif(QA_MASK_TIF, valid_byte, anchor, dtype="uint8", nodata=0)

    # 6) Stats + metadata
    stats = {}
    if m_ok.any():
        vals = dvp_masked[m_ok].astype("float64")
        stats = {
            "count_valid": int(vals.size),
            "min": float(np.min(vals)),
            "p25": float(np.percentile(vals, 25)),
            "mean": float(np.mean(vals)),
            "p75": float(np.percentile(vals, 75)),
            "max": float(np.max(vals))
        }
        pp(f"DV⁺ stats (masked): count={stats['count_valid']}, "
           f"min/mean/max={stats['min']:.2f}/{stats['mean']:.2f}/{stats['max']:.2f} mm/yr")
    else:
        pp("WARNING: No valid DV⁺ pixels after QA mask.")

    meta = {
        "step": "3.6 DV+",
        "created_utc": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "inputs": {
            "dv_tif": DV_TIF,
            "coherence_tif": COH_TIF if coh is not None else None,
            "count_tif": COUNT_TIF if cnt is not None else None,
            "dot_tif": DOT_TIF if dot is not None else None,
            "mask_tif": STUDY_MASK,
            "anchor_tif": ANCHOR_TIF
        },
        "params": {
            "min_count": int(MIN_COUNT),
            "min_coherence": float(MIN_COHERENCE),
            "require_positive_dot": bool(REQUIRE_POSITIVE_DOT),
            "dv_pos_cap_mm_yr": float(DV_POS_CAP_MM_YR) if DV_POS_CAP_MM_YR is not None else None,
            "nodata": float(NODATA)
        },
        "outputs": {
            "dv_plus_tif": DVPLUS_TIF,
            "dv_plus_masked_tif": DVPLUS_MASKED_TIF,
            "valid_mask_tif": QA_MASK_TIF
        },
        "stats_masked": stats
    }
    with open(META_JSON, "w") as f:
        json.dump(meta, f, indent=2)

    pp(f"[OK] Wrote: {DVPLUS_TIF}")
    pp(f"[OK] Wrote: {DVPLUS_MASKED_TIF}")
    pp(f"[OK] Wrote: {QA_MASK_TIF}")
    pp(f"[OK] Metadata: {META_JSON}")
    pp("=== Done (Step 3.6) ===")

#-------------RUN-----------------
if __name__ == "__main__":
    main()


# Updated Code for Visualization of DV⁺ (downslope-only velocity)

In [ ]:
# =========================================================
# Step 3.6 — DV⁺ (downslope-only velocity) Visualization
# =========================================================
import os, time, stat, traceback
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Polygon, Rectangle, PathPatch
from matplotlib.path import Path as MplPath
from matplotlib.ticker import FuncFormatter, MultipleLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import FixedLocator, MultipleLocator
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import transform as rio_transform, transform_geom
from rasterio import features as rio_features
from matplotlib.ticker import MultipleLocator, FixedLocator

# ---------------- Optional libraries ----------------
try:
    from scipy.spatial import cKDTree
    from scipy.interpolate import griddata, RBFInterpolator
    from scipy.ndimage import binary_erosion
    from scipy.stats import gaussian_kde
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
    print("[WARN] SciPy not available; using simpler fallbacks")

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False
    print("[WARN] pyproj not available; limited CRS transforms")

try:
    import geopandas as gpd
    from shapely.geometry import shape as shp_shape, Polygon as ShpPolygon
    from shapely.ops import unary_union
    HAVE_GPD = True
    HAVE_SHAPELY = True
except Exception:
    HAVE_GPD = False
    try:
        from shapely.geometry import shape as shp_shape, Polygon as ShpPolygon
        from shapely.ops import unary_union
        HAVE_SHAPELY = True
    except Exception:
        HAVE_SHAPELY = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

# =========================================================
# PATHS
# =========================================================
BASE = os.environ.get("LSM_DATA_DIR", "./data")

STUDY_MASK  = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP   = os.path.join(BASE, "Study_Area.shp")
ROAD_SHP    = os.path.join(BASE, "NTD_Road.shp")

OUT_DIR  = os.path.join(BASE, "Results", "Step-3.6-DV⁺")
os.makedirs(OUT_DIR, exist_ok=True)

DVPLUS_MASKED_TIF = os.path.join(OUT_DIR, "dv_plus_masked_mm_yr_32645_12p5m_float32.tif")

# =========================================================
# Plot Theme 
# =========================================================
MAP_FIGSIZE  = (1, 1)
KDE_FIGSIZE  = (1, 1)
HIST_FIGSIZE = (1, 1)

SAVE_DPI    = 600
FS_LABEL    = 2.0
FS_TICKS    = 1.5
FS_LEGEND   = 2.0
FS_CBAR     = 2.2
FS_CBARLAB  = 3.0
FS_TEXT     = 2.6
LW_AXIS     = 0.3
LW_BORDERS  = 0.4
LW_ROAD     = 0.4
TICK_PAD    = 0.4
XLABEL_PAD  = 0.5
YLABEL_PAD  = 0.5
CBAR_PAD    = 1.0

# Right inset colorbar 
CAX_SIZE    = "8%"
CAX_PAD     = 0.002

# North arrow placement
NA_SIZE     = 0.08
NA_PAD      = 0.06
NA_ARROW_DOWNSHIFT = 0.015
NA_LABEL_UPSHIFT   = 0.010

# Legend box
LEG_BBOX = (0.80, 0.15)

# =========================================================
# Display / scale options
# =========================================================
CONTRAST_PERCENTILE = 98
ABS_CAP_MM_PER_YR   = 75.0
GRID_RES            = 200
MASK_EROSION_ITER   = 2

# --- Export toggles ---
SAVE_PDF = True
SAVE_SVG = True
SAVE_EPS = True   

# Colormap 
VELOCITY_COLORS = [
    '#E6F2FF','#CFE6FF','#B7DAFF','#9FCEFF','#87C2FF','#6FB6FF',
    '#57AAFF','#3E9EFF','#268FFF','#0F7FFF','#0069D1',
    '#FFB3B3','#FF9999','#FF7F7F','#FF6666','#FF4C4C','#FF3333','#CC0000'
]
VELOCITY_CMAP = mcolors.LinearSegmentedColormap.from_list('dvplus', VELOCITY_COLORS, N=256)

# =========================================================
# Safe save helpers 
# =========================================================
def safe_savefig(fig, path, *, dpi=SAVE_DPI, bbox_inches='tight', facecolor='white',
                 retries=3, wait_sec=0.6, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path):
        try: os.chmod(path, stat.S_IWRITE)
        except Exception: pass
    for i in range(retries):
        try:
            fig.savefig(path, dpi=dpi, bbox_inches=bbox_inches, facecolor=facecolor, **kwargs)
            return path
        except PermissionError:
            if i < retries - 1:
                time.sleep(wait_sec)
            else:
                root, ext = os.path.splitext(path)
                alt = f"{root}_{int(time.time())}{ext}"
                fig.savefig(alt, dpi=dpi, bbox_inches=bbox_inches, facecolor=facecolor, **kwargs)
                return alt

def _opaque_rgba(arr):
    if arr is None or len(arr) == 0:
        return arr
    arr = np.array(arr, copy=True)
    if arr.ndim == 2 and arr.shape[1] == 4:
        arr[:, 3] = 1.0
    return arr

def _strip_transparency_for_eps(fig):
    """Make all artists fully opaque to avoid EPS alpha warnings."""
    from matplotlib.collections import Collection
    from matplotlib.image import AxesImage
    from matplotlib.lines import Line2D

    for ax in fig.axes:
        # collections
        for coll in list(getattr(ax, "collections", [])):
            try:
                coll.set_alpha(None)
                fc = getattr(coll, "get_facecolors", lambda: None)()
                ec = getattr(coll, "get_edgecolors", lambda: None)()
                if fc is not None and len(fc): coll.set_facecolors(_opaque_rgba(fc))
                if ec is not None and len(ec): coll.set_edgecolors(_opaque_rgba(ec))
            except Exception:
                pass
        # patches
        for p in list(getattr(ax, "patches", [])):
            try:
                p.set_alpha(None)
                fc = p.get_facecolor(); ec = p.get_edgecolor()
                if fc is not None and len(np.atleast_1d(fc)) >= 4:
                    rgba = np.array(fc, copy=True); rgba[..., 3] = 1.0; p.set_facecolor(rgba)
                if ec is not None and len(np.atleast_1d(ec)) >= 4:
                    rgba = np.array(ec, copy=True); rgba[..., 3] = 1.0; p.set_edgecolor(rgba)
            except Exception:
                pass
        # lines
        for ln in list(getattr(ax, "lines", [])):
            try:
                ln.set_alpha(None)
                c = ln.get_color()
                if isinstance(c, (tuple, list)) and len(c) == 4:
                    rgba = list(c); rgba[3] = 1.0; ln.set_color(tuple(rgba))
            except Exception:
                pass
        # images
        for im in list(getattr(ax, "images", [])):
            try:
                if isinstance(im, AxesImage):
                    im.set_alpha(1.0)
            except Exception:
                pass
        # legend frame
        leg = ax.get_legend()
        if leg is not None and leg.get_frame() is not None:
            try:
                fr = leg.get_frame(); fr.set_alpha(None)
                fc = fr.get_facecolor(); ec = fr.get_edgecolor()
                if fc is not None and len(np.atleast_1d(fc)) >= 4:
                    rgba = np.array(fc, copy=True); rgba[..., 3] = 1.0; fr.set_facecolor(rgba)
                if ec is not None and len(np.atleast_1d(ec)) >= 4:
                    rgba = np.array(ec, copy=True); rgba[..., 3] = 1.0; fr.set_edgecolor(rgba)
            except Exception:
                pass
        # ensure grids off
        try: ax.grid(False)
        except Exception: pass

def save_all(fig, base_no_ext):
    saved = {}
    saved["png"] = safe_savefig(fig, base_no_ext + ".png")
    if SAVE_PDF:
        saved["pdf"] = safe_savefig(fig, base_no_ext + ".pdf")
    if SAVE_SVG:
        saved["svg"] = safe_savefig(fig, base_no_ext + ".svg", format="svg")
    if SAVE_EPS:
        _strip_transparency_for_eps(fig)
        saved["eps"] = safe_savefig(fig, base_no_ext + ".eps", format="eps")
    return saved

# =========================================================
# Axis helpers 
# =========================================================
def dms_label(dec, is_lon=True):
    a = abs(float(dec))
    d = int(a)
    rem_min = (a - d) * 60.0
    m = int(rem_min)
    s = int(round((rem_min - m) * 60.0))  # integer seconds

    # carry if rounding bumps to 60s or 60m
    if s == 60:
        s = 0
        m += 1
    if m == 60:
        m = 0
        d += 1

    hemi = ('E' if dec >= 0 else 'W') if is_lon else ('N' if dec >= 0 else 'S')
    return f"{d}°{m:02d}′{s:02d}″{hemi}"

def add_north_arrow_tri(ax, size=NA_SIZE, pad=NA_PAD, label='N',
                        arrow_downshift=NA_ARROW_DOWNSHIFT, label_upshift=NA_LABEL_UPSHIFT):
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10 + label_upshift
    x0 = pad; y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72; y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size * 0.09)
    ax.add_patch(Polygon([apex, left_base, tail],  closed=True,
                         transform=ax.transAxes, facecolor='white',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True,
                         transform=ax.transAxes, facecolor='black',
                         edgecolor='black', lw=0.6, zorder=6))
    ax.text(label_anchor_x, label_anchor_y, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=FS_LABEL, fontweight='bold',
            color='black', zorder=7)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10 ** exp
    for m in (1, 2, 5, 10):
        if m * base >= target: return m * base
    return 10 * base

def add_checker_scale_bar(ax, west, east, south, north,
                          crs_type='wgs84', mid_lat_deg=None, segments=4, fontsize=FS_TEXT):
    width_data = east - west; height_data = north - south
    if crs_type == 'utm':
        width_m = width_data; units_per_meter_x = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat_deg)
        width_m = width_data * mpd; units_per_meter_x = 1.0 / mpd
    total_len_m = nice_len_125(width_m / 5.0); seg_len_m = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x; seg_len_units = seg_len_m * units_per_meter_x
    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.016 * height_data
    pad_x   = total_len_units * 0.06; pad_y = 0.028 * height_data

    ax.add_patch(Rectangle((x_start - pad_x, y_base - pad_y),
                           total_len_units + 2 * pad_x, bar_h + 2 * pad_y,
                           transform=ax.transData, facecolor='white',
                           edgecolor='none', zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        face = 'black' if i % 2 == 0 else 'white'
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData, facecolor=face,
                               edgecolor='none', zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False, edgecolor='black',
                           lw=0.3, zorder=5))
    ax.text(x_start + total_len_units / 2.0, y_base + bar_h + 0.010 * height_data, "km",
            ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.010 * height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.010 * height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Boundary / Roads I/O + plotting
# =========================================================
def load_roads(road_shp_path, target_crs_str):
    if not os.path.exists(road_shp_path): return None
    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_shp_path)
            if roads.crs is None: roads = roads.set_crs(epsg=4326)
            roads = roads.to_crs(target_crs_str)
            return roads
        except Exception:
            pass
    if HAVE_FIONA:
        try:
            geoms = []
            with fiona.open(road_shp_path, 'r') as shp:
                src_crs = shp.crs
                for feat in shp:
                    geom = feat['geometry']
                    if geom is None: continue
                    try:
                        geom_t = transform_geom(src_crs, target_crs_str, geom)
                    except Exception:
                        geom_t = geom
                    geoms.append(geom_t)
            return {'geometries': geoms, 'method': 'fiona', 'crs': target_crs_str}
        except Exception:
            pass
    return None

def plot_roads_on_map(ax, roads, line_color='red', line_width=LW_ROAD, line_style='-', alpha=1.0, zorder=850):
    if roads is None: return
    try:
        if HAVE_GPD and hasattr(roads, 'plot'):
            try:
                roads.geometry = roads.geometry.apply(
                    lambda g: g if g is None or g.geom_type in ("LineString","MultiLineString") else None
                )
            except Exception:
                pass
            roads.plot(ax=ax, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
        elif isinstance(roads, dict) and 'geometries' in roads:
            for geom in roads['geometries']:
                if geom is None: continue
                if geom['type'] == 'LineString':
                    xs = [p[0] for p in geom['coordinates']]; ys = [p[1] for p in geom['coordinates']]
                    ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
                elif geom['type'] == 'MultiLineString':
                    for line in geom['coordinates']:
                        xs = [p[0] for p in line]; ys = [p[1] for p in line]
                        ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
    except Exception:
        pass

def draw_boundary_from_mask(ax, mask_tif_path, raster_crs, zorder=999, color="black", linewidth=LW_BORDERS):
    if not os.path.exists(mask_tif_path): return False
    try:
        with rasterio.open(mask_tif_path) as ms:
            if ms.crs != raster_crs:
                return False
            mask = ms.read(1)
            b = ms.bounds
            extent_mask = [b.left, b.right, b.bottom, b.top]
        ax.contour(mask, levels=[0.5], colors=color, linewidths=linewidth,
                   extent=extent_mask, origin='upper', zorder=zorder)
        return True
    except Exception:
        return False

def draw_boundary_from_shp(ax, shp_path, target_crs_str, zorder=998, color="black", linewidth=LW_BORDERS):
    if not os.path.exists(shp_path): return False
    try:
        if HAVE_GPD:
            gdf = gpd.read_file(shp_path).to_crs(target_crs_str)
            gdf.boundary.plot(ax=ax, color=color, linewidth=linewidth, alpha=1.0, zorder=zorder)
            return True
        elif HAVE_FIONA:
            geoms = []
            with fiona.open(shp_path, 'r') as shp:
                src_crs = shp.crs
                for feat in shp:
                    geom = feat['geometry']
                    if geom is None: continue
                    geoms.append(transform_geom(src_crs, target_crs_str, geom))
            for geom in geoms:
                if geom['type'] == 'Polygon':
                    xs = [p[0] for p in geom['coordinates'][0]]
                    ys = [p[1] for p in geom['coordinates'][0]]
                    ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
                elif geom['type'] == 'MultiPolygon':
                    for poly in geom['coordinates']:
                        xs = [p[0] for p in poly[0]]
                        ys = [p[1] for p in poly[0]]
                        ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
            return True
    except Exception:
        return False
    return False

def get_study_polygon_in_crs(target_crs_str):
    if not HAVE_SHAPELY: return None
    if os.path.exists(STUDY_SHP):
        try:
            if HAVE_GPD:
                gdf = gpd.read_file(STUDY_SHP)
                if gdf.crs is None: gdf = gdf.set_crs(epsg=4326)
                gdf = gdf.to_crs(target_crs_str)
                return unary_union(gdf.geometry)
            elif HAVE_FIONA:
                geoms = []
                with fiona.open(STUDY_SHP, 'r') as src:
                    src_crs = src.crs
                    for ft in src:
                        if ft['geometry'] is None: continue
                        geoms.append(shp_shape(transform_geom(src_crs, target_crs_str, ft['geometry'])))
                if geoms: return unary_union(geoms)
        except Exception:
            pass
    if os.path.exists(STUDY_MASK):
        try:
            with rasterio.open(STUDY_MASK) as ds:
                data = ds.read(1); trans = ds.transform; crs = ds.crs
                polys = []
                for geom, val in rio_features.shapes(data.astype('uint8'),
                                                     mask=(data > 0), transform=trans):
                    if val != 1: continue
                    g = shp_shape(geom)
                    if HAVE_GPD:
                        g = gpd.GeoSeries([g], crs=crs).to_crs(target_crs_str).geometry.iloc[0]
                    polys.append(g)
                if polys: return unary_union(polys)
        except Exception:
            pass
    return None

def polygon_to_pathpatch(ax, poly):
    if not (HAVE_SHAPELY and (poly is not None)): return None
    def _ring_to_vertices(ring):
        xs, ys = ring.coords.xy
        verts = list(zip(xs, ys))
        if verts[0] != verts[-1]: verts.append(verts[0])
        codes = [MplPath.MOVETO] + [MplPath.LINETO]*(len(verts)-2) + [MplPath.CLOSEPOLY]
        return verts, codes
    polys = [poly] if isinstance(poly, ShpPolygon) else list(poly.geoms)
    vertices = []; codes = []
    for p in polys:
        v, c = _ring_to_vertices(p.exterior); vertices += v; codes += c
        for hole in p.interiors:
            v, c = _ring_to_vertices(hole); vertices += v; codes += c
    return PathPatch(MplPath(vertices, codes), transform=ax.transData,
                     facecolor='none', edgecolor='none')

# =========================================================
# Interpolation (RBF → cubic → IDW → naive)
# =========================================================
def interpolate_grid(x, y, v, west, east, south, north, grid_res=GRID_RES):
    gx = np.linspace(west,  east,  grid_res)
    gy = np.linspace(south, north, grid_res)
    Xg, Yg = np.meshgrid(gx, gy)
    pts = np.column_stack([x, y])

    if HAVE_SCIPY:
        grid_pts = np.column_stack([Xg.ravel(), Yg.ravel()])
        try:
            rbf = RBFInterpolator(pts, v, kernel='thin_plate_spline', smoothing=0.1, epsilon=0.1)
            Vg = rbf(grid_pts).reshape(grid_res, grid_res)
            return Xg, Yg, Vg
        except Exception:
            pass
        try:
            Vg = griddata(pts, v, grid_pts, method='cubic', fill_value=np.nan).reshape(grid_res, grid_res)
            return Xg, Yg, Vg
        except Exception:
            pass
        try:
            tree = cKDTree(pts)
            k = min(64, len(pts))
            dist, idx = tree.query(grid_pts, k=k)
            w  = 1.0 / (dist + 1e-12)
            vv = v[idx] if k > 1 else v[idx][:, None]
            V_flat = (w * vv).sum(axis=1) / w.sum(axis=1)
            return Xg, Yg, V_flat.reshape(grid_res, grid_res)
        except Exception:
            pass

    # Naive Shepard
    Vg = np.zeros((grid_res, grid_res), dtype=float)
    for i in range(grid_res):
        Xi = Xg[i]; Yi = Yg[i]
        for j in range(grid_res):
            d = np.hypot(pts[:, 0] - Xi[j], pts[:, 1] - Yi[j])
            w = 1.0 / (d + 1e-12)
            Vg[i, j] = (w * v).sum() / w.sum()
    return Xg, Yg, Vg

# =========================================================
# Main DV⁺ visualization (Map → KDE → Histogram)
# =========================================================
def create_dvplus_visualizations(out_prefix="DVplus"):
    print("="*60); print("DV⁺ VISUALIZATION"); print("="*60)
    if not os.path.exists(DVPLUS_MASKED_TIF):
        raise FileNotFoundError(f"Missing DV⁺ raster: {DVPLUS_MASKED_TIF}")

    with rasterio.open(DVPLUS_MASKED_TIF) as ds:
        arr = ds.read(1).astype(np.float32)
        nodata = ds.nodata
        b = ds.bounds
        raster_crs = ds.crs
        transform = ds.transform
        extent = [b.left, b.right, b.bottom, b.top]
        crs_is_geographic = bool(raster_crs and raster_crs.is_geographic)

    # Keep positive DV+ only
    if nodata is not None:
        arr[arr == nodata] = np.nan
    arr[~np.isfinite(arr)] = np.nan
    arr[arr < 0] = np.nan

    rows, cols = np.where(np.isfinite(arr))
    if rows.size == 0:
        raise RuntimeError("No valid DV⁺ values found.")
    xs = transform.c + (cols + 0.5) * transform.a + (rows + 0.5) * transform.b
    ys = transform.f + (cols + 0.5) * transform.d + (rows + 0.5) * transform.e
    x = np.asarray(xs); y = np.asarray(ys); v = arr[rows, cols].astype(float)

    # Erode mask & clip points
    if os.path.exists(STUDY_MASK):
        try:
            with rasterio.open(STUDY_MASK) as ms:
                mask = ms.read(1)
                m_crs = ms.crs; m_tx = ms.transform
                mask_eroded = binary_erosion(mask > 0, iterations=MASK_EROSION_ITER).astype(np.uint8) if HAVE_SCIPY else (mask > 0).astype(np.uint8)
                if m_crs != raster_crs:
                    mx, my = rio_transform(raster_crs, m_crs, x, y)
                else:
                    mx, my = x, y
                r, c = rowcol(m_tx, mx, my)
                keep = (r >= 0) & (r < mask_eroded.shape[0]) & (c >= 0) & (c < mask_eroded.shape[1]) & (mask_eroded[r, c] > 0)
                x, y, v = x[keep], y[keep], v[keep]
        except Exception:
            pass

    # Color scale
    vmax_data = float(np.percentile(v, CONTRAST_PERCENTILE))
    vmin, vmax = 0.0, float(min(ABS_CAP_MM_PER_YR, vmax_data))

    # Slight buffer on extent (2%)
    west, east, south, north = extent
    bx = (east - west) * 0.02 or 1.0
    by = (north - south) * 0.02 or 1.0
    extent_buf = [west - bx, east + bx, south - by, north + by]

    # Interpolate grid
    Xg, Yg, Vgrid = interpolate_grid(x, y, v, *extent_buf, grid_res=GRID_RES)

    # Mask grid to AOI
    if os.path.exists(STUDY_MASK):
        try:
            with rasterio.open(STUDY_MASK) as ms:
                mask = ms.read(1)
                m_crs = ms.crs; m_tx = ms.transform
                mask_eroded = binary_erosion(mask > 0, iterations=MASK_EROSION_ITER).astype(np.uint8) if HAVE_SCIPY else (mask > 0).astype(np.uint8)
                H, W = Vgrid.shape
                for i in range(H):
                    for j in range(W):
                        gx, gy = Xg[i, j], Yg[i, j]
                        if m_crs != raster_crs:
                            mx, my = rio_transform(raster_crs, m_crs, [gx], [gy]); mx, my = mx[0], my[0]
                        else:
                            mx, my = gx, gy
                        rr, cc = rowcol(m_tx, mx, my)
                        inside = (0 <= rr < mask_eroded.shape[0]) and (0 <= cc < mask_eroded.shape[1]) and (mask_eroded[int(rr), int(cc)] > 0)
                        if not inside: Vgrid[i, j] = np.nan
        except Exception:
            pass

    # DMS tick formatters
    xmid = 0.5 * (extent[0] + extent[1]); ymid = 0.5 * (extent[2] + extent[3])
    if crs_is_geographic:
        fmt_lon = lambda xv, pos: dms_label(xv, True)
        fmt_lat = lambda yv, pos: dms_label(yv, False)
        crs_type = 'wgs84'
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError("Projected raster requires pyproj for DMS axis labels.")
        to_ll = Transformer.from_crs(raster_crs, "EPSG:4326", always_xy=True)
        fmt_lon = lambda xv, pos: dms_label(to_ll.transform(xv, ymid)[0], True)
        fmt_lat = lambda yv, pos: dms_label(to_ll.transform(xmid, yv)[1], False)
        crs_type = 'utm'

    # Overlays + strict clip
    target_crs_str = str(raster_crs)
    roads = load_roads(ROAD_SHP, target_crs_str)
    study_poly = get_study_polygon_in_crs(target_crs_str)

    # ================= MAP =================
    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE, facecolor='white')
    ax.set_facecolor('white')
    for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)

    # Add some room for right inset colorbar
    plt.subplots_adjust(left=0.18, right=0.86, bottom=0.18, top=0.98)

    im = ax.imshow(Vgrid, cmap=VELOCITY_CMAP, extent=extent_buf,
                   vmin=vmin, vmax=vmax, origin='lower', interpolation='nearest', zorder=1)
    pts = ax.scatter(x, y, c=v, s=0.25, cmap=VELOCITY_CMAP,
                     vmin=vmin, vmax=vmax, edgecolors='none', zorder=2)

    # Clip both image and points to polygon (if available)
    if study_poly is not None:
        clip_patch = polygon_to_pathpatch(ax, study_poly)
        if clip_patch is not None:
            try:
                ax.add_patch(clip_patch); im.set_clip_path(clip_patch); pts.set_clip_path(clip_patch)
            except Exception:
                pass

    # Roads + boundary
    plot_roads_on_map(ax, roads, line_color='red', line_width=LW_ROAD, line_style='-', alpha=1.0, zorder=850)
    if not draw_boundary_from_mask(ax, STUDY_MASK, raster_crs, zorder=999):
        draw_boundary_from_shp(ax, STUDY_SHP, target_crs_str, zorder=998)

    # Axis labels (DMS) and ticks — 6′ on X, 10′ on Y
    #ax.set_xlabel("Longitude", fontsize=FS_LABEL, fontweight='bold')
    #ax.set_ylabel("Latitude",  fontsize=FS_LABEL, fontweight='bold')
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_lon))  # DMS labels
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_lat))  # DMS labels
        
    SIX_MIN =  6.0/60.0     # 0.1°  -> 6 minutes
    TEN_MIN = 8.0/60.0     # ≈0.1666667° -> 10 minutes
    
    if crs_is_geographic:
        # Axes are already in degrees
        ax.xaxis.set_major_locator(MultipleLocator(SIX_MIN))  # 6′ for longitude
        ax.yaxis.set_major_locator(MultipleLocator(SIX_MIN))  # 8′ for latitude
    else:
        # Axes are in meters (e.g., UTM); place ticks every 6′ (lon) and 10′ (lat)
        to_ll = Transformer.from_crs(raster_crs, "EPSG:4326", always_xy=True)
        to_xy = Transformer.from_crs("EPSG:4326", raster_crs, always_xy=True)
    
        west, east, south, north = extent_buf
    
        # bounds in degrees
        w_deg, s_deg = to_ll.transform(west,  south)
        e_deg, n_deg = to_ll.transform(east,  north)
    
        # mid-lines (to convert 1D grids cleanly)
        x_mid_m  = 0.5*(west + east)
        y_mid_m  = 0.5*(south + north)
        x_mid_deg, y_mid_deg = to_ll.transform(x_mid_m, y_mid_m)
    
        def frange(start, stop, step):
            vals = []
            v = np.floor(start/step)*step
            while v <= stop + 1e-9:
                vals.append(v); v += step
            return vals
    
        # Different step sizes per axis
        lons_6min = frange(min(w_deg, e_deg), max(w_deg, e_deg), SIX_MIN)   # 6′ grid for X
        lats_10min = frange(min(s_deg, n_deg), max(s_deg, n_deg), SIX_MIN)  # 10′ grid for Y
    
        # Convert degree grids to data coords (meters)
        xticks_m = [to_xy.transform(lon, y_mid_deg)[0] for lon in lons_6min]  # vary lon
        yticks_m = [to_xy.transform(x_mid_deg, lat)[1] for lat in lats_10min] # vary lat
    
        ax.xaxis.set_major_locator(FixedLocator(xticks_m))
        ax.yaxis.set_major_locator(FixedLocator(yticks_m))
      
    # Styling
    ax.tick_params(axis='x', which='major', labelsize=FS_TICKS, width=LW_AXIS, pad=TICK_PAD, length=1.3, direction='out', rotation=45)
    ax.tick_params(axis='y', which='major', labelsize=FS_TICKS, width=LW_AXIS, pad=TICK_PAD, length=1.3, direction='out', rotation=90)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(35); lbl.set_ha('right')
    ax.xaxis.labelpad = XLABEL_PAD; ax.yaxis.labelpad = YLABEL_PAD
    ax.set_xlim(extent_buf[0], extent_buf[1])
    ax.set_ylim(extent_buf[2], extent_buf[3])

    # Decorations: north arrow + checker scale bar
    add_north_arrow_tri(ax)
    add_checker_scale_bar(ax, *extent_buf, crs_type=crs_type, mid_lat_deg=ymid if crs_is_geographic else None)

    # Legend 
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='red',   lw=LW_ROAD,    label='Highway'),
        Line2D([0], [0], color='black', lw=LW_BORDERS, label='Study Area')
    ]
    leg = ax.legend(handles=legend_elements, loc='center',
                    bbox_to_anchor=LEG_BBOX, bbox_transform=ax.transAxes,
                    frameon=True, framealpha=1.0,
                    fontsize=1.5, labelspacing=0.3, borderpad=0.3,
                    handlelength=1.2, handletextpad=0.4, borderaxespad=0.0)
    fr = leg.get_frame(); fr.set_facecolor('white'); fr.set_edgecolor('lightgray'); fr.set_linewidth(0.3)

    # Right inset colorbar 
    cax = inset_axes(ax, width=CAX_SIZE, height="100%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.0, 1.0, 1.0),
                     bbox_transform=ax.transAxes, borderpad=0)
    cbar = plt.colorbar(im, cax=cax, orientation="vertical",
                        ticks=np.linspace(vmin, vmax, 5))
    cbar.set_label('DV⁺ (mm/year)', fontsize=FS_CBARLAB, fontweight='bold', labelpad=CBAR_PAD)
    cbar.ax.tick_params(labelsize=FS_CBAR, width=LW_AXIS, pad=0.4, length=1.3, direction='out')
    if hasattr(cbar, "outline"): cbar.outline.set_linewidth(0.3)

    # Save map (PNG/PDF/SVG/EPS)
    base_map = os.path.join(OUT_DIR, f"{out_prefix} Map")
    _ = save_all(plt.gcf(), base_map)
    print(f"[OK] Map saved under: {OUT_DIR}")
    plt.show()

    # ================= KDE (dual-y) =================
    fig, ax = plt.subplots(1, 1, figsize=KDE_FIGSIZE, facecolor="white")
    for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)
    plt.subplots_adjust(left=0.18, right=0.97, bottom=0.18, top=0.98)

    data = np.clip(v[np.isfinite(v)], vmin, vmax)
    xs = np.linspace(vmin, vmax, 600)
    bin_width = (vmax - vmin) / 20.0
    n = len(data)

    if HAVE_SCIPY:
        kde = gaussian_kde(data, bw_method="scott"); density = kde(xs)
    else:
        hist_y, hist_x = np.histogram(data, bins=100, range=(vmin, vmax), density=True)
        x_centers = 0.5 * (hist_x[:-1] + hist_x[1:])
        density = np.interp(xs, x_centers, hist_y)

    ax.plot(xs, density, linewidth=0.6)
    ax.fill_between(xs, 0, density)

    ax.set_xlabel("DV⁺ (mm/year)", fontsize=FS_LABEL, fontweight="bold")
    ax.set_ylabel("Density",        fontsize=FS_LABEL, fontweight="bold")
    ax.set_xlim(vmin, vmax)
    ax.tick_params(axis="both", labelsize=FS_TICKS, width=LW_AXIS, pad=TICK_PAD)
    ax.xaxis.labelpad = XLABEL_PAD; ax.yaxis.labelpad = YLABEL_PAD

    # Dual y-axis ≈ counts per bin
    def density_to_counts(y): return np.asarray(y) * n * bin_width
    ax2 = ax.twinx()
    ymin, ymax = ax.get_ylim()
    ax2.set_ylim(density_to_counts(ymin), density_to_counts(ymax))
    ax2.set_ylabel(f"Approx. pixels per {bin_width:.02f} bin",
                   fontsize=FS_LABEL, fontweight="bold")
    ax2.tick_params(axis="y", labelsize=FS_TICKS, width=LW_AXIS, pad=TICK_PAD)

    base_kde = os.path.join(OUT_DIR, f"{out_prefix} Density (KDE)")
    _ = save_all(plt.gcf(), base_kde)
    print(f"[OK] KDE saved under: {OUT_DIR}")
    plt.show()

    # ================= HISTOGRAM =================
    fig, ax = plt.subplots(1, 1, figsize=HIST_FIGSIZE, facecolor='white')
    for sp in ax.spines.values(): sp.set_linewidth(LW_AXIS)
    plt.subplots_adjust(left=0.18, right=0.97, bottom=0.18, top=0.98)

    bins = np.linspace(vmin, vmax, 21)
    counts, edges = np.histogram(data, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2.0
    ax.bar(centers, counts, width=(bins[1]-bins[0])*0.9,
           edgecolor='black', linewidth=0.3)

    mean_v = float(np.mean(data)); med_v = float(np.median(data)); p95_v = float(np.percentile(data, 95))
    ax.axvline(mean_v, color='r',      ls='--', lw=0.7, label=f"Mean: {mean_v:.2f}")
    ax.axvline(med_v,  color='orange', ls='--', lw=0.7, label=f"Median: {med_v:.2f}")
    ax.axvline(p95_v,  color='green',  ls='--', lw=0.7, label=f"95th %ile: {p95_v:.2f}")
    ax.legend(fontsize=FS_LEGEND, loc='upper right', framealpha=1.0)

    ax.set_xlabel('DV⁺ (mm/year)',    fontsize=FS_LABEL, fontweight='bold')
    ax.set_ylabel('Number of Pixels', fontsize=FS_LABEL, fontweight='bold')
    ax.set_xlim(vmin, vmax)
    ax.tick_params(axis='both', labelsize=FS_TICKS, width=LW_AXIS, pad=TICK_PAD)
    ax.xaxis.labelpad = XLABEL_PAD; ax.yaxis.labelpad = YLABEL_PAD

    base_hist = os.path.join(OUT_DIR, f"{out_prefix} Histogram")
    _ = save_all(plt.gcf(), base_hist)
    print(f"[OK] Histogram saved under: {OUT_DIR}")
    plt.show()

    # -------- Summary --------
    print("\n" + "="*60)
    print("DV⁺ VISUALIZATION SUMMARY")
    print("="*60)
    print(f"Total valid pixels: {len(data)}")
    print(f"Display range (mm/yr): [{vmin:.2f}, {vmax:.2f}]")
    print(f"Mean/Median/95th (mm/yr): {mean_v:.2f} / {med_v:.2f} / {p95_v:.2f}")
    print("="*60)
    return True

# ---------------- Main ----------------
if __name__ == "__main__":
    try:
        ok = create_dvplus_visualizations(out_prefix="DVplus")
        print("\nVISUALIZATION COMPLETE!" if ok else "\nVISUALIZATION FAILED")
    except Exception as e:
        print("[ERROR] DV⁺ visualization failed:", e)
        traceback.print_exc()


# Step-3.7: Densify DV⁺ (downslope-only velocity)

In [ ]:
# =========================================================
# Step 3.7 — DV+ Visualization
# =========================================================
#
# Scientific plotting rules:
#   1. Reads the Step 3.7 filled DV+ raster directly.
#   2. Adds no new RBF, IDW, cubic, or scatter reconstruction during plotting.
#   3. Uses the final Step 3.1 coherence map style:
#        - Viridis colour map
#        - 2 × 4 inch map; 2 × 2 inch histogram
#        - 2% buffered display extent
#        - four longitude DMS labels (LinearLocator, rotated 45°)
#        - three latitude DMS labels (LinearLocator, rotated 90°)
#        - bottom-most latitude label anchored above its tick (va="bottom")
#          so it cannot collide with the first longitude label at the corner
#        - no "Latitude" / "Longitude" axis titles
#        - north arrow
#        - checkerboard scale bar
#        - highway and study-area overlays
#        - compact legend and side colour bar
#        - PNG, PDF, and SVG outputs at 900 dpi
#   4. The Step 3.7 input raster may already contain IDW-filled values because
#      densification is part of the analytical Step 3.7 workflow. This
#      visualization does not perform any additional spatial interpolation.
#   5. The same valid raster cells are used for the map, histogram, and stats.
# =========================================================

from __future__ import annotations
import os

import csv
import json
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_geom

import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator, LinearLocator
from matplotlib.patches import Polygon, Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False


# =========================================================
# PATHS
# =========================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
ROAD_SHP = BASE / "NTD_Road.shp"

DVPLUS_DIR = BASE / "Results" / "Step-3.7-DVplus_Densify"

# Analytical output from Step 3.7
DVPLUS_FILLED_TIF = (
    DVPLUS_DIR / "dv_plus_filled_mm_yr_32645_12p5m_float32.tif"
)
DVPLUS_META_JSON = DVPLUS_DIR / "dvplus_densify_metadata.json"

# Visualization outputs
OUT_MAP_PNG = DVPLUS_DIR / "DVplus Map.png"
OUT_MAP_PDF = DVPLUS_DIR / "DVplus Map.pdf"
OUT_MAP_SVG = DVPLUS_DIR / "DVplus Map.svg"

OUT_HIST_PNG = DVPLUS_DIR / "DVplus Histogram.png"
OUT_HIST_PDF = DVPLUS_DIR / "DVplus Histogram.pdf"
OUT_HIST_SVG = DVPLUS_DIR / "DVplus Histogram.svg"

OUT_SUMMARY_JSON = DVPLUS_DIR / "dvplus_visualization_summary.json"
OUT_STATS_CSV = DVPLUS_DIR / "dvplus_visualization_statistics.csv"

DVPLUS_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# DISPLAY SETTINGS
# =========================================================
# Final Step 3.1 coherence map size for this corridor.
MAP_FIGSIZE = (2, 4)
HIST_FIGSIZE = (2, 2)

CAX_SIZE = "5%"
CAX_PAD = 0.01
FIG_DPI = 900

# Same colour map as the final SNDR plot
DVPLUS_CMAP = plt.get_cmap("viridis")

DVPLUS_INFO = {
    "title": "Downslope Deformation Velocity",
    "units": "DV+ (mm/yr)",
}

# Direct raster display only
RASTER_DISPLAY_INTERPOLATION = "nearest"

# Same map buffer and DMS settings as the final Step 3.1 map
DISPLAY_BUFFER_FRACTION = 0.02
LONGITUDE_DMS_TICK_COUNT = 4
LATITUDE_DMS_TICK_COUNT = 3
DMS_TICK_FONTSIZE = 5

# DV+ is nonnegative by definition in Step 3.7.
DISPLAY_VMIN = 0.0

# Step 3.7 currently applies DVPLUS_CAP = 150 mm/yr.
# Reading the metadata cap keeps the display consistent with the analytical
# raster. If metadata are unavailable, this value is used as a fallback.
USE_METADATA_DVPLUS_CAP = True
FALLBACK_DISPLAY_VMAX = 150.0

SHOW_FIGURES = True
SAVE_FIGURES = True

# Font convention used by the final Step 3.1 map
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = [
    "Times New Roman",
    "DejaVu Serif",
]
plt.rcParams["axes.unicode_minus"] = False


# =========================================================
# GENERAL HELPERS
# =========================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json_optional(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None

    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return None


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


# =========================================================
# RASTER HELPERS
# =========================================================
def same_grid(
    dataset: rasterio.io.DatasetReader,
    reference: dict[str, Any],
) -> bool:
    return (
        dataset.crs == reference["crs"]
        and dataset.transform == reference["transform"]
        and dataset.width == reference["width"]
        and dataset.height == reference["height"]
    )


def clean_float_nodata(
    array: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    output = array.astype("float64", copy=True)

    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if np.isfinite(nodata_value):
                output[output == nodata_value] = np.nan
        except (TypeError, ValueError):
            pass

    output[output == -9999.0] = np.nan
    output[output == -32768.0] = np.nan
    output[np.abs(output) > 1.0e20] = np.nan

    return output


def read_dvplus_raster(
    path: Path,
) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(path, "Step 3.7 filled DV+ raster")

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"DV+ raster has no CRS: {path}")

        raw = dataset.read(1)
        dvplus = clean_float_nodata(raw, dataset.nodata)

        reference = {
            "crs": dataset.crs,
            "transform": dataset.transform,
            "width": dataset.width,
            "height": dataset.height,
            "bounds": dataset.bounds,
            "epsg": dataset.crs.to_epsg(),
            "resolution_x": abs(dataset.transform.a),
            "resolution_y": abs(dataset.transform.e),
        }

    materially_negative = (
        np.isfinite(dvplus) & (dvplus < -1.0e-6)
    )

    if materially_negative.any():
        raise ValueError(
            f"DV+ contains {int(materially_negative.sum()):,} materially "
            "negative cells. Recheck Step 3.7."
        )

    # Numerical safeguard only. This is not renormalization.
    dvplus = np.where(
        np.isfinite(dvplus),
        np.maximum(dvplus, 0.0),
        np.nan,
    )

    return dvplus, reference


def read_mask_to_reference(
    path: Path,
    reference: dict[str, Any],
    description: str,
) -> np.ndarray:
    require_file(path, description)

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        raw = dataset.read(1)
        nodata = dataset.nodata

        if same_grid(dataset, reference):
            values = raw.astype("float64")

            if nodata is not None:
                values[raw == nodata] = 0.0

            values[~np.isfinite(values)] = 0.0
            return values > 0.5

        destination = np.zeros(
            (reference["height"], reference["width"]),
            dtype="float64",
        )

        reproject(
            source=raw,
            destination=destination,
            src_transform=dataset.transform,
            src_crs=dataset.crs,
            src_nodata=nodata,
            dst_transform=reference["transform"],
            dst_crs=reference["crs"],
            dst_nodata=0.0,
            resampling=Resampling.nearest,
        )

        destination[~np.isfinite(destination)] = 0.0
        return destination > 0.5


def resolve_display_limits(
    values: np.ndarray,
    metadata: dict[str, Any] | None,
) -> tuple[float, float, str]:
    vmin = float(DISPLAY_VMIN)
    vmax: float | None = None
    source = "fallback"

    if USE_METADATA_DVPLUS_CAP and metadata:
        try:
            cap = metadata.get("dvplus_post", {}).get("cap")
            if cap is not None:
                cap_value = float(cap)
                if np.isfinite(cap_value) and cap_value > vmin:
                    vmax = cap_value
                    source = "Step 3.7 metadata cap"
        except Exception:
            vmax = None

    if vmax is None:
        fallback = float(FALLBACK_DISPLAY_VMAX)
        if np.isfinite(fallback) and fallback > vmin:
            vmax = fallback
            source = "configured fallback"

    if vmax is None or vmax <= vmin:
        vmax = float(np.nanmax(values))
        source = "observed maximum"

    if not np.isfinite(vmax) or vmax <= vmin:
        raise RuntimeError("Could not determine a valid DV+ display maximum.")

    return vmin, vmax, source


# =========================================================
# CARTOGRAPHIC HELPERS
# =========================================================
def to_dms(value: float, is_lon: bool = True) -> str:
    absolute = abs(value)
    degrees = int(absolute)

    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_lon
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def add_north_arrow_tri(
    axis,
    size: float = 0.09,
    pad: float = 0.07,
    label: str = "N",
    arrow_downshift: float = 0.025,
    label_upshift: float = 0.0,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (
        (1.0 - pad)
        + size * 0.10
        + label_upshift
    )

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=8,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base
        if candidate >= target_metres:
            return candidate

    return 10 * base


def add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    crs_type: str = "wgs84",
    mid_lat_deg: float | None = None,
    segments: int = 4,
    fontsize: float = 4,
    frame_lw: float = 0.4,
) -> None:
    width_data = east - west
    height_data = north - south

    if crs_type == "utm":
        width_metres = width_data
        data_units_per_metre = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = 0.5 * (south + north)

        metres_per_degree = metres_per_degree_lon(mid_lat_deg)
        width_metres = width_data * metres_per_degree
        data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = total_length_metres * data_units_per_metre
    segment_length_units = segment_length_metres * data_units_per_metre

    x_start = (
        west
        + 0.5 * width_data
        - 0.5 * total_length_units
    )
    y_base = south + 0.06 * height_data
    bar_height = 0.018 * height_data

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_data

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (
                    x_start + index * segment_length_units,
                    y_base,
                ),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=(
                    "black" if index % 2 == 0 else "white"
                ),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_data,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_data,
        "0",
        ha="center",
        va="top",
        fontsize=fontsize,
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_data,
        end_label,
        ha="center",
        va="top",
        fontsize=fontsize,
        zorder=15,
    )


# =========================================================
# ROAD AND BOUNDARY OVERLAYS
# =========================================================
def load_roads(
    road_path: Path,
    target_crs,
):
    if not road_path.exists():
        log(f"[WARN] Road shapefile not found: {road_path}")
        return None

    target_crs_text = str(target_crs)

    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_path)

            if roads.crs is None:
                raise ValueError(
                    "Road shapefile has no CRS. Assign its correct CRS "
                    "before plotting."
                )

            roads = roads.to_crs(target_crs_text)
            return roads

        except Exception as exc:
            log(f"[WARN] GeoPandas road loading failed: {exc}")

    if HAVE_FIONA:
        try:
            geometries = []

            with fiona.open(road_path, "r") as source:
                source_crs = source.crs

                if not source_crs:
                    raise ValueError(
                        "Road shapefile has no CRS. Assign its correct CRS "
                        "before plotting."
                    )

                for feature in source:
                    geometry = feature["geometry"]
                    if geometry is None:
                        continue

                    geometries.append(
                        transform_geom(
                            source_crs,
                            target_crs_text,
                            geometry,
                        )
                    )

            return {"geometries": geometries}

        except Exception as exc:
            log(f"[WARN] Fiona road loading failed: {exc}")

    return None


def plot_roads(
    axis,
    roads,
    colour: str = "red",
    linewidth: float = 0.7,
    alpha: float = 0.9,
    zorder: int = 850,
) -> None:
    if roads is None:
        return

    try:
        if HAVE_GPD and hasattr(roads, "plot"):
            roads.plot(
                ax=axis,
                color=colour,
                linewidth=linewidth,
                alpha=alpha,
                zorder=zorder,
            )
            return

        if isinstance(roads, dict):
            for geometry in roads["geometries"]:
                if geometry is None:
                    continue

                if geometry["type"] == "LineString":
                    line_groups = [geometry["coordinates"]]
                elif geometry["type"] == "MultiLineString":
                    line_groups = geometry["coordinates"]
                else:
                    continue

                for line in line_groups:
                    axis.plot(
                        [point[0] for point in line],
                        [point[1] for point in line],
                        color=colour,
                        linewidth=linewidth,
                        alpha=alpha,
                        zorder=zorder,
                    )

    except Exception as exc:
        log(f"[WARN] Road plotting failed: {exc}")


def draw_study_boundary(
    axis,
    study_mask: np.ndarray,
    reference: dict[str, Any],
) -> None:
    bounds = reference["bounds"]
    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    axis.contour(
        study_mask.astype("uint8"),
        levels=[0.5],
        colors="black",
        linewidths=0.7,
        extent=extent,
        origin="upper",
        zorder=999,
    )


# =========================================================
# STATISTICS
# =========================================================
def calculate_statistics(
    values: np.ndarray,
) -> dict[str, float | int]:
    values = values[np.isfinite(values)].astype("float64")

    if values.size == 0:
        return {"n": 0}

    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "positive_fraction": float(np.mean(values > 0.0)),
        "unique_values_rounded_6dp": int(
            np.unique(np.round(values, 6)).size
        ),
    }


# =========================================================
# MAP
# =========================================================
def create_dvplus_map(
    dvplus: np.ndarray,
    valid_mask: np.ndarray,
    study_mask: np.ndarray,
    reference: dict[str, Any],
    display_vmin: float,
    display_vmax: float,
) -> None:
    bounds = reference["bounds"]

    raster_extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    buffer_x = (
        bounds.right - bounds.left
    ) * DISPLAY_BUFFER_FRACTION
    buffer_y = (
        bounds.top - bounds.bottom
    ) * DISPLAY_BUFFER_FRACTION

    west = bounds.left - buffer_x
    east = bounds.right + buffer_x
    south = bounds.bottom - buffer_y
    north = bounds.top + buffer_y

    display_array = np.ma.masked_invalid(
        np.where(valid_mask, dvplus, np.nan)
    )

    raster_crs = reference["crs"]
    crs_is_geographic = bool(
        raster_crs and raster_crs.is_geographic
    )

    x_mid = 0.5 * (bounds.left + bounds.right)
    y_mid = 0.5 * (bounds.bottom + bounds.top)

    if crs_is_geographic:
        def format_longitude(value, position):
            return to_dms(value, True)

        def format_latitude(value, position):
            return to_dms(value, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError(
                "Projected DV+ raster requires pyproj for DMS labels."
            )

        to_geographic = Transformer.from_crs(
            raster_crs,
            "EPSG:4326",
            always_xy=True,
        )

        def format_longitude(value, position):
            longitude, _ = to_geographic.transform(
                value,
                y_mid,
            )
            return to_dms(longitude, True)

        def format_latitude(value, position):
            _, latitude = to_geographic.transform(
                x_mid,
                value,
            )
            return to_dms(latitude, False)

    roads = load_roads(ROAD_SHP, raster_crs)

    figure, axis = plt.subplots(
        1,
        1,
        figsize=MAP_FIGSIZE,
        facecolor="white",
    )
    axis.set_facecolor("white")

    # Direct display of the Step 3.7 raster. No new spatial interpolation.
    image = axis.imshow(
        display_array,
        cmap=DVPLUS_CMAP,
        extent=raster_extent,
        vmin=display_vmin,
        vmax=display_vmax,
        origin="upper",
        interpolation=RASTER_DISPLAY_INTERPOLATION,
        alpha=1.0,
        zorder=1,
    )

    axis.set_xlim(west, east)
    axis.set_ylim(south, north)

    plot_roads(
        axis,
        roads,
        colour="red",
        linewidth=0.7,
        alpha=0.9,
        zorder=850,
    )
    draw_study_boundary(
        axis,
        study_mask,
        reference,
    )

    # No "Latitude" / "Longitude" axis titles (Step 3.1 style).

    axis.xaxis.set_major_locator(
        LinearLocator(LONGITUDE_DMS_TICK_COUNT)
    )

    # Fixed latitude tick count so the rotated DMS labels
    # cannot overlap (Step 3.1 style: three latitude labels).
    # Applies for both geographic and projected rasters.
    axis.yaxis.set_major_locator(
        LinearLocator(LATITUDE_DMS_TICK_COUNT)
    )

    axis.xaxis.set_major_formatter(
        FuncFormatter(format_longitude)
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(format_latitude)
    )

    axis.tick_params(
        axis="x",
        which="major",
        labelsize=DMS_TICK_FONTSIZE,
    )
    axis.tick_params(
        axis="y",
        which="major",
        labelsize=DMS_TICK_FONTSIZE,
    )

    for tick_label in axis.get_xticklabels():
        tick_label.set_rotation(45)
        tick_label.set_ha("right")
        tick_label.set_fontsize(DMS_TICK_FONTSIZE)

    y_tick_labels = axis.get_yticklabels()
    for tick_label in y_tick_labels:
        tick_label.set_fontsize(DMS_TICK_FONTSIZE)
        tick_label.set_rotation(90)
        tick_label.set_va("center")

    # Lift the bottom-most latitude label fully above its tick so it
    # clears the first (45°-rotated) longitude label at the corner.
    if y_tick_labels:
        y_tick_labels[0].set_va("bottom")
        log(
            "[ADJUST] Bottom latitude DMS label anchored above its "
            "tick (va='bottom') to avoid corner overlap with the "
            "first longitude label."
        )

    add_north_arrow_tri(
        axis,
        size=0.09,
        pad=0.09,
    )

    crs_type = (
        "wgs84" if crs_is_geographic else "utm"
    )

    add_checker_scale_bar(
        axis,
        west,
        east,
        south,
        north,
        crs_type=crs_type,
        mid_lat_deg=(
            y_mid if crs_is_geographic else None
        ),
        segments=4,
        fontsize=4,
    )

    legend_elements = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=1.0,
            label="Highway",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=1.0,
            label="Study Area",
        ),
    ]

    legend = axis.legend(
        handles=legend_elements,
        loc="center",
        bbox_to_anchor=(0.82, 0.15),
        bbox_transform=axis.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=2.5,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0,
    )
    legend.get_frame().set_edgecolor("lightgray")
    legend.get_frame().set_linewidth(0.3)

    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes(
        "right",
        size=CAX_SIZE,
        pad=CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(
        labelsize=5,
    )
    colorbar.set_label(
        DVPLUS_INFO["units"],
        fontsize=4,
        fontweight="bold",
    )
    colorbar.set_ticks(
        np.linspace(display_vmin, display_vmax, 5)
    )

    figure.tight_layout()

    if SAVE_FIGURES:
        figure.savefig(
            OUT_MAP_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_MAP_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_MAP_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )

        log(f"[MAP] {OUT_MAP_PNG}")
        log(f"[MAP] {OUT_MAP_PDF}")
        log(f"[MAP] {OUT_MAP_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# HISTOGRAM
# =========================================================
def create_dvplus_histogram(
    values: np.ndarray,
    statistics: dict[str, float | int],
    display_vmin: float,
    display_vmax: float,
) -> None:
    figure, axis = plt.subplots(
        1,
        1,
        figsize=HIST_FIGSIZE,
        facecolor="white",
    )

    bins = np.linspace(display_vmin, display_vmax, 21)
    counts, edges = np.histogram(values, bins=bins)
    centres = 0.5 * (edges[:-1] + edges[1:])

    normalized_centres = (
        (centres - display_vmin)
        / max(display_vmax - display_vmin, 1.0e-12)
    )
    colours = DVPLUS_CMAP(
        np.clip(normalized_centres, 0.0, 1.0)
    )

    axis.bar(
        centres,
        counts,
        width=(bins[1] - bins[0]) * 0.9,
        color=colours,
        edgecolor="black",
        linewidth=0.4,
        alpha=0.9,
    )

    axis.axvline(
        statistics["mean"],
        color="darkred",
        linestyle="--",
        linewidth=0.7,
        label=f"Mean: {statistics['mean']:.2f}",
    )
    axis.axvline(
        statistics["median"],
        color="orange",
        linestyle="--",
        linewidth=0.7,
        label=f"Median: {statistics['median']:.2f}",
    )
    axis.axvline(
        statistics["p95"],
        color="green",
        linestyle="--",
        linewidth=0.7,
        label=f"95th %ile: {statistics['p95']:.2f}",
    )

    axis.legend(
        fontsize=3,
        loc="upper right",
        framealpha=0.9,
    )
    axis.set_xlabel(
        DVPLUS_INFO["units"],
        fontsize=4,
        fontweight="bold",
    )
    axis.set_ylabel(
        "Number of Pixels",
        fontsize=4,
        fontweight="bold",
    )
    axis.tick_params(
        axis="both",
        which="major",
        labelsize=4,
    )
    axis.set_xlim(display_vmin, display_vmax)
    axis.grid(
        True,
        alpha=0.25,
        linestyle="-",
        linewidth=0.4,
        axis="y",
    )
    axis.set_axisbelow(True)

    figure.tight_layout(pad=1.2)

    if SAVE_FIGURES:
        figure.savefig(
            OUT_HIST_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_HIST_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_HIST_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )

        log(f"[HISTOGRAM] {OUT_HIST_PNG}")
        log(f"[HISTOGRAM] {OUT_HIST_PDF}")
        log(f"[HISTOGRAM] {OUT_HIST_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# OUTPUT SUMMARY
# =========================================================
def write_statistics_csv(
    path: Path,
    statistics: dict[str, float | int],
    coverage: dict[str, float | int],
    display: dict[str, Any],
) -> None:
    rows: list[tuple[str, Any]] = []

    for key, value in coverage.items():
        rows.append((f"coverage_{key}", value))

    for key, value in statistics.items():
        rows.append((f"DVplus_{key}", value))

    for key, value in display.items():
        rows.append((f"display_{key}", value))

    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["metric", "value"])
        writer.writerows(rows)


# =========================================================
# MAIN
# =========================================================
def main() -> None:
    log("=== Step 3.7 visualization: direct DV+ raster ===")

    dvplus, reference = read_dvplus_raster(DVPLUS_FILLED_TIF)

    log(f"[INPUT] Step 3.7 raster: {DVPLUS_FILLED_TIF}")
    log(
        "[PLOT] direct raster=True | cmap=viridis | "
        "origin=upper | interpolation=nearest"
    )

    study_mask = read_mask_to_reference(
        STUDY_MASK_TIF,
        reference,
        "study-area mask",
    )

    valid_mask = (
        study_mask
        & np.isfinite(dvplus)
        & (dvplus >= 0.0)
    )

    if not valid_mask.any():
        raise RuntimeError(
            "No valid Step 3.7 DV+ cells occur inside the study area."
        )

    values = dvplus[valid_mask]
    statistics = calculate_statistics(values)

    metadata = load_json_optional(DVPLUS_META_JSON)

    display_vmin, display_vmax, display_limit_source = (
        resolve_display_limits(values, metadata)
    )

    study_pixels = int(study_mask.sum())
    valid_pixels = int(valid_mask.sum())

    coverage = {
        "study_pixels": study_pixels,
        "valid_DVplus_pixels": valid_pixels,
        "valid_percent_of_study": (
            100.0 * valid_pixels / max(study_pixels, 1)
        ),
    }

    display_settings = {
        "vmin_mm_yr": display_vmin,
        "vmax_mm_yr": display_vmax,
        "vmax_source": display_limit_source,
    }

    log(
        f"[RASTER] EPSG:{reference['epsg']} | "
        f"{reference['width']} × {reference['height']} | "
        f"{reference['resolution_x']:.3f} m"
    )
    log(
        f"[COVERAGE] {valid_pixels:,}/{study_pixels:,} study pixels "
        f"({coverage['valid_percent_of_study']:.3f}%)"
    )
    log(
        f"[DV+] min={statistics['min']:.3f} | "
        f"median={statistics['median']:.3f} | "
        f"mean={statistics['mean']:.3f} | "
        f"p95={statistics['p95']:.3f} | "
        f"max={statistics['max']:.3f} mm/yr"
    )
    log(
        f"[DISPLAY] {display_vmin:.3f} to {display_vmax:.3f} mm/yr "
        f"({display_limit_source})"
    )
    log(
        "[STYLE] 2×4 inch map | four longitude DMS labels | "
        "three latitude DMS labels (rotated 90°, bottom label "
        "lifted) | no axis titles | 900 dpi | PNG/PDF/SVG"
    )

    summary = {
        "generated_at_utc": utc_now_iso(),
        "purpose": (
            "Direct-raster visualization of Step 3.7 filled DV+ using "
            "the final Step 3.1 coherence cartographic style."
        ),
        "scientific_plotting": {
            "direct_step37_DVplus_raster": True,
            "additional_RBF_used_for_visualization": False,
            "additional_IDW_used_for_visualization": False,
            "additional_cubic_interpolation_used_for_visualization": False,
            "scatter_reconstruction_used_for_visualization": False,
            "display_interpolation": RASTER_DISPLAY_INTERPOLATION,
            "values_renormalized_for_visualization": False,
            "input_raster_may_contain_step37_IDW_fills": True,
        },
        "inputs": {
            "DVplus_filled_raster": str(DVPLUS_FILLED_TIF),
            "study_mask": str(STUDY_MASK_TIF),
            "road": str(ROAD_SHP),
            "Step37_metadata": (
                str(DVPLUS_META_JSON)
                if metadata is not None
                else None
            ),
        },
        "coverage": coverage,
        "statistics_mm_yr": statistics,
        "display": display_settings,
        "visual_style": {
            "map_figsize_inches": list(MAP_FIGSIZE),
            "histogram_figsize_inches": list(HIST_FIGSIZE),
            "colormap": "viridis",
            "axis_titles_shown": False,
            "display_buffer_fraction": DISPLAY_BUFFER_FRACTION,
            "longitude_DMS_tick_count": LONGITUDE_DMS_TICK_COUNT,
            "latitude_DMS_tick_count": LATITUDE_DMS_TICK_COUNT,
            "latitude_DMS_label_rotation_deg": 90,
            "bottom_latitude_label_va": "bottom",
            "DMS_tick_fontsize": DMS_TICK_FONTSIZE,
            "north_arrow": True,
            "checkerboard_scale_bar": True,
            "compact_legend": True,
            "side_colour_bar": True,
            "figure_dpi": FIG_DPI,
        },
        "outputs": {
            "map_png": str(OUT_MAP_PNG),
            "map_pdf": str(OUT_MAP_PDF),
            "map_svg": str(OUT_MAP_SVG),
            "histogram_png": str(OUT_HIST_PNG),
            "histogram_pdf": str(OUT_HIST_PDF),
            "histogram_svg": str(OUT_HIST_SVG),
            "summary_json": str(OUT_SUMMARY_JSON),
            "statistics_csv": str(OUT_STATS_CSV),
        },
    }

    with open(OUT_SUMMARY_JSON, "w", encoding="utf-8") as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    write_statistics_csv(
        OUT_STATS_CSV,
        statistics,
        coverage,
        display_settings,
    )

    create_dvplus_map(
        dvplus=dvplus,
        valid_mask=valid_mask,
        study_mask=study_mask,
        reference=reference,
        display_vmin=display_vmin,
        display_vmax=display_vmax,
    )

    create_dvplus_histogram(
        values=values,
        statistics=statistics,
        display_vmin=display_vmin,
        display_vmax=display_vmax,
    )

    log(f"[SUMMARY] {OUT_SUMMARY_JSON}")
    log(f"[SUMMARY] {OUT_STATS_CSV}")
    log("=== Step 3.7 visualization completed successfully ===")


def create_dvplus_visualization() -> None:
    try:
        main()
    except Exception as exc:
        print(
            f"[ERROR] Failed to create Step 3.7 DV+ visualization: {exc}"
        )
        traceback.print_exc()
        raise


if __name__ == "__main__":
    create_dvplus_visualization()

In [ ]:
# =========================================================
# Step 3.7 — DV+ Visualization
# =========================================================
#
# Scientific plotting rules:
#   1. Reads the Step 3.7 filled DV+ raster directly.
#   2. Adds no new RBF, IDW, cubic, or scatter reconstruction during plotting.
#   3. Uses the final Step 3.1 coherence map style:
#        - Viridis colour map
#        - 2.5 × 5 inch map; 2.5 × 2.5 inch histogram
#        - 2% buffered display extent
#        - four longitude DMS labels (LinearLocator, rotated 45°, bold)
#        - three latitude DMS labels (LinearLocator, rotated 90°, bold)
#        - bottom-most latitude label anchored above its tick (va="bottom")
#          so it cannot collide with the first longitude label at the corner
#        - no "Latitude" / "Longitude" axis titles
#        - north arrow
#        - checkerboard scale bar
#        - highway and study-area overlays
#        - compact legend and side colour bar
#        - PNG, PDF, and SVG outputs at 900 dpi
#   4. The Step 3.7 input raster may already contain IDW-filled values because
#      densification is part of the analytical Step 3.7 workflow. This
#      visualization does not perform any additional spatial interpolation.
#   5. The same valid raster cells are used for the map, histogram, and stats.
# =========================================================

from __future__ import annotations
import os

import csv
import json
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_geom

import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator, LinearLocator
from matplotlib.patches import Polygon, Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False


# =========================================================
# PATHS
# =========================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
ROAD_SHP = BASE / "NTD_Road.shp"

DVPLUS_DIR = BASE / "Results" / "Step-3.7-DVplus_Densify"

# Analytical output from Step 3.7
DVPLUS_FILLED_TIF = (
    DVPLUS_DIR / "dv_plus_filled_mm_yr_32645_12p5m_float32.tif"
)
DVPLUS_META_JSON = DVPLUS_DIR / "dvplus_densify_metadata.json"

# Visualization outputs
OUT_MAP_PNG = DVPLUS_DIR / "DVplus Map.png"
OUT_MAP_PDF = DVPLUS_DIR / "DVplus Map.pdf"
OUT_MAP_SVG = DVPLUS_DIR / "DVplus Map.svg"

OUT_HIST_PNG = DVPLUS_DIR / "DVplus Histogram.png"
OUT_HIST_PDF = DVPLUS_DIR / "DVplus Histogram.pdf"
OUT_HIST_SVG = DVPLUS_DIR / "DVplus Histogram.svg"

OUT_SUMMARY_JSON = DVPLUS_DIR / "dvplus_visualization_summary.json"
OUT_STATS_CSV = DVPLUS_DIR / "dvplus_visualization_statistics.csv"

DVPLUS_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# DISPLAY SETTINGS
# =========================================================
# Final Step 3.1 coherence map size for this corridor.
MAP_FIGSIZE = (2.5, 5)
HIST_FIGSIZE = (2.5, 2.5)

CAX_SIZE = "5%"
CAX_PAD = 0.01
FIG_DPI = 900

# Same colour map as the final SNDR plot
DVPLUS_CMAP = plt.get_cmap("viridis")

DVPLUS_INFO = {
    "title": "Downslope Deformation Velocity",
    "units": "DV+ (mm/yr)",
}

# Direct raster display only
RASTER_DISPLAY_INTERPOLATION = "nearest"

# Same map buffer and DMS settings as the final Step 3.1 map
DISPLAY_BUFFER_FRACTION = 0.02
LONGITUDE_DMS_TICK_COUNT = 4
LATITUDE_DMS_TICK_COUNT = 3
DMS_TICK_FONTSIZE = 6.2

# DV+ is nonnegative by definition in Step 3.7.
DISPLAY_VMIN = 0.0

# Step 3.7 currently applies DVPLUS_CAP = 150 mm/yr.
# Reading the metadata cap keeps the display consistent with the analytical
# raster. If metadata are unavailable, this value is used as a fallback.
USE_METADATA_DVPLUS_CAP = True
FALLBACK_DISPLAY_VMAX = 150.0

SHOW_FIGURES = True
SAVE_FIGURES = True

# Font convention used by the final Step 3.1 map
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = [
    "Times New Roman",
    "DejaVu Serif",
]
plt.rcParams["axes.unicode_minus"] = False


# =========================================================
# GENERAL HELPERS
# =========================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json_optional(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None

    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return None


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


# =========================================================
# RASTER HELPERS
# =========================================================
def same_grid(
    dataset: rasterio.io.DatasetReader,
    reference: dict[str, Any],
) -> bool:
    return (
        dataset.crs == reference["crs"]
        and dataset.transform == reference["transform"]
        and dataset.width == reference["width"]
        and dataset.height == reference["height"]
    )


def clean_float_nodata(
    array: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    output = array.astype("float64", copy=True)

    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if np.isfinite(nodata_value):
                output[output == nodata_value] = np.nan
        except (TypeError, ValueError):
            pass

    output[output == -9999.0] = np.nan
    output[output == -32768.0] = np.nan
    output[np.abs(output) > 1.0e20] = np.nan

    return output


def read_dvplus_raster(
    path: Path,
) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(path, "Step 3.7 filled DV+ raster")

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"DV+ raster has no CRS: {path}")

        raw = dataset.read(1)
        dvplus = clean_float_nodata(raw, dataset.nodata)

        reference = {
            "crs": dataset.crs,
            "transform": dataset.transform,
            "width": dataset.width,
            "height": dataset.height,
            "bounds": dataset.bounds,
            "epsg": dataset.crs.to_epsg(),
            "resolution_x": abs(dataset.transform.a),
            "resolution_y": abs(dataset.transform.e),
        }

    materially_negative = (
        np.isfinite(dvplus) & (dvplus < -1.0e-6)
    )

    if materially_negative.any():
        raise ValueError(
            f"DV+ contains {int(materially_negative.sum()):,} materially "
            "negative cells. Recheck Step 3.7."
        )

    # Numerical safeguard only. This is not renormalization.
    dvplus = np.where(
        np.isfinite(dvplus),
        np.maximum(dvplus, 0.0),
        np.nan,
    )

    return dvplus, reference


def read_mask_to_reference(
    path: Path,
    reference: dict[str, Any],
    description: str,
) -> np.ndarray:
    require_file(path, description)

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        raw = dataset.read(1)
        nodata = dataset.nodata

        if same_grid(dataset, reference):
            values = raw.astype("float64")

            if nodata is not None:
                values[raw == nodata] = 0.0

            values[~np.isfinite(values)] = 0.0
            return values > 0.5

        destination = np.zeros(
            (reference["height"], reference["width"]),
            dtype="float64",
        )

        reproject(
            source=raw,
            destination=destination,
            src_transform=dataset.transform,
            src_crs=dataset.crs,
            src_nodata=nodata,
            dst_transform=reference["transform"],
            dst_crs=reference["crs"],
            dst_nodata=0.0,
            resampling=Resampling.nearest,
        )

        destination[~np.isfinite(destination)] = 0.0
        return destination > 0.5


def resolve_display_limits(
    values: np.ndarray,
    metadata: dict[str, Any] | None,
) -> tuple[float, float, str]:
    vmin = float(DISPLAY_VMIN)
    vmax: float | None = None
    source = "fallback"

    if USE_METADATA_DVPLUS_CAP and metadata:
        try:
            cap = metadata.get("dvplus_post", {}).get("cap")
            if cap is not None:
                cap_value = float(cap)
                if np.isfinite(cap_value) and cap_value > vmin:
                    vmax = cap_value
                    source = "Step 3.7 metadata cap"
        except Exception:
            vmax = None

    if vmax is None:
        fallback = float(FALLBACK_DISPLAY_VMAX)
        if np.isfinite(fallback) and fallback > vmin:
            vmax = fallback
            source = "configured fallback"

    if vmax is None or vmax <= vmin:
        vmax = float(np.nanmax(values))
        source = "observed maximum"

    if not np.isfinite(vmax) or vmax <= vmin:
        raise RuntimeError("Could not determine a valid DV+ display maximum.")

    return vmin, vmax, source


# =========================================================
# CARTOGRAPHIC HELPERS
# =========================================================
def to_dms(value: float, is_lon: bool = True) -> str:
    absolute = abs(value)
    degrees = int(absolute)

    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_lon
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def add_north_arrow_tri(
    axis,
    size: float = 0.09,
    pad: float = 0.07,
    label: str = "N",
    arrow_downshift: float = 0.025,
    label_upshift: float = 0.0,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (
        (1.0 - pad)
        + size * 0.10
        + label_upshift
    )

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base
        if candidate >= target_metres:
            return candidate

    return 10 * base


def add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    crs_type: str = "wgs84",
    mid_lat_deg: float | None = None,
    segments: int = 4,
    fontsize: float = 5.6,
    frame_lw: float = 0.4,
) -> None:
    width_data = east - west
    height_data = north - south

    if crs_type == "utm":
        width_metres = width_data
        data_units_per_metre = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = 0.5 * (south + north)

        metres_per_degree = metres_per_degree_lon(mid_lat_deg)
        width_metres = width_data * metres_per_degree
        data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = total_length_metres * data_units_per_metre
    segment_length_units = segment_length_metres * data_units_per_metre

    x_start = (
        west
        + 0.5 * width_data
        - 0.5 * total_length_units
    )
    y_base = south + 0.06 * height_data
    bar_height = 0.018 * height_data

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_data

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (
                    x_start + index * segment_length_units,
                    y_base,
                ),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=(
                    "black" if index % 2 == 0 else "white"
                ),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_data,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
        color="black",
        alpha=1.0,
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_data,
        "0",
        ha="center",
        va="top",
        fontsize=5,
        fontweight="bold",
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_data,
        end_label,
        ha="center",
        va="top",
        fontsize=5,
        fontweight="bold",
        zorder=15,
    )


# =========================================================
# ROAD AND BOUNDARY OVERLAYS
# =========================================================
def load_roads(
    road_path: Path,
    target_crs,
):
    if not road_path.exists():
        log(f"[WARN] Road shapefile not found: {road_path}")
        return None

    target_crs_text = str(target_crs)

    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_path)

            if roads.crs is None:
                raise ValueError(
                    "Road shapefile has no CRS. Assign its correct CRS "
                    "before plotting."
                )

            roads = roads.to_crs(target_crs_text)
            return roads

        except Exception as exc:
            log(f"[WARN] GeoPandas road loading failed: {exc}")

    if HAVE_FIONA:
        try:
            geometries = []

            with fiona.open(road_path, "r") as source:
                source_crs = source.crs

                if not source_crs:
                    raise ValueError(
                        "Road shapefile has no CRS. Assign its correct CRS "
                        "before plotting."
                    )

                for feature in source:
                    geometry = feature["geometry"]
                    if geometry is None:
                        continue

                    geometries.append(
                        transform_geom(
                            source_crs,
                            target_crs_text,
                            geometry,
                        )
                    )

            return {"geometries": geometries}

        except Exception as exc:
            log(f"[WARN] Fiona road loading failed: {exc}")

    return None


def plot_roads(
    axis,
    roads,
    colour: str = "red",
    linewidth: float = 0.7,
    alpha: float = 0.9,
    zorder: int = 850,
) -> None:
    if roads is None:
        return

    try:
        if HAVE_GPD and hasattr(roads, "plot"):
            roads.plot(
                ax=axis,
                color=colour,
                linewidth=linewidth,
                alpha=alpha,
                zorder=zorder,
            )
            return

        if isinstance(roads, dict):
            for geometry in roads["geometries"]:
                if geometry is None:
                    continue

                if geometry["type"] == "LineString":
                    line_groups = [geometry["coordinates"]]
                elif geometry["type"] == "MultiLineString":
                    line_groups = geometry["coordinates"]
                else:
                    continue

                for line in line_groups:
                    axis.plot(
                        [point[0] for point in line],
                        [point[1] for point in line],
                        color=colour,
                        linewidth=linewidth,
                        alpha=alpha,
                        zorder=zorder,
                    )

    except Exception as exc:
        log(f"[WARN] Road plotting failed: {exc}")


def draw_study_boundary(
    axis,
    study_mask: np.ndarray,
    reference: dict[str, Any],
) -> None:
    bounds = reference["bounds"]
    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    axis.contour(
        study_mask.astype("uint8"),
        levels=[0.5],
        colors="black",
        linewidths=0.7,
        extent=extent,
        origin="upper",
        zorder=999,
    )


# =========================================================
# STATISTICS
# =========================================================
def calculate_statistics(
    values: np.ndarray,
) -> dict[str, float | int]:
    values = values[np.isfinite(values)].astype("float64")

    if values.size == 0:
        return {"n": 0}

    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "positive_fraction": float(np.mean(values > 0.0)),
        "unique_values_rounded_6dp": int(
            np.unique(np.round(values, 6)).size
        ),
    }


# =========================================================
# MAP
# =========================================================
def create_dvplus_map(
    dvplus: np.ndarray,
    valid_mask: np.ndarray,
    study_mask: np.ndarray,
    reference: dict[str, Any],
    display_vmin: float,
    display_vmax: float,
) -> None:
    bounds = reference["bounds"]

    raster_extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    buffer_x = (
        bounds.right - bounds.left
    ) * DISPLAY_BUFFER_FRACTION
    buffer_y = (
        bounds.top - bounds.bottom
    ) * DISPLAY_BUFFER_FRACTION

    west = bounds.left - buffer_x
    east = bounds.right + buffer_x
    south = bounds.bottom - buffer_y
    north = bounds.top + buffer_y

    display_array = np.ma.masked_invalid(
        np.where(valid_mask, dvplus, np.nan)
    )

    raster_crs = reference["crs"]
    crs_is_geographic = bool(
        raster_crs and raster_crs.is_geographic
    )

    x_mid = 0.5 * (bounds.left + bounds.right)
    y_mid = 0.5 * (bounds.bottom + bounds.top)

    if crs_is_geographic:
        def format_longitude(value, position):
            return to_dms(value, True)

        def format_latitude(value, position):
            return to_dms(value, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError(
                "Projected DV+ raster requires pyproj for DMS labels."
            )

        to_geographic = Transformer.from_crs(
            raster_crs,
            "EPSG:4326",
            always_xy=True,
        )

        def format_longitude(value, position):
            longitude, _ = to_geographic.transform(
                value,
                y_mid,
            )
            return to_dms(longitude, True)

        def format_latitude(value, position):
            _, latitude = to_geographic.transform(
                x_mid,
                value,
            )
            return to_dms(latitude, False)

    roads = load_roads(ROAD_SHP, raster_crs)

    figure, axis = plt.subplots(
        1,
        1,
        figsize=MAP_FIGSIZE,
        facecolor="white",
    )
    axis.set_facecolor("white")

    # Direct display of the Step 3.7 raster. No new spatial interpolation.
    image = axis.imshow(
        display_array,
        cmap=DVPLUS_CMAP,
        extent=raster_extent,
        vmin=display_vmin,
        vmax=display_vmax,
        origin="upper",
        interpolation=RASTER_DISPLAY_INTERPOLATION,
        alpha=1.0,
        zorder=1,
    )

    axis.set_xlim(west, east)
    axis.set_ylim(south, north)

    plot_roads(
        axis,
        roads,
        colour="red",
        linewidth=0.7,
        alpha=0.9,
        zorder=850,
    )
    draw_study_boundary(
        axis,
        study_mask,
        reference,
    )

    # No "Latitude" / "Longitude" axis titles (Step 3.1 style).

    axis.xaxis.set_major_locator(
        LinearLocator(LONGITUDE_DMS_TICK_COUNT)
    )

    # Fixed latitude tick count so the rotated DMS labels
    # cannot overlap (Step 3.1 style: three latitude labels).
    # Applies for both geographic and projected rasters.
    axis.yaxis.set_major_locator(
        LinearLocator(LATITUDE_DMS_TICK_COUNT)
    )

    axis.xaxis.set_major_formatter(
        FuncFormatter(format_longitude)
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(format_latitude)
    )

    axis.tick_params(
        axis="x",
        which="major",
        labelsize=DMS_TICK_FONTSIZE,
        width=0.8,
        length=2.5,
    )
    axis.tick_params(
        axis="y",
        which="major",
        labelsize=DMS_TICK_FONTSIZE,
        width=0.8,
        length=2.5,
    )

    for tick_label in axis.get_xticklabels():
        tick_label.set_rotation(45)
        tick_label.set_ha("right")
        tick_label.set_fontsize(DMS_TICK_FONTSIZE)
        tick_label.set_fontweight("bold")

    y_tick_labels = axis.get_yticklabels()
    for tick_label in y_tick_labels:
        tick_label.set_fontsize(DMS_TICK_FONTSIZE)
        tick_label.set_fontweight("bold")
        tick_label.set_rotation(90)
        tick_label.set_va("center")

    # Lift the bottom-most latitude label fully above its tick so it
    # clears the first (45°-rotated) longitude label at the corner.
    if y_tick_labels:
        y_tick_labels[0].set_va("bottom")
        log(
            "[ADJUST] Bottom latitude DMS label anchored above its "
            "tick (va='bottom') to avoid corner overlap with the "
            "first longitude label."
        )

    add_north_arrow_tri(
        axis,
        size=0.09,
        pad=0.09,
    )

    crs_type = (
        "wgs84" if crs_is_geographic else "utm"
    )

    add_checker_scale_bar(
        axis,
        west,
        east,
        south,
        north,
        crs_type=crs_type,
        mid_lat_deg=(
            y_mid if crs_is_geographic else None
        ),
        segments=4,
        fontsize=5.6,
    )

    legend_elements = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=1.0,
            label="Road",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=1.0,
            label="Study Area",
        ),
    ]

    legend = axis.legend(
        handles=legend_elements,
        loc="lower right",
        bbox_to_anchor=(0.98, 0.09),
        bbox_transform=axis.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=3.1,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0,
    )
    legend.get_frame().set_edgecolor("lightgray")
    legend.get_frame().set_linewidth(0.3)
    for txt in legend.get_texts():
        txt.set_fontweight("bold")

    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes(
        "right",
        size=CAX_SIZE,
        pad=CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(
        labelsize=5.4,
        width=0.6,
        length=2,
    )
    for tick_label in colorbar.ax.get_yticklabels():
        tick_label.set_fontweight("bold")
    colorbar.set_label(
        DVPLUS_INFO["units"],
        fontsize=6.0,
        fontweight="bold",
    )
    colorbar.set_ticks(
        np.linspace(display_vmin, display_vmax, 5)
    )

    figure.tight_layout()

    if SAVE_FIGURES:
        figure.savefig(
            OUT_MAP_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_MAP_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_MAP_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )

        log(f"[MAP] {OUT_MAP_PNG}")
        log(f"[MAP] {OUT_MAP_PDF}")
        log(f"[MAP] {OUT_MAP_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# HISTOGRAM
# =========================================================
def create_dvplus_histogram(
    values: np.ndarray,
    statistics: dict[str, float | int],
    display_vmin: float,
    display_vmax: float,
) -> None:
    figure, axis = plt.subplots(
        1,
        1,
        figsize=HIST_FIGSIZE,
        facecolor="white",
    )

    bins = np.linspace(display_vmin, display_vmax, 21)
    counts, edges = np.histogram(values, bins=bins)
    centres = 0.5 * (edges[:-1] + edges[1:])

    normalized_centres = (
        (centres - display_vmin)
        / max(display_vmax - display_vmin, 1.0e-12)
    )
    colours = DVPLUS_CMAP(
        np.clip(normalized_centres, 0.0, 1.0)
    )

    axis.bar(
        centres,
        counts,
        width=(bins[1] - bins[0]) * 0.9,
        color=colours,
        edgecolor="black",
        linewidth=0.4,
        alpha=0.9,
    )

    axis.axvline(
        statistics["mean"],
        color="darkred",
        linestyle="--",
        linewidth=0.7,
        label=f"Mean: {statistics['mean']:.2f}",
    )
    axis.axvline(
        statistics["median"],
        color="orange",
        linestyle="--",
        linewidth=0.7,
        label=f"Median: {statistics['median']:.2f}",
    )
    axis.axvline(
        statistics["p95"],
        color="green",
        linestyle="--",
        linewidth=0.7,
        label=f"95th %ile: {statistics['p95']:.2f}",
    )

    axis.legend(
        fontsize=5.0,
        loc="upper right",
        framealpha=0.9,
    )
    axis.set_xlabel(
        DVPLUS_INFO["units"],
        fontsize=7,
        fontweight="bold",
    )
    axis.set_ylabel(
        "Number of Pixels",
        fontsize=7,
        fontweight="bold",
    )
    axis.tick_params(
        axis="both",
        which="major",
        labelsize=6,
    )
    for tick_label in axis.get_xticklabels() + axis.get_yticklabels():
        tick_label.set_fontweight("bold")

    axis.set_xlim(display_vmin, display_vmax)
    axis.grid(
        True,
        alpha=0.25,
        linestyle="-",
        linewidth=0.4,
        axis="y",
    )
    axis.set_axisbelow(True)

    figure.tight_layout(pad=1.2)

    if SAVE_FIGURES:
        figure.savefig(
            OUT_HIST_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_HIST_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_HIST_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )

        log(f"[HISTOGRAM] {OUT_HIST_PNG}")
        log(f"[HISTOGRAM] {OUT_HIST_PDF}")
        log(f"[HISTOGRAM] {OUT_HIST_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# OUTPUT SUMMARY
# =========================================================
def write_statistics_csv(
    path: Path,
    statistics: dict[str, float | int],
    coverage: dict[str, float | int],
    display: dict[str, Any],
) -> None:
    rows: list[tuple[str, Any]] = []

    for key, value in coverage.items():
        rows.append((f"coverage_{key}", value))

    for key, value in statistics.items():
        rows.append((f"DVplus_{key}", value))

    for key, value in display.items():
        rows.append((f"display_{key}", value))

    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["metric", "value"])
        writer.writerows(rows)


# =========================================================
# MAIN
# =========================================================
def main() -> None:
    log("=== Step 3.7 visualization: direct DV+ raster ===")

    dvplus, reference = read_dvplus_raster(DVPLUS_FILLED_TIF)

    log(f"[INPUT] Step 3.7 raster: {DVPLUS_FILLED_TIF}")
    log(
        "[PLOT] direct raster=True | cmap=viridis | "
        "origin=upper | interpolation=nearest"
    )

    study_mask = read_mask_to_reference(
        STUDY_MASK_TIF,
        reference,
        "study-area mask",
    )

    valid_mask = (
        study_mask
        & np.isfinite(dvplus)
        & (dvplus >= 0.0)
    )

    if not valid_mask.any():
        raise RuntimeError(
            "No valid Step 3.7 DV+ cells occur inside the study area."
        )

    values = dvplus[valid_mask]
    statistics = calculate_statistics(values)

    metadata = load_json_optional(DVPLUS_META_JSON)

    display_vmin, display_vmax, display_limit_source = (
        resolve_display_limits(values, metadata)
    )

    study_pixels = int(study_mask.sum())
    valid_pixels = int(valid_mask.sum())

    coverage = {
        "study_pixels": study_pixels,
        "valid_DVplus_pixels": valid_pixels,
        "valid_percent_of_study": (
            100.0 * valid_pixels / max(study_pixels, 1)
        ),
    }

    display_settings = {
        "vmin_mm_yr": display_vmin,
        "vmax_mm_yr": display_vmax,
        "vmax_source": display_limit_source,
    }

    log(
        f"[RASTER] EPSG:{reference['epsg']} | "
        f"{reference['width']} × {reference['height']} | "
        f"{reference['resolution_x']:.3f} m"
    )
    log(
        f"[COVERAGE] {valid_pixels:,}/{study_pixels:,} study pixels "
        f"({coverage['valid_percent_of_study']:.3f}%)"
    )
    log(
        f"[DV+] min={statistics['min']:.3f} | "
        f"median={statistics['median']:.3f} | "
        f"mean={statistics['mean']:.3f} | "
        f"p95={statistics['p95']:.3f} | "
        f"max={statistics['max']:.3f} mm/yr"
    )
    log(
        f"[DISPLAY] {display_vmin:.3f} to {display_vmax:.3f} mm/yr "
        f"({display_limit_source})"
    )
    log(
        "[STYLE] 2.5×5 inch map | four longitude DMS labels (bold) | "
        "three latitude DMS labels (rotated 90°, bold, bottom label "
        "lifted) | no axis titles | 900 dpi | PNG/PDF/SVG"
    )

    summary = {
        "generated_at_utc": utc_now_iso(),
        "purpose": (
            "Direct-raster visualization of Step 3.7 filled DV+ using "
            "the final Step 3.1 coherence cartographic style."
        ),
        "scientific_plotting": {
            "direct_step37_DVplus_raster": True,
            "additional_RBF_used_for_visualization": False,
            "additional_IDW_used_for_visualization": False,
            "additional_cubic_interpolation_used_for_visualization": False,
            "scatter_reconstruction_used_for_visualization": False,
            "display_interpolation": RASTER_DISPLAY_INTERPOLATION,
            "values_renormalized_for_visualization": False,
            "input_raster_may_contain_step37_IDW_fills": True,
        },
        "inputs": {
            "DVplus_filled_raster": str(DVPLUS_FILLED_TIF),
            "study_mask": str(STUDY_MASK_TIF),
            "road": str(ROAD_SHP),
            "Step37_metadata": (
                str(DVPLUS_META_JSON)
                if metadata is not None
                else None
            ),
        },
        "coverage": coverage,
        "statistics_mm_yr": statistics,
        "display": display_settings,
        "visual_style": {
            "map_figsize_inches": list(MAP_FIGSIZE),
            "histogram_figsize_inches": list(HIST_FIGSIZE),
            "colormap": "viridis",
            "axis_titles_shown": False,
            "display_buffer_fraction": DISPLAY_BUFFER_FRACTION,
            "longitude_DMS_tick_count": LONGITUDE_DMS_TICK_COUNT,
            "latitude_DMS_tick_count": LATITUDE_DMS_TICK_COUNT,
            "latitude_DMS_label_rotation_deg": 90,
            "bottom_latitude_label_va": "bottom",
            "DMS_tick_fontsize": DMS_TICK_FONTSIZE,
            "north_arrow": True,
            "checkerboard_scale_bar": True,
            "compact_legend": True,
            "side_colour_bar": True,
            "figure_dpi": FIG_DPI,
            "style_reference": "Step 5.1b revised cartographic style",
            "DMS_labels_bold": True,
            "map_legend_fontsize": 3.1,
            "colorbar_tick_fontsize": 4.4,
            "colorbar_label_fontsize": 5.2,
            "north_arrow_label_fontsize": 9,
            "scale_bar_unit_fontsize": 5.6,
        },
        "outputs": {
            "map_png": str(OUT_MAP_PNG),
            "map_pdf": str(OUT_MAP_PDF),
            "map_svg": str(OUT_MAP_SVG),
            "histogram_png": str(OUT_HIST_PNG),
            "histogram_pdf": str(OUT_HIST_PDF),
            "histogram_svg": str(OUT_HIST_SVG),
            "summary_json": str(OUT_SUMMARY_JSON),
            "statistics_csv": str(OUT_STATS_CSV),
        },
    }

    with open(OUT_SUMMARY_JSON, "w", encoding="utf-8") as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    write_statistics_csv(
        OUT_STATS_CSV,
        statistics,
        coverage,
        display_settings,
    )

    create_dvplus_map(
        dvplus=dvplus,
        valid_mask=valid_mask,
        study_mask=study_mask,
        reference=reference,
        display_vmin=display_vmin,
        display_vmax=display_vmax,
    )

    create_dvplus_histogram(
        values=values,
        statistics=statistics,
        display_vmin=display_vmin,
        display_vmax=display_vmax,
    )

    log(f"[SUMMARY] {OUT_SUMMARY_JSON}")
    log(f"[SUMMARY] {OUT_STATS_CSV}")
    log("=== Step 3.7 visualization completed successfully ===")


def create_dvplus_visualization() -> None:
    try:
        main()
    except Exception as exc:
        print(
            f"[ERROR] Failed to create Step 3.7 DV+ visualization: {exc}"
        )
        traceback.print_exc()
        raise


if __name__ == "__main__":
    create_dvplus_visualization()

# Phase-4: Standardization (winsorized z-scores)

# Step-4.1: Limits

In [ ]:
# =========================================================
# Step 4.1 — Limits (winsor caps & stats)
# u ∈ { DV⁺ (mm/yr), TWI, SM (m³/m³) }
# =========================================================
import os, json
from datetime import datetime
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# --------------------------
# CONFIG
# --------------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

# Anchor
ANCHOR_TIF = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")

# Masks
# DV+ limits should be computed inside PS footprint
DYN_MASK_TIF = os.path.join(BASE, "Results", "Step-0.3-Quality_Masks", "dynamic_mask.tif")
CONTEXT_MASK_TIF = os.path.join(BASE, "Results", "Step-0.3-Quality_Masks", "context_mask.tif")
STUDY_MASK_TIF   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")

# Inputs
DVPLUS_FILLED_TIF = os.path.join(
    BASE, "Results", "Step-3.7-DVplus_Densify", "dv_plus_filled_mm_yr_32645_12p5m_float32.tif"
)
TWI_TIF = os.path.join(BASE, "Results", "Step-1.6-TWI", "twi.tif")
SM_TIF  = os.path.join(BASE, "Results", "Step-2.3-SM",  "sm_monsoon_mean_2020_2023_32645_12p5m_float32.tif")

# Optional 
QUALITY_MASK_STATS_JSON = os.path.join(BASE, "Results", "Step-0.3-Quality_Masks", "quality_mask_stats.json")

# Output
OUT_DIR = os.path.join(BASE, "Results", "Step-4.1-Limits")
os.makedirs(OUT_DIR, exist_ok=True)
LIMITS_JSON = os.path.join(OUT_DIR, "winsor_limits.json")

# Winsorization percentiles
P_LOW, P_HIGH = 2.0, 98.0

# NODATA handling
NODATA = -9999.0

# --------------------------
# Helpers
# --------------------------
def pp(msg):
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

def read_anchor_spec(path):
    if not os.path.exists(path):
        raise SystemExit(f"[ANCHOR] Missing: {path}")
    with rasterio.open(path) as ds:
        return {
            "tf": ds.transform,
            "crs": ds.crs,
            "width": ds.width,
            "height": ds.height,
            "profile": ds.profile,
            "epsg": int(ds.crs.to_epsg()) if ds.crs else None,
        }

def load_and_snap(path, anchor, resampling=Resampling.bilinear):
    """Load a raster and snap to anchor grid if needed. Returns float64 with NaN for nodata."""
    if not os.path.exists(path):
        return None
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float64")
        nd = src.nodata
        if nd is not None and np.isfinite(nd):
            arr = np.where(arr == float(nd), np.nan, arr)

        same_grid = (
            src.crs == anchor["crs"]
            and src.transform == anchor["tf"]
            and src.width == anchor["width"]
            and src.height == anchor["height"]
        )
        if same_grid:
            return arr

        pp(f"[SNAP] Reprojecting {os.path.basename(path)} to anchor grid …")
        dst = np.full((anchor["height"], anchor["width"]), np.nan, dtype="float64")
        reproject(
            source=arr,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=anchor["tf"],
            dst_crs=anchor["crs"],
            resampling=resampling,
            src_nodata=np.nan,
            dst_nodata=np.nan,
        )
        return dst

def load_mask_as_bool(path, anchor):
    """Reads any mask-like raster and returns boolean snapped to anchor (nearest)."""
    if not os.path.exists(path):
        return None
    with rasterio.open(path) as ms:
        same = (
            ms.crs == anchor["crs"]
            and ms.transform == anchor["tf"]
            and ms.width == anchor["width"]
            and ms.height == anchor["height"]
        )
        if same:
            m = ms.read(1).astype("float64")
            nd = ms.nodata
            if nd is not None and np.isfinite(nd):
                m = np.where(m == float(nd), 0.0, m)
        else:
            m = np.zeros((anchor["height"], anchor["width"]), dtype="float64")
            reproject(
                source=ms.read(1).astype("float64"),
                destination=m,
                src_transform=ms.transform,
                src_crs=ms.crs,
                dst_transform=anchor["tf"],
                dst_crs=anchor["crs"],
                resampling=Resampling.nearest,
                src_nodata=0,
                dst_nodata=0,
            )
    return (m > 0.5)

def layer_stats(arr, valid_bool, clamp_nonneg=False, units=""):
    """Compute stats and winsor caps on arr over valid_bool."""
    if arr is None:
        return {"present": False, "reason": "file not found"}

    x = arr.copy()
    neg_count = None
    if clamp_nonneg:
        neg_count = int(np.sum((x < 0) & valid_bool & np.isfinite(x)))
        x[(x < 0) & np.isfinite(x)] = 0.0

    m = valid_bool & np.isfinite(x)
    n = int(m.sum())
    if n == 0:
        return {"present": True, "n_valid": 0, "error": "no finite samples under mask"}

    vals = x[m]

    mean = float(np.mean(vals))
    std  = float(np.std(vals, ddof=0))
    med  = float(np.median(vals))
    mad  = float(1.4826 * np.median(np.abs(vals - med)))  # robust σ

    prqs = [0, 1, 2, 5, 25, 50, 75, 95, 98, 99, 100]
    perc = np.percentile(vals, prqs).astype("float64")
    pmap = {f"p{p:02d}" if p < 10 else f"p{p}": float(v) for p, v in zip(prqs, perc)}

    lo = float(np.percentile(vals, P_LOW))
    hi = float(np.percentile(vals, P_HIGH))
    if not (hi > lo):
        lo = float(np.min(vals))
        hi = float(np.max(vals))

    out = {
        "present": True,
        "n_valid": n,
        "units": units,
        "mean": mean,
        "std": std,
        "median": med,
        "mad_robust": mad,
        "min": float(np.min(vals)),
        "max": float(np.max(vals)),
        "winsor_low": lo,
        "winsor_high": hi,
        "percentiles": pmap,
    }
    if neg_count is not None:
        out["negatives_clamped_to_zero"] = neg_count
    return out

# --------------------------
# MAIN
# --------------------------
def main():
    pp("=== Step 4.1 — Limits (winsor caps & stats) [using Step-3.7 DV+ filled] ===")

    # 1) Anchor
    anchor = read_anchor_spec(ANCHOR_TIF)
    pp(f"[ANCHOR] EPSG:{anchor['epsg']}  grid={anchor['width']}×{anchor['height']}")

    # 2) Masks
    dyn = load_mask_as_bool(DYN_MASK_TIF, anchor)
    if dyn is None:
        raise SystemExit(f"[MASK] Missing dynamic_mask: {DYN_MASK_TIF}")

    ctx = None
    if os.path.exists(CONTEXT_MASK_TIF):
        ctx = load_mask_as_bool(CONTEXT_MASK_TIF, anchor)
        pp(f"[MASK] Using context_mask: {CONTEXT_MASK_TIF}")
    else:
        ctx = load_mask_as_bool(STUDY_MASK_TIF, anchor)
        if ctx is None:
            raise SystemExit(f"[MASK] Missing study_mask: {STUDY_MASK_TIF}")
        pp(f"[MASK] context_mask missing; using study_mask instead: {STUDY_MASK_TIF}")

    pp(f"[MASK] dynamic valid: {int(dyn.sum()):,} / {dyn.size:,} ({100*dyn.sum()/dyn.size:.2f}%)")
    pp(f"[MASK] context valid: {int(ctx.sum()):,} / {ctx.size:,} ({100*ctx.sum()/ctx.size:.2f}%)")

    # Optional notes
    qm_note = {}
    if os.path.exists(QUALITY_MASK_STATS_JSON):
        try:
            with open(QUALITY_MASK_STATS_JSON, "r") as f:
                qm = json.load(f)
            qm_note = {
                "coherence_threshold": qm.get("coherence_threshold"),
                "dilate_pixels": qm.get("dilate_pixels"),
                "valid_area_km2": qm.get("valid_area_km2"),
            }
        except Exception:
            qm_note = {}

    # 3) Load layers (snap to anchor)
    dv  = load_and_snap(DVPLUS_FILLED_TIF, anchor, resampling=Resampling.nearest)   # DV+ filled (mm/yr)
    twi = load_and_snap(TWI_TIF,          anchor, resampling=Resampling.bilinear)  # TWI
    sm  = load_and_snap(SM_TIF,           anchor, resampling=Resampling.bilinear)  # SM

    if dv is None:
        raise SystemExit(f"[INPUT] Missing DV+ filled from Step-3.7: {DVPLUS_FILLED_TIF}")
    if twi is None:
        raise SystemExit(f"[INPUT] Missing TWI: {TWI_TIF}")
    if sm is None:
        raise SystemExit(f"[INPUT] Missing SM: {SM_TIF}")

    # 4) Compute stats & winsor caps
    # DVplus caps on PS footprint; TWI/SM caps on context
    dv_stats  = layer_stats(dv,  dyn, clamp_nonneg=True,  units="mm/yr")
    twi_stats = layer_stats(twi, ctx, clamp_nonneg=False, units="")
    sm_stats  = layer_stats(sm,  ctx, clamp_nonneg=False, units="m3/m3")

    # 5) Write JSON in a Step-4.2-friendly structure
    # (Your Step-4.2 searches for limits under lim['layers'][name] and keys like winsor_low/high.)
    limits = {
        "generated_at": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "anchor_epsg": anchor["epsg"],
        "winsor_percentiles": {"low": float(P_LOW), "high": float(P_HIGH)},
        "masks": {
            "dynamic_mask_path": DYN_MASK_TIF,
            "context_mask_path": CONTEXT_MASK_TIF if os.path.exists(CONTEXT_MASK_TIF) else STUDY_MASK_TIF,
            "dynamic_valid_count": int(dyn.sum()),
            "context_valid_count": int(ctx.sum()),
            **({"quality_mask_notes": qm_note} if qm_note else {}),
        },
        "layers": {
            "DVplus": {
                "path": DVPLUS_FILLED_TIF,
                **dv_stats
            },
            "TWI": {
                "path": TWI_TIF,
                **twi_stats
            },
            "SM": {
                "path": SM_TIF,
                **sm_stats
            },
        },
        "notes": [
            "DVplus limits computed on dynamic_mask (PS footprint).",
            "TWI and SM limits computed on context_mask (or study_mask fallback).",
            "DVplus negatives (if any) clamped to 0 before stats/caps.",
        ],
    }

    with open(LIMITS_JSON, "w") as f:
        json.dump(limits, f, indent=2)
    pp(f"[OUT] winsor_limits.json → {LIMITS_JSON}")

    # 6) Quick console summary
    def brief(tag, s):
        if not s.get("present", False):
            return f"{tag:6s}: MISSING"
        if s.get("n_valid", 0) == 0:
            return f"{tag:6s}: n=0 (no finite samples)"
        p = s.get("percentiles", {})
        return (f"{tag:6s}: n={s['n_valid']:,} "
                f"min={s['min']:.4f} p02={p.get('p02', np.nan):.4f} "
                f"mean={s['mean']:.4f} p98={p.get('p98', np.nan):.4f} "
                f"max={s['max']:.4f} | caps=({s['winsor_low']:.4f}, {s['winsor_high']:.4f})")

    pp(brief("DV⁺", dv_stats))
    pp(brief("TWI", twi_stats))
    pp(brief("SM",  sm_stats))
    pp("=== Done (Step 4.1) ===")

if __name__ == "__main__":
    main()


# Step-4.2: Standarization (μ, σ and winsorized z-scores)

In [ ]:
# ============================================================
# Step 4.2 — μ, σ and winsorized z-scores for {DV+, TWI, SM}
# ============================================================
import os, json
from datetime import datetime
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# ----------------PATHS--------------------
BASE = os.environ.get("LSM_DATA_DIR", "./data")

ANCHOR_TIF  = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
LIMITS_JSON = os.path.join(BASE, "Results", "Step-4.1-Limits", "winsor_limits.json")

TWI_TIF = os.path.join(BASE, "Results", "Step-1.6-TWI", "twi.tif")
SM_TIF  = os.path.join(BASE, "Results", "Step-2.3-SM",  "sm_monsoon_mean_2020_2023_32645_12p5m_float32.tif")

# Masks
STUDY_MASK_TIF   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")
DYN_MASK_TIF     = os.path.join(BASE, "Results", "Step-0.3-Quality_Masks", "dynamic_mask.tif")
CONTEXT_MASK_TIF = os.path.join(BASE, "Results", "Step-0.3-Quality_Masks", "context_mask.tif")

# Observed DV source distance 
DVPLUS_FILLED_TIF = os.path.join(BASE, "Results", "Step-3.7-DVplus_Densify", "dv_plus_filled_mm_yr_32645_12p5m_float32.tif")
DV_SOURCE_DIST_TIF = os.path.join(BASE, "Results", "Step-3.7-DVplus_Densify", "dv_plus_source_dist_m_32645_12p5m_float32.tif")

# Output
OUT_DIR = os.path.join(BASE, "Results", "Step-4.2-Standardization")
os.makedirs(OUT_DIR, exist_ok=True)

Z_META_JSON         = os.path.join(OUT_DIR, "zscore_metadata.json")
DERIVED_LIMITS_JSON = os.path.join(OUT_DIR, "winsor_caps_used.json")

# --------------------------
# CONSTANTS
# --------------------------
NODATA = -9999.0
EPS = 1e-12

# Only used if limits JSON missing/broken for a layer
FALLBACK_Q_LOW  = 1.0
FALLBACK_Q_HIGH = 99.0

# DV+ observed selection
OBS_DIST_EPS_M = 1e-6  # treat <= this as "source pixel"

def pp(msg):
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

# --------------------------
# IO helpers
# --------------------------
def read_anchor(path):
    if not os.path.exists(path):
        raise SystemExit(f"[ANCHOR] Missing: {path}")
    with rasterio.open(path) as ds:
        prof = ds.profile.copy()
        return {
            "tf": ds.transform, "crs": ds.crs,
            "width": ds.width,  "height": ds.height,
            "profile": prof,
            "epsg": int(ds.crs.to_epsg()) if ds.crs else None,
        }

def read_and_snap(path, anchor, resampling=Resampling.bilinear):
    if not os.path.exists(path):
        return None
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float64")
        nd = src.nodata

        same = (
            src.crs == anchor["crs"] and src.transform == anchor["tf"] and
            src.width == anchor["width"] and src.height == anchor["height"]
        )
        if same:
            if nd is not None and np.isfinite(nd):
                arr = np.where(arr == float(nd), np.nan, arr)
            return arr

        dst = np.full((anchor["height"], anchor["width"]), np.nan, dtype="float64")
        reproject(
            source=arr, destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=anchor["tf"],  dst_crs=anchor["crs"],
            resampling=resampling,
            src_nodata=nd if (nd is not None and np.isfinite(nd)) else None,
            dst_nodata=np.nan
        )
        return dst

def read_mask_bool(path, anchor):
    if not os.path.exists(path):
        return None
    with rasterio.open(path) as ms:
        m = ms.read(1).astype("float64")
        nd = ms.nodata
        if nd is not None and np.isfinite(nd):
            m = np.where(m == float(nd), 0.0, m)

        same = (
            ms.crs == anchor["crs"] and ms.transform == anchor["tf"] and
            ms.width == anchor["width"] and ms.height == anchor["height"]
        )
        if same:
            return (m > 0.5)

        dst = np.zeros((anchor["height"], anchor["width"]), dtype="float64")
        reproject(
            source=m, destination=dst,
            src_transform=ms.transform, src_crs=ms.crs,
            dst_transform=anchor["tf"],  dst_crs=anchor["crs"],
            resampling=Resampling.nearest,
            src_nodata=0, dst_nodata=0
        )
        return (dst > 0.5)

def write_tif(path, arr_float64, anchor, nodata=NODATA):
    prof = anchor["profile"].copy()
    prof.update(
        driver="GTiff", count=1, dtype="float32", nodata=nodata,
        compress="LZW", tiled=True, bigtiff="IF_NEEDED",
        transform=anchor["tf"], crs=anchor["crs"],
        width=anchor["width"], height=anchor["height"]
    )
    tmp = path + ".part"
    with rasterio.open(tmp, "w", **prof) as ds:
        ds.write(arr_float64.astype("float32"), 1)
    os.replace(tmp, path)

# --------------------------
# Limits loading
# --------------------------
_LOW_KEYS  = {"winsor_low","low","lo","cap_low","clip_low","lower","lower_cap","p01","q01","p1","q1"}
_HIGH_KEYS = {"winsor_high","high","hi","cap_high","clip_high","upper","upper_cap","p99","q99","p99th","q99th"}

_LAYER_ALIASES = {
    "DVplus": ["dvplus","dv+","dv_plus","dv⁺","dvplus_mm_yr","dv"],
    "TWI":    ["twi","topographic wetness index"],
    "SM":     ["sm","soil moisture","era5l_sm","era5_l_sm"]
}

def _extract_caps_from_dict(d):
    if not isinstance(d, dict):
        return None, None
    dnorm = {str(k).strip().lower(): v for k, v in d.items()}
    low = high = None
    for k in _LOW_KEYS:
        if k in dnorm:
            low = float(dnorm[k]); break
    for k in _HIGH_KEYS:
        if k in dnorm:
            high = float(dnorm[k]); break
    if (low is not None) and (high is not None):
        return low, high
    for nest in ("winsor","winsor_caps","caps","limits"):
        if nest in dnorm and isinstance(dnorm[nest], dict):
            low, high = _extract_caps_from_dict(dnorm[nest])
            if (low is not None) and (high is not None):
                return low, high
    return None, None

def _find_layer_section(lim_dict, logical_name):
    if not isinstance(lim_dict, dict):
        return None
    layers = lim_dict.get("layers")
    if isinstance(layers, dict):
        keys_lower = {k.lower(): k for k in layers.keys()}
        if logical_name.lower() in keys_lower:
            return layers[keys_lower[logical_name.lower()]]
        for alias in _LAYER_ALIASES.get(logical_name, []):
            if alias in keys_lower:
                return layers[keys_lower[alias]]
    keys_lower = {k.lower(): k for k in lim_dict.keys()}
    if logical_name.lower() in keys_lower:
        return lim_dict[keys_lower[logical_name.lower()]]
    for alias in _LAYER_ALIASES.get(logical_name, []):
        if alias in keys_lower:
            return lim_dict[keys_lower[alias]]
    return None

def get_caps_or_fallback(lim_dict, arr, mask_bool, name, qlow=FALLBACK_Q_LOW, qhigh=FALLBACK_Q_HIGH):
    section = _find_layer_section(lim_dict, name) if lim_dict else None
    lo = hi = None
    if section is not None:
        lo, hi = _extract_caps_from_dict(section)

    if (lo is None) or (hi is None) or (not np.isfinite(lo)) or (not np.isfinite(hi)) or (hi <= lo):
        ok = np.isfinite(arr) & mask_bool
        if not ok.any():
            raise SystemExit(f"[LIMITS] Cannot derive caps for {name}: no finite samples under its mask.")
        lo = float(np.nanpercentile(arr[ok], qlow))
        hi = float(np.nanpercentile(arr[ok], qhigh))
        if hi <= lo:
            span = (np.nanmax(arr[ok]) - np.nanmin(arr[ok])) or 1.0
            lo = float(lo - 0.01 * span)
            hi = float(hi + 0.01 * span)
        pp(f"[LIMITS] {name}: caps derived {qlow:.1f}–{qhigh:.1f}% → [{lo:.6g}, {hi:.6g}]")
        derived = True
    else:
        pp(f"[LIMITS] {name}: caps from JSON → [{lo:.6g}, {hi:.6g}]")
        derived = False

    return lo, hi, derived

# --------------------------
# Winsor + z
# --------------------------
def winsorize(arr, lo, hi):
    out = arr.copy()
    ok = np.isfinite(out)
    out[ok & (out < lo)] = lo
    out[ok & (out > hi)] = hi
    return out

# --------------------------
# MAIN
# --------------------------
def main():
    pp("=== Step 4.2: Standardization  ===")

    anchor = read_anchor(ANCHOR_TIF)
    pp(f"[ANCHOR] EPSG:{anchor['epsg']} grid={anchor['width']}×{anchor['height']}")

    # masks
    study = read_mask_bool(STUDY_MASK_TIF, anchor)
    if study is None:
        raise SystemExit(f"[MASK] Missing study_mask: {STUDY_MASK_TIF}")

    dyn = read_mask_bool(DYN_MASK_TIF, anchor)
    if dyn is None:
        raise SystemExit(f"[MASK] Missing dynamic_mask: {DYN_MASK_TIF}")
    dyn = dyn & study

    if os.path.exists(CONTEXT_MASK_TIF):
        ctx = read_mask_bool(CONTEXT_MASK_TIF, anchor)
        if ctx is None:
            raise SystemExit(f"[MASK] context_mask exists but unreadable: {CONTEXT_MASK_TIF}")
        ctx_path = CONTEXT_MASK_TIF
    else:
        ctx = study.copy()
        ctx_path = STUDY_MASK_TIF
    ctx = ctx & study

    pp(f"[MASK] study={int(study.sum()):,} dyn={int(dyn.sum()):,} ctx={int(ctx.sum()):,}")

    # limits json
    lim = None
    if os.path.exists(LIMITS_JSON):
        with open(LIMITS_JSON, "r") as f:
            lim = json.load(f)
        pp(f"[LIMITS] Loaded {LIMITS_JSON}")
    else:
        pp("[WARN] winsor_limits.json missing; will derive caps from data.")

    # load layers
    dv  = read_and_snap(DVPLUS_FILLED_TIF, anchor, resampling=Resampling.nearest)
    twi = read_and_snap(TWI_TIF, anchor, resampling=Resampling.bilinear)
    sm  = read_and_snap(SM_TIF,  anchor, resampling=Resampling.bilinear)

    if dv is None:  raise SystemExit(f"[INPUT] Missing: {DVPLUS_FILLED_TIF}")
    if twi is None: raise SystemExit(f"[INPUT] Missing: {TWI_TIF}")
    if sm is None:  raise SystemExit(f"[INPUT] Missing: {SM_TIF}")

    # Observed DV mask 
    dv_src_dist = read_and_snap(DV_SOURCE_DIST_TIF, anchor, resampling=Resampling.nearest)
    if dv_src_dist is None:
        raise SystemExit(f"[INPUT] Missing DV source distance raster: {DV_SOURCE_DIST_TIF}")

    dv_observed = dyn & np.isfinite(dv_src_dist) & (dv_src_dist <= OBS_DIST_EPS_M)
    pp(f"[DV] observed (source_dist==0) pixels: {int(dv_observed.sum()):,}")

    if dv_observed.sum() < 50:
        raise SystemExit("[DV] Too few observed DV pixels for stable stats. Check dv_plus_source_dist_m raster.")

    # Output paths
    LAYERS = {
        "DVplus": {
            "arr": dv,
            "domain_mask": dyn,          # output domain
            "stats_mask": dv_observed,   # caps/mu/sigma domain
            "winsor_tif": os.path.join(OUT_DIR, "dvplus_winsor_mm_yr_32645_12p5m_float32.tif"),
            "z_tif":      os.path.join(OUT_DIR, "z_dvplus_32645_12p5m_float32.tif"),
            "units": "mm/yr",
            "clamp_nonneg": True
        },
        "TWI": {
            "arr": twi,
            "domain_mask": ctx,
            "stats_mask": ctx,
            "winsor_tif": os.path.join(OUT_DIR, "twi_winsor_32645_12p5m_float32.tif"),
            "z_tif":      os.path.join(OUT_DIR, "z_twi_32645_12p5m_float32.tif"),
            "units": "",
            "clamp_nonneg": False
        },
        "SM": {
            "arr": sm,
            "domain_mask": ctx,
            "stats_mask": ctx,
            "winsor_tif": os.path.join(OUT_DIR, "sm_winsor_m3m3_32645_12p5m_float32.tif"),
            "z_tif":      os.path.join(OUT_DIR, "z_sm_32645_12p5m_float32.tif"),
            "units": "m3/m3",
            "clamp_nonneg": False
        }
    }

    caps_used = {}
    derived_flags = {}
    meta = {
        "generated_at": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "anchor_epsg": anchor["epsg"],
        "limits_json": LIMITS_JSON if os.path.exists(LIMITS_JSON) else None,
        "masks": {
            "study_mask": STUDY_MASK_TIF,
            "dynamic_mask": DYN_MASK_TIF,
            "context_mask": ctx_path,
            "dv_observed_mask": f"dynamic_mask∩study_mask∩(dv_source_dist<= {OBS_DIST_EPS_M} m)"
        },
        "dv_source_dist_tif": DV_SOURCE_DIST_TIF,
        "fallback_quantiles_pct": {"low": FALLBACK_Q_LOW, "high": FALLBACK_Q_HIGH},
        "winsor_caps_used": {},
        "caps_derived_from_data": {},
        "layers": {}
    }

    for name, cfg in LAYERS.items():
        arr = cfg["arr"].copy()

        if cfg["clamp_nonneg"]:
            neg = np.isfinite(arr) & (arr < 0)
            nneg = int(neg.sum())
            if nneg:
                arr[neg] = 0.0
            pp(f"[{name}] negatives clamped: {nneg:,}")

        stats_mask = cfg["stats_mask"]
        domain_mask = cfg["domain_mask"]

        lo, hi, derived = get_caps_or_fallback(lim, arr, stats_mask, name)
        caps_used[name] = {"low": float(lo), "high": float(hi)}
        derived_flags[name] = bool(derived)

        w = winsorize(arr, lo, hi)

        # μ, σ computed on STATS mask only
        mw = stats_mask & np.isfinite(w)
        if not mw.any():
            raise SystemExit(f"[ERROR] No finite samples for {name} inside stats_mask.")
        vals = w[mw]
        mu = float(np.mean(vals))
        sigma = float(np.std(vals, ddof=0))
        if sigma < EPS:
            sigma = 1.0
            pp(f"[WARN] {name} σ≈0 forced to 1.0")

        z = (w - mu) / sigma

        # write winsor + z on DOMAIN mask (not stats_mask)
        w_out = np.full(w.shape, NODATA, dtype="float64")
        mwd = domain_mask & np.isfinite(w)
        w_out[mwd] = w[mwd]
        write_tif(cfg["winsor_tif"], w_out, anchor, nodata=NODATA)

        z_out = np.full(z.shape, NODATA, dtype="float64")
        mzd = domain_mask & np.isfinite(z)
        z_out[mzd] = z[mzd]
        write_tif(cfg["z_tif"], z_out, anchor, nodata=NODATA)

        pp(f"[OUT] {name}: {cfg['z_tif']} (μ={mu:.6g}, σ={sigma:.6g}, n_stats={int(mw.sum()):,}, n_domain={int(mzd.sum()):,})")

        meta["layers"][name] = {
            "src": DVPLUS_FILLED_TIF if name == "DVplus" else (TWI_TIF if name == "TWI" else SM_TIF),
            "winsor_tif": cfg["winsor_tif"],
            "z_tif": cfg["z_tif"],
            "units": cfg["units"],
            "winsor_low": float(lo),
            "winsor_high": float(hi),
            "mu_winsor_stats": mu,
            "sigma_winsor_stats": sigma,
            "n_stats": int(mw.sum()),
            "n_domain_written": int(mzd.sum()),
            "domain": "dynamic_mask∩study_mask" if name == "DVplus" else "context_mask∩study_mask",
            "stats_domain": "OBSERVED DV sources (source_dist==0)" if name == "DVplus" else "context_mask∩study_mask"
        }

    meta["winsor_caps_used"] = caps_used
    meta["caps_derived_from_data"] = derived_flags

    with open(Z_META_JSON, "w") as f:
        json.dump(meta, f, indent=2)
    with open(DERIVED_LIMITS_JSON, "w") as f:
        json.dump(
            {
                "winsor_caps_used": caps_used,
                "caps_derived_from_data": derived_flags,
                "fallback_quantiles_pct": {"low": FALLBACK_Q_LOW, "high": FALLBACK_Q_HIGH},
            },
            f,
            indent=2,
        )

    pp(f"[META] {Z_META_JSON}")
    pp(f"[META] {DERIVED_LIMITS_JSON}")
    pp("=== Done (Step 4.2) ===")

if __name__ == "__main__":
    main()


# Step-4.3: Z-maps

In [ ]:
# =========================================================
# Step 4.3 — Z-maps 
# =========================================================
import os, json
from datetime import datetime
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

BASE = os.environ.get("LSM_DATA_DIR", "./data")

ANCHOR_TIF       = os.path.join(BASE, "Results", "Step-0.1-Grid_CRS", "DEM_anchor_32645.tif")
STUDY_MASK_TIF   = os.path.join(BASE, "Results", "Step-0.2-Study_Mask", "study_mask.tif")
DYN_MASK_TIF     = os.path.join(BASE, "Results", "Step-0.3-Quality_Masks", "dynamic_mask.tif")
CONTEXT_MASK_TIF = os.path.join(BASE, "Results", "Step-0.3-Quality_Masks", "context_mask.tif")

STDZ_DIR  = os.path.join(BASE, "Results", "Step-4.2-Standardization")
Z_DVPLUS  = os.path.join(STDZ_DIR, "z_dvplus_32645_12p5m_float32.tif")
Z_TWI     = os.path.join(STDZ_DIR, "z_twi_32645_12p5m_float32.tif")
Z_SM      = os.path.join(STDZ_DIR, "z_sm_32645_12p5m_float32.tif")

OUT_DIR = os.path.join(BASE, "Results", "Step-4.3-Z-Maps")
os.makedirs(OUT_DIR, exist_ok=True)

OUTS = {
    "DVplus": {
        "src": Z_DVPLUS,
        "masked":  os.path.join(OUT_DIR, "z_dvplus_masked_32645_12p5m_float32.tif"),
        "display": os.path.join(OUT_DIR, "z_dvplus_display_32645_12p5m_float32.tif"),
        "png_crop": os.path.join(OUT_DIR, "z_dvplus_display_cropped.png"),
        "png_full": os.path.join(OUT_DIR, "z_dvplus_display_full.png"),
    },
    "TWI": {
        "src": Z_TWI,
        "masked":  os.path.join(OUT_DIR, "z_twi_masked_32645_12p5m_float32.tif"),
        "display": os.path.join(OUT_DIR, "z_twi_display_32645_12p5m_float32.tif"),
        "png_crop": os.path.join(OUT_DIR, "z_twi_display_cropped.png"),
        "png_full": os.path.join(OUT_DIR, "z_twi_display_full.png"),
    },
    "SM": {
        "src": Z_SM,
        "masked":  os.path.join(OUT_DIR, "z_sm_masked_32645_12p5m_float32.tif"),
        "display": os.path.join(OUT_DIR, "z_sm_display_32645_12p5m_float32.tif"),
        "png_crop": os.path.join(OUT_DIR, "z_sm_display_cropped.png"),
        "png_full": os.path.join(OUT_DIR, "z_sm_display_full.png"),
    }
}

DISPLAY_CLAMP_PER_LAYER = {
    "DVplus": (-1.0, 8.0),
    "TWI":    (-3.0, 3.0),
    "SM":     (-3.0, 1.5),
}

MAKE_PNGS           = True
WRITE_DISPLAY_TIFS  = True
CROP_PNG_TO_DATA    = True
ALSO_WRITE_FULL_PNG = True
PNG_DPI             = 220
NODATA = -9999.0

def pp(msg):
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

def read_anchor(path):
    if not os.path.exists(path):
        raise SystemExit(f"[ANCHOR] Missing: {path}")
    with rasterio.open(path) as ds:
        prof = ds.profile.copy()
        return {
            "tf": ds.transform, "crs": ds.crs,
            "width": ds.width, "height": ds.height,
            "profile": prof,
            "epsg": int(ds.crs.to_epsg()) if ds.crs else None
        }

def read_and_snap(path, anchor, resampling=Resampling.nearest):
    if not os.path.exists(path):
        raise SystemExit(f"[INPUT] Missing: {path}")
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float64")
        nd = src.nodata

        same = (
            src.crs == anchor["crs"] and src.transform == anchor["tf"] and
            src.width == anchor["width"] and src.height == anchor["height"]
        )
        if same:
            if nd is not None and np.isfinite(nd):
                arr = np.where(arr == float(nd), np.nan, arr)
            return arr

        dst = np.full((anchor["height"], anchor["width"]), np.nan, dtype="float64")
        reproject(
            source=arr, destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=anchor["tf"],  dst_crs=anchor["crs"],
            resampling=resampling,
            src_nodata=nd if (nd is not None and np.isfinite(nd)) else None,
            dst_nodata=np.nan
        )
        return dst

def read_mask_bool(path, anchor):
    if not os.path.exists(path):
        return None
    with rasterio.open(path) as ms:
        m = ms.read(1).astype("float64")
        nd = ms.nodata
        if nd is not None and np.isfinite(nd):
            m = np.where(m == float(nd), 0.0, m)

        same = (
            ms.crs == anchor["crs"] and ms.transform == anchor["tf"] and
            ms.width == anchor["width"] and ms.height == anchor["height"]
        )
        if same:
            return (m > 0.5)

        dst = np.zeros((anchor["height"], anchor["width"]), dtype="float64")
        reproject(
            source=m, destination=dst,
            src_transform=ms.transform, src_crs=ms.crs,
            dst_transform=anchor["tf"],  dst_crs=anchor["crs"],
            resampling=Resampling.nearest,
            src_nodata=0, dst_nodata=0
        )
        return (dst > 0.5)

def write_tif(path, arr_float64, anchor, nodata=NODATA):
    prof = anchor["profile"].copy()
    prof.update(
        driver="GTiff", count=1, dtype="float32", nodata=nodata,
        compress="LZW", tiled=True, bigtiff="IF_NEEDED",
        transform=anchor["tf"], crs=anchor["crs"],
        width=anchor["width"], height=anchor["height"]
    )
    tmp = path + ".part"
    with rasterio.open(tmp, "w", **prof) as dst:
        dst.write(arr_float64.astype("float32"), 1)
    os.replace(tmp, path)

def clamp_vals(z, lo, hi):
    out = z.copy()
    ok = np.isfinite(out)
    out[ok & (out < lo)] = lo
    out[ok & (out > hi)] = hi
    return out

def _crop_to_finite(arr):
    m = np.isfinite(arr)
    if not m.any():
        return arr
    rows = np.where(m.any(axis=1))[0]
    cols = np.where(m.any(axis=0))[0]
    return arr[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]

def quicklook_png(path_png, arr, clamp, title, crop=True):
    if not MAKE_PNGS:
        return
    try:
        import matplotlib.pyplot as plt
        lo, hi = clamp
        A = arr.copy()
        A[~np.isfinite(A)] = np.nan
        if crop:
            A = _crop_to_finite(A)

        h, w = A.shape
        figw = 6
        figh = max(2.0, 6 * (h / max(w, 1)))

        fig, ax = plt.subplots(figsize=(figw, figh), dpi=PNG_DPI)
        im = ax.imshow(A, vmin=lo, vmax=hi, cmap="RdBu_r", interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
        cbar = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
        cbar.set_label("z-score")
        plt.tight_layout()
        fig.savefig(path_png, bbox_inches="tight")
        plt.close(fig)
    except Exception as e:
        pp(f"[WARN] quicklook failed for {title}: {e}")

def main():
    pp("=== Step-4.3: Z-maps ===")
    anchor = read_anchor(ANCHOR_TIF)
    pp(f"[ANCHOR] EPSG:{anchor['epsg']}  grid={anchor['width']}×{anchor['height']}")

    study = read_mask_bool(STUDY_MASK_TIF, anchor)
    if study is None:
        raise SystemExit(f"[MASK] Missing study_mask: {STUDY_MASK_TIF}")

    dyn = read_mask_bool(DYN_MASK_TIF, anchor)
    if dyn is None:
        raise SystemExit(f"[MASK] Missing dynamic_mask: {DYN_MASK_TIF}")
    dyn = dyn & study

    if os.path.exists(CONTEXT_MASK_TIF):
        ctx = read_mask_bool(CONTEXT_MASK_TIF, anchor)
        if ctx is None:
            raise SystemExit(f"[MASK] context_mask exists but unreadable: {CONTEXT_MASK_TIF}")
        ctx_path = CONTEXT_MASK_TIF
    else:
        ctx = study.copy()
        ctx_path = STUDY_MASK_TIF
    ctx = ctx & study

    DOMAIN_MASK = {"DVplus": dyn, "TWI": ctx, "SM": ctx}
    pp(f"[MASK] study={int(study.sum()):,} dyn={int(dyn.sum()):,} ctx={int(ctx.sum()):,}")

    summary = {}
    for name, cfg in OUTS.items():
        z = read_and_snap(cfg["src"], anchor, resampling=Resampling.nearest)
        mask = DOMAIN_MASK[name]

        # masked tif
        z_masked = np.full(z.shape, NODATA, dtype="float64")
        m = mask & np.isfinite(z)
        z_masked[m] = z[m]
        write_tif(cfg["masked"], z_masked, anchor, nodata=NODATA)

        # display tif + png
        lo, hi = DISPLAY_CLAMP_PER_LAYER.get(name, (-3.0, 3.0))
        z_disp = z_masked.copy()
        z_disp[z_disp == NODATA] = np.nan
        z_disp = clamp_vals(z_disp, lo, hi)

        if WRITE_DISPLAY_TIFS:
            z_disp_tif = np.full(z.shape, NODATA, dtype="float64")
            md = mask & np.isfinite(z_disp)
            z_disp_tif[md] = z_disp[md]
            write_tif(cfg["display"], z_disp_tif, anchor, nodata=NODATA)

        if MAKE_PNGS:
            quicklook_png(cfg["png_crop"], z_disp, (lo, hi), f"{name} z (clamped {lo}..{hi}, cropped)", crop=CROP_PNG_TO_DATA)
            if ALSO_WRITE_FULL_PNG:
                quicklook_png(cfg["png_full"], z_disp, (lo, hi), f"{name} z (clamped {lo}..{hi}, full)", crop=False)

        v = z_disp[np.isfinite(z_disp)]
        summary[name] = {
            "n_finite": int(v.size),
            "min": float(v.min()) if v.size else None,
            "mean": float(v.mean()) if v.size else None,
            "p05": float(np.percentile(v, 5)) if v.size else None,
            "p50": float(np.percentile(v, 50)) if v.size else None,
            "p95": float(np.percentile(v, 95)) if v.size else None,
            "max": float(v.max()) if v.size else None,
            "display_clamp": [float(lo), float(hi)]
        }
        pp(f"[OUT] {name}: n={summary[name]['n_finite']:,}")

    meta = {
        "generated_at_utc": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "anchor_epsg": anchor["epsg"],
        "anchor_path": ANCHOR_TIF,
        "masks": {
            "study_mask": STUDY_MASK_TIF,
            "dynamic_mask": DYN_MASK_TIF,
            "context_mask": ctx_path,
        },
        "inputs": {k: OUTS[k]["src"] for k in OUTS.keys()},
        "outputs": OUTS,
        "display_clamp_per_layer": DISPLAY_CLAMP_PER_LAYER,
        "summary": summary,
    }
    META_OUT = os.path.join(OUT_DIR, "zmaps_metadata.json")
    with open(META_OUT, "w") as f:
        json.dump(meta, f, indent=2)
    pp(f"[META] {META_OUT}")
    pp("=== Done (Step 4.3) ===")

if __name__ == "__main__":
    main()


# Visualization: Z-Map

In [ ]:
# =========================================================
# Step 4.3 — Z-Maps Visualization 
# =========================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import FuncFormatter, MultipleLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
import traceback
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import transform as rio_transform, transform_geom

# Optional libs 
try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    from scipy.ndimage import binary_erosion
    from scipy.spatial import cKDTree
    from scipy.interpolate import griddata, RBFInterpolator
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

# =========================================================
# PATHS
# =========================================================
BASE = os.environ.get("LSM_DATA_DIR", "./data")

STUDY_MASK = os.path.join(BASE, r"Results\Step-0.2-Study_Mask", "study_mask.tif")
STUDY_SHP  = os.path.join(BASE, "Study_Area.shp")
ROAD_SHP   = os.path.join(BASE, "NTD_Road.shp")

ZMAP_DIR   = os.path.join(BASE, "Results", "Step-4.3-Z-Maps")
Z_DVPLUS_DISPLAY = os.path.join(ZMAP_DIR, "z_dvplus_display_32645_12p5m_float32.tif")
Z_TWI_DISPLAY    = os.path.join(ZMAP_DIR, "z_twi_display_32645_12p5m_float32.tif")
Z_SM_DISPLAY     = os.path.join(ZMAP_DIR, "z_sm_display_32645_12p5m_float32.tif")
os.makedirs(ZMAP_DIR, exist_ok=True)

# =========================================================
# DISPLAY SETTINGS 
# =========================================================
MAP_FIGSIZE  = (2, 2)
HIST_FIGSIZE = (2, 2)
CAX_SIZE     = "5%"
CAX_PAD      = 0.01

# Interpolation settings 
GRID_RES                 = 200
USE_RBF                  = True
IDW_NEIGHBORS            = 64
MASK_EROSION_ITERATIONS  = 2

# Variable-specific display ranges (vmin, vmax)
DISPLAY_RANGES = {
    "DVplus": (-8.0, 8.0),
    "TWI":    (-3.0, 3.0),
    "SM":     (-3.0, 3.0),
}

VAR_INFO = {
    "DVplus": {"title": "DV⁺ Z-Score", "units": "Z-score"},
    "TWI":    {"title": "TWI Z-Score", "units": "Z-score"},
    "SM":     {"title": "SM Z-Score",  "units": "Z-score"},
}

# Z-score diverging colormap 
ZSCORE_COLORS = [
    '#00008B','#0000CD','#0000FF','#4169E1','#6495ED','#87CEEB','#B0E0E6','#E0F6FF',
    '#FFFFFF',
    '#FFE0E0','#FFB6C1','#FFA07A','#FF6347','#FF4500','#FF0000','#DC143C','#B22222','#8B0000'
]
ZSCORE_CMAP = mcolors.LinearSegmentedColormap.from_list('zscore_diverging', ZSCORE_COLORS, N=256)

# =========================================================
# Helpers: DMS, north arrow, checker scale bar
# =========================================================
def to_dms(val, is_lon=True):
    a = abs(val); d = int(a)
    m_float = (a - d) * 60.0; m = int(m_float)
    s = (m_float - m) * 60.0
    hemi = ('E' if val >= 0 else 'W') if is_lon else ('N' if val >= 0 else 'S')
    return f"{d}°{m:02d}'{s:04.1f}\"{hemi}"

def add_north_arrow_tri(ax, size=0.09, pad=0.09, label='N'):
    x0 = pad; y0 = 1.0 - pad - size
    x1 = x0 + size * 0.72; y1 = y0 + size
    apex = ((x0 + x1)/2.0, y1)
    left_base, right_base = (x0, y0), (x1, y0)
    tail = ((x0 + x1)/2.0, y0 - size*0.09)
    ax.add_patch(Polygon([apex, left_base, tail], closed=True, transform=ax.transAxes,
                         facecolor='white', edgecolor='black', lw=1.0, zorder=10))
    ax.add_patch(Polygon([apex, tail, right_base], closed=True, transform=ax.transAxes,
                         facecolor='black', edgecolor='black', lw=1.0, zorder=10))
    ax.text(apex[0], y1 + size*0.12, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=9, fontweight='bold', color='black', zorder=11)

def meters_per_degree_lon(lat_deg):
    return 111320.0 * np.cos(np.radians(lat_deg))

def nice_len_125(target_m):
    target = max(float(target_m), 1e-6)
    exp = int(np.floor(np.log10(target))); base = 10**exp
    for m in (1, 2, 5, 10):
        if m*base >= target: return m*base
    return 10*base

def add_checker_scale_bar(ax, west, east, south, north, crs_is_geographic=False,
                          segments=4, fontsize=4):
    width_data  = east - west
    height_data = north - south
    if crs_is_geographic:
        mid_lat = (south + north) / 2.0
        mpd = meters_per_degree_lon(mid_lat)
        width_m = width_data * mpd
        units_per_meter_x = 1.0 / mpd
    else:
        width_m = width_data
        units_per_meter_x = 1.0
    total_len_m = nice_len_125(width_m / 5.0)
    seg_len_m   = total_len_m / float(segments)
    total_len_units = total_len_m * units_per_meter_x
    seg_len_units   = seg_len_m   * units_per_meter_x
    x_start = west + 0.5 * width_data - 0.5 * total_len_units
    y_base  = south + 0.06 * height_data
    bar_h   = 0.018 * height_data
    ax.add_patch(Rectangle((x_start - total_len_units*0.08, y_base - 0.035*height_data),
                           total_len_units*1.16, bar_h + 0.07*height_data,
                           transform=ax.transData, facecolor='white', edgecolor='none', alpha=0.7, zorder=3))
    for i in range(segments):
        xs = x_start + i * seg_len_units
        color = 'black' if i % 2 == 0 else 'white'
        ax.add_patch(Rectangle((xs, y_base), seg_len_units, bar_h,
                               transform=ax.transData, facecolor=color,
                               edgecolor='black', lw=0.8, zorder=4))
    ax.add_patch(Rectangle((x_start, y_base), total_len_units, bar_h,
                           transform=ax.transData, fill=False,
                           edgecolor='black', lw=1.0, zorder=5))
    ax.text(x_start + total_len_units/2.0, y_base + bar_h + 0.012*height_data,
            "km", ha='center', va='bottom', fontsize=fontsize, color='black', zorder=6)
    ax.text(x_start, y_base - 0.012*height_data, "0",
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)
    end_km = total_len_m / 1000.0
    end_label = f"{int(round(end_km))}" if abs(end_km - int(round(end_km))) < 1e-6 else f"{end_km:.1f}"
    ax.text(x_start + total_len_units, y_base - 0.012*height_data, end_label,
            ha='center', va='top', fontsize=fontsize, color='black', zorder=6)

# =========================================================
# Interpolation 
# =========================================================
def interpolate_grid(x, y, v, west, east, south, north, grid_res=200):
    gx = np.linspace(west,  east,  grid_res)
    gy = np.linspace(south, north, grid_res)
    Xg, Yg = np.meshgrid(gx, gy)
    pts = np.column_stack([x, y])

    if HAVE_SCIPY:
        grid_pts = np.column_stack([Xg.ravel(), Yg.ravel()])
        try:
            if USE_RBF:
                rbf = RBFInterpolator(pts, v, kernel='thin_plate_spline', smoothing=0.1, epsilon=0.1)
                Vg = rbf(grid_pts).reshape(grid_res, grid_res)
            else:
                Vg = griddata(pts, v, grid_pts, method='cubic', fill_value=np.nan).reshape(grid_res, grid_res)
            tree = cKDTree(pts)
            distances, _ = tree.query(grid_pts, k=1)
            Vg[distances.reshape(grid_res, grid_res) > ((east - west) * 0.01)] = np.nan
            return Xg, Yg, Vg
        except Exception as e:
            print(f"[WARN] SciPy interpolation failed ({e}); falling back to IDW...")

    if HAVE_SCIPY:
        tree = cKDTree(pts)
        grid_pts = np.column_stack([Xg.ravel(), Yg.ravel()])
        k = min(IDW_NEIGHBORS, len(pts))
        dist, idx = tree.query(grid_pts, k=k)
        w = 1.0 / (dist + 1e-12)
        vv = v[idx] if k > 1 else v[idx][:, None]
        V_flat = (w * vv).sum(axis=1) / w.sum(axis=1)
        Vg = V_flat.reshape(grid_res, grid_res)
        return Xg, Yg, Vg

    # simple IDW fallback
    Vg = np.zeros((grid_res, grid_res), dtype=float)
    for i in range(grid_res):
        Xi = Xg[i]; Yi = Yg[i]
        for j in range(grid_res):
            d = np.hypot(pts[:, 0] - Xi[j], pts[:, 1] - Yi[j])
            w = 1.0 / (d + 1e-12)
            Vg[i, j] = (w * v).sum() / w.sum()
    return Xg, Yg, Vg

# =========================================================
# Highway & Boundary Helpers 
# =========================================================
def load_roads(road_shp_path, target_crs):
    if not os.path.exists(road_shp_path):
        print(f"[WARN] Road shapefile not found: {road_shp_path}")
        return None
    target_crs_str = str(target_crs) if not isinstance(target_crs, str) else target_crs
    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_shp_path)
            if roads.crs is None:
                print("[WARN] Road CRS is None; assuming EPSG:4326")
                roads = roads.set_crs(epsg=4326)
            roads = roads.to_crs(target_crs_str)
            print(f"[OK] Roads (GeoPandas): {len(roads)} features; CRS={roads.crs}")
            return roads
        except Exception as e:
            print(f"[WARN] GeoPandas failed for roads: {e}")
    if HAVE_FIONA:
        try:
            geoms = []
            with fiona.open(road_shp_path, 'r') as shp:
                src_crs = shp.crs
                for feat in shp:
                    geom = feat['geometry']
                    if geom is None: continue
                    try:
                        geom_t = transform_geom(src_crs, target_crs_str, geom)
                    except Exception as e:
                        print(f"[WARN] Road transform failed; keeping original: {e}")
                        geom_t = geom
                    geoms.append(geom_t)
            print(f"[OK] Roads (Fiona): {len(geoms)} features")
            return {'geometries': geoms, 'method': 'fiona', 'crs': target_crs_str}
        except Exception as e:
            print(f"[WARN] Fiona failed for roads: {e}")
    print("[WARN] Unable to load roads")
    return None

def plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850):
    if roads is None: return
    try:
        if HAVE_GPD and hasattr(roads, 'plot'):
            try:
                roads.geometry = roads.geometry.apply(
                    lambda g: g if g is None or g.geom_type in ("LineString","MultiLineString") else None
                )
            except Exception:
                pass
            roads.plot(ax=ax, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (GeoPandas)")
        elif isinstance(roads, dict) and 'geometries' in roads:
            for geom in roads['geometries']:
                if geom is None: continue
                if geom['type'] == 'LineString':
                    xs = [p[0] for p in geom['coordinates']]
                    ys = [p[1] for p in geom['coordinates']]
                    ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
                elif geom['type'] == 'MultiLineString':
                    for line in geom['coordinates']:
                        xs = [p[0] for p in line]
                        ys = [p[1] for p in line]
                        ax.plot(xs, ys, color=line_color, linewidth=line_width, linestyle=line_style, alpha=alpha, zorder=zorder)
            print("[OK] Roads plotted (Fiona fallback)")
    except Exception as e:
        print(f"[WARN] Failed to plot roads: {e}")

def draw_boundary_from_mask(ax, mask_tif_path, raster_crs, zorder=999, color="black", linewidth=0.7):
    if not os.path.exists(mask_tif_path):
        return False
    try:
        with rasterio.open(mask_tif_path) as ms:
            if ms.crs != raster_crs:
                print("[INFO] Mask CRS differs from raster CRS; skip mask boundary")
                return False
            mask = ms.read(1)
            b = ms.bounds
            extent_mask = [b.left, b.right, b.bottom, b.top]
        ax.contour(mask, levels=[0.5], colors=color, linewidths=linewidth,
                   extent=extent_mask, origin='upper', zorder=zorder)
        print("[OK] Boundary drawn from mask")
        return True
    except Exception as ex:
        print(f"[WARN] Could not draw boundary from mask: {ex}")
        return False

def draw_boundary_from_shp(ax, shp_path, target_crs, zorder=998, color="black", linewidth=0.7):
    if not os.path.exists(shp_path):
        return False
    try:
        target_crs_str = str(target_crs) if not isinstance(target_crs, str) else target_crs
        if HAVE_GPD:
            gdf = gpd.read_file(shp_path).to_crs(target_crs_str)
            gdf.boundary.plot(ax=ax, color=color, linewidth=linewidth, alpha=1.0, zorder=zorder)
            print("[OK] Boundary drawn from shapefile (GeoPandas)")
            return True
        elif HAVE_FIONA:
            geoms = []
            with fiona.open(shp_path, 'r') as shp:
                src_crs = shp.crs
                for feat in shp:
                    geom = feat['geometry']
                    if geom is None: continue
                    geoms.append(transform_geom(src_crs, target_crs_str, geom))
            for geom in geoms:
                if geom['type'] == 'Polygon':
                    xs = [p[0] for p in geom['coordinates'][0]]
                    ys = [p[1] for p in geom['coordinates'][0]]
                    ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
                elif geom['type'] == 'MultiPolygon':
                    for poly in geom['coordinates']:
                        xs = [p[0] for p in poly[0]]
                        ys = [p[1] for p in poly[0]]
                        ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=zorder)
            print("[OK] Boundary drawn from shapefile (Fiona)")
            return True
    except Exception as ex:
        print(f"[WARN] Boundary-from-shp failed: {ex}")
    return False

# =========================================================
# One-variable Z map (map + histogram)
# =========================================================
def plot_zmap_variable(tif_path, var_name):
    if not os.path.exists(tif_path):
        print(f"[WARN] Missing z-map file: {tif_path}")
        return

    print("="*60)
    print(f"{var_name.upper()} Z-MAP")
    print("="*60)

    with rasterio.open(tif_path) as ds:
        arr = ds.read(1).astype(np.float32)
        nodata = ds.nodata
        raster_crs = ds.crs
        transform = ds.transform
        b = ds.bounds
        extent = (b.left, b.right, b.bottom, b.top)
        crs_is_geographic = bool(raster_crs and raster_crs.is_geographic)

    # Clean
    arr_clean = arr.copy()
    if nodata is not None:
        arr_clean[arr == nodata] = np.nan
    arr_clean[~np.isfinite(arr_clean)] = np.nan

    rows, cols = np.where(np.isfinite(arr_clean))
    if rows.size == 0:
        print(f"[WARN] No valid pixels for {var_name}")
        return

    # Pixel center coordinates from affine 
    xs = transform.c + (cols + 0.5) * transform.a + (rows + 0.5) * transform.b
    ys = transform.f + (cols + 0.5) * transform.d + (rows + 0.5) * transform.e
    x, y, v = np.asarray(xs), np.asarray(ys), arr_clean[rows, cols]

    # Boundary erosion to clean edge/outsider points 
    if os.path.exists(STUDY_MASK):
        try:
            with rasterio.open(STUDY_MASK) as ms:
                mask_data = ms.read(1)
                mask_crs = ms.crs
                mask_transform = ms.transform
                if HAVE_SCIPY:
                    mask_eroded = binary_erosion(mask_data > 0, iterations=MASK_EROSION_ITERATIONS).astype(np.uint8)
                else:
                    mask_eroded = (mask_data > 0).astype(np.uint8)

                # Transform sample points to mask CRS if needed
                if mask_crs != raster_crs:
                    mx, my = rio_transform(raster_crs, mask_crs, x, y)
                else:
                    mx, my = x, y

                keep = []
                for xi, yi in zip(mx, my):
                    r, c = rowcol(mask_transform, xi, yi)
                    inside = (0 <= r < mask_eroded.shape[0]) and (0 <= c < mask_eroded.shape[1]) and (mask_eroded[int(r), int(c)] > 0)
                    keep.append(inside)
                keep = np.array(keep, dtype=bool)
                x, y, v = x[keep], y[keep], v[keep]
                print(f"[OK] Removed {np.sum(~keep)} edge/outsider points")
        except Exception as ex:
            print(f"[WARN] Boundary erosion failed: {ex}")
            traceback.print_exc()

    # Color scaling 
    vmin, vmax = DISPLAY_RANGES[var_name]
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    print(f"[INFO] Display range: [{vmin:.2f}, {vmax:.2f}] z-score")

    # Small buffered extent
    west, east = extent[0], extent[1]
    south, north = extent[2], extent[3]
    bx = (east - west) * 0.02 or 1.0
    by = (north - south) * 0.02 or 1.0
    west, east = west - bx, east + bx
    south, north = south - by, north + by
    extent_buf = [west, east, south, north]

    # Background interpolation
    print("[INFO] Interpolating background grid…")
    Xg, Yg, Vgrid = interpolate_grid(x, y, v, west, east, south, north, grid_res=GRID_RES)

    # Mask grid to study area with CRS safety
    if os.path.exists(STUDY_MASK):
        try:
            with rasterio.open(STUDY_MASK) as ms:
                mask_data = ms.read(1)
                mask_crs = ms.crs
                mask_transform = ms.transform
                if HAVE_SCIPY:
                    mask_eroded = binary_erosion(mask_data > 0, iterations=MASK_EROSION_ITERATIONS).astype(np.uint8)
                else:
                    mask_eroded = (mask_data > 0).astype(np.uint8)
                H, W = Vgrid.shape
                for i in range(H):
                    for j in range(W):
                        gx, gy = Xg[i, j], Yg[i, j]
                        if mask_crs != raster_crs:
                            mx, my = rio_transform(raster_crs, mask_crs, [gx], [gy])
                            mx, my = mx[0], my[0]
                        else:
                            mx, my = gx, gy
                        r, c = rowcol(mask_transform, mx, my)
                        keep = (0 <= r < mask_eroded.shape[0]) and (0 <= c < mask_eroded.shape[1]) and (mask_eroded[int(r), int(c)] > 0)
                        if not keep:
                            Vgrid[i, j] = np.nan
            print("[OK] Grid masked to boundary")
        except Exception as ex:
            print(f"[WARN] Grid boundary masking failed: {ex}")

    # DMS tick formatters 
    xmid = (extent[0] + extent[1]) / 2.0
    ymid = (extent[2] + extent[3]) / 2.0
    if crs_is_geographic:
        def fmt_lon(xv, pos): return to_dms(xv, True)
        def fmt_lat(yv, pos): return to_dms(yv, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError("Raster is projected; DMS tick labels require 'pyproj'.")
        to_ll = Transformer.from_crs(raster_crs, "EPSG:4326", always_xy=True)
        def fmt_lon(xv, pos):
            lon, lat = to_ll.transform(xv, ymid)
            return to_dms(lon, True)
        def fmt_lat(yv, pos):
            lon, lat = to_ll.transform(xmid, yv)
            return to_dms(lat, False)

    # Overlays
    roads = load_roads(ROAD_SHP, raster_crs) if os.path.exists(ROAD_SHP) else None

    # ================= MAP =================
    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE, facecolor='white')
    ax.set_facecolor('white')
    im = ax.imshow(Vgrid, cmap=ZSCORE_CMAP, extent=extent_buf, norm=norm,
                   origin='lower', interpolation='bilinear', alpha=1.0, zorder=1)
    ax.scatter(x, y, c=v, s=0.6, cmap=ZSCORE_CMAP, norm=norm,
               edgecolors='none', alpha=0.9, zorder=2)

    # Overlays: Highway + Boundary
    plot_roads_on_map(ax, roads, line_color='red', line_width=0.7, line_style='-', alpha=0.9, zorder=850)
    drew = draw_boundary_from_mask(ax, STUDY_MASK, raster_crs, zorder=999, color="black", linewidth=0.7)
    if not drew:
        draw_boundary_from_shp(ax, STUDY_SHP, raster_crs, zorder=998, color="black", linewidth=0.7)

    # Axis labels + DMS ticks 
    if crs_is_geographic:
        ax.set_xlabel("Longitude", fontsize=5, fontweight='bold')
        ax.set_ylabel("Latitude",  fontsize=5, fontweight='bold')
        ax.xaxis.set_major_locator(MultipleLocator(0.1))
        # ax.yaxis.set_major_locator(MultipleLocator(0.1))
    else:
        ax.set_xlabel("Longitude", fontsize=5, fontweight='bold')
        ax.set_ylabel("Latitude", fontsize=5, fontweight='bold')

    ax.xaxis.set_major_formatter(FuncFormatter(fmt_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_lat))
    ax.tick_params(axis='x', which='major', labelsize=3)
    ax.tick_params(axis='y', which='major', labelsize=3)
    for t in ax.get_xticklabels():
        t.set_rotation(45); t.set_ha('right'); t.set_fontsize(3)

    # Decorations
    add_north_arrow_tri(ax, size=0.09, pad=0.09)
    add_checker_scale_bar(ax, *extent_buf, crs_is_geographic=crs_is_geographic, segments=4, fontsize=4)

    # Tiny legend (consistent placement & style)
    from matplotlib.lines import Line2D
    legend_elems = [
        Line2D([0], [0], color='red',   lw=1.0, label='Highway'),
        Line2D([0], [0], color='black', lw=1.0, label='Study Area')
    ]
    legend_box = ax.legend(
        handles=legend_elems,
        loc='center',
        bbox_to_anchor=(0.82, 0.15),
        bbox_transform=ax.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=3,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0
    )
    legend_box.get_frame().set_edgecolor('lightgray')
    legend_box.get_frame().set_linewidth(0.3)

    # Side colorbar
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=CAX_SIZE, pad=CAX_PAD)
    cb = plt.colorbar(im, cax=cax)
    cb.ax.tick_params(labelsize=3)
    cb.set_label(VAR_INFO[var_name]["units"], fontsize=4, fontweight='bold')
    cb.set_ticks(np.linspace(vmin, vmax, 5))

    # Save
    map_path = os.path.join(ZMAP_DIR, f"{var_name.upper()} Zmap.png")
    plt.tight_layout()
    plt.savefig(map_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"[OK] Saved map → {map_path}")
    plt.show()

    # ================= HISTOGRAM =================
    vals_hist = np.clip(v, vmin, vmax)
    fig2, ax2 = plt.subplots(1, 1, figsize=HIST_FIGSIZE, facecolor='white')
    bins = np.linspace(vmin, vmax, 21)
    counts, edges = np.histogram(vals_hist, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2.0

    # Blue for <0, Red for >0 (visual echo of diverging cmap)
    bar_colors = ['#4A90E2' if c < 0 else '#E24A4A' if c > 0 else '#888888' for c in centers]
    ax2.bar(centers, counts, width=(bins[1]-bins[0])*0.9,
            color=bar_colors, edgecolor="black", linewidth=0.4, alpha=0.9)
    mean_v = float(np.mean(v)); med_v  = float(np.median(v))
    ax2.axvline(0,      color='k',       lw=0.8, label='Zero')
    ax2.axvline(mean_v, color='darkred', ls='--', lw=0.7, label=f"Mean: {mean_v:.2f}")
    ax2.axvline(med_v,  color='orange',  ls='--', lw=0.7, label=f"Median: {med_v:.2f}")
    ax2.legend(fontsize=4, loc='upper right', framealpha=0.9)
    ax2.set_xlabel("Z-score", fontsize=4, fontweight='bold')
    ax2.set_ylabel("Number of Pixels", fontsize=4, fontweight='bold')
    ax2.tick_params(axis='both', which='major', labelsize=4)
    ax2.set_xlim(vmin, vmax)
    ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.4, axis='y')
    ax2.set_axisbelow(True)

    out_hist = os.path.join(ZMAP_DIR, f"{var_name.upper()}_Zmap Histogram.png")
    plt.tight_layout(pad=1.2)
    plt.savefig(out_hist, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"[OK] Saved histogram → {out_hist}")
    plt.show()

# =========================================================
# Main driver
# =========================================================
def create_zmaps_visualization():
    print("\n" + "="*60)
    print("Z-MAPS VISUALIZATION")
    print("="*60 + "\n")

    zmap_files = {
        "DVplus": Z_DVPLUS_DISPLAY,
        "TWI":    Z_TWI_DISPLAY,
        "SM":     Z_SM_DISPLAY,
    }

    for var_name, tif_path in zmap_files.items():
        try:
            plot_zmap_variable(tif_path, var_name)
        except Exception as e:
            print(f"[ERROR] Failed to process {var_name}: {e}")
            traceback.print_exc()

    print("\n" + "="*60)
    print("Z-MAPS VISUALIZATION COMPLETE")
    print("="*60)
    print("\nGenerated files:")
    for var_name in zmap_files.keys():
        print(f"  - {var_name.upper()} Zmap.png")
        print(f"  - {var_name.upper()}_Zmap Histogram.png")

# ================= RUN =================
if __name__ == "__main__":
    create_zmaps_visualization()


# Phase-5: Physics-guided indices

# Step-5.1: Geometric Observability

In [ ]:
# ============================================================================
# Step 5.1 — Pure Geometric Observability (G)
#
# Scientific definition:
#     signed_projection = l_hat · s_hat
#     G = |l_hat · s_hat|
#
# Purpose:
#   1. Calculate ONLY the terrain-satellite geometric observability term.
#   2. Keep G independent of coherence, PS density, and the dynamic mask.
#   3. Preserve NoData where the downslope or LOS direction is undefined.
#   4. Produce a resolution-preserving map and histogram directly from G.
#
# Required preceding steps:
#   Step 3.3 — Downslope unit vector (s_hat)
#   Step 3.4 — LOS unit vector (l_hat)
#
# This script intentionally does NOT calculate composite observability or IRS.
# IRS must be calculated separately in Step 5.2 using coherence, PS density,
# and this pure geometric term G.
# ============================================================================

from __future__ import annotations

import json
import math
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

# Matplotlib is used only for direct raster visualization. No interpolation,
# RBF, cubic surface fitting, IDW, or boundary erosion is applied.
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.ticker import FuncFormatter, MaxNLocator


# =============================================================================
# USER CONFIGURATION
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

# Anchor and study mask
ANCHOR_TIF = BASE / "Results" / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
STUDY_MASK_TIF = BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"

# Step 3.3: downslope unit vector, ENU components
S_DIR = BASE / "Results" / "Step-3.3-Downslope unit vector (ŝ)"
S_E_TIF = S_DIR / "s_hat_east_32645_12p5m_float32.tif"
S_N_TIF = S_DIR / "s_hat_north_32645_12p5m_float32.tif"
S_U_TIF = S_DIR / "s_hat_up_32645_12p5m_float32.tif"

# Step 3.4: LOS unit vector, ENU components
L_DIR = BASE / "Results" / "Step-3.4-LOS unit (l̂)"
L_E_TIF = L_DIR / "l_hat_east_32645_12p5m_float32.tif"
L_N_TIF = L_DIR / "l_hat_north_32645_12p5m_float32.tif"
L_U_TIF = L_DIR / "l_hat_up_32645_12p5m_float32.tif"
L_META_JSON = L_DIR / "los_geometry_metadata.json"

# Optional cartographic overlay
ROAD_SHP = BASE / "NTD_Road.shp"

# Outputs
OUT_DIR = BASE / "Results" / "Step-5.1-Observability"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SIGNED_DOT_TIF = OUT_DIR / "signed_los_downslope_projection_32645_12p5m_float32.tif"
G_TIF = OUT_DIR / "geometric_observability_G_0_1_32645_12p5m_float32.tif"
G_VALID_MASK_TIF = OUT_DIR / "geometric_observability_valid_mask_32645_12p5m_byte.tif"
G_LOW_FLAG_TIF = OUT_DIR / "geometric_observability_low_flag_32645_12p5m_byte.tif"

# Compatibility alias for the existing Step 5.2 path. This is an exact copy of
# G_TIF, not a composite observability product.
LEGACY_G_TIF = OUT_DIR / "obs_term_sensitivity_0_1_32645_12p5m_float32.tif"

META_JSON = OUT_DIR / "geometric_observability_metadata.json"
SUMMARY_JSON = OUT_DIR / "geometric_observability_summary.json"
MAP_PNG = OUT_DIR / "Geometric_Observability_G_Map.png"
MAP_PDF = OUT_DIR / "Geometric_Observability_G_Map.pdf"
HIST_PNG = OUT_DIR / "Geometric_Observability_G_Histogram.png"
HIST_PDF = OUT_DIR / "Geometric_Observability_G_Histogram.pdf"

# Existing composite outputs from the earlier Step 5.1 implementation are not
# used by this corrected script. They are only reported as stale files if found.
STALE_COMPOSITE_FILES = [
    OUT_DIR / "observability_0_1_32645_12p5m_float32.tif",
    OUT_DIR / "observability_soft_0_1_32645_12p5m_float32.tif",
    OUT_DIR / "obs_term_coherence_0_1_32645_12p5m_float32.tif",
    OUT_DIR / "obs_term_pscount_0_1_32645_12p5m_float32.tif",
]

# Scientific and QA settings
NODATA_FLOAT = -9999.0
NODATA_BYTE = 255
VECTOR_NORM_MIN = 1.0e-8
VECTOR_NORM_WARN_TOL = 1.0e-3
DOT_CLIP_TOL = 1.0e-6

# This threshold is diagnostic only. It matches the 0.05 denominator floor used
# in the existing downslope-velocity inversion, corresponding to a maximum
# geometric amplification of 20. It does not modify G itself.
LOW_G_THRESHOLD = 0.05

# Descriptive, not calibrated, summary intervals
DESCRIPTIVE_LOW_MAX = 0.30
DESCRIPTIVE_MODERATE_MAX = 0.70

# Figure settings
MAKE_FIGURES = True
SHOW_FIGURES = True
MAP_FIGSIZE = (6.2, 9.0)
HIST_FIGSIZE = (6.2, 4.2)
MAP_DPI = 300


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


# =============================================================================
# RASTER I/O
# =============================================================================
def read_anchor(path: Path) -> dict[str, Any]:
    require_file(path, "anchor raster")
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"Anchor raster has no CRS: {path}")
        return {
            "transform": ds.transform,
            "crs": ds.crs,
            "width": ds.width,
            "height": ds.height,
            "profile": ds.profile.copy(),
            "bounds": ds.bounds,
            "epsg": ds.crs.to_epsg(),
            "res_x": abs(ds.transform.a),
            "res_y": abs(ds.transform.e),
        }


def same_grid(ds: rasterio.io.DatasetReader, anchor: dict[str, Any]) -> bool:
    return (
        ds.crs == anchor["crs"]
        and ds.transform == anchor["transform"]
        and ds.width == anchor["width"]
        and ds.height == anchor["height"]
    )


def scrub_common_nodata(arr: np.ndarray, tagged_nodata: float | int | None) -> np.ndarray:
    out = arr.astype("float64", copy=False)
    if tagged_nodata is not None:
        try:
            nd = float(tagged_nodata)
            if np.isfinite(nd):
                out = np.where(out == nd, np.nan, out)
        except (TypeError, ValueError):
            pass

    # Defensive handling for common geospatial sentinels when metadata are absent
    out = np.where(out == -9999.0, np.nan, out)
    out = np.where(out == -32768.0, np.nan, out)
    out = np.where(np.abs(out) > 1.0e20, np.nan, out)
    return out


def read_continuous_to_anchor(
    path: Path,
    anchor: dict[str, Any],
    description: str,
    resampling: Resampling = Resampling.bilinear,
) -> np.ndarray:
    """
    Read a continuous raster and align it to the anchor grid.

    Important: the source is reprojected before converting tagged NoData to NaN,
    so src_nodata remains meaningful to rasterio during reprojection.
    """
    require_file(path, description)

    with rasterio.open(path) as src:
        raw = src.read(1).astype("float64")
        src_nodata = src.nodata

        if src.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        if same_grid(src, anchor):
            return scrub_common_nodata(raw, src_nodata)

        destination = np.full(
            (anchor["height"], anchor["width"]),
            np.nan,
            dtype="float64",
        )

        reproject(
            source=raw,
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )

        return scrub_common_nodata(destination, None)


def read_mask_to_anchor(path: Path, anchor: dict[str, Any]) -> np.ndarray:
    require_file(path, "study mask")

    with rasterio.open(path) as src:
        raw = src.read(1)
        src_nodata = src.nodata

        if same_grid(src, anchor):
            arr = raw.astype("float64")
            if src_nodata is not None:
                arr = np.where(arr == src_nodata, 0.0, arr)
            return np.isfinite(arr) & (arr > 0.5)

        destination = np.zeros(
            (anchor["height"], anchor["width"]),
            dtype="float64",
        )
        reproject(
            source=raw,
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=0,
            resampling=Resampling.nearest,
        )
        return np.isfinite(destination) & (destination > 0.5)


def write_float_tif(path: Path, arr: np.ndarray, anchor: dict[str, Any]) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=NODATA_FLOAT,
        compress="LZW",
        tiled=True,
        bigtiff="IF_SAFER",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    tmp = Path(str(path) + ".part")
    output = np.where(np.isfinite(arr), arr, NODATA_FLOAT).astype("float32")
    with rasterio.open(tmp, "w", **profile) as dst:
        dst.write(output, 1)
    os.replace(tmp, path)

    try:
        with rasterio.open(path, "r+") as dst:
            dst.build_overviews([2, 4, 8, 16], Resampling.average)
            dst.update_tags(ns="rio_overview", resampling="average")
    except Exception as exc:
        log(f"[WARN] Could not build overviews for {path.name}: {exc}")


def write_byte_tif(
    path: Path,
    arr: np.ndarray,
    anchor: dict[str, Any],
    nodata: int = NODATA_BYTE,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="uint8",
        count=1,
        nodata=nodata,
        compress="LZW",
        tiled=True,
        bigtiff="IF_SAFER",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    tmp = Path(str(path) + ".part")
    output = arr.astype("uint8")
    with rasterio.open(tmp, "w", **profile) as dst:
        dst.write(output, 1)
    os.replace(tmp, path)

    try:
        with rasterio.open(path, "r+") as dst:
            dst.build_overviews([2, 4, 8, 16], Resampling.nearest)
            dst.update_tags(ns="rio_overview", resampling="nearest")
    except Exception as exc:
        log(f"[WARN] Could not build overviews for {path.name}: {exc}")


# =============================================================================
# VECTOR AND STATISTICAL HELPERS
# =============================================================================
def normalize_vector_triplet(
    east: np.ndarray,
    north: np.ndarray,
    up: np.ndarray,
    vector_name: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict[str, float | int]]:
    """Normalize an ENU vector triplet and return QA statistics."""
    finite_components = np.isfinite(east) & np.isfinite(north) & np.isfinite(up)
    magnitude = np.sqrt(east * east + north * north + up * up)
    valid = finite_components & np.isfinite(magnitude) & (magnitude > VECTOR_NORM_MIN)

    before = magnitude[valid]
    qa = {
        "valid_count": int(valid.sum()),
        "mean_norm_before": float(np.mean(before)) if before.size else float("nan"),
        "std_norm_before": float(np.std(before)) if before.size else float("nan"),
        "max_abs_deviation_before": (
            float(np.max(np.abs(before - 1.0))) if before.size else float("nan")
        ),
    }

    east_n = np.full_like(east, np.nan, dtype="float64")
    north_n = np.full_like(north, np.nan, dtype="float64")
    up_n = np.full_like(up, np.nan, dtype="float64")

    east_n[valid] = east[valid] / magnitude[valid]
    north_n[valid] = north[valid] / magnitude[valid]
    up_n[valid] = up[valid] / magnitude[valid]

    magnitude_after = np.sqrt(east_n * east_n + north_n * north_n + up_n * up_n)
    after = magnitude_after[valid]
    qa.update(
        {
            "mean_norm_after": float(np.mean(after)) if after.size else float("nan"),
            "std_norm_after": float(np.std(after)) if after.size else float("nan"),
            "max_abs_deviation_after": (
                float(np.max(np.abs(after - 1.0))) if after.size else float("nan")
            ),
        }
    )

    if after.size == 0:
        raise RuntimeError(f"No valid pixels remain after normalizing {vector_name}.")

    if qa["max_abs_deviation_after"] > VECTOR_NORM_WARN_TOL:
        log(
            f"[WARN] {vector_name} norm QA exceeds tolerance: "
            f"{qa['max_abs_deviation_after']:.3e}"
        )

    return east_n, north_n, up_n, qa


def detailed_stats(arr: np.ndarray, mask: np.ndarray) -> dict[str, float | int]:
    values = arr[mask & np.isfinite(arr)]
    if values.size == 0:
        return {"count": 0}

    rounded_unique = np.unique(np.round(values, 6)).size
    return {
        "count": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "near_zero_lt_0_01_fraction": float(np.mean(values < 0.01)),
        "low_geometry_lt_threshold_fraction": float(np.mean(values < LOW_G_THRESHOLD)),
        "unique_values_rounded_6dp": int(rounded_unique),
    }


def descriptive_class_counts(values: np.ndarray) -> dict[str, dict[str, float | int]]:
    n = int(values.size)
    if n == 0:
        return {}

    low = (values >= 0.0) & (values < DESCRIPTIVE_LOW_MAX)
    moderate = (values >= DESCRIPTIVE_LOW_MAX) & (values < DESCRIPTIVE_MODERATE_MAX)
    high = (values >= DESCRIPTIVE_MODERATE_MAX) & (values <= 1.0)

    def item(mask: np.ndarray) -> dict[str, float | int]:
        count = int(mask.sum())
        return {"count": count, "percentage": 100.0 * count / n}

    return {
        "low_0_to_0.3": item(low),
        "moderate_0.3_to_0.7": item(moderate),
        "high_0.7_to_1.0": item(high),
    }


# =============================================================================
# OPTIONAL CARTOGRAPHIC HELPERS
# =============================================================================
def try_load_roads(target_crs: Any):
    if not ROAD_SHP.exists():
        log(f"[INFO] Road overlay skipped; file not found: {ROAD_SHP}")
        return None

    try:
        import geopandas as gpd

        roads = gpd.read_file(ROAD_SHP)
        if roads.crs is None:
            raise ValueError("Road shapefile has no CRS.")
        return roads.to_crs(target_crs)
    except Exception as exc:
        log(f"[WARN] Road overlay could not be loaded: {exc}")
        return None


def add_scale_bar_utm(ax, bounds, target_fraction: float = 0.20) -> None:
    """Add a simple 1-2-5 scale bar for projected metre coordinates."""
    west, south, east, north = bounds.left, bounds.bottom, bounds.right, bounds.top
    width = east - west
    target = max(width * target_fraction, 1.0)

    exponent = math.floor(math.log10(target))
    base = 10.0 ** exponent
    candidates = [1.0 * base, 2.0 * base, 5.0 * base, 10.0 * base]
    length_m = min(candidates, key=lambda x: abs(x - target))

    x0 = west + 0.07 * width
    y0 = south + 0.06 * (north - south)
    segment = length_m / 4.0
    height = 0.012 * (north - south)

    from matplotlib.patches import Rectangle

    for idx in range(4):
        face = "black" if idx % 2 == 0 else "white"
        ax.add_patch(
            Rectangle(
                (x0 + idx * segment, y0),
                segment,
                height,
                facecolor=face,
                edgecolor="black",
                linewidth=0.5,
                zorder=8,
            )
        )

    label = f"{length_m / 1000.0:g} km"
    ax.text(
        x0 + length_m / 2.0,
        y0 + 1.6 * height,
        label,
        ha="center",
        va="bottom",
        fontsize=8,
        zorder=9,
    )


def add_north_arrow(ax) -> None:
    ax.annotate(
        "N",
        xy=(0.08, 0.95),
        xytext=(0.08, 0.86),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        arrowprops=dict(facecolor="black", edgecolor="black", width=2.0, headwidth=8.0),
        zorder=10,
    )


def install_dms_formatters(ax, crs, bounds) -> bool:
    """Use DMS labels when pyproj is available; otherwise retain UTM labels."""
    try:
        from pyproj import Transformer
    except Exception:
        log("[INFO] pyproj unavailable; map axes remain in projected coordinates.")
        return False

    transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
    x_mid = (bounds.left + bounds.right) / 2.0
    y_mid = (bounds.bottom + bounds.top) / 2.0

    def dms(value: float, longitude: bool) -> str:
        absolute = abs(value)
        degrees = int(absolute)
        minutes_float = (absolute - degrees) * 60.0
        minutes = int(minutes_float)
        seconds = (minutes_float - minutes) * 60.0
        if longitude:
            hemi = "E" if value >= 0 else "W"
        else:
            hemi = "N" if value >= 0 else "S"
        return f"{degrees}°{minutes:02d}'{seconds:04.1f}\"{hemi}"

    def x_formatter(x: float, _position: int) -> str:
        lon, _lat = transformer.transform(x, y_mid)
        return dms(lon, True)

    def y_formatter(y: float, _position: int) -> str:
        _lon, lat = transformer.transform(x_mid, y)
        return dms(lat, False)

    ax.xaxis.set_major_formatter(FuncFormatter(x_formatter))
    ax.yaxis.set_major_formatter(FuncFormatter(y_formatter))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=7))
    return True


# =============================================================================
# FIGURES: DIRECT RASTER DISPLAY, NO SURFACE INTERPOLATION
# =============================================================================
def create_figures(
    g: np.ndarray,
    valid_geometry: np.ndarray,
    study_mask: np.ndarray,
    anchor: dict[str, Any],
    stats: dict[str, float | int],
) -> None:
    if not MAKE_FIGURES:
        return

    bounds = anchor["bounds"]
    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    g_display = np.ma.masked_invalid(np.where(valid_geometry, g, np.nan))

    # ----------------------------- Map -----------------------------
    fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
    image = ax.imshow(
        g_display,
        extent=extent,
        origin="upper",
        interpolation="nearest",
        cmap="viridis",
        norm=Normalize(vmin=0.0, vmax=1.0),
        zorder=1,
    )

    # Study-area boundary from the original, non-eroded mask
    ax.contour(
        study_mask.astype("uint8"),
        levels=[0.5],
        extent=extent,
        origin="upper",
        linewidths=0.8,
        colors="black",
        zorder=6,
    )

    roads = try_load_roads(anchor["crs"])
    if roads is not None and not roads.empty:
        roads.plot(ax=ax, color="red", linewidth=0.8, zorder=7)

    ax.set_xlim(bounds.left, bounds.right)
    ax.set_ylim(bounds.bottom, bounds.top)
    ax.set_title(r"Geometric observability $G=|\hat{l}\cdot\hat{s}|$", fontsize=11)

    dms_enabled = install_dms_formatters(ax, anchor["crs"], bounds)
    ax.set_xlabel("Longitude" if dms_enabled else "Easting (m)", fontsize=9)
    ax.set_ylabel("Latitude" if dms_enabled else "Northing (m)", fontsize=9)
    ax.tick_params(labelsize=7)
    if dms_enabled:
        plt.setp(ax.get_xticklabels(), rotation=35, ha="right")

    add_north_arrow(ax)
    add_scale_bar_utm(ax, bounds)

    colorbar = fig.colorbar(image, ax=ax, fraction=0.035, pad=0.02)
    colorbar.set_label("Geometric observability G (0–1)", fontsize=9)
    colorbar.ax.tick_params(labelsize=8)

    coverage = 100.0 * int(valid_geometry.sum()) / max(int(study_mask.sum()), 1)
    annotation = (
        f"Valid geometry coverage: {coverage:.1f}%\n"
        f"Mean G: {stats.get('mean', float('nan')):.3f}\n"
        f"Median G: {stats.get('median', float('nan')):.3f}\n"
        f"G < {LOW_G_THRESHOLD:.2f}: "
        f"{100.0 * stats.get('low_geometry_lt_threshold_fraction', 0.0):.1f}%"
    )
    ax.text(
        0.985,
        0.015,
        annotation,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=7.5,
        bbox=dict(facecolor="white", edgecolor="0.6", alpha=0.88, boxstyle="round,pad=0.3"),
        zorder=12,
    )

    fig.tight_layout()
    fig.savefig(MAP_PNG, dpi=MAP_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(MAP_PDF, dpi=MAP_DPI, bbox_inches="tight", facecolor="white")
    log(f"[FIGURE] {MAP_PNG}")
    log(f"[FIGURE] {MAP_PDF}")

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)

    # --------------------------- Histogram --------------------------
    values = g[valid_geometry & np.isfinite(g)]
    fig_h, ax_h = plt.subplots(figsize=HIST_FIGSIZE)
    bins = np.linspace(0.0, 1.0, 31)
    ax_h.hist(values, bins=bins, edgecolor="black", linewidth=0.5)
    ax_h.axvline(float(np.mean(values)), linestyle="--", linewidth=1.0, label=f"Mean = {np.mean(values):.3f}")
    ax_h.axvline(float(np.median(values)), linestyle=":", linewidth=1.2, label=f"Median = {np.median(values):.3f}")
    ax_h.axvline(LOW_G_THRESHOLD, linestyle="-.", linewidth=1.0, label=f"Low-geometry threshold = {LOW_G_THRESHOLD:.2f}")
    ax_h.set_xlim(0.0, 1.0)
    ax_h.set_xlabel("Geometric observability G", fontsize=9)
    ax_h.set_ylabel("Number of study-area pixels", fontsize=9)
    ax_h.set_title("Distribution of pure geometric observability", fontsize=11)
    ax_h.grid(axis="y", alpha=0.25)
    ax_h.legend(fontsize=8)
    ax_h.tick_params(labelsize=8)
    fig_h.tight_layout()
    fig_h.savefig(HIST_PNG, dpi=MAP_DPI, bbox_inches="tight", facecolor="white")
    fig_h.savefig(HIST_PDF, dpi=MAP_DPI, bbox_inches="tight", facecolor="white")
    log(f"[FIGURE] {HIST_PNG}")
    log(f"[FIGURE] {HIST_PDF}")

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig_h)


# =============================================================================
# MAIN WORKFLOW
# =============================================================================
def main() -> None:
    log("=== Step 5.1: Pure Geometric Observability G ===")

    required_inputs = [
        (ANCHOR_TIF, "anchor raster"),
        (STUDY_MASK_TIF, "study mask"),
        (S_E_TIF, "s_hat east component"),
        (S_N_TIF, "s_hat north component"),
        (S_U_TIF, "s_hat up component"),
        (L_E_TIF, "l_hat east component"),
        (L_N_TIF, "l_hat north component"),
        (L_U_TIF, "l_hat up component"),
    ]
    for path, description in required_inputs:
        require_file(path, description)

    anchor = read_anchor(ANCHOR_TIF)
    study_mask = read_mask_to_anchor(STUDY_MASK_TIF, anchor)

    study_count = int(study_mask.sum())
    if study_count == 0:
        raise RuntimeError("The study mask contains no valid study-area pixels.")

    log(
        f"[ANCHOR] EPSG:{anchor['epsg']} | "
        f"{anchor['width']} x {anchor['height']} | "
        f"resolution {anchor['res_x']:.3f} x {anchor['res_y']:.3f} m"
    )
    log(f"[MASK] Study-area pixels: {study_count:,}")

    # Vector components are continuous quantities. Bilinear component
    # interpolation followed by vector renormalization preserves direction more
    # faithfully than nearest-neighbour when reprojection is actually required.
    s_e = read_continuous_to_anchor(S_E_TIF, anchor, "s_hat east", Resampling.bilinear)
    s_n = read_continuous_to_anchor(S_N_TIF, anchor, "s_hat north", Resampling.bilinear)
    s_u = read_continuous_to_anchor(S_U_TIF, anchor, "s_hat up", Resampling.bilinear)

    l_e = read_continuous_to_anchor(L_E_TIF, anchor, "l_hat east", Resampling.bilinear)
    l_n = read_continuous_to_anchor(L_N_TIF, anchor, "l_hat north", Resampling.bilinear)
    l_u = read_continuous_to_anchor(L_U_TIF, anchor, "l_hat up", Resampling.bilinear)

    s_e, s_n, s_u, s_qa = normalize_vector_triplet(s_e, s_n, s_u, "s_hat")
    l_e, l_n, l_u, l_qa = normalize_vector_triplet(l_e, l_n, l_u, "l_hat")

    log(
        f"[QA] s_hat norm after renormalization: "
        f"mean={s_qa['mean_norm_after']:.8f}, "
        f"max deviation={s_qa['max_abs_deviation_after']:.3e}"
    )
    log(
        f"[QA] l_hat norm after renormalization: "
        f"mean={l_qa['mean_norm_after']:.8f}, "
        f"max deviation={l_qa['max_abs_deviation_after']:.3e}"
    )

    vector_valid = (
        study_mask
        & np.isfinite(s_e)
        & np.isfinite(s_n)
        & np.isfinite(s_u)
        & np.isfinite(l_e)
        & np.isfinite(l_n)
        & np.isfinite(l_u)
    )

    valid_count = int(vector_valid.sum())
    if valid_count == 0:
        raise RuntimeError(
            "No overlapping valid s_hat and l_hat pixels occur inside the study area."
        )

    # Signed projection coefficient. Its sign depends on whether the LOS vector
    # is defined ground-to-satellite or satellite-to-ground. G is invariant to
    # that sign convention because it uses the absolute value.
    signed_dot = np.full((anchor["height"], anchor["width"]), np.nan, dtype="float64")
    signed_values = (
        l_e[vector_valid] * s_e[vector_valid]
        + l_n[vector_valid] * s_n[vector_valid]
        + l_u[vector_valid] * s_u[vector_valid]
    )

    raw_outside = np.sum((signed_values < -1.0 - DOT_CLIP_TOL) | (signed_values > 1.0 + DOT_CLIP_TOL))
    if raw_outside > 0:
        raise RuntimeError(
            f"Dot product exceeded the physically valid [-1, 1] interval at "
            f"{int(raw_outside):,} pixels. Check vector inputs and CRS alignment."
        )

    signed_values = np.clip(signed_values, -1.0, 1.0)
    signed_dot[vector_valid] = signed_values

    g = np.full_like(signed_dot, np.nan, dtype="float64")
    g[vector_valid] = np.abs(signed_dot[vector_valid])
    g = np.where(vector_valid, np.clip(g, 0.0, 1.0), np.nan)

    # Masks retain the distinction between valid low G, valid higher G, and
    # outside/undefined geometry by using 255 as NoData.
    valid_mask = np.full(g.shape, NODATA_BYTE, dtype="uint8")
    valid_mask[study_mask] = 0
    valid_mask[vector_valid] = 1

    low_flag = np.full(g.shape, NODATA_BYTE, dtype="uint8")
    low_flag[study_mask] = 0
    low_flag[vector_valid & (g < LOW_G_THRESHOLD)] = 1

    stats_g = detailed_stats(g, vector_valid)
    stats_signed = detailed_stats(signed_dot, vector_valid)
    coverage = 100.0 * valid_count / study_count
    values = g[vector_valid]
    class_counts = descriptive_class_counts(values)

    log(f"[COVERAGE] Valid geometric direction: {valid_count:,}/{study_count:,} ({coverage:.2f}%)")
    log(
        f"[G] min={stats_g['min']:.4f}, median={stats_g['median']:.4f}, "
        f"mean={stats_g['mean']:.4f}, p95={stats_g['p95']:.4f}, max={stats_g['max']:.4f}"
    )
    log(
        f"[G] fraction below {LOW_G_THRESHOLD:.2f}: "
        f"{100.0 * stats_g['low_geometry_lt_threshold_fraction']:.2f}%"
    )
    log(f"[G] unique values rounded to 6 decimals: {stats_g['unique_values_rounded_6dp']:,}")

    # Save core scientific outputs
    write_float_tif(SIGNED_DOT_TIF, signed_dot, anchor)
    write_float_tif(G_TIF, g, anchor)
    write_byte_tif(G_VALID_MASK_TIF, valid_mask, anchor)
    write_byte_tif(G_LOW_FLAG_TIF, low_flag, anchor)

    # Compatibility copy for the existing Step 5.2 input path
    shutil.copy2(G_TIF, LEGACY_G_TIF)

    log(f"[OUTPUT] Signed projection: {SIGNED_DOT_TIF}")
    log(f"[OUTPUT] Pure G: {G_TIF}")
    log(f"[OUTPUT] Legacy G alias: {LEGACY_G_TIF}")
    log(f"[OUTPUT] Validity mask: {G_VALID_MASK_TIF}")
    log(f"[OUTPUT] Low-G flag: {G_LOW_FLAG_TIF}")

    los_metadata = None
    if L_META_JSON.exists():
        try:
            with open(L_META_JSON, "r", encoding="utf-8") as handle:
                los_metadata = json.load(handle)
        except Exception as exc:
            log(f"[WARN] Could not read LOS metadata: {exc}")

    stale_found = [str(path) for path in STALE_COMPOSITE_FILES if path.exists()]
    if stale_found:
        log("[IMPORTANT] Older composite Step 5.1 files still exist and are not used:")
        for item in stale_found:
            log(f"            {item}")
        log("            Do not use these files as pure G or as the revised IRS.")

    summary = {
        "generated_at_utc": utc_now_iso(),
        "step": "5.1 Pure Geometric Observability",
        "formula": {
            "signed_projection": "l_hat dot s_hat",
            "geometric_observability_G": "abs(l_hat dot s_hat)",
        },
        "scientific_interpretation": {
            "G_near_0": "weak LOS sensitivity to assumed downslope motion",
            "G_near_1": "strong LOS sensitivity to assumed downslope motion",
            "sign_note": (
                "The signed coefficient depends on LOS vector orientation; "
                "G is invariant to LOS sign convention."
            ),
            "threshold_note": (
                f"G < {LOW_G_THRESHOLD} is a diagnostic low-geometry flag only; "
                "G is not thresholded or rescaled."
            ),
        },
        "domain": {
            "study_area_pixels": study_count,
            "valid_geometry_pixels": valid_count,
            "valid_geometry_coverage_percent": coverage,
            "definition": (
                "study_mask AND finite normalized s_hat AND finite normalized l_hat"
            ),
            "dynamic_mask_used": False,
            "coherence_used": False,
            "ps_density_used": False,
        },
        "statistics_G": stats_g,
        "statistics_signed_projection": stats_signed,
        "descriptive_intervals": {
            "warning": "Intervals are descriptive, not calibrated detection probabilities.",
            "counts": class_counts,
        },
        "vector_QA": {"s_hat": s_qa, "l_hat": l_qa},
        "inputs": {
            "anchor": str(ANCHOR_TIF),
            "study_mask": str(STUDY_MASK_TIF),
            "s_hat": {
                "east": str(S_E_TIF),
                "north": str(S_N_TIF),
                "up": str(S_U_TIF),
            },
            "l_hat": {
                "east": str(L_E_TIF),
                "north": str(L_N_TIF),
                "up": str(L_U_TIF),
                "metadata": str(L_META_JSON) if L_META_JSON.exists() else None,
                "metadata_content": los_metadata,
            },
        },
        "outputs": {
            "signed_projection_tif": str(SIGNED_DOT_TIF),
            "geometric_observability_G_tif": str(G_TIF),
            "legacy_G_alias_tif": str(LEGACY_G_TIF),
            "valid_mask_tif": str(G_VALID_MASK_TIF),
            "low_G_flag_tif": str(G_LOW_FLAG_TIF),
            "map_png": str(MAP_PNG) if MAKE_FIGURES else None,
            "map_pdf": str(MAP_PDF) if MAKE_FIGURES else None,
            "histogram_png": str(HIST_PNG) if MAKE_FIGURES else None,
            "histogram_pdf": str(HIST_PDF) if MAKE_FIGURES else None,
        },
        "stale_composite_files_found": stale_found,
    }

    with open(SUMMARY_JSON, "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2, ensure_ascii=False, default=json_safe)

    # Metadata file retained at a predictable Step 5.1 filename
    with open(META_JSON, "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2, ensure_ascii=False, default=json_safe)

    log(f"[METADATA] {META_JSON}")
    log(f"[SUMMARY] {SUMMARY_JSON}")

    create_figures(g, vector_valid, study_mask, anchor, stats_g)

    log("=== Step 5.1 completed successfully ===")
    log("Next: use G_TIF (or its legacy alias) as the geometry term in corrected Step 5.2 IRS.")


if __name__ == "__main__":
    main()


# Visualization: Observability

In [ ]:
# ============================================================================
# Step 5.1 — Visualization of Corrected Pure Geometric Observability (G)
#
# This script is ONLY for visualization and summary statistics.
# It reads the corrected Step 5.1 output:
#
#   geometric_observability_G_0_1_32645_12p5m_float32.tif
#
# Scientific safeguards:
#   - No RBF, cubic, IDW, or other surface interpolation
#   - No boundary erosion
#   - No automatic hard/soft observability fallback
#   - Same original raster pixels used for map, histogram, and statistics
#   - Coverage calculated relative to the study-area mask
#   - Low/moderate/high intervals are descriptive, not calibrated probabilities
# ============================================================================

from __future__ import annotations
import os

import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D


# =============================================================================
# CONFIGURATION
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

OBS_DIR = BASE / "Results" / "Step-5.1-Observability"

# Corrected Step 5.1 scientific output
G_TIF = OBS_DIR / "geometric_observability_G_0_1_32645_12p5m_float32.tif"

# Optional validity mask produced by corrected Step 5.1
G_VALID_MASK_TIF = (
    OBS_DIR / "geometric_observability_valid_mask_32645_12p5m_byte.tif"
)

# Study-area and optional overlay
STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
ROAD_SHP = BASE / "NTD_Road.shp"

# Corrected metadata
META_JSON = OBS_DIR / "geometric_observability_metadata.json"

# Figure outputs
MAP_PNG = OBS_DIR / "Geometric_Observability_G_Map.png"
MAP_PDF = OBS_DIR / "Geometric_Observability_G_Map.pdf"
HIST_PNG = OBS_DIR / "Geometric_Observability_G_Histogram.png"
HIST_PDF = OBS_DIR / "Geometric_Observability_G_Histogram.pdf"
SUMMARY_JSON = OBS_DIR / "geometric_observability_visualization_summary.json"
SUMMARY_CSV = OBS_DIR / "geometric_observability_visualization_statistics.csv"

# Display settings
SHOW_FIGURES = True
SAVE_FIGURES = True
MAP_DPI = 300
MAP_FIGSIZE = (6.3, 9.2)
HIST_FIGSIZE = (6.3, 4.3)
CMAP = "viridis"
RASTER_DISPLAY_INTERPOLATION = "nearest"

# DMS tick interval for the main map
DMS_INTERVAL_MINUTES = 5

# Descriptive intervals only
LOW_MAX = 0.30
MODERATE_MAX = 0.70

# Diagnostic geometric threshold used in the downslope inversion
LOW_G_THRESHOLD = 0.05

# Font preference
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "DejaVu Serif"]
plt.rcParams["axes.unicode_minus"] = False


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


# =============================================================================
# RASTER HELPERS
# =============================================================================
def same_grid(ds: rasterio.io.DatasetReader, ref: dict[str, Any]) -> bool:
    return (
        ds.crs == ref["crs"]
        and ds.transform == ref["transform"]
        and ds.width == ref["width"]
        and ds.height == ref["height"]
    )


def clean_nodata(
    arr: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    out = arr.astype("float64", copy=True)

    if nodata is not None:
        try:
            nd = float(nodata)
            if np.isfinite(nd):
                out[out == nd] = np.nan
        except (TypeError, ValueError):
            pass

    out[out == -9999.0] = np.nan
    out[out == -32768.0] = np.nan
    out[np.abs(out) > 1.0e20] = np.nan
    return out


def read_g_raster(path: Path) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(path, "corrected geometric-observability raster")

    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"Raster has no CRS: {path}")

        raw = ds.read(1)
        g = clean_nodata(raw, ds.nodata)

        ref = {
            "crs": ds.crs,
            "transform": ds.transform,
            "width": ds.width,
            "height": ds.height,
            "bounds": ds.bounds,
            "resolution_x": abs(ds.transform.a),
            "resolution_y": abs(ds.transform.e),
            "epsg": ds.crs.to_epsg(),
        }

    outside = np.isfinite(g) & ((g < -1e-6) | (g > 1.0 + 1e-6))
    if outside.any():
        raise ValueError(
            f"G contains {int(outside.sum()):,} values outside [0,1]. "
            "Recheck the corrected Step 5.1 calculation."
        )

    g = np.where(np.isfinite(g), np.clip(g, 0.0, 1.0), np.nan)
    return g, ref


def read_mask_to_grid(
    path: Path,
    ref: dict[str, Any],
    description: str,
    optional: bool = False,
) -> np.ndarray | None:
    if not path.exists():
        if optional:
            return None
        raise FileNotFoundError(f"Missing {description}: {path}")

    with rasterio.open(path) as ds:
        raw = ds.read(1)
        src_nodata = ds.nodata

        if same_grid(ds, ref):
            arr = raw.astype("float64")
            if src_nodata is not None:
                arr[arr == src_nodata] = 0.0
            return np.isfinite(arr) & (arr > 0.5)

        destination = np.zeros(
            (ref["height"], ref["width"]),
            dtype="float64",
        )

        reproject(
            source=raw,
            destination=destination,
            src_transform=ds.transform,
            src_crs=ds.crs,
            src_nodata=src_nodata,
            dst_transform=ref["transform"],
            dst_crs=ref["crs"],
            dst_nodata=0,
            resampling=Resampling.nearest,
        )

        return np.isfinite(destination) & (destination > 0.5)


# =============================================================================
# STATISTICS
# =============================================================================
def detailed_stats(values: np.ndarray) -> dict[str, float | int]:
    if values.size == 0:
        return {"count": 0}

    return {
        "count": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "near_zero_lt_0_01_fraction": float(np.mean(values < 0.01)),
        "low_geometry_lt_0_05_fraction": float(
            np.mean(values < LOW_G_THRESHOLD)
        ),
        "unique_values_rounded_6dp": int(
            np.unique(np.round(values, 6)).size
        ),
    }


def descriptive_counts(values: np.ndarray) -> dict[str, dict[str, float | int]]:
    n = int(values.size)

    masks = {
        "low_0_to_0.3": (values >= 0.0) & (values < LOW_MAX),
        "moderate_0.3_to_0.7": (
            (values >= LOW_MAX) & (values < MODERATE_MAX)
        ),
        "high_0.7_to_1.0": (
            (values >= MODERATE_MAX) & (values <= 1.0)
        ),
    }

    result: dict[str, dict[str, float | int]] = {}
    for name, mask in masks.items():
        count = int(mask.sum())
        result[name] = {
            "count": count,
            "percentage": 100.0 * count / max(n, 1),
        }

    return result


# =============================================================================
# CARTOGRAPHIC HELPERS
# =============================================================================
def try_load_roads(target_crs):
    if not ROAD_SHP.exists():
        log(f"[INFO] Road overlay skipped: {ROAD_SHP}")
        return None

    try:
        import geopandas as gpd

        roads = gpd.read_file(ROAD_SHP)
        if roads.crs is None:
            raise ValueError("The road shapefile has no CRS.")
        roads = roads.to_crs(target_crs)
        log(f"[ROAD] Loaded {len(roads):,} feature(s)")
        return roads
    except Exception as exc:
        log(f"[WARN] Road overlay unavailable: {exc}")
        return None


def add_north_arrow(ax) -> None:
    ax.annotate(
        "N",
        xy=(0.085, 0.955),
        xytext=(0.085, 0.865),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        arrowprops={
            "facecolor": "black",
            "edgecolor": "black",
            "width": 2.0,
            "headwidth": 8.5,
        },
        zorder=20,
    )


def nice_length_125(target: float) -> float:
    target = max(float(target), 1.0)
    exponent = math.floor(math.log10(target))
    base = 10.0 ** exponent

    for multiplier in (1.0, 2.0, 5.0, 10.0):
        value = multiplier * base
        if value >= target:
            return value

    return 10.0 * base


def add_scale_bar_utm(ax, bounds, fraction: float = 0.20) -> None:
    width = bounds.right - bounds.left
    height = bounds.top - bounds.bottom
    total_length = nice_length_125(width * fraction)

    segment_length = total_length / 4.0
    bar_height = height * 0.010

    x0 = bounds.left + width * 0.08
    y0 = bounds.bottom + height * 0.055

    from matplotlib.patches import Rectangle

    for idx in range(4):
        face = "black" if idx % 2 == 0 else "white"
        ax.add_patch(
            Rectangle(
                (x0 + idx * segment_length, y0),
                segment_length,
                bar_height,
                facecolor=face,
                edgecolor="black",
                linewidth=0.55,
                zorder=15,
            )
        )

    ax.text(
        x0,
        y0 - bar_height * 0.55,
        "0",
        ha="center",
        va="top",
        fontsize=8,
        zorder=16,
    )
    ax.text(
        x0 + total_length,
        y0 - bar_height * 0.55,
        f"{total_length / 1000.0:g}",
        ha="center",
        va="top",
        fontsize=8,
        zorder=16,
    )
    ax.text(
        x0 + total_length / 2.0,
        y0 + bar_height * 1.40,
        "km",
        ha="center",
        va="bottom",
        fontsize=8,
        zorder=16,
    )


def dms_label(value: float, longitude: bool) -> str:
    absolute = abs(value)
    degree = int(absolute)
    minute_float = (absolute - degree) * 60.0
    minute = int(round(minute_float))

    if minute == 60:
        degree += 1
        minute = 0

    if longitude:
        hemisphere = "E" if value >= 0 else "W"
    else:
        hemisphere = "N" if value >= 0 else "S"

    return f"{degree}°{minute:02d}'{hemisphere}"


def multiples_in_range(
    minimum: float,
    maximum: float,
    interval: float,
) -> np.ndarray:
    start = math.ceil(minimum / interval) * interval
    stop = math.floor(maximum / interval) * interval

    if stop < start:
        return np.array([], dtype="float64")

    count = int(round((stop - start) / interval)) + 1
    return start + np.arange(count, dtype="float64") * interval


def install_fixed_dms_ticks(
    ax,
    raster_crs,
    bounds,
    interval_minutes: int = DMS_INTERVAL_MINUTES,
) -> bool:
    """
    Install approximate longitude/latitude ticks at a fixed angular interval.

    Because the raster is projected, longitude ticks are positioned along the
    map's mid-latitude and latitude ticks along the map's mid-longitude.
    """
    try:
        from pyproj import Transformer
    except Exception:
        log("[INFO] pyproj unavailable; axes remain in UTM coordinates.")
        return False

    to_ll = Transformer.from_crs(
        raster_crs,
        "EPSG:4326",
        always_xy=True,
    )
    from_ll = Transformer.from_crs(
        "EPSG:4326",
        raster_crs,
        always_xy=True,
    )

    corners_x = [
        bounds.left,
        bounds.right,
        bounds.left,
        bounds.right,
    ]
    corners_y = [
        bounds.bottom,
        bounds.bottom,
        bounds.top,
        bounds.top,
    ]
    lons, lats = to_ll.transform(corners_x, corners_y)

    lon_min = float(np.min(lons))
    lon_max = float(np.max(lons))
    lat_min = float(np.min(lats))
    lat_max = float(np.max(lats))

    lon_mid = 0.5 * (lon_min + lon_max)
    lat_mid = 0.5 * (lat_min + lat_max)

    interval_degree = float(interval_minutes) / 60.0

    lon_ticks = multiples_in_range(lon_min, lon_max, interval_degree)
    lat_ticks = multiples_in_range(lat_min, lat_max, interval_degree)

    if lon_ticks.size:
        x_positions, _ = from_ll.transform(
            lon_ticks,
            np.full(lon_ticks.shape, lat_mid),
        )
        x_positions = np.asarray(x_positions)
        valid_x = (
            np.isfinite(x_positions)
            & (x_positions >= bounds.left)
            & (x_positions <= bounds.right)
        )
        ax.set_xticks(x_positions[valid_x])
        ax.set_xticklabels(
            [dms_label(value, True) for value in lon_ticks[valid_x]]
        )

    if lat_ticks.size:
        _, y_positions = from_ll.transform(
            np.full(lat_ticks.shape, lon_mid),
            lat_ticks,
        )
        y_positions = np.asarray(y_positions)
        valid_y = (
            np.isfinite(y_positions)
            & (y_positions >= bounds.bottom)
            & (y_positions <= bounds.top)
        )
        ax.set_yticks(y_positions[valid_y])
        ax.set_yticklabels(
            [dms_label(value, False) for value in lat_ticks[valid_y]]
        )

    return bool(lon_ticks.size or lat_ticks.size)


# =============================================================================
# FIGURE CREATION
# =============================================================================
def make_map(
    g: np.ndarray,
    valid: np.ndarray,
    study: np.ndarray,
    ref: dict[str, Any],
    stats: dict[str, float | int],
) -> None:
    bounds = ref["bounds"]
    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    display = np.ma.masked_invalid(
        np.where(valid, g, np.nan)
    )

    fig, ax = plt.subplots(figsize=MAP_FIGSIZE, facecolor="white")

    image = ax.imshow(
        display,
        extent=extent,
        origin="upper",
        interpolation=RASTER_DISPLAY_INTERPOLATION,
        cmap=CMAP,
        norm=Normalize(vmin=0.0, vmax=1.0),
        zorder=1,
    )

    ax.contour(
        study.astype("uint8"),
        levels=[0.5],
        extent=extent,
        origin="upper",
        colors="black",
        linewidths=0.85,
        zorder=8,
    )

    roads = try_load_roads(ref["crs"])
    if roads is not None and not roads.empty:
        roads.plot(
            ax=ax,
            color="red",
            linewidth=0.85,
            zorder=9,
        )

    ax.set_xlim(bounds.left, bounds.right)
    ax.set_ylim(bounds.bottom, bounds.top)

    dms_installed = install_fixed_dms_ticks(
        ax,
        ref["crs"],
        bounds,
        interval_minutes=DMS_INTERVAL_MINUTES,
    )

    ax.set_xlabel(
        "Longitude" if dms_installed else "Easting (m)",
        fontsize=10,
        fontweight="bold",
    )
    ax.set_ylabel(
        "Latitude" if dms_installed else "Northing (m)",
        fontsize=10,
        fontweight="bold",
    )
    ax.tick_params(axis="both", labelsize=8)

    if dms_installed:
        plt.setp(
            ax.get_xticklabels(),
            rotation=45,
            ha="right",
        )

    add_north_arrow(ax)
    add_scale_bar_utm(ax, bounds)

    colorbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.035,
        pad=0.025,
    )
    colorbar.set_label(
        "Geometric observability, G (0–1)",
        fontsize=10,
        fontweight="bold",
    )
    colorbar.ax.tick_params(labelsize=8)
    colorbar.set_ticks(np.linspace(0.0, 1.0, 5))

    legend_elements = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=1.2,
            label="China–Nepal Friendship Highway",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=1.1,
            label="Study area",
        ),
    ]
    ax.legend(
        handles=legend_elements,
        loc="lower right",
        fontsize=8,
        frameon=True,
        framealpha=0.92,
    )

    study_pixels = int(study.sum())
    valid_pixels = int(valid.sum())
    coverage = 100.0 * valid_pixels / max(study_pixels, 1)

    annotation = (
        f"Valid geometry coverage: {coverage:.1f}%\n"
        f"Mean G: {stats['mean']:.3f}\n"
        f"Median G: {stats['median']:.3f}\n"
        f"G < {LOW_G_THRESHOLD:.2f}: "
        f"{100.0 * stats['low_geometry_lt_0_05_fraction']:.1f}%"
    )
    ax.text(
        0.985,
        0.015,
        annotation,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=8,
        bbox={
            "facecolor": "white",
            "edgecolor": "0.55",
            "alpha": 0.90,
            "boxstyle": "round,pad=0.30",
        },
        zorder=20,
    )

    fig.tight_layout()

    if SAVE_FIGURES:
        fig.savefig(
            MAP_PNG,
            dpi=MAP_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        fig.savefig(
            MAP_PDF,
            dpi=MAP_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        log(f"[MAP] {MAP_PNG}")
        log(f"[MAP] {MAP_PDF}")

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)


def make_histogram(
    values: np.ndarray,
    stats: dict[str, float | int],
) -> None:
    fig, ax = plt.subplots(
        figsize=HIST_FIGSIZE,
        facecolor="white",
    )

    bins = np.linspace(0.0, 1.0, 31)
    counts, edges, patches = ax.hist(
        values,
        bins=bins,
        edgecolor="black",
        linewidth=0.55,
    )

    cmap = plt.get_cmap(CMAP)
    centres = 0.5 * (edges[:-1] + edges[1:])
    for centre, patch in zip(centres, patches):
        patch.set_facecolor(cmap(centre))

    ax.axvline(
        stats["mean"],
        linestyle="--",
        linewidth=1.1,
        label=f"Mean = {stats['mean']:.3f}",
    )
    ax.axvline(
        stats["median"],
        linestyle=":",
        linewidth=1.3,
        label=f"Median = {stats['median']:.3f}",
    )
    ax.axvline(
        LOW_G_THRESHOLD,
        linestyle="-.",
        linewidth=1.1,
        label=f"Diagnostic low-G threshold = {LOW_G_THRESHOLD:.2f}",
    )

    ax.set_xlim(0.0, 1.0)
    ax.set_xlabel(
        "Geometric observability, G",
        fontsize=10,
        fontweight="bold",
    )
    ax.set_ylabel(
        "Number of study-area pixels",
        fontsize=10,
        fontweight="bold",
    )
    ax.tick_params(axis="both", labelsize=8)
    ax.grid(axis="y", alpha=0.25, linewidth=0.5)
    ax.legend(fontsize=8, framealpha=0.92)

    fig.tight_layout()

    if SAVE_FIGURES:
        fig.savefig(
            HIST_PNG,
            dpi=MAP_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        fig.savefig(
            HIST_PDF,
            dpi=MAP_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        log(f"[HISTOGRAM] {HIST_PNG}")
        log(f"[HISTOGRAM] {HIST_PDF}")

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)


# =============================================================================
# SUMMARY OUTPUT
# =============================================================================
def write_summary_csv(
    path: Path,
    stats: dict[str, float | int],
    study_pixels: int,
    valid_pixels: int,
) -> None:
    import csv

    coverage = 100.0 * valid_pixels / max(study_pixels, 1)

    rows = [
        ("study_area_pixels", study_pixels),
        ("valid_G_pixels", valid_pixels),
        ("valid_geometry_coverage_percent", coverage),
    ]
    rows.extend(stats.items())

    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["metric", "value"])
        writer.writerows(rows)


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=== Step 5.1 visualization: pure geometric observability G ===")

    g, ref = read_g_raster(G_TIF)
    study = read_mask_to_grid(
        STUDY_MASK_TIF,
        ref,
        "study-area mask",
    )

    corrected_valid_mask = read_mask_to_grid(
        G_VALID_MASK_TIF,
        ref,
        "corrected G-validity mask",
        optional=True,
    )

    raster_valid = (
        study
        & np.isfinite(g)
        & (g >= 0.0)
        & (g <= 1.0)
    )

    if corrected_valid_mask is not None:
        valid = raster_valid & corrected_valid_mask
    else:
        valid = raster_valid

    study_pixels = int(study.sum())
    valid_pixels = int(valid.sum())

    if study_pixels == 0:
        raise RuntimeError("The study mask contains no study-area pixels.")

    if valid_pixels == 0:
        raise RuntimeError(
            "No valid G values occur inside the study area."
        )

    values = g[valid]
    stats = detailed_stats(values)
    classes = descriptive_counts(values)
    coverage = 100.0 * valid_pixels / study_pixels

    log(
        f"[RASTER] EPSG:{ref['epsg']} | "
        f"{ref['width']} x {ref['height']} | "
        f"{ref['resolution_x']:.3f} m"
    )
    log(
        f"[COVERAGE] {valid_pixels:,}/{study_pixels:,} "
        f"study-area pixels ({coverage:.2f}%)"
    )
    log(
        f"[G] min={stats['min']:.4f}, "
        f"median={stats['median']:.4f}, "
        f"mean={stats['mean']:.4f}, "
        f"p95={stats['p95']:.4f}, "
        f"max={stats['max']:.4f}"
    )
    log(
        f"[G] G < {LOW_G_THRESHOLD:.2f}: "
        f"{100.0 * stats['low_geometry_lt_0_05_fraction']:.2f}%"
    )

    print("\nDescriptive observability intervals")
    print(
        "These are descriptive classes, not calibrated "
        "deformation-detection probabilities."
    )
    for name, item in classes.items():
        print(
            f"  {name}: {item['count']:,} "
            f"({item['percentage']:.2f}%)"
        )

    metadata_content = None
    if META_JSON.exists():
        try:
            with open(META_JSON, "r", encoding="utf-8") as handle:
                metadata_content = json.load(handle)
            log(f"[METADATA] Loaded {META_JSON.name}")
        except Exception as exc:
            log(f"[WARN] Metadata could not be read: {exc}")

    summary = {
        "generated_at_utc": utc_now_iso(),
        "purpose": (
            "Visualization and summary of corrected pure geometric "
            "observability G. No scientific raster interpolation was used."
        ),
        "input": {
            "G_tif": str(G_TIF),
            "study_mask": str(STUDY_MASK_TIF),
            "valid_mask": (
                str(G_VALID_MASK_TIF)
                if corrected_valid_mask is not None
                else None
            ),
            "metadata": (
                str(META_JSON)
                if META_JSON.exists()
                else None
            ),
        },
        "raster": {
            "epsg": ref["epsg"],
            "width": ref["width"],
            "height": ref["height"],
            "resolution_x_m": ref["resolution_x"],
            "resolution_y_m": ref["resolution_y"],
        },
        "coverage": {
            "study_area_pixels": study_pixels,
            "valid_G_pixels": valid_pixels,
            "valid_geometry_coverage_percent": coverage,
        },
        "statistics": stats,
        "descriptive_intervals": {
            "warning": (
                "Low, moderate, and high intervals are descriptive only."
            ),
            "counts": classes,
        },
        "visualization_settings": {
            "surface_interpolation_used": False,
            "boundary_erosion_used": False,
            "raster_display_interpolation": (
                RASTER_DISPLAY_INTERPOLATION
            ),
            "DMS_tick_interval_minutes": DMS_INTERVAL_MINUTES,
            "colormap": CMAP,
        },
        "source_step_5_1_metadata": metadata_content,
        "outputs": {
            "map_png": str(MAP_PNG),
            "map_pdf": str(MAP_PDF),
            "histogram_png": str(HIST_PNG),
            "histogram_pdf": str(HIST_PDF),
            "summary_json": str(SUMMARY_JSON),
            "statistics_csv": str(SUMMARY_CSV),
        },
    }

    with open(SUMMARY_JSON, "w", encoding="utf-8") as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    write_summary_csv(
        SUMMARY_CSV,
        stats,
        study_pixels,
        valid_pixels,
    )

    make_map(g, valid, study, ref, stats)
    make_histogram(values, stats)

    log(f"[SUMMARY] {SUMMARY_JSON}")
    log(f"[SUMMARY] {SUMMARY_CSV}")
    log("=== Step 5.1 visualization completed successfully ===")


if __name__ == "__main__":
    main()


# Step 5.1b — Geometric Observability and Blind-Zone Representativeness Audit

In [ ]:
# =============================================================================
# Step 5.1b — Geometric Observability and Blind-Zone Representativeness Audit
# =============================================================================
# Retains the corrected Step 5.1 formulation:
#     C = l_hat · s_hat
#     G = |C|
#
# Produces the quantitative evidence requested by Reviewer 1, Comment 3:
#   • near-blind and low-observability area statistics;
#   • threshold sensitivity and cumulative area curve;
#   • aspect and slope-aspect dependence;
#   • PS-location and landslide-inventory representativeness;
#   • publication-ready maps, tables, QA and a four-panel reviewer figure.
#
# Important interpretation:
#   IRS can attenuate unreliable kinematic evidence, but it cannot recover
#   motion that is intrinsically unobservable from one ascending orbit.
# =============================================================================

from __future__ import annotations

import csv
import json
import math
import os
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.features import geometry_mask, rasterize
from rasterio.transform import rowcol
from rasterio.windows import Window, from_bounds

import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon, Rectangle
from matplotlib.ticker import FuncFormatter, LinearLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    import geopandas as gpd
    from shapely.geometry import mapping
    HAVE_GEOPANDAS = True
except Exception:
    gpd = None
    HAVE_GEOPANDAS = False

# Compatibility imports for vector reading.
#
# Some older GeoPandas releases call fiona.path inside gpd.read_file().
# Fiona 1.10 removed that attribute, which causes:
#     AttributeError: module 'fiona' has no attribute 'path'
#
# This script avoids that incompatible code path. It uses Pyogrio when
# available, otherwise direct fiona.open() plus GeoDataFrame.from_features().
try:
    import fiona
    HAVE_FIONA = True
except Exception:
    fiona = None
    HAVE_FIONA = False

try:
    import pyogrio
    HAVE_PYOGRIO = True
except Exception:
    pyogrio = None
    HAVE_PYOGRIO = False

# =============================================================================
# PATHS
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

ANCHOR_TIF = BASE / "Results" / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
STUDY_MASK_TIF = BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
OBS_DIR = BASE / "Results" / "Step-5.1-Observability"

G_TIF_CANDIDATES = [
    OBS_DIR / "geometric_observability_G_0_1_32645_12p5m_float32.tif",
    OBS_DIR / "obs_term_sensitivity_0_1_32645_12p5m_float32.tif",
    OBS_DIR / "observability_0_1_32645_12p5m_float32.tif",
]
SIGNED_C_TIF_CANDIDATES = [
    OBS_DIR / "signed_los_slope_projection_32645_12p5m_float32.tif",
    OBS_DIR / "signed_projection_C_32645_12p5m_float32.tif",
]
SLOPE_TIF_CANDIDATES = [
    BASE / "Results" / "Step-1.1-Slope_Aspect" / "slope_deg.tif",
    BASE / "Results" / "Step-1.1-Slope_Aspect" / "slope_deg_32645_12p5m_float32.tif",
    BASE / "Results" / "Step-3.3-Downslope unit vector (ŝ)" / "slope_deg_32645_12p5m_float32.tif",
]
ASPECT_TIF_CANDIDATES = [
    BASE / "Results" / "Step-1.1-Slope_Aspect" / "aspect_deg.tif",
    BASE / "Results" / "Step-1.1-Slope_Aspect" / "aspect_deg_32645_12p5m_float32.tif",
    BASE / "Results" / "Step-1.1-Slope_Aspect" / "aspect.tif",
]
PS_POINTS_CSV_CANDIDATES = [
    BASE / "Results" / "Step-3.1-Coherence" / "ps_points_coherence.csv",
    BASE / "Results" / "Step-3.1-Coherence" / "PS_points_coherence.csv",
]

# Set this explicitly if automatic inventory discovery does not identify the
# correct 199-site polygon inventory.
LANDSLIDE_INVENTORY_OVERRIDE: Path | None = None
LANDSLIDE_INVENTORY_CANDIDATES = [
    BASE / "NTD_Landslide_Inventory.shp",
    BASE / "NTD_landslide_inventory.shp",
    BASE / "Landslide_Inventory.shp",
    BASE / "LandslideInventory.shp",
    BASE / "Inventory" / "NTD_Landslide_Inventory.shp",
    BASE / "Data" / "NTD_Landslide_Inventory.shp",
]

ROAD_SHP = BASE / "NTD_Road.shp"
STUDY_AREA_SHP = BASE / "Study_Area.shp"

OUT_DIR = BASE / "Results" / "Step-5.1b-Geometric_Representativeness"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Rasters
SLOPE_VALID_MASK_TIF = OUT_DIR / "geometric_observability_slope_valid_mask_32645_12p5m_byte.tif"
OBS_CLASS_TIF = OUT_DIR / "geometric_observability_classes_32645_12p5m_byte.tif"
NEAR_BLIND_MASK_TIF = OUT_DIR / "G_lt_0p05_near_blind_mask_32645_12p5m_byte.tif"
VERY_LOW_MASK_TIF = OUT_DIR / "G_lt_0p10_very_low_mask_32645_12p5m_byte.tif"
LOW_OBS_MASK_TIF = OUT_DIR / "G_lt_0p20_low_observability_mask_32645_12p5m_byte.tif"
MODERATE_MASK_TIF = OUT_DIR / "G_lt_0p50_moderate_or_lower_mask_32645_12p5m_byte.tif"

# Tables
AREA_STATS_CSV = OUT_DIR / "geometric_observability_area_statistics.csv"
THRESHOLD_CSV = OUT_DIR / "geometric_observability_threshold_sensitivity.csv"
CLASS_STATS_CSV = OUT_DIR / "geometric_observability_class_statistics.csv"
CDF_DATA_CSV = OUT_DIR / "geometric_observability_cumulative_area_curve.csv"
ASPECT_STATS_CSV = OUT_DIR / "geometric_observability_by_aspect.csv"
SLOPE_ASPECT_LOW_CSV = OUT_DIR / "G_low_fraction_by_slope_aspect.csv"
SLOPE_ASPECT_MEDIAN_CSV = OUT_DIR / "G_median_by_slope_aspect.csv"
SLOPE_ASPECT_COUNT_CSV = OUT_DIR / "G_pixel_count_by_slope_aspect.csv"
LANDSLIDE_ZONAL_CSV = OUT_DIR / "geometric_observability_landslide_zonal_statistics.csv"
LANDSLIDE_SUMMARY_CSV = OUT_DIR / "geometric_observability_landslide_summary.csv"
DOMAIN_COMPARISON_CSV = OUT_DIR / "geometric_observability_domain_comparison.csv"

# Figures
CONT_MAP = OUT_DIR / "Geometric Observability G Map"
CLASS_MAP = OUT_DIR / "Geometric Observability Classes Map"
CDF_FIG = OUT_DIR / "Geometric Observability Cumulative Area Curve"
ASPECT_FIG = OUT_DIR / "Low Observability by Aspect"
SLOPE_ASPECT_FIG = OUT_DIR / "Low Observability Slope Aspect Heatmap"
DOMAIN_ECDF_FIG = OUT_DIR / "Observability Domain ECDF Comparison"
REVIEWER_FIG = OUT_DIR / "Reviewer Figure Geometric Observability"

QA_JSON = OUT_DIR / "geometric_observability_quality_assurance.json"
META_JSON = OUT_DIR / "geometric_observability_representativeness_metadata.json"
REPORT_TXT = OUT_DIR / "Geometric Observability Reviewer Report.txt"

# =============================================================================
# SETTINGS
# =============================================================================
NODATA_FLOAT = -9999.0
NODATA_BYTE = 255
REQUIRE_EXACT_GRID = True
REQUIRE_LANDSLIDE_INVENTORY = True

FLAT_SLOPE_THRESHOLD_DEG = 5.0
PRIMARY_NEAR_BLIND_THRESHOLD = 0.05
BROADER_LOW_OBSERVABILITY_THRESHOLD = 0.20
THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

CLASS_LABELS = [
    "Near-blind/capped (G < 0.05)",
    "Very low (0.05 ≤ G < 0.10)",
    "Low (0.10 ≤ G < 0.20)",
    "Moderate (0.20 ≤ G < 0.50)",
    "High (G ≥ 0.50)",
]
ASPECT_LABELS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
SLOPE_EDGES = [5.0, 15.0, 25.0, 35.0, 45.0, 60.0, 90.0001]
SLOPE_LABELS = ["5–15°", "15–25°", "25–35°", "35–45°", "45–60°", ">60°"]

G_TOL = 1.0e-6
C_TOL = 1.0e-5

MAP_FIGSIZE = (2, 4)
PLOT_FIGSIZE = (3.4, 2.8)
HEATMAP_FIGSIZE = (4.6, 3.2)
REVIEWER_FIGSIZE = (7.2, 6.2)
FIG_DPI = 900
REVIEWER_DPI = 600
DISPLAY_BUFFER_FRACTION = 0.02
LONGITUDE_DMS_TICK_COUNT = 4
LATITUDE_DMS_TICK_COUNT = 3
DMS_TICK_FONTSIZE = 5
SHOW_FIGURES = True
SAVE_FIGURES = True

CONT_CMAP = plt.get_cmap("viridis").copy()
CONT_CMAP.set_bad(color="white", alpha=1.0)
CLASS_COLOURS = plt.get_cmap("viridis")(np.linspace(0.05, 0.95, 5))
CLASS_CMAP = ListedColormap(CLASS_COLOURS)
CLASS_NORM = BoundaryNorm(np.arange(0.5, 6.5, 1.0), CLASS_CMAP.N)

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "DejaVu Serif"]
plt.rcParams["axes.unicode_minus"] = False

# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def pick_existing(candidates: Iterable[Path], description: str, required: bool = True) -> Path | None:
    for path in candidates:
        if path.exists():
            log(f"[PATH] {description}: {path}")
            return path
    if required:
        raise FileNotFoundError(
            f"Could not find {description}. Tried:\n" + "\n".join(map(str, candidates))
        )
    log(f"[WARN] Optional {description} not found.")
    return None


def resolve_inventory() -> Path | None:
    if LANDSLIDE_INVENTORY_OVERRIDE is not None:
        path = Path(LANDSLIDE_INVENTORY_OVERRIDE)
        require_file(path, "landslide inventory")
        return path
    for path in LANDSLIDE_INVENTORY_CANDIDATES:
        if path.exists():
            log(f"[PATH] Landslide inventory: {path}")
            return path
    found = []
    try:
        for path in BASE.rglob("*.shp"):
            name = path.name.lower()
            if "landslide" in name and ("invent" in name or "site" in name or "polygon" in name):
                found.append(path)
    except Exception:
        found = []
    found = sorted(set(found))
    if len(found) == 1:
        log(f"[PATH] Auto-discovered inventory: {found[0]}")
        return found[0]
    if len(found) > 1:
        raise RuntimeError(
            "Multiple possible inventories found. Set LANDSLIDE_INVENTORY_OVERRIDE:\n"
            + "\n".join(map(str, found))
        )
    if REQUIRE_LANDSLIDE_INVENTORY:
        raise FileNotFoundError(
            "No landslide inventory found. Set LANDSLIDE_INVENTORY_OVERRIDE."
        )
    return None

# =============================================================================
# RASTER I/O
# =============================================================================
def read_anchor(path: Path) -> dict[str, Any]:
    require_file(path, "anchor raster")
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError("Anchor raster has no CRS.")
        tf = ds.transform
        cell_area = abs(tf.a * tf.e - tf.b * tf.d)
        return {
            "crs": ds.crs,
            "transform": tf,
            "width": ds.width,
            "height": ds.height,
            "bounds": ds.bounds,
            "profile": ds.profile.copy(),
            "epsg": ds.crs.to_epsg(),
            "cell_area_m2": float(cell_area),
        }


def same_grid(ds, anchor: dict[str, Any]) -> bool:
    return (
        ds.crs == anchor["crs"]
        and ds.transform == anchor["transform"]
        and ds.width == anchor["width"]
        and ds.height == anchor["height"]
    )


def clean_nodata(array: np.ndarray, nodata) -> np.ndarray:
    out = array.astype("float64", copy=True)
    if nodata is not None:
        try:
            nd = float(nodata)
            if np.isfinite(nd):
                out[out == nd] = np.nan
        except (TypeError, ValueError):
            pass
    out[out == -9999.0] = np.nan
    out[out == -32768.0] = np.nan
    out[np.abs(out) > 1.0e20] = np.nan
    return out


def read_continuous_exact(path: Path, anchor: dict[str, Any], description: str) -> np.ndarray:
    require_file(path, description)
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")
        if REQUIRE_EXACT_GRID and not same_grid(ds, anchor):
            raise ValueError(
                f"{description} does not match the anchor grid. Repair upstream: {path}"
            )
        return clean_nodata(ds.read(1), ds.nodata)


def read_mask_exact(path: Path, anchor: dict[str, Any], description: str) -> np.ndarray:
    require_file(path, description)
    with rasterio.open(path) as ds:
        if REQUIRE_EXACT_GRID and not same_grid(ds, anchor):
            raise ValueError(f"{description} does not match anchor grid: {path}")
        raw = ds.read(1)
        values = raw.astype("float64")
        if ds.nodata is not None:
            values[raw == ds.nodata] = 0.0
        values[~np.isfinite(values)] = 0.0
        return values > 0.5


def validate_G(G: np.ndarray, study: np.ndarray) -> np.ndarray:
    finite = study & np.isfinite(G)
    low = finite & (G < -G_TOL)
    high = finite & (G > 1.0 + G_TOL)
    if low.any() or high.any():
        raise ValueError(
            f"G outside [0,1]: below={int(low.sum())}, above={int(high.sum())}"
        )
    out = G.copy()
    out[finite] = np.clip(out[finite], 0.0, 1.0)
    return out


def write_byte(path: Path, values: np.ndarray, anchor: dict[str, Any], domain: np.ndarray, description: str) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff", count=1, dtype="uint8", nodata=NODATA_BYTE,
        compress="LZW", tiled=True, bigtiff="IF_NEEDED",
        transform=anchor["transform"], crs=anchor["crs"],
        width=anchor["width"], height=anchor["height"],
    )
    out = np.full(values.shape, NODATA_BYTE, dtype="uint8")
    out[domain] = values[domain].astype("uint8")
    tmp = Path(str(path) + ".part")
    with rasterio.open(tmp, "w", **profile) as ds:
        ds.write(out, 1)
        ds.set_band_description(1, description)
    os.replace(tmp, path)

# =============================================================================
# CORE ANALYSIS
# =============================================================================
def stats(values: np.ndarray) -> dict[str, float | int]:
    values = values[np.isfinite(values)].astype("float64")
    if values.size == 0:
        return {"n": 0}
    return {
        "n": int(values.size),
        "min": float(values.min()),
        "p05": float(np.percentile(values, 5)),
        "p10": float(np.percentile(values, 10)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p90": float(np.percentile(values, 90)),
        "p95": float(np.percentile(values, 95)),
        "max": float(values.max()),
        "fraction_G_lt_0p05": float(np.mean(values < 0.05)),
        "fraction_G_lt_0p10": float(np.mean(values < 0.10)),
        "fraction_G_lt_0p20": float(np.mean(values < 0.20)),
        "fraction_G_ge_0p50": float(np.mean(values >= 0.50)),
    }


def classify_G(G: np.ndarray, domain: np.ndarray) -> np.ndarray:
    classes = np.zeros(G.shape, dtype="uint8")
    classes[domain & (G < 0.05)] = 1
    classes[domain & (G >= 0.05) & (G < 0.10)] = 2
    classes[domain & (G >= 0.10) & (G < 0.20)] = 3
    classes[domain & (G >= 0.20) & (G < 0.50)] = 4
    classes[domain & (G >= 0.50)] = 5
    if np.any(domain & (classes == 0)):
        raise RuntimeError("Some slope-valid G cells were not classified.")
    return classes


def km2(pixel_count: int, cell_area_m2: float) -> float:
    return pixel_count * cell_area_m2 / 1.0e6


def area_tables(G, study, slope_valid, classes, cell_area_m2):
    n_study = int(study.sum())
    n_valid = int(slope_valid.sum())
    n_finite = int((study & np.isfinite(G)).sum())
    n_flat = int((study & np.isfinite(G) & ~slope_valid).sum())

    area_df = pd.DataFrame([
        ["Full study mask", n_study, km2(n_study, cell_area_m2), 100.0],
        ["Finite G in study mask", n_finite, km2(n_finite, cell_area_m2), 100*n_finite/max(n_study,1)],
        [f"Flat terrain, slope < {FLAT_SLOPE_THRESHOLD_DEG:g}°", n_flat, km2(n_flat, cell_area_m2), 100*n_flat/max(n_study,1)],
        [f"Slope-valid domain, slope ≥ {FLAT_SLOPE_THRESHOLD_DEG:g}°", n_valid, km2(n_valid, cell_area_m2), 100*n_valid/max(n_study,1)],
    ], columns=["domain", "pixel_count", "area_km2", "percent_of_study"])

    threshold_rows = []
    previous = -1
    for threshold in THRESHOLDS:
        count = int((slope_valid & (G < threshold)).sum())
        if count < previous:
            raise RuntimeError("Threshold counts are not monotonic.")
        previous = count
        if math.isclose(threshold, 0.05):
            interpretation = "Primary near-blind/capped domain"
        elif math.isclose(threshold, 0.20):
            interpretation = "Broader low-observability domain"
        else:
            interpretation = "Threshold sensitivity"
        threshold_rows.append({
            "threshold_G_lt": threshold,
            "interpretation": interpretation,
            "pixel_count": count,
            "area_km2": km2(count, cell_area_m2),
            "percent_of_slope_valid_area": 100*count/max(n_valid,1),
            "percent_of_total_study_area": 100*count/max(n_study,1),
            "uncapped_amplification_gt": 1.0/threshold,
        })
    threshold_df = pd.DataFrame(threshold_rows)

    class_rows = []
    for code, label in enumerate(CLASS_LABELS, start=1):
        count = int((slope_valid & (classes == code)).sum())
        class_rows.append({
            "class_code": code,
            "class_label": label,
            "pixel_count": count,
            "area_km2": km2(count, cell_area_m2),
            "percent_of_slope_valid_area": 100*count/max(n_valid,1),
            "percent_of_total_study_area": 100*count/max(n_study,1),
        })
    class_df = pd.DataFrame(class_rows)
    if int(class_df.pixel_count.sum()) != n_valid:
        raise RuntimeError("Class totals do not equal slope-valid domain.")
    return area_df, threshold_df, class_df


def cdf_table(values: np.ndarray) -> pd.DataFrame:
    values = np.sort(values[np.isfinite(values)])
    unique, counts = np.unique(values, return_counts=True)
    cumulative = 100*np.cumsum(counts)/values.size
    return pd.DataFrame({"G": unique, "cumulative_percent_of_slope_valid_area": cumulative})


def aspect_code(aspect: np.ndarray) -> np.ndarray:
    return np.floor(((np.mod(aspect, 360.0) + 22.5) % 360.0) / 45.0).astype("int16")


def aspect_and_slope_tables(G, slope, aspect, slope_valid, cell_area_m2):
    valid = slope_valid & np.isfinite(slope) & np.isfinite(aspect)
    a_code = np.full(G.shape, -1, dtype="int16")
    a_code[valid] = aspect_code(aspect[valid])
    rows = []
    for code, label in enumerate(ASPECT_LABELS):
        mask = valid & (a_code == code)
        s = stats(G[mask])
        count = int(mask.sum())
        rows.append({
            "aspect_class": label,
            "pixel_count": count,
            "area_km2": km2(count, cell_area_m2),
            "mean_G": s.get("mean"),
            "median_G": s.get("median"),
            "percent_G_lt_0p05": 100*s.get("fraction_G_lt_0p05", np.nan),
            "percent_G_lt_0p20": 100*s.get("fraction_G_lt_0p20", np.nan),
            "percent_G_ge_0p50": 100*s.get("fraction_G_ge_0p50", np.nan),
        })
    aspect_df = pd.DataFrame(rows)

    s_code = np.full(G.shape, -1, dtype="int16")
    for i in range(len(SLOPE_LABELS)):
        s_code[valid & (slope >= SLOPE_EDGES[i]) & (slope < SLOPE_EDGES[i+1])] = i

    low = np.full((len(SLOPE_LABELS), len(ASPECT_LABELS)), np.nan)
    median = np.full_like(low, np.nan)
    count = np.zeros_like(low, dtype="int64")
    for i in range(len(SLOPE_LABELS)):
        for j in range(len(ASPECT_LABELS)):
            values = G[valid & (s_code == i) & (a_code == j)]
            count[i, j] = values.size
            if values.size:
                low[i, j] = 100*np.mean(values < 0.20)
                median[i, j] = np.median(values)
    return (
        aspect_df,
        pd.DataFrame(low, index=SLOPE_LABELS, columns=ASPECT_LABELS),
        pd.DataFrame(median, index=SLOPE_LABELS, columns=ASPECT_LABELS),
        pd.DataFrame(count, index=SLOPE_LABELS, columns=ASPECT_LABELS),
    )

# =============================================================================
# PS AND LANDSLIDE REPRESENTATIVENESS
# =============================================================================
def sample_PS(csv_path: Path | None, G, slope_valid, anchor) -> np.ndarray:
    if csv_path is None:
        return np.array([], dtype="float64")
    df = pd.read_csv(csv_path)
    lon = next((c for c in df.columns if "lon" in c.lower()), None)
    lat = next((c for c in df.columns if "lat" in c.lower()), None)
    if lon is None or lat is None:
        raise ValueError("PS CSV must contain longitude and latitude columns.")
    lonv = pd.to_numeric(df[lon], errors="coerce").to_numpy(float)
    latv = pd.to_numeric(df[lat], errors="coerce").to_numpy(float)
    finite = np.isfinite(lonv) & np.isfinite(latv)
    lonv, latv = lonv[finite], latv[finite]
    if anchor["crs"].is_geographic:
        x, y = lonv, latv
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError("pyproj is required for PS coordinate transformation.")
        x, y = Transformer.from_crs("EPSG:4326", anchor["crs"], always_xy=True).transform(lonv, latv)
    rows, cols = rowcol(anchor["transform"], x, y, op=np.floor)
    rows, cols = np.asarray(rows, int), np.asarray(cols, int)
    inside = (rows >= 0) & (cols >= 0) & (rows < anchor["height"]) & (cols < anchor["width"])
    rows, cols = rows[inside], cols[inside]
    good = slope_valid[rows, cols]
    values = G[rows[good], cols[good]]
    return values[np.isfinite(values)]


def _fiona_feature_to_mapping(feature) -> dict:
    """
    Convert a Fiona Feature model object into a plain GeoJSON-like mapping.
    This avoids GeoPandas' file-reading wrapper and the removed fiona.path
    attribute.
    """
    geometry = feature["geometry"]

    if geometry is None:
        geometry_mapping = None
    elif hasattr(geometry, "__geo_interface__"):
        geometry_mapping = geometry.__geo_interface__
    else:
        geometry_mapping = dict(geometry)

    properties = feature["properties"]
    property_mapping = (
        {} if properties is None else dict(properties)
    )

    try:
        feature_id = feature["id"]
    except Exception:
        feature_id = getattr(feature, "id", None)

    return {
        "type": "Feature",
        "id": None if feature_id is None else str(feature_id),
        "properties": property_mapping,
        "geometry": geometry_mapping,
    }


def read_vector_compat(
    path: Path,
    target_crs,
    description: str,
):
    """
    Read a vector dataset with a Fiona/GeoPandas compatibility fallback.

    Reading order:
      1. pyogrio.read_dataframe(), when Pyogrio is installed;
      2. fiona.open() + GeoDataFrame.from_features().

    The incompatible GeoPandas file-reading route is intentionally not used.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}"
        )

    if not HAVE_GEOPANDAS:
        raise RuntimeError(
            "GeoPandas and Shapely are required for vector analysis."
        )

    errors = []
    gdf = None
    engine_used = None

    if HAVE_PYOGRIO:
        try:
            gdf = pyogrio.read_dataframe(str(path))
            engine_used = "pyogrio.read_dataframe"
        except Exception as exc:
            errors.append(
                f"Pyogrio failed: {type(exc).__name__}: {exc}"
            )
            gdf = None

    if gdf is None and HAVE_FIONA:
        try:
            features = []

            with fiona.open(str(path), mode="r") as collection:
                source_crs = (
                    collection.crs_wkt
                    if getattr(collection, "crs_wkt", None)
                    else collection.crs
                )

                for feature in collection:
                    converted = _fiona_feature_to_mapping(feature)
                    if converted["geometry"] is not None:
                        features.append(converted)

            if not features:
                raise RuntimeError(
                    f"{description} contains no readable geometries."
                )

            gdf = gpd.GeoDataFrame.from_features(
                features,
                crs=source_crs,
            )
            engine_used = (
                "fiona.open + GeoDataFrame.from_features"
            )

        except Exception as exc:
            errors.append(
                f"Direct Fiona failed: "
                f"{type(exc).__name__}: {exc}"
            )
            gdf = None

    if gdf is None:
        geopandas_version = getattr(
            gpd,
            "__version__",
            "unknown",
        )
        fiona_version = (
            getattr(fiona, "__version__", "unknown")
            if HAVE_FIONA
            else "not installed"
        )
        pyogrio_version = (
            getattr(pyogrio, "__version__", "unknown")
            if HAVE_PYOGRIO
            else "not installed"
        )
        details = (
            "\n".join(errors)
            if errors
            else "No compatible vector-reading engine was available."
        )

        raise RuntimeError(
            f"Could not read {description}: {path}\n"
            f"GeoPandas={geopandas_version}, "
            f"Fiona={fiona_version}, "
            f"Pyogrio={pyogrio_version}\n"
            f"{details}\n"
            "The script deliberately avoided the incompatible "
            "GeoPandas/Fiona wrapper."
        )

    if gdf.crs is None:
        raise ValueError(
            f"{description} has no CRS. Assign its correct CRS "
            f"before running Step 5.1b: {path}"
        )

    if target_crs is not None:
        gdf = gdf.to_crs(target_crs)

    gdf = gdf[
        gdf.geometry.notna()
        & (~gdf.geometry.is_empty)
    ].copy()

    if gdf.empty:
        raise RuntimeError(
            f"{description} contains no valid geometries."
        )

    log(
        f"[VECTOR] {description}: {len(gdf):,} features "
        f"using {engine_used}"
    )

    return gdf


def load_inventory(path: Path, target_crs):
    gdf = read_vector_compat(
        path=path,
        target_crs=target_crs,
        description="landslide inventory",
    )

    invalid = ~gdf.geometry.is_valid
    invalid_count = int(invalid.sum())

    if invalid_count:
        log(
            f"[VECTOR] Repairing {invalid_count:,} invalid "
            "inventory geometries using buffer(0)."
        )
        gdf.loc[invalid, "geometry"] = (
            gdf.loc[invalid, "geometry"].buffer(0)
        )

    still_invalid = (
        gdf.geometry.notna()
        & (~gdf.geometry.is_empty)
        & (~gdf.geometry.is_valid)
    )

    if still_invalid.any():
        raise RuntimeError(
            f"{int(still_invalid.sum()):,} inventory geometries "
            "remain invalid after repair."
        )

    return gdf


def inventory_id_field(gdf) -> str | None:
    candidates = ["landslide_id", "landslideid", "ls_id", "site_id", "id", "fid", "objectid"]
    lower = {c.lower(): c for c in gdf.columns if c != gdf.geometry.name}
    return next((lower[c] for c in candidates if c in lower), None)


def clipped_window(geom, anchor):
    """
    Return an integer, raster-clipped window for a geometry.

    The explicit integer conversion avoids zero/float window dimensions
    across different Rasterio versions.
    """
    raster_bounds = anchor["bounds"]
    left, bottom, right, top = geom.bounds

    left = max(float(left), float(raster_bounds.left))
    bottom = max(float(bottom), float(raster_bounds.bottom))
    right = min(float(right), float(raster_bounds.right))
    top = min(float(top), float(raster_bounds.top))

    if left >= right or bottom >= top:
        return None

    raw_window = from_bounds(
        left,
        bottom,
        right,
        top,
        transform=anchor["transform"],
    )

    col_start = max(0, int(np.floor(raw_window.col_off)))
    row_start = max(0, int(np.floor(raw_window.row_off)))
    col_stop = min(
        int(anchor["width"]),
        int(np.ceil(raw_window.col_off + raw_window.width)),
    )
    row_stop = min(
        int(anchor["height"]),
        int(np.ceil(raw_window.row_off + raw_window.height)),
    )

    width = col_stop - col_start
    height = row_stop - row_start

    if width <= 0 or height <= 0:
        return None

    return Window(
        col_off=col_start,
        row_off=row_start,
        width=width,
        height=height,
    )


def scalar_row_col(transform, x_value, y_value) -> tuple[int, int]:
    """
    Convert one coordinate pair to integer raster indices.

    Rasterio may return NumPy floating scalars when rowcol() is supplied
    with op=np.floor. NumPy arrays cannot be indexed by those floating
    scalars. Using the default floor operation and explicitly converting
    each scalar to Python int makes the result version-independent.
    """
    row_value, col_value = rowcol(
        transform,
        float(x_value),
        float(y_value),
    )

    row_index = int(np.asarray(row_value).item())
    col_index = int(np.asarray(col_value).item())

    return row_index, col_index


def landslide_zonal(gdf, G, slope_valid, anchor):
    """
    Calculate landslide-level geometric-observability statistics.

    Polygon cells are used first. When a polygon is smaller than one raster
    cell or contains no slope-valid cell centre, a representative interior
    point is sampled as a documented fallback.
    """
    if G.shape != slope_valid.shape:
        raise ValueError(
            "G and slope_valid must have identical array dimensions."
        )

    expected_shape = (
        int(anchor["height"]),
        int(anchor["width"]),
    )
    if G.shape != expected_shape:
        raise ValueError(
            f"Raster array shape {G.shape} does not match anchor "
            f"shape {expected_shape}."
        )

    id_field = inventory_id_field(gdf)
    records = []
    rasterize_shapes = []

    for feature_number, (idx, row) in enumerate(
        gdf.iterrows(),
        start=1,
    ):
        geom = row.geometry

        if geom is None or geom.is_empty:
            continue

        rasterize_shapes.append(
            (geom.__geo_interface__, 1)
        )

        landslide_id = (
            row[id_field]
            if id_field is not None
            else idx
        )

        values = np.array([], dtype="float64")
        sampling_method = "polygon_cells"
        sampled_pixel_count = 0

        window = clipped_window(geom, anchor)

        if window is not None:
            row_start = int(window.row_off)
            col_start = int(window.col_off)
            row_stop = row_start + int(window.height)
            col_stop = col_start + int(window.width)

            local_shape = (
                int(window.height),
                int(window.width),
            )

            if local_shape[0] > 0 and local_shape[1] > 0:
                local_transform = rasterio.windows.transform(
                    window,
                    anchor["transform"],
                )

                inside_polygon = geometry_mask(
                    [geom.__geo_interface__],
                    out_shape=local_shape,
                    transform=local_transform,
                    invert=True,
                    all_touched=False,
                )

                local_G = G[
                    row_start:row_stop,
                    col_start:col_stop,
                ]
                local_valid = slope_valid[
                    row_start:row_stop,
                    col_start:col_stop,
                ]

                if (
                    inside_polygon.shape != local_G.shape
                    or local_G.shape != local_valid.shape
                ):
                    raise RuntimeError(
                        "Local polygon mask and raster windows have "
                        "inconsistent dimensions."
                    )

                selected = (
                    inside_polygon
                    & local_valid
                    & np.isfinite(local_G)
                )

                values = local_G[selected].astype(
                    "float64",
                    copy=False,
                )
                sampled_pixel_count = int(selected.sum())

        if values.size == 0:
            point = geom.representative_point()
            row_index, col_index = scalar_row_col(
                anchor["transform"],
                point.x,
                point.y,
            )

            inside_raster = (
                0 <= row_index < int(anchor["height"])
                and 0 <= col_index < int(anchor["width"])
            )

            if (
                inside_raster
                and bool(slope_valid[row_index, col_index])
                and np.isfinite(G[row_index, col_index])
            ):
                values = np.array(
                    [float(G[row_index, col_index])],
                    dtype="float64",
                )
                sampled_pixel_count = 1
                sampling_method = "representative_point"
            else:
                sampling_method = "no_slope_valid_G"

        statistics = stats(values)

        records.append(
            {
                "landslide_id": str(landslide_id),
                "feature_number": int(feature_number),
                "geometry_area_m2": float(geom.area),
                "sampling_method": sampling_method,
                "sampled_slope_valid_pixels": int(
                    sampled_pixel_count
                ),
                "mean_G": statistics.get("mean"),
                "median_G": statistics.get("median"),
                "min_G": statistics.get("min"),
                "p10_G": statistics.get("p10"),
                "p90_G": statistics.get("p90"),
                "max_G": statistics.get("max"),
                "fraction_G_lt_0p05": statistics.get(
                    "fraction_G_lt_0p05"
                ),
                "fraction_G_lt_0p10": statistics.get(
                    "fraction_G_lt_0p10"
                ),
                "fraction_G_lt_0p20": statistics.get(
                    "fraction_G_lt_0p20"
                ),
                "fraction_G_ge_0p50": statistics.get(
                    "fraction_G_ge_0p50"
                ),
            }
        )

    zonal = pd.DataFrame(records)

    if rasterize_shapes:
        inventory_mask = rasterize(
            rasterize_shapes,
            out_shape=expected_shape,
            transform=anchor["transform"],
            fill=0,
            all_touched=False,
            dtype="uint8",
        ).astype(bool)
    else:
        inventory_mask = np.zeros(
            expected_shape,
            dtype=bool,
        )

    inventory_mask &= slope_valid

    return zonal, inventory_mask


def summarize_landslides(zonal: pd.DataFrame) -> pd.DataFrame:
    valid = zonal[pd.to_numeric(zonal.get("median_G"), errors="coerce").notna()].copy()
    if valid.empty:
        return pd.DataFrame([{"metric": "landslides_with_valid_G", "value": 0}])
    median_G = pd.to_numeric(valid["median_G"], errors="coerce").to_numpy(float)
    low_fraction = pd.to_numeric(valid["fraction_G_lt_0p20"], errors="coerce").to_numpy(float)
    rows = [
        ["landslides_total_in_inventory", len(zonal)],
        ["landslides_with_valid_G", len(valid)],
        ["landslides_with_median_G_lt_0p05", int(np.sum(median_G < 0.05))],
        ["percent_landslides_with_median_G_lt_0p05", 100*np.mean(median_G < 0.05)],
        ["landslides_with_median_G_lt_0p20", int(np.sum(median_G < 0.20))],
        ["percent_landslides_with_median_G_lt_0p20", 100*np.mean(median_G < 0.20)],
        ["landslides_with_more_than_half_area_G_lt_0p20", int(np.sum(low_fraction > 0.50))],
        ["percent_landslides_with_more_than_half_area_G_lt_0p20", 100*np.mean(low_fraction > 0.50)],
        ["landslides_with_median_G_ge_0p50", int(np.sum(median_G >= 0.50))],
        ["percent_landslides_with_median_G_ge_0p50", 100*np.mean(median_G >= 0.50)],
        ["median_of_landslide_median_G", np.median(median_G)],
    ]
    return pd.DataFrame(rows, columns=["metric", "value"])


def domain_comparison(study_values, ps_values, landslide_values):
    rows = []
    for name, values in [
        ("Slope-valid study area", study_values),
        ("PS locations", ps_values),
        ("Landslide polygon pixels", landslide_values),
    ]:
        s = stats(values)
        rows.append({
            "domain": name, "n": s.get("n", 0),
            "mean_G": s.get("mean"), "median_G": s.get("median"),
            "percent_G_lt_0p05": 100*s.get("fraction_G_lt_0p05", np.nan),
            "percent_G_lt_0p20": 100*s.get("fraction_G_lt_0p20", np.nan),
            "percent_G_ge_0p50": 100*s.get("fraction_G_ge_0p50", np.nan),
        })
    return pd.DataFrame(rows)

# =============================================================================
# PLOTTING
# =============================================================================
# All map panels use the same final cartographic style as:
#   "Geometric Observability Classes Map InMap Legend Final"
#
# Required map style:
#   - standalone maps: 2 x 4 inches;
#   - four longitude DMS labels, rotated 45 degrees;
#   - three latitude DMS labels, rotated 90 degrees;
#   - bottom-most latitude label lifted above its tick to prevent overlap;
#   - no explicit "Longitude" or "Latitude" axis titles;
#   - same north arrow, scale bar, road and study-area styling;
#   - bold, fully opaque "km" scale-bar label;
#   - compact in-map class-threshold legend for observability-class maps.
# =============================================================================
def dms(value: float, lon: bool) -> str:
    absolute = abs(float(value))
    degrees = int(absolute)
    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0
    hemisphere = ("E" if value >= 0 else "W") if lon else ("N" if value >= 0 else "S")
    return f"{degrees}°{minutes:02d}'{seconds:04.1f}\"{hemisphere}"


def add_north(axis, size: float = 0.09, pad: float = 0.09, compact: bool = False) -> None:
    x0 = pad
    y0 = 1 - pad - size - 0.025
    x1 = x0 + size * 0.72
    y1 = y0 + size
    apex = ((x0 + x1) / 2, y1)
    tail = ((x0 + x1) / 2, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, (x0, y0), tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, (x1, y0)],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1,
            zorder=20,
        )
    )
    axis.text(
        pad + size * 0.36,
        1 - pad + size * 0.10,
        "N",
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=6.5 if compact else 8,
        fontweight="bold",
        zorder=21,
    )


def nice_length(target: float) -> float:
    exponent = int(np.floor(np.log10(max(target, 1.0))))
    base = 10 ** exponent
    for multiplier in (1, 2, 5, 10):
        if multiplier * base >= target:
            return multiplier * base
    return 10 * base


def add_scale(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    projected: bool,
    mid_lat: float | None = None,
    compact: bool = False,
) -> None:
    width = east - west
    height = north - south

    if projected:
        width_m = width
        units_per_m = 1.0
    else:
        reference_latitude = mid_lat if mid_lat is not None else (south + north) / 2
        metres_per_degree = 111320 * np.cos(np.radians(reference_latitude))
        width_m = width * metres_per_degree
        units_per_m = 1.0 / metres_per_degree

    total_m = nice_length(width_m / 5)
    total_u = total_m * units_per_m
    segment_u = total_u / 4

    x0 = west + width / 2 - total_u / 2
    y0 = south + 0.06 * height
    bar_height = 0.018 * height

    axis.add_patch(
        Rectangle(
            (x0 - total_u * 0.08, y0 - 0.035 * height),
            total_u * 1.16,
            bar_height + 0.07 * height,
            facecolor="white",
            edgecolor="none",
            alpha=0.7,
            zorder=12,
        )
    )

    for index in range(4):
        axis.add_patch(
            Rectangle(
                (x0 + index * segment_u, y0),
                segment_u,
                bar_height,
                facecolor="black" if index % 2 == 0 else "white",
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x0, y0),
            total_u,
            bar_height,
            fill=False,
            edgecolor="black",
            linewidth=0.4,
            zorder=14,
        )
    )

    unit_fontsize = 3.5 if compact else 4.5
    value_fontsize = 3.1 if compact else 4.0

    axis.text(
        x0 + total_u / 2,
        y0 + bar_height + 0.012 * height,
        "km",
        ha="center",
        va="bottom",
        fontsize=unit_fontsize,
        fontweight="bold",
        color="black",
        alpha=1.0,
        zorder=16,
    )
    axis.text(
        x0,
        y0 - 0.012 * height,
        "0",
        ha="center",
        va="top",
        fontsize=value_fontsize,
        zorder=16,
    )

    end_km = total_m / 1000
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - round(end_km)) < 1.0e-6
        else f"{end_km:.1f}"
    )
    axis.text(
        x0 + total_u,
        y0 - 0.012 * height,
        end_label,
        ha="center",
        va="top",
        fontsize=value_fontsize,
        zorder=16,
    )


def load_vector(path: Path, crs):
    """
    Read optional map overlays with the compatibility reader.
    Overlay failure does not stop the numerical audit.
    """
    if not path.exists():
        log(f"[WARN] Optional vector not found: {path}")
        return None

    try:
        return read_vector_compat(
            path=path,
            target_crs=crs,
            description=path.stem,
        )
    except Exception as exc:
        log(f"[WARN] Could not load {path.name}: {exc}")
        return None


def extent(anchor):
    bounds = anchor["bounds"]
    dx = (bounds.right - bounds.left) * DISPLAY_BUFFER_FRACTION
    dy = (bounds.top - bounds.bottom) * DISPLAY_BUFFER_FRACTION
    return (
        [bounds.left, bounds.right, bounds.bottom, bounds.top],
        bounds.left - dx,
        bounds.right + dx,
        bounds.bottom - dy,
        bounds.top + dy,
    )


def configure_axes(
    axis,
    anchor,
    west: float,
    east: float,
    south: float,
    north: float,
    compact: bool = False,
) -> None:
    """
    Apply the final map-coordinate style consistently to every map axis.

    The longitude and latitude coordinate values remain visible as DMS tick
    labels, but the words "Longitude" and "Latitude" are intentionally omitted.
    """
    xmid = (anchor["bounds"].left + anchor["bounds"].right) / 2
    ymid = (anchor["bounds"].bottom + anchor["bounds"].top) / 2

    if anchor["crs"].is_geographic:
        longitude_formatter = lambda value, position: dms(value, True)
        latitude_formatter = lambda value, position: dms(value, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError("pyproj is required for DMS coordinate labels.")

        transformer = Transformer.from_crs(
            anchor["crs"],
            "EPSG:4326",
            always_xy=True,
        )
        longitude_formatter = lambda value, position: dms(
            transformer.transform(value, ymid)[0],
            True,
        )
        latitude_formatter = lambda value, position: dms(
            transformer.transform(xmid, value)[1],
            False,
        )

    axis.set_xlim(west, east)
    axis.set_ylim(south, north)

    # Exactly four longitude labels and three latitude labels.
    axis.xaxis.set_major_locator(LinearLocator(LONGITUDE_DMS_TICK_COUNT))
    axis.xaxis.set_major_formatter(FuncFormatter(longitude_formatter))
    axis.yaxis.set_major_locator(LinearLocator(LATITUDE_DMS_TICK_COUNT))
    axis.yaxis.set_major_formatter(FuncFormatter(latitude_formatter))

    tick_fontsize = 4 if compact else DMS_TICK_FONTSIZE
    axis.tick_params(axis="both", labelsize=tick_fontsize)

    # Longitude DMS labels: 45-degree rotation and right alignment.
    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")
        label.set_fontsize(tick_fontsize)

    # Latitude DMS labels: 90-degree rotation.
    latitude_labels = axis.get_yticklabels()
    for label in latitude_labels:
        label.set_rotation(90)
        label.set_va("center")
        label.set_fontsize(tick_fontsize)

    # Lift the bottom-most latitude label above its tick to avoid collision
    # with the first 45-degree longitude label at the lower-left corner.
    if latitude_labels:
        latitude_labels[0].set_va("bottom")

    # Intentionally suppress the literal axis-title words.
    axis.set_xlabel("")
    axis.set_ylabel("")
    axis.xaxis.label.set_visible(False)
    axis.yaxis.label.set_visible(False)


def plot_map(
    axis,
    array,
    anchor,
    roads,
    boundary,
    cmap,
    norm=None,
    vmin=None,
    vmax=None,
    compact: bool = False,
):
    ext, west, east, south, north = extent(anchor)

    image = axis.imshow(
        np.ma.masked_invalid(array),
        extent=ext,
        origin="upper",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
        vmin=vmin,
        vmax=vmax,
        zorder=1,
    )

    if roads is not None:
        roads.plot(
            ax=axis,
            color="red",
            linewidth=0.7,
            zorder=850,
        )

    if boundary is not None:
        boundary.boundary.plot(
            ax=axis,
            color="black",
            linewidth=0.7,
            zorder=999,
        )

    configure_axes(
        axis,
        anchor,
        west,
        east,
        south,
        north,
        compact=compact,
    )
    add_north(
        axis,
        size=0.075 if compact else 0.09,
        pad=0.08 if compact else 0.09,
        compact=compact,
    )
    add_scale(
        axis,
        west,
        east,
        south,
        north,
        not anchor["crs"].is_geographic,
        (south + north) / 2 if anchor["crs"].is_geographic else None,
        compact=compact,
    )
    return image


def add_map_feature_legend(axis, compact: bool = False):
    """Add the same compact Road/Study Area legend used by the final class map."""
    handles = [
        Line2D([0], [0], color="red", linewidth=1.0, label="Road"),
        Line2D([0], [0], color="black", linewidth=1.0, label="Study Area"),
    ]

    legend = axis.legend(
        handles=handles,
        loc="lower right",
        bbox_to_anchor=(0.98, 0.09),
        bbox_transform=axis.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=2.0 if compact else 2.5,
        labelspacing=0.25 if compact else 0.30,
        borderpad=0.25 if compact else 0.30,
        handlelength=1.0 if compact else 1.2,
        handletextpad=0.30 if compact else 0.40,
        borderaxespad=0.0,
    )
    legend.get_frame().set_edgecolor("lightgray")
    legend.get_frame().set_linewidth(0.3)
    legend.set_zorder(1001)
    return legend


def add_class_threshold_legend(axis, compact: bool = False):
    """Add the five class thresholds inside an observability-class map."""
    handles = [
        Patch(
            facecolor=CLASS_COLOURS[0],
            edgecolor="black",
            linewidth=0.3,
            label=r"Near-blind: $G<0.05$",
        ),
        Patch(
            facecolor=CLASS_COLOURS[1],
            edgecolor="black",
            linewidth=0.3,
            label=r"Very low: $0.05\leq G<0.10$",
        ),
        Patch(
            facecolor=CLASS_COLOURS[2],
            edgecolor="black",
            linewidth=0.3,
            label=r"Low: $0.10\leq G<0.20$",
        ),
        Patch(
            facecolor=CLASS_COLOURS[3],
            edgecolor="black",
            linewidth=0.3,
            label=r"Moderate: $0.20\leq G<0.50$",
        ),
        Patch(
            facecolor=CLASS_COLOURS[4],
            edgecolor="black",
            linewidth=0.3,
            label=r"High: $G\geq0.50$",
        ),
    ]

    legend = axis.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=(0.03, 0.67),
        fontsize=1.75 if compact else 2.25,
        title="Class thresholds",
        title_fontsize=1.95 if compact else 2.45,
        frameon=True,
        framealpha=0.92,
        facecolor="white",
        edgecolor="0.6",
        borderpad=0.20 if compact else 0.25,
        labelspacing=0.15 if compact else 0.20,
        handlelength=0.80 if compact else 0.95,
        handletextpad=0.25 if compact else 0.35,
        borderaxespad=0.0,
    )
    legend.set_zorder(1000)
    return legend


def save_figure(fig, stem: Path, dpi: int = FIG_DPI) -> None:
    if SAVE_FIGURES:
        for suffix in (".png", ".pdf", ".svg"):
            fig.savefig(
                Path(str(stem) + suffix),
                dpi=dpi,
                bbox_inches="tight",
                facecolor="white",
            )

    if SHOW_FIGURES:
        display(fig) if HAVE_IPYTHON_DISPLAY else plt.show()

    plt.close(fig)


def create_figures(
    G,
    classes,
    slope_valid,
    anchor,
    roads,
    boundary,
    cdf_df,
    aspect_df,
    low_df,
    study_values,
    ps_values,
    landslide_values,
):
    # -------------------------------------------------------------------------
    # 1. Continuous geometric-observability map
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=MAP_FIGSIZE, facecolor="white")
    image = plot_map(
        ax,
        np.where(slope_valid, G, np.nan),
        anchor,
        roads,
        boundary,
        CONT_CMAP,
        vmin=0,
        vmax=1,
    )

    divider = make_axes_locatable(ax)
    colorbar_axis = divider.append_axes("right", size="5%", pad=0.01)
    colorbar = fig.colorbar(image, cax=colorbar_axis)
    colorbar.set_label(
        "Geometric observability, G",
        fontsize=4,
        fontweight="bold",
    )
    colorbar.ax.tick_params(labelsize=5)

    add_map_feature_legend(ax)

    fig.tight_layout()
    save_figure(fig, CONT_MAP)

    # -------------------------------------------------------------------------
    # 2. Geometric-observability class map
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=MAP_FIGSIZE, facecolor="white")
    image = plot_map(
        ax,
        np.where(slope_valid, classes.astype(float), np.nan),
        anchor,
        roads,
        boundary,
        CLASS_CMAP,
        norm=CLASS_NORM,
    )

    divider = make_axes_locatable(ax)
    colorbar_axis = divider.append_axes("right", size="5%", pad=0.01)
    colorbar = fig.colorbar(
        image,
        cax=colorbar_axis,
        ticks=np.arange(1, 6),
    )
    colorbar.ax.set_yticklabels(
        ["Near-blind", "Very low", "Low", "Moderate", "High"]
    )
    colorbar.ax.tick_params(labelsize=3.2)
    colorbar.set_label(
        "Observability class",
        fontsize=4,
        fontweight="bold",
    )

    threshold_legend = add_class_threshold_legend(ax)
    ax.add_artist(threshold_legend)
    add_map_feature_legend(ax)

    fig.tight_layout()
    save_figure(fig, CLASS_MAP)

    # -------------------------------------------------------------------------
    # 3. Cumulative area curve
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, facecolor="white")
    ax.plot(
        cdf_df.G,
        cdf_df.cumulative_percent_of_slope_valid_area,
        linewidth=1.2,
    )
    for threshold in (0.05, 0.10, 0.20, 0.50):
        ax.axvline(
            threshold,
            linestyle="--",
            linewidth=0.8,
            label=f"G = {threshold:.2f}",
        )
    ax.set(xlim=(0, 1), ylim=(0, 100))
    ax.set_xlabel(
        "Geometric observability, G",
        fontsize=7,
        fontweight="bold",
    )
    ax.set_ylabel(
        "Cumulative slope-valid area (%)",
        fontsize=7,
        fontweight="bold",
    )
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.25, linewidth=0.4)
    ax.legend(fontsize=5, loc="lower right")
    fig.tight_layout()
    save_figure(fig, CDF_FIG)

    # -------------------------------------------------------------------------
    # 4. Low observability by aspect
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, facecolor="white")
    x = np.arange(len(ASPECT_LABELS))
    ax.bar(
        x,
        aspect_df.percent_G_lt_0p20,
        edgecolor="black",
        linewidth=0.4,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(ASPECT_LABELS)
    ax.set_xlabel(
        "Slope aspect class",
        fontsize=7,
        fontweight="bold",
    )
    ax.set_ylabel(
        "Area with G < 0.20 (%)",
        fontsize=7,
        fontweight="bold",
    )
    ax.tick_params(labelsize=6)
    ax.grid(True, axis="y", alpha=0.25, linewidth=0.4)
    ax.set_axisbelow(True)
    fig.tight_layout()
    save_figure(fig, ASPECT_FIG)

    # -------------------------------------------------------------------------
    # 5. Slope-aspect heatmap
    # -------------------------------------------------------------------------
    matrix = low_df.to_numpy(float)
    fig, ax = plt.subplots(figsize=HEATMAP_FIGSIZE, facecolor="white")
    image = ax.imshow(
        matrix,
        aspect="auto",
        origin="upper",
        interpolation="nearest",
        vmin=0,
        vmax=100,
        cmap=CONT_CMAP,
    )
    ax.set_xticks(np.arange(len(ASPECT_LABELS)))
    ax.set_xticklabels(ASPECT_LABELS)
    ax.set_yticks(np.arange(len(SLOPE_LABELS)))
    ax.set_yticklabels(SLOPE_LABELS)
    ax.set_xlabel(
        "Slope aspect class",
        fontsize=7,
        fontweight="bold",
    )
    ax.set_ylabel(
        "Slope angle class",
        fontsize=7,
        fontweight="bold",
    )
    ax.tick_params(labelsize=6)

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            if np.isfinite(matrix[row_index, column_index]):
                ax.text(
                    column_index,
                    row_index,
                    f"{matrix[row_index, column_index]:.1f}",
                    ha="center",
                    va="center",
                    fontsize=5,
                )

    colorbar = fig.colorbar(image, ax=ax, pad=0.02)
    colorbar.set_label(
        "Area with G < 0.20 (%)",
        fontsize=7,
        fontweight="bold",
    )
    colorbar.ax.tick_params(labelsize=6)
    fig.tight_layout()
    save_figure(fig, SLOPE_ASPECT_FIG)

    # -------------------------------------------------------------------------
    # 6. Domain ECDF comparison
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE, facecolor="white")
    for label, values in [
        ("Slope-valid study area", study_values),
        ("PS locations", ps_values),
        ("Landslide polygon pixels", landslide_values),
    ]:
        values = np.sort(values[np.isfinite(values)])
        if values.size:
            ax.plot(
                values,
                100 * np.arange(1, values.size + 1) / values.size,
                linewidth=1.1,
                label=label,
            )

    ax.axvline(
        0.05,
        linestyle="--",
        linewidth=0.8,
        label="Near-blind threshold",
    )
    ax.axvline(
        0.20,
        linestyle=":",
        linewidth=0.9,
        label="Low-observability threshold",
    )
    ax.set(xlim=(0, 1), ylim=(0, 100))
    ax.set_xlabel(
        "Geometric observability, G",
        fontsize=7,
        fontweight="bold",
    )
    ax.set_ylabel(
        "Cumulative proportion (%)",
        fontsize=7,
        fontweight="bold",
    )
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.25, linewidth=0.4)
    ax.legend(fontsize=4.8, loc="lower right")
    fig.tight_layout()
    save_figure(fig, DOMAIN_ECDF_FIG)

    # -------------------------------------------------------------------------
    # 7. Reviewer four-panel figure
    # -------------------------------------------------------------------------
    fig = plt.figure(figsize=REVIEWER_FIGSIZE, facecolor="white")
    grid = fig.add_gridspec(2, 2, hspace=0.30, wspace=0.28)

    axis_a = fig.add_subplot(grid[0, 0])
    axis_b = fig.add_subplot(grid[0, 1])
    axis_c = fig.add_subplot(grid[1, 0])
    axis_d = fig.add_subplot(grid[1, 1])

    image_a = plot_map(
        axis_a,
        np.where(slope_valid, G, np.nan),
        anchor,
        roads,
        boundary,
        CONT_CMAP,
        vmin=0,
        vmax=1,
        compact=True,
    )
    axis_a.set_title(
        "(a) Continuous geometric observability",
        fontsize=7,
        fontweight="bold",
    )
    colorbar_a = fig.colorbar(
        image_a,
        ax=axis_a,
        fraction=0.04,
        pad=0.02,
    )
    colorbar_a.ax.tick_params(labelsize=5)
    add_map_feature_legend(axis_a, compact=True)

    image_b = plot_map(
        axis_b,
        np.where(slope_valid, classes.astype(float), np.nan),
        anchor,
        roads,
        boundary,
        CLASS_CMAP,
        norm=CLASS_NORM,
        compact=True,
    )
    axis_b.set_title(
        "(b) Geometric observability classes",
        fontsize=7,
        fontweight="bold",
    )
    colorbar_b = fig.colorbar(
        image_b,
        ax=axis_b,
        fraction=0.04,
        pad=0.02,
        ticks=np.arange(1, 6),
    )
    colorbar_b.ax.set_yticklabels(
        ["Near-blind", "Very low", "Low", "Moderate", "High"]
    )
    colorbar_b.ax.tick_params(labelsize=4.5)

    compact_threshold_legend = add_class_threshold_legend(
        axis_b,
        compact=True,
    )
    axis_b.add_artist(compact_threshold_legend)
    add_map_feature_legend(axis_b, compact=True)

    axis_c.plot(
        cdf_df.G,
        cdf_df.cumulative_percent_of_slope_valid_area,
        linewidth=1.2,
    )
    for threshold in (0.05, 0.10, 0.20, 0.50):
        axis_c.axvline(
            threshold,
            linestyle="--",
            linewidth=0.7,
        )
    axis_c.set(xlim=(0, 1), ylim=(0, 100))
    axis_c.set_xlabel("Geometric observability, G", fontsize=6)
    axis_c.set_ylabel("Cumulative area (%)", fontsize=6)
    axis_c.tick_params(labelsize=5)
    axis_c.grid(True, alpha=0.25, linewidth=0.4)
    axis_c.set_title(
        "(c) Threshold sensitivity",
        fontsize=7,
        fontweight="bold",
    )

    axis_d.bar(
        x,
        aspect_df.percent_G_lt_0p20,
        edgecolor="black",
        linewidth=0.4,
    )
    axis_d.set_xticks(x)
    axis_d.set_xticklabels(ASPECT_LABELS)
    axis_d.set_xlabel("Slope aspect class", fontsize=6)
    axis_d.set_ylabel("Area with G < 0.20 (%)", fontsize=6)
    axis_d.tick_params(labelsize=5)
    axis_d.grid(True, axis="y", alpha=0.25, linewidth=0.4)
    axis_d.set_title(
        "(d) Low observability by aspect",
        fontsize=7,
        fontweight="bold",
    )

    save_figure(fig, REVIEWER_FIG, REVIEWER_DPI)

    log(
        "[STYLE] All map panels use four longitude DMS ticks rotated 45°, "
        "three latitude DMS ticks rotated 90°, the bottom latitude label "
        "lifted above its tick, and no Longitude/Latitude axis titles."
    )

# =============================================================================
# REPORTING
# =============================================================================
def threshold_row(df: pd.DataFrame, threshold: float) -> pd.Series:
    return df[np.isclose(df.threshold_G_lt, threshold)].iloc[0]


def write_report(threshold_df, aspect_df, landslide_summary, domain_df, C_qa):
    blind=threshold_row(threshold_df,0.05); low=threshold_row(threshold_df,0.20)
    highest=aspect_df.iloc[aspect_df.percent_G_lt_0p20.astype(float).argmax()]
    metrics=dict(zip(landslide_summary.metric,landslide_summary.value))
    with open(REPORT_TXT,"w",encoding="utf-8") as f:
        f.write("STEP 5.1b GEOMETRIC OBSERVABILITY REVIEWER REPORT\n"+"="*70+"\n\n")
        f.write("C = l_hat · s_hat; G = |C|\n\n")
        f.write("Primary near-blind/capped domain: G < 0.05, corresponding to epsilon = 0.05 and uncapped amplification > 20.\n")
        f.write(f"Flat terrain (slope < {FLAT_SLOPE_THRESHOLD_DEG:g}°) is reported separately.\n\n")
        f.write(f"G < 0.05: {blind.area_km2:.4f} km², {blind.percent_of_slope_valid_area:.2f}% of slope-valid area, {blind.percent_of_total_study_area:.2f}% of total study area.\n")
        f.write(f"G < 0.20: {low.area_km2:.4f} km², {low.percent_of_slope_valid_area:.2f}% of slope-valid area, {low.percent_of_total_study_area:.2f}% of total study area.\n\n")
        f.write(f"Highest low-observability proportion: {highest.aspect_class} aspect ({highest.percent_G_lt_0p20:.2f}%).\n\n")
        f.write(f"Landslides with valid G: {metrics.get('landslides_with_valid_G','NA')}\n")
        f.write(f"Percent with median G < 0.05: {float(metrics.get('percent_landslides_with_median_G_lt_0p05',np.nan)):.2f}%\n")
        f.write(f"Percent with median G < 0.20: {float(metrics.get('percent_landslides_with_median_G_lt_0p20',np.nan)):.2f}%\n")
        f.write(f"Percent with >50% sampled area at G < 0.20: {float(metrics.get('percent_landslides_with_more_than_half_area_G_lt_0p20',np.nan)):.2f}%\n\n")
        f.write("Domain comparison\n-----------------\n"+domain_df.to_string(index=False)+"\n\n")
        f.write("Signed C consistency\n--------------------\n"+json.dumps(C_qa,indent=2)+"\n\n")
        f.write("Limitation: IRS can attenuate low-confidence kinematic evidence, but cannot recover motion intrinsically unobservable from the single ascending viewing geometry.\n")

# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=== Step 5.1b: geometric observability representativeness audit ===")
    log(
        "[VERSIONS] "
        f"GeoPandas={getattr(gpd, '__version__', 'not installed')} | "
        f"Fiona={getattr(fiona, '__version__', 'not installed')} | "
        f"Pyogrio={getattr(pyogrio, '__version__', 'not installed')}"
    )
    anchor=read_anchor(ANCHOR_TIF)
    study=read_mask_exact(STUDY_MASK_TIF,anchor,"study mask")
    G_path=pick_existing(G_TIF_CANDIDATES,"corrected pure G",True)
    slope_path=pick_existing(SLOPE_TIF_CANDIDATES,"slope raster",True)
    aspect_path=pick_existing(ASPECT_TIF_CANDIDATES,"aspect raster",True)
    C_path=pick_existing(SIGNED_C_TIF_CANDIDATES,"signed projection C",False)
    ps_path=pick_existing(PS_POINTS_CSV_CANDIDATES,"PS point CSV",False)
    inventory_path=resolve_inventory()

    G=validate_G(read_continuous_exact(G_path,anchor,"G"),study)
    slope=read_continuous_exact(slope_path,anchor,"slope")
    aspect=read_continuous_exact(aspect_path,anchor,"aspect")
    invalid_slope=study & np.isfinite(slope) & ((slope < -1e-6) | (slope > 90+1e-6))
    if invalid_slope.any(): raise ValueError(f"{int(invalid_slope.sum())} slope cells outside [0,90].")
    slope_valid=study & np.isfinite(G) & np.isfinite(slope) & (slope >= FLAT_SLOPE_THRESHOLD_DEG)
    if not slope_valid.any(): raise RuntimeError("Slope-valid G domain is empty.")
    classes=classify_G(G,slope_valid)

    C_qa={"signed_C_available": C_path is not None}
    if C_path is not None:
        C=read_continuous_exact(C_path,anchor,"signed C")
        mask=slope_valid & np.isfinite(C)
        diff=np.abs(np.abs(C[mask])-G[mask])
        C_qa.update({
            "n_compared": int(diff.size),
            "mean_abs_difference_absC_vs_G": float(np.mean(diff)) if diff.size else None,
            "max_abs_difference_absC_vs_G": float(np.max(diff)) if diff.size else None,
            "tolerance": C_TOL,
            "pass": bool(diff.size and np.max(diff)<=C_TOL),
            "fraction_C_negative": float(np.mean(C[mask]<0)) if mask.any() else None,
            "fraction_C_positive": float(np.mean(C[mask]>0)) if mask.any() else None,
        })
        if diff.size and np.max(diff)>C_TOL:
            raise RuntimeError("G is inconsistent with |signed C|.")

    write_byte(SLOPE_VALID_MASK_TIF,slope_valid.astype("uint8"),anchor,study,"slope_valid_observability_domain")
    write_byte(OBS_CLASS_TIF,classes,anchor,slope_valid,"geometric_observability_classes_1_to_5")
    for path,t,label in [(NEAR_BLIND_MASK_TIF,0.05,"G_lt_0p05"),(VERY_LOW_MASK_TIF,0.10,"G_lt_0p10"),(LOW_OBS_MASK_TIF,0.20,"G_lt_0p20"),(MODERATE_MASK_TIF,0.50,"G_lt_0p50")]:
        mask=slope_valid & (G<t); write_byte(path,mask.astype("uint8"),anchor,study,label)

    area_df,threshold_df,class_df=area_tables(G,study,slope_valid,classes,anchor["cell_area_m2"])
    area_df.to_csv(AREA_STATS_CSV,index=False); threshold_df.to_csv(THRESHOLD_CSV,index=False); class_df.to_csv(CLASS_STATS_CSV,index=False)
    study_values=G[slope_valid]; cdf_df=cdf_table(study_values); cdf_df.to_csv(CDF_DATA_CSV,index=False)

    aspect_df,low_df,median_df,count_df=aspect_and_slope_tables(G,slope,aspect,slope_valid,anchor["cell_area_m2"])
    aspect_df.to_csv(ASPECT_STATS_CSV,index=False); low_df.to_csv(SLOPE_ASPECT_LOW_CSV); median_df.to_csv(SLOPE_ASPECT_MEDIAN_CSV); count_df.to_csv(SLOPE_ASPECT_COUNT_CSV)

    ps_values=sample_PS(ps_path,G,slope_valid,anchor)
    inventory=load_inventory(inventory_path,anchor["crs"])
    zonal,inv_mask=landslide_zonal(inventory,G,slope_valid,anchor)
    zonal.to_csv(LANDSLIDE_ZONAL_CSV,index=False)
    landslide_summary=summarize_landslides(zonal); landslide_summary.to_csv(LANDSLIDE_SUMMARY_CSV,index=False)
    landslide_values=G[inv_mask & np.isfinite(G)]
    domain_df=domain_comparison(study_values,ps_values,landslide_values); domain_df.to_csv(DOMAIN_COMPARISON_CSV,index=False)

    roads=load_vector(ROAD_SHP,anchor["crs"]); boundary=load_vector(STUDY_AREA_SHP,anchor["crs"])
    create_figures(G,classes,slope_valid,anchor,roads,boundary,cdf_df,aspect_df,low_df,study_values,ps_values,landslide_values)

    blind=threshold_row(threshold_df,0.05); low=threshold_row(threshold_df,0.20)
    qa={
        "generated_at_utc": utc_now_iso(), "status": "PASS",
        "grid": {"epsg":anchor["epsg"],"width":anchor["width"],"height":anchor["height"],"cell_area_m2":anchor["cell_area_m2"],"exact_grid_required":REQUIRE_EXACT_GRID},
        "counts": {"study_pixels":int(study.sum()),"finite_G_study_pixels":int((study & np.isfinite(G)).sum()),"slope_valid_pixels":int(slope_valid.sum()),"PS_samples":int(ps_values.size),"landslide_pixels":int(landslide_values.size),"landslide_records":int(len(zonal))},
        "primary_results": {
            "G_lt_0p05_area_km2":float(blind.area_km2),"G_lt_0p05_percent_slope_valid":float(blind.percent_of_slope_valid_area),"G_lt_0p05_percent_total_study":float(blind.percent_of_total_study_area),
            "G_lt_0p20_area_km2":float(low.area_km2),"G_lt_0p20_percent_slope_valid":float(low.percent_of_slope_valid_area),"G_lt_0p20_percent_total_study":float(low.percent_of_total_study_area),
        },
        "G_statistics": {"study_finite":stats(G[study & np.isfinite(G)]),"slope_valid":stats(study_values)},
        "signed_C_consistency":C_qa,
        "checks":{"G_within_0_1":True,"class_totals_equal_domain":True,"threshold_counts_monotonic":True,"flat_terrain_separate":True,"near_blind_threshold_matches_epsilon_0p05":True},
    }
    with open(QA_JSON,"w",encoding="utf-8") as f: json.dump(qa,f,indent=2,ensure_ascii=False)

    metadata={
        "generated_at_utc":utc_now_iso(),"step":"5.1b geometric observability representativeness audit",
        "equations":{"signed_projection":"C = l_hat dot s_hat","geometric_observability":"G = abs(C)","uncapped_amplification":"A_G = 1/G"},
        "interpretation":{"primary_near_blind":"G < 0.05; amplification >20; epsilon=0.05","broader_low_observability":"G < 0.20","flat_policy":f"slope < {FLAT_SLOPE_THRESHOLD_DEG:g}° excluded from blind-zone denominator","limitation":"IRS cannot recover intrinsically unobservable motion"},
        "inputs":{"G":str(G_path),"signed_C":str(C_path) if C_path else None,"slope":str(slope_path),"aspect":str(aspect_path),"PS_points":str(ps_path) if ps_path else None,"landslide_inventory":str(inventory_path)},
        "outputs":{"threshold_table":str(THRESHOLD_CSV),"class_table":str(CLASS_STATS_CSV),"aspect_table":str(ASPECT_STATS_CSV),"landslide_zonal":str(LANDSLIDE_ZONAL_CSV),"domain_comparison":str(DOMAIN_COMPARISON_CSV),"reviewer_figure":str(REVIEWER_FIG)+".png","QA":str(QA_JSON),"report":str(REPORT_TXT)},
    }
    with open(META_JSON,"w",encoding="utf-8") as f: json.dump(metadata,f,indent=2,ensure_ascii=False)
    write_report(threshold_df,aspect_df,landslide_summary,domain_df,C_qa)

    log(f"[RESULT] G < 0.05: {blind.area_km2:.4f} km², {blind.percent_of_slope_valid_area:.2f}% of slope-valid area")
    log(f"[RESULT] G < 0.20: {low.area_km2:.4f} km², {low.percent_of_slope_valid_area:.2f}% of slope-valid area")
    log(f"[OUT] {REVIEWER_FIG}.png")
    log(f"[OUT] {THRESHOLD_CSV}")
    log(f"[OUT] {LANDSLIDE_ZONAL_CSV}")
    log(f"[QA] {QA_JSON}")
    log("=== Step 5.1b completed successfully ===")


def run_step_5_1b() -> None:
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 5.1b failed: {exc}")
        traceback.print_exc()
        raise


if __name__ == "__main__":
    run_step_5_1b()


# Step 5.1b — Geometric Observability Classes Map

In [ ]:
# =============================================================================
# Step 5.1b — Geometric Observability Classes Map
# =============================================================================
#
# Purpose:
#   Recreate only the Geometric Observability Classes Map, keeping the same
#   plotting style as the existing Step 5.1b figure, but adding the
#   class-threshold legend INSIDE the map in the free middle-left area.
#
# What is preserved:
#   - same class raster
#   - same viridis class colours
#   - same class colorbar on the right
#   - same road and study-area boundary plotting
#
# Style updates (matched to the final Step 3.1 coherence map style):
#   - 2 x 4 inch map figure
#   - four longitude DMS labels (LinearLocator, rotated 45 deg)
#   - three latitude DMS labels (LinearLocator, rotated 90 deg)
#   - bottom-most latitude label anchored above its tick (va="bottom")
#     so it cannot collide with the first longitude label at the corner
#   - no "Latitude" / "Longitude" axis titles
#
# What is added:
#   - compact in-map legend box showing:
#       Near-blind   : G < 0.05
#       Very low     : 0.05 <= G < 0.10
#       Low          : 0.10 <= G < 0.20
#       Moderate     : 0.20 <= G < 0.50
#       High         : G >= 0.50
#   - compact Road and Study Area legend at the lower-right corner
#   - clearer, bold, fully opaque "km" scale-bar unit label
#
# This script reads the existing Step 5.1b outputs and writes a new figure.
# =============================================================================

from __future__ import annotations
import os

import traceback
from datetime import datetime
from pathlib import Path

import numpy as np
import rasterio
import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon, Rectangle
from matplotlib.ticker import FuncFormatter, LinearLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    import geopandas as gpd
    HAVE_GEOPANDAS = True
except Exception:
    gpd = None
    HAVE_GEOPANDAS = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    fiona = None
    HAVE_FIONA = False

try:
    import pyogrio
    HAVE_PYOGRIO = True
except Exception:
    pyogrio = None
    HAVE_PYOGRIO = False


# =============================================================================
# PATHS
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

ANCHOR_TIF = BASE / "Results" / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
OBS_DIR = BASE / "Results" / "Step-5.1b-Geometric_Representativeness"

CLASS_TIF = OBS_DIR / "geometric_observability_classes_32645_12p5m_byte.tif"
SLOPE_VALID_MASK_TIF = OBS_DIR / "geometric_observability_slope_valid_mask_32645_12p5m_byte.tif"

ROAD_SHP = BASE / "NTD_Road.shp"
STUDY_AREA_SHP = BASE / "Study_Area.shp"

OUT_STEM = OBS_DIR / "Geometric Observability Classes Map InMap Legend Final"


# =============================================================================
# SETTINGS
# =============================================================================
NODATA_BYTE = 255
MAP_FIGSIZE = (2.5, 5)
FIG_DPI = 900
DISPLAY_BUFFER_FRACTION = 0.02
LONGITUDE_DMS_TICK_COUNT = 4
LATITUDE_DMS_TICK_COUNT = 3
DMS_TICK_FONTSIZE = 6.2
SHOW_FIGURES = True
SAVE_FIGURES = True

CLASS_COLOURS = plt.get_cmap("viridis")(np.linspace(0.05, 0.95, 5))
CLASS_CMAP = ListedColormap(CLASS_COLOURS)
CLASS_NORM = BoundaryNorm(np.arange(0.5, 5.6, 1.0), CLASS_CMAP.N)

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "DejaVu Serif"]
plt.rcParams["axes.unicode_minus"] = False


# =============================================================================
# HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def read_anchor(path: Path):
    require_file(path, "anchor raster")
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"Anchor raster has no CRS: {path}")
        return {
            "transform": ds.transform,
            "crs": ds.crs,
            "width": ds.width,
            "height": ds.height,
            "bounds": ds.bounds,
        }


def same_grid(ds, anchor) -> bool:
    return (
        ds.crs == anchor["crs"]
        and ds.transform == anchor["transform"]
        and ds.width == anchor["width"]
        and ds.height == anchor["height"]
    )


def read_mask_exact(path: Path, anchor) -> np.ndarray:
    require_file(path, f"mask {path.name}")
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"Mask has no CRS: {path}")
        if not same_grid(ds, anchor):
            raise ValueError(f"Mask does not match anchor grid: {path}")
        arr = ds.read(1).astype("float64")
        if ds.nodata is not None:
            arr[arr == ds.nodata] = 0.0
        arr[~np.isfinite(arr)] = 0.0
        return arr > 0.5


def read_classes_exact(path: Path, anchor) -> np.ndarray:
    require_file(path, "class raster")
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"Class raster has no CRS: {path}")
        if not same_grid(ds, anchor):
            raise ValueError(f"Class raster does not match anchor grid: {path}")
        raw = ds.read(1)
        arr = raw.astype("float64")
        nodata = ds.nodata if ds.nodata is not None else NODATA_BYTE
        arr[raw == nodata] = np.nan

    finite = np.isfinite(arr)
    invalid = finite & ((arr < 1) | (arr > 5) | (arr != np.floor(arr)))
    if invalid.any():
        raise ValueError(f"Invalid class codes found: {np.unique(arr[invalid]).tolist()}")
    return arr


# -------------------------
# Vector compatibility read
# -------------------------
def _fiona_feature_to_mapping(feature):
    geometry = feature["geometry"]
    if geometry is None:
        geometry_mapping = None
    elif hasattr(geometry, "__geo_interface__"):
        geometry_mapping = geometry.__geo_interface__
    else:
        geometry_mapping = dict(geometry)

    properties = feature["properties"]
    property_mapping = {} if properties is None else dict(properties)

    try:
        feature_id = feature["id"]
    except Exception:
        feature_id = getattr(feature, "id", None)

    return {
        "type": "Feature",
        "id": None if feature_id is None else str(feature_id),
        "properties": property_mapping,
        "geometry": geometry_mapping,
    }


def read_vector_compat(path: Path, target_crs, description: str):
    if not path.exists():
        log(f"[WARN] Optional {description} not found: {path}")
        return None

    if not HAVE_GEOPANDAS:
        log(f"[WARN] GeoPandas unavailable; {description} will not be plotted.")
        return None

    errors = []
    vector = None

    if HAVE_PYOGRIO:
        try:
            vector = pyogrio.read_dataframe(str(path))
        except Exception as exc:
            errors.append(f"Pyogrio failed: {type(exc).__name__}: {exc}")
            vector = None

    if vector is None and HAVE_FIONA:
        try:
            features = []
            with fiona.open(str(path), mode="r") as collection:
                source_crs = collection.crs_wkt if getattr(collection, "crs_wkt", None) else collection.crs
                for feature in collection:
                    converted = _fiona_feature_to_mapping(feature)
                    if converted["geometry"] is not None:
                        features.append(converted)

            if not features:
                raise RuntimeError(f"{description} contains no readable geometries.")

            vector = gpd.GeoDataFrame.from_features(features, crs=source_crs)

        except Exception as exc:
            errors.append(f"Direct Fiona failed: {type(exc).__name__}: {exc}")
            vector = None

    if vector is None:
        log(f"[WARN] Could not load {description}. " + " | ".join(errors))
        return None

    if vector.crs is None:
        raise ValueError(f"{description} has no CRS: {path}")

    vector = vector.to_crs(target_crs)
    vector = vector[vector.geometry.notna() & (~vector.geometry.is_empty)].copy()
    return vector


# -------------------------
# Cartographic helpers
# -------------------------
def dms(value: float, longitude: bool) -> str:
    absolute = abs(float(value))
    degrees = int(absolute)
    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0
    hemisphere = ("E" if value >= 0 else "W") if longitude else ("N" if value >= 0 else "S")
    return f"{degrees}°{minutes:02d}'{seconds:04.1f}\"{hemisphere}"


def add_north(axis, size=0.09, pad=0.09):
    x0, y0 = pad, 1 - pad - size - 0.025
    x1, y1 = x0 + size * 0.72, y0 + size
    apex = ((x0 + x1) / 2, y1)
    tail = ((x0 + x1) / 2, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, (x0, y0), tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, (x1, y0)],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1,
            zorder=20,
        )
    )
    axis.text(
        pad + size * 0.36,
        1 - pad + size * 0.10,
        "N",
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        zorder=21,
    )


def nice_length(target):
    exponent = int(np.floor(np.log10(max(target, 1.0))))
    base = 10 ** exponent
    for multiplier in (1, 2, 5, 10):
        if multiplier * base >= target:
            return multiplier * base
    return 10 * base


def add_scale(axis, west, east, south, north, projected, mid_lat=None):
    width, height = east - west, north - south
    if projected:
        width_m, units_per_m = width, 1.0
    else:
        mpd = 111320 * np.cos(np.radians(mid_lat if mid_lat is not None else (south + north) / 2))
        width_m, units_per_m = width * mpd, 1.0 / mpd

    total_m = nice_length(width_m / 5)
    total_u = total_m * units_per_m
    seg_u = total_u / 4

    x0 = west + width / 2 - total_u / 2
    y0 = south + 0.06 * height
    h = 0.018 * height

    axis.add_patch(
        Rectangle(
            (x0 - total_u * 0.08, y0 - 0.035 * height),
            total_u * 1.16,
            h + 0.07 * height,
            facecolor="white",
            edgecolor="none",
            alpha=0.7,
            zorder=12,
        )
    )

    for i in range(4):
        axis.add_patch(
            Rectangle(
                (x0 + i * seg_u, y0),
                seg_u,
                h,
                facecolor="black" if i % 2 == 0 else "white",
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x0, y0),
            total_u,
            h,
            fill=False,
            edgecolor="black",
            linewidth=0.4,
            zorder=14,
        )
    )

    axis.text(
        x0 + total_u / 2,
        y0 + h + 0.012 * height,
        "km",
        ha="center",
        va="bottom",
        fontsize=5.6,
        fontweight="bold",
        color="black",
        alpha=1.0,
        zorder=16,
    )
    axis.text(x0, y0 - 0.012 * height, "0", ha="center", va="top", fontsize=5, fontweight="bold")
    end_km = total_m / 1000
    label = f"{int(round(end_km))}" if abs(end_km - round(end_km)) < 1e-6 else f"{end_km:.1f}"
    axis.text(x0 + total_u, y0 - 0.012 * height, label, ha="center", va="top", fontsize=5, fontweight="bold")


def extent(anchor):
    bounds = anchor["bounds"]
    dx = (bounds.right - bounds.left) * DISPLAY_BUFFER_FRACTION
    dy = (bounds.top - bounds.bottom) * DISPLAY_BUFFER_FRACTION
    return [bounds.left, bounds.right, bounds.bottom, bounds.top], bounds.left - dx, bounds.right + dx, bounds.bottom - dy, bounds.top + dy


def configure_axes(axis, anchor, west, east, south, north):
    xmid = (anchor["bounds"].left + anchor["bounds"].right) / 2
    ymid = (anchor["bounds"].bottom + anchor["bounds"].top) / 2

    if anchor["crs"].is_geographic:
        flon = lambda v, p: dms(v, True)
        flat = lambda v, p: dms(v, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError("pyproj required for DMS labels.")
        tr = Transformer.from_crs(anchor["crs"], "EPSG:4326", always_xy=True)
        flon = lambda v, p: dms(tr.transform(v, ymid)[0], True)
        flat = lambda v, p: dms(tr.transform(xmid, v)[1], False)

    axis.set_xlim(west, east)
    axis.set_ylim(south, north)

    # No "Latitude" / "Longitude" axis titles (Step 3.1 style).

    axis.xaxis.set_major_locator(LinearLocator(LONGITUDE_DMS_TICK_COUNT))
    axis.xaxis.set_major_formatter(FuncFormatter(flon))

    # Fixed latitude tick count so the rotated DMS labels
    # cannot overlap (Step 3.1 style: three latitude labels).
    axis.yaxis.set_major_locator(LinearLocator(LATITUDE_DMS_TICK_COUNT))
    axis.yaxis.set_major_formatter(FuncFormatter(flat))

    axis.tick_params(axis="both", labelsize=DMS_TICK_FONTSIZE, width=0.8, length=2.5)

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")
        label.set_fontsize(DMS_TICK_FONTSIZE)
        label.set_fontweight("bold")

    y_tick_labels = axis.get_yticklabels()
    for label in y_tick_labels:
        label.set_fontsize(DMS_TICK_FONTSIZE)
        label.set_fontweight("bold")
        label.set_rotation(90)
        label.set_va("center")

    # Lift the bottom-most latitude label fully above its tick so it
    # clears the first (45 deg rotated) longitude label at the corner.
    if y_tick_labels:
        y_tick_labels[0].set_va("bottom")
        log(
            "[ADJUST] Bottom latitude DMS label anchored above its "
            "tick (va='bottom') to avoid corner overlap with the "
            "first longitude label."
        )


def plot_map(axis, array, anchor, roads, boundary, cmap, norm=None):
    ext, west, east, south, north = extent(anchor)

    image = axis.imshow(
        np.ma.masked_invalid(array),
        extent=ext,
        origin="upper",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
        zorder=1,
    )

    if roads is not None:
        roads.plot(ax=axis, color="red", linewidth=0.7, zorder=850)

    if boundary is not None:
        boundary.boundary.plot(ax=axis, color="black", linewidth=0.7, zorder=999)

    configure_axes(axis, anchor, west, east, south, north)
    add_north(axis, size=0.09, pad=0.09)
    add_scale(
        axis,
        west,
        east,
        south,
        north,
        not anchor["crs"].is_geographic,
        (south + north) / 2 if anchor["crs"].is_geographic else None,
    )
    return image


def save_figure(fig, stem: Path, dpi=FIG_DPI):
    if SAVE_FIGURES:
        for suffix in (".png", ".pdf", ".svg"):
            fig.savefig(Path(str(stem) + suffix), dpi=dpi, bbox_inches="tight", facecolor="white")
    if SHOW_FIGURES:
        display(fig) if HAVE_IPYTHON_DISPLAY else plt.show()
    plt.close(fig)


# =============================================================================
# MAIN
# =============================================================================
def create_class_map():
    log("=== Creating in-map-legend geometric observability class map ===")

    anchor = read_anchor(ANCHOR_TIF)
    slope_valid = read_mask_exact(SLOPE_VALID_MASK_TIF, anchor)
    classes = read_classes_exact(CLASS_TIF, anchor)

    roads = read_vector_compat(ROAD_SHP, anchor["crs"], "road")
    boundary = read_vector_compat(STUDY_AREA_SHP, anchor["crs"], "study area")

    class_array = np.where(slope_valid, classes.astype(float), np.nan)

    fig, ax = plt.subplots(figsize=MAP_FIGSIZE, facecolor="white")

    im = plot_map(ax, class_array, anchor, roads, boundary, CLASS_CMAP, norm=CLASS_NORM)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.01)

    cb = fig.colorbar(im, cax=cax, ticks=np.arange(1, 6))
    cb.ax.set_yticklabels(["Near-blind", "Very low", "Low", "Moderate", "High"])
    cb.ax.tick_params(labelsize=4.4, width=0.6, length=2)
    for label in cb.ax.get_yticklabels():
        label.set_fontweight("bold")
    cb.set_label("Observability class", fontsize=5.2, fontweight="bold")

    # -------------------------------------------------------------------------
    # Compact in-map legend box
    # -------------------------------------------------------------------------
    legend_handles = [
        Patch(facecolor=CLASS_COLOURS[0], edgecolor="black", linewidth=0.3, label=r"Near-blind: $G<0.05$"),
        Patch(facecolor=CLASS_COLOURS[1], edgecolor="black", linewidth=0.3, label=r"Very low: $0.05\leq G<0.10$"),
        Patch(facecolor=CLASS_COLOURS[2], edgecolor="black", linewidth=0.3, label=r"Low: $0.10\leq G<0.20$"),
        Patch(facecolor=CLASS_COLOURS[3], edgecolor="black", linewidth=0.3, label=r"Moderate: $0.20\leq G<0.50$"),
        Patch(facecolor=CLASS_COLOURS[4], edgecolor="black", linewidth=0.3, label=r"High: $G\geq0.50$"),
    ]

    class_legend = ax.legend(
        handles=legend_handles,
        loc="center left",
        bbox_to_anchor=(0.03, 0.65),   # middle-left free area inside map
        fontsize=3.1,
        title="Class thresholds",
        title_fontsize=3.5,
        frameon=True,
        framealpha=0.92,
        facecolor="white",
        edgecolor="0.6",
        borderpad=0.25,
        labelspacing=0.20,
        handlelength=0.95,
        handletextpad=0.35,
        borderaxespad=0.0,
    )
    class_legend.set_zorder(1000)
    class_legend.get_title().set_fontweight("bold")
    for txt in class_legend.get_texts():
        txt.set_fontweight("bold")

    # Preserve the class-threshold legend when adding the map-feature legend.
    ax.add_artist(class_legend)

    # -------------------------------------------------------------------------
    # Road and Study Area legend — same compact style used in Steps 5.3/5.4
    # -------------------------------------------------------------------------
    map_legend_handles = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=1.0,
            label="Road",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=1.0,
            label="Study Area",
        ),
    ]

    map_legend = ax.legend(
        handles=map_legend_handles,
        loc="lower right",
        bbox_to_anchor=(0.98, 0.09),
        bbox_transform=ax.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=3.1,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0,
    )
    map_legend.get_frame().set_edgecolor("lightgray")
    map_legend.get_frame().set_linewidth(0.3)
    for txt in map_legend.get_texts():
        txt.set_fontweight("bold")
    map_legend.set_zorder(1001)

    fig.tight_layout()
    save_figure(fig, OUT_STEM)

    log(
        "[STYLE] 2x4 inch map | four longitude DMS labels | "
        "three latitude DMS labels (rotated 90 deg, bold, bottom label "
        "lifted) | no axis titles | 900 dpi | PNG/PDF/SVG"
    )
    log(f"[OUT] {Path(str(OUT_STEM)+'.png')}")
    log(f"[OUT] {Path(str(OUT_STEM)+'.pdf')}")
    log(f"[OUT] {Path(str(OUT_STEM)+'.svg')}")
    log("=== Done ===")


def run():
    try:
        create_class_map()
    except Exception as exc:
        print(f"[ERROR] Class-map creation failed: {exc}")
        traceback.print_exc()
        raise


if __name__ == "__main__":
    run()

# Step 5.1c — Stabilization-Parameter ε Sensitivity of DV+, HKFI and SNDR

In [ ]:
# =============================================================================
# Step 5.1c — Stabilization-Parameter ε Sensitivity of DV+, HKFI and SNDR
# =============================================================================
# Reviewer issue addressed
# ------------------------
#   DV = v_LOS / [sign(C) * max(|C|, ε)]
#   C  = l_hat · s_hat
#   G  = |C|
#   DV+ = max(DV, 0)
#
# Tests ε = 0.025, 0.050, 0.075, 0.100, 0.150 and 0.200, with ε=0.050
# as the primary reference. The perturbation is propagated through the exact
# existing DV+ preparation logic, Step 4.2 standardization, revised HKFI, and
# corrected SNDR while all non-ε terms are held fixed.
#
# Important scientific rules
# --------------------------
# 1. G and signed C are fixed; ε changes only the stabilized denominator.
# 2. IRS is fixed and is never recalculated or renormalized.
# 3. The same LOS support, coherence mask, PS support, IDW neighbours, fixed
#    winsor caps, TWI term, soil-moisture term, and slope denominator are used
#    for all ε scenarios.
# 4. Step 4.2 μ and σ are recomputed for each ε on the same observed-DV source
#    domain, exactly as a rerun of the pipeline would do.
# 5. HKFI uses fixed equal weights K=T=S=1/3 and P2-P98 normalization.
# 6. SNDR retains the corrected zero-preserving P98 normalization.
# 7. Class-change and top-decile overlap use thresholds defined once from the
#    ε=0.05 reference and reused for every scenario.
# 8. This is map-level sensitivity. It is not susceptibility-model sensitivity
#    unless selected ε scenarios are later propagated through the unchanged
#    spatial-CV machine-learning pipeline.
# =============================================================================

from __future__ import annotations
import os

import json
import math
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

try:
    from scipy.spatial import cKDTree
    from scipy.stats import spearmanr
except Exception as exc:
    raise ImportError(
        "Step 5.1c requires SciPy (cKDTree and spearmanr)."
    ) from exc

import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass
import matplotlib.pyplot as plt


# =============================================================================
# CONFIGURATION
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
RESULTS = BASE / "Results"

EPSILON_VALUES = [0.025, 0.050, 0.075, 0.100, 0.150, 0.200]
REFERENCE_EPSILON = 0.050

# Existing Step 3.6 / 3.7 parameters retained exactly for controlled sensitivity.
MIN_LOS_COUNT = 1.0
MIN_COHERENCE = 0.25
DVPLUS_CAP_MM_YR = 150.0
SUPPORT_RADIUS_M = 100.0
K_NEIGHBORS = 16
IDW_POWER = 2.0
MAX_IDW_DIST_M = 200.0
IDW_EPS = 1.0e-12
QUERY_CHUNK_SIZE = 150_000

# Existing Step 4.2 fallback only; normally exact caps are recovered from its
# metadata or Step 4.1 limits JSON.
FALLBACK_Q_LOW = 1.0
FALLBACK_Q_HIGH = 99.0
MIN_STATS_PIXELS = 50

# Revised Step 5.3.
HKFI_WEIGHTS = {"K": 1.0 / 3.0, "T": 1.0 / 3.0, "S": 1.0 / 3.0}
HKFI_P_LOW = 2.0
HKFI_P_HIGH = 98.0

# Corrected Step 5.4.
SNDR_P_UPPER = 98.0

# Metrics.
TOP_FRACTION = 0.10
DV_ABS_CHANGE_THRESHOLD_MM_YR = 5.0
DV_REL_CHANGE_THRESHOLD = 0.10
DV_REL_DENOMINATOR_FLOOR_MM_YR = 1.0

# Reproduction QA. These assess whether the reconstructed ε=0.05 chain is
# sufficiently consistent with the authoritative outputs before interpreting
# the sensitivity experiment.
BASELINE_WARN_SPEARMAN = 0.95
BASELINE_HARD_FAIL_SPEARMAN = 0.80

WRITE_SCENARIO_RASTERS = True
MAKE_FIGURE = True
SHOW_FIGURE = True
FIG_DPI = 600
FIGSIZE = (8.2, 6.6)

REQUIRE_EXACT_GRID = True
NODATA_FLOAT = -9999.0


# =============================================================================
# PATHS
# =============================================================================
ANCHOR_TIF = RESULTS / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
STUDY_MASK_TIF = RESULTS / "Step-0.2-Study_Mask" / "study_mask.tif"
CONTEXT_MASK_TIF = RESULTS / "Step-0.3-Quality_Masks" / "context_mask.tif"
DYNAMIC_MASK_TIF = RESULTS / "Step-0.3-Quality_Masks" / "dynamic_mask.tif"
PS_DISTANCE_TIF = RESULTS / "Step-0.3-Quality_Masks" / "ps_distance_m.tif"

LOS_MEAN_TIF = (
    RESULTS / "Step-3.2-LOS" / "los_velocity_mean_mm_yr_32645_12p5m_float32.tif"
)
LOS_COUNT_TIF = (
    RESULTS / "Step-3.2-LOS" / "los_velocity_count_32645_12p5m_int16.tif"
)
COHERENCE_TIF = (
    RESULTS / "Step-3.1-Coherence" / "coherence_ps_mean_32645_12p5m_float32.tif"
)

SIGNED_C_TIF = (
    RESULTS / "Step-5.1-Observability"
    / "signed_los_downslope_projection_32645_12p5m_float32.tif"
)
G_TIF = (
    RESULTS / "Step-5.1-Observability"
    / "geometric_observability_G_0_1_32645_12p5m_float32.tif"
)
SLOPE_VALID_MASK_TIF = (
    RESULTS / "Step-5.1b-Geometric_Representativeness"
    / "geometric_observability_slope_valid_mask_32645_12p5m_byte.tif"
)

STEP36_DVPLUS_MASKED_TIF = (
    RESULTS / "Step-3.6-DV⁺" / "dv_plus_masked_mm_yr_32645_12p5m_float32.tif"
)
STEP37_DVPLUS_FILLED_TIF = (
    RESULTS / "Step-3.7-DVplus_Densify"
    / "dv_plus_filled_mm_yr_32645_12p5m_float32.tif"
)
DV_SOURCE_DIST_TIF = (
    RESULTS / "Step-3.7-DVplus_Densify"
    / "dv_plus_source_dist_m_32645_12p5m_float32.tif"
)

STEP41_LIMITS_JSON = RESULTS / "Step-4.1-Limits" / "winsor_limits.json"
STEP42_DIR = RESULTS / "Step-4.2-Standardization"
STEP42_META_JSON = STEP42_DIR / "zscore_metadata.json"
AUTHORITATIVE_DVPLUS_WINSOR_TIF = (
    STEP42_DIR / "dvplus_winsor_mm_yr_32645_12p5m_float32.tif"
)
AUTHORITATIVE_Z_DVPLUS_TIF = (
    STEP42_DIR / "z_dvplus_32645_12p5m_float32.tif"
)

IRS_DIR = RESULTS / "Step-5.2-IRS"
IRS_TIF = IRS_DIR / "irs_0_1_dense_32645_12p5m_float32.tif"
IRS_SUPPORT_MASK_TIF = IRS_DIR / "irs_observation_support_mask_32645_12p5m_byte.tif"

HKFI_DIR = RESULTS / "Step-5.3-HKFI"
HKFI_T_TERM_TIF = HKFI_DIR / "hkfi_term_T_ztwi_32645_12p5m_float32.tif"
HKFI_S_TERM_TIF = (
    HKFI_DIR / "hkfi_term_S_zsm_support_preserved_32645_12p5m_float32.tif"
)
HKFI_IRS_EFFECTIVE_TIF = (
    HKFI_DIR / "irs_effective_for_hkfi_0_1_32645_12p5m_float32.tif"
)
HKFI_VALID_MASK_TIF = HKFI_DIR / "hkfi_valid_mask_32645_12p5m_byte.tif"
AUTHORITATIVE_HKFI_CANDIDATES = [
    HKFI_DIR / "hkfi_multiscale_norm_0_1_32645_12p5m_float32.tif",
    HKFI_DIR / "hkfi_norm_0_1_dense_32645_12p5m_float32.tif",
    HKFI_DIR / "hkfi_norm_0_1_32645_12p5m_float32.tif",
]

SNDR_DIR = RESULTS / "Step-5.4-SNDR"
SNDR_IRS_EFFECTIVE_TIF = (
    SNDR_DIR / "irs_effective_for_sndr_0_1_32645_12p5m_float32.tif"
)
SNDR_DENOMINATOR_TIF = (
    SNDR_DIR / "sndr_sin_slope_plus_delta_32645_12p5m_float32.tif"
)
SNDR_VALID_MASK_TIF = SNDR_DIR / "sndr_valid_domain_mask_32645_12p5m_byte.tif"
SNDR_KINEMATIC_SUPPORT_MASK_TIF = (
    SNDR_DIR / "sndr_kinematic_support_mask_32645_12p5m_byte.tif"
)
AUTHORITATIVE_SNDR_TIF = SNDR_DIR / "sndr_0_1_32645_12p5m_float32.tif"

MODEL_READY_DATASET_CSV = (
    RESULTS / "Step-6.4-Model_Ready_QA" / "model_ready_dataset_A_to_F.csv"
)

OUT_DIR = RESULTS / "Step-5.1c-Epsilon_Sensitivity"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIO_DIR = OUT_DIR / "scenario_rasters"
SCENARIO_DIR.mkdir(parents=True, exist_ok=True)

GEOMETRIC_CSV = OUT_DIR / "epsilon_geometric_sensitivity.csv"
DVPLUS_CSV = OUT_DIR / "epsilon_dvplus_sensitivity.csv"
HKFI_CSV = OUT_DIR / "epsilon_hkfi_sensitivity.csv"
SNDR_CSV = OUT_DIR / "epsilon_sndr_sensitivity.csv"
SUMMARY_CSV = OUT_DIR / "epsilon_sensitivity_summary.csv"
MODEL_SITES_CSV = OUT_DIR / "epsilon_sensitivity_model_sites.csv"
BASELINE_QA_CSV = OUT_DIR / "epsilon_0p05_baseline_reproduction_QA.csv"
META_JSON = OUT_DIR / "Step_5_1c_epsilon_sensitivity_metadata.json"
QA_REPORT_TXT = OUT_DIR / "Step_5_1c_epsilon_sensitivity_QA_report.txt"
FIG_PNG = OUT_DIR / "Epsilon_Sensitivity_Reviewer_Figure.png"
FIG_PDF = OUT_DIR / "Epsilon_Sensitivity_Reviewer_Figure.pdf"


# =============================================================================
# HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def epsilon_tag(value: float) -> str:
    return f"{value:.3f}".replace(".", "p")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def choose_existing(candidates: Iterable[Path], description: str) -> Path:
    for path in candidates:
        if path.exists():
            log(f"[PATH] {description}: {path}")
            return path
    raise FileNotFoundError(
        f"Could not locate {description}. Tried:\n"
        + "\n".join(str(p) for p in candidates)
    )


# =============================================================================
# RASTER I/O
# =============================================================================
def read_anchor(path: Path) -> Dict[str, Any]:
    require_file(path, "anchor raster")
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"Anchor raster has no CRS: {path}")
        tf = ds.transform
        return {
            "transform": tf,
            "crs": ds.crs,
            "width": ds.width,
            "height": ds.height,
            "profile": ds.profile.copy(),
            "bounds": ds.bounds,
            "epsg": ds.crs.to_epsg(),
            "cell_area_m2": float(abs(tf.a * tf.e - tf.b * tf.d)),
        }


def same_grid(ds: rasterio.io.DatasetReader, anchor: Dict[str, Any]) -> bool:
    return (
        ds.crs == anchor["crs"]
        and ds.transform == anchor["transform"]
        and ds.width == anchor["width"]
        and ds.height == anchor["height"]
    )


def clean_nodata(array: np.ndarray, nodata: float | int | None) -> np.ndarray:
    out = array.astype("float64", copy=True)
    if nodata is not None:
        try:
            nd = float(nodata)
            if np.isfinite(nd):
                out[out == nd] = np.nan
        except (TypeError, ValueError):
            pass
    out[out == -9999.0] = np.nan
    out[out == -32768.0] = np.nan
    out[np.abs(out) > 1.0e20] = np.nan
    return out


def read_continuous_exact(path: Path, anchor: Dict[str, Any], description: str) -> np.ndarray:
    require_file(path, description)
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")
        if REQUIRE_EXACT_GRID and not same_grid(ds, anchor):
            raise ValueError(f"{description} does not match the anchor grid: {path}")
        return clean_nodata(ds.read(1), ds.nodata)


def read_continuous_to_anchor(
    path: Path,
    anchor: Dict[str, Any],
    description: str,
    resampling: Resampling,
) -> np.ndarray:
    require_file(path, description)
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")
        raw = ds.read(1).astype("float64")
        if same_grid(ds, anchor):
            return clean_nodata(raw, ds.nodata)
        dst = np.full((anchor["height"], anchor["width"]), np.nan, dtype="float64")
        reproject(
            source=raw,
            destination=dst,
            src_transform=ds.transform,
            src_crs=ds.crs,
            src_nodata=ds.nodata,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
        return clean_nodata(dst, None)


def read_mask_exact(
    path: Path,
    anchor: Dict[str, Any],
    description: str,
    required: bool = True,
) -> np.ndarray | None:
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Missing {description}: {path}")
        return None
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")
        if REQUIRE_EXACT_GRID and not same_grid(ds, anchor):
            raise ValueError(f"{description} does not match the anchor grid: {path}")
        raw = ds.read(1)
        values = raw.astype("float64")
        if ds.nodata is not None:
            values[raw == ds.nodata] = 0.0
        values[~np.isfinite(values)] = 0.0
        return values > 0.5


def write_float_tif(path: Path, array: np.ndarray, anchor: Dict[str, Any], description: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff", count=1, dtype="float32", nodata=NODATA_FLOAT,
        compress="LZW", tiled=True, bigtiff="IF_NEEDED",
        transform=anchor["transform"], crs=anchor["crs"],
        width=anchor["width"], height=anchor["height"],
    )
    out = np.where(np.isfinite(array), array, NODATA_FLOAT).astype("float32")
    tmp = Path(str(path) + ".part")
    if tmp.exists():
        tmp.unlink()
    with rasterio.open(tmp, "w", **profile) as ds:
        ds.write(out, 1)
        ds.set_band_description(1, description)
    if path.exists():
        path.unlink()
    tmp.replace(path)


# =============================================================================
# STEP 4.2 CAP RECOVERY
# =============================================================================
def recover_fixed_dvplus_caps() -> Tuple[float, float, str]:
    # First preference: the exact caps recorded by the completed Step 4.2 run.
    if STEP42_META_JSON.exists():
        try:
            meta = json.loads(STEP42_META_JSON.read_text(encoding="utf-8"))
            layer = meta.get("layers", {}).get("DVplus", {})
            lo = float(layer.get("winsor_low"))
            hi = float(layer.get("winsor_high"))
            if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
                return lo, hi, "Step 4.2 zscore_metadata.json"
        except Exception:
            pass

    # Second preference: Step 4.1 limits JSON, using the same aliases as old 4.2.
    if STEP41_LIMITS_JSON.exists():
        try:
            data = json.loads(STEP41_LIMITS_JSON.read_text(encoding="utf-8"))
            candidates = []
            if isinstance(data.get("layers"), dict):
                candidates.append(data["layers"].get("DVplus"))
                for key, value in data["layers"].items():
                    if str(key).lower() in {"dvplus", "dv+", "dv_plus", "dv⁺", "dv"}:
                        candidates.append(value)
            candidates.append(data.get("DVplus"))
            for section in candidates:
                if not isinstance(section, dict):
                    continue
                lower_keys = ["winsor_low", "low", "lo", "cap_low", "lower"]
                upper_keys = ["winsor_high", "high", "hi", "cap_high", "upper"]
                lo = next((section[k] for k in lower_keys if k in section), None)
                hi = next((section[k] for k in upper_keys if k in section), None)
                if lo is not None and hi is not None:
                    lo = float(lo); hi = float(hi)
                    if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
                        return lo, hi, "Step 4.1 winsor_limits.json"
        except Exception:
            pass

    return float("nan"), float("nan"), "fallback quantiles required"


# =============================================================================
# ε-SPECIFIC DV+ SOURCE COMPUTATION
# =============================================================================
def compute_masked_source_dvplus(
    los_mean: np.ndarray,
    signed_c: np.ndarray,
    count: np.ndarray,
    coherence: np.ndarray,
    study: np.ndarray,
    epsilon: float,
) -> Tuple[np.ndarray, np.ndarray]:
    if epsilon <= 0.0:
        raise ValueError("epsilon must be > 0")

    base_valid = (
        study
        & np.isfinite(los_mean)
        & np.isfinite(signed_c)
        & np.isfinite(count)
        & (count >= MIN_LOS_COUNT)
        & (np.abs(signed_c) > 1.0e-12)
    )

    denom = np.full_like(signed_c, np.nan, dtype="float64")
    denom[base_valid] = (
        np.sign(signed_c[base_valid])
        * np.maximum(np.abs(signed_c[base_valid]), float(epsilon))
    )

    dv = np.full_like(los_mean, np.nan, dtype="float64")
    dv[base_valid] = los_mean[base_valid] / denom[base_valid]

    finite_dv = np.isfinite(dv)
    dv[finite_dv] = np.minimum(dv[finite_dv], DVPLUS_CAP_MM_YR)

    dvplus = np.full_like(dv, np.nan, dtype="float64")
    dvplus[finite_dv] = np.maximum(dv[finite_dv], 0.0)

    qa_valid = (
        base_valid
        & np.isfinite(coherence)
        & (coherence >= MIN_COHERENCE)
        & np.isfinite(dvplus)
    )

    masked = np.full_like(dvplus, np.nan, dtype="float64")
    masked[qa_valid] = dvplus[qa_valid]
    return masked, qa_valid


# =============================================================================
# IDENTICAL IDW DENSIFICATION FOR ALL ε
# =============================================================================
def rows_cols_to_xy(rows: np.ndarray, cols: np.ndarray, transform) -> np.ndarray:
    rr = rows.astype("float64") + 0.5
    cc = cols.astype("float64") + 0.5
    x = transform.c + cc * transform.a + rr * transform.b
    y = transform.f + cc * transform.d + rr * transform.e
    return np.column_stack([x, y])


def densify_all_scenarios(
    masked_source_by_epsilon: Dict[float, np.ndarray],
    source_mask: np.ndarray,
    domain: np.ndarray,
    anchor: Dict[str, Any],
) -> Dict[float, np.ndarray]:
    src_r, src_c = np.where(source_mask)
    if src_r.size < 50:
        raise RuntimeError(f"Too few source cells for IDW: {src_r.size}")

    need = domain & ~source_mask
    tgt_r, tgt_c = np.where(need)
    src_xy = rows_cols_to_xy(src_r, src_c, anchor["transform"])
    tgt_xy = rows_cols_to_xy(tgt_r, tgt_c, anchor["transform"])

    order = list(EPSILON_VALUES)
    source_values = np.vstack([
        masked_source_by_epsilon[e][source_mask] for e in order
    ]).astype("float64")

    outputs: Dict[float, np.ndarray] = {}
    for e in order:
        arr = np.full((anchor["height"], anchor["width"]), np.nan, dtype="float32")
        arr[source_mask] = masked_source_by_epsilon[e][source_mask].astype("float32")
        outputs[e] = arr

    tree = cKDTree(src_xy)
    k = min(K_NEIGHBORS, src_r.size)
    log(f"[IDW] sources={src_r.size:,}, targets={tgt_r.size:,}, k={k}")

    for start in range(0, tgt_r.size, QUERY_CHUNK_SIZE):
        stop = min(start + QUERY_CHUNK_SIZE, tgt_r.size)
        try:
            dist, idx = tree.query(
                tgt_xy[start:stop], k=k,
                distance_upper_bound=MAX_IDW_DIST_M, workers=-1,
            )
        except TypeError:
            dist, idx = tree.query(
                tgt_xy[start:stop], k=k,
                distance_upper_bound=MAX_IDW_DIST_M,
            )

        if k == 1:
            dist = dist[:, None]
            idx = idx[:, None]

        valid_nb = np.isfinite(dist) & (dist <= MAX_IDW_DIST_M) & (idx < src_r.size)
        safe_idx = np.where(valid_nb, idx, 0)
        weights = np.zeros_like(dist, dtype="float64")
        weights[valid_nb] = 1.0 / np.maximum(dist[valid_nb], IDW_EPS) ** IDW_POWER
        wsum = weights.sum(axis=1)
        fillable = wsum > 0.0

        rr = tgt_r[start:stop]
        cc = tgt_c[start:stop]
        for s, e in enumerate(order):
            vals = source_values[s, safe_idx]
            estimate = np.full(stop - start, np.nan, dtype="float64")
            estimate[fillable] = (weights * vals).sum(axis=1)[fillable] / wsum[fillable]
            estimate[fillable] = np.minimum(estimate[fillable], DVPLUS_CAP_MM_YR)
            outputs[e][rr[fillable], cc[fillable]] = estimate[fillable].astype("float32")

        log(f"[IDW] processed {stop:,}/{tgt_r.size:,} target cells")

    return outputs


# =============================================================================
# STEP 4.2 STANDARDIZATION
# =============================================================================
def standardize_dvplus_like_step42(
    filled: np.ndarray,
    dynamic_domain: np.ndarray,
    observed_stats_mask: np.ndarray,
    fixed_lo: float,
    fixed_hi: float,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    stats_ok = observed_stats_mask & np.isfinite(filled)
    if int(stats_ok.sum()) < MIN_STATS_PIXELS:
        raise RuntimeError("Too few observed DV+ cells for stable Step 4.2 statistics.")

    lo, hi = fixed_lo, fixed_hi
    cap_source = "fixed existing pipeline caps"
    if not (np.isfinite(lo) and np.isfinite(hi) and hi > lo):
        values = filled[stats_ok]
        lo = float(np.percentile(values, FALLBACK_Q_LOW))
        hi = float(np.percentile(values, FALLBACK_Q_HIGH))
        cap_source = f"fallback {FALLBACK_Q_LOW:g}-{FALLBACK_Q_HIGH:g} percentile"

    winsor = np.full_like(filled, np.nan, dtype="float64")
    finite = np.isfinite(filled)
    winsor[finite] = np.clip(filled[finite], lo, hi)

    stats_ok = observed_stats_mask & np.isfinite(winsor)
    mu = float(np.mean(winsor[stats_ok]))
    sigma = float(np.std(winsor[stats_ok], ddof=0))
    if not np.isfinite(sigma) or sigma <= 1.0e-12:
        raise RuntimeError("Invalid Step 4.2 DV+ standard deviation.")

    z = np.full_like(filled, np.nan, dtype="float64")
    domain = dynamic_domain & np.isfinite(winsor)
    z[domain] = (winsor[domain] - mu) / sigma

    winsor_domain = np.full_like(filled, np.nan, dtype="float64")
    winsor_domain[domain] = winsor[domain]

    return winsor_domain, z, {
        "winsor_low": float(lo),
        "winsor_high": float(hi),
        "cap_source": cap_source,
        "mu_winsor_observed": mu,
        "sigma_winsor_observed": sigma,
        "n_stats": int(stats_ok.sum()),
        "n_domain": int(domain.sum()),
    }


# =============================================================================
# NORMALIZATION AND METRICS
# =============================================================================
def robust_minmax(
    array: np.ndarray,
    domain: np.ndarray,
    p_low: float,
    p_high: float,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    valid = domain & np.isfinite(array)
    values = array[valid]
    if values.size < 100:
        raise RuntimeError("Too few HKFI cells for robust normalization.")
    lo = float(np.percentile(values, p_low))
    hi = float(np.percentile(values, p_high))
    if hi <= lo:
        raise RuntimeError("Invalid HKFI normalization range.")
    out = np.full_like(array, np.nan, dtype="float64")
    out[valid] = np.clip((array[valid] - lo) / (hi - lo), 0.0, 1.0)
    return out, {"lower": lo, "upper": hi, "n": int(values.size)}


def zero_preserving_p98(
    raw: np.ndarray,
    stats_mask: np.ndarray,
    apply_mask: np.ndarray,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    valid_stats = stats_mask & np.isfinite(raw) & (raw >= 0.0)
    values = raw[valid_stats]
    if values.size < 100:
        raise RuntimeError("Too few supported SNDR cells for P98 normalization.")
    p98 = float(np.percentile(values, SNDR_P_UPPER))
    if not np.isfinite(p98) or p98 <= 0.0:
        raise RuntimeError("Invalid SNDR P98 normalization value.")
    out = np.full_like(raw, np.nan, dtype="float64")
    valid_apply = apply_mask & np.isfinite(raw)
    out[valid_apply] = np.clip(raw[valid_apply] / p98, 0.0, 1.0)
    return out, {"p98": p98, "n_stats": int(values.size)}


def safe_spearman(a: np.ndarray, b: np.ndarray) -> float:
    valid = np.isfinite(a) & np.isfinite(b)
    if int(valid.sum()) < 3:
        return float("nan")
    av, bv = a[valid], b[valid]
    if np.std(av) <= 0.0 or np.std(bv) <= 0.0:
        return float("nan")
    return float(spearmanr(av, bv).statistic)


def fixed_classes(values: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    out = np.full(values.shape, -1, dtype="int8")
    finite = np.isfinite(values)
    out[finite] = np.digitize(values[finite], thresholds, right=False).astype("int8")
    return out


def comparison_metrics(
    reference: np.ndarray,
    scenario: np.ndarray,
    domain: np.ndarray,
    class_thresholds: np.ndarray,
    top_threshold: float,
) -> Dict[str, Any]:
    valid = domain & np.isfinite(reference) & np.isfinite(scenario)
    n = int(valid.sum())
    if n < 3:
        return {
            "comparison_n": n, "spearman_rho": np.nan, "rmse": np.nan,
            "normalized_rmse": np.nan, "mae": np.nan,
            "class_change_fraction": np.nan, "top_decile_jaccard": np.nan,
        }

    ref, scn = reference[valid], scenario[valid]
    diff = scn - ref
    rmse = float(np.sqrt(np.mean(diff ** 2)))
    scale = float(np.percentile(ref, 98.0) - np.percentile(ref, 2.0))

    ref_class = fixed_classes(reference, class_thresholds)
    scn_class = fixed_classes(scenario, class_thresholds)

    ref_top = valid & (reference >= top_threshold)
    scn_top = valid & (scenario >= top_threshold)
    union = ref_top | scn_top

    return {
        "comparison_n": n,
        "spearman_rho": safe_spearman(ref, scn),
        "rmse": rmse,
        "normalized_rmse": rmse / scale if scale > 0.0 else np.nan,
        "mae": float(np.mean(np.abs(diff))),
        "class_change_fraction": float(np.mean(ref_class[valid] != scn_class[valid])),
        "top_decile_jaccard": (
            float((ref_top & scn_top).sum() / union.sum()) if int(union.sum()) else np.nan
        ),
    }


def distribution_stats(values: np.ndarray) -> Dict[str, Any]:
    values = values[np.isfinite(values)]
    if values.size == 0:
        return {k: np.nan for k in ["mean", "median", "p90", "p95", "p99", "max"]}
    return {
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "p90": float(np.percentile(values, 90.0)),
        "p95": float(np.percentile(values, 95.0)),
        "p99": float(np.percentile(values, 99.0)),
        "max": float(np.max(values)),
    }


# =============================================================================
# FIXED STEP 6.4 MODEL-SITE GEOMETRY DIAGNOSTIC
# =============================================================================
def model_site_diagnostic(g: np.ndarray) -> pd.DataFrame:
    if not MODEL_READY_DATASET_CSV.exists():
        log("[WARN] Step 6.4 model-ready dataset absent; skipping model-site diagnostic.")
        return pd.DataFrame()

    df = pd.read_csv(MODEL_READY_DATASET_CSV)
    if not {"row", "col"}.issubset(df.columns):
        log("[WARN] Step 6.4 dataset has no row/col; skipping model-site diagnostic.")
        return pd.DataFrame()

    rr = pd.to_numeric(df["row"], errors="coerce")
    cc = pd.to_numeric(df["col"], errors="coerce")
    ok = rr.notna() & cc.notna() & (rr >= 0) & (cc >= 0) & (rr < g.shape[0]) & (cc < g.shape[1])
    work = df.loc[ok].copy()
    r = rr.loc[ok].astype(int).to_numpy()
    c = cc.loc[ok].astype(int).to_numpy()
    work["G"] = g[r, c]

    rows = []
    groups = [("all_model_sites", work)]
    if "label" in work.columns:
        groups += [("positive_sites", work[work["label"] == 1]),
                   ("negative_sites", work[work["label"] == 0])]

    for epsilon in EPSILON_VALUES:
        for name, sub in groups:
            vals = sub["G"].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            n = int(vals.size)
            affected = int(np.sum(vals < epsilon))
            rows.append({
                "epsilon": epsilon,
                "group": name,
                "n_with_finite_G": n,
                "n_G_lt_epsilon": affected,
                "percent_G_lt_epsilon": 100.0 * affected / n if n else np.nan,
            })
    return pd.DataFrame(rows)


# =============================================================================
# BASELINE REPRODUCTION QA
# =============================================================================
def compare_to_authoritative(
    product: str,
    calculated: np.ndarray,
    authoritative_path: Path,
    domain: np.ndarray,
    anchor: Dict[str, Any],
) -> Dict[str, Any]:
    if not authoritative_path.exists():
        return {
            "product": product, "path": str(authoritative_path), "n": 0,
            "spearman_rho": np.nan, "rmse": np.nan, "mae": np.nan,
            "status": "SKIPPED_MISSING_AUTHORITATIVE_PRODUCT",
        }
    authoritative = read_continuous_exact(authoritative_path, anchor, f"authoritative {product}")
    valid = domain & np.isfinite(calculated) & np.isfinite(authoritative)
    n = int(valid.sum())
    if n < 3:
        return {
            "product": product, "path": str(authoritative_path), "n": n,
            "spearman_rho": np.nan, "rmse": np.nan, "mae": np.nan,
            "status": "INSUFFICIENT_OVERLAP",
        }
    diff = calculated[valid] - authoritative[valid]
    rho = safe_spearman(calculated[valid], authoritative[valid])
    status = "PASS"
    if np.isfinite(rho) and rho < BASELINE_HARD_FAIL_SPEARMAN:
        status = "FAIL_LOW_BASELINE_REPRODUCTION"
    elif np.isfinite(rho) and rho < BASELINE_WARN_SPEARMAN:
        status = "PASS_WITH_WARNING"
    return {
        "product": product, "path": str(authoritative_path), "n": n,
        "spearman_rho": rho,
        "rmse": float(np.sqrt(np.mean(diff ** 2))),
        "mae": float(np.mean(np.abs(diff))),
        "status": status,
    }


# =============================================================================
# FIGURE
# =============================================================================
def make_figure(geometric_df: pd.DataFrame, dv_df: pd.DataFrame,
                hkfi_df: pd.DataFrame, sndr_df: pd.DataFrame) -> None:
    if not MAKE_FIGURE:
        return
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["Times New Roman", "DejaVu Serif"]
    fig, axes = plt.subplots(2, 2, figsize=FIGSIZE, facecolor="white")

    ax = axes[0, 0]
    ax.plot(geometric_df["epsilon"], geometric_df["percent_slope_valid_G_lt_epsilon"], marker="o")
    ax.set_xlabel(r"Stabilization parameter, $\varepsilon$")
    ax.set_ylabel(r"Slope-valid area with $G<\varepsilon$ (%)")
    ax.set_title("(a) Geometric domain affected")
    ax.grid(True, alpha=0.25)

    for ax, df, title in [
        (axes[0, 1], dv_df, "(b) DV+ stability"),
        (axes[1, 0], hkfi_df, "(c) HKFI stability"),
        (axes[1, 1], sndr_df, "(d) SNDR stability"),
    ]:
        ax.plot(df["epsilon"], df["spearman_rho"], marker="o", label="Spearman ρ")
        ax.plot(df["epsilon"], df["top_decile_jaccard"], marker="s", label="Top-decile Jaccard")
        ax.set_xlabel(r"$\varepsilon$")
        ax.set_ylabel("Agreement with ε=0.05")
        ax.set_ylim(0.0, 1.02)
        ax.set_title(title)
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False, fontsize=8)

    fig.tight_layout()
    fig.savefig(FIG_PNG, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(FIG_PDF, bbox_inches="tight", facecolor="white")
    if SHOW_FIGURE:
        plt.show()
    else:
        plt.close(fig)


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=" * 82)
    log("Step 5.1c — Stabilization-parameter ε sensitivity")
    log("=" * 82)

    if REFERENCE_EPSILON not in EPSILON_VALUES:
        raise ValueError("REFERENCE_EPSILON must be included in EPSILON_VALUES.")

    # -------------------------------------------------------------------------
    # A. Fixed inputs and masks
    # -------------------------------------------------------------------------
    anchor = read_anchor(ANCHOR_TIF)
    study = read_mask_exact(STUDY_MASK_TIF, anchor, "study mask")
    dynamic = read_mask_exact(DYNAMIC_MASK_TIF, anchor, "dynamic mask")
    context = read_mask_exact(CONTEXT_MASK_TIF, anchor, "context mask", required=False)
    assert study is not None and dynamic is not None
    dynamic = dynamic & study
    context = (context & study) if context is not None else study.copy()

    los_mean = read_continuous_to_anchor(LOS_MEAN_TIF, anchor, "LOS mean velocity", Resampling.bilinear)
    los_count = read_continuous_to_anchor(LOS_COUNT_TIF, anchor, "LOS count", Resampling.nearest)
    coherence = read_continuous_to_anchor(COHERENCE_TIF, anchor, "PS coherence", Resampling.bilinear)
    ps_distance = read_continuous_to_anchor(PS_DISTANCE_TIF, anchor, "PS distance", Resampling.nearest)
    signed_c = read_continuous_exact(SIGNED_C_TIF, anchor, "signed geometric projection C")
    g = read_continuous_exact(G_TIF, anchor, "geometric observability G")

    geometry_domain = study & np.isfinite(signed_c) & np.isfinite(g)
    if np.any(geometry_domain):
        max_consistency_error = float(np.max(np.abs(np.abs(signed_c[geometry_domain]) - g[geometry_domain])))
        if max_consistency_error > 1.0e-5:
            raise RuntimeError(
                f"Signed C and G are inconsistent: max |abs(C)-G|={max_consistency_error:.3e}"
            )

    slope_valid = read_mask_exact(
        SLOPE_VALID_MASK_TIF, anchor, "Step 5.1b slope-valid mask", required=False
    )
    if slope_valid is None:
        slope_valid = geometry_domain.copy()
        log("[WARN] Step 5.1b slope-valid mask missing; finite geometry used instead.")
    else:
        slope_valid &= geometry_domain

    irs_support = read_mask_exact(
        IRS_SUPPORT_MASK_TIF, anchor, "IRS observation-support mask", required=False
    )
    if irs_support is None:
        irs_support = np.zeros_like(study, dtype=bool)

    # -------------------------------------------------------------------------
    # B. Source support is frozen across all ε
    # -------------------------------------------------------------------------
    densification_domain = (
        study & dynamic & np.isfinite(ps_distance) & (ps_distance <= SUPPORT_RADIUS_M)
    )

    masked_source_by_epsilon: Dict[float, np.ndarray] = {}
    source_masks = []
    for epsilon in EPSILON_VALUES:
        masked, source_mask = compute_masked_source_dvplus(
            los_mean, signed_c, los_count, coherence, study, epsilon
        )
        masked_source_by_epsilon[epsilon] = masked
        source_masks.append(source_mask)

    source_mask = source_masks[0]
    for test_mask in source_masks[1:]:
        if not np.array_equal(source_mask, test_mask):
            raise RuntimeError("DV+ source support changed with ε; the sensitivity design is not controlled.")
    source_mask &= densification_domain

    if int(source_mask.sum()) < 50:
        raise RuntimeError(f"Too few fixed DV+ source cells: {int(source_mask.sum())}")

    # Restrict source arrays exactly to the controlled Step 3.7 source mask.
    for epsilon in EPSILON_VALUES:
        arr = np.full_like(masked_source_by_epsilon[epsilon], np.nan)
        arr[source_mask] = masked_source_by_epsilon[epsilon][source_mask]
        masked_source_by_epsilon[epsilon] = arr

    # -------------------------------------------------------------------------
    # C. Identical IDW densification for all scenarios
    # -------------------------------------------------------------------------
    filled_by_epsilon = densify_all_scenarios(
        masked_source_by_epsilon, source_mask, densification_domain, anchor
    )

    # Fixed Step 4.2 observed statistics domain. Existing source-distance==0 is
    # preferred because that is exactly what the completed Step 4.2 used.
    if DV_SOURCE_DIST_TIF.exists():
        dv_source_dist = read_continuous_to_anchor(
            DV_SOURCE_DIST_TIF, anchor, "DV source-distance raster", Resampling.nearest
        )
        observed_stats_mask = dynamic & np.isfinite(dv_source_dist) & (dv_source_dist <= 1.0e-6)
    else:
        observed_stats_mask = source_mask.copy()
        log("[WARN] DV source-distance raster missing; fixed source mask used for Step 4.2 statistics.")

    fixed_lo, fixed_hi, cap_source = recover_fixed_dvplus_caps()
    log(f"[STEP 4.2] DV+ caps: {fixed_lo:.6g}, {fixed_hi:.6g} ({cap_source})")

    # -------------------------------------------------------------------------
    # D. Fixed HKFI and SNDR terms
    # -------------------------------------------------------------------------
    hkfi_t = read_continuous_exact(HKFI_T_TERM_TIF, anchor, "HKFI T term")
    hkfi_s = read_continuous_exact(HKFI_S_TERM_TIF, anchor, "HKFI support-preserved S term")
    hkfi_irs = read_continuous_exact(HKFI_IRS_EFFECTIVE_TIF, anchor, "effective IRS for HKFI")
    hkfi_valid = read_mask_exact(HKFI_VALID_MASK_TIF, anchor, "HKFI valid mask")
    assert hkfi_valid is not None
    hkfi_domain = (
        study & context & hkfi_valid
        & np.isfinite(hkfi_t) & np.isfinite(hkfi_s) & np.isfinite(hkfi_irs)
    )

    sndr_irs = read_continuous_exact(SNDR_IRS_EFFECTIVE_TIF, anchor, "effective IRS for SNDR")
    sndr_den = read_continuous_exact(SNDR_DENOMINATOR_TIF, anchor, "SNDR slope denominator")
    sndr_valid = read_mask_exact(SNDR_VALID_MASK_TIF, anchor, "SNDR valid mask")
    sndr_support = read_mask_exact(
        SNDR_KINEMATIC_SUPPORT_MASK_TIF, anchor, "SNDR kinematic support", required=False
    )
    assert sndr_valid is not None
    sndr_domain = (
        study & sndr_valid & np.isfinite(sndr_irs) & np.isfinite(sndr_den) & (sndr_den > 0.0)
    )
    if sndr_support is None:
        sndr_support = sndr_domain & (sndr_irs > 0.0)
    sndr_stats_domain = sndr_domain & sndr_support

    # -------------------------------------------------------------------------
    # E. Propagate every ε through Step 4.2, HKFI and SNDR
    # -------------------------------------------------------------------------
    winsor_by_epsilon: Dict[float, np.ndarray] = {}
    z_by_epsilon: Dict[float, np.ndarray] = {}
    hkfi_by_epsilon: Dict[float, np.ndarray] = {}
    sndr_by_epsilon: Dict[float, np.ndarray] = {}
    step42_meta: Dict[str, Any] = {}
    hkfi_meta: Dict[str, Any] = {}
    sndr_meta: Dict[str, Any] = {}

    for epsilon in EPSILON_VALUES:
        tag = epsilon_tag(epsilon)
        scenario_dir = SCENARIO_DIR / f"epsilon_{tag}"
        scenario_dir.mkdir(parents=True, exist_ok=True)
        filled = filled_by_epsilon[epsilon].astype("float64")

        winsor, z_dv, m42 = standardize_dvplus_like_step42(
            filled, dynamic, observed_stats_mask, fixed_lo, fixed_hi
        )
        winsor_by_epsilon[epsilon] = winsor
        z_by_epsilon[epsilon] = z_dv
        step42_meta[tag] = m42

        z_placeholder = np.where(np.isfinite(z_dv), z_dv, 0.0)
        hkfi_raw = np.full_like(g, np.nan, dtype="float64")
        hkfi_raw[hkfi_domain] = (
            HKFI_WEIGHTS["K"] * hkfi_irs[hkfi_domain] * z_placeholder[hkfi_domain]
            + HKFI_WEIGHTS["T"] * hkfi_t[hkfi_domain]
            + HKFI_WEIGHTS["S"] * hkfi_s[hkfi_domain]
        )
        hkfi_norm, mh = robust_minmax(hkfi_raw, hkfi_domain, HKFI_P_LOW, HKFI_P_HIGH)
        hkfi_by_epsilon[epsilon] = hkfi_norm
        hkfi_meta[tag] = mh

        sndr_raw = np.full_like(g, np.nan, dtype="float64")
        sndr_raw[sndr_domain] = 0.0
        dv_available = sndr_domain & np.isfinite(winsor)
        sndr_raw[dv_available] = (
            sndr_irs[dv_available] * winsor[dv_available] / sndr_den[dv_available]
        )
        sndr_raw[sndr_domain] = np.maximum(sndr_raw[sndr_domain], 0.0)
        sndr_norm, ms = zero_preserving_p98(sndr_raw, sndr_stats_domain, sndr_domain)
        sndr_by_epsilon[epsilon] = sndr_norm
        sndr_meta[tag] = ms

        if WRITE_SCENARIO_RASTERS:
            write_float_tif(
                scenario_dir / f"dvplus_filled_epsilon_{tag}_mm_yr_32645_12p5m_float32.tif",
                filled, anchor, f"DVplus_filled_epsilon_{epsilon:.3f}_mm_yr",
            )
            write_float_tif(
                scenario_dir / f"dvplus_winsor_epsilon_{tag}_mm_yr_32645_12p5m_float32.tif",
                winsor, anchor, f"DVplus_winsor_epsilon_{epsilon:.3f}_mm_yr",
            )
            write_float_tif(
                scenario_dir / f"z_dvplus_epsilon_{tag}_32645_12p5m_float32.tif",
                z_dv, anchor, f"z_DVplus_epsilon_{epsilon:.3f}",
            )
            write_float_tif(
                scenario_dir / f"hkfi_epsilon_{tag}_0_1_32645_12p5m_float32.tif",
                hkfi_norm, anchor, f"HKFI_epsilon_{epsilon:.3f}_0_1",
            )
            write_float_tif(
                scenario_dir / f"sndr_epsilon_{tag}_0_1_32645_12p5m_float32.tif",
                sndr_norm, anchor, f"SNDR_epsilon_{epsilon:.3f}_0_1",
            )

        log(f"[SCENARIO] ε={epsilon:.3f}, nominal amplification cap={1.0/epsilon:.2f}")

    # -------------------------------------------------------------------------
    # F. Fixed reference thresholds from ε=0.05
    # -------------------------------------------------------------------------
    ref_dv = winsor_by_epsilon[REFERENCE_EPSILON]
    ref_hkfi = hkfi_by_epsilon[REFERENCE_EPSILON]
    ref_sndr = sndr_by_epsilon[REFERENCE_EPSILON]

    dv_domain = dynamic & np.isfinite(ref_dv)
    hkfi_ref_domain = hkfi_domain & np.isfinite(ref_hkfi)
    sndr_ref_domain = sndr_domain & np.isfinite(ref_sndr)

    dv_quintiles = np.quantile(ref_dv[dv_domain], [0.2, 0.4, 0.6, 0.8])
    hkfi_quintiles = np.quantile(ref_hkfi[hkfi_ref_domain], [0.2, 0.4, 0.6, 0.8])
    sndr_quintiles = np.quantile(ref_sndr[sndr_ref_domain], [0.2, 0.4, 0.6, 0.8])
    dv_top = float(np.quantile(ref_dv[dv_domain], 1.0 - TOP_FRACTION))
    hkfi_top = float(np.quantile(ref_hkfi[hkfi_ref_domain], 1.0 - TOP_FRACTION))
    sndr_top = float(np.quantile(ref_sndr[sndr_ref_domain], 1.0 - TOP_FRACTION))

    # -------------------------------------------------------------------------
    # G. Geometric statistics
    # -------------------------------------------------------------------------
    slope_n = int(slope_valid.sum())
    finite_study_g = study & np.isfinite(g)
    irs_g_domain = study & irs_support & np.isfinite(g)
    geometric_rows = []
    for epsilon in EPSILON_VALUES:
        a = slope_valid & (g < epsilon)
        b = finite_study_g & (g < epsilon)
        c = irs_g_domain & (g < epsilon)
        geometric_rows.append({
            "epsilon": epsilon,
            "maximum_nominal_amplification": 1.0 / epsilon,
            "G_lt_epsilon_slope_valid_pixels": int(a.sum()),
            "G_lt_epsilon_slope_valid_area_km2": float(a.sum() * anchor["cell_area_m2"] / 1e6),
            "percent_slope_valid_G_lt_epsilon": 100.0 * a.sum() / max(slope_n, 1),
            "percent_finite_study_G_lt_epsilon": 100.0 * b.sum() / max(int(finite_study_g.sum()), 1),
            "percent_IRS_supported_G_lt_epsilon": 100.0 * c.sum() / max(int(irs_g_domain.sum()), 1),
        })
    geometric_df = pd.DataFrame(geometric_rows)
    geometric_df.to_csv(GEOMETRIC_CSV, index=False, float_format="%.8f")

    # -------------------------------------------------------------------------
    # H. DV+, HKFI and SNDR metrics
    # -------------------------------------------------------------------------
    dv_rows, hkfi_rows, sndr_rows = [], [], []
    for epsilon in EPSILON_VALUES:
        scenario_dv = winsor_by_epsilon[epsilon]
        mdv = comparison_metrics(ref_dv, scenario_dv, dv_domain, dv_quintiles, dv_top)
        valid = dv_domain & np.isfinite(scenario_dv)
        abs_change = np.abs(scenario_dv[valid] - ref_dv[valid])
        rel_change = abs_change / np.maximum(np.abs(ref_dv[valid]), DV_REL_DENOMINATOR_FLOOR_MM_YR)
        stw = distribution_stats(scenario_dv[valid])
        filled = filled_by_epsilon[epsilon].astype("float64")
        stf = distribution_stats(filled[dynamic & np.isfinite(filled)])
        dv_rows.append({
            "epsilon": epsilon,
            "maximum_nominal_amplification": 1.0 / epsilon,
            **mdv,
            "percent_abs_change_gt_5_mm_yr": 100.0 * np.mean(abs_change > DV_ABS_CHANGE_THRESHOLD_MM_YR),
            "percent_relative_change_gt_10pct": 100.0 * np.mean(rel_change > DV_REL_CHANGE_THRESHOLD),
            **{f"winsor_{k}_mm_yr": v for k, v in stw.items()},
            **{f"filled_{k}_mm_yr": v for k, v in stf.items()},
            "step42_mu": step42_meta[epsilon_tag(epsilon)]["mu_winsor_observed"],
            "step42_sigma": step42_meta[epsilon_tag(epsilon)]["sigma_winsor_observed"],
        })

        mh = comparison_metrics(
            ref_hkfi, hkfi_by_epsilon[epsilon], hkfi_ref_domain, hkfi_quintiles, hkfi_top
        )
        hkfi_rows.append({
            "epsilon": epsilon,
            "maximum_nominal_amplification": 1.0 / epsilon,
            **mh,
            "class_change_percent": 100.0 * mh["class_change_fraction"],
            "normalization_p2": hkfi_meta[epsilon_tag(epsilon)]["lower"],
            "normalization_p98": hkfi_meta[epsilon_tag(epsilon)]["upper"],
        })

        ms = comparison_metrics(
            ref_sndr, sndr_by_epsilon[epsilon], sndr_ref_domain, sndr_quintiles, sndr_top
        )
        sndr_rows.append({
            "epsilon": epsilon,
            "maximum_nominal_amplification": 1.0 / epsilon,
            **ms,
            "class_change_percent": 100.0 * ms["class_change_fraction"],
            "normalization_p98": sndr_meta[epsilon_tag(epsilon)]["p98"],
        })

    dv_df = pd.DataFrame(dv_rows)
    hkfi_df = pd.DataFrame(hkfi_rows)
    sndr_df = pd.DataFrame(sndr_rows)
    dv_df.to_csv(DVPLUS_CSV, index=False, float_format="%.8f")
    hkfi_df.to_csv(HKFI_CSV, index=False, float_format="%.8f")
    sndr_df.to_csv(SNDR_CSV, index=False, float_format="%.8f")

    # -------------------------------------------------------------------------
    # I. Fixed modelling-site geometry diagnostic
    # -------------------------------------------------------------------------
    model_sites_df = model_site_diagnostic(g)
    if not model_sites_df.empty:
        model_sites_df.to_csv(MODEL_SITES_CSV, index=False, float_format="%.8f")

    # -------------------------------------------------------------------------
    # J. ε=0.05 baseline reproduction QA
    # -------------------------------------------------------------------------
    authoritative_hkfi = choose_existing(AUTHORITATIVE_HKFI_CANDIDATES, "authoritative HKFI")
    qa_rows = [
        compare_to_authoritative(
            "DVplus_winsor", ref_dv, AUTHORITATIVE_DVPLUS_WINSOR_TIF, dynamic, anchor
        ),
        compare_to_authoritative(
            "HKFI", ref_hkfi, authoritative_hkfi, hkfi_ref_domain, anchor
        ),
        compare_to_authoritative(
            "SNDR", ref_sndr, AUTHORITATIVE_SNDR_TIF, sndr_ref_domain, anchor
        ),
    ]
    baseline_qa_df = pd.DataFrame(qa_rows)
    baseline_qa_df.to_csv(BASELINE_QA_CSV, index=False, float_format="%.10g")

    failed = baseline_qa_df[baseline_qa_df["status"] == "FAIL_LOW_BASELINE_REPRODUCTION"]
    if not failed.empty:
        raise RuntimeError(
            "ε=0.05 baseline reproduction failed for: "
            + ", ".join(failed["product"].astype(str))
            + f". Inspect {BASELINE_QA_CSV} before interpreting sensitivity."
        )

    # -------------------------------------------------------------------------
    # K. Combined reviewer summary
    # -------------------------------------------------------------------------
    summary_df = geometric_df[[
        "epsilon", "maximum_nominal_amplification",
        "percent_slope_valid_G_lt_epsilon", "percent_IRS_supported_G_lt_epsilon",
    ]].merge(
        dv_df[["epsilon", "spearman_rho", "normalized_rmse", "class_change_fraction", "top_decile_jaccard"]].rename(columns={
            "spearman_rho": "DVplus_spearman_rho",
            "normalized_rmse": "DVplus_normalized_rmse",
            "class_change_fraction": "DVplus_class_change_fraction",
            "top_decile_jaccard": "DVplus_top_decile_jaccard",
        }), on="epsilon"
    ).merge(
        hkfi_df[["epsilon", "spearman_rho", "normalized_rmse", "class_change_fraction", "top_decile_jaccard"]].rename(columns={
            "spearman_rho": "HKFI_spearman_rho",
            "normalized_rmse": "HKFI_normalized_rmse",
            "class_change_fraction": "HKFI_class_change_fraction",
            "top_decile_jaccard": "HKFI_top_decile_jaccard",
        }), on="epsilon"
    ).merge(
        sndr_df[["epsilon", "spearman_rho", "normalized_rmse", "class_change_fraction", "top_decile_jaccard"]].rename(columns={
            "spearman_rho": "SNDR_spearman_rho",
            "normalized_rmse": "SNDR_normalized_rmse",
            "class_change_fraction": "SNDR_class_change_fraction",
            "top_decile_jaccard": "SNDR_top_decile_jaccard",
        }), on="epsilon"
    )
    summary_df.to_csv(SUMMARY_CSV, index=False, float_format="%.8f")

    make_figure(geometric_df, dv_df, hkfi_df, sndr_df)

    # -------------------------------------------------------------------------
    # L. Metadata and report
    # -------------------------------------------------------------------------
    warnings = []
    for row in baseline_qa_df.itertuples(index=False):
        if row.status == "PASS_WITH_WARNING":
            warnings.append(
                f"Baseline reproduction for {row.product}: Spearman rho={row.spearman_rho:.4f}."
            )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "step": "5.1c Stabilization-Parameter Epsilon Sensitivity",
        "equations": {
            "DV": "v_LOS / [sign(C) * max(abs(C), epsilon)]",
            "C": "l_hat dot s_hat",
            "G": "abs(C)",
            "DVplus": "max(DV, 0)",
            "HKFI": "(K + T + S) / 3, K=IRS_HKFI*z(DVplus)",
            "SNDR": "IRS_SNDR*DVplus/(sin(theta)+delta), zero-preserving P98 normalization",
        },
        "epsilon_values": EPSILON_VALUES,
        "reference_epsilon": REFERENCE_EPSILON,
        "amplification_caps": {epsilon_tag(e): 1.0 / e for e in EPSILON_VALUES},
        "fixed_inputs": {
            "LOS_mean": str(LOS_MEAN_TIF),
            "signed_C": str(SIGNED_C_TIF),
            "G": str(G_TIF),
            "coherence": str(COHERENCE_TIF),
            "LOS_count": str(LOS_COUNT_TIF),
            "IRS": str(IRS_TIF),
            "HKFI_T": str(HKFI_T_TERM_TIF),
            "HKFI_S": str(HKFI_S_TERM_TIF),
            "HKFI_effective_IRS": str(HKFI_IRS_EFFECTIVE_TIF),
            "SNDR_effective_IRS": str(SNDR_IRS_EFFECTIVE_TIF),
            "SNDR_denominator": str(SNDR_DENOMINATOR_TIF),
        },
        "controlled_upstream_settings": {
            "min_LOS_count": MIN_LOS_COUNT,
            "min_coherence": MIN_COHERENCE,
            "DVplus_cap_mm_yr": DVPLUS_CAP_MM_YR,
            "support_radius_m": SUPPORT_RADIUS_M,
            "IDW_k": K_NEIGHBORS,
            "IDW_power": IDW_POWER,
            "max_IDW_distance_m": MAX_IDW_DIST_M,
            "same_source_support_all_epsilon": True,
            "same_IDW_neighbour_geometry_all_epsilon": True,
        },
        "step42": {
            "fixed_winsor_low": fixed_lo,
            "fixed_winsor_high": fixed_hi,
            "caps_source": cap_source,
            "scenario_statistics": step42_meta,
        },
        "HKFI": {
            "weights": HKFI_WEIGHTS,
            "normalization_percentiles": [HKFI_P_LOW, HKFI_P_HIGH],
            "scenario_normalization": hkfi_meta,
        },
        "SNDR": {
            "normalization_upper_percentile": SNDR_P_UPPER,
            "scenario_normalization": sndr_meta,
        },
        "fixed_reference_thresholds": {
            "DVplus_quintiles": dv_quintiles.tolist(),
            "HKFI_quintiles": hkfi_quintiles.tolist(),
            "SNDR_quintiles": sndr_quintiles.tolist(),
            "DVplus_top_decile_threshold": dv_top,
            "HKFI_top_decile_threshold": hkfi_top,
            "SNDR_top_decile_threshold": sndr_top,
        },
        "baseline_reproduction_QA": baseline_qa_df.to_dict(orient="records"),
        "warnings": warnings,
        "scope_note": (
            "Map-level epsilon sensitivity only. Susceptibility-model sensitivity requires "
            "rerunning selected epsilon scenarios through the unchanged spatial-CV pipeline."
        ),
        "outputs": {
            "geometric_csv": str(GEOMETRIC_CSV),
            "DVplus_csv": str(DVPLUS_CSV),
            "HKFI_csv": str(HKFI_CSV),
            "SNDR_csv": str(SNDR_CSV),
            "summary_csv": str(SUMMARY_CSV),
            "model_sites_csv": str(MODEL_SITES_CSV) if not model_sites_df.empty else None,
            "baseline_QA_csv": str(BASELINE_QA_CSV),
            "figure_png": str(FIG_PNG),
            "figure_pdf": str(FIG_PDF),
            "scenario_directory": str(SCENARIO_DIR),
            "QA_report": str(QA_REPORT_TXT),
        },
    }
    META_JSON.write_text(json.dumps(metadata, indent=2, ensure_ascii=False, default=json_safe), encoding="utf-8")

    with open(QA_REPORT_TXT, "w", encoding="utf-8") as handle:
        handle.write("STEP 5.1c — ε SENSITIVITY OF DV+, HKFI AND SNDR\n")
        handle.write("=" * 82 + "\n\n")
        handle.write(f"Reference ε: {REFERENCE_EPSILON:.3f}\n")
        handle.write(f"Tested ε: {EPSILON_VALUES}\n")
        handle.write(f"Fixed DV+ winsor caps: [{fixed_lo:.6g}, {fixed_hi:.6g}] from {cap_source}\n\n")
        handle.write("ε=0.05 baseline reproduction QA\n")
        handle.write("-" * 82 + "\n")
        handle.write(baseline_qa_df.to_string(index=False))
        handle.write("\n\nReviewer summary\n")
        handle.write("-" * 82 + "\n")
        handle.write(summary_df.to_string(index=False))
        if not model_sites_df.empty:
            handle.write("\n\nFixed Step 6.4 modelling-site geometry diagnostic\n")
            handle.write("-" * 82 + "\n")
            handle.write(model_sites_df.to_string(index=False))
        handle.write("\n\nScope\n")
        handle.write("-" * 82 + "\n")
        handle.write(
            "These outputs quantify map-level sensitivity of DV+, HKFI and SNDR. "
            "They are not susceptibility-model sensitivity results unless selected ε "
            "scenarios are propagated through the unchanged spatial-CV modelling pipeline.\n"
        )
        if warnings:
            handle.write("\nWarnings\n" + "-" * 82 + "\n")
            for warning in warnings:
                handle.write(f"- {warning}\n")

    log("=" * 82)
    log("Step 5.1c completed successfully")
    log(f"[OUT] {SUMMARY_CSV}")
    log(f"[OUT] {FIG_PNG}")
    log(f"[OUT] {META_JSON}")
    log(f"[OUT] {QA_REPORT_TXT}")
    log("=" * 82)


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 5.1c failed: {exc}")
        traceback.print_exc()
        raise


# Step-5.2: InSAR Reliability Score (IRS)

In [ ]:
# ============================================================================
# Step 5.2 — InSAR Reliability Score (IRS), corrected implementation
#
# Scientific definition 
#
#     IRS = gamma_norm * rho_PS_norm * G
#
# where:
#   gamma_norm  = locally supported, normalized interferometric coherence
#   rho_PS_norm = normalized local persistent-scatterer availability
#   G           = pure geometric observability |l_hat · s_hat| from Step 5.1
#
# Key scientific safeguards in this corrected version:
#   1. IRS is calculated over the complete study mask, not only dynamic_mask.
#   2. G is read from the corrected pure-geometry Step 5.1 output and is not
#      conditioned on coherence, PS density, or deformation availability.
#   3. Legitimate zero PS counts are preserved as zero, even when the source
#      raster incorrectly tags 0 as NoData.
#   4. Coherence is locally aggregated only where PS support exists.
#   5. PS availability is calculated in an explicit circular metric window.
#   6. The final multiplicative IRS is NOT percentile-renormalized.
#   7. Missing observation support inside the study area yields IRS = 0,
#      meaning low/absent measurement reliability, not terrain stability.
#   8. Maps and histograms use the original analysis grid without RBF, cubic,
#      IDW, or other surface interpolation.
#
# Required preceding steps:
#   Step 3.1 — coherence raster
#   Step 3.2 — LOS velocity count / PS-count raster
#   Step 5.1 — corrected pure geometric observability G
# ============================================================================

from __future__ import annotations

import csv
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.ticker import FuncFormatter
from rasterio.enums import Resampling
from rasterio.warp import reproject

try:
    from scipy.ndimage import convolve
except Exception as exc:  # pragma: no cover - local environment dependency
    raise ImportError(
        "Step 5.2 requires scipy.ndimage.convolve for a reproducible circular "
        "neighbourhood calculation. Install SciPy before running this step."
    ) from exc


# =============================================================================
# USER CONFIGURATION
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

# Anchor and masks
ANCHOR_TIF = BASE / "Results" / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
STUDY_MASK_TIF = BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"

# Optional diagnostic mask only. It is NOT used to define the IRS domain.
DYNAMIC_MASK_TIF = (
    BASE / "Results" / "Step-0.3-Quality_Masks" / "dynamic_mask.tif"
)

# Observation inputs
COHERENCE_TIF = (
    BASE
    / "Results"
    / "Step-3.1-Coherence"
    / "coherence_ps_mean_32645_12p5m_float32.tif"
)
PSCOUNT_TIF = (
    BASE
    / "Results"
    / "Step-3.2-LOS"
    / "los_velocity_count_32645_12p5m_int16.tif"
)

# Corrected Step 5.1 output: pure terrain-satellite geometry only
GEOMETRIC_G_TIF = (
    BASE
    / "Results"
    / "Step-5.1-Observability"
    / "geometric_observability_G_0_1_32645_12p5m_float32.tif"
)

# Optional cartographic overlay
ROAD_SHP = BASE / "NTD_Road.shp"

# Output directory
OUT_DIR = BASE / "Results" / "Step-5.2-IRS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Analysis outputs
COH_LOCAL_TIF = (
    OUT_DIR / "coherence_local_mean_32645_12p5m_float32.tif"
)
COH_NORM_TIF = (
    OUT_DIR / "coherence_0_1_used_32645_12p5m_float32.tif"
)
LOCAL_PS_COUNT_TIF = (
    OUT_DIR / "ps_local_count_100m_32645_12p5m_float32.tif"
)
PSD_NORM_TIF = (
    OUT_DIR / "ps_density_0_1_dense_32645_12p5m_float32.tif"
)
GEOM_USED_TIF = (
    OUT_DIR / "geom_obs_0_1_used_32645_12p5m_float32.tif"
)
SUPPORT_MASK_TIF = (
    OUT_DIR / "irs_observation_support_mask_32645_12p5m_byte.tif"
)
IRS_TIF = OUT_DIR / "irs_0_1_dense_32645_12p5m_float32.tif"

# Metadata, QA, and figures
META_JSON = OUT_DIR / "irs_metadata.json"
QA_JSON = OUT_DIR / "irs_quality_assurance.json"
QA_CSV = OUT_DIR / "irs_component_statistics.csv"
MAP_PNG = OUT_DIR / "IRS_Map.png"
MAP_PDF = OUT_DIR / "IRS_Map.pdf"
HIST_PNG = OUT_DIR / "IRS_Histogram.png"
HIST_PDF = OUT_DIR / "IRS_Histogram.pdf"

# Raster output settings
NODATA_FLOAT = -9999.0
NODATA_BYTE = 255

# -----------------------------------------------------------------------------
# Coherence normalization
# -----------------------------------------------------------------------------
# A fixed, physically interpretable linear normalization is used instead of
# study-specific percentile stretching. Coherence <= 0.30 receives reliability
# 0; coherence >= 0.95 receives reliability 1. The values can be adjusted only
# if supported by the PS-InSAR processing and sensitivity analysis.
COHERENCE_LOW = 0.30
COHERENCE_HIGH = 0.95

# -----------------------------------------------------------------------------
# Local observation support
# -----------------------------------------------------------------------------
# Radius of the circular neighbourhood used for both local PS availability and
# local PS-count-weighted coherence. Unlike a Gaussian sigma of 100 m, this is
# an explicit 100 m radius and therefore has a clearly defined spatial support.
LOCAL_RADIUS_M = 100.0

# Local PS-count normalization. The default uses the 95th percentile of positive
# local counts as a robust saturation reference. This preserves zero as zero,
# avoids unstable min-max scaling, and records the saturation value in metadata.
# Options: "p95_saturation" or "fixed_saturation".
PS_DENSITY_NORM_MODE = "p95_saturation"
PS_DENSITY_SATURATION_PERCENTILE = 95.0
PS_DENSITY_FIXED_SATURATION_COUNT = 5.0

# A cell is considered locally observation-supported when at least this many PS
# points occur in its circular neighbourhood.
MIN_LOCAL_PS_COUNT = 1.0

# Final IRS is never renormalized. Keep this False for scientific analysis.
IRS_RENORMALIZE = False

# Quality-control thresholds; these do not modify the output.
NEAR_ZERO_THRESHOLD = 0.01
WARN_NEAR_ZERO_FRACTION = 0.85
WARN_UNIQUE_VALUES_MIN = 100

# Descriptive intervals only, not calibrated reliability probabilities.
LOW_IRS_MAX = 0.30
MODERATE_IRS_MAX = 0.70

# Figures
MAKE_FIGURES = True
SHOW_FIGURES = True
MAP_FIGSIZE = (6.2, 9.0)
HIST_FIGSIZE = (6.2, 4.2)
FIG_DPI = 300


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


# =============================================================================
# RASTER I/O
# =============================================================================
def read_anchor(path: Path) -> dict[str, Any]:
    require_file(path, "anchor raster")
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"Anchor raster has no CRS: {path}")
        return {
            "transform": ds.transform,
            "crs": ds.crs,
            "width": ds.width,
            "height": ds.height,
            "profile": ds.profile.copy(),
            "bounds": ds.bounds,
            "epsg": ds.crs.to_epsg(),
            "res_x": abs(ds.transform.a),
            "res_y": abs(ds.transform.e),
        }


def same_grid(ds: rasterio.io.DatasetReader, anchor: dict[str, Any]) -> bool:
    return (
        ds.crs == anchor["crs"]
        and ds.transform == anchor["transform"]
        and ds.width == anchor["width"]
        and ds.height == anchor["height"]
    )


def scrub_common_nodata(
    arr: np.ndarray,
    tagged_nodata: float | int | None,
) -> np.ndarray:
    out = arr.astype("float64", copy=False)

    if tagged_nodata is not None:
        try:
            nd = float(tagged_nodata)
            if np.isfinite(nd):
                out = np.where(out == nd, np.nan, out)
        except (TypeError, ValueError):
            pass

    out = np.where(out == -9999.0, np.nan, out)
    out = np.where(out == -32768.0, np.nan, out)
    out = np.where(np.abs(out) > 1.0e20, np.nan, out)
    return out


def read_continuous_to_anchor(
    path: Path,
    anchor: dict[str, Any],
    description: str,
    resampling: Resampling = Resampling.bilinear,
) -> tuple[np.ndarray, dict[str, Any]]:
    """
    Read a continuous raster and align it to the anchor.

    Reprojection occurs before tagged NoData is converted to NaN, preserving
    the meaning of src_nodata during rasterio.warp.reproject.
    """
    require_file(path, description)

    with rasterio.open(path) as src:
        raw = src.read(1).astype("float64")
        source_info = {
            "path": str(path),
            "dtype": src.dtypes[0],
            "nodata": json_safe(src.nodata),
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "same_as_anchor": same_grid(src, anchor),
        }

        if src.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        if same_grid(src, anchor):
            return scrub_common_nodata(raw, src.nodata), source_info

        destination = np.full(
            (anchor["height"], anchor["width"]),
            np.nan,
            dtype="float64",
        )

        reproject(
            source=raw,
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )

        return scrub_common_nodata(destination, None), source_info


def read_pscount_to_anchor(
    path: Path,
    anchor: dict[str, Any],
    study_mask: np.ndarray,
) -> tuple[np.ndarray, dict[str, Any]]:
    """
    Read PS-count data while preserving legitimate zero-count cells.

    A PS-count value of 0 means no persistent scatterer in the cell. It must not
    be converted to NaN even when a source raster incorrectly tags 0 as NoData.
    Outside the study area, the returned array is NaN.
    """
    require_file(path, "PS-count raster")

    with rasterio.open(path) as src:
        if src.crs is None:
            raise ValueError(f"PS-count raster has no CRS: {path}")

        raw = src.read(1).astype("float64")
        tagged_nodata = src.nodata
        nodata_is_zero = False
        if tagged_nodata is not None:
            try:
                nodata_is_zero = math.isclose(float(tagged_nodata), 0.0)
            except (TypeError, ValueError):
                nodata_is_zero = False

        source_info = {
            "path": str(path),
            "dtype": src.dtypes[0],
            "nodata": json_safe(tagged_nodata),
            "nodata_is_zero": bool(nodata_is_zero),
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "same_as_anchor": same_grid(src, anchor),
        }

        if same_grid(src, anchor):
            arr = raw
            # Preserve zero. Convert only non-zero tagged NoData values.
            if tagged_nodata is not None and not nodata_is_zero:
                try:
                    nd = float(tagged_nodata)
                    if np.isfinite(nd):
                        arr = np.where(arr == nd, np.nan, arr)
                except (TypeError, ValueError):
                    pass
        else:
            destination = np.zeros(
                (anchor["height"], anchor["width"]),
                dtype="float64",
            )

            # If 0 is tagged as NoData, do not pass it as src_nodata because it
            # is scientifically meaningful for a count raster.
            src_nodata = None if nodata_is_zero else tagged_nodata

            reproject(
                source=raw,
                destination=destination,
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=src_nodata,
                dst_transform=anchor["transform"],
                dst_crs=anchor["crs"],
                dst_nodata=0.0,
                resampling=Resampling.nearest,
            )
            arr = destination

        # Defensive cleaning. Inside the study area, nonfinite and negative
        # values represent no available PS and become 0.
        arr = np.where(np.isfinite(arr), arr, 0.0)
        arr = np.where(arr < 0.0, 0.0, arr)
        arr = np.where(study_mask, arr, np.nan)

        return arr.astype("float64"), source_info


def read_mask_to_anchor(
    path: Path,
    anchor: dict[str, Any],
    required: bool = True,
) -> np.ndarray | None:
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Missing mask raster: {path}")
        return None

    with rasterio.open(path) as src:
        raw = src.read(1)
        if src.crs is None:
            raise ValueError(f"Mask raster has no CRS: {path}")

        if same_grid(src, anchor):
            arr = raw.astype("float64")
            if src.nodata is not None:
                arr = np.where(arr == src.nodata, 0.0, arr)
            return np.isfinite(arr) & (arr > 0.5)

        destination = np.zeros(
            (anchor["height"], anchor["width"]),
            dtype="float64",
        )
        reproject(
            source=raw,
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=0.0,
            resampling=Resampling.nearest,
        )
        return np.isfinite(destination) & (destination > 0.5)


def write_float_tif(
    path: Path,
    arr: np.ndarray,
    anchor: dict[str, Any],
    description: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=NODATA_FLOAT,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    output = np.where(np.isfinite(arr), arr, NODATA_FLOAT).astype("float32")
    temporary = path.with_suffix(path.suffix + ".part")

    with rasterio.open(temporary, "w", **profile) as dst:
        dst.write(output, 1)
        dst.update_tags(
            step="5.2",
            variable=description,
            generated_utc=utc_now_iso(),
        )

    os.replace(temporary, path)

    try:
        with rasterio.open(path, "r+") as ds:
            ds.build_overviews([2, 4, 8, 16], Resampling.average)
            ds.update_tags(ns="rio_overview", resampling="average")
    except Exception as exc:
        log(f"[WARN] Could not create overviews for {path.name}: {exc}")


def write_byte_tif(
    path: Path,
    arr: np.ndarray,
    anchor: dict[str, Any],
    study_mask: np.ndarray,
    description: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="uint8",
        count=1,
        nodata=NODATA_BYTE,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    output = np.full(arr.shape, NODATA_BYTE, dtype="uint8")
    output[study_mask] = np.asarray(arr[study_mask], dtype="uint8")
    temporary = path.with_suffix(path.suffix + ".part")

    with rasterio.open(temporary, "w", **profile) as dst:
        dst.write(output, 1)
        dst.update_tags(
            step="5.2",
            variable=description,
            generated_utc=utc_now_iso(),
        )

    os.replace(temporary, path)


# =============================================================================
# NUMERICAL HELPERS
# =============================================================================
def circular_kernel(radius_m: float, anchor: dict[str, Any]) -> tuple[np.ndarray, dict[str, Any]]:
    pixel_size = float(max(anchor["res_x"], anchor["res_y"]))
    if pixel_size <= 0:
        raise ValueError("Anchor pixel size must be positive.")

    radius_px_float = radius_m / pixel_size
    radius_px = max(1, int(math.ceil(radius_px_float)))

    yy, xx = np.ogrid[-radius_px : radius_px + 1, -radius_px : radius_px + 1]
    distance_m = np.sqrt((xx * anchor["res_x"]) ** 2 + (yy * anchor["res_y"]) ** 2)
    kernel = (distance_m <= radius_m + 1.0e-9).astype("float64")

    if kernel.sum() <= 0:
        raise RuntimeError("Circular neighbourhood kernel contains no cells.")

    information = {
        "radius_m": float(radius_m),
        "radius_px_float": float(radius_px_float),
        "radius_px_ceiling": int(radius_px),
        "kernel_rows": int(kernel.shape[0]),
        "kernel_cols": int(kernel.shape[1]),
        "kernel_cell_count": int(kernel.sum()),
        "nominal_window_area_m2": float(kernel.sum() * anchor["res_x"] * anchor["res_y"]),
    }
    return kernel, information


def detailed_stats(arr: np.ndarray, mask: np.ndarray) -> dict[str, float | int | None]:
    valid = mask & np.isfinite(arr)
    values = arr[valid].astype("float64")

    if values.size == 0:
        return {
            "n": 0,
            "min": None,
            "p01": None,
            "p05": None,
            "p25": None,
            "median": None,
            "mean": None,
            "std": None,
            "p75": None,
            "p95": None,
            "p99": None,
            "max": None,
            "zero_fraction": None,
            "near_zero_fraction": None,
            "unique_values_rounded_6dp": 0,
        }

    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "near_zero_fraction": float(np.mean(values < NEAR_ZERO_THRESHOLD)),
        "unique_values_rounded_6dp": int(np.unique(np.round(values, 6)).size),
    }


def descriptive_class_counts(
    values: np.ndarray,
) -> dict[str, dict[str, float | int]]:
    values = values[np.isfinite(values)]
    total = int(values.size)
    if total == 0:
        return {}

    definitions = {
        "low_0_to_lt_0.3": (values >= 0.0) & (values < LOW_IRS_MAX),
        "moderate_0.3_to_lt_0.7": (
            (values >= LOW_IRS_MAX) & (values < MODERATE_IRS_MAX)
        ),
        "high_0.7_to_1.0": (
            (values >= MODERATE_IRS_MAX) & (values <= 1.0)
        ),
    }

    result: dict[str, dict[str, float | int]] = {}
    for label, condition in definitions.items():
        count = int(np.sum(condition))
        result[label] = {
            "count": count,
            "percent": float(100.0 * count / total),
        }
    return result


def compute_local_components(
    coherence: np.ndarray,
    pscount: np.ndarray,
    geometric_g: np.ndarray,
    study_mask: np.ndarray,
    kernel: np.ndarray,
) -> dict[str, Any]:
    """
    Calculate locally supported coherence, PS availability, G used, and IRS.
    """
    # ------------------------------------------------------------------
    # PS-count surface: zeros are retained as scientifically meaningful.
    # ------------------------------------------------------------------
    ps_clean = np.zeros_like(pscount, dtype="float64")
    ps_clean[study_mask] = np.where(
        np.isfinite(pscount[study_mask]) & (pscount[study_mask] > 0.0),
        pscount[study_mask],
        0.0,
    )

    local_ps_count = convolve(
        ps_clean,
        kernel,
        mode="constant",
        cval=0.0,
    )
    local_ps_count = np.where(study_mask, local_ps_count, np.nan)

    positive_local_counts = local_ps_count[
        study_mask & np.isfinite(local_ps_count) & (local_ps_count > 0.0)
    ]

    if positive_local_counts.size == 0:
        raise RuntimeError(
            "No positive local PS counts were found in the study area. Check "
            "the PS-count raster and its NoData metadata."
        )

    if PS_DENSITY_NORM_MODE == "p95_saturation":
        saturation_count = float(
            np.percentile(
                positive_local_counts,
                PS_DENSITY_SATURATION_PERCENTILE,
            )
        )
        if not np.isfinite(saturation_count) or saturation_count <= 0.0:
            raise RuntimeError("Invalid PS-density saturation count.")
    elif PS_DENSITY_NORM_MODE == "fixed_saturation":
        saturation_count = float(PS_DENSITY_FIXED_SATURATION_COUNT)
        if saturation_count <= 0.0:
            raise ValueError("PS_DENSITY_FIXED_SATURATION_COUNT must be > 0.")
    else:
        raise ValueError(
            "PS_DENSITY_NORM_MODE must be 'p95_saturation' or "
            "'fixed_saturation'."
        )

    ps_density_norm = np.zeros_like(local_ps_count, dtype="float64")
    ps_density_norm[study_mask] = np.clip(
        local_ps_count[study_mask] / saturation_count,
        0.0,
        1.0,
    )
    ps_density_norm[~study_mask] = np.nan

    # ------------------------------------------------------------------
    # Local PS-count-weighted coherence.
    # Coherence is averaged only where a finite coherence observation and a
    # positive PS count coexist. No support yields coherence reliability 0.
    # ------------------------------------------------------------------
    coherence_clipped = np.where(
        np.isfinite(coherence),
        np.clip(coherence, 0.0, 1.0),
        np.nan,
    )

    coherence_weight = np.zeros_like(coherence_clipped, dtype="float64")
    coherence_weight[
        study_mask
        & np.isfinite(coherence_clipped)
        & np.isfinite(pscount)
        & (pscount > 0.0)
    ] = pscount[
        study_mask
        & np.isfinite(coherence_clipped)
        & np.isfinite(pscount)
        & (pscount > 0.0)
    ]

    coherence_numerator_input = np.zeros_like(coherence_clipped, dtype="float64")
    coherence_numerator_input[coherence_weight > 0.0] = (
        coherence_clipped[coherence_weight > 0.0]
        * coherence_weight[coherence_weight > 0.0]
    )

    coherence_numerator = convolve(
        coherence_numerator_input,
        kernel,
        mode="constant",
        cval=0.0,
    )
    coherence_denominator = convolve(
        coherence_weight,
        kernel,
        mode="constant",
        cval=0.0,
    )

    coherence_local = np.full_like(coherence_clipped, np.nan, dtype="float64")
    np.divide(
        coherence_numerator,
        coherence_denominator,
        out=coherence_local,
        where=coherence_denominator > 0.0,
    )
    coherence_local = np.where(study_mask, coherence_local, np.nan)

    coherence_norm = np.zeros_like(coherence_local, dtype="float64")
    coherence_supported = study_mask & np.isfinite(coherence_local)
    coherence_norm[coherence_supported] = np.clip(
        (coherence_local[coherence_supported] - COHERENCE_LOW)
        / (COHERENCE_HIGH - COHERENCE_LOW),
        0.0,
        1.0,
    )
    coherence_norm[~study_mask] = np.nan

    # ------------------------------------------------------------------
    # Pure geometric observability from corrected Step 5.1.
    # Missing G inside the study area is treated as zero reliability for IRS,
    # but the missing fraction is separately reported in QA metadata.
    # ------------------------------------------------------------------
    geometric_g_clipped = np.where(
        np.isfinite(geometric_g),
        np.clip(geometric_g, 0.0, 1.0),
        np.nan,
    )
    geometric_g_used = np.zeros_like(geometric_g_clipped, dtype="float64")
    geometric_valid = study_mask & np.isfinite(geometric_g_clipped)
    geometric_g_used[geometric_valid] = geometric_g_clipped[geometric_valid]
    geometric_g_used[~study_mask] = np.nan

    # Observation support is descriptive. It requires local PS availability,
    # local coherence support, and a defined geometric term.
    support_mask = (
        study_mask
        & (local_ps_count >= MIN_LOCAL_PS_COUNT)
        & np.isfinite(coherence_local)
        & geometric_valid
    )

    # ------------------------------------------------------------------
    # Final IRS. No post-product percentile renormalization is allowed.
    # ------------------------------------------------------------------
    irs = np.full_like(coherence_norm, np.nan, dtype="float64")
    irs[study_mask] = (
        coherence_norm[study_mask]
        * ps_density_norm[study_mask]
        * geometric_g_used[study_mask]
    )
    irs[study_mask] = np.clip(irs[study_mask], 0.0, 1.0)

    if IRS_RENORMALIZE:
        raise RuntimeError(
            "IRS_RENORMALIZE must remain False. Post-product percentile "
            "renormalization changes the physical meaning of IRS = 0 and 1."
        )

    return {
        "local_ps_count": local_ps_count,
        "ps_density_norm": ps_density_norm,
        "ps_density_saturation_count": saturation_count,
        "coherence_local": coherence_local,
        "coherence_norm": coherence_norm,
        "coherence_denominator": coherence_denominator,
        "geometric_g_used": geometric_g_used,
        "geometric_valid": geometric_valid,
        "support_mask": support_mask,
        "irs": irs,
    }


# =============================================================================
# CARTOGRAPHIC HELPERS
# =============================================================================
def try_load_roads(target_crs: Any):
    if not ROAD_SHP.exists():
        return None
    try:
        import geopandas as gpd

        roads = gpd.read_file(ROAD_SHP)
        if roads.crs is None:
            log("[WARN] Road shapefile has no CRS; assuming EPSG:4326.")
            roads = roads.set_crs("EPSG:4326")
        return roads.to_crs(target_crs)
    except Exception as exc:
        log(f"[WARN] Road overlay could not be loaded: {exc}")
        return None


def add_scale_bar_utm(ax, bounds, target_fraction: float = 0.20) -> None:
    width = bounds.right - bounds.left
    height = bounds.top - bounds.bottom
    target = width * target_fraction

    if target <= 0:
        return

    exponent = math.floor(math.log10(target))
    base = 10.0**exponent
    candidates = [1.0, 2.0, 5.0, 10.0]
    length = next((candidate * base for candidate in candidates if candidate * base >= target), 10.0 * base)

    x0 = bounds.left + 0.08 * width
    y0 = bounds.bottom + 0.06 * height
    x1 = x0 + length

    ax.plot([x0, x1], [y0, y0], color="black", linewidth=2.0, zorder=20)
    ax.plot([x0, x0], [y0 - 0.005 * height, y0 + 0.005 * height], color="black", linewidth=1.0, zorder=20)
    ax.plot([x1, x1], [y0 - 0.005 * height, y0 + 0.005 * height], color="black", linewidth=1.0, zorder=20)

    if length >= 1000.0:
        label = f"{length / 1000.0:g} km"
    else:
        label = f"{length:g} m"

    ax.text(
        (x0 + x1) / 2.0,
        y0 + 0.012 * height,
        label,
        ha="center",
        va="bottom",
        fontsize=8,
        color="black",
        zorder=20,
    )


def add_north_arrow(ax) -> None:
    ax.annotate(
        "N",
        xy=(0.09, 0.92),
        xytext=(0.09, 0.82),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        arrowprops=dict(facecolor="black", width=3, headwidth=9),
        zorder=30,
    )


def install_dms_formatters(ax, crs, bounds) -> bool:
    try:
        from pyproj import Transformer

        transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
        x_mid = (bounds.left + bounds.right) / 2.0
        y_mid = (bounds.bottom + bounds.top) / 2.0

        def decimal_to_dms(value: float, longitude: bool) -> str:
            absolute = abs(value)
            degrees = int(absolute)
            minutes_float = (absolute - degrees) * 60.0
            minutes = int(minutes_float)
            seconds = (minutes_float - minutes) * 60.0
            if longitude:
                hemisphere = "E" if value >= 0 else "W"
            else:
                hemisphere = "N" if value >= 0 else "S"
            return f"{degrees}°{minutes:02d}'{seconds:03.0f}\"{hemisphere}"

        def format_x(x_value, _position):
            longitude, _ = transformer.transform(x_value, y_mid)
            return decimal_to_dms(longitude, True)

        def format_y(y_value, _position):
            _, latitude = transformer.transform(x_mid, y_value)
            return decimal_to_dms(latitude, False)

        ax.xaxis.set_major_formatter(FuncFormatter(format_x))
        ax.yaxis.set_major_formatter(FuncFormatter(format_y))
        return True
    except Exception as exc:
        log(f"[WARN] DMS labels unavailable: {exc}")
        return False


def create_figures(
    irs: np.ndarray,
    study_mask: np.ndarray,
    anchor: dict[str, Any],
) -> None:
    if not MAKE_FIGURES:
        return

    bounds = anchor["bounds"]
    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    plot_array = np.where(study_mask, irs, np.nan)
    valid_values = plot_array[study_mask & np.isfinite(plot_array)]

    if valid_values.size == 0:
        log("[WARN] No finite IRS values available for figures.")
        return

    roads = try_load_roads(anchor["crs"])

    # Map: direct raster display, no scientific surface interpolation.
    figure, axis = plt.subplots(figsize=MAP_FIGSIZE, facecolor="white")
    image = axis.imshow(
        plot_array,
        extent=extent,
        origin="upper",
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        interpolation="nearest",
    )

    if roads is not None:
        roads.plot(ax=axis, color="red", linewidth=0.8, zorder=10)

    add_north_arrow(axis)
    if anchor["crs"].is_projected:
        add_scale_bar_utm(axis, bounds)
        install_dms_formatters(axis, anchor["crs"], bounds)
        axis.set_xlabel("Longitude", fontsize=9)
        axis.set_ylabel("Latitude", fontsize=9)
    else:
        axis.set_xlabel("Longitude", fontsize=9)
        axis.set_ylabel("Latitude", fontsize=9)

    axis.set_title("InSAR Reliability Score (IRS)", fontsize=11, fontweight="bold")
    axis.tick_params(labelsize=7)
    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")

    colorbar = figure.colorbar(image, ax=axis, fraction=0.035, pad=0.025)
    colorbar.set_label("IRS (0–1)", fontsize=9)
    colorbar.ax.tick_params(labelsize=7)

    figure.tight_layout()
    figure.savefig(MAP_PNG, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    figure.savefig(MAP_PDF, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    log(f"[FIGURE] {MAP_PNG}")

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(figure)

    # Histogram: exact same study-area values used in the map.
    figure_h, axis_h = plt.subplots(figsize=HIST_FIGSIZE, facecolor="white")
    axis_h.hist(
        valid_values,
        bins=np.linspace(0.0, 1.0, 31),
        edgecolor="black",
        linewidth=0.5,
        alpha=0.85,
    )

    mean_value = float(np.mean(valid_values))
    median_value = float(np.median(valid_values))
    p95_value = float(np.percentile(valid_values, 95))

    axis_h.axvline(mean_value, linestyle="--", linewidth=1.0, label=f"Mean = {mean_value:.3f}")
    axis_h.axvline(median_value, linestyle="--", linewidth=1.0, label=f"Median = {median_value:.3f}")
    axis_h.axvline(p95_value, linestyle="--", linewidth=1.0, label=f"P95 = {p95_value:.3f}")

    axis_h.set_xlim(0.0, 1.0)
    axis_h.set_xlabel("IRS (0–1)", fontsize=9)
    axis_h.set_ylabel("Number of study-area pixels", fontsize=9)
    axis_h.set_title("Distribution of InSAR Reliability Score", fontsize=11, fontweight="bold")
    axis_h.tick_params(labelsize=8)
    axis_h.grid(axis="y", alpha=0.25)
    axis_h.legend(fontsize=8)

    figure_h.tight_layout()
    figure_h.savefig(HIST_PNG, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    figure_h.savefig(HIST_PDF, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    log(f"[FIGURE] {HIST_PNG}")

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(figure_h)


# =============================================================================
# QA OUTPUTS
# =============================================================================
def write_statistics_csv(
    path: Path,
    component_statistics: dict[str, dict[str, Any]],
) -> None:
    fieldnames = [
        "component",
        "n",
        "min",
        "p01",
        "p05",
        "p25",
        "median",
        "mean",
        "std",
        "p75",
        "p95",
        "p99",
        "max",
        "zero_fraction",
        "near_zero_fraction",
        "unique_values_rounded_6dp",
    ]

    with open(path, "w", newline="", encoding="utf-8") as file_handle:
        writer = csv.DictWriter(file_handle, fieldnames=fieldnames)
        writer.writeheader()
        for component_name, statistics in component_statistics.items():
            row = {"component": component_name}
            row.update(statistics)
            writer.writerow(row)


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=" * 76)
    log("Step 5.2 — Corrected InSAR Reliability Score (IRS)")
    log("=" * 76)

    for input_path, description in [
        (ANCHOR_TIF, "anchor raster"),
        (STUDY_MASK_TIF, "study mask"),
        (COHERENCE_TIF, "coherence raster"),
        (PSCOUNT_TIF, "PS-count raster"),
        (GEOMETRIC_G_TIF, "corrected geometric observability G raster"),
    ]:
        require_file(input_path, description)

    if IRS_RENORMALIZE:
        raise RuntimeError(
            "IRS_RENORMALIZE must be False in the corrected scientific workflow."
        )

    if not (0.0 <= COHERENCE_LOW < COHERENCE_HIGH <= 1.0):
        raise ValueError(
            "Require 0 <= COHERENCE_LOW < COHERENCE_HIGH <= 1."
        )

    anchor = read_anchor(ANCHOR_TIF)
    study_mask = read_mask_to_anchor(STUDY_MASK_TIF, anchor, required=True)
    assert study_mask is not None
    dynamic_mask = read_mask_to_anchor(DYNAMIC_MASK_TIF, anchor, required=False)

    log(
        f"[ANCHOR] {anchor['width']} x {anchor['height']}; "
        f"EPSG:{anchor['epsg']}; resolution="
        f"{anchor['res_x']:.3f} x {anchor['res_y']:.3f} m"
    )
    log(f"[DOMAIN] Study-area pixels: {int(study_mask.sum()):,}")
    if dynamic_mask is not None:
        log(
            f"[DIAGNOSTIC ONLY] dynamic_mask pixels inside study: "
            f"{int(np.sum(dynamic_mask & study_mask)):,}. "
            "This mask does not restrict IRS calculation."
        )

    coherence, coherence_source_info = read_continuous_to_anchor(
        COHERENCE_TIF,
        anchor,
        "coherence raster",
        resampling=Resampling.bilinear,
    )
    pscount, pscount_source_info = read_pscount_to_anchor(
        PSCOUNT_TIF,
        anchor,
        study_mask,
    )
    geometric_g, geometric_source_info = read_continuous_to_anchor(
        GEOMETRIC_G_TIF,
        anchor,
        "geometric observability G",
        resampling=Resampling.bilinear,
    )

    coherence = np.where(
        study_mask & np.isfinite(coherence),
        np.clip(coherence, 0.0, 1.0),
        np.nan,
    )
    geometric_g = np.where(
        study_mask & np.isfinite(geometric_g),
        np.clip(geometric_g, 0.0, 1.0),
        np.nan,
    )

    log(
        f"[PSCOUNT] Source nodata={pscount_source_info['nodata']}; "
        f"nodata_is_zero={pscount_source_info['nodata_is_zero']}"
    )
    log(
        f"[PSCOUNT] Positive source cells in study="
        f"{int(np.sum(study_mask & np.isfinite(pscount) & (pscount > 0.0))):,}; "
        f"max cell count={float(np.nanmax(pscount[study_mask])):.3f}"
    )

    kernel, kernel_info = circular_kernel(LOCAL_RADIUS_M, anchor)
    log(
        f"[WINDOW] Circular radius={LOCAL_RADIUS_M:.1f} m; "
        f"kernel cells={kernel_info['kernel_cell_count']}; "
        f"array={kernel_info['kernel_rows']} x {kernel_info['kernel_cols']}"
    )

    components = compute_local_components(
        coherence=coherence,
        pscount=pscount,
        geometric_g=geometric_g,
        study_mask=study_mask,
        kernel=kernel,
    )

    local_ps_count = components["local_ps_count"]
    ps_density_norm = components["ps_density_norm"]
    coherence_local = components["coherence_local"]
    coherence_norm = components["coherence_norm"]
    geometric_g_used = components["geometric_g_used"]
    geometric_valid = components["geometric_valid"]
    support_mask = components["support_mask"]
    irs = components["irs"]
    ps_density_saturation_count = components["ps_density_saturation_count"]

    # Preserve study domain in all outputs.
    coherence_local_out = np.where(study_mask, coherence_local, np.nan)
    coherence_norm_out = np.where(study_mask, coherence_norm, np.nan)
    local_ps_count_out = np.where(study_mask, local_ps_count, np.nan)
    ps_density_norm_out = np.where(study_mask, ps_density_norm, np.nan)
    geometric_g_out = np.where(study_mask, geometric_g_used, np.nan)
    irs_out = np.where(study_mask, irs, np.nan)

    # ------------------------------------------------------------------
    # Detailed QA
    # ------------------------------------------------------------------
    component_statistics = {
        "coherence_source": detailed_stats(coherence, study_mask),
        "coherence_local_mean": detailed_stats(coherence_local_out, study_mask),
        "coherence_normalized": detailed_stats(coherence_norm_out, study_mask),
        "local_ps_count": detailed_stats(local_ps_count_out, study_mask),
        "ps_density_normalized": detailed_stats(ps_density_norm_out, study_mask),
        "geometric_G_used": detailed_stats(geometric_g_out, study_mask),
        "IRS": detailed_stats(irs_out, study_mask),
    }

    for name, statistics in component_statistics.items():
        log(f"[STATS] {name}: {statistics}")

    study_pixel_count = int(study_mask.sum())
    support_pixel_count = int(np.sum(support_mask))
    source_coherence_pixel_count = int(np.sum(study_mask & np.isfinite(coherence)))
    source_ps_positive_cell_count = int(
        np.sum(study_mask & np.isfinite(pscount) & (pscount > 0.0))
    )
    geometric_valid_count = int(np.sum(geometric_valid))

    dynamic_diagnostics: dict[str, Any] | None = None
    if dynamic_mask is not None:
        dynamic_inside = dynamic_mask & study_mask
        dynamic_count = int(dynamic_inside.sum())
        dynamic_diagnostics = {
            "dynamic_mask_pixels_inside_study": dynamic_count,
            "dynamic_mask_fraction_of_study": (
                float(dynamic_count / study_pixel_count)
                if study_pixel_count > 0
                else None
            ),
            "IRS_mean_inside_dynamic_mask": (
                float(np.nanmean(irs_out[dynamic_inside]))
                if np.any(dynamic_inside)
                else None
            ),
            "IRS_mean_outside_dynamic_mask_but_inside_study": (
                float(np.nanmean(irs_out[study_mask & ~dynamic_inside]))
                if np.any(study_mask & ~dynamic_inside)
                else None
            ),
        }

    warnings: list[str] = []
    irs_statistics = component_statistics["IRS"]

    if irs_statistics["near_zero_fraction"] is not None:
        if float(irs_statistics["near_zero_fraction"]) > WARN_NEAR_ZERO_FRACTION:
            warnings.append(
                "More than the configured fraction of study-area IRS values "
                f"are below {NEAR_ZERO_THRESHOLD}. Inspect coherence support, "
                "PS density, and G before continuing to HKFI/SNDR."
            )

    if int(irs_statistics["unique_values_rounded_6dp"] or 0) < WARN_UNIQUE_VALUES_MIN:
        warnings.append(
            "IRS has fewer unique rounded values than expected for a continuous "
            "reliability surface. Do not continue to HKFI/SNDR until checked."
        )

    if geometric_valid_count < study_pixel_count:
        warnings.append(
            f"Geometric G is undefined in {study_pixel_count - geometric_valid_count:,} "
            "study-area pixels; IRS was set to 0 in those cells and the missing "
            "fraction is reported explicitly."
        )

    if pscount_source_info["nodata_is_zero"]:
        warnings.append(
            "The PS-count raster tags 0 as NoData. This code intentionally "
            "preserved zero as a valid count, but the source raster should be "
            "regenerated with a distinct NoData value for long-term clarity."
        )

    for warning in warnings:
        log(f"[QA WARNING] {warning}")

    irs_values = irs_out[study_mask & np.isfinite(irs_out)]
    class_counts = descriptive_class_counts(irs_values)

    qa = {
        "generated_at_utc": utc_now_iso(),
        "equation": "IRS = gamma_norm * rho_PS_norm * G [manuscript Eq. 2]",
        "study_pixel_count": study_pixel_count,
        "source_coherence_finite_pixel_count": source_coherence_pixel_count,
        "source_ps_positive_cell_count": source_ps_positive_cell_count,
        "geometric_G_valid_pixel_count": geometric_valid_count,
        "local_observation_support_pixel_count": support_pixel_count,
        "local_observation_support_fraction": float(
            support_pixel_count / study_pixel_count
        ),
        "component_statistics": component_statistics,
        "descriptive_IRS_classes": class_counts,
        "dynamic_mask_diagnostics": dynamic_diagnostics,
        "warnings": warnings,
        "pass_basic_range_check": bool(
            np.nanmin(irs_values) >= -1.0e-9
            and np.nanmax(irs_values) <= 1.0 + 1.0e-9
        ),
    }

    # ------------------------------------------------------------------
    # Write outputs
    # ------------------------------------------------------------------
    write_float_tif(
        COH_LOCAL_TIF,
        coherence_local_out,
        anchor,
        "local PS-count-weighted mean coherence",
    )
    write_float_tif(
        COH_NORM_TIF,
        coherence_norm_out,
        anchor,
        "normalized locally supported coherence gamma_norm",
    )
    write_float_tif(
        LOCAL_PS_COUNT_TIF,
        local_ps_count_out,
        anchor,
        f"local PS count within {LOCAL_RADIUS_M:g} m circular radius",
    )
    write_float_tif(
        PSD_NORM_TIF,
        ps_density_norm_out,
        anchor,
        "normalized local persistent-scatterer availability rho_PS_norm",
    )
    write_float_tif(
        GEOM_USED_TIF,
        geometric_g_out,
        anchor,
        "pure geometric observability G used in IRS",
    )
    write_byte_tif(
        SUPPORT_MASK_TIF,
        support_mask.astype("uint8"),
        anchor,
        study_mask,
        "local IRS observation-support mask",
    )
    write_float_tif(
        IRS_TIF,
        irs_out,
        anchor,
        "InSAR Reliability Score IRS, no post-product renormalization",
    )

    write_statistics_csv(QA_CSV, component_statistics)

    with open(QA_JSON, "w", encoding="utf-8") as file_handle:
        json.dump(qa, file_handle, indent=2, default=json_safe)

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "step": "5.2",
        "scientific_name": "InSAR Reliability Score",
        "abbreviation": "IRS",
        "manuscript_equation_number": 2,
        "formula": "IRS = gamma_norm * rho_PS_norm * G",
        "interpretation": (
            "IRS is a relative observation-reliability weight. Low IRS indicates "
            "weak or absent interferometric support and must not be interpreted "
            "as terrain stability."
        ),
        "domain": {
            "analysis_domain": "complete study mask",
            "study_mask": str(STUDY_MASK_TIF),
            "dynamic_mask": str(DYNAMIC_MASK_TIF),
            "dynamic_mask_role": "diagnostic only; not used to restrict IRS",
            "study_pixels": study_pixel_count,
        },
        "anchor": {
            "path": str(ANCHOR_TIF),
            "epsg": anchor["epsg"],
            "width": anchor["width"],
            "height": anchor["height"],
            "resolution_x_m": anchor["res_x"],
            "resolution_y_m": anchor["res_y"],
        },
        "inputs": {
            "coherence": coherence_source_info,
            "pscount": pscount_source_info,
            "geometric_G": geometric_source_info,
        },
        "coherence_processing": {
            "local_aggregation": "PS-count-weighted mean in circular window",
            "normalization": "fixed linear clipping",
            "coherence_low_maps_to_0": COHERENCE_LOW,
            "coherence_high_maps_to_1": COHERENCE_HIGH,
        },
        "ps_density_processing": {
            "local_measure": "sum of PS counts in circular window",
            "window": kernel_info,
            "normalization_mode": PS_DENSITY_NORM_MODE,
            "saturation_percentile": (
                PS_DENSITY_SATURATION_PERCENTILE
                if PS_DENSITY_NORM_MODE == "p95_saturation"
                else None
            ),
            "fixed_saturation_count": (
                PS_DENSITY_FIXED_SATURATION_COUNT
                if PS_DENSITY_NORM_MODE == "fixed_saturation"
                else None
            ),
            "realized_saturation_count": ps_density_saturation_count,
            "minimum_local_PS_count_for_support": MIN_LOCAL_PS_COUNT,
        },
        "geometric_processing": {
            "definition": "G = |l_hat dot s_hat|",
            "source": str(GEOMETRIC_G_TIF),
            "post_normalization": "none; clipped only to [0,1]",
        },
        "final_IRS_processing": {
            "operator": "multiplicative product",
            "post_product_renormalization": False,
            "missing_observation_support_inside_study": (
                "components are assigned reliability 0; output remains within "
                "the study domain"
            ),
        },
        "outputs": {
            "coherence_local_mean": str(COH_LOCAL_TIF),
            "coherence_normalized": str(COH_NORM_TIF),
            "local_ps_count": str(LOCAL_PS_COUNT_TIF),
            "ps_density_normalized": str(PSD_NORM_TIF),
            "geometric_G_used": str(GEOM_USED_TIF),
            "support_mask": str(SUPPORT_MASK_TIF),
            "IRS": str(IRS_TIF),
            "QA_json": str(QA_JSON),
            "QA_csv": str(QA_CSV),
        },
    }

    with open(META_JSON, "w", encoding="utf-8") as file_handle:
        json.dump(metadata, file_handle, indent=2, default=json_safe)

    create_figures(irs_out, study_mask, anchor)

    log("[OUTPUTS]")
    for output_path in [
        COH_LOCAL_TIF,
        COH_NORM_TIF,
        LOCAL_PS_COUNT_TIF,
        PSD_NORM_TIF,
        GEOM_USED_TIF,
        SUPPORT_MASK_TIF,
        IRS_TIF,
        META_JSON,
        QA_JSON,
        QA_CSV,
    ]:
        log(f"  {output_path}")

    log("=" * 76)
    if warnings:
        log(
            "Step 5.2 completed with QA warnings. Review irs_quality_assurance.json "
            "before running HKFI or SNDR."
        )
    else:
        log("Step 5.2 completed without QA warnings.")
    log("=" * 76)


if __name__ == "__main__":
    main()

# Visualization: InSAR Reliability Score (IRS)

In [ ]:
# ============================================================================
# Step 5.2 — Visualization of the Corrected InSAR Reliability Score (IRS)
#
# This script is ONLY for visualization and summary statistics.
# It reads the corrected analytical IRS raster generated by Step 5.2:
#
#   irs_0_1_dense_32645_12p5m_float32.tif
#
# Scientific safeguards:
#   - Uses the original 12.5 m analysis grid directly
#   - No RBF, cubic, IDW, or other surface interpolation
#   - No boundary erosion
#   - Includes legitimate IRS = 0 values inside the study area
#   - Calculates coverage relative to the study-area mask
#   - Separately reports observation-support coverage
#   - Uses low/moderate/high IRS intervals descriptively only
#   - Does not renormalize or alter IRS values for visualization
#
# ============================================================================

from __future__ import annotations
import os

import csv
import json
import math
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon, Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False


# =============================================================================
# PATHS
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

IRS_DIR = BASE / "Results" / "Step-5.2-IRS"

# Corrected analytical IRS output from the revised Step 5.2
IRS_TIF = IRS_DIR / "irs_0_1_dense_32645_12p5m_float32.tif"

# Corrected observation-support mask from the revised Step 5.2
SUPPORT_MASK_TIF = (
    IRS_DIR / "irs_observation_support_mask_32645_12p5m_byte.tif"
)

# Metadata and QA generated by the revised Step 5.2
IRS_META_JSON = IRS_DIR / "irs_metadata.json"
IRS_QA_JSON = IRS_DIR / "irs_quality_assurance.json"

# Study-area mask and optional highway overlay
STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
ROAD_SHP = BASE / "NTD_Road.shp"

# Separate visualization outputs.
# These do not overwrite the figures generated inside the analytical Step 5.2.
OUT_MAP_PNG = IRS_DIR / "IRS Map.png"
OUT_MAP_PDF = IRS_DIR / "IRS Map.pdf"
OUT_HIST_PNG = IRS_DIR / "IRS Histogram.png"
OUT_HIST_PDF = IRS_DIR / "IRS Histogram.pdf"
OUT_SUMMARY_JSON = IRS_DIR / "irs_visualization_summary.json"
OUT_STATS_CSV = IRS_DIR / "irs_visualization_statistics.csv"

IRS_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# DISPLAY SETTINGS
# =============================================================================
MAP_FIGSIZE = (2, 2)
HIST_FIGSIZE = (2, 2)

FIG_DPI = 300
CAX_SIZE = "5%"
CAX_PAD = 0.01

IRS_CMAP = plt.cm.plasma
RASTER_DISPLAY_INTERPOLATION = "nearest"

# Preserve the main-map DMS convention.
DMS_INTERVAL_MINUTES = 5

# Descriptive intervals only, not calibrated reliability probabilities.
LOW_IRS_MAX = 0.30
MODERATE_IRS_MAX = 0.70

# QA threshold used by the corrected analytical Step 5.2.
NEAR_ZERO_THRESHOLD = 0.01

SHOW_FIGURES = True
SAVE_FIGURES = True
SHOW_SUPPORT_BOUNDARY = False
HISTOGRAM_LOG_Y = False

# Compact typography for the requested 2 x 2 inch figures.
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "DejaVu Serif"]
plt.rcParams["axes.unicode_minus"] = False


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def load_json_optional(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return None


# =============================================================================
# RASTER HELPERS
# =============================================================================
def same_grid(ds: rasterio.io.DatasetReader, ref: dict[str, Any]) -> bool:
    return (
        ds.crs == ref["crs"]
        and ds.transform == ref["transform"]
        and ds.width == ref["width"]
        and ds.height == ref["height"]
    )


def clean_float_nodata(
    array: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    result = array.astype("float64", copy=True)

    if nodata is not None:
        try:
            nd = float(nodata)
            if np.isfinite(nd):
                result[result == nd] = np.nan
        except (TypeError, ValueError):
            pass

    result[result == -9999.0] = np.nan
    result[result == -32768.0] = np.nan
    result[np.abs(result) > 1.0e20] = np.nan
    return result


def read_irs_raster(path: Path) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(path, "corrected IRS raster")

    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"IRS raster has no CRS: {path}")

        raw = ds.read(1)
        irs = clean_float_nodata(raw, ds.nodata)

        ref = {
            "crs": ds.crs,
            "transform": ds.transform,
            "width": ds.width,
            "height": ds.height,
            "bounds": ds.bounds,
            "epsg": ds.crs.to_epsg(),
            "resolution_x": abs(ds.transform.a),
            "resolution_y": abs(ds.transform.e),
            "profile": ds.profile.copy(),
        }

    invalid_range = (
        np.isfinite(irs)
        & ((irs < -1.0e-6) | (irs > 1.0 + 1.0e-6))
    )
    if invalid_range.any():
        raise ValueError(
            f"IRS contains {int(invalid_range.sum()):,} values outside [0,1]. "
            "Do not visualize this raster until the analytical Step 5.2 "
            "range error is resolved."
        )

    # Defensive floating-point clipping only. This is not renormalization.
    irs = np.where(
        np.isfinite(irs),
        np.clip(irs, 0.0, 1.0),
        np.nan,
    )
    return irs, ref


def read_mask_to_reference(
    path: Path,
    ref: dict[str, Any],
    description: str,
    optional: bool = False,
) -> np.ndarray | None:
    if not path.exists():
        if optional:
            return None
        raise FileNotFoundError(f"Missing {description}: {path}")

    with rasterio.open(path) as ds:
        raw = ds.read(1)
        src_nodata = ds.nodata

        if same_grid(ds, ref):
            values = raw.astype("float64")
            if src_nodata is not None:
                values[raw == src_nodata] = 0.0
            values[~np.isfinite(values)] = 0.0
            return values > 0.5

        destination = np.zeros(
            (ref["height"], ref["width"]),
            dtype="float64",
        )

        reproject(
            source=raw,
            destination=destination,
            src_transform=ds.transform,
            src_crs=ds.crs,
            src_nodata=src_nodata,
            dst_transform=ref["transform"],
            dst_crs=ref["crs"],
            dst_nodata=0,
            resampling=Resampling.nearest,
        )

        destination[~np.isfinite(destination)] = 0.0
        return destination > 0.5


# =============================================================================
# STATISTICS
# =============================================================================
def detailed_stats(values: np.ndarray) -> dict[str, float | int]:
    values = values[np.isfinite(values)].astype("float64")

    if values.size == 0:
        return {"n": 0}

    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "near_zero_fraction_lt_0_01": float(
            np.mean(values < NEAR_ZERO_THRESHOLD)
        ),
        "unique_values_rounded_6dp": int(
            np.unique(np.round(values, 6)).size
        ),
    }


def descriptive_classes(
    values: np.ndarray,
) -> dict[str, dict[str, float | int]]:
    values = values[np.isfinite(values)]
    total = int(values.size)

    if total == 0:
        return {}

    definitions = {
        "low_0_to_lt_0.3": (
            (values >= 0.0) & (values < LOW_IRS_MAX)
        ),
        "moderate_0.3_to_lt_0.7": (
            (values >= LOW_IRS_MAX)
            & (values < MODERATE_IRS_MAX)
        ),
        "high_0.7_to_1.0": (
            (values >= MODERATE_IRS_MAX)
            & (values <= 1.0)
        ),
    }

    result: dict[str, dict[str, float | int]] = {}

    for label, condition in definitions.items():
        count = int(condition.sum())
        result[label] = {
            "count": count,
            "percentage": 100.0 * count / max(total, 1),
        }

    return result


# =============================================================================
# CARTOGRAPHIC HELPERS
# =============================================================================
def to_dms_label(value: float, is_longitude: bool) -> str:
    absolute = abs(value)
    degrees = int(absolute)
    minutes_float = (absolute - degrees) * 60.0
    minutes = int(round(minutes_float))

    if minutes == 60:
        degrees += 1
        minutes = 0

    if is_longitude:
        hemisphere = "E" if value >= 0 else "W"
    else:
        hemisphere = "N" if value >= 0 else "S"

    return f"{degrees}°{minutes:02d}'{hemisphere}"


def multiples_in_range(
    minimum: float,
    maximum: float,
    interval: float,
) -> np.ndarray:
    start = math.ceil(minimum / interval) * interval
    stop = math.floor(maximum / interval) * interval

    if stop < start:
        return np.array([], dtype="float64")

    count = int(round((stop - start) / interval)) + 1
    return start + np.arange(count, dtype="float64") * interval


def install_fixed_dms_ticks(
    axis,
    raster_crs,
    bounds,
    interval_minutes: int = DMS_INTERVAL_MINUTES,
) -> bool:
    try:
        from pyproj import Transformer
    except Exception:
        log("[INFO] pyproj unavailable; UTM coordinates will be displayed.")
        return False

    to_ll = Transformer.from_crs(
        raster_crs,
        "EPSG:4326",
        always_xy=True,
    )
    from_ll = Transformer.from_crs(
        "EPSG:4326",
        raster_crs,
        always_xy=True,
    )

    corner_x = [
        bounds.left,
        bounds.right,
        bounds.left,
        bounds.right,
    ]
    corner_y = [
        bounds.bottom,
        bounds.bottom,
        bounds.top,
        bounds.top,
    ]

    longitudes, latitudes = to_ll.transform(corner_x, corner_y)

    lon_min = float(np.min(longitudes))
    lon_max = float(np.max(longitudes))
    lat_min = float(np.min(latitudes))
    lat_max = float(np.max(latitudes))

    lon_mid = 0.5 * (lon_min + lon_max)
    lat_mid = 0.5 * (lat_min + lat_max)

    interval_degrees = float(interval_minutes) / 60.0
    lon_ticks = multiples_in_range(lon_min, lon_max, interval_degrees)
    lat_ticks = multiples_in_range(lat_min, lat_max, interval_degrees)

    if lon_ticks.size:
        x_positions, _ = from_ll.transform(
            lon_ticks,
            np.full(lon_ticks.shape, lat_mid),
        )
        x_positions = np.asarray(x_positions)
        keep_x = (
            np.isfinite(x_positions)
            & (x_positions >= bounds.left)
            & (x_positions <= bounds.right)
        )
        axis.set_xticks(x_positions[keep_x])
        axis.set_xticklabels(
            [
                to_dms_label(value, True)
                for value in lon_ticks[keep_x]
            ]
        )

    if lat_ticks.size:
        _, y_positions = from_ll.transform(
            np.full(lat_ticks.shape, lon_mid),
            lat_ticks,
        )
        y_positions = np.asarray(y_positions)
        keep_y = (
            np.isfinite(y_positions)
            & (y_positions >= bounds.bottom)
            & (y_positions <= bounds.top)
        )
        axis.set_yticks(y_positions[keep_y])
        axis.set_yticklabels(
            [
                to_dms_label(value, False)
                for value in lat_ticks[keep_y]
            ]
        )

    return bool(lon_ticks.size or lat_ticks.size)


def add_north_arrow_tri(
    axis,
    size: float = 0.09,
    pad: float = 0.09,
    label: str = "N",
    arrow_downshift: float = 0.025,
) -> None:
    label_x = pad + size * 0.36
    label_y = (1.0 - pad) + size * 0.10

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=0.7,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=0.7,
            zorder=20,
        )
    )
    axis.text(
        label_x,
        label_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=6,
        fontweight="bold",
        zorder=21,
    )


def nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = math.floor(math.log10(target_metres))
    base = 10.0 ** exponent

    for multiplier in (1.0, 2.0, 5.0, 10.0):
        candidate = multiplier * base
        if candidate >= target_metres:
            return candidate

    return 10.0 * base


def add_checker_scale_bar(
    axis,
    bounds,
    segments: int = 4,
) -> None:
    map_width = bounds.right - bounds.left
    map_height = bounds.top - bounds.bottom

    total_length = nice_length_125(map_width / 5.0)
    segment_length = total_length / float(segments)

    x_start = bounds.left + 0.5 * map_width - 0.5 * total_length
    y_base = bounds.bottom + 0.06 * map_height
    bar_height = 0.018 * map_height

    background_pad_x = total_length * 0.08
    background_pad_y = 0.035 * map_height

    axis.add_patch(
        Rectangle(
            (
                x_start - background_pad_x,
                y_base - background_pad_y,
            ),
            total_length + 2.0 * background_pad_x,
            bar_height + 2.0 * background_pad_y,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (
                    x_start + index * segment_length,
                    y_base,
                ),
                segment_length,
                bar_height,
                facecolor=(
                    "black" if index % 2 == 0 else "white"
                ),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length,
            bar_height,
            fill=False,
            edgecolor="black",
            linewidth=0.35,
            zorder=14,
        )
    )

    axis.text(
        x_start,
        y_base - 0.012 * map_height,
        "0",
        ha="center",
        va="top",
        fontsize=3.5,
        zorder=15,
    )
    axis.text(
        x_start + total_length,
        y_base - 0.012 * map_height,
        f"{total_length / 1000.0:g}",
        ha="center",
        va="top",
        fontsize=3.5,
        zorder=15,
    )
    axis.text(
        x_start + total_length / 2.0,
        y_base + bar_height + 0.012 * map_height,
        "km",
        ha="center",
        va="bottom",
        fontsize=3.5,
        zorder=15,
    )


def load_roads(target_crs):
    if not ROAD_SHP.exists():
        log(f"[INFO] Road overlay skipped: {ROAD_SHP}")
        return None

    try:
        import geopandas as gpd

        roads = gpd.read_file(ROAD_SHP)

        if roads.crs is None:
            raise ValueError(
                "The road shapefile has no CRS. Assign its correct CRS "
                "before plotting."
            )

        roads = roads.to_crs(target_crs)
        log(f"[ROAD] Loaded {len(roads):,} feature(s)")
        return roads

    except Exception as exc:
        log(f"[WARN] Road overlay unavailable: {exc}")
        return None


# =============================================================================
# MAP
# =============================================================================
def create_map(
    irs: np.ndarray,
    valid_mask: np.ndarray,
    study_mask: np.ndarray,
    support_mask: np.ndarray | None,
    ref: dict[str, Any],
) -> None:
    bounds = ref["bounds"]
    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    plot_array = np.ma.masked_invalid(
        np.where(valid_mask, irs, np.nan)
    )

    figure, axis = plt.subplots(
        figsize=MAP_FIGSIZE,
        facecolor="white",
    )
    axis.set_facecolor("white")

    image = axis.imshow(
        plot_array,
        cmap=IRS_CMAP,
        extent=extent,
        origin="upper",
        interpolation=RASTER_DISPLAY_INTERPOLATION,
        norm=Normalize(vmin=0.0, vmax=1.0),
        zorder=1,
    )

    # Original, non-eroded study-area boundary.
    axis.contour(
        study_mask.astype("uint8"),
        levels=[0.5],
        colors="black",
        linewidths=0.55,
        extent=extent,
        origin="upper",
        zorder=9,
    )

    if SHOW_SUPPORT_BOUNDARY and support_mask is not None:
        axis.contour(
            support_mask.astype("uint8"),
            levels=[0.5],
            colors="white",
            linewidths=0.35,
            linestyles="--",
            extent=extent,
            origin="upper",
            zorder=8,
        )

    roads = load_roads(ref["crs"])
    if roads is not None and not roads.empty:
        roads.plot(
            ax=axis,
            color="red",
            linewidth=0.55,
            zorder=10,
        )

    axis.set_xlim(bounds.left, bounds.right)
    axis.set_ylim(bounds.bottom, bounds.top)

    dms_installed = install_fixed_dms_ticks(
        axis,
        ref["crs"],
        bounds,
        interval_minutes=DMS_INTERVAL_MINUTES,
    )

    axis.set_xlabel(
        "Longitude" if dms_installed else "Easting (m)",
        fontsize=5,
        fontweight="bold",
    )
    axis.set_ylabel(
        "Latitude" if dms_installed else "Northing (m)",
        fontsize=5,
        fontweight="bold",
    )
    axis.tick_params(axis="both", labelsize=3, pad=1)

    if dms_installed:
        plt.setp(
            axis.get_xticklabels(),
            rotation=45,
            ha="right",
        )

    add_north_arrow_tri(axis)
    add_checker_scale_bar(axis, bounds)

    legend_elements = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=0.8,
            label="Highway",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=0.8,
            label="Study Area",
        ),
    ]

    if SHOW_SUPPORT_BOUNDARY and support_mask is not None:
        legend_elements.append(
            Line2D(
                [0],
                [0],
                color="white",
                markeredgecolor="black",
                linewidth=0.8,
                linestyle="--",
                label="Observation Support",
            )
        )

    legend = axis.legend(
        handles=legend_elements,
        loc="center",
        bbox_to_anchor=(0.82, 0.15),
        bbox_transform=axis.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=2.5,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0,
    )
    legend.get_frame().set_edgecolor("lightgray")
    legend.get_frame().set_linewidth(0.3)

    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes(
        "right",
        size=CAX_SIZE,
        pad=CAX_PAD,
    )
    colorbar = figure.colorbar(image, cax=colorbar_axis)
    colorbar.ax.tick_params(labelsize=3, pad=1)
    colorbar.set_label(
        "IRS (0–1)",
        fontsize=4,
        fontweight="bold",
    )
    colorbar.set_ticks(np.linspace(0.0, 1.0, 5))

    figure.tight_layout(pad=0.3)

    if SAVE_FIGURES:
        figure.savefig(
            OUT_MAP_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_MAP_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        log(f"[MAP] {OUT_MAP_PNG}")
        log(f"[MAP] {OUT_MAP_PDF}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()
    plt.close(figure)


# =============================================================================
# HISTOGRAM
# =============================================================================
def create_histogram(
    values: np.ndarray,
    stats: dict[str, float | int],
) -> None:
    figure, axis = plt.subplots(
        figsize=HIST_FIGSIZE,
        facecolor="white",
    )

    bins = np.linspace(0.0, 1.0, 21)
    counts, edges = np.histogram(values, bins=bins)
    centres = 0.5 * (edges[:-1] + edges[1:])
    colours = IRS_CMAP(centres)

    axis.bar(
        centres,
        counts,
        width=(bins[1] - bins[0]) * 0.90,
        color=colours,
        edgecolor="black",
        linewidth=0.35,
        alpha=0.95,
    )

    axis.axvline(
        stats["mean"],
        linestyle="--",
        linewidth=0.65,
        label=f"Mean: {stats['mean']:.3f}",
    )
    axis.axvline(
        stats["median"],
        linestyle=":",
        linewidth=0.75,
        label=f"Median: {stats['median']:.3f}",
    )
    axis.axvline(
        stats["p95"],
        linestyle="-.",
        linewidth=0.65,
        label=f"P95: {stats['p95']:.3f}",
    )

    if HISTOGRAM_LOG_Y:
        axis.set_yscale("log")

    axis.set_xlim(0.0, 1.0)
    axis.set_xlabel(
        "IRS (0–1)",
        fontsize=4,
        fontweight="bold",
    )
    axis.set_ylabel(
        "Number of Pixels",
        fontsize=4,
        fontweight="bold",
    )
    axis.tick_params(axis="both", labelsize=4, pad=1)
    axis.grid(
        True,
        axis="y",
        alpha=0.25,
        linewidth=0.35,
    )
    axis.set_axisbelow(True)
    axis.legend(
        fontsize=3.2,
        loc="upper right",
        framealpha=0.9,
        borderpad=0.3,
        labelspacing=0.25,
        handlelength=1.2,
    )

    figure.tight_layout(pad=0.8)

    if SAVE_FIGURES:
        figure.savefig(
            OUT_HIST_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_HIST_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        log(f"[HISTOGRAM] {OUT_HIST_PNG}")
        log(f"[HISTOGRAM] {OUT_HIST_PDF}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()
    plt.close(figure)


# =============================================================================
# SUMMARY OUTPUT
# =============================================================================
def write_stats_csv(
    path: Path,
    stats: dict[str, float | int],
    classes: dict[str, dict[str, float | int]],
    coverage: dict[str, float | int | None],
) -> None:
    rows: list[tuple[str, float | int | str | None]] = []

    for key, value in coverage.items():
        rows.append((f"coverage_{key}", value))

    for key, value in stats.items():
        rows.append((f"IRS_{key}", value))

    for class_name, item in classes.items():
        rows.append(
            (
                f"{class_name}_count",
                item["count"],
            )
        )
        rows.append(
            (
                f"{class_name}_percentage",
                item["percentage"],
            )
        )

    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["metric", "value"])
        writer.writerows(rows)


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=== Step 5.2 visualization: corrected IRS ===")

    irs, ref = read_irs_raster(IRS_TIF)

    study_mask = read_mask_to_reference(
        STUDY_MASK_TIF,
        ref,
        "study-area mask",
    )

    support_mask = read_mask_to_reference(
        SUPPORT_MASK_TIF,
        ref,
        "IRS observation-support mask",
        optional=True,
    )

    valid_mask = (
        study_mask
        & np.isfinite(irs)
        & (irs >= 0.0)
        & (irs <= 1.0)
    )

    study_pixels = int(study_mask.sum())
    valid_pixels = int(valid_mask.sum())

    if study_pixels == 0:
        raise RuntimeError("The study-area mask contains no valid pixels.")

    if valid_pixels == 0:
        raise RuntimeError(
            "No finite IRS values occur within the study area."
        )

    values = irs[valid_mask]
    stats = detailed_stats(values)
    classes = descriptive_classes(values)

    valid_coverage_percent = (
        100.0 * valid_pixels / study_pixels
    )

    if support_mask is not None:
        support_pixels = int((support_mask & study_mask).sum())
        support_fraction = support_pixels / study_pixels
        support_percent = 100.0 * support_fraction
    else:
        support_pixels = None
        support_fraction = None
        support_percent = None

    coverage = {
        "study_area_pixels": study_pixels,
        "valid_IRS_pixels": valid_pixels,
        "valid_IRS_coverage_percent": valid_coverage_percent,
        "observation_support_pixels": support_pixels,
        "observation_support_fraction": support_fraction,
        "observation_support_percent": support_percent,
    }

    metadata = load_json_optional(IRS_META_JSON)
    qa = load_json_optional(IRS_QA_JSON)

    log(
        f"[RASTER] EPSG:{ref['epsg']} | "
        f"{ref['width']} x {ref['height']} | "
        f"{ref['resolution_x']:.3f} m"
    )
    log(
        f"[COVERAGE] Valid IRS: {valid_pixels:,}/{study_pixels:,} "
        f"study pixels ({valid_coverage_percent:.2f}%)"
    )

    if support_percent is not None:
        log(
            f"[SUPPORT] {support_pixels:,}/{study_pixels:,} "
            f"study pixels ({support_percent:.2f}%)"
        )

    log(
        f"[IRS] min={stats['min']:.4f}, "
        f"median={stats['median']:.4f}, "
        f"mean={stats['mean']:.4f}, "
        f"p95={stats['p95']:.4f}, "
        f"max={stats['max']:.4f}"
    )
    log(
        f"[IRS] zero={100.0 * stats['zero_fraction']:.2f}% | "
        f"<{NEAR_ZERO_THRESHOLD:.2f}="
        f"{100.0 * stats['near_zero_fraction_lt_0_01']:.2f}% | "
        f"unique(6 dp)={stats['unique_values_rounded_6dp']:,}"
    )

    if metadata:
        formula = metadata.get(
            "formula",
            "IRS = gamma_norm * rho_PS_norm * G",
        )
        post_renorm = (
            metadata
            .get("final_IRS_processing", {})
            .get("post_product_renormalization")
        )
        log(f"[METADATA] Formula: {formula}")
        log(
            "[METADATA] Post-product renormalization: "
            f"{post_renorm}"
        )

    if qa:
        warnings = qa.get("warnings", [])
        if warnings:
            log("[QA WARNINGS]")
            for warning in warnings:
                log(f"  - {warning}")
        else:
            log("[QA] No warnings recorded by analytical Step 5.2.")

    print("\nDescriptive IRS intervals")
    print(
        "These intervals are descriptive only and are not calibrated "
        "probabilities of observation success."
    )
    for class_name, item in classes.items():
        print(
            f"  {class_name}: {item['count']:,} "
            f"({item['percentage']:.2f}%)"
        )

    summary = {
        "generated_at_utc": utc_now_iso(),
        "purpose": (
            "Visualization and descriptive summary of the corrected "
            "analytical IRS raster."
        ),
        "scientific_interpretation": (
            "Low IRS denotes weak or absent observation reliability and "
            "must not be interpreted as terrain stability."
        ),
        "formula": (
            metadata.get("formula")
            if metadata
            else "IRS = gamma_norm * rho_PS_norm * G"
        ),
        "post_product_renormalization": (
            metadata
            .get("final_IRS_processing", {})
            .get("post_product_renormalization")
            if metadata
            else False
        ),
        "inputs": {
            "IRS_raster": str(IRS_TIF),
            "study_mask": str(STUDY_MASK_TIF),
            "support_mask": (
                str(SUPPORT_MASK_TIF)
                if support_mask is not None
                else None
            ),
            "metadata": (
                str(IRS_META_JSON)
                if metadata is not None
                else None
            ),
            "quality_assurance": (
                str(IRS_QA_JSON)
                if qa is not None
                else None
            ),
        },
        "raster": {
            "epsg": ref["epsg"],
            "width": ref["width"],
            "height": ref["height"],
            "resolution_x_m": ref["resolution_x"],
            "resolution_y_m": ref["resolution_y"],
        },
        "coverage": coverage,
        "statistics": stats,
        "descriptive_classes": {
            "warning": (
                "The 0–0.3, 0.3–0.7, and 0.7–1.0 intervals are "
                "descriptive and are not calibrated reliability classes."
            ),
            "counts": classes,
        },
        "visualization_settings": {
            "map_figsize_inches": list(MAP_FIGSIZE),
            "histogram_figsize_inches": list(HIST_FIGSIZE),
            "surface_interpolation_used": False,
            "boundary_erosion_used": False,
            "raster_display_interpolation": (
                RASTER_DISPLAY_INTERPOLATION
            ),
            "colormap": "plasma",
            "DMS_interval_minutes": DMS_INTERVAL_MINUTES,
            "histogram_log_y": HISTOGRAM_LOG_Y,
        },
        "analytical_QA_warnings": (
            qa.get("warnings", [])
            if qa
            else None
        ),
        "outputs": {
            "map_png": str(OUT_MAP_PNG),
            "map_pdf": str(OUT_MAP_PDF),
            "histogram_png": str(OUT_HIST_PNG),
            "histogram_pdf": str(OUT_HIST_PDF),
            "summary_json": str(OUT_SUMMARY_JSON),
            "statistics_csv": str(OUT_STATS_CSV),
        },
    }

    with open(
        OUT_SUMMARY_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    write_stats_csv(
        OUT_STATS_CSV,
        stats,
        classes,
        coverage,
    )

    create_map(
        irs,
        valid_mask,
        study_mask,
        support_mask,
        ref,
    )
    create_histogram(values, stats)

    log(f"[SUMMARY] {OUT_SUMMARY_JSON}")
    log(f"[SUMMARY] {OUT_STATS_CSV}")
    log("=== IRS visualization completed successfully ===")


def create_irs_visualization() -> None:
    """Notebook-friendly wrapper retained from the earlier code."""
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] {exc}")
        traceback.print_exc()
        raise


if __name__ == "__main__":
    create_irs_visualization()


# Step 5.3 — Multiscale Hydro–Kinematic Fusion Index (HKFI)

In [ ]:
# =============================================================================
# Step 5.3 — Multiscale Hydro–Kinematic Fusion Index (HKFI)
# =============================================================================
#
# Revised scientific formulation
# ------------------------------
# Local kinematic component:
#
#     K(x) = IRS_HKFI(x) * z(DV+(x))
#
# Local topographic-wetness component:
#
#     T(x) = z(TWI(x))
#
# Regional soil-moisture component:
#
#     S_j = z_native(SM_j)
#
# where SM_j is the monsoon-mean soil moisture of coarse support cell j.
# S_j is standardized across UNIQUE coarse support cells and assigned to every
# 12.5 m pixel belonging to cell j using nearest-neighbour support preservation.
#
# Local hydro-kinematic term:
#
#     H_local(x) = [K(x) + T(x)] / 2
#
# Primary multiscale HKFI:
#
#     HKFI_MS_raw(x)
#       = (1 - lambda_SM) * H_local(x) + lambda_SM * S_j(x)
#
# with lambda_SM = 1/3:
#
#     HKFI_MS_raw(x) = [K(x) + T(x) + S_j(x)] / 3
#
# The nominal component weights therefore remain equal, but the spatial support
# of ERA5-Land soil moisture is explicitly preserved. Within a coarse support
# cell, the soil-moisture term is constant; local differentiation is supplied
# by K and T.
#
# IMPORTANT
# ---------
# 1. The old 12.5 m z(SM) raster is NOT used as the primary SM input.
#    It is read only as an optional diagnostic comparator.
#
# 2. Preferred support-source hierarchy:
#       a. exact native/coarse ERA5-Land raster;
#       b. exact regular coarse grid reconstructed from the monthly CSV when
#          longitude, latitude, and soil-moisture columns are present;
#       c. clearly labelled approximate 9 km support grid generated from the
#          physical 12.5 m monsoon-mean SM raster.
#
# 3. Standardization of SM is performed across UNIQUE support-cell values,
#    not across repeated 12.5 m pixels. This avoids area-weighting the same
#    coarse observation thousands of times.
#
# 4. The approximate 9 km fallback is scientifically useful, but it must not
#    be described as the exact native ERA5 grid.
#
# 5. IRS remains a [0,1] reliability weight and is not re-standardized.
#
# 6. Missing z(DV+) sets I_DV=0 and IRS_HKFI=IRS*I_DV=0. The kinematic
#    contribution is neutral zero without interpreting missing deformation as
#    observed stability.
#
# Required preceding steps
# ------------------------
# Step 4.2:
#   z_dvplus_32645_12p5m_float32.tif
#   z_twi_32645_12p5m_float32.tif
#
# Step 5.2:
#   corrected dense IRS raster
#
# Soil moisture:
#   Results/Step-2.3-SM/
#       sm_monsoon_mean_2020_2023_32645_12p5m_float32.tif
#       sm_monsoon_monthly_2020_2023.csv
# =============================================================================

from __future__ import annotations

import csv
import json
import math
import os
import re
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.warp import reproject, transform_bounds


# =============================================================================
# USER CONFIGURATION
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

# Anchor and masks
ANCHOR_TIF = (
    BASE / "Results" / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
)
STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
CONTEXT_MASK_TIF = (
    BASE / "Results" / "Step-0.3-Quality_Masks" / "context_mask.tif"
)
DYNAMIC_MASK_TIF = (
    BASE / "Results" / "Step-0.3-Quality_Masks" / "dynamic_mask.tif"
)

# Step 4.2 standardized local inputs
STANDARDIZATION_DIR = BASE / "Results" / "Step-4.2-Standardization"
Z_DV_TIF = STANDARDIZATION_DIR / "z_dvplus_32645_12p5m_float32.tif"
Z_TWI_TIF = STANDARDIZATION_DIR / "z_twi_32645_12p5m_float32.tif"

# Old/interpolated z(SM), diagnostic only
OLD_Z_SM_TIF = STANDARDIZATION_DIR / "z_sm_32645_12p5m_float32.tif"
ZSCORE_META_JSON = STANDARDIZATION_DIR / "zscore_metadata.json"

# Step 5.2 corrected IRS
IRS_DIR = BASE / "Results" / "Step-5.2-IRS"
IRS_TIF = IRS_DIR / "irs_0_1_dense_32645_12p5m_float32.tif"
IRS_SUPPORT_MASK_TIF = (
    IRS_DIR / "irs_observation_support_mask_32645_12p5m_byte.tif"
)
IRS_META_JSON = IRS_DIR / "irs_metadata.json"

# Soil-moisture products confirmed by the user
SM_SOURCE_DIR = BASE / "Results" / "Step-2.3-SM"
SM_MONSOON_MEAN_TIF = (
    SM_SOURCE_DIR
    / "sm_monsoon_mean_2020_2023_32645_12p5m_float32.tif"
)
SM_MONSOON_MONTHLY_CSV = (
    SM_SOURCE_DIR
    / "sm_monsoon_monthly_2020_2023.csv"
)

# Optional exact native/coarse source raster.
# Set this manually when the original ERA5-Land raster is available.
ERA5_NATIVE_SM_OVERRIDE: Path | None = None

ERA5_NATIVE_SM_CANDIDATES = [
    SM_SOURCE_DIR / "era5_land_soil_moisture_native.tif",
    SM_SOURCE_DIR / "soil_moisture_native.tif",
    SM_SOURCE_DIR / "sm_native.tif",
    BASE / "Data" / "ERA5-Land" / "era5_land_soil_moisture_native.tif",
    BASE / "Data" / "ERA5-Land" / "soil_moisture_native.tif",
    BASE / "Data" / "ERA5" / "era5_soil_moisture_native.tif",
]

ERA5_NATIVE_BAND = 1
MIN_EXPECTED_NATIVE_CELL_SIZE_M = 1000.0

# Approximate support fallback
ALLOW_APPROXIMATE_9KM_AS_PRIMARY = True
APPROXIMATE_SUPPORT_CELL_SIZE_M = 9000.0

# Primary multiscale weight
LAMBDA_SM = 1.0 / 3.0

# Optional sensitivity scenarios written by this step
WRITE_WEIGHT_SENSITIVITY_RASTERS = True
WEIGHT_SCENARIOS = {
    "no_sm": 0.00,
    "reduced_sm": 0.20,
    "primary_equal_component": 1.0 / 3.0,
    "enhanced_sm": 0.50,
}

# Output directory
OUT_DIR = BASE / "Results" / "Step-5.3-HKFI"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Primary compatibility outputs used by downstream code
HKFI_RAW_TIF = (
    OUT_DIR / "hkfi_raw_zmean_32645_12p5m_float32.tif"
)
HKFI_NORM_TIF = (
    OUT_DIR / "hkfi_norm_0_1_dense_32645_12p5m_float32.tif"
)
HKFI_COMPAT_TIF = (
    OUT_DIR / "hkfi_norm_0_1_32645_12p5m_float32.tif"
)

# Explicit multiscale aliases
HKFI_MS_RAW_TIF = (
    OUT_DIR / "hkfi_multiscale_raw_32645_12p5m_float32.tif"
)
HKFI_MS_NORM_TIF = (
    OUT_DIR / "hkfi_multiscale_norm_0_1_32645_12p5m_float32.tif"
)

# Component and support outputs
KINEMATIC_TERM_TIF = (
    OUT_DIR / "hkfi_term_K_irs_times_zdv_32645_12p5m_float32.tif"
)
TWI_TERM_TIF = (
    OUT_DIR / "hkfi_term_T_ztwi_32645_12p5m_float32.tif"
)
SM_SUPPORT_PHYSICAL_TIF = (
    OUT_DIR / "sm_support_preserved_physical_32645_12p5m_float32.tif"
)
SM_SUPPORT_Z_TIF = (
    OUT_DIR / "hkfi_term_S_zsm_support_preserved_32645_12p5m_float32.tif"
)
SM_SUPPORT_CELL_ID_TIF = (
    OUT_DIR / "sm_support_cell_id_32645_12p5m_int32.tif"
)
LOCAL_HYDRO_KINEMATIC_TIF = (
    OUT_DIR / "hkfi_local_K_plus_T_over_2_32645_12p5m_float32.tif"
)
HKFI_OLD_SM_DIFFERENCE_TIF = (
    OUT_DIR / "hkfi_multiscale_minus_old_interpolated_sm_raw_32645_12p5m_float32.tif"
)

# Missing-data and domain outputs
HKFI_VALID_MASK_TIF = (
    OUT_DIR / "hkfi_valid_mask_32645_12p5m_byte.tif"
)
HKFI_DV_AVAILABILITY_MASK_TIF = (
    OUT_DIR / "hkfi_dv_availability_mask_32645_12p5m_byte.tif"
)
HKFI_MISSING_ZDV_MASK_TIF = (
    OUT_DIR / "hkfi_missing_zdv_mask_32645_12p5m_byte.tif"
)
HKFI_UNRESOLVED_MASK_TIF = (
    OUT_DIR
    / "hkfi_missing_zdv_nonzero_original_irs_mask_32645_12p5m_byte.tif"
)
IRS_EFFECTIVE_HKFI_TIF = (
    OUT_DIR / "irs_effective_for_hkfi_0_1_32645_12p5m_float32.tif"
)

# Sensitivity raster names
SENSITIVITY_DIR = OUT_DIR / "SM_weight_sensitivity"
SENSITIVITY_DIR.mkdir(parents=True, exist_ok=True)

# Metadata and QA
META_JSON = OUT_DIR / "hkfi_metadata.json"
QA_JSON = OUT_DIR / "hkfi_quality_assurance.json"
QA_CSV = OUT_DIR / "hkfi_component_statistics.csv"
SUPPORT_CELL_TABLE_CSV = OUT_DIR / "sm_support_cell_values.csv"
SM_CSV_PROFILE_JSON = OUT_DIR / "sm_monthly_csv_profile.json"
SM_TEMPORAL_SUMMARY_CSV = OUT_DIR / "sm_monthly_temporal_summary.csv"
SM_SUPPORT_MODE_JSON = OUT_DIR / "sm_support_mode.json"
WEIGHT_SENSITIVITY_CSV = OUT_DIR / "hkfi_sm_weight_sensitivity.csv"

# Optional QA figures
MAKE_QA_FIGURES = True
SHOW_QA_FIGURES = True
QA_MAP_PNG = OUT_DIR / "HKFI_QA_Map.png"
QA_HIST_PNG = OUT_DIR / "HKFI_QA_Histogram.png"
QA_MAP_FIGSIZE = (6.2, 9.0)
QA_HIST_FIGSIZE = (6.2, 4.2)
QA_FIG_DPI = 300

# Raster settings
NODATA_FLOAT = -9999.0
NODATA_BYTE = 255
NODATA_INT32 = -2147483648

# Scientific controls
IRS_EFFECTIVE_ZERO_TOL = 1.0e-6
REQUIRE_EXACT_GRID_FOR_LOCAL_INPUTS = True

# SM standardization at unique support-cell level
SM_WINSOR_P_LOW = 2.0
SM_WINSOR_P_HIGH = 98.0

# Final HKFI normalization
HKFI_NORM_P_LOW = 2.0
HKFI_NORM_P_HIGH = 98.0

# QA thresholds
WARN_INPUT_Z_MEAN_ABS = 0.25
WARN_INPUT_Z_STD_LOW = 0.70
WARN_INPUT_Z_STD_HIGH = 1.30
WARN_HKFI_UNIQUE_VALUES_MIN = 100
WARN_VALID_DOMAIN_FRACTION = 0.90
WARN_IRS_NEAR_ZERO_FRACTION = 0.85
WARN_APPROXIMATE_SUPPORT = True
WARN_K_ZERO_FRACTION = 0.70

EPS = 1.0e-12


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def load_json_optional(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return None


def normalize_column_name(name: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).strip().lower(),
    ).strip("_")


def detect_column(
    columns: list[str],
    exact_names: Iterable[str],
    required_tokens: Iterable[str] = (),
) -> str | None:
    normalized = {
        normalize_column_name(column): column
        for column in columns
    }

    for candidate in exact_names:
        key = normalize_column_name(candidate)
        if key in normalized:
            return normalized[key]

    required_tokens = tuple(required_tokens)
    if required_tokens:
        for normalized_name, original in normalized.items():
            if all(token in normalized_name for token in required_tokens):
                return original

    return None


# =============================================================================
# RASTER I/O
# =============================================================================
def read_anchor(path: Path) -> dict[str, Any]:
    require_file(path, "anchor raster")
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"Anchor raster has no CRS: {path}")

        transform = dataset.transform
        cell_area_m2 = abs(
            transform.a * transform.e
            - transform.b * transform.d
        )

        return {
            "transform": transform,
            "crs": dataset.crs,
            "width": dataset.width,
            "height": dataset.height,
            "profile": dataset.profile.copy(),
            "bounds": dataset.bounds,
            "epsg": dataset.crs.to_epsg(),
            "res_x": abs(transform.a),
            "res_y": abs(transform.e),
            "cell_area_m2": float(cell_area_m2),
        }


def same_grid(
    dataset: rasterio.io.DatasetReader,
    anchor: dict[str, Any],
) -> bool:
    return (
        dataset.crs == anchor["crs"]
        and dataset.transform == anchor["transform"]
        and dataset.width == anchor["width"]
        and dataset.height == anchor["height"]
    )


def scrub_common_nodata(
    array: np.ndarray,
    tagged_nodata: float | int | None,
) -> np.ndarray:
    result = array.astype("float64", copy=True)

    if tagged_nodata is not None:
        try:
            nodata_value = float(tagged_nodata)
            if np.isfinite(nodata_value):
                result[result == nodata_value] = np.nan
        except (TypeError, ValueError):
            pass

    result[result == -9999.0] = np.nan
    result[result == -32768.0] = np.nan
    result[np.abs(result) > 1.0e20] = np.nan
    return result


def read_continuous_exact(
    path: Path,
    anchor: dict[str, Any],
    description: str,
) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(path, description)

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        is_same = same_grid(dataset, anchor)

        if REQUIRE_EXACT_GRID_FOR_LOCAL_INPUTS and not is_same:
            raise ValueError(
                f"{description} does not match the anchor grid. "
                f"Regenerate the upstream raster rather than silently "
                f"resampling it:\n{path}"
            )

        array = scrub_common_nodata(
            dataset.read(1),
            dataset.nodata,
        )

        information = {
            "path": str(path),
            "dtype": dataset.dtypes[0],
            "nodata": json_safe(dataset.nodata),
            "crs": str(dataset.crs),
            "width": dataset.width,
            "height": dataset.height,
            "same_as_anchor": bool(is_same),
        }

    return array, information


def read_mask(
    path: Path,
    anchor: dict[str, Any],
    required: bool,
    description: str,
) -> tuple[np.ndarray | None, dict[str, Any] | None]:
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Missing {description}: {path}")
        return None, None

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        raw = dataset.read(1)
        is_same = same_grid(dataset, anchor)

        if is_same:
            values = raw.astype("float64")
            if dataset.nodata is not None:
                values[raw == dataset.nodata] = 0.0
            values[~np.isfinite(values)] = 0.0
            output = values > 0.5
        else:
            destination = np.zeros(
                (anchor["height"], anchor["width"]),
                dtype="float64",
            )
            reproject(
                source=raw,
                destination=destination,
                src_transform=dataset.transform,
                src_crs=dataset.crs,
                src_nodata=dataset.nodata,
                dst_transform=anchor["transform"],
                dst_crs=anchor["crs"],
                dst_nodata=0.0,
                resampling=Resampling.nearest,
            )
            destination[~np.isfinite(destination)] = 0.0
            output = destination > 0.5

        information = {
            "path": str(path),
            "crs": str(dataset.crs),
            "width": dataset.width,
            "height": dataset.height,
            "same_as_anchor": bool(is_same),
        }

    return output, information


def write_float_tif(
    path: Path,
    array: np.ndarray,
    anchor: dict[str, Any],
    variable: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=NODATA_FLOAT,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    output = np.where(
        np.isfinite(array),
        array,
        NODATA_FLOAT,
    ).astype("float32")

    temporary = Path(str(path) + ".part")
    with rasterio.open(temporary, "w", **profile) as dataset:
        dataset.write(output, 1)
        dataset.set_band_description(1, variable)
        dataset.update_tags(
            step="5.3_multiscale_HKFI",
            variable=variable,
            generated_utc=utc_now_iso(),
        )

    os.replace(temporary, path)


def write_byte_tif(
    path: Path,
    boolean_array: np.ndarray,
    anchor: dict[str, Any],
    study_mask: np.ndarray,
    variable: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="uint8",
        count=1,
        nodata=NODATA_BYTE,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    output = np.full(
        boolean_array.shape,
        NODATA_BYTE,
        dtype="uint8",
    )
    output[study_mask] = boolean_array[study_mask].astype("uint8")

    temporary = Path(str(path) + ".part")
    with rasterio.open(temporary, "w", **profile) as dataset:
        dataset.write(output, 1)
        dataset.set_band_description(1, variable)
        dataset.update_tags(
            step="5.3_multiscale_HKFI",
            variable=variable,
            generated_utc=utc_now_iso(),
        )

    os.replace(temporary, path)


def write_int32_tif(
    path: Path,
    array: np.ndarray,
    anchor: dict[str, Any],
    variable: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="int32",
        count=1,
        nodata=NODATA_INT32,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    output = np.where(
        np.isfinite(array),
        array,
        NODATA_INT32,
    ).astype("int32")

    temporary = Path(str(path) + ".part")
    with rasterio.open(temporary, "w", **profile) as dataset:
        dataset.write(output, 1)
        dataset.set_band_description(1, variable)
        dataset.update_tags(
            step="5.3_multiscale_HKFI",
            variable=variable,
            generated_utc=utc_now_iso(),
        )

    os.replace(temporary, path)


# =============================================================================
# NUMERICAL HELPERS
# =============================================================================
def detailed_stats(
    array: np.ndarray,
    mask: np.ndarray,
) -> dict[str, float | int | None]:
    valid = mask & np.isfinite(array)
    values = array[valid].astype("float64")

    if values.size == 0:
        return {
            "n": 0,
            "min": None,
            "p01": None,
            "p05": None,
            "p25": None,
            "median": None,
            "mean": None,
            "std": None,
            "variance": None,
            "p75": None,
            "p95": None,
            "p99": None,
            "max": None,
            "zero_fraction": None,
            "near_zero_fraction": None,
            "unique_values_rounded_6dp": 0,
        }

    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "variance": float(np.var(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "near_zero_fraction": float(
            np.mean(np.abs(values) < 0.01)
        ),
        "unique_values_rounded_6dp": int(
            np.unique(np.round(values, 6)).size
        ),
    }


def safe_pearson(
    first: np.ndarray,
    second: np.ndarray,
    mask: np.ndarray,
) -> float | None:
    valid = (
        mask
        & np.isfinite(first)
        & np.isfinite(second)
    )

    if int(valid.sum()) < 3:
        return None

    x = first[valid].astype("float64")
    y = second[valid].astype("float64")

    if np.std(x) <= EPS or np.std(y) <= EPS:
        return None

    return float(np.corrcoef(x, y)[0, 1])


def robust_minmax_normalize(
    array: np.ndarray,
    stats_mask: np.ndarray,
    apply_mask: np.ndarray,
    p_low: float,
    p_high: float,
) -> tuple[np.ndarray, dict[str, float | int]]:
    values = array[
        stats_mask & np.isfinite(array)
    ].astype("float64")

    if values.size < 50:
        raise RuntimeError(
            "Too few valid HKFI pixels for stable normalization."
        )

    lower = float(np.percentile(values, p_low))
    upper = float(np.percentile(values, p_high))

    if (
        not np.isfinite(lower)
        or not np.isfinite(upper)
        or upper <= lower + EPS
    ):
        raise RuntimeError(
            "Invalid HKFI normalization bounds."
        )

    normalized = np.full_like(
        array,
        np.nan,
        dtype="float64",
    )
    valid_apply = apply_mask & np.isfinite(array)

    scaled = (
        array[valid_apply] - lower
    ) / (upper - lower)

    normalized[valid_apply] = np.clip(
        scaled,
        0.0,
        1.0,
    )

    information = {
        "p_low": float(p_low),
        "p_high": float(p_high),
        "lower_bound": lower,
        "upper_bound": upper,
        "n_stats": int(values.size),
        "n_applied": int(valid_apply.sum()),
        "lower_clipped_fraction": float(
            np.mean(array[valid_apply] <= lower)
        ),
        "upper_clipped_fraction": float(
            np.mean(array[valid_apply] >= upper)
        ),
    }

    return normalized, information


# =============================================================================
# SOIL-MOISTURE SOURCE DISCOVERY AND SUPPORT PRESERVATION
# =============================================================================
def approximate_cell_size_metres(
    dataset: rasterio.io.DatasetReader,
) -> tuple[float, float]:
    if dataset.crs is None:
        raise ValueError("Raster has no CRS.")

    transform = dataset.transform

    if dataset.crs.is_projected:
        return abs(float(transform.a)), abs(float(transform.e))

    row = dataset.height // 2
    col = dataset.width // 2

    left, top = transform * (col, row)
    right, bottom = transform * (col + 1, row + 1)

    bounds_3857 = transform_bounds(
        dataset.crs,
        "EPSG:3857",
        min(left, right),
        min(bottom, top),
        max(left, right),
        max(bottom, top),
        densify_pts=5,
    )

    return (
        float(abs(bounds_3857[2] - bounds_3857[0])),
        float(abs(bounds_3857[3] - bounds_3857[1])),
    )


def choose_native_raster() -> Path | None:
    if ERA5_NATIVE_SM_OVERRIDE is not None:
        path = Path(ERA5_NATIVE_SM_OVERRIDE)
        require_file(path, "native ERA5-Land soil-moisture raster")
        return path

    for candidate in ERA5_NATIVE_SM_CANDIDATES:
        if not candidate.exists():
            continue

        try:
            with rasterio.open(candidate) as dataset:
                width_m, height_m = approximate_cell_size_metres(dataset)
                if (
                    min(width_m, height_m)
                    >= MIN_EXPECTED_NATIVE_CELL_SIZE_M
                ):
                    return candidate
        except Exception:
            continue

    return None


def inspect_monthly_csv(
    path: Path,
) -> tuple[
    pd.DataFrame | None,
    dict[str, Any],
    pd.DataFrame,
]:
    if not path.exists():
        return (
            None,
            {
                "path": str(path),
                "exists": False,
                "spatial_grid_reconstructable": False,
            },
            pd.DataFrame(),
        )

    dataframe = pd.read_csv(path)

    if dataframe.empty:
        raise RuntimeError(
            f"Monthly soil-moisture CSV is empty: {path}"
        )

    columns = list(dataframe.columns)

    longitude_column = detect_column(
        columns,
        ["longitude", "lon", "long", "grid_longitude"],
    )
    latitude_column = detect_column(
        columns,
        ["latitude", "lat", "grid_latitude"],
    )
    year_column = detect_column(
        columns,
        ["year", "yr"],
    )
    month_column = detect_column(
        columns,
        ["month", "mon", "month_number"],
    )
    date_column = detect_column(
        columns,
        ["date", "time", "datetime", "valid_time"],
    )

    value_column = detect_column(
        columns,
        [
            "soil_moisture",
            "sm",
            "sm_mean",
            "mean_sm",
            "monthly_sm",
            "monthly_mean_sm",
            "swvl",
            "swvl1",
            "swvl2",
            "volumetric_soil_water",
            "value",
        ],
    )

    if value_column is None:
        scored: list[tuple[int, float, str]] = []

        for column in columns:
            numeric = pd.to_numeric(
                dataframe[column],
                errors="coerce",
            )
            finite_fraction = float(
                numeric.notna().mean()
            )

            if finite_fraction < 0.5:
                continue

            normalized = normalize_column_name(column)
            score = 0

            for token, weight in [
                ("soil", 4),
                ("moisture", 4),
                ("swvl", 5),
                ("sm", 2),
                ("mean", 1),
                ("value", 1),
            ]:
                if token in normalized:
                    score += weight

            if column in {
                longitude_column,
                latitude_column,
                year_column,
                month_column,
            }:
                score -= 20

            scored.append((score, finite_fraction, column))

        scored.sort(reverse=True)

        if scored and scored[0][0] > 0:
            value_column = scored[0][2]

    working = dataframe.copy()

    if date_column is not None:
        parsed_date = pd.to_datetime(
            working[date_column],
            errors="coerce",
        )

        if year_column is None and parsed_date.notna().any():
            working["_detected_year"] = parsed_date.dt.year
            year_column = "_detected_year"

        if month_column is None and parsed_date.notna().any():
            working["_detected_month"] = parsed_date.dt.month
            month_column = "_detected_month"

    temporal_summary = pd.DataFrame()

    if value_column is not None:
        working["_sm_numeric"] = pd.to_numeric(
            working[value_column],
            errors="coerce",
        )

        groups = [
            column
            for column in [year_column, month_column]
            if column is not None
        ]

        if groups:
            temporal_summary = (
                working
                .dropna(subset=groups + ["_sm_numeric"])
                .groupby(groups, dropna=False)["_sm_numeric"]
                .agg(
                    n="count",
                    mean="mean",
                    std="std",
                    minimum="min",
                    median="median",
                    maximum="max",
                )
                .reset_index()
            )

    spatial_grid_reconstructable = bool(
        longitude_column is not None
        and latitude_column is not None
        and value_column is not None
    )

    profile = {
        "path": str(path),
        "exists": True,
        "row_count": int(len(dataframe)),
        "columns": columns,
        "detected_columns": {
            "longitude": longitude_column,
            "latitude": latitude_column,
            "year": year_column,
            "month": month_column,
            "date": date_column,
            "soil_moisture_value": value_column,
        },
        "spatial_grid_reconstructable": (
            spatial_grid_reconstructable
        ),
    }

    return working, profile, temporal_summary


def reproject_native_raster_to_anchor(
    path: Path,
    anchor: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(
                f"Native ERA5 raster has no CRS: {path}"
            )

        if (
            ERA5_NATIVE_BAND < 1
            or ERA5_NATIVE_BAND > dataset.count
        ):
            raise ValueError(
                f"ERA5_NATIVE_BAND={ERA5_NATIVE_BAND} is invalid "
                f"for {dataset.count} band(s)."
            )

        width_m, height_m = approximate_cell_size_metres(
            dataset
        )

        if (
            min(width_m, height_m)
            < MIN_EXPECTED_NATIVE_CELL_SIZE_M
        ):
            raise ValueError(
                "Selected ERA5 raster does not appear to be native/coarse. "
                f"Estimated cell size: {width_m:.1f} x {height_m:.1f} m."
            )

        native_physical = scrub_common_nodata(
            dataset.read(ERA5_NATIVE_BAND),
            dataset.nodata,
        )
        native_valid = np.isfinite(native_physical)

        native_ids = (
            np.arange(
                dataset.height * dataset.width,
                dtype="int64",
            ).reshape(dataset.height, dataset.width)
            + 1
        )
        native_ids = np.where(
            native_valid,
            native_ids,
            NODATA_INT32,
        ).astype("int32")

        physical_anchor = np.full(
            (anchor["height"], anchor["width"]),
            np.nan,
            dtype="float64",
        )
        id_anchor = np.full(
            (anchor["height"], anchor["width"]),
            NODATA_INT32,
            dtype="int32",
        )

        reproject(
            source=native_physical,
            destination=physical_anchor,
            src_transform=dataset.transform,
            src_crs=dataset.crs,
            src_nodata=np.nan,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=np.nan,
            resampling=Resampling.nearest,
        )

        reproject(
            source=native_ids,
            destination=id_anchor,
            src_transform=dataset.transform,
            src_crs=dataset.crs,
            src_nodata=NODATA_INT32,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=NODATA_INT32,
            resampling=Resampling.nearest,
        )

        id_anchor_float = id_anchor.astype("float64")
        id_anchor_float[
            id_anchor == NODATA_INT32
        ] = np.nan

        metadata = {
            "support_mode": "exact_native_era5_raster",
            "is_exact_native_era5_grid": True,
            "source": str(path),
            "source_crs": dataset.crs.to_string(),
            "source_width": dataset.width,
            "source_height": dataset.height,
            "estimated_cell_width_m": width_m,
            "estimated_cell_height_m": height_m,
        }

    return physical_anchor, id_anchor_float, metadata


def reconstruct_csv_grid_to_anchor(
    dataframe: pd.DataFrame | None,
    profile: dict[str, Any],
    anchor: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]] | None:
    if (
        dataframe is None
        or not profile.get(
            "spatial_grid_reconstructable",
            False,
        )
    ):
        return None

    detected = profile["detected_columns"]
    lon_column = detected["longitude"]
    lat_column = detected["latitude"]
    value_column = detected["soil_moisture_value"]

    working = pd.DataFrame(
        {
            "longitude": pd.to_numeric(
                dataframe[lon_column],
                errors="coerce",
            ),
            "latitude": pd.to_numeric(
                dataframe[lat_column],
                errors="coerce",
            ),
            "sm": pd.to_numeric(
                dataframe[value_column],
                errors="coerce",
            ),
        }
    ).dropna()

    working = working[
        working["longitude"].between(-180.0, 180.0)
        & working["latitude"].between(-90.0, 90.0)
    ]

    if working.empty:
        return None

    cell_means = (
        working
        .groupby(
            ["longitude", "latitude"],
            as_index=False,
        )["sm"]
        .mean()
    )

    x_values = np.sort(
        cell_means["longitude"].unique()
    )
    y_values = np.sort(
        cell_means["latitude"].unique()
    )[::-1]

    if x_values.size < 2 or y_values.size < 2:
        return None

    dx = float(np.median(np.diff(x_values)))
    dy = float(np.median(np.abs(np.diff(y_values))))

    if dx <= 0 or dy <= 0:
        return None

    x_deviation = float(
        np.max(np.abs(np.diff(x_values) - dx))
        / max(abs(dx), EPS)
    )
    y_deviation = float(
        np.max(np.abs(np.abs(np.diff(y_values)) - dy))
        / max(abs(dy), EPS)
    )

    if x_deviation > 0.05 or y_deviation > 0.05:
        log(
            "[WARN] CSV coordinates do not define a sufficiently regular grid."
        )
        return None

    x_lookup = {
        float(value): index
        for index, value in enumerate(x_values)
    }
    y_lookup = {
        float(value): index
        for index, value in enumerate(y_values)
    }

    native_array = np.full(
        (y_values.size, x_values.size),
        np.nan,
        dtype="float64",
    )

    for row in cell_means.itertuples(index=False):
        native_array[
            y_lookup[float(row.latitude)],
            x_lookup[float(row.longitude)],
        ] = float(row.sm)

    if float(np.isfinite(native_array).mean()) < 0.5:
        return None

    native_transform = from_origin(
        float(x_values.min() - dx / 2.0),
        float(y_values.max() + dy / 2.0),
        dx,
        dy,
    )

    native_ids = (
        np.arange(
            native_array.size,
            dtype="int64",
        ).reshape(native_array.shape)
        + 1
    )
    native_ids = np.where(
        np.isfinite(native_array),
        native_ids,
        NODATA_INT32,
    ).astype("int32")

    physical_anchor = np.full(
        (anchor["height"], anchor["width"]),
        np.nan,
        dtype="float64",
    )
    id_anchor = np.full(
        (anchor["height"], anchor["width"]),
        NODATA_INT32,
        dtype="int32",
    )

    reproject(
        source=native_array,
        destination=physical_anchor,
        src_transform=native_transform,
        src_crs="EPSG:4326",
        src_nodata=np.nan,
        dst_transform=anchor["transform"],
        dst_crs=anchor["crs"],
        dst_nodata=np.nan,
        resampling=Resampling.nearest,
    )

    reproject(
        source=native_ids,
        destination=id_anchor,
        src_transform=native_transform,
        src_crs="EPSG:4326",
        src_nodata=NODATA_INT32,
        dst_transform=anchor["transform"],
        dst_crs=anchor["crs"],
        dst_nodata=NODATA_INT32,
        resampling=Resampling.nearest,
    )

    id_anchor_float = id_anchor.astype("float64")
    id_anchor_float[
        id_anchor == NODATA_INT32
    ] = np.nan

    metadata = {
        "support_mode": "exact_native_grid_reconstructed_from_csv",
        "is_exact_native_era5_grid": True,
        "source": str(SM_MONSOON_MONTHLY_CSV),
        "native_crs": "EPSG:4326",
        "native_width": int(x_values.size),
        "native_height": int(y_values.size),
        "native_dx_degrees": dx,
        "native_dy_degrees": dy,
        "aggregation": (
            "Mean of monthly values at each longitude-latitude support cell"
        ),
    }

    return physical_anchor, id_anchor_float, metadata


def build_approximate_support_ids(
    anchor: dict[str, Any],
    domain: np.ndarray,
    cell_size_m: float,
) -> tuple[np.ndarray, dict[str, Any]]:
    if not anchor["crs"].is_projected:
        raise RuntimeError(
            "Approximate support fallback requires a projected anchor CRS."
        )

    transform = anchor["transform"]

    if abs(transform.b) > 1.0e-10 or abs(transform.d) > 1.0e-10:
        raise RuntimeError(
            "Approximate support fallback requires a north-up anchor grid."
        )

    x_centres = (
        transform.c
        + (np.arange(anchor["width"], dtype="float64") + 0.5)
        * transform.a
    )
    y_centres = (
        transform.f
        + (np.arange(anchor["height"], dtype="float64") + 0.5)
        * transform.e
    )

    origin_x = (
        math.floor(float(anchor["bounds"].left) / cell_size_m)
        * cell_size_m
    )
    origin_y = (
        math.floor(float(anchor["bounds"].bottom) / cell_size_m)
        * cell_size_m
    )

    col_group = np.floor(
        (x_centres - origin_x) / cell_size_m
    ).astype("int64")
    row_group = np.floor(
        (y_centres - origin_y) / cell_size_m
    ).astype("int64")

    row_index = row_group - int(row_group.min())
    col_index = col_group - int(col_group.min())
    n_columns = int(col_index.max()) + 1

    ids = (
        row_index[:, None] * n_columns
        + col_index[None, :]
        + 1
    ).astype("float64")
    ids[~domain] = np.nan

    metadata = {
        "support_mode": "approximate_9km_from_physical_sm",
        "is_exact_native_era5_grid": False,
        "cell_size_m": float(cell_size_m),
        "origin_x": float(origin_x),
        "origin_y": float(origin_y),
        "interpretation": (
            "Approximate support grid used because the exact native ERA5 "
            "geometry was unavailable. Do not report as exact native cells."
        ),
    }

    return ids, metadata


def aggregate_physical_sm_by_support(
    physical_sm: np.ndarray,
    support_ids: np.ndarray,
    domain: np.ndarray,
) -> tuple[np.ndarray, pd.DataFrame]:
    valid = (
        domain
        & np.isfinite(physical_sm)
        & np.isfinite(support_ids)
        & (support_ids > 0)
    )

    if valid.sum() < 2:
        raise RuntimeError(
            "Insufficient physical SM values for support aggregation."
        )

    ids = support_ids[valid].astype("int64")
    values = physical_sm[valid].astype("float64")

    dataframe = pd.DataFrame(
        {
            "support_id": ids,
            "physical_sm": values,
        }
    )

    cell_table = (
        dataframe
        .groupby("support_id", as_index=False)
        .agg(
            physical_sm_mean=("physical_sm", "mean"),
            physical_sm_median=("physical_sm", "median"),
            fine_pixel_count=("physical_sm", "size"),
        )
    )

    value_lookup = dict(
        zip(
            cell_table["support_id"].astype("int64"),
            cell_table["physical_sm_mean"].astype("float64"),
        )
    )

    repeated = np.full(
        physical_sm.shape,
        np.nan,
        dtype="float64",
    )

    unique_ids = np.unique(ids)
    for support_id in unique_ids:
        repeated[
            valid & (support_ids.astype("int64") == support_id)
        ] = value_lookup[int(support_id)]

    return repeated, cell_table


def standardize_unique_support_values(
    support_physical: np.ndarray,
    support_ids: np.ndarray,
    domain: np.ndarray,
) -> tuple[np.ndarray, pd.DataFrame, dict[str, Any]]:
    """
    Winsorize and standardize one value per UNIQUE support cell.

    Fine-pixel replication is performed only after support-cell z-scores are
    calculated, preventing repeated 12.5 m pixels from weighting the same
    coarse observation multiple times.
    """
    valid = (
        domain
        & np.isfinite(support_physical)
        & np.isfinite(support_ids)
        & (support_ids > 0)
    )

    if valid.sum() < 2:
        raise RuntimeError(
            "Insufficient support-preserved SM values for standardization."
        )

    dataframe = pd.DataFrame(
        {
            "support_id": support_ids[valid].astype("int64"),
            "physical_sm": support_physical[valid].astype("float64"),
        }
    )

    cells = (
        dataframe
        .groupby("support_id", as_index=False)
        .agg(
            physical_sm=("physical_sm", "median"),
            fine_pixel_count=("physical_sm", "size"),
        )
    )

    if len(cells) < 2:
        raise RuntimeError(
            "Fewer than two unique SM support cells intersect the study domain."
        )

    unique_values = cells["physical_sm"].to_numpy(dtype="float64")

    lower = float(
        np.percentile(unique_values, SM_WINSOR_P_LOW)
    )
    upper = float(
        np.percentile(unique_values, SM_WINSOR_P_HIGH)
    )

    if upper <= lower + EPS:
        raise RuntimeError(
            "Support-cell SM values are effectively constant."
        )

    winsorized = np.clip(
        unique_values,
        lower,
        upper,
    )

    mean = float(np.mean(winsorized))
    std = float(np.std(winsorized))

    if std <= EPS:
        raise RuntimeError(
            "Support-cell SM standard deviation is zero."
        )

    z_values = (winsorized - mean) / std

    cells["winsorized_sm"] = winsorized
    cells["z_sm_support"] = z_values

    z_lookup = dict(
        zip(
            cells["support_id"].astype("int64"),
            cells["z_sm_support"].astype("float64"),
        )
    )

    z_raster = np.full(
        support_physical.shape,
        np.nan,
        dtype="float64",
    )

    integer_ids = np.where(
        np.isfinite(support_ids),
        support_ids,
        -1,
    ).astype("int64")

    for support_id, z_value in z_lookup.items():
        z_raster[
            valid & (integer_ids == int(support_id))
        ] = float(z_value)

    metadata = {
        "standardization_level": "unique_support_cells",
        "number_of_unique_support_cells": int(len(cells)),
        "winsor_p_low": SM_WINSOR_P_LOW,
        "winsor_p_high": SM_WINSOR_P_HIGH,
        "winsor_lower": lower,
        "winsor_upper": upper,
        "winsor_mean_across_unique_cells": mean,
        "winsor_std_across_unique_cells": std,
        "z_mean_across_unique_cells": float(
            np.mean(z_values)
        ),
        "z_std_across_unique_cells": float(
            np.std(z_values)
        ),
        "within_support_cell_z_std_expected": 0.0,
    }

    return z_raster, cells, metadata


def prepare_support_preserved_sm(
    anchor: dict[str, Any],
    study_domain: np.ndarray,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    dict[str, Any],
    pd.DataFrame,
    dict[str, Any],
]:
    require_file(
        SM_MONSOON_MEAN_TIF,
        "physical monsoon-mean soil-moisture raster",
    )

    physical_fine, physical_info = read_continuous_exact(
        SM_MONSOON_MEAN_TIF,
        anchor,
        "physical monsoon-mean SM raster",
    )

    csv_dataframe, csv_profile, temporal_summary = inspect_monthly_csv(
        SM_MONSOON_MONTHLY_CSV
    )

    with open(
        SM_CSV_PROFILE_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            csv_profile,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    if not temporal_summary.empty:
        temporal_summary.to_csv(
            SM_TEMPORAL_SUMMARY_CSV,
            index=False,
        )

    native_path = choose_native_raster()

    support_physical: np.ndarray
    support_ids: np.ndarray
    support_metadata: dict[str, Any]

    if native_path is not None:
        (
            support_physical,
            support_ids,
            support_metadata,
        ) = reproject_native_raster_to_anchor(
            native_path,
            anchor,
        )

    else:
        csv_result = reconstruct_csv_grid_to_anchor(
            csv_dataframe,
            csv_profile,
            anchor,
        )

        if csv_result is not None:
            (
                support_physical,
                support_ids,
                support_metadata,
            ) = csv_result

        else:
            if not ALLOW_APPROXIMATE_9KM_AS_PRIMARY:
                raise RuntimeError(
                    "No exact native/coarse ERA5 support was recovered and "
                    "ALLOW_APPROXIMATE_9KM_AS_PRIMARY=False."
                )

            support_ids, support_metadata = (
                build_approximate_support_ids(
                    anchor,
                    study_domain & np.isfinite(physical_fine),
                    APPROXIMATE_SUPPORT_CELL_SIZE_M,
                )
            )

            support_physical, approximate_cell_table = (
                aggregate_physical_sm_by_support(
                    physical_fine,
                    support_ids,
                    study_domain,
                )
            )
            support_metadata["approximate_cell_table_rows"] = int(
                len(approximate_cell_table)
            )

    support_domain = (
        study_domain
        & np.isfinite(support_physical)
        & np.isfinite(support_ids)
        & (support_ids > 0)
    )

    z_sm_support, support_cell_table, standardization_metadata = (
        standardize_unique_support_values(
            support_physical,
            support_ids,
            support_domain,
        )
    )

    support_metadata.update(
        {
            "physical_fine_input": str(SM_MONSOON_MEAN_TIF),
            "monthly_csv_input": str(SM_MONSOON_MONTHLY_CSV),
            "physical_fine_input_info": physical_info,
            "csv_profile": csv_profile,
            "standardization": standardization_metadata,
            "support_domain_pixel_count": int(
                support_domain.sum()
            ),
        }
    )

    return (
        support_physical,
        z_sm_support,
        support_ids,
        support_metadata,
        support_cell_table,
        physical_info,
    )


# =============================================================================
# HKFI COMPUTATION
# =============================================================================
def compute_multiscale_hkfi(
    z_dv: np.ndarray,
    z_twi: np.ndarray,
    z_sm_support: np.ndarray,
    irs: np.ndarray,
    context_domain: np.ndarray,
    lambda_sm: float,
) -> dict[str, np.ndarray | int]:
    if not (0.0 <= lambda_sm <= 1.0):
        raise ValueError("lambda_sm must be within [0,1].")

    irs_original = np.full_like(
        irs,
        np.nan,
        dtype="float64",
    )
    finite_irs = np.isfinite(irs)
    irs_original[finite_irs] = np.clip(
        irs[finite_irs],
        0.0,
        1.0,
    )

    regional_hydro_valid = (
        context_domain
        & np.isfinite(z_twi)
        & np.isfinite(z_sm_support)
        & np.isfinite(irs_original)
    )

    dv_available = (
        regional_hydro_valid
        & np.isfinite(z_dv)
    )
    missing_zdv = (
        regional_hydro_valid
        & ~np.isfinite(z_dv)
    )

    missing_zdv_nonzero_irs = (
        missing_zdv
        & (irs_original > IRS_EFFECTIVE_ZERO_TOL)
    )

    irs_for_hkfi = np.full_like(
        irs_original,
        np.nan,
        dtype="float64",
    )
    irs_for_hkfi[regional_hydro_valid] = (
        irs_original[regional_hydro_valid]
    )
    irs_for_hkfi[missing_zdv] = 0.0

    z_dv_effective = np.full_like(
        z_dv,
        np.nan,
        dtype="float64",
    )
    z_dv_effective[dv_available] = z_dv[dv_available]
    z_dv_effective[missing_zdv] = 0.0

    compute_domain = regional_hydro_valid

    K = np.full_like(z_dv, np.nan, dtype="float64")
    T = np.full_like(z_twi, np.nan, dtype="float64")
    S = np.full_like(z_sm_support, np.nan, dtype="float64")

    K[compute_domain] = (
        irs_for_hkfi[compute_domain]
        * z_dv_effective[compute_domain]
    )
    T[compute_domain] = z_twi[compute_domain]
    S[compute_domain] = z_sm_support[compute_domain]

    H_local = np.full_like(
        z_twi,
        np.nan,
        dtype="float64",
    )
    H_local[compute_domain] = (
        K[compute_domain] + T[compute_domain]
    ) / 2.0

    hkfi_raw = np.full_like(
        z_twi,
        np.nan,
        dtype="float64",
    )
    hkfi_raw[compute_domain] = (
        (1.0 - lambda_sm)
        * H_local[compute_domain]
        + lambda_sm
        * S[compute_domain]
    )

    # Algebraic equal-component reference for lambda=1/3.
    equal_component_reference = np.full_like(
        z_twi,
        np.nan,
        dtype="float64",
    )
    equal_component_reference[compute_domain] = (
        K[compute_domain]
        + T[compute_domain]
        + S[compute_domain]
    ) / 3.0

    return {
        "irs_original": irs_original,
        "irs_for_hkfi": irs_for_hkfi,
        "compute_domain": compute_domain,
        "dv_available_mask": dv_available,
        "missing_zdv_mask": missing_zdv,
        "missing_zdv_nonzero_irs_mask": (
            missing_zdv_nonzero_irs
        ),
        "K": K,
        "T": T,
        "S": S,
        "H_local": H_local,
        "hkfi_raw": hkfi_raw,
        "equal_component_reference": (
            equal_component_reference
        ),
        "n_missing_zdv": int(missing_zdv.sum()),
        "n_missing_zdv_nonzero_irs": int(
            missing_zdv_nonzero_irs.sum()
        ),
    }


# =============================================================================
# QA FIGURES
# =============================================================================
def make_qa_figures(
    hkfi_norm: np.ndarray,
    compute_domain: np.ndarray,
    study_mask: np.ndarray,
    anchor: dict[str, Any],
) -> None:
    if not MAKE_QA_FIGURES:
        return

    import matplotlib.pyplot as plt

    bounds = anchor["bounds"]
    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    display_array = np.where(
        compute_domain,
        hkfi_norm,
        np.nan,
    )
    values = hkfi_norm[
        compute_domain & np.isfinite(hkfi_norm)
    ]

    if values.size == 0:
        return

    figure, axis = plt.subplots(
        figsize=QA_MAP_FIGSIZE,
        facecolor="white",
    )
    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        interpolation="nearest",
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
    )
    axis.contour(
        study_mask.astype("uint8"),
        levels=[0.5],
        extent=extent,
        origin="upper",
        colors="black",
        linewidths=0.7,
    )
    axis.set_title(
        "Multiscale Hydro–Kinematic Fusion Index",
        fontsize=11,
        fontweight="bold",
    )
    axis.set_xlabel("Easting (m)", fontsize=9)
    axis.set_ylabel("Northing (m)", fontsize=9)
    axis.tick_params(labelsize=7)

    colorbar = figure.colorbar(
        image,
        ax=axis,
        fraction=0.035,
        pad=0.025,
    )
    colorbar.set_label(
        "Study-relative multiscale HKFI (0–1)",
        fontsize=9,
    )

    figure.tight_layout()
    figure.savefig(
        QA_MAP_PNG,
        dpi=QA_FIG_DPI,
        bbox_inches="tight",
        facecolor="white",
    )

    if SHOW_QA_FIGURES:
        plt.show()
    else:
        plt.close(figure)

    figure_h, axis_h = plt.subplots(
        figsize=QA_HIST_FIGSIZE,
        facecolor="white",
    )
    axis_h.hist(
        values,
        bins=np.linspace(0.0, 1.0, 31),
        edgecolor="black",
        linewidth=0.5,
        alpha=0.85,
    )
    axis_h.axvline(
        float(np.mean(values)),
        linestyle="--",
        linewidth=1.0,
        label=f"Mean = {float(np.mean(values)):.3f}",
    )
    axis_h.axvline(
        float(np.median(values)),
        linestyle=":",
        linewidth=1.2,
        label=f"Median = {float(np.median(values)):.3f}",
    )
    axis_h.set_xlim(0.0, 1.0)
    axis_h.set_xlabel(
        "Study-relative multiscale HKFI (0–1)",
        fontsize=9,
    )
    axis_h.set_ylabel("Number of pixels", fontsize=9)
    axis_h.grid(axis="y", alpha=0.25)
    axis_h.legend(fontsize=8)
    figure_h.tight_layout()
    figure_h.savefig(
        QA_HIST_PNG,
        dpi=QA_FIG_DPI,
        bbox_inches="tight",
        facecolor="white",
    )

    if SHOW_QA_FIGURES:
        plt.show()
    else:
        plt.close(figure_h)


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=== Step 5.3: multiscale HKFI ===")

    if not math.isclose(
        LAMBDA_SM,
        1.0 / 3.0,
        rel_tol=0.0,
        abs_tol=1.0e-12,
    ):
        log(
            "[WARN] LAMBDA_SM differs from the primary equal-component "
            "value 1/3. Update the manuscript equation if this is intentional."
        )

    anchor = read_anchor(ANCHOR_TIF)

    study, study_info = read_mask(
        STUDY_MASK_TIF,
        anchor,
        required=True,
        description="study mask",
    )
    assert study is not None

    context, context_info = read_mask(
        CONTEXT_MASK_TIF,
        anchor,
        required=False,
        description="context mask",
    )
    if context is None:
        context = study.copy()
        context_path_used = STUDY_MASK_TIF
    else:
        context = context & study
        context_path_used = CONTEXT_MASK_TIF

    dynamic, dynamic_info = read_mask(
        DYNAMIC_MASK_TIF,
        anchor,
        required=False,
        description="dynamic mask",
    )
    if dynamic is not None:
        dynamic = dynamic & study

    irs_support, irs_support_info = read_mask(
        IRS_SUPPORT_MASK_TIF,
        anchor,
        required=False,
        description="IRS support mask",
    )
    if irs_support is not None:
        irs_support = irs_support & study

    z_dv, z_dv_info = read_continuous_exact(
        Z_DV_TIF,
        anchor,
        "z(DV+) raster",
    )
    z_twi, z_twi_info = read_continuous_exact(
        Z_TWI_TIF,
        anchor,
        "z(TWI) raster",
    )
    irs, irs_info = read_continuous_exact(
        IRS_TIF,
        anchor,
        "corrected IRS raster",
    )

    invalid_irs = (
        np.isfinite(irs)
        & (
            (irs < -1.0e-6)
            | (irs > 1.0 + 1.0e-6)
        )
    )
    if invalid_irs.any():
        raise ValueError(
            f"IRS contains {int(invalid_irs.sum()):,} values outside [0,1]."
        )

    irs = np.where(
        np.isfinite(irs),
        np.clip(irs, 0.0, 1.0),
        np.nan,
    )

    old_z_sm = None
    old_z_sm_info = None
    if OLD_Z_SM_TIF.exists():
        old_z_sm, old_z_sm_info = read_continuous_exact(
            OLD_Z_SM_TIF,
            anchor,
            "old interpolated z(SM) diagnostic raster",
        )

    # -------------------------------------------------------------------------
    # Prepare support-preserved regional SM
    # -------------------------------------------------------------------------
    (
        sm_support_physical,
        z_sm_support,
        support_ids,
        support_metadata,
        support_cell_table,
        sm_physical_info,
    ) = prepare_support_preserved_sm(
        anchor=anchor,
        study_domain=context,
    )

    support_cell_table.to_csv(
        SUPPORT_CELL_TABLE_CSV,
        index=False,
    )

    with open(
        SM_SUPPORT_MODE_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            support_metadata,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    # -------------------------------------------------------------------------
    # Compute primary multiscale HKFI
    # -------------------------------------------------------------------------
    result = compute_multiscale_hkfi(
        z_dv=z_dv,
        z_twi=z_twi,
        z_sm_support=z_sm_support,
        irs=irs,
        context_domain=context,
        lambda_sm=LAMBDA_SM,
    )

    compute_domain = result["compute_domain"]
    dv_available = result["dv_available_mask"]
    missing_zdv = result["missing_zdv_mask"]
    unresolved = result[
        "missing_zdv_nonzero_irs_mask"
    ]
    irs_original = result["irs_original"]
    irs_effective = result["irs_for_hkfi"]
    K = result["K"]
    T = result["T"]
    S = result["S"]
    H_local = result["H_local"]
    hkfi_raw = result["hkfi_raw"]
    equal_component_reference = result[
        "equal_component_reference"
    ]

    if not np.any(compute_domain):
        raise RuntimeError(
            "No valid pixels remain for multiscale HKFI calculation."
        )

    if math.isclose(
        LAMBDA_SM,
        1.0 / 3.0,
        rel_tol=0.0,
        abs_tol=1.0e-12,
    ):
        equality_difference = (
            hkfi_raw[compute_domain]
            - equal_component_reference[compute_domain]
        )
        equality_rmse = float(
            np.sqrt(
                np.mean(equality_difference ** 2)
            )
        )
        equality_max_abs = float(
            np.max(np.abs(equality_difference))
        )

        if equality_max_abs > 1.0e-10:
            raise RuntimeError(
                "Multiscale lambda formulation does not match the equal-"
                "component equation at lambda=1/3."
            )
    else:
        equality_rmse = None
        equality_max_abs = None

    hkfi_norm, norm_info = robust_minmax_normalize(
        hkfi_raw,
        stats_mask=compute_domain,
        apply_mask=compute_domain,
        p_low=HKFI_NORM_P_LOW,
        p_high=HKFI_NORM_P_HIGH,
    )

    # -------------------------------------------------------------------------
    # Optional comparison with old interpolated-SM formulation
    # -------------------------------------------------------------------------
    old_hkfi_raw = None
    old_hkfi_difference = None
    old_comparison = None

    if old_z_sm is not None:
        old_result = compute_multiscale_hkfi(
            z_dv=z_dv,
            z_twi=z_twi,
            z_sm_support=old_z_sm,
            irs=irs,
            context_domain=context,
            lambda_sm=LAMBDA_SM,
        )
        old_hkfi_raw = old_result["hkfi_raw"]

        old_common = (
            compute_domain
            & old_result["compute_domain"]
            & np.isfinite(old_hkfi_raw)
            & np.isfinite(hkfi_raw)
        )

        old_hkfi_difference = np.full_like(
            hkfi_raw,
            np.nan,
            dtype="float64",
        )
        old_hkfi_difference[old_common] = (
            hkfi_raw[old_common]
            - old_hkfi_raw[old_common]
        )

        old_comparison = {
            "n_common": int(old_common.sum()),
            "pearson_r": safe_pearson(
                hkfi_raw,
                old_hkfi_raw,
                old_common,
            ),
            "rmse": float(
                np.sqrt(
                    np.mean(
                        old_hkfi_difference[old_common] ** 2
                    )
                )
            ),
            "mae": float(
                np.mean(
                    np.abs(
                        old_hkfi_difference[old_common]
                    )
                )
            ),
        }

    # -------------------------------------------------------------------------
    # Weight sensitivity
    # -------------------------------------------------------------------------
    sensitivity_rows: list[dict[str, Any]] = []

    for scenario_name, lambda_value in WEIGHT_SCENARIOS.items():
        scenario = compute_multiscale_hkfi(
            z_dv=z_dv,
            z_twi=z_twi,
            z_sm_support=z_sm_support,
            irs=irs,
            context_domain=context,
            lambda_sm=lambda_value,
        )
        scenario_raw = scenario["hkfi_raw"]
        scenario_domain = scenario["compute_domain"]

        scenario_norm, scenario_norm_info = robust_minmax_normalize(
            scenario_raw,
            stats_mask=scenario_domain,
            apply_mask=scenario_domain,
            p_low=HKFI_NORM_P_LOW,
            p_high=HKFI_NORM_P_HIGH,
        )

        common = (
            compute_domain
            & scenario_domain
            & np.isfinite(hkfi_raw)
            & np.isfinite(scenario_raw)
            & np.isfinite(hkfi_norm)
            & np.isfinite(scenario_norm)
        )

        raw_difference = (
            scenario_raw[common]
            - hkfi_raw[common]
        )

        sensitivity_rows.append(
            {
                "scenario": scenario_name,
                "lambda_sm": lambda_value,
                "weight_K": (1.0 - lambda_value) / 2.0,
                "weight_T": (1.0 - lambda_value) / 2.0,
                "weight_S": lambda_value,
                "raw_pearson_with_primary": safe_pearson(
                    hkfi_raw,
                    scenario_raw,
                    common,
                ),
                "norm_pearson_with_primary": safe_pearson(
                    hkfi_norm,
                    scenario_norm,
                    common,
                ),
                "raw_rmse_vs_primary": float(
                    np.sqrt(np.mean(raw_difference ** 2))
                ),
                "raw_mae_vs_primary": float(
                    np.mean(np.abs(raw_difference))
                ),
                "normalization_lower": (
                    scenario_norm_info["lower_bound"]
                ),
                "normalization_upper": (
                    scenario_norm_info["upper_bound"]
                ),
            }
        )

        if (
            WRITE_WEIGHT_SENSITIVITY_RASTERS
            and scenario_name != "primary_equal_component"
        ):
            write_float_tif(
                SENSITIVITY_DIR
                / (
                    f"hkfi_{scenario_name}_raw_"
                    "32645_12p5m_float32.tif"
                ),
                np.where(
                    scenario_domain,
                    scenario_raw,
                    np.nan,
                ),
                anchor,
                f"HKFI_{scenario_name}_raw",
            )
            write_float_tif(
                SENSITIVITY_DIR
                / (
                    f"hkfi_{scenario_name}_norm_0_1_"
                    "32645_12p5m_float32.tif"
                ),
                np.where(
                    scenario_domain,
                    scenario_norm,
                    np.nan,
                ),
                anchor,
                f"HKFI_{scenario_name}_norm_0_1",
            )

    pd.DataFrame(
        sensitivity_rows
    ).to_csv(
        WEIGHT_SENSITIVITY_CSV,
        index=False,
    )

    # -------------------------------------------------------------------------
    # QA statistics
    # -------------------------------------------------------------------------
    stats = {
        "z_dv_compute_domain": detailed_stats(
            z_dv,
            compute_domain & np.isfinite(z_dv),
        ),
        "z_twi_compute_domain": detailed_stats(
            z_twi,
            compute_domain,
        ),
        "z_sm_support_compute_domain": detailed_stats(
            z_sm_support,
            compute_domain,
        ),
        "irs_original_compute_domain": detailed_stats(
            irs_original,
            compute_domain,
        ),
        "irs_effective_for_hkfi": detailed_stats(
            irs_effective,
            compute_domain,
        ),
        "K_kinematic_term": detailed_stats(
            K,
            compute_domain,
        ),
        "T_twi_term": detailed_stats(
            T,
            compute_domain,
        ),
        "S_support_preserved_sm_term": detailed_stats(
            S,
            compute_domain,
        ),
        "H_local": detailed_stats(
            H_local,
            compute_domain,
        ),
        "hkfi_multiscale_raw": detailed_stats(
            hkfi_raw,
            compute_domain,
        ),
        "hkfi_multiscale_norm": detailed_stats(
            hkfi_norm,
            compute_domain,
        ),
    }

    correlations = {
        "K_vs_T": safe_pearson(
            K,
            T,
            compute_domain,
        ),
        "K_vs_S": safe_pearson(
            K,
            S,
            compute_domain,
        ),
        "T_vs_S": safe_pearson(
            T,
            S,
            compute_domain,
        ),
        "K_vs_HKFI": safe_pearson(
            K,
            hkfi_raw,
            compute_domain,
        ),
        "T_vs_HKFI": safe_pearson(
            T,
            hkfi_raw,
            compute_domain,
        ),
        "S_vs_HKFI": safe_pearson(
            S,
            hkfi_raw,
            compute_domain,
        ),
    }

    warnings: list[str] = []

    study_n = int(study.sum())
    context_n = int(context.sum())
    compute_n = int(compute_domain.sum())
    compute_fraction = compute_n / max(context_n, 1)

    if compute_fraction < WARN_VALID_DOMAIN_FRACTION:
        warnings.append(
            f"Only {100.0 * compute_fraction:.2f}% of the context domain "
            "has computable multiscale HKFI."
        )

    for label in (
        "z_dv_compute_domain",
        "z_twi_compute_domain",
        "z_sm_support_compute_domain",
    ):
        component = stats[label]
        mean_value = component["mean"]
        std_value = component["std"]

        if (
            mean_value is not None
            and abs(float(mean_value)) > WARN_INPUT_Z_MEAN_ABS
        ):
            warnings.append(
                f"{label} mean is {float(mean_value):.3f} on the fine-pixel "
                "HKFI domain. For z(SM), note that standardization was correctly "
                "performed across unique support cells, so the repeated fine-"
                "pixel mean need not be exactly zero."
            )

        if (
            label != "z_sm_support_compute_domain"
            and std_value is not None
            and (
                float(std_value) < WARN_INPUT_Z_STD_LOW
                or float(std_value) > WARN_INPUT_Z_STD_HIGH
            )
        ):
            warnings.append(
                f"{label} standard deviation is {float(std_value):.3f}."
            )

    k_zero_fraction = stats[
        "K_kinematic_term"
    ]["zero_fraction"]

    if (
        k_zero_fraction is not None
        and float(k_zero_fraction) > WARN_K_ZERO_FRACTION
    ):
        warnings.append(
            f"K is exactly zero over "
            f"{100.0 * float(k_zero_fraction):.2f}% of the compute domain. "
            "Report full-domain and InSAR-supported-domain contributions "
            "separately in Step 5.3b."
        )

    irs_near_zero = stats[
        "irs_original_compute_domain"
    ]["near_zero_fraction"]

    if (
        irs_near_zero is not None
        and float(irs_near_zero)
        > WARN_IRS_NEAR_ZERO_FRACTION
    ):
        warnings.append(
            f"{100.0 * float(irs_near_zero):.2f}% of IRS values are near zero."
        )

    hkfi_unique = stats[
        "hkfi_multiscale_norm"
    ]["unique_values_rounded_6dp"]

    if int(hkfi_unique) < WARN_HKFI_UNIQUE_VALUES_MIN:
        warnings.append(
            f"Normalized HKFI has only {int(hkfi_unique):,} unique values "
            "at six decimal places."
        )

    if (
        WARN_APPROXIMATE_SUPPORT
        and not support_metadata.get(
            "is_exact_native_era5_grid",
            False,
        )
    ):
        warnings.append(
            "The primary HKFI used an approximate 9 km support grid because "
            "the exact native ERA5 geometry was unavailable. This is a "
            "support-scale correction, not exact native-cell reconstruction."
        )

    # Check within-cell constancy of S
    valid_support = (
        compute_domain
        & np.isfinite(S)
        & np.isfinite(support_ids)
    )
    support_integer = np.where(
        np.isfinite(support_ids),
        support_ids,
        -1,
    ).astype("int64")

    within_cell_std_values: list[float] = []
    for support_id in np.unique(
        support_integer[valid_support]
    ):
        values = S[
            valid_support
            & (support_integer == support_id)
        ]
        if values.size:
            within_cell_std_values.append(
                float(np.std(values))
            )

    maximum_within_cell_s_std = (
        max(within_cell_std_values)
        if within_cell_std_values
        else None
    )

    if (
        maximum_within_cell_s_std is not None
        and maximum_within_cell_s_std > 1.0e-10
    ):
        raise RuntimeError(
            "Support-preserved z(SM) is not constant within support cells."
        )

    # -------------------------------------------------------------------------
    # Write outputs
    # -------------------------------------------------------------------------
    write_float_tif(
        IRS_EFFECTIVE_HKFI_TIF,
        np.where(
            compute_domain,
            irs_effective,
            np.nan,
        ),
        anchor,
        "IRS_effective_for_HKFI",
    )
    write_float_tif(
        KINEMATIC_TERM_TIF,
        np.where(compute_domain, K, np.nan),
        anchor,
        "K_IRS_times_zDVplus",
    )
    write_float_tif(
        TWI_TERM_TIF,
        np.where(compute_domain, T, np.nan),
        anchor,
        "T_zTWI",
    )
    write_float_tif(
        SM_SUPPORT_PHYSICAL_TIF,
        np.where(
            compute_domain,
            sm_support_physical,
            np.nan,
        ),
        anchor,
        "SM_support_preserved_physical",
    )
    write_float_tif(
        SM_SUPPORT_Z_TIF,
        np.where(compute_domain, S, np.nan),
        anchor,
        "S_zSM_support_preserved",
    )
    write_int32_tif(
        SM_SUPPORT_CELL_ID_TIF,
        np.where(
            compute_domain,
            support_ids,
            np.nan,
        ),
        anchor,
        "SM_support_cell_identifier",
    )
    write_float_tif(
        LOCAL_HYDRO_KINEMATIC_TIF,
        np.where(
            compute_domain,
            H_local,
            np.nan,
        ),
        anchor,
        "H_local_K_plus_T_over_2",
    )

    # Primary compatibility outputs
    write_float_tif(
        HKFI_RAW_TIF,
        np.where(
            compute_domain,
            hkfi_raw,
            np.nan,
        ),
        anchor,
        "HKFI_multiscale_raw",
    )
    write_float_tif(
        HKFI_NORM_TIF,
        np.where(
            compute_domain,
            hkfi_norm,
            np.nan,
        ),
        anchor,
        "HKFI_multiscale_norm_0_1",
    )
    write_float_tif(
        HKFI_COMPAT_TIF,
        np.where(
            compute_domain,
            hkfi_norm,
            np.nan,
        ),
        anchor,
        "HKFI_multiscale_norm_compatibility_copy",
    )

    # Explicit aliases
    write_float_tif(
        HKFI_MS_RAW_TIF,
        np.where(
            compute_domain,
            hkfi_raw,
            np.nan,
        ),
        anchor,
        "HKFI_multiscale_raw_alias",
    )
    write_float_tif(
        HKFI_MS_NORM_TIF,
        np.where(
            compute_domain,
            hkfi_norm,
            np.nan,
        ),
        anchor,
        "HKFI_multiscale_norm_alias",
    )

    if old_hkfi_difference is not None:
        write_float_tif(
            HKFI_OLD_SM_DIFFERENCE_TIF,
            old_hkfi_difference,
            anchor,
            "HKFI_multiscale_minus_old_interpolated_SM_raw",
        )

    write_byte_tif(
        HKFI_VALID_MASK_TIF,
        compute_domain,
        anchor,
        study,
        "HKFI_valid_mask",
    )
    write_byte_tif(
        HKFI_DV_AVAILABILITY_MASK_TIF,
        dv_available,
        anchor,
        study,
        "HKFI_DV_availability_mask",
    )
    write_byte_tif(
        HKFI_MISSING_ZDV_MASK_TIF,
        missing_zdv,
        anchor,
        study,
        "HKFI_missing_zDV_mask",
    )
    write_byte_tif(
        HKFI_UNRESOLVED_MASK_TIF,
        unresolved,
        anchor,
        study,
        "HKFI_missing_zDV_nonzero_original_IRS_mask",
    )

    # -------------------------------------------------------------------------
    # Metadata and QA
    # -------------------------------------------------------------------------
    zscore_metadata = load_json_optional(
        ZSCORE_META_JSON
    )
    irs_metadata = load_json_optional(
        IRS_META_JSON
    )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "step": "5.3",
        "formulation": "multiscale_HKFI",
        "equation": {
            "K": "K = IRS_HKFI * z(DV+)",
            "T": "T = z(TWI)",
            "S": (
                "S_j = z_native(SM_j), standardized across unique "
                "coarse support cells"
            ),
            "H_local": "H_local = (K + T) / 2",
            "HKFI_MS": (
                "HKFI_MS_raw = (1-lambda_SM)*H_local + lambda_SM*S"
            ),
            "lambda_SM": LAMBDA_SM,
            "equivalent_primary_equation": (
                "At lambda_SM=1/3: HKFI_MS_raw = (K + T + S) / 3"
            ),
        },
        "scientific_interpretation": {
            "multiscale_structure": (
                "K and T provide local 12.5 m differentiation. "
                "S represents regional soil-moisture loading and is "
                "constant within each coarse support cell."
            ),
            "SM_standardization": (
                "SM was winsorized and z-standardized across unique "
                "support-cell values before assignment to the 12.5 m grid."
            ),
            "IRS_role": (
                "IRS attenuates uncertain signed kinematic evidence "
                "toward neutral zero and is not used as an independent "
                "predictor."
            ),
            "missing_zDV_rule": (
                "Where z(DV+) is unavailable, I_DV=0 and "
                "IRS_HKFI=IRS*I_DV=0. This is neutral missing evidence, "
                "not observed stability."
            ),
            "normalization": (
                "The final 0–1 HKFI is a study-relative robust P2-P98 "
                "mapping and is not a failure probability."
            ),
        },
        "support_metadata": support_metadata,
        "inputs": {
            "anchor": str(ANCHOR_TIF),
            "study_mask": str(STUDY_MASK_TIF),
            "context_mask_used": str(context_path_used),
            "dynamic_mask_diagnostic": (
                str(DYNAMIC_MASK_TIF)
                if dynamic is not None
                else None
            ),
            "z_dv": z_dv_info,
            "z_twi": z_twi_info,
            "irs": irs_info,
            "old_z_sm_diagnostic": old_z_sm_info,
            "physical_sm": sm_physical_info,
            "monthly_sm_csv": str(SM_MONSOON_MONTHLY_CSV),
            "irs_support_mask": irs_support_info,
            "zscore_metadata": (
                str(ZSCORE_META_JSON)
                if zscore_metadata is not None
                else None
            ),
            "irs_metadata": (
                str(IRS_META_JSON)
                if irs_metadata is not None
                else None
            ),
        },
        "domain": {
            "n_study": study_n,
            "n_context": context_n,
            "n_compute": compute_n,
            "compute_fraction_of_context": compute_fraction,
            "n_dv_available": int(dv_available.sum()),
            "n_missing_zdv": int(missing_zdv.sum()),
            "n_missing_zdv_nonzero_irs": int(
                unresolved.sum()
            ),
            "dynamic_mask_used_to_define_domain": False,
        },
        "normalization": norm_info,
        "equal_component_equivalence_QA": {
            "applicable": bool(
                math.isclose(
                    LAMBDA_SM,
                    1.0 / 3.0,
                    rel_tol=0.0,
                    abs_tol=1.0e-12,
                )
            ),
            "rmse": equality_rmse,
            "maximum_absolute_difference": (
                equality_max_abs
            ),
        },
        "support_constancy_QA": {
            "maximum_within_support_cell_std_of_zSM": (
                maximum_within_cell_s_std
            ),
            "expected": 0.0,
        },
        "old_interpolated_SM_comparison": old_comparison,
        "statistics": stats,
        "correlations": correlations,
        "warnings": warnings,
        "outputs": {
            "hkfi_raw_primary": str(HKFI_RAW_TIF),
            "hkfi_norm_primary": str(HKFI_NORM_TIF),
            "hkfi_norm_compatibility": str(HKFI_COMPAT_TIF),
            "hkfi_multiscale_raw_alias": str(
                HKFI_MS_RAW_TIF
            ),
            "hkfi_multiscale_norm_alias": str(
                HKFI_MS_NORM_TIF
            ),
            "K": str(KINEMATIC_TERM_TIF),
            "T": str(TWI_TERM_TIF),
            "S_support_preserved": str(SM_SUPPORT_Z_TIF),
            "SM_support_physical": str(
                SM_SUPPORT_PHYSICAL_TIF
            ),
            "SM_support_cell_id": str(
                SM_SUPPORT_CELL_ID_TIF
            ),
            "H_local": str(LOCAL_HYDRO_KINEMATIC_TIF),
            "old_SM_difference": (
                str(HKFI_OLD_SM_DIFFERENCE_TIF)
                if old_hkfi_difference is not None
                else None
            ),
            "weight_sensitivity": str(
                WEIGHT_SENSITIVITY_CSV
            ),
            "support_cell_table": str(
                SUPPORT_CELL_TABLE_CSV
            ),
            "metadata": str(META_JSON),
            "quality_assurance": str(QA_JSON),
            "statistics_csv": str(QA_CSV),
        },
    }

    quality_assurance = {
        "generated_at_utc": utc_now_iso(),
        "status": (
            "PASS"
            if not warnings
            else "PASS_WITH_WARNINGS"
        ),
        "warnings": warnings,
        "support_mode": support_metadata.get(
            "support_mode"
        ),
        "is_exact_native_era5_grid": (
            support_metadata.get(
                "is_exact_native_era5_grid"
            )
        ),
        "support_constancy": metadata[
            "support_constancy_QA"
        ],
        "equal_component_equivalence": metadata[
            "equal_component_equivalence_QA"
        ],
        "coverage": metadata["domain"],
        "normalization": norm_info,
        "statistics": stats,
        "correlations": correlations,
        "old_interpolated_SM_comparison": (
            old_comparison
        ),
    }

    with open(
        META_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            metadata,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    with open(
        QA_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            quality_assurance,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    rows: list[tuple[str, str, Any]] = []
    for component_name, component_stats in stats.items():
        for metric_name, metric_value in component_stats.items():
            rows.append(
                (
                    component_name,
                    metric_name,
                    metric_value,
                )
            )

    with open(
        QA_CSV,
        "w",
        newline="",
        encoding="utf-8",
    ) as handle:
        writer = csv.writer(handle)
        writer.writerow(
            ["component", "metric", "value"]
        )
        writer.writerows(rows)

    # -------------------------------------------------------------------------
    # Console summary
    # -------------------------------------------------------------------------
    log(
        f"[SUPPORT] mode="
        f"{support_metadata.get('support_mode')} | "
        f"exact_native="
        f"{support_metadata.get('is_exact_native_era5_grid')}"
    )
    log(
        f"[DOMAIN] compute={compute_n:,}/{context_n:,} "
        f"({100.0 * compute_fraction:.2f}%)"
    )
    log(
        f"[K] zero fraction="
        f"{100.0 * float(stats['K_kinematic_term']['zero_fraction']):.2f}%"
    )
    log(
        f"[HKFI_RAW] mean="
        f"{stats['hkfi_multiscale_raw']['mean']:.4f} | "
        f"std={stats['hkfi_multiscale_raw']['std']:.4f}"
    )
    log(
        f"[HKFI_NORM] mean="
        f"{stats['hkfi_multiscale_norm']['mean']:.4f} | "
        f"median={stats['hkfi_multiscale_norm']['median']:.4f}"
    )
    log(
        f"[SM SUPPORT QA] maximum within-cell z(SM) std="
        f"{maximum_within_cell_s_std:.3e}"
    )

    if warnings:
        log("[QA] PASS WITH WARNINGS")
        for warning in warnings:
            log(f"  - {warning}")
    else:
        log("[QA] PASS")

    log(f"[OUT] {HKFI_NORM_TIF}")
    log(f"[META] {META_JSON}")
    log(f"[QA] {QA_JSON}")

    make_qa_figures(
        hkfi_norm=hkfi_norm,
        compute_domain=compute_domain,
        study_mask=study,
        anchor=anchor,
    )

    log("=== Step 5.3 completed successfully ===")


def run_step_5_3() -> None:
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 5.3 failed: {exc}")
        traceback.print_exc()
        raise


if __name__ == "__main__":
    run_step_5_3()


In [ ]:
# =============================================================================
# Step 5.3 — Multiscale Hydro–Kinematic Fusion Index (HKFI)
# =============================================================================
# Components:
#
#     K(x) = IRS_HKFI(x) * z(DV+(x))
#     T(x) = z(TWI(x))
#     S(x) = support-preserved z(SM_j)
#
# General multiscale weighted formulation:
#
#     HKFI_raw(x) = w_K*K(x) + w_T*T(x) + w_S*S(x)
#
# subject to:
#
#     w_K >= 0, w_T >= 0, w_S >= 0, and w_K + w_T + w_S = 1
#
# The primary formulation uses transparent a priori equal nominal weights:
#
#     w_K = w_T = w_S = 1/3
#
# Equal nominal weights are treated as a neutral reference, not as evidence
# that the three components have equal physical or statistical influence.
#
# The script also generates:
#   1. named physically interpretable alternative-weight HKFI rasters;
#   2. component-ablation rasters (no K, no TWI, no SM);
#   3. a systematic simplex sensitivity table over all non-negative
#      K/T/S weights at a configurable increment;
#   4. a manifest of normalized scenario rasters for downstream spatial-CV
#      susceptibility-model sensitivity analysis.
#
# Spatial-support rules
# ---------------------
# - K and T are local 12.5 m components.
# - S is standardized across UNIQUE coarse support cells and assigned to the
#   12.5 m grid without interpolation; S is constant within each support cell.
# - The old interpolated 12.5 m z(SM) raster is diagnostic only.
# - IRS remains on [0,1] and is not re-standardized.
# - Missing z(DV+) gives I_DV=0 and K=0, representing neutral missing
#   kinematic evidence rather than observed stability.
#
# Important scope
# ---------------
# This step generates the alternative HKFI rasters required by Reviewer 1,
# Comment 7. Their effect on susceptibility AUC and mapped susceptibility must
# be evaluated later using identical samples, folds, preprocessing, and model
# settings in the downstream spatial cross-validation pipeline.
# =============================================================================

from __future__ import annotations

import csv
import json
import math
import os
import re
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.warp import reproject, transform_bounds


# =============================================================================
# USER CONFIGURATION
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

# Anchor and masks
ANCHOR_TIF = (
    BASE / "Results" / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
)
STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
CONTEXT_MASK_TIF = (
    BASE / "Results" / "Step-0.3-Quality_Masks" / "context_mask.tif"
)
DYNAMIC_MASK_TIF = (
    BASE / "Results" / "Step-0.3-Quality_Masks" / "dynamic_mask.tif"
)

# Step 4.2 standardized local inputs
STANDARDIZATION_DIR = BASE / "Results" / "Step-4.2-Standardization"
Z_DV_TIF = STANDARDIZATION_DIR / "z_dvplus_32645_12p5m_float32.tif"
Z_TWI_TIF = STANDARDIZATION_DIR / "z_twi_32645_12p5m_float32.tif"

# Old/interpolated z(SM), diagnostic only
OLD_Z_SM_TIF = STANDARDIZATION_DIR / "z_sm_32645_12p5m_float32.tif"
ZSCORE_META_JSON = STANDARDIZATION_DIR / "zscore_metadata.json"

# Step 5.2 corrected IRS
IRS_DIR = BASE / "Results" / "Step-5.2-IRS"
IRS_TIF = IRS_DIR / "irs_0_1_dense_32645_12p5m_float32.tif"
IRS_SUPPORT_MASK_TIF = (
    IRS_DIR / "irs_observation_support_mask_32645_12p5m_byte.tif"
)
IRS_META_JSON = IRS_DIR / "irs_metadata.json"

# Soil-moisture products confirmed by the user
SM_SOURCE_DIR = BASE / "Results" / "Step-2.3-SM"
SM_MONSOON_MEAN_TIF = (
    SM_SOURCE_DIR
    / "sm_monsoon_mean_2020_2023_32645_12p5m_float32.tif"
)
SM_MONSOON_MONTHLY_CSV = (
    SM_SOURCE_DIR
    / "sm_monsoon_monthly_2020_2023.csv"
)

# Optional exact native/coarse source raster.
# Set this manually when the original ERA5-Land raster is available.
ERA5_NATIVE_SM_OVERRIDE: Path | None = None

ERA5_NATIVE_SM_CANDIDATES = [
    SM_SOURCE_DIR / "era5_land_soil_moisture_native.tif",
    SM_SOURCE_DIR / "soil_moisture_native.tif",
    SM_SOURCE_DIR / "sm_native.tif",
    BASE / "Data" / "ERA5-Land" / "era5_land_soil_moisture_native.tif",
    BASE / "Data" / "ERA5-Land" / "soil_moisture_native.tif",
    BASE / "Data" / "ERA5" / "era5_soil_moisture_native.tif",
]

ERA5_NATIVE_BAND = 1
MIN_EXPECTED_NATIVE_CELL_SIZE_M = 4000.0
MAX_EXPECTED_NATIVE_CELL_SIZE_M = 20000.0

# Approximate support fallback
ALLOW_APPROXIMATE_9KM_AS_PRIMARY = True
APPROXIMATE_SUPPORT_CELL_SIZE_M = 9000.0

# Primary a priori reference weights.
# These are nominal WLC weights, not assumed physical contribution fractions.
PRIMARY_WEIGHTS = {
    "K": 1.0 / 3.0,
    "T": 1.0 / 3.0,
    "S": 1.0 / 3.0,
}

# Named physically interpretable scenarios for Reviewer 1, Comment 7.
# All scenarios are declared before model evaluation.
WRITE_NAMED_WEIGHT_SCENARIO_RASTERS = True
NAMED_WEIGHT_SCENARIOS = {
    "primary_equal_component": {
        "K": 1.0 / 3.0,
        "T": 1.0 / 3.0,
        "S": 1.0 / 3.0,
    },
    "kinematic_emphasis": {
        "K": 0.50,
        "T": 0.25,
        "S": 0.25,
    },
    "twi_emphasis": {
        "K": 0.25,
        "T": 0.50,
        "S": 0.25,
    },
    "sm_emphasis": {
        "K": 0.25,
        "T": 0.25,
        "S": 0.50,
    },
    "local_emphasis": {
        "K": 0.40,
        "T": 0.40,
        "S": 0.20,
    },
    "hydrological_emphasis": {
        "K": 0.20,
        "T": 0.40,
        "S": 0.40,
    },
    "no_kinematic_K": {
        "K": 0.00,
        "T": 0.50,
        "S": 0.50,
    },
    "no_TWI": {
        "K": 0.50,
        "T": 0.00,
        "S": 0.50,
    },
    "no_SM": {
        "K": 0.50,
        "T": 0.50,
        "S": 0.00,
    },
}

NAMED_WEIGHT_SCENARIO_PURPOSES = {
    "primary_equal_component": (
        "Transparent a priori non-preferential reference"
    ),
    "kinematic_emphasis": (
        "Tests stronger reliability-weighted deformation influence"
    ),
    "twi_emphasis": (
        "Tests stronger local topographic-wetness influence"
    ),
    "sm_emphasis": (
        "Tests stronger regional soil-moisture loading"
    ),
    "local_emphasis": (
        "Tests greater combined local K and T influence"
    ),
    "hydrological_emphasis": (
        "Tests greater combined TWI and soil-moisture influence"
    ),
    "no_kinematic_K": (
        "Kinematic component ablation"
    ),
    "no_TWI": (
        "Topographic-wetness component ablation"
    ),
    "no_SM": (
        "Soil-moisture component ablation"
    ),
}

# Systematic simplex sensitivity. A 0.10 increment produces 66 combinations.
RUN_SIMPLEX_WEIGHT_SENSITIVITY = True
SIMPLEX_WEIGHT_INCREMENT = 0.10
WRITE_SIMPLEX_WEIGHT_RASTERS = False

# Output directory
OUT_DIR = BASE / "Results" / "Step-5.3-HKFI"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Primary compatibility outputs used by downstream code
HKFI_RAW_TIF = (
    OUT_DIR / "hkfi_raw_zmean_32645_12p5m_float32.tif"
)
HKFI_NORM_TIF = (
    OUT_DIR / "hkfi_norm_0_1_dense_32645_12p5m_float32.tif"
)
HKFI_COMPAT_TIF = (
    OUT_DIR / "hkfi_norm_0_1_32645_12p5m_float32.tif"
)

# Explicit multiscale aliases
HKFI_MS_RAW_TIF = (
    OUT_DIR / "hkfi_multiscale_raw_32645_12p5m_float32.tif"
)
HKFI_MS_NORM_TIF = (
    OUT_DIR / "hkfi_multiscale_norm_0_1_32645_12p5m_float32.tif"
)

# Component and support outputs
KINEMATIC_TERM_TIF = (
    OUT_DIR / "hkfi_term_K_irs_times_zdv_32645_12p5m_float32.tif"
)
TWI_TERM_TIF = (
    OUT_DIR / "hkfi_term_T_ztwi_32645_12p5m_float32.tif"
)
SM_SUPPORT_PHYSICAL_TIF = (
    OUT_DIR / "sm_support_preserved_physical_32645_12p5m_float32.tif"
)
SM_SUPPORT_Z_TIF = (
    OUT_DIR / "hkfi_term_S_zsm_support_preserved_32645_12p5m_float32.tif"
)
SM_SUPPORT_CELL_ID_TIF = (
    OUT_DIR / "sm_support_cell_id_32645_12p5m_int32.tif"
)
LOCAL_HYDRO_KINEMATIC_TIF = (
    OUT_DIR / "hkfi_local_K_plus_T_over_2_32645_12p5m_float32.tif"
)
HKFI_OLD_SM_DIFFERENCE_TIF = (
    OUT_DIR / "hkfi_multiscale_minus_old_interpolated_sm_raw_32645_12p5m_float32.tif"
)

# Missing-data and domain outputs
HKFI_VALID_MASK_TIF = (
    OUT_DIR / "hkfi_valid_mask_32645_12p5m_byte.tif"
)
HKFI_DV_AVAILABILITY_MASK_TIF = (
    OUT_DIR / "hkfi_dv_availability_mask_32645_12p5m_byte.tif"
)
HKFI_MISSING_ZDV_MASK_TIF = (
    OUT_DIR / "hkfi_missing_zdv_mask_32645_12p5m_byte.tif"
)
HKFI_UNRESOLVED_MASK_TIF = (
    OUT_DIR
    / "hkfi_missing_zdv_nonzero_original_irs_mask_32645_12p5m_byte.tif"
)
IRS_EFFECTIVE_HKFI_TIF = (
    OUT_DIR / "irs_effective_for_hkfi_0_1_32645_12p5m_float32.tif"
)

# Weight-sensitivity outputs
WEIGHT_SENSITIVITY_DIR = OUT_DIR / "weight_sensitivity"
NAMED_WEIGHT_RASTER_DIR = WEIGHT_SENSITIVITY_DIR / "named_scenarios"
SIMPLEX_WEIGHT_RASTER_DIR = WEIGHT_SENSITIVITY_DIR / "simplex_scenarios"
NAMED_WEIGHT_RASTER_DIR.mkdir(parents=True, exist_ok=True)
SIMPLEX_WEIGHT_RASTER_DIR.mkdir(parents=True, exist_ok=True)

# Metadata and QA
META_JSON = OUT_DIR / "hkfi_metadata.json"
QA_JSON = OUT_DIR / "hkfi_quality_assurance.json"
QA_CSV = OUT_DIR / "hkfi_component_statistics.csv"
SUPPORT_CELL_TABLE_CSV = OUT_DIR / "sm_support_cell_values.csv"
SM_CSV_PROFILE_JSON = OUT_DIR / "sm_monthly_csv_profile.json"
SM_TEMPORAL_SUMMARY_CSV = OUT_DIR / "sm_monthly_temporal_summary.csv"
SM_SUPPORT_MODE_JSON = OUT_DIR / "sm_support_mode.json"

NAMED_WEIGHT_SENSITIVITY_CSV = (
    WEIGHT_SENSITIVITY_DIR / "hkfi_named_weight_sensitivity.csv"
)
SIMPLEX_WEIGHT_SENSITIVITY_CSV = (
    WEIGHT_SENSITIVITY_DIR / "hkfi_simplex_weight_sensitivity.csv"
)
WEIGHT_SCENARIO_MANIFEST_CSV = (
    WEIGHT_SENSITIVITY_DIR / "hkfi_weight_scenario_manifest.csv"
)
# Compatibility name retained for older downstream references.
WEIGHT_SENSITIVITY_CSV = (
    WEIGHT_SENSITIVITY_DIR / "hkfi_weight_sensitivity.csv"
)

# Optional QA figures
MAKE_QA_FIGURES = True
SHOW_QA_FIGURES = True
QA_MAP_PNG = OUT_DIR / "HKFI_QA_Map.png"
QA_HIST_PNG = OUT_DIR / "HKFI_QA_Histogram.png"
QA_MAP_FIGSIZE = (6.2, 9.0)
QA_HIST_FIGSIZE = (6.2, 4.2)
QA_FIG_DPI = 300

# Raster settings
NODATA_FLOAT = -9999.0
NODATA_BYTE = 255
NODATA_INT32 = -2147483648

# Scientific controls
IRS_EFFECTIVE_ZERO_TOL = 1.0e-6
REQUIRE_EXACT_GRID_FOR_LOCAL_INPUTS = True

# SM standardization at unique support-cell level
SM_WINSOR_P_LOW = 2.0
SM_WINSOR_P_HIGH = 98.0

# Final HKFI normalization
HKFI_NORM_P_LOW = 2.0
HKFI_NORM_P_HIGH = 98.0

# QA thresholds
WARN_INPUT_Z_MEAN_ABS = 0.25
WARN_INPUT_Z_STD_LOW = 0.70
WARN_INPUT_Z_STD_HIGH = 1.30
WARN_HKFI_UNIQUE_VALUES_MIN = 100
WARN_VALID_DOMAIN_FRACTION = 0.90
WARN_IRS_NEAR_ZERO_FRACTION = 0.85
WARN_APPROXIMATE_SUPPORT = True
WARN_K_ZERO_FRACTION = 0.70

EPS = 1.0e-12


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def load_json_optional(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return None


def normalize_column_name(name: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).strip().lower(),
    ).strip("_")


def detect_column(
    columns: list[str],
    exact_names: Iterable[str],
    required_tokens: Iterable[str] = (),
) -> str | None:
    normalized = {
        normalize_column_name(column): column
        for column in columns
    }

    for candidate in exact_names:
        key = normalize_column_name(candidate)
        if key in normalized:
            return normalized[key]

    required_tokens = tuple(required_tokens)
    if required_tokens:
        for normalized_name, original in normalized.items():
            if all(token in normalized_name for token in required_tokens):
                return original

    return None


# =============================================================================
# RASTER I/O
# =============================================================================
def read_anchor(path: Path) -> dict[str, Any]:
    require_file(path, "anchor raster")
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"Anchor raster has no CRS: {path}")

        transform = dataset.transform
        cell_area_m2 = abs(
            transform.a * transform.e
            - transform.b * transform.d
        )

        return {
            "transform": transform,
            "crs": dataset.crs,
            "width": dataset.width,
            "height": dataset.height,
            "profile": dataset.profile.copy(),
            "bounds": dataset.bounds,
            "epsg": dataset.crs.to_epsg(),
            "res_x": abs(transform.a),
            "res_y": abs(transform.e),
            "cell_area_m2": float(cell_area_m2),
        }


def same_grid(
    dataset: rasterio.io.DatasetReader,
    anchor: dict[str, Any],
) -> bool:
    return (
        dataset.crs == anchor["crs"]
        and dataset.transform == anchor["transform"]
        and dataset.width == anchor["width"]
        and dataset.height == anchor["height"]
    )


def scrub_common_nodata(
    array: np.ndarray,
    tagged_nodata: float | int | None,
) -> np.ndarray:
    result = array.astype("float64", copy=True)

    if tagged_nodata is not None:
        try:
            nodata_value = float(tagged_nodata)
            if np.isfinite(nodata_value):
                result[result == nodata_value] = np.nan
        except (TypeError, ValueError):
            pass

    result[result == -9999.0] = np.nan
    result[result == -32768.0] = np.nan
    result[np.abs(result) > 1.0e20] = np.nan
    return result


def read_continuous_exact(
    path: Path,
    anchor: dict[str, Any],
    description: str,
) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(path, description)

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        is_same = same_grid(dataset, anchor)

        if REQUIRE_EXACT_GRID_FOR_LOCAL_INPUTS and not is_same:
            raise ValueError(
                f"{description} does not match the anchor grid. "
                f"Regenerate the upstream raster rather than silently "
                f"resampling it:\n{path}"
            )

        array = scrub_common_nodata(
            dataset.read(1),
            dataset.nodata,
        )

        information = {
            "path": str(path),
            "dtype": dataset.dtypes[0],
            "nodata": json_safe(dataset.nodata),
            "crs": str(dataset.crs),
            "width": dataset.width,
            "height": dataset.height,
            "same_as_anchor": bool(is_same),
        }

    return array, information


def read_mask(
    path: Path,
    anchor: dict[str, Any],
    required: bool,
    description: str,
) -> tuple[np.ndarray | None, dict[str, Any] | None]:
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Missing {description}: {path}")
        return None, None

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        raw = dataset.read(1)
        is_same = same_grid(dataset, anchor)

        if is_same:
            values = raw.astype("float64")
            if dataset.nodata is not None:
                values[raw == dataset.nodata] = 0.0
            values[~np.isfinite(values)] = 0.0
            output = values > 0.5
        else:
            destination = np.zeros(
                (anchor["height"], anchor["width"]),
                dtype="float64",
            )
            reproject(
                source=raw,
                destination=destination,
                src_transform=dataset.transform,
                src_crs=dataset.crs,
                src_nodata=dataset.nodata,
                dst_transform=anchor["transform"],
                dst_crs=anchor["crs"],
                dst_nodata=0.0,
                resampling=Resampling.nearest,
            )
            destination[~np.isfinite(destination)] = 0.0
            output = destination > 0.5

        information = {
            "path": str(path),
            "crs": str(dataset.crs),
            "width": dataset.width,
            "height": dataset.height,
            "same_as_anchor": bool(is_same),
        }

    return output, information


def write_float_tif(
    path: Path,
    array: np.ndarray,
    anchor: dict[str, Any],
    variable: str,
    extra_tags: dict[str, Any] | None = None,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=NODATA_FLOAT,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    output = np.where(
        np.isfinite(array),
        array,
        NODATA_FLOAT,
    ).astype("float32")

    temporary = Path(str(path) + ".part")
    with rasterio.open(temporary, "w", **profile) as dataset:
        dataset.write(output, 1)
        dataset.set_band_description(1, variable)
        tags = {
            "step": "5.3_multiscale_HKFI",
            "variable": variable,
            "generated_utc": utc_now_iso(),
        }
        if extra_tags:
            tags.update(
                {
                    str(key): str(value)
                    for key, value in extra_tags.items()
                }
            )
        dataset.update_tags(**tags)

    os.replace(temporary, path)


def write_byte_tif(
    path: Path,
    boolean_array: np.ndarray,
    anchor: dict[str, Any],
    study_mask: np.ndarray,
    variable: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="uint8",
        count=1,
        nodata=NODATA_BYTE,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    output = np.full(
        boolean_array.shape,
        NODATA_BYTE,
        dtype="uint8",
    )
    output[study_mask] = boolean_array[study_mask].astype("uint8")

    temporary = Path(str(path) + ".part")
    with rasterio.open(temporary, "w", **profile) as dataset:
        dataset.write(output, 1)
        dataset.set_band_description(1, variable)
        dataset.update_tags(
            step="5.3_multiscale_HKFI",
            variable=variable,
            generated_utc=utc_now_iso(),
        )

    os.replace(temporary, path)


def write_int32_tif(
    path: Path,
    array: np.ndarray,
    anchor: dict[str, Any],
    variable: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="int32",
        count=1,
        nodata=NODATA_INT32,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    output = np.where(
        np.isfinite(array),
        array,
        NODATA_INT32,
    ).astype("int32")

    temporary = Path(str(path) + ".part")
    with rasterio.open(temporary, "w", **profile) as dataset:
        dataset.write(output, 1)
        dataset.set_band_description(1, variable)
        dataset.update_tags(
            step="5.3_multiscale_HKFI",
            variable=variable,
            generated_utc=utc_now_iso(),
        )

    os.replace(temporary, path)


# =============================================================================
# NUMERICAL HELPERS
# =============================================================================
def detailed_stats(
    array: np.ndarray,
    mask: np.ndarray,
) -> dict[str, float | int | None]:
    valid = mask & np.isfinite(array)
    values = array[valid].astype("float64")

    if values.size == 0:
        return {
            "n": 0,
            "min": None,
            "p01": None,
            "p05": None,
            "p25": None,
            "median": None,
            "mean": None,
            "std": None,
            "variance": None,
            "p75": None,
            "p95": None,
            "p99": None,
            "max": None,
            "zero_fraction": None,
            "near_zero_fraction": None,
            "unique_values_rounded_6dp": 0,
        }

    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "variance": float(np.var(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "near_zero_fraction": float(
            np.mean(np.abs(values) < 0.01)
        ),
        "unique_values_rounded_6dp": int(
            np.unique(np.round(values, 6)).size
        ),
    }


def safe_pearson(
    first: np.ndarray,
    second: np.ndarray,
    mask: np.ndarray,
) -> float | None:
    valid = (
        mask
        & np.isfinite(first)
        & np.isfinite(second)
    )

    if int(valid.sum()) < 3:
        return None

    x = first[valid].astype("float64")
    y = second[valid].astype("float64")

    if np.std(x) <= EPS or np.std(y) <= EPS:
        return None

    return float(np.corrcoef(x, y)[0, 1])



def safe_spearman(
    first: np.ndarray,
    second: np.ndarray,
    mask: np.ndarray,
) -> float | None:
    valid = (
        mask
        & np.isfinite(first)
        & np.isfinite(second)
    )

    if int(valid.sum()) < 3:
        return None

    x = first[valid].astype("float64")
    y = second[valid].astype("float64")

    x_rank = pd.Series(x).rank(
        method="average"
    ).to_numpy(dtype="float64")
    y_rank = pd.Series(y).rank(
        method="average"
    ).to_numpy(dtype="float64")

    if np.std(x_rank) <= EPS or np.std(y_rank) <= EPS:
        return None

    return float(np.corrcoef(x_rank, y_rank)[0, 1])


def validate_weights(
    weights: dict[str, float],
    label: str,
    tolerance: float = 1.0e-10,
) -> dict[str, float]:
    required = {"K", "T", "S"}

    if set(weights) != required:
        raise ValueError(
            f"{label} must define exactly K, T and S weights. "
            f"Received keys: {sorted(weights)}"
        )

    validated = {
        name: float(weights[name])
        for name in ("K", "T", "S")
    }

    if any(
        (not np.isfinite(value)) or value < 0.0
        for value in validated.values()
    ):
        raise ValueError(
            f"{label} contains a negative or non-finite weight: {validated}"
        )

    total = float(sum(validated.values()))

    if abs(total - 1.0) > tolerance:
        raise ValueError(
            f"{label} weights must sum to 1.0; received {total:.12f}."
        )

    return validated


def generate_simplex_weight_scenarios(
    increment: float,
) -> list[tuple[str, dict[str, float]]]:
    if not np.isfinite(increment) or increment <= 0.0 or increment > 1.0:
        raise ValueError(
            "SIMPLEX_WEIGHT_INCREMENT must be within (0,1]."
        )

    steps_float = 1.0 / increment
    steps = int(round(steps_float))

    if not math.isclose(
        steps * increment,
        1.0,
        rel_tol=0.0,
        abs_tol=1.0e-10,
    ):
        raise ValueError(
            "SIMPLEX_WEIGHT_INCREMENT must divide 1.0 exactly, "
            "for example 0.05, 0.10, 0.20 or 0.25."
        )

    scenarios: list[tuple[str, dict[str, float]]] = []

    for k_step in range(steps + 1):
        for t_step in range(steps - k_step + 1):
            s_step = steps - k_step - t_step

            weights = {
                "K": k_step / steps,
                "T": t_step / steps,
                "S": s_step / steps,
            }
            label = (
                f"simplex_K{k_step:02d}_T{t_step:02d}_S{s_step:02d}"
            )
            scenarios.append((label, weights))

    return scenarios


def combine_hkfi_components(
    K: np.ndarray,
    T: np.ndarray,
    S: np.ndarray,
    compute_domain: np.ndarray,
    weights: dict[str, float],
    label: str,
) -> np.ndarray:
    weights = validate_weights(weights, label)

    valid = (
        compute_domain
        & np.isfinite(K)
        & np.isfinite(T)
        & np.isfinite(S)
    )

    output = np.full_like(
        K,
        np.nan,
        dtype="float64",
    )
    output[valid] = (
        weights["K"] * K[valid]
        + weights["T"] * T[valid]
        + weights["S"] * S[valid]
    )
    return output


def scenario_comparison_metrics(
    primary_raw: np.ndarray,
    scenario_raw: np.ndarray,
    primary_norm: np.ndarray,
    scenario_norm: np.ndarray,
    common: np.ndarray,
) -> dict[str, float | int | None]:
    valid = (
        common
        & np.isfinite(primary_raw)
        & np.isfinite(scenario_raw)
        & np.isfinite(primary_norm)
        & np.isfinite(scenario_norm)
    )

    if int(valid.sum()) < 3:
        raise RuntimeError(
            "Too few common pixels for HKFI weight-sensitivity comparison."
        )

    raw_difference = (
        scenario_raw[valid] - primary_raw[valid]
    )
    norm_difference = (
        scenario_norm[valid] - primary_norm[valid]
    )

    primary_values = primary_norm[valid]
    scenario_values = scenario_norm[valid]

    primary_quintile = np.minimum(
        np.floor(primary_values * 5.0),
        4,
    ).astype("int16")
    scenario_quintile = np.minimum(
        np.floor(scenario_values * 5.0),
        4,
    ).astype("int16")

    primary_p90 = float(
        np.percentile(primary_values, 90)
    )
    scenario_p90 = float(
        np.percentile(scenario_values, 90)
    )

    primary_top = primary_values >= primary_p90
    scenario_top = scenario_values >= scenario_p90

    intersection = int(
        np.sum(primary_top & scenario_top)
    )
    union = int(
        np.sum(primary_top | scenario_top)
    )

    return {
        "n_common": int(valid.sum()),
        "raw_pearson_with_primary": safe_pearson(
            primary_raw,
            scenario_raw,
            valid,
        ),
        "raw_spearman_with_primary": safe_spearman(
            primary_raw,
            scenario_raw,
            valid,
        ),
        "norm_pearson_with_primary": safe_pearson(
            primary_norm,
            scenario_norm,
            valid,
        ),
        "norm_spearman_with_primary": safe_spearman(
            primary_norm,
            scenario_norm,
            valid,
        ),
        "raw_rmse_vs_primary": float(
            np.sqrt(np.mean(raw_difference ** 2))
        ),
        "raw_mae_vs_primary": float(
            np.mean(np.abs(raw_difference))
        ),
        "norm_rmse_vs_primary": float(
            np.sqrt(np.mean(norm_difference ** 2))
        ),
        "norm_mae_vs_primary": float(
            np.mean(np.abs(norm_difference))
        ),
        "quintile_change_fraction": float(
            np.mean(
                primary_quintile != scenario_quintile
            )
        ),
        "mean_absolute_quintile_shift": float(
            np.mean(
                np.abs(
                    primary_quintile - scenario_quintile
                )
            )
        ),
        "top_decile_jaccard": (
            float(intersection / union)
            if union > 0
            else None
        ),
    }


def robust_minmax_normalize(
    array: np.ndarray,
    stats_mask: np.ndarray,
    apply_mask: np.ndarray,
    p_low: float,
    p_high: float,
) -> tuple[np.ndarray, dict[str, float | int]]:
    values = array[
        stats_mask & np.isfinite(array)
    ].astype("float64")

    if values.size < 50:
        raise RuntimeError(
            "Too few valid HKFI pixels for stable normalization."
        )

    lower = float(np.percentile(values, p_low))
    upper = float(np.percentile(values, p_high))

    if (
        not np.isfinite(lower)
        or not np.isfinite(upper)
        or upper <= lower + EPS
    ):
        raise RuntimeError(
            "Invalid HKFI normalization bounds."
        )

    normalized = np.full_like(
        array,
        np.nan,
        dtype="float64",
    )
    valid_apply = apply_mask & np.isfinite(array)

    scaled = (
        array[valid_apply] - lower
    ) / (upper - lower)

    normalized[valid_apply] = np.clip(
        scaled,
        0.0,
        1.0,
    )

    information = {
        "p_low": float(p_low),
        "p_high": float(p_high),
        "lower_bound": lower,
        "upper_bound": upper,
        "n_stats": int(values.size),
        "n_applied": int(valid_apply.sum()),
        "lower_clipped_fraction": float(
            np.mean(array[valid_apply] <= lower)
        ),
        "upper_clipped_fraction": float(
            np.mean(array[valid_apply] >= upper)
        ),
    }

    return normalized, information


# =============================================================================
# SOIL-MOISTURE SOURCE DISCOVERY AND SUPPORT PRESERVATION
# =============================================================================
def approximate_cell_size_metres(
    dataset: rasterio.io.DatasetReader,
) -> tuple[float, float]:
    if dataset.crs is None:
        raise ValueError("Raster has no CRS.")

    transform = dataset.transform

    if dataset.crs.is_projected:
        return abs(float(transform.a)), abs(float(transform.e))

    row = dataset.height // 2
    col = dataset.width // 2

    left, top = transform * (col, row)
    right, bottom = transform * (col + 1, row + 1)

    bounds_3857 = transform_bounds(
        dataset.crs,
        "EPSG:3857",
        min(left, right),
        min(bottom, top),
        max(left, right),
        max(bottom, top),
        densify_pts=5,
    )

    return (
        float(abs(bounds_3857[2] - bounds_3857[0])),
        float(abs(bounds_3857[3] - bounds_3857[1])),
    )


def choose_native_raster() -> Path | None:
    if ERA5_NATIVE_SM_OVERRIDE is not None:
        path = Path(ERA5_NATIVE_SM_OVERRIDE)
        require_file(path, "native ERA5-Land soil-moisture raster")
        return path

    for candidate in ERA5_NATIVE_SM_CANDIDATES:
        if not candidate.exists():
            continue

        try:
            with rasterio.open(candidate) as dataset:
                width_m, height_m = approximate_cell_size_metres(dataset)
                if (
                    min(width_m, height_m)
                    >= MIN_EXPECTED_NATIVE_CELL_SIZE_M
                    and max(width_m, height_m)
                    <= MAX_EXPECTED_NATIVE_CELL_SIZE_M
                ):
                    return candidate
        except Exception:
            continue

    return None


def inspect_monthly_csv(
    path: Path,
) -> tuple[
    pd.DataFrame | None,
    dict[str, Any],
    pd.DataFrame,
]:
    if not path.exists():
        return (
            None,
            {
                "path": str(path),
                "exists": False,
                "spatial_grid_reconstructable": False,
            },
            pd.DataFrame(),
        )

    dataframe = pd.read_csv(path)

    if dataframe.empty:
        raise RuntimeError(
            f"Monthly soil-moisture CSV is empty: {path}"
        )

    columns = list(dataframe.columns)

    longitude_column = detect_column(
        columns,
        ["longitude", "lon", "long", "grid_longitude"],
    )
    latitude_column = detect_column(
        columns,
        ["latitude", "lat", "grid_latitude"],
    )
    year_column = detect_column(
        columns,
        ["year", "yr"],
    )
    month_column = detect_column(
        columns,
        ["month", "mon", "month_number"],
    )
    date_column = detect_column(
        columns,
        ["date", "time", "datetime", "valid_time"],
    )

    value_column = detect_column(
        columns,
        [
            "soil_moisture",
            "sm",
            "sm_mean",
            "mean_sm",
            "monthly_sm",
            "monthly_mean_sm",
            "swvl",
            "swvl1",
            "swvl2",
            "volumetric_soil_water",
            "value",
        ],
    )

    if value_column is None:
        scored: list[tuple[int, float, str]] = []

        for column in columns:
            numeric = pd.to_numeric(
                dataframe[column],
                errors="coerce",
            )
            finite_fraction = float(
                numeric.notna().mean()
            )

            if finite_fraction < 0.5:
                continue

            normalized = normalize_column_name(column)
            score = 0

            for token, weight in [
                ("soil", 4),
                ("moisture", 4),
                ("swvl", 5),
                ("sm", 2),
                ("mean", 1),
                ("value", 1),
            ]:
                if token in normalized:
                    score += weight

            if column in {
                longitude_column,
                latitude_column,
                year_column,
                month_column,
            }:
                score -= 20

            scored.append((score, finite_fraction, column))

        scored.sort(reverse=True)

        if scored and scored[0][0] > 0:
            value_column = scored[0][2]

    working = dataframe.copy()

    if date_column is not None:
        parsed_date = pd.to_datetime(
            working[date_column],
            errors="coerce",
        )

        if year_column is None and parsed_date.notna().any():
            working["_detected_year"] = parsed_date.dt.year
            year_column = "_detected_year"

        if month_column is None and parsed_date.notna().any():
            working["_detected_month"] = parsed_date.dt.month
            month_column = "_detected_month"

    temporal_summary = pd.DataFrame()

    if value_column is not None:
        working["_sm_numeric"] = pd.to_numeric(
            working[value_column],
            errors="coerce",
        )

        groups = [
            column
            for column in [year_column, month_column]
            if column is not None
        ]

        if groups:
            temporal_summary = (
                working
                .dropna(subset=groups + ["_sm_numeric"])
                .groupby(groups, dropna=False)["_sm_numeric"]
                .agg(
                    n="count",
                    mean="mean",
                    std="std",
                    minimum="min",
                    median="median",
                    maximum="max",
                )
                .reset_index()
            )

    spatial_grid_reconstructable = bool(
        longitude_column is not None
        and latitude_column is not None
        and value_column is not None
    )

    profile = {
        "path": str(path),
        "exists": True,
        "row_count": int(len(dataframe)),
        "columns": columns,
        "detected_columns": {
            "longitude": longitude_column,
            "latitude": latitude_column,
            "year": year_column,
            "month": month_column,
            "date": date_column,
            "soil_moisture_value": value_column,
        },
        "spatial_grid_reconstructable": (
            spatial_grid_reconstructable
        ),
    }

    return working, profile, temporal_summary


def reproject_native_raster_to_anchor(
    path: Path,
    anchor: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(
                f"Native ERA5 raster has no CRS: {path}"
            )

        if (
            ERA5_NATIVE_BAND < 1
            or ERA5_NATIVE_BAND > dataset.count
        ):
            raise ValueError(
                f"ERA5_NATIVE_BAND={ERA5_NATIVE_BAND} is invalid "
                f"for {dataset.count} band(s)."
            )

        width_m, height_m = approximate_cell_size_metres(
            dataset
        )

        if (
            min(width_m, height_m)
            < MIN_EXPECTED_NATIVE_CELL_SIZE_M
            or max(width_m, height_m)
            > MAX_EXPECTED_NATIVE_CELL_SIZE_M
        ):
            raise ValueError(
                "Selected ERA5 raster does not appear to have the expected "
                "coarse ERA5-Land support. Estimated cell size: "
                f"{width_m:.1f} x {height_m:.1f} m."
            )

        native_physical = scrub_common_nodata(
            dataset.read(ERA5_NATIVE_BAND),
            dataset.nodata,
        )
        native_valid = np.isfinite(native_physical)

        native_ids = (
            np.arange(
                dataset.height * dataset.width,
                dtype="int64",
            ).reshape(dataset.height, dataset.width)
            + 1
        )
        native_ids = np.where(
            native_valid,
            native_ids,
            NODATA_INT32,
        ).astype("int32")

        physical_anchor = np.full(
            (anchor["height"], anchor["width"]),
            np.nan,
            dtype="float64",
        )
        id_anchor = np.full(
            (anchor["height"], anchor["width"]),
            NODATA_INT32,
            dtype="int32",
        )

        reproject(
            source=native_physical,
            destination=physical_anchor,
            src_transform=dataset.transform,
            src_crs=dataset.crs,
            src_nodata=np.nan,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=np.nan,
            resampling=Resampling.nearest,
        )

        reproject(
            source=native_ids,
            destination=id_anchor,
            src_transform=dataset.transform,
            src_crs=dataset.crs,
            src_nodata=NODATA_INT32,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=NODATA_INT32,
            resampling=Resampling.nearest,
        )

        id_anchor_float = id_anchor.astype("float64")
        id_anchor_float[
            id_anchor == NODATA_INT32
        ] = np.nan

        metadata = {
            "support_mode": "exact_native_era5_raster",
            "is_exact_native_era5_grid": True,
            "source": str(path),
            "source_crs": dataset.crs.to_string(),
            "source_width": dataset.width,
            "source_height": dataset.height,
            "estimated_cell_width_m": width_m,
            "estimated_cell_height_m": height_m,
        }

    return physical_anchor, id_anchor_float, metadata


def reconstruct_csv_grid_to_anchor(
    dataframe: pd.DataFrame | None,
    profile: dict[str, Any],
    anchor: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]] | None:
    if (
        dataframe is None
        or not profile.get(
            "spatial_grid_reconstructable",
            False,
        )
    ):
        return None

    detected = profile["detected_columns"]
    lon_column = detected["longitude"]
    lat_column = detected["latitude"]
    value_column = detected["soil_moisture_value"]

    working = pd.DataFrame(
        {
            "longitude": pd.to_numeric(
                dataframe[lon_column],
                errors="coerce",
            ),
            "latitude": pd.to_numeric(
                dataframe[lat_column],
                errors="coerce",
            ),
            "sm": pd.to_numeric(
                dataframe[value_column],
                errors="coerce",
            ),
        }
    ).dropna()

    working = working[
        working["longitude"].between(-180.0, 180.0)
        & working["latitude"].between(-90.0, 90.0)
    ]

    if working.empty:
        return None

    cell_means = (
        working
        .groupby(
            ["longitude", "latitude"],
            as_index=False,
        )["sm"]
        .mean()
    )

    x_values = np.sort(
        cell_means["longitude"].unique()
    )
    y_values = np.sort(
        cell_means["latitude"].unique()
    )[::-1]

    if x_values.size < 2 or y_values.size < 2:
        return None

    dx = float(np.median(np.diff(x_values)))
    dy = float(np.median(np.abs(np.diff(y_values))))

    if dx <= 0 or dy <= 0:
        return None

    x_deviation = float(
        np.max(np.abs(np.diff(x_values) - dx))
        / max(abs(dx), EPS)
    )
    y_deviation = float(
        np.max(np.abs(np.abs(np.diff(y_values)) - dy))
        / max(abs(dy), EPS)
    )

    if x_deviation > 0.05 or y_deviation > 0.05:
        log(
            "[WARN] CSV coordinates do not define a sufficiently regular grid."
        )
        return None

    median_latitude = float(
        np.median(cell_means["latitude"].to_numpy(dtype="float64"))
    )
    estimated_width_m = float(
        abs(dx)
        * 111320.0
        * max(
            math.cos(math.radians(median_latitude)),
            1.0e-6,
        )
    )
    estimated_height_m = float(
        abs(dy) * 110574.0
    )

    if (
        min(estimated_width_m, estimated_height_m)
        < MIN_EXPECTED_NATIVE_CELL_SIZE_M
        or max(estimated_width_m, estimated_height_m)
        > MAX_EXPECTED_NATIVE_CELL_SIZE_M
    ):
        log(
            "[WARN] CSV coordinates define a regular grid, but its "
            "estimated cell size is inconsistent with the expected "
            "coarse ERA5-Land support: "
            f"{estimated_width_m:.1f} x {estimated_height_m:.1f} m."
        )
        return None

    x_lookup = {
        float(value): index
        for index, value in enumerate(x_values)
    }
    y_lookup = {
        float(value): index
        for index, value in enumerate(y_values)
    }

    native_array = np.full(
        (y_values.size, x_values.size),
        np.nan,
        dtype="float64",
    )

    for row in cell_means.itertuples(index=False):
        native_array[
            y_lookup[float(row.latitude)],
            x_lookup[float(row.longitude)],
        ] = float(row.sm)

    if float(np.isfinite(native_array).mean()) < 0.5:
        return None

    native_transform = from_origin(
        float(x_values.min() - dx / 2.0),
        float(y_values.max() + dy / 2.0),
        dx,
        dy,
    )

    native_ids = (
        np.arange(
            native_array.size,
            dtype="int64",
        ).reshape(native_array.shape)
        + 1
    )
    native_ids = np.where(
        np.isfinite(native_array),
        native_ids,
        NODATA_INT32,
    ).astype("int32")

    physical_anchor = np.full(
        (anchor["height"], anchor["width"]),
        np.nan,
        dtype="float64",
    )
    id_anchor = np.full(
        (anchor["height"], anchor["width"]),
        NODATA_INT32,
        dtype="int32",
    )

    reproject(
        source=native_array,
        destination=physical_anchor,
        src_transform=native_transform,
        src_crs="EPSG:4326",
        src_nodata=np.nan,
        dst_transform=anchor["transform"],
        dst_crs=anchor["crs"],
        dst_nodata=np.nan,
        resampling=Resampling.nearest,
    )

    reproject(
        source=native_ids,
        destination=id_anchor,
        src_transform=native_transform,
        src_crs="EPSG:4326",
        src_nodata=NODATA_INT32,
        dst_transform=anchor["transform"],
        dst_crs=anchor["crs"],
        dst_nodata=NODATA_INT32,
        resampling=Resampling.nearest,
    )

    id_anchor_float = id_anchor.astype("float64")
    id_anchor_float[
        id_anchor == NODATA_INT32
    ] = np.nan

    metadata = {
        "support_mode": "exact_native_grid_reconstructed_from_csv",
        "is_exact_native_era5_grid": True,
        "source": str(SM_MONSOON_MONTHLY_CSV),
        "native_crs": "EPSG:4326",
        "native_width": int(x_values.size),
        "native_height": int(y_values.size),
        "native_dx_degrees": dx,
        "native_dy_degrees": dy,
        "estimated_cell_width_m": estimated_width_m,
        "estimated_cell_height_m": estimated_height_m,
        "aggregation": (
            "Mean of monthly values at each longitude-latitude support cell"
        ),
    }

    return physical_anchor, id_anchor_float, metadata


def build_approximate_support_ids(
    anchor: dict[str, Any],
    domain: np.ndarray,
    cell_size_m: float,
) -> tuple[np.ndarray, dict[str, Any]]:
    if not anchor["crs"].is_projected:
        raise RuntimeError(
            "Approximate support fallback requires a projected anchor CRS."
        )

    transform = anchor["transform"]

    if abs(transform.b) > 1.0e-10 or abs(transform.d) > 1.0e-10:
        raise RuntimeError(
            "Approximate support fallback requires a north-up anchor grid."
        )

    x_centres = (
        transform.c
        + (np.arange(anchor["width"], dtype="float64") + 0.5)
        * transform.a
    )
    y_centres = (
        transform.f
        + (np.arange(anchor["height"], dtype="float64") + 0.5)
        * transform.e
    )

    origin_x = (
        math.floor(float(anchor["bounds"].left) / cell_size_m)
        * cell_size_m
    )
    origin_y = (
        math.floor(float(anchor["bounds"].bottom) / cell_size_m)
        * cell_size_m
    )

    col_group = np.floor(
        (x_centres - origin_x) / cell_size_m
    ).astype("int64")
    row_group = np.floor(
        (y_centres - origin_y) / cell_size_m
    ).astype("int64")

    row_index = row_group - int(row_group.min())
    col_index = col_group - int(col_group.min())
    n_columns = int(col_index.max()) + 1

    ids = (
        row_index[:, None] * n_columns
        + col_index[None, :]
        + 1
    ).astype("float64")
    ids[~domain] = np.nan

    metadata = {
        "support_mode": "approximate_9km_from_physical_sm",
        "is_exact_native_era5_grid": False,
        "cell_size_m": float(cell_size_m),
        "origin_x": float(origin_x),
        "origin_y": float(origin_y),
        "interpretation": (
            "Approximate support grid used because the exact native ERA5 "
            "geometry was unavailable. Do not report as exact native cells."
        ),
    }

    return ids, metadata


def aggregate_physical_sm_by_support(
    physical_sm: np.ndarray,
    support_ids: np.ndarray,
    domain: np.ndarray,
) -> tuple[np.ndarray, pd.DataFrame]:
    valid = (
        domain
        & np.isfinite(physical_sm)
        & np.isfinite(support_ids)
        & (support_ids > 0)
    )

    if valid.sum() < 2:
        raise RuntimeError(
            "Insufficient physical SM values for support aggregation."
        )

    ids = support_ids[valid].astype("int64")
    values = physical_sm[valid].astype("float64")

    dataframe = pd.DataFrame(
        {
            "support_id": ids,
            "physical_sm": values,
        }
    )

    cell_table = (
        dataframe
        .groupby("support_id", as_index=False)
        .agg(
            physical_sm_mean=("physical_sm", "mean"),
            physical_sm_median=("physical_sm", "median"),
            fine_pixel_count=("physical_sm", "size"),
        )
    )

    value_lookup = dict(
        zip(
            cell_table["support_id"].astype("int64"),
            cell_table["physical_sm_mean"].astype("float64"),
        )
    )

    repeated = np.full(
        physical_sm.shape,
        np.nan,
        dtype="float64",
    )

    integer_ids = np.where(
        np.isfinite(support_ids),
        support_ids,
        -1,
    ).astype("int64")

    unique_ids = np.unique(ids)
    for support_id in unique_ids:
        repeated[
            valid & (integer_ids == support_id)
        ] = value_lookup[int(support_id)]

    return repeated, cell_table


def standardize_unique_support_values(
    support_physical: np.ndarray,
    support_ids: np.ndarray,
    domain: np.ndarray,
) -> tuple[np.ndarray, pd.DataFrame, dict[str, Any]]:
    """
    Winsorize and standardize one value per UNIQUE support cell.

    Fine-pixel replication is performed only after support-cell z-scores are
    calculated, preventing repeated 12.5 m pixels from weighting the same
    coarse observation multiple times.
    """
    valid = (
        domain
        & np.isfinite(support_physical)
        & np.isfinite(support_ids)
        & (support_ids > 0)
    )

    if valid.sum() < 2:
        raise RuntimeError(
            "Insufficient support-preserved SM values for standardization."
        )

    dataframe = pd.DataFrame(
        {
            "support_id": support_ids[valid].astype("int64"),
            "physical_sm": support_physical[valid].astype("float64"),
        }
    )

    cells = (
        dataframe
        .groupby("support_id", as_index=False)
        .agg(
            physical_sm=("physical_sm", "median"),
            fine_pixel_count=("physical_sm", "size"),
        )
    )

    if len(cells) < 2:
        raise RuntimeError(
            "Fewer than two unique SM support cells intersect the study domain."
        )

    unique_values = cells["physical_sm"].to_numpy(dtype="float64")

    lower = float(
        np.percentile(unique_values, SM_WINSOR_P_LOW)
    )
    upper = float(
        np.percentile(unique_values, SM_WINSOR_P_HIGH)
    )

    if upper <= lower + EPS:
        raise RuntimeError(
            "Support-cell SM values are effectively constant."
        )

    winsorized = np.clip(
        unique_values,
        lower,
        upper,
    )

    mean = float(np.mean(winsorized))
    std = float(np.std(winsorized))

    if std <= EPS:
        raise RuntimeError(
            "Support-cell SM standard deviation is zero."
        )

    z_values = (winsorized - mean) / std

    cells["winsorized_sm"] = winsorized
    cells["z_sm_support"] = z_values

    z_lookup = dict(
        zip(
            cells["support_id"].astype("int64"),
            cells["z_sm_support"].astype("float64"),
        )
    )

    z_raster = np.full(
        support_physical.shape,
        np.nan,
        dtype="float64",
    )

    integer_ids = np.where(
        np.isfinite(support_ids),
        support_ids,
        -1,
    ).astype("int64")

    for support_id, z_value in z_lookup.items():
        z_raster[
            valid & (integer_ids == int(support_id))
        ] = float(z_value)

    metadata = {
        "standardization_level": "unique_support_cells",
        "number_of_unique_support_cells": int(len(cells)),
        "winsor_p_low": SM_WINSOR_P_LOW,
        "winsor_p_high": SM_WINSOR_P_HIGH,
        "winsor_lower": lower,
        "winsor_upper": upper,
        "winsor_mean_across_unique_cells": mean,
        "winsor_std_across_unique_cells": std,
        "z_mean_across_unique_cells": float(
            np.mean(z_values)
        ),
        "z_std_across_unique_cells": float(
            np.std(z_values)
        ),
        "within_support_cell_z_std_expected": 0.0,
    }

    return z_raster, cells, metadata


def prepare_support_preserved_sm(
    anchor: dict[str, Any],
    study_domain: np.ndarray,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    dict[str, Any],
    pd.DataFrame,
    dict[str, Any],
]:
    require_file(
        SM_MONSOON_MEAN_TIF,
        "physical monsoon-mean soil-moisture raster",
    )

    physical_fine, physical_info = read_continuous_exact(
        SM_MONSOON_MEAN_TIF,
        anchor,
        "physical monsoon-mean SM raster",
    )

    csv_dataframe, csv_profile, temporal_summary = inspect_monthly_csv(
        SM_MONSOON_MONTHLY_CSV
    )

    with open(
        SM_CSV_PROFILE_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            csv_profile,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    if not temporal_summary.empty:
        temporal_summary.to_csv(
            SM_TEMPORAL_SUMMARY_CSV,
            index=False,
        )

    native_path = choose_native_raster()

    support_physical: np.ndarray
    support_ids: np.ndarray
    support_metadata: dict[str, Any]

    if native_path is not None:
        (
            support_physical,
            support_ids,
            support_metadata,
        ) = reproject_native_raster_to_anchor(
            native_path,
            anchor,
        )

    else:
        csv_result = reconstruct_csv_grid_to_anchor(
            csv_dataframe,
            csv_profile,
            anchor,
        )

        if csv_result is not None:
            (
                support_physical,
                support_ids,
                support_metadata,
            ) = csv_result

        else:
            if not ALLOW_APPROXIMATE_9KM_AS_PRIMARY:
                raise RuntimeError(
                    "No exact native/coarse ERA5 support was recovered and "
                    "ALLOW_APPROXIMATE_9KM_AS_PRIMARY=False."
                )

            support_ids, support_metadata = (
                build_approximate_support_ids(
                    anchor,
                    study_domain & np.isfinite(physical_fine),
                    APPROXIMATE_SUPPORT_CELL_SIZE_M,
                )
            )

            support_physical, approximate_cell_table = (
                aggregate_physical_sm_by_support(
                    physical_fine,
                    support_ids,
                    study_domain,
                )
            )
            support_metadata["approximate_cell_table_rows"] = int(
                len(approximate_cell_table)
            )

    support_domain = (
        study_domain
        & np.isfinite(support_physical)
        & np.isfinite(support_ids)
        & (support_ids > 0)
    )

    z_sm_support, support_cell_table, standardization_metadata = (
        standardize_unique_support_values(
            support_physical,
            support_ids,
            support_domain,
        )
    )

    support_metadata.update(
        {
            "physical_fine_input": str(SM_MONSOON_MEAN_TIF),
            "monthly_csv_input": str(SM_MONSOON_MONTHLY_CSV),
            "physical_fine_input_info": physical_info,
            "csv_profile": csv_profile,
            "standardization": standardization_metadata,
            "support_domain_pixel_count": int(
                support_domain.sum()
            ),
        }
    )

    return (
        support_physical,
        z_sm_support,
        support_ids,
        support_metadata,
        support_cell_table,
        physical_info,
    )


# =============================================================================
# HKFI COMPUTATION
# =============================================================================
def compute_multiscale_hkfi(
    z_dv: np.ndarray,
    z_twi: np.ndarray,
    z_sm_support: np.ndarray,
    irs: np.ndarray,
    context_domain: np.ndarray,
    weights: dict[str, float],
    label: str = "HKFI",
) -> dict[str, np.ndarray | int | dict[str, float]]:
    weights = validate_weights(weights, label)

    irs_original = np.full_like(
        irs,
        np.nan,
        dtype="float64",
    )
    finite_irs = np.isfinite(irs)
    irs_original[finite_irs] = np.clip(
        irs[finite_irs],
        0.0,
        1.0,
    )

    regional_hydro_valid = (
        context_domain
        & np.isfinite(z_twi)
        & np.isfinite(z_sm_support)
        & np.isfinite(irs_original)
    )

    dv_available = (
        regional_hydro_valid
        & np.isfinite(z_dv)
    )
    missing_zdv = (
        regional_hydro_valid
        & ~np.isfinite(z_dv)
    )

    missing_zdv_nonzero_irs = (
        missing_zdv
        & (irs_original > IRS_EFFECTIVE_ZERO_TOL)
    )

    irs_for_hkfi = np.full_like(
        irs_original,
        np.nan,
        dtype="float64",
    )
    irs_for_hkfi[regional_hydro_valid] = (
        irs_original[regional_hydro_valid]
    )
    irs_for_hkfi[missing_zdv] = 0.0

    z_dv_effective = np.full_like(
        z_dv,
        np.nan,
        dtype="float64",
    )
    z_dv_effective[dv_available] = z_dv[dv_available]
    z_dv_effective[missing_zdv] = 0.0

    compute_domain = regional_hydro_valid

    K = np.full_like(z_dv, np.nan, dtype="float64")
    T = np.full_like(z_twi, np.nan, dtype="float64")
    S = np.full_like(z_sm_support, np.nan, dtype="float64")

    K[compute_domain] = (
        irs_for_hkfi[compute_domain]
        * z_dv_effective[compute_domain]
    )
    T[compute_domain] = z_twi[compute_domain]
    S[compute_domain] = z_sm_support[compute_domain]

    H_local = np.full_like(
        z_twi,
        np.nan,
        dtype="float64",
    )
    H_local[compute_domain] = (
        K[compute_domain] + T[compute_domain]
    ) / 2.0

    hkfi_raw = combine_hkfi_components(
        K=K,
        T=T,
        S=S,
        compute_domain=compute_domain,
        weights=weights,
        label=label,
    )

    equal_component_reference = np.full_like(
        z_twi,
        np.nan,
        dtype="float64",
    )
    equal_component_reference[compute_domain] = (
        K[compute_domain]
        + T[compute_domain]
        + S[compute_domain]
    ) / 3.0

    return {
        "weights": weights,
        "irs_original": irs_original,
        "irs_for_hkfi": irs_for_hkfi,
        "compute_domain": compute_domain,
        "dv_available_mask": dv_available,
        "missing_zdv_mask": missing_zdv,
        "missing_zdv_nonzero_irs_mask": (
            missing_zdv_nonzero_irs
        ),
        "K": K,
        "T": T,
        "S": S,
        "H_local": H_local,
        "hkfi_raw": hkfi_raw,
        "equal_component_reference": (
            equal_component_reference
        ),
        "n_missing_zdv": int(missing_zdv.sum()),
        "n_missing_zdv_nonzero_irs": int(
            missing_zdv_nonzero_irs.sum()
        ),
    }


# =============================================================================
# QA FIGURES
# =============================================================================
def make_qa_figures(
    hkfi_norm: np.ndarray,
    compute_domain: np.ndarray,
    study_mask: np.ndarray,
    anchor: dict[str, Any],
) -> None:
    if not MAKE_QA_FIGURES:
        return

    import matplotlib.pyplot as plt

    bounds = anchor["bounds"]
    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    display_array = np.where(
        compute_domain,
        hkfi_norm,
        np.nan,
    )
    values = hkfi_norm[
        compute_domain & np.isfinite(hkfi_norm)
    ]

    if values.size == 0:
        return

    figure, axis = plt.subplots(
        figsize=QA_MAP_FIGSIZE,
        facecolor="white",
    )
    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        interpolation="nearest",
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
    )
    axis.contour(
        study_mask.astype("uint8"),
        levels=[0.5],
        extent=extent,
        origin="upper",
        colors="black",
        linewidths=0.7,
    )
    axis.set_title(
        "Multiscale Hydro–Kinematic Fusion Index",
        fontsize=11,
        fontweight="bold",
    )
    axis.set_xlabel("Easting (m)", fontsize=9)
    axis.set_ylabel("Northing (m)", fontsize=9)
    axis.tick_params(labelsize=7)

    colorbar = figure.colorbar(
        image,
        ax=axis,
        fraction=0.035,
        pad=0.025,
    )
    colorbar.set_label(
        "Study-relative multiscale HKFI (0–1)",
        fontsize=9,
    )

    figure.tight_layout()
    figure.savefig(
        QA_MAP_PNG,
        dpi=QA_FIG_DPI,
        bbox_inches="tight",
        facecolor="white",
    )

    if SHOW_QA_FIGURES:
        plt.show()
    else:
        plt.close(figure)

    figure_h, axis_h = plt.subplots(
        figsize=QA_HIST_FIGSIZE,
        facecolor="white",
    )
    axis_h.hist(
        values,
        bins=np.linspace(0.0, 1.0, 31),
        edgecolor="black",
        linewidth=0.5,
        alpha=0.85,
    )
    axis_h.axvline(
        float(np.mean(values)),
        linestyle="--",
        linewidth=1.0,
        label=f"Mean = {float(np.mean(values)):.3f}",
    )
    axis_h.axvline(
        float(np.median(values)),
        linestyle=":",
        linewidth=1.2,
        label=f"Median = {float(np.median(values)):.3f}",
    )
    axis_h.set_xlim(0.0, 1.0)
    axis_h.set_xlabel(
        "Study-relative multiscale HKFI (0–1)",
        fontsize=9,
    )
    axis_h.set_ylabel("Number of pixels", fontsize=9)
    axis_h.grid(axis="y", alpha=0.25)
    axis_h.legend(fontsize=8)
    figure_h.tight_layout()
    figure_h.savefig(
        QA_HIST_PNG,
        dpi=QA_FIG_DPI,
        bbox_inches="tight",
        facecolor="white",
    )

    if SHOW_QA_FIGURES:
        plt.show()
    else:
        plt.close(figure_h)


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=== Step 5.3: multiscale HKFI ===")

    primary_weights = validate_weights(
        PRIMARY_WEIGHTS,
        "PRIMARY_WEIGHTS",
    )

    equal_primary = all(
        math.isclose(
            primary_weights[name],
            1.0 / 3.0,
            rel_tol=0.0,
            abs_tol=1.0e-12,
        )
        for name in ("K", "T", "S")
    )

    if not equal_primary:
        log(
            "[WARN] PRIMARY_WEIGHTS are not equal one-third weights. "
            "Update the primary manuscript equation and interpretation."
        )

    validated_named_scenarios = {
        name: validate_weights(
            weights,
            f"NAMED_WEIGHT_SCENARIOS[{name}]",
        )
        for name, weights in NAMED_WEIGHT_SCENARIOS.items()
    }

    if "primary_equal_component" not in validated_named_scenarios:
        raise ValueError(
            "NAMED_WEIGHT_SCENARIOS must include primary_equal_component."
        )

    if any(
        abs(
            validated_named_scenarios[
                "primary_equal_component"
            ][component]
            - primary_weights[component]
        ) > 1.0e-12
        for component in ("K", "T", "S")
    ):
        raise ValueError(
            "primary_equal_component scenario does not match PRIMARY_WEIGHTS."
        )

    anchor = read_anchor(ANCHOR_TIF)

    study, study_info = read_mask(
        STUDY_MASK_TIF,
        anchor,
        required=True,
        description="study mask",
    )
    assert study is not None

    context, context_info = read_mask(
        CONTEXT_MASK_TIF,
        anchor,
        required=False,
        description="context mask",
    )
    if context is None:
        context = study.copy()
        context_path_used = STUDY_MASK_TIF
    else:
        context = context & study
        context_path_used = CONTEXT_MASK_TIF

    dynamic, dynamic_info = read_mask(
        DYNAMIC_MASK_TIF,
        anchor,
        required=False,
        description="dynamic mask",
    )
    if dynamic is not None:
        dynamic = dynamic & study

    irs_support, irs_support_info = read_mask(
        IRS_SUPPORT_MASK_TIF,
        anchor,
        required=False,
        description="IRS support mask",
    )
    if irs_support is not None:
        irs_support = irs_support & study

    z_dv, z_dv_info = read_continuous_exact(
        Z_DV_TIF,
        anchor,
        "z(DV+) raster",
    )
    z_twi, z_twi_info = read_continuous_exact(
        Z_TWI_TIF,
        anchor,
        "z(TWI) raster",
    )
    irs, irs_info = read_continuous_exact(
        IRS_TIF,
        anchor,
        "corrected IRS raster",
    )

    invalid_irs = (
        np.isfinite(irs)
        & (
            (irs < -1.0e-6)
            | (irs > 1.0 + 1.0e-6)
        )
    )
    if invalid_irs.any():
        raise ValueError(
            f"IRS contains {int(invalid_irs.sum()):,} values outside [0,1]."
        )

    irs = np.where(
        np.isfinite(irs),
        np.clip(irs, 0.0, 1.0),
        np.nan,
    )

    old_z_sm = None
    old_z_sm_info = None
    if OLD_Z_SM_TIF.exists():
        old_z_sm, old_z_sm_info = read_continuous_exact(
            OLD_Z_SM_TIF,
            anchor,
            "old interpolated z(SM) diagnostic raster",
        )

    # -------------------------------------------------------------------------
    # Prepare support-preserved regional SM
    # -------------------------------------------------------------------------
    (
        sm_support_physical,
        z_sm_support,
        support_ids,
        support_metadata,
        support_cell_table,
        sm_physical_info,
    ) = prepare_support_preserved_sm(
        anchor=anchor,
        study_domain=context,
    )

    support_cell_table.to_csv(
        SUPPORT_CELL_TABLE_CSV,
        index=False,
    )

    with open(
        SM_SUPPORT_MODE_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            support_metadata,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    # -------------------------------------------------------------------------
    # Compute primary multiscale HKFI
    # -------------------------------------------------------------------------
    result = compute_multiscale_hkfi(
        z_dv=z_dv,
        z_twi=z_twi,
        z_sm_support=z_sm_support,
        irs=irs,
        context_domain=context,
        weights=primary_weights,
        label="primary_HKFI",
    )

    compute_domain = result["compute_domain"]
    dv_available = result["dv_available_mask"]
    missing_zdv = result["missing_zdv_mask"]
    unresolved = result[
        "missing_zdv_nonzero_irs_mask"
    ]
    irs_original = result["irs_original"]
    irs_effective = result["irs_for_hkfi"]
    K = result["K"]
    T = result["T"]
    S = result["S"]
    H_local = result["H_local"]
    hkfi_raw = result["hkfi_raw"]
    equal_component_reference = result[
        "equal_component_reference"
    ]

    if not np.any(compute_domain):
        raise RuntimeError(
            "No valid pixels remain for multiscale HKFI calculation."
        )

    if equal_primary:
        equality_difference = (
            hkfi_raw[compute_domain]
            - equal_component_reference[compute_domain]
        )
        equality_rmse = float(
            np.sqrt(
                np.mean(equality_difference ** 2)
            )
        )
        equality_max_abs = float(
            np.max(np.abs(equality_difference))
        )

        if equality_max_abs > 1.0e-10:
            raise RuntimeError(
                "Multiscale lambda formulation does not match the equal-"
                "component equation at lambda=1/3."
            )
    else:
        equality_rmse = None
        equality_max_abs = None

    hkfi_norm, norm_info = robust_minmax_normalize(
        hkfi_raw,
        stats_mask=compute_domain,
        apply_mask=compute_domain,
        p_low=HKFI_NORM_P_LOW,
        p_high=HKFI_NORM_P_HIGH,
    )

    # -------------------------------------------------------------------------
    # Optional comparison with old interpolated-SM formulation
    # -------------------------------------------------------------------------
    old_hkfi_raw = None
    old_hkfi_difference = None
    old_comparison = None

    if old_z_sm is not None:
        old_result = compute_multiscale_hkfi(
            z_dv=z_dv,
            z_twi=z_twi,
            z_sm_support=old_z_sm,
            irs=irs,
            context_domain=context,
            weights=primary_weights,
            label="old_interpolated_SM_primary_weights",
        )
        old_hkfi_raw = old_result["hkfi_raw"]

        old_common = (
            compute_domain
            & old_result["compute_domain"]
            & np.isfinite(old_hkfi_raw)
            & np.isfinite(hkfi_raw)
        )

        old_hkfi_difference = np.full_like(
            hkfi_raw,
            np.nan,
            dtype="float64",
        )
        old_hkfi_difference[old_common] = (
            hkfi_raw[old_common]
            - old_hkfi_raw[old_common]
        )

        old_comparison = {
            "n_common": int(old_common.sum()),
            "pearson_r": safe_pearson(
                hkfi_raw,
                old_hkfi_raw,
                old_common,
            ),
            "rmse": float(
                np.sqrt(
                    np.mean(
                        old_hkfi_difference[old_common] ** 2
                    )
                )
            ),
            "mae": float(
                np.mean(
                    np.abs(
                        old_hkfi_difference[old_common]
                    )
                )
            ),
        }

    # -------------------------------------------------------------------------
    # Independent K/T/S weight sensitivity for Reviewer 1, Comment 7
    # -------------------------------------------------------------------------
    sensitivity_rows: list[dict[str, Any]] = []
    manifest_rows: list[dict[str, Any]] = []

    for scenario_name, scenario_weights in (
        validated_named_scenarios.items()
    ):
        scenario_raw = combine_hkfi_components(
            K=K,
            T=T,
            S=S,
            compute_domain=compute_domain,
            weights=scenario_weights,
            label=f"named_scenario_{scenario_name}",
        )

        scenario_norm, scenario_norm_info = robust_minmax_normalize(
            scenario_raw,
            stats_mask=compute_domain,
            apply_mask=compute_domain,
            p_low=HKFI_NORM_P_LOW,
            p_high=HKFI_NORM_P_HIGH,
        )

        common = (
            compute_domain
            & np.isfinite(hkfi_raw)
            & np.isfinite(scenario_raw)
            & np.isfinite(hkfi_norm)
            & np.isfinite(scenario_norm)
        )

        metrics = scenario_comparison_metrics(
            primary_raw=hkfi_raw,
            scenario_raw=scenario_raw,
            primary_norm=hkfi_norm,
            scenario_norm=scenario_norm,
            common=common,
        )

        raw_path: Path
        norm_path: Path

        if scenario_name == "primary_equal_component":
            raw_path = HKFI_MS_RAW_TIF
            norm_path = HKFI_MS_NORM_TIF
        else:
            raw_path = (
                NAMED_WEIGHT_RASTER_DIR
                / (
                    f"hkfi_{scenario_name}_raw_"
                    "32645_12p5m_float32.tif"
                )
            )
            norm_path = (
                NAMED_WEIGHT_RASTER_DIR
                / (
                    f"hkfi_{scenario_name}_norm_0_1_"
                    "32645_12p5m_float32.tif"
                )
            )

            if WRITE_NAMED_WEIGHT_SCENARIO_RASTERS:
                raster_tags = {
                    "scenario": scenario_name,
                    "weight_K": scenario_weights["K"],
                    "weight_T": scenario_weights["T"],
                    "weight_S": scenario_weights["S"],
                    "primary_reference": False,
                }
                write_float_tif(
                    raw_path,
                    np.where(
                        compute_domain,
                        scenario_raw,
                        np.nan,
                    ),
                    anchor,
                    f"HKFI_{scenario_name}_raw",
                    extra_tags=raster_tags,
                )
                write_float_tif(
                    norm_path,
                    np.where(
                        compute_domain,
                        scenario_norm,
                        np.nan,
                    ),
                    anchor,
                    f"HKFI_{scenario_name}_norm_0_1",
                    extra_tags=raster_tags,
                )

        row = {
            "scenario": scenario_name,
            "scenario_purpose": (
                NAMED_WEIGHT_SCENARIO_PURPOSES.get(
                    scenario_name,
                    "Predefined sensitivity scenario",
                )
            ),
            "weight_K": scenario_weights["K"],
            "weight_T": scenario_weights["T"],
            "weight_S": scenario_weights["S"],
            "is_primary": (
                scenario_name
                == "primary_equal_component"
            ),
            **metrics,
            "normalization_lower": (
                scenario_norm_info["lower_bound"]
            ),
            "normalization_upper": (
                scenario_norm_info["upper_bound"]
            ),
            "normalization_lower_clipped_fraction": (
                scenario_norm_info[
                    "lower_clipped_fraction"
                ]
            ),
            "normalization_upper_clipped_fraction": (
                scenario_norm_info[
                    "upper_clipped_fraction"
                ]
            ),
            "raw_raster": str(raw_path),
            "norm_raster": str(norm_path),
            "raster_written": bool(
                scenario_name == "primary_equal_component"
                or WRITE_NAMED_WEIGHT_SCENARIO_RASTERS
            ),
        }
        sensitivity_rows.append(row)

        manifest_rows.append(
            {
                "scenario": scenario_name,
                "scenario_purpose": (
                    NAMED_WEIGHT_SCENARIO_PURPOSES.get(
                        scenario_name,
                        "Predefined sensitivity scenario",
                    )
                ),
                "weight_K": scenario_weights["K"],
                "weight_T": scenario_weights["T"],
                "weight_S": scenario_weights["S"],
                "raw_raster": str(raw_path),
                "norm_raster": str(norm_path),
                "is_primary": (
                    scenario_name
                    == "primary_equal_component"
                ),
                "use_for_downstream_spatial_CV": True,
            }
        )

    named_sensitivity_df = pd.DataFrame(
        sensitivity_rows
    )
    named_sensitivity_df.to_csv(
        NAMED_WEIGHT_SENSITIVITY_CSV,
        index=False,
    )
    # Compatibility copy.
    named_sensitivity_df.to_csv(
        WEIGHT_SENSITIVITY_CSV,
        index=False,
    )

    pd.DataFrame(
        manifest_rows
    ).to_csv(
        WEIGHT_SCENARIO_MANIFEST_CSV,
        index=False,
    )

    # -------------------------------------------------------------------------
    # Systematic simplex sensitivity
    # -------------------------------------------------------------------------
    simplex_rows: list[dict[str, Any]] = []

    if RUN_SIMPLEX_WEIGHT_SENSITIVITY:
        simplex_scenarios = (
            generate_simplex_weight_scenarios(
                SIMPLEX_WEIGHT_INCREMENT
            )
        )

        for scenario_name, scenario_weights in simplex_scenarios:
            scenario_raw = combine_hkfi_components(
                K=K,
                T=T,
                S=S,
                compute_domain=compute_domain,
                weights=scenario_weights,
                label=scenario_name,
            )

            scenario_norm, scenario_norm_info = robust_minmax_normalize(
                scenario_raw,
                stats_mask=compute_domain,
                apply_mask=compute_domain,
                p_low=HKFI_NORM_P_LOW,
                p_high=HKFI_NORM_P_HIGH,
            )

            common = (
                compute_domain
                & np.isfinite(hkfi_raw)
                & np.isfinite(scenario_raw)
                & np.isfinite(hkfi_norm)
                & np.isfinite(scenario_norm)
            )

            metrics = scenario_comparison_metrics(
                primary_raw=hkfi_raw,
                scenario_raw=scenario_raw,
                primary_norm=hkfi_norm,
                scenario_norm=scenario_norm,
                common=common,
            )

            raw_path = (
                SIMPLEX_WEIGHT_RASTER_DIR
                / (
                    f"hkfi_{scenario_name}_raw_"
                    "32645_12p5m_float32.tif"
                )
            )
            norm_path = (
                SIMPLEX_WEIGHT_RASTER_DIR
                / (
                    f"hkfi_{scenario_name}_norm_0_1_"
                    "32645_12p5m_float32.tif"
                )
            )

            if WRITE_SIMPLEX_WEIGHT_RASTERS:
                raster_tags = {
                    "scenario": scenario_name,
                    "weight_K": scenario_weights["K"],
                    "weight_T": scenario_weights["T"],
                    "weight_S": scenario_weights["S"],
                    "simplex_increment": (
                        SIMPLEX_WEIGHT_INCREMENT
                    ),
                }
                write_float_tif(
                    raw_path,
                    scenario_raw,
                    anchor,
                    f"HKFI_{scenario_name}_raw",
                    extra_tags=raster_tags,
                )
                write_float_tif(
                    norm_path,
                    scenario_norm,
                    anchor,
                    f"HKFI_{scenario_name}_norm_0_1",
                    extra_tags=raster_tags,
                )

            simplex_rows.append(
                {
                    "scenario": scenario_name,
                    "weight_K": scenario_weights["K"],
                    "weight_T": scenario_weights["T"],
                    "weight_S": scenario_weights["S"],
                    **metrics,
                    "normalization_lower": (
                        scenario_norm_info["lower_bound"]
                    ),
                    "normalization_upper": (
                        scenario_norm_info["upper_bound"]
                    ),
                    "raw_raster": (
                        str(raw_path)
                        if WRITE_SIMPLEX_WEIGHT_RASTERS
                        else None
                    ),
                    "norm_raster": (
                        str(norm_path)
                        if WRITE_SIMPLEX_WEIGHT_RASTERS
                        else None
                    ),
                }
            )

    simplex_sensitivity_df = pd.DataFrame(
        simplex_rows
    )
    simplex_sensitivity_df.to_csv(
        SIMPLEX_WEIGHT_SENSITIVITY_CSV,
        index=False,
    )

    # -------------------------------------------------------------------------
    # QA statistics
    # -------------------------------------------------------------------------
    stats = {
        "z_dv_compute_domain": detailed_stats(
            z_dv,
            compute_domain & np.isfinite(z_dv),
        ),
        "z_twi_compute_domain": detailed_stats(
            z_twi,
            compute_domain,
        ),
        "z_sm_support_compute_domain": detailed_stats(
            z_sm_support,
            compute_domain,
        ),
        "irs_original_compute_domain": detailed_stats(
            irs_original,
            compute_domain,
        ),
        "irs_effective_for_hkfi": detailed_stats(
            irs_effective,
            compute_domain,
        ),
        "K_kinematic_term": detailed_stats(
            K,
            compute_domain,
        ),
        "T_twi_term": detailed_stats(
            T,
            compute_domain,
        ),
        "S_support_preserved_sm_term": detailed_stats(
            S,
            compute_domain,
        ),
        "H_local": detailed_stats(
            H_local,
            compute_domain,
        ),
        "hkfi_multiscale_raw": detailed_stats(
            hkfi_raw,
            compute_domain,
        ),
        "hkfi_multiscale_norm": detailed_stats(
            hkfi_norm,
            compute_domain,
        ),
    }

    correlations = {
        "K_vs_T": safe_pearson(
            K,
            T,
            compute_domain,
        ),
        "K_vs_S": safe_pearson(
            K,
            S,
            compute_domain,
        ),
        "T_vs_S": safe_pearson(
            T,
            S,
            compute_domain,
        ),
        "K_vs_HKFI": safe_pearson(
            K,
            hkfi_raw,
            compute_domain,
        ),
        "T_vs_HKFI": safe_pearson(
            T,
            hkfi_raw,
            compute_domain,
        ),
        "S_vs_HKFI": safe_pearson(
            S,
            hkfi_raw,
            compute_domain,
        ),
    }

    warnings: list[str] = []

    study_n = int(study.sum())
    context_n = int(context.sum())
    compute_n = int(compute_domain.sum())
    compute_fraction = compute_n / max(context_n, 1)

    if compute_fraction < WARN_VALID_DOMAIN_FRACTION:
        warnings.append(
            f"Only {100.0 * compute_fraction:.2f}% of the context domain "
            "has computable multiscale HKFI."
        )

    for label in (
        "z_dv_compute_domain",
        "z_twi_compute_domain",
        "z_sm_support_compute_domain",
    ):
        component = stats[label]
        mean_value = component["mean"]
        std_value = component["std"]

        if (
            mean_value is not None
            and abs(float(mean_value)) > WARN_INPUT_Z_MEAN_ABS
        ):
            warnings.append(
                f"{label} mean is {float(mean_value):.3f} on the fine-pixel "
                "HKFI domain. For z(SM), note that standardization was correctly "
                "performed across unique support cells, so the repeated fine-"
                "pixel mean need not be exactly zero."
            )

        if (
            label != "z_sm_support_compute_domain"
            and std_value is not None
            and (
                float(std_value) < WARN_INPUT_Z_STD_LOW
                or float(std_value) > WARN_INPUT_Z_STD_HIGH
            )
        ):
            warnings.append(
                f"{label} standard deviation is {float(std_value):.3f}."
            )

    k_zero_fraction = stats[
        "K_kinematic_term"
    ]["zero_fraction"]

    if (
        k_zero_fraction is not None
        and float(k_zero_fraction) > WARN_K_ZERO_FRACTION
    ):
        warnings.append(
            f"K is exactly zero over "
            f"{100.0 * float(k_zero_fraction):.2f}% of the compute domain. "
            "Report full-domain and InSAR-supported-domain contributions "
            "separately in Step 5.3b."
        )

    irs_near_zero = stats[
        "irs_original_compute_domain"
    ]["near_zero_fraction"]

    if (
        irs_near_zero is not None
        and float(irs_near_zero)
        > WARN_IRS_NEAR_ZERO_FRACTION
    ):
        warnings.append(
            f"{100.0 * float(irs_near_zero):.2f}% of IRS values are near zero."
        )

    hkfi_unique = stats[
        "hkfi_multiscale_norm"
    ]["unique_values_rounded_6dp"]

    if int(hkfi_unique) < WARN_HKFI_UNIQUE_VALUES_MIN:
        warnings.append(
            f"Normalized HKFI has only {int(hkfi_unique):,} unique values "
            "at six decimal places."
        )

    if (
        WARN_APPROXIMATE_SUPPORT
        and not support_metadata.get(
            "is_exact_native_era5_grid",
            False,
        )
    ):
        warnings.append(
            "The primary HKFI used an approximate 9 km support grid because "
            "the exact native ERA5 geometry was unavailable. This is a "
            "support-scale correction, not exact native-cell reconstruction."
        )

    # Check within-cell constancy of S
    valid_support = (
        compute_domain
        & np.isfinite(S)
        & np.isfinite(support_ids)
    )
    support_integer = np.where(
        np.isfinite(support_ids),
        support_ids,
        -1,
    ).astype("int64")

    within_cell_std_values: list[float] = []
    for support_id in np.unique(
        support_integer[valid_support]
    ):
        values = S[
            valid_support
            & (support_integer == support_id)
        ]
        if values.size:
            within_cell_std_values.append(
                float(np.std(values))
            )

    maximum_within_cell_s_std = (
        max(within_cell_std_values)
        if within_cell_std_values
        else None
    )

    if (
        maximum_within_cell_s_std is not None
        and maximum_within_cell_s_std > 1.0e-10
    ):
        raise RuntimeError(
            "Support-preserved z(SM) is not constant within support cells."
        )

    # -------------------------------------------------------------------------
    # Write outputs
    # -------------------------------------------------------------------------
    write_float_tif(
        IRS_EFFECTIVE_HKFI_TIF,
        np.where(
            compute_domain,
            irs_effective,
            np.nan,
        ),
        anchor,
        "IRS_effective_for_HKFI",
    )
    write_float_tif(
        KINEMATIC_TERM_TIF,
        np.where(compute_domain, K, np.nan),
        anchor,
        "K_IRS_times_zDVplus",
    )
    write_float_tif(
        TWI_TERM_TIF,
        np.where(compute_domain, T, np.nan),
        anchor,
        "T_zTWI",
    )
    write_float_tif(
        SM_SUPPORT_PHYSICAL_TIF,
        np.where(
            compute_domain,
            sm_support_physical,
            np.nan,
        ),
        anchor,
        "SM_support_preserved_physical",
    )
    write_float_tif(
        SM_SUPPORT_Z_TIF,
        np.where(compute_domain, S, np.nan),
        anchor,
        "S_zSM_support_preserved",
    )
    write_int32_tif(
        SM_SUPPORT_CELL_ID_TIF,
        np.where(
            compute_domain,
            support_ids,
            np.nan,
        ),
        anchor,
        "SM_support_cell_identifier",
    )
    write_float_tif(
        LOCAL_HYDRO_KINEMATIC_TIF,
        np.where(
            compute_domain,
            H_local,
            np.nan,
        ),
        anchor,
        "H_local_K_plus_T_over_2",
    )

    # Primary compatibility outputs
    write_float_tif(
        HKFI_RAW_TIF,
        np.where(
            compute_domain,
            hkfi_raw,
            np.nan,
        ),
        anchor,
        "HKFI_multiscale_raw",
    )
    write_float_tif(
        HKFI_NORM_TIF,
        np.where(
            compute_domain,
            hkfi_norm,
            np.nan,
        ),
        anchor,
        "HKFI_multiscale_norm_0_1",
    )
    write_float_tif(
        HKFI_COMPAT_TIF,
        np.where(
            compute_domain,
            hkfi_norm,
            np.nan,
        ),
        anchor,
        "HKFI_multiscale_norm_compatibility_copy",
    )

    # Explicit aliases
    write_float_tif(
        HKFI_MS_RAW_TIF,
        np.where(
            compute_domain,
            hkfi_raw,
            np.nan,
        ),
        anchor,
        "HKFI_multiscale_raw_alias",
    )
    write_float_tif(
        HKFI_MS_NORM_TIF,
        np.where(
            compute_domain,
            hkfi_norm,
            np.nan,
        ),
        anchor,
        "HKFI_multiscale_norm_alias",
    )

    if old_hkfi_difference is not None:
        write_float_tif(
            HKFI_OLD_SM_DIFFERENCE_TIF,
            old_hkfi_difference,
            anchor,
            "HKFI_multiscale_minus_old_interpolated_SM_raw",
        )

    write_byte_tif(
        HKFI_VALID_MASK_TIF,
        compute_domain,
        anchor,
        study,
        "HKFI_valid_mask",
    )
    write_byte_tif(
        HKFI_DV_AVAILABILITY_MASK_TIF,
        dv_available,
        anchor,
        study,
        "HKFI_DV_availability_mask",
    )
    write_byte_tif(
        HKFI_MISSING_ZDV_MASK_TIF,
        missing_zdv,
        anchor,
        study,
        "HKFI_missing_zDV_mask",
    )
    write_byte_tif(
        HKFI_UNRESOLVED_MASK_TIF,
        unresolved,
        anchor,
        study,
        "HKFI_missing_zDV_nonzero_original_IRS_mask",
    )

    # -------------------------------------------------------------------------
    # Metadata and QA
    # -------------------------------------------------------------------------
    zscore_metadata = load_json_optional(
        ZSCORE_META_JSON
    )
    irs_metadata = load_json_optional(
        IRS_META_JSON
    )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "step": "5.3",
        "formulation": "multiscale_HKFI",
        "equation": {
            "K": "K = IRS_HKFI * z(DV+)",
            "T": "T = z(TWI)",
            "S": (
                "S_j = z_support(SM_j), standardized across unique "
                "coarse support cells"
            ),
            "general_HKFI": (
                "HKFI_raw = w_K*K + w_T*T + w_S*S, "
                "with non-negative weights summing to one"
            ),
            "primary_weights": primary_weights,
            "primary_equation": (
                "HKFI_raw = (K + T + S) / 3"
            ),
            "H_local_diagnostic": (
                "H_local = (K + T) / 2; retained as a local-component "
                "diagnostic and for backward-compatible output"
            ),
            # Compatibility field for the earlier lambda representation.
            "lambda_SM": primary_weights["S"],
        },
        "scientific_interpretation": {
            "multiscale_structure": (
                "Only K and T can vary within a support cell, whereas "
                "S represents regional soil-moisture loading and is "
                "constant within each coarse support cell. Their observed "
                "effective contributions are quantified separately in "
                "Step 5.3b."
            ),
            "SM_standardization": (
                "SM was winsorized and z-standardized across unique "
                "support-cell values before assignment to the 12.5 m grid."
            ),
            "IRS_role": (
                "IRS attenuates uncertain signed kinematic evidence "
                "toward neutral zero and is not used as an independent "
                "predictor."
            ),
            "missing_zDV_rule": (
                "Where z(DV+) is unavailable, I_DV=0 and "
                "IRS_HKFI=IRS*I_DV=0. This is neutral missing evidence, "
                "not observed stability."
            ),
            "normalization": (
                "The final 0–1 HKFI is a study-relative robust P2-P98 "
                "mapping and is not a failure probability."
            ),
            "weighting_rationale": (
                "Equal one-third nominal weights are an a priori transparent "
                "reference, not a claim of equal physical influence. Named "
                "independent K/T/S scenarios, component ablations and a "
                "systematic simplex analysis are generated for robustness "
                "testing. Susceptibility-output robustness must be evaluated "
                "in the downstream spatial-CV pipeline."
            ),
        },
        "support_metadata": support_metadata,
        "inputs": {
            "anchor": str(ANCHOR_TIF),
            "study_mask": str(STUDY_MASK_TIF),
            "context_mask_used": str(context_path_used),
            "dynamic_mask_diagnostic": (
                str(DYNAMIC_MASK_TIF)
                if dynamic is not None
                else None
            ),
            "z_dv": z_dv_info,
            "z_twi": z_twi_info,
            "irs": irs_info,
            "old_z_sm_diagnostic": old_z_sm_info,
            "physical_sm": sm_physical_info,
            "monthly_sm_csv": str(SM_MONSOON_MONTHLY_CSV),
            "irs_support_mask": irs_support_info,
            "zscore_metadata": (
                str(ZSCORE_META_JSON)
                if zscore_metadata is not None
                else None
            ),
            "irs_metadata": (
                str(IRS_META_JSON)
                if irs_metadata is not None
                else None
            ),
        },
        "domain": {
            "n_study": study_n,
            "n_context": context_n,
            "n_compute": compute_n,
            "compute_fraction_of_context": compute_fraction,
            "n_dv_available": int(dv_available.sum()),
            "n_missing_zdv": int(missing_zdv.sum()),
            "n_missing_zdv_nonzero_irs": int(
                unresolved.sum()
            ),
            "dynamic_mask_used_to_define_domain": False,
        },
        "normalization": norm_info,
        "equal_component_equivalence_QA": {
            "applicable": bool(equal_primary),
            "primary_weights": primary_weights,
            "rmse": equality_rmse,
            "maximum_absolute_difference": (
                equality_max_abs
            ),
        },
        "weight_sensitivity_design": {
            "named_scenarios": validated_named_scenarios,
            "named_scenario_purposes": (
                NAMED_WEIGHT_SCENARIO_PURPOSES
            ),
            "simplex_enabled": RUN_SIMPLEX_WEIGHT_SENSITIVITY,
            "simplex_increment": SIMPLEX_WEIGHT_INCREMENT,
            "simplex_scenario_count": int(
                len(simplex_sensitivity_df)
            ),
            "named_scenario_rasters_written": (
                WRITE_NAMED_WEIGHT_SCENARIO_RASTERS
            ),
            "simplex_rasters_written": (
                WRITE_SIMPLEX_WEIGHT_RASTERS
            ),
            "downstream_requirement": (
                "Propagate named scenario normalized rasters through "
                "identical spatial-CV susceptibility models."
            ),
        },
        "support_constancy_QA": {
            "maximum_within_support_cell_std_of_zSM": (
                maximum_within_cell_s_std
            ),
            "expected": 0.0,
        },
        "old_interpolated_SM_comparison": old_comparison,
        "statistics": stats,
        "correlations": correlations,
        "warnings": warnings,
        "outputs": {
            "hkfi_raw_primary": str(HKFI_RAW_TIF),
            "hkfi_norm_primary": str(HKFI_NORM_TIF),
            "hkfi_norm_compatibility": str(HKFI_COMPAT_TIF),
            "hkfi_multiscale_raw_alias": str(
                HKFI_MS_RAW_TIF
            ),
            "hkfi_multiscale_norm_alias": str(
                HKFI_MS_NORM_TIF
            ),
            "K": str(KINEMATIC_TERM_TIF),
            "T": str(TWI_TERM_TIF),
            "S_support_preserved": str(SM_SUPPORT_Z_TIF),
            "SM_support_physical": str(
                SM_SUPPORT_PHYSICAL_TIF
            ),
            "SM_support_cell_id": str(
                SM_SUPPORT_CELL_ID_TIF
            ),
            "H_local": str(LOCAL_HYDRO_KINEMATIC_TIF),
            "old_SM_difference": (
                str(HKFI_OLD_SM_DIFFERENCE_TIF)
                if old_hkfi_difference is not None
                else None
            ),
            "named_weight_sensitivity": str(
                NAMED_WEIGHT_SENSITIVITY_CSV
            ),
            "simplex_weight_sensitivity": str(
                SIMPLEX_WEIGHT_SENSITIVITY_CSV
            ),
            "weight_scenario_manifest": str(
                WEIGHT_SCENARIO_MANIFEST_CSV
            ),
            "weight_sensitivity_compatibility": str(
                WEIGHT_SENSITIVITY_CSV
            ),
            "support_cell_table": str(
                SUPPORT_CELL_TABLE_CSV
            ),
            "metadata": str(META_JSON),
            "quality_assurance": str(QA_JSON),
            "statistics_csv": str(QA_CSV),
        },
    }

    quality_assurance = {
        "generated_at_utc": utc_now_iso(),
        "status": (
            "PASS"
            if not warnings
            else "PASS_WITH_WARNINGS"
        ),
        "warnings": warnings,
        "support_mode": support_metadata.get(
            "support_mode"
        ),
        "is_exact_native_era5_grid": (
            support_metadata.get(
                "is_exact_native_era5_grid"
            )
        ),
        "support_constancy": metadata[
            "support_constancy_QA"
        ],
        "equal_component_equivalence": metadata[
            "equal_component_equivalence_QA"
        ],
        "coverage": metadata["domain"],
        "normalization": norm_info,
        "statistics": stats,
        "correlations": correlations,
        "old_interpolated_SM_comparison": (
            old_comparison
        ),
        "primary_weights": primary_weights,
        "named_weight_scenario_count": int(
            len(named_sensitivity_df)
        ),
        "simplex_weight_scenario_count": int(
            len(simplex_sensitivity_df)
        ),
        "weight_sensitivity_outputs": {
            "named": str(
                NAMED_WEIGHT_SENSITIVITY_CSV
            ),
            "simplex": str(
                SIMPLEX_WEIGHT_SENSITIVITY_CSV
            ),
            "manifest": str(
                WEIGHT_SCENARIO_MANIFEST_CSV
            ),
        },
    }

    with open(
        META_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            metadata,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    with open(
        QA_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            quality_assurance,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    rows: list[tuple[str, str, Any]] = []
    for component_name, component_stats in stats.items():
        for metric_name, metric_value in component_stats.items():
            rows.append(
                (
                    component_name,
                    metric_name,
                    metric_value,
                )
            )

    with open(
        QA_CSV,
        "w",
        newline="",
        encoding="utf-8",
    ) as handle:
        writer = csv.writer(handle)
        writer.writerow(
            ["component", "metric", "value"]
        )
        writer.writerows(rows)

    # -------------------------------------------------------------------------
    # Console summary
    # -------------------------------------------------------------------------
    log(
        f"[SUPPORT] mode="
        f"{support_metadata.get('support_mode')} | "
        f"exact_native="
        f"{support_metadata.get('is_exact_native_era5_grid')}"
    )
    log(
        f"[DOMAIN] compute={compute_n:,}/{context_n:,} "
        f"({100.0 * compute_fraction:.2f}%)"
    )
    log(
        f"[K] zero fraction="
        f"{100.0 * float(stats['K_kinematic_term']['zero_fraction']):.2f}%"
    )
    log(
        f"[HKFI_RAW] mean="
        f"{stats['hkfi_multiscale_raw']['mean']:.4f} | "
        f"std={stats['hkfi_multiscale_raw']['std']:.4f}"
    )
    log(
        f"[HKFI_NORM] mean="
        f"{stats['hkfi_multiscale_norm']['mean']:.4f} | "
        f"median={stats['hkfi_multiscale_norm']['median']:.4f}"
    )
    log(
        f"[SM SUPPORT QA] maximum within-cell z(SM) std="
        f"{maximum_within_cell_s_std:.3e}"
    )

    if warnings:
        log("[QA] PASS WITH WARNINGS")
        for warning in warnings:
            log(f"  - {warning}")
    else:
        log("[QA] PASS")

    log(f"[OUT] {HKFI_NORM_TIF}")
    log(
        f"[WEIGHT SENSITIVITY] named={len(named_sensitivity_df)} | "
        f"simplex={len(simplex_sensitivity_df)}"
    )
    log(f"[OUT] {NAMED_WEIGHT_SENSITIVITY_CSV}")
    log(f"[OUT] {SIMPLEX_WEIGHT_SENSITIVITY_CSV}")
    log(f"[OUT] {WEIGHT_SCENARIO_MANIFEST_CSV}")
    log(f"[META] {META_JSON}")
    log(f"[QA] {QA_JSON}")

    make_qa_figures(
        hkfi_norm=hkfi_norm,
        compute_domain=compute_domain,
        study_mask=study,
        anchor=anchor,
    )

    log("=== Step 5.3 completed successfully ===")


def run_step_5_3() -> None:
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 5.3 failed: {exc}")
        traceback.print_exc()
        raise


if __name__ == "__main__":
    run_step_5_3()


# Step 5.3 — Revised Multiscale HKFI Visualization

In [ ]:
# =========================================================
# Step 5.3 — Multiscale HKFI Visualization
# =========================================================
#
# Purpose
# -------
# Creates the final map and histogram for the revised multiscale
# Hydro–Kinematic Fusion Index (HKFI).
#
# Scientific safeguards
# ---------------------
# 1. Reads the explicit multiscale HKFI output first, with a compatibility
#    fallback only when the explicit file is unavailable.
# 2. Displays the authoritative raster directly.
# 3. Uses no RBF, IDW, cubic interpolation, smoothing, or surface
#    reconstruction.
# 4. Uses interpolation="nearest" only for raster display.
# 5. Requires the analytical HKFI valid mask to match the HKFI raster grid
#    exactly.
# 6. Reads and records the Step 5.3 soil-moisture support mode.
# 7. Does not renormalize HKFI for visualization.
# 8. Uses the same valid cells for the map, histogram, interval counts, and
#    descriptive statistics.
#
# Cartographic style
# ------------------
# Replicates the final Step 3.7 DV+ / Step 3.1 coherence map style:
# - 2.5 × 5 inch map; 2.5 × 2.5 inch histogram
# - viridis colour map
# - 2% buffered display extent
# - four longitude DMS labels, rotated 45°
# - three latitude DMS labels, rotated 90°
# - bottom-most latitude label anchored above its tick (va="bottom")
# - no "Latitude" / "Longitude" axis titles
# - north arrow
# - checkerboard scale bar
# - highway and study-area overlays
# - compact legend
# - narrow side colour bar
# - PNG, PDF, and SVG outputs at 900 dpi
# =========================================================

from __future__ import annotations

import csv
import json
import os
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_geom

import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator, LinearLocator
from matplotlib.patches import Polygon, Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False


# =========================================================
# PATHS
# =========================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
CONTEXT_MASK_TIF = (
    BASE / "Results" / "Step-0.3-Quality_Masks" / "context_mask.tif"
)

STUDY_SHP = BASE / "Study_Area.shp"
ROAD_SHP = BASE / "NTD_Road.shp"

HKFI_DIR = BASE / "Results" / "Step-5.3-HKFI"

# Prefer the explicit revised multiscale raster.
# The second path is retained only for downstream compatibility.
HKFI_NORM_CANDIDATES = [
    HKFI_DIR / "hkfi_multiscale_norm_0_1_32645_12p5m_float32.tif",
    HKFI_DIR / "hkfi_norm_0_1_dense_32645_12p5m_float32.tif",
    HKFI_DIR / "hkfi_norm_0_1_32645_12p5m_float32.tif",
]

HKFI_VALID_MASK_TIF = (
    HKFI_DIR / "hkfi_valid_mask_32645_12p5m_byte.tif"
)
HKFI_MISSING_ZDV_MASK_TIF = (
    HKFI_DIR / "hkfi_missing_zdv_mask_32645_12p5m_byte.tif"
)
HKFI_MISMATCH_MASK_TIF = (
    HKFI_DIR
    / "hkfi_missing_zdv_nonzero_original_irs_mask_32645_12p5m_byte.tif"
)

HKFI_META_JSON = HKFI_DIR / "hkfi_metadata.json"
HKFI_QA_JSON = HKFI_DIR / "hkfi_quality_assurance.json"
SM_SUPPORT_MODE_JSON = HKFI_DIR / "sm_support_mode.json"

# Revised output names prevent overwriting figures from the former formulation.
OUT_MAP_PNG = HKFI_DIR / "Multiscale_HKFI_Map.png"
OUT_MAP_PDF = HKFI_DIR / "Multiscale_HKFI_Map.pdf"
OUT_MAP_SVG = HKFI_DIR / "Multiscale_HKFI_Map.svg"

OUT_HIST_PNG = HKFI_DIR / "Multiscale_HKFI_Histogram.png"
OUT_HIST_PDF = HKFI_DIR / "Multiscale_HKFI_Histogram.pdf"
OUT_HIST_SVG = HKFI_DIR / "Multiscale_HKFI_Histogram.svg"

OUT_SUMMARY_JSON = (
    HKFI_DIR / "multiscale_hkfi_visualization_summary.json"
)
OUT_STATS_CSV = (
    HKFI_DIR / "multiscale_hkfi_visualization_statistics.csv"
)

HKFI_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# DISPLAY SETTINGS
# =========================================================
MAP_FIGSIZE = (2.5, 5)
HIST_FIGSIZE = (2.5, 2.5)

CAX_SIZE = "5%"
CAX_PAD = 0.01
FIG_DPI = 900

HKFI_CMAP = plt.get_cmap("viridis")

HKFI_INFO = {
    "title": "HKFI",
    "units": "HKFI (0–1)",
}

# Direct display of the authoritative raster.
RASTER_DISPLAY_INTERPOLATION = "nearest"

DISPLAY_BUFFER_FRACTION = 0.02
LONGITUDE_DMS_TICK_COUNT = 4
LATITUDE_DMS_TICK_COUNT = 3
DMS_TICK_FONTSIZE = 6.2

# Descriptive intervals only; not calibrated probability classes.
LOW_MAX = 0.30
MODERATE_MAX = 0.60

SHOW_FIGURES = True
SAVE_FIGURES = True

# Optional diagnostic overlay; disabled for the manuscript map.
SHOW_MISMATCH_OVERLAY = False
MISMATCH_MARKER_SIZE = 1.2

# Require exact grid identity for analytical Step 5.3 masks.
REQUIRE_EXACT_ANALYTICAL_MASK_GRID = True

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = [
    "Times New Roman",
    "DejaVu Serif",
]
plt.rcParams["axes.unicode_minus"] = False


# =========================================================
# GENERAL HELPERS
# =========================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def choose_existing_raster(
    candidates: list[Path],
    description: str,
) -> Path:
    for path in candidates:
        if path.exists():
            log(f"[PATH] {description}: {path}")
            return path

    attempted = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not locate {description}. Tried:\n{attempted}"
    )


def load_json_optional(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None

    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return None


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


# =========================================================
# RASTER HELPERS
# =========================================================
def same_grid(
    dataset: rasterio.io.DatasetReader,
    reference: dict[str, Any],
) -> bool:
    return (
        dataset.crs == reference["crs"]
        and dataset.transform == reference["transform"]
        and dataset.width == reference["width"]
        and dataset.height == reference["height"]
    )


def clean_float_nodata(
    array: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    output = array.astype("float64", copy=True)

    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if np.isfinite(nodata_value):
                output[output == nodata_value] = np.nan
        except (TypeError, ValueError):
            pass

    output[output == -9999.0] = np.nan
    output[output == -32768.0] = np.nan
    output[np.abs(output) > 1.0e20] = np.nan
    return output


def read_hkfi_raster(
    path: Path,
) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(path, "revised multiscale normalized HKFI raster")

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"HKFI raster has no CRS: {path}")

        raw = dataset.read(1)
        hkfi = clean_float_nodata(raw, dataset.nodata)

        reference = {
            "crs": dataset.crs,
            "transform": dataset.transform,
            "width": dataset.width,
            "height": dataset.height,
            "bounds": dataset.bounds,
            "epsg": dataset.crs.to_epsg(),
            "resolution_x": abs(dataset.transform.a),
            "resolution_y": abs(dataset.transform.e),
            "path": str(path),
        }

    outside_range = (
        np.isfinite(hkfi)
        & ((hkfi < -1.0e-6) | (hkfi > 1.0 + 1.0e-6))
    )

    if outside_range.any():
        raise ValueError(
            f"HKFI contains {int(outside_range.sum()):,} values outside "
            "[0,1]. Recheck Step 5.3 normalization."
        )

    # Floating-point safeguard only; this is not renormalization.
    hkfi = np.where(
        np.isfinite(hkfi),
        np.clip(hkfi, 0.0, 1.0),
        np.nan,
    )

    return hkfi, reference


def read_mask_to_reference(
    path: Path,
    reference: dict[str, Any],
    description: str,
    optional: bool = False,
    require_exact_grid: bool = False,
) -> np.ndarray | None:
    if not path.exists():
        if optional:
            return None
        raise FileNotFoundError(f"Missing {description}: {path}")

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        is_same = same_grid(dataset, reference)

        if require_exact_grid and not is_same:
            raise ValueError(
                f"{description} does not match the revised HKFI raster grid. "
                f"Regenerate the analytical mask rather than resampling it:\n"
                f"{path}"
            )

        raw = dataset.read(1)
        nodata = dataset.nodata

        if is_same:
            values = raw.astype("float64")

            if nodata is not None:
                values[raw == nodata] = 0.0

            values[~np.isfinite(values)] = 0.0
            return values > 0.5

        destination = np.zeros(
            (reference["height"], reference["width"]),
            dtype="float64",
        )

        reproject(
            source=raw,
            destination=destination,
            src_transform=dataset.transform,
            src_crs=dataset.crs,
            src_nodata=nodata,
            dst_transform=reference["transform"],
            dst_crs=reference["crs"],
            dst_nodata=0.0,
            resampling=Resampling.nearest,
        )

        destination[~np.isfinite(destination)] = 0.0
        return destination > 0.5


# =========================================================
# CARTOGRAPHIC HELPERS
# =========================================================
def to_dms(value: float, is_lon: bool = True) -> str:
    absolute = abs(value)
    degrees = int(absolute)

    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_lon
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def add_north_arrow_tri(
    axis,
    size: float = 0.09,
    pad: float = 0.07,
    label: str = "N",
    arrow_downshift: float = 0.025,
    label_upshift: float = 0.0,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (
        (1.0 - pad)
        + size * 0.10
        + label_upshift
    )

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base
        if candidate >= target_metres:
            return candidate

    return 10 * base


def add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    crs_type: str = "wgs84",
    mid_lat_deg: float | None = None,
    segments: int = 4,
    fontsize: float = 5.6,
    frame_lw: float = 0.4,
) -> None:
    width_data = east - west
    height_data = north - south

    if crs_type == "utm":
        width_metres = width_data
        data_units_per_metre = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = 0.5 * (south + north)

        metres_per_degree = metres_per_degree_lon(mid_lat_deg)
        width_metres = width_data * metres_per_degree
        data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = (
        total_length_metres * data_units_per_metre
    )
    segment_length_units = (
        segment_length_metres * data_units_per_metre
    )

    x_start = (
        west
        + 0.5 * width_data
        - 0.5 * total_length_units
    )
    y_base = south + 0.06 * height_data
    bar_height = 0.018 * height_data

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_data

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (
                    x_start + index * segment_length_units,
                    y_base,
                ),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=(
                    "black" if index % 2 == 0 else "white"
                ),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_data,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
        color="black",
        alpha=1.0,
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_data,
        "0",
        ha="center",
        va="top",
        fontsize=5,
        fontweight="bold",
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_data,
        end_label,
        ha="center",
        va="top",
        fontsize=5,
        fontweight="bold",
        zorder=15,
    )


# =========================================================
# ROAD AND BOUNDARY OVERLAYS
# =========================================================
def load_roads(
    road_path: Path,
    target_crs,
):
    if not road_path.exists():
        log(f"[WARN] Road shapefile not found: {road_path}")
        return None

    target_crs_text = str(target_crs)

    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_path)

            if roads.crs is None:
                log(
                    "[WARN] Road CRS is missing; assuming EPSG:4326."
                )
                roads = roads.set_crs(epsg=4326)

            return roads.to_crs(target_crs_text)

        except Exception as exc:
            log(f"[WARN] GeoPandas road loading failed: {exc}")

    if HAVE_FIONA:
        try:
            geometries = []

            with fiona.open(road_path, "r") as source:
                source_crs = source.crs

                for feature in source:
                    geometry = feature["geometry"]
                    if geometry is None:
                        continue

                    geometries.append(
                        transform_geom(
                            source_crs,
                            target_crs_text,
                            geometry,
                        )
                    )

            return {"geometries": geometries}

        except Exception as exc:
            log(f"[WARN] Fiona road loading failed: {exc}")

    return None


def plot_roads(
    axis,
    roads,
    colour: str = "red",
    linewidth: float = 0.7,
    alpha: float = 0.9,
    zorder: int = 850,
) -> None:
    if roads is None:
        return

    try:
        if HAVE_GPD and hasattr(roads, "plot"):
            roads.plot(
                ax=axis,
                color=colour,
                linewidth=linewidth,
                alpha=alpha,
                zorder=zorder,
            )
            return

        if isinstance(roads, dict):
            for geometry in roads["geometries"]:
                if geometry is None:
                    continue

                if geometry["type"] == "LineString":
                    line_groups = [geometry["coordinates"]]
                elif geometry["type"] == "MultiLineString":
                    line_groups = geometry["coordinates"]
                else:
                    continue

                for line in line_groups:
                    axis.plot(
                        [point[0] for point in line],
                        [point[1] for point in line],
                        color=colour,
                        linewidth=linewidth,
                        alpha=alpha,
                        zorder=zorder,
                    )

    except Exception as exc:
        log(f"[WARN] Road plotting failed: {exc}")


def draw_study_boundary(
    axis,
    study_mask: np.ndarray,
    reference: dict[str, Any],
) -> None:
    bounds = reference["bounds"]
    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    axis.contour(
        study_mask.astype("uint8"),
        levels=[0.5],
        colors="black",
        linewidths=0.7,
        extent=extent,
        origin="upper",
        zorder=999,
    )


def mismatch_pixel_centres(
    mask: np.ndarray,
    transform,
) -> tuple[np.ndarray, np.ndarray]:
    rows, columns = np.where(mask)

    x = (
        transform.c
        + (columns + 0.5) * transform.a
        + (rows + 0.5) * transform.b
    )
    y = (
        transform.f
        + (columns + 0.5) * transform.d
        + (rows + 0.5) * transform.e
    )

    return np.asarray(x), np.asarray(y)


# =========================================================
# STATISTICS
# =========================================================
def calculate_statistics(
    values: np.ndarray,
) -> dict[str, float | int]:
    values = values[np.isfinite(values)].astype("float64")

    if values.size == 0:
        return {"n": 0}

    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "one_fraction": float(np.mean(values == 1.0)),
        "unique_values_rounded_6dp": int(
            np.unique(np.round(values, 6)).size
        ),
    }


def calculate_descriptive_intervals(
    values: np.ndarray,
) -> dict[str, dict[str, float | int]]:
    total = int(values.size)

    definitions = {
        "low_0_to_lt_0.3": (
            (values >= 0.0) & (values < LOW_MAX)
        ),
        "moderate_0.3_to_lt_0.6": (
            (values >= LOW_MAX) & (values < MODERATE_MAX)
        ),
        "high_0.6_to_1.0": (
            (values >= MODERATE_MAX) & (values <= 1.0)
        ),
    }

    output: dict[str, dict[str, float | int]] = {}

    for name, condition in definitions.items():
        count = int(condition.sum())

        output[name] = {
            "count": count,
            "percentage": 100.0 * count / max(total, 1),
        }

    return output


# =========================================================
# MAP
# =========================================================
def create_hkfi_map(
    hkfi: np.ndarray,
    valid_mask: np.ndarray,
    study_mask: np.ndarray,
    mismatch_mask: np.ndarray | None,
    reference: dict[str, Any],
) -> None:
    bounds = reference["bounds"]

    raster_extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    buffer_x = (
        bounds.right - bounds.left
    ) * DISPLAY_BUFFER_FRACTION
    buffer_y = (
        bounds.top - bounds.bottom
    ) * DISPLAY_BUFFER_FRACTION

    west = bounds.left - buffer_x
    east = bounds.right + buffer_x
    south = bounds.bottom - buffer_y
    north = bounds.top + buffer_y

    display_array = np.ma.masked_invalid(
        np.where(valid_mask, hkfi, np.nan)
    )

    raster_crs = reference["crs"]
    crs_is_geographic = bool(
        raster_crs and raster_crs.is_geographic
    )

    x_mid = 0.5 * (bounds.left + bounds.right)
    y_mid = 0.5 * (bounds.bottom + bounds.top)

    if crs_is_geographic:
        def format_longitude(value, position):
            return to_dms(value, True)

        def format_latitude(value, position):
            return to_dms(value, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError(
                "Projected HKFI raster requires pyproj for DMS labels."
            )

        to_geographic = Transformer.from_crs(
            raster_crs,
            "EPSG:4326",
            always_xy=True,
        )

        def format_longitude(value, position):
            longitude, _ = to_geographic.transform(
                value,
                y_mid,
            )
            return to_dms(longitude, True)

        def format_latitude(value, position):
            _, latitude = to_geographic.transform(
                x_mid,
                value,
            )
            return to_dms(latitude, False)

    roads = load_roads(ROAD_SHP, raster_crs)

    figure, axis = plt.subplots(
        1,
        1,
        figsize=MAP_FIGSIZE,
        facecolor="white",
    )
    axis.set_facecolor("white")

    image = axis.imshow(
        display_array,
        cmap=HKFI_CMAP,
        extent=raster_extent,
        vmin=0.0,
        vmax=1.0,
        origin="upper",
        interpolation=RASTER_DISPLAY_INTERPOLATION,
        alpha=1.0,
        zorder=1,
    )

    axis.set_xlim(west, east)
    axis.set_ylim(south, north)

    plot_roads(
        axis,
        roads,
        colour="red",
        linewidth=0.7,
        alpha=0.9,
        zorder=850,
    )
    draw_study_boundary(
        axis,
        study_mask,
        reference,
    )

    if (
        SHOW_MISMATCH_OVERLAY
        and mismatch_mask is not None
        and mismatch_mask.any()
    ):
        mismatch_x, mismatch_y = mismatch_pixel_centres(
            mismatch_mask,
            reference["transform"],
        )

        axis.scatter(
            mismatch_x,
            mismatch_y,
            s=MISMATCH_MARKER_SIZE,
            marker="x",
            color="white",
            linewidths=0.3,
            zorder=900,
        )

    # No "Latitude" / "Longitude" axis titles (Step 3.7 style).

    axis.xaxis.set_major_locator(
        LinearLocator(LONGITUDE_DMS_TICK_COUNT)
    )

    # Fixed latitude tick count to match the Step 3.7 map style.
    axis.yaxis.set_major_locator(
        LinearLocator(LATITUDE_DMS_TICK_COUNT)
    )

    axis.xaxis.set_major_formatter(
        FuncFormatter(format_longitude)
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(format_latitude)
    )

    axis.tick_params(
        axis="x",
        which="major",
        labelsize=DMS_TICK_FONTSIZE,
        width=0.8,
        length=2.5,
    )
    axis.tick_params(
        axis="y",
        which="major",
        labelsize=DMS_TICK_FONTSIZE,
        width=0.8,
        length=2.5,
    )

    for tick_label in axis.get_xticklabels():
        tick_label.set_rotation(45)
        tick_label.set_ha("right")
        tick_label.set_fontsize(DMS_TICK_FONTSIZE)
        tick_label.set_fontweight("bold")

    y_tick_labels = axis.get_yticklabels()
    for tick_label in y_tick_labels:
        tick_label.set_fontsize(DMS_TICK_FONTSIZE)
        tick_label.set_fontweight("bold")
        tick_label.set_rotation(90)
        tick_label.set_va("center")

    # Lift the bottom-most latitude label above its tick so it does not
    # collide with the first 45°-rotated longitude label at the corner.
    if y_tick_labels:
        y_tick_labels[0].set_va("bottom")
        log(
            "[ADJUST] Bottom latitude DMS label anchored above its "
            "tick (va='bottom') to avoid corner overlap with the "
            "first longitude label."
        )

    add_north_arrow_tri(
        axis,
        size=0.09,
        pad=0.09,
    )

    crs_type = (
        "wgs84" if crs_is_geographic else "utm"
    )

    add_checker_scale_bar(
        axis,
        west,
        east,
        south,
        north,
        crs_type=crs_type,
        mid_lat_deg=(
            y_mid if crs_is_geographic else None
        ),
        segments=4,
        fontsize=5.6,
    )

    legend_elements = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=1.0,
            label="Road",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=1.0,
            label="Study Area",
        ),
    ]

    legend = axis.legend(
        handles=legend_elements,
        loc="lower right",
        bbox_to_anchor=(0.98, 0.09),
        bbox_transform=axis.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=3.1,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0,
    )
    legend.get_frame().set_edgecolor("lightgray")
    legend.get_frame().set_linewidth(0.3)
    for txt in legend.get_texts():
        txt.set_fontweight("bold")

    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes(
        "right",
        size=CAX_SIZE,
        pad=CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(
        labelsize=4.4,
        width=0.6,
        length=2,
    )
    for tick_label in colorbar.ax.get_yticklabels():
        tick_label.set_fontweight("bold")
    colorbar.set_label(
        HKFI_INFO["units"],
        fontsize=5.2,
        fontweight="bold",
    )
    colorbar.set_ticks(
        np.linspace(0.0, 1.0, 5)
    )

    figure.tight_layout()

    if SAVE_FIGURES:
        figure.savefig(
            OUT_MAP_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_MAP_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_MAP_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )

        log(f"[MAP] {OUT_MAP_PNG}")
        log(f"[MAP] {OUT_MAP_PDF}")
        log(f"[MAP] {OUT_MAP_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# HISTOGRAM
# =========================================================
def create_hkfi_histogram(
    values: np.ndarray,
    statistics: dict[str, float | int],
) -> None:
    figure, axis = plt.subplots(
        1,
        1,
        figsize=HIST_FIGSIZE,
        facecolor="white",
    )

    bins = np.linspace(0.0, 1.0, 21)
    counts, edges = np.histogram(values, bins=bins)
    centres = 0.5 * (edges[:-1] + edges[1:])
    colours = HKFI_CMAP(centres)

    axis.bar(
        centres,
        counts,
        width=(bins[1] - bins[0]) * 0.9,
        color=colours,
        edgecolor="black",
        linewidth=0.4,
        alpha=0.9,
    )

    axis.axvline(
        statistics["mean"],
        color="darkred",
        linestyle="--",
        linewidth=0.7,
        label=f"Mean: {statistics['mean']:.3f}",
    )
    axis.axvline(
        statistics["median"],
        color="orange",
        linestyle="--",
        linewidth=0.7,
        label=f"Median: {statistics['median']:.3f}",
    )
    axis.axvline(
        statistics["p95"],
        color="green",
        linestyle="--",
        linewidth=0.7,
        label=f"95th %ile: {statistics['p95']:.3f}",
    )

    axis.legend(
        fontsize=5.0,
        loc="upper right",
        framealpha=0.9,
    )
    axis.set_xlabel(
        HKFI_INFO["units"],
        fontsize=7,
        fontweight="bold",
    )
    axis.set_ylabel(
        "Number of Pixels",
        fontsize=7,
        fontweight="bold",
    )
    axis.tick_params(
        axis="both",
        which="major",
        labelsize=6,
    )
    for tick_label in axis.get_xticklabels() + axis.get_yticklabels():
        tick_label.set_fontweight("bold")
    axis.set_xlim(0.0, 1.0)
    axis.grid(
        True,
        alpha=0.25,
        linestyle="-",
        linewidth=0.4,
        axis="y",
    )
    axis.set_axisbelow(True)

    figure.tight_layout(pad=1.2)

    if SAVE_FIGURES:
        figure.savefig(
            OUT_HIST_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_HIST_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_HIST_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )

        log(f"[HISTOGRAM] {OUT_HIST_PNG}")
        log(f"[HISTOGRAM] {OUT_HIST_PDF}")
        log(f"[HISTOGRAM] {OUT_HIST_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# OUTPUT SUMMARY
# =========================================================
def write_statistics_csv(
    path: Path,
    statistics: dict[str, float | int],
    intervals: dict[str, dict[str, float | int]],
    coverage: dict[str, float | int],
    diagnostics: dict[str, float | int | None],
    provenance: dict[str, Any],
) -> None:
    rows: list[tuple[str, Any]] = []

    for key, value in provenance.items():
        rows.append((f"provenance_{key}", value))

    for key, value in coverage.items():
        rows.append((f"coverage_{key}", value))

    for key, value in statistics.items():
        rows.append((f"HKFI_{key}", value))

    for interval_name, interval_data in intervals.items():
        rows.append(
            (
                f"{interval_name}_count",
                interval_data["count"],
            )
        )
        rows.append(
            (
                f"{interval_name}_percentage",
                interval_data["percentage"],
            )
        )

    for key, value in diagnostics.items():
        rows.append((f"diagnostic_{key}", value))

    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["metric", "value"])
        writer.writerows(rows)


# =========================================================
# MAIN
# =========================================================
def main() -> None:
    log(
        "=== Step 5.3 visualization: revised multiscale HKFI raster ==="
    )

    hkfi_norm_path = choose_existing_raster(
        HKFI_NORM_CANDIDATES,
        "revised multiscale normalized HKFI raster",
    )

    hkfi, reference = read_hkfi_raster(
        hkfi_norm_path
    )

    log(f"[INPUT] Step 5.3 raster: {hkfi_norm_path}")
    log(
        "[PLOT] direct raster=True | cmap=viridis | "
        "origin=upper | interpolation=nearest"
    )

    study_mask = read_mask_to_reference(
        STUDY_MASK_TIF,
        reference,
        "study-area mask",
        optional=False,
        require_exact_grid=False,
    )
    assert study_mask is not None

    context_mask = read_mask_to_reference(
        CONTEXT_MASK_TIF,
        reference,
        "context mask",
        optional=True,
        require_exact_grid=False,
    )

    if context_mask is None:
        context_mask = study_mask.copy()
        context_path_used = STUDY_MASK_TIF
        log("[MASK] Context mask unavailable; study mask used.")
    else:
        context_mask = context_mask & study_mask
        context_path_used = CONTEXT_MASK_TIF

    revised_valid_mask = read_mask_to_reference(
        HKFI_VALID_MASK_TIF,
        reference,
        "revised analytical HKFI valid mask",
        optional=False,
        require_exact_grid=REQUIRE_EXACT_ANALYTICAL_MASK_GRID,
    )
    assert revised_valid_mask is not None

    finite_hkfi = (
        study_mask
        & context_mask
        & np.isfinite(hkfi)
        & (hkfi >= 0.0)
        & (hkfi <= 1.0)
    )

    valid_mask = finite_hkfi & revised_valid_mask

    if not valid_mask.any():
        raise RuntimeError(
            "No valid revised multiscale HKFI values occur "
            "in the context domain."
        )

    missing_zdv_mask = read_mask_to_reference(
        HKFI_MISSING_ZDV_MASK_TIF,
        reference,
        "missing z(DV+) diagnostic mask",
        optional=True,
        require_exact_grid=REQUIRE_EXACT_ANALYTICAL_MASK_GRID,
    )
    mismatch_mask = read_mask_to_reference(
        HKFI_MISMATCH_MASK_TIF,
        reference,
        "missing z(DV+) with nonzero original IRS mask",
        optional=True,
        require_exact_grid=REQUIRE_EXACT_ANALYTICAL_MASK_GRID,
    )

    if missing_zdv_mask is not None:
        missing_zdv_mask = (
            missing_zdv_mask
            & context_mask
            & study_mask
        )

    if mismatch_mask is not None:
        mismatch_mask = (
            mismatch_mask
            & context_mask
            & study_mask
        )

    values = hkfi[valid_mask]

    statistics = calculate_statistics(values)
    intervals = calculate_descriptive_intervals(values)

    study_pixels = int(study_mask.sum())
    context_pixels = int(context_mask.sum())
    valid_pixels = int(valid_mask.sum())

    missing_zdv_pixels = (
        int(missing_zdv_mask.sum())
        if missing_zdv_mask is not None
        else None
    )
    mismatch_pixels = (
        int(mismatch_mask.sum())
        if mismatch_mask is not None
        else None
    )

    coverage = {
        "study_pixels": study_pixels,
        "context_pixels": context_pixels,
        "valid_HKFI_pixels": valid_pixels,
        "valid_percent_of_context": (
            100.0 * valid_pixels / max(context_pixels, 1)
        ),
        "valid_percent_of_study": (
            100.0 * valid_pixels / max(study_pixels, 1)
        ),
    }

    diagnostics = {
        "missing_zDV_pixels": missing_zdv_pixels,
        "missing_zDV_nonzero_original_IRS_pixels": mismatch_pixels,
    }

    metadata = load_json_optional(HKFI_META_JSON)
    quality_assurance = load_json_optional(HKFI_QA_JSON)
    support_document = load_json_optional(
        SM_SUPPORT_MODE_JSON
    ) or {}

    # Step 5.3 writes the support dictionary directly to sm_support_mode.json.
    # The nested fallback supports older variants.
    support_metadata = support_document.get(
        "support_metadata",
        support_document,
    )

    support_mode = support_metadata.get(
        "support_mode",
        "unknown",
    )
    is_exact_native = bool(
        support_metadata.get(
            "is_exact_native_era5_grid",
            False,
        )
    )

    log(
        f"[SM SUPPORT] mode={support_mode} | "
        f"exact_native={is_exact_native}"
    )

    if support_mode == "approximate_9km_from_physical_sm":
        log(
            "[SM SUPPORT WARNING] Interpret cells as approximate 9 km "
            "support cells, not exact native ERA5-Land cells."
        )

    log(
        f"[RASTER] EPSG:{reference['epsg']} | "
        f"{reference['width']} × {reference['height']} | "
        f"{reference['resolution_x']:.3f} m"
    )
    log(
        f"[COVERAGE] {valid_pixels:,}/{context_pixels:,} context pixels "
        f"({coverage['valid_percent_of_context']:.3f}%)"
    )
    log(
        f"[HKFI] min={statistics['min']:.4f} | "
        f"median={statistics['median']:.4f} | "
        f"mean={statistics['mean']:.4f} | "
        f"p95={statistics['p95']:.4f} | "
        f"max={statistics['max']:.4f}"
    )
    log(
        "[STYLE] 2.5×5 inch map | four longitude DMS labels | "
        "three latitude DMS labels (rotated 90°, bottom label "
        "lifted) | no axis titles | 900 dpi | PNG/PDF/SVG"
    )

    if metadata:
        log(
            "[METADATA] Equation: "
            f"{metadata.get('equation', 'Unknown')}"
        )

    qa_status = None
    qa_warnings: list[str] = []

    if quality_assurance:
        qa_status = quality_assurance.get("status")
        qa_warnings = quality_assurance.get(
            "warnings",
            [],
        ) or []

        log(f"[QA] {qa_status}")
        for warning in qa_warnings:
            log(f"[QA WARNING] {warning}")

    print("\nDescriptive multiscale HKFI intervals")
    print(
        "The intervals are descriptive only. Normalized HKFI is "
        "study-relative and is not an absolute landslide probability."
    )

    for name, item in intervals.items():
        print(
            f"  {name}: {item['count']:,} "
            f"({item['percentage']:.2f}%)"
        )

    provenance = {
        "HKFI_raster_used": str(hkfi_norm_path),
        "support_mode": support_mode,
        "is_exact_native_era5_grid": is_exact_native,
        "display_interpolation": RASTER_DISPLAY_INTERPOLATION,
        "values_renormalized_for_visualization": False,
    }

    summary = {
        "generated_at_utc": utc_now_iso(),
        "purpose": (
            "Direct-raster visualization of the revised multiscale HKFI "
            "using support-preserved soil moisture and the final Step 3.7 "
            "DV+ cartographic style."
        ),
        "scientific_plotting": {
            "direct_original_HKFI_raster": True,
            "RBF_used": False,
            "IDW_used": False,
            "cubic_interpolation_used": False,
            "scatter_reconstruction_used": False,
            "display_interpolation": RASTER_DISPLAY_INTERPOLATION,
            "values_renormalized_for_visualization": False,
            "analytical_valid_mask_exact_grid_required": (
                REQUIRE_EXACT_ANALYTICAL_MASK_GRID
            ),
        },
        "soil_moisture_support": {
            "support_mode": support_mode,
            "is_exact_native_era5_grid": is_exact_native,
            "interpretation": support_metadata.get(
                "interpretation"
            ),
            "cell_size_m": support_metadata.get(
                "cell_size_m"
            ),
            "source": support_metadata.get("source"),
        },
        "inputs": {
            "HKFI_raster": str(hkfi_norm_path),
            "HKFI_valid_mask": str(HKFI_VALID_MASK_TIF),
            "study_mask": str(STUDY_MASK_TIF),
            "context_mask_used": str(context_path_used),
            "missing_zDV_mask": (
                str(HKFI_MISSING_ZDV_MASK_TIF)
                if missing_zdv_mask is not None
                else None
            ),
            "mismatch_mask": (
                str(HKFI_MISMATCH_MASK_TIF)
                if mismatch_mask is not None
                else None
            ),
            "HKFI_metadata": (
                str(HKFI_META_JSON)
                if metadata is not None
                else None
            ),
            "HKFI_quality_assurance": (
                str(HKFI_QA_JSON)
                if quality_assurance is not None
                else None
            ),
            "SM_support_metadata": (
                str(SM_SUPPORT_MODE_JSON)
                if support_document
                else None
            ),
        },
        "equation": (
            metadata.get("equation")
            if metadata
            else None
        ),
        "coverage": coverage,
        "statistics": statistics,
        "descriptive_intervals": {
            "warning": (
                "These intervals are descriptive and are not "
                "calibrated landslide probabilities."
            ),
            "counts": intervals,
        },
        "diagnostics": diagnostics,
        "analytical_QA_status": qa_status,
        "analytical_QA_warnings": qa_warnings,
        "visual_style": {
            "map_figsize_inches": list(MAP_FIGSIZE),
            "histogram_figsize_inches": list(HIST_FIGSIZE),
            "colormap": "viridis",
            "axis_titles_shown": False,
            "display_buffer_fraction": DISPLAY_BUFFER_FRACTION,
            "longitude_DMS_tick_count": LONGITUDE_DMS_TICK_COUNT,
            "latitude_DMS_tick_count": LATITUDE_DMS_TICK_COUNT,
            "latitude_DMS_label_rotation_deg": 90,
            "bottom_latitude_label_va": "bottom",
            "DMS_tick_fontsize": DMS_TICK_FONTSIZE,
            "north_arrow": True,
            "checkerboard_scale_bar": True,
            "compact_legend": True,
            "side_colour_bar": True,
            "figure_dpi": FIG_DPI,
        },
        "outputs": {
            "map_png": str(OUT_MAP_PNG),
            "map_pdf": str(OUT_MAP_PDF),
            "map_svg": str(OUT_MAP_SVG),
            "histogram_png": str(OUT_HIST_PNG),
            "histogram_pdf": str(OUT_HIST_PDF),
            "histogram_svg": str(OUT_HIST_SVG),
            "summary_json": str(OUT_SUMMARY_JSON),
            "statistics_csv": str(OUT_STATS_CSV),
        },
    }

    with open(
        OUT_SUMMARY_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    write_statistics_csv(
        OUT_STATS_CSV,
        statistics,
        intervals,
        coverage,
        diagnostics,
        provenance,
    )

    create_hkfi_map(
        hkfi=hkfi,
        valid_mask=valid_mask,
        study_mask=study_mask,
        mismatch_mask=mismatch_mask,
        reference=reference,
    )
    create_hkfi_histogram(
        values=values,
        statistics=statistics,
    )

    log(f"[SUMMARY] {OUT_SUMMARY_JSON}")
    log(f"[SUMMARY] {OUT_STATS_CSV}")
    log(
        "=== Step 5.3 multiscale HKFI visualization completed successfully ==="
    )


def create_hkfi_visualization() -> None:
    try:
        main()
    except Exception as exc:
        print(
            f"[ERROR] Failed to create multiscale HKFI visualization: {exc}"
        )
        traceback.print_exc()
        raise


if __name__ == "__main__":
    create_hkfi_visualization()


# Step 5.3b — Multiscale HKFI Validation and Independent Weight Sensitivity

In [ ]:
# =============================================================================
# Step 5.3b — Multiscale HKFI Validation and Independent Weight Sensitivity
# =============================================================================
#
# Purpose
# -------
# This step independently validates the authoritative outputs from Step 5.3 and

# Important scope
# ---------------
# This script evaluates HKFI-level robustness and validates all alternative
# HKFI inputs. Reviewer Comment 7 also asks whether weights affect final
# susceptibility outputs. That part must be completed by propagating the named
# normalized HKFI rasters through the unchanged downstream spatial-CV model
# pipeline. This script can summarize such results when a model-results CSV is
# supplied, but it does not replace the downstream model training step.
# =============================================================================

from __future__ import annotations

import csv
import json
import math
import os
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import rasterio

try:
    from scipy.stats import spearmanr
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass

import matplotlib.pyplot as plt

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False


# =============================================================================
# USER CONFIGURATION
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

ANCHOR_TIF = (
    BASE / "Results" / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
)
STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
CONTEXT_MASK_TIF = (
    BASE / "Results" / "Step-0.3-Quality_Masks" / "context_mask.tif"
)

HKFI_DIR = BASE / "Results" / "Step-5.3-HKFI"

# Authoritative Step 5.3 component outputs
K_TERM_TIF = (
    HKFI_DIR / "hkfi_term_K_irs_times_zdv_32645_12p5m_float32.tif"
)
T_TERM_TIF = (
    HKFI_DIR / "hkfi_term_T_ztwi_32645_12p5m_float32.tif"
)
S_TERM_TIF = (
    HKFI_DIR
    / "hkfi_term_S_zsm_support_preserved_32645_12p5m_float32.tif"
)
H_LOCAL_TIF = (
    HKFI_DIR / "hkfi_local_K_plus_T_over_2_32645_12p5m_float32.tif"
)
SUPPORT_CELL_ID_TIF = (
    HKFI_DIR / "sm_support_cell_id_32645_12p5m_int32.tif"
)
IRS_EFFECTIVE_HKFI_TIF = (
    HKFI_DIR / "irs_effective_for_hkfi_0_1_32645_12p5m_float32.tif"
)
DV_AVAILABILITY_MASK_TIF = (
    HKFI_DIR / "hkfi_dv_availability_mask_32645_12p5m_byte.tif"
)
HKFI_VALID_MASK_TIF = (
    HKFI_DIR / "hkfi_valid_mask_32645_12p5m_byte.tif"
)

HKFI_RAW_CANDIDATES = [
    HKFI_DIR / "hkfi_multiscale_raw_32645_12p5m_float32.tif",
    HKFI_DIR / "hkfi_raw_zmean_32645_12p5m_float32.tif",
]
HKFI_NORM_CANDIDATES = [
    HKFI_DIR / "hkfi_multiscale_norm_0_1_32645_12p5m_float32.tif",
    HKFI_DIR / "hkfi_norm_0_1_dense_32645_12p5m_float32.tif",
]

STEP53_META_JSON = HKFI_DIR / "hkfi_metadata.json"
STEP53_QA_JSON = HKFI_DIR / "hkfi_quality_assurance.json"
STEP53_SUPPORT_MODE_JSON = HKFI_DIR / "sm_support_mode.json"

STEP53_WEIGHT_DIR = HKFI_DIR / "weight_sensitivity"
STEP53_NAMED_SENSITIVITY_CSV = (
    STEP53_WEIGHT_DIR / "hkfi_named_weight_sensitivity.csv"
)
STEP53_SIMPLEX_SENSITIVITY_CSV = (
    STEP53_WEIGHT_DIR / "hkfi_simplex_weight_sensitivity.csv"
)
STEP53_WEIGHT_MANIFEST_CSV = (
    STEP53_WEIGHT_DIR / "hkfi_weight_scenario_manifest.csv"
)

# Former interpolated z(SM), diagnostic only
OLD_Z_SM_TIF = (
    BASE
    / "Results"
    / "Step-4.2-Standardization"
    / "z_sm_32645_12p5m_float32.tif"
)

# Optional terrain-context inputs. Missing files do not stop the core analysis.
SLOPE_TIF_CANDIDATES = [
    BASE / "Results" / "Step-1.1-Slope_Aspect" / "slope_deg.tif",
    BASE
    / "Results"
    / "Step-1.1-Slope_Aspect"
    / "slope_deg_32645_12p5m_float32.tif",
    BASE
    / "Results"
    / "Step-3.3-Downslope unit vector (ŝ)"
    / "slope_deg_32645_12p5m_float32.tif",
]
G_TIF_CANDIDATES = [
    BASE
    / "Results"
    / "Step-5.1-Observability"
    / "geometric_observability_G_0_1_32645_12p5m_float32.tif",
    BASE
    / "Results"
    / "Step-5.1-Observability"
    / "observability_0_1_32645_12p5m_float32.tif",
]

# Optional downstream model-results CSV. Leave as None until the susceptibility
# models have been rerun with the named HKFI scenarios. When supplied, required
# columns are: scenario, model, fold, auc. Additional metrics are optional.
DOWNSTREAM_MODEL_RESULTS_CSV: Path | None = None

# Output directory
OUT_DIR = BASE / "Results" / "Step-5.3b-HKFI_Revised_Validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Comment 4 outputs
EQUATION_QA_CSV = OUT_DIR / "hkfi_equation_reproduction_QA.csv"
SM_CONSTANCY_QA_CSV = OUT_DIR / "hkfi_sm_within_support_constancy_QA.csv"
SM_CONSTANCY_DETAIL_CSV = OUT_DIR / "hkfi_sm_support_cell_constancy_details.csv"
COMPONENT_STATS_BY_DOMAIN_CSV = OUT_DIR / "hkfi_component_statistics_by_domain.csv"
COMPONENT_CORRELATIONS_BY_DOMAIN_CSV = OUT_DIR / "hkfi_component_correlations_by_domain.csv"
CONTRIBUTIONS_BY_DOMAIN_CSV = OUT_DIR / "hkfi_covariance_contributions_by_domain.csv"
WITHIN_BETWEEN_CSV = OUT_DIR / "hkfi_within_between_variance_partition.csv"
WITHIN_CONTRIBUTIONS_CSV = OUT_DIR / "hkfi_within_cell_component_contributions.csv"
BETWEEN_CONTRIBUTIONS_CSV = OUT_DIR / "hkfi_between_cell_component_contributions.csv"
SUPPORT_CELL_STATS_CSV = OUT_DIR / "hkfi_support_cell_statistics.csv"
OLD_VS_REVISED_CSV = OUT_DIR / "hkfi_old_vs_revised_comparison.csv"

# Comment 7 outputs
NAMED_WEIGHT_VALIDATION_CSV = OUT_DIR / "hkfi_named_weight_validation.csv"
NAMED_WEIGHT_BY_DOMAIN_CSV = OUT_DIR / "hkfi_named_weight_sensitivity_by_domain.csv"
NAMED_WEIGHT_CONTRIBUTIONS_CSV = OUT_DIR / "hkfi_named_weight_contributions_by_domain.csv"
NAMED_WEIGHT_CONTEXT_CSV = OUT_DIR / "hkfi_named_weight_sensitivity_by_terrain_context.csv"
PRIMARY_QUINTILE_THRESHOLDS_CSV = (
    OUT_DIR / "hkfi_primary_equal_reference_quintile_thresholds.csv"
)
SIMPLEX_RECOMPUTED_CSV = OUT_DIR / "hkfi_simplex_weight_sensitivity_recomputed.csv"
SIMPLEX_CROSSCHECK_CSV = OUT_DIR / "hkfi_simplex_step53_crosscheck.csv"
SIMPLEX_SUMMARY_CSV = OUT_DIR / "hkfi_simplex_weight_sensitivity_summary.csv"
DOWNSTREAM_MANIFEST_CSV = OUT_DIR / "hkfi_downstream_spatial_CV_manifest.csv"
DOWNSTREAM_RESULTS_TEMPLATE_CSV = OUT_DIR / "hkfi_downstream_model_results_template.csv"
DOWNSTREAM_RESULTS_SUMMARY_CSV = OUT_DIR / "hkfi_downstream_model_results_summary.csv"
DOWNSTREAM_PAIRED_AUC_CSV = OUT_DIR / "hkfi_downstream_paired_fold_AUC_differences.csv"

# Historical diagnostic rasters
OLD_HKFI_RAW_TIF = OUT_DIR / "hkfi_old_interpolated_sm_raw_32645_12p5m_float32.tif"
OLD_HKFI_NORM_TIF = OUT_DIR / "hkfi_old_interpolated_sm_norm_0_1_32645_12p5m_float32.tif"
REVISED_MINUS_OLD_TIF = OUT_DIR / "hkfi_revised_minus_old_raw_32645_12p5m_float32.tif"

# Reports, metadata and figures
QA_JSON = OUT_DIR / "hkfi_comments_4_and_7_quality_assurance.json"
META_JSON = OUT_DIR / "hkfi_comments_4_and_7_metadata.json"
COMMENT4_REPORT_TXT = OUT_DIR / "Reviewer_1_Comment_4_HKFI_Report.txt"
COMMENT7_REPORT_TXT = OUT_DIR / "Reviewer_1_Comment_7_HKFI_Weight_Report.txt"

COMMENT4_FIGURE_PNG = OUT_DIR / "HKFI_Reviewer_Comment_4_Figure.png"
COMMENT4_FIGURE_PDF = OUT_DIR / "HKFI_Reviewer_Comment_4_Figure.pdf"
COMMENT4_FIGURE_SVG = OUT_DIR / "HKFI_Reviewer_Comment_4_Figure.svg"
COMMENT7_FIGURE_PNG = OUT_DIR / "HKFI_Reviewer_Comment_7_Named_Weights.png"
COMMENT7_FIGURE_PDF = OUT_DIR / "HKFI_Reviewer_Comment_7_Named_Weights.pdf"
COMMENT7_FIGURE_SVG = OUT_DIR / "HKFI_Reviewer_Comment_7_Named_Weights.svg"
SIMPLEX_FIGURE_PNG = OUT_DIR / "HKFI_Reviewer_Comment_7_Simplex.png"
SIMPLEX_FIGURE_PDF = OUT_DIR / "HKFI_Reviewer_Comment_7_Simplex.pdf"
SIMPLEX_FIGURE_SVG = OUT_DIR / "HKFI_Reviewer_Comment_7_Simplex.svg"
OLD_VS_REVISED_FIGURE_PNG = OUT_DIR / "HKFI_Old_vs_Revised_Diagnostic.png"

# Scientific settings
REQUIRE_EXACT_GRID = True
NODATA_FLOAT = -9999.0
NODATA_INT32 = -2147483648

IRS_SUPPORT_TOL = 1.0e-6
HIGH_CONFIDENCE_IRS_THRESHOLD = 0.20

MIN_DOMAIN_PIXELS = 100
MIN_FINE_PIXELS_PER_SUPPORT_CELL = 10

EQUATION_RMSE_TOL = 1.0e-6
EQUATION_MAX_ABS_TOL = 1.0e-5
SM_WITHIN_CELL_STD_TOL = 1.0e-7
CONTRIBUTION_CLOSURE_TOL = 1.0e-6
WEIGHT_TOL = 1.0e-10
STEP53_METRIC_CROSSCHECK_TOL = 1.0e-6

HKFI_NORM_P_LOW = 2.0
HKFI_NORM_P_HIGH = 98.0

# True HKFI quintile boundaries are calculated ONCE from the authoritative
# primary equal-weight normalized HKFI over the full HKFI analysis domain.
# The resulting Q20/Q40/Q60/Q80 thresholds are then reused unchanged for the
# primary map and every alternative weight scenario. This makes class-change
# metrics directly comparable across scenarios.
REFERENCE_QUINTILE_PERCENTILES = (20.0, 40.0, 60.0, 80.0)

SIMPLEX_WEIGHT_INCREMENT = 0.10

# Spearman rank correlation may require substantial memory for millions of
# pixels and 66 simplex scenarios. Error, quintile and overlap metrics are
# always calculated on all valid pixels. Rank metrics use a fixed reproducible
# sample only when the domain exceeds this limit. Set to None for exact ranks.
MAX_RANK_SAMPLE_PIXELS: int | None = 300_000
RANK_SAMPLE_SEED = 5307

WRITE_HISTORICAL_DIAGNOSTIC_RASTERS = True
MAKE_FIGURES = True
SHOW_FIGURES = True
FIG_DPI = 600

EXPECTED_NAMED_SCENARIOS: dict[str, dict[str, Any]] = {
    "primary_equal_component": {
        "weights": {"K": 1.0 / 3.0, "T": 1.0 / 3.0, "S": 1.0 / 3.0},
        "purpose": "Transparent a priori non-preferential reference",
    },
    "kinematic_emphasis": {
        "weights": {"K": 0.50, "T": 0.25, "S": 0.25},
        "purpose": "Tests stronger reliability-weighted deformation influence",
    },
    "twi_emphasis": {
        "weights": {"K": 0.25, "T": 0.50, "S": 0.25},
        "purpose": "Tests stronger local topographic-wetness influence",
    },
    "sm_emphasis": {
        "weights": {"K": 0.25, "T": 0.25, "S": 0.50},
        "purpose": "Tests stronger regional soil-moisture loading",
    },
    "local_emphasis": {
        "weights": {"K": 0.40, "T": 0.40, "S": 0.20},
        "purpose": "Tests greater combined local K and T influence",
    },
    "hydrological_emphasis": {
        "weights": {"K": 0.20, "T": 0.40, "S": 0.40},
        "purpose": "Tests greater combined TWI and soil-moisture influence",
    },
    "no_kinematic_K": {
        "weights": {"K": 0.00, "T": 0.50, "S": 0.50},
        "purpose": "Kinematic component ablation",
    },
    "no_TWI": {
        "weights": {"K": 0.50, "T": 0.00, "S": 0.50},
        "purpose": "Topographic-wetness component ablation",
    },
    "no_SM": {
        "weights": {"K": 0.50, "T": 0.50, "S": 0.00},
        "purpose": "Soil-moisture component ablation",
    },
}

EPS = 1.0e-12


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def load_json_optional(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        log(f"[WARN] Could not read JSON {path.name}: {exc}")
        return None


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def choose_existing(candidates: Iterable[Path], description: str) -> Path:
    candidates = list(candidates)
    for path in candidates:
        if path.exists():
            log(f"[PATH] {description}: {path}")
            return path
    attempted = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not locate {description}. Candidates:\n{attempted}"
    )


def choose_optional(candidates: Iterable[Path], description: str) -> Path | None:
    for path in candidates:
        if path.exists():
            log(f"[OPTIONAL PATH] {description}: {path}")
            return path
    log(f"[OPTIONAL] {description} not found; related context analysis skipped.")
    return None


def pretty_scenario(name: str) -> str:
    replacements = {
        "primary_equal_component": "Primary equal",
        "kinematic_emphasis": "Kinematic emphasis",
        "twi_emphasis": "TWI emphasis",
        "sm_emphasis": "SM emphasis",
        "local_emphasis": "Local emphasis",
        "hydrological_emphasis": "Hydrological emphasis",
        "no_kinematic_K": "No K",
        "no_TWI": "No TWI",
        "no_SM": "No SM",
    }
    return replacements.get(name, name.replace("_", " "))


# =============================================================================
# RASTER I/O
# =============================================================================
def read_anchor(path: Path) -> dict[str, Any]:
    require_file(path, "anchor raster")
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"Anchor has no CRS: {path}")
        transform = dataset.transform
        cell_area_m2 = abs(
            transform.a * transform.e - transform.b * transform.d
        )
        return {
            "crs": dataset.crs,
            "transform": transform,
            "width": dataset.width,
            "height": dataset.height,
            "bounds": dataset.bounds,
            "profile": dataset.profile.copy(),
            "epsg": dataset.crs.to_epsg(),
            "cell_area_m2": float(cell_area_m2),
        }


def same_grid(dataset: rasterio.io.DatasetReader, anchor: dict[str, Any]) -> bool:
    return (
        dataset.crs == anchor["crs"]
        and dataset.transform == anchor["transform"]
        and dataset.width == anchor["width"]
        and dataset.height == anchor["height"]
    )


def clean_continuous(
    array: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    output = array.astype("float64", copy=True)
    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if np.isfinite(nodata_value):
                output[output == nodata_value] = np.nan
        except Exception:
            pass
    output[output == -9999.0] = np.nan
    output[output == -32768.0] = np.nan
    output[np.abs(output) > 1.0e20] = np.nan
    return output


def read_continuous_exact(
    path: Path,
    anchor: dict[str, Any],
    description: str,
) -> np.ndarray:
    require_file(path, description)
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")
        if REQUIRE_EXACT_GRID and not same_grid(dataset, anchor):
            raise ValueError(
                f"{description} does not match the anchor grid:\n{path}"
            )
        return clean_continuous(dataset.read(1), dataset.nodata)


def read_mask_exact(
    path: Path,
    anchor: dict[str, Any],
    description: str,
    required: bool = True,
) -> np.ndarray | None:
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Missing {description}: {path}")
        return None
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")
        if REQUIRE_EXACT_GRID and not same_grid(dataset, anchor):
            raise ValueError(
                f"{description} does not match the anchor grid:\n{path}"
            )
        raw = dataset.read(1)
        values = raw.astype("float64")
        if dataset.nodata is not None:
            values[raw == dataset.nodata] = 0.0
        values[~np.isfinite(values)] = 0.0
        return values > 0.5


def read_cell_ids_exact(path: Path, anchor: dict[str, Any]) -> np.ndarray:
    require_file(path, "soil-moisture support-cell ID raster")
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"Support-cell ID raster has no CRS: {path}")
        if REQUIRE_EXACT_GRID and not same_grid(dataset, anchor):
            raise ValueError(
                f"Support-cell ID raster does not match anchor:\n{path}"
            )
        raw = dataset.read(1)
        output = raw.astype("float64")
        if dataset.nodata is not None:
            output[raw == dataset.nodata] = np.nan
        output[
            (output <= 0)
            | (output == NODATA_INT32)
            | ~np.isfinite(output)
        ] = np.nan
        return output


def write_float_tif(
    path: Path,
    array: np.ndarray,
    anchor: dict[str, Any],
    description: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        count=1,
        dtype="float32",
        nodata=NODATA_FLOAT,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )
    output = np.where(np.isfinite(array), array, NODATA_FLOAT).astype("float32")
    temporary = Path(str(path) + ".part")
    with rasterio.open(temporary, "w", **profile) as dataset:
        dataset.write(output, 1)
        dataset.set_band_description(1, description)
        dataset.update_tags(
            step="5.3b_HKFI_comments_4_and_7",
            variable=description,
            generated_utc=utc_now_iso(),
        )
    os.replace(temporary, path)


# =============================================================================
# WEIGHT AND EQUATION HELPERS
# =============================================================================
def validate_weights(
    weights: dict[str, float],
    label: str,
    tolerance: float = WEIGHT_TOL,
) -> dict[str, float]:
    required = {"K", "T", "S"}
    if set(weights) != required:
        raise ValueError(
            f"{label} must define exactly K, T and S. Received {sorted(weights)}"
        )
    result = {name: float(weights[name]) for name in ("K", "T", "S")}
    if any((not np.isfinite(value)) or value < 0.0 for value in result.values()):
        raise ValueError(f"{label} has negative or non-finite weights: {result}")
    total = float(sum(result.values()))
    if abs(total - 1.0) > tolerance:
        raise ValueError(
            f"{label} weights must sum to 1.0; received {total:.12f}."
        )
    return result


def generate_simplex_weight_scenarios(
    increment: float,
) -> list[tuple[str, dict[str, float]]]:
    if not np.isfinite(increment) or increment <= 0.0 or increment > 1.0:
        raise ValueError("SIMPLEX_WEIGHT_INCREMENT must be within (0,1].")
    steps_float = 1.0 / increment
    steps = int(round(steps_float))
    if not math.isclose(
        steps * increment,
        1.0,
        rel_tol=0.0,
        abs_tol=1.0e-10,
    ):
        raise ValueError("Simplex increment must divide 1.0 exactly.")
    scenarios: list[tuple[str, dict[str, float]]] = []
    for k_step in range(steps + 1):
        for t_step in range(steps - k_step + 1):
            s_step = steps - k_step - t_step
            weights = {
                "K": k_step / steps,
                "T": t_step / steps,
                "S": s_step / steps,
            }
            label = f"simplex_K{k_step:02d}_T{t_step:02d}_S{s_step:02d}"
            scenarios.append((label, weights))
    return scenarios


def combine_hkfi_components(
    K: np.ndarray,
    T: np.ndarray,
    S: np.ndarray,
    domain: np.ndarray,
    weights: dict[str, float],
    label: str,
) -> np.ndarray:
    weights = validate_weights(weights, label)
    valid = (
        domain
        & np.isfinite(K)
        & np.isfinite(T)
        & np.isfinite(S)
    )
    output = np.full(K.shape, np.nan, dtype="float64")
    output[valid] = (
        weights["K"] * K[valid]
        + weights["T"] * T[valid]
        + weights["S"] * S[valid]
    )
    return output


def equation_comparison_row(
    label: str,
    expected: np.ndarray,
    observed: np.ndarray,
    domain: np.ndarray,
    rmse_tolerance: float = EQUATION_RMSE_TOL,
    max_abs_tolerance: float = EQUATION_MAX_ABS_TOL,
) -> dict[str, Any]:
    valid = domain & np.isfinite(expected) & np.isfinite(observed)
    if int(valid.sum()) < MIN_DOMAIN_PIXELS:
        raise RuntimeError(f"Too few pixels for equation QA: {label}")
    difference = observed[valid] - expected[valid]
    rmse = float(np.sqrt(np.mean(difference ** 2)))
    mae = float(np.mean(np.abs(difference)))
    maximum = float(np.max(np.abs(difference)))
    failed = int(np.sum(np.abs(difference) > max_abs_tolerance))
    return {
        "check": label,
        "n": int(valid.sum()),
        "rmse": rmse,
        "mae": mae,
        "maximum_absolute_difference": maximum,
        "pixels_exceeding_max_abs_tolerance": failed,
        "rmse_tolerance": rmse_tolerance,
        "max_abs_tolerance": max_abs_tolerance,
        "pass": bool(
            rmse <= rmse_tolerance
            and maximum <= max_abs_tolerance
        ),
    }


# =============================================================================
# STATISTICAL HELPERS
# =============================================================================
def descriptive_stats(values: np.ndarray) -> dict[str, float | int | None]:
    values = values[np.isfinite(values)].astype("float64")
    if values.size == 0:
        return {"n": 0}
    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p02": float(np.percentile(values, 2)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "variance": float(np.var(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p98": float(np.percentile(values, 98)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(
            np.mean(np.isclose(values, 0.0, atol=1.0e-12))
        ),
        "unique_values": int(np.unique(values).size),
    }


def pearson_correlation(x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if int(valid.sum()) < 2:
        return float("nan")
    xv = x[valid].astype("float64")
    yv = y[valid].astype("float64")
    if np.std(xv) <= EPS or np.std(yv) <= EPS:
        return float("nan")
    return float(np.corrcoef(xv, yv)[0, 1])


def deterministic_rank_sample(
    n: int,
    maximum: int | None,
    seed: int,
) -> np.ndarray | None:
    if maximum is None or n <= maximum:
        return None
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(n, size=maximum, replace=False))


def spearman_correlation_values(
    x: np.ndarray,
    y: np.ndarray,
    sample_positions: np.ndarray | None = None,
) -> tuple[float, int, bool]:
    valid = np.isfinite(x) & np.isfinite(y)
    xv = x[valid].astype("float64")
    yv = y[valid].astype("float64")
    if xv.size < 2:
        return float("nan"), int(xv.size), False
    sampled = False
    if sample_positions is not None:
        positions = sample_positions[sample_positions < xv.size]
        xv = xv[positions]
        yv = yv[positions]
        sampled = True
    if np.std(xv) <= EPS or np.std(yv) <= EPS:
        return float("nan"), int(xv.size), sampled
    if HAVE_SCIPY:
        result = spearmanr(xv, yv)
        return float(result.statistic), int(xv.size), sampled
    xr = pd.Series(xv).rank(method="average").to_numpy(dtype="float64")
    yr = pd.Series(yv).rank(method="average").to_numpy(dtype="float64")
    return pearson_correlation(xr, yr), int(xv.size), sampled


def robust_normalize_0_1(
    array: np.ndarray,
    domain: np.ndarray,
    p_low: float = HKFI_NORM_P_LOW,
    p_high: float = HKFI_NORM_P_HIGH,
) -> tuple[np.ndarray, dict[str, float | int]]:
    valid = domain & np.isfinite(array)
    if int(valid.sum()) < MIN_DOMAIN_PIXELS:
        raise RuntimeError("Too few valid pixels for robust normalization.")
    values = array[valid].astype("float64")
    lower = float(np.percentile(values, p_low))
    upper = float(np.percentile(values, p_high))
    if upper <= lower + EPS:
        raise RuntimeError("Invalid robust normalization bounds.")
    output = np.full(array.shape, np.nan, dtype="float64")
    output[valid] = np.clip(
        (array[valid] - lower) / (upper - lower),
        0.0,
        1.0,
    )
    return output, {
        "p_low": p_low,
        "p_high": p_high,
        "lower": lower,
        "upper": upper,
        "n": int(valid.sum()),
        "fraction_clipped_low": float(np.mean(array[valid] <= lower)),
        "fraction_clipped_high": float(np.mean(array[valid] >= upper)),
    }


def calculate_reference_quintile_thresholds(
    primary_norm: np.ndarray,
    reference_domain: np.ndarray,
    percentiles: tuple[float, ...] = REFERENCE_QUINTILE_PERCENTILES,
) -> np.ndarray:
    """
    Calculate true quantile boundaries from the authoritative primary HKFI.

    The thresholds are calculated once from the primary equal-weight normalized
    HKFI over the full reference domain and are subsequently reused unchanged
    for all scenario maps. For five classes, the boundaries are Q20, Q40, Q60
    and Q80.
    """
    valid = reference_domain & np.isfinite(primary_norm)
    if int(valid.sum()) < MIN_DOMAIN_PIXELS:
        raise RuntimeError(
            "Too few valid primary-HKFI pixels to calculate reference quintiles."
        )

    values = primary_norm[valid].astype("float64")
    thresholds = np.asarray(
        np.percentile(values, percentiles),
        dtype="float64",
    )

    if thresholds.ndim != 1 or thresholds.size != len(percentiles):
        raise RuntimeError("Unexpected reference-quintile threshold shape.")
    if not np.all(np.isfinite(thresholds)):
        raise RuntimeError("Reference-quintile thresholds contain non-finite values.")
    if np.any(np.diff(thresholds) <= EPS):
        raise RuntimeError(
            "Primary HKFI quantile boundaries are not strictly increasing. "
            "This indicates excessive ties around one or more quintile boundaries."
        )
    return thresholds


def classify_with_fixed_thresholds(
    values: np.ndarray,
    thresholds: np.ndarray,
) -> np.ndarray:
    """
    Assign ordinal classes 0..len(thresholds) using fixed reference thresholds.

    The same thresholds must be used for the primary and every sensitivity
    scenario. np.searchsorted(..., side='right') places values exactly equal to
    a boundary in the higher class and provides deterministic classification.
    """
    values = np.asarray(values, dtype="float64")
    thresholds = np.asarray(thresholds, dtype="float64")

    if thresholds.ndim != 1 or thresholds.size == 0:
        raise ValueError("Fixed class thresholds must be a non-empty 1-D array.")
    if not np.all(np.isfinite(thresholds)):
        raise ValueError("Fixed class thresholds contain non-finite values.")
    if np.any(np.diff(thresholds) <= 0.0):
        raise ValueError("Fixed class thresholds must be strictly increasing.")

    return np.searchsorted(thresholds, values, side="right").astype("int16")


def comparison_metrics(
    primary_raw: np.ndarray,
    scenario_raw: np.ndarray,
    primary_norm: np.ndarray,
    scenario_norm: np.ndarray,
    domain: np.ndarray,
    rank_seed: int,
    reference_quintile_thresholds: np.ndarray,
) -> dict[str, float | int | bool | str]:
    valid_raw = domain & np.isfinite(primary_raw) & np.isfinite(scenario_raw)
    valid_norm = domain & np.isfinite(primary_norm) & np.isfinite(scenario_norm)
    if int(valid_raw.sum()) < 2 or int(valid_norm.sum()) < 2:
        raise RuntimeError("Insufficient common pixels for scenario comparison.")

    raw_primary = primary_raw[valid_raw].astype("float64")
    raw_scenario = scenario_raw[valid_raw].astype("float64")
    norm_primary = primary_norm[valid_norm].astype("float64")
    norm_scenario = scenario_norm[valid_norm].astype("float64")

    raw_difference = raw_scenario - raw_primary
    norm_difference = norm_scenario - norm_primary

    sample_positions = deterministic_rank_sample(
        int(norm_primary.size),
        MAX_RANK_SAMPLE_PIXELS,
        rank_seed,
    )
    raw_sample_positions = deterministic_rank_sample(
        int(raw_primary.size),
        MAX_RANK_SAMPLE_PIXELS,
        rank_seed,
    )
    raw_rho, raw_rank_n, raw_sampled = spearman_correlation_values(
        raw_primary,
        raw_scenario,
        raw_sample_positions,
    )
    norm_rho, norm_rank_n, norm_sampled = spearman_correlation_values(
        norm_primary,
        norm_scenario,
        sample_positions,
    )

    # True quintile classes are defined by Q20/Q40/Q60/Q80 from the
    # authoritative primary equal-weight HKFI over the full analysis domain.
    # The SAME fixed thresholds are then applied to both maps. This measures
    # genuine movement across reference HKFI classes rather than changes caused
    # by re-estimating class boundaries separately for each scenario.
    primary_quintile = classify_with_fixed_thresholds(
        norm_primary,
        reference_quintile_thresholds,
    )
    scenario_quintile = classify_with_fixed_thresholds(
        norm_scenario,
        reference_quintile_thresholds,
    )

    primary_threshold = float(np.percentile(norm_primary, 90))
    scenario_threshold = float(np.percentile(norm_scenario, 90))
    primary_top = norm_primary >= primary_threshold
    scenario_top = norm_scenario >= scenario_threshold
    intersection = int(np.sum(primary_top & scenario_top))
    union = int(np.sum(primary_top | scenario_top))

    return {
        "n_raw": int(valid_raw.sum()),
        "n_norm": int(valid_norm.sum()),
        "raw_pearson_r": pearson_correlation(raw_primary, raw_scenario),
        "raw_spearman_rho": raw_rho,
        "raw_rank_n": raw_rank_n,
        "raw_rank_sampled": raw_sampled,
        "raw_mae": float(np.mean(np.abs(raw_difference))),
        "raw_rmse": float(np.sqrt(np.mean(raw_difference ** 2))),
        "raw_mean_difference_scenario_minus_primary": float(
            np.mean(raw_difference)
        ),
        "norm_pearson_r": pearson_correlation(norm_primary, norm_scenario),
        "norm_spearman_rho": norm_rho,
        "norm_rank_n": norm_rank_n,
        "norm_rank_sampled": norm_sampled,
        "norm_mae": float(np.mean(np.abs(norm_difference))),
        "norm_rmse": float(np.sqrt(np.mean(norm_difference ** 2))),
        "quintile_change_fraction": float(
            np.mean(primary_quintile != scenario_quintile)
        ),
        "mean_absolute_quintile_shift": float(
            np.mean(np.abs(primary_quintile - scenario_quintile))
        ),
        "quintile_definition": (
            "fixed_primary_equal_weight_Q20_Q40_Q60_Q80"
        ),
        "reference_quintile_q20": float(reference_quintile_thresholds[0]),
        "reference_quintile_q40": float(reference_quintile_thresholds[1]),
        "reference_quintile_q60": float(reference_quintile_thresholds[2]),
        "reference_quintile_q80": float(reference_quintile_thresholds[3]),
        "top_decile_jaccard": (
            float(intersection / union) if union else float("nan")
        ),
        "primary_top_decile_threshold": primary_threshold,
        "scenario_top_decile_threshold": scenario_threshold,
    }


# =============================================================================
# COMPONENT AND SUPPORT-CELL ANALYSIS
# =============================================================================
def covariance_contributions(
    components: dict[str, np.ndarray],
    weights: dict[str, float],
    total: np.ndarray,
    domain: np.ndarray,
    domain_name: str,
    scenario_name: str = "primary_equal_component",
) -> pd.DataFrame:
    weights = validate_weights(weights, f"weights[{scenario_name}]")
    valid = domain & np.isfinite(total)
    for array in components.values():
        valid &= np.isfinite(array)
    if int(valid.sum()) < MIN_DOMAIN_PIXELS:
        return pd.DataFrame()
    total_values = total[valid].astype("float64")
    total_variance = float(np.var(total_values))
    if total_variance <= EPS:
        raise RuntimeError(
            f"HKFI variance is zero in domain {domain_name}, scenario {scenario_name}."
        )
    rows: list[dict[str, Any]] = []
    for name, array in components.items():
        weighted_values = weights[name] * array[valid].astype("float64")
        covariance = float(
            np.mean(
                (weighted_values - np.mean(weighted_values))
                * (total_values - np.mean(total_values))
            )
        )
        contribution = covariance / total_variance
        rows.append(
            {
                "scenario": scenario_name,
                "domain": domain_name,
                "domain_pixel_count": int(valid.sum()),
                "component": name,
                "weight": weights[name],
                "component_zero_fraction": float(
                    np.mean(np.isclose(array[valid], 0.0, atol=1.0e-12))
                ),
                "weighted_component_variance": float(np.var(weighted_values)),
                "covariance_with_total": covariance,
                "total_hkfi_variance": total_variance,
                "contribution_fraction": contribution,
                "contribution_percent": 100.0 * contribution,
            }
        )
    dataframe = pd.DataFrame(rows)
    dataframe["contribution_sum_check"] = float(
        dataframe["contribution_fraction"].sum()
    )
    return dataframe


def prepare_cell_dataframe(
    support_ids: np.ndarray,
    domain: np.ndarray,
    arrays: dict[str, np.ndarray],
) -> pd.DataFrame:
    valid = domain & np.isfinite(support_ids) & (support_ids > 0)
    for array in arrays.values():
        valid &= np.isfinite(array)
    data: dict[str, np.ndarray] = {
        "support_id": support_ids[valid].astype("int64"),
    }
    for name, array in arrays.items():
        data[name] = array[valid].astype("float64")
    dataframe = pd.DataFrame(data)
    counts = dataframe.groupby("support_id").size()
    eligible = counts[counts >= MIN_FINE_PIXELS_PER_SUPPORT_CELL].index
    dataframe = dataframe[dataframe["support_id"].isin(eligible)].copy()
    if dataframe.empty:
        raise RuntimeError("No support cell has enough fine-grid pixels.")
    return dataframe


def support_constancy_tables(
    support_ids: np.ndarray,
    S: np.ndarray,
    domain: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    dataframe = prepare_cell_dataframe(support_ids, domain, {"S": S})
    grouped = dataframe.groupby("support_id", sort=True)["S"]
    detail = grouped.agg(
        n_pixels="size",
        mean="mean",
        minimum="min",
        maximum="max",
    ).reset_index()
    detail["std"] = grouped.std(ddof=0).to_numpy()
    detail["range"] = detail["maximum"] - detail["minimum"]
    detail["exceeds_tolerance"] = detail["std"] > SM_WITHIN_CELL_STD_TOL
    summary = pd.DataFrame(
        [
            {
                "support_cell_count": int(len(detail)),
                "fine_pixel_count": int(detail["n_pixels"].sum()),
                "mean_within_cell_std": float(detail["std"].mean()),
                "median_within_cell_std": float(detail["std"].median()),
                "maximum_within_cell_std": float(detail["std"].max()),
                "maximum_within_cell_range": float(detail["range"].max()),
                "std_tolerance": SM_WITHIN_CELL_STD_TOL,
                "cells_exceeding_tolerance": int(
                    detail["exceeds_tolerance"].sum()
                ),
                "pass": bool(not detail["exceeds_tolerance"].any()),
            }
        ]
    )
    return summary, detail


def weighted_mean(values: np.ndarray, weights: np.ndarray) -> float:
    return float(np.sum(values * weights) / np.sum(weights))


def weighted_variance(values: np.ndarray, weights: np.ndarray) -> float:
    mean = weighted_mean(values, weights)
    return float(np.sum(weights * (values - mean) ** 2) / np.sum(weights))


def weighted_covariance(
    x: np.ndarray,
    y: np.ndarray,
    weights: np.ndarray,
) -> float:
    x_mean = weighted_mean(x, weights)
    y_mean = weighted_mean(y, weights)
    return float(
        np.sum(weights * (x - x_mean) * (y - y_mean)) / np.sum(weights)
    )


def within_between_analysis(
    dataframe: pd.DataFrame,
    component_names: list[str],
    total_name: str,
    component_weights: dict[str, float],
    domain_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    grouped = dataframe.groupby("support_id", sort=True)
    cell_counts = grouped.size().rename("n_pixels")
    variables = component_names + [total_name]
    cell_means = grouped[variables].mean()
    cell_variances = grouped[variables].var(ddof=0)
    cell_summary = cell_means.copy()
    for variable in variables:
        cell_summary[f"{variable}_within_variance"] = cell_variances[variable]
    cell_summary["n_pixels"] = cell_counts
    cell_summary["domain"] = domain_name
    cell_summary = cell_summary.reset_index()
    total_pixel_count = float(cell_counts.sum())

    variance_rows: list[dict[str, Any]] = []
    for variable in variables:
        pixel_values = dataframe[variable].to_numpy(dtype="float64")
        total_variance = float(np.var(pixel_values))
        within_variance = float(
            np.sum(
                cell_counts.to_numpy(dtype="float64")
                * cell_variances[variable].to_numpy(dtype="float64")
            )
            / total_pixel_count
        )
        between_variance = weighted_variance(
            cell_means[variable].to_numpy(dtype="float64"),
            cell_counts.to_numpy(dtype="float64"),
        )
        variance_rows.append(
            {
                "domain": domain_name,
                "variable": variable,
                "support_cell_count": int(cell_counts.size),
                "fine_pixel_count": int(total_pixel_count),
                "total_variance": total_variance,
                "within_support_variance": within_variance,
                "between_support_variance": between_variance,
                "within_fraction_of_total": (
                    within_variance / total_variance
                    if total_variance > EPS
                    else np.nan
                ),
                "between_fraction_of_total": (
                    between_variance / total_variance
                    if total_variance > EPS
                    else np.nan
                ),
                "law_total_variance_closure_error": (
                    total_variance - within_variance - between_variance
                ),
            }
        )
    variance_df = pd.DataFrame(variance_rows)

    within_dataframe = dataframe.copy()
    for variable in variables:
        within_dataframe[f"{variable}_within"] = (
            within_dataframe[variable]
            - grouped[variable].transform("mean")
        )
    within_total = within_dataframe[f"{total_name}_within"].to_numpy(
        dtype="float64"
    )
    within_total_variance = float(np.var(within_total))
    within_rows: list[dict[str, Any]] = []
    for component in component_names:
        weighted_component = (
            component_weights[component]
            * within_dataframe[f"{component}_within"].to_numpy(dtype="float64")
        )
        covariance = float(np.mean(weighted_component * within_total))
        contribution = (
            covariance / within_total_variance
            if within_total_variance > EPS
            else np.nan
        )
        within_rows.append(
            {
                "domain": domain_name,
                "component": component,
                "weight": component_weights[component],
                "within_covariance_with_HKFI": covariance,
                "within_HKFI_variance": within_total_variance,
                "within_contribution_fraction": contribution,
                "within_contribution_percent": (
                    100.0 * contribution if np.isfinite(contribution) else np.nan
                ),
            }
        )
    within_df = pd.DataFrame(within_rows)
    within_df["contribution_sum_check"] = float(
        within_df["within_contribution_fraction"].sum()
    )

    between_total = cell_means[total_name].to_numpy(dtype="float64")
    support_weights = cell_counts.to_numpy(dtype="float64")
    between_total_variance = weighted_variance(between_total, support_weights)
    between_rows: list[dict[str, Any]] = []
    for component in component_names:
        weighted_component_means = (
            component_weights[component]
            * cell_means[component].to_numpy(dtype="float64")
        )
        covariance = weighted_covariance(
            weighted_component_means,
            between_total,
            support_weights,
        )
        contribution = (
            covariance / between_total_variance
            if between_total_variance > EPS
            else np.nan
        )
        between_rows.append(
            {
                "domain": domain_name,
                "component": component,
                "weight": component_weights[component],
                "between_covariance_with_HKFI": covariance,
                "between_HKFI_variance": between_total_variance,
                "between_contribution_fraction": contribution,
                "between_contribution_percent": (
                    100.0 * contribution if np.isfinite(contribution) else np.nan
                ),
            }
        )
    between_df = pd.DataFrame(between_rows)
    between_df["contribution_sum_check"] = float(
        between_df["between_contribution_fraction"].sum()
    )
    return variance_df, within_df, between_df, cell_summary


# =============================================================================
# STEP 5.3 WEIGHT MANIFEST AND TERRAIN CONTEXTS
# =============================================================================
def primary_weights_from_metadata(metadata: dict[str, Any]) -> dict[str, float]:
    equation = metadata.get("equation", {})
    weights = equation.get("primary_weights")
    if isinstance(weights, dict):
        return validate_weights(weights, "Step 5.3 primary_weights")
    lambda_sm = float(equation.get("lambda_SM", 1.0 / 3.0))
    local = (1.0 - lambda_sm) / 2.0
    return validate_weights(
        {"K": local, "T": local, "S": lambda_sm},
        "Step 5.3 legacy lambda-derived primary weights",
    )


def load_named_scenarios(
    primary_weights: dict[str, float],
) -> tuple[pd.DataFrame, dict[str, dict[str, Any]]]:
    scenarios: dict[str, dict[str, Any]] = {}
    manifest_rows: list[dict[str, Any]] = []

    if STEP53_WEIGHT_MANIFEST_CSV.exists():
        manifest = pd.read_csv(STEP53_WEIGHT_MANIFEST_CSV)
        required_columns = {
            "scenario",
            "weight_K",
            "weight_T",
            "weight_S",
            "raw_raster",
            "norm_raster",
        }
        missing = required_columns - set(manifest.columns)
        if missing:
            raise ValueError(
                "Step 5.3 weight manifest is missing columns: "
                + ", ".join(sorted(missing))
            )
        for row in manifest.itertuples(index=False):
            name = str(row.scenario)
            weights = validate_weights(
                {
                    "K": float(row.weight_K),
                    "T": float(row.weight_T),
                    "S": float(row.weight_S),
                },
                f"manifest[{name}]",
            )
            purpose = str(
                getattr(row, "scenario_purpose", "Predefined sensitivity scenario")
            )
            raw_path = Path(str(row.raw_raster))
            norm_path = Path(str(row.norm_raster))
            scenarios[name] = {
                "weights": weights,
                "purpose": purpose,
                "raw_raster": raw_path,
                "norm_raster": norm_path,
                "is_primary": bool(getattr(row, "is_primary", False)),
                "use_for_downstream_spatial_CV": bool(
                    getattr(row, "use_for_downstream_spatial_CV", True)
                ),
            }
            manifest_rows.append(
                {
                    "scenario": name,
                    "scenario_purpose": purpose,
                    "weight_K": weights["K"],
                    "weight_T": weights["T"],
                    "weight_S": weights["S"],
                    "raw_raster": str(raw_path),
                    "norm_raster": str(norm_path),
                    "is_primary": scenarios[name]["is_primary"],
                    "use_for_downstream_spatial_CV": scenarios[name][
                        "use_for_downstream_spatial_CV"
                    ],
                }
            )
    else:
        log(
            "[WARN] Step 5.3 weight manifest not found; constructing expected "
            "scenario paths from the declared scenario definitions."
        )
        named_dir = STEP53_WEIGHT_DIR / "named_scenarios"
        for name, definition in EXPECTED_NAMED_SCENARIOS.items():
            weights = validate_weights(
                definition["weights"],
                f"EXPECTED_NAMED_SCENARIOS[{name}]",
            )
            if name == "primary_equal_component":
                raw_path = choose_existing(HKFI_RAW_CANDIDATES, "primary raw HKFI")
                norm_path = choose_existing(HKFI_NORM_CANDIDATES, "primary normalized HKFI")
            else:
                raw_path = named_dir / f"hkfi_{name}_raw_32645_12p5m_float32.tif"
                norm_path = named_dir / f"hkfi_{name}_norm_0_1_32645_12p5m_float32.tif"
            scenarios[name] = {
                "weights": weights,
                "purpose": definition["purpose"],
                "raw_raster": raw_path,
                "norm_raster": norm_path,
                "is_primary": name == "primary_equal_component",
                "use_for_downstream_spatial_CV": True,
            }
            manifest_rows.append(
                {
                    "scenario": name,
                    "scenario_purpose": definition["purpose"],
                    "weight_K": weights["K"],
                    "weight_T": weights["T"],
                    "weight_S": weights["S"],
                    "raw_raster": str(raw_path),
                    "norm_raster": str(norm_path),
                    "is_primary": name == "primary_equal_component",
                    "use_for_downstream_spatial_CV": True,
                }
            )

    if "primary_equal_component" not in scenarios:
        raise ValueError("Named scenarios must include primary_equal_component.")
    primary_from_manifest = scenarios["primary_equal_component"]["weights"]
    for component in ("K", "T", "S"):
        if abs(primary_from_manifest[component] - primary_weights[component]) > WEIGHT_TOL:
            raise ValueError(
                "Manifest primary weights do not match Step 5.3 metadata."
            )

    expected_names = set(EXPECTED_NAMED_SCENARIOS)
    missing_expected = expected_names - set(scenarios)
    if missing_expected:
        raise ValueError(
            "Step 5.3 named scenarios are missing: "
            + ", ".join(sorted(missing_expected))
        )

    return pd.DataFrame(manifest_rows), scenarios


def build_context_masks(
    common_domain: np.ndarray,
    irs_effective: np.ndarray,
    dv_availability: np.ndarray | None,
    slope: np.ndarray | None,
    G: np.ndarray | None,
) -> dict[str, dict[str, Any]]:
    contexts: dict[str, dict[str, Any]] = {}

    def add(name: str, group: str, mask: np.ndarray, label: str) -> None:
        count = int(mask.sum())
        if count >= MIN_DOMAIN_PIXELS:
            contexts[name] = {
                "group": group,
                "label": label,
                "mask": mask,
                "pixel_count": count,
            }

    add(
        "full_HKFI_domain",
        "analysis_domain",
        common_domain,
        "Full HKFI domain",
    )

    insar_supported = common_domain & (irs_effective > IRS_SUPPORT_TOL)
    high_confidence = common_domain & (
        irs_effective >= HIGH_CONFIDENCE_IRS_THRESHOLD
    )
    if dv_availability is not None:
        insar_supported &= dv_availability
        high_confidence &= dv_availability
    add(
        "InSAR_supported_IRS_gt_0",
        "IRS_support",
        insar_supported,
        "InSAR-supported IRS > 0",
    )
    add(
        "higher_confidence_IRS_ge_0p20",
        "IRS_support",
        high_confidence,
        "Higher-confidence IRS ≥ 0.20",
    )
    add(
        "IRS_unsupported_or_zero",
        "IRS_support",
        common_domain & ~(irs_effective > IRS_SUPPORT_TOL),
        "IRS unsupported or zero",
    )
    add(
        "IRS_low_gt_0_lt_0p20",
        "IRS_support",
        common_domain
        & (irs_effective > IRS_SUPPORT_TOL)
        & (irs_effective < HIGH_CONFIDENCE_IRS_THRESHOLD),
        "Low effective IRS (0–0.20)",
    )

    if slope is not None:
        valid_slope = common_domain & np.isfinite(slope)
        slope_classes = [
            ("slope_lt_5", -np.inf, 5.0, "Slope < 5°"),
            ("slope_5_15", 5.0, 15.0, "Slope 5–15°"),
            ("slope_15_30", 15.0, 30.0, "Slope 15–30°"),
            ("slope_30_45", 30.0, 45.0, "Slope 30–45°"),
            ("slope_ge_45", 45.0, np.inf, "Slope ≥ 45°"),
        ]
        for name, lower, upper, label in slope_classes:
            mask = valid_slope & (slope >= lower) & (slope < upper)
            add(name, "slope", mask, label)

    if G is not None:
        valid_g = common_domain & np.isfinite(G)
        g_classes = [
            ("G_lt_0p20", -np.inf, 0.20, "G < 0.20"),
            ("G_0p20_0p50", 0.20, 0.50, "0.20 ≤ G < 0.50"),
            ("G_ge_0p50", 0.50, np.inf, "G ≥ 0.50"),
        ]
        for name, lower, upper, label in g_classes:
            mask = valid_g & (G >= lower) & (G < upper)
            add(name, "geometric_observability", mask, label)

    return contexts


# =============================================================================
# DOWNSTREAM MODEL-RESULT INTEGRATION
# =============================================================================
def write_downstream_template(
    manifest_df: pd.DataFrame,
) -> None:
    rows: list[dict[str, Any]] = []
    for manifest_row in manifest_df.itertuples(index=False):
        if not bool(manifest_row.use_for_downstream_spatial_CV):
            continue
        for model in ("XGB", "RF", "LR"):
            for fold in range(1, 6):
                rows.append(
                    {
                        "scenario": manifest_row.scenario,
                        "model": model,
                        "fold": fold,
                        "auc": np.nan,
                        "precision": np.nan,
                        "recall": np.nan,
                        "f1": np.nan,
                        "accuracy": np.nan,
                        "high_area_fraction": np.nan,
                        "very_high_area_fraction": np.nan,
                        "high_zone_jaccard_vs_primary": np.nan,
                    }
                )
    pd.DataFrame(rows).to_csv(DOWNSTREAM_RESULTS_TEMPLATE_CSV, index=False)


def summarize_downstream_results(
    path: Path | None,
    primary_scenario: str = "primary_equal_component",
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    if path is None or not path.exists():
        return pd.DataFrame(), pd.DataFrame(), "PENDING"
    dataframe = pd.read_csv(path)
    required = {"scenario", "model", "fold", "auc"}
    missing = required - set(dataframe.columns)
    if missing:
        raise ValueError(
            "Downstream model-results CSV is missing columns: "
            + ", ".join(sorted(missing))
        )
    dataframe["fold"] = pd.to_numeric(dataframe["fold"], errors="coerce")
    dataframe["auc"] = pd.to_numeric(dataframe["auc"], errors="coerce")
    dataframe = dataframe.dropna(subset=["scenario", "model", "fold", "auc"])
    if dataframe.empty:
        return pd.DataFrame(), pd.DataFrame(), "PENDING_EMPTY"

    metric_columns = [
        column
        for column in [
            "auc",
            "precision",
            "recall",
            "f1",
            "accuracy",
            "high_area_fraction",
            "very_high_area_fraction",
            "high_zone_jaccard_vs_primary",
        ]
        if column in dataframe.columns
    ]
    for column in metric_columns:
        dataframe[column] = pd.to_numeric(dataframe[column], errors="coerce")

    summary_rows: list[dict[str, Any]] = []
    for (scenario, model), group in dataframe.groupby(["scenario", "model"]):
        row: dict[str, Any] = {
            "scenario": scenario,
            "model": model,
            "fold_count": int(group["fold"].nunique()),
        }
        for metric in metric_columns:
            values = group[metric].dropna().to_numpy(dtype="float64")
            row[f"{metric}_mean"] = float(np.mean(values)) if values.size else np.nan
            row[f"{metric}_std"] = float(np.std(values, ddof=1)) if values.size > 1 else np.nan
            row[f"{metric}_min"] = float(np.min(values)) if values.size else np.nan
            row[f"{metric}_max"] = float(np.max(values)) if values.size else np.nan
        summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows)

    paired_rows: list[dict[str, Any]] = []
    for model, model_df in dataframe.groupby("model"):
        primary = model_df[model_df["scenario"] == primary_scenario][
            ["fold", "auc"]
        ].rename(columns={"auc": "primary_auc"})
        for scenario, scenario_df in model_df.groupby("scenario"):
            paired = primary.merge(
                scenario_df[["fold", "auc"]].rename(columns={"auc": "scenario_auc"}),
                on="fold",
                how="inner",
            )
            if paired.empty:
                continue
            differences = paired["scenario_auc"] - paired["primary_auc"]
            paired_rows.append(
                {
                    "model": model,
                    "scenario": scenario,
                    "paired_fold_count": int(len(paired)),
                    "mean_delta_AUC_vs_primary": float(differences.mean()),
                    "std_delta_AUC_vs_primary": (
                        float(differences.std(ddof=1)) if len(differences) > 1 else np.nan
                    ),
                    "minimum_delta_AUC": float(differences.min()),
                    "maximum_delta_AUC": float(differences.max()),
                }
            )
    paired_df = pd.DataFrame(paired_rows)
    return summary_df, paired_df, "COMPLETE"


# =============================================================================
# FIGURES
# =============================================================================
def save_figure(figure, png: Path, pdf: Path, svg: Path) -> None:
    figure.savefig(png, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    figure.savefig(pdf, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    figure.savefig(svg, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()
    plt.close(figure)


def create_comment4_figure(
    contribution_df: pd.DataFrame,
    variance_df: pd.DataFrame,
    within_df: pd.DataFrame,
    named_full_df: pd.DataFrame,
) -> None:
    if not MAKE_FIGURES:
        return
    figure = plt.figure(figsize=(8.2, 6.8), facecolor="white")
    grid = figure.add_gridspec(2, 2, hspace=0.42, wspace=0.34)
    axis_a = figure.add_subplot(grid[0, 0])
    axis_b = figure.add_subplot(grid[0, 1])
    axis_c = figure.add_subplot(grid[1, 0])
    axis_d = figure.add_subplot(grid[1, 1])

    primary = contribution_df[
        contribution_df["scenario"] == "primary_equal_component"
    ]
    pivot = primary.pivot(
        index="domain",
        columns="component",
        values="contribution_percent",
    ).reindex(
        [
            "full_HKFI_domain",
            "InSAR_supported_IRS_gt_0",
            "higher_confidence_IRS_ge_0p20",
        ]
    )
    pivot.plot(kind="bar", ax=axis_a, edgecolor="black", linewidth=0.5)
    axis_a.axhline(0.0, linewidth=0.6)
    axis_a.set_ylabel("Contribution to raw HKFI variance (%)")
    axis_a.set_xlabel("")
    axis_a.set_title("(a) Component contribution by domain")
    axis_a.tick_params(axis="x", rotation=20, labelsize=7)
    axis_a.legend(title="Component", fontsize=7, title_fontsize=7)
    axis_a.grid(axis="y", alpha=0.25)

    hkfi_row = variance_df[
        (variance_df["domain"] == "full_HKFI_domain")
        & (variance_df["variable"] == "HKFI_raw")
    ].iloc[0]
    axis_b.bar(
        ["Within cells", "Between cells"],
        [
            100.0 * hkfi_row["within_fraction_of_total"],
            100.0 * hkfi_row["between_fraction_of_total"],
        ],
        edgecolor="black",
        linewidth=0.5,
    )
    axis_b.set_ylabel("Fraction of raw HKFI variance (%)")
    axis_b.set_title("(b) Support-scale variance partition")
    axis_b.grid(axis="y", alpha=0.25)

    within_selected = within_df[within_df["domain"] == "full_HKFI_domain"]
    axis_c.bar(
        within_selected["component"],
        within_selected["within_contribution_percent"],
        edgecolor="black",
        linewidth=0.5,
    )
    axis_c.axhline(0.0, linewidth=0.6)
    axis_c.set_ylabel("Within-cell contribution (%)")
    axis_c.set_title("(c) Source of within-cell differentiation")
    axis_c.grid(axis="y", alpha=0.25)

    sm_scenarios = [
        "no_SM",
        "local_emphasis",
        "primary_equal_component",
        "sm_emphasis",
    ]
    selected = named_full_df[
        named_full_df["scenario"].isin(sm_scenarios)
    ].copy()
    selected["order"] = selected["scenario"].map(
        {name: index for index, name in enumerate(sm_scenarios)}
    )
    selected = selected.sort_values("order")
    axis_d.bar(
        [pretty_scenario(name) for name in selected["scenario"]],
        selected["norm_spearman_rho"],
        edgecolor="black",
        linewidth=0.5,
    )
    axis_d.set_ylim(0.0, 1.02)
    axis_d.set_ylabel("Spearman correlation with primary HKFI")
    axis_d.set_title("(d) Soil-moisture weight sensitivity")
    axis_d.tick_params(axis="x", rotation=22, labelsize=7)
    axis_d.grid(axis="y", alpha=0.25)

    save_figure(
        figure,
        COMMENT4_FIGURE_PNG,
        COMMENT4_FIGURE_PDF,
        COMMENT4_FIGURE_SVG,
    )


def create_comment7_named_figure(named_full_df: pd.DataFrame) -> None:
    if not MAKE_FIGURES:
        return
    selected = named_full_df.copy()
    scenario_order = list(EXPECTED_NAMED_SCENARIOS)
    selected["order"] = selected["scenario"].map(
        {name: index for index, name in enumerate(scenario_order)}
    )
    selected = selected.sort_values("order")
    labels = [pretty_scenario(name) for name in selected["scenario"]]

    figure = plt.figure(figsize=(9.2, 7.2), facecolor="white")
    grid = figure.add_gridspec(2, 2, hspace=0.48, wspace=0.30)
    axes = [figure.add_subplot(grid[i, j]) for i in range(2) for j in range(2)]

    axes[0].bar(labels, selected["norm_spearman_rho"], edgecolor="black", linewidth=0.5)
    axes[0].set_ylim(0.0, 1.02)
    axes[0].set_ylabel("Spearman correlation")
    axes[0].set_title("(a) Rank agreement with equal-weight HKFI")

    axes[1].bar(
        labels,
        100.0 * selected["quintile_change_fraction"],
        edgecolor="black",
        linewidth=0.5,
    )
    axes[1].set_ylabel("Pixels changing HKFI quintile (%)")
    axes[1].set_title("(b) Quintile sensitivity")

    axes[2].bar(labels, selected["top_decile_jaccard"], edgecolor="black", linewidth=0.5)
    axes[2].set_ylim(0.0, 1.02)
    axes[2].set_ylabel("Top-decile Jaccard overlap")
    axes[2].set_title("(c) High-HKFI spatial overlap")

    axes[3].bar(labels, selected["norm_rmse"], edgecolor="black", linewidth=0.5)
    axes[3].set_ylabel("Normalized HKFI RMSE")
    axes[3].set_title("(d) Magnitude of map change")

    for axis in axes:
        axis.tick_params(axis="x", rotation=35, labelsize=7)
        axis.grid(axis="y", alpha=0.25)

    save_figure(
        figure,
        COMMENT7_FIGURE_PNG,
        COMMENT7_FIGURE_PDF,
        COMMENT7_FIGURE_SVG,
    )


def simplex_xy(weight_K: np.ndarray, weight_T: np.ndarray, weight_S: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    del weight_K
    x = weight_T + 0.5 * weight_S
    y = (math.sqrt(3.0) / 2.0) * weight_S
    return x, y


def create_simplex_figure(simplex_df: pd.DataFrame) -> None:
    if not MAKE_FIGURES or simplex_df.empty:
        return
    x, y = simplex_xy(
        simplex_df["weight_K"].to_numpy(dtype="float64"),
        simplex_df["weight_T"].to_numpy(dtype="float64"),
        simplex_df["weight_S"].to_numpy(dtype="float64"),
    )
    figure, axes = plt.subplots(1, 2, figsize=(9.0, 4.2), facecolor="white")
    scatter_a = axes[0].scatter(
        x,
        y,
        c=simplex_df["norm_spearman_rho"],
        s=45,
        edgecolors="black",
        linewidths=0.35,
    )
    scatter_b = axes[1].scatter(
        x,
        y,
        c=100.0 * simplex_df["quintile_change_fraction"],
        s=45,
        edgecolors="black",
        linewidths=0.35,
    )
    triangle_x = [0.0, 1.0, 0.5, 0.0]
    triangle_y = [0.0, 0.0, math.sqrt(3.0) / 2.0, 0.0]
    for axis in axes:
        axis.plot(triangle_x, triangle_y, linewidth=0.8)
        axis.text(-0.03, -0.04, "K=1", ha="right", va="top", fontsize=8)
        axis.text(1.03, -0.04, "T=1", ha="left", va="top", fontsize=8)
        axis.text(0.5, math.sqrt(3.0) / 2.0 + 0.04, "S=1", ha="center", fontsize=8)
        axis.set_aspect("equal")
        axis.set_xlim(-0.08, 1.08)
        axis.set_ylim(-0.08, math.sqrt(3.0) / 2.0 + 0.10)
        axis.set_xticks([])
        axis.set_yticks([])
    #axes[0].set_title("(a) Spearman agreement with equal weights")
    #axes[1].set_title("(b) Pixels changing HKFI quintile (%)")
    figure.colorbar(scatter_a, ax=axes[0], fraction=0.046, pad=0.04)
    figure.colorbar(scatter_b, ax=axes[1], fraction=0.046, pad=0.04)
    figure.tight_layout()
    save_figure(
        figure,
        SIMPLEX_FIGURE_PNG,
        SIMPLEX_FIGURE_PDF,
        SIMPLEX_FIGURE_SVG,
    )


def create_old_vs_revised_figure(revised_minus_old: np.ndarray | None) -> None:
    if not MAKE_FIGURES or revised_minus_old is None:
        return
    valid = np.isfinite(revised_minus_old)
    if not valid.any():
        return
    limit = max(
        float(np.percentile(np.abs(revised_minus_old[valid]), 98)),
        EPS,
    )
    figure, axis = plt.subplots(figsize=(5.5, 7.0), facecolor="white")
    image = axis.imshow(
        np.ma.masked_invalid(revised_minus_old),
        interpolation="nearest",
        vmin=-limit,
        vmax=limit,
    )
    axis.set_title("Revised HKFI − former interpolated-SM HKFI")
    axis.set_xticks([])
    axis.set_yticks([])
    colorbar = figure.colorbar(image, ax=axis, fraction=0.04, pad=0.03)
    colorbar.set_label("Raw HKFI difference")
    figure.savefig(
        OLD_VS_REVISED_FIGURE_PNG,
        dpi=FIG_DPI,
        bbox_inches="tight",
        facecolor="white",
    )
    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()
    plt.close(figure)


# =============================================================================
# REPORTS
# =============================================================================
def write_comment4_report(
    support_metadata: dict[str, Any],
    equation_df: pd.DataFrame,
    constancy_df: pd.DataFrame,
    contribution_df: pd.DataFrame,
    variance_df: pd.DataFrame,
    within_df: pd.DataFrame,
    named_full_df: pd.DataFrame,
    old_df: pd.DataFrame,
) -> None:
    support_mode = support_metadata.get("support_mode", "unknown")
    is_exact = bool(support_metadata.get("is_exact_native_era5_grid", False))
    primary = contribution_df[
        contribution_df["scenario"] == "primary_equal_component"
    ]
    hkfi_variance = variance_df[
        (variance_df["domain"] == "full_HKFI_domain")
        & (variance_df["variable"] == "HKFI_raw")
    ].iloc[0]
    within_full = within_df[
        within_df["domain"] == "full_HKFI_domain"
    ].set_index("component")
    constancy = constancy_df.iloc[0]

    with open(COMMENT4_REPORT_TXT, "w", encoding="utf-8") as handle:
        handle.write("REVIEWER 1, COMMENT 4 — REVISED MULTISCALE HKFI\n")
        handle.write("=" * 76 + "\n\n")
        handle.write("Equation validation\n-------------------\n")
        handle.write(equation_df.to_string(index=False))
        handle.write("\n\n")
        handle.write(
            f"Support mode: {support_mode}\n"
            f"Exact native/reconstructed ERA5 support: {is_exact}\n\n"
        )
        handle.write(
            "Within-cell soil-moisture constancy\n"
            "-----------------------------------\n"
            f"Mean within-cell SD: {constancy['mean_within_cell_std']:.6e}\n"
            f"Median within-cell SD: {constancy['median_within_cell_std']:.6e}\n"
            f"Maximum within-cell SD: {constancy['maximum_within_cell_std']:.6e}\n"
            f"Cells exceeding tolerance: {int(constancy['cells_exceeding_tolerance'])}\n\n"
        )
        handle.write("Primary covariance-aware component contributions (%)\n")
        handle.write("---------------------------------------------------\n")
        for domain in [
            "full_HKFI_domain",
            "InSAR_supported_IRS_gt_0",
            "higher_confidence_IRS_ge_0p20",
        ]:
            table = primary[primary["domain"] == domain].set_index("component")
            handle.write(domain + "\n")
            for component in ("K", "T", "S"):
                if component in table.index:
                    handle.write(
                        f"  {component}: {table.loc[component, 'contribution_percent']:.3f}%\n"
                    )
            handle.write("\n")
        handle.write(
            "Full-domain support-scale partition\n"
            "-----------------------------------\n"
            f"Within cells: {100.0 * hkfi_variance['within_fraction_of_total']:.3f}%\n"
            f"Between cells: {100.0 * hkfi_variance['between_fraction_of_total']:.3f}%\n\n"
        )
        handle.write("Full-domain within-cell contribution\n")
        handle.write("------------------------------------\n")
        for component in ("K", "T", "S"):
            if component in within_full.index:
                handle.write(
                    f"{component}: {within_full.loc[component, 'within_contribution_percent']:.6f}%\n"
                )
        handle.write("\nNamed soil-moisture-related sensitivity scenarios\n")
        handle.write("-----------------------------------------------\n")
        handle.write(
            named_full_df[
                named_full_df["scenario"].isin(
                    ["no_SM", "local_emphasis", "primary_equal_component", "sm_emphasis"]
                )
            ][
                [
                    "scenario",
                    "weight_K",
                    "weight_T",
                    "weight_S",
                    "norm_spearman_rho",
                    "quintile_change_fraction",
                    "top_decile_jaccard",
                ]
            ].to_string(index=False)
        )
        handle.write("\n\n")
        if not old_df.empty:
            handle.write("Former interpolated SM comparison\n")
            handle.write("---------------------------------\n")
            handle.write(old_df.to_string(index=False))
            handle.write("\n\n")
        if not is_exact:
            handle.write(
                "Limitation: the support geometry is approximate. Describe it as "
                "an approximate 9 km soil-moisture support grid, not exact native "
                "ERA5-Land cells.\n"
            )
        handle.write(
            "\nCovariance-aware contributions describe association with raw HKFI "
            "variance, not causal influence or predictive importance.\n"
        )


def write_comment7_report(
    primary_weights: dict[str, float],
    reference_quintile_thresholds: np.ndarray,
    named_full_df: pd.DataFrame,
    simplex_df: pd.DataFrame,
    simplex_summary_df: pd.DataFrame,
    context_df: pd.DataFrame,
    downstream_status: str,
    downstream_summary_df: pd.DataFrame,
    downstream_paired_df: pd.DataFrame,
) -> None:
    with open(COMMENT7_REPORT_TXT, "w", encoding="utf-8") as handle:
        handle.write("REVIEWER 1, COMMENT 7 — HKFI WEIGHT SENSITIVITY\n")
        handle.write("=" * 76 + "\n\n")
        handle.write(
            "Primary weights are an a priori transparent reference and are not "
            "interpreted as equal physical contribution fractions.\n"
            f"Primary weights: K={primary_weights['K']:.6f}, "
            f"T={primary_weights['T']:.6f}, S={primary_weights['S']:.6f}\n\n"
        )
        handle.write(
            "Fixed primary-reference HKFI quintile thresholds\n"
            "-----------------------------------------------\n"
            "True quintile boundaries were calculated once from the authoritative "
            "primary equal-weight normalized HKFI over the full HKFI domain and "
            "reused unchanged for every scenario.\n"
            f"Q20={reference_quintile_thresholds[0]:.8f}, "
            f"Q40={reference_quintile_thresholds[1]:.8f}, "
            f"Q60={reference_quintile_thresholds[2]:.8f}, "
            f"Q80={reference_quintile_thresholds[3]:.8f}\n\n"
        )
        handle.write("Named independent K/T/S scenarios — full domain\n")
        handle.write("------------------------------------------------\n")
        columns = [
            "scenario",
            "weight_K",
            "weight_T",
            "weight_S",
            "norm_spearman_rho",
            "norm_rmse",
            "quintile_change_fraction",
            "top_decile_jaccard",
        ]
        handle.write(named_full_df[columns].to_string(index=False))
        handle.write("\n\n")
        handle.write("Systematic simplex summary\n")
        handle.write("--------------------------\n")
        handle.write(simplex_summary_df.to_string(index=False))
        handle.write("\n\n")
        if not context_df.empty:
            handle.write("Terrain-context sensitivity summary\n")
            handle.write("-----------------------------------\n")
            context_summary = (
                context_df.groupby(["context_group", "context_name"])[
                    "norm_spearman_rho"
                ]
                .agg(["min", "median", "max"])
                .reset_index()
            )
            handle.write(context_summary.to_string(index=False))
            handle.write("\n\n")
        handle.write(
            "Downstream susceptibility-output analysis status\n"
            "------------------------------------------------\n"
            f"Status: {downstream_status}\n"
        )
        if downstream_status == "COMPLETE":
            handle.write("\nModel summary\n")
            handle.write(downstream_summary_df.to_string(index=False))
            handle.write("\n\nPaired fold AUC differences\n")
            handle.write(downstream_paired_df.to_string(index=False))
            handle.write("\n")
        else:
            handle.write(
                "The named normalized HKFI rasters and downstream manifest are "
                "ready. The final Comment 7 response remains incomplete until "
                "the unchanged spatial-CV susceptibility pipeline is rerun for "
                "the named scenarios and fold-level AUC/map outputs are supplied.\n"
            )
        handle.write(
            "\nThe simplex analysis characterizes HKFI-map sensitivity. It must not "
            "be described as a susceptibility-model sensitivity result unless the "
            "alternative rasters have been propagated through model training.\n"
        )


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=== Step 5.3b: HKFI validation ===")

    anchor = read_anchor(ANCHOR_TIF)
    study = read_mask_exact(STUDY_MASK_TIF, anchor, "study mask")
    assert study is not None
    context = read_mask_exact(
        CONTEXT_MASK_TIF,
        anchor,
        "context mask",
        required=False,
    )
    if context is None:
        context = study.copy()
        context_path_used = STUDY_MASK_TIF
    else:
        context = context & study
        context_path_used = CONTEXT_MASK_TIF

    valid_mask = read_mask_exact(
        HKFI_VALID_MASK_TIF,
        anchor,
        "HKFI valid mask",
    )
    assert valid_mask is not None
    dv_availability = read_mask_exact(
        DV_AVAILABILITY_MASK_TIF,
        anchor,
        "DV availability mask",
        required=False,
    )

    K = read_continuous_exact(K_TERM_TIF, anchor, "authoritative K component")
    T = read_continuous_exact(T_TERM_TIF, anchor, "authoritative T component")
    S = read_continuous_exact(
        S_TERM_TIF,
        anchor,
        "authoritative support-preserved S component",
    )
    H_local_existing = read_continuous_exact(
        H_LOCAL_TIF,
        anchor,
        "authoritative H_local diagnostic",
    )
    support_ids = read_cell_ids_exact(SUPPORT_CELL_ID_TIF, anchor)
    irs_effective = read_continuous_exact(
        IRS_EFFECTIVE_HKFI_TIF,
        anchor,
        "effective IRS for HKFI",
    )

    raw_path = choose_existing(HKFI_RAW_CANDIDATES, "primary raw HKFI")
    norm_path = choose_existing(HKFI_NORM_CANDIDATES, "primary normalized HKFI")
    hkfi_raw = read_continuous_exact(raw_path, anchor, "primary raw HKFI")
    hkfi_norm = read_continuous_exact(norm_path, anchor, "primary normalized HKFI")

    step53_metadata = load_json_optional(STEP53_META_JSON) or {}
    step53_qa = load_json_optional(STEP53_QA_JSON) or {}
    separate_support_metadata = load_json_optional(STEP53_SUPPORT_MODE_JSON) or {}
    primary_weights = primary_weights_from_metadata(step53_metadata)

    support_metadata = step53_metadata.get("support_metadata", {})
    if not support_metadata:
        support_metadata = separate_support_metadata.get(
            "support_metadata",
            separate_support_metadata,
        )

    common_domain = (
        study
        & context
        & valid_mask
        & np.isfinite(K)
        & np.isfinite(T)
        & np.isfinite(S)
        & np.isfinite(H_local_existing)
        & np.isfinite(hkfi_raw)
        & np.isfinite(hkfi_norm)
        & np.isfinite(irs_effective)
        & np.isfinite(support_ids)
        & (support_ids > 0)
    )
    if int(common_domain.sum()) < MIN_DOMAIN_PIXELS:
        raise RuntimeError("The common revised-HKFI domain is too small.")

    # -------------------------------------------------------------------------
    # Fixed reference quintiles for all HKFI weight-sensitivity comparisons
    # -------------------------------------------------------------------------
    # Calculate Q20/Q40/Q60/Q80 once from the authoritative primary equal-weight
    # normalized HKFI over the full common analysis domain. These boundaries are
    # reused unchanged for every named scenario, terrain-context comparison,
    # simplex scenario, and historical diagnostic comparison.
    reference_quintile_thresholds = calculate_reference_quintile_thresholds(
        hkfi_norm,
        common_domain,
    )
    quintile_threshold_df = pd.DataFrame(
        {
            "boundary": ["Q20", "Q40", "Q60", "Q80"],
            "percentile": list(REFERENCE_QUINTILE_PERCENTILES),
            "normalized_HKFI_threshold": reference_quintile_thresholds,
            "reference_scenario": "primary_equal_component",
            "reference_domain": "full_HKFI_domain",
        }
    )
    quintile_threshold_df.to_csv(
        PRIMARY_QUINTILE_THRESHOLDS_CSV,
        index=False,
    )
    log(
        "[REFERENCE QUINTILES] "
        + ", ".join(
            f"{name}={value:.6f}"
            for name, value in zip(
                ("Q20", "Q40", "Q60", "Q80"),
                reference_quintile_thresholds,
            )
        )
    )

    # Optional terrain rasters
    slope_path = choose_optional(SLOPE_TIF_CANDIDATES, "slope raster")
    G_path = choose_optional(G_TIF_CANDIDATES, "geometric observability raster")
    slope = (
        read_continuous_exact(slope_path, anchor, "slope raster")
        if slope_path is not None
        else None
    )
    G = (
        read_continuous_exact(G_path, anchor, "geometric observability raster")
        if G_path is not None
        else None
    )

    # -------------------------------------------------------------------------
    # 1. Equation reproduction and normalization QA
    # -------------------------------------------------------------------------
    H_local_reproduced = np.full(K.shape, np.nan, dtype="float64")
    H_local_reproduced[common_domain] = (
        K[common_domain] + T[common_domain]
    ) / 2.0
    hkfi_primary_reproduced = combine_hkfi_components(
        K,
        T,
        S,
        common_domain,
        primary_weights,
        "primary_weights",
    )
    hkfi_norm_reproduced, norm_metadata = robust_normalize_0_1(
        hkfi_raw,
        common_domain,
    )
    equation_rows = [
        equation_comparison_row(
            "H_local_existing_vs_(K+T)/2",
            H_local_reproduced,
            H_local_existing,
            common_domain,
        ),
        equation_comparison_row(
            "HKFI_raw_existing_vs_wK_K_plus_wT_T_plus_wS_S",
            hkfi_primary_reproduced,
            hkfi_raw,
            common_domain,
        ),
        equation_comparison_row(
            "HKFI_norm_existing_vs_recomputed_P2_P98",
            hkfi_norm_reproduced,
            hkfi_norm,
            common_domain,
        ),
    ]
    equal_primary = all(
        abs(primary_weights[name] - 1.0 / 3.0) <= WEIGHT_TOL
        for name in ("K", "T", "S")
    )
    if equal_primary:
        equal_reference = np.full(K.shape, np.nan, dtype="float64")
        equal_reference[common_domain] = (
            K[common_domain] + T[common_domain] + S[common_domain]
        ) / 3.0
        equation_rows.append(
            equation_comparison_row(
                "HKFI_raw_existing_vs_(K+T+S)/3",
                equal_reference,
                hkfi_raw,
                common_domain,
            )
        )
    equation_df = pd.DataFrame(equation_rows)
    equation_df.to_csv(EQUATION_QA_CSV, index=False)
    if not bool(equation_df["pass"].all()):
        raise RuntimeError(
            "One or more Step 5.3 equation checks failed:\n"
            + equation_df[~equation_df["pass"]].to_string(index=False)
        )

    # -------------------------------------------------------------------------
    # 2. Support-preserved soil-moisture constancy
    # -------------------------------------------------------------------------
    sm_constancy_df, sm_constancy_detail_df = support_constancy_tables(
        support_ids,
        S,
        common_domain,
    )
    sm_constancy_df.to_csv(SM_CONSTANCY_QA_CSV, index=False)
    sm_constancy_detail_df.to_csv(SM_CONSTANCY_DETAIL_CSV, index=False)
    if not bool(sm_constancy_df.iloc[0]["pass"]):
        raise RuntimeError("Support-preserved S is not constant within all cells.")

    # -------------------------------------------------------------------------
    # 3. Domains and primary component attribution
    # -------------------------------------------------------------------------
    contexts = build_context_masks(
        common_domain,
        irs_effective,
        dv_availability,
        slope,
        G,
    )
    main_domain_names = [
        name
        for name in [
            "full_HKFI_domain",
            "InSAR_supported_IRS_gt_0",
            "higher_confidence_IRS_ge_0p20",
        ]
        if name in contexts
    ]

    component_arrays = {"K": K, "T": T, "S": S, "HKFI_raw": hkfi_raw}
    statistics_rows: list[dict[str, Any]] = []
    correlation_rows: list[dict[str, Any]] = []
    primary_contribution_frames: list[pd.DataFrame] = []

    for domain_name in main_domain_names:
        domain_mask = contexts[domain_name]["mask"]
        n_domain = int(domain_mask.sum())
        for component_name, array in component_arrays.items():
            row = descriptive_stats(array[domain_mask])
            row.update(
                {
                    "domain": domain_name,
                    "component": component_name,
                    "domain_pixel_count": n_domain,
                    "fraction_of_full_domain": n_domain / max(int(common_domain.sum()), 1),
                    "pearson_with_HKFI_raw": pearson_correlation(
                        array[domain_mask],
                        hkfi_raw[domain_mask],
                    ),
                }
            )
            rho, rank_n, sampled = spearman_correlation_values(
                array[domain_mask],
                hkfi_raw[domain_mask],
                deterministic_rank_sample(
                    n_domain,
                    MAX_RANK_SAMPLE_PIXELS,
                    RANK_SAMPLE_SEED + len(statistics_rows),
                ),
            )
            row["spearman_with_HKFI_raw"] = rho
            row["spearman_rank_n"] = rank_n
            row["spearman_sampled"] = sampled
            statistics_rows.append(row)

        names = list(component_arrays)
        for first_index, first_name in enumerate(names):
            for second_name in names[first_index + 1 :]:
                rho, rank_n, sampled = spearman_correlation_values(
                    component_arrays[first_name][domain_mask],
                    component_arrays[second_name][domain_mask],
                    deterministic_rank_sample(
                        n_domain,
                        MAX_RANK_SAMPLE_PIXELS,
                        RANK_SAMPLE_SEED + len(correlation_rows),
                    ),
                )
                correlation_rows.append(
                    {
                        "domain": domain_name,
                        "first_variable": first_name,
                        "second_variable": second_name,
                        "pearson_r": pearson_correlation(
                            component_arrays[first_name][domain_mask],
                            component_arrays[second_name][domain_mask],
                        ),
                        "spearman_rho": rho,
                        "spearman_rank_n": rank_n,
                        "spearman_sampled": sampled,
                        "n": n_domain,
                    }
                )

        contribution = covariance_contributions(
            components={"K": K, "T": T, "S": S},
            weights=primary_weights,
            total=hkfi_raw,
            domain=domain_mask,
            domain_name=domain_name,
            scenario_name="primary_equal_component",
        )
        if not contribution.empty:
            primary_contribution_frames.append(contribution)

    component_stats_df = pd.DataFrame(statistics_rows)
    component_correlations_df = pd.DataFrame(correlation_rows)
    primary_contribution_df = pd.concat(
        primary_contribution_frames,
        ignore_index=True,
    )
    component_stats_df.to_csv(COMPONENT_STATS_BY_DOMAIN_CSV, index=False)
    component_correlations_df.to_csv(
        COMPONENT_CORRELATIONS_BY_DOMAIN_CSV,
        index=False,
    )
    primary_contribution_df.to_csv(CONTRIBUTIONS_BY_DOMAIN_CSV, index=False)

    # -------------------------------------------------------------------------
    # 4. Within/between support-cell analysis for primary HKFI
    # -------------------------------------------------------------------------
    variance_frames: list[pd.DataFrame] = []
    within_frames: list[pd.DataFrame] = []
    between_frames: list[pd.DataFrame] = []
    support_frames: list[pd.DataFrame] = []
    for domain_name in main_domain_names:
        domain_mask = contexts[domain_name]["mask"]
        cell_dataframe = prepare_cell_dataframe(
            support_ids,
            domain_mask,
            {"K": K, "T": T, "S": S, "HKFI_raw": hkfi_raw},
        )
        variance_part, within_part, between_part, support_part = within_between_analysis(
            dataframe=cell_dataframe,
            component_names=["K", "T", "S"],
            total_name="HKFI_raw",
            component_weights=primary_weights,
            domain_name=domain_name,
        )
        variance_frames.append(variance_part)
        within_frames.append(within_part)
        between_frames.append(between_part)
        support_frames.append(support_part)
    variance_df = pd.concat(variance_frames, ignore_index=True)
    within_df = pd.concat(within_frames, ignore_index=True)
    between_df = pd.concat(between_frames, ignore_index=True)
    support_cell_df = pd.concat(support_frames, ignore_index=True)
    variance_df.to_csv(WITHIN_BETWEEN_CSV, index=False)
    within_df.to_csv(WITHIN_CONTRIBUTIONS_CSV, index=False)
    between_df.to_csv(BETWEEN_CONTRIBUTIONS_CSV, index=False)
    support_cell_df.to_csv(SUPPORT_CELL_STATS_CSV, index=False)

    full_s_within = within_df[
        (within_df["domain"] == "full_HKFI_domain")
        & (within_df["component"] == "S")
    ]
    if full_s_within.empty:
        raise RuntimeError("Could not calculate full-domain within-cell S contribution.")
    full_s_within_fraction = float(
        full_s_within.iloc[0]["within_contribution_fraction"]
    )
    if abs(full_s_within_fraction) > 1.0e-6:
        raise RuntimeError(
            "Support-preserved S contributes non-negligibly to within-cell HKFI variance."
        )

    # -------------------------------------------------------------------------
    # 5. Named independent K/T/S weight sensitivity
    # -------------------------------------------------------------------------
    manifest_df, named_scenarios = load_named_scenarios(primary_weights)
    manifest_df.to_csv(DOWNSTREAM_MANIFEST_CSV, index=False)
    write_downstream_template(manifest_df)

    step53_named_df = (
        pd.read_csv(STEP53_NAMED_SENSITIVITY_CSV)
        if STEP53_NAMED_SENSITIVITY_CSV.exists()
        else pd.DataFrame()
    )

    named_validation_rows: list[dict[str, Any]] = []
    named_by_domain_rows: list[dict[str, Any]] = []
    named_contribution_frames: list[pd.DataFrame] = []
    scenario_cache: dict[str, dict[str, np.ndarray]] = {}

    for scenario_index, (scenario_name, scenario_info) in enumerate(named_scenarios.items()):
        weights = scenario_info["weights"]
        scenario_raw = combine_hkfi_components(
            K,
            T,
            S,
            common_domain,
            weights,
            f"named_scenario_{scenario_name}",
        )
        scenario_norm, scenario_norm_meta = robust_normalize_0_1(
            scenario_raw,
            common_domain,
        )
        scenario_cache[scenario_name] = {
            "raw": scenario_raw,
            "norm": scenario_norm,
        }

        raw_raster_path = Path(scenario_info["raw_raster"])
        norm_raster_path = Path(scenario_info["norm_raster"])
        raw_raster_exists = raw_raster_path.exists()
        norm_raster_exists = norm_raster_path.exists()
        raw_qa = None
        norm_qa = None
        if raw_raster_exists:
            observed_raw = read_continuous_exact(
                raw_raster_path,
                anchor,
                f"Step 5.3 raw scenario {scenario_name}",
            )
            raw_qa = equation_comparison_row(
                f"{scenario_name}_raw_raster_vs_recomputed",
                scenario_raw,
                observed_raw,
                common_domain,
            )
        if norm_raster_exists:
            observed_norm = read_continuous_exact(
                norm_raster_path,
                anchor,
                f"Step 5.3 normalized scenario {scenario_name}",
            )
            norm_qa = equation_comparison_row(
                f"{scenario_name}_norm_raster_vs_recomputed",
                scenario_norm,
                observed_norm,
                common_domain,
            )

        if scenario_info["use_for_downstream_spatial_CV"] and (
            not raw_raster_exists or not norm_raster_exists
        ):
            raise FileNotFoundError(
                f"Named scenario {scenario_name} is marked for downstream spatial CV "
                f"but one or both rasters are missing:\n{raw_raster_path}\n{norm_raster_path}"
            )

        named_validation_rows.append(
            {
                "scenario": scenario_name,
                "weight_K": weights["K"],
                "weight_T": weights["T"],
                "weight_S": weights["S"],
                "raw_raster": str(raw_raster_path),
                "norm_raster": str(norm_raster_path),
                "raw_raster_exists": raw_raster_exists,
                "norm_raster_exists": norm_raster_exists,
                "raw_equation_rmse": raw_qa["rmse"] if raw_qa else np.nan,
                "raw_equation_max_abs": raw_qa["maximum_absolute_difference"] if raw_qa else np.nan,
                "raw_equation_pass": raw_qa["pass"] if raw_qa else False,
                "norm_equation_rmse": norm_qa["rmse"] if norm_qa else np.nan,
                "norm_equation_max_abs": norm_qa["maximum_absolute_difference"] if norm_qa else np.nan,
                "norm_equation_pass": norm_qa["pass"] if norm_qa else False,
                "normalization_lower": scenario_norm_meta["lower"],
                "normalization_upper": scenario_norm_meta["upper"],
            }
        )

        for domain_offset, domain_name in enumerate(main_domain_names):
            domain_mask = contexts[domain_name]["mask"]
            metrics = comparison_metrics(
                hkfi_raw,
                scenario_raw,
                hkfi_norm,
                scenario_norm,
                domain_mask,
                rank_seed=(
                    RANK_SAMPLE_SEED
                    + scenario_index * 100
                    + domain_offset
                ),
                reference_quintile_thresholds=reference_quintile_thresholds,
            )
            named_by_domain_rows.append(
                {
                    "scenario": scenario_name,
                    "scenario_purpose": scenario_info["purpose"],
                    "domain": domain_name,
                    "weight_K": weights["K"],
                    "weight_T": weights["T"],
                    "weight_S": weights["S"],
                    **metrics,
                }
            )
            contribution = covariance_contributions(
                components={"K": K, "T": T, "S": S},
                weights=weights,
                total=scenario_raw,
                domain=domain_mask,
                domain_name=domain_name,
                scenario_name=scenario_name,
            )
            if not contribution.empty:
                named_contribution_frames.append(contribution)

    named_validation_df = pd.DataFrame(named_validation_rows)
    named_by_domain_df = pd.DataFrame(named_by_domain_rows)
    named_contribution_df = pd.concat(
        named_contribution_frames,
        ignore_index=True,
    )
    named_validation_df.to_csv(NAMED_WEIGHT_VALIDATION_CSV, index=False)
    named_by_domain_df.to_csv(NAMED_WEIGHT_BY_DOMAIN_CSV, index=False)
    named_contribution_df.to_csv(NAMED_WEIGHT_CONTRIBUTIONS_CSV, index=False)

    if not bool(
        named_validation_df["raw_equation_pass"].all()
        and named_validation_df["norm_equation_pass"].all()
    ):
        raise RuntimeError(
            "One or more Step 5.3 named scenario rasters do not reproduce the "
            "independent K/T/S equations."
        )

    # Cross-check selected Step 5.3 sensitivity metrics when available.
    named_crosscheck_warnings: list[str] = []
    if not step53_named_df.empty:
        recomputed_full = named_by_domain_df[
            named_by_domain_df["domain"] == "full_HKFI_domain"
        ]
        for row in recomputed_full.itertuples(index=False):
            original = step53_named_df[
                step53_named_df["scenario"].astype(str) == str(row.scenario)
            ]
            if original.empty:
                named_crosscheck_warnings.append(
                    f"Scenario {row.scenario} absent from Step 5.3 named sensitivity CSV."
                )
                continue
            original_row = original.iloc[0]
            # Step 5.3b now uses true primary-reference quintiles (Q20/Q40/Q60/Q80),
            # whereas older Step 5.3 outputs may contain equal-width 0.2 classes.
            # Therefore quintile_change_fraction is intentionally NOT cross-checked
            # against the older Step 5.3 sensitivity CSV.
            metric_mapping = {
                "norm_spearman_with_primary": "norm_spearman_rho",
                "norm_rmse_vs_primary": "norm_rmse",
                "top_decile_jaccard": "top_decile_jaccard",
            }
            for step53_column, step53b_column in metric_mapping.items():
                if step53_column not in original.columns:
                    continue
                old_value = float(original_row[step53_column])
                new_value = float(getattr(row, step53b_column))
                # Spearman may differ slightly if Step 5.3 used a different
                # rank-sampling strategy. Flag, but do not fail, that metric.
                tolerance = (
                    5.0e-4
                    if "spearman" in step53_column
                    else STEP53_METRIC_CROSSCHECK_TOL
                )
                if (
                    np.isfinite(old_value)
                    and np.isfinite(new_value)
                    and abs(old_value - new_value) > tolerance
                ):
                    named_crosscheck_warnings.append(
                        f"{row.scenario} {step53_column}: "
                        f"Step 5.3={old_value:.8f}, Step 5.3b={new_value:.8f}."
                    )

    # -------------------------------------------------------------------------
    # 6. Named sensitivity by terrain context
    # -------------------------------------------------------------------------
    context_rows: list[dict[str, Any]] = []
    for scenario_index, (scenario_name, scenario_info) in enumerate(named_scenarios.items()):
        scenario_raw = scenario_cache[scenario_name]["raw"]
        scenario_norm = scenario_cache[scenario_name]["norm"]
        for context_index, (context_name, context_info) in enumerate(contexts.items()):
            mask = context_info["mask"]
            if int(mask.sum()) < MIN_DOMAIN_PIXELS:
                continue
            metrics = comparison_metrics(
                hkfi_raw,
                scenario_raw,
                hkfi_norm,
                scenario_norm,
                mask,
                rank_seed=(
                    RANK_SAMPLE_SEED
                    + 10_000
                    + scenario_index * 100
                    + context_index
                ),
                reference_quintile_thresholds=reference_quintile_thresholds,
            )
            context_rows.append(
                {
                    "scenario": scenario_name,
                    "scenario_purpose": scenario_info["purpose"],
                    "context_name": context_name,
                    "context_group": context_info["group"],
                    "context_label": context_info["label"],
                    "context_pixel_count": context_info["pixel_count"],
                    "weight_K": scenario_info["weights"]["K"],
                    "weight_T": scenario_info["weights"]["T"],
                    "weight_S": scenario_info["weights"]["S"],
                    **metrics,
                }
            )
    context_df = pd.DataFrame(context_rows)
    context_df.to_csv(NAMED_WEIGHT_CONTEXT_CSV, index=False)

    # -------------------------------------------------------------------------
    # 7. Systematic simplex sensitivity (66 scenarios at 0.10 increment)
    # -------------------------------------------------------------------------
    simplex_rows: list[dict[str, Any]] = []
    simplex_scenarios = generate_simplex_weight_scenarios(SIMPLEX_WEIGHT_INCREMENT)
    full_mask = contexts["full_HKFI_domain"]["mask"]
    for simplex_index, (scenario_name, weights) in enumerate(simplex_scenarios):
        scenario_raw = combine_hkfi_components(
            K,
            T,
            S,
            common_domain,
            weights,
            scenario_name,
        )
        scenario_norm, normalization = robust_normalize_0_1(
            scenario_raw,
            common_domain,
        )
        metrics = comparison_metrics(
            hkfi_raw,
            scenario_raw,
            hkfi_norm,
            scenario_norm,
            full_mask,
            rank_seed=RANK_SAMPLE_SEED + 20_000 + simplex_index,
            reference_quintile_thresholds=reference_quintile_thresholds,
        )
        simplex_rows.append(
            {
                "scenario": scenario_name,
                "weight_K": weights["K"],
                "weight_T": weights["T"],
                "weight_S": weights["S"],
                **metrics,
                "normalization_lower": normalization["lower"],
                "normalization_upper": normalization["upper"],
            }
        )
    simplex_df = pd.DataFrame(simplex_rows)
    simplex_df.to_csv(SIMPLEX_RECOMPUTED_CSV, index=False)

    # Cross-check Step 5.3 simplex output when available.
    simplex_crosscheck_rows: list[dict[str, Any]] = []
    if STEP53_SIMPLEX_SENSITIVITY_CSV.exists():
        step53_simplex = pd.read_csv(STEP53_SIMPLEX_SENSITIVITY_CSV)
        for row in simplex_df.itertuples(index=False):
            original = step53_simplex[
                step53_simplex["scenario"].astype(str) == str(row.scenario)
            ]
            if original.empty:
                simplex_crosscheck_rows.append(
                    {
                        "scenario": row.scenario,
                        "status": "MISSING_IN_STEP53",
                    }
                )
                continue
            old = original.iloc[0]
            cross_row: dict[str, Any] = {
                "scenario": row.scenario,
                "status": "COMPARED",
            }
            passes = []
            # Quintile-change metrics are not cross-checked against older
            # Step 5.3 outputs because Step 5.3b now uses fixed true quintiles
            # from the primary equal-weight HKFI rather than equal-width classes.
            metric_mapping = {
                "weight_K": "weight_K",
                "weight_T": "weight_T",
                "weight_S": "weight_S",
                "norm_rmse_vs_primary": "norm_rmse",
                "top_decile_jaccard": "top_decile_jaccard",
            }
            for step53_column, step53b_column in metric_mapping.items():
                if step53_column not in original.columns:
                    continue
                difference = (
                    float(getattr(row, step53b_column))
                    - float(old[step53_column])
                )
                cross_row[f"difference_{step53_column}"] = difference
                passes.append(
                    abs(difference) <= STEP53_METRIC_CROSSCHECK_TOL
                )
            cross_row["pass"] = bool(all(passes)) if passes else False
            simplex_crosscheck_rows.append(cross_row)
    simplex_crosscheck_df = pd.DataFrame(simplex_crosscheck_rows)
    simplex_crosscheck_df.to_csv(SIMPLEX_CROSSCHECK_CSV, index=False)

    simplex_summary_rows: list[dict[str, Any]] = []
    for metric in [
        "norm_spearman_rho",
        "norm_rmse",
        "quintile_change_fraction",
        "top_decile_jaccard",
    ]:
        values = simplex_df[metric].to_numpy(dtype="float64")
        simplex_summary_rows.append(
            {
                "metric": metric,
                "scenario_count": int(len(values)),
                "minimum": float(np.nanmin(values)),
                "p05": float(np.nanpercentile(values, 5)),
                "p25": float(np.nanpercentile(values, 25)),
                "median": float(np.nanmedian(values)),
                "mean": float(np.nanmean(values)),
                "p75": float(np.nanpercentile(values, 75)),
                "p95": float(np.nanpercentile(values, 95)),
                "maximum": float(np.nanmax(values)),
            }
        )
    simplex_summary_rows.extend(
        [
            {
                "metric": "count_norm_spearman_ge_0p90",
                "scenario_count": int(len(simplex_df)),
                "minimum": int((simplex_df["norm_spearman_rho"] >= 0.90).sum()),
            },
            {
                "metric": "count_norm_spearman_ge_0p95",
                "scenario_count": int(len(simplex_df)),
                "minimum": int((simplex_df["norm_spearman_rho"] >= 0.95).sum()),
            },
        ]
    )
    simplex_summary_df = pd.DataFrame(simplex_summary_rows)
    simplex_summary_df.to_csv(SIMPLEX_SUMMARY_CSV, index=False)

    # -------------------------------------------------------------------------
    # 8. Former interpolated SM versus revised support-preserved SM
    # -------------------------------------------------------------------------
    old_rows: list[dict[str, Any]] = []
    revised_minus_old = None
    if OLD_Z_SM_TIF.exists():
        old_S = read_continuous_exact(
            OLD_Z_SM_TIF,
            anchor,
            "former interpolated z(SM)",
        )
        old_domain = common_domain & np.isfinite(old_S)
        old_raw = combine_hkfi_components(
            K,
            T,
            old_S,
            old_domain,
            primary_weights,
            "former_interpolated_SM",
        )
        old_norm, old_norm_meta = robust_normalize_0_1(old_raw, old_domain)
        old_metrics = comparison_metrics(
            hkfi_raw,
            old_raw,
            hkfi_norm,
            old_norm,
            old_domain,
            rank_seed=RANK_SAMPLE_SEED + 30_000,
            reference_quintile_thresholds=reference_quintile_thresholds,
        )
        old_constancy_summary, _ = support_constancy_tables(
            support_ids,
            old_S,
            old_domain,
        )
        old_constancy = old_constancy_summary.iloc[0]
        revised_constancy = sm_constancy_df.iloc[0]
        old_metrics.update(
            {
                "comparison": "former_interpolated_SM_vs_revised_support_preserved_SM",
                "old_SM_mean_within_cell_std": float(old_constancy["mean_within_cell_std"]),
                "old_SM_median_within_cell_std": float(old_constancy["median_within_cell_std"]),
                "old_SM_max_within_cell_std": float(old_constancy["maximum_within_cell_std"]),
                "revised_SM_mean_within_cell_std": float(revised_constancy["mean_within_cell_std"]),
                "revised_SM_median_within_cell_std": float(revised_constancy["median_within_cell_std"]),
                "revised_SM_max_within_cell_std": float(revised_constancy["maximum_within_cell_std"]),
                "old_vs_revised_SM_pearson": pearson_correlation(
                    old_S[old_domain],
                    S[old_domain],
                ),
                "old_normalization_lower": old_norm_meta["lower"],
                "old_normalization_upper": old_norm_meta["upper"],
            }
        )
        old_rows.append(old_metrics)
        revised_minus_old = np.full(K.shape, np.nan, dtype="float64")
        revised_minus_old[old_domain] = hkfi_raw[old_domain] - old_raw[old_domain]
        if WRITE_HISTORICAL_DIAGNOSTIC_RASTERS:
            write_float_tif(OLD_HKFI_RAW_TIF, old_raw, anchor, "HKFI_former_interpolated_SM_raw")
            write_float_tif(OLD_HKFI_NORM_TIF, old_norm, anchor, "HKFI_former_interpolated_SM_norm")
            write_float_tif(REVISED_MINUS_OLD_TIF, revised_minus_old, anchor, "HKFI_revised_minus_former_raw")
    else:
        old_rows.append(
            {
                "comparison": "former_interpolated_SM_vs_revised_support_preserved_SM",
                "status": "SKIPPED",
                "reason": f"Former z(SM) raster not found: {OLD_Z_SM_TIF}",
            }
        )
    old_df = pd.DataFrame(old_rows)
    old_df.to_csv(OLD_VS_REVISED_CSV, index=False)

    # -------------------------------------------------------------------------
    # 9. Downstream model-result integration
    # -------------------------------------------------------------------------
    downstream_summary_df, downstream_paired_df, downstream_status = (
        summarize_downstream_results(DOWNSTREAM_MODEL_RESULTS_CSV)
    )
    if not downstream_summary_df.empty:
        downstream_summary_df.to_csv(DOWNSTREAM_RESULTS_SUMMARY_CSV, index=False)
    else:
        pd.DataFrame().to_csv(DOWNSTREAM_RESULTS_SUMMARY_CSV, index=False)
    if not downstream_paired_df.empty:
        downstream_paired_df.to_csv(DOWNSTREAM_PAIRED_AUC_CSV, index=False)
    else:
        pd.DataFrame().to_csv(DOWNSTREAM_PAIRED_AUC_CSV, index=False)

    # -------------------------------------------------------------------------
    # 10. Closure QA
    # -------------------------------------------------------------------------
    contribution_closure = (
        primary_contribution_df.groupby(["scenario", "domain"])[
            "contribution_fraction"
        ].sum()
    )
    named_contribution_closure = (
        named_contribution_df.groupby(["scenario", "domain"])[
            "contribution_fraction"
        ].sum()
    )
    within_closure = within_df.groupby("domain")[
        "within_contribution_fraction"
    ].sum()
    between_closure = between_df.groupby("domain")[
        "between_contribution_fraction"
    ].sum()

    contribution_pass = bool(
        np.all(np.abs(contribution_closure.to_numpy() - 1.0) <= CONTRIBUTION_CLOSURE_TOL)
    )
    named_contribution_pass = bool(
        np.all(np.abs(named_contribution_closure.to_numpy() - 1.0) <= CONTRIBUTION_CLOSURE_TOL)
    )
    within_pass = bool(
        np.all(np.abs(within_closure.to_numpy() - 1.0) <= CONTRIBUTION_CLOSURE_TOL)
    )
    between_pass = bool(
        np.all(np.abs(between_closure.to_numpy() - 1.0) <= CONTRIBUTION_CLOSURE_TOL)
    )
    if not all(
        [
            contribution_pass,
            named_contribution_pass,
            within_pass,
            between_pass,
        ]
    ):
        raise RuntimeError("One or more contribution closure checks failed.")

    # -------------------------------------------------------------------------
    # 11. Figures and reports
    # -------------------------------------------------------------------------
    named_full_df = named_by_domain_df[
        named_by_domain_df["domain"] == "full_HKFI_domain"
    ].copy()
    create_comment4_figure(
        primary_contribution_df,
        variance_df,
        within_df,
        named_full_df,
    )
    create_comment7_named_figure(named_full_df)
    create_simplex_figure(simplex_df)
    create_old_vs_revised_figure(revised_minus_old)

    write_comment4_report(
        support_metadata,
        equation_df,
        sm_constancy_df,
        primary_contribution_df,
        variance_df,
        within_df,
        named_full_df,
        old_df,
    )
    write_comment7_report(
        primary_weights,
        reference_quintile_thresholds,
        named_full_df,
        simplex_df,
        simplex_summary_df,
        context_df,
        downstream_status,
        downstream_summary_df,
        downstream_paired_df,
    )

    # -------------------------------------------------------------------------
    # 12. Metadata and QA
    # -------------------------------------------------------------------------
    support_mode = support_metadata.get("support_mode", "unknown")
    is_exact_native = bool(
        support_metadata.get("is_exact_native_era5_grid", False)
    )
    warnings: list[str] = list(named_crosscheck_warnings)
    if not is_exact_native:
        warnings.append(
            "Step 5.3 used approximate support geometry. Describe the cells as "
            "approximate 9 km soil-moisture support cells, not exact native ERA5 cells."
        )
    if slope is None:
        warnings.append("Slope-context weight sensitivity was skipped because no slope raster was found.")
    if G is None:
        warnings.append("Geometric-observability-context sensitivity was skipped because no G raster was found.")
    if downstream_status != "COMPLETE":
        warnings.append(
            "Comment 7 susceptibility-output sensitivity remains pending until the "
            "named HKFI scenarios are propagated through the downstream spatial-CV models."
        )

    qa = {
        "generated_at_utc": utc_now_iso(),
        "status": "PASS" if not warnings else "PASS_WITH_WARNINGS",
        "warnings": warnings,
        "grid": {
            "epsg": anchor["epsg"],
            "width": anchor["width"],
            "height": anchor["height"],
            "cell_area_m2": anchor["cell_area_m2"],
            "exact_grid_required": REQUIRE_EXACT_GRID,
        },
        "primary_weights": primary_weights,
        "reference_quintiles": {
            "definition": (
                "Q20/Q40/Q60/Q80 calculated once from primary equal-weight "
                "normalized HKFI over the full HKFI domain and reused unchanged"
            ),
            "percentiles": list(REFERENCE_QUINTILE_PERCENTILES),
            "thresholds": reference_quintile_thresholds.tolist(),
            "output_csv": str(PRIMARY_QUINTILE_THRESHOLDS_CSV),
        },
        "equation_QA": equation_rows,
        "SM_support_constancy": sm_constancy_df.iloc[0].to_dict(),
        "support_mode": support_mode,
        "is_exact_native_era5_grid": is_exact_native,
        "domain_summary": {
            name: {
                "group": info["group"],
                "label": info["label"],
                "pixel_count": info["pixel_count"],
                "fraction_of_full_domain": (
                    info["pixel_count"] / max(int(common_domain.sum()), 1)
                ),
            }
            for name, info in contexts.items()
        },
        "high_confidence_IRS_threshold": HIGH_CONFIDENCE_IRS_THRESHOLD,
        "contribution_closure_pass": contribution_pass,
        "named_weight_contribution_closure_pass": named_contribution_pass,
        "within_contribution_closure_pass": within_pass,
        "between_contribution_closure_pass": between_pass,
        "within_cell_S_contribution_fraction_full_domain": full_s_within_fraction,
        "named_scenario_count": int(len(named_scenarios)),
        "simplex_scenario_count": int(len(simplex_df)),
        "rank_metric_sampling": {
            "maximum_pixels": MAX_RANK_SAMPLE_PIXELS,
            "seed": RANK_SAMPLE_SEED,
            "all_error_overlap_metrics_use_full_domain": True,
        },
        "downstream_susceptibility_output_status": downstream_status,
        "checks": {
            "authoritative_step53_components_read_directly": True,
            "support_grid_not_reconstructed_in_step53b": True,
            "component_analysis_before_final_normalization": True,
            "IRS_not_restandardized": True,
            "SM_constant_within_support_cells": True,
            "domain_specific_K_contribution_evaluated": True,
            "independent_K_T_S_named_sensitivity_completed": True,
            "component_ablations_completed": True,
            "simplex_weight_sensitivity_completed": True,
            "true_primary_reference_quintiles_used_for_class_change": True,
            "fixed_quintile_thresholds_reused_across_all_scenarios": True,
            "terrain_context_sensitivity_completed_for_available_inputs": True,
            "downstream_manifest_created": True,
            "downstream_model_results_analyzed": downstream_status == "COMPLETE",
        },
    }

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "step": "5.3b",
        "title": "Multiscale HKFI validation and independent K/T/S weight sensitivity",
        "equations": {
            "K": "K = IRS_HKFI * z(DV+)",
            "T": "T = z(TWI)",
            "S": "S = support-preserved z(SM_j)",
            "general_HKFI": "HKFI_raw = w_K*K + w_T*T + w_S*S",
            "weight_constraints": "w_K,w_T,w_S >= 0 and w_K+w_T+w_S=1",
            "primary": "HKFI_raw = (K+T+S)/3",
            "reference_quintiles": (
                "Q20,Q40,Q60,Q80 from the primary equal-weight normalized HKFI "
                "over the full HKFI domain; thresholds fixed for all scenarios"
            ),
            "covariance_contribution": "C_m = Cov(w_m X_m, HKFI_raw) / Var(HKFI_raw)",
        },
        "support_metadata": support_metadata,
        "primary_weights": primary_weights,
        "inputs": {
            "anchor": str(ANCHOR_TIF),
            "study_mask": str(STUDY_MASK_TIF),
            "context_mask": str(context_path_used),
            "K": str(K_TERM_TIF),
            "T": str(T_TERM_TIF),
            "S": str(S_TERM_TIF),
            "H_local": str(H_LOCAL_TIF),
            "support_cell_ids": str(SUPPORT_CELL_ID_TIF),
            "IRS_effective": str(IRS_EFFECTIVE_HKFI_TIF),
            "HKFI_raw": str(raw_path),
            "HKFI_norm": str(norm_path),
            "Step53_metadata": str(STEP53_META_JSON),
            "Step53_QA": str(STEP53_QA_JSON),
            "Step53_weight_manifest": str(STEP53_WEIGHT_MANIFEST_CSV),
            "Step53_named_sensitivity": str(STEP53_NAMED_SENSITIVITY_CSV),
            "Step53_simplex_sensitivity": str(STEP53_SIMPLEX_SENSITIVITY_CSV),
            "slope_optional": str(slope_path) if slope_path else None,
            "G_optional": str(G_path) if G_path else None,
            "former_interpolated_zSM": str(OLD_Z_SM_TIF) if OLD_Z_SM_TIF.exists() else None,
            "downstream_model_results": str(DOWNSTREAM_MODEL_RESULTS_CSV) if DOWNSTREAM_MODEL_RESULTS_CSV else None,
        },
        "outputs": {
            "comment4_tables": [
                str(EQUATION_QA_CSV),
                str(SM_CONSTANCY_QA_CSV),
                str(COMPONENT_STATS_BY_DOMAIN_CSV),
                str(CONTRIBUTIONS_BY_DOMAIN_CSV),
                str(WITHIN_BETWEEN_CSV),
                str(WITHIN_CONTRIBUTIONS_CSV),
            ],
            "comment7_tables": [
                str(PRIMARY_QUINTILE_THRESHOLDS_CSV),
                str(NAMED_WEIGHT_VALIDATION_CSV),
                str(NAMED_WEIGHT_BY_DOMAIN_CSV),
                str(NAMED_WEIGHT_CONTRIBUTIONS_CSV),
                str(NAMED_WEIGHT_CONTEXT_CSV),
                str(SIMPLEX_RECOMPUTED_CSV),
                str(SIMPLEX_SUMMARY_CSV),
                str(DOWNSTREAM_MANIFEST_CSV),
                str(DOWNSTREAM_RESULTS_TEMPLATE_CSV),
            ],
            "reports": [str(COMMENT4_REPORT_TXT), str(COMMENT7_REPORT_TXT)],
            "figures": [
                str(COMMENT4_FIGURE_PNG),
                str(COMMENT7_FIGURE_PNG),
                str(SIMPLEX_FIGURE_PNG),
            ],
            "QA": str(QA_JSON),
            "metadata": str(META_JSON),
        },
    }

    with open(QA_JSON, "w", encoding="utf-8") as handle:
        json.dump(qa, handle, indent=2, ensure_ascii=False, default=json_safe)
    with open(META_JSON, "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2, ensure_ascii=False, default=json_safe)

    log(f"[SUPPORT] mode={support_mode} | exact_native={is_exact_native}")
    log(f"[DOMAIN] full={int(common_domain.sum()):,}")
    log(f"[NAMED WEIGHTS] scenarios={len(named_scenarios)}")
    log(
        "[QUINTILES] fixed primary-reference thresholds="
        + ", ".join(f"{value:.6f}" for value in reference_quintile_thresholds)
    )
    log(f"[SIMPLEX] scenarios={len(simplex_df)}")
    log(
        "[SM CONSTANCY] maximum within-cell SD="
        f"{float(sm_constancy_df.iloc[0]['maximum_within_cell_std']):.3e}"
    )
    log(
        "[WITHIN S CONTRIBUTION] "
        f"{100.0 * full_s_within_fraction:.6f}%"
    )
    log(f"[DOWNSTREAM MODEL STATUS] {downstream_status}")
    log(f"[OUT] {COMMENT4_REPORT_TXT}")
    log(f"[OUT] {COMMENT7_REPORT_TXT}")
    log(f"[OUT] {QA_JSON}")
    log("=== Step 5.3b completed successfully ===")


def run_step_5_3b() -> None:
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 5.3b failed: {exc}")
        traceback.print_exc()
        raise


if __name__ == "__main__":
    run_step_5_3b()


# Step-5.4: Slope-Normalized Deformation Ratio (SNDR)

In [ ]:
# ============================================================================
# Step 5.4 — Slope-Normalized Deformation Ratio (SNDR)
#
#     SNDR_raw = (IRS_SNDR * DV+) / (sin(theta) + delta)
#
# Scientific corrections:
#   - Uses corrected dense IRS from Step 5.2 and its support mask.
#   - Keeps original IRS unchanged; writes IRS effective for SNDR separately.
#   - Missing DV+ or unsupported IRS is neutral evidence, not measured zero.
#   - Produces a dense ML predictor with explicit support/availability masks.
#   - Keeps zero fixed during normalization:
#         SNDR_0_1 = clip(SNDR_raw / P98_supported, 0, 1)
#   - Requires exact common-grid alignment by default.
# ============================================================================

from __future__ import annotations

import csv
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject


# =============================================================================
# PATHS
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

ANCHOR_TIF = BASE / "Results" / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
STUDY_MASK_TIF = BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
CONTEXT_MASK_TIF = BASE / "Results" / "Step-0.3-Quality_Masks" / "context_mask.tif"
DYNAMIC_MASK_TIF = BASE / "Results" / "Step-0.3-Quality_Masks" / "dynamic_mask.tif"

DVPLUS_TIF = (
    BASE / "Results" / "Step-4.2-Standardization"
    / "dvplus_winsor_mm_yr_32645_12p5m_float32.tif"
)
IRS_TIF = (
    BASE / "Results" / "Step-5.2-IRS"
    / "irs_0_1_dense_32645_12p5m_float32.tif"
)
IRS_SUPPORT_MASK_TIF = (
    BASE / "Results" / "Step-5.2-IRS"
    / "irs_observation_support_mask_32645_12p5m_byte.tif"
)
SLOPE_TIF = (
    BASE / "Results" / "Step-3.3-Downslope unit vector (ŝ)"
    / "slope_deg_32645_12p5m_float32.tif"
)

OUT_DIR = BASE / "Results" / "Step-5.4-SNDR"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SNDR_RATIO_TIF = OUT_DIR / "sndr_ratio_32645_12p5m_float32.tif"
SNDR_01_TIF = OUT_DIR / "sndr_0_1_32645_12p5m_float32.tif"
IRS_EFFECTIVE_TIF = OUT_DIR / "irs_effective_for_sndr_0_1_32645_12p5m_float32.tif"
WEIGHTED_DV_TIF = OUT_DIR / "sndr_weighted_dv_mm_yr_32645_12p5m_float32.tif"
DENOMINATOR_TIF = OUT_DIR / "sndr_sin_slope_plus_delta_32645_12p5m_float32.tif"
AMPLIFICATION_TIF = OUT_DIR / "sndr_slope_normalization_factor_32645_12p5m_float32.tif"
SNDR_SUPPORTED_ONLY_TIF = OUT_DIR / "sndr_ratio_supported_only_32645_12p5m_float32.tif"

VALID_MASK_TIF = OUT_DIR / "sndr_valid_domain_mask_32645_12p5m_byte.tif"
KINEMATIC_SUPPORT_MASK_TIF = OUT_DIR / "sndr_kinematic_support_mask_32645_12p5m_byte.tif"
DV_AVAILABILITY_MASK_TIF = OUT_DIR / "sndr_dv_availability_mask_32645_12p5m_byte.tif"
FLAT_SLOPE_MASK_TIF = OUT_DIR / "sndr_flat_slope_mask_32645_12p5m_byte.tif"
NO_EVIDENCE_MASK_TIF = OUT_DIR / "sndr_no_kinematic_evidence_mask_32645_12p5m_byte.tif"
IRS_MISMATCH_MASK_TIF = OUT_DIR / "sndr_nonzero_irs_outside_support_mask_32645_12p5m_byte.tif"

SNDR_PNG = OUT_DIR / "sndr_quicklook.png"
META_JSON = OUT_DIR / "sndr_metadata.json"
QA_JSON = OUT_DIR / "sndr_quality_assurance.json"
STATS_CSV = OUT_DIR / "sndr_component_statistics.csv"


# =============================================================================
# SETTINGS
# =============================================================================
NODATA = -9999.0
REQUIRE_EXACT_GRID = True

DELTA = 1.0e-3
SLOPE_FLAT_THRESHOLD_DEG = 5.0
SLOPE_MIN_DEG = 0.0
SLOPE_MAX_DEG = 89.9

IRS_RANGE_TOL = 1.0e-6
DV_NEGATIVE_TOL_MM_YR = 1.0e-6
NORMALIZATION_UPPER_PERCENTILE = 98.0
MIN_NORMALIZATION_PIXELS = 100

MAKE_PNG = True
PNG_DPI = 300


def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def read_anchor(path: Path) -> dict[str, Any]:
    require_file(path, "anchor raster")
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"Anchor has no CRS: {path}")
        return {
            "transform": ds.transform,
            "crs": ds.crs,
            "width": ds.width,
            "height": ds.height,
            "profile": ds.profile.copy(),
            "bounds": ds.bounds,
            "epsg": ds.crs.to_epsg(),
        }


def same_grid(ds: rasterio.io.DatasetReader, anchor: dict[str, Any]) -> bool:
    return (
        ds.crs == anchor["crs"]
        and ds.transform == anchor["transform"]
        and ds.width == anchor["width"]
        and ds.height == anchor["height"]
    )


def clean_nodata(array: np.ndarray, nodata: float | int | None) -> np.ndarray:
    out = array.astype("float64", copy=True)
    if nodata is not None:
        try:
            nd = float(nodata)
            if np.isfinite(nd):
                out[out == nd] = np.nan
        except (TypeError, ValueError):
            pass
    out[out == NODATA] = np.nan
    out[out == -32768.0] = np.nan
    out[np.abs(out) > 1.0e20] = np.nan
    return out


def read_continuous(
    path: Path,
    anchor: dict[str, Any],
    description: str,
    resampling_if_allowed: Resampling = Resampling.bilinear,
) -> np.ndarray:
    require_file(path, description)
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")
        src = clean_nodata(ds.read(1), ds.nodata)
        if same_grid(ds, anchor):
            return src
        if REQUIRE_EXACT_GRID:
            raise ValueError(
                f"{description} does not match the anchor grid. "
                f"Recreate it upstream on the common grid: {path}"
            )
        dst = np.full((anchor["height"], anchor["width"]), np.nan, dtype="float64")
        reproject(
            source=src,
            destination=dst,
            src_transform=ds.transform,
            src_crs=ds.crs,
            src_nodata=np.nan,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=np.nan,
            resampling=resampling_if_allowed,
        )
        return dst


def read_mask(
    path: Path,
    anchor: dict[str, Any],
    description: str,
    optional: bool = False,
) -> np.ndarray | None:
    if not path.exists():
        if optional:
            return None
        raise FileNotFoundError(f"Missing {description}: {path}")
    with rasterio.open(path) as ds:
        if ds.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")
        raw = ds.read(1)
        if same_grid(ds, anchor):
            values = raw.astype("float64")
            if ds.nodata is not None:
                values[raw == ds.nodata] = 0.0
            values[~np.isfinite(values)] = 0.0
            return values > 0.5
        dst = np.zeros((anchor["height"], anchor["width"]), dtype="float64")
        reproject(
            source=raw,
            destination=dst,
            src_transform=ds.transform,
            src_crs=ds.crs,
            src_nodata=ds.nodata,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=0.0,
            resampling=Resampling.nearest,
        )
        dst[~np.isfinite(dst)] = 0.0
        return dst > 0.5


def validate_irs(irs: np.ndarray, domain: np.ndarray) -> np.ndarray:
    finite = domain & np.isfinite(irs)
    low = finite & (irs < -IRS_RANGE_TOL)
    high = finite & (irs > 1.0 + IRS_RANGE_TOL)
    if low.any() or high.any():
        raise ValueError(
            "IRS contains material values outside [0,1]: "
            f"below={int(low.sum()):,}, above={int(high.sum()):,}."
        )
    out = irs.copy()
    out[finite] = np.clip(out[finite], 0.0, 1.0)
    return out


def write_float_tif(path: Path, array: np.ndarray, anchor: dict[str, Any], description: str) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff", count=1, dtype="float32", nodata=NODATA,
        compress="LZW", tiled=True, bigtiff="IF_NEEDED",
        transform=anchor["transform"], crs=anchor["crs"],
        width=anchor["width"], height=anchor["height"],
    )
    out = np.where(np.isfinite(array), array, NODATA).astype("float32")
    tmp = Path(str(path) + ".part")
    with rasterio.open(tmp, "w", **profile) as ds:
        ds.write(out, 1)
        ds.set_band_description(1, description)
    os.replace(tmp, path)
    try:
        with rasterio.open(path, "r+") as ds:
            ds.build_overviews([2, 4, 8, 16], Resampling.average)
            ds.update_tags(ns="rio_overview", resampling="average")
    except Exception:
        pass


def write_byte_tif(
    path: Path,
    mask: np.ndarray,
    anchor: dict[str, Any],
    study: np.ndarray,
    description: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff", count=1, dtype="uint8", nodata=255,
        compress="LZW", tiled=True, bigtiff="IF_NEEDED",
        transform=anchor["transform"], crs=anchor["crs"],
        width=anchor["width"], height=anchor["height"],
    )
    out = np.full(mask.shape, 255, dtype="uint8")
    out[study] = 0
    out[study & mask] = 1
    tmp = Path(str(path) + ".part")
    with rasterio.open(tmp, "w", **profile) as ds:
        ds.write(out, 1)
        ds.set_band_description(1, description)
    os.replace(tmp, path)


def detailed_stats(array: np.ndarray, mask: np.ndarray) -> dict[str, float | int]:
    values = array[mask & np.isfinite(array)].astype("float64")
    if values.size == 0:
        return {"n": 0}
    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p02": float(np.percentile(values, 2)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p98": float(np.percentile(values, 98)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "one_fraction": float(np.mean(values == 1.0)),
        "unique_values_rounded_6dp": int(np.unique(np.round(values, 6)).size),
    }


def safe_pearson(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> float | None:
    valid = mask & np.isfinite(a) & np.isfinite(b)
    if valid.sum() < 3:
        return None
    av = a[valid]
    bv = b[valid]
    if np.std(av) == 0.0 or np.std(bv) == 0.0:
        return None
    return float(np.corrcoef(av, bv)[0, 1])


def zero_preserving_normalize(
    raw: np.ndarray,
    reference_mask: np.ndarray,
    output_domain: np.ndarray,
) -> tuple[np.ndarray, float]:
    values = raw[reference_mask & np.isfinite(raw)]
    if values.size < MIN_NORMALIZATION_PIXELS:
        raise RuntimeError(
            f"Too few supported SNDR cells for normalization: {values.size:,}."
        )
    positive = values[values > 0.0]
    if positive.size == 0:
        raise RuntimeError("Supported SNDR values are all zero.")
    upper = float(np.percentile(values, NORMALIZATION_UPPER_PERCENTILE))
    if not np.isfinite(upper) or upper <= 0.0:
        upper = float(np.max(positive))
    if not np.isfinite(upper) or upper <= 0.0:
        raise RuntimeError("Could not determine a positive SNDR upper bound.")
    out = np.full_like(raw, np.nan, dtype="float64")
    out[output_domain] = np.clip(raw[output_domain] / upper, 0.0, 1.0)
    return out, upper


def save_quicklook(normalized: np.ndarray, valid_domain: np.ndarray, path: Path) -> None:
    if not MAKE_PNG:
        return
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        array = np.ma.masked_invalid(np.where(valid_domain, normalized, np.nan))
        cmap = plt.get_cmap("plasma").copy()
        cmap.set_bad(color="#e6e6e6", alpha=1.0)
        fig, ax = plt.subplots(figsize=(6, 5), dpi=PNG_DPI)
        image = ax.imshow(
            array, cmap=cmap, vmin=0.0, vmax=1.0,
            origin="upper", interpolation="nearest"
        )
        ax.set_title("Slope-Normalized Deformation Ratio (SNDR)")
        ax.axis("off")
        cbar = fig.colorbar(image, ax=ax, shrink=0.75, pad=0.02)
        cbar.set_label("SNDR (0–1)")
        fig.tight_layout()
        fig.savefig(path, dpi=PNG_DPI, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        log(f"[PNG] {path}")
    except Exception as exc:
        log(f"[WARN] Quicklook failed: {exc}")


def compute_sndr(
    dvplus: np.ndarray,
    irs: np.ndarray,
    slope_deg: np.ndarray,
    study: np.ndarray,
    context: np.ndarray,
    dynamic: np.ndarray,
    irs_support: np.ndarray,
) -> dict[str, Any]:
    base_domain = study & context & np.isfinite(slope_deg) & np.isfinite(irs)

    invalid_slope = base_domain & (
        (slope_deg < SLOPE_MIN_DEG) | (slope_deg > SLOPE_MAX_DEG)
    )
    if invalid_slope.any():
        raise ValueError(
            f"{int(invalid_slope.sum()):,} slope cells fall outside "
            f"[{SLOPE_MIN_DEG}, {SLOPE_MAX_DEG}] degrees."
        )

    dv_finite = np.isfinite(dvplus)
    dv_available = base_domain & dynamic & dv_finite
    finite_dv_outside_dynamic = base_domain & (~dynamic) & dv_finite
    missing_dv_inside_dynamic = base_domain & dynamic & (~dv_finite)

    negative_dv = dv_available & (dvplus < -DV_NEGATIVE_TOL_MM_YR)
    if negative_dv.any():
        raise ValueError(
            f"{int(negative_dv.sum()):,} materially negative DV+ cells found."
        )

    dv_nonnegative = np.full_like(dvplus, np.nan, dtype="float64")
    dv_nonnegative[dv_finite] = np.maximum(dvplus[dv_finite], 0.0)

    support_mismatch = base_domain & (~irs_support) & (irs > IRS_RANGE_TOL)
    slope_eligible = base_domain & (slope_deg >= SLOPE_FLAT_THRESHOLD_DEG)
    flat_slope = base_domain & (slope_deg < SLOPE_FLAT_THRESHOLD_DEG)
    kinematic_support = slope_eligible & dv_available & irs_support
    no_evidence = base_domain & (~kinematic_support)

    irs_effective = np.full_like(irs, np.nan, dtype="float64")
    irs_effective[base_domain] = 0.0
    irs_effective[kinematic_support] = irs[kinematic_support]

    dv_effective = np.zeros_like(dvplus, dtype="float64")
    dv_effective[kinematic_support] = dv_nonnegative[kinematic_support]

    weighted_dv = np.full_like(dvplus, np.nan, dtype="float64")
    weighted_dv[base_domain] = (
        irs_effective[base_domain] * dv_effective[base_domain]
    )

    slope_rad = np.deg2rad(slope_deg)
    denominator = np.full_like(slope_deg, np.nan, dtype="float64")
    denominator[base_domain] = np.sin(slope_rad[base_domain]) + DELTA

    amplification = np.full_like(slope_deg, np.nan, dtype="float64")
    amplification[base_domain] = 1.0 / denominator[base_domain]

    sndr_raw = np.full_like(dvplus, np.nan, dtype="float64")
    sndr_raw[base_domain] = 0.0
    sndr_raw[kinematic_support] = (
        weighted_dv[kinematic_support] / denominator[kinematic_support]
    )

    supported_only = np.full_like(sndr_raw, np.nan, dtype="float64")
    supported_only[kinematic_support] = sndr_raw[kinematic_support]

    sndr_01, upper = zero_preserving_normalize(
        sndr_raw, kinematic_support, base_domain
    )

    return {
        "base_domain": base_domain,
        "dv_available": dv_available,
        "finite_dv_outside_dynamic": finite_dv_outside_dynamic,
        "missing_dv_inside_dynamic": missing_dv_inside_dynamic,
        "flat_slope": flat_slope,
        "kinematic_support": kinematic_support,
        "no_evidence": no_evidence,
        "support_mismatch": support_mismatch,
        "irs_effective": irs_effective,
        "weighted_dv": weighted_dv,
        "denominator": denominator,
        "amplification": amplification,
        "sndr_raw": sndr_raw,
        "supported_only": supported_only,
        "sndr_01": sndr_01,
        "upper_bound": upper,
    }


def main() -> None:
    log("=== Step 5.4: SNDR ===")
    log(
        "Eq. 5: SNDR = (IRS_SNDR × DV+) / "
        f"(sin(theta) + {DELTA})"
    )

    anchor = read_anchor(ANCHOR_TIF)
    study = read_mask(STUDY_MASK_TIF, anchor, "study-area mask")
    assert study is not None

    context = read_mask(CONTEXT_MASK_TIF, anchor, "context mask", optional=True)
    if context is None:
        context = study.copy()
        context_path_used = STUDY_MASK_TIF
        log("[MASK] Context mask unavailable; study mask used.")
    else:
        context = context & study
        context_path_used = CONTEXT_MASK_TIF

    dynamic = read_mask(DYNAMIC_MASK_TIF, anchor, "dynamic mask")
    irs_support = read_mask(
        IRS_SUPPORT_MASK_TIF, anchor, "IRS observation-support mask"
    )
    assert dynamic is not None and irs_support is not None
    irs_support = irs_support & study

    dvplus = read_continuous(DVPLUS_TIF, anchor, "winsorized DV+")
    irs = read_continuous(IRS_TIF, anchor, "corrected dense IRS")
    slope = read_continuous(SLOPE_TIF, anchor, "slope")
    irs = validate_irs(irs, study & context)

    result = compute_sndr(
        dvplus, irs, slope, study, context, dynamic, irs_support
    )

    base_domain = result["base_domain"]
    dv_available = result["dv_available"]
    finite_dv_outside_dynamic = result["finite_dv_outside_dynamic"]
    missing_dv_inside_dynamic = result["missing_dv_inside_dynamic"]
    flat_slope = result["flat_slope"]
    support = result["kinematic_support"]
    no_evidence = result["no_evidence"]
    mismatch = result["support_mismatch"]
    irs_effective = result["irs_effective"]
    weighted_dv = result["weighted_dv"]
    denominator = result["denominator"]
    amplification = result["amplification"]
    sndr_raw = result["sndr_raw"]
    supported_only = result["supported_only"]
    sndr_01 = result["sndr_01"]
    upper = result["upper_bound"]

    statistics = {
        "DVplus_supported_mm_yr": detailed_stats(dvplus, support),
        "IRS_original_base_domain": detailed_stats(irs, base_domain),
        "IRS_original_supported": detailed_stats(irs, support),
        "IRS_effective_for_SNDR": detailed_stats(irs_effective, base_domain),
        "weighted_DV_supported_mm_yr": detailed_stats(weighted_dv, support),
        "slope_supported_deg": detailed_stats(slope, support),
        "denominator_supported": detailed_stats(denominator, support),
        "slope_normalization_factor_supported": detailed_stats(amplification, support),
        "SNDR_raw_dense_mm_yr": detailed_stats(sndr_raw, base_domain),
        "SNDR_raw_supported_mm_yr": detailed_stats(sndr_raw, support),
        "SNDR_0_1_dense": detailed_stats(sndr_01, base_domain),
        "SNDR_0_1_supported": detailed_stats(sndr_01, support),
    }

    correlations = {
        "SNDR_raw_vs_DVplus_supported": safe_pearson(sndr_raw, dvplus, support),
        "SNDR_raw_vs_IRS_supported": safe_pearson(sndr_raw, irs, support),
        "SNDR_raw_vs_slope_supported": safe_pearson(sndr_raw, slope, support),
        "SNDR_raw_vs_weighted_DV_supported": safe_pearson(
            sndr_raw, weighted_dv, support
        ),
    }

    base_n = int(base_domain.sum())
    support_n = int(support.sum())
    support_fraction = support_n / max(base_n, 1)
    warnings: list[str] = []

    if finite_dv_outside_dynamic.any():
        warnings.append(
            f"{int(finite_dv_outside_dynamic.sum()):,} finite DV+ cells "
            "occur outside the dynamic mask and were not used."
        )
    if missing_dv_inside_dynamic.any():
        warnings.append(
            f"{int(missing_dv_inside_dynamic.sum()):,} dynamic-mask cells "
            "lack DV+ and receive neutral dense SNDR=0."
        )
    if mismatch.any():
        warnings.append(
            f"{int(mismatch.sum()):,} cells have nonzero IRS outside the IRS "
            "support mask; effective IRS was set to zero."
        )
    if support_fraction < 0.05:
        warnings.append(
            f"Kinematic support covers only {100.0 * support_fraction:.2f}% "
            "of the dense SNDR domain."
        )

    supported_norm_stats = statistics["SNDR_0_1_supported"]
    if supported_norm_stats.get("one_fraction", 0.0) > 0.05:
        warnings.append(
            f"{100.0 * supported_norm_stats['one_fraction']:.2f}% of "
            "supported SNDR cells are clipped at 1."
        )
    if supported_norm_stats.get("unique_values_rounded_6dp", 0) < 100:
        warnings.append(
            "Supported normalized SNDR has fewer than 100 unique values at "
            "six-decimal precision."
        )

    write_float_tif(SNDR_RATIO_TIF, sndr_raw, anchor, "SNDR_raw_dense_mm_per_year")
    write_float_tif(SNDR_01_TIF, sndr_01, anchor, "SNDR_normalized_0_1_dense")
    write_float_tif(IRS_EFFECTIVE_TIF, irs_effective, anchor, "IRS_effective_for_SNDR")
    write_float_tif(WEIGHTED_DV_TIF, weighted_dv, anchor, "IRS_times_DVplus_mm_per_year")
    write_float_tif(DENOMINATOR_TIF, denominator, anchor, "sin_slope_plus_delta")
    write_float_tif(AMPLIFICATION_TIF, amplification, anchor, "one_over_sin_slope_plus_delta")
    write_float_tif(SNDR_SUPPORTED_ONLY_TIF, supported_only, anchor, "SNDR_raw_supported_only_mm_per_year")

    write_byte_tif(VALID_MASK_TIF, base_domain, anchor, study, "SNDR_dense_valid_domain")
    write_byte_tif(KINEMATIC_SUPPORT_MASK_TIF, support, anchor, study, "SNDR_kinematic_support")
    write_byte_tif(DV_AVAILABILITY_MASK_TIF, dv_available, anchor, study, "SNDR_DV_availability")
    write_byte_tif(FLAT_SLOPE_MASK_TIF, flat_slope, anchor, study, "SNDR_flat_slope_mask")
    write_byte_tif(NO_EVIDENCE_MASK_TIF, no_evidence, anchor, study, "SNDR_no_kinematic_evidence")
    write_byte_tif(IRS_MISMATCH_MASK_TIF, mismatch, anchor, study, "SNDR_nonzero_IRS_outside_support")

    with open(STATS_CSV, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["component", "statistic", "value"])
        for component, component_stats in statistics.items():
            for statistic, value in component_stats.items():
                writer.writerow([component, statistic, value])
        for name, value in correlations.items():
            writer.writerow(["correlation", name, value])

    qa = {
        "generated_at_utc": utc_now_iso(),
        "status": "PASS_WITH_WARNINGS" if warnings else "PASS",
        "warnings": warnings,
        "counts": {
            "study_pixels": int(study.sum()),
            "context_pixels": int(context.sum()),
            "base_domain_pixels": base_n,
            "dynamic_pixels": int(dynamic.sum()),
            "IRS_support_pixels": int(irs_support.sum()),
            "DV_available_pixels": int(dv_available.sum()),
            "flat_slope_pixels": int(flat_slope.sum()),
            "kinematic_support_pixels": support_n,
            "no_kinematic_evidence_pixels": int(no_evidence.sum()),
            "finite_DV_outside_dynamic_pixels": int(finite_dv_outside_dynamic.sum()),
            "missing_DV_inside_dynamic_pixels": int(missing_dv_inside_dynamic.sum()),
            "nonzero_IRS_outside_support_pixels": int(mismatch.sum()),
        },
        "fractions": {
            "kinematic_support_fraction_of_base_domain": support_fraction,
            "no_kinematic_evidence_fraction_of_base_domain": (
                int(no_evidence.sum()) / max(base_n, 1)
            ),
        },
        "normalization": {
            "lower_bound": 0.0,
            "upper_percentile": NORMALIZATION_UPPER_PERCENTILE,
            "upper_bound_raw_SNDR_mm_yr": upper,
            "reference_domain": (
                "study∩context∩dynamic∩finite(DV+)∩IRS_support∩"
                "slope>=flat_threshold"
            ),
        },
        "statistics": statistics,
        "correlations": correlations,
    }
    with open(QA_JSON, "w", encoding="utf-8") as handle:
        json.dump(qa, handle, indent=2, ensure_ascii=False)

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "anchor_epsg": anchor["epsg"],
        "equation": {
            "manuscript_equation": "Eq. 5",
            "formula": (
                "SNDR_raw = (IRS_SNDR * DVplus) / "
                "(sin(slope_radians) + delta)"
            ),
            "effective_reliability": (
                "IRS_SNDR = IRS * I_IRS_support * I_DV_available * "
                "I_slope_eligible"
            ),
            "units_raw": "mm/yr",
            "units_normalized": "dimensionless 0-1",
        },
        "interpretation": {
            "dense_zero_policy": (
                "SNDR=0 outside usable kinematic support is neutral no-"
                "reliable-kinematic-evidence for ML compatibility; it is "
                "not measured zero deformation."
            ),
            "flat_rule": (
                f"Slope < {SLOPE_FLAT_THRESHOLD_DEG} degrees receives SNDR=0."
            ),
            "normalization": (
                "Lower bound fixed at zero; upper bound is P98 of raw SNDR "
                "over supported, non-flat kinematic cells."
            ),
            "probability_warning": (
                "SNDR_0_1 is a study-relative predictor, not a probability."
            ),
        },
        "inputs": {
            "anchor": str(ANCHOR_TIF),
            "study_mask": str(STUDY_MASK_TIF),
            "context_mask_used": str(context_path_used),
            "dynamic_mask": str(DYNAMIC_MASK_TIF),
            "DVplus_winsor_mm_yr": str(DVPLUS_TIF),
            "corrected_dense_IRS": str(IRS_TIF),
            "IRS_observation_support_mask": str(IRS_SUPPORT_MASK_TIF),
            "slope_deg": str(SLOPE_TIF),
        },
        "settings": {
            "require_exact_grid": REQUIRE_EXACT_GRID,
            "delta": DELTA,
            "flat_threshold_deg": SLOPE_FLAT_THRESHOLD_DEG,
            "slope_valid_range_deg": [SLOPE_MIN_DEG, SLOPE_MAX_DEG],
            "normalization_upper_percentile": NORMALIZATION_UPPER_PERCENTILE,
            "nodata": NODATA,
        },
        "domains": {
            "dense_output_domain": "study∩context∩finite(slope)∩finite(IRS)",
            "kinematic_support_domain": (
                "dense_domain∩dynamic∩finite(DV+)∩IRS_support∩"
                "slope>=flat_threshold"
            ),
        },
        "outputs": {
            "SNDR_raw_dense": str(SNDR_RATIO_TIF),
            "SNDR_0_1_dense": str(SNDR_01_TIF),
            "IRS_effective_for_SNDR": str(IRS_EFFECTIVE_TIF),
            "weighted_DV": str(WEIGHTED_DV_TIF),
            "denominator": str(DENOMINATOR_TIF),
            "slope_normalization_factor": str(AMPLIFICATION_TIF),
            "SNDR_supported_only": str(SNDR_SUPPORTED_ONLY_TIF),
            "valid_domain_mask": str(VALID_MASK_TIF),
            "kinematic_support_mask": str(KINEMATIC_SUPPORT_MASK_TIF),
            "DV_availability_mask": str(DV_AVAILABILITY_MASK_TIF),
            "flat_slope_mask": str(FLAT_SLOPE_MASK_TIF),
            "no_kinematic_evidence_mask": str(NO_EVIDENCE_MASK_TIF),
            "IRS_support_mismatch_mask": str(IRS_MISMATCH_MASK_TIF),
            "quicklook": str(SNDR_PNG),
            "metadata": str(META_JSON),
            "quality_assurance": str(QA_JSON),
            "component_statistics": str(STATS_CSV),
        },
    }
    with open(META_JSON, "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2, ensure_ascii=False)

    save_quicklook(sndr_01, base_domain, SNDR_PNG)

    log(
        f"[DOMAIN] dense={base_n:,} | support={support_n:,} "
        f"({100.0 * support_fraction:.2f}%)"
    )
    log(
        f"[NORMALIZATION] lower=0 | P{NORMALIZATION_UPPER_PERCENTILE:g}="
        f"{upper:.6g} mm/yr"
    )
    log(f"[OUT] {SNDR_RATIO_TIF}")
    log(f"[OUT] {SNDR_01_TIF}")
    log(f"[QA] {QA_JSON}")
    log(f"[META] {META_JSON}")
    for warning in warnings:
        log(f"[QA WARNING] {warning}")
    log("=== Step 5.4 completed ===")


if __name__ == "__main__":
    main()


# Step 5.4 — SNDR Visualization -Corrected

In [ ]:
# =========================================================
# Step 5.4 — SNDR Visualization 
# =========================================================
#
# Final recommended visualization:
#   1. Uses the revised Step 5.4 v2 SNDR raster directly.
#   2. Uses NO RBF, cubic, IDW, resampling surface, or scatter reconstruction.
#   3. Uses the established final DV+/HKFI manuscript plotting style:
#        - viridis colour map
#        - 2.5 × 5 inch map; 2.5 × 2.5 inch histogram
#        - 2% buffered display extent
#        - four longitude DMS labels, rotated 45°
#        - three latitude DMS labels, rotated 90°
#        - bottom-most latitude label anchored above its tick
#        - no "Latitude" / "Longitude" axis titles
#        - north arrow
#        - checkerboard scale bar
#        - highway and study-area overlays
#        - compact legend and side colour bar
#        - PNG, PDF, and SVG outputs at 900 dpi
#
# The same valid raster cells are used for:
#   - the map
#   - the histogram
#   - descriptive interval counts
#   - summary statistics
# =========================================================

from __future__ import annotations
import os

import csv
import json
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_geom

import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LinearLocator
from matplotlib.patches import Polygon, Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False


# =========================================================
# PATHS
# =========================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
CONTEXT_MASK_TIF = (
    BASE / "Results" / "Step-0.3-Quality_Masks" / "context_mask.tif"
)
STUDY_SHP = BASE / "Study_Area.shp"
ROAD_SHP = BASE / "NTD_Road.shp"

SNDR_DIR = BASE / "Results" / "Step-5.4-SNDR"

# Corrected Step 5.4 outputs
SNDR_NORM_TIF = (
    SNDR_DIR / "sndr_0_1_32645_12p5m_float32.tif"
)
SNDR_VALID_MASK_TIF = (
    SNDR_DIR / "sndr_valid_domain_mask_32645_12p5m_byte.tif"
)
SNDR_KINEMATIC_SUPPORT_MASK_TIF = (
    SNDR_DIR / "sndr_kinematic_support_mask_32645_12p5m_byte.tif"
)
SNDR_NO_EVIDENCE_MASK_TIF = (
    SNDR_DIR / "sndr_no_kinematic_evidence_mask_32645_12p5m_byte.tif"
)
SNDR_MISMATCH_MASK_TIF = (
    SNDR_DIR
    / "sndr_nonzero_irs_outside_support_mask_32645_12p5m_byte.tif"
)
SNDR_META_JSON = SNDR_DIR / "sndr_metadata.json"
SNDR_QA_JSON = SNDR_DIR / "sndr_quality_assurance.json"

# Visualization outputs
OUT_MAP_PNG = SNDR_DIR / "SNDR Map.png"
OUT_MAP_PDF = SNDR_DIR / "SNDR Map.pdf"
OUT_MAP_SVG = SNDR_DIR / "SNDR Map.svg"
OUT_HIST_PNG = SNDR_DIR / "SNDR Histogram.png"
OUT_HIST_PDF = SNDR_DIR / "SNDR Histogram.pdf"
OUT_HIST_SVG = SNDR_DIR / "SNDR Histogram.svg"
OUT_SUMMARY_JSON = SNDR_DIR / "sndr_visualization_summary.json"
OUT_STATS_CSV = SNDR_DIR / "sndr_visualization_statistics.csv"

SNDR_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# DISPLAY SETTINGS 
# =========================================================
MAP_FIGSIZE = (2.5, 5)
HIST_FIGSIZE = (2.5, 2.5)

CAX_SIZE = "5%"
CAX_PAD = 0.01
FIG_DPI = 900

# Exact colormap used by the revised Step 5.4 analytical plot
SNDR_CMAP = plt.get_cmap("viridis")


SNDR_INFO = {
    "title": "Slope-Normalized Deformation Ratio",
    "units": "SNDR (0–1)",
}

# Scientifically correct display of the original raster.
RASTER_DISPLAY_INTERPOLATION = "nearest"

# Same old display buffer.
DISPLAY_BUFFER_FRACTION = 0.02

# Same DMS tick configuration as the final DV+/HKFI maps.
# LinearLocator includes the western/eastern and southern/northern limits.
LONGITUDE_DMS_TICK_COUNT = 4
LATITUDE_DMS_TICK_COUNT = 3
DMS_TICK_FONTSIZE = 6.2

# Descriptive intervals retained from the earlier visualization.
# These are not calibrated probability thresholds.
LOW_MAX = 0.30
MODERATE_MAX = 0.60

SHOW_FIGURES = True
SAVE_FIGURES = True

# Optional diagnostic overlay; disabled for the main manuscript map.
SHOW_MISMATCH_OVERLAY = False
MISMATCH_MARKER_SIZE = 1.2

# Font convention
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = [
    "Times New Roman",
    "DejaVu Serif",
]
plt.rcParams["axes.unicode_minus"] = False


# =========================================================
# GENERAL HELPERS
# =========================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json_optional(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None

    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return None


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


# =========================================================
# RASTER HELPERS
# =========================================================
def same_grid(
    dataset: rasterio.io.DatasetReader,
    reference: dict[str, Any],
) -> bool:
    return (
        dataset.crs == reference["crs"]
        and dataset.transform == reference["transform"]
        and dataset.width == reference["width"]
        and dataset.height == reference["height"]
    )


def clean_float_nodata(
    array: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    output = array.astype("float64", copy=True)

    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if np.isfinite(nodata_value):
                output[output == nodata_value] = np.nan
        except (TypeError, ValueError):
            pass

    output[output == -9999.0] = np.nan
    output[output == -32768.0] = np.nan
    output[np.abs(output) > 1.0e20] = np.nan
    return output


def read_sndr_raster(
    path: Path,
) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(path, "revised normalized SNDR raster")

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"SNDR raster has no CRS: {path}")

        raw = dataset.read(1)
        sndr = clean_float_nodata(raw, dataset.nodata)

        reference = {
            "crs": dataset.crs,
            "transform": dataset.transform,
            "width": dataset.width,
            "height": dataset.height,
            "bounds": dataset.bounds,
            "epsg": dataset.crs.to_epsg(),
            "resolution_x": abs(dataset.transform.a),
            "resolution_y": abs(dataset.transform.e),
        }

    outside_range = (
        np.isfinite(sndr)
        & ((sndr < -1.0e-6) | (sndr > 1.0 + 1.0e-6))
    )

    if outside_range.any():
        raise ValueError(
            f"SNDR contains {int(outside_range.sum()):,} values outside "
            "[0,1]. Recheck Step 5.4 normalization."
        )

    # Floating-point safeguard only; this is not renormalization.
    sndr = np.where(
        np.isfinite(sndr),
        np.clip(sndr, 0.0, 1.0),
        np.nan,
    )

    return sndr, reference


def read_mask_to_reference(
    path: Path,
    reference: dict[str, Any],
    description: str,
    optional: bool = False,
) -> np.ndarray | None:
    if not path.exists():
        if optional:
            return None
        raise FileNotFoundError(f"Missing {description}: {path}")

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        raw = dataset.read(1)
        nodata = dataset.nodata

        if same_grid(dataset, reference):
            values = raw.astype("float64")

            if nodata is not None:
                values[raw == nodata] = 0.0

            values[~np.isfinite(values)] = 0.0
            return values > 0.5

        destination = np.zeros(
            (reference["height"], reference["width"]),
            dtype="float64",
        )

        reproject(
            source=raw,
            destination=destination,
            src_transform=dataset.transform,
            src_crs=dataset.crs,
            src_nodata=nodata,
            dst_transform=reference["transform"],
            dst_crs=reference["crs"],
            dst_nodata=0.0,
            resampling=Resampling.nearest,
        )

        destination[~np.isfinite(destination)] = 0.0
        return destination > 0.5


# =========================================================
# CARTOGRAPHIC HELPERS 
# =========================================================
def to_dms(value: float, is_lon: bool = True) -> str:
    absolute = abs(value)
    degrees = int(absolute)

    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_lon
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def add_north_arrow_tri(
    axis,
    size: float = 0.09,
    pad: float = 0.07,
    label: str = "N",
    arrow_downshift: float = 0.025,
    label_upshift: float = 0.0,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (
        (1.0 - pad)
        + size * 0.10
        + label_upshift
    )

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base
        if candidate >= target_metres:
            return candidate

    return 10 * base


def add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    crs_type: str = "wgs84",
    mid_lat_deg: float | None = None,
    segments: int = 4,
    fontsize: float = 5.6,
    frame_lw: float = 0.4,
) -> None:
    width_data = east - west
    height_data = north - south

    if crs_type == "utm":
        width_metres = width_data
        data_units_per_metre = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = 0.5 * (south + north)

        metres_per_degree = metres_per_degree_lon(mid_lat_deg)
        width_metres = width_data * metres_per_degree
        data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = (
        total_length_metres * data_units_per_metre
    )
    segment_length_units = (
        segment_length_metres * data_units_per_metre
    )

    x_start = (
        west
        + 0.5 * width_data
        - 0.5 * total_length_units
    )
    y_base = south + 0.06 * height_data
    bar_height = 0.018 * height_data

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_data

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (
                    x_start + index * segment_length_units,
                    y_base,
                ),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=(
                    "black" if index % 2 == 0 else "white"
                ),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_data,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
        color="black",
        alpha=1.0,
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_data,
        "0",
        ha="center",
        va="top",
        fontsize=5,
        fontweight="bold",
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_data,
        end_label,
        ha="center",
        va="top",
        fontsize=5,
        fontweight="bold",
        zorder=15,
    )


# =========================================================
# ROAD AND BOUNDARY OVERLAYS
# =========================================================
def load_roads(
    road_path: Path,
    target_crs,
):
    if not road_path.exists():
        log(f"[WARN] Road shapefile not found: {road_path}")
        return None

    target_crs_text = str(target_crs)

    if HAVE_GPD:
        try:
            roads = gpd.read_file(road_path)

            if roads.crs is None:
                log(
                    "[WARN] Road CRS is missing; assuming EPSG:4326."
                )
                roads = roads.set_crs(epsg=4326)

            roads = roads.to_crs(target_crs_text)
            return roads

        except Exception as exc:
            log(f"[WARN] GeoPandas road loading failed: {exc}")

    if HAVE_FIONA:
        try:
            geometries = []

            with fiona.open(road_path, "r") as source:
                source_crs = source.crs

                for feature in source:
                    geometry = feature["geometry"]
                    if geometry is None:
                        continue

                    geometries.append(
                        transform_geom(
                            source_crs,
                            target_crs_text,
                            geometry,
                        )
                    )

            return {"geometries": geometries}

        except Exception as exc:
            log(f"[WARN] Fiona road loading failed: {exc}")

    return None


def plot_roads(
    axis,
    roads,
    colour: str = "red",
    linewidth: float = 0.7,
    alpha: float = 0.9,
    zorder: int = 850,
) -> None:
    if roads is None:
        return

    try:
        if HAVE_GPD and hasattr(roads, "plot"):
            roads.plot(
                ax=axis,
                color=colour,
                linewidth=linewidth,
                alpha=alpha,
                zorder=zorder,
            )
            return

        if isinstance(roads, dict):
            for geometry in roads["geometries"]:
                if geometry is None:
                    continue

                if geometry["type"] == "LineString":
                    line_groups = [geometry["coordinates"]]
                elif geometry["type"] == "MultiLineString":
                    line_groups = geometry["coordinates"]
                else:
                    continue

                for line in line_groups:
                    axis.plot(
                        [point[0] for point in line],
                        [point[1] for point in line],
                        color=colour,
                        linewidth=linewidth,
                        alpha=alpha,
                        zorder=zorder,
                    )

    except Exception as exc:
        log(f"[WARN] Road plotting failed: {exc}")


def draw_study_boundary(
    axis,
    study_mask: np.ndarray,
    reference: dict[str, Any],
) -> None:
    bounds = reference["bounds"]
    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    axis.contour(
        study_mask.astype("uint8"),
        levels=[0.5],
        colors="black",
        linewidths=0.7,
        extent=extent,
        origin="upper",
        zorder=999,
    )


def mismatch_pixel_centres(
    mask: np.ndarray,
    transform,
) -> tuple[np.ndarray, np.ndarray]:
    rows, columns = np.where(mask)

    x = (
        transform.c
        + (columns + 0.5) * transform.a
        + (rows + 0.5) * transform.b
    )
    y = (
        transform.f
        + (columns + 0.5) * transform.d
        + (rows + 0.5) * transform.e
    )

    return np.asarray(x), np.asarray(y)


# =========================================================
# STATISTICS
# =========================================================
def calculate_statistics(
    values: np.ndarray,
) -> dict[str, float | int]:
    values = values[np.isfinite(values)].astype("float64")

    if values.size == 0:
        return {"n": 0}

    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "zero_fraction": float(np.mean(values == 0.0)),
        "one_fraction": float(np.mean(values == 1.0)),
        "unique_values_rounded_6dp": int(
            np.unique(np.round(values, 6)).size
        ),
    }


def calculate_descriptive_intervals(
    values: np.ndarray,
) -> dict[str, dict[str, float | int]]:
    total = int(values.size)

    definitions = {
        "low_0_to_lt_0.3": (
            (values >= 0.0) & (values < LOW_MAX)
        ),
        "moderate_0.3_to_lt_0.6": (
            (values >= LOW_MAX) & (values < MODERATE_MAX)
        ),
        "high_0.6_to_1.0": (
            (values >= MODERATE_MAX) & (values <= 1.0)
        ),
    }

    output: dict[str, dict[str, float | int]] = {}

    for name, condition in definitions.items():
        count = int(condition.sum())

        output[name] = {
            "count": count,
            "percentage": 100.0 * count / max(total, 1),
        }

    return output


# =========================================================
# MAP
# =========================================================
def create_sndr_map(
    sndr: np.ndarray,
    valid_mask: np.ndarray,
    study_mask: np.ndarray,
    mismatch_mask: np.ndarray | None,
    reference: dict[str, Any],
) -> None:
    bounds = reference["bounds"]

    raster_extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    buffer_x = (
        bounds.right - bounds.left
    ) * DISPLAY_BUFFER_FRACTION
    buffer_y = (
        bounds.top - bounds.bottom
    ) * DISPLAY_BUFFER_FRACTION

    west = bounds.left - buffer_x
    east = bounds.right + buffer_x
    south = bounds.bottom - buffer_y
    north = bounds.top + buffer_y

    display_array = np.ma.masked_invalid(
        np.where(valid_mask, sndr, np.nan)
    )

    raster_crs = reference["crs"]
    crs_is_geographic = bool(
        raster_crs and raster_crs.is_geographic
    )

    x_mid = 0.5 * (bounds.left + bounds.right)
    y_mid = 0.5 * (bounds.bottom + bounds.top)

    if crs_is_geographic:
        def format_longitude(value, position):
            return to_dms(value, True)

        def format_latitude(value, position):
            return to_dms(value, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError(
                "Projected SNDR raster requires pyproj for DMS labels."
            )

        to_geographic = Transformer.from_crs(
            raster_crs,
            "EPSG:4326",
            always_xy=True,
        )

        def format_longitude(value, position):
            longitude, _ = to_geographic.transform(
                value,
                y_mid,
            )
            return to_dms(longitude, True)

        def format_latitude(value, position):
            _, latitude = to_geographic.transform(
                x_mid,
                value,
            )
            return to_dms(latitude, False)

    roads = load_roads(ROAD_SHP, raster_crs)

    figure, axis = plt.subplots(
        1,
        1,
        figsize=MAP_FIGSIZE,
        facecolor="white",
    )
    axis.set_facecolor("white")

    # Scientifically authoritative plot: original revised raster.
    image = axis.imshow(
        display_array,
        cmap=SNDR_CMAP,
        extent=raster_extent,
        vmin=0.0,
        vmax=1.0,
        origin="upper",
        interpolation=RASTER_DISPLAY_INTERPOLATION,
        alpha=1.0,
        zorder=1,
    )

    axis.set_xlim(west, east)
    axis.set_ylim(south, north)

    plot_roads(
        axis,
        roads,
        colour="red",
        linewidth=0.7,
        alpha=0.9,
        zorder=850,
    )
    draw_study_boundary(
        axis,
        study_mask,
        reference,
    )

    if (
        SHOW_MISMATCH_OVERLAY
        and mismatch_mask is not None
        and mismatch_mask.any()
    ):
        mismatch_x, mismatch_y = mismatch_pixel_centres(
            mismatch_mask,
            reference["transform"],
        )

        axis.scatter(
            mismatch_x,
            mismatch_y,
            s=MISMATCH_MARKER_SIZE,
            marker="x",
            color="white",
            linewidths=0.3,
            zorder=900,
        )

    # No "Latitude" / "Longitude" axis titles.
    # This matches the final DV+/HKFI manuscript map style.

    axis.xaxis.set_major_locator(
        LinearLocator(LONGITUDE_DMS_TICK_COUNT)
    )

    # Fixed three latitude labels for both geographic and projected rasters.
    axis.yaxis.set_major_locator(
        LinearLocator(LATITUDE_DMS_TICK_COUNT)
    )

    axis.xaxis.set_major_formatter(
        FuncFormatter(format_longitude)
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(format_latitude)
    )

    axis.tick_params(
        axis="x",
        which="major",
        labelsize=DMS_TICK_FONTSIZE,
        width=0.8,
        length=2.5,
    )
    axis.tick_params(
        axis="y",
        which="major",
        labelsize=DMS_TICK_FONTSIZE,
        width=0.8,
        length=2.5,
    )

    for tick_label in axis.get_xticklabels():
        tick_label.set_rotation(45)
        tick_label.set_ha("right")
        tick_label.set_fontsize(DMS_TICK_FONTSIZE)
        tick_label.set_fontweight("bold")

    y_tick_labels = axis.get_yticklabels()
    for tick_label in y_tick_labels:
        tick_label.set_fontsize(DMS_TICK_FONTSIZE)
        tick_label.set_fontweight("bold")
        tick_label.set_rotation(90)
        tick_label.set_va("center")

    # Lift the bottom-most latitude label above its tick so it does not
    # collide with the first 45°-rotated longitude label.
    if y_tick_labels:
        y_tick_labels[0].set_va("bottom")
        log(
            "[ADJUST] Bottom latitude DMS label anchored above its "
            "tick (va='bottom') to avoid corner overlap with the "
            "first longitude label."
        )

    add_north_arrow_tri(
        axis,
        size=0.09,
        pad=0.09,
    )

    crs_type = (
        "wgs84" if crs_is_geographic else "utm"
    )

    add_checker_scale_bar(
        axis,
        west,
        east,
        south,
        north,
        crs_type=crs_type,
        mid_lat_deg=(
            y_mid if crs_is_geographic else None
        ),
        segments=4,
        fontsize=5.6,
    )

    legend_elements = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=1.0,
            label="Road",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=1.0,
            label="Study Area",
        ),
    ]

    legend = axis.legend(
        handles=legend_elements,
        loc="lower right",
        bbox_to_anchor=(0.98, 0.09),
        bbox_transform=axis.transAxes,
        frameon=True,
        framealpha=0.9,
        fontsize=3.1,
        labelspacing=0.3,
        borderpad=0.3,
        handlelength=1.2,
        handletextpad=0.4,
        borderaxespad=0.0,
    )
    legend.get_frame().set_edgecolor("lightgray")
    legend.get_frame().set_linewidth(0.3)
    for txt in legend.get_texts():
        txt.set_fontweight("bold")

    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes(
        "right",
        size=CAX_SIZE,
        pad=CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(
        labelsize=4.4,
        width=0.6,
        length=2,
    )
    for label in colorbar.ax.get_yticklabels():
        label.set_fontweight("bold")
    colorbar.set_label(
        SNDR_INFO["units"],
        fontsize=5.2,
        fontweight="bold",
    )
    colorbar.set_ticks(
        np.linspace(0.0, 1.0, 5)
    )

    figure.tight_layout()

    if SAVE_FIGURES:
        figure.savefig(
            OUT_MAP_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_MAP_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_MAP_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        
        log(f"[MAP] {OUT_MAP_PNG}")
        log(f"[MAP] {OUT_MAP_PDF}")
        log(f"[MAP] {OUT_MAP_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# HISTOGRAM
# =========================================================
def create_sndr_histogram(
    values: np.ndarray,
    statistics: dict[str, float | int],
) -> None:
    figure, axis = plt.subplots(
        1,
        1,
        figsize=HIST_FIGSIZE,
        facecolor="white",
    )

    bins = np.linspace(0.0, 1.0, 21)
    counts, edges = np.histogram(values, bins=bins)
    centres = 0.5 * (edges[:-1] + edges[1:])
    colours = SNDR_CMAP(centres)

    axis.bar(
        centres,
        counts,
        width=(bins[1] - bins[0]) * 0.9,
        color=colours,
        edgecolor="black",
        linewidth=0.4,
        alpha=0.9,
    )

    axis.axvline(
        statistics["mean"],
        color="darkred",
        linestyle="--",
        linewidth=0.7,
        label=f"Mean: {statistics['mean']:.3f}",
    )
    axis.axvline(
        statistics["median"],
        color="orange",
        linestyle="--",
        linewidth=0.7,
        label=f"Median: {statistics['median']:.3f}",
    )
    axis.axvline(
        statistics["p95"],
        color="green",
        linestyle="--",
        linewidth=0.7,
        label=f"95th %ile: {statistics['p95']:.3f}",
    )

    axis.legend(
        fontsize=5,
        loc="upper right",
        framealpha=0.9,
    )
    axis.set_xlabel(
        SNDR_INFO["units"],
        fontsize=7,
        fontweight="bold",
    )
    axis.set_ylabel(
        "Number of Pixels",
        fontsize=7,
        fontweight="bold",
    )
    axis.tick_params(
        axis="both",
        which="major",
        labelsize=6,
        width=0.8,
        length=3,
    )
    for tick_label in axis.get_xticklabels() + axis.get_yticklabels():
        tick_label.set_fontweight("bold")
    axis.set_xlim(0.0, 1.0)
    axis.grid(
        True,
        alpha=0.25,
        linestyle="-",
        linewidth=0.4,
        axis="y",
    )
    axis.set_axisbelow(True)

    figure.tight_layout(pad=1.2)

    if SAVE_FIGURES:
        figure.savefig(
            OUT_HIST_PNG,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_HIST_PDF,
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )
        figure.savefig(
            OUT_HIST_SVG,
            format="svg",
            dpi=FIG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )

        log(f"[HISTOGRAM] {OUT_HIST_PNG}")
        log(f"[HISTOGRAM] {OUT_HIST_PDF}")
        log(f"[HISTOGRAM] {OUT_HIST_SVG}")

    if SHOW_FIGURES:
        if HAVE_IPYTHON_DISPLAY:
            display(figure)
        else:
            plt.show()

    plt.close(figure)


# =========================================================
# OUTPUT SUMMARY
# =========================================================
def write_statistics_csv(
    path: Path,
    statistics: dict[str, float | int],
    intervals: dict[str, dict[str, float | int]],
    coverage: dict[str, float | int],
    diagnostics: dict[str, float | int | None],
) -> None:
    rows: list[tuple[str, Any]] = []

    for key, value in coverage.items():
        rows.append((f"coverage_{key}", value))

    for key, value in statistics.items():
        rows.append((f"SNDR_{key}", value))

    for interval_name, interval_data in intervals.items():
        rows.append(
            (
                f"{interval_name}_count",
                interval_data["count"],
            )
        )
        rows.append(
            (
                f"{interval_name}_percentage",
                interval_data["percentage"],
            )
        )

    for key, value in diagnostics.items():
        rows.append((f"diagnostic_{key}", value))

    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["metric", "value"])
        writer.writerows(rows)


# =========================================================
# MAIN
# =========================================================
def main() -> None:
    log("=== Step 5.4 visualization: revised scientific SNDR raster ===")

    sndr, reference = read_sndr_raster(SNDR_NORM_TIF)

    log(f"[INPUT] Step 5.4 raster: {SNDR_NORM_TIF}")
    log(
        "[PLOT] direct raster=True | cmap=viridis | "
        "origin=upper | interpolation=nearest"
    )

    study_mask = read_mask_to_reference(
        STUDY_MASK_TIF,
        reference,
        "study-area mask",
    )
    assert study_mask is not None

    context_mask = read_mask_to_reference(
        CONTEXT_MASK_TIF,
        reference,
        "context mask",
        optional=True,
    )

    if context_mask is None:
        context_mask = study_mask.copy()
        context_path_used = STUDY_MASK_TIF
        log("[MASK] Context mask unavailable; study mask used.")
    else:
        context_mask = context_mask & study_mask
        context_path_used = CONTEXT_MASK_TIF

    revised_valid_mask = read_mask_to_reference(
        SNDR_VALID_MASK_TIF,
        reference,
        "revised SNDR valid mask",
        optional=True,
    )

    finite_sndr = (
        study_mask
        & context_mask
        & np.isfinite(sndr)
        & (sndr >= 0.0)
        & (sndr <= 1.0)
    )

    if revised_valid_mask is not None:
        valid_mask = finite_sndr & revised_valid_mask
    else:
        valid_mask = finite_sndr
        log(
            "[WARN] Revised SNDR valid mask was unavailable; "
            "finite SNDR values were used."
        )

    if not valid_mask.any():
        raise RuntimeError(
            "No valid revised SNDR values occur in the context domain."
        )

    kinematic_support_mask = read_mask_to_reference(
        SNDR_KINEMATIC_SUPPORT_MASK_TIF,
        reference,
        "SNDR kinematic-support mask",
        optional=True,
    )
    no_evidence_mask = read_mask_to_reference(
        SNDR_NO_EVIDENCE_MASK_TIF,
        reference,
        "no-kinematic-evidence diagnostic mask",
        optional=True,
    )
    mismatch_mask = read_mask_to_reference(
        SNDR_MISMATCH_MASK_TIF,
        reference,
        "nonzero IRS outside observation-support mask",
        optional=True,
    )

    if kinematic_support_mask is not None:
        kinematic_support_mask = (
            kinematic_support_mask & context_mask
        )

    if no_evidence_mask is not None:
        no_evidence_mask = no_evidence_mask & context_mask

    if mismatch_mask is not None:
        mismatch_mask = mismatch_mask & context_mask

    values = sndr[valid_mask]

    statistics = calculate_statistics(values)
    intervals = calculate_descriptive_intervals(values)

    study_pixels = int(study_mask.sum())
    context_pixels = int(context_mask.sum())
    valid_pixels = int(valid_mask.sum())

    kinematic_support_pixels = (
        int(kinematic_support_mask.sum())
        if kinematic_support_mask is not None
        else None
    )
    no_evidence_pixels = (
        int(no_evidence_mask.sum())
        if no_evidence_mask is not None
        else None
    )
    mismatch_pixels = (
        int(mismatch_mask.sum())
        if mismatch_mask is not None
        else None
    )

    coverage = {
        "study_pixels": study_pixels,
        "context_pixels": context_pixels,
        "valid_SNDR_pixels": valid_pixels,
        "valid_percent_of_context": (
            100.0 * valid_pixels / max(context_pixels, 1)
        ),
        "valid_percent_of_study": (
            100.0 * valid_pixels / max(study_pixels, 1)
        ),
    }

    diagnostics = {
        "kinematic_support_pixels": kinematic_support_pixels,
        "no_kinematic_evidence_pixels": no_evidence_pixels,
        "nonzero_IRS_outside_support_pixels": mismatch_pixels,
    }

    metadata = load_json_optional(SNDR_META_JSON)
    quality_assurance = load_json_optional(SNDR_QA_JSON)

    log(
        f"[RASTER] EPSG:{reference['epsg']} | "
        f"{reference['width']} × {reference['height']} | "
        f"{reference['resolution_x']:.3f} m"
    )
    log(
        f"[COVERAGE] {valid_pixels:,}/{context_pixels:,} context pixels "
        f"({coverage['valid_percent_of_context']:.3f}%)"
    )
    log(
        f"[SNDR] min={statistics['min']:.4f} | "
        f"median={statistics['median']:.4f} | "
        f"mean={statistics['mean']:.4f} | "
        f"p95={statistics['p95']:.4f} | "
        f"max={statistics['max']:.4f}"
    )
    log(
        "[STYLE] 2.5×5 inch map | four longitude DMS labels | "
        "three latitude DMS labels (rotated 90°, bottom label "
        "lifted) | no axis titles | 900 dpi | PNG/PDF/SVG"
    )

    if metadata:
        log(
            "[METADATA] Equation: "
            f"{metadata.get('equation', 'Unknown')}"
        )

    qa_status = None
    qa_warnings: list[str] = []

    if quality_assurance:
        qa_status = quality_assurance.get("status")
        qa_warnings = quality_assurance.get("warnings", []) or []

        log(f"[QA] {qa_status}")
        for warning in qa_warnings:
            log(f"[QA WARNING] {warning}")

    print("\nDescriptive SNDR intervals")
    print(
        "The intervals are descriptive only. Normalized SNDR is "
        "study-relative and is not an absolute probability."
    )

    for name, item in intervals.items():
        print(
            f"  {name}: {item['count']:,} "
            f"({item['percentage']:.2f}%)"
        )

    summary = {
        "generated_at_utc": utc_now_iso(),
        "purpose": (
            "Scientific direct-raster visualization of the revised "
            "Step 5.4 SNDR using the final Step 5.3 cartographic style."
        ),
        "scientific_plotting": {
            "direct_original_SNDR_raster": True,
            "RBF_used": False,
            "IDW_used": False,
            "cubic_interpolation_used": False,
            "scatter_reconstruction_used": False,
            "display_interpolation": RASTER_DISPLAY_INTERPOLATION,
            "values_renormalized_for_visualization": False,
        },
        "inputs": {
            "SNDR_raster": str(SNDR_NORM_TIF),
            "SNDR_valid_mask": (
                str(SNDR_VALID_MASK_TIF)
                if revised_valid_mask is not None
                else None
            ),
            "study_mask": str(STUDY_MASK_TIF),
            "context_mask_used": str(context_path_used),
            "kinematic_support_mask": (
                str(SNDR_KINEMATIC_SUPPORT_MASK_TIF)
                if kinematic_support_mask is not None
                else None
            ),
            "no_kinematic_evidence_mask": (
                str(SNDR_NO_EVIDENCE_MASK_TIF)
                if no_evidence_mask is not None
                else None
            ),
            "mismatch_mask": (
                str(SNDR_MISMATCH_MASK_TIF)
                if mismatch_mask is not None
                else None
            ),
        },
        "equation": (
            metadata.get("equation")
            if metadata
            else None
        ),
        "coverage": coverage,
        "statistics": statistics,
        "descriptive_intervals": {
            "warning": (
                "These intervals are descriptive and are not "
                "calibrated landslide probabilities."
            ),
            "counts": intervals,
        },
        "diagnostics": diagnostics,
        "analytical_QA_status": qa_status,
        "analytical_QA_warnings": qa_warnings,
        "visual_style": {
            "map_figsize_inches": list(MAP_FIGSIZE),
            "histogram_figsize_inches": list(HIST_FIGSIZE),
            "colormap": "viridis",
            "axis_titles_shown": False,
            "display_buffer_fraction": DISPLAY_BUFFER_FRACTION,
            "longitude_DMS_tick_count": LONGITUDE_DMS_TICK_COUNT,
            "latitude_DMS_tick_count": LATITUDE_DMS_TICK_COUNT,
            "longitude_DMS_label_rotation_deg": 45,
            "latitude_DMS_label_rotation_deg": 90,
            "bottom_latitude_label_va": "bottom",
            "DMS_tick_fontsize": DMS_TICK_FONTSIZE,
            "north_arrow": True,
            "checkerboard_scale_bar": True,
            "DMS_axes": True,
            "compact_legend": True,
            "side_colour_bar": True,
            "figure_dpi": FIG_DPI,
        },
        "outputs": {
            "map_png": str(OUT_MAP_PNG),
            "map_pdf": str(OUT_MAP_PDF),
            "map_svg": str(OUT_MAP_SVG),
            "histogram_png": str(OUT_HIST_PNG),
            "histogram_pdf": str(OUT_HIST_PDF),
            "histogram_svg": str(OUT_HIST_SVG),
            "summary_json": str(OUT_SUMMARY_JSON),
            "statistics_csv": str(OUT_STATS_CSV),
        },
    }

    with open(OUT_SUMMARY_JSON, "w", encoding="utf-8") as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    write_statistics_csv(
        OUT_STATS_CSV,
        statistics,
        intervals,
        coverage,
        diagnostics,
    )

    create_sndr_map(
        sndr=sndr,
        valid_mask=valid_mask,
        study_mask=study_mask,
        mismatch_mask=mismatch_mask,
        reference=reference,
    )
    create_sndr_histogram(
        values=values,
        statistics=statistics,
    )

    log(f"[SUMMARY] {OUT_SUMMARY_JSON}")
    log(f"[SUMMARY] {OUT_STATS_CSV}")
    log("=== Step 5.4 visualization completed successfully ===")


def create_sndr_visualization() -> None:
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Failed to create SNDR visualization: {exc}")
        traceback.print_exc()
        raise


if __name__ == "__main__":
    create_sndr_visualization()


# STEP 5.5 — SPATIAL CONSISTENCY OF PS-InSAR DEFORMATION WITH INDEPENDENTLY MAPPED LANDSLIDES   

In [ ]:
# =============================================================================
# STEP 5.5 — REVISED SPATIAL CONSISTENCY OF PS-InSAR DEFORMATION WITH
#            INDEPENDENTLY MAPPED LANDSLIDES
# =============================================================================
#
# Reviewer target
# ---------------
# Reviewer #1, Comment #12 requested independent validation of the InSAR-derived
# deformation, or at minimum a quantitative spatial-overlap analysis with known
# historical landslides / independent displacement records.
#
# This script implements the strongest defensible analysis available when
# colocated GNSS/GPS/extensometer measurements are unavailable:
#
#   A. complete source-inventory and domain accounting;
#   B. PS-InSAR observability and support audit;
#   C. quantitative spatial overlap with independently mapped landslides;
#   D. conservative local observability-matched background comparisons;
#   E. bootstrap uncertainty and paired non-parametric inference;
#   F. explicit spatial-overlap rasters and maps.
#
# INTERPRETATION BOUNDARY
# -----------------------
# The historical/geomorphic landslide polygons were mapped independently of the
# PS-InSAR processing, but they are not direct measurements of displacement
# magnitude. Therefore this is an independent spatial-consistency / event-
# overlap assessment, not direct geodetic validation of velocity.
#
# Corrections relative to the earlier Step 5.5
# ---------------------------------------------
# 1. Uses authoritative Step 4.2 winsorized DV+, not the earlier un-winsorized
#    Step 3.7 filled field.
# 2. Uses the explicit Step 5.4 DV-availability and kinematic-support masks.
# 3. Primary physical fields are:
#       - winsorized DV+
#       - IRS-weighted DV+
#       - supported raw SNDR
#    HKFI K and normalized SNDR are supplementary only.
# 4. Preserves every source inventory record; no polygon is silently dropped.
# 5. Direct PS source cells are treated as sparse observations; no 50% polygon
#    direct-source-coverage requirement is used.
# 6. G thresholds are descriptive observability classes, not support masks.
# 7. Principal inference uses local background matched by G class, slope, IRS,
#    and metric-specific support.
# 8. Both polygon median and P90 are tested to capture localized active sectors.
# 9. Effect estimates, bootstrap intervals, and Wilcoxon tests use the same
#    paired differences.
# 10. Global top-decile overlap is secondary descriptive evidence only.
# =============================================================================

from __future__ import annotations

import json
import math
import os
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Sequence

import fiona
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from pyproj import CRS, Transformer
from rasterio import features
from rasterio.enums import Resampling
from rasterio.windows import Window, from_bounds
from rasterio.windows import transform as window_transform
from scipy.stats import binomtest, wilcoxon
from shapely.geometry import box, mapping, shape
from shapely.ops import transform as shp_transform


# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    ANCHOR_TIF = RESULTS_DIR / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
    STUDY_MASK_TIF = RESULTS_DIR / "Step-0.2-Study_Mask" / "study_mask.tif"

    TRAIN_LANDSLIDE_SHP = (
        BASE / "Landslide Historical Map" / "Training Data"
        / "Landslide_only_train.shp"
    )
    VALID_LANDSLIDE_SHP = (
        BASE / "Landslide Historical Map" / "Validation Data"
        / "Landslide_only_valid.shp"
    )

    FILTER_SYMBOLID_IF_PRESENT = True
    LANDSLIDE_SYMBOL_VALUE = 1
    EXPECTED_TRAIN_RECORDS = 186
    EXPECTED_VALID_RECORDS = 46
    EXPECTED_TOTAL_SOURCE_RECORDS = 232
    ENFORCE_EXPECTED_SOURCE_COUNTS = True

    GEOMETRIC_G_TIF = (
        RESULTS_DIR / "Step-5.1-Observability"
        / "geometric_observability_G_0_1_32645_12p5m_float32.tif"
    )

    IRS_DIR = RESULTS_DIR / "Step-5.2-IRS"
    IRS_TIF = IRS_DIR / "irs_0_1_dense_32645_12p5m_float32.tif"
    IRS_SUPPORT_MASK_TIF = (
        IRS_DIR / "irs_observation_support_mask_32645_12p5m_byte.tif"
    )
    LOCAL_PS_COUNT_100M_TIF = (
        IRS_DIR / "ps_local_count_100m_32645_12p5m_float32.tif"
    )
    RAW_PS_COUNT_TIF = (
        RESULTS_DIR / "Step-3.2-LOS"
        / "los_velocity_count_32645_12p5m_int16.tif"
    )

    DVPLUS_WINSOR_TIF = (
        RESULTS_DIR / "Step-4.2-Standardization"
        / "dvplus_winsor_mm_yr_32645_12p5m_float32.tif"
    )

    STEP54_DIR = RESULTS_DIR / "Step-5.4-SNDR"
    WEIGHTED_DV_TIF = (
        STEP54_DIR / "sndr_weighted_dv_mm_yr_32645_12p5m_float32.tif"
    )
    SNDR_SUPPORTED_ONLY_TIF = (
        STEP54_DIR / "sndr_ratio_supported_only_32645_12p5m_float32.tif"
    )
    SNDR_01_TIF = STEP54_DIR / "sndr_0_1_32645_12p5m_float32.tif"
    DV_AVAILABILITY_MASK_TIF = (
        STEP54_DIR / "sndr_dv_availability_mask_32645_12p5m_byte.tif"
    )
    KINEMATIC_SUPPORT_MASK_TIF = (
        STEP54_DIR / "sndr_kinematic_support_mask_32645_12p5m_byte.tif"
    )
    SLOPE_TIF = (
        RESULTS_DIR / "Step-3.3-Downslope unit vector (ŝ)"
        / "slope_deg_32645_12p5m_float32.tif"
    )

    HKFI_K_TIF = (
        RESULTS_DIR / "Step-5.3-HKFI"
        / "hkfi_term_K_irs_times_zdv_32645_12p5m_float32.tif"
    )
    HKFI_DV_AVAILABILITY_MASK_TIF = (
        RESULTS_DIR / "Step-5.3-HKFI"
        / "hkfi_dv_availability_mask_32645_12p5m_byte.tif"
    )

    OUT_DIR = RESULTS_DIR / "Step-5.5-Spatial_Consistency"

    POLYGON_ALL_TOUCHED = True
    MIN_VALID_PIXELS_PER_POLYGON = 3
    MIN_VALID_COVERAGE_FRACTION = 0.10

    LOCAL_BACKGROUND_RADIUS_M = 2000.0
    BACKGROUND_EXCLUSION_BUFFER_M = 500.0
    MIN_MATCHED_BACKGROUND_PIXELS = 50

    MATCHING_TIERS = (
        {
            "tier": 1,
            "g_class_tolerance": 0,
            "slope_tolerance_deg": 5.0,
            "irs_tolerance": 0.10,
            "primary_inference": True,
        },
        {
            "tier": 2,
            "g_class_tolerance": 0,
            "slope_tolerance_deg": 10.0,
            "irs_tolerance": 0.20,
            "primary_inference": True,
        },
        {
            "tier": 3,
            "g_class_tolerance": 1,
            "slope_tolerance_deg": 15.0,
            "irs_tolerance": 0.30,
            "primary_inference": True,
        },
        {
            "tier": 4,
            "g_class_tolerance": None,
            "slope_tolerance_deg": None,
            "irs_tolerance": None,
            "primary_inference": False,
        },
    )

    GLOBAL_TOP_FRACTION = 0.10
    N_BOOTSTRAP = 5000
    CI_LEVEL = 0.95
    RANDOM_STATE = 55172

    WRITE_POLYGON_GPKG = True
    WRITE_SPATIAL_OVERLAP_RASTERS = True
    MAKE_FIGURES = True
    SHOW_FIGURES = False
    FIG_DPI = 300

    NODATA_FLOAT = -9999.0
    NODATA_BYTE = 255


PRIMARY_METRICS = (
    "DVplus_winsor",
    "IRS_weighted_DV",
    "SNDR_raw_supported",
)

SUPPLEMENTARY_METRICS = (
    "SNDR_0_1_supported",
    "HKFI_K_standardized",
)

ALL_METRICS = PRIMARY_METRICS + SUPPLEMENTARY_METRICS

METRIC_LABELS = {
    "DVplus_winsor": "Winsorized DV+ (mm/yr)",
    "IRS_weighted_DV": "IRS-weighted DV+ (mm/yr)",
    "SNDR_raw_supported": "Supported raw SNDR",
    "SNDR_0_1_supported": "Supported normalized SNDR (0–1)",
    "HKFI_K_standardized": "HKFI kinematic term K (supplementary)",
}


def log(message: str) -> None:
    print(
        f"[{datetime.now():%H:%M:%S}] {message}",
        flush=True,
    )

def utc_now_iso() -> str:
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )

def require_file(
    path: Path,
    description: str,
) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}"
        )

def json_safe(
    value: Any,
) -> Any:
    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    return value

def clean_float_nodata(
    array: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    output = np.asarray(
        array,
        dtype="float64",
    ).copy()

    if nodata is not None:
        try:
            nodata_value = float(
                nodata
            )

            if np.isfinite(
                nodata_value
            ):
                output[
                    output
                    == nodata_value
                ] = np.nan

        except (
            TypeError,
            ValueError,
        ):
            pass

    output[
        output
        == -9999.0
    ] = np.nan

    output[
        output
        == -32768.0
    ] = np.nan

    output[
        np.abs(
            output
        )
        > 1.0e20
    ] = np.nan

    return output

def to_pyproj_crs(
    crs_like: Any,
) -> CRS:
    if isinstance(
        crs_like,
        CRS,
    ):
        return crs_like

    if hasattr(
        crs_like,
        "to_wkt",
    ):
        return CRS.from_wkt(
            crs_like.to_wkt()
        )

    return CRS.from_user_input(
        crs_like
    )

def find_case_insensitive_field(
    properties: Dict[str, Any],
    requested: str,
) -> str | None:
    requested_lower = requested.lower()

    for key in properties:
        if str(
            key
        ).lower() == requested_lower:
            return str(
                key
            )

    return None

def assert_same_grid(
    dataset,
    reference: Dict[str, Any],
    description: str,
) -> None:
    same = (
        dataset.crs
        == reference[
            "crs"
        ]
        and dataset.transform
        == reference[
            "transform"
        ]
        and dataset.width
        == reference[
            "width"
        ]
        and dataset.height
        == reference[
            "height"
        ]
    )

    if not same:
        raise RuntimeError(
            f"{description} does not exactly match the "
            "Step 5.5 reference grid."
        )

def read_continuous_exact(
    path: Path,
    reference: Dict[str, Any],
    description: str,
) -> np.ndarray:
    require_file(
        path,
        description,
    )

    with rasterio.open(
        path
    ) as dataset:
        assert_same_grid(
            dataset,
            reference,
            description,
        )

        return clean_float_nodata(
            dataset.read(
                1
            ),
            dataset.nodata,
        )

def read_mask_exact(
    path: Path,
    reference: Dict[str, Any],
    description: str,
) -> np.ndarray:
    require_file(
        path,
        description,
    )

    with rasterio.open(
        path
    ) as dataset:
        assert_same_grid(
            dataset,
            reference,
            description,
        )

        raw = dataset.read(
            1
        )

        nodata = dataset.nodata

        values = raw.astype(
            "float64"
        )

        if nodata is not None:
            values[
                raw
                == nodata
            ] = 0.0

        values[
            ~np.isfinite(
                values
            )
        ] = 0.0

        return (
            values
            > 0.5
        )

def safe_window_for_geometry(
    geometry,
    reference: Dict[str, Any],
) -> Window | None:
    if geometry.is_empty:
        return None

    try:
        raw_window = from_bounds(
            *geometry.bounds,
            transform=reference[
                "transform"
            ],
        )

    except Exception:
        return None

    col_off = max(
        0,
        int(
            math.floor(
                raw_window.col_off
            )
        )
        - 1,
    )

    row_off = max(
        0,
        int(
            math.floor(
                raw_window.row_off
            )
        )
        - 1,
    )

    col_end = min(
        reference[
            "width"
        ],
        int(
            math.ceil(
                raw_window.col_off
                + raw_window.width
            )
        )
        + 1,
    )

    row_end = min(
        reference[
            "height"
        ],
        int(
            math.ceil(
                raw_window.row_off
                + raw_window.height
            )
        )
        + 1,
    )

    width = (
        col_end
        - col_off
    )

    height = (
        row_end
        - row_off
    )

    if (
        width
        <= 0
        or height
        <= 0
    ):
        return None

    return Window(
        col_off=col_off,
        row_off=row_off,
        width=width,
        height=height,
    )

def geometry_mask_in_window(
    geometry,
    window: Window,
    reference: Dict[str, Any],
    all_touched: bool,
) -> np.ndarray:
    out_shape = (
        int(
            window.height
        ),
        int(
            window.width
        ),
    )

    transform = window_transform(
        window,
        reference[
            "transform"
        ],
    )

    return features.geometry_mask(
        [
            mapping(
                geometry
            )
        ],
        out_shape=out_shape,
        transform=transform,
        invert=True,
        all_touched=all_touched,
    )

def array_window(
    array: np.ndarray,
    window: Window,
) -> np.ndarray:
    row_start = int(
        window.row_off
    )

    row_end = row_start + int(
        window.height
    )

    col_start = int(
        window.col_off
    )

    col_end = col_start + int(
        window.width
    )

    return array[
        row_start:row_end,
        col_start:col_end,
    ]

def describe_values(
    values: np.ndarray,
) -> Dict[str, float | int]:
    values = np.asarray(
        values,
        dtype="float64",
    )

    values = values[
        np.isfinite(
            values
        )
    ]

    if values.size == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "median": np.nan,
            "p90": np.nan,
            "max": np.nan,
        }

    return {
        "n": int(
            values.size
        ),
        "mean": float(
            np.mean(
                values
            )
        ),
        "median": float(
            np.median(
                values
            )
        ),
        "p90": float(
            np.percentile(
                values,
                90.0,
            )
        ),
        "max": float(
            np.max(
                values
            )
        ),
    }

def benjamini_hochberg(
    p_values: Sequence[float],
) -> np.ndarray:
    values = np.asarray(
        p_values,
        dtype="float64",
    )

    adjusted = np.full(
        values.shape,
        np.nan,
        dtype="float64",
    )

    finite_indices = np.flatnonzero(
        np.isfinite(
            values
        )
    )

    if finite_indices.size == 0:
        return adjusted

    finite_values = values[
        finite_indices
    ]

    order = np.argsort(
        finite_values
    )

    ranked = finite_values[
        order
    ]

    m = len(
        ranked
    )

    raw_adjusted = (
        ranked
        * m
        / np.arange(
            1,
            m
            + 1,
        )
    )

    monotonic = np.minimum.accumulate(
        raw_adjusted[
            ::-1
        ]
    )[
        ::-1
    ]

    monotonic = np.clip(
        monotonic,
        0.0,
        1.0,
    )

    restored = np.empty_like(
        monotonic
    )

    restored[
        order
    ] = monotonic

    adjusted[
        finite_indices
    ] = restored

    return adjusted


def finite_percent(numerator: int, denominator: int) -> float:
    if denominator <= 0:
        return float("nan")
    return 100.0 * float(numerator) / float(denominator)


def read_anchor(path: Path) -> Dict[str, Any]:
    require_file(path, "anchor raster")
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"Anchor has no CRS: {path}")
        return {
            "transform": dataset.transform,
            "crs": dataset.crs,
            "width": int(dataset.width),
            "height": int(dataset.height),
            "bounds": dataset.bounds,
            "profile": dataset.profile.copy(),
            "epsg": dataset.crs.to_epsg(),
            "res_x": float(abs(dataset.transform.a)),
            "res_y": float(abs(dataset.transform.e)),
        }


def read_count_preserve_zero_exact(
    path: Path,
    anchor: Dict[str, Any],
    study_mask: np.ndarray,
    description: str,
) -> np.ndarray:
    """Preserve legitimate zero PS counts even when zero is tagged as NoData."""
    require_file(path, description)
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")
        assert_same_grid(dataset, anchor, description)
        values = dataset.read(1).astype("float64")

        nodata_is_zero = False
        if dataset.nodata is not None:
            try:
                nodata_is_zero = math.isclose(float(dataset.nodata), 0.0)
            except (TypeError, ValueError):
                nodata_is_zero = False

        if dataset.nodata is not None and not nodata_is_zero:
            try:
                nodata_value = float(dataset.nodata)
                if np.isfinite(nodata_value):
                    values[values == nodata_value] = np.nan
            except (TypeError, ValueError):
                pass

        values[~np.isfinite(values)] = 0.0
        values[values < 0.0] = 0.0
        values[~study_mask] = np.nan
        return values


def write_byte_raster(
    path: Path,
    array: np.ndarray,
    anchor: Dict[str, Any],
    description: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        count=1,
        dtype="uint8",
        nodata=Config.NODATA_BYTE,
        compress="LZW",
        tiled=True,
        bigtiff="IF_NEEDED",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )
    temporary_path = Path(str(path) + ".part")
    with rasterio.open(temporary_path, "w", **profile) as dataset:
        dataset.write(np.asarray(array, dtype="uint8"), 1)
        dataset.set_band_description(1, description)
    os.replace(temporary_path, path)
    try:
        with rasterio.open(path, "r+") as dataset:
            dataset.build_overviews([2, 4, 8, 16], Resampling.nearest)
            dataset.update_tags(ns="rio_overview", resampling="nearest")
    except Exception:
        pass


def geometric_observability_class(geometric_g: np.ndarray) -> np.ndarray:
    output = np.zeros(geometric_g.shape, dtype="uint8")
    finite = np.isfinite(geometric_g)
    output[finite & (geometric_g < 0.05)] = 1
    output[finite & (geometric_g >= 0.05) & (geometric_g < 0.10)] = 2
    output[finite & (geometric_g >= 0.10) & (geometric_g < 0.20)] = 3
    output[finite & (geometric_g >= 0.20) & (geometric_g < 0.50)] = 4
    output[finite & (geometric_g >= 0.50)] = 5
    return output


def load_inventory_source(
    path: Path,
    source_label: str,
    target_crs: CRS,
) -> list[Dict[str, Any]]:
    """Preserve every retained source record, including invalid geometry."""
    require_file(path, f"{source_label} inventory")
    records: list[Dict[str, Any]] = []

    with fiona.open(path) as source:
        if getattr(source, "crs_wkt", None):
            source_crs = CRS.from_user_input(source.crs_wkt)
        elif getattr(source, "crs", None):
            source_crs = CRS.from_user_input(source.crs)
        else:
            raise ValueError(f"Could not determine CRS for {path}")

        transformer = None
        if source_crs != target_crs:
            transformer = Transformer.from_crs(
                source_crs,
                target_crs,
                always_xy=True,
            )

        retained_index = 0
        for feature_index, feature in enumerate(source):
            properties = dict(feature.get("properties", {}))
            symbol_field = find_case_insensitive_field(properties, "SymbolID")

            if Config.FILTER_SYMBOLID_IF_PRESENT and symbol_field is not None:
                try:
                    symbol_value = int(float(properties.get(symbol_field)))
                except (TypeError, ValueError):
                    symbol_value = None
                if symbol_value != Config.LANDSLIDE_SYMBOL_VALUE:
                    continue

            retained_index += 1
            geometry = None
            geometry_repaired = False
            geometry_error = ""

            raw_geometry = feature.get("geometry")
            if raw_geometry:
                try:
                    geometry = shape(raw_geometry)

                    if geometry is not None and not geometry.is_empty and not geometry.is_valid:
                        repaired = geometry.buffer(0)
                        if repaired is not None and not repaired.is_empty and repaired.is_valid:
                            geometry = repaired
                            geometry_repaired = True

                    if geometry is not None and not geometry.is_empty and transformer is not None:
                        geometry = shp_transform(transformer.transform, geometry)

                    if geometry is not None and not geometry.is_empty and not geometry.is_valid:
                        repaired = geometry.buffer(0)
                        if repaired is not None and not repaired.is_empty and repaired.is_valid:
                            geometry = repaired
                            geometry_repaired = True

                    if geometry is not None and geometry.is_empty:
                        geometry = None

                except Exception as exc:
                    geometry = None
                    geometry_error = f"{type(exc).__name__}: {exc}"

            records.append(
                {
                    "source_dataset": source_label,
                    "source_path": str(path),
                    "source_feature_id": str(feature.get("id", feature_index)),
                    "source_retained_index": int(retained_index),
                    "geometry": geometry,
                    "geometry_repaired": bool(geometry_repaired),
                    "geometry_error": geometry_error,
                }
            )

    log(f"[INVENTORY] {source_label}: {len(records)} retained source records")
    return records


def normalized_geometry_key(geometry) -> str | None:
    if geometry is None:
        return None
    try:
        if hasattr(geometry, "normalize"):
            normalized = geometry.normalize()
            if normalized is not None:
                geometry = normalized
    except Exception:
        pass
    try:
        return geometry.wkb_hex
    except Exception:
        return geometry.wkb.hex()


def mark_duplicate_geometries(records: list[Dict[str, Any]]) -> None:
    seen: Dict[str, str] = {}
    for record in records:
        key = normalized_geometry_key(record["geometry"])
        record["duplicate_geometry_key"] = key
        record["duplicate_of_inventory_id"] = None
        if key is None:
            continue
        if key in seen:
            record["duplicate_of_inventory_id"] = seen[key]
        else:
            seen[key] = record["inventory_id"]


def load_complete_inventory(
    target_crs: CRS,
) -> tuple[list[Dict[str, Any]], pd.DataFrame]:
    training_records = load_inventory_source(
        Config.TRAIN_LANDSLIDE_SHP,
        "training_inventory",
        target_crs,
    )
    validation_records = load_inventory_source(
        Config.VALID_LANDSLIDE_SHP,
        "validation_inventory",
        target_crs,
    )

    source_count_df = pd.DataFrame(
        [
            {
                "source_dataset": "training_inventory",
                "expected_records": Config.EXPECTED_TRAIN_RECORDS,
                "retained_records": len(training_records),
            },
            {
                "source_dataset": "validation_inventory",
                "expected_records": Config.EXPECTED_VALID_RECORDS,
                "retained_records": len(validation_records),
            },
            {
                "source_dataset": "combined",
                "expected_records": Config.EXPECTED_TOTAL_SOURCE_RECORDS,
                "retained_records": len(training_records) + len(validation_records),
            },
        ]
    )
    source_count_df["count_matches_expected"] = (
        source_count_df["expected_records"] == source_count_df["retained_records"]
    )

    if (
        Config.ENFORCE_EXPECTED_SOURCE_COUNTS
        and not bool(source_count_df["count_matches_expected"].all())
    ):
        raise RuntimeError(
            "Source inventory counts do not reproduce the revised contract:\n"
            + source_count_df.to_string(index=False)
        )

    records = training_records + validation_records
    for inventory_index, record in enumerate(records, start=1):
        record["inventory_index"] = int(inventory_index)
        record["inventory_id"] = f"LS_{inventory_index:04d}"

        geometry = record["geometry"]
        if geometry is None:
            record["geometry_valid"] = False
            record["polygon_area_m2"] = np.nan
            record["polygon_area_ha"] = np.nan
        else:
            record["geometry_valid"] = bool(geometry.is_valid and not geometry.is_empty)
            record["polygon_area_m2"] = float(geometry.area)
            record["polygon_area_ha"] = float(geometry.area / 10000.0)

    mark_duplicate_geometries(records)
    return records, source_count_df


def load_analysis_products() -> tuple[
    Dict[str, np.ndarray],
    Dict[str, np.ndarray],
    Dict[str, np.ndarray],
    Dict[str, Any],
]:
    anchor = read_anchor(Config.ANCHOR_TIF)
    study_mask = read_mask_exact(Config.STUDY_MASK_TIF, anchor, "study-area mask")

    geometric_g = read_continuous_exact(
        Config.GEOMETRIC_G_TIF,
        anchor,
        "Step 5.1 pure geometric observability G",
    )
    geometric_g_class = geometric_observability_class(geometric_g)

    irs = read_continuous_exact(
        Config.IRS_TIF,
        anchor,
        "Step 5.2 corrected dense IRS",
    )
    irs_support = read_mask_exact(
        Config.IRS_SUPPORT_MASK_TIF,
        anchor,
        "Step 5.2 IRS observation-support mask",
    )
    local_ps_count_100m = read_continuous_exact(
        Config.LOCAL_PS_COUNT_100M_TIF,
        anchor,
        "Step 5.2 local 100 m PS-count raster",
    )
    raw_ps_count = read_count_preserve_zero_exact(
        Config.RAW_PS_COUNT_TIF,
        anchor,
        study_mask,
        "raw PS-count / LOS velocity-count raster",
    )
    slope_deg = read_continuous_exact(
        Config.SLOPE_TIF,
        anchor,
        "slope raster used by corrected Step 5.4",
    )

    dvplus_winsor = read_continuous_exact(
        Config.DVPLUS_WINSOR_TIF,
        anchor,
        "authoritative Step 4.2 winsorized DV+",
    )
    weighted_dv = read_continuous_exact(
        Config.WEIGHTED_DV_TIF,
        anchor,
        "Step 5.4 IRS-weighted DV+",
    )
    sndr_raw_supported = read_continuous_exact(
        Config.SNDR_SUPPORTED_ONLY_TIF,
        anchor,
        "Step 5.4 supported raw SNDR",
    )
    sndr_01 = read_continuous_exact(
        Config.SNDR_01_TIF,
        anchor,
        "Step 5.4 normalized SNDR 0–1",
    )
    dv_availability = read_mask_exact(
        Config.DV_AVAILABILITY_MASK_TIF,
        anchor,
        "Step 5.4 DV-availability mask",
    )
    kinematic_support = read_mask_exact(
        Config.KINEMATIC_SUPPORT_MASK_TIF,
        anchor,
        "Step 5.4 kinematic-support mask",
    )

    hkfi_k = read_continuous_exact(
        Config.HKFI_K_TIF,
        anchor,
        "Step 5.3 HKFI kinematic component K",
    )
    hkfi_dv_availability = read_mask_exact(
        Config.HKFI_DV_AVAILABILITY_MASK_TIF,
        anchor,
        "Step 5.3 HKFI DV-availability mask",
    )

    metric_arrays = {
        "DVplus_winsor": dvplus_winsor,
        "IRS_weighted_DV": weighted_dv,
        "SNDR_raw_supported": sndr_raw_supported,
        "SNDR_0_1_supported": sndr_01,
        "HKFI_K_standardized": hkfi_k,
    }

    metric_supports = {
        "DVplus_winsor": (
            study_mask & dv_availability & np.isfinite(dvplus_winsor)
        ),
        "IRS_weighted_DV": (
            study_mask & kinematic_support & np.isfinite(weighted_dv)
        ),
        "SNDR_raw_supported": (
            study_mask & kinematic_support & np.isfinite(sndr_raw_supported)
        ),
        "SNDR_0_1_supported": (
            study_mask & kinematic_support & np.isfinite(sndr_01)
        ),
        "HKFI_K_standardized": (
            study_mask & hkfi_dv_availability & np.isfinite(hkfi_k)
        ),
    }

    auxiliary = {
        "study_mask": study_mask,
        "geometric_g": geometric_g,
        "geometric_g_class": geometric_g_class,
        "irs": irs,
        "irs_support": study_mask & irs_support,
        "local_ps_count_100m": local_ps_count_100m,
        "raw_ps_count": raw_ps_count,
        "direct_ps_source_cell": (
            study_mask & np.isfinite(raw_ps_count) & (raw_ps_count > 0.0)
        ),
        "slope_deg": slope_deg,
        "dv_availability": study_mask & dv_availability,
        "kinematic_support": study_mask & kinematic_support,
        "hkfi_dv_availability": study_mask & hkfi_dv_availability,
    }

    return metric_arrays, metric_supports, auxiliary, anchor



# =============================================================================
# INVENTORY DOMAIN ACCOUNTING — NEVER SILENTLY DROP A RECORD
# =============================================================================
def determine_record_domain_status(
    record: Dict[str, Any],
    anchor: Dict[str, Any],
    study_mask: np.ndarray,
) -> Dict[str, Any]:
    geometry = record["geometry"]
    raster_extent = box(*anchor["bounds"])

    result = {
        "domain_status": "",
        "intersects_raster_extent": False,
        "polygon_raster_pixels_in_extent": 0,
        "polygon_raster_pixels_in_study": 0,
        "analysis_unique_geometry": (
            record["duplicate_of_inventory_id"] is None
        ),
    }

    if geometry is None or geometry.is_empty or not record["geometry_valid"]:
        result["domain_status"] = "INVALID_GEOMETRY"
        return result

    result["intersects_raster_extent"] = bool(
        geometry.intersects(raster_extent)
    )

    if not result["intersects_raster_extent"]:
        result["domain_status"] = "OUTSIDE_RASTER_EXTENT"
        return result

    window = safe_window_for_geometry(geometry, anchor)

    if window is None:
        result["domain_status"] = "OUTSIDE_RASTER_EXTENT"
        return result

    polygon_mask = geometry_mask_in_window(
        geometry=geometry,
        window=window,
        reference=anchor,
        all_touched=Config.POLYGON_ALL_TOUCHED,
    )

    study_window = array_window(study_mask, window)

    result["polygon_raster_pixels_in_extent"] = int(polygon_mask.sum())
    result["polygon_raster_pixels_in_study"] = int(
        (polygon_mask & study_window).sum()
    )

    if result["polygon_raster_pixels_in_study"] <= 0:
        result["domain_status"] = "OUTSIDE_STUDY_MASK"
        return result

    if record["duplicate_of_inventory_id"] is not None:
        result["domain_status"] = "DUPLICATE_GEOMETRY_IN_STUDY"
        return result

    result["domain_status"] = "VALID_UNIQUE_IN_STUDY"
    return result


def apply_inventory_domain_accounting(
    records: list[Dict[str, Any]],
    anchor: Dict[str, Any],
    study_mask: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for record in records:
        status = determine_record_domain_status(
            record=record,
            anchor=anchor,
            study_mask=study_mask,
        )
        record.update(status)

        rows.append(
            {
                "inventory_id": record["inventory_id"],
                "inventory_index": int(record["inventory_index"]),
                "source_dataset": record["source_dataset"],
                "source_feature_id": record["source_feature_id"],
                "source_retained_index": int(record["source_retained_index"]),
                "geometry_valid": bool(record["geometry_valid"]),
                "geometry_repaired": bool(record["geometry_repaired"]),
                "geometry_error": record["geometry_error"],
                "polygon_area_m2": record["polygon_area_m2"],
                "polygon_area_ha": record["polygon_area_ha"],
                "duplicate_of_inventory_id": record["duplicate_of_inventory_id"],
                "analysis_unique_geometry": bool(
                    record["analysis_unique_geometry"]
                ),
                "intersects_raster_extent": bool(
                    record["intersects_raster_extent"]
                ),
                "polygon_raster_pixels_in_extent": int(
                    record["polygon_raster_pixels_in_extent"]
                ),
                "polygon_raster_pixels_in_study": int(
                    record["polygon_raster_pixels_in_study"]
                ),
                "domain_status": record["domain_status"],
            }
        )

    accounting_df = pd.DataFrame(rows)

    if len(accounting_df) != Config.EXPECTED_TOTAL_SOURCE_RECORDS:
        raise RuntimeError(
            "Inventory accounting did not preserve every source record. "
            f"Expected {Config.EXPECTED_TOTAL_SOURCE_RECORDS}, "
            f"found {len(accounting_df)}."
        )

    return accounting_df


def build_inventory_accounting_summary(
    accounting_df: pd.DataFrame,
) -> pd.DataFrame:
    status_order = [
        "VALID_UNIQUE_IN_STUDY",
        "DUPLICATE_GEOMETRY_IN_STUDY",
        "OUTSIDE_STUDY_MASK",
        "OUTSIDE_RASTER_EXTENT",
        "INVALID_GEOMETRY",
    ]

    rows = []

    for status in status_order:
        count = int((accounting_df["domain_status"] == status).sum())
        rows.append(
            {
                "domain_status": status,
                "record_count": count,
                "percent_of_all_source_records": finite_percent(
                    count,
                    len(accounting_df),
                ),
            }
        )

    rows.append(
        {
            "domain_status": "ALL_SOURCE_RECORDS",
            "record_count": int(len(accounting_df)),
            "percent_of_all_source_records": 100.0,
        }
    )

    return pd.DataFrame(rows)


# =============================================================================
# INVENTORY MASKS
# =============================================================================
def valid_geometries_for_rasterization(
    records: Sequence[Dict[str, Any]],
    include_duplicates: bool,
) -> list:
    geometries = []

    for record in records:
        geometry = record["geometry"]

        if geometry is None or geometry.is_empty:
            continue

        if record["polygon_raster_pixels_in_study"] <= 0:
            continue

        if (
            not include_duplicates
            and record["duplicate_of_inventory_id"] is not None
        ):
            continue

        geometries.append(geometry)

    return geometries


def rasterize_background_exclusion_mask(
    records: Sequence[Dict[str, Any]],
    anchor: Dict[str, Any],
    study_mask: np.ndarray,
) -> np.ndarray:
    geometries = valid_geometries_for_rasterization(
        records,
        include_duplicates=True,
    )

    if not geometries:
        raise RuntimeError(
            "No in-study landslide geometries are available for "
            "background exclusion."
        )

    buffered_geometries = [
        mapping(geometry.buffer(Config.BACKGROUND_EXCLUSION_BUFFER_M))
        for geometry in geometries
    ]

    exclusion = features.rasterize(
        ((geometry, 1) for geometry in buffered_geometries),
        out_shape=(anchor["height"], anchor["width"]),
        transform=anchor["transform"],
        fill=0,
        all_touched=True,
        dtype="uint8",
    ) > 0

    return exclusion & study_mask


def rasterize_unique_inventory_union(
    records: Sequence[Dict[str, Any]],
    anchor: Dict[str, Any],
    study_mask: np.ndarray,
) -> np.ndarray:
    geometries = valid_geometries_for_rasterization(
        records,
        include_duplicates=False,
    )

    if not geometries:
        raise RuntimeError(
            "No valid unique in-study landslide geometries are available."
        )

    inventory_union = features.rasterize(
        ((mapping(geometry), 1) for geometry in geometries),
        out_shape=(anchor["height"], anchor["width"]),
        transform=anchor["transform"],
        fill=0,
        all_touched=Config.POLYGON_ALL_TOUCHED,
        dtype="uint8",
    ) > 0

    return inventory_union & study_mask


# =============================================================================
# STATISTICAL HELPERS
# =============================================================================
def bootstrap_statistic_ci(
    values: np.ndarray,
    statistic: str,
    seed: int,
) -> tuple[float, float, float]:
    values = np.asarray(values, dtype="float64")
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan, np.nan, np.nan

    if statistic == "median":
        statistic_function = np.median
    elif statistic == "mean":
        statistic_function = np.mean
    else:
        raise ValueError(f"Unsupported statistic: {statistic}")

    observed = float(statistic_function(values))
    rng = np.random.default_rng(seed)
    bootstrap_values = np.empty(Config.N_BOOTSTRAP, dtype="float64")

    for index in range(Config.N_BOOTSTRAP):
        sample = rng.choice(values, size=values.size, replace=True)
        bootstrap_values[index] = statistic_function(sample)

    alpha = (1.0 - Config.CI_LEVEL) / 2.0
    low, high = np.quantile(
        bootstrap_values,
        [alpha, 1.0 - alpha],
    )

    return observed, float(low), float(high)


def paired_rank_biserial(
    differences: np.ndarray,
) -> float:
    """Signed-rank rank-biserial correlation."""
    differences = np.asarray(differences, dtype="float64")
    differences = differences[np.isfinite(differences)]
    differences = differences[differences != 0.0]

    if differences.size == 0:
        return 0.0

    ranks = pd.Series(np.abs(differences)).rank(
        method="average"
    ).to_numpy(dtype=float)

    positive_rank_sum = float(ranks[differences > 0.0].sum())
    negative_rank_sum = float(ranks[differences < 0.0].sum())
    denominator = positive_rank_sum + negative_rank_sum

    if denominator == 0.0:
        return 0.0

    return (
        positive_rank_sum - negative_rank_sum
    ) / denominator


def paired_wilcoxon_p_value(
    differences: np.ndarray,
) -> float:
    differences = np.asarray(differences, dtype="float64")
    differences = differences[np.isfinite(differences)]

    if differences.size == 0:
        return np.nan

    if np.allclose(differences, 0.0):
        return 1.0

    try:
        result = wilcoxon(
            differences,
            zero_method="wilcox",
            correction=False,
            alternative="two-sided",
            method="auto",
        )
        return float(result.pvalue)
    except ValueError:
        return 1.0


def paired_sign_test_p_value(
    differences: np.ndarray,
) -> float:
    differences = np.asarray(differences, dtype="float64")
    differences = differences[np.isfinite(differences)]

    positive = int(np.sum(differences > 0.0))
    negative = int(np.sum(differences < 0.0))
    n_nonzero = positive + negative

    if n_nonzero == 0:
        return 1.0

    return float(
        binomtest(
            k=positive,
            n=n_nonzero,
            p=0.5,
            alternative="two-sided",
        ).pvalue
    )


# =============================================================================
# SECONDARY GLOBAL HOTSPOT THRESHOLDS
# =============================================================================
def calculate_global_hotspot_thresholds(
    metric_arrays: Dict[str, np.ndarray],
    metric_supports: Dict[str, np.ndarray],
) -> pd.DataFrame:
    rows = []
    percentile = 100.0 * (1.0 - Config.GLOBAL_TOP_FRACTION)

    for metric_name in PRIMARY_METRICS:
        support = (
            metric_supports[metric_name]
            & np.isfinite(metric_arrays[metric_name])
        )
        values = metric_arrays[metric_name][support]

        if values.size == 0:
            raise RuntimeError(
                f"No valid supported values for {metric_name}."
            )

        threshold = float(np.percentile(values, percentile))
        actual_fraction = float(np.mean(values >= threshold))

        rows.append(
            {
                "metric": metric_name,
                "metric_label": METRIC_LABELS[metric_name],
                "analysis_role": "SECONDARY_GLOBAL_OVERLAP",
                "threshold_percentile": float(percentile),
                "threshold": threshold,
                "supported_domain_pixels": int(values.size),
                "actual_supported_domain_fraction_at_or_above_threshold": (
                    actual_fraction
                ),
            }
        )

    return pd.DataFrame(rows)


# =============================================================================
# LOCAL OBSERVABILITY-MATCHED BACKGROUND
# =============================================================================
def choose_matched_background(
    candidate_base: np.ndarray,
    focal_valid_mask: np.ndarray,
    geometric_g: np.ndarray,
    geometric_g_class: np.ndarray,
    slope_deg: np.ndarray,
    irs: np.ndarray,
) -> Dict[str, Any]:
    focal_g_values = geometric_g[
        focal_valid_mask & np.isfinite(geometric_g)
    ]
    focal_slope_values = slope_deg[
        focal_valid_mask & np.isfinite(slope_deg)
    ]
    focal_irs_values = irs[
        focal_valid_mask & np.isfinite(irs)
    ]

    empty_result = {
        "mask": np.zeros_like(candidate_base, dtype=bool),
        "match_tier": 0,
        "primary_inference_match": False,
        "matched_pixel_count": 0,
        "candidate_pool_count": int(candidate_base.sum()),
        "focal_median_G": np.nan,
        "focal_G_class": 0,
        "focal_median_slope_deg": np.nan,
        "focal_median_IRS": np.nan,
    }

    if (
        focal_g_values.size == 0
        or focal_slope_values.size == 0
        or focal_irs_values.size == 0
    ):
        return empty_result

    focal_median_g = float(np.median(focal_g_values))
    focal_median_slope = float(np.median(focal_slope_values))
    focal_median_irs = float(np.median(focal_irs_values))
    focal_g_class = int(
        geometric_observability_class(
            np.asarray([focal_median_g], dtype=float)
        )[0]
    )

    candidate_pool_count = int(candidate_base.sum())

    for tier_definition in Config.MATCHING_TIERS:
        tier = int(tier_definition["tier"])

        if tier == 4:
            matched = candidate_base.copy()
        else:
            matched = (
                candidate_base
                & (geometric_g_class > 0)
                & (
                    np.abs(
                        geometric_g_class.astype("int16")
                        - focal_g_class
                    )
                    <= int(tier_definition["g_class_tolerance"])
                )
                & np.isfinite(slope_deg)
                & (
                    np.abs(slope_deg - focal_median_slope)
                    <= float(tier_definition["slope_tolerance_deg"])
                )
                & np.isfinite(irs)
                & (
                    np.abs(irs - focal_median_irs)
                    <= float(tier_definition["irs_tolerance"])
                )
            )

        matched_pixel_count = int(matched.sum())

        if matched_pixel_count >= Config.MIN_MATCHED_BACKGROUND_PIXELS:
            return {
                "mask": matched,
                "match_tier": tier,
                "primary_inference_match": bool(
                    tier_definition["primary_inference"]
                ),
                "matched_pixel_count": matched_pixel_count,
                "candidate_pool_count": candidate_pool_count,
                "focal_median_G": focal_median_g,
                "focal_G_class": focal_g_class,
                "focal_median_slope_deg": focal_median_slope,
                "focal_median_IRS": focal_median_irs,
            }

    empty_result.update(
        {
            "candidate_pool_count": candidate_pool_count,
            "focal_median_G": focal_median_g,
            "focal_G_class": focal_g_class,
            "focal_median_slope_deg": focal_median_slope,
            "focal_median_IRS": focal_median_irs,
        }
    )
    return empty_result


# =============================================================================
# COMPLETE POLYGON-LEVEL ANALYSIS
# =============================================================================
def analyze_inventory_polygons(
    records: Sequence[Dict[str, Any]],
    metric_arrays: Dict[str, np.ndarray],
    metric_supports: Dict[str, np.ndarray],
    auxiliary: Dict[str, np.ndarray],
    anchor: Dict[str, Any],
    background_exclusion_mask: np.ndarray,
    hotspot_thresholds_df: pd.DataFrame,
) -> pd.DataFrame:
    threshold_lookup = hotspot_thresholds_df.set_index(
        "metric"
    )["threshold"].to_dict()

    rows = []
    study_mask = auxiliary["study_mask"]

    for record_number, record in enumerate(records, start=1):
        row: Dict[str, Any] = {
            "inventory_id": record["inventory_id"],
            "inventory_index": int(record["inventory_index"]),
            "source_dataset": record["source_dataset"],
            "source_feature_id": record["source_feature_id"],
            "domain_status": record["domain_status"],
            "duplicate_of_inventory_id": record["duplicate_of_inventory_id"],
            "geometry_valid": bool(record["geometry_valid"]),
            "geometry_repaired": bool(record["geometry_repaired"]),
            "polygon_area_m2": record["polygon_area_m2"],
            "polygon_area_ha": record["polygon_area_ha"],
            "polygon_raster_pixels_in_extent": int(
                record["polygon_raster_pixels_in_extent"]
            ),
            "polygon_raster_pixels_in_study": int(
                record["polygon_raster_pixels_in_study"]
            ),
            "primary_inference_geometry": bool(
                record["domain_status"] == "VALID_UNIQUE_IN_STUDY"
            ),
        }

        geometry = record["geometry"]

        if (
            geometry is None
            or record["polygon_raster_pixels_in_study"] <= 0
        ):
            rows.append(row)
            continue

        local_geometry = geometry.buffer(
            Config.LOCAL_BACKGROUND_RADIUS_M
        )
        window = safe_window_for_geometry(local_geometry, anchor)

        if window is None:
            row["analysis_window_status"] = "NO_VALID_WINDOW"
            rows.append(row)
            continue

        focal_geometry_mask = geometry_mask_in_window(
            geometry=geometry,
            window=window,
            reference=anchor,
            all_touched=Config.POLYGON_ALL_TOUCHED,
        )
        local_geometry_mask = geometry_mask_in_window(
            geometry=local_geometry,
            window=window,
            reference=anchor,
            all_touched=False,
        )

        study_window = array_window(study_mask, window)
        exclusion_window = array_window(
            background_exclusion_mask,
            window,
        )

        focal_base = focal_geometry_mask & study_window
        local_background_base = (
            local_geometry_mask
            & study_window
            & ~exclusion_window
        )

        focal_pixel_count = int(focal_base.sum())
        row["analysis_window_status"] = "ANALYZED"
        row["focal_study_pixel_count"] = focal_pixel_count

        if focal_pixel_count <= 0:
            rows.append(row)
            continue

        # ---------------------------------------------------------------------
        # Step 5.1 pure geometric observability — descriptive classes only.
        # ---------------------------------------------------------------------
        g_window = array_window(auxiliary["geometric_g"], window)
        g_class_window = array_window(
            auxiliary["geometric_g_class"],
            window,
        )
        g_values = g_window[
            focal_base & np.isfinite(g_window)
        ]
        g_stats = describe_values(g_values)

        row["G_valid_pixel_count"] = int(g_stats["n"])
        row["G_mean"] = g_stats["mean"]
        row["G_median"] = g_stats["median"]

        g_class_names = {
            1: "near_blind_G_lt_0p05",
            2: "very_low_G_0p05_to_0p10",
            3: "low_G_0p10_to_0p20",
            4: "moderate_G_0p20_to_0p50",
            5: "high_G_ge_0p50",
        }

        for class_code, class_name in g_class_names.items():
            class_count = int(
                (focal_base & (g_class_window == class_code)).sum()
            )
            row[f"G_class_{class_name}_pixel_count"] = class_count
            row[f"G_class_{class_name}_fraction"] = (
                class_count / focal_pixel_count
            )

        # ---------------------------------------------------------------------
        # Step 5.2 IRS and direct/local PS support.
        # ---------------------------------------------------------------------
        irs_window = array_window(auxiliary["irs"], window)
        irs_support_window = array_window(
            auxiliary["irs_support"],
            window,
        )
        local_ps_count_window = array_window(
            auxiliary["local_ps_count_100m"],
            window,
        )
        raw_ps_count_window = array_window(
            auxiliary["raw_ps_count"],
            window,
        )
        direct_ps_window = array_window(
            auxiliary["direct_ps_source_cell"],
            window,
        )

        irs_support_count = int(
            (focal_base & irs_support_window).sum()
        )
        row["IRS_observation_support_pixel_count"] = irs_support_count
        row["IRS_observation_support_fraction"] = (
            irs_support_count / focal_pixel_count
        )

        irs_values = irs_window[
            focal_base & np.isfinite(irs_window)
        ]
        irs_stats = describe_values(irs_values)
        row["IRS_mean"] = irs_stats["mean"]
        row["IRS_median"] = irs_stats["median"]

        direct_ps_cell_count = int(
            (focal_base & direct_ps_window).sum()
        )
        row["direct_PS_source_cell_count"] = direct_ps_cell_count
        row["intersects_any_direct_PS_source_cell"] = bool(
            direct_ps_cell_count > 0
        )

        polygon_area_ha = row["polygon_area_ha"]
        if np.isfinite(polygon_area_ha) and polygon_area_ha > 0.0:
            row["direct_PS_source_cells_per_ha"] = (
                direct_ps_cell_count / polygon_area_ha
            )
        else:
            row["direct_PS_source_cells_per_ha"] = np.nan

        direct_ps_counts = raw_ps_count_window[
            focal_base
            & np.isfinite(raw_ps_count_window)
            & (raw_ps_count_window > 0.0)
        ]
        row["direct_PS_total_count_sum"] = (
            float(np.sum(direct_ps_counts))
            if direct_ps_counts.size > 0
            else 0.0
        )

        local_ps_values = local_ps_count_window[
            focal_base & np.isfinite(local_ps_count_window)
        ]
        local_ps_stats = describe_values(local_ps_values)
        row["local_PS_count_100m_mean"] = local_ps_stats["mean"]
        row["local_PS_count_100m_median"] = local_ps_stats["median"]
        row["local_PS_count_100m_p90"] = local_ps_stats["p90"]

        # ---------------------------------------------------------------------
        # Authoritative Step 5.4 support fractions.
        # ---------------------------------------------------------------------
        dv_availability_window = array_window(
            auxiliary["dv_availability"],
            window,
        )
        kinematic_support_window = array_window(
            auxiliary["kinematic_support"],
            window,
        )

        dv_available_count = int(
            (focal_base & dv_availability_window).sum()
        )
        kinematic_support_count = int(
            (focal_base & kinematic_support_window).sum()
        )

        row["DV_availability_pixel_count"] = dv_available_count
        row["DV_availability_fraction"] = (
            dv_available_count / focal_pixel_count
        )
        row["kinematic_support_pixel_count"] = kinematic_support_count
        row["kinematic_support_fraction"] = (
            kinematic_support_count / focal_pixel_count
        )

        # Arrays reused for matching.
        local_g = array_window(
            auxiliary["geometric_g"],
            window,
        )
        local_g_class = array_window(
            auxiliary["geometric_g_class"],
            window,
        )
        local_slope = array_window(
            auxiliary["slope_deg"],
            window,
        )
        local_irs = array_window(
            auxiliary["irs"],
            window,
        )

        # ---------------------------------------------------------------------
        # Metric-specific consistency and matching.
        # ---------------------------------------------------------------------
        for metric_name in ALL_METRICS:
            metric_array_window = array_window(
                metric_arrays[metric_name],
                window,
            )
            metric_support_window = array_window(
                metric_supports[metric_name],
                window,
            )

            focal_valid = (
                focal_base
                & metric_support_window
                & np.isfinite(metric_array_window)
            )
            focal_values = metric_array_window[focal_valid]
            focal_stats = describe_values(focal_values)

            valid_count = int(focal_stats["n"])
            coverage_fraction = valid_count / focal_pixel_count

            row[f"{metric_name}_valid_pixel_count"] = valid_count
            row[f"{metric_name}_coverage_fraction"] = coverage_fraction
            row[f"{metric_name}_adequate_support"] = bool(
                record["domain_status"] == "VALID_UNIQUE_IN_STUDY"
                and valid_count >= Config.MIN_VALID_PIXELS_PER_POLYGON
                and coverage_fraction
                >= Config.MIN_VALID_COVERAGE_FRACTION
            )

            row[f"{metric_name}_mean"] = focal_stats["mean"]
            row[f"{metric_name}_median"] = focal_stats["median"]
            row[f"{metric_name}_p90"] = focal_stats["p90"]
            row[f"{metric_name}_max"] = focal_stats["max"]

            # Secondary global overlap for primary physical metrics only.
            if metric_name in threshold_lookup:
                threshold = float(threshold_lookup[metric_name])
                row[f"{metric_name}_global_top10_threshold"] = threshold

                if valid_count > 0:
                    row[f"{metric_name}_global_top10_fraction"] = float(
                        np.mean(focal_values >= threshold)
                    )
                    row[f"{metric_name}_intersects_global_top10"] = bool(
                        np.any(focal_values >= threshold)
                    )
                else:
                    row[f"{metric_name}_global_top10_fraction"] = np.nan
                    row[f"{metric_name}_intersects_global_top10"] = False

            candidate_background = (
                local_background_base
                & metric_support_window
                & np.isfinite(metric_array_window)
            )

            match_result = choose_matched_background(
                candidate_base=candidate_background,
                focal_valid_mask=focal_valid,
                geometric_g=local_g,
                geometric_g_class=local_g_class,
                slope_deg=local_slope,
                irs=local_irs,
            )

            matched_mask = match_result["mask"]
            background_values = metric_array_window[matched_mask]
            background_stats = describe_values(background_values)

            row[f"{metric_name}_match_tier"] = int(
                match_result["match_tier"]
            )
            row[f"{metric_name}_primary_inference_match"] = bool(
                match_result["primary_inference_match"]
            )
            row[f"{metric_name}_candidate_background_pixel_count"] = int(
                match_result["candidate_pool_count"]
            )
            row[f"{metric_name}_matched_background_pixel_count"] = int(
                match_result["matched_pixel_count"]
            )
            row[f"{metric_name}_matching_reference_median_G"] = (
                match_result["focal_median_G"]
            )
            row[f"{metric_name}_matching_reference_G_class"] = int(
                match_result["focal_G_class"]
            )
            row[f"{metric_name}_matching_reference_median_slope_deg"] = (
                match_result["focal_median_slope_deg"]
            )
            row[f"{metric_name}_matching_reference_median_IRS"] = (
                match_result["focal_median_IRS"]
            )

            row[f"{metric_name}_matched_background_mean"] = (
                background_stats["mean"]
            )
            row[f"{metric_name}_matched_background_median"] = (
                background_stats["median"]
            )
            row[f"{metric_name}_matched_background_p90"] = (
                background_stats["p90"]
            )

            if (
                np.isfinite(focal_stats["median"])
                and np.isfinite(background_stats["median"])
            ):
                row[f"{metric_name}_delta_median"] = float(
                    focal_stats["median"]
                    - background_stats["median"]
                )
            else:
                row[f"{metric_name}_delta_median"] = np.nan

            if (
                np.isfinite(focal_stats["p90"])
                and np.isfinite(background_stats["p90"])
            ):
                row[f"{metric_name}_delta_p90"] = float(
                    focal_stats["p90"]
                    - background_stats["p90"]
                )
            else:
                row[f"{metric_name}_delta_p90"] = np.nan

            # Localized activity: fraction of focal pixels above the matched
            # background P90, compared with the matched background itself.
            if (
                valid_count > 0
                and background_stats["n"] > 0
                and np.isfinite(background_stats["p90"])
            ):
                background_p90 = float(background_stats["p90"])
                focal_hotspot_fraction = float(
                    np.mean(focal_values >= background_p90)
                )
                background_hotspot_fraction = float(
                    np.mean(background_values >= background_p90)
                )

                row[
                    f"{metric_name}_"
                    "focal_fraction_ge_matched_background_p90"
                ] = focal_hotspot_fraction
                row[
                    f"{metric_name}_"
                    "matched_background_fraction_ge_its_p90"
                ] = background_hotspot_fraction
                row[
                    f"{metric_name}_delta_local_hotspot_fraction"
                ] = (
                    focal_hotspot_fraction
                    - background_hotspot_fraction
                )
            else:
                row[
                    f"{metric_name}_"
                    "focal_fraction_ge_matched_background_p90"
                ] = np.nan
                row[
                    f"{metric_name}_"
                    "matched_background_fraction_ge_its_p90"
                ] = np.nan
                row[
                    f"{metric_name}_delta_local_hotspot_fraction"
                ] = np.nan

        rows.append(row)

        if (
            record_number % 25 == 0
            or record_number == len(records)
        ):
            log(
                "[POLYGON] processed "
                f"{record_number}/{len(records)} source records"
            )

    polygon_df = pd.DataFrame(rows)

    if len(polygon_df) != Config.EXPECTED_TOTAL_SOURCE_RECORDS:
        raise RuntimeError(
            "Polygon-level output failed to preserve every source record."
        )

    return polygon_df


# =============================================================================
# SUMMARY TABLES
# =============================================================================
def build_observability_support_summary(
    polygon_df: pd.DataFrame,
) -> pd.DataFrame:
    in_study = polygon_df.loc[
        polygon_df["domain_status"] == "VALID_UNIQUE_IN_STUDY"
    ].copy()
    denominator = len(in_study)

    rows = []

    direct_ps_any = (
        in_study["intersects_any_direct_PS_source_cell"]
        .fillna(False)
        .astype(bool)
    )

    rows.append(
        {
            "measure": "Direct PS source-cell intersection",
            "denominator_unique_in_study_polygons": denominator,
            "polygons_with_any_support": int(direct_ps_any.sum()),
            "percent_with_any_support": finite_percent(
                int(direct_ps_any.sum()),
                denominator,
            ),
            "median_polygon_coverage_fraction": np.nan,
            "median_direct_PS_source_cells_per_ha": float(
                pd.to_numeric(
                    in_study["direct_PS_source_cells_per_ha"],
                    errors="coerce",
                ).median()
            ),
            "interpretation": (
                "Sparse point-like PS source intersection; no areal "
                "50% direct-source-coverage requirement is imposed."
            ),
        }
    )

    support_specs = [
        (
            "IRS observation support",
            "IRS_observation_support_fraction",
        ),
        (
            "Authoritative Step 5.4 DV availability",
            "DV_availability_fraction",
        ),
        (
            "Authoritative Step 5.4 kinematic support",
            "kinematic_support_fraction",
        ),
    ]

    for label, coverage_column in support_specs:
        coverage = pd.to_numeric(
            in_study[coverage_column],
            errors="coerce",
        ).fillna(0.0)
        any_support = coverage > 0.0

        rows.append(
            {
                "measure": label,
                "denominator_unique_in_study_polygons": denominator,
                "polygons_with_any_support": int(any_support.sum()),
                "percent_with_any_support": finite_percent(
                    int(any_support.sum()),
                    denominator,
                ),
                "median_polygon_coverage_fraction": float(
                    np.median(coverage)
                ),
                "median_direct_PS_source_cells_per_ha": np.nan,
                "interpretation": (
                    "Areal support fraction evaluated over unique "
                    "in-study inventory polygons."
                ),
            }
        )

    return pd.DataFrame(rows)


def build_geometric_observability_summary(
    polygon_df: pd.DataFrame,
) -> pd.DataFrame:
    in_study = polygon_df.loc[
        polygon_df["domain_status"] == "VALID_UNIQUE_IN_STUDY"
    ].copy()

    class_specs = [
        (
            "Near-blind",
            "G_class_near_blind_G_lt_0p05_fraction",
            "G < 0.05",
        ),
        (
            "Very low",
            "G_class_very_low_G_0p05_to_0p10_fraction",
            "0.05 <= G < 0.10",
        ),
        (
            "Low",
            "G_class_low_G_0p10_to_0p20_fraction",
            "0.10 <= G < 0.20",
        ),
        (
            "Moderate",
            "G_class_moderate_G_0p20_to_0p50_fraction",
            "0.20 <= G < 0.50",
        ),
        (
            "High",
            "G_class_high_G_ge_0p50_fraction",
            "G >= 0.50",
        ),
    ]

    rows = []

    for class_name, fraction_column, threshold in class_specs:
        fractions = pd.to_numeric(
            in_study[fraction_column],
            errors="coerce",
        )

        polygons_intersecting = int((fractions > 0.0).sum())

        rows.append(
            {
                "G_class": class_name,
                "threshold": threshold,
                "unique_in_study_polygons": int(len(in_study)),
                "polygons_intersecting_class": polygons_intersecting,
                "percent_polygons_intersecting_class": finite_percent(
                    polygons_intersecting,
                    len(in_study),
                ),
                "mean_polygon_fraction": float(fractions.mean()),
                "median_polygon_fraction": float(fractions.median()),
            }
        )

    return pd.DataFrame(rows)


def build_match_tier_qa(
    polygon_df: pd.DataFrame,
) -> pd.DataFrame:
    descriptions = {
        0: "No adequate matched background",
        1: "Same G class; slope ±5°; IRS ±0.10",
        2: "Same G class; slope ±10°; IRS ±0.20",
        3: "G class ±1; slope ±15°; IRS ±0.30",
        4: "Support-only fallback; descriptive only",
    }

    rows = []

    for metric_name in ALL_METRICS:
        subset = polygon_df.loc[
            (
                polygon_df["domain_status"]
                == "VALID_UNIQUE_IN_STUDY"
            )
            & polygon_df[
                f"{metric_name}_adequate_support"
            ].fillna(False)
        ].copy()

        tiers = pd.to_numeric(
            subset[f"{metric_name}_match_tier"],
            errors="coerce",
        ).fillna(0)

        for tier in (0, 1, 2, 3, 4):
            count = int((tiers == tier).sum())
            rows.append(
                {
                    "metric": metric_name,
                    "metric_label": METRIC_LABELS[metric_name],
                    "match_tier": tier,
                    "match_tier_description": descriptions[tier],
                    "adequately_supported_polygons": int(len(subset)),
                    "polygon_count": count,
                    "percent_of_adequately_supported": finite_percent(
                        count,
                        len(subset),
                    ),
                    "included_in_primary_inference": bool(
                        tier in (1, 2, 3)
                    ),
                }
            )

    return pd.DataFrame(rows)


def build_global_hotspot_overlap_summary(
    polygon_df: pd.DataFrame,
    metric_arrays: Dict[str, np.ndarray],
    metric_supports: Dict[str, np.ndarray],
    inventory_union_mask: np.ndarray,
    hotspot_thresholds_df: pd.DataFrame,
) -> pd.DataFrame:
    threshold_lookup = hotspot_thresholds_df.set_index(
        "metric"
    ).to_dict(orient="index")

    rows = []

    for metric_index, metric_name in enumerate(PRIMARY_METRICS):
        threshold = float(
            threshold_lookup[metric_name]["threshold"]
        )
        global_fraction = float(
            threshold_lookup[metric_name][
                "actual_supported_domain_fraction_at_or_above_threshold"
            ]
        )

        valid_inventory_pixels = (
            inventory_union_mask
            & metric_supports[metric_name]
            & np.isfinite(metric_arrays[metric_name])
        )

        n_valid_inventory_pixels = int(valid_inventory_pixels.sum())
        n_hotspot_inventory_pixels = int(
            (
                valid_inventory_pixels
                & (metric_arrays[metric_name] >= threshold)
            ).sum()
        )

        inventory_hotspot_fraction = (
            n_hotspot_inventory_pixels / n_valid_inventory_pixels
            if n_valid_inventory_pixels > 0
            else np.nan
        )

        enrichment_ratio = (
            inventory_hotspot_fraction / global_fraction
            if (
                np.isfinite(inventory_hotspot_fraction)
                and global_fraction > 0.0
            )
            else np.nan
        )

        supported_polygons = polygon_df.loc[
            (
                polygon_df["domain_status"]
                == "VALID_UNIQUE_IN_STUDY"
            )
            & (
                pd.to_numeric(
                    polygon_df[f"{metric_name}_valid_pixel_count"],
                    errors="coerce",
                ).fillna(0)
                > 0
            )
        ].copy()

        intersection_values = (
            supported_polygons[
                f"{metric_name}_intersects_global_top10"
            ]
            .fillna(False)
            .astype(float)
            .to_numpy()
        )

        (
            intersection_fraction,
            intersection_ci_low,
            intersection_ci_high,
        ) = bootstrap_statistic_ci(
            values=intersection_values,
            statistic="mean",
            seed=Config.RANDOM_STATE + 1000 + metric_index,
        )

        rows.append(
            {
                "metric": metric_name,
                "metric_label": METRIC_LABELS[metric_name],
                "analysis_role": (
                    "SECONDARY_GLOBAL_DESCRIPTIVE_OVERLAP"
                ),
                "global_hotspot_threshold": threshold,
                "global_supported_domain_hotspot_fraction": global_fraction,
                "inventory_supported_pixels": n_valid_inventory_pixels,
                "inventory_hotspot_pixels": n_hotspot_inventory_pixels,
                "inventory_hotspot_fraction": inventory_hotspot_fraction,
                "global_pixel_enrichment_ratio": enrichment_ratio,
                "supported_unique_in_study_polygons": int(
                    len(supported_polygons)
                ),
                "polygon_hotspot_intersection_fraction": (
                    intersection_fraction
                ),
                "polygon_hotspot_intersection_bootstrap_ci_low": (
                    intersection_ci_low
                ),
                "polygon_hotspot_intersection_bootstrap_ci_high": (
                    intersection_ci_high
                ),
            }
        )

    return pd.DataFrame(rows)


def build_primary_paired_tests(
    polygon_df: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    effect_specs = [
        (
            "median",
            "delta_median",
            "Polygon median minus matched-background median",
        ),
        (
            "p90",
            "delta_p90",
            "Polygon P90 minus matched-background P90",
        ),
        (
            "local_hotspot_fraction",
            "delta_local_hotspot_fraction",
            (
                "Fraction >= matched-background P90: "
                "polygon minus matched background"
            ),
        ),
    ]

    seed_counter = 0

    for metric_name in PRIMARY_METRICS:
        eligible = (
            (
                polygon_df["domain_status"]
                == "VALID_UNIQUE_IN_STUDY"
            )
            & polygon_df[
                f"{metric_name}_adequate_support"
            ].fillna(False)
            & polygon_df[
                f"{metric_name}_primary_inference_match"
            ].fillna(False)
            & pd.to_numeric(
                polygon_df[f"{metric_name}_match_tier"],
                errors="coerce",
            ).isin([1, 2, 3])
        )

        subset = polygon_df.loc[eligible].copy()

        for effect_name, delta_suffix, effect_description in effect_specs:
            seed_counter += 1

            differences = pd.to_numeric(
                subset[f"{metric_name}_{delta_suffix}"],
                errors="coerce",
            ).to_numpy(dtype=float)
            differences = differences[np.isfinite(differences)]

            (
                median_difference,
                ci_low,
                ci_high,
            ) = bootstrap_statistic_ci(
                values=differences,
                statistic="median",
                seed=(
                    Config.RANDOM_STATE
                    + 2000
                    + seed_counter
                ),
            )

            positive_fraction = (
                float(np.mean(differences > 0.0))
                if differences.size > 0
                else np.nan
            )

            zero_fraction = (
                float(np.mean(differences == 0.0))
                if differences.size > 0
                else np.nan
            )

            rows.append(
                {
                    "metric": metric_name,
                    "metric_label": METRIC_LABELS[metric_name],
                    "effect": effect_name,
                    "effect_description": effect_description,
                    "n_paired_unique_landslides": int(
                        differences.size
                    ),
                    "median_paired_difference": median_difference,
                    "bootstrap_median_ci_low": ci_low,
                    "bootstrap_median_ci_high": ci_high,
                    "fraction_positive_difference": positive_fraction,
                    "fraction_zero_difference": zero_fraction,
                    "wilcoxon_signed_rank_two_sided_p": (
                        paired_wilcoxon_p_value(differences)
                    ),
                    "exact_sign_test_two_sided_p": (
                        paired_sign_test_p_value(differences)
                    ),
                    "signed_rank_rank_biserial": (
                        paired_rank_biserial(differences)
                    ),
                    "matching_tiers_included": "1,2,3",
                    "tier4_support_only_fallback_excluded": True,
                }
            )

    result = pd.DataFrame(rows)

    if not result.empty:
        result["wilcoxon_BH_FDR_p"] = benjamini_hochberg(
            result[
                "wilcoxon_signed_rank_two_sided_p"
            ].to_numpy(dtype=float)
        )
        result["sign_test_BH_FDR_p"] = benjamini_hochberg(
            result[
                "exact_sign_test_two_sided_p"
            ].to_numpy(dtype=float)
        )

    return result



# =============================================================================
# REVIEWER-FOCUSED KEY STATISTICS
# =============================================================================
def build_reviewer_key_statistics(
    source_count_df: pd.DataFrame,
    accounting_summary_df: pd.DataFrame,
    observability_summary_df: pd.DataFrame,
    geometric_summary_df: pd.DataFrame,
    match_tier_df: pd.DataFrame,
    global_overlap_df: pd.DataFrame,
    paired_tests_df: pd.DataFrame,
) -> Dict[str, Any]:
    valid_unique_row = accounting_summary_df.loc[
        accounting_summary_df["domain_status"]
        == "VALID_UNIQUE_IN_STUDY"
    ]

    valid_unique_count = (
        int(valid_unique_row.iloc[0]["record_count"])
        if not valid_unique_row.empty
        else 0
    )

    return {
        "generated_at_utc": utc_now_iso(),
        "reviewer_comment": (
            "Reviewer #1 Comment #12 — independent deformation validation / "
            "minimum spatial overlap with historical events."
        ),
        "interpretation_boundary": (
            "Quantitative spatial consistency against historical/geomorphic "
            "landslide polygons mapped independently of the PS-InSAR processing; "
            "not direct GNSS/GPS displacement validation."
        ),
        "source_inventory_count_QA": source_count_df.to_dict(
            orient="records"
        ),
        "inventory_domain_accounting": accounting_summary_df.to_dict(
            orient="records"
        ),
        "valid_unique_in_study_landslides": valid_unique_count,
        "observability_support": observability_summary_df.to_dict(
            orient="records"
        ),
        "geometric_observability": geometric_summary_df.to_dict(
            orient="records"
        ),
        "matching_tier_QA": match_tier_df.to_dict(
            orient="records"
        ),
        "primary_local_matched_tests": paired_tests_df.to_dict(
            orient="records"
        ),
        "secondary_global_overlap": global_overlap_df.to_dict(
            orient="records"
        ),
    }


# =============================================================================
# SPATIAL OVERLAP RASTERS / MAPS
# =============================================================================
def create_overlap_category_raster(
    metric_name: str,
    metric_array: np.ndarray,
    metric_support: np.ndarray,
    inventory_union_mask: np.ndarray,
    study_mask: np.ndarray,
    threshold: float,
    anchor: Dict[str, Any],
    out_dir: Path,
) -> tuple[Path, np.ndarray]:
    """
    Category codes:
      0 = supported background, below global hotspot threshold
      1 = supported hotspot outside mapped landslide
      2 = supported mapped landslide, below hotspot threshold
      3 = supported hotspot inside mapped landslide
      4 = mapped landslide outside metric support
    255 = outside study / unclassified
    """
    output = np.full(
        study_mask.shape,
        Config.NODATA_BYTE,
        dtype="uint8",
    )

    supported = (
        study_mask
        & metric_support
        & np.isfinite(metric_array)
    )
    hotspot = supported & (metric_array >= threshold)

    output[supported & ~hotspot & ~inventory_union_mask] = 0
    output[hotspot & ~inventory_union_mask] = 1
    output[supported & ~hotspot & inventory_union_mask] = 2
    output[hotspot & inventory_union_mask] = 3
    output[
        study_mask
        & inventory_union_mask
        & ~supported
    ] = 4

    output_path = (
        out_dir
        / (
            f"Step_5_5_{metric_name}_"
            "Spatial_Overlap_Categories.tif"
        )
    )

    write_byte_raster(
        path=output_path,
        array=output,
        anchor=anchor,
        description=(
            f"Step 5.5 {metric_name} landslide spatial-overlap categories"
        ),
    )

    return output_path, output


def create_overlap_quicklook(
    category_raster: np.ndarray,
    anchor: Dict[str, Any],
    metric_name: str,
    output_path: Path,
) -> None:
    if not Config.MAKE_FIGURES:
        return

    display = category_raster.astype("float64")
    display[
        category_raster
        == Config.NODATA_BYTE
    ] = np.nan

    figure, axis = plt.subplots(
        figsize=(7.0, 5.2)
    )

    cmap = plt.get_cmap("tab10", 5)
    norm = BoundaryNorm(
        np.arange(-0.5, 5.5, 1.0),
        cmap.N,
    )

    extent = [
        anchor["bounds"].left,
        anchor["bounds"].right,
        anchor["bounds"].bottom,
        anchor["bounds"].top,
    ]

    axis.imshow(
        display,
        extent=extent,
        origin="upper",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
    )

    legend_labels = [
        "Supported background",
        "Global hotspot outside inventory",
        "Mapped landslide below hotspot",
        "Hotspot ∩ mapped landslide",
        "Mapped landslide outside metric support",
    ]

    handles = [
        Patch(
            facecolor=cmap(index),
            label=label,
        )
        for index, label in enumerate(legend_labels)
    ]

    axis.legend(
        handles=handles,
        loc="best",
        frameon=True,
        fontsize=8,
    )

    axis.set_title(
        f"{METRIC_LABELS[metric_name]}: spatial overlap"
    )
    axis.set_xlabel("Easting (m)")
    axis.set_ylabel("Northing (m)")
    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=Config.FIG_DPI,
        bbox_inches="tight",
    )

    if Config.SHOW_FIGURES:
        plt.show()

    plt.close(figure)


# =============================================================================
# FIGURES
# =============================================================================
def create_inventory_accounting_figure(
    accounting_summary_df: pd.DataFrame,
    output_path: Path,
) -> None:
    if not Config.MAKE_FIGURES:
        return

    plot_df = accounting_summary_df.loc[
        accounting_summary_df["domain_status"]
        != "ALL_SOURCE_RECORDS"
    ].copy()

    figure, axis = plt.subplots(
        figsize=(7.0, 4.5)
    )

    x = np.arange(len(plot_df))
    axis.bar(
        x,
        plot_df["record_count"],
    )
    axis.set_xticks(x)
    axis.set_xticklabels(
        plot_df["domain_status"].str.replace(
            "_",
            " ",
            regex=False,
        ),
        rotation=30,
        ha="right",
    )
    axis.set_ylabel("Source inventory records")
    axis.grid(True, axis="y", alpha=0.25)
    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=Config.FIG_DPI,
        bbox_inches="tight",
    )

    if Config.SHOW_FIGURES:
        plt.show()

    plt.close(figure)


def create_observability_support_figure(
    observability_summary_df: pd.DataFrame,
    output_path: Path,
) -> None:
    if not Config.MAKE_FIGURES:
        return

    figure, axis = plt.subplots(
        figsize=(7.5, 4.5)
    )

    x = np.arange(len(observability_summary_df))
    axis.bar(
        x,
        observability_summary_df["percent_with_any_support"],
    )
    axis.set_xticks(x)
    axis.set_xticklabels(
        observability_summary_df["measure"],
        rotation=25,
        ha="right",
    )
    axis.set_ylabel(
        "Unique in-study landslides with any support (%)"
    )
    axis.set_ylim(0.0, 105.0)
    axis.grid(True, axis="y", alpha=0.25)
    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=Config.FIG_DPI,
        bbox_inches="tight",
    )

    if Config.SHOW_FIGURES:
        plt.show()

    plt.close(figure)


def create_global_overlap_figure(
    global_overlap_df: pd.DataFrame,
    output_path: Path,
) -> None:
    if not Config.MAKE_FIGURES:
        return

    figure, axis = plt.subplots(
        figsize=(5.8, 4.2)
    )

    x = np.arange(len(global_overlap_df))
    axis.bar(
        x,
        global_overlap_df["global_pixel_enrichment_ratio"],
    )
    axis.axhline(
        1.0,
        linestyle="--",
        linewidth=1.0,
        label="No enrichment",
    )
    axis.set_xticks(x)
    axis.set_xticklabels(
        [
            "DV+",
            "IRS-weighted DV+",
            "SNDR raw",
        ],
    )
    axis.set_ylabel(
        "Global top-decile enrichment ratio"
    )
    axis.set_title(
        "Secondary descriptive global overlap"
    )
    axis.grid(True, axis="y", alpha=0.25)
    axis.legend(frameon=False)
    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=Config.FIG_DPI,
        bbox_inches="tight",
    )

    if Config.SHOW_FIGURES:
        plt.show()

    plt.close(figure)


def create_metric_matched_boxplots(
    polygon_df: pd.DataFrame,
    out_dir: Path,
) -> None:
    if not Config.MAKE_FIGURES:
        return

    for metric_name in PRIMARY_METRICS:
        eligible = (
            (
                polygon_df["domain_status"]
                == "VALID_UNIQUE_IN_STUDY"
            )
            & polygon_df[
                f"{metric_name}_adequate_support"
            ].fillna(False)
            & polygon_df[
                f"{metric_name}_primary_inference_match"
            ].fillna(False)
        )

        subset = polygon_df.loc[eligible].copy()

        plot_specs = [
            (
                "Median",
                f"{metric_name}_matched_background_median",
                f"{metric_name}_median",
                METRIC_LABELS[metric_name],
            ),
            (
                "P90",
                f"{metric_name}_matched_background_p90",
                f"{metric_name}_p90",
                METRIC_LABELS[metric_name],
            ),
            (
                "Local_Hotspot_Fraction",
                (
                    f"{metric_name}_"
                    "matched_background_fraction_ge_its_p90"
                ),
                (
                    f"{metric_name}_"
                    "focal_fraction_ge_matched_background_p90"
                ),
                "Fraction of pixels ≥ matched-background P90",
            ),
        ]

        for (
            statistic_name,
            background_column,
            focal_column,
            y_label,
        ) in plot_specs:
            background_values = pd.to_numeric(
                subset[background_column],
                errors="coerce",
            ).to_numpy(dtype=float)
            focal_values = pd.to_numeric(
                subset[focal_column],
                errors="coerce",
            ).to_numpy(dtype=float)

            valid = (
                np.isfinite(background_values)
                & np.isfinite(focal_values)
            )

            if not np.any(valid):
                continue

            figure, axis = plt.subplots(
                figsize=(4.8, 4.2)
            )
            axis.boxplot(
                [
                    background_values[valid],
                    focal_values[valid],
                ],
                labels=[
                    "Matched background",
                    "Mapped landslide",
                ],
                showfliers=False,
            )
            axis.set_ylabel(y_label)
            axis.set_title(
                f"{METRIC_LABELS[metric_name]} — {statistic_name}"
            )
            axis.grid(True, axis="y", alpha=0.25)
            figure.tight_layout()

            figure.savefig(
                (
                    out_dir
                    / (
                        f"Step_5_5_{metric_name}_"
                        f"{statistic_name}_Matched_Background.png"
                    )
                ),
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
            )

            if Config.SHOW_FIGURES:
                plt.show()

            plt.close(figure)


# =============================================================================
# OPTIONAL POLYGON GPKG
# =============================================================================
def safe_float_or_none(value: Any) -> float | None:
    try:
        result = float(value)
    except (TypeError, ValueError):
        return None
    if not np.isfinite(result):
        return None
    return result


def safe_int_or_default(
    value: Any,
    default: int = -1,
) -> int:
    """
    Convert a scalar to int without failing on None, NaN, pandas.NA,
    non-finite floats, or non-numeric values.

    GPKG convention used here:
      -1 = not evaluated / not applicable
       0 = evaluated but no adequate matched background
     1-4 = valid matching tier
    """
    if value is None:
        return int(default)

    try:
        if pd.isna(value):
            return int(default)
    except (TypeError, ValueError):
        pass

    try:
        numeric = float(value)
    except (TypeError, ValueError, OverflowError):
        return int(default)

    if not np.isfinite(numeric):
        return int(default)

    return int(numeric)


def safe_bool_to_int(
    value: Any,
    missing: int = -1,
) -> int:
    """
    Convert a scalar boolean-like value to an integer safely.

    GPKG convention:
      -1 = not evaluated / missing
       0 = False
       1 = True

    This explicitly avoids bool(np.nan), which incorrectly evaluates to True.
    """
    if value is None:
        return int(missing)

    try:
        if pd.isna(value):
            return int(missing)
    except (TypeError, ValueError):
        pass

    if isinstance(value, str):
        normalized = value.strip().lower()

        if normalized in {"true", "1", "yes", "y"}:
            return 1

        if normalized in {"false", "0", "no", "n"}:
            return 0

        return int(missing)

    return int(bool(value))


def write_polygon_metrics_gpkg(
    records: Sequence[Dict[str, Any]],
    polygon_df: pd.DataFrame,
    target_crs: CRS,
    output_path: Path,
) -> None:
    if not Config.WRITE_POLYGON_GPKG:
        return

    if output_path.exists():
        output_path.unlink()

    metrics_lookup = (
        polygon_df.set_index("inventory_id")
        .to_dict(orient="index")
    )

    schema = {
        "geometry": "Unknown",
        "properties": {
            "inv_id": "str:20",
            "src": "str:30",
            "status": "str:40",
            "area_ha": "float",
            "g_med": "float",
            "irs_cov": "float",
            "dv_cov": "float",
            "kin_cov": "float",
            "ps_any": "int",
            "dv_med": "float",
            "wdv_med": "float",
            "sndr_med": "float",
            "dv_tier": "int",
            "wdv_tier": "int",
            "sndr_tier": "int",
        },
    }

    with fiona.open(
        output_path,
        "w",
        driver="GPKG",
        crs_wkt=target_crs.to_wkt(),
        schema=schema,
        layer="landslide_spatial_consistency",
    ) as sink:
        for record in records:
            geometry = record["geometry"]

            if geometry is None or geometry.is_empty:
                continue

            metrics = metrics_lookup.get(
                record["inventory_id"],
                {},
            )

            sink.write(
                {
                    "geometry": mapping(geometry),
                    "properties": {
                        "inv_id": record["inventory_id"],
                        "src": record["source_dataset"],
                        "status": record["domain_status"],
                        "area_ha": safe_float_or_none(
                            record["polygon_area_ha"]
                        ),
                        "g_med": safe_float_or_none(
                            metrics.get("G_median")
                        ),
                        "irs_cov": safe_float_or_none(
                            metrics.get(
                                "IRS_observation_support_fraction"
                            )
                        ),
                        "dv_cov": safe_float_or_none(
                            metrics.get("DV_availability_fraction")
                        ),
                        "kin_cov": safe_float_or_none(
                            metrics.get("kinematic_support_fraction")
                        ),
                        "ps_any": safe_bool_to_int(
                            metrics.get(
                                "intersects_any_direct_PS_source_cell"
                            ),
                            missing=-1,
                        ),
                        "dv_med": safe_float_or_none(
                            metrics.get("DVplus_winsor_median")
                        ),
                        "wdv_med": safe_float_or_none(
                            metrics.get("IRS_weighted_DV_median")
                        ),
                        "sndr_med": safe_float_or_none(
                            metrics.get("SNDR_raw_supported_median")
                        ),
                        "dv_tier": safe_int_or_default(
                            metrics.get(
                                "DVplus_winsor_match_tier"
                            ),
                            default=-1,
                        ),
                        "wdv_tier": safe_int_or_default(
                            metrics.get(
                                "IRS_weighted_DV_match_tier"
                            ),
                            default=-1,
                        ),
                        "sndr_tier": safe_int_or_default(
                            metrics.get(
                                "SNDR_raw_supported_match_tier"
                            ),
                            default=-1,
                        ),
                    },
                }
            )


# =============================================================================
# REPORT / METADATA
# =============================================================================
def write_reviewer_report(
    output_path: Path,
    source_count_df: pd.DataFrame,
    accounting_summary_df: pd.DataFrame,
    observability_summary_df: pd.DataFrame,
    geometric_summary_df: pd.DataFrame,
    match_tier_df: pd.DataFrame,
    global_overlap_df: pd.DataFrame,
    paired_tests_df: pd.DataFrame,
) -> None:
    with open(output_path, "w", encoding="utf-8") as handle:
        handle.write(
            "STEP 5.5 — REVISED SPATIAL CONSISTENCY OF PS-InSAR "
            "DEFORMATION WITH INDEPENDENTLY MAPPED LANDSLIDES\n"
        )
        handle.write("=" * 110 + "\n\n")
        handle.write(f"Generated UTC: {utc_now_iso()}\n\n")

        handle.write(
            "REVIEWER #1 COMMENT #12 — INTERPRETATION BOUNDARY\n"
        )
        handle.write("-" * 110 + "\n")
        handle.write(
            "This analysis quantitatively evaluates spatial consistency "
            "between PS-InSAR-derived deformation fields and historical/"
            "geomorphic landslide polygons mapped independently of the "
            "PS-InSAR processing. It is an event-overlap / spatial-consistency "
            "assessment, not direct GNSS/GPS validation of displacement "
            "magnitude.\n\n"
        )

        handle.write("PRIMARY PHYSICAL INDICATORS\n")
        handle.write("-" * 110 + "\n")
        handle.write(
            "1. Step 4.2 winsorized DV+ on the authoritative Step 5.4 "
            "DV-availability domain.\n"
        )
        handle.write(
            "2. Step 5.4 IRS-weighted DV+ on the authoritative kinematic-"
            "support domain.\n"
        )
        handle.write(
            "3. Step 5.4 supported raw SNDR on the authoritative kinematic-"
            "support domain.\n"
        )
        handle.write(
            "The signed standardized HKFI K term and normalized SNDR 0–1 are "
            "supplementary diagnostics only.\n\n"
        )

        sections = [
            ("SOURCE INVENTORY COUNT QA", source_count_df),
            (
                "COMPLETE INVENTORY / DOMAIN ACCOUNTING",
                accounting_summary_df,
            ),
            (
                "PS-InSAR OBSERVABILITY AND SUPPORT",
                observability_summary_df,
            ),
            (
                "GEOMETRIC OBSERVABILITY CLASSES",
                geometric_summary_df,
            ),
            (
                "LOCAL BACKGROUND MATCH-TIER QA",
                match_tier_df,
            ),
            (
                "SECONDARY GLOBAL HOTSPOT OVERLAP",
                global_overlap_df,
            ),
            (
                "PRIMARY LOCAL MATCHED-BACKGROUND TESTS",
                paired_tests_df,
            ),
        ]

        for title, dataframe in sections:
            handle.write(title + "\n")
            handle.write("-" * 110 + "\n")
            handle.write(dataframe.to_string(index=False))
            handle.write("\n\n")

        handle.write("PRIMARY INFERENCE RULES\n")
        handle.write("-" * 110 + "\n")
        handle.write(
            f"- Minimum focal support: "
            f"{Config.MIN_VALID_PIXELS_PER_POLYGON} pixels and "
            f"{100.0 * Config.MIN_VALID_COVERAGE_FRACTION:.1f}% polygon "
            "coverage.\n"
        )
        handle.write(
            f"- Local search radius: "
            f"{Config.LOCAL_BACKGROUND_RADIUS_M:.0f} m.\n"
        )
        handle.write(
            f"- Exclusion around every mapped landslide: "
            f"{Config.BACKGROUND_EXCLUSION_BUFFER_M:.0f} m.\n"
        )
        handle.write(
            "- Primary inference uses matching tiers 1–3 only; support-only "
            "tier 4 is descriptive and excluded from inferential tests.\n"
        )
        handle.write(
            "- Effect estimates are median paired differences. Bootstrap "
            "confidence intervals and Wilcoxon tests use the same paired "
            "differences.\n"
        )
        handle.write(
            "- Polygon P90 and local-hotspot fraction are included to detect "
            "localized deformation within partially active historical "
            "landslide polygons.\n"
        )


def write_metadata(
    output_path: Path,
) -> None:
    metadata = {
        "generated_at_utc": utc_now_iso(),
        "step": (
            "Step 5.5 — Revised spatial consistency of PS-InSAR deformation "
            "with independently mapped landslides"
        ),
        "reviewer_target": "Reviewer #1 Comment #12",
        "interpretation_boundary": (
            "Independent spatial-consistency/event-overlap assessment; "
            "not direct geodetic displacement validation."
        ),
        "inventory_contract": {
            "training_source": str(Config.TRAIN_LANDSLIDE_SHP),
            "validation_source": str(Config.VALID_LANDSLIDE_SHP),
            "expected_training_records": Config.EXPECTED_TRAIN_RECORDS,
            "expected_validation_records": Config.EXPECTED_VALID_RECORDS,
            "expected_total_source_records": (
                Config.EXPECTED_TOTAL_SOURCE_RECORDS
            ),
            "all_source_records_preserved_in_accounting": True,
            "duplicate_geometries_preserved_for_provenance_but_excluded_from_inference": True,
        },
        "primary_indicators": {
            "DVplus_winsor": str(Config.DVPLUS_WINSOR_TIF),
            "IRS_weighted_DV": str(Config.WEIGHTED_DV_TIF),
            "SNDR_raw_supported": str(
                Config.SNDR_SUPPORTED_ONLY_TIF
            ),
        },
        "authoritative_support_masks": {
            "DV_availability": str(
                Config.DV_AVAILABILITY_MASK_TIF
            ),
            "kinematic_support": str(
                Config.KINEMATIC_SUPPORT_MASK_TIF
            ),
            "IRS_observation_support": str(
                Config.IRS_SUPPORT_MASK_TIF
            ),
        },
        "supplementary_indicators": {
            "normalized_SNDR": str(Config.SNDR_01_TIF),
            "HKFI_K": str(Config.HKFI_K_TIF),
        },
        "observability_inputs": {
            "pure_G": str(Config.GEOMETRIC_G_TIF),
            "corrected_IRS": str(Config.IRS_TIF),
            "local_PS_count_100m": str(
                Config.LOCAL_PS_COUNT_100M_TIF
            ),
            "raw_PS_count": str(Config.RAW_PS_COUNT_TIF),
        },
        "matching": {
            "local_background_radius_m": (
                Config.LOCAL_BACKGROUND_RADIUS_M
            ),
            "exclude_within_m_of_any_landslide": (
                Config.BACKGROUND_EXCLUSION_BUFFER_M
            ),
            "minimum_matched_background_pixels": (
                Config.MIN_MATCHED_BACKGROUND_PIXELS
            ),
            "tiers": json_safe(Config.MATCHING_TIERS),
            "primary_inference_tiers": [1, 2, 3],
            "tier4_support_only_fallback_excluded_from_primary_inference": True,
        },
        "statistics": {
            "bootstrap_iterations": Config.N_BOOTSTRAP,
            "ci_level": Config.CI_LEVEL,
            "paired_test": (
                "Wilcoxon signed-rank on the same paired differences "
                "reported by the effect estimate"
            ),
            "secondary_test": "exact two-sided sign test",
            "effect_size": "signed-rank rank-biserial correlation",
            "multiple_testing": (
                "Benjamini-Hochberg FDR across primary local-matched tests"
            ),
            "random_state": Config.RANDOM_STATE,
        },
        "global_top_decile_role": (
            "Secondary descriptive overlap only; principal inference uses "
            "local observability-matched background."
        ),
        "gpkg_integer_sentinels": {
            "ps_any": {
                "-1": "not evaluated / missing",
                "0": "no direct PS source-cell intersection",
                "1": "intersects at least one direct PS source cell",
            },
            "match_tier": {
                "-1": "not evaluated / not applicable",
                "0": "evaluated but no adequate matched background",
                "1": "same G class; slope ±5 degrees; IRS ±0.10",
                "2": "same G class; slope ±10 degrees; IRS ±0.20",
                "3": "G class ±1; slope ±15 degrees; IRS ±0.30",
                "4": "support-only fallback; descriptive only",
            },
        },
    }

    with open(output_path, "w", encoding="utf-8") as handle:
        json.dump(
            json_safe(metadata),
            handle,
            indent=2,
            ensure_ascii=False,
        )


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    Config.OUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    log("=" * 110)
    log(
        "STEP 5.5 — REVISED SPATIAL CONSISTENCY OF PS-InSAR DEFORMATION "
        "WITH INDEPENDENTLY MAPPED LANDSLIDES"
    )
    log("=" * 110)

    # A. Authoritative Step 5.1–5.4 products.
    (
        metric_arrays,
        metric_supports,
        auxiliary,
        anchor,
    ) = load_analysis_products()

    target_crs = to_pyproj_crs(anchor["crs"])

    # B. Complete 232-source-record accounting.
    records, source_count_df = load_complete_inventory(
        target_crs=target_crs
    )

    source_count_df.to_csv(
        Config.OUT_DIR
        / "Step_5_5_Source_Inventory_Count_QA.csv",
        index=False,
    )

    accounting_df = apply_inventory_domain_accounting(
        records=records,
        anchor=anchor,
        study_mask=auxiliary["study_mask"],
    )

    accounting_df.to_csv(
        Config.OUT_DIR
        / "Step_5_5_Inventory_Accounting_All_232_Records.csv",
        index=False,
        float_format="%.10f",
    )

    accounting_summary_df = build_inventory_accounting_summary(
        accounting_df
    )

    accounting_summary_df.to_csv(
        Config.OUT_DIR
        / "Step_5_5_Inventory_Accounting_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    log(
        "[ACCOUNTING]\n"
        + accounting_summary_df.to_string(index=False)
    )

    # C. Unique inventory union and conservative exclusion.
    inventory_union_mask = rasterize_unique_inventory_union(
        records=records,
        anchor=anchor,
        study_mask=auxiliary["study_mask"],
    )

    background_exclusion_mask = rasterize_background_exclusion_mask(
        records=records,
        anchor=anchor,
        study_mask=auxiliary["study_mask"],
    )

    # D. Secondary global thresholds.
    hotspot_thresholds_df = calculate_global_hotspot_thresholds(
        metric_arrays=metric_arrays,
        metric_supports=metric_supports,
    )

    hotspot_thresholds_df.to_csv(
        Config.OUT_DIR
        / "Step_5_5_Global_Hotspot_Thresholds_SECONDARY.csv",
        index=False,
        float_format="%.10f",
    )

    # E. Complete polygon-level analysis — all source records retained.
    polygon_df = analyze_inventory_polygons(
        records=records,
        metric_arrays=metric_arrays,
        metric_supports=metric_supports,
        auxiliary=auxiliary,
        anchor=anchor,
        background_exclusion_mask=background_exclusion_mask,
        hotspot_thresholds_df=hotspot_thresholds_df,
    )

    polygon_df.to_csv(
        Config.OUT_DIR
        / "Step_5_5_Polygon_Level_Spatial_Consistency_All_232_Records.csv",
        index=False,
        float_format="%.10f",
    )

    # F. Observability/support summaries.
    observability_summary_df = build_observability_support_summary(
        polygon_df
    )
    observability_summary_df.to_csv(
        Config.OUT_DIR
        / "Step_5_5_Observability_and_Support_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    geometric_summary_df = build_geometric_observability_summary(
        polygon_df
    )
    geometric_summary_df.to_csv(
        Config.OUT_DIR
        / "Step_5_5_Geometric_Observability_Class_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    # G. Matching QA.
    match_tier_df = build_match_tier_qa(polygon_df)
    match_tier_df.to_csv(
        Config.OUT_DIR
        / "Step_5_5_Local_Background_Match_Tier_QA.csv",
        index=False,
        float_format="%.10f",
    )

    # H. Secondary global overlap.
    global_overlap_df = build_global_hotspot_overlap_summary(
        polygon_df=polygon_df,
        metric_arrays=metric_arrays,
        metric_supports=metric_supports,
        inventory_union_mask=inventory_union_mask,
        hotspot_thresholds_df=hotspot_thresholds_df,
    )
    global_overlap_df.to_csv(
        Config.OUT_DIR
        / "Step_5_5_Global_Hotspot_Overlap_Summary_SECONDARY.csv",
        index=False,
        float_format="%.10f",
    )

    # I. Primary local matched-background inference.
    paired_tests_df = build_primary_paired_tests(polygon_df)
    paired_tests_df.to_csv(
        Config.OUT_DIR
        / "Step_5_5_Primary_Local_Matched_Paired_Tests.csv",
        index=False,
        float_format="%.10f",
    )

    # J. Explicit spatial-overlap rasters/maps.
    if Config.WRITE_SPATIAL_OVERLAP_RASTERS:
        threshold_lookup = hotspot_thresholds_df.set_index(
            "metric"
        )["threshold"].to_dict()

        for metric_name in PRIMARY_METRICS:
            _, overlap_categories = create_overlap_category_raster(
                metric_name=metric_name,
                metric_array=metric_arrays[metric_name],
                metric_support=metric_supports[metric_name],
                inventory_union_mask=inventory_union_mask,
                study_mask=auxiliary["study_mask"],
                threshold=float(threshold_lookup[metric_name]),
                anchor=anchor,
                out_dir=Config.OUT_DIR,
            )

            create_overlap_quicklook(
                category_raster=overlap_categories,
                anchor=anchor,
                metric_name=metric_name,
                output_path=(
                    Config.OUT_DIR
                    / (
                        f"Step_5_5_{metric_name}_"
                        "Spatial_Overlap_Map.png"
                    )
                ),
            )

    # K. Polygon GPKG.
    write_polygon_metrics_gpkg(
        records=records,
        polygon_df=polygon_df,
        target_crs=target_crs,
        output_path=(
            Config.OUT_DIR
            / "Step_5_5_Landslide_Spatial_Consistency_REVISED.gpkg"
        ),
    )

    # L. Figures.
    create_inventory_accounting_figure(
        accounting_summary_df=accounting_summary_df,
        output_path=(
            Config.OUT_DIR
            / "Step_5_5_Inventory_Accounting.png"
        ),
    )

    create_observability_support_figure(
        observability_summary_df=observability_summary_df,
        output_path=(
            Config.OUT_DIR
            / "Step_5_5_Observability_and_Support.png"
        ),
    )

    create_global_overlap_figure(
        global_overlap_df=global_overlap_df,
        output_path=(
            Config.OUT_DIR
            / "Step_5_5_Global_Hotspot_Overlap_SECONDARY.png"
        ),
    )

    create_metric_matched_boxplots(
        polygon_df=polygon_df,
        out_dir=Config.OUT_DIR,
    )

    # M. Reviewer-focused report / key statistics / metadata.
    reviewer_statistics = build_reviewer_key_statistics(
        source_count_df=source_count_df,
        accounting_summary_df=accounting_summary_df,
        observability_summary_df=observability_summary_df,
        geometric_summary_df=geometric_summary_df,
        match_tier_df=match_tier_df,
        global_overlap_df=global_overlap_df,
        paired_tests_df=paired_tests_df,
    )

    with open(
        Config.OUT_DIR
        / "Step_5_5_Reviewer_Comment_12_Key_Statistics.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(reviewer_statistics),
            handle,
            indent=2,
            ensure_ascii=False,
        )

    write_reviewer_report(
        output_path=(
            Config.OUT_DIR
            / "Step_5_5_Reviewer_Comment_12_Report.txt"
        ),
        source_count_df=source_count_df,
        accounting_summary_df=accounting_summary_df,
        observability_summary_df=observability_summary_df,
        geometric_summary_df=geometric_summary_df,
        match_tier_df=match_tier_df,
        global_overlap_df=global_overlap_df,
        paired_tests_df=paired_tests_df,
    )

    write_metadata(
        output_path=(
            Config.OUT_DIR
            / "Step_5_5_Spatial_Consistency_REVISED_Metadata.json"
        )
    )

    log("")
    log("=" * 110)
    log("STEP 5.5 REVISED ANALYSIS COMPLETED SUCCESSFULLY")
    log("=" * 110)
    log(f"[OUT] {Config.OUT_DIR}")

    print("\nSource inventory count QA:")
    print(source_count_df.to_string(index=False))

    print("\nInventory accounting summary:")
    print(accounting_summary_df.to_string(index=False))

    print("\nObservability/support summary:")
    print(observability_summary_df.to_string(index=False))

    print("\nPrimary local matched-background tests:")
    print(paired_tests_df.to_string(index=False))

    print("\nSecondary global overlap summary:")
    print(global_overlap_df.to_string(index=False))


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(
            f"[ERROR] Revised Step 5.5 failed: {exc}",
            flush=True,
        )
        traceback.print_exc()
        raise


# Phase-6: Data Preparation and Quality Check

# Step-6.1: Factor Stack

In [ ]:
# =============================================================================
# Step 6.1 — Authoritative Predictor Stack and Ablation-Scenario Foundation
# =============================================================================
#
# Purpose
# -------
# Build one harmonized predictor stack and explicit scenario manifests for the
# revised landslide-susceptibility modelling workflow.
#
# This step is designed to support the following controlled comparisons:
#
#   Model A — Static baseline
#       12 static predictors
#
#   Model B — Raw deformation
#       12 static predictors + direct DV+
#
#   Model C — HKFI only
#       12 static predictors + HKFI
#
#   Model D — SNDR only
#       12 static predictors + SNDR
#
#   Model E — Proposed physics-guided model
#       12 static predictors + HKFI + SNDR
#
#   Model F — Untransformed source-variable benchmark
#       12 static predictors + direct DV+ + TWI + support-preserved SM
#
# Scientific rules
# ----------------
# 1. Exactly ONE TPI predictor is retained: TPI_s5.
# 2. TWI and SM are excluded from the 12-factor static baseline because they
#    are HKFI sub-components in the primary integrated model.
# 3. IRS is NOT included as an ML predictor. It remains an internal reliability
#    weight used during HKFI/SNDR construction.
# 4. The dynamic mask is diagnostic only and does NOT restrict the modelling
#    domain.
# 5. No global standardization is performed here. XGB/RF can use the raw stack;
#    LR preprocessing must be fitted within each training fold downstream.
# 6. Final Step 5 physics-guided products must match the analysis grid exactly;
#    they are never silently resampled.
# 7. Static/source rasters may be snapped to the anchor grid using an explicitly
#    declared resampling method.
# 8. All A–F scenario definitions are written to machine-readable manifests so
#    the downstream sample extraction and spatial-CV code can use identical
#    rows, folds, preprocessing, and model settings.
# 9. A common A–F complete-case mask is written for diagnostics. It does not
#    clip or overwrite any predictor raster.
#
# Reviewer relevance
# ------------------
# - Reviewer 1 Comment 8: raw DV+ ablation versus HKFI/SNDR.
# - Reviewer 1 Comment 7: later propagation of alternative HKFI weight rasters.
# - Reviewer 1 Comment 14: LR ablation can reuse the same scenario definitions.
# =============================================================================

from __future__ import annotations

import csv
import json
import os
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject


# =============================================================================
# USER CONFIGURATION
# =============================================================================
BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))

# -----------------------------------------------------------------------------
# Anchor and masks
# -----------------------------------------------------------------------------
ANCHOR_TIF = (
    BASE / "Results" / "Step-0.1-Grid_CRS" / "DEM_anchor_32645.tif"
)
STUDY_MASK_TIF = (
    BASE / "Results" / "Step-0.2-Study_Mask" / "study_mask.tif"
)
CONTEXT_MASK_TIF = (
    BASE / "Results" / "Step-0.3-Quality_Masks" / "context_mask.tif"
)
DYNAMIC_MASK_TIF = (
    BASE / "Results" / "Step-0.3-Quality_Masks" / "dynamic_mask.tif"
)

# -----------------------------------------------------------------------------
# Exactly 12 static predictors
# TWI and SM are intentionally NOT part of Static12.
# Exactly one TPI is retained: tpi_s5.
# -----------------------------------------------------------------------------
SLOPE_TIF = (
    BASE / "Results" / "Step-1.1-Slope_Aspect" / "slope_deg.tif"
)
NORTHNESS_TIF = (
    BASE
    / "Results"
    / "Step-1.2-Northness_Eastness"
    / "northness.tif"
)
EASTNESS_TIF = (
    BASE
    / "Results"
    / "Step-1.2-Northness_Eastness"
    / "eastness.tif"
)
PROFILE_CURV_TIF = (
    BASE
    / "Results"
    / "Step-1.7-Curvatures"
    / "profile_curv_32645_12p5m_float32.tif"
)
PLAN_CURV_TIF = (
    BASE
    / "Results"
    / "Step-1.7-Curvatures"
    / "plan_curv_32645_12p5m_float32.tif"
)
TPI_TIF = (
    BASE
    / "Results"
    / "Step-1.8-TPI"
    / "tpi_s5_32645_12p5m_float32.tif"
)
SPI_TIF = (
    BASE
    / "Results"
    / "Step-1.9-SPI"
    / "spi_ln_32645_12p5m_float32.tif"
)
DIST_RIVER_TIF = (
    BASE
    / "Results"
    / "Step-1.11-DistanceFromRiver"
    / "distance_from_river_m_32645_12p5m_float32.tif"
)
DIST_ROAD_TIF = (
    BASE
    / "Results"
    / "Step-1.12-DistanceFromRoad"
    / "distance_from_road_m_32645_12p5m_float32.tif"
)
AAP_TIF = (
    BASE
    / "Results"
    / "Step-2.2-AAP"
    / "aap_2020_2023_32645_12p5m_float32.tif"
)
NDVI_TIF = (
    BASE
    / "Results"
    / "Step-2.1-NDVI"
    / "ndvi_median_2020_2023_32645_12p5m_int16.tif"
)
LANDCOVER_TIF = (
    BASE
    / "Results"
    / "Step-2.4-LandCover"
    / "ESA_WorldCover_Final_Reprojected.tif"
)

# -----------------------------------------------------------------------------
# Source variables needed for reviewer ablations
# -----------------------------------------------------------------------------
# Direct DV+ in physical units, before HKFI/SNDR fusion. The Step 4.2 winsorized
# product is preferred because it retains physical mm/yr units while applying
# the same documented outlier control used before index construction.
DVPLUS_CANDIDATES = [
    BASE
    / "Results"
    / "Step-4.2-Standardization"
    / "dvplus_winsor_mm_yr_32645_12p5m_float32.tif",
    BASE
    / "Results"
    / "Step-3.7-DVplus_Densify"
    / "dv_plus_filled_mm_yr_32645_12p5m_float32.tif",
]

# Untransformed TWI source used for Model F.
TWI_SOURCE_TIF = (
    BASE / "Results" / "Step-1.6-TWI" / "twi.tif"
)

# -----------------------------------------------------------------------------
# Corrected Step 5 products
# -----------------------------------------------------------------------------
HKFI_DIR = BASE / "Results" / "Step-5.3-HKFI"
SNDR_DIR = BASE / "Results" / "Step-5.4-SNDR"

# Prefer the explicit revised multiscale HKFI output.
HKFI_CANDIDATES = [
    HKFI_DIR / "hkfi_multiscale_norm_0_1_32645_12p5m_float32.tif",
    HKFI_DIR / "hkfi_norm_0_1_dense_32645_12p5m_float32.tif",
    HKFI_DIR / "hkfi_norm_0_1_32645_12p5m_float32.tif",
]

SNDR_TIF = (
    SNDR_DIR / "sndr_0_1_32645_12p5m_float32.tif"
)

# Support-preserved physical soil moisture generated by the revised Step 5.3.
# This is used in Model F instead of the former interpolated fine-grid SM.
SM_SUPPORT_PHYSICAL_TIF = (
    HKFI_DIR
    / "sm_support_preserved_physical_32645_12p5m_float32.tif"
)

# Optional provenance only; IRS is deliberately NOT included in the ML stack.
IRS_TIF = (
    BASE
    / "Results"
    / "Step-5.2-IRS"
    / "irs_0_1_dense_32645_12p5m_float32.tif"
)
SM_SUPPORT_MODE_JSON = HKFI_DIR / "sm_support_mode.json"
HKFI_METADATA_JSON = HKFI_DIR / "hkfi_metadata.json"
SNDR_METADATA_JSON = SNDR_DIR / "sndr_metadata.json"

# -----------------------------------------------------------------------------
# Outputs
# -----------------------------------------------------------------------------
OUT_DIR = BASE / "Results" / "Step-6.1-Stacks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

STATIC12_STACK_TIF = (
    OUT_DIR / "stack_static12_raw_studydomain_32645_12p5m_float32.tif"
)
MASTER_STACK_TIF = (
    OUT_DIR / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
)

ANALYSIS_DOMAIN_MASK_TIF = (
    OUT_DIR / "analysis_domain_mask_32645_12p5m_byte.tif"
)
COMMON_ABLATION_VALID_MASK_TIF = (
    OUT_DIR / "common_A_to_F_valid_mask_32645_12p5m_byte.tif"
)

SCENARIO_MASK_DIR = OUT_DIR / "scenario_valid_masks"
SCENARIO_MASK_DIR.mkdir(parents=True, exist_ok=True)

SCENARIO_STACK_DIR = OUT_DIR / "scenario_stacks"
SCENARIO_STACK_DIR.mkdir(parents=True, exist_ok=True)

STACK_MANIFEST_JSON = OUT_DIR / "stack_manifest.json"
BAND_MANIFEST_CSV = OUT_DIR / "master_candidate_band_manifest.csv"
SCENARIO_MANIFEST_JSON = OUT_DIR / "model_ablation_scenarios.json"
SCENARIO_MANIFEST_CSV = OUT_DIR / "model_ablation_scenarios.csv"
QC_REPORT_TXT = OUT_DIR / "Data Quality Report.txt"
STATIC_CORR_CSV = OUT_DIR / "Cross Correlation Static12.csv"
MASTER_CORR_CSV = OUT_DIR / "Cross Correlation Master Candidates.csv"

# Writing six scenario stacks repeats the 12 static bands many times and can
# consume substantial disk space. The master stack + manifests are sufficient
# for downstream modelling. Enable only when separate GeoTIFFs are desired.
WRITE_SCENARIO_STACKS = False

NODATA_FLOAT = -9999.0
NODATA_BYTE = 255
CORR_MAX_SAMPLES = 200_000
RANDOM_SEED = 42

# Strict coverage rule for final physics products. This checks grid identity;
# it does not require every pixel to be finite.
REQUIRE_EXACT_GRID_FOR_STEP5_PRODUCTS = True

# Fail if a required predictor has no valid values inside the analysis domain.
FAIL_ON_EMPTY_PREDICTOR = True


# =============================================================================
# AUTHORITATIVE FEATURE DEFINITIONS
# =============================================================================
STATIC12_FEATURES = [
    "slope_deg",
    "northness",
    "eastness",
    "profile_curv",
    "plan_curv",
    "tpi_s5",
    "spi_ln",
    "dist_river_m",
    "dist_road_m",
    "aap_mm",
    "ndvi",
    "landcover",
]

EXTRA_CANDIDATE_FEATURES = [
    "dvplus_mm_yr",
    "hkfi_0_1",
    "sndr_0_1",
    "twi_raw",
    "sm_support_physical",
]

MASTER_FEATURES = STATIC12_FEATURES + EXTRA_CANDIDATE_FEATURES

MODEL_SCENARIOS: dict[str, dict[str, Any]] = {
    "A_static_baseline": {
        "label": "Model A — Static baseline",
        "purpose": "Reference model using the 12 static predictors only.",
        "features": STATIC12_FEATURES,
    },
    "B_raw_deformation": {
        "label": "Model B — Raw deformation",
        "purpose": (
            "Tests the added value of direct DV+ without HKFI/SNDR fusion; "
            "directly addresses Reviewer 1 Comment 8."
        ),
        "features": STATIC12_FEATURES + ["dvplus_mm_yr"],
    },
    "C_hkfi_only": {
        "label": "Model C — HKFI only",
        "purpose": "Isolates the added value of HKFI relative to Static12.",
        "features": STATIC12_FEATURES + ["hkfi_0_1"],
    },
    "D_sndr_only": {
        "label": "Model D — SNDR only",
        "purpose": "Isolates the added value of SNDR relative to Static12.",
        "features": STATIC12_FEATURES + ["sndr_0_1"],
    },
    "E_physics_guided": {
        "label": "Model E — Proposed physics-guided model",
        "purpose": (
            "Primary integrated formulation: Static12 + HKFI + SNDR."
        ),
        "features": STATIC12_FEATURES + ["hkfi_0_1", "sndr_0_1"],
    },
    "F_untransformed_sources": {
        "label": "Model F — Untransformed source-variable benchmark",
        "purpose": (
            "Tests whether physics-guided fusion adds value beyond supplying "
            "direct DV+, TWI, and support-preserved soil moisture separately."
        ),
        "features": STATIC12_FEATURES
        + ["dvplus_mm_yr", "twi_raw", "sm_support_physical"],
    },
}


# =============================================================================
# LOGGING AND GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    return value


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def pick_existing(candidates: list[Path], description: str) -> Path:
    for path in candidates:
        if path.exists():
            log(f"[PATH] {description}: {path}")
            return path
    attempted = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not locate {description}. Tried:\n{attempted}"
    )


def load_json_optional(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return None


# =============================================================================
# RASTER HELPERS
# =============================================================================
def read_anchor(path: Path) -> dict[str, Any]:
    require_file(path, "anchor raster")
    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"Anchor raster has no CRS: {path}")
        return {
            "crs": dataset.crs,
            "transform": dataset.transform,
            "width": dataset.width,
            "height": dataset.height,
            "bounds": dataset.bounds,
            "profile": dataset.profile.copy(),
            "epsg": dataset.crs.to_epsg(),
            "res_x": abs(dataset.transform.a),
            "res_y": abs(dataset.transform.e),
        }


def same_grid(dataset: rasterio.io.DatasetReader, anchor: dict[str, Any]) -> bool:
    return (
        dataset.crs == anchor["crs"]
        and dataset.transform == anchor["transform"]
        and dataset.width == anchor["width"]
        and dataset.height == anchor["height"]
    )


def clean_float(array: np.ndarray, nodata: float | int | None) -> np.ndarray:
    output = array.astype("float64", copy=True)
    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if np.isfinite(nodata_value):
                output[output == nodata_value] = np.nan
        except Exception:
            pass
    output[output == -9999.0] = np.nan
    output[output == -32768.0] = np.nan
    output[np.abs(output) > 1.0e20] = np.nan
    return output


def read_mask_to_anchor(
    path: Path,
    anchor: dict[str, Any],
    required: bool,
    description: str,
) -> np.ndarray | None:
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Missing {description}: {path}")
        return None

    with rasterio.open(path) as dataset:
        raw = dataset.read(1)
        nodata = dataset.nodata

        if same_grid(dataset, anchor):
            values = raw.astype("float64")
            if nodata is not None:
                values[raw == nodata] = 0.0
            values[~np.isfinite(values)] = 0.0
            return values > 0.5

        destination = np.zeros(
            (anchor["height"], anchor["width"]),
            dtype="float64",
        )
        reproject(
            source=raw,
            destination=destination,
            src_transform=dataset.transform,
            src_crs=dataset.crs,
            src_nodata=nodata,
            dst_transform=anchor["transform"],
            dst_crs=anchor["crs"],
            dst_nodata=0.0,
            resampling=Resampling.nearest,
        )
        destination[~np.isfinite(destination)] = 0.0
        return destination > 0.5


def read_static_or_source_to_anchor(
    path: Path,
    anchor: dict[str, Any],
    description: str,
    resampling: Resampling,
    ndvi_autoscale: bool = False,
    categorical_round: bool = False,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Read a static/source raster; snap only when grid identity differs."""
    require_file(path, description)

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        raw = dataset.read(1)
        array = clean_float(raw, dataset.nodata)
        exact = same_grid(dataset, anchor)

        if exact:
            output = array
        else:
            output = np.full(
                (anchor["height"], anchor["width"]),
                np.nan,
                dtype="float64",
            )
            reproject(
                source=array,
                destination=output,
                src_transform=dataset.transform,
                src_crs=dataset.crs,
                src_nodata=np.nan,
                dst_transform=anchor["transform"],
                dst_crs=anchor["crs"],
                dst_nodata=np.nan,
                resampling=resampling,
            )

    if ndvi_autoscale:
        finite = np.isfinite(output)
        if finite.any():
            p01 = float(np.nanpercentile(output[finite], 0.1))
            p99 = float(np.nanpercentile(output[finite], 99.9))
            if p99 > 2.0 or p01 < -2.0:
                output = output / 10000.0
        output = np.where(
            np.isfinite(output),
            np.clip(output, -1.0, 1.0),
            np.nan,
        )

    if categorical_round:
        finite = np.isfinite(output)
        output[finite] = np.rint(output[finite])

    metadata = {
        "path": str(path),
        "exact_grid_before_read": bool(exact),
        "resampling_if_needed": resampling.name,
    }
    return output, metadata


def read_exact_continuous(
    path: Path,
    anchor: dict[str, Any],
    description: str,
    expected_0_1: bool = False,
    nonnegative: bool = False,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Read an authoritative downstream raster and refuse silent resampling."""
    require_file(path, description)

    with rasterio.open(path) as dataset:
        if dataset.crs is None:
            raise ValueError(f"{description} has no CRS: {path}")

        exact = same_grid(dataset, anchor)
        if REQUIRE_EXACT_GRID_FOR_STEP5_PRODUCTS and not exact:
            raise ValueError(
                f"{description} does not exactly match the anchor grid. "
                f"Regenerate the upstream product rather than resampling it:\n{path}"
            )

        raw = dataset.read(1)
        array = clean_float(raw, dataset.nodata)

    if expected_0_1:
        outside = np.isfinite(array) & (
            (array < -1.0e-6) | (array > 1.0 + 1.0e-6)
        )
        if outside.any():
            raise ValueError(
                f"{description} contains {int(outside.sum()):,} values outside [0,1]."
            )
        array = np.where(
            np.isfinite(array),
            np.clip(array, 0.0, 1.0),
            np.nan,
        )

    if nonnegative:
        clearly_negative = np.isfinite(array) & (array < -1.0e-6)
        if clearly_negative.any():
            raise ValueError(
                f"{description} contains {int(clearly_negative.sum()):,} "
                "negative values although DV+ should be non-negative."
            )
        array = np.where(
            np.isfinite(array),
            np.maximum(array, 0.0),
            np.nan,
        )

    return array, {
        "path": str(path),
        "exact_grid": bool(exact),
    }


def write_multiband_tif(
    path: Path,
    feature_names: list[str],
    arrays_by_name: dict[str, np.ndarray],
    anchor: dict[str, Any],
    stack_role: str,
) -> None:
    if not feature_names:
        raise ValueError("Cannot write a stack with zero bands.")

    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        count=len(feature_names),
        nodata=NODATA_FLOAT,
        compress="LZW",
        tiled=True,
        BIGTIFF="IF_SAFER",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    temporary = Path(str(path) + ".part")
    with rasterio.open(temporary, "w", **profile) as dataset:
        for band_index, feature_name in enumerate(feature_names, start=1):
            array = arrays_by_name[feature_name]
            output = np.where(
                np.isfinite(array),
                array,
                NODATA_FLOAT,
            ).astype("float32")
            dataset.write(output, band_index)
            dataset.set_band_description(band_index, feature_name)
            dataset.update_tags(
                band_index,
                feature_name=feature_name,
            )

        dataset.update_tags(
            generated_at_utc=utc_now_iso(),
            stack_role=stack_role,
            preprocessing=(
                "No global standardization in Step 6.1; "
                "fold-specific preprocessing is required downstream."
            ),
        )

        try:
            dataset.build_overviews([2, 4, 8, 16], Resampling.average)
            dataset.update_tags(ns="rio_overview", resampling="average")
        except Exception:
            pass

    if path.exists():
        path.unlink()
    os.replace(temporary, path)


def write_mask_tif(
    path: Path,
    mask: np.ndarray,
    anchor: dict[str, Any],
    description: str,
) -> None:
    profile = anchor["profile"].copy()
    profile.update(
        driver="GTiff",
        dtype="uint8",
        count=1,
        nodata=NODATA_BYTE,
        compress="LZW",
        tiled=True,
        BIGTIFF="IF_SAFER",
        transform=anchor["transform"],
        crs=anchor["crs"],
        width=anchor["width"],
        height=anchor["height"],
    )

    temporary = Path(str(path) + ".part")
    with rasterio.open(temporary, "w", **profile) as dataset:
        dataset.write(mask.astype("uint8"), 1)
        dataset.set_band_description(1, description)
        dataset.update_tags(description=description)
        try:
            dataset.build_overviews([2, 4, 8, 16], Resampling.nearest)
            dataset.update_tags(ns="rio_overview", resampling="nearest")
        except Exception:
            pass

    if path.exists():
        path.unlink()
    os.replace(temporary, path)


# =============================================================================
# QA HELPERS
# =============================================================================
def feature_statistics(
    array: np.ndarray,
    domain: np.ndarray,
) -> dict[str, Any]:
    valid = domain & np.isfinite(array)
    values = array[valid]

    if values.size == 0:
        return {
            "n_valid": 0,
            "coverage_percent": 0.0,
        }

    return {
        "n_valid": int(values.size),
        "coverage_percent": float(
            100.0 * values.size / max(int(domain.sum()), 1)
        ),
        "min": float(np.min(values)),
        "p01": float(np.percentile(values, 1)),
        "p05": float(np.percentile(values, 5)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": float(np.max(values)),
        "n_unique_rounded_6dp": int(
            np.unique(np.round(values, 6)).size
        ),
    }


def complete_case_mask(
    feature_names: list[str],
    arrays_by_name: dict[str, np.ndarray],
    domain: np.ndarray,
) -> np.ndarray:
    mask = domain.copy()
    for feature_name in feature_names:
        mask &= np.isfinite(arrays_by_name[feature_name])
    return mask


def build_correlation_matrix(
    feature_names: list[str],
    arrays_by_name: dict[str, np.ndarray],
    domain: np.ndarray,
    output_csv: Path,
    max_samples: int = CORR_MAX_SAMPLES,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    valid_index = np.flatnonzero(domain.ravel())

    if valid_index.size == 0:
        raise RuntimeError("Correlation domain contains no pixels.")

    if valid_index.size > max_samples:
        valid_index = rng.choice(
            valid_index,
            size=max_samples,
            replace=False,
        )

    flat = {
        name: arrays_by_name[name].ravel()
        for name in feature_names
    }

    n_features = len(feature_names)
    matrix = np.full(
        (n_features, n_features),
        np.nan,
        dtype="float64",
    )
    np.fill_diagonal(matrix, 1.0)

    for i, name_i in enumerate(feature_names):
        values_i = flat[name_i][valid_index]
        for j in range(i + 1, n_features):
            name_j = feature_names[j]
            values_j = flat[name_j][valid_index]
            valid = np.isfinite(values_i) & np.isfinite(values_j)
            if int(valid.sum()) >= 10:
                correlation = float(
                    np.corrcoef(
                        values_i[valid],
                        values_j[valid],
                    )[0, 1]
                )
                matrix[i, j] = correlation
                matrix[j, i] = correlation

    with open(output_csv, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow([""] + feature_names)
        for i, feature_name in enumerate(feature_names):
            writer.writerow(
                [feature_name]
                + [
                    f"{matrix[i, j]:.6f}"
                    if np.isfinite(matrix[i, j])
                    else "NaN"
                    for j in range(n_features)
                ]
            )

    return matrix


def validate_scenario_definitions() -> None:
    if len(STATIC12_FEATURES) != 12:
        raise RuntimeError(
            f"Static12 must contain exactly 12 predictors; found {len(STATIC12_FEATURES)}."
        )

    if len(set(STATIC12_FEATURES)) != 12:
        raise RuntimeError("Static12 contains duplicate feature names.")

    if "tpi_s25" in STATIC12_FEATURES:
        raise RuntimeError(
            "tpi_s25 must not be included. Only tpi_s5 is retained."
        )

    if "tpi_s5" not in STATIC12_FEATURES:
        raise RuntimeError("The retained single TPI, tpi_s5, is missing.")

    if "irs_0_1" in MASTER_FEATURES:
        raise RuntimeError("IRS must not be included as an ML predictor.")

    expected_counts = {
        "A_static_baseline": 12,
        "B_raw_deformation": 13,
        "C_hkfi_only": 13,
        "D_sndr_only": 13,
        "E_physics_guided": 14,
        "F_untransformed_sources": 15,
    }

    for scenario_name, expected_count in expected_counts.items():
        features = MODEL_SCENARIOS[scenario_name]["features"]
        if len(features) != expected_count:
            raise RuntimeError(
                f"{scenario_name} should have {expected_count} features; "
                f"found {len(features)}."
            )
        missing = [
            feature for feature in features
            if feature not in MASTER_FEATURES
        ]
        if missing:
            raise RuntimeError(
                f"{scenario_name} references unknown features: {missing}"
            )


# =============================================================================
# MAIN WORKFLOW
# =============================================================================
def main() -> None:
    log("=== Step 6.1: authoritative predictor stack and ablation foundation ===")
    validate_scenario_definitions()

    anchor = read_anchor(ANCHOR_TIF)
    log(
        f"[ANCHOR] EPSG:{anchor['epsg']} | "
        f"{anchor['width']} x {anchor['height']} | "
        f"{anchor['res_x']:.3f} x {anchor['res_y']:.3f} m"
    )

    # -------------------------------------------------------------------------
    # Analysis domain: context ∩ study, with context fallback to study.
    # Dynamic mask is read only for diagnostics and NEVER restricts predictors.
    # -------------------------------------------------------------------------
    study = read_mask_to_anchor(
        STUDY_MASK_TIF,
        anchor,
        required=True,
        description="study-area mask",
    )
    assert study is not None

    context = read_mask_to_anchor(
        CONTEXT_MASK_TIF,
        anchor,
        required=False,
        description="context mask",
    )
    if context is None:
        analysis_domain = study.copy()
        context_path_used = STUDY_MASK_TIF
        log("[MASK] context_mask unavailable; study_mask used as analysis domain.")
    else:
        analysis_domain = study & context
        context_path_used = CONTEXT_MASK_TIF

    dynamic = read_mask_to_anchor(
        DYNAMIC_MASK_TIF,
        anchor,
        required=False,
        description="dynamic mask",
    )
    if dynamic is not None:
        dynamic = dynamic & study

    if not analysis_domain.any():
        raise RuntimeError("The analysis domain contains no valid pixels.")

    write_mask_tif(
        ANALYSIS_DOMAIN_MASK_TIF,
        analysis_domain,
        anchor,
        "analysis_domain_context_intersection_study",
    )

    log(
        f"[MASK] study={int(study.sum()):,} | "
        f"analysis_domain={int(analysis_domain.sum()):,}"
    )
    if dynamic is not None:
        log(
            f"[MASK] dynamic diagnostic only={int(dynamic.sum()):,}; "
            "NOT used to restrict Step 6.1."
        )

    # -------------------------------------------------------------------------
    # Resolve authoritative upstream outputs.
    # -------------------------------------------------------------------------
    dvplus_path = pick_existing(
        DVPLUS_CANDIDATES,
        "direct DV+ raster",
    )
    hkfi_path = pick_existing(
        HKFI_CANDIDATES,
        "revised multiscale HKFI raster",
    )

    # -------------------------------------------------------------------------
    # Load exactly 12 static predictors.
    # No global winsorization, log transform, or standardization is performed.
    # -------------------------------------------------------------------------
    static_specs = [
        ("slope_deg", SLOPE_TIF, Resampling.bilinear, False, False, "deg", "Slope angle"),
        ("northness", NORTHNESS_TIF, Resampling.bilinear, False, False, "unitless", "Cosine of aspect"),
        ("eastness", EASTNESS_TIF, Resampling.bilinear, False, False, "unitless", "Sine of aspect"),
        ("profile_curv", PROFILE_CURV_TIF, Resampling.bilinear, False, False, "1/m", "Profile curvature"),
        ("plan_curv", PLAN_CURV_TIF, Resampling.bilinear, False, False, "1/m", "Plan curvature"),
        ("tpi_s5", TPI_TIF, Resampling.bilinear, False, False, "m", "Topographic Position Index; retained single TPI"),
        ("spi_ln", SPI_TIF, Resampling.bilinear, False, False, "ln-unit", "Log-transformed Stream Power Index from source step"),
        ("dist_river_m", DIST_RIVER_TIF, Resampling.bilinear, False, False, "m", "Distance to river"),
        ("dist_road_m", DIST_ROAD_TIF, Resampling.bilinear, False, False, "m", "Distance to road"),
        ("aap_mm", AAP_TIF, Resampling.bilinear, False, False, "mm", "Average annual precipitation"),
        ("ndvi", NDVI_TIF, Resampling.bilinear, True, False, "-1 to 1", "NDVI"),
        ("landcover", LANDCOVER_TIF, Resampling.nearest, False, True, "class", "ESA WorldCover class code"),
    ]

    arrays_by_name: dict[str, np.ndarray] = {}
    provenance_by_name: dict[str, dict[str, Any]] = {}
    units_by_name: dict[str, str] = {}
    descriptions_by_name: dict[str, str] = {}

    log("[LOAD] Reading Static12 predictors...")
    for (
        name,
        path,
        resampling,
        ndvi_autoscale,
        categorical_round,
        units,
        description,
    ) in static_specs:
        array, provenance = read_static_or_source_to_anchor(
            path=path,
            anchor=anchor,
            description=name,
            resampling=resampling,
            ndvi_autoscale=ndvi_autoscale,
            categorical_round=categorical_round,
        )
        array = np.where(analysis_domain, array, np.nan)
        arrays_by_name[name] = array
        provenance_by_name[name] = provenance
        units_by_name[name] = units
        descriptions_by_name[name] = description

    # -------------------------------------------------------------------------
    # Load direct DV+, revised HKFI, revised SNDR, raw TWI, and support-preserved
    # physical soil moisture.
    # -------------------------------------------------------------------------
    dvplus, dvplus_provenance = read_exact_continuous(
        dvplus_path,
        anchor,
        "direct DV+ raster",
        nonnegative=True,
    )
    hkfi, hkfi_provenance = read_exact_continuous(
        hkfi_path,
        anchor,
        "revised multiscale HKFI raster",
        expected_0_1=True,
    )
    sndr, sndr_provenance = read_exact_continuous(
        SNDR_TIF,
        anchor,
        "corrected Slope-Normalized Deformation Ratio raster",
        expected_0_1=True,
    )

    twi_raw, twi_provenance = read_static_or_source_to_anchor(
        path=TWI_SOURCE_TIF,
        anchor=anchor,
        description="untransformed TWI source raster",
        resampling=Resampling.bilinear,
    )

    sm_support_physical, sm_provenance = read_exact_continuous(
        SM_SUPPORT_PHYSICAL_TIF,
        anchor,
        "support-preserved physical soil-moisture raster",
    )

    additional_arrays = {
        "dvplus_mm_yr": dvplus,
        "hkfi_0_1": hkfi,
        "sndr_0_1": sndr,
        "twi_raw": twi_raw,
        "sm_support_physical": sm_support_physical,
    }

    additional_provenance = {
        "dvplus_mm_yr": dvplus_provenance,
        "hkfi_0_1": hkfi_provenance,
        "sndr_0_1": sndr_provenance,
        "twi_raw": twi_provenance,
        "sm_support_physical": sm_provenance,
    }

    additional_units = {
        "dvplus_mm_yr": "mm/yr",
        "hkfi_0_1": "0-1",
        "sndr_0_1": "0-1",
        "twi_raw": "unitless",
        "sm_support_physical": "m3/m3",
    }

    additional_descriptions = {
        "dvplus_mm_yr": (
            "Direct positive downslope deformation in physical units; "
            "not reliability-fused into HKFI/SNDR."
        ),
        "hkfi_0_1": "Revised support-preserved multiscale HKFI.",
        "sndr_0_1": "Corrected Slope-Normalized Deformation Ratio.",
        "twi_raw": "Untransformed Topographic Wetness Index source variable.",
        "sm_support_physical": (
            "Support-preserved physical ERA5-Land soil-moisture representation; "
            "constant within each coarse support cell."
        ),
    }

    for name, array in additional_arrays.items():
        arrays_by_name[name] = np.where(
            analysis_domain,
            array,
            np.nan,
        )
        provenance_by_name[name] = additional_provenance[name]
        units_by_name[name] = additional_units[name]
        descriptions_by_name[name] = additional_descriptions[name]

    # -------------------------------------------------------------------------
    # Validate all 17 candidate predictors and summarize coverage.
    # -------------------------------------------------------------------------
    if list(arrays_by_name.keys()) != MASTER_FEATURES:
        # The check is deliberately strict so band order remains stable.
        missing = [name for name in MASTER_FEATURES if name not in arrays_by_name]
        extra = [name for name in arrays_by_name if name not in MASTER_FEATURES]
        if missing or extra:
            raise RuntimeError(
                f"Master feature mismatch. Missing={missing}; extra={extra}"
            )

    stats_by_name: dict[str, dict[str, Any]] = {}
    log("[QA] Predictor coverage and statistics:")
    for name in MASTER_FEATURES:
        stats = feature_statistics(
            arrays_by_name[name],
            analysis_domain,
        )
        stats_by_name[name] = stats

        if FAIL_ON_EMPTY_PREDICTOR and stats.get("n_valid", 0) == 0:
            raise RuntimeError(
                f"Required predictor {name} has no valid values in the analysis domain."
            )

        log(
            f"  {name:<24s} coverage={stats.get('coverage_percent', 0.0):7.3f}% "
            f"n={stats.get('n_valid', 0):,}"
        )

    # -------------------------------------------------------------------------
    # Write authoritative stacks.
    # -------------------------------------------------------------------------
    write_multiband_tif(
        STATIC12_STACK_TIF,
        STATIC12_FEATURES,
        arrays_by_name,
        anchor,
        stack_role="authoritative_static12_raw_predictor_stack",
    )
    log(f"[OUT] Static12 stack -> {STATIC12_STACK_TIF}")

    write_multiband_tif(
        MASTER_STACK_TIF,
        MASTER_FEATURES,
        arrays_by_name,
        anchor,
        stack_role="master_candidate_stack_for_models_A_to_F",
    )
    log(f"[OUT] Master 17-band candidate stack -> {MASTER_STACK_TIF}")

    # -------------------------------------------------------------------------
    # Scenario-specific complete-case masks.
    # These allow later site extraction to enforce identical rows when required.
    # -------------------------------------------------------------------------
    scenario_rows: list[dict[str, Any]] = []
    scenario_masks: dict[str, np.ndarray] = {}

    for scenario_name, scenario in MODEL_SCENARIOS.items():
        feature_names = list(scenario["features"])
        scenario_mask = complete_case_mask(
            feature_names,
            arrays_by_name,
            analysis_domain,
        )
        scenario_masks[scenario_name] = scenario_mask

        mask_path = (
            SCENARIO_MASK_DIR
            / f"{scenario_name}_valid_mask_32645_12p5m_byte.tif"
        )
        write_mask_tif(
            mask_path,
            scenario_mask,
            anchor,
            f"complete_case_mask_{scenario_name}",
        )

        band_indices = [
            MASTER_FEATURES.index(name) + 1
            for name in feature_names
        ]

        scenario_stack_path = (
            SCENARIO_STACK_DIR
            / f"{scenario_name}_32645_12p5m_float32.tif"
        )
        if WRITE_SCENARIO_STACKS:
            write_multiband_tif(
                scenario_stack_path,
                feature_names,
                arrays_by_name,
                anchor,
                stack_role=scenario_name,
            )

        scenario_rows.append(
            {
                "scenario": scenario_name,
                "label": scenario["label"],
                "purpose": scenario["purpose"],
                "n_features": len(feature_names),
                "features": "|".join(feature_names),
                "master_band_indices_1_based": "|".join(
                    str(index) for index in band_indices
                ),
                "valid_pixels": int(scenario_mask.sum()),
                "valid_percent_of_analysis_domain": float(
                    100.0
                    * scenario_mask.sum()
                    / max(int(analysis_domain.sum()), 1)
                ),
                "valid_mask_path": str(mask_path),
                "scenario_stack_path": (
                    str(scenario_stack_path)
                    if WRITE_SCENARIO_STACKS
                    else ""
                ),
            }
        )

        log(
            f"[SCENARIO] {scenario_name}: {len(feature_names)} predictors | "
            f"complete pixels={int(scenario_mask.sum()):,} "
            f"({100.0 * scenario_mask.sum() / max(int(analysis_domain.sum()), 1):.3f}%)"
        )

    # Common complete-case domain for all A–F candidate variables.
    common_ablation_valid = analysis_domain.copy()
    for scenario_mask in scenario_masks.values():
        common_ablation_valid &= scenario_mask

    write_mask_tif(
        COMMON_ABLATION_VALID_MASK_TIF,
        common_ablation_valid,
        anchor,
        "common_complete_case_mask_across_models_A_to_F",
    )
    log(
        f"[MASK] Common A–F complete-case pixels: "
        f"{int(common_ablation_valid.sum()):,}/"
        f"{int(analysis_domain.sum()):,} "
        f"({100.0 * common_ablation_valid.sum() / max(int(analysis_domain.sum()), 1):.3f}%)"
    )

    # -------------------------------------------------------------------------
    # Correlation QA. This is descriptive only and does not alter predictors.
    # -------------------------------------------------------------------------
    build_correlation_matrix(
        STATIC12_FEATURES,
        arrays_by_name,
        analysis_domain,
        STATIC_CORR_CSV,
    )
    build_correlation_matrix(
        MASTER_FEATURES,
        arrays_by_name,
        analysis_domain,
        MASTER_CORR_CSV,
    )
    log(f"[OUT] Static correlation matrix -> {STATIC_CORR_CSV}")
    log(f"[OUT] Master correlation matrix -> {MASTER_CORR_CSV}")

    # -------------------------------------------------------------------------
    # Band manifest.
    # -------------------------------------------------------------------------
    with open(BAND_MANIFEST_CSV, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "band_index_1_based",
                "feature",
                "role",
                "units",
                "description",
                "source_path",
                "coverage_percent",
                "exact_grid",
                "resampling_if_needed",
            ],
        )
        writer.writeheader()

        for band_index, feature_name in enumerate(MASTER_FEATURES, start=1):
            provenance = provenance_by_name[feature_name]
            writer.writerow(
                {
                    "band_index_1_based": band_index,
                    "feature": feature_name,
                    "role": (
                        "Static12"
                        if feature_name in STATIC12_FEATURES
                        else "Ablation/source candidate"
                    ),
                    "units": units_by_name[feature_name],
                    "description": descriptions_by_name[feature_name],
                    "source_path": provenance.get("path", ""),
                    "coverage_percent": stats_by_name[feature_name].get(
                        "coverage_percent"
                    ),
                    "exact_grid": provenance.get(
                        "exact_grid",
                        provenance.get("exact_grid_before_read", ""),
                    ),
                    "resampling_if_needed": provenance.get(
                        "resampling_if_needed",
                        "none",
                    ),
                }
            )

    # -------------------------------------------------------------------------
    # Scenario manifests.
    # -------------------------------------------------------------------------
    with open(
        SCENARIO_MANIFEST_CSV,
        "w",
        newline="",
        encoding="utf-8",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=list(scenario_rows[0].keys()),
        )
        writer.writeheader()
        writer.writerows(scenario_rows)

    scenario_json = {
        "generated_at_utc": utc_now_iso(),
        "scientific_rule": (
            "All downstream A–F comparisons must use identical sampled rows, "
            "spatial folds, preprocessing rules, and model settings."
        ),
        "primary_model": "E_physics_guided",
        "reviewer_comment_8_direct_comparison": {
            "baseline": "A_static_baseline",
            "raw_deformation": "B_raw_deformation",
            "physics_guided": "E_physics_guided",
            "source_variable_benchmark": "F_untransformed_sources",
            "key_contrasts": [
                "B minus A: added value of direct DV+",
                "E minus A: total added value of HKFI + SNDR",
                "E minus B: physics-guided formulation versus direct DV+",
                "E minus F: physics-guided formulation versus separate source variables",
            ],
        },
        "scenarios": {
            name: {
                **scenario,
                "n_features": len(scenario["features"]),
                "master_band_indices_1_based": [
                    MASTER_FEATURES.index(feature) + 1
                    for feature in scenario["features"]
                ],
                "valid_mask": str(
                    SCENARIO_MASK_DIR
                    / f"{name}_valid_mask_32645_12p5m_byte.tif"
                ),
            }
            for name, scenario in MODEL_SCENARIOS.items()
        },
    }

    with open(
        SCENARIO_MANIFEST_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            scenario_json,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    # -------------------------------------------------------------------------
    # Provenance and QA metadata.
    # -------------------------------------------------------------------------
    sm_support_mode = load_json_optional(SM_SUPPORT_MODE_JSON)
    hkfi_metadata = load_json_optional(HKFI_METADATA_JSON)
    sndr_metadata = load_json_optional(SNDR_METADATA_JSON)

    stack_manifest = {
        "generated_at_utc": utc_now_iso(),
        "anchor": {
            "path": str(ANCHOR_TIF),
            "epsg": anchor["epsg"],
            "width": anchor["width"],
            "height": anchor["height"],
            "resolution_x": anchor["res_x"],
            "resolution_y": anchor["res_y"],
        },
        "analysis_domain": {
            "definition": "context_mask intersect study_mask; study_mask fallback when context is unavailable",
            "path_used_for_context": str(context_path_used),
            "analysis_domain_mask": str(ANALYSIS_DOMAIN_MASK_TIF),
            "n_pixels": int(analysis_domain.sum()),
            "dynamic_mask_used_for_modelling": False,
            "dynamic_mask_note": "Diagnostic only; not used to restrict predictor coverage.",
        },
        "predictor_design": {
            "n_static": len(STATIC12_FEATURES),
            "retained_tpi": "tpi_s5",
            "tpi_s25_included": False,
            "TWI_in_static12": False,
            "SM_in_static12": False,
            "IRS_as_predictor": False,
            "global_standardization_in_step_6_1": False,
            "LR_scaling_rule": "Fit preprocessing only within each training fold downstream.",
            "XGB_RF_scaling_rule": "No scaling required by default.",
        },
        "stacks": {
            "static12": str(STATIC12_STACK_TIF),
            "master_candidates": str(MASTER_STACK_TIF),
            "master_n_bands": len(MASTER_FEATURES),
            "master_band_order": MASTER_FEATURES,
        },
        "common_ablation_valid_mask": {
            "path": str(COMMON_ABLATION_VALID_MASK_TIF),
            "n_pixels": int(common_ablation_valid.sum()),
            "percent_of_analysis_domain": float(
                100.0
                * common_ablation_valid.sum()
                / max(int(analysis_domain.sum()), 1)
            ),
            "note": (
                "Diagnostic complete-case domain across all A–F predictors. "
                "Final site-level common rows are established downstream after sample extraction."
            ),
        },
        "features": {
            name: {
                "band_index_1_based": MASTER_FEATURES.index(name) + 1,
                "role": (
                    "Static12"
                    if name in STATIC12_FEATURES
                    else "Ablation/source candidate"
                ),
                "units": units_by_name[name],
                "description": descriptions_by_name[name],
                "provenance": provenance_by_name[name],
                "statistics": stats_by_name[name],
            }
            for name in MASTER_FEATURES
        },
        "scenario_manifest_json": str(SCENARIO_MANIFEST_JSON),
        "scenario_manifest_csv": str(SCENARIO_MANIFEST_CSV),
        "band_manifest_csv": str(BAND_MANIFEST_CSV),
        "supplementary_provenance": {
            "sm_support_mode": sm_support_mode,
            "hkfi_metadata": hkfi_metadata,
            "sndr_metadata": sndr_metadata,
            "IRS_path_recorded_only_not_predictor": str(IRS_TIF),
        },
    }

    with open(
        STACK_MANIFEST_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            stack_manifest,
            handle,
            indent=2,
            ensure_ascii=False,
            default=json_safe,
        )

    # -------------------------------------------------------------------------
    # Human-readable QC report.
    # -------------------------------------------------------------------------
    with open(QC_REPORT_TXT, "w", encoding="utf-8") as handle:
        handle.write("=" * 88 + "\n")
        handle.write("STEP 6.1 — AUTHORITATIVE PREDICTOR STACK QA REPORT\n")
        handle.write("=" * 88 + "\n\n")
        handle.write(f"Generated UTC: {utc_now_iso()}\n")
        handle.write(f"Anchor EPSG: {anchor['epsg']}\n")
        handle.write(f"Analysis-domain pixels: {int(analysis_domain.sum()):,}\n")
        handle.write("Dynamic mask used for modelling: NO\n")
        handle.write("Global standardization performed in Step 6.1: NO\n")
        handle.write("IRS included as predictor: NO\n")
        handle.write("Retained TPI: tpi_s5 only\n")
        handle.write("TWI/SM in Static12: NO\n\n")

        handle.write("STATIC12 FEATURES\n")
        handle.write("-" * 88 + "\n")
        for index, name in enumerate(STATIC12_FEATURES, start=1):
            handle.write(f"{index:02d}. {name}\n")

        handle.write("\nMASTER CANDIDATE FEATURES\n")
        handle.write("-" * 88 + "\n")
        for index, name in enumerate(MASTER_FEATURES, start=1):
            stats = stats_by_name[name]
            handle.write(
                f"{index:02d}. {name:<24s} "
                f"coverage={stats.get('coverage_percent', 0.0):8.3f}% "
                f"n={stats.get('n_valid', 0):,}\n"
            )

        handle.write("\nMODEL ABLATION SCENARIOS\n")
        handle.write("-" * 88 + "\n")
        for row in scenario_rows:
            handle.write(
                f"{row['scenario']}: {row['n_features']} predictors | "
                f"valid={row['valid_pixels']:,} "
                f"({row['valid_percent_of_analysis_domain']:.3f}%)\n"
            )
            handle.write(f"  {row['features']}\n")

        handle.write("\nCOMMON A–F COMPLETE-CASE DOMAIN\n")
        handle.write("-" * 88 + "\n")
        handle.write(
            f"Pixels: {int(common_ablation_valid.sum()):,}/"
            f"{int(analysis_domain.sum()):,} "
            f"({100.0 * common_ablation_valid.sum() / max(int(analysis_domain.sum()), 1):.3f}%)\n"
        )
        handle.write(
            "This raster-level mask is diagnostic. Final identical model rows must be fixed "
            "after site-level sample extraction.\n"
        )

        handle.write("\nREVIEWER 1 COMMENT 8 CONTRASTS\n")
        handle.write("-" * 88 + "\n")
        handle.write("B - A : added predictive value of direct DV+ alone\n")
        handle.write("E - A : total added value of HKFI + SNDR\n")
        handle.write("E - B : physics-guided formulation versus direct DV+\n")
        handle.write("E - F : physics-guided formulation versus separate source variables\n")

    log(f"[OUT] Band manifest -> {BAND_MANIFEST_CSV}")
    log(f"[OUT] Scenario manifest JSON -> {SCENARIO_MANIFEST_JSON}")
    log(f"[OUT] Scenario manifest CSV -> {SCENARIO_MANIFEST_CSV}")
    log(f"[OUT] Stack manifest -> {STACK_MANIFEST_JSON}")
    log(f"[OUT] QC report -> {QC_REPORT_TXT}")

    log("=== Step 6.1 completed successfully ===")
    log("Authoritative primary model: Static12 + HKFI + SNDR (14 predictors).")
    log("Reviewer Comment 8 raw-DV+ and source-variable ablation foundations are ready.")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 6.1 failed: {exc}")
        traceback.print_exc()
        raise


# Visualization: Cross-Correlation Matrix Analysis for LSM

In [ ]:
import os
# ============================================================================
# Step 6.1b — Cross-Correlation Matrices for Models A–F
# Uses the authoritative Step 6.1 master stack and model scenario manifest.
# Produces separate cross-correlation matrices for:
#   A — Primary static baseline
#   B — Raw deformation baseline
#   C — HKFI only
#   D — SNDR only
#   E — Proposed integrated model
#   F — Untransformed raw-variable benchmark
#
# Plotting style intentionally follows the earlier cross-correlation
# visualization style:
#   - Times New Roman
#   - compact 2 × 2 inch figure
#   - annotated heatmap
#   - same custom diverging colour ramp
# ============================================================================

import json
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import font_manager as fm

# ----------------------------------------------------------------------------
# Global style
# ----------------------------------------------------------------------------
warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

try:
    _available_fonts = {f.name for f in fm.fontManager.ttflist}
    if "Times New Roman" not in _available_fonts:
        print("[WARN] 'Times New Roman' not found. Using serif fallback.")
except Exception:
    pass


# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    BASE_DIR = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE_DIR / "Results"
    STACK_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    OUT_DIR = STACK_DIR / "Cross_Correlation_Matrices_Models_A_to_F"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Step 6.1 outputs
    STACK_MANIFEST_JSON = STACK_DIR / "stack_manifest.json"
    SCENARIO_MANIFEST_JSON = STACK_DIR / "model_ablation_scenarios.json"
    MASTER_STACK_TIF = STACK_DIR / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
    COMMON_MASK_TIF = STACK_DIR / "common_A_to_F_valid_mask_32645_12p5m_byte.tif"
    ANALYSIS_MASK_TIF = STACK_DIR / "analysis_domain_mask_32645_12p5m_byte.tif"

    # Recommended mask mode for direct comparison across A–F.
    # Options: "common", "scenario", "analysis"
    MASK_MODE = "common"

    # Analysis settings
    NODATA_VALUE = -9999.0
    MIN_VALID_PIXELS = 30
    HIGH_CORRELATION_THRESHOLD = 0.7
    PERFECT_CORRELATION_THRESHOLD = 0.99
    MAX_SAMPLE_PIXELS = 200000
    RANDOM_SEED = 42

    # Visualization settings — same style as earlier code
    FIGURE_SIZE = (2, 2)
    FIGURE_DPI = 600
    SAVE_DPI = 600

    # Only needed if landcover is one-hot; harmless otherwise
    COLLAPSE_LANDCOVER_ONE_HOT = True
    LANDCOVER_ONE_HOT_PREFIX = "landcover_c"
    LANDCOVER_SINGLE_NAME = "landcover"

    FACTOR_DISPLAY_NAMES = {
        "slope_deg": "Slope",
        "northness": "Northness",
        "eastness": "Eastness",
        "profile_curv": "Prof. Curv",
        "plan_curv": "Plan Curv",
        "tpi_s5": "TPI",
        "spi_ln": "SPI",
        "dist_river_m": "Dist.River",
        "dist_road_m": "Dist.Road",
        "aap_mm": "AAP",
        "ndvi": "NDVI",
        "landcover": "Land Use",
        "dvplus_mm_yr": "DV+",
        "hkfi_0_1": "HKFI",
        "sndr_0_1": "SNDR",
        "twi_raw": "TWI",
        "sm_support_physical": "SM",
    }

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]


# ============================================================================
# LOGGING
# ============================================================================
class Logger:
    @staticmethod
    def log(msg: str) -> None:
        print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

    @staticmethod
    def section(title: str) -> None:
        Logger.log("=" * 78)
        Logger.log(title)
        Logger.log("=" * 78)


# ============================================================================
# HELPERS
# ============================================================================
def normalize_name(x: str) -> str:
    return x.strip().lower()


def load_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Missing JSON file: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_mask(path: Path, expected_shape: Tuple[int, int], mask_name: str) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(f"Missing {mask_name}: {path}")
    with rasterio.open(path) as ds:
        raw = ds.read(1, masked=True)
        arr = np.where(raw.mask, 0, raw.data)
    if arr.shape != expected_shape:
        raise ValueError(f"{mask_name} shape mismatch: {arr.shape} vs {expected_shape}")
    return arr > 0


def load_master_stack(stack_path: Path, band_names: List[str]) -> Tuple[np.ndarray, List[str]]:
    if not stack_path.exists():
        raise FileNotFoundError(f"Master stack not found: {stack_path}")

    Logger.log(f"[LOAD] Master stack: {stack_path.name}")
    with rasterio.open(stack_path) as ds:
        stack = ds.read(masked=True)
        nodata = ds.nodata if ds.nodata is not None else Config.NODATA_VALUE
        desc = [ds.descriptions[i] if ds.descriptions[i] else None for i in range(ds.count)]

    stack = np.where(stack.mask, np.nan, stack.data).astype("float64")
    stack[stack == nodata] = np.nan

    actual_n = stack.shape[0]
    if desc and all(d is not None for d in desc):
        names = list(desc)
    else:
        if len(band_names) < actual_n:
            raise ValueError(
                f"Manifest lists {len(band_names)} bands but stack has {actual_n} bands."
            )
        names = band_names[:actual_n]

    Logger.log(f"[OK] Master stack shape = {stack.shape}")
    Logger.log(f"[OK] Band names = {names}")
    return stack, names


def stack_to_named_dict(stack: np.ndarray, names: List[str]) -> Dict[str, np.ndarray]:
    if stack.shape[0] != len(names):
        raise ValueError("Band count does not match band-name count.")
    return {names[i]: stack[i, :, :] for i in range(len(names))}


def _parse_landcover_code(name: str) -> Optional[int]:
    if not name.lower().startswith(Config.LANDCOVER_ONE_HOT_PREFIX):
        return None
    try:
        return int(name.lower().replace(Config.LANDCOVER_ONE_HOT_PREFIX, ""))
    except Exception:
        return None


def collapse_landcover_one_hot(arrays: List[np.ndarray], names: List[str]) -> Tuple[List[np.ndarray], List[str]]:
    if not Config.COLLAPSE_LANDCOVER_ONE_HOT:
        return arrays, names

    lc_indices = []
    lc_codes = []
    for i, nm in enumerate(names):
        code = _parse_landcover_code(nm)
        if code is not None:
            lc_indices.append(i)
            lc_codes.append(code)

    if len(lc_indices) == 0:
        return arrays, names

    Logger.log(f"[LC] Collapsing {len(lc_indices)} one-hot landcover bands to one numeric band")
    lc_stack = np.stack([arrays[i] for i in lc_indices], axis=0)
    k, h, w = lc_stack.shape
    lc_bin = (lc_stack > 0.5).astype("float64")
    s = np.nansum(lc_bin, axis=0)
    exactly_one = s == 1
    arg = np.nanargmax(lc_bin, axis=0)
    codes = np.asarray(lc_codes, dtype="float64")
    codes_arr = np.full((h, w), np.nan, dtype="float64")
    codes_arr[exactly_one] = codes[arg[exactly_one]]

    keep_arrays = [arrays[i] for i in range(len(names)) if i not in lc_indices]
    keep_names = [names[i] for i in range(len(names)) if i not in lc_indices]
    keep_arrays.append(codes_arr)
    keep_names.append(Config.LANDCOVER_SINGLE_NAME)
    return keep_arrays, keep_names


def display_names(names: List[str]) -> List[str]:
    return [Config.FACTOR_DISPLAY_NAMES.get(n, n) for n in names]


def build_pixel_matrix(arrays: List[np.ndarray], mask: np.ndarray, max_samples: Optional[int]) -> np.ndarray:
    n_bands = len(arrays)
    h, w = arrays[0].shape
    idx = np.flatnonzero(mask.ravel())
    if idx.size == 0:
        raise ValueError("Mask has 0 valid pixels.")

    Logger.log(f"[PIXELS] Valid mask pixels: {idx.size:,}")
    if max_samples is not None and idx.size > max_samples:
        rng = np.random.default_rng(Config.RANDOM_SEED)
        idx = rng.choice(idx, size=max_samples, replace=False)
        Logger.log(f"[SAMPLE] Using {len(idx):,} random pixels for correlation")

    matrix = np.full((n_bands, len(idx)), np.nan, dtype="float64")
    for i, arr in enumerate(arrays):
        matrix[i, :] = arr.reshape(h * w)[idx]
    return matrix


def compute_corr(mat: np.ndarray, min_valid: int) -> np.ndarray:
    n = mat.shape[0]
    C = np.full((n, n), np.nan, dtype="float64")
    Logger.log(f"[COMPUTE] Correlation matrix {n}×{n} (min_valid={min_valid})")
    for i in range(n):
        C[i, i] = 1.0
        xi = mat[i]
        for j in range(i + 1, n):
            xj = mat[j]
            ok = np.isfinite(xi) & np.isfinite(xj)
            if ok.sum() >= min_valid:
                a = xi[ok]
                b = xj[ok]
                if a.std() > 1e-12 and b.std() > 1e-12:
                    r = np.corrcoef(a, b)[0, 1]
                    if np.isfinite(r):
                        C[i, j] = r
                        C[j, i] = r
    return C


def analyze_corr(C: np.ndarray, names: List[str]) -> Dict[str, Any]:
    mask = ~np.eye(len(names), dtype=bool)
    vals = C[mask]
    vals = vals[np.isfinite(vals)]

    out = {
        "mean_abs": None,
        "max": None,
        "min": None,
        "std": None,
        "high_pairs": [],
        "perfect_pairs": [],
    }
    if vals.size == 0:
        return out

    out["mean_abs"] = float(np.mean(np.abs(vals)))
    out["max"] = float(np.max(vals))
    out["min"] = float(np.min(vals))
    out["std"] = float(np.std(vals))

    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            r = C[i, j]
            if not np.isfinite(r):
                continue
            ar = abs(r)
            info = {
                "var1": names[i],
                "var2": names[j],
                "display_var1": Config.FACTOR_DISPLAY_NAMES.get(names[i], names[i]),
                "display_var2": Config.FACTOR_DISPLAY_NAMES.get(names[j], names[j]),
                "correlation": float(r),
            }
            if ar >= Config.PERFECT_CORRELATION_THRESHOLD:
                out["perfect_pairs"].append(info)
            elif ar >= Config.HIGH_CORRELATION_THRESHOLD:
                out["high_pairs"].append(info)
    return out


# ============================================================================
# VISUALIZATION — same style as previous code
# ============================================================================
def save_heatmap_matplotlib(C: np.ndarray, names: List[str], out_png: Path, out_pdf: Path, out_svg: Path) -> None:
    plt.style.use("default")
    labels = display_names(names)
    n = len(labels)

    base_font_size = max(2, 8 - n * 0.3)
    tick_font_size = max(1.5, base_font_size - 0.5)
    value_font_size = max(1, base_font_size - 2)
    colorbar_font_size = max(2, base_font_size - 0.5)

    fig, ax = plt.subplots(figsize=Config.FIGURE_SIZE, dpi=Config.FIGURE_DPI)

    colors = [
        "#053061", "#2166ac", "#4393c3", "#92c5de", "#d1e5f0",
        "#f7f7f7",
        "#fdbf6f", "#fd8d3c", "#e31a1c", "#bd0026", "#67001f",
    ]
    cmap = LinearSegmentedColormap.from_list("RdBu_custom", colors)

    C_flipped = np.flipud(C)
    im = ax.imshow(C_flipped, cmap=cmap, vmin=-1, vmax=1, aspect="equal", interpolation="nearest")

    for i in range(n):
        for j in range(n):
            orig_i = n - 1 - i
            if not np.isnan(C[orig_i, j]):
                val = C[orig_i, j]
                text_color = "white" if abs(val) > 0.6 else "black"

                if abs(val) < 0.01:
                    text = "0"
                elif abs(val) >= 0.995:
                    text = "1" if val > 0 else "-1"
                else:
                    text = f"{val:.2f}"

                ax.text(
                    j, i, text,
                    ha="center", va="center",
                    color=text_color,
                    fontsize=value_font_size,
                    fontweight="bold" if abs(val) > 0.7 else "normal",
                )

    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=tick_font_size)
    ax.set_yticklabels(labels[::-1], rotation=0, fontsize=tick_font_size)

    cbar = plt.colorbar(im, ax=ax, shrink=0.6, pad=0.01, aspect=15)
    cbar.set_label("Correlation", fontsize=colorbar_font_size + 1, labelpad=2, fontweight="bold")
    cbar.ax.tick_params(labelsize=colorbar_font_size, width=0.5, length=2)

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(-0.5, n - 0.5)

    for i in range(n + 1):
        ax.axhline(i - 0.5, color="white", linewidth=0.3, alpha=0.8)
        ax.axvline(i - 0.5, color="white", linewidth=0.3, alpha=0.8)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.5)
        spine.set_color("#333333")

    plt.tight_layout(pad=0.1)

    for path in (out_png, out_pdf, out_svg):
        if path.exists():
            try:
                path.unlink()
            except Exception:
                pass

    save_kwargs = {
        "dpi": Config.SAVE_DPI,
        "bbox_inches": "tight",
        "facecolor": "white",
        "edgecolor": "none",
        "pad_inches": 0.02,
    }
    fig.savefig(out_png, format="png", **save_kwargs)
    fig.savefig(out_pdf, format="pdf", **save_kwargs)
    fig.savefig(out_svg, format="svg", bbox_inches="tight", facecolor="white", edgecolor="none", pad_inches=0.02)
    plt.show()
    plt.close(fig)


def save_csv(C: np.ndarray, names: List[str], out_csv: Path) -> None:
    df = pd.DataFrame(C, index=names, columns=names)
    df.to_csv(out_csv, float_format="%.4f")
    Logger.log(f"[CSV] Saved: {out_csv.name}")


def save_summary(stats: Dict[str, Any], names: List[str], out_txt: Path, scenario_label: str, mask_mode: str) -> None:
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write("=" * 90 + "\n")
        f.write(f"CROSS-CORRELATION MATRIX REPORT — {scenario_label}\n")
        f.write("=" * 90 + "\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M:%S}\n")
        f.write(f"Mask mode: {mask_mode}\n\n")

        f.write(f"Predictors analyzed ({len(names)}):\n")
        f.write("-" * 50 + "\n")
        for i, nm in enumerate(names, 1):
            display_nm = Config.FACTOR_DISPLAY_NAMES.get(nm, nm)
            f.write(f"  {i:2d}. {nm:<24s} ({display_nm})\n")
        f.write("\n")

        f.write("Correlation Statistics:\n")
        f.write("-" * 50 + "\n")
        if stats["mean_abs"] is None:
            f.write("  No valid correlations computed.\n\n")
        else:
            f.write(f"  Mean |r|     : {stats['mean_abs']:.3f}\n")
            f.write(f"  Max r        : {stats['max']:.3f}\n")
            f.write(f"  Min r        : {stats['min']:.3f}\n")
            f.write(f"  Std r        : {stats['std']:.3f}\n")
            f.write(f"  Total pairs  : {len(names) * (len(names) - 1) // 2}\n\n")

        f.write(f"Perfect correlations (|r| ≥ {Config.PERFECT_CORRELATION_THRESHOLD}):\n")
        f.write("-" * 70 + "\n")
        if stats["perfect_pairs"]:
            for p in stats["perfect_pairs"]:
                f.write(f"  {p['display_var1']} ↔ {p['display_var2']}: r={p['correlation']:.4f}\n")
        else:
            f.write("  None\n")
        f.write("\n")

        f.write(f"High correlations ({Config.HIGH_CORRELATION_THRESHOLD:.1f} ≤ |r| < {Config.PERFECT_CORRELATION_THRESHOLD}):\n")
        f.write("-" * 80 + "\n")
        if stats["high_pairs"]:
            for p in stats["high_pairs"]:
                f.write(f"  {p['display_var1']} ↔ {p['display_var2']}: r={p['correlation']:.4f}\n")
        else:
            f.write("  None\n")


# ============================================================================
# MODEL-SPECIFIC ANALYSIS
# ============================================================================
def get_scenario_mask_path(scenario_name: str, scenario_info: Dict[str, Any]) -> Path:
    if "valid_mask" in scenario_info:
        return Path(scenario_info["valid_mask"])
    raise KeyError(f"Scenario {scenario_name} does not define 'valid_mask'.")


def choose_mask(
    mask_mode: str,
    scenario_name: str,
    scenario_info: Dict[str, Any],
    expected_shape: Tuple[int, int],
) -> np.ndarray:
    if mask_mode == "common":
        return load_mask(Config.COMMON_MASK_TIF, expected_shape, "common A–F complete-case mask")
    if mask_mode == "analysis":
        return load_mask(Config.ANALYSIS_MASK_TIF, expected_shape, "analysis-domain mask")
    if mask_mode == "scenario":
        scenario_mask_path = get_scenario_mask_path(scenario_name, scenario_info)
        return load_mask(scenario_mask_path, expected_shape, f"scenario mask for {scenario_name}")
    raise ValueError(f"Unsupported mask mode: {mask_mode}")


def analyze_model_scenario(
    scenario_name: str,
    scenario_info: Dict[str, Any],
    arrays_by_name: Dict[str, np.ndarray],
    expected_shape: Tuple[int, int],
) -> Dict[str, Any]:
    features = list(scenario_info["features"])
    scenario_label = scenario_info.get("label", scenario_name)

    arrays = []
    names = []
    for feature in features:
        if feature not in arrays_by_name:
            raise KeyError(f"Feature '{feature}' required by {scenario_name} not found in master stack")
        arrays.append(arrays_by_name[feature])
        names.append(feature)

    arrays, names = collapse_landcover_one_hot(arrays, names)

    mask = choose_mask(Config.MASK_MODE, scenario_name, scenario_info, expected_shape)
    Logger.log(f"[MODEL] {scenario_name} | predictors={len(names)} | mask pixels={int(mask.sum()):,}")

    mat = build_pixel_matrix(arrays, mask, Config.MAX_SAMPLE_PIXELS)
    C = compute_corr(mat, Config.MIN_VALID_PIXELS)
    stats = analyze_corr(C, names)

    short_name = scenario_name.replace("_", "-")
    prefix = f"{short_name}_Cross-Correlation_Matrix"
    out_png = Config.OUT_DIR / f"{prefix}.png"
    out_pdf = Config.OUT_DIR / f"{prefix}.pdf"
    out_svg = Config.OUT_DIR / f"{prefix}.svg"
    out_csv = Config.OUT_DIR / f"{prefix}.csv"
    out_txt = Config.OUT_DIR / f"{prefix}_Summary.txt"

    save_heatmap_matplotlib(C, names, out_png, out_pdf, out_svg)
    save_csv(C, names, out_csv)
    save_summary(stats, names, out_txt, scenario_label, Config.MASK_MODE)

    return {
        "scenario_name": scenario_name,
        "scenario_label": scenario_label,
        "n_predictors": len(names),
        "predictors": names,
        "mask_mode": Config.MASK_MODE,
        "mask_pixels": int(mask.sum()),
        "mean_abs": stats["mean_abs"],
        "max_r": stats["max"],
        "min_r": stats["min"],
        "std_r": stats["std"],
        "n_high_pairs": len(stats["high_pairs"]),
        "n_perfect_pairs": len(stats["perfect_pairs"]),
        "png": str(out_png),
        "pdf": str(out_pdf),
        "svg": str(out_svg),
        "csv": str(out_csv),
        "summary_txt": str(out_txt),
    }


# ============================================================================
# MAIN
# ============================================================================
def main() -> None:
    Logger.section("Step 6.1b — Cross-Correlation Matrices for Models A–F")

    stack_manifest = load_json(Config.STACK_MANIFEST_JSON)
    scenario_manifest = load_json(Config.SCENARIO_MANIFEST_JSON)

    master_band_order = stack_manifest["stacks"]["master_band_order"]
    master_stack, master_names = load_master_stack(Config.MASTER_STACK_TIF, master_band_order)
    arrays_by_name = stack_to_named_dict(master_stack, master_names)
    expected_shape = master_stack.shape[1:]

    scenarios = scenario_manifest["scenarios"]

    summary_rows = []
    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in scenarios:
            raise KeyError(f"Scenario missing from scenario manifest: {scenario_name}")
        Logger.section(f"Analyzing {scenario_name}")
        row = analyze_model_scenario(
            scenario_name=scenario_name,
            scenario_info=scenarios[scenario_name],
            arrays_by_name=arrays_by_name,
            expected_shape=expected_shape,
        )
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = Config.OUT_DIR / "Cross-Correlation_Matrix_All_Models_Summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    Logger.log(f"[OUT] Summary table: {summary_csv}")

    summary_json = Config.OUT_DIR / "Cross-Correlation_Matrix_All_Models_Summary.json"
    with open(summary_json, "w", encoding="utf-8") as f:
        json.dump(summary_rows, f, indent=2)
    Logger.log(f"[OUT] Summary JSON: {summary_json}")

    Logger.section("Completed")
    Logger.log(f"Mask mode used for all models: {Config.MASK_MODE}")
    Logger.log(f"Output directory: {Config.OUT_DIR}")


if __name__ == "__main__":
    main()


# Visualization:Multicollinearity Testing-VIF/TOL

In [ ]:
# =============================================================================
# Step 6.1c — Primary VIF and Tolerance Analysis 
# =============================================================================
#
# Purpose
# -------
# Perform a separate multicollinearity analysis for each of the six predictor
# configurations defined in the revised Step 6.1 scenario manifest:
#
#   Model A — Primary static baseline
#   Model B — Raw deformation baseline
#   Model C — HKFI only
#   Model D — SNDR only
#   Model E — Proposed integrated model
#   Model F — Untransformed raw-variable benchmark
#
# PRIMARY VIF DESIGN
# ------------------
# 1. All six analyses use the SAME reproducible sample of pixels from the common
#    A–F complete-case mask produced by Step 6.1.
#
# 2. Continuous predictors are treated as one-degree-of-freedom model terms.
#
# 3. Land Use is treated correctly as a categorical factor:
#       - internally expanded to K-1 dummy variables;
#       - analyzed as ONE grouped model term using generalized VIF (GVIF);
#       - reported on a VIF-comparable variance scale as:
#
#             VIF_equivalent = GVIF ** (1 / df)
#
#         where df is the number of dummy columns representing the factor.
#
#       For a one-degree-of-freedom predictor, GVIF ** (1/df) is exactly the
#       ordinary VIF. Therefore the same VIF=10 and Tolerance=0.10 thresholds
#       remain directly interpretable in the primary summary plots.
#
# 4. The corresponding grouped tolerance is:
#
#             Tolerance = 1 / VIF_equivalent
#
# 5. A detailed dummy-level VIF table is also written for QA, but the primary
#    plots contain one bar per original conditioning factor, including one
#    grouped "Land Use" bar.
#
# 6. Standardization is used only for numerical conditioning. VIF/GVIF is
#    invariant to linear rescaling of continuous predictors.
#
# 7. This analysis is predictor-only and can be run immediately after Step 6.1.
#    Landslide/non-landslide labels are NOT required.
#
# Plot style
# ----------
# Matches the previous VIF/Tolerance visualization:
#   - Times New Roman
#   - 2 × 2 inch figures
#   - 900 dpi
#   - horizontal bars
#   - red threshold lines
#   - separate VIF and Tolerance plots for every model
# =============================================================================

from __future__ import annotations

import json
import os
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"
    STACK_DIR = RESULTS_DIR / "Step-6.1-Stacks"

    # Authoritative Step 6.1 outputs
    STACK_MANIFEST_JSON = STACK_DIR / "stack_manifest.json"
    SCENARIO_MANIFEST_JSON = STACK_DIR / "model_ablation_scenarios.json"
    MASTER_STACK_TIF = (
        STACK_DIR
        / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
    )
    COMMON_MASK_TIF = (
        STACK_DIR
        / "common_A_to_F_valid_mask_32645_12p5m_byte.tif"
    )

    OUT_DIR = STACK_DIR / "Primary_VIF_Tolerance_Models_A_to_F"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Scenario order
    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    # Only Land Use is categorical in the current predictor design
    CATEGORICAL_FEATURES = {"landcover"}

    # Multicollinearity thresholds
    VIF_THRESHOLD = 10.0
    TOLERANCE_THRESHOLD = 0.10

    # Common reproducible raster sample
    MAX_SAMPLES_FOR_VIF = 50_000
    RANDOM_SEED = 42

    # Numerical safeguards
    MIN_ROWS = 100
    NEAR_CONSTANT_STD = 1.0e-12
    CORR_EIGENVALUE_FLOOR = 1.0e-10

    # Plot style — same as previous script
    CHART_SIZE = (2, 2)
    DISPLAY_DPI = 900
    SAVE_DPI = 900

    DISPLAY_LABELS = {
        "slope_deg": "Slope",
        "northness": "Northness",
        "eastness": "Eastness",
        "profile_curv": "Prof. Curv",
        "plan_curv": "Plan Curv",
        "tpi_s5": "TPI",
        "spi_ln": "SPI",
        "dist_river_m": "Dist.River",
        "dist_road_m": "Dist.Road",
        "aap_mm": "AAP",
        "ndvi": "NDVI",
        "landcover": "Land Use",
        "dvplus_mm_yr": "DV+",
        "hkfi_0_1": "HKFI",
        "sndr_0_1": "SNDR",
        "twi_raw": "TWI",
        "sm_support_physical": "SM",
    }


# =============================================================================
# LOGGING
# =============================================================================
def pp(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


# =============================================================================
# GENERAL I/O
# =============================================================================
def load_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Missing JSON file: {path}")
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def load_master_stack(
    path: Path,
    manifest_band_order: Sequence[str],
) -> Tuple[np.ndarray, List[str]]:
    if not path.exists():
        raise FileNotFoundError(f"Master stack not found: {path}")

    pp(f"[LOAD] {path}")

    with rasterio.open(path) as dataset:
        masked = dataset.read(masked=True)
        nodata = dataset.nodata

        descriptions = [
            dataset.descriptions[index]
            for index in range(dataset.count)
        ]

    stack = np.where(
        masked.mask,
        np.nan,
        masked.data,
    ).astype("float64")

    if nodata is not None and np.isfinite(float(nodata)):
        stack[stack == float(nodata)] = np.nan

    # Prefer embedded band descriptions from Step 6.1
    if descriptions and all(description for description in descriptions):
        names = [str(description) for description in descriptions]
    else:
        names = list(manifest_band_order[: stack.shape[0]])

    if len(names) != stack.shape[0]:
        raise ValueError(
            f"Master stack has {stack.shape[0]} bands but only "
            f"{len(names)} usable names were found."
        )

    pp(f"[STACK] shape={stack.shape}")
    pp(f"[STACK] bands={names}")

    return stack, names


def load_common_mask(
    path: Path,
    expected_shape: Tuple[int, int],
) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(f"Common A–F mask not found: {path}")

    with rasterio.open(path) as dataset:
        masked = dataset.read(1, masked=True)
        values = np.where(masked.mask, 0, masked.data)

    if values.shape != expected_shape:
        raise ValueError(
            f"Common mask shape {values.shape} does not match "
            f"master-stack shape {expected_shape}."
        )

    mask = values > 0

    if not mask.any():
        raise RuntimeError("Common A–F complete-case mask contains no valid pixels.")

    return mask


def choose_common_sample_indices(
    common_mask: np.ndarray,
    max_samples: int,
    seed: int,
) -> np.ndarray:
    valid_index = np.flatnonzero(common_mask.ravel())

    if valid_index.size < Config.MIN_ROWS:
        raise RuntimeError(
            f"Only {valid_index.size} common valid pixels are available; "
            f"at least {Config.MIN_ROWS} are required."
        )

    rng = np.random.default_rng(seed)

    if valid_index.size > max_samples:
        valid_index = rng.choice(
            valid_index,
            size=max_samples,
            replace=False,
        )

    # Sort for stable/reproducible downstream extraction
    valid_index = np.sort(valid_index)

    pp(
        f"[SAMPLE] Common A–F VIF sample: {len(valid_index):,} pixels "
        f"(seed={seed})"
    )

    return valid_index


def build_array_lookup(
    stack: np.ndarray,
    names: Sequence[str],
) -> Dict[str, np.ndarray]:
    return {
        str(name): stack[index]
        for index, name in enumerate(names)
    }


# =============================================================================
# DATA PREPARATION
# =============================================================================
def extract_scenario_dataframe(
    arrays_by_name: Dict[str, np.ndarray],
    features: Sequence[str],
    common_sample_indices: np.ndarray,
) -> pd.DataFrame:
    rows = {}

    for feature in features:
        if feature not in arrays_by_name:
            raise KeyError(
                f"Feature '{feature}' is required by the scenario but "
                "was not found in the Step 6.1 master stack."
            )

        values = arrays_by_name[feature].ravel()[common_sample_indices]
        rows[feature] = values

    df = pd.DataFrame(rows)

    # Step 6.1 common mask should already guarantee completeness, but retain
    # this strict QA check rather than silently using different rows by model.
    finite_numeric = []
    for feature in features:
        if feature in Config.CATEGORICAL_FEATURES:
            finite_numeric.append(df[feature].notna())
        else:
            numeric = pd.to_numeric(df[feature], errors="coerce")
            df[feature] = numeric
            finite_numeric.append(np.isfinite(numeric))

    complete = np.logical_and.reduce(
        [series.to_numpy() for series in finite_numeric]
    )

    if not complete.all():
        n_bad = int((~complete).sum())
        raise RuntimeError(
            f"Scenario has {n_bad:,} incomplete rows inside the common A–F "
            "mask. Re-run Step 6.1 and verify the common mask."
        )

    return df


def build_vif_design_matrix(
    df: pd.DataFrame,
    features: Sequence[str],
) -> Tuple[pd.DataFrame, Dict[str, List[str]], Dict[str, Any]]:
    """
    Build the internal design matrix used for VIF/GVIF.

    Continuous predictors:
        one standardized column each.

    Land Use:
        K-1 dummy columns, with the lowest sorted category used as reference.

    Returns
    -------
    design_df
        Numeric design matrix WITHOUT intercept.
    term_columns
        Mapping from original model term -> design-matrix column(s).
    encoding_metadata
        Metadata describing categorical reference categories.
    """
    design_parts = []
    term_columns: Dict[str, List[str]] = {}
    encoding_metadata: Dict[str, Any] = {}

    continuous_features = [
        feature
        for feature in features
        if feature not in Config.CATEGORICAL_FEATURES
    ]

    # -------------------------------------------------------------------------
    # Continuous predictors: standardize only for numerical conditioning.
    # -------------------------------------------------------------------------
    if continuous_features:
        continuous = df[continuous_features].astype(float).copy()

        near_constant = [
            column
            for column in continuous.columns
            if float(continuous[column].std(ddof=0)) < Config.NEAR_CONSTANT_STD
        ]
        if near_constant:
            raise RuntimeError(
                f"Near-constant continuous predictors detected: {near_constant}"
            )

        scaler = StandardScaler()
        standardized = pd.DataFrame(
            scaler.fit_transform(continuous),
            columns=continuous_features,
            index=df.index,
        )
        design_parts.append(standardized)

        for feature in continuous_features:
            term_columns[feature] = [feature]

    # -------------------------------------------------------------------------
    # Categorical predictors: K-1 dummy coding.
    # -------------------------------------------------------------------------
    for feature in features:
        if feature not in Config.CATEGORICAL_FEATURES:
            continue

        categorical = df[feature].copy()

        # Preserve actual source classes but use strings as stable dummy labels.
        unique_values = sorted(
            pd.Series(categorical)
            .dropna()
            .unique()
            .tolist()
        )

        if len(unique_values) < 2:
            raise RuntimeError(
                f"Categorical predictor '{feature}' has fewer than two "
                "categories in the common VIF sample."
            )

        reference_value = unique_values[0]

        dummies = pd.get_dummies(
            categorical.astype(str),
            prefix=feature,
            prefix_sep="__",
            drop_first=True,
            dtype=float,
        )

        if dummies.shape[1] == 0:
            raise RuntimeError(
                f"Dummy encoding produced no columns for categorical factor "
                f"'{feature}'."
            )

        design_parts.append(dummies)
        term_columns[feature] = list(dummies.columns)

        encoding_metadata[feature] = {
            "n_categories": len(unique_values),
            "categories": [
                value.item() if isinstance(value, np.generic) else value
                for value in unique_values
            ],
            "reference_category": (
                reference_value.item()
                if isinstance(reference_value, np.generic)
                else reference_value
            ),
            "dummy_columns": list(dummies.columns),
        }

    design_df = pd.concat(
        design_parts,
        axis=1,
    )

    if design_df.isna().any().any():
        raise RuntimeError("Design matrix unexpectedly contains NaN values.")

    # Remove any accidental zero-variance dummy column.
    zero_variance_columns = [
        column
        for column in design_df.columns
        if float(design_df[column].std(ddof=0)) < Config.NEAR_CONSTANT_STD
    ]

    if zero_variance_columns:
        raise RuntimeError(
            "Zero-variance columns detected after encoding: "
            f"{zero_variance_columns}"
        )

    if len(design_df) <= design_df.shape[1] + 1:
        raise RuntimeError(
            f"Too few rows ({len(design_df)}) for "
            f"{design_df.shape[1]} internal design columns."
        )

    return design_df, term_columns, encoding_metadata


# =============================================================================
# VIF / GVIF CALCULATION
# =============================================================================
def safe_logdet_correlation(
    matrix: np.ndarray,
    eigen_floor: float,
) -> Tuple[float, bool]:
    """
    Return a stable log-determinant for a positive semi-definite correlation
    matrix. If numerical singularity is detected, eigenvalues are floored.
    """
    matrix = np.asarray(matrix, dtype="float64")

    if matrix.shape == (1, 1):
        return 0.0, False

    sign, logdet = np.linalg.slogdet(matrix)

    if sign > 0 and np.isfinite(logdet):
        return float(logdet), False

    # Numerical fallback using a symmetrized eigen-decomposition
    symmetric = 0.5 * (matrix + matrix.T)
    eigenvalues = np.linalg.eigvalsh(symmetric)
    clipped = np.clip(
        eigenvalues,
        eigen_floor,
        None,
    )
    regularized_logdet = float(
        np.sum(np.log(clipped))
    )

    return regularized_logdet, True


def calculate_term_level_gvif(
    design_df: pd.DataFrame,
    term_columns: Dict[str, List[str]],
) -> pd.DataFrame:
    """
    Calculate generalized VIF by original model term.

    Formula:
        GVIF_j = det(R_jj) * det(R_-j,-j) / det(R)

    Reported primary VIF-equivalent:
        GVIF_j ** (1 / df_j)

    This equals ordinary VIF when df_j = 1.
    """
    X = design_df.astype(float).to_numpy()
    column_names = list(design_df.columns)

    correlation = np.corrcoef(
        X,
        rowvar=False,
    )

    if correlation.ndim != 2:
        raise RuntimeError("Could not construct a valid predictor correlation matrix.")

    full_logdet, full_regularized = safe_logdet_correlation(
        correlation,
        Config.CORR_EIGENVALUE_FLOOR,
    )

    results = []

    for term, columns in term_columns.items():
        indices_term = [
            column_names.index(column)
            for column in columns
        ]
        indices_other = [
            index
            for index in range(len(column_names))
            if index not in indices_term
        ]

        df_term = len(indices_term)

        R_term = correlation[
            np.ix_(indices_term, indices_term)
        ]

        term_logdet, term_regularized = safe_logdet_correlation(
            R_term,
            Config.CORR_EIGENVALUE_FLOOR,
        )

        if indices_other:
            R_other = correlation[
                np.ix_(indices_other, indices_other)
            ]
            other_logdet, other_regularized = safe_logdet_correlation(
                R_other,
                Config.CORR_EIGENVALUE_FLOOR,
            )
        else:
            other_logdet = 0.0
            other_regularized = False

        log_gvif = (
            term_logdet
            + other_logdet
            - full_logdet
        )

        # Prevent overflow while preserving diagnostic meaning.
        log_gvif = float(
            np.clip(
                log_gvif,
                -700.0,
                700.0,
            )
        )

        gvif = float(np.exp(log_gvif))
        gvif = max(gvif, 1.0)

        vif_equivalent = float(
            gvif ** (1.0 / max(df_term, 1))
        )
        tolerance = (
            1.0 / vif_equivalent
            if np.isfinite(vif_equivalent) and vif_equivalent > 0
            else np.nan
        )

        gvif_adjusted_1_over_2df = float(
            gvif ** (
                1.0
                / (
                    2.0
                    * max(df_term, 1)
                )
            )
        )

        if (
            np.isfinite(vif_equivalent)
            and vif_equivalent >= Config.VIF_THRESHOLD
        ):
            status = "High Multicollinearity"
        else:
            status = "Acceptable"

        results.append(
            {
                "Variable": term,
                "DF": int(df_term),
                "Statistic_Type": (
                    "VIF"
                    if df_term == 1
                    else "Grouped GVIF^(1/DF)"
                ),
                "GVIF_Raw": gvif,
                "GVIF_Adjusted_1_over_2DF": gvif_adjusted_1_over_2df,
                "VIF": vif_equivalent,
                "Tolerance": tolerance,
                "Status": status,
                "Numerical_Regularization_Used": bool(
                    full_regularized
                    or term_regularized
                    or other_regularized
                ),
                "Internal_Columns": "|".join(columns),
            }
        )

    return pd.DataFrame(results)


def calculate_dummy_level_vif(
    design_df: pd.DataFrame,
    term_columns: Dict[str, List[str]],
) -> pd.DataFrame:
    """
    QA-only internal VIF table for every design column, including individual
    Land Use dummy variables.

    An intercept is explicitly added so each auxiliary regression includes a
    constant term.
    """
    X = design_df.astype(float).copy()
    X_with_constant = X.copy()
    X_with_constant.insert(
        0,
        "__intercept__",
        1.0,
    )

    reverse_term_lookup = {}
    for term, columns in term_columns.items():
        for column in columns:
            reverse_term_lookup[column] = term

    rows = []

    for column_index, column in enumerate(
        X_with_constant.columns
    ):
        if column == "__intercept__":
            continue

        try:
            vif = float(
                variance_inflation_factor(
                    X_with_constant.to_numpy(),
                    column_index,
                )
            )
        except Exception:
            vif = np.nan

        tolerance = (
            1.0 / vif
            if np.isfinite(vif) and vif > 0
            else np.nan
        )

        rows.append(
            {
                "Original_Term": reverse_term_lookup.get(
                    column,
                    column,
                ),
                "Internal_Column": column,
                "Dummy_Level_VIF": vif,
                "Dummy_Level_Tolerance": tolerance,
            }
        )

    return pd.DataFrame(rows)


# =============================================================================
# PLOTTING — SAME STYLE AS PREVIOUS VIF/TOLERANCE FIGURES
# =============================================================================
def distinct_colors(
    n: int,
    base_palette: str = "tab20",
):
    base = (
        list(getattr(plt.cm, base_palette).colors)
        if hasattr(plt.cm, base_palette)
        else list(plt.cm.tab20.colors)
    )

    if n <= len(base):
        return base[:n]

    extra = n - len(base)
    extras = [
        mcolors.hsv_to_rgb(
            (
                i / max(extra, 1),
                0.55,
                0.90,
            )
        )
        for i in range(extra)
    ]

    return base + extras


def apply_times_new_roman() -> None:
    mpl.rcParams.update(
        {
            "font.family": "serif",
            "font.serif": [
                "Times New Roman",
                "Times",
                "DejaVu Serif",
            ],
            "mathtext.fontset": "stix",
            "axes.titlesize": 5,
            "axes.labelsize": 5,
            "xtick.labelsize": 4,
            "ytick.labelsize": 4,
            "legend.fontsize": 4,
            "font.size": 4,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def apply_tiny_axes_style(
    ax,
    xlabel: str,
    ylabel: str,
) -> None:
    ax.set_xlabel(
        xlabel,
        fontsize=5,
        fontweight="bold",
        labelpad=2,
    )
    ax.set_ylabel(
        ylabel,
        fontsize=5,
        fontweight="bold",
        labelpad=2,
    )

    ax.tick_params(
        axis="x",
        labelsize=4,
    )
    ax.tick_params(
        axis="y",
        labelsize=4,
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.3)
        spine.set_color("gray")

    ax.grid(
        True,
        axis="x",
        alpha=0.3,
        linestyle="-",
        linewidth=0.4,
    )
    ax.set_axisbelow(True)


def display_label(variable: str) -> str:
    return Config.DISPLAY_LABELS.get(
        variable,
        variable,
    )


def save_figure_triplet(
    fig,
    png_path: Path,
    pdf_path: Path,
    svg_path: Path,
) -> None:
    for path in (
        png_path,
        pdf_path,
        svg_path,
    ):
        if path.exists():
            try:
                path.unlink()
            except Exception:
                pass

    save_kwargs = {
        "dpi": Config.SAVE_DPI,
        "bbox_inches": "tight",
        "facecolor": "white",
        "pad_inches": 0.02,
    }

    fig.savefig(
        png_path,
        format="png",
        **save_kwargs,
    )
    fig.savefig(
        pdf_path,
        format="pdf",
        **save_kwargs,
    )
    fig.savefig(
        svg_path,
        format="svg",
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.02,
    )


def create_tolerance_chart(
    table: pd.DataFrame,
    out_png: Path,
    out_pdf: Path,
    out_svg: Path,
) -> None:
    apply_times_new_roman()

    data = table.dropna(
        subset=["Tolerance"]
    ).copy()

    if data.empty:
        raise RuntimeError(
            "No valid tolerance values are available for plotting."
        )

    data["Display_Label"] = data["Variable"].map(
        display_label
    )
    data = data.sort_values(
        "Tolerance",
        ascending=True,
    )

    plt.style.use("default")

    fig, ax = plt.subplots(
        figsize=Config.CHART_SIZE,
        dpi=Config.DISPLAY_DPI,
    )

    colors = distinct_colors(
        len(data),
        base_palette="tab20",
    )

    bars = ax.barh(
        range(len(data)),
        data["Tolerance"],
        color=colors,
        alpha=0.95,
        height=0.55,
    )

    for bar, tolerance in zip(
        bars,
        data["Tolerance"],
    ):
        if tolerance > 0.92:
            ax.text(
                tolerance - 0.03,
                bar.get_y()
                + bar.get_height() / 2,
                f"{tolerance:.2f}",
                va="center",
                ha="right",
                fontsize=2.5,
                fontweight="bold",
                color="white",
                fontfamily="Times New Roman",
            )
        else:
            ax.text(
                tolerance + 0.02,
                bar.get_y()
                + bar.get_height() / 2,
                f"{tolerance:.2f}",
                va="center",
                ha="left",
                fontsize=2.5,
                fontweight="bold",
                fontfamily="Times New Roman",
            )

    ax.axvline(
        x=Config.TOLERANCE_THRESHOLD,
        color="red",
        linestyle="--",
        linewidth=1.0,
        alpha=0.9,
    )
    ax.text(
        Config.TOLERANCE_THRESHOLD,
        len(data) - 0.5,
        f"{Config.TOLERANCE_THRESHOLD:.2f}",
        ha="center",
        va="bottom",
        fontsize=3.0,
        color="red",
        fontweight="bold",
        fontfamily="Times New Roman",
    )

    ax.set_xlim(
        0,
        1.0,
    )
    ax.set_yticks(
        range(len(data))
    )
    ax.set_yticklabels(
        data["Display_Label"],
        fontsize=4,
        fontfamily="Times New Roman",
    )

    apply_tiny_axes_style(
        ax,
        xlabel="Tolerance",
        ylabel="Conditioning Factors",
    )

    plt.tight_layout(
        pad=0.1
    )

    save_figure_triplet(
        fig,
        out_png,
        out_pdf,
        out_svg,
    )

    plt.show()
    plt.close(fig)
    mpl.rcParams.update(
        mpl.rcParamsDefault
    )

    pp(f"[FIGURE] Tolerance chart -> {out_png}")


def create_vif_chart(
    table: pd.DataFrame,
    out_png: Path,
    out_pdf: Path,
    out_svg: Path,
) -> None:
    apply_times_new_roman()

    data = table.dropna(
        subset=["VIF"]
    ).copy()

    if data.empty:
        raise RuntimeError(
            "No valid VIF values are available for plotting."
        )

    data["Display_Label"] = data["Variable"].map(
        display_label
    )
    data = data.sort_values(
        "VIF",
        ascending=True,
    )

    plt.style.use("default")

    fig, ax = plt.subplots(
        figsize=Config.CHART_SIZE,
        dpi=Config.DISPLAY_DPI,
    )

    colors = distinct_colors(
        len(data),
        base_palette="tab20",
    )

    bars = ax.barh(
        range(len(data)),
        data["VIF"],
        color=colors,
        alpha=0.95,
        height=0.55,
    )

    finite_vif = data["VIF"][
        np.isfinite(
            data["VIF"]
        )
    ]

    max_vif = (
        float(finite_vif.max())
        if len(finite_vif)
        else Config.VIF_THRESHOLD
    )

    dynamic_xmax = max(
        20.0,
        float(
            np.ceil(
                max_vif
                / 10.0
            )
            * 10.0
            + 10.0
        ),
    )

    # Prevent pathological values from producing an unreadable axis.
    dynamic_xmax = min(
        dynamic_xmax,
        1000.0,
    )

    for bar, vif in zip(
        bars,
        data["VIF"],
    ):
        if not np.isfinite(vif):
            continue

        x_text = min(
            vif,
            dynamic_xmax
            * 0.97,
        )

        ax.text(
            x_text
            + dynamic_xmax * 0.01,
            bar.get_y()
            + bar.get_height() / 2,
            f"{vif:.2f}",
            va="center",
            ha="left",
            fontsize=2.5,
            fontweight="bold",
            fontfamily="Times New Roman",
            clip_on=False,
        )

    ax.axvline(
        x=Config.VIF_THRESHOLD,
        color="red",
        linestyle="--",
        linewidth=1.0,
        alpha=0.9,
    )

    ax.text(
        Config.VIF_THRESHOLD,
        len(data) - 0.5,
        f"{int(Config.VIF_THRESHOLD)}",
        ha="center",
        va="bottom",
        fontsize=3.0,
        color="red",
        fontweight="bold",
        fontfamily="Times New Roman",
    )

    ax.set_xlim(
        0,
        dynamic_xmax,
    )

    ax.set_yticks(
        range(len(data))
    )
    ax.set_yticklabels(
        data["Display_Label"],
        fontsize=4,
        fontfamily="Times New Roman",
    )

    apply_tiny_axes_style(
        ax,
        xlabel="VIF",
        ylabel="Conditioning Factors",
    )

    plt.tight_layout(
        pad=0.1
    )

    save_figure_triplet(
        fig,
        out_png,
        out_pdf,
        out_svg,
    )

    plt.show()
    plt.close(fig)
    mpl.rcParams.update(
        mpl.rcParamsDefault
    )

    pp(f"[FIGURE] VIF chart -> {out_png}")


# =============================================================================
# REPORTING
# =============================================================================
def write_scenario_report(
    scenario_name: str,
    scenario_info: Dict[str, Any],
    primary_table: pd.DataFrame,
    dummy_table: pd.DataFrame,
    encoding_metadata: Dict[str, Any],
    n_common_pixels_total: int,
    n_sampled_pixels: int,
    report_path: Path,
) -> None:
    with open(
        report_path,
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(
            "=" * 100
            + "\n"
        )
        handle.write(
            f"PRIMARY VIF AND TOLERANCE ANALYSIS — "
            f"{scenario_info.get('label', scenario_name)}\n"
        )
        handle.write(
            "=" * 100
            + "\n\n"
        )

        handle.write(
            f"Generated: {datetime.now():%Y-%m-%d %H:%M:%S}\n"
        )
        handle.write(
            f"Scenario: {scenario_name}\n"
        )
        handle.write(
            f"Purpose: {scenario_info.get('purpose', '')}\n"
        )
        handle.write(
            f"Common A–F complete-case pixels available: "
            f"{n_common_pixels_total:,}\n"
        )
        handle.write(
            f"Pixels sampled identically for all models: "
            f"{n_sampled_pixels:,}\n"
        )
        handle.write(
            f"Random seed: {Config.RANDOM_SEED}\n\n"
        )

        handle.write(
            "PRIMARY METHODOLOGY\n"
        )
        handle.write(
            "-" * 64
            + "\n"
        )
        handle.write(
            "Continuous predictors are one-degree-of-freedom terms and use "
            "ordinary VIF.\n"
        )
        handle.write(
            "Land Use is dummy-coded internally and summarized as one grouped "
            "factor using GVIF^(1/DF), which equals ordinary VIF for a one-"
            "degree-of-freedom predictor.\n"
        )
        handle.write(
            "Grouped tolerance is the reciprocal of GVIF^(1/DF).\n"
        )
        handle.write(
            f"High multicollinearity threshold: VIF >= "
            f"{Config.VIF_THRESHOLD:.1f} or Tolerance <= "
            f"{Config.TOLERANCE_THRESHOLD:.2f}.\n"
        )
        handle.write(
            "All models use the same common raster sample; no landslide labels "
            "are required.\n\n"
        )

        if encoding_metadata:
            handle.write(
                "CATEGORICAL ENCODING\n"
            )
            handle.write(
                "-" * 64
                + "\n"
            )

            for variable, metadata in encoding_metadata.items():
                handle.write(
                    f"{variable}: categories={metadata['categories']}\n"
                )
                handle.write(
                    f"{variable}: reference category="
                    f"{metadata['reference_category']}\n"
                )
                handle.write(
                    f"{variable}: internal dummy columns="
                    f"{metadata['dummy_columns']}\n"
                )

            handle.write(
                "\n"
            )

        handle.write(
            "PRIMARY TERM-LEVEL RESULTS\n"
        )
        handle.write(
            "-" * 64
            + "\n"
        )

        ordered = primary_table.sort_values(
            "VIF",
            ascending=False,
        )

        for _, row in ordered.iterrows():
            handle.write(
                f"{row['Variable']:<24s} "
                f"DF={int(row['DF']):2d} "
                f"VIF={row['VIF']:10.4f} "
                f"Tolerance={row['Tolerance']:9.4f} "
                f"{row['Status']}\n"
            )

        handle.write(
            "\nINTERNAL DUMMY-LEVEL QA RESULTS\n"
        )
        handle.write(
            "-" * 64
            + "\n"
        )

        for _, row in dummy_table.iterrows():
            handle.write(
                f"{row['Original_Term']:<20s} "
                f"{row['Internal_Column']:<35s} "
                f"VIF={row['Dummy_Level_VIF']:10.4f} "
                f"Tolerance={row['Dummy_Level_Tolerance']:9.4f}\n"
            )


# =============================================================================
# SCENARIO ANALYSIS
# =============================================================================
def analyze_scenario(
    scenario_name: str,
    scenario_info: Dict[str, Any],
    arrays_by_name: Dict[str, np.ndarray],
    common_sample_indices: np.ndarray,
    n_common_pixels_total: int,
) -> Dict[str, Any]:
    features = list(
        scenario_info["features"]
    )

    pp("")
    pp(
        "=" * 92
    )
    pp(
        f"=== {scenario_info.get('label', scenario_name)} "
        f"({len(features)} predictors) ==="
    )
    pp(
        "=" * 92
    )

    scenario_df = extract_scenario_dataframe(
        arrays_by_name=arrays_by_name,
        features=features,
        common_sample_indices=common_sample_indices,
    )

    design_df, term_columns, encoding_metadata = (
        build_vif_design_matrix(
            df=scenario_df,
            features=features,
        )
    )

    pp(
        f"[DESIGN] Original terms={len(features)} | "
        f"internal columns={design_df.shape[1]} | "
        f"rows={len(design_df):,}"
    )

    primary_table = calculate_term_level_gvif(
        design_df=design_df,
        term_columns=term_columns,
    )

    # Preserve scenario feature order in the primary CSV/summary.
    order_map = {
        feature: index
        for index, feature in enumerate(features)
    }
    primary_table["_scenario_order"] = primary_table[
        "Variable"
    ].map(order_map)

    primary_table = primary_table.sort_values(
        "_scenario_order"
    ).drop(
        columns=["_scenario_order"]
    ).reset_index(
        drop=True
    )

    dummy_table = calculate_dummy_level_vif(
        design_df=design_df,
        term_columns=term_columns,
    )

    scenario_prefix = scenario_name.replace(
        "_",
        "-",
    )

    primary_csv = (
        Config.OUT_DIR
        / f"{scenario_prefix}_Primary_VIF_Tolerance.csv"
    )
    dummy_csv = (
        Config.OUT_DIR
        / f"{scenario_prefix}_Internal_Dummy_Level_VIF_QA.csv"
    )

    vif_png = (
        Config.OUT_DIR
        / f"{scenario_prefix}_VIF_Analysis.png"
    )
    vif_pdf = (
        Config.OUT_DIR
        / f"{scenario_prefix}_VIF_Analysis.pdf"
    )
    vif_svg = (
        Config.OUT_DIR
        / f"{scenario_prefix}_VIF_Analysis.svg"
    )

    tolerance_png = (
        Config.OUT_DIR
        / f"{scenario_prefix}_Tolerance_Analysis.png"
    )
    tolerance_pdf = (
        Config.OUT_DIR
        / f"{scenario_prefix}_Tolerance_Analysis.pdf"
    )
    tolerance_svg = (
        Config.OUT_DIR
        / f"{scenario_prefix}_Tolerance_Analysis.svg"
    )

    report_path = (
        Config.OUT_DIR
        / f"{scenario_prefix}_Primary_VIF_Tolerance_Report.txt"
    )

    primary_table.to_csv(
        primary_csv,
        index=False,
        float_format="%.8f",
    )
    dummy_table.to_csv(
        dummy_csv,
        index=False,
        float_format="%.8f",
    )

    create_vif_chart(
        table=primary_table,
        out_png=vif_png,
        out_pdf=vif_pdf,
        out_svg=vif_svg,
    )

    create_tolerance_chart(
        table=primary_table,
        out_png=tolerance_png,
        out_pdf=tolerance_pdf,
        out_svg=tolerance_svg,
    )

    write_scenario_report(
        scenario_name=scenario_name,
        scenario_info=scenario_info,
        primary_table=primary_table,
        dummy_table=dummy_table,
        encoding_metadata=encoding_metadata,
        n_common_pixels_total=n_common_pixels_total,
        n_sampled_pixels=len(common_sample_indices),
        report_path=report_path,
    )

    high = primary_table[
        primary_table["VIF"]
        >= Config.VIF_THRESHOLD
    ]

    pp(
        f"[RESULT] High-multicollinearity terms: {len(high)}"
    )

    for _, row in primary_table.sort_values(
        "VIF",
        ascending=False,
    ).iterrows():
        pp(
            f"  {row['Variable']:<24s} "
            f"VIF={row['VIF']:9.4f} "
            f"Tol={row['Tolerance']:8.4f} "
            f"{row['Status']}"
        )

    return {
        "Scenario": scenario_name,
        "Label": scenario_info.get(
            "label",
            scenario_name,
        ),
        "N_Original_Predictors": len(features),
        "N_Internal_Design_Columns": design_df.shape[1],
        "N_Common_Pixels_Available": int(
            n_common_pixels_total
        ),
        "N_Common_Pixels_Sampled": int(
            len(common_sample_indices)
        ),
        "N_High_Multicollinearity_Terms": int(
            len(high)
        ),
        "Maximum_VIF": float(
            primary_table["VIF"].max()
        ),
        "Minimum_Tolerance": float(
            primary_table["Tolerance"].min()
        ),
        "Primary_CSV": str(
            primary_csv
        ),
        "Dummy_QA_CSV": str(
            dummy_csv
        ),
        "VIF_PNG": str(
            vif_png
        ),
        "Tolerance_PNG": str(
            tolerance_png
        ),
        "Report": str(
            report_path
        ),
    }


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    pp(
        "=" * 96
    )
    pp(
        "=== Step 6.1c: Primary VIF and Tolerance Analysis for Models A–F ==="
    )
    pp(
        "=" * 96
    )

    stack_manifest = load_json(
        Config.STACK_MANIFEST_JSON
    )
    scenario_manifest = load_json(
        Config.SCENARIO_MANIFEST_JSON
    )

    master_band_order = stack_manifest[
        "stacks"
    ]["master_band_order"]

    master_stack, master_names = load_master_stack(
        Config.MASTER_STACK_TIF,
        master_band_order,
    )

    arrays_by_name = build_array_lookup(
        master_stack,
        master_names,
    )

    common_mask = load_common_mask(
        Config.COMMON_MASK_TIF,
        master_stack.shape[1:],
    )

    n_common_pixels_total = int(
        common_mask.sum()
    )

    common_sample_indices = choose_common_sample_indices(
        common_mask=common_mask,
        max_samples=Config.MAX_SAMPLES_FOR_VIF,
        seed=Config.RANDOM_SEED,
    )

    scenarios = scenario_manifest[
        "scenarios"
    ]

    summary_rows = []

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in scenarios:
            raise KeyError(
                f"Scenario '{scenario_name}' is missing from "
                "model_ablation_scenarios.json"
            )

        summary = analyze_scenario(
            scenario_name=scenario_name,
            scenario_info=scenarios[scenario_name],
            arrays_by_name=arrays_by_name,
            common_sample_indices=common_sample_indices,
            n_common_pixels_total=n_common_pixels_total,
        )

        summary_rows.append(
            summary
        )

    # -------------------------------------------------------------------------
    # Combined summary
    # -------------------------------------------------------------------------
    summary_df = pd.DataFrame(
        summary_rows
    )

    summary_csv = (
        Config.OUT_DIR
        / "All_Models_A_to_F_Primary_VIF_Tolerance_Summary.csv"
    )
    summary_json = (
        Config.OUT_DIR
        / "All_Models_A_to_F_Primary_VIF_Tolerance_Summary.json"
    )

    summary_df.to_csv(
        summary_csv,
        index=False,
        float_format="%.8f",
    )

    with open(
        summary_json,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary_rows,
            handle,
            indent=2,
        )

    # -------------------------------------------------------------------------
    # Methodology note
    # -------------------------------------------------------------------------
    methodology_path = (
        Config.OUT_DIR
        / "Primary_VIF_Tolerance_Methodology.txt"
    )

    with open(
        methodology_path,
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(
            "PRIMARY VIF AND TOLERANCE METHODOLOGY FOR MODELS A–F\n"
        )
        handle.write(
            "=" * 76
            + "\n\n"
        )
        handle.write(
            "All six model configurations are evaluated on the same fixed "
            "sample of pixels drawn from the Step 6.1 common A–F complete-case "
            "mask.\n\n"
        )
        handle.write(
            "Continuous predictors are treated as one-degree-of-freedom terms.\n"
        )
        handle.write(
            "Land Use is internally dummy-coded with one reference category "
            "and is summarized as one grouped model term using generalized VIF "
            "(GVIF).\n"
        )
        handle.write(
            "The primary grouped VIF-equivalent is GVIF^(1/DF), which reduces "
            "exactly to ordinary VIF for one-degree-of-freedom predictors.\n"
        )
        handle.write(
            "Tolerance is defined as the reciprocal of this VIF-equivalent.\n"
        )
        handle.write(
            f"Primary thresholds: VIF >= {Config.VIF_THRESHOLD:.1f}; "
            f"Tolerance <= {Config.TOLERANCE_THRESHOLD:.2f}.\n"
        )
        handle.write(
            "Detailed dummy-level VIF values are written only as internal QA; "
            "the primary figures contain one Land Use bar.\n"
        )
        handle.write(
            "Standardization of continuous variables is used only for numerical "
            "conditioning and does not change VIF/GVIF under linear rescaling.\n"
        )

    pp("")
    pp(
        "=" * 96
    )
    pp(
        "=== STEP 6.1c COMPLETE ==="
    )
    pp(
        "=" * 96
    )
    pp(
        f"[OUT] {Config.OUT_DIR}"
    )
    pp(
        f"[OUT] {summary_csv}"
    )
    pp(
        f"[OUT] {summary_json}"
    )
    pp(
        f"[OUT] {methodology_path}"
    )


if __name__ == "__main__":
    main()


# Step 6.2 — Model-Scenario QA and Common Sampling-Domain Preparation

In [ ]:
# =============================================================================
# Step 6.2 — Model-Scenario QA and Common Sampling-Domain Preparation
# =============================================================================
# Validates the revised Step 6.1 Models A–F architecture and writes one common
# sampling-domain contract for the corrected site-level Step 6.3 workflow.
# =============================================================================

from __future__ import annotations

import json
import os
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling

BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
RESULTS_DIR = BASE / "Results"
STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"

STACK_MANIFEST_JSON = STEP61_DIR / "stack_manifest.json"
SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"
MASTER_STACK_TIF = STEP61_DIR / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
ANALYSIS_DOMAIN_MASK_TIF = STEP61_DIR / "analysis_domain_mask_32645_12p5m_byte.tif"
STEP61_COMMON_MASK_TIF = STEP61_DIR / "common_A_to_F_valid_mask_32645_12p5m_byte.tif"

OUT_DIR = RESULTS_DIR / "Step-6.2-Model_Scenario_QA"
OUT_DIR.mkdir(parents=True, exist_ok=True)
COMMON_SAMPLING_DOMAIN_TIF = OUT_DIR / "common_sampling_domain_A_to_F_32645_12p5m_byte.tif"
SCENARIO_COVERAGE_CSV = OUT_DIR / "model_scenario_coverage_QA.csv"
FEATURE_COVERAGE_CSV = OUT_DIR / "master_candidate_feature_coverage_QA.csv"
MEMBERSHIP_CSV = OUT_DIR / "model_scenario_predictor_membership.csv"
SAMPLING_CONTRACT_JSON = OUT_DIR / "Step_6_2_sampling_contract.json"
QA_REPORT_TXT = OUT_DIR / "Step_6_2_Model_Scenario_QA_Report.txt"

NODATA_BYTE = 255

STATIC12 = [
    "slope_deg", "northness", "eastness", "profile_curv", "plan_curv",
    "tpi_s5", "spi_ln", "dist_river_m", "dist_road_m", "aap_mm",
    "ndvi", "landcover",
]
MASTER17 = STATIC12 + [
    "dvplus_mm_yr", "hkfi_0_1", "sndr_0_1", "twi_raw", "sm_support_physical"
]
SCENARIOS: Dict[str, List[str]] = {
    "A_static_baseline": STATIC12,
    "B_raw_deformation": STATIC12 + ["dvplus_mm_yr"],
    "C_hkfi_only": STATIC12 + ["hkfi_0_1"],
    "D_sndr_only": STATIC12 + ["sndr_0_1"],
    "E_physics_guided": STATIC12 + ["hkfi_0_1", "sndr_0_1"],
    "F_untransformed_sources": STATIC12 + ["dvplus_mm_yr", "twi_raw", "sm_support_physical"],
}
SCENARIO_ORDER = list(SCENARIOS)


def log(msg: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")


def load_json(path: Path) -> Dict[str, Any]:
    require(path, "JSON input")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def exact_grid(ds: rasterio.io.DatasetReader, profile: Dict[str, Any]) -> bool:
    return (
        ds.crs == profile["crs"]
        and ds.transform == profile["transform"]
        and ds.width == profile["width"]
        and ds.height == profile["height"]
    )


def read_mask(path: Path, profile: Dict[str, Any], label: str) -> np.ndarray:
    require(path, label)
    with rasterio.open(path) as ds:
        if not exact_grid(ds, profile):
            raise ValueError(f"{label} is not exactly aligned to master stack: {path}")
        m = ds.read(1, masked=True)
        arr = np.where(m.mask, 0, m.data)
    return arr > 0


def write_mask(path: Path, mask: np.ndarray, profile: Dict[str, Any], description: str) -> None:
    p = profile.copy()
    p.update(driver="GTiff", dtype="uint8", count=1, nodata=NODATA_BYTE,
             compress="LZW", tiled=True, BIGTIFF="IF_SAFER")
    tmp = Path(str(path) + ".part")
    with rasterio.open(tmp, "w", **p) as ds:
        ds.write(mask.astype("uint8"), 1)
        ds.set_band_description(1, description)
        ds.update_tags(description=description, generated_at_utc=utc_now())
        try:
            ds.build_overviews([2, 4, 8, 16], Resampling.nearest)
            ds.update_tags(ns="rio_overview", resampling="nearest")
        except Exception:
            pass
    if path.exists():
        path.unlink()
    os.replace(tmp, path)


def validate_design(stack_manifest: Dict[str, Any], scenario_manifest: Dict[str, Any], names: List[str]) -> None:
    if names != MASTER17:
        raise RuntimeError(f"Master predictor order mismatch.\nExpected: {MASTER17}\nFound: {names}")
    if "tpi_s25" in names or "tpi_s5" not in names:
        raise RuntimeError("Exactly one TPI is required: tpi_s5.")
    if any("irs" in n.lower() for n in names):
        raise RuntimeError("IRS must not be present in the ML predictor stack.")

    manifest_scenarios = scenario_manifest.get("scenarios", {})
    for scenario, expected in SCENARIOS.items():
        if scenario not in manifest_scenarios:
            raise RuntimeError(f"Missing scenario: {scenario}")
        found = list(manifest_scenarios[scenario].get("features", []))
        if found != expected:
            raise RuntimeError(
                f"Scenario mismatch for {scenario}.\nExpected: {expected}\nFound: {found}"
            )

    if len(SCENARIOS["E_physics_guided"]) != 14:
        raise RuntimeError("Model E must contain exactly 14 predictors.")
    log("[QA] Step 6.1 predictor/scenario architecture: PASS")


def main() -> None:
    log("=== Step 6.2: Model-Scenario QA and Common Sampling Domain ===")

    stack_manifest = load_json(STACK_MANIFEST_JSON)
    scenario_manifest = load_json(SCENARIO_MANIFEST_JSON)
    require(MASTER_STACK_TIF, "Step 6.1 master candidate stack")

    with rasterio.open(MASTER_STACK_TIF) as ds:
        profile = ds.profile.copy()
        nodata = ds.nodata
        descriptions = list(ds.descriptions)
        m = ds.read(masked=True)

    stack = np.where(m.mask, np.nan, m.data).astype("float64")
    if nodata is not None and np.isfinite(float(nodata)):
        stack[stack == float(nodata)] = np.nan

    if descriptions and all(d for d in descriptions):
        names = [str(d) for d in descriptions]
    else:
        names = list(stack_manifest["stacks"]["master_band_order"])
    if len(names) != stack.shape[0]:
        raise RuntimeError("Band-name count does not match master-stack band count.")

    validate_design(stack_manifest, scenario_manifest, names)

    analysis_domain = read_mask(ANALYSIS_DOMAIN_MASK_TIF, profile, "analysis-domain mask")
    step61_common = read_mask(STEP61_COMMON_MASK_TIF, profile, "Step 6.1 common A–F mask")

    recomputed_common = analysis_domain & np.all(np.isfinite(stack), axis=0)
    mismatch = np.logical_xor(recomputed_common, step61_common)
    mismatch_n = int(mismatch.sum())
    if mismatch_n:
        raise RuntimeError(
            f"Step 6.1 common mask differs from independently recomputed mask at {mismatch_n:,} pixels."
        )
    if not recomputed_common.any():
        raise RuntimeError("Validated common sampling domain contains no pixels.")

    write_mask(
        COMMON_SAMPLING_DOMAIN_TIF,
        recomputed_common,
        profile,
        "validated_common_sampling_domain_for_models_A_to_F",
    )

    log(
        f"[MASK] common sampling domain = {int(recomputed_common.sum()):,}/"
        f"{int(analysis_domain.sum()):,} analysis-domain pixels "
        f"({100.0 * recomputed_common.sum() / max(int(analysis_domain.sum()), 1):.3f}%)"
    )

    # Feature coverage QA
    feature_rows = []
    lookup = {name: stack[i] for i, name in enumerate(names)}
    for i, name in enumerate(names, start=1):
        valid = analysis_domain & np.isfinite(lookup[name])
        feature_rows.append({
            "band_index_1_based": i,
            "feature": name,
            "valid_pixels_in_analysis_domain": int(valid.sum()),
            "coverage_percent_of_analysis_domain": float(
                100.0 * valid.sum() / max(int(analysis_domain.sum()), 1)
            ),
        })
    pd.DataFrame(feature_rows).to_csv(FEATURE_COVERAGE_CSV, index=False, float_format="%.8f")

    # Scenario coverage QA
    scenario_rows = []
    manifest_scenarios = scenario_manifest["scenarios"]
    for scenario in SCENARIO_ORDER:
        features = SCENARIOS[scenario]
        valid = analysis_domain.copy()
        for feature in features:
            valid &= np.isfinite(lookup[feature])
        scenario_rows.append({
            "scenario": scenario,
            "label": manifest_scenarios[scenario].get("label", scenario),
            "n_predictors": len(features),
            "predictors": "|".join(features),
            "valid_pixels_scenario_specific": int(valid.sum()),
            "coverage_percent_of_analysis_domain": float(
                100.0 * valid.sum() / max(int(analysis_domain.sum()), 1)
            ),
            "common_sampling_domain_pixels": int(recomputed_common.sum()),
            "common_sampling_domain_percent_of_analysis_domain": float(
                100.0 * recomputed_common.sum() / max(int(analysis_domain.sum()), 1)
            ),
            "scenario_specific_minus_common_pixels": int(valid.sum() - recomputed_common.sum()),
        })
    pd.DataFrame(scenario_rows).to_csv(SCENARIO_COVERAGE_CSV, index=False, float_format="%.8f")

    # Membership matrix
    membership_rows = []
    for feature in names:
        row = {"feature": feature}
        for scenario in SCENARIO_ORDER:
            row[scenario] = int(feature in SCENARIOS[scenario])
        membership_rows.append(row)
    pd.DataFrame(membership_rows).to_csv(MEMBERSHIP_CSV, index=False)

    # Sampling contract consumed by corrected Step 6.3
    contract = {
        "generated_at_utc": utc_now(),
        "step": "Step-6.2-Model_Scenario_QA",
        "status": "PASS",
        "purpose": (
            "Validate Models A–F and define one common predictor domain before one-time site-level sampling."
        ),
        "authoritative_inputs": {
            "master_stack": str(MASTER_STACK_TIF),
            "stack_manifest": str(STACK_MANIFEST_JSON),
            "scenario_manifest": str(SCENARIO_MANIFEST_JSON),
            "analysis_domain_mask": str(ANALYSIS_DOMAIN_MASK_TIF),
            "step61_common_mask": str(STEP61_COMMON_MASK_TIF),
        },
        "validated_outputs": {
            "common_sampling_domain_mask": str(COMMON_SAMPLING_DOMAIN_TIF),
            "scenario_coverage_csv": str(SCENARIO_COVERAGE_CSV),
            "feature_coverage_csv": str(FEATURE_COVERAGE_CSV),
            "membership_csv": str(MEMBERSHIP_CSV),
        },
        "predictor_design": {
            "static12": STATIC12,
            "master_candidates": MASTER17,
            "n_static": 12,
            "n_master_candidates": 17,
            "retained_tpi": "tpi_s5",
            "tpi_s25_included": False,
            "irs_as_predictor": False,
            "dynamic_mask_used_for_modelling_domain": False,
            "new_weighted_fusion_index_created": False,
        },
        "scenarios": {
            scenario: {"features": SCENARIOS[scenario], "n_predictors": len(SCENARIOS[scenario])}
            for scenario in SCENARIO_ORDER
        },
        "common_sampling_rule_for_step_6_3": {
            "sample_once": True,
            "use_same_site_records_for_all_A_to_F_models": True,
            "sampling_domain": str(COMMON_SAMPLING_DOMAIN_TIF),
            "do_not_create_separate_context_and_integrated_sample_tables": True,
            "do_not_restrict_to_dynamic_mask": True,
            "do_not_restrict_to_old_core_valid_mask": True,
            "do_not_include_irs_as_predictor": True,
            "recommended_presence_absence_ratio": "1:1",
            "recommended_primary_landslide_exclusion_buffer_m": 100.0,
            "later_buffer_sensitivity_m": [500.0, 1000.0],
            "spatial_cv_blocks": 5,
            "positive_unit": "one site-level landslide record per mapped landslide site",
            "negative_unit": "one matched non-landslide site per positive site",
        },
        "coverage": {
            "analysis_domain_pixels": int(analysis_domain.sum()),
            "common_sampling_domain_pixels": int(recomputed_common.sum()),
            "common_sampling_domain_percent": float(
                100.0 * recomputed_common.sum() / max(int(analysis_domain.sum()), 1)
            ),
            "step61_common_mask_mismatch_pixels": mismatch_n,
        },
    }
    with open(SAMPLING_CONTRACT_JSON, "w", encoding="utf-8") as f:
        json.dump(contract, f, indent=2, ensure_ascii=False)

    # Human-readable report
    with open(QA_REPORT_TXT, "w", encoding="utf-8") as f:
        f.write("=" * 96 + "\n")
        f.write("STEP 6.2 — MODEL-SCENARIO QA AND COMMON SAMPLING-DOMAIN REPORT\n")
        f.write("=" * 96 + "\n\n")
        f.write(f"Generated UTC: {utc_now()}\nStatus: PASS\n\n")
        f.write("FORMER FUSION STEP\n" + "-" * 72 + "\n")
        f.write(
            "No additional manually weighted fusion index is generated. The revised workflow compares "
            "Models A–F directly. A second fusion of DV+, HKFI, SNDR and selected static predictors would "
            "double-count information, reintroduce arbitrary weights, and obstruct the controlled ablations.\n\n"
        )
        f.write("AUTHORITATIVE MODEL CONFIGURATIONS\n" + "-" * 72 + "\n")
        for scenario in SCENARIO_ORDER:
            f.write(f"{scenario}: {len(SCENARIOS[scenario])} predictors\n")
            f.write("  " + ", ".join(SCENARIOS[scenario]) + "\n")
        f.write("\nCOMMON SAMPLING DOMAIN\n" + "-" * 72 + "\n")
        f.write(f"Analysis-domain pixels: {int(analysis_domain.sum()):,}\n")
        f.write(f"Validated common A–F sampling pixels: {int(recomputed_common.sum()):,}\n")
        f.write(
            f"Coverage: {100.0 * recomputed_common.sum() / max(int(analysis_domain.sum()), 1):.3f}%\n"
        )
        f.write(f"Mismatch against Step 6.1 common mask: {mismatch_n:,} pixels\n\n")
        f.write("STEP 6.3 CONTRACT\n" + "-" * 72 + "\n")
        f.write("1. Use one common site-level sample table for Models A–F.\n")
        f.write("2. Use 199 mapped landslide sites as independent positive units, after inventory reconciliation.\n")
        f.write("3. Use 1:1 matched non-landslide sampling for the primary analysis.\n")
        f.write("4. Use a 100 m primary exclusion buffer; test 500 m and 1 km later for sensitivity.\n")
        f.write("5. Use five contiguous longitudinal spatial blocks downstream.\n")
        f.write("6. Do not use the dynamic mask or former core_valid mask to create a separate integrated dataset.\n")
        f.write("7. Do not include IRS as an ML predictor.\n")

    log(f"[OUT] {COMMON_SAMPLING_DOMAIN_TIF}")
    log(f"[OUT] {SCENARIO_COVERAGE_CSV}")
    log(f"[OUT] {FEATURE_COVERAGE_CSV}")
    log(f"[OUT] {MEMBERSHIP_CSV}")
    log(f"[OUT] {SAMPLING_CONTRACT_JSON}")
    log(f"[OUT] {QA_REPORT_TXT}")
    log("=== Step 6.2 completed successfully ===")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 6.2 failed: {exc}")
        traceback.print_exc()
        raise


# Step 6.3 — Common Site-Level Sampling for Models A–F

In [ ]:
# =============================================================================
# Step 6.3 — Common Site-Level Sampling for Models A–F
# Robust road-backbone version with 100/500/1000 m buffer sensitivity
# =============================================================================
#
# This version fixes the earlier failure:
#
#   "NTD_Road.shp does not form one dominant continuous route.
#    Longest component contains only 78.04% of total line length."
#
# The revised road logic does NOT require NTD_Road.shp to be a single perfectly
# connected LineString. It:
#
#   1. merges all available road linework;
#   2. identifies disconnected/branched merged components;
#   3. builds a minimum-gap component tree;
#   4. extracts the longest corridor-like path through that tree;
#   5. orients the selected components consistently;
#   6. represents topology gaps only as virtual cumulative chainage offsets;
#   7. projects samples only to real road components (never to virtual gaps);
#   8. writes full road-backbone QA, component offsets, and gap diagnostics.
#
# The reconstructed backbone is used ONLY for longitudinal chainage and
# five-block assignment. It is NOT used as a conditioning factor and does NOT
# replace the original road-distance raster.
#
# Main scientific design
# ----------------------
# - One unique source landslide feature = one positive site-level record.
# - Training and validation inventory shapefiles are merged as one inventory;
#   their former split is retained only as source metadata.
# - If a SymbolID field exists, only SymbolID == 1 is retained as landslide.
# - One representative common-domain raster cell is selected per landslide.
# - Five contiguous highway-chainage blocks are defined from retained positives.
# - 1:1 block-matched pseudo-absence sampling is performed for 100, 500, 1000 m.
# - 100 m is the primary analysis.
# - All 17 Step 6.1 candidate predictors are extracted once.
# - Models A–F use identical rows within each buffer experiment.
# - No dynamic-mask restriction, no former core-valid mask, no IRS predictor,
#   no global standardization, and no pixel replication of landslide polygons.
# =============================================================================

from __future__ import annotations

import json
import math
import os
import traceback
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import fiona
import numpy as np
import pandas as pd
import rasterio
from pyproj import CRS, Transformer
from rasterio import features
from rasterio.enums import Resampling
from rasterio.windows import Window, from_bounds
from rasterio.windows import transform as window_transform
from shapely.geometry import (
    GeometryCollection,
    LineString,
    MultiLineString,
    MultiPolygon,
    Point,
    Polygon,
    mapping,
    shape,
)
from shapely.ops import linemerge, transform as shp_transform, unary_union


# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # User-supplied vectors
    ROAD_SHP = BASE / "NTD_Road.shp"

    # Optional manually cleaned/verified chainage backbone.
    # Recommended when the automatic reconstruction requires a long synthetic
    # connector. The file may be a Shapefile or GeoPackage containing one
    # continuous corridor line after merging.
    MANUAL_ROAD_BACKBONE = (
        BASE / "NTD_Road_chainage_backbone_MANUAL.gpkg"
    )

    # Allowed modes:
    #   "manual_required"     -> always use MANUAL_ROAD_BACKBONE and fail if absent
    #   "manual_if_available" -> use manual backbone when present; otherwise
    #                            attempt automatic reconstruction with QA gate
    #   "automatic"           -> always reconstruct from ROAD_SHP
    ROAD_BACKBONE_MODE = "manual_if_available"

    # Long topology gaps are retained only as VIRTUAL chainage offsets.
    # They are never used for point-to-road projection. This prevents a long
    # straight synthetic connector from attracting samples and distorting
    # chainage or spatial-block assignment.
    ROAD_REQUIRE_REVIEW_ABOVE_GAP_M = 5000.0

    # Automatic component ordering is accepted only when the selected corridor
    # path retains at least this fraction of the original road linework.
    ROAD_MIN_RETAINED_FRACTION_FOR_AUTOMATIC_CHAINAGE = 0.95

    # Legacy metadata flag. It is no longer needed to bypass long-gap failure
    # because virtual gaps are not used as spatial projection geometry.
    ROAD_BACKBONE_VISUALLY_APPROVED = False

    # Standardize chainage direction for reproducibility: Block 1 starts at
    # the more southerly endpoint and increases generally northward.
    ROAD_ORIENT_SOUTH_TO_NORTH = True

    STUDY_AREA_SHP = BASE / "Study_Area.shp"

    TRAIN_LANDSLIDE_SHP = (
        BASE
        / "Landslide Historical Map"
        / "Training Data"
        / "Landslide_only_train.shp"
    )
    VALID_LANDSLIDE_SHP = (
        BASE
        / "Landslide Historical Map"
        / "Validation Data"
        / "Landslide_only_valid.shp"
    )

    # Historical expectations are QA references only, not hard constraints.
    EXPECTED_TRAIN_SOURCE_FEATURES = 186
    EXPECTED_VALID_SOURCE_FEATURES = 46
    EXPECTED_TOTAL_LANDSLIDE_SITES = 232

    FILTER_SYMBOLID_IF_PRESENT = True
    LANDSLIDE_SYMBOL_VALUE = 1

    # Step 6.1
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    MASTER_STACK_TIF = (
        STEP61_DIR
        / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
    )
    STACK_MANIFEST_JSON = STEP61_DIR / "stack_manifest.json"
    SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"
    ANALYSIS_DOMAIN_MASK_TIF = (
        STEP61_DIR
        / "analysis_domain_mask_32645_12p5m_byte.tif"
    )

    # Step 6.2
    STEP62_DIR = RESULTS_DIR / "Step-6.2-Model_Scenario_QA"
    COMMON_SAMPLING_DOMAIN_TIF = (
        STEP62_DIR
        / "common_sampling_domain_A_to_F_32645_12p5m_byte.tif"
    )
    SAMPLING_CONTRACT_JSON = STEP62_DIR / "Step_6_2_sampling_contract.json"

    # Study mask
    STUDY_MASK_TIF = RESULTS_DIR / "Step-0.2-Study_Mask" / "study_mask.tif"

    # Outputs
    OUT_DIR = RESULTS_DIR / "Step-6.3-Site_Level_Sampling"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    INVENTORY_AUDIT_CSV = OUT_DIR / "landslide_inventory_source_count_audit.csv"
    RECONCILIATION_CSV = OUT_DIR / "landslide_site_reconciliation.csv"
    RETAINED_POSITIVE_CSV = OUT_DIR / "retained_landslide_sites.csv"

    ROAD_QA_JSON = OUT_DIR / "NTD_Road_backbone_QA.json"
    ROAD_BACKBONE_GPKG = OUT_DIR / "NTD_Road_chainage_backbone.gpkg"
    ROAD_COMPONENT_CHAINAGE_CSV = OUT_DIR / "NTD_Road_component_chainage_offsets.csv"
    ROAD_DIAGNOSTIC_GPKG = OUT_DIR / "NTD_Road_backbone_diagnostic.gpkg"
    ROAD_REVIEW_TXT = OUT_DIR / "NTD_Road_backbone_review.txt"

    BLOCK_BOUNDARIES_CSV = OUT_DIR / "spatial_block_boundaries.csv"
    BLOCK_ASSIGNMENTS_CSV = OUT_DIR / "spatial_block_assignments.csv"

    PRIMARY_MASTER_CSV = OUT_DIR / "common_site_level_samples_A_to_F.csv"
    METADATA_JSON = OUT_DIR / "Step_6_3_site_level_sampling_metadata.json"
    QA_REPORT_TXT = OUT_DIR / "Step_6_3_site_level_sampling_QA_report.txt"

    # Sampling
    RANDOM_SEED = 42
    N_SPATIAL_BLOCKS = 5
    PRESENCE_ABSENCE_RATIO = 1.0

    PRIMARY_BUFFER_M = 100
    BUFFER_SENSITIVITY_M = [100, 500, 1000]

    ALL_TOUCHED_FALLBACK = True

    WRITE_MODEL_SCENARIO_VIEWS = True
    WRITE_POINT_GPKGS = True
    WRITE_NEGATIVE_CANDIDATE_MASKS = True
    WRITE_ROAD_BACKBONE_GPKG = True

    MASK_NODATA = 255

    # QA only: do not fail on disconnected road linework.
    ROAD_WARN_RETAINED_ORIGINAL_FRACTION = 0.70
    ROAD_WARN_MAX_SYNTHETIC_GAP_M = 5000.0
    ROAD_WARN_POSITIVE_DISTANCE_TO_BACKBONE_M = 5000.0

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]


# =============================================================================
# BASIC HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json(path: Path) -> Dict[str, Any]:
    require_file(path, "JSON input")
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def to_json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_safe(v) for v in value]
    return value


def to_crs(crs_like: Any) -> CRS:
    if isinstance(crs_like, CRS):
        return crs_like
    if hasattr(crs_like, "to_wkt"):
        return CRS.from_wkt(crs_like.to_wkt())
    return CRS.from_user_input(crs_like)


def make_reprojector(source_crs: CRS, target_crs: CRS):
    if source_crs == target_crs:
        return lambda geometry: geometry
    transformer = Transformer.from_crs(
        source_crs,
        target_crs,
        always_xy=True,
    )
    return lambda geometry: shp_transform(transformer.transform, geometry)


def source_crs_from_fiona(source) -> CRS:
    if getattr(source, "crs_wkt", None):
        return CRS.from_user_input(source.crs_wkt)
    if getattr(source, "crs", None):
        return CRS.from_user_input(source.crs)
    raise RuntimeError("Vector CRS could not be determined.")


# =============================================================================
# EXACT RASTER INPUTS
# =============================================================================
def read_exact_mask(
    path: Path,
    reference_profile: Dict[str, Any],
    description: str,
) -> np.ndarray:
    require_file(path, description)

    with rasterio.open(path) as dataset:
        if (
            dataset.crs != reference_profile["crs"]
            or dataset.transform != reference_profile["transform"]
            or dataset.width != reference_profile["width"]
            or dataset.height != reference_profile["height"]
        ):
            raise ValueError(
                f"{description} does not exactly match the master-stack grid: {path}"
            )

        masked = dataset.read(1, masked=True)
        values = np.where(masked.mask, 0, masked.data)

    return values > 0


def load_master_stack_and_masks():
    require_file(Config.MASTER_STACK_TIF, "Step 6.1 master predictor stack")
    stack_manifest = load_json(Config.STACK_MANIFEST_JSON)
    scenario_manifest = load_json(Config.SCENARIO_MANIFEST_JSON)

    with rasterio.open(Config.MASTER_STACK_TIF) as dataset:
        profile = dataset.profile.copy()
        masked = dataset.read(masked=True)
        nodata = dataset.nodata
        descriptions = list(dataset.descriptions)

    stack = np.where(masked.mask, np.nan, masked.data).astype("float64")

    if nodata is not None and np.isfinite(float(nodata)):
        stack[stack == float(nodata)] = np.nan

    if descriptions and all(descriptions):
        band_names = [str(name) for name in descriptions]
    else:
        band_names = list(stack_manifest["stacks"]["master_band_order"])

    if len(band_names) != stack.shape[0]:
        raise RuntimeError(
            "Master-stack band count does not match the available band-name count."
        )

    common_domain = read_exact_mask(
        Config.COMMON_SAMPLING_DOMAIN_TIF,
        profile,
        "Step 6.2 common A–F sampling-domain mask",
    )
    study_mask = read_exact_mask(
        Config.STUDY_MASK_TIF,
        profile,
        "study mask",
    )
    analysis_domain = read_exact_mask(
        Config.ANALYSIS_DOMAIN_MASK_TIF,
        profile,
        "Step 6.1 analysis-domain mask",
    )

    finite_all = np.all(np.isfinite(stack), axis=0)
    invalid_inside_common = common_domain & ~finite_all

    if invalid_inside_common.any():
        raise RuntimeError(
            "The common A–F sampling domain contains "
            f"{int(invalid_inside_common.sum()):,} pixels with missing "
            "master-stack predictors."
        )

    # Step 6.2 contract is useful but not allowed to cause another brittle stop.
    sampling_contract = {}
    if Config.SAMPLING_CONTRACT_JSON.exists():
        sampling_contract = load_json(Config.SAMPLING_CONTRACT_JSON)
    else:
        log(
            "[WARN] Step 6.2 sampling contract JSON was not found. "
            "The validated common-domain raster will still be used."
        )

    return (
        stack,
        band_names,
        profile,
        common_domain,
        study_mask,
        analysis_domain,
        stack_manifest,
        scenario_manifest,
        sampling_contract,
    )


# =============================================================================
# LANDSLIDE INVENTORY
# =============================================================================
def repair_polygonal_geometry(geometry):
    repaired = False

    if geometry is None or geometry.is_empty:
        return None, repaired

    if not geometry.is_valid:
        geometry = geometry.buffer(0)
        repaired = True

    if geometry.is_empty:
        return None, repaired

    if isinstance(geometry, (Polygon, MultiPolygon)):
        return geometry, repaired

    if isinstance(geometry, GeometryCollection):
        polygonal = [
            part
            for part in geometry.geoms
            if isinstance(part, (Polygon, MultiPolygon))
        ]
        if polygonal:
            geometry = unary_union(polygonal)
            repaired = True
            if isinstance(geometry, (Polygon, MultiPolygon)):
                return geometry, repaired

    return None, repaired


def find_symbolid_field(property_names: Sequence[str]) -> Optional[str]:
    lookup = {str(name).lower(): str(name) for name in property_names}
    for candidate in ["symbolid", "symbol_id", "symbol"]:
        if candidate in lookup:
            return lookup[candidate]
    return None


def read_landslide_inventory(
    shp_path: Path,
    source_tag: str,
    site_prefix: str,
    target_crs: CRS,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    require_file(shp_path, f"{source_tag} landslide inventory")

    retained_records: List[Dict[str, Any]] = []

    with fiona.open(shp_path) as source:
        source_crs = source_crs_from_fiona(source)
        reproj = make_reprojector(source_crs, target_crs)

        property_names = list(source.schema.get("properties", {}).keys())
        symbol_field = find_symbolid_field(property_names)

        source_feature_count = 0
        filtered_nonlandslide_count = 0
        invalid_geometry_count = 0
        symbol_counts: Dict[str, int] = defaultdict(int)

        for feature_index, feature in enumerate(source):
            source_feature_count += 1
            properties = dict(feature.get("properties", {}) or {})

            symbol_value = None
            if symbol_field is not None:
                symbol_value = properties.get(symbol_field)
                symbol_counts[str(symbol_value)] += 1

            if Config.FILTER_SYMBOLID_IF_PRESENT and symbol_field is not None:
                try:
                    symbol_int = int(symbol_value)
                except Exception:
                    symbol_int = None

                if symbol_int != int(Config.LANDSLIDE_SYMBOL_VALUE):
                    filtered_nonlandslide_count += 1
                    continue

            geometry_projected = reproj(shape(feature["geometry"]))
            geometry, repaired = repair_polygonal_geometry(geometry_projected)

            if geometry is None:
                invalid_geometry_count += 1

            original_id = None
            for candidate in [
                "id",
                "ID",
                "OBJECTID",
                "ObjectID",
                "FID",
                "fid",
            ]:
                if candidate in properties:
                    original_id = properties[candidate]
                    break

            retained_records.append(
                {
                    "site_id": f"{site_prefix}_{feature_index + 1:05d}",
                    "source_tag": source_tag,
                    "source_file": str(shp_path),
                    "source_feature_index": int(feature_index),
                    "source_original_id": (
                        None if original_id is None else str(original_id)
                    ),
                    "geometry": geometry,
                    "geometry_repaired": bool(repaired),
                    "symbol_field": symbol_field,
                    "symbol_value": symbol_value,
                }
            )

    audit = {
        "source_tag": source_tag,
        "source_file": str(shp_path),
        "source_features_total": int(source_feature_count),
        "symbol_field": symbol_field,
        "symbol_counts": dict(symbol_counts),
        "filtered_nonlandslide_features": int(filtered_nonlandslide_count),
        "retained_landslide_records": int(len(retained_records)),
        "invalid_polygonal_geometries": int(invalid_geometry_count),
    }

    return retained_records, audit


# =============================================================================
# ROBUST NTD ROAD BACKBONE
# =============================================================================
def extract_line_parts(geometry) -> List[LineString]:
    if geometry is None or geometry.is_empty:
        return []
    if isinstance(geometry, LineString):
        return [geometry]
    if isinstance(geometry, MultiLineString):
        return list(geometry.geoms)
    if isinstance(geometry, GeometryCollection):
        parts: List[LineString] = []
        for child in geometry.geoms:
            parts.extend(extract_line_parts(child))
        return parts
    return []


def component_endpoints(line: LineString) -> Tuple[np.ndarray, np.ndarray]:
    coords = np.asarray(line.coords, dtype="float64")
    return coords[0, :2], coords[-1, :2]


def endpoint_gap(line_a: LineString, line_b: LineString) -> float:
    a0, a1 = component_endpoints(line_a)
    b0, b1 = component_endpoints(line_b)

    distances = [
        np.linalg.norm(a0 - b0),
        np.linalg.norm(a0 - b1),
        np.linalg.norm(a1 - b0),
        np.linalg.norm(a1 - b1),
    ]
    return float(min(distances))


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> bool:
        ra = self.find(a)
        rb = self.find(b)

        if ra == rb:
            return False

        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra

        self.parent[rb] = ra

        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

        return True


def build_component_mst(components: Sequence[LineString]):
    n = len(components)

    if n == 1:
        return {0: []}, []

    edges = []

    for i in range(n):
        for j in range(i + 1, n):
            gap = endpoint_gap(components[i], components[j])
            edges.append((gap, i, j))

    edges.sort(key=lambda item: item[0])

    uf = UnionFind(n)
    adjacency = {i: [] for i in range(n)}
    mst_edges = []

    for gap, i, j in edges:
        if uf.union(i, j):
            adjacency[i].append((j, float(gap)))
            adjacency[j].append((i, float(gap)))
            mst_edges.append((i, j, float(gap)))

            if len(mst_edges) == n - 1:
                break

    if len(mst_edges) != n - 1:
        raise RuntimeError("Could not build a connected component tree for NTD_Road.shp.")

    return adjacency, mst_edges


def farthest_tree_node(
    start: int,
    adjacency: Dict[int, List[Tuple[int, float]]],
    component_lengths: Sequence[float],
):
    stack = [(start, -1, float(component_lengths[start]))]
    distances = {start: float(component_lengths[start])}
    parents = {start: -1}

    while stack:
        node, parent, distance = stack.pop()

        for neighbor, gap in adjacency[node]:
            if neighbor == parent:
                continue

            new_distance = (
                distance
                + float(gap)
                + float(component_lengths[neighbor])
            )

            distances[neighbor] = new_distance
            parents[neighbor] = node
            stack.append((neighbor, node, new_distance))

    farthest = max(distances, key=distances.get)
    return farthest, distances, parents


def tree_diameter_path(
    adjacency: Dict[int, List[Tuple[int, float]]],
    component_lengths: Sequence[float],
) -> List[int]:
    if len(component_lengths) == 1:
        return [0]

    start, _, _ = farthest_tree_node(
        0,
        adjacency,
        component_lengths,
    )

    end, _, parents = farthest_tree_node(
        start,
        adjacency,
        component_lengths,
    )

    path = []
    current = end

    while current != -1:
        path.append(current)

        if current == start:
            break

        current = parents[current]

    path.reverse()

    return path


def orient_component_for_next(
    component: LineString,
    next_component: Optional[LineString],
) -> List[Tuple[float, float]]:
    coords = list(component.coords)

    if next_component is None:
        return [(float(x), float(y)) for x, y, *rest in coords]

    next_start, next_end = component_endpoints(next_component)
    current_start = np.asarray(coords[0][:2], dtype="float64")
    current_end = np.asarray(coords[-1][:2], dtype="float64")

    distance_if_keep = min(
        np.linalg.norm(current_end - next_start),
        np.linalg.norm(current_end - next_end),
    )
    distance_if_reverse = min(
        np.linalg.norm(current_start - next_start),
        np.linalg.norm(current_start - next_end),
    )

    if distance_if_reverse < distance_if_keep:
        coords = coords[::-1]

    return [(float(coord[0]), float(coord[1])) for coord in coords]


def orient_component_from_previous(
    component: LineString,
    previous_end: Tuple[float, float],
) -> List[Tuple[float, float]]:
    coords = list(component.coords)

    start = np.asarray(coords[0][:2], dtype="float64")
    end = np.asarray(coords[-1][:2], dtype="float64")
    previous = np.asarray(previous_end, dtype="float64")

    if np.linalg.norm(previous - end) < np.linalg.norm(previous - start):
        coords = coords[::-1]

    return [(float(coord[0]), float(coord[1])) for coord in coords]


def stitch_component_path(
    components: Sequence[LineString],
    component_path: Sequence[int],
):
    if not component_path:
        raise RuntimeError("Road-backbone component path is empty.")

    stitched_coords: List[Tuple[float, float]] = []
    gap_lengths: List[float] = []
    connector_lines: List[LineString] = []

    for path_position, component_index in enumerate(component_path):
        component = components[component_index]

        if path_position == 0:
            next_component = (
                components[component_path[1]]
                if len(component_path) > 1
                else None
            )

            coords = orient_component_for_next(
                component,
                next_component,
            )

            stitched_coords.extend(coords)
            continue

        previous_end = stitched_coords[-1]

        coords = orient_component_from_previous(
            component,
            previous_end,
        )

        start = coords[0]

        gap = float(
            math.hypot(
                start[0] - previous_end[0],
                start[1] - previous_end[1],
            )
        )

        gap_lengths.append(gap)

        # Store every synthetic connector explicitly for QA. The connector is
        # part of the chainage line only; it is never used as an ML predictor.
        if gap > 0:
            connector_lines.append(
                LineString([previous_end, start])
            )
            stitched_coords.append(start)

        stitched_coords.extend(coords[1:])

    backbone = LineString(stitched_coords)

    return backbone, gap_lengths, connector_lines


def orient_backbone_consistently(backbone: LineString) -> LineString:
    """Orient the final line reproducibly from south to north when requested."""
    if not Config.ROAD_ORIENT_SOUTH_TO_NORTH:
        return backbone

    coords = list(backbone.coords)
    if len(coords) < 2:
        return backbone

    first = coords[0]
    last = coords[-1]

    reverse = False
    if float(first[1]) > float(last[1]):
        reverse = True
    elif math.isclose(float(first[1]), float(last[1]), abs_tol=1.0e-9):
        # Deterministic fallback for nearly equal northings.
        reverse = float(first[0]) > float(last[0])

    if reverse:
        return LineString(coords[::-1])

    return backbone


@dataclass
class RoadChainageModel:
    """Ordered real-road components used for corridor chainage."""
    components: List[LineString]
    offsets_m: List[float]
    gap_lengths_m: List[float]
    total_chainage_length_m: float
    source_mode: str


def _reverse_linestring(line: LineString) -> LineString:
    return LineString(list(line.coords)[::-1])


def prepare_virtual_component_chainage(
    components: Sequence[LineString],
    component_path: Sequence[int],
    source_mode: str,
):
    """
    Order and orient disconnected real-road components.

    The stitched line is written only for QA visualization. Actual sample
    chainage is calculated by projection to the nearest REAL selected road
    component plus a cumulative component offset. Virtual connectors therefore
    cannot attract samples or alter nearest-road assignment.
    """
    if not component_path:
        raise RuntimeError("Road-backbone component path is empty.")

    oriented_components: List[LineString] = []

    for path_position, component_index in enumerate(component_path):
        component = components[component_index]
        if path_position == 0:
            next_component = (
                components[component_path[1]]
                if len(component_path) > 1
                else None
            )
            coords = orient_component_for_next(component, next_component)
        else:
            previous_end = tuple(oriented_components[-1].coords[-1][:2])
            coords = orient_component_from_previous(component, previous_end)
        oriented_components.append(LineString(coords))

    # Reproducible chainage direction: generally south -> north.
    if Config.ROAD_ORIENT_SOUTH_TO_NORTH and oriented_components:
        first_xy = oriented_components[0].coords[0]
        last_xy = oriented_components[-1].coords[-1]
        reverse = (
            float(first_xy[1]) > float(last_xy[1])
            or (
                math.isclose(float(first_xy[1]), float(last_xy[1]), abs_tol=1.0e-9)
                and float(first_xy[0]) > float(last_xy[0])
            )
        )
        if reverse:
            oriented_components = [
                _reverse_linestring(line)
                for line in oriented_components[::-1]
            ]

    gap_lengths: List[float] = []
    connector_lines: List[LineString] = []
    for index in range(len(oriented_components) - 1):
        end_xy = tuple(oriented_components[index].coords[-1][:2])
        start_xy = tuple(oriented_components[index + 1].coords[0][:2])
        gap = float(math.hypot(start_xy[0] - end_xy[0], start_xy[1] - end_xy[1]))
        gap_lengths.append(gap)
        if gap > 0.0:
            connector_lines.append(LineString([end_xy, start_xy]))

    offsets_m: List[float] = []
    running = 0.0
    for index, component in enumerate(oriented_components):
        offsets_m.append(float(running))
        running += float(component.length)
        if index < len(gap_lengths):
            running += float(gap_lengths[index])

    model = RoadChainageModel(
        components=oriented_components,
        offsets_m=offsets_m,
        gap_lengths_m=gap_lengths,
        total_chainage_length_m=float(running),
        source_mode=source_mode,
    )

    stitched_coords: List[Tuple[float, float]] = []
    for index, component in enumerate(oriented_components):
        coords = [(float(c[0]), float(c[1])) for c in component.coords]
        if index == 0:
            stitched_coords.extend(coords)
        else:
            # QA visualization only: draw the virtual gap to the next component.
            stitched_coords.append(coords[0])
            stitched_coords.extend(coords[1:])

    display_backbone = LineString(stitched_coords)
    return display_backbone, gap_lengths, connector_lines, oriented_components, model


def chainage_model_from_continuous_line(
    line: LineString,
    source_mode: str,
) -> RoadChainageModel:
    return RoadChainageModel(
        components=[line],
        offsets_m=[0.0],
        gap_lengths_m=[],
        total_chainage_length_m=float(line.length),
        source_mode=source_mode,
    )


def write_component_chainage_table(
    path: Path,
    model: RoadChainageModel,
) -> None:
    rows = []
    for index, (component, offset) in enumerate(zip(model.components, model.offsets_m)):
        start = component.coords[0]
        end = component.coords[-1]
        rows.append(
            {
                "component_order": int(index + 1),
                "chainage_offset_m": float(offset),
                "component_length_m": float(component.length),
                "virtual_gap_before_m": float(0.0 if index == 0 else model.gap_lengths_m[index - 1]),
                "start_x": float(start[0]),
                "start_y": float(start[1]),
                "end_x": float(end[0]),
                "end_y": float(end[1]),
                "source_mode": model.source_mode,
            }
        )
    pd.DataFrame(rows).to_csv(path, index=False, float_format="%.8f")


def read_continuous_manual_backbone(
    manual_path: Path,
    target_crs: CRS,
) -> Tuple[LineString, RoadChainageModel, Dict[str, Any], List[Dict[str, Any]]]:
    """
    Read a user-cleaned corridor line.

    The geometry must merge to exactly one continuous LineString. This avoids
    silently reintroducing large synthetic connectors into a manually verified
    chainage backbone.
    """
    require_file(manual_path, "manually cleaned NTD chainage backbone")

    raw_parts: List[LineString] = []

    with fiona.open(manual_path) as source:
        source_crs = source_crs_from_fiona(source)
        reproj = make_reprojector(source_crs, target_crs)
        source_feature_count = 0

        for feature in source:
            source_feature_count += 1
            geometry = reproj(shape(feature["geometry"]))
            raw_parts.extend(extract_line_parts(geometry))

    raw_parts = [
        line
        for line in raw_parts
        if line is not None and not line.is_empty and line.length > 0
    ]

    if not raw_parts:
        raise RuntimeError(
            "The manually cleaned road-backbone file contains no usable linework."
        )

    unioned = unary_union(raw_parts)
    if isinstance(unioned, LineString):
        merged = unioned
    else:
        try:
            merged = linemerge(unioned)
        except Exception:
            merged = unioned

    merged_parts = extract_line_parts(merged)

    if len(merged_parts) != 1:
        raise RuntimeError(
            "The manually cleaned road backbone does not merge to one continuous "
            f"LineString; {len(merged_parts)} disconnected components remain. "
            "Please clean/snap the linework before using it for spatial-CV chainage."
        )

    backbone = orient_backbone_consistently(merged_parts[0])

    qa_features: List[Dict[str, Any]] = []
    for index, line in enumerate(raw_parts):
        qa_features.append(
            {
                "geometry": line,
                "feature_type": "manual_source_part",
                "component_id": int(index),
                "selected": 1,
                "length_m": float(line.length),
                "gap_m": 0.0,
            }
        )

    metadata = {
        "source_path": str(manual_path),
        "source_feature_count": int(source_feature_count),
        "raw_line_part_count": int(len(raw_parts)),
        "merged_component_count": 1,
        "largest_merged_component_fraction": 1.0,
        "largest_merged_component_percent": 100.0,
        "backbone_component_indices": [0],
        "backbone_component_count": 1,
        "total_original_line_length_m": float(sum(line.length for line in raw_parts)),
        "retained_original_line_length_m": float(backbone.length),
        "retained_original_length_fraction": 1.0,
        "retained_original_length_percent": 100.0,
        "omitted_branch_or_off_backbone_length_m": 0.0,
        "synthetic_connector_count": 0,
        "synthetic_connector_lengths_m": [],
        "total_synthetic_connector_length_m": 0.0,
        "max_synthetic_gap_m": 0.0,
        "final_backbone_length_m": float(backbone.length),
        "method": "user-cleaned continuous corridor backbone",
        "scientific_use": (
            "chainage and spatial-block ordering only; not used as an ML predictor"
        ),
        "backbone_source_mode": "manual_cleaned",
        "manual_backbone_path": str(manual_path),
        "visual_approval_declared": True,
        "warnings": [],
    }

    chainage_model = chainage_model_from_continuous_line(
        backbone,
        source_mode="manual_cleaned",
    )

    return backbone, chainage_model, metadata, qa_features


def write_road_backbone_gpkg(
    path: Path,
    backbone: LineString,
    crs: CRS,
    metadata: Dict[str, Any],
) -> None:
    if not Config.WRITE_ROAD_BACKBONE_GPKG:
        return

    if path.exists():
        path.unlink()

    schema = {
        "geometry": "LineString",
        "properties": {
            "length_m": "float",
            "components": "int",
            "retained_pc": "float",
            "gap_max_m": "float",
            "source_mode": "str:32",
        },
    }

    with fiona.open(
        path,
        mode="w",
        driver="GPKG",
        crs_wkt=crs.to_wkt(),
        schema=schema,
        layer="chainage_backbone",
    ) as sink:
        sink.write(
            {
                "geometry": mapping(backbone),
                "properties": {
                    "length_m": float(backbone.length),
                    "components": int(metadata["backbone_component_count"]),
                    "retained_pc": float(
                        100.0 * metadata["retained_original_length_fraction"]
                    ),
                    "gap_max_m": float(metadata["max_synthetic_gap_m"]),
                    "source_mode": str(
                        metadata.get("backbone_source_mode", "unknown")
                    )[:32],
                },
            }
        )


def write_road_diagnostic_gpkg(
    path: Path,
    qa_features: Sequence[Dict[str, Any]],
    crs: CRS,
) -> None:
    """
    Write selected/omitted components and synthetic connectors in one QA layer.
    This file is intended for visual inspection before spatial-CV modelling.
    """
    if path.exists():
        path.unlink()

    schema = {
        "geometry": "LineString",
        "properties": {
            "type": "str:40",
            "comp_id": "int",
            "selected": "int",
            "length_m": "float",
            "gap_m": "float",
        },
    }

    with fiona.open(
        path,
        mode="w",
        driver="GPKG",
        crs_wkt=crs.to_wkt(),
        schema=schema,
        layer="road_backbone_QA",
    ) as sink:
        for feature_record in qa_features:
            geometry = feature_record.get("geometry")
            if geometry is None or geometry.is_empty:
                continue

            sink.write(
                {
                    "geometry": mapping(geometry),
                    "properties": {
                        "type": str(
                            feature_record.get("feature_type", "unknown")
                        )[:40],
                        "comp_id": int(
                            feature_record.get("component_id", -1)
                        ),
                        "selected": int(
                            feature_record.get("selected", 0)
                        ),
                        "length_m": float(
                            feature_record.get("length_m", geometry.length)
                        ),
                        "gap_m": float(
                            feature_record.get("gap_m", 0.0)
                        ),
                    },
                }
            )


def build_automatic_road_backbone(
    road_path: Path,
    target_crs: CRS,
) -> Tuple[LineString, RoadChainageModel, Dict[str, Any], List[Dict[str, Any]]]:
    require_file(road_path, "NTD road shapefile")

    raw_line_parts: List[LineString] = []

    with fiona.open(road_path) as source:
        source_crs = source_crs_from_fiona(source)
        reproj = make_reprojector(source_crs, target_crs)

        source_feature_count = 0

        for feature in source:
            source_feature_count += 1
            geometry = reproj(shape(feature["geometry"]))
            raw_line_parts.extend(extract_line_parts(geometry))

    raw_line_parts = [
        line
        for line in raw_line_parts
        if line is not None and not line.is_empty and line.length > 0
    ]

    if not raw_line_parts:
        raise RuntimeError("No usable line geometry was found in NTD_Road.shp.")

    total_original_length = float(sum(line.length for line in raw_line_parts))

    unioned = unary_union(raw_line_parts)

    if isinstance(unioned, LineString):
        merged = unioned
    else:
        try:
            merged = linemerge(unioned)
        except Exception:
            merged = unioned

    merged_components = extract_line_parts(merged)

    if not merged_components:
        merged_components = raw_line_parts

    merged_components = [
        line
        for line in merged_components
        if line.length > 0
    ]

    merged_component_lengths = [
        float(line.length)
        for line in merged_components
    ]

    largest_component_fraction = (
        max(merged_component_lengths) / total_original_length
        if total_original_length > 0
        else 1.0
    )

    if len(merged_components) == 1:
        component_path = [0]
        mst_edges = []
    else:
        adjacency, mst_edges = build_component_mst(merged_components)
        component_path = tree_diameter_path(
            adjacency,
            merged_component_lengths,
        )

    (
        backbone,
        gap_lengths,
        connector_lines,
        oriented_components,
        chainage_model,
    ) = prepare_virtual_component_chainage(
        merged_components,
        component_path,
        source_mode="automatic_virtual_component_chainage",
    )

    retained_original_length = float(
        sum(
            merged_component_lengths[index]
            for index in component_path
        )
    )

    retained_original_fraction = (
        retained_original_length / total_original_length
        if total_original_length > 0
        else 1.0
    )

    omitted_original_length = max(
        0.0,
        total_original_length - retained_original_length,
    )

    total_synthetic_gap = float(sum(gap_lengths)) if gap_lengths else 0.0
    max_synthetic_gap = float(max(gap_lengths)) if gap_lengths else 0.0

    selected_component_set = set(component_path)
    qa_features: List[Dict[str, Any]] = []

    for index, component in enumerate(merged_components):
        selected = int(index in selected_component_set)
        qa_features.append(
            {
                "geometry": component,
                "feature_type": (
                    "selected_component"
                    if selected
                    else "omitted_component"
                ),
                "component_id": int(index),
                "selected": selected,
                "length_m": float(component.length),
                "gap_m": 0.0,
            }
        )

    for connector_index, connector in enumerate(connector_lines):
        qa_features.append(
            {
                "geometry": connector,
                "feature_type": "synthetic_connector",
                "component_id": int(connector_index),
                "selected": 1,
                "length_m": float(connector.length),
                "gap_m": float(connector.length),
            }
        )

    metadata = {
        "source_path": str(road_path),
        "source_feature_count": int(source_feature_count),
        "raw_line_part_count": int(len(raw_line_parts)),
        "merged_component_count": int(len(merged_components)),
        "largest_merged_component_fraction": float(largest_component_fraction),
        "largest_merged_component_percent": float(
            100.0 * largest_component_fraction
        ),
        "backbone_component_indices": [int(i) for i in component_path],
        "backbone_component_count": int(len(component_path)),
        "total_original_line_length_m": float(total_original_length),
        "retained_original_line_length_m": float(retained_original_length),
        "retained_original_length_fraction": float(retained_original_fraction),
        "retained_original_length_percent": float(
            100.0 * retained_original_fraction
        ),
        "omitted_branch_or_off_backbone_length_m": float(omitted_original_length),
        "synthetic_connector_count": int(len(gap_lengths)),
        "synthetic_connector_lengths_m": [float(value) for value in gap_lengths],
        "total_synthetic_connector_length_m": float(total_synthetic_gap),
        "max_synthetic_gap_m": float(max_synthetic_gap),
        "final_backbone_length_m": float(chainage_model.total_chainage_length_m),
        "method": (
            "minimum-gap component tree plus longest corridor path; "
            "points projected only to ordered real-road components; "
            "disconnected gaps represented as virtual cumulative chainage offsets"
        ),
        "chainage_projection_mode": "nearest_real_component_plus_virtual_gap_offsets",
        "synthetic_connectors_used_for_point_projection": False,
        "virtual_gap_chainage_only": True,
        "scientific_use": (
            "chainage and spatial-block ordering only; not used as an ML predictor"
        ),
        "backbone_source_mode": "automatic_virtual_component_chainage",
        "manual_backbone_path": str(Config.MANUAL_ROAD_BACKBONE),
        "visual_approval_declared": bool(
            Config.ROAD_BACKBONE_VISUALLY_APPROVED
        ),
    }

    warnings_list = []

    if (
        retained_original_fraction
        < Config.ROAD_WARN_RETAINED_ORIGINAL_FRACTION
    ):
        warnings_list.append(
            "Backbone retains less than the configured warning fraction of "
            "original road linework."
        )

    if max_synthetic_gap > Config.ROAD_WARN_MAX_SYNTHETIC_GAP_M:
        warnings_list.append(
            "At least one synthetic topology-gap connector is longer than the "
            "configured warning threshold."
        )

    metadata["warnings"] = warnings_list

    return backbone, chainage_model, metadata, qa_features


def choose_road_backbone(
    target_crs: CRS,
) -> Tuple[LineString, RoadChainageModel, Dict[str, Any], List[Dict[str, Any]]]:
    mode = str(Config.ROAD_BACKBONE_MODE).strip().lower()
    allowed = {
        "manual_required",
        "manual_if_available",
        "automatic",
    }

    if mode not in allowed:
        raise ValueError(
            "ROAD_BACKBONE_MODE must be one of "
            f"{sorted(allowed)}; received {Config.ROAD_BACKBONE_MODE!r}."
        )

    manual_exists = Config.MANUAL_ROAD_BACKBONE.exists()

    if mode == "manual_required":
        if not manual_exists:
            raise FileNotFoundError(
                "ROAD_BACKBONE_MODE='manual_required', but the cleaned backbone "
                f"does not exist:\n{Config.MANUAL_ROAD_BACKBONE}"
            )
        return read_continuous_manual_backbone(
            Config.MANUAL_ROAD_BACKBONE,
            target_crs,
        )

    if mode == "manual_if_available" and manual_exists:
        log(
            "[ROAD] Using manually cleaned/verified chainage backbone: "
            f"{Config.MANUAL_ROAD_BACKBONE}"
        )
        return read_continuous_manual_backbone(
            Config.MANUAL_ROAD_BACKBONE,
            target_crs,
        )

    if mode == "manual_if_available" and not manual_exists:
        log(
            "[ROAD] Manual chainage backbone not found; automatic ordered-component "
            "chainage will be used with virtual gaps and QA checks."
        )

    return build_automatic_road_backbone(
        Config.ROAD_SHP,
        target_crs,
    )


def evaluate_road_backbone_gate(
    metadata: Dict[str, Any],
) -> Tuple[bool, str]:
    """
    Validate the chainage representation.

    Long disconnected gaps do not fail when virtual-component chainage is used,
    because points are projected only to real road components. Automatic use is
    blocked only when the selected corridor path omits too much original road
    linework, indicating an ambiguous branch/path selection.
    """
    source_mode = str(metadata.get("backbone_source_mode", "unknown"))

    if source_mode == "manual_cleaned":
        return True, "PASS: manually cleaned continuous backbone used."

    retained_fraction = float(metadata.get("retained_original_length_fraction", 0.0))
    minimum_fraction = float(
        Config.ROAD_MIN_RETAINED_FRACTION_FOR_AUTOMATIC_CHAINAGE
    )

    if retained_fraction < minimum_fraction:
        return False, (
            "FAIL_ROAD_PATH_AMBIGUOUS: the automatic corridor path retains only "
            f"{100.0 * retained_fraction:.2f}% of original road linework, below "
            f"the required {100.0 * minimum_fraction:.2f}%. Provide a manually "
            "cleaned NTD corridor backbone before spatial-CV sampling."
        )

    max_gap = float(metadata.get("max_synthetic_gap_m", 0.0))
    threshold = float(Config.ROAD_REQUIRE_REVIEW_ABOVE_GAP_M)

    if max_gap > threshold:
        return True, (
            "PASS_WITH_VIRTUAL_GAP_WARNING: the ordered road components contain "
            f"a {max_gap:.2f} m disconnected gap. The gap is used only as a "
            "cumulative chainage offset; points are projected exclusively to "
            "real road components, so the virtual gap cannot attract samples or "
            "distort nearest-road assignment. Visual inspection remains recommended."
        )

    return True, "PASS: automatic ordered-component chainage passed QA."


def write_road_review_text(
    path: Path,
    metadata: Dict[str, Any],
) -> None:
    with open(path, "w", encoding="utf-8") as handle:
        handle.write("STEP 6.3 — NTD ROAD BACKBONE REVIEW\n")
        handle.write("=" * 72 + "\n\n")
        handle.write(
            f"Backbone source mode: "
            f"{metadata.get('backbone_source_mode', 'unknown')}\n"
        )
        handle.write(
            f"Merged components: "
            f"{metadata.get('merged_component_count', 0)}\n"
        )
        handle.write(
            f"Retained original road length: "
            f"{metadata.get('retained_original_length_percent', float('nan')):.2f}%\n"
        )
        handle.write(
            f"Synthetic connector count: "
            f"{metadata.get('synthetic_connector_count', 0)}\n"
        )
        handle.write(
            f"Maximum synthetic connector: "
            f"{metadata.get('max_synthetic_gap_m', 0.0):.2f} m\n"
        )
        handle.write(
            f"Sampling gate passed: "
            f"{metadata.get('sampling_gate_passed', False)}\n"
        )
        handle.write(
            f"Gate reason: "
            f"{metadata.get('sampling_gate_reason', '')}\n\n"
        )
        handle.write(
            "Files to inspect:\n"
            f"  Final backbone: {Config.ROAD_BACKBONE_GPKG}\n"
            f"  Diagnostic linework: {Config.ROAD_DIAGNOSTIC_GPKG}\n\n"
        )
        handle.write(
            "Interpretation:\n"
            "The final backbone is used only to order samples longitudinally and "
            "define the five contiguous spatial-CV blocks. A long or incorrectly "
            "placed synthetic connector can therefore distort block ordering even "
            "though it is not an ML predictor.\n"
        )


# =============================================================================
# REPRESENTATIVE POSITIVE SITE SELECTION
# =============================================================================
def clamp_window_to_raster(
    window: Window,
    width: int,
    height: int,
) -> Optional[Window]:
    col_off = max(0, int(math.floor(window.col_off)))
    row_off = max(0, int(math.floor(window.row_off)))

    col_end = min(
        width,
        int(math.ceil(window.col_off + window.width)),
    )
    row_end = min(
        height,
        int(math.ceil(window.row_off + window.height)),
    )

    if col_end <= col_off or row_end <= row_off:
        return None

    return Window(
        col_off=col_off,
        row_off=row_off,
        width=col_end - col_off,
        height=row_end - row_off,
    )


def candidate_cells_for_geometry(
    geometry,
    mask: np.ndarray,
    raster_transform,
    all_touched: bool,
) -> Tuple[np.ndarray, np.ndarray]:
    height, width = mask.shape

    window = from_bounds(
        *geometry.bounds,
        transform=raster_transform,
    )

    window = clamp_window_to_raster(
        window,
        width,
        height,
    )

    if window is None:
        return (
            np.array([], dtype=int),
            np.array([], dtype=int),
        )

    row_off = int(window.row_off)
    col_off = int(window.col_off)
    window_height = int(window.height)
    window_width = int(window.width)

    local_transform = window_transform(
        window,
        raster_transform,
    )

    geometry_mask = features.geometry_mask(
        [mapping(geometry)],
        out_shape=(window_height, window_width),
        transform=local_transform,
        invert=True,
        all_touched=all_touched,
    )

    local_valid = (
        geometry_mask
        & mask[
            row_off : row_off + window_height,
            col_off : col_off + window_width,
        ]
    )

    local_rows, local_cols = np.where(local_valid)

    return (
        local_rows + row_off,
        local_cols + col_off,
    )


def choose_representative_cell(
    geometry,
    common_domain: np.ndarray,
    study_mask: np.ndarray,
    raster_transform,
) -> Dict[str, Any]:
    if geometry is None or geometry.is_empty:
        return {"status": "invalid_geometry"}

    study_rows, study_cols = candidate_cells_for_geometry(
        geometry,
        study_mask,
        raster_transform,
        all_touched=False,
    )

    if study_rows.size == 0 and Config.ALL_TOUCHED_FALLBACK:
        study_rows, study_cols = candidate_cells_for_geometry(
            geometry,
            study_mask,
            raster_transform,
            all_touched=True,
        )

    if study_rows.size == 0:
        return {"status": "outside_study_area_or_no_study_cell"}

    rows, cols = candidate_cells_for_geometry(
        geometry,
        common_domain,
        raster_transform,
        all_touched=False,
    )
    method = "cell_center_inside_nearest_centroid"

    if rows.size == 0 and Config.ALL_TOUCHED_FALLBACK:
        rows, cols = candidate_cells_for_geometry(
            geometry,
            common_domain,
            raster_transform,
            all_touched=True,
        )
        method = "all_touched_fallback_nearest_centroid"

    if rows.size == 0:
        return {"status": "no_valid_common_domain_pixel"}

    xs, ys = rasterio.transform.xy(
        raster_transform,
        rows,
        cols,
        offset="center",
    )

    xs = np.asarray(xs, dtype="float64")
    ys = np.asarray(ys, dtype="float64")

    centroid = geometry.centroid

    distance_squared = (
        (xs - centroid.x) ** 2
        + (ys - centroid.y) ** 2
    )

    selected = int(np.argmin(distance_squared))

    return {
        "status": "retained_common_domain",
        "row": int(rows[selected]),
        "col": int(cols[selected]),
        "x": float(xs[selected]),
        "y": float(ys[selected]),
        "representative_method": method,
        "candidate_valid_cells": int(rows.size),
        "distance_to_polygon_centroid_m": float(
            math.sqrt(distance_squared[selected])
        ),
        "polygon_area_m2": float(geometry.area),
    }


# =============================================================================
# CHAINAGE AND FIVE CONTIGUOUS BLOCKS
# =============================================================================
def calculate_chainage(
    road_chainage_model: RoadChainageModel,
    x: float,
    y: float,
) -> Tuple[float, float]:
    """Project a point only to the nearest REAL ordered road component."""
    point = Point(float(x), float(y))

    if not road_chainage_model.components:
        raise RuntimeError("Road chainage model contains no components.")

    distances = np.asarray(
        [float(point.distance(component)) for component in road_chainage_model.components],
        dtype="float64",
    )
    component_index = int(np.argmin(distances))
    component = road_chainage_model.components[component_index]

    local_chainage = float(component.project(point))
    chainage = float(
        road_chainage_model.offsets_m[component_index] + local_chainage
    )
    return chainage, float(distances[component_index])


def diagnose_point_against_synthetic_connectors(
    x: float,
    y: float,
    distance_to_backbone_m: float,
    road_qa_features: Sequence[Dict[str, Any]],
    tolerance_m: float = 1.0e-6,
) -> Tuple[float, bool]:
    """
    Return distance to the nearest synthetic connector and whether that
    connector is also the nearest part of the final backbone.

    This is a QA diagnostic only and does not alter sample selection.
    """
    connectors = [
        record["geometry"]
        for record in road_qa_features
        if record.get("feature_type") == "synthetic_connector"
        and record.get("geometry") is not None
        and not record["geometry"].is_empty
    ]

    if not connectors:
        return float("nan"), False

    point = Point(float(x), float(y))
    minimum = min(float(point.distance(line)) for line in connectors)

    projected_to_connector = bool(
        abs(minimum - float(distance_to_backbone_m)) <= tolerance_m
    )

    return float(minimum), projected_to_connector


def define_balanced_contiguous_blocks(
    positive_df: pd.DataFrame,
    n_blocks: int,
):
    if len(positive_df) < n_blocks:
        raise RuntimeError(
            f"Only {len(positive_df)} positive sites are available for "
            f"{n_blocks} spatial blocks."
        )

    ordered = positive_df.sort_values(
        ["chainage_m", "site_id"]
    ).reset_index(drop=True)

    row_groups = np.array_split(
        np.arange(len(ordered)),
        n_blocks,
    )

    ordered["spatial_block"] = 0

    for block_number, row_indices in enumerate(
        row_groups,
        start=1,
    ):
        ordered.loc[row_indices, "spatial_block"] = block_number

    boundaries: List[float] = []

    for block_number in range(1, n_blocks):
        current_max = float(
            ordered.loc[
                ordered["spatial_block"] == block_number,
                "chainage_m",
            ].max()
        )
        next_min = float(
            ordered.loc[
                ordered["spatial_block"] == block_number + 1,
                "chainage_m",
            ].min()
        )
        boundaries.append((current_max + next_min) / 2.0)

    boundary_rows = []

    for block_number in range(1, n_blocks + 1):
        block_df = ordered[
            ordered["spatial_block"] == block_number
        ]

        lower = 0.0 if block_number == 1 else boundaries[block_number - 2]
        upper = None if block_number == n_blocks else boundaries[block_number - 1]

        boundary_rows.append(
            {
                "spatial_block": int(block_number),
                "lower_chainage_m_inclusive": float(lower),
                "upper_chainage_m_exclusive": (
                    None if upper is None else float(upper)
                ),
                "n_positive_sites": int(len(block_df)),
                "min_positive_chainage_m": float(block_df["chainage_m"].min()),
                "max_positive_chainage_m": float(block_df["chainage_m"].max()),
            }
        )

    output = positive_df.merge(
        ordered[["site_id", "spatial_block"]],
        on="site_id",
        how="left",
        validate="one_to_one",
    )

    return output, boundaries, pd.DataFrame(boundary_rows)


def assign_block_from_chainage(
    chainage_m: float,
    boundaries: Sequence[float],
) -> int:
    return int(
        np.searchsorted(
            np.asarray(boundaries, dtype="float64"),
            float(chainage_m),
            side="right",
        )
        + 1
    )


# =============================================================================
# NEGATIVE CANDIDATE MASKS AND 1:1 BLOCK-MATCHED SAMPLING
# =============================================================================
def write_mask_tif(
    path: Path,
    mask: np.ndarray,
    reference_profile: Dict[str, Any],
    description: str,
) -> None:
    profile = reference_profile.copy()
    profile.update(
        driver="GTiff",
        dtype="uint8",
        count=1,
        nodata=Config.MASK_NODATA,
        compress="LZW",
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

    temporary = Path(str(path) + ".part")

    with rasterio.open(temporary, "w", **profile) as dataset:
        dataset.write(mask.astype("uint8"), 1)
        dataset.set_band_description(1, description)

        try:
            dataset.build_overviews(
                [2, 4, 8, 16],
                Resampling.nearest,
            )
            dataset.update_tags(
                ns="rio_overview",
                resampling="nearest",
            )
        except Exception:
            pass

    if path.exists():
        path.unlink()

    os.replace(temporary, path)


def build_negative_candidate_mask(
    common_domain: np.ndarray,
    landslide_geometries: Sequence[Any],
    buffer_m: float,
    raster_transform,
) -> np.ndarray:
    buffered = []

    for geometry in landslide_geometries:
        if geometry is None or geometry.is_empty:
            continue

        geometry_buffer = geometry.buffer(float(buffer_m))

        if not geometry_buffer.is_empty:
            buffered.append(geometry_buffer)

    if not buffered:
        raise RuntimeError(
            f"No landslide buffers could be created for {buffer_m} m."
        )

    exclusion_union = unary_union(buffered)

    exclusion_mask = features.rasterize(
        [(exclusion_union, 1)],
        out_shape=common_domain.shape,
        transform=raster_transform,
        fill=0,
        dtype="uint8",
        all_touched=True,
    ).astype(bool)

    return common_domain & ~exclusion_mask


def sample_block_matched_negatives(
    candidate_mask: np.ndarray,
    priority_flat_indices: np.ndarray,
    positive_counts_by_block: Dict[int, int],
    road_chainage_model: RoadChainageModel,
    boundaries: Sequence[float],
    raster_transform,
    raster_width: int,
    buffer_m: int,
) -> pd.DataFrame:
    required_by_block = {
        int(block): int(count)
        for block, count in positive_counts_by_block.items()
    }

    selected_counts = defaultdict(int)
    selected_rows = []

    total_required = int(sum(required_by_block.values()))

    for priority_rank, flat_index in enumerate(priority_flat_indices):
        if not candidate_mask.ravel()[flat_index]:
            continue

        row = int(flat_index // raster_width)
        col = int(flat_index % raster_width)

        x, y = rasterio.transform.xy(
            raster_transform,
            row,
            col,
            offset="center",
        )

        chainage_m, distance_to_backbone_m = calculate_chainage(
            road_chainage_model,
            float(x),
            float(y),
        )

        spatial_block = assign_block_from_chainage(
            chainage_m,
            boundaries,
        )

        if spatial_block not in required_by_block:
            continue

        if selected_counts[spatial_block] >= required_by_block[spatial_block]:
            continue

        selected_counts[spatial_block] += 1

        sample_id = (
            f"NEG_B{int(buffer_m):04d}_"
            f"{len(selected_rows) + 1:05d}"
        )

        selected_rows.append(
            {
                "sample_id": sample_id,
                "site_id": sample_id,
                "label": 0,
                "class_name": "pseudo_absence",
                "inventory_source": "none",
                "source_feature_index": np.nan,
                "source_original_id": None,
                "row": row,
                "col": col,
                "x": float(x),
                "y": float(y),
                "chainage_m": float(chainage_m),
                "distance_to_road_backbone_m": float(distance_to_backbone_m),
                "spatial_block": int(spatial_block),
                "buffer_m": int(buffer_m),
                "negative_priority_rank": int(priority_rank),
                "representative_method": "random_common_domain_block_matched",
            }
        )

        if len(selected_rows) >= total_required:
            break

    missing = {
        block: required_by_block[block] - selected_counts[block]
        for block in required_by_block
        if selected_counts[block] < required_by_block[block]
    }

    if missing:
        raise RuntimeError(
            f"Not enough pseudo-absence candidates for buffer {buffer_m} m. "
            f"Missing counts by spatial block: {missing}"
        )

    negative_df = pd.DataFrame(selected_rows)

    observed = (
        negative_df["spatial_block"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    expected = dict(sorted(required_by_block.items()))

    if observed != expected:
        raise RuntimeError(
            f"Block matching failed for buffer {buffer_m} m.\n"
            f"Expected: {expected}\nObserved: {observed}"
        )

    return negative_df


# =============================================================================
# PREDICTOR EXTRACTION
# =============================================================================
def extract_predictors_for_sites(
    sites_df: pd.DataFrame,
    master_stack: np.ndarray,
    band_names: Sequence[str],
) -> pd.DataFrame:
    output = sites_df.copy()

    rows = output["row"].astype(int).to_numpy()
    cols = output["col"].astype(int).to_numpy()

    values = master_stack[:, rows, cols].T

    complete = np.isfinite(values).all(axis=1)

    if not complete.all():
        bad = np.where(~complete)[0]
        raise RuntimeError(
            "Missing predictor values were found despite using the common "
            f"A–F domain. Bad sample rows: {bad[:20].tolist()}"
        )

    for band_index, band_name in enumerate(band_names):
        output[band_name] = values[:, band_index].astype(float)

    return output


# =============================================================================
# POINT / LINE OUTPUTS
# =============================================================================
def write_points_gpkg(
    path: Path,
    dataframe: pd.DataFrame,
    crs: CRS,
    layer_name: str,
) -> None:
    if not Config.WRITE_POINT_GPKGS:
        return

    if path.exists():
        path.unlink()

    schema = {
        "geometry": "Point",
        "properties": {
            "sample_id": "str:80",
            "site_id": "str:80",
            "label": "int",
            "block": "int",
            "buffer_m": "int",
            "chainage_m": "float",
            "backbone_d": "float",
            "source": "str:40",
        },
    }

    with fiona.open(
        path,
        mode="w",
        driver="GPKG",
        crs_wkt=crs.to_wkt(),
        schema=schema,
        layer=layer_name,
    ) as sink:
        for _, row in dataframe.iterrows():
            sink.write(
                {
                    "geometry": mapping(
                        Point(float(row["x"]), float(row["y"]))
                    ),
                    "properties": {
                        "sample_id": str(row["sample_id"]),
                        "site_id": str(row["site_id"]),
                        "label": int(row["label"]),
                        "block": int(row["spatial_block"]),
                        "buffer_m": int(row.get("buffer_m", 0)),
                        "chainage_m": float(row["chainage_m"]),
                        "backbone_d": float(
                            row["distance_to_road_backbone_m"]
                        ),
                        "source": str(
                            row.get("inventory_source", "none")
                        )[:40],
                    },
                }
            )


# =============================================================================
# MODEL A–F VIEWS
# =============================================================================
def write_model_views(
    master_df: pd.DataFrame,
    scenario_manifest: Dict[str, Any],
    buffer_m: int,
) -> Dict[str, str]:
    if not Config.WRITE_MODEL_SCENARIO_VIEWS:
        return {}

    metadata_columns = [
        "sample_id",
        "site_id",
        "label",
        "class_name",
        "inventory_source",
        "source_feature_index",
        "source_original_id",
        "row",
        "col",
        "x",
        "y",
        "chainage_m",
        "distance_to_road_backbone_m",
        "spatial_block",
        "buffer_m",
        "representative_method",
    ]

    if "negative_priority_rank" in master_df.columns:
        metadata_columns.append("negative_priority_rank")

    outputs = {}

    for scenario_name in Config.SCENARIO_ORDER:
        features_list = list(
            scenario_manifest["scenarios"][scenario_name]["features"]
        )

        view = master_df[
            metadata_columns + features_list
        ].copy()

        path = (
            Config.OUT_DIR
            / f"{scenario_name}_samples_buffer{int(buffer_m)}m.csv"
        )

        view.to_csv(
            path,
            index=False,
            float_format="%.8f",
        )

        outputs[scenario_name] = str(path)

    return outputs


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=== Step 6.3: Common Site-Level Sampling for Models A–F ===")

    (
        master_stack,
        band_names,
        raster_profile,
        common_domain,
        study_mask,
        analysis_domain,
        stack_manifest,
        scenario_manifest,
        sampling_contract,
    ) = load_master_stack_and_masks()

    raster_crs = to_crs(raster_profile["crs"])
    raster_transform = raster_profile["transform"]
    raster_width = int(raster_profile["width"])

    log(f"[STACK] master predictors = {len(band_names)}")
    log(f"[MASK] common A–F domain = {int(common_domain.sum()):,} pixels")

    # Validate scenario feature availability.
    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in scenario_manifest.get("scenarios", {}):
            raise RuntimeError(
                f"Scenario '{scenario_name}' is missing from the Step 6.1 manifest."
            )

        missing = [
            feature
            for feature in scenario_manifest["scenarios"][scenario_name]["features"]
            if feature not in band_names
        ]

        if missing:
            raise RuntimeError(
                f"{scenario_name} references missing master predictors: {missing}"
            )

    # -------------------------------------------------------------------------
    # 1. Inventory
    # -------------------------------------------------------------------------
    train_sites, train_audit = read_landslide_inventory(
        Config.TRAIN_LANDSLIDE_SHP,
        source_tag="training_inventory",
        site_prefix="LS_TR",
        target_crs=raster_crs,
    )

    valid_sites, valid_audit = read_landslide_inventory(
        Config.VALID_LANDSLIDE_SHP,
        source_tag="validation_inventory",
        site_prefix="LS_VA",
        target_crs=raster_crs,
    )

    inventory_audit_df = pd.DataFrame(
        [
            {
                **train_audit,
                "historical_expected_source_features": (
                    Config.EXPECTED_TRAIN_SOURCE_FEATURES
                ),
            },
            {
                **valid_audit,
                "historical_expected_source_features": (
                    Config.EXPECTED_VALID_SOURCE_FEATURES
                ),
            },
        ]
    )

    inventory_audit_df.to_csv(
        Config.INVENTORY_AUDIT_CSV,
        index=False,
    )

    all_sites = train_sites + valid_sites

    log(
        f"[INVENTORY] TRAIN source features={train_audit['source_features_total']} "
        f"-> retained landslide records={len(train_sites)}"
    )
    log(
        f"[INVENTORY] VALID source features={valid_audit['source_features_total']} "
        f"-> retained landslide records={len(valid_sites)}"
    )

    if train_audit["symbol_field"] is not None:
        log(
            f"[INVENTORY] TRAIN {train_audit['symbol_field']} counts="
            f"{train_audit['symbol_counts']}"
        )

    if valid_audit["symbol_field"] is not None:
        log(
            f"[INVENTORY] VALID {valid_audit['symbol_field']} counts="
            f"{valid_audit['symbol_counts']}"
        )

    log(
        f"[INVENTORY] Combined retained landslide records={len(all_sites)}"
    )

    if len(all_sites) != Config.EXPECTED_TOTAL_LANDSLIDE_SITES:
        log(
            "[WARN] Combined retained inventory count differs from the historical "
            f"reference of {Config.EXPECTED_TOTAL_LANDSLIDE_SITES}. "
            "The actual retained source records will be used."
        )

    # -------------------------------------------------------------------------
    # 2. Robust highway backbone
    # -------------------------------------------------------------------------
    road_backbone, road_chainage_model, road_meta, road_qa_features = choose_road_backbone(
        raster_crs,
    )

    log(
        f"[ROAD] original merged components={road_meta['merged_component_count']}"
    )
    log(
        f"[ROAD] largest single component="
        f"{road_meta['largest_merged_component_percent']:.2f}% of total line length"
    )
    log(
        f"[ROAD] reconstructed backbone uses "
        f"{road_meta['backbone_component_count']} components and retains "
        f"{road_meta['retained_original_length_percent']:.2f}% of original line length"
    )
    log(
        f"[ROAD] synthetic connectors={road_meta['synthetic_connector_count']}, "
        f"maximum gap={road_meta['max_synthetic_gap_m']:.2f} m"
    )
    log(
        f"[ROAD] final chainage backbone length="
        f"{road_meta['final_backbone_length_m'] / 1000.0:.3f} km"
    )

    for warning in road_meta.get("warnings", []):
        log(f"[ROAD-WARN] {warning}")

    # Write both the final chainage line and the diagnostic source linework
    # BEFORE applying the sampling gate, so a failed automatic reconstruction
    # can still be inspected visually.
    write_road_backbone_gpkg(
        Config.ROAD_BACKBONE_GPKG,
        road_backbone,
        raster_crs,
        road_meta,
    )

    write_road_diagnostic_gpkg(
        Config.ROAD_DIAGNOSTIC_GPKG,
        road_qa_features,
        raster_crs,
    )

    write_component_chainage_table(
        Config.ROAD_COMPONENT_CHAINAGE_CSV,
        road_chainage_model,
    )

    gate_passed, gate_reason = evaluate_road_backbone_gate(
        road_meta,
    )
    road_meta["sampling_gate_passed"] = bool(gate_passed)
    road_meta["sampling_gate_reason"] = str(gate_reason)
    road_meta["review_threshold_m"] = float(
        Config.ROAD_REQUIRE_REVIEW_ABOVE_GAP_M
    )

    with open(
        Config.ROAD_QA_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            to_json_safe(road_meta),
            handle,
            indent=2,
        )

    write_road_review_text(
        Config.ROAD_REVIEW_TXT,
        road_meta,
    )

    log(f"[ROAD-GATE] {gate_reason}")

    if not gate_passed:
        raise RuntimeError(
            gate_reason
            + "\nQA outputs were written before stopping:\n"
            + f"  {Config.ROAD_BACKBONE_GPKG}\n"
            + f"  {Config.ROAD_DIAGNOSTIC_GPKG}\n"
            + f"  {Config.ROAD_QA_JSON}\n"
            + f"  {Config.ROAD_REVIEW_TXT}"
        )

    # -------------------------------------------------------------------------
    # 3. One representative common-domain cell per landslide
    # -------------------------------------------------------------------------
    reconciliation_rows = []
    retained_positive_rows = []

    for site in all_sites:
        selection = choose_representative_cell(
            geometry=site["geometry"],
            common_domain=common_domain,
            study_mask=study_mask,
            raster_transform=raster_transform,
        )

        audit_row = {
            "site_id": site["site_id"],
            "inventory_source": site["source_tag"],
            "source_file": site["source_file"],
            "source_feature_index": site["source_feature_index"],
            "source_original_id": site["source_original_id"],
            "symbol_field": site["symbol_field"],
            "symbol_value": site["symbol_value"],
            "geometry_repaired": site["geometry_repaired"],
            "status": selection["status"],
        }

        if selection["status"] == "retained_common_domain":
            chainage_m, backbone_distance_m = calculate_chainage(
                road_chainage_model,
                selection["x"],
                selection["y"],
            )

            (
                distance_to_synthetic_connector_m,
                _legacy_connector_would_be_nearest,
            ) = diagnose_point_against_synthetic_connectors(
                selection["x"],
                selection["y"],
                backbone_distance_m,
                road_qa_features,
            )
            # Chainage projection uses only real road components.
            projects_to_synthetic_connector = False

            audit_row.update(
                {
                    "row": selection["row"],
                    "col": selection["col"],
                    "x": selection["x"],
                    "y": selection["y"],
                    "representative_method": (
                        selection["representative_method"]
                    ),
                    "candidate_valid_cells": (
                        selection["candidate_valid_cells"]
                    ),
                    "distance_to_polygon_centroid_m": (
                        selection["distance_to_polygon_centroid_m"]
                    ),
                    "polygon_area_m2": selection["polygon_area_m2"],
                    "chainage_m": chainage_m,
                    "distance_to_road_backbone_m": backbone_distance_m,
                    "distance_to_synthetic_connector_m": (
                        distance_to_synthetic_connector_m
                    ),
                    "projects_to_synthetic_connector": bool(
                        projects_to_synthetic_connector
                    ),
                }
            )

            retained_positive_rows.append(
                {
                    "sample_id": site["site_id"],
                    "site_id": site["site_id"],
                    "label": 1,
                    "class_name": "landslide",
                    "inventory_source": site["source_tag"],
                    "source_feature_index": (
                        site["source_feature_index"]
                    ),
                    "source_original_id": site["source_original_id"],
                    "row": selection["row"],
                    "col": selection["col"],
                    "x": selection["x"],
                    "y": selection["y"],
                    "chainage_m": chainage_m,
                    "distance_to_road_backbone_m": backbone_distance_m,
                    "distance_to_synthetic_connector_m": (
                        distance_to_synthetic_connector_m
                    ),
                    "projects_to_synthetic_connector": bool(
                        projects_to_synthetic_connector
                    ),
                    "buffer_m": 0,
                    "representative_method": (
                        selection["representative_method"]
                    ),
                }
            )

        reconciliation_rows.append(audit_row)

    reconciliation_df = pd.DataFrame(reconciliation_rows)

    reconciliation_df.to_csv(
        Config.RECONCILIATION_CSV,
        index=False,
        float_format="%.8f",
    )

    positive_df = pd.DataFrame(retained_positive_rows)

    if positive_df.empty:
        raise RuntimeError(
            "No landslide sites were retained in the common A–F domain."
        )

    status_counts = reconciliation_df["status"].value_counts().to_dict()

    log(
        f"[POSITIVES] retained common-domain sites="
        f"{len(positive_df)}/{len(all_sites)}"
    )
    log(f"[POSITIVES] reconciliation status={status_counts}")

    positive_projects_to_connector_count = int(
        positive_df.get(
            "projects_to_synthetic_connector",
            pd.Series(False, index=positive_df.index),
        ).fillna(False).astype(bool).sum()
    )
    positive_projects_to_connector_fraction = float(
        positive_projects_to_connector_count / max(len(positive_df), 1)
    )

    if positive_projects_to_connector_count > 0:
        log(
            "[ROAD-WARN] "
            f"{positive_projects_to_connector_count}/{len(positive_df)} retained "
            "positive sites project most closely to a synthetic connector. "
            "Review the diagnostic road GeoPackage before interpreting chainage "
            "or spatial-block membership."
        )

    max_positive_backbone_distance = float(
        positive_df["distance_to_road_backbone_m"].max()
    )

    if (
        max_positive_backbone_distance
        > Config.ROAD_WARN_POSITIVE_DISTANCE_TO_BACKBONE_M
    ):
        log(
            "[ROAD-WARN] At least one retained positive site is more than "
            f"{Config.ROAD_WARN_POSITIVE_DISTANCE_TO_BACKBONE_M:.0f} m from "
            "the reconstructed highway backbone. Review the road-backbone GPKG "
            "and the reconciliation CSV."
        )

    # -------------------------------------------------------------------------
    # 4. Five contiguous longitudinal blocks
    # -------------------------------------------------------------------------
    positive_df, block_boundaries, block_boundary_df = (
        define_balanced_contiguous_blocks(
            positive_df,
            Config.N_SPATIAL_BLOCKS,
        )
    )

    positive_counts_by_block = (
        positive_df["spatial_block"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    log(
        f"[BLOCKS] positive counts by block={positive_counts_by_block}"
    )

    block_boundary_df.to_csv(
        Config.BLOCK_BOUNDARIES_CSV,
        index=False,
        float_format="%.8f",
    )

    positive_df[
        [
            "site_id",
            "chainage_m",
            "distance_to_road_backbone_m",
            "spatial_block",
        ]
    ].sort_values(
        ["spatial_block", "chainage_m"]
    ).to_csv(
        Config.BLOCK_ASSIGNMENTS_CSV,
        index=False,
        float_format="%.8f",
    )

    positive_df.to_csv(
        Config.RETAINED_POSITIVE_CSV,
        index=False,
        float_format="%.8f",
    )

    positive_with_predictors = extract_predictors_for_sites(
        positive_df,
        master_stack,
        band_names,
    )

    positive_gpkg = Config.OUT_DIR / "landslide_representative_sites.gpkg"

    write_points_gpkg(
        positive_gpkg,
        positive_df,
        raster_crs,
        "positive_sites",
    )

    # -------------------------------------------------------------------------
    # 5. Common reproducible negative priority
    # -------------------------------------------------------------------------
    common_flat_indices = np.flatnonzero(common_domain.ravel())

    rng = np.random.default_rng(Config.RANDOM_SEED)

    priority_flat_indices = common_flat_indices.copy()
    rng.shuffle(priority_flat_indices)

    landslide_geometries = [
        site["geometry"]
        for site in all_sites
        if site["geometry"] is not None
    ]

    # -------------------------------------------------------------------------
    # 6. 100 / 500 / 1000 m experiments
    # -------------------------------------------------------------------------
    buffer_metadata = {}

    for buffer_m in Config.BUFFER_SENSITIVITY_M:
        log("")
        log("=" * 84)
        log(f"[BUFFER] {buffer_m} m exclusion experiment")
        log("=" * 84)

        candidate_mask = build_negative_candidate_mask(
            common_domain=common_domain,
            landslide_geometries=landslide_geometries,
            buffer_m=float(buffer_m),
            raster_transform=raster_transform,
        )

        log(
            f"[BUFFER {buffer_m}] pseudo-absence candidate pixels="
            f"{int(candidate_mask.sum()):,}"
        )

        candidate_mask_path = (
            Config.OUT_DIR
            / (
                f"negative_candidates_buffer{int(buffer_m)}m_"
                f"32645_12p5m_byte.tif"
            )
        )

        if Config.WRITE_NEGATIVE_CANDIDATE_MASKS:
            write_mask_tif(
                candidate_mask_path,
                candidate_mask,
                raster_profile,
                (
                    f"negative_candidate_mask_buffer_{int(buffer_m)}m"
                ),
            )

        negative_df = sample_block_matched_negatives(
            candidate_mask=candidate_mask,
            priority_flat_indices=priority_flat_indices,
            positive_counts_by_block=positive_counts_by_block,
            road_chainage_model=road_chainage_model,
            boundaries=block_boundaries,
            raster_transform=raster_transform,
            raster_width=raster_width,
            buffer_m=int(buffer_m),
        )

        negative_with_predictors = extract_predictors_for_sites(
            negative_df,
            master_stack,
            band_names,
        )

        positive_buffer_df = positive_with_predictors.copy()
        positive_buffer_df["buffer_m"] = int(buffer_m)

        combined_df = pd.concat(
            [
                positive_buffer_df,
                negative_with_predictors,
            ],
            ignore_index=True,
            sort=False,
        )

        combined_df = combined_df.sort_values(
            [
                "spatial_block",
                "label",
                "chainage_m",
                "sample_id",
            ],
            ascending=[
                True,
                False,
                True,
                True,
            ],
        ).reset_index(drop=True)

        block_class_table = pd.crosstab(
            combined_df["spatial_block"],
            combined_df["label"],
        )

        for block_number in range(
            1,
            Config.N_SPATIAL_BLOCKS + 1,
        ):
            positive_count = int(
                block_class_table.loc[block_number, 1]
            )
            negative_count = int(
                block_class_table.loc[block_number, 0]
            )

            if positive_count != negative_count:
                raise RuntimeError(
                    f"1:1 block matching failed for buffer {buffer_m} m, "
                    f"block {block_number}: positive={positive_count}, "
                    f"negative={negative_count}"
                )

        n_positive = int((combined_df["label"] == 1).sum())
        n_negative = int((combined_df["label"] == 0).sum())

        if n_positive != n_negative:
            raise RuntimeError(
                f"Overall 1:1 sampling failed for {buffer_m} m: "
                f"positive={n_positive}, negative={n_negative}"
            )

        master_buffer_csv = (
            Config.OUT_DIR
            / (
                f"common_site_level_samples_A_to_F_"
                f"buffer{int(buffer_m)}m.csv"
            )
        )

        combined_df.to_csv(
            master_buffer_csv,
            index=False,
            float_format="%.8f",
        )

        if int(buffer_m) == int(Config.PRIMARY_BUFFER_M):
            combined_df.to_csv(
                Config.PRIMARY_MASTER_CSV,
                index=False,
                float_format="%.8f",
            )

        negative_csv = (
            Config.OUT_DIR
            / f"matched_negative_sites_buffer{int(buffer_m)}m.csv"
        )

        negative_with_predictors.to_csv(
            negative_csv,
            index=False,
            float_format="%.8f",
        )

        negative_gpkg = (
            Config.OUT_DIR
            / f"matched_negative_sites_buffer{int(buffer_m)}m.gpkg"
        )

        write_points_gpkg(
            negative_gpkg,
            negative_with_predictors,
            raster_crs,
            f"negative_{int(buffer_m)}m",
        )

        model_views = write_model_views(
            combined_df,
            scenario_manifest,
            int(buffer_m),
        )

        block_counts = {}

        for block in sorted(
            combined_df["spatial_block"].unique()
        ):
            block_counts[str(int(block))] = {
                "positive": int(
                    (
                        (
                            combined_df["spatial_block"] == block
                        )
                        & (combined_df["label"] == 1)
                    ).sum()
                ),
                "negative": int(
                    (
                        (
                            combined_df["spatial_block"] == block
                        )
                        & (combined_df["label"] == 0)
                    ).sum()
                ),
            }

        buffer_metadata[str(int(buffer_m))] = {
            "buffer_m": int(buffer_m),
            "is_primary": bool(
                int(buffer_m)
                == int(Config.PRIMARY_BUFFER_M)
            ),
            "candidate_pixels": int(candidate_mask.sum()),
            "n_positive_sites": n_positive,
            "n_negative_sites": n_negative,
            "total_rows": int(len(combined_df)),
            "block_counts": block_counts,
            "master_csv": str(master_buffer_csv),
            "negative_csv": str(negative_csv),
            "negative_candidate_mask": str(candidate_mask_path),
            "model_view_csvs": model_views,
        }

        log(
            f"[BUFFER {buffer_m}] positive={n_positive}, "
            f"negative={n_negative}, total={len(combined_df)}"
        )
        log(
            f"[BUFFER {buffer_m}] block counts={block_counts}"
        )
        log(f"[OUT] {master_buffer_csv}")

    # -------------------------------------------------------------------------
    # 7. Metadata
    # -------------------------------------------------------------------------
    metadata = {
        "step_name": "Step-6.3-Common-Site-Level-Sampling",
        "generated_at_utc": utc_now_iso(),
        "inventory": {
            "training_audit": train_audit,
            "validation_audit": valid_audit,
            "combined_retained_landslide_records": int(len(all_sites)),
            "retained_common_domain_positive_sites": int(len(positive_df)),
            "reconciliation_status_counts": status_counts,
            "historical_reference_total_sites": int(
                Config.EXPECTED_TOTAL_LANDSLIDE_SITES
            ),
        },
        "road_backbone": road_meta,
        "road_chainage_model": {
            "projection_rule": "nearest real ordered road component",
            "virtual_gaps_used_for_projection": False,
            "component_count": int(len(road_chainage_model.components)),
            "component_offsets_m": [float(v) for v in road_chainage_model.offsets_m],
            "virtual_gap_lengths_m": [float(v) for v in road_chainage_model.gap_lengths_m],
            "total_chainage_length_m": float(road_chainage_model.total_chainage_length_m),
            "component_chainage_csv": str(Config.ROAD_COMPONENT_CHAINAGE_CSV),
        },
        "sampling_domain": {
            "common_domain_path": str(
                Config.COMMON_SAMPLING_DOMAIN_TIF
            ),
            "common_domain_pixels": int(common_domain.sum()),
            "analysis_domain_pixels": int(analysis_domain.sum()),
            "dynamic_mask_used": False,
            "former_core_valid_mask_used": False,
        },
        "positive_sampling": {
            "unit": (
                "one retained source landslide feature equals one positive "
                "site-level observation"
            ),
            "representative_rule": (
                "common-domain raster cell nearest polygon centroid"
            ),
            "all_touched_fallback": bool(
                Config.ALL_TOUCHED_FALLBACK
            ),
            "positive_sites_fixed_across_buffers": True,
        },
        "spatial_blocks": {
            "n_blocks": int(Config.N_SPATIAL_BLOCKS),
            "definition": (
                "five contiguous reconstructed-highway-chainage blocks with "
                "approximately equal retained positive-site counts"
            ),
            "chainage_boundaries_m": [
                float(value)
                for value in block_boundaries
            ],
            "positive_counts_by_block": {
                str(int(block)): int(count)
                for block, count in positive_counts_by_block.items()
            },
            "maximum_positive_distance_to_backbone_m": (
                max_positive_backbone_distance
            ),
            "positive_sites_projecting_to_synthetic_connector_count": int(
                positive_projects_to_connector_count
            ),
            "positive_sites_projecting_to_synthetic_connector_fraction": float(
                positive_projects_to_connector_fraction
            ),
        },
        "negative_sampling": {
            "presence_absence_ratio": 1.0,
            "primary_buffer_m": int(
                Config.PRIMARY_BUFFER_M
            ),
            "buffer_sensitivity_m": [
                int(value)
                for value in Config.BUFFER_SENSITIVITY_M
            ],
            "block_matched": True,
            "same_random_priority_across_buffers": True,
            "random_seed": int(Config.RANDOM_SEED),
        },
        "buffer_experiments": buffer_metadata,
        "predictors": {
            "master_stack": str(Config.MASTER_STACK_TIF),
            "n_candidate_predictors": int(len(band_names)),
            "band_names": list(band_names),
            "scenario_manifest": str(
                Config.SCENARIO_MANIFEST_JSON
            ),
        },
        "primary_dataset": str(Config.PRIMARY_MASTER_CSV),
    }

    with open(
        Config.METADATA_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            to_json_safe(metadata),
            handle,
            indent=2,
            ensure_ascii=False,
        )

    # -------------------------------------------------------------------------
    # 8. Human-readable QA report
    # -------------------------------------------------------------------------
    with open(
        Config.QA_REPORT_TXT,
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write("=" * 100 + "\n")
        handle.write(
            "STEP 6.3 — COMMON SITE-LEVEL SAMPLING QA REPORT\n"
        )
        handle.write("=" * 100 + "\n\n")

        handle.write(f"Generated UTC: {utc_now_iso()}\n\n")

        handle.write("INVENTORY\n")
        handle.write("-" * 72 + "\n")
        handle.write(
            f"Training source features: "
            f"{train_audit['source_features_total']}\n"
        )
        handle.write(
            f"Training retained landslide records: {len(train_sites)}\n"
        )
        handle.write(
            f"Validation source features: "
            f"{valid_audit['source_features_total']}\n"
        )
        handle.write(
            f"Validation retained landslide records: {len(valid_sites)}\n"
        )
        handle.write(
            f"Combined retained source landslide records: {len(all_sites)}\n"
        )
        handle.write(
            f"Retained common-domain positive sites: {len(positive_df)}\n"
        )
        handle.write(
            f"Reconciliation status counts: {status_counts}\n\n"
        )

        handle.write("ROAD BACKBONE\n")
        handle.write("-" * 72 + "\n")
        handle.write(
            f"Backbone source mode: "
            f"{road_meta.get('backbone_source_mode', 'unknown')}\n"
        )
        handle.write(
            f"Sampling gate passed: "
            f"{road_meta.get('sampling_gate_passed', False)}\n"
        )
        handle.write(
            f"Sampling gate reason: "
            f"{road_meta.get('sampling_gate_reason', '')}\n"
        )
        handle.write(
            f"Merged road components: "
            f"{road_meta['merged_component_count']}\n"
        )
        handle.write(
            f"Largest single component: "
            f"{road_meta['largest_merged_component_percent']:.2f}% "
            f"of original line length\n"
        )
        handle.write(
            f"Backbone components retained: "
            f"{road_meta['backbone_component_count']}\n"
        )
        handle.write(
            f"Retained original road length: "
            f"{road_meta['retained_original_length_percent']:.2f}%\n"
        )
        handle.write(
            f"Synthetic connector count: "
            f"{road_meta['synthetic_connector_count']}\n"
        )
        handle.write(
            f"Maximum synthetic gap: "
            f"{road_meta['max_synthetic_gap_m']:.2f} m\n"
        )
        handle.write(
            f"Final chainage-backbone length: "
            f"{road_meta['final_backbone_length_m'] / 1000.0:.3f} km\n"
        )
        handle.write(
            f"Positive sites projecting to synthetic connectors: "
            f"{positive_projects_to_connector_count}/{len(positive_df)} "
            f"({100.0 * positive_projects_to_connector_fraction:.2f}%)\n\n"
        )

        handle.write("SPATIAL BLOCKS\n")
        handle.write("-" * 72 + "\n")
        handle.write(
            f"Positive counts by block: {positive_counts_by_block}\n"
        )
        handle.write(
            f"Chainage boundaries (m): {block_boundaries}\n"
        )
        handle.write(
            f"Maximum retained-positive distance to backbone: "
            f"{max_positive_backbone_distance:.2f} m\n\n"
        )

        handle.write("BUFFER EXPERIMENTS\n")
        handle.write("-" * 72 + "\n")

        for key, details in buffer_metadata.items():
            handle.write(
                f"{key} m: positive={details['n_positive_sites']}, "
                f"negative={details['n_negative_sites']}, "
                f"total={details['total_rows']}, "
                f"candidate pixels={details['candidate_pixels']}\n"
            )
            handle.write(
                f"  block counts={details['block_counts']}\n"
            )

        handle.write("\nPRIMARY ANALYSIS\n")
        handle.write("-" * 72 + "\n")
        handle.write(
            f"Primary exclusion buffer: {Config.PRIMARY_BUFFER_M} m\n"
        )
        handle.write(
            f"Primary dataset: {Config.PRIMARY_MASTER_CSV}\n"
        )
        handle.write(
            "All Models A–F use identical sample rows within each buffer "
            "experiment; only predictor columns differ.\n"
        )

    log(f"[OUT] Inventory audit -> {Config.INVENTORY_AUDIT_CSV}")
    log(f"[OUT] Road QA -> {Config.ROAD_QA_JSON}")
    log(f"[OUT] Road backbone -> {Config.ROAD_BACKBONE_GPKG}")
    log(f"[OUT] Road component chainage -> {Config.ROAD_COMPONENT_CHAINAGE_CSV}")
    log(f"[OUT] Reconciliation -> {Config.RECONCILIATION_CSV}")
    log(f"[OUT] Primary dataset -> {Config.PRIMARY_MASTER_CSV}")
    log(f"[OUT] Metadata -> {Config.METADATA_JSON}")
    log(f"[OUT] QA report -> {Config.QA_REPORT_TXT}")
    log("=== Step 6.3 completed successfully ===")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 6.3 failed: {exc}")
        traceback.print_exc()
        raise


# Step 6.4 — Model-Ready Dataset Validation and Spatial-CV Setup

In [ ]:
# =============================================================================
# Step 6.4 — Model-Ready Dataset Validation and Spatial-CV Setup
# =============================================================================
#
# Final QA/CV-freezing stage before ML modelling.
#
# Validates:
#   - the primary 100 m site-level dataset;
#   - the 100/500/1000 m buffer datasets;
#   - all 17 candidate predictors;
#   - the six A–F model configurations;
#   - five contiguous spatial blocks and 1:1 class balance;
#   - identical positive sites and block assignments across buffer experiments.
#
# Freezes:
#   - one authoritative model-ready dataset;
#   - A–F predictor manifests;
#   - sample-to-fold assignments;
#   - five outer spatial-CV folds;
#   - fold-safe preprocessing rules.
#
# No model is trained here. No predictor is globally standardized or encoded.
# =============================================================================

from __future__ import annotations

import json
import os
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple

import fiona
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import CRS, Transformer
from shapely.geometry import GeometryCollection, LineString, MultiLineString, MultiPolygon, Polygon, shape
from shapely.ops import transform as shp_transform


class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    STACK_MANIFEST_JSON = STEP61_DIR / "stack_manifest.json"
    SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"

    STEP63_DIR = RESULTS_DIR / "Step-6.3-Site_Level_Sampling"
    PRIMARY_DATASET_CSV = STEP63_DIR / "common_site_level_samples_A_to_F.csv"

    BUFFER_DATASETS = {
        100: STEP63_DIR / "common_site_level_samples_A_to_F_buffer100m.csv",
        500: STEP63_DIR / "common_site_level_samples_A_to_F_buffer500m.csv",
        1000: STEP63_DIR / "common_site_level_samples_A_to_F_buffer1000m.csv",
    }

    ROAD_QA_JSON = STEP63_DIR / "NTD_Road_backbone_QA.json"
    ROAD_BACKBONE_GPKG = STEP63_DIR / "NTD_Road_chainage_backbone.gpkg"
    STUDY_AREA_SHP = BASE / "Study_Area.shp"

    OUT_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    MODEL_READY_DATASET_CSV = OUT_DIR / "model_ready_dataset_A_to_F.csv"
    MODEL_CONFIG_CSV = OUT_DIR / "model_configuration_manifest.csv"
    MODEL_CONFIG_JSON = OUT_DIR / "model_configuration_manifest.json"
    SAMPLE_FOLD_ASSIGNMENTS_CSV = OUT_DIR / "spatial_cv_sample_assignments.csv"
    FOLD_MANIFEST_CSV = OUT_DIR / "spatial_cv_fold_manifest.csv"
    PREPROCESSING_MANIFEST_JSON = OUT_DIR / "model_preprocessing_manifest.json"
    BUFFER_QA_CSV = OUT_DIR / "buffer_dataset_consistency_QA.csv"
    FINAL_MANIFEST_JSON = OUT_DIR / "Step_6_4_model_ready_manifest.json"
    QA_REPORT_TXT = OUT_DIR / "Step_6_4_model_ready_QA_report.txt"
    BLOCK_QA_MAP_PNG = OUT_DIR / "Spatial_CV_Block_QA_Map.png"
    BLOCK_QA_MAP_PDF = OUT_DIR / "Spatial_CV_Block_QA_Map.pdf"

    STATIC12 = [
        "slope_deg",
        "northness",
        "eastness",
        "profile_curv",
        "plan_curv",
        "tpi_s5",
        "spi_ln",
        "dist_river_m",
        "dist_road_m",
        "aap_mm",
        "ndvi",
        "landcover",
    ]

    MASTER17 = STATIC12 + [
        "dvplus_mm_yr",
        "hkfi_0_1",
        "sndr_0_1",
        "twi_raw",
        "sm_support_physical",
    ]

    EXPECTED_SCENARIOS = {
        "A_static_baseline": STATIC12,
        "B_raw_deformation": STATIC12 + ["dvplus_mm_yr"],
        "C_hkfi_only": STATIC12 + ["hkfi_0_1"],
        "D_sndr_only": STATIC12 + ["sndr_0_1"],
        "E_physics_guided": STATIC12 + ["hkfi_0_1", "sndr_0_1"],
        "F_untransformed_sources": STATIC12 + [
            "dvplus_mm_yr",
            "twi_raw",
            "sm_support_physical",
        ],
    }

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    EXPECTED_PREDICTOR_COUNTS = {
        "A_static_baseline": 12,
        "B_raw_deformation": 13,
        "C_hkfi_only": 13,
        "D_sndr_only": 13,
        "E_physics_guided": 14,
        "F_untransformed_sources": 15,
    }

    REQUIRED_METADATA_COLUMNS = [
        "sample_id",
        "site_id",
        "label",
        "class_name",
        "row",
        "col",
        "x",
        "y",
        "chainage_m",
        "distance_to_road_backbone_m",
        "spatial_block",
        "buffer_m",
    ]

    CATEGORICAL_FEATURES = ["landcover"]
    N_SPATIAL_BLOCKS = 5
    PRIMARY_BUFFER_M = 100

    EXPECTED_PRIMARY_POSITIVE_COUNT_QA = 165
    WARN_IF_ROAD_SYNTHETIC_GAP_EXCEEDS_M = 5000.0

    MAP_FIGSIZE = (8, 8)
    MAP_DPI = 600


def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json(path: Path) -> Dict[str, Any]:
    require_file(path, "JSON file")
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def to_json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): to_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_safe(item) for item in value]
    return value


def load_dataset(path: Path, label: str) -> pd.DataFrame:
    require_file(path, label)
    dataframe = pd.read_csv(path)
    if dataframe.empty:
        raise RuntimeError(f"{label} is empty: {path}")
    return dataframe


def canonicalize_for_comparison(
    dataframe: pd.DataFrame,
    columns: Sequence[str],
) -> pd.DataFrame:
    return (
        dataframe[list(columns)]
        .copy()
        .sort_values("sample_id")
        .reset_index(drop=True)
    )


def frames_equal(
    left: pd.DataFrame,
    right: pd.DataFrame,
    columns: Sequence[str],
) -> bool:
    left_c = canonicalize_for_comparison(left, columns)
    right_c = canonicalize_for_comparison(right, columns)

    if left_c.shape != right_c.shape:
        return False

    return left_c.equals(right_c)


def validate_required_columns(
    dataframe: pd.DataFrame,
    dataset_label: str,
) -> None:
    required = Config.REQUIRED_METADATA_COLUMNS + Config.MASTER17

    missing = [
        column
        for column in required
        if column not in dataframe.columns
    ]

    if missing:
        raise RuntimeError(
            f"{dataset_label} is missing required columns: {missing}"
        )

    irs_columns = [
        column
        for column in dataframe.columns
        if "irs" in str(column).lower()
    ]

    if irs_columns:
        raise RuntimeError(
            f"{dataset_label} contains IRS-related columns: {irs_columns}. "
            "IRS must not be used as an ML predictor."
        )

    if "tpi_s25" in dataframe.columns:
        raise RuntimeError(
            f"{dataset_label} contains tpi_s25. Only tpi_s5 is retained."
        )


def validate_dataset_integrity(
    dataframe: pd.DataFrame,
    dataset_label: str,
) -> Dict[str, Any]:
    validate_required_columns(dataframe, dataset_label)

    if dataframe["sample_id"].isna().any():
        raise RuntimeError(f"{dataset_label} contains missing sample_id values.")

    duplicate_ids = dataframe["sample_id"].duplicated(keep=False)

    if duplicate_ids.any():
        raise RuntimeError(
            f"{dataset_label} contains duplicated sample_id values: "
            f"{dataframe.loc[duplicate_ids, 'sample_id'].tolist()[:20]}"
        )

    duplicated_cells = dataframe.duplicated(
        subset=["row", "col"],
        keep=False,
    )

    if duplicated_cells.any():
        raise RuntimeError(
            f"{dataset_label} contains duplicated raster cells:\n"
            + dataframe.loc[
                duplicated_cells,
                ["sample_id", "label", "row", "col"],
            ].head(20).to_string(index=False)
        )

    labels = set(
        dataframe["label"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    if labels != {0, 1}:
        raise RuntimeError(
            f"{dataset_label} must contain exactly labels 0 and 1. Found {labels}."
        )

    blocks = sorted(
        dataframe["spatial_block"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    expected_blocks = list(range(1, Config.N_SPATIAL_BLOCKS + 1))

    if blocks != expected_blocks:
        raise RuntimeError(
            f"{dataset_label} must contain blocks {expected_blocks}. "
            f"Found {blocks}."
        )

    predictors = dataframe[Config.MASTER17].apply(
        pd.to_numeric,
        errors="coerce",
    )

    values = predictors.to_numpy(dtype="float64")

    if not np.isfinite(values).all():
        bad_rows, bad_columns = np.where(~np.isfinite(values))

        examples = [
            (
                str(dataframe.iloc[row_index]["sample_id"]),
                Config.MASTER17[column_index],
            )
            for row_index, column_index in zip(
                bad_rows[:20],
                bad_columns[:20],
            )
        ]

        raise RuntimeError(
            f"{dataset_label} contains missing/non-finite predictors. "
            f"Examples: {examples}"
        )

    class_counts = dataframe["label"].astype(int).value_counts()

    n_positive = int(class_counts.get(1, 0))
    n_negative = int(class_counts.get(0, 0))

    if n_positive != n_negative:
        raise RuntimeError(
            f"{dataset_label} is not 1:1 balanced: "
            f"positive={n_positive}, negative={n_negative}"
        )

    block_class = pd.crosstab(
        dataframe["spatial_block"].astype(int),
        dataframe["label"].astype(int),
    )

    for block in expected_blocks:
        if block not in block_class.index or 0 not in block_class.columns or 1 not in block_class.columns:
            raise RuntimeError(
                f"{dataset_label} is missing one class in block {block}."
            )

        positive_count = int(block_class.loc[block, 1])
        negative_count = int(block_class.loc[block, 0])

        if positive_count != negative_count:
            raise RuntimeError(
                f"{dataset_label} is not 1:1 balanced in block {block}: "
                f"positive={positive_count}, negative={negative_count}"
            )

    positive_sites = dataframe.loc[
        dataframe["label"].astype(int) == 1,
        "site_id",
    ]

    if positive_sites.duplicated().any():
        raise RuntimeError(
            f"{dataset_label} contains duplicated positive site IDs."
        )

    return {
        "dataset": dataset_label,
        "n_rows": int(len(dataframe)),
        "n_positive": n_positive,
        "n_negative": n_negative,
        "n_unique_positive_sites": int(positive_sites.nunique()),
        "n_unique_sample_ids": int(dataframe["sample_id"].nunique()),
        "n_blocks": int(len(blocks)),
        "block_class_counts": {
            str(int(block)): {
                "negative": int(block_class.loc[block, 0]),
                "positive": int(block_class.loc[block, 1]),
            }
            for block in block_class.index
        },
    }


def validate_and_write_model_configurations(
    scenario_manifest: Dict[str, Any],
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    scenarios = scenario_manifest.get("scenarios", {})

    rows = []
    json_scenarios = {}

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in scenarios:
            raise RuntimeError(
                f"Missing scenario in Step 6.1 manifest: {scenario_name}"
            )

        found = list(scenarios[scenario_name].get("features", []))
        expected = list(Config.EXPECTED_SCENARIOS[scenario_name])

        if found != expected:
            raise RuntimeError(
                f"Scenario mismatch for {scenario_name}.\n"
                f"Expected: {expected}\n"
                f"Found:    {found}"
            )

        expected_count = Config.EXPECTED_PREDICTOR_COUNTS[scenario_name]

        if len(found) != expected_count:
            raise RuntimeError(
                f"{scenario_name} should contain {expected_count} predictors, "
                f"found {len(found)}."
            )

        additional = [
            feature
            for feature in found
            if feature not in Config.STATIC12
        ]

        rows.append(
            {
                "scenario": scenario_name,
                "label": scenarios[scenario_name].get("label", scenario_name),
                "n_predictors": len(found),
                "predictors": "|".join(found),
                "additional_predictors_beyond_static12": "|".join(additional),
                "same_rows_required": True,
                "same_labels_required": True,
                "same_spatial_folds_required": True,
            }
        )

        json_scenarios[scenario_name] = {
            "label": scenarios[scenario_name].get("label", scenario_name),
            "n_predictors": len(found),
            "features": found,
            "additional_predictors_beyond_static12": additional,
        }

    dataframe = pd.DataFrame(rows)

    manifest = {
        "generated_at_utc": utc_now_iso(),
        "static12": Config.STATIC12,
        "master_candidate_features": Config.MASTER17,
        "n_master_candidate_features": len(Config.MASTER17),
        "irs_as_predictor": False,
        "tpi_s25_included": False,
        "retained_tpi": "tpi_s5",
        "scenarios": json_scenarios,
    }

    dataframe.to_csv(
        Config.MODEL_CONFIG_CSV,
        index=False,
    )

    with open(
        Config.MODEL_CONFIG_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(manifest, handle, indent=2)

    return dataframe, manifest


def create_spatial_cv_manifests(
    model_ready_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    assignments = model_ready_df[
        [
            "sample_id",
            "site_id",
            "label",
            "class_name",
            "x",
            "y",
            "chainage_m",
            "spatial_block",
        ]
    ].copy()

    assignments["cv_fold"] = assignments["spatial_block"].astype(int)
    assignments["held_out_in_fold"] = assignments["cv_fold"]

    assignments = assignments.sort_values(
        ["cv_fold", "label", "chainage_m", "sample_id"],
        ascending=[True, False, True, True],
    ).reset_index(drop=True)

    assignments.to_csv(
        Config.SAMPLE_FOLD_ASSIGNMENTS_CSV,
        index=False,
        float_format="%.8f",
    )

    fold_rows = []
    all_blocks = list(range(1, Config.N_SPATIAL_BLOCKS + 1))

    for fold_id in all_blocks:
        held_out_block = fold_id
        training_blocks = [
            block
            for block in all_blocks
            if block != held_out_block
        ]

        train_df = model_ready_df.loc[
            model_ready_df["spatial_block"].astype(int) != held_out_block
        ]

        test_df = model_ready_df.loc[
            model_ready_df["spatial_block"].astype(int) == held_out_block
        ]

        train_counts = train_df["label"].astype(int).value_counts()
        test_counts = test_df["label"].astype(int).value_counts()

        fold_rows.append(
            {
                "cv_fold": fold_id,
                "held_out_block": held_out_block,
                "training_blocks": "|".join(map(str, training_blocks)),
                "n_train": len(train_df),
                "n_train_positive": int(train_counts.get(1, 0)),
                "n_train_negative": int(train_counts.get(0, 0)),
                "n_test": len(test_df),
                "n_test_positive": int(test_counts.get(1, 0)),
                "n_test_negative": int(test_counts.get(0, 0)),
            }
        )

    fold_df = pd.DataFrame(fold_rows)

    fold_df.to_csv(
        Config.FOLD_MANIFEST_CSV,
        index=False,
    )

    return assignments, fold_df


def write_preprocessing_manifest() -> Dict[str, Any]:
    continuous_features = [
        feature
        for feature in Config.MASTER17
        if feature not in Config.CATEGORICAL_FEATURES
    ]

    manifest = {
        "generated_at_utc": utc_now_iso(),
        "global_rules": {
            "fit_preprocessing_on_full_dataset": False,
            "fit_preprocessing_inside_each_outer_training_fold": True,
            "global_standardization": False,
            "global_one_hot_encoding": False,
            "imputation_in_step_6_4": False,
            "categorical_features": Config.CATEGORICAL_FEATURES,
            "continuous_features": continuous_features,
        },
        "algorithms": {
            "XGBoost": {
                "continuous_scaling": "none",
                "landcover_handling": (
                    "one-hot encode inside each training fold; fit encoder on "
                    "training data and apply to held-out data"
                ),
                "unknown_category_handling": "ignore",
                "hyperparameter_tuning": (
                    "training data only; outer held-out block must remain unseen"
                ),
            },
            "Random_Forest": {
                "continuous_scaling": "none",
                "landcover_handling": (
                    "one-hot encode inside each training fold to avoid ordinal "
                    "interpretation of nominal land-use codes"
                ),
                "unknown_category_handling": "ignore",
                "hyperparameter_tuning": (
                    "training data only; outer held-out block must remain unseen"
                ),
            },
            "Logistic_Regression": {
                "continuous_scaling": (
                    "StandardScaler fitted on continuous predictors using "
                    "outer-training data only"
                ),
                "landcover_handling": (
                    "one-hot encode inside each training fold with one reference "
                    "category removed"
                ),
                "unknown_category_handling": "ignore",
                "hyperparameter_tuning": (
                    "training data only; outer held-out block must remain unseen"
                ),
            },
        },
    }

    with open(
        Config.PREPROCESSING_MANIFEST_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(manifest, handle, indent=2)

    return manifest


def compare_buffer_datasets(
    datasets: Dict[int, pd.DataFrame],
) -> pd.DataFrame:
    primary = datasets[Config.PRIMARY_BUFFER_M]

    primary_positive = primary.loc[
        primary["label"].astype(int) == 1
    ].copy()

    primary_positive_ids = set(
        primary_positive["site_id"].astype(str)
    )

    primary_block_map = (
        primary_positive
        .set_index("site_id")["spatial_block"]
        .astype(int)
        .to_dict()
    )

    positive_compare_columns = [
        "sample_id",
        "site_id",
        "row",
        "col",
        "x",
        "y",
        "chainage_m",
        "spatial_block",
    ] + Config.MASTER17

    rows = []

    for buffer_m in sorted(datasets):
        dataframe = datasets[buffer_m]

        integrity = validate_dataset_integrity(
            dataframe,
            f"buffer {buffer_m} m dataset",
        )

        positive_df = dataframe.loc[
            dataframe["label"].astype(int) == 1
        ].copy()

        negative_df = dataframe.loc[
            dataframe["label"].astype(int) == 0
        ].copy()

        same_positive_ids = (
            set(positive_df["site_id"].astype(str))
            == primary_positive_ids
        )

        same_blocks = (
            positive_df
            .set_index("site_id")["spatial_block"]
            .astype(int)
            .to_dict()
            == primary_block_map
        )

        same_positive_values = frames_equal(
            primary_positive,
            positive_df,
            positive_compare_columns,
        )

        duplicate_negative_cells = int(
            negative_df.duplicated(
                subset=["row", "col"]
            ).sum()
        )

        status = (
            "PASS"
            if (
                same_positive_ids
                and same_blocks
                and same_positive_values
                and duplicate_negative_cells == 0
            )
            else "FAIL"
        )

        rows.append(
            {
                "buffer_m": buffer_m,
                "is_primary": buffer_m == Config.PRIMARY_BUFFER_M,
                "n_rows": integrity["n_rows"],
                "n_positive": integrity["n_positive"],
                "n_negative": integrity["n_negative"],
                "same_positive_site_ids_as_100m": same_positive_ids,
                "same_positive_block_assignments_as_100m": same_blocks,
                "same_positive_locations_and_predictors_as_100m": (
                    same_positive_values
                ),
                "duplicate_negative_cells": duplicate_negative_cells,
                "block_class_counts": json.dumps(
                    integrity["block_class_counts"],
                    sort_keys=True,
                ),
                "status": status,
            }
        )

    qa_df = pd.DataFrame(rows)

    if (qa_df["status"] != "PASS").any():
        raise RuntimeError(
            "Buffer-dataset consistency QA failed:\n"
            + qa_df.loc[
                qa_df["status"] != "PASS"
            ].to_string(index=False)
        )

    qa_df.to_csv(
        Config.BUFFER_QA_CSV,
        index=False,
    )

    return qa_df


def extract_line_geometries(geometry) -> List[LineString]:
    if geometry is None or geometry.is_empty:
        return []
    if isinstance(geometry, LineString):
        return [geometry]
    if isinstance(geometry, MultiLineString):
        return list(geometry.geoms)
    if isinstance(geometry, GeometryCollection):
        output = []
        for child in geometry.geoms:
            output.extend(extract_line_geometries(child))
        return output
    return []


def extract_polygon_geometries(geometry) -> List[Polygon]:
    if geometry is None or geometry.is_empty:
        return []
    if isinstance(geometry, Polygon):
        return [geometry]
    if isinstance(geometry, MultiPolygon):
        return list(geometry.geoms)
    if isinstance(geometry, GeometryCollection):
        output = []
        for child in geometry.geoms:
            output.extend(extract_polygon_geometries(child))
        return output
    return []


def read_vector_geometries(
    path: Path,
    target_crs: CRS,
) -> List[Any]:
    require_file(path, "QA-map vector input")

    geometries = []

    with fiona.open(path) as source:
        if getattr(source, "crs_wkt", None):
            source_crs = CRS.from_user_input(source.crs_wkt)
        elif getattr(source, "crs", None):
            source_crs = CRS.from_user_input(source.crs)
        else:
            raise RuntimeError(f"Could not determine CRS for {path}")

        if source_crs == target_crs:
            reproj = lambda geometry: geometry
        else:
            transformer = Transformer.from_crs(
                source_crs,
                target_crs,
                always_xy=True,
            )
            reproj = lambda geometry: shp_transform(
                transformer.transform,
                geometry,
            )

        for feature in source:
            geometries.append(
                reproj(shape(feature["geometry"]))
            )

    return geometries


def create_spatial_cv_block_qa_map(
    primary_df: pd.DataFrame,
    road_qa: Dict[str, Any],
) -> None:
    target_crs = CRS.from_epsg(32645)

    study_geometries = read_vector_geometries(
        Config.STUDY_AREA_SHP,
        target_crs,
    )

    road_geometries = read_vector_geometries(
        Config.ROAD_BACKBONE_GPKG,
        target_crs,
    )

    figure, axis = plt.subplots(
        figsize=Config.MAP_FIGSIZE,
        dpi=Config.MAP_DPI,
    )

    for geometry in study_geometries:
        for polygon in extract_polygon_geometries(geometry):
            x, y = polygon.exterior.xy
            axis.plot(
                x,
                y,
                linewidth=0.7,
                alpha=0.8,
            )

    for geometry in road_geometries:
        for line in extract_line_geometries(geometry):
            x, y = line.xy
            axis.plot(
                x,
                y,
                linewidth=1.0,
                alpha=0.9,
            )

    positive_df = primary_df.loc[
        primary_df["label"].astype(int) == 1
    ]

    negative_df = primary_df.loc[
        primary_df["label"].astype(int) == 0
    ]

    positive_scatter = axis.scatter(
        positive_df["x"],
        positive_df["y"],
        c=positive_df["spatial_block"],
        cmap="tab10",
        s=18,
        marker="o",
        label="Landslide sites",
        alpha=0.9,
    )

    axis.scatter(
        negative_df["x"],
        negative_df["y"],
        c=negative_df["spatial_block"],
        cmap="tab10",
        s=15,
        marker="x",
        label="100 m pseudo-absence sites",
        alpha=0.8,
    )

    colorbar = figure.colorbar(
        positive_scatter,
        ax=axis,
        fraction=0.035,
        pad=0.02,
    )

    colorbar.set_label("Spatial block")

    max_gap_m = float(
        road_qa.get(
            "max_synthetic_gap_m",
            np.nan,
        )
    )

    retained_percent = float(
        road_qa.get(
            "retained_original_length_percent",
            np.nan,
        )
    )

    title = (
        "Five-block spatial-CV QA map\n"
        f"Primary 100 m dataset: "
        f"{int((primary_df['label'] == 1).sum())} positive + "
        f"{int((primary_df['label'] == 0).sum())} negative sites"
    )

    if np.isfinite(max_gap_m):
        title += (
            f"\nRoad-backbone QA: max synthetic gap "
            f"{max_gap_m / 1000.0:.2f} km"
        )

    if np.isfinite(retained_percent):
        title += (
            f"; retained road linework "
            f"{retained_percent:.1f}%"
        )

    axis.set_title(title, fontsize=10)
    axis.set_xlabel("Easting (m)")
    axis.set_ylabel("Northing (m)")
    axis.set_aspect("equal", adjustable="box")
    axis.legend(loc="best", fontsize=8)
    axis.grid(True, alpha=0.2)

    figure.tight_layout()

    figure.savefig(
        Config.BLOCK_QA_MAP_PNG,
        dpi=Config.MAP_DPI,
        bbox_inches="tight",
    )

    figure.savefig(
        Config.BLOCK_QA_MAP_PDF,
        bbox_inches="tight",
    )

    plt.close(figure)


def main() -> None:
    log(
        "=== Step 6.4: Model-Ready Dataset Validation and Spatial-CV Setup ==="
    )

    # 1. Load manifests and datasets
    scenario_manifest = load_json(
        Config.SCENARIO_MANIFEST_JSON
    )

    road_qa = load_json(
        Config.ROAD_QA_JSON
    )

    primary_df = load_dataset(
        Config.PRIMARY_DATASET_CSV,
        "primary 100 m model dataset",
    )

    datasets = {
        buffer_m: load_dataset(
            path,
            f"{buffer_m} m buffer dataset",
        )
        for buffer_m, path in Config.BUFFER_DATASETS.items()
    }

    # 2. Verify canonical primary alias = explicit 100 m dataset
    canonical_columns = [
        "sample_id",
        "site_id",
        "label",
        "row",
        "col",
        "spatial_block",
    ] + Config.MASTER17

    if not frames_equal(
        primary_df,
        datasets[100],
        canonical_columns,
    ):
        raise RuntimeError(
            "The canonical primary dataset does not exactly match the "
            "explicit 100 m dataset."
        )

    log(
        "[QA] Canonical primary dataset exactly matches buffer100m dataset: PASS"
    )

    # 3. Primary integrity
    primary_integrity = validate_dataset_integrity(
        primary_df,
        "primary 100 m dataset",
    )

    log(
        f"[PRIMARY] rows={primary_integrity['n_rows']}, "
        f"positive={primary_integrity['n_positive']}, "
        f"negative={primary_integrity['n_negative']}"
    )

    log(
        f"[PRIMARY] block counts="
        f"{primary_integrity['block_class_counts']}"
    )

    if (
        primary_integrity["n_positive"]
        != Config.EXPECTED_PRIMARY_POSITIVE_COUNT_QA
    ):
        log(
            "[WARN] Primary positive count differs from the current QA "
            f"reference of {Config.EXPECTED_PRIMARY_POSITIVE_COUNT_QA}. "
            "The actual validated count will be used."
        )

    # 4. Freeze A–F model configurations
    model_config_df, model_config_json = (
        validate_and_write_model_configurations(
            scenario_manifest
        )
    )

    log(
        "[QA] Models A–F predictor definitions: PASS"
    )

    for _, row in model_config_df.iterrows():
        log(
            f"[MODEL] {row['scenario']}: "
            f"{int(row['n_predictors'])} predictors"
        )

    # 5. Create authoritative model-ready dataset
    model_ready_df = primary_df.copy()
    model_ready_df["cv_fold"] = (
        model_ready_df["spatial_block"].astype(int)
    )

    metadata_columns = [
        column
        for column in [
            "sample_id",
            "site_id",
            "label",
            "class_name",
            "inventory_source",
            "source_feature_index",
            "source_original_id",
            "row",
            "col",
            "x",
            "y",
            "chainage_m",
            "distance_to_road_backbone_m",
            "spatial_block",
            "cv_fold",
            "buffer_m",
            "representative_method",
            "negative_priority_rank",
        ]
        if column in model_ready_df.columns
    ]

    other_columns = [
        column
        for column in model_ready_df.columns
        if (
            column not in metadata_columns
            and column not in Config.MASTER17
        )
    ]

    model_ready_df = model_ready_df[
        metadata_columns
        + other_columns
        + Config.MASTER17
    ]

    model_ready_df.to_csv(
        Config.MODEL_READY_DATASET_CSV,
        index=False,
        float_format="%.8f",
    )

    # 6. Freeze five outer spatial folds
    assignments_df, fold_manifest_df = (
        create_spatial_cv_manifests(
            model_ready_df
        )
    )

    log(
        "[CV] Five outer spatial folds frozen successfully."
    )

    for _, row in fold_manifest_df.iterrows():
        log(
            f"[CV] Fold {int(row['cv_fold'])}: "
            f"train={int(row['n_train'])}, "
            f"test={int(row['n_test'])}, "
            f"held-out block={int(row['held_out_block'])}"
        )

    # 7. Freeze fold-safe preprocessing rules
    preprocessing_manifest = write_preprocessing_manifest()

    log(
        "[PREPROCESSING] Fold-safe preprocessing manifest written."
    )

    # 8. Validate 100/500/1000 m sensitivity datasets
    buffer_qa_df = compare_buffer_datasets(
        datasets
    )

    log(
        "[QA] Buffer-sensitivity datasets are internally consistent: PASS"
    )

    for _, row in buffer_qa_df.iterrows():
        log(
            f"[BUFFER-QA] {int(row['buffer_m'])} m: "
            f"positive={int(row['n_positive'])}, "
            f"negative={int(row['n_negative'])}, "
            f"status={row['status']}"
        )

    # 9. Road-backbone warning
    max_gap_m = float(
        road_qa.get(
            "max_synthetic_gap_m",
            np.nan,
        )
    )

    road_warning = None

    if (
        np.isfinite(max_gap_m)
        and max_gap_m
        > Config.WARN_IF_ROAD_SYNTHETIC_GAP_EXCEEDS_M
    ):
        road_warning = (
            "The reconstructed NTD road backbone contains a large synthetic "
            f"connector ({max_gap_m / 1000.0:.2f} km). Visually inspect the "
            "spatial-block QA map before treating downstream ML results as final."
        )

        log(
            f"[ROAD-WARN] {road_warning}"
        )

    # 10. Spatial CV QA map
    create_spatial_cv_block_qa_map(
        model_ready_df,
        road_qa,
    )

    log(
        f"[OUT] Spatial CV QA map -> {Config.BLOCK_QA_MAP_PNG}"
    )

    # 11. Final manifest
    final_manifest = {
        "step_name": (
            "Step-6.4-Model-Ready-Dataset-Validation-and-Spatial-CV-Setup"
        ),
        "generated_at_utc": utc_now_iso(),
        "status": (
            "PASS_WITH_ROAD_QA_WARNING"
            if road_warning
            else "PASS"
        ),
        "primary_sampling_design": {
            "buffer_m": Config.PRIMARY_BUFFER_M,
            "n_rows": primary_integrity["n_rows"],
            "n_positive": primary_integrity["n_positive"],
            "n_negative": primary_integrity["n_negative"],
            "n_spatial_blocks": Config.N_SPATIAL_BLOCKS,
            "block_class_counts": (
                primary_integrity["block_class_counts"]
            ),
        },
        "model_architecture": model_config_json,
        "spatial_cv": {
            "outer_cv_type": (
                "five contiguous longitudinal spatial blocks"
            ),
            "fold_rule": (
                "outer fold k holds out spatial block k"
            ),
            "sample_assignments_csv": str(
                Config.SAMPLE_FOLD_ASSIGNMENTS_CSV
            ),
            "fold_manifest_csv": str(
                Config.FOLD_MANIFEST_CSV
            ),
        },
        "preprocessing": preprocessing_manifest,
        "buffer_sensitivity": {
            "buffers_m": [100, 500, 1000],
            "primary_buffer_m": 100,
            "qa_csv": str(Config.BUFFER_QA_CSV),
            "all_buffer_datasets_pass_consistency_qa": True,
        },
        "road_backbone_qa": {
            "road_qa_json": str(Config.ROAD_QA_JSON),
            "qa_map_png": str(Config.BLOCK_QA_MAP_PNG),
            "maximum_synthetic_gap_m": (
                None
                if not np.isfinite(max_gap_m)
                else max_gap_m
            ),
            "warning": road_warning,
        },
        "model_ready_dataset": str(
            Config.MODEL_READY_DATASET_CSV
        ),
    }

    with open(
        Config.FINAL_MANIFEST_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            to_json_safe(final_manifest),
            handle,
            indent=2,
            ensure_ascii=False,
        )

    # 12. QA report
    with open(
        Config.QA_REPORT_TXT,
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write("=" * 100 + "\n")
        handle.write(
            "STEP 6.4 — MODEL-READY DATASET AND SPATIAL-CV QA REPORT\n"
        )
        handle.write("=" * 100 + "\n\n")

        handle.write(
            f"Generated UTC: {utc_now_iso()}\n"
        )
        handle.write(
            f"Status: {final_manifest['status']}\n\n"
        )

        handle.write("PRIMARY 100 m DATASET\n")
        handle.write("-" * 72 + "\n")
        handle.write(
            f"Rows: {primary_integrity['n_rows']}\n"
        )
        handle.write(
            f"Positive sites: {primary_integrity['n_positive']}\n"
        )
        handle.write(
            f"Negative sites: {primary_integrity['n_negative']}\n"
        )
        handle.write(
            f"Spatial blocks: {primary_integrity['n_blocks']}\n"
        )
        handle.write(
            f"Block class counts: "
            f"{primary_integrity['block_class_counts']}\n\n"
        )

        handle.write("MODEL CONFIGURATIONS\n")
        handle.write("-" * 72 + "\n")

        for _, row in model_config_df.iterrows():
            handle.write(
                f"{row['scenario']}: "
                f"{int(row['n_predictors'])} predictors\n"
            )
            handle.write(
                f"  {row['predictors']}\n"
            )

        handle.write("\nSPATIAL CROSS-VALIDATION\n")
        handle.write("-" * 72 + "\n")

        for _, row in fold_manifest_df.iterrows():
            handle.write(
                f"Fold {int(row['cv_fold'])}: "
                f"held-out block={int(row['held_out_block'])}; "
                f"training blocks={row['training_blocks']}; "
                f"train n={int(row['n_train'])}; "
                f"test n={int(row['n_test'])}; "
                f"test positive={int(row['n_test_positive'])}; "
                f"test negative={int(row['n_test_negative'])}\n"
            )

        handle.write("\nPREPROCESSING RULES\n")
        handle.write("-" * 72 + "\n")
        handle.write(
            "No preprocessing is fitted globally before outer spatial CV.\n"
        )
        handle.write(
            "Land Use is treated as categorical and encoded inside each "
            "training fold.\n"
        )
        handle.write(
            "Logistic Regression scales continuous predictors inside each "
            "training fold only.\n"
        )
        handle.write(
            "XGBoost and Random Forest do not standardize continuous predictors.\n\n"
        )

        handle.write("BUFFER-SENSITIVITY DATASETS\n")
        handle.write("-" * 72 + "\n")

        for _, row in buffer_qa_df.iterrows():
            handle.write(
                f"{int(row['buffer_m'])} m: "
                f"rows={int(row['n_rows'])}; "
                f"positive={int(row['n_positive'])}; "
                f"negative={int(row['n_negative'])}; "
                f"same positives={row['same_positive_site_ids_as_100m']}; "
                f"same blocks={row['same_positive_block_assignments_as_100m']}; "
                f"status={row['status']}\n"
            )

        handle.write("\nROAD-BACKBONE QA\n")
        handle.write("-" * 72 + "\n")

        if road_warning:
            handle.write(road_warning + "\n")
        else:
            handle.write(
                "No configured large-gap warning was triggered.\n"
            )

        handle.write("\nFINAL ML CONTRACT\n")
        handle.write("-" * 72 + "\n")
        handle.write(
            "1. Use model_ready_dataset_A_to_F.csv as the authoritative "
            "primary modelling dataset.\n"
        )
        handle.write(
            "2. Use the frozen cv_fold/spatial_block assignments exactly as "
            "written; never regenerate folds separately by algorithm.\n"
        )
        handle.write(
            "3. Models A–F use identical rows, labels and outer spatial folds.\n"
        )
        handle.write(
            "4. Fit preprocessing and hyperparameter searches using training "
            "data only within each outer fold.\n"
        )
        handle.write(
            "5. Reuse the same modelling pipeline for 500 m and 1000 m buffer "
            "sensitivity analyses.\n"
        )

    log(
        f"[OUT] Model-ready dataset -> "
        f"{Config.MODEL_READY_DATASET_CSV}"
    )
    log(
        f"[OUT] Model configuration -> "
        f"{Config.MODEL_CONFIG_CSV}"
    )
    log(
        f"[OUT] CV sample assignments -> "
        f"{Config.SAMPLE_FOLD_ASSIGNMENTS_CSV}"
    )
    log(
        f"[OUT] CV fold manifest -> "
        f"{Config.FOLD_MANIFEST_CSV}"
    )
    log(
        f"[OUT] Preprocessing manifest -> "
        f"{Config.PREPROCESSING_MANIFEST_JSON}"
    )
    log(
        f"[OUT] Buffer QA -> "
        f"{Config.BUFFER_QA_CSV}"
    )
    log(
        f"[OUT] Final manifest -> "
        f"{Config.FINAL_MANIFEST_JSON}"
    )
    log(
        f"[OUT] QA report -> "
        f"{Config.QA_REPORT_TXT}"
    )

    log(
        "=== Step 6.4 completed successfully ==="
    )


if __name__ == "__main__":
    try:
        main()

    except Exception as exc:
        print(
            f"[ERROR] Step 6.4 failed: {exc}"
        )
        traceback.print_exc()
        raise


# Step 6.5 — Landslide Conditioning-Factor Relative Relevance for Models A–F

In [ ]:
# =============================================================================
# Step 6.5 — Landslide Conditioning-Factor Relative Relevance for Models A–F
# =============================================================================
# Uses the authoritative Step 6.4 model-ready dataset and frozen five spatial
# folds. Produces six separate compact importance-style figures (one per model).
#
# IMPORTANT:
# - Scores are WITHIN-MODEL relative relevance percentages.
# - Percentages are not directly comparable between different A–F models.
# - This is a pre-model filter/relevance analysis, not permutation importance
#   or SHAP.
# =============================================================================

from __future__ import annotations
import os

import json
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"

    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    MODEL_READY_DATASET_CSV = STEP64_DIR / "model_ready_dataset_A_to_F.csv"
    STEP64_MODEL_CONFIG_JSON = STEP64_DIR / "model_configuration_manifest.json"

    OUT_DIR = RESULTS_DIR / "Step-6.5-Factor_Relevance_Models_A_to_F"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    SAMPLE_ID_COLUMN = "sample_id"

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    MODEL_DESCRIPTIONS = {
        "A_static_baseline": "Static baseline",
        "B_raw_deformation": "Static + DV+",
        "C_hkfi_only": "Static + HKFI",
        "D_sndr_only": "Static + SNDR",
        "E_physics_guided": "Static + HKFI + SNDR",
        "F_untransformed_sources": "Static + DV+ + TWI + SM",
    }

    DISPLAY_LABELS = {
        "slope_deg": "Slope",
        "northness": "Northness",
        "eastness": "Eastness",
        "profile_curv": "Prof.Curv",
        "plan_curv": "Plan.Curv",
        "tpi_s5": "TPI",
        "spi_ln": "SPI",
        "dist_river_m": "Dist.River",
        "dist_road_m": "Dist.Road",
        "aap_mm": "AAP",
        "ndvi": "NDVI",
        "landcover": "Land Use",
        "dvplus_mm_yr": "DV+",
        "hkfi_0_1": "HKFI",
        "sndr_0_1": "SNDR",
        "twi_raw": "TWI",
        "sm_support_physical": "SM",
    }

    CATEGORICAL_FEATURES = {"landcover"}

    N_QUANTILE_BINS = 10
    WOE_EPS = 0.5

    RANDOM_STATE = 42
    LOGISTIC_C = 1.0
    MAX_ITER = 5000

    FIGSIZE = (2, 2)
    FIGURE_DPI = 900
    SAVE_DPI = 900
    ROUND_LABELS_DECIMALS = 1
    SHOW_MODEL_TITLE = False


# =============================================================================
# HELPERS
# =============================================================================
def pp(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json(path: Path) -> Dict[str, Any]:
    require_file(path, "JSON file")
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def is_categorical(feature: str) -> bool:
    return feature in Config.CATEGORICAL_FEATURES


# =============================================================================
# LOAD / VALIDATE AUTHORITATIVE STEP 6.4 DATASET
# =============================================================================
def load_and_validate_dataset(
    scenarios: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    require_file(
        Config.MODEL_READY_DATASET_CSV,
        "Step 6.4 authoritative model-ready dataset",
    )

    df = pd.read_csv(Config.MODEL_READY_DATASET_CSV)

    required_metadata = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        "spatial_block",
    ]

    missing_metadata = [
        c for c in required_metadata if c not in df.columns
    ]

    if missing_metadata:
        raise RuntimeError(
            f"Missing required Step 6.4 columns: {missing_metadata}"
        )

    all_features: List[str] = []

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in scenarios:
            raise RuntimeError(
                f"Scenario missing from Step 6.1 manifest: {scenario_name}"
            )

        for feature in scenarios[scenario_name]["features"]:
            if feature not in all_features:
                all_features.append(feature)

    missing_features = [
        f for f in all_features if f not in df.columns
    ]

    if missing_features:
        raise RuntimeError(
            f"Model-ready dataset is missing predictors: {missing_features}"
        )

    irs_columns = [
        c for c in df.columns if "irs" in str(c).lower()
    ]

    if irs_columns:
        raise RuntimeError(
            f"IRS-related columns are present: {irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError(
            "tpi_s25 is present. Only tpi_s5 is allowed."
        )

    if df[Config.SAMPLE_ID_COLUMN].duplicated().any():
        raise RuntimeError("Duplicated sample_id values found.")

    if len(df) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Expected {Config.EXPECTED_N_ROWS} rows, found {len(df)}. "
            "No silent row dropping is allowed."
        )

    df[Config.TARGET_COLUMN] = pd.to_numeric(
        df[Config.TARGET_COLUMN],
        errors="raise",
    ).astype(int)

    class_counts = (
        df[Config.TARGET_COLUMN]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    if class_counts != {
        0: Config.EXPECTED_N_NEGATIVE,
        1: Config.EXPECTED_N_POSITIVE,
    }:
        raise RuntimeError(
            f"Unexpected class counts: {class_counts}"
        )

    df[Config.FOLD_COLUMN] = pd.to_numeric(
        df[Config.FOLD_COLUMN],
        errors="raise",
    ).astype(int)

    observed_folds = sorted(
        df[Config.FOLD_COLUMN].unique().tolist()
    )

    if observed_folds != Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected frozen folds {Config.EXPECTED_FOLDS}, "
            f"found {observed_folds}."
        )

    spatial_blocks = pd.to_numeric(
        df["spatial_block"],
        errors="raise",
    ).astype(int)

    if not np.array_equal(
        df[Config.FOLD_COLUMN].to_numpy(),
        spatial_blocks.to_numpy(),
    ):
        raise RuntimeError(
            "cv_fold and spatial_block are not identical."
        )

    for feature in all_features:
        if is_categorical(feature):
            if df[feature].isna().any():
                raise RuntimeError(
                    f"Missing values found in categorical predictor {feature}."
                )
        else:
            numeric = pd.to_numeric(
                df[feature],
                errors="coerce",
            )

            if (
                numeric.isna().any()
                or not np.isfinite(
                    numeric.to_numpy(dtype="float64")
                ).all()
            ):
                raise RuntimeError(
                    f"Missing/non-finite values found in predictor {feature}."
                )

            df[feature] = numeric.astype(float)

    pp(f"[PATH] {Config.MODEL_READY_DATASET_CSV}")
    pp(
        f"[SAMPLES] rows={len(df)} | "
        f"positive={class_counts[1]} | negative={class_counts[0]}"
    )
    pp(f"[FOLDS] frozen spatial folds={observed_folds}")
    pp("[COMMON ROWS] No rows dropped; identical rows used for Models A–F.")

    return df


# =============================================================================
# BINNING
# =============================================================================
def make_bins(
    x: pd.Series,
    categorical: bool,
) -> pd.Series:
    if categorical:
        return x.astype(str).fillna("Missing")

    numeric = pd.to_numeric(
        x,
        errors="raise",
    ).astype(float)

    n_unique = int(numeric.nunique(dropna=True))

    if n_unique <= 1:
        return pd.Series(
            ["constant"] * len(numeric),
            index=numeric.index,
        )

    n_bins = min(
        Config.N_QUANTILE_BINS,
        n_unique,
    )

    try:
        return pd.qcut(
            numeric,
            q=n_bins,
            duplicates="drop",
        ).astype(str)

    except Exception:
        return pd.cut(
            numeric,
            bins=n_bins,
            duplicates="drop",
        ).astype(str)


# =============================================================================
# METRICS
# =============================================================================
def anova_f_metric(
    x: pd.Series,
    y: pd.Series,
    categorical: bool,
) -> float:
    if categorical:
        dummies = pd.get_dummies(
            x.astype(str),
            drop_first=False,
        )

        scores, _ = f_classif(
            dummies.astype(float).values,
            y.values,
        )

        scores = np.nan_to_num(
            scores,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return float(np.max(scores)) if scores.size else 0.0

    values = pd.to_numeric(
        x,
        errors="raise",
    ).astype(float).to_numpy().reshape(-1, 1)

    scores, _ = f_classif(
        values,
        y.values,
    )

    return float(
        max(
            np.nan_to_num(
                scores[0],
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            ),
            0.0,
        )
    )


def point_biserial_metric(
    x: pd.Series,
    y: pd.Series,
    categorical: bool,
) -> float:
    if categorical:
        dummies = pd.get_dummies(
            x.astype(str),
            drop_first=False,
        )

        values = []

        for column in dummies.columns:
            a = dummies[column].astype(float).to_numpy()

            if np.std(a) <= 1e-12:
                continue

            r = np.corrcoef(
                a,
                y.to_numpy(),
            )[0, 1]

            if np.isfinite(r):
                values.append(abs(float(r)))

        return max(values) if values else 0.0

    a = pd.to_numeric(
        x,
        errors="raise",
    ).astype(float).to_numpy()

    if np.std(a) <= 1e-12:
        return 0.0

    r = np.corrcoef(
        a,
        y.to_numpy(),
    )[0, 1]

    return abs(float(r)) if np.isfinite(r) else 0.0


def iv_and_woe_metrics(
    x: pd.Series,
    y: pd.Series,
    categorical: bool,
) -> Tuple[float, float]:
    bins = make_bins(x, categorical)

    table = pd.DataFrame(
        {
            "bin": bins,
            "y": y.to_numpy(),
        }
    )

    grouped = (
        table.groupby("bin", observed=False)["y"]
        .agg(count="count", events="sum")
    )

    grouped["non_events"] = (
        grouped["count"] - grouped["events"]
    )

    total_events = float(grouped["events"].sum())
    total_non_events = float(grouped["non_events"].sum())

    if total_events <= 0 or total_non_events <= 0:
        return 0.0, 0.0

    k = max(len(grouped), 1)
    eps = Config.WOE_EPS

    dist_events = (
        grouped["events"].astype(float) + eps
    ) / (
        total_events + eps * k
    )

    dist_non_events = (
        grouped["non_events"].astype(float) + eps
    ) / (
        total_non_events + eps * k
    )

    woe = np.log(
        dist_events / dist_non_events
    )

    iv = np.sum(
        (dist_events - dist_non_events) * woe
    )

    weights = (
        grouped["count"].astype(float)
        / float(grouped["count"].sum())
    )

    weighted_abs_woe = np.sum(
        weights * np.abs(woe)
    )

    return (
        float(iv) if np.isfinite(iv) else 0.0,
        float(weighted_abs_woe)
        if np.isfinite(weighted_abs_woe)
        else 0.0,
    )


def chi_square_metric(
    x: pd.Series,
    y: pd.Series,
    categorical: bool,
) -> float:
    bins = make_bins(x, categorical)

    contingency = pd.crosstab(
        bins,
        y,
    )

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        return 0.0

    chi2, _, _, _ = chi2_contingency(
        contingency.values,
        correction=False,
    )

    return float(chi2) if np.isfinite(chi2) else 0.0


def make_one_feature_pipeline(
    categorical: bool,
) -> Pipeline:
    if categorical:
        try:
            encoder = OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            )
        except TypeError:
            encoder = OneHotEncoder(
                handle_unknown="ignore",
                sparse=False,
            )

        return Pipeline(
            [
                ("encode", encoder),
                (
                    "model",
                    LogisticRegression(
                        C=Config.LOGISTIC_C,
                        solver="liblinear",
                        max_iter=Config.MAX_ITER,
                        class_weight="balanced",
                        random_state=Config.RANDOM_STATE,
                    ),
                ),
            ]
        )

    return Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    C=Config.LOGISTIC_C,
                    solver="liblinear",
                    max_iter=Config.MAX_ITER,
                    class_weight="balanced",
                    random_state=Config.RANDOM_STATE,
                ),
            ),
        ]
    )


def spatial_oof_univariate_auc(
    x: pd.Series,
    y: pd.Series,
    categorical: bool,
    folds: pd.Series,
) -> Tuple[float, float]:
    if categorical:
        X = x.astype(str).to_numpy().reshape(-1, 1)
    else:
        X = (
            pd.to_numeric(
                x,
                errors="raise",
            )
            .astype(float)
            .to_numpy()
            .reshape(-1, 1)
        )

    y_values = y.astype(int).to_numpy()
    fold_values = folds.astype(int).to_numpy()

    oof = np.full(
        len(y_values),
        np.nan,
        dtype=float,
    )

    for fold_value in Config.EXPECTED_FOLDS:
        test_mask = fold_values == fold_value
        train_mask = ~test_mask

        pipeline = make_one_feature_pipeline(
            categorical
        )

        pipeline.fit(
            X[train_mask],
            y_values[train_mask],
        )

        oof[test_mask] = pipeline.predict_proba(
            X[test_mask]
        )[:, 1]

    if not np.isfinite(oof).all():
        raise RuntimeError(
            "Incomplete spatial OOF probabilities."
        )

    raw_auc = float(
        roc_auc_score(
            y_values,
            oof,
        )
    )

    auc_strength = max(
        raw_auc,
        1.0 - raw_auc,
    )

    return float(auc_strength), raw_auc


# =============================================================================
# COMPOSITE RELATIVE RELEVANCE
# =============================================================================
def minmax_metric(
    values: pd.Series,
) -> pd.Series:
    values = (
        pd.to_numeric(
            values,
            errors="coerce",
        )
        .fillna(0.0)
        .astype(float)
    )

    minimum = float(values.min())
    maximum = float(values.max())

    if maximum <= minimum + 1e-15:
        return pd.Series(
            np.zeros(len(values)),
            index=values.index,
        )

    return (
        100.0
        * (values - minimum)
        / (maximum - minimum)
    )


def compute_scenario_relevance(
    df: pd.DataFrame,
    features: List[str],
) -> pd.DataFrame:
    y = df[Config.TARGET_COLUMN].astype(int)
    folds = df[Config.FOLD_COLUMN].astype(int)

    rows = []

    for feature in features:
        x = df[feature]
        categorical = is_categorical(feature)

        iv, woe_abs = iv_and_woe_metrics(
            x,
            y,
            categorical,
        )

        auc_strength, raw_auc = spatial_oof_univariate_auc(
            x,
            y,
            categorical,
            folds,
        )

        rows.append(
            {
                "Variable": feature,
                "ANOVA_F": anova_f_metric(
                    x,
                    y,
                    categorical,
                ),
                "Information_Value": iv,
                "Point_Biserial_Abs": point_biserial_metric(
                    x,
                    y,
                    categorical,
                ),
                "Weighted_Abs_WoE": woe_abs,
                "Spatial_OOF_AUC_Raw": raw_auc,
                "Spatial_OOF_AUC_Strength": auc_strength,
                "Chi_Square": chi_square_metric(
                    x,
                    y,
                    categorical,
                ),
            }
        )

    result = pd.DataFrame(rows)

    metric_columns = [
        "ANOVA_F",
        "Information_Value",
        "Point_Biserial_Abs",
        "Weighted_Abs_WoE",
        "Spatial_OOF_AUC_Strength",
        "Chi_Square",
    ]

    normalized_columns = []

    for metric in metric_columns:
        normalized_name = (
            f"{metric}_Normalized_0_100"
        )

        result[normalized_name] = minmax_metric(
            result[metric]
        )

        normalized_columns.append(
            normalized_name
        )

    result["MultiMetric_Relevance_Raw"] = (
        result[normalized_columns].mean(axis=1)
    )

    total = float(
        result["MultiMetric_Relevance_Raw"].sum()
    )

    if total <= 0:
        result[
            "Within_Model_Relative_Relevance_Percent"
        ] = (
            100.0 / len(result)
        )
    else:
        result[
            "Within_Model_Relative_Relevance_Percent"
        ] = (
            100.0
            * result["MultiMetric_Relevance_Raw"]
            / total
        )

    result["Rank"] = (
        result[
            "Within_Model_Relative_Relevance_Percent"
        ]
        .rank(
            method="min",
            ascending=False,
        )
        .astype(int)
    )

    return result.sort_values(
        [
            "Within_Model_Relative_Relevance_Percent",
            "Variable",
        ],
        ascending=[False, True],
    ).reset_index(drop=True)


# =============================================================================
# PLOTTING — SAME COMPACT STYLE AS EARLIER
# =============================================================================
def distinct_colors(n: int):
    import matplotlib.colors as mcolors

    base = list(plt.cm.tab20.colors)

    if n <= len(base):
        return base[:n]

    extra = n - len(base)

    extras = [
        mcolors.hsv_to_rgb(
            (
                i / max(extra, 1),
                0.55,
                0.90,
            )
        )
        for i in range(extra)
    ]

    return base + extras


def force_rounded_labels_sum_to_100(
    values: Sequence[float],
    decimals: int = 1,
    target: float = 100.0,
) -> np.ndarray:
    values = np.asarray(
        values,
        dtype="float64",
    )

    rounded = np.round(
        values,
        decimals,
    )

    difference = np.round(
        target - rounded.sum(),
        decimals,
    )

    if abs(difference) > 0 and len(rounded) > 0:
        index = int(np.argmax(values))
        rounded[index] = np.round(
            rounded[index] + difference,
            decimals,
        )

    return rounded


def plot_relevance(
    scenario_name: str,
    relevance_df: pd.DataFrame,
    png_path: Path,
    pdf_path: Path,
    svg_path: Path,
) -> None:
    mpl.rcParams.update(
        {
            "font.family": "serif",
            "font.serif": [
                "Times New Roman",
                "Times",
                "DejaVu Serif",
            ],
            "mathtext.fontset": "stix",
            "axes.titlesize": 4,
            "axes.labelsize": 4,
            "xtick.labelsize": 4,
            "ytick.labelsize": 4,
            "font.size": 4,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )

    plot_df = relevance_df.sort_values(
        "Within_Model_Relative_Relevance_Percent",
        ascending=True,
    ).copy()

    plot_df["Label"] = plot_df["Variable"].map(
        lambda value: Config.DISPLAY_LABELS.get(
            value,
            value,
        )
    )

    values = plot_df[
        "Within_Model_Relative_Relevance_Percent"
    ].to_numpy(dtype="float64")

    rounded_labels = force_rounded_labels_sum_to_100(
        values,
        decimals=Config.ROUND_LABELS_DECIMALS,
        target=100.0,
    )

    plt.style.use("default")

    fig, ax = plt.subplots(
        figsize=Config.FIGSIZE,
        dpi=Config.FIGURE_DPI,
    )

    bars = ax.barh(
        range(len(plot_df)),
        values,
        color=distinct_colors(len(plot_df)),
        alpha=0.95,
        height=0.55,
    )

    max_value = float(values.max()) if len(values) else 1.0

    for bar, true_value, label_value in zip(
        bars,
        values,
        rounded_labels,
    ):
        ax.text(
            true_value + max_value * 0.02,
            bar.get_y() + bar.get_height() / 2,
            f"{label_value:.{Config.ROUND_LABELS_DECIMALS}f}%",
            va="center",
            ha="left",
            fontsize=3.0,
            fontweight="bold",
            fontfamily="Times New Roman",
        )

    ax.set_yticks(range(len(plot_df)))
    ax.set_yticklabels(
        plot_df["Label"],
        fontsize=4,
    )

    ax.set_xlabel(
        "Relative relevance (%)",
        fontsize=4,
        fontweight="bold",
        labelpad=2,
    )

    ax.set_ylabel(
        "Conditioning factors",
        fontsize=4,
        fontweight="bold",
        labelpad=2,
    )

    if Config.SHOW_MODEL_TITLE:
        ax.set_title(
            f"{Config.MODEL_SHORT_LABELS[scenario_name]} — "
            f"{Config.MODEL_DESCRIPTIONS[scenario_name]}",
            fontsize=4,
            fontweight="bold",
            pad=2,
        )

    ax.grid(
        True,
        axis="x",
        alpha=0.3,
        linestyle="-",
        linewidth=0.4,
    )

    ax.set_axisbelow(True)

    ax.set_xlim(
        0,
        max_value * 1.20 if max_value > 0 else 1.0,
    )

    fig.subplots_adjust(
        left=0.35,
        bottom=0.15,
        right=0.95,
        top=0.95,
    )

    for path in (
        png_path,
        pdf_path,
        svg_path,
    ):
        if path.exists():
            path.unlink()

    fig.savefig(
        png_path,
        dpi=Config.SAVE_DPI,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.02,
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.02,
    )

    fig.savefig(
        svg_path,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.02,
    )

    plt.show()
    plt.close(fig)

    mpl.rcParams.update(
        mpl.rcParamsDefault
    )


# =============================================================================
# REPORTING
# =============================================================================
def write_scenario_report(
    scenario_name: str,
    relevance_df: pd.DataFrame,
    report_path: Path,
) -> None:
    with open(
        report_path,
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write("=" * 96 + "\n")
        handle.write(
            f"{Config.MODEL_SHORT_LABELS[scenario_name]} — "
            f"{Config.MODEL_DESCRIPTIONS[scenario_name]}\n"
        )
        handle.write("=" * 96 + "\n\n")

        handle.write(
            f"Generated: {datetime.now():%Y-%m-%d %H:%M:%S}\n"
        )
        handle.write(
            f"Rows analyzed: {Config.EXPECTED_N_ROWS}\n"
        )
        handle.write(
            "Cross-validation: fixed five-block spatial OOF for "
            "the univariate AUC metric\n\n"
        )

        handle.write(
            "Interpretation: values are within-model relative relevance "
            "percentages and should not be compared numerically across "
            "different A–F models.\n\n"
        )

        for _, row in relevance_df.iterrows():
            variable = row["Variable"]
            label = Config.DISPLAY_LABELS.get(
                variable,
                variable,
            )

            handle.write(
                f"{int(row['Rank']):2d}. "
                f"{variable:<24s} "
                f"({label:<15s}) "
                f"{row['Within_Model_Relative_Relevance_Percent']:8.4f}%\n"
            )


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    pp("=" * 94)
    pp(
        "=== Step 6.5: Conditioning-Factor Relative Relevance "
        "for Models A–F ==="
    )
    pp("=" * 94)

    scenario_manifest = load_json(
        Config.SCENARIO_MANIFEST_JSON
    )

    scenarios = scenario_manifest["scenarios"]

    # Cross-check Step 6.4 frozen configuration when available.
    if Config.STEP64_MODEL_CONFIG_JSON.exists():
        frozen = load_json(
            Config.STEP64_MODEL_CONFIG_JSON
        ).get(
            "scenarios",
            {},
        )

        for scenario_name in Config.SCENARIO_ORDER:
            if scenario_name not in frozen:
                raise RuntimeError(
                    f"Step 6.4 frozen configuration is missing "
                    f"{scenario_name}."
                )

            if list(
                frozen[scenario_name]["features"]
            ) != list(
                scenarios[scenario_name]["features"]
            ):
                raise RuntimeError(
                    f"Step 6.1 and Step 6.4 predictor definitions differ "
                    f"for {scenario_name}."
                )

        pp(
            "[QA] Step 6.1 scenario manifest matches Step 6.4 "
            "frozen configuration: PASS"
        )

    df = load_and_validate_dataset(
        scenarios
    )

    summaries = []
    long_tables = []

    for scenario_name in Config.SCENARIO_ORDER:
        features = list(
            scenarios[scenario_name]["features"]
        )

        pp("")
        pp("=" * 94)
        pp(
            f"=== {Config.MODEL_SHORT_LABELS[scenario_name]}: "
            f"{Config.MODEL_DESCRIPTIONS[scenario_name]} "
            f"({len(features)} predictors) ==="
        )
        pp("=" * 94)

        relevance_df = compute_scenario_relevance(
            df,
            features,
        )

        prefix = scenario_name.replace("_", "-")

        csv_path = (
            Config.OUT_DIR
            / f"{prefix}_Conditioning_Factor_Relative_Relevance.csv"
        )

        png_path = (
            Config.OUT_DIR
            / f"{prefix}_Conditioning_Factor_Relative_Relevance.png"
        )

        pdf_path = (
            Config.OUT_DIR
            / f"{prefix}_Conditioning_Factor_Relative_Relevance.pdf"
        )

        svg_path = (
            Config.OUT_DIR
            / f"{prefix}_Conditioning_Factor_Relative_Relevance.svg"
        )

        report_path = (
            Config.OUT_DIR
            / f"{prefix}_Conditioning_Factor_Relative_Relevance.txt"
        )

        relevance_df.to_csv(
            csv_path,
            index=False,
            float_format="%.8f",
        )

        plot_relevance(
            scenario_name,
            relevance_df,
            png_path,
            pdf_path,
            svg_path,
        )

        write_scenario_report(
            scenario_name,
            relevance_df,
            report_path,
        )

        top = relevance_df.iloc[0]

        summaries.append(
            {
                "Scenario": scenario_name,
                "Model": Config.MODEL_SHORT_LABELS[scenario_name],
                "Description": Config.MODEL_DESCRIPTIONS[scenario_name],
                "N_Predictors": len(features),
                "N_Rows": len(df),
                "Top_Factor": top["Variable"],
                "Top_Factor_Display": Config.DISPLAY_LABELS.get(
                    top["Variable"],
                    top["Variable"],
                ),
                "Top_Factor_Relative_Relevance_Percent": float(
                    top[
                        "Within_Model_Relative_Relevance_Percent"
                    ]
                ),
                "Relevance_Sum": float(
                    relevance_df[
                        "Within_Model_Relative_Relevance_Percent"
                    ].sum()
                ),
            }
        )

        long_df = relevance_df.copy()
        long_df.insert(
            0,
            "Scenario",
            scenario_name,
        )
        long_df.insert(
            1,
            "Model",
            Config.MODEL_SHORT_LABELS[scenario_name],
        )
        long_df.insert(
            2,
            "Description",
            Config.MODEL_DESCRIPTIONS[scenario_name],
        )

        long_tables.append(long_df)

        pp("[TOP FACTORS]")

        for rank, (_, row) in enumerate(
            relevance_df.head(10).iterrows(),
            start=1,
        ):
            display = Config.DISPLAY_LABELS.get(
                row["Variable"],
                row["Variable"],
            )

            pp(
                f"{rank:2d}. "
                f"{display:<15s} "
                f"{row['Within_Model_Relative_Relevance_Percent']:7.3f}%"
            )

    summary_df = pd.DataFrame(summaries)

    summary_path = (
        Config.OUT_DIR
        / "All_Models_A_to_F_Relative_Relevance_Summary.csv"
    )

    summary_df.to_csv(
        summary_path,
        index=False,
    )

    long_all = pd.concat(
        long_tables,
        ignore_index=True,
    )

    long_path = (
        Config.OUT_DIR
        / "All_Models_A_to_F_Relative_Relevance_Long.csv"
    )

    long_all.to_csv(
        long_path,
        index=False,
        float_format="%.8f",
    )

    methodology_path = (
        Config.OUT_DIR
        / "Step_6_5_Conditioning_Factor_Relevance_Methodology.txt"
    )

    with open(
        methodology_path,
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(
            "STEP 6.5 — MODEL-INDEPENDENT CONDITIONING-FACTOR RELEVANCE\n\n"
        )
        handle.write(
            f"Input: {Config.MODEL_READY_DATASET_CSV}\n\n"
        )
        handle.write(
            "All six A–F models use the same 330 Step 6.4 observations "
            "and the same frozen five spatial folds. No rows are dropped.\n\n"
        )
        handle.write(
            "The six metrics are ANOVA F, Information Value, absolute "
            "point-biserial association, weighted absolute WoE, fixed "
            "five-block spatial OOF univariate AUC, and chi-square.\n\n"
        )
        handle.write(
            "Each metric is min-max normalized within each scenario, then "
            "the six normalized metrics are averaged and rescaled to sum "
            "to 100% within that scenario.\n\n"
        )
        handle.write(
            "The resulting percentages are within-model relative relevance "
            "scores. They are not directly comparable across Models A–F "
            "because the predictor sets and normalization pools differ.\n\n"
        )
        handle.write(
            "This pre-model analysis is distinct from downstream "
            "permutation importance and SHAP."
        )

    pp("")
    pp("=" * 94)
    pp("=== STEP 6.5 COMPLETE ===")
    pp("=" * 94)
    pp(f"[OUT] {Config.OUT_DIR}")
    pp(f"[OUT] {summary_path}")
    pp(f"[OUT] {long_path}")
    pp(f"[OUT] {methodology_path}")


if __name__ == "__main__":
    main()


# Step 6.6 — Landslide Inventory Completeness and ML-Sample Representation Audit

In [ ]:
# =============================================================================
# Step 6.6 — Landslide Inventory Completeness and ML-Sample Representation Audit
# =============================================================================
# Diagnostic only: this step does not modify the 165-site ML dataset.
#
# Main outputs:
#   - polygon areas for all mapped landslides + Step 6.3 retention status
#   - empirical area-frequency rollover from KDE mode in log10(area)
#   - bootstrap uncertainty and binning/bandwidth sensitivity
#   - retained-versus-excluded area statistics and size-bias tests
#   - Supplementary Figure S7
#
# IMPORTANT:
# The rollover is an empirical indicator of approximate size-dependent catalogue
# completeness, not a hard detection limit and not proof of absolute completeness.
# =============================================================================

from __future__ import annotations
import os

import json
import math
import traceback
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

import fiona
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import CRS, Transformer
from scipy.stats import fisher_exact, gaussian_kde, ks_2samp, mannwhitneyu
from shapely.geometry import GeometryCollection, MultiPolygon, Polygon, shape
from shapely.ops import transform as shp_transform, unary_union


class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    TRAIN_SHP = (
        BASE / "Landslide Historical Map" / "Training Data" /
        "Landslide_only_train.shp"
    )
    VALID_SHP = (
        BASE / "Landslide Historical Map" / "Validation Data" /
        "Landslide_only_valid.shp"
    )

    FILTER_SYMBOLID_IF_PRESENT = True
    LANDSLIDE_SYMBOL_VALUE = 1

    TRAIN_SOURCE_TAG = "training_inventory"
    VALID_SOURCE_TAG = "validation_inventory"
    TRAIN_SITE_PREFIX = "LS_TR"
    VALID_SITE_PREFIX = "LS_VA"

    EXPECTED_TRAIN = 186
    EXPECTED_VALID = 46
    EXPECTED_TOTAL = 232

    STEP63_DIR = RESULTS_DIR / "Step-6.3-Site_Level_Sampling"
    RECONCILIATION_CSV = STEP63_DIR / "landslide_site_reconciliation.csv"

    OUT_DIR = RESULTS_DIR / "Step-6.6-Inventory_Completeness"

    INVENTORY_AREA_CSV = OUT_DIR / "landslide_polygon_area_and_retention_status.csv"
    FREQUENCY_CSV = OUT_DIR / "inventory_area_frequency_bins.csv"
    ROLLOVER_SENSITIVITY_CSV = OUT_DIR / "inventory_rollover_sensitivity.csv"
    BOOTSTRAP_CSV = OUT_DIR / "bootstrap_rollover_thresholds.csv"
    GROUP_STATS_CSV = OUT_DIR / "retained_vs_excluded_area_statistics.csv"
    STATUS_STATS_CSV = OUT_DIR / "area_statistics_by_reconciliation_status.csv"
    BIAS_TESTS_CSV = OUT_DIR / "retained_vs_excluded_area_bias_tests.csv"
    SUMMARY_JSON = OUT_DIR / "inventory_completeness_summary.json"
    QA_REPORT_TXT = OUT_DIR / "Step_6_6_inventory_completeness_QA_report.txt"

    FIGURE_COMBINED_PNG = OUT_DIR / "Figure_S7_inventory_completeness_and_sample_representation.png"
    FIGURE_COMBINED_PDF = OUT_DIR / "Figure_S7_inventory_completeness_and_sample_representation.pdf"
    FIGURE_COMBINED_SVG = OUT_DIR / "Figure_S7_inventory_completeness_and_sample_representation.svg"
    FIGURE_A_PNG = OUT_DIR / "Figure_S7a_inventory_area_frequency.png"
    FIGURE_A_PDF = OUT_DIR / "Figure_S7a_inventory_area_frequency.pdf"
    FIGURE_A_SVG = OUT_DIR / "Figure_S7a_inventory_area_frequency.svg"
    FIGURE_B_PNG = OUT_DIR / "Figure_S7b_retained_vs_excluded_area.png"
    FIGURE_B_PDF = OUT_DIR / "Figure_S7b_retained_vs_excluded_area.pdf"
    FIGURE_B_SVG = OUT_DIR / "Figure_S7b_retained_vs_excluded_area.svg"

    TARGET_EPSG = 32645
    RANDOM_SEED = 42
    KDE_GRID_SIZE = 2048
    N_BOOTSTRAP = 1000
    KDE_BANDWIDTH_FACTORS = (0.75, 1.00, 1.25)
    HISTOGRAM_RULES = ("fd", "scott", "sturges", "doane")
    SAVE_DPI = 600


def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def source_crs_from_fiona(src) -> CRS:
    if getattr(src, "crs_wkt", None):
        return CRS.from_user_input(src.crs_wkt)
    if getattr(src, "crs", None):
        return CRS.from_user_input(src.crs)
    raise RuntimeError("Could not determine vector CRS.")


def find_symbol_field(names: Sequence[str]) -> Optional[str]:
    lookup = {str(n).lower(): str(n) for n in names}
    for candidate in ("symbolid", "symbol_id", "symbol"):
        if candidate in lookup:
            return lookup[candidate]
    return None


def repair_polygon(geometry):
    repaired = False
    if geometry is None or geometry.is_empty:
        return None, repaired

    if not geometry.is_valid:
        geometry = geometry.buffer(0)
        repaired = True

    if geometry is None or geometry.is_empty:
        return None, repaired

    if isinstance(geometry, (Polygon, MultiPolygon)):
        return geometry, repaired

    if isinstance(geometry, GeometryCollection):
        parts = [
            g for g in geometry.geoms
            if isinstance(g, (Polygon, MultiPolygon))
        ]
        if parts:
            geometry = unary_union(parts)
            repaired = True
            if isinstance(geometry, (Polygon, MultiPolygon)):
                return geometry, repaired

    return None, repaired


def read_inventory(
    path: Path,
    source_tag: str,
    site_prefix: str,
    target_crs: CRS,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    require_file(path, f"{source_tag} inventory")

    rows = []
    counts = defaultdict(int)

    with fiona.open(path) as src:
        source_crs = source_crs_from_fiona(src)
        transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)
        property_names = list(src.schema.get("properties", {}).keys())
        symbol_field = find_symbol_field(property_names)

        n_source = 0
        n_filtered = 0
        n_invalid = 0
        n_repaired = 0

        for feature_index, feature in enumerate(src):
            n_source += 1
            props = dict(feature.get("properties", {}) or {})

            symbol_value = props.get(symbol_field) if symbol_field else None
            if symbol_field:
                counts[str(symbol_value)] += 1

            if Config.FILTER_SYMBOLID_IF_PRESENT and symbol_field:
                try:
                    symbol_int = int(symbol_value)
                except Exception:
                    symbol_int = None
                if symbol_int != Config.LANDSLIDE_SYMBOL_VALUE:
                    n_filtered += 1
                    continue

            geometry = None
            repaired = False
            if feature.get("geometry") is not None:
                raw = shape(feature["geometry"])
                projected = shp_transform(transformer.transform, raw)
                geometry, repaired = repair_polygon(projected)

            if geometry is None:
                n_invalid += 1
            if repaired:
                n_repaired += 1

            original_id = None
            for key in ("id", "ID", "OBJECTID", "ObjectID", "FID", "fid"):
                if key in props:
                    original_id = props[key]
                    break

            area_m2 = float(geometry.area) if geometry is not None else np.nan

            rows.append({
                "site_id": f"{site_prefix}_{feature_index + 1:05d}",
                "inventory_source": source_tag,
                "source_file": str(path),
                "source_feature_index": int(feature_index),
                "source_original_id": None if original_id is None else str(original_id),
                "symbol_field": symbol_field,
                "symbol_value": symbol_value,
                "geometry_repaired": bool(repaired),
                "valid_polygon_geometry": bool(geometry is not None),
                "polygon_area_m2": area_m2,
                "polygon_area_ha": area_m2 / 10000.0 if np.isfinite(area_m2) else np.nan,
                "log10_polygon_area_m2": (
                    math.log10(area_m2)
                    if np.isfinite(area_m2) and area_m2 > 0 else np.nan
                ),
            })

    df = pd.DataFrame(rows)
    audit = {
        "source_tag": source_tag,
        "source_file": str(path),
        "source_crs": str(source_crs),
        "analysis_crs": f"EPSG:{Config.TARGET_EPSG}",
        "source_features_total": int(n_source),
        "symbol_field": symbol_field,
        "symbol_counts": dict(counts),
        "filtered_nonlandslide_features": int(n_filtered),
        "retained_landslide_records": int(len(df)),
        "invalid_polygonal_geometries": int(n_invalid),
        "repaired_polygonal_geometries": int(n_repaired),
    }
    return df, audit


def load_reconciliation() -> pd.DataFrame:
    require_file(Config.RECONCILIATION_CSV, "Step 6.3 reconciliation CSV")
    df = pd.read_csv(Config.RECONCILIATION_CSV)

    for column in ("site_id", "status"):
        if column not in df.columns:
            raise RuntimeError(
                f"Step 6.3 reconciliation is missing required column '{column}'."
            )

    if df["site_id"].duplicated().any():
        raise RuntimeError("Duplicate site_id values found in Step 6.3 reconciliation.")

    return df


def merge_reconciliation(inventory: pd.DataFrame, reconciliation: pd.DataFrame) -> pd.DataFrame:
    columns = [
        c for c in (
            "site_id", "status", "representative_method",
            "candidate_valid_cells", "distance_to_polygon_centroid_m",
            "chainage_m", "distance_to_road_backbone_m", "spatial_block"
        )
        if c in reconciliation.columns
    ]
    rec = reconciliation[columns].copy()

    merged = inventory.merge(rec, on="site_id", how="left", validate="one_to_one")

    if merged["status"].isna().any():
        missing = merged.loc[merged["status"].isna(), "site_id"].tolist()
        raise RuntimeError(
            "Some inventory polygons could not be matched to Step 6.3: "
            f"{missing[:20]}"
        )

    extra = set(reconciliation["site_id"].astype(str)) - set(merged["site_id"].astype(str))
    if extra:
        raise RuntimeError(
            "Step 6.3 contains site IDs not found in the current inventories: "
            f"{sorted(extra)[:20]}"
        )

    merged["ml_sample_group"] = np.where(
        merged["status"].astype(str) == "retained_common_domain",
        "ML-retained",
        "Excluded",
    )
    return merged


def kde_rollover(log_area: np.ndarray, bandwidth_factor: float = 1.0) -> Dict[str, Any]:
    values = np.asarray(log_area, dtype=float)
    values = values[np.isfinite(values)]

    if values.size < 5 or np.allclose(values, values[0]):
        raise RuntimeError("Insufficient area variation for KDE rollover estimation.")

    kde = gaussian_kde(
        values,
        bw_method=lambda estimator: estimator.scotts_factor() * bandwidth_factor,
    )

    span = max(float(values.max() - values.min()), 0.1)
    grid = np.linspace(
        values.min() - 0.05 * span,
        values.max() + 0.05 * span,
        Config.KDE_GRID_SIZE,
    )
    density = kde(grid)
    idx = int(np.argmax(density))
    log_roll = float(grid[idx])
    area = float(10 ** log_roll)

    observed_span = max(float(values.max() - values.min()), 1e-12)
    relative_position = float((log_roll - values.min()) / observed_span)
    near_edge = bool(relative_position < 0.03 or relative_position > 0.97)

    return {
        "method": f"KDE mode of log10(area), Scott bandwidth × {bandwidth_factor:.2f}",
        "bandwidth_factor": float(bandwidth_factor),
        "rollover_log10_area_m2": log_roll,
        "rollover_area_m2": area,
        "rollover_area_ha": area / 10000.0,
        "mode_near_observed_range_edge": near_edge,
    }


def histogram_rollover(log_area: np.ndarray, rule: str) -> Dict[str, Any]:
    values = np.asarray(log_area, dtype=float)
    values = values[np.isfinite(values)]
    counts, edges = np.histogram(values, bins=rule)

    idx = int(np.argmax(counts))
    log_roll = float((edges[idx] + edges[idx + 1]) / 2.0)
    area = float(10 ** log_roll)

    return {
        "method": f"Histogram modal bin ({rule})",
        "rollover_log10_area_m2": log_roll,
        "rollover_area_m2": area,
        "rollover_area_ha": area / 10000.0,
        "n_bins": int(len(counts)),
        "modal_bin_count": int(counts[idx]),
    }


def bootstrap_rollovers(log_area: np.ndarray) -> np.ndarray:
    values = np.asarray(log_area, dtype=float)
    values = values[np.isfinite(values)]
    rng = np.random.default_rng(Config.RANDOM_SEED)
    estimates = []

    log(f"[ROLLOVER] Running {Config.N_BOOTSTRAP} bootstrap resamples...")

    for i in range(Config.N_BOOTSTRAP):
        sample = rng.choice(values, size=len(values), replace=True)
        try:
            estimates.append(kde_rollover(sample, 1.0)["rollover_area_m2"])
        except Exception:
            pass

        if (i + 1) % 100 == 0:
            log(f"[ROLLOVER] Bootstrap {i + 1}/{Config.N_BOOTSTRAP}")

    estimates = np.asarray(estimates, dtype=float)

    if len(estimates) < max(100, int(0.8 * Config.N_BOOTSTRAP)):
        raise RuntimeError(
            f"Too many bootstrap failures: {len(estimates)}/{Config.N_BOOTSTRAP}"
        )

    return estimates


def area_frequency_table(area_m2: np.ndarray) -> pd.DataFrame:
    area = np.asarray(area_m2, dtype=float)
    area = area[np.isfinite(area) & (area > 0)]
    log_area = np.log10(area)

    edges = np.histogram_bin_edges(log_area, bins="fd")
    if len(edges) < 6:
        edges = np.linspace(log_area.min(), log_area.max(), 11)

    counts, _ = np.histogram(log_area, bins=edges)
    rows = []

    for i, count in enumerate(counts):
        lo, hi = float(edges[i]), float(edges[i + 1])
        width = hi - lo
        rows.append({
            "bin_index": i + 1,
            "lower_log10_area_m2": lo,
            "upper_log10_area_m2": hi,
            "lower_area_m2": 10 ** lo,
            "upper_area_m2": 10 ** hi,
            "geometric_mid_area_m2": 10 ** ((lo + hi) / 2.0),
            "count": int(count),
            "fraction_of_inventory": float(count / max(counts.sum(), 1)),
            "count_per_log10_area_unit": float(count / width) if width > 0 else np.nan,
        })

    return pd.DataFrame(rows)


def summarize_group(df: pd.DataFrame, name: str, rollover: float) -> Dict[str, Any]:
    values = df["polygon_area_m2"].to_numpy(dtype=float)
    values = values[np.isfinite(values) & (values > 0)]

    if not len(values):
        return {"group": name, "n": 0}

    q25, q75 = np.percentile(values, [25, 75])
    above = values >= rollover

    return {
        "group": name,
        "n": int(len(values)),
        "min_area_m2": float(values.min()),
        "q25_area_m2": float(q25),
        "median_area_m2": float(np.median(values)),
        "q75_area_m2": float(q75),
        "iqr_area_m2": float(q75 - q25),
        "mean_area_m2": float(values.mean()),
        "max_area_m2": float(values.max()),
        "n_at_or_above_rollover": int(above.sum()),
        "pct_at_or_above_rollover": float(100 * above.mean()),
        "n_below_rollover": int((~above).sum()),
        "pct_below_rollover": float(100 * (~above).mean()),
    }


def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    differences = x[:, None] - y[None, :]
    return float(
        (np.sum(differences > 0) - np.sum(differences < 0)) /
        (len(x) * len(y))
    )


def delta_magnitude(delta: float) -> str:
    a = abs(delta)
    if a < 0.147:
        return "negligible"
    if a < 0.33:
        return "small"
    if a < 0.474:
        return "medium"
    return "large"


def retained_excluded_tests(df: pd.DataFrame, rollover: float) -> pd.DataFrame:
    retained = df.loc[
        df["ml_sample_group"] == "ML-retained", "polygon_area_m2"
    ].dropna().to_numpy(dtype=float)
    excluded = df.loc[
        df["ml_sample_group"] == "Excluded", "polygon_area_m2"
    ].dropna().to_numpy(dtype=float)

    retained = retained[retained > 0]
    excluded = excluded[excluded > 0]

    mw = mannwhitneyu(retained, excluded, alternative="two-sided")
    ks = ks_2samp(retained, excluded, alternative="two-sided", method="auto")
    delta = cliffs_delta(retained, excluded)

    table = np.array([
        [np.sum(retained >= rollover), np.sum(retained < rollover)],
        [np.sum(excluded >= rollover), np.sum(excluded < rollover)],
    ], dtype=int)

    fisher = fisher_exact(table, alternative="two-sided")

    return pd.DataFrame([
        {
            "test": "Mann-Whitney U",
            "statistic": float(mw.statistic),
            "p_value": float(mw.pvalue),
            "effect_size": delta,
            "effect_size_name": "Cliff's delta; positive means retained polygons tend to be larger",
            "effect_size_magnitude": delta_magnitude(delta),
        },
        {
            "test": "Kolmogorov-Smirnov",
            "statistic": float(ks.statistic),
            "p_value": float(ks.pvalue),
            "effect_size": np.nan,
            "effect_size_name": "",
            "effect_size_magnitude": "",
        },
        {
            "test": "Fisher exact: retention × above/below rollover",
            "statistic": float(fisher.statistic),
            "p_value": float(fisher.pvalue),
            "effect_size": np.nan,
            "effect_size_name": "Odds ratio",
            "effect_size_magnitude": "",
        },
    ])


def ecdf(values: np.ndarray):
    values = np.asarray(values, dtype=float)
    values = np.sort(values[np.isfinite(values) & (values > 0)])
    return values, np.arange(1, len(values) + 1) / len(values)


def make_figures(
    df: pd.DataFrame,
    freq: pd.DataFrame,
    rollover: float,
    ci_low: float,
    ci_high: float,
) -> None:
    mpl.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 12,          # Base text size
        "axes.labelsize": 13,     # X- and Y-axis label size
        "axes.labelweight": "bold",
        "axes.titlesize": 12,     # Panel-title size
        "axes.titleweight": "bold",
        "xtick.labelsize": 11,    # X-axis tick-label size
        "ytick.labelsize": 11,    # Y-axis tick-label size
        "legend.fontsize": 10,    # Legend text size
    })

    areas = df["polygon_area_m2"].dropna().to_numpy(dtype=float)
    areas = areas[areas > 0]

    edges = np.r_[
        freq["lower_area_m2"].to_numpy(dtype=float),
        float(freq["upper_area_m2"].iloc[-1]),
    ]

    def draw_a(ax):
        ax.hist(areas, bins=edges, alpha=0.75, edgecolor="black", linewidth=0.5)
        ax.axvspan(ci_low, ci_high, alpha=0.15, label="Bootstrap 95% interval")
        ax.axvline(rollover, linestyle="--", linewidth=1.2, label="Empirical rollover")
        ax.set_xscale("log")
        ax.set_xlabel("Landslide polygon area (m²)", fontsize=13, fontweight="bold")
        ax.set_ylabel("Frequency", fontsize=13, fontweight="bold")
        ax.tick_params(axis="both", which="major", labelsize=11)
        ax.grid(True, axis="y", alpha=0.25, linewidth=0.5)
        ax.legend(frameon=False)

    def draw_b(ax):
        retained = df.loc[
            df["ml_sample_group"] == "ML-retained", "polygon_area_m2"
        ].dropna().to_numpy(dtype=float)
        excluded = df.loc[
            df["ml_sample_group"] == "Excluded", "polygon_area_m2"
        ].dropna().to_numpy(dtype=float)

        xr, yr = ecdf(retained)
        xe, ye = ecdf(excluded)

        ax.step(xr, yr, where="post", linewidth=1.4,
                label=f"ML-retained (n={len(xr)})")
        ax.step(xe, ye, where="post", linewidth=1.4, linestyle="--",
                label=f"Excluded (n={len(xe)})")
        ax.axvline(rollover, linestyle=":", linewidth=1.2,
                   label="Empirical rollover")
        ax.set_xscale("log")
        ax.set_xlabel("Landslide polygon area (m²)", fontsize=13, fontweight="bold")
        ax.set_ylabel("Empirical cumulative probability", fontsize=13, fontweight="bold")
        ax.tick_params(axis="both", which="major", labelsize=11)
        ax.set_ylim(0, 1.02)
        ax.grid(True, alpha=0.25, linewidth=0.5)
        ax.legend(frameon=False)

    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))
    draw_a(axes[0])
    draw_b(axes[1])
    axes[0].set_title("(a) Inventory area–frequency distribution")
    axes[1].set_title("(b) ML-retained versus excluded polygons")
    fig.tight_layout()
    fig.savefig(Config.FIGURE_COMBINED_PNG, dpi=Config.SAVE_DPI,
                bbox_inches="tight", facecolor="white")
    fig.savefig(Config.FIGURE_COMBINED_PDF, bbox_inches="tight", facecolor="white")
    fig.savefig(Config.FIGURE_COMBINED_SVG, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(4.2, 3.2))
    draw_a(ax)
    fig.tight_layout()
    fig.savefig(Config.FIGURE_A_PNG, dpi=Config.SAVE_DPI,
                bbox_inches="tight", facecolor="white")
    fig.savefig(Config.FIGURE_A_PDF, bbox_inches="tight", facecolor="white")
    fig.savefig(Config.FIGURE_A_SVG, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(4.2, 3.2))
    draw_b(ax)
    fig.tight_layout()
    fig.savefig(Config.FIGURE_B_PNG, dpi=Config.SAVE_DPI,
                bbox_inches="tight", facecolor="white")
    fig.savefig(Config.FIGURE_B_PDF, bbox_inches="tight", facecolor="white")
    fig.savefig(Config.FIGURE_B_SVG, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

    mpl.rcParams.update(mpl.rcParamsDefault)


def write_report(
    summary: Dict[str, Any],
    group_stats: pd.DataFrame,
    status_stats: pd.DataFrame,
    tests: pd.DataFrame,
) -> None:
    rollover = summary["primary_rollover"]

    with open(Config.QA_REPORT_TXT, "w", encoding="utf-8") as f:
        f.write("=" * 92 + "\n")
        f.write("STEP 6.6 — INVENTORY COMPLETENESS AND ML-SAMPLE REPRESENTATION AUDIT\n")
        f.write("=" * 92 + "\n\n")
        f.write(f"Generated UTC: {summary['generated_at_utc']}\n")
        f.write(f"Analysis CRS: EPSG:{Config.TARGET_EPSG}\n\n")

        f.write("INVENTORY COUNTS\n" + "-" * 72 + "\n")
        for k, v in summary["inventory_counts"].items():
            f.write(f"{k}: {v}\n")

        f.write("\nSTEP 6.3 STATUS COUNTS\n" + "-" * 72 + "\n")
        for k, v in summary["reconciliation_status_counts"].items():
            f.write(f"{k}: {v}\n")

        f.write("\nEMPIRICAL ROLLOVER\n" + "-" * 72 + "\n")
        f.write(f"Primary method: {rollover['method']}\n")
        f.write(
            f"Rollover: {rollover['area_m2']:.3f} m² "
            f"({rollover['area_ha']:.6f} ha)\n"
        )
        f.write(
            f"Bootstrap 95% interval: "
            f"{rollover['bootstrap_ci_low_m2']:.3f}–"
            f"{rollover['bootstrap_ci_high_m2']:.3f} m²\n"
        )
        f.write(
            f"KDE mode near observed range edge: "
            f"{rollover['mode_near_observed_range_edge']}\n"
        )

        f.write("\nINTERPRETATION WARNING\n" + "-" * 72 + "\n")
        f.write(
            "The rollover is an empirical indicator of approximate size-dependent "
            "mapping completeness. It is not a hard detection limit and does not "
            "prove that every landslide above the threshold was mapped.\n"
        )

        f.write("\nML-RETAINED VS EXCLUDED\n" + "-" * 72 + "\n")
        f.write(group_stats.to_string(index=False))

        f.write("\n\nBY RECONCILIATION STATUS\n" + "-" * 72 + "\n")
        f.write(status_stats.to_string(index=False))

        f.write("\n\nSIZE-BIAS TESTS\n" + "-" * 72 + "\n")
        f.write(tests.to_string(index=False))

        f.write("\n\nDECISION NOTE\n" + "-" * 72 + "\n")
        f.write(
            "Do not automatically remove below-rollover landslides. Use the "
            "proportion of ML-retained sites above/below the rollover together "
            "with the retained-versus-excluded distribution tests to decide "
            "whether a separate completeness-filtered ML sensitivity experiment "
            "is scientifically warranted.\n"
        )


def main() -> None:
    log("=== Step 6.6: Inventory Completeness and ML-Sample Representation Audit ===")
    Config.OUT_DIR.mkdir(parents=True, exist_ok=True)

    target_crs = CRS.from_epsg(Config.TARGET_EPSG)

    train, train_audit = read_inventory(
        Config.TRAIN_SHP, Config.TRAIN_SOURCE_TAG,
        Config.TRAIN_SITE_PREFIX, target_crs
    )
    valid, valid_audit = read_inventory(
        Config.VALID_SHP, Config.VALID_SOURCE_TAG,
        Config.VALID_SITE_PREFIX, target_crs
    )

    inventory = pd.concat([train, valid], ignore_index=True)

    log(
        f"[INVENTORY] TRAIN={len(train)}, VALID={len(valid)}, TOTAL={len(inventory)}"
    )

    if len(train) != Config.EXPECTED_TRAIN:
        log(f"[WARN] Expected {Config.EXPECTED_TRAIN} TRAIN records, found {len(train)}.")
    if len(valid) != Config.EXPECTED_VALID:
        log(f"[WARN] Expected {Config.EXPECTED_VALID} VALID records, found {len(valid)}.")
    if len(inventory) != Config.EXPECTED_TOTAL:
        log(f"[WARN] Expected {Config.EXPECTED_TOTAL} total records, found {len(inventory)}.")

    reconciliation = load_reconciliation()
    merged = merge_reconciliation(inventory, reconciliation)

    status_counts = merged["status"].value_counts().to_dict()
    group_counts = merged["ml_sample_group"].value_counts().to_dict()

    log(f"[RECONCILIATION] status={status_counts}")
    log(f"[RECONCILIATION] groups={group_counts}")

    valid_area = np.isfinite(merged["polygon_area_m2"]) & (merged["polygon_area_m2"] > 0)
    analysis = merged.loc[valid_area].copy()

    if len(analysis) < 30:
        raise RuntimeError("Too few valid polygon areas for the completeness audit.")

    invalid_area_count = int((~valid_area).sum())
    if invalid_area_count:
        log(f"[WARN] {invalid_area_count} records have no valid positive polygon area.")

    log_area = analysis["log10_polygon_area_m2"].to_numpy(dtype=float)

    primary = kde_rollover(log_area, 1.0)
    rollover = float(primary["rollover_area_m2"])
    log(f"[ROLLOVER] Primary estimate={rollover:.3f} m²")

    boot = bootstrap_rollovers(log_area)
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    pd.DataFrame({
        "bootstrap_iteration": np.arange(1, len(boot) + 1),
        "rollover_area_m2": boot,
        "rollover_area_ha": boot / 10000.0,
        "log10_rollover_area_m2": np.log10(boot),
    }).to_csv(Config.BOOTSTRAP_CSV, index=False, float_format="%.10f")

    sensitivity_rows = []
    for factor in Config.KDE_BANDWIDTH_FACTORS:
        result = kde_rollover(log_area, factor)
        sensitivity_rows.append({
            "estimator_family": "KDE",
            "method": result["method"],
            "parameter": f"bandwidth_factor={factor:.2f}",
            "rollover_area_m2": result["rollover_area_m2"],
            "rollover_area_ha": result["rollover_area_ha"],
            "is_primary": bool(math.isclose(factor, 1.0)),
        })

    for rule in Config.HISTOGRAM_RULES:
        result = histogram_rollover(log_area, rule)
        sensitivity_rows.append({
            "estimator_family": "Histogram",
            "method": result["method"],
            "parameter": f"bin_rule={rule}",
            "rollover_area_m2": result["rollover_area_m2"],
            "rollover_area_ha": result["rollover_area_ha"],
            "is_primary": False,
        })

    sensitivity = pd.DataFrame(sensitivity_rows)
    sensitivity.to_csv(
        Config.ROLLOVER_SENSITIVITY_CSV, index=False, float_format="%.10f"
    )

    merged["empirical_rollover_area_m2"] = rollover
    merged["relative_to_empirical_rollover"] = np.where(
        ~np.isfinite(merged["polygon_area_m2"]),
        "invalid_area",
        np.where(
            merged["polygon_area_m2"] >= rollover,
            "at_or_above_rollover",
            "below_rollover",
        ),
    )
    merged.to_csv(Config.INVENTORY_AREA_CSV, index=False, float_format="%.10f")

    freq = area_frequency_table(analysis["polygon_area_m2"].to_numpy(dtype=float))
    freq["empirical_rollover_area_m2"] = rollover
    freq.to_csv(Config.FREQUENCY_CSV, index=False, float_format="%.10f")

    group_rows = [
        summarize_group(analysis, "All mapped polygons", rollover),
        summarize_group(
            analysis.loc[analysis["ml_sample_group"] == "ML-retained"],
            "ML-retained", rollover
        ),
        summarize_group(
            analysis.loc[analysis["ml_sample_group"] == "Excluded"],
            "Excluded", rollover
        ),
    ]
    group_stats = pd.DataFrame(group_rows)
    group_stats.to_csv(Config.GROUP_STATS_CSV, index=False, float_format="%.10f")

    status_rows = [
        summarize_group(subset, str(status), rollover)
        for status, subset in analysis.groupby("status", dropna=False)
    ]
    status_stats = pd.DataFrame(status_rows)
    status_stats.to_csv(Config.STATUS_STATS_CSV, index=False, float_format="%.10f")

    tests = retained_excluded_tests(analysis, rollover)
    tests.to_csv(Config.BIAS_TESTS_CSV, index=False, float_format="%.10f")

    all_summary = group_rows[0]
    retained_summary = group_rows[1]
    excluded_summary = group_rows[2]

    summary = {
        "step": "Step 6.6 — Inventory Completeness and ML-Sample Representation Audit",
        "generated_at_utc": utc_now_iso(),
        "analysis_crs": f"EPSG:{Config.TARGET_EPSG}",
        "inputs": {
            "training_inventory": Config.TRAIN_SHP,
            "validation_inventory": Config.VALID_SHP,
            "step63_reconciliation": Config.RECONCILIATION_CSV,
        },
        "inventory_audit": {
            "training": train_audit,
            "validation": valid_audit,
        },
        "inventory_counts": {
            "training_retained": len(train),
            "validation_retained": len(valid),
            "combined_retained": len(inventory),
            "valid_area_polygons": len(analysis),
            "invalid_area_polygons": invalid_area_count,
        },
        "reconciliation_status_counts": {
            str(k): int(v) for k, v in status_counts.items()
        },
        "ml_group_counts": {
            str(k): int(v) for k, v in group_counts.items()
        },
        "primary_rollover": {
            "method": primary["method"],
            "area_m2": rollover,
            "area_ha": rollover / 10000.0,
            "log10_area_m2": primary["rollover_log10_area_m2"],
            "bootstrap_ci_low_m2": float(ci_low),
            "bootstrap_ci_high_m2": float(ci_high),
            "bootstrap_ci_low_ha": float(ci_low / 10000.0),
            "bootstrap_ci_high_ha": float(ci_high / 10000.0),
            "bootstrap_successful_fits": len(boot),
            "bootstrap_requested_fits": Config.N_BOOTSTRAP,
            "mode_near_observed_range_edge": primary["mode_near_observed_range_edge"],
        },
        "all_mapped_polygon_summary": all_summary,
        "ml_retained_summary": retained_summary,
        "excluded_summary": excluded_summary,
        "interpretation_note": (
            "The empirical rollover is an approximate indicator of size-dependent "
            "inventory completeness, not a hard detection limit."
        ),
        "workflow_note": (
            "This step does not modify the 165-site ML dataset. Decide on a "
            "completeness-filtered ML sensitivity experiment only after reviewing "
            "the rollover and retained-versus-excluded size-bias results."
        ),
    }

    with open(Config.SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(json_safe(summary), f, indent=2)

    make_figures(analysis, freq, rollover, float(ci_low), float(ci_high))
    write_report(summary, group_stats, status_stats, tests)

    log(
        f"[RESULT] Rollover={rollover:.3f} m²; "
        f"bootstrap 95% interval={ci_low:.3f}–{ci_high:.3f} m²"
    )
    log(
        "[RESULT] All mapped at/above rollover="
        f"{all_summary['n_at_or_above_rollover']}/{all_summary['n']} "
        f"({all_summary['pct_at_or_above_rollover']:.2f}%)"
    )
    log(
        "[RESULT] ML-retained at/above rollover="
        f"{retained_summary['n_at_or_above_rollover']}/{retained_summary['n']} "
        f"({retained_summary['pct_at_or_above_rollover']:.2f}%)"
    )
    log(
        "[RESULT] Excluded at/above rollover="
        f"{excluded_summary['n_at_or_above_rollover']}/{excluded_summary['n']} "
        f"({excluded_summary['pct_at_or_above_rollover']:.2f}%)"
    )

    log(f"[OUT] {Config.INVENTORY_AREA_CSV}")
    log(f"[OUT] {Config.FREQUENCY_CSV}")
    log(f"[OUT] {Config.ROLLOVER_SENSITIVITY_CSV}")
    log(f"[OUT] {Config.BOOTSTRAP_CSV}")
    log(f"[OUT] {Config.GROUP_STATS_CSV}")
    log(f"[OUT] {Config.BIAS_TESTS_CSV}")
    log(f"[OUT] {Config.SUMMARY_JSON}")
    log(f"[OUT] {Config.QA_REPORT_TXT}")
    log(f"[OUT] {Config.FIGURE_COMBINED_PNG}")
    log(f"[OUT] {Config.FIGURE_COMBINED_PDF}")
    log(f"[OUT] {Config.FIGURE_COMBINED_SVG}")
    log(f"[OUT] {Config.FIGURE_A_PNG}")
    log(f"[OUT] {Config.FIGURE_A_PDF}")
    log(f"[OUT] {Config.FIGURE_A_SVG}")
    log(f"[OUT] {Config.FIGURE_B_PNG}")
    log(f"[OUT] {Config.FIGURE_B_PDF}")
    log(f"[OUT] {Config.FIGURE_B_SVG}")
    log("=== Step 6.6 completed successfully ===")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 6.6 failed: {exc}")
        traceback.print_exc()
        raise


# Phase-7:Susceptibility ML modeling

# Step 7.1 — XGBoost Models A–F with Nested Spatial Cross-Validation

In [ ]:
# =============================================================================
# Step 7.1 — XGBoost Models A–F with Nested Spatial Cross-Validation
# =============================================================================
#
# Purpose
# -------
# Train and evaluate XGBoost for all six frozen A–F predictor configurations
# using the primary frozen Step 6.4 ML dataset and EXACT frozen five spatial folds.
#
# Design
# ------
# Outer CV (unbiased evaluation):
#   - One frozen spatial block is held out.
#   - The remaining four blocks are used for all model selection.
#
# Inner CV (hyperparameter tuning):
#   - Each of the four remaining spatial blocks serves once as the inner
#     validation fold via PredefinedSplit.
#   - Grid-search AUC is computed only inside the outer-training data.
#
# After nested-CV evaluation:
#   - OOF predictions are pooled across the five outer folds.
#   - A separate full-data spatial grid search can be enabled later to select
#     final deployment/map models. These full-data models are NOT used for
#     validation statistics.
#
# Important scientific rules
# --------------------------
# 1. Never regenerate spatial folds. Use Step 6.4 cv_fold exactly.
# 2. Models A–F use identical rows, labels and outer folds.
# 3. No global imputation, scaling, downsampling or train/test split.
# 4. Land cover is treated as categorical and one-hot encoded inside each fit.
# 5. XGBoost continuous predictors are not standardized.
# 6. IRS is never used as an ML predictor.
# 7. Outer validation data are never used for tuning or early stopping.
# 8. Binary metrics use the fixed threshold 0.50. AUC remains primary.
# 9. Cluster-bootstrap confidence intervals resample the five spatial blocks,
#    preserving the spatial grouping structure better than sample-wise bootstrap.
# 10. Final susceptibility visualization uses the direct Model E deployment
#     probability raster; it never substitutes for nested spatial OOF validation.
#
# Primary reviewer comparisons supported
# --------------------------------------
# B - A : added value of raw DV+
# C - A : added value of HKFI alone
# D - A : added value of SNDR alone
# E - A : total added value of the proposed physics-guided indices
# F - A : added value of supplying DV+, TWI and SM as separate source variables
# E - B : physics-guided indices versus raw deformation alone
# E - F : transformed indices versus supplying source variables separately
# =============================================================================

from __future__ import annotations
import os
# =============================================================================
# CRITICAL FIX: Prevent CPU oversubscription when using GridSearchCV with XGBoost
# =============================================================================
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"


import json
import os
import re
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Sequence, Tuple

import joblib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Polygon, Rectangle
from matplotlib.ticker import FuncFormatter, LinearLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling,
    transform_geom,
)
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# Optional cartographic/vector libraries.
try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False


# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Step 6.1 frozen predictor definitions and raster stack.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"
    MASTER_STACK_TIF = (
        STEP61_DIR
        / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
    )
    COMMON_AF_MASK_TIF = (
        STEP61_DIR / "common_A_to_F_valid_mask_32645_12p5m_byte.tif"
    )

    # Step 6.4 frozen modelling contract.
    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    MODEL_READY_DATASET_CSV = STEP64_DIR / "model_ready_dataset_A_to_F.csv"
    STEP64_MODEL_CONFIG_JSON = STEP64_DIR / "model_configuration_manifest.json"

    # Outputs.
    OUT_DIR = RESULTS_DIR / "Step-7.1-XGB-Models_A_to_F"
    MODEL_DIR = OUT_DIR / "final_models"
    MAP_DIR = OUT_DIR / "susceptibility_maps"
    SEARCH_DIR = OUT_DIR / "grid_search_details"

    OOF_PREDICTIONS_CSV = OUT_DIR / "XGB_A_to_F_OOF_predictions.csv"
    FOLD_METRICS_CSV = OUT_DIR / "XGB_A_to_F_outer_fold_metrics.csv"
    SUMMARY_METRICS_CSV = OUT_DIR / "XGB_A_to_F_summary_metrics.csv"
    AUC_COMPARISONS_CSV = OUT_DIR / "XGB_A_to_F_paired_AUC_comparisons.csv"
    OUTER_PARAMS_JSON = OUT_DIR / "XGB_A_to_F_outer_fold_best_parameters.json"
    FINAL_PARAMS_JSON = OUT_DIR / "XGB_A_to_F_final_full_data_best_parameters.json"
    INNER_SEARCH_CSV = SEARCH_DIR / "XGB_A_to_F_nested_inner_grid_results.csv"
    FINAL_SEARCH_CSV = SEARCH_DIR / "XGB_A_to_F_full_data_grid_results.csv"
    ROC_FIG_PNG = OUT_DIR / "XGB_A_to_F_OOF_ROC_Curves.png"
    ROC_FIG_PDF = OUT_DIR / "XGB_A_to_F_OOF_ROC_Curves.pdf"
    ROC_FIG_SVG = OUT_DIR / "XGB_A_to_F_OOF_ROC_Curves.svg"
    FOLD_AUC_FIG_PNG = OUT_DIR / "XGB_A_to_F_Fold_AUC.png"
    FOLD_AUC_FIG_PDF = OUT_DIR / "XGB_A_to_F_Fold_AUC.pdf"
    FOLD_AUC_FIG_SVG = OUT_DIR / "XGB_A_to_F_Fold_AUC.svg"
    QA_REPORT_TXT = OUT_DIR / "Step_7_1_XGB_A_to_F_QA_Report.txt"
    METADATA_JSON = OUT_DIR / "Step_7_1_XGB_A_to_F_Metadata.json"

    # Manuscript-ready Excel workbook containing the compact A-F performance
    # table, paired AUC comparisons, fold-level AUCs and ensemble summary.
    MANUSCRIPT_TABLES_XLSX = (
        OUT_DIR / "XGB_A_to_F_Manuscript_Performance_Tables.xlsx"
    )

    # Cartographic overlays used by the established Step 5.3 map style.
    STUDY_SHP = BASE / "Study_Area.shp"
    ROAD_SHP = BASE / "NTD_Road.shp"
    VISUALIZATION_DIR = MAP_DIR / "visualization"

    # Primary A-F model retained for scientific interpretation.
    PRIMARY_VISUALIZATION_SCENARIO = "E_physics_guided"
    GENERATE_AE_ENSEMBLE_LSM = True
    ENSEMBLE_STATIC_SCENARIO = "A_static_baseline"
    ENSEMBLE_ENHANCED_SCENARIO = "E_physics_guided"
    ENSEMBLE_ENHANCED_WEIGHT = 0.70

    ENSEMBLE_STATIC_FULL_DOMAIN_TIF = (
        MAP_DIR / "A-static-baseline_XGB_probability_full_available_domain.tif"
    )
    ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF = (
        MAP_DIR / "E-physics-guided_XGB_probability_full_available_domain.tif"
    )
    ENSEMBLE_PROBABILITY_TIF = (
        MAP_DIR / "XGB_Ensemble_A30_E70_probability_full_study_domain.tif"
    )
    ENSEMBLE_SUPPORT_TIF = (
        MAP_DIR / "XGB_Ensemble_A30_E70_information_support.tif"
    )
    ENSEMBLE_OOF_PREDICTIONS_CSV = (
        OUT_DIR / "XGB_Ensemble_A30_E70_OOF_predictions.csv"
    )
    ENSEMBLE_OOF_METRICS_JSON = (
        OUT_DIR / "XGB_Ensemble_A30_E70_OOF_metrics.json"
    )

    # Step 5.3-style map/histogram settings.
    # Publication-size figures.
    # The two requested maps are explicitly controlled independently so that
    # later changes to other figures cannot accidentally alter their dimensions.
    ENSEMBLE_MAP_FIGSIZE = (4, 6)
    SUPPORT_MAP_FIGSIZE = (4, 6)

    # Other visualization products.
    MAP_FIGSIZE = (4, 4)
    HIST_FIGSIZE = (4, 4)
    AREA_PERCENT_FIGSIZE = (4, 4)

    # Enlarged cartographic text for the 4 x 4 inch figure.
    DMS_TICK_FONTSIZE = 7  # Reduced from 9 to prevent latitude label overlap
    LEGEND_FONTSIZE = 7
    LEGEND_TITLE_FONTSIZE = 8
    COLORBAR_TICK_FONTSIZE = 8
    COLORBAR_LABEL_FONTSIZE = 9
    NORTH_ARROW_FONTSIZE = 8  # Slightly reduced for balance
    SCALEBAR_FONTSIZE = 7

    CAX_SIZE = "5%"
    CAX_PAD = 0.01
    FIG_DPI = 900
    DISPLAY_BUFFER_FRACTION = 0.00
    LONGITUDE_DMS_TICK_COUNT = 4
    RASTER_DISPLAY_INTERPOLATION = "nearest"
    SHOW_LSM_FIGURES = True
    SAVE_LSM_FIGURES = True

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"
    CATEGORICAL_FEATURE_ALIASES = {"landcover", "landuse", "land_use"}

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    MODEL_DESCRIPTIONS = {
        "A_static_baseline": "Static baseline",
        "B_raw_deformation": "Static + DV+",
        "C_hkfi_only": "Static + HKFI",
        "D_sndr_only": "Static + SNDR",
        "E_physics_guided": "Static + HKFI + SNDR",
        "F_untransformed_sources": "Static + DV+ + TWI + SM",
    }

    # Compact, reproducible grid. This is intentionally not enormous because
    # nested spatial CV already fits the grid 5 outer folds x 6 models.
    PARAM_GRID = {
        "model__n_estimators": [200, 500],
        "model__max_depth": [2, 3, 4],
        "model__learning_rate": [0.03, 0.07],
        "model__min_child_weight": [1.0, 3.0],
        "model__reg_lambda": [1.0, 5.0],
    }

    # Fixed XGBoost settings shared by A–F.
    XGB_FIXED_PARAMS = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "reg_alpha": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": 1.0,
        "random_state": 42,
        # Keep one thread per fit; GridSearchCV parallelizes candidate fits.
        "n_jobs": 1,
    }

    GRIDSEARCH_N_JOBS = 1  # CRITICAL FIX: Changed from -1 to prevent oversubscription deadlock
    GRIDSEARCH_VERBOSE = 2  # Increased from 1 to show per-fold progress
    RANDOM_STATE = 42
    BINARY_THRESHOLD = 0.50

    # Spatial-block cluster bootstrap.
    N_CLUSTER_BOOTSTRAP = 5000
    CI_LEVEL = 0.95

    # Complete execution controls.
    #
    # True  -> after nested spatial-CV evaluation, tune and fit the final
    #          full-data deployment models for A-F.
    # False -> stop after leakage-safe nested spatial OOF evaluation.
    RUN_FINAL_DEPLOYMENT_TUNING = True

    # True  -> generate A-F full-domain probability rasters from the final
    #          deployment models. Requires RUN_FINAL_DEPLOYMENT_TUNING=True.
    GENERATE_RASTER_MAPS = True

    # True  -> create the publication-style primary Model E landslide
    #          susceptibility map and probability histogram.
    GENERATE_PRIMARY_LSM_VISUALIZATION = True
    RASTER_BLOCK_ROWS = 512
    OUTPUT_NODATA = -9999.0

    # Keep nested-CV fold estimators only if needed for later diagnostics.
    SAVE_OUTER_FOLD_MODELS = False


# Create output folders once the script is imported/run.
for _directory in (
    Config.OUT_DIR,
    Config.MODEL_DIR,
    Config.MAP_DIR,
    Config.SEARCH_DIR,
    Config.VISUALIZATION_DIR,
):
    _directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")



# =============================================================================
# FIGURE STYLE HELPERS
# =============================================================================
AXIS_LABEL_FONTSIZE = 11
TICK_LABEL_FONTSIZE = 10
LEGEND_FONTSIZE = 9
TITLE_FONTSIZE = 11

def apply_axis_text_style(ax: plt.Axes, legend_fontsize: float = LEGEND_FONTSIZE) -> None:
    """Apply bold axis labels, bold tick labels, and slightly larger in-figure text."""
    ax.xaxis.label.set_fontsize(AXIS_LABEL_FONTSIZE)
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontsize(AXIS_LABEL_FONTSIZE)
    ax.yaxis.label.set_fontweight("bold")
    ax.title.set_fontsize(TITLE_FONTSIZE)
    ax.title.set_fontweight("bold")
    ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_FONTSIZE)
    for tick_label in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        tick_label.set_fontsize(TICK_LABEL_FONTSIZE)
        tick_label.set_fontweight("bold")
    legend = ax.get_legend()
    if legend is not None:
        for label in legend.get_texts():
            label.set_fontsize(legend_fontsize)

def save_figure_triplet(
    fig: plt.Figure,
    png_path: Path,
    pdf_path: Path,
    svg_path: Path,
    dpi: int = 600,
) -> None:
    fig.tight_layout()
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    fig.savefig(svg_path, bbox_inches="tight", facecolor="white", format="svg")
    plt.close(fig)


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json(path: Path) -> Dict[str, Any]:
    require_file(path, "JSON file")
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


def scenario_prefix(scenario_name: str) -> str:
    return scenario_name.replace("_", "-")


def normalize_feature_name(name: str) -> str:
    """Normalize a feature name for robust alias matching."""
    return re.sub(r"[^a-z0-9_]+", "_", str(name).strip().lower()).strip("_")


def is_categorical_feature(name: str) -> bool:
    """Identify categorical predictors using conservative land-cover aliases."""
    normalized = normalize_feature_name(name)
    compact = normalized.replace("_", "")
    aliases = {alias.lower() for alias in Config.CATEGORICAL_FEATURE_ALIASES}
    compact_aliases = {alias.replace("_", "") for alias in aliases}
    return normalized in aliases or compact in compact_aliases


# =============================================================================
# DATA / MANIFEST VALIDATION
# =============================================================================
def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """Cross-check Step 6.1 and frozen Step 6.4 A–F definitions."""
    step61 = load_json(Config.SCENARIO_MANIFEST_JSON).get("scenarios", {})
    step64 = load_json(Config.STEP64_MODEL_CONFIG_JSON).get("scenarios", {})

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                f"Step 6.1 scenario manifest is missing {scenario_name}."
            )
        if scenario_name not in step64:
            raise RuntimeError(
                f"Step 6.4 frozen model manifest is missing {scenario_name}."
            )

        features_61 = list(step61[scenario_name].get("features", []))
        features_64 = list(step64[scenario_name].get("features", []))

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any("irs" in feature.lower() for feature in features_61):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; IRS must not be an ML predictor."
            )

    log("[QA] Step 6.1 and Step 6.4 A–F scenario definitions match: PASS")
    return step64


def load_and_validate_model_ready_dataset(
    scenarios: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    """Load the primary frozen 330-row Step 6.4 ML dataset with strict QA."""
    require_file(
        Config.MODEL_READY_DATASET_CSV,
        "Step 6.4 primary frozen model-ready ML dataset",
    )

    df = pd.read_csv(Config.MODEL_READY_DATASET_CSV)

    required_metadata = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ]

    all_features: List[str] = []
    for scenario_name in Config.SCENARIO_ORDER:
        for feature in scenarios[scenario_name]["features"]:
            if feature not in all_features:
                all_features.append(feature)

    missing = [
        column
        for column in required_metadata + all_features
        if column not in df.columns
    ]
    if missing:
        raise RuntimeError(f"Model-ready dataset is missing columns: {missing}")

    if len(df) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Expected {Config.EXPECTED_N_ROWS} rows; found {len(df)}. "
            "No silent row dropping is allowed."
        )

    if df[Config.SAMPLE_ID_COLUMN].isna().any():
        raise RuntimeError("Missing sample_id values found.")
    if df[Config.SAMPLE_ID_COLUMN].duplicated().any():
        raise RuntimeError("Duplicated sample_id values found.")

    df[Config.TARGET_COLUMN] = pd.to_numeric(
        df[Config.TARGET_COLUMN], errors="raise"
    ).astype(int)
    df[Config.FOLD_COLUMN] = pd.to_numeric(
        df[Config.FOLD_COLUMN], errors="raise"
    ).astype(int)
    df[Config.BLOCK_COLUMN] = pd.to_numeric(
        df[Config.BLOCK_COLUMN], errors="raise"
    ).astype(int)

    class_counts = df[Config.TARGET_COLUMN].value_counts().sort_index().to_dict()
    expected_counts = {
        0: Config.EXPECTED_N_NEGATIVE,
        1: Config.EXPECTED_N_POSITIVE,
    }
    if class_counts != expected_counts:
        raise RuntimeError(
            f"Unexpected class counts: {class_counts}; expected {expected_counts}."
        )

    folds = sorted(df[Config.FOLD_COLUMN].unique().tolist())
    if folds != Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected frozen folds {Config.EXPECTED_FOLDS}; found {folds}."
        )

    if not np.array_equal(
        df[Config.FOLD_COLUMN].to_numpy(),
        df[Config.BLOCK_COLUMN].to_numpy(),
    ):
        raise RuntimeError("cv_fold and spatial_block are not identical.")

    # Every frozen block must contain exactly 33 positives and 33 pseudo-absences.
    block_class = pd.crosstab(
        df[Config.FOLD_COLUMN],
        df[Config.TARGET_COLUMN],
    )
    for required_class in (0, 1):
        if required_class not in block_class.columns:
            raise RuntimeError(
                f"Frozen fold table is missing class {required_class}."
            )

    expected_per_class_per_fold = Config.EXPECTED_N_POSITIVE // len(Config.EXPECTED_FOLDS)
    if expected_per_class_per_fold * len(Config.EXPECTED_FOLDS) != Config.EXPECTED_N_POSITIVE:
        raise RuntimeError(
            "Expected positive count is not divisible evenly across the frozen folds."
        )

    for fold in Config.EXPECTED_FOLDS:
        if fold not in block_class.index:
            raise RuntimeError(f"Missing fold {fold}.")
        n0 = int(block_class.loc[fold, 0])
        n1 = int(block_class.loc[fold, 1])
        if n0 != expected_per_class_per_fold or n1 != expected_per_class_per_fold:
            raise RuntimeError(
                f"Fold {fold} violates the frozen Step 6.4 contract: "
                f"negative={n0}, positive={n1}; expected "
                f"{expected_per_class_per_fold}+{expected_per_class_per_fold}=66."
            )

    log(
        "[QA] Frozen fold balance: each fold contains "
        f"{expected_per_class_per_fold} positives + "
        f"{expected_per_class_per_fold} pseudo-absences: PASS"
    )

    # Step 6.4 should already guarantee complete predictors; verify again.
    predictor_values = df[all_features].apply(pd.to_numeric, errors="coerce")
    if not np.isfinite(predictor_values.to_numpy(dtype="float64")).all():
        bad = np.argwhere(
            ~np.isfinite(predictor_values.to_numpy(dtype="float64"))
        )
        examples = [
            (
                str(df.iloc[row_i][Config.SAMPLE_ID_COLUMN]),
                all_features[col_i],
            )
            for row_i, col_i in bad[:20]
        ]
        raise RuntimeError(
            "Missing/non-finite predictors remain in the model-ready dataset. "
            f"Examples: {examples}"
        )

    irs_columns = [
        column for column in df.columns if "irs" in str(column).lower()
    ]
    if irs_columns:
        raise RuntimeError(
            f"IRS-related columns found in ML dataset: {irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError("tpi_s25 found; only tpi_s5 is allowed.")

    log(
        "[QA] Model-ready dataset: "
        f"rows={len(df)}, positive={class_counts[1]}, negative={class_counts[0]}, "
        f"folds={folds}: PASS"
    )
    return df


# =============================================================================
# MODEL BUILDING / NESTED SPATIAL CV
# =============================================================================
def build_xgb_pipeline(features: Sequence[str]) -> Pipeline:
    """Create fold-safe preprocessing + XGBoost pipeline."""
    categorical = [
        feature for feature in features if is_categorical_feature(feature)
    ]
    numeric = [
        feature for feature in features if not is_categorical_feature(feature)
    ]

    transformers = []
    if numeric:
        transformers.append(("numeric", "passthrough", numeric))
    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = XGBClassifier(**Config.XGB_FIXED_PARAMS)

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", classifier),
        ]
    )


def make_inner_predefined_split(outer_train_df: pd.DataFrame) -> PredefinedSplit:
    """
    Build a four-fold spatial inner CV from the four blocks remaining after one
    outer block has been held out.
    """
    blocks = sorted(outer_train_df[Config.FOLD_COLUMN].unique().tolist())
    if len(blocks) != 4:
        raise RuntimeError(
            f"Nested CV expected four inner blocks; found {blocks}."
        )

    block_to_test_fold = {
        block: inner_index for inner_index, block in enumerate(blocks)
    }
    test_fold = outer_train_df[Config.FOLD_COLUMN].map(block_to_test_fold).to_numpy()
    return PredefinedSplit(test_fold=test_fold)


def make_full_data_predefined_split(df: pd.DataFrame) -> PredefinedSplit:
    """Use the frozen five blocks for final deployment-model tuning."""
    test_fold = df[Config.FOLD_COLUMN].to_numpy(dtype=int) - 1
    return PredefinedSplit(test_fold=test_fold)


def classification_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = Config.BINARY_THRESHOLD,
) -> Dict[str, Any]:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def cluster_bootstrap_auc_ci(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> Tuple[float, float, int]:
    """Cluster bootstrap AUC by resampling whole spatial blocks."""
    rng = np.random.default_rng(seed)
    unique_blocks = np.unique(blocks)
    values: List[float] = []

    indices_by_block = {
        block: np.flatnonzero(blocks == block) for block in unique_blocks
    }

    for i in range(n_bootstrap):
        if (i + 1) % 1000 == 0:
            log(f"  Bootstrap progress: {i + 1}/{n_bootstrap}")
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(unique_blocks),
            replace=True,
        )
        sampled_indices = np.concatenate(
            [indices_by_block[block] for block in sampled_blocks]
        )
        y_b = y_true[sampled_indices]
        p_b = y_prob[sampled_indices]

        if np.unique(y_b).size < 2:
            continue

        values.append(float(roc_auc_score(y_b, p_b)))

    if not values:
        return float("nan"), float("nan"), 0

    alpha = (1.0 - ci_level) / 2.0
    low, high = np.quantile(values, [alpha, 1.0 - alpha])
    return float(low), float(high), len(values)


def paired_cluster_bootstrap_auc_difference(
    y_true: np.ndarray,
    prob_left: np.ndarray,
    prob_right: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> Tuple[float, float, float, int]:
    """
    Paired spatial-block bootstrap for AUC(left) - AUC(right), preserving the
    identical samples/folds used by all A–F models.
    """
    observed = float(
        roc_auc_score(y_true, prob_left) - roc_auc_score(y_true, prob_right)
    )

    rng = np.random.default_rng(seed)
    unique_blocks = np.unique(blocks)
    indices_by_block = {
        block: np.flatnonzero(blocks == block) for block in unique_blocks
    }
    differences: List[float] = []

    for i in range(n_bootstrap):
        if (i + 1) % 1000 == 0:
            log(f"  Bootstrap progress: {i + 1}/{n_bootstrap}")
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(unique_blocks),
            replace=True,
        )
        sampled_indices = np.concatenate(
            [indices_by_block[block] for block in sampled_blocks]
        )
        y_b = y_true[sampled_indices]

        if np.unique(y_b).size < 2:
            continue

        difference = (
            roc_auc_score(y_b, prob_left[sampled_indices])
            - roc_auc_score(y_b, prob_right[sampled_indices])
        )
        differences.append(float(difference))

    if not differences:
        return observed, float("nan"), float("nan"), 0

    alpha = (1.0 - ci_level) / 2.0
    low, high = np.quantile(differences, [alpha, 1.0 - alpha])
    return observed, float(low), float(high), len(differences)


def normalize_grid_results(
    cv_results: Dict[str, Any],
    scenario_name: str,
    search_stage: str,
    outer_fold: int | None,
) -> pd.DataFrame:
    """Convert GridSearchCV.cv_results_ to a compact audit table."""
    frame = pd.DataFrame(cv_results)

    keep = [
        column
        for column in frame.columns
        if column.startswith("param_")
        or column
        in {
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "mean_fit_time",
            "std_fit_time",
        }
    ]

    frame = frame[keep].copy()
    frame.insert(0, "scenario", scenario_name)
    frame.insert(1, "model", Config.MODEL_SHORT_LABELS[scenario_name])
    frame.insert(2, "search_stage", search_stage)
    frame.insert(3, "outer_fold", outer_fold)
    return frame


def run_nested_spatial_cv_for_scenario(
    df: pd.DataFrame,
    scenario_name: str,
    features: Sequence[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any], pd.DataFrame]:
    """Run five outer spatial folds with four-block inner spatial grid search."""
    log("")
    log("=" * 96)
    log(
        f"[NESTED-CV] {Config.MODEL_SHORT_LABELS[scenario_name]} — "
        f"{Config.MODEL_DESCRIPTIONS[scenario_name]} ({len(features)} predictors)"
    )
    log("=" * 96)

    oof_probability = np.full(len(df), np.nan, dtype="float64")
    oof_predicted = np.full(len(df), -1, dtype="int16")

    fold_rows: List[Dict[str, Any]] = []
    best_params: Dict[str, Any] = {}
    search_tables: List[pd.DataFrame] = []

    for outer_fold in Config.EXPECTED_FOLDS:
        outer_train_mask = df[Config.FOLD_COLUMN].to_numpy() != outer_fold
        outer_valid_mask = df[Config.FOLD_COLUMN].to_numpy() == outer_fold

        train_df = df.loc[outer_train_mask].copy()
        valid_df = df.loc[outer_valid_mask].copy()

        if train_df[Config.FOLD_COLUMN].nunique() != 4:
            raise RuntimeError(
                f"Outer fold {outer_fold}: expected four training blocks."
            )
        if valid_df[Config.FOLD_COLUMN].nunique() != 1:
            raise RuntimeError(
                f"Outer fold {outer_fold}: validation must contain one block only."
            )

        X_train = train_df[list(features)]
        y_train = train_df[Config.TARGET_COLUMN].to_numpy(dtype=int)
        X_valid = valid_df[list(features)]
        y_valid = valid_df[Config.TARGET_COLUMN].to_numpy(dtype=int)

        inner_cv = make_inner_predefined_split(train_df)
        pipeline = build_xgb_pipeline(features)

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=Config.PARAM_GRID,
            scoring="roc_auc",
            cv=inner_cv,
            refit=True,
            n_jobs=Config.GRIDSEARCH_N_JOBS,
            verbose=Config.GRIDSEARCH_VERBOSE,
            return_train_score=False,
            error_score="raise",
        )

        log(
            f"[OUTER {outer_fold}] Tuning on blocks "
            f"{sorted(train_df[Config.FOLD_COLUMN].unique().tolist())}; "
            f"holding out block {outer_fold}."
        )
        search.fit(X_train, y_train)

        probability = search.best_estimator_.predict_proba(X_valid)[:, 1]
        predicted = (probability >= Config.BINARY_THRESHOLD).astype(int)

        valid_indices = np.flatnonzero(outer_valid_mask)
        oof_probability[valid_indices] = probability
        oof_predicted[valid_indices] = predicted

        metrics = classification_metrics(y_valid, probability)

        row = {
            "scenario": scenario_name,
            "model": Config.MODEL_SHORT_LABELS[scenario_name],
            "description": Config.MODEL_DESCRIPTIONS[scenario_name],
            "outer_fold": int(outer_fold),
            "n_predictors": int(len(features)),
            "n_train": int(len(train_df)),
            "n_validation": int(len(valid_df)),
            "n_train_positive": int(y_train.sum()),
            "n_train_negative": int(len(y_train) - y_train.sum()),
            "n_validation_positive": int(y_valid.sum()),
            "n_validation_negative": int(len(y_valid) - y_valid.sum()),
            "inner_best_mean_auc": float(search.best_score_),
            **metrics,
        }
        fold_rows.append(row)

        best_params[str(outer_fold)] = {
            "inner_best_mean_auc": float(search.best_score_),
            "best_params": json_safe(search.best_params_),
        }

        search_tables.append(
            normalize_grid_results(
                search.cv_results_,
                scenario_name=scenario_name,
                search_stage="nested_inner",
                outer_fold=outer_fold,
            )
        )

        if Config.SAVE_OUTER_FOLD_MODELS:
            fold_model_path = (
                Config.MODEL_DIR
                / f"{scenario_prefix(scenario_name)}_outer-fold-{outer_fold}.joblib"
            )
            joblib.dump(search.best_estimator_, fold_model_path)

        log(
            f"[OUTER {outer_fold}] inner best AUC={search.best_score_:.4f}; "
            f"held-out AUC={metrics['roc_auc']:.4f}; "
            f"F1={metrics['f1']:.4f}"
        )

    if not np.isfinite(oof_probability).all():
        missing_count = int((~np.isfinite(oof_probability)).sum())
        raise RuntimeError(
            f"{scenario_name}: {missing_count} samples did not receive OOF predictions."
        )
    if (oof_predicted < 0).any():
        raise RuntimeError(f"{scenario_name}: incomplete OOF class predictions.")

    oof_df = df[
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
        ]
    ].copy()
    oof_df.insert(0, "scenario", scenario_name)
    oof_df.insert(1, "model", Config.MODEL_SHORT_LABELS[scenario_name])
    oof_df.insert(2, "description", Config.MODEL_DESCRIPTIONS[scenario_name])
    oof_df["oof_probability"] = oof_probability
    oof_df["oof_predicted_class_0p5"] = oof_predicted

    fold_df = pd.DataFrame(fold_rows)
    search_df = pd.concat(search_tables, ignore_index=True)

    return oof_df, fold_df, best_params, search_df




def build_fixed_ae_oof_ensemble(
    all_oof: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Build and evaluate the fixed Model A / Model E OOF ensemble.

    This mirrors the old XGBoost blend:
        P_ensemble = 0.30 * P_static + 0.70 * P_enhanced

    Because the weight is fixed before inspecting the OOF ensemble result,
    the resulting OOF metric can be reported as a separate, pre-specified
    ensemble diagnostic. It does not replace the primary A-F ablation results.
    """
    static_name = Config.ENSEMBLE_STATIC_SCENARIO
    enhanced_name = Config.ENSEMBLE_ENHANCED_SCENARIO
    enhanced_weight = float(Config.ENSEMBLE_ENHANCED_WEIGHT)
    static_weight = 1.0 - enhanced_weight

    static_oof = all_oof.loc[
        all_oof["scenario"] == static_name,
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
            "oof_probability",
        ],
    ].copy()
    static_oof = static_oof.rename(
        columns={"oof_probability": "probability_static_A"}
    )

    enhanced_oof = all_oof.loc[
        all_oof["scenario"] == enhanced_name,
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
            "oof_probability",
        ],
    ].copy()
    enhanced_oof = enhanced_oof.rename(
        columns={"oof_probability": "probability_enhanced_E"}
    )

    merged = static_oof.merge(
        enhanced_oof,
        on=Config.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=("_A", "_E"),
        validate="one_to_one",
    )

    expected_rows = Config.EXPECTED_N_ROWS
    if len(merged) != expected_rows:
        raise RuntimeError(
            f"A/E ensemble OOF merge returned {len(merged)} rows; "
            f"expected {expected_rows}."
        )

    for column_name in (
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ):
        left = merged[f"{column_name}_A"].to_numpy()
        right = merged[f"{column_name}_E"].to_numpy()
        if not np.array_equal(left, right):
            raise RuntimeError(
                f"A/E ensemble OOF mismatch in {column_name}."
            )

    merged[Config.TARGET_COLUMN] = merged[
        f"{Config.TARGET_COLUMN}_A"
    ].astype(int)
    merged[Config.FOLD_COLUMN] = merged[
        f"{Config.FOLD_COLUMN}_A"
    ].astype(int)
    merged[Config.BLOCK_COLUMN] = merged[
        f"{Config.BLOCK_COLUMN}_A"
    ].astype(int)

    merged["oof_probability_ensemble"] = (
        static_weight * merged["probability_static_A"].to_numpy(dtype=float)
        + enhanced_weight * merged["probability_enhanced_E"].to_numpy(dtype=float)
    )

    merged["oof_predicted_class_0p5"] = (
        merged["oof_probability_ensemble"].to_numpy(dtype=float)
        >= Config.BINARY_THRESHOLD
    ).astype(int)

    y_true = merged[Config.TARGET_COLUMN].to_numpy(dtype=int)
    y_prob = merged["oof_probability_ensemble"].to_numpy(dtype=float)
    blocks = merged[Config.BLOCK_COLUMN].to_numpy(dtype=int)

    metrics = classification_metrics(
        y_true,
        y_prob,
        threshold=Config.BINARY_THRESHOLD,
    )
    ci_low, ci_high, n_valid_bootstrap = cluster_bootstrap_auc_ci(
        y_true=y_true,
        y_prob=y_prob,
        blocks=blocks,
        n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
        seed=Config.RANDOM_STATE + 9001,
        ci_level=Config.CI_LEVEL,
    )

    # Paired comparison against the primary Model E OOF prediction.
    observed_difference, diff_low, diff_high, n_valid_diff = (
        paired_cluster_bootstrap_auc_difference(
            y_true=y_true,
            prob_left=y_prob,
            prob_right=merged["probability_enhanced_E"].to_numpy(dtype=float),
            blocks=blocks,
            n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
            seed=Config.RANDOM_STATE + 9002,
            ci_level=Config.CI_LEVEL,
        )
    )

    metrics_summary = {
        "ensemble_name": "XGB A30-E70 fixed ensemble",
        "formula": (
            f"{static_weight:.2f} * Model A + "
            f"{enhanced_weight:.2f} * Model E"
        ),
        "weight_selection": (
            "Fixed a priori to mirror the previous static/integrated "
            "XGBoost ensemble; not optimized on OOF predictions."
        ),
        **metrics,
        "auc_cluster_bootstrap_ci_low": float(ci_low),
        "auc_cluster_bootstrap_ci_high": float(ci_high),
        "auc_cluster_bootstrap_valid_iterations": int(n_valid_bootstrap),
        "auc_difference_ensemble_minus_model_E": float(observed_difference),
        "auc_difference_ci_low": float(diff_low),
        "auc_difference_ci_high": float(diff_high),
        "auc_difference_valid_iterations": int(n_valid_diff),
    }

    output_columns = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
        "probability_static_A",
        "probability_enhanced_E",
        "oof_probability_ensemble",
        "oof_predicted_class_0p5",
    ]
    output_df = merged[output_columns].copy()

    output_df.to_csv(
        Config.ENSEMBLE_OOF_PREDICTIONS_CSV,
        index=False,
        float_format="%.10f",
    )
    with open(
        Config.ENSEMBLE_OOF_METRICS_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(metrics_summary),
            handle,
            indent=2,
        )

    log(
        "[ENSEMBLE-OOF] A30/E70 fixed ensemble: "
        f"AUC={metrics_summary['roc_auc']:.4f} "
        f"[{ci_low:.4f}, {ci_high:.4f}], "
        f"ΔAUC vs Model E={observed_difference:+.4f} "
        f"[{diff_low:+.4f}, {diff_high:+.4f}]"
    )

    if observed_difference < 0:
        log(
            "[ENSEMBLE-WARN] The fixed A30/E70 ensemble has lower pooled "
            "OOF AUC than Model E. The ensemble map can still be generated "
            "for comparison, but Model E should remain the primary scientific "
            "map unless there is another justified reason to prefer the ensemble."
        )

    return output_df, metrics_summary


# =============================================================================
# FINAL FULL-DATA MODEL FOR MAPPING / DEPLOYMENT
# =============================================================================
def tune_and_fit_final_model(
    df: pd.DataFrame,
    scenario_name: str,
    features: Sequence[str],
) -> Tuple[Pipeline, Dict[str, Any], pd.DataFrame]:
    """
    Tune on all 330 samples using the frozen five spatial folds, then refit on
    all samples. This model is for raster prediction only, not performance claims.
    """
    X = df[list(features)]
    y = df[Config.TARGET_COLUMN].to_numpy(dtype=int)

    pipeline = build_xgb_pipeline(features)
    spatial_cv = make_full_data_predefined_split(df)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=Config.PARAM_GRID,
        scoring="roc_auc",
        cv=spatial_cv,
        refit=True,
        n_jobs=Config.GRIDSEARCH_N_JOBS,
        verbose=Config.GRIDSEARCH_VERBOSE,
        return_train_score=False,
        error_score="raise",
    )

    log(
        f"[FINAL-MODEL] {Config.MODEL_SHORT_LABELS[scenario_name]}: "
        "full-data spatial grid search for deployment model."
    )
    search.fit(X, y)

    model_path = Config.MODEL_DIR / f"{scenario_prefix(scenario_name)}_XGB.joblib"
    joblib.dump(search.best_estimator_, model_path)

    info = {
        "best_mean_spatial_cv_auc_for_model_selection_only": float(
            search.best_score_
        ),
        "best_params": json_safe(search.best_params_),
        "model_path": str(model_path),
        "note": (
            "This full-data tuning score is not used as the reported validation "
            "performance; reported performance comes only from nested outer OOF predictions."
        ),
    }

    search_df = normalize_grid_results(
        search.cv_results_,
        scenario_name=scenario_name,
        search_stage="full_data_deployment_tuning",
        outer_fold=None,
    )

    return search.best_estimator_, info, search_df


# =============================================================================
# RASTER PREDICTION
# =============================================================================
def read_stack_band_map(stack_path: Path) -> Dict[str, int]:
    require_file(stack_path, "Step 6.1 master predictor stack")
    with rasterio.open(stack_path) as dataset:
        descriptions = list(dataset.descriptions)

    band_map: Dict[str, int] = {}
    for band_index, description in enumerate(descriptions, start=1):
        if description:
            band_map[str(description)] = band_index

    return band_map


def predict_scenario_raster(
    fitted_pipeline: Pipeline,
    scenario_name: str,
    features: Sequence[str],
) -> Path:
    """
    Predict one susceptibility raster on the common A–F valid domain so all six
    maps have an identical comparison footprint.
    """
    require_file(Config.MASTER_STACK_TIF, "Step 6.1 master predictor stack")
    require_file(Config.COMMON_AF_MASK_TIF, "common A–F valid mask")

    band_map = read_stack_band_map(Config.MASTER_STACK_TIF)
    missing_bands = [feature for feature in features if feature not in band_map]
    if missing_bands:
        raise RuntimeError(
            f"Master stack is missing bands required by {scenario_name}: {missing_bands}"
        )

    band_indexes = [band_map[feature] for feature in features]
    output_path = (
        Config.MAP_DIR
        / f"{scenario_prefix(scenario_name)}_XGB_probability_0_1_common_AF_domain.tif"
    )

    with rasterio.open(Config.MASTER_STACK_TIF) as stack_ds, rasterio.open(
        Config.COMMON_AF_MASK_TIF
    ) as mask_ds:
        if (
            stack_ds.width != mask_ds.width
            or stack_ds.height != mask_ds.height
            or stack_ds.transform != mask_ds.transform
            or stack_ds.crs != mask_ds.crs
        ):
            raise RuntimeError("Master stack and common A–F mask are not aligned.")

        profile = stack_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(temporary_path, "w", **profile) as output_ds:
            for row_start in range(0, stack_ds.height, Config.RASTER_BLOCK_ROWS):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    stack_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=stack_ds.width,
                    height=block_height,
                )

                data = stack_ds.read(band_indexes, window=window, masked=True)
                common_mask = mask_ds.read(1, window=window, masked=True)

                output = np.full(
                    (block_height, stack_ds.width),
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                values = np.asarray(data.filled(np.nan), dtype="float64")
                valid = (
                    np.asarray(common_mask.filled(0)) > 0
                ) & np.all(np.isfinite(values), axis=0)

                if np.any(valid):
                    pixel_matrix = values[:, valid].T
                    pixel_df = pd.DataFrame(pixel_matrix, columns=list(features))
                    probabilities = fitted_pipeline.predict_proba(pixel_df)[:, 1]
                    output[valid] = probabilities.astype("float32")

                output_ds.write(output, 1, window=window)

            output_ds.set_band_description(
                1,
                f"{Config.MODEL_SHORT_LABELS[scenario_name]} XGB susceptibility probability",
            )
            output_ds.update_tags(
                scenario=scenario_name,
                model="XGBoost",
                prediction_domain="common_A_to_F_valid_domain",
                validation_note=(
                    "Raster generated from final full-data deployment model; "
                    "validation metrics come from nested spatial OOF predictions."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(temporary_path, output_path)
    log(f"[MAP] Saved: {output_path}")
    return output_path






def predict_scenario_raster_on_own_feature_domain(
    fitted_pipeline: Pipeline,
    scenario_name: str,
    features: Sequence[str],
    output_path: Path,
) -> Path:
    """
    Predict a scenario wherever all predictors required by that scenario are
    finite in the study-domain master stack.

    Unlike the A-F comparison maps, this function does NOT restrict prediction
    to the common A-F mask. This is required to reproduce the earlier ensemble
    logic, where the static model supplies the broad susceptibility background
    and the enhanced model is blended only where its additional predictors exist.
    """
    require_file(
        Config.MASTER_STACK_TIF,
        "Step 6.1 master predictor stack",
    )

    band_map = read_stack_band_map(Config.MASTER_STACK_TIF)
    missing_bands = [
        feature
        for feature in features
        if feature not in band_map
    ]
    if missing_bands:
        raise RuntimeError(
            f"Master stack is missing bands required by "
            f"{scenario_name}: {missing_bands}"
        )

    band_indexes = [
        band_map[feature]
        for feature in features
    ]

    with rasterio.open(Config.MASTER_STACK_TIF) as stack_ds:
        profile = stack_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        valid_pixel_count = 0

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_ds:
            for row_start in range(
                0,
                stack_ds.height,
                Config.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    stack_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=stack_ds.width,
                    height=block_height,
                )

                data = stack_ds.read(
                    band_indexes,
                    window=window,
                    masked=True,
                )

                output = np.full(
                    (block_height, stack_ds.width),
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                values = np.asarray(
                    data.filled(np.nan),
                    dtype="float64",
                )
                valid = np.all(
                    np.isfinite(values),
                    axis=0,
                )

                if np.any(valid):
                    pixel_matrix = values[:, valid].T
                    pixel_df = pd.DataFrame(
                        pixel_matrix,
                        columns=list(features),
                    )
                    probabilities = (
                        fitted_pipeline.predict_proba(pixel_df)[:, 1]
                    )
                    output[valid] = probabilities.astype("float32")
                    valid_pixel_count += int(valid.sum())

                output_ds.write(
                    output,
                    1,
                    window=window,
                )

            output_ds.set_band_description(
                1,
                (
                    f"{Config.MODEL_SHORT_LABELS[scenario_name]} "
                    "XGB susceptibility probability"
                ),
            )
            output_ds.update_tags(
                scenario=scenario_name,
                model="XGBoost",
                prediction_domain="scenario_specific_valid_feature_domain",
                validation_note=(
                    "Raster generated from the final full-data deployment model. "
                    "Reported validation metrics come from nested spatial OOF "
                    "predictions, not from this raster."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        f"[MAP-FULL-DOMAIN] {Config.MODEL_SHORT_LABELS[scenario_name]}: "
        f"{valid_pixel_count:,} valid pixels -> {output_path}"
    )

    return output_path


def blend_static_and_enhanced_probability_rasters(
    static_raster_path: Path,
    enhanced_raster_path: Path,
    output_path: Path,
    enhanced_weight: float,
) -> Path:
    """
    Reproduce the old XGBoost ensemble logic for the revised A-F framework.

    Where both models are valid:
        ensemble = (1-w) * Model A + w * Model E

    Where only Model A is valid:
        retain Model A as the susceptibility background.

    Where only Model E is valid:
        retain Model E.

    Outside both valid domains:
        nodata.

    This is intentionally different from averaging all A-F configurations.
    Models B-F are ablation experiments and should not be indiscriminately
    averaged into the final susceptibility map.
    """
    if not 0.0 <= enhanced_weight <= 1.0:
        raise ValueError(
            "ENSEMBLE_ENHANCED_WEIGHT must lie between 0 and 1."
        )

    static_weight = 1.0 - enhanced_weight

    require_file(
        static_raster_path,
        "Model A full-domain probability raster",
    )
    require_file(
        enhanced_raster_path,
        "Model E full-domain probability raster",
    )

    with rasterio.open(static_raster_path) as static_ds, rasterio.open(
        enhanced_raster_path
    ) as enhanced_ds:
        aligned = (
            static_ds.width == enhanced_ds.width
            and static_ds.height == enhanced_ds.height
            and static_ds.transform == enhanced_ds.transform
            and static_ds.crs == enhanced_ds.crs
        )
        if not aligned:
            raise RuntimeError(
                "Model A and Model E full-domain rasters are not aligned."
            )

        profile = static_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        n_static_background = 0
        n_blended = 0
        n_enhanced_only = 0

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_ds:
            for row_start in range(
                0,
                static_ds.height,
                Config.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    static_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=static_ds.width,
                    height=block_height,
                )

                static_arr = static_ds.read(
                    1,
                    window=window,
                    masked=True,
                )
                enhanced_arr = enhanced_ds.read(
                    1,
                    window=window,
                    masked=True,
                )

                static_values = np.asarray(
                    static_arr.filled(np.nan),
                    dtype="float64",
                )
                enhanced_values = np.asarray(
                    enhanced_arr.filled(np.nan),
                    dtype="float64",
                )

                static_valid = np.isfinite(static_values)
                enhanced_valid = np.isfinite(enhanced_values)

                both = static_valid & enhanced_valid
                static_only = static_valid & ~enhanced_valid
                enhanced_only = enhanced_valid & ~static_valid

                output = np.full(
                    static_values.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                output[static_only] = static_values[
                    static_only
                ].astype("float32")

                output[enhanced_only] = enhanced_values[
                    enhanced_only
                ].astype("float32")

                output[both] = (
                    static_weight * static_values[both]
                    + enhanced_weight * enhanced_values[both]
                ).astype("float32")

                n_static_background += int(static_only.sum())
                n_blended += int(both.sum())
                n_enhanced_only += int(enhanced_only.sum())

                output_ds.write(
                    output,
                    1,
                    window=window,
                )

            output_ds.set_band_description(
                1,
                "XGBoost A30-E70 ensemble susceptibility probability",
            )
            output_ds.update_tags(
                model="XGBoost",
                ensemble="Model_A_static_plus_Model_E_physics_guided",
                static_weight=f"{static_weight:.6f}",
                enhanced_weight=f"{enhanced_weight:.6f}",
                blend_rule=(
                    "weighted blend where both are valid; "
                    "retain static background elsewhere"
                ),
                validation_note=(
                    "The fixed A/E ensemble is separately evaluated from nested "
                    "OOF probabilities. A-F ablation metrics remain the primary "
                    "model-comparison results."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[ENSEMBLE-MAP] Saved A30/E70 ensemble: "
        f"blended={n_blended:,}, "
        f"static-background={n_static_background:,}, "
        f"enhanced-only={n_enhanced_only:,} -> "
        f"{output_path}"
    )

    return output_path


def create_revised_ae_ensemble_probability_map(
    final_models: dict[str, Pipeline],
    scenarios: dict[str, dict[str, Any]],
) -> Path:
    """
    Generate the revised equivalent of the previous XGB blended ensemble map.

    Model A supplies the broad static susceptibility background.
    Model E contributes the deformation-informed physics-guided prediction
    wherever HKFI and SNDR are available.
    """
    static_scenario = Config.ENSEMBLE_STATIC_SCENARIO
    enhanced_scenario = Config.ENSEMBLE_ENHANCED_SCENARIO

    for scenario_name in (
        static_scenario,
        enhanced_scenario,
    ):
        if scenario_name not in final_models:
            raise RuntimeError(
                f"Final deployment model unavailable for {scenario_name}."
            )
        if scenario_name not in scenarios:
            raise RuntimeError(
                f"Scenario definition unavailable for {scenario_name}."
            )

    predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=final_models[static_scenario],
        scenario_name=static_scenario,
        features=list(scenarios[static_scenario]["features"]),
        output_path=Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
    )

    predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=final_models[enhanced_scenario],
        scenario_name=enhanced_scenario,
        features=list(scenarios[enhanced_scenario]["features"]),
        output_path=Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
    )

    return blend_static_and_enhanced_probability_rasters(
        static_raster_path=Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
        enhanced_raster_path=Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
        output_path=Config.ENSEMBLE_PROBABILITY_TIF,
        enhanced_weight=float(Config.ENSEMBLE_ENHANCED_WEIGHT),
    )


# =============================================================================
# STEP 5.3-STYLE LANDSLIDE SUSCEPTIBILITY VISUALIZATION
# =============================================================================

LSM_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "xgb_landslide_susceptibility",
    [
        "#006400",
        "#32CD32",
        "#FFFF00",
        "#FFA500",
        "#FF4500",
        "#8B0000",
    ],
    N=256,
)



SUSCEPTIBILITY_CLASSES = {
    "Very Low": (0.0, 0.2),
    "Low": (0.2, 0.4),
    "Moderate": (0.4, 0.6),
    "High": (0.6, 0.8),
    "Very High": (0.8, 1.0),
}

SUSCEPTIBILITY_CLASS_COLORS = {
    "Very Low": "#006400",
    "Low": "#32CD32",
    "Moderate": "#FFFF00",
    "High": "#FFA500",
    "Very High": "#8B0000",
}


def add_old_style_susceptibility_legend(
    axis,
    include_road: bool = True,
) -> None:
    """Reproduce the earlier LSM legend layout."""
    legend_elements: list[Line2D] = []

    if include_road:
        legend_elements.append(
            Line2D(
                [0],
                [0],
                color="red",
                linewidth=1.5,
                label="Highway/Roads",
            )
        )

    for class_name in (
        "Very High",
        "High",
        "Moderate",
        "Low",
        "Very Low",
    ):
        legend_elements.append(
            Line2D(
                [0],
                [0],
                marker="s",
                color="white",
                markerfacecolor=(
                    SUSCEPTIBILITY_CLASS_COLORS[class_name]
                ),
                markersize=7,
                markeredgecolor="black",
                markeredgewidth=0.5,
                label=class_name,
            )
        )

    legend = axis.legend(
        handles=legend_elements,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.50), # # Changed here to move the Map Up
        fontsize=Config.LEGEND_FONTSIZE,
        title="Legend",
        title_fontsize=Config.LEGEND_TITLE_FONTSIZE,
        frameon=True,
        fancybox=True,
        shadow=True,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0,
    )
    legend.get_title().set_fontweight("bold")
    legend.set_zorder(2000)


def reproject_probability_raster_to_epsg4326(
    source_path: Path,
    output_path: Path,
) -> Path:
    """
    Reproject only for cartographic display, matching the earlier LSM workflow.

    The source analytical probability raster is never modified.
    Bilinear resampling is used only for display reprojection of the continuous
    probability surface.
    """
    require_file(
        source_path,
        "susceptibility probability raster for visualization",
    )

    with rasterio.open(source_path) as source:
        if source.crs is None:
            raise RuntimeError(
                f"Probability raster has no CRS: {source_path}"
            )

        if source.crs.to_epsg() == 4326:
            return source_path

        destination_crs = "EPSG:4326"
        destination_transform, width, height = (
            calculate_default_transform(
                source.crs,
                destination_crs,
                source.width,
                source.height,
                *source.bounds,
            )
        )

        profile = source.profile.copy()
        profile.update(
            crs=destination_crs,
            transform=destination_transform,
            width=width,
            height=height,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as destination:
            reproject(
                source=rasterio.band(source, 1),
                destination=rasterio.band(destination, 1),
                src_transform=source.transform,
                src_crs=source.crs,
                src_nodata=source.nodata,
                dst_transform=destination_transform,
                dst_crs=destination_crs,
                dst_nodata=Config.OUTPUT_NODATA,
                resampling=Resampling.bilinear,
            )

            destination.set_band_description(
                1,
                "XGBoost ensemble susceptibility probability — EPSG:4326 visualization copy",
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[VIZ-REPROJECT] Display raster reprojected to EPSG:4326: "
        f"{output_path}"
    )
    return output_path


def ensemble_visualization_output_paths() -> dict[str, Path]:
    stem = (
        Config.VISUALIZATION_DIR
        / "XGB_Susceptibility_Map_Ensemble_A30_E70"
    )
    return {
        "map_png": Path(str(stem) + ".png"),
        "map_pdf": Path(str(stem) + ".pdf"),
        "map_svg": Path(str(stem) + ".svg"),
        "histogram_png": Path(
            str(Config.VISUALIZATION_DIR / "XGB_Ensemble_Histogram_A30_E70")
            + ".png"
        ),
        "histogram_pdf": Path(
            str(Config.VISUALIZATION_DIR / "XGB_Ensemble_Histogram_A30_E70")
            + ".pdf"
        ),
        "histogram_svg": Path(
            str(Config.VISUALIZATION_DIR / "XGB_Ensemble_Histogram_A30_E70")
            + ".svg"
        ),
        "summary_json": (
            Config.VISUALIZATION_DIR
            / "XGB_Ensemble_A30_E70_Visualization_Summary.json"
        ),
        "statistics_csv": (
            Config.VISUALIZATION_DIR
            / "XGB_Ensemble_A30_E70_Statistics.csv"
        ),
        "area_percentage_png": (
            Config.VISUALIZATION_DIR
            / "XGB_Ensemble_A30_E70_Area_Percentage.png"
        ),
        "area_percentage_pdf": (
            Config.VISUALIZATION_DIR
            / "XGB_Ensemble_A30_E70_Area_Percentage.pdf"
        ),
        "area_percentage_svg": (
            Config.VISUALIZATION_DIR
            / "XGB_Ensemble_A30_E70_Area_Percentage.svg"
        ),
        "area_percentage_csv": (
            Config.VISUALIZATION_DIR
            / "XGB_Ensemble_A30_E70_Area_Percentage.csv"
        ),
        "support_map_png": (
            Config.VISUALIZATION_DIR
            / "XGB_Ensemble_A30_E70_Information_Support_Map.png"
        ),
        "support_map_pdf": (
            Config.VISUALIZATION_DIR
            / "XGB_Ensemble_A30_E70_Information_Support_Map.pdf"
        ),
        "support_map_svg": (
            Config.VISUALIZATION_DIR
            / "XGB_Ensemble_A30_E70_Information_Support_Map.svg"
        ),
        "display_raster_epsg4326": (
            Config.VISUALIZATION_DIR
            / "XGB_Ensemble_A30_E70_Susceptibility_EPSG4326.tif"
        ),
    }


def create_old_style_ensemble_lsm_map(
    probability_raster_path: Path,
    outputs: dict[str, Path],
) -> tuple[np.ndarray, dict[str, Any]]:
    """
    Draw the ensemble LSM using the earlier successful XGB map approach.

    Key elements reproduced:
      - static susceptibility background retained across the broader study domain;
      - EPSG:4326 display raster;
      - exact raster extent (no added map buffer);
      - white opaque map background;
      - vector clip path to the study boundary;
      - thick black study boundary;
      - red highway;
      - north arrow;
      - checker scale bar;
      - susceptibility-class legend plus continuous colour bar.
    """
    display_raster_path = reproject_probability_raster_to_epsg4326(
        source_path=probability_raster_path,
        output_path=outputs["display_raster_epsg4326"],
    )

    probability, reference = read_probability_raster(
        display_raster_path
    )
    values = probability[np.isfinite(probability)]

    bounds = reference["bounds"]
    extent = (
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    )

    display_array = np.ma.masked_invalid(
        probability
    )

    # Opaque white for any nodata cells that remain.
    map_cmap = LSM_CMAP.copy()
    map_cmap.set_bad(
        color="white",
        alpha=1.0,
    )

    study_boundary = load_vector_overlay(
        Config.STUDY_SHP,
        "EPSG:4326",
    )
    roads = load_vector_overlay(
        Config.ROAD_SHP,
        "EPSG:4326",
    )

    figure, axis = plt.subplots(
        1,
        1,
        figsize=Config.ENSEMBLE_MAP_FIGSIZE,
        facecolor="white",
    )
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        vmin=0.0,
        vmax=1.0,
        cmap=map_cmap,
        interpolation="nearest",
        alpha=1.0,
        zorder=1,
    )

    # The raster is clipped instead of relying on alpha transparency.
    clip_patch = build_clip_patch_from_vector_overlay(
        study_boundary,
        axis,
    )
    if clip_patch is not None:
        image.set_clip_path(
            clip_patch
        )
        log(
            "[ENSEMBLE-VIZ] Raster clipped to study boundary; "
            "opaque white background retained."
        )

    # Match the visual hierarchy of the earlier map.
    plot_vector_overlay(
        axis,
        study_boundary,
        colour="black",
        linewidth=2.0,
        zorder=1000,
        polygon_boundary_only=True,
    )
    plot_vector_overlay(
        axis,
        roads,
        colour="red",
        linewidth=1.0,
        zorder=1100,
        polygon_boundary_only=False,
    )

    axis.set_xlim(
        extent[0],
        extent[1],
    )
    axis.set_ylim(
        extent[2],
        extent[3],
    )

    axis.xaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: to_dms(
                value,
                True,
            )
        )
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: to_dms(
                value,
                False,
            )
        )
    )
    axis.tick_params(
        axis="both",
        which="major",
        labelsize=Config.DMS_TICK_FONTSIZE,
    )

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")

    for label in axis.get_yticklabels():
        label.set_rotation(90)
        label.set_va("center")

    add_north_arrow_tri(
        axis,
        size=0.08,
        pad=0.06,
    )

    add_checker_scale_bar(
        axis,
        extent[0],
        extent[1],
        extent[2],
        extent[3],
        crs_type="wgs84",
        mid_lat_deg=(
            0.5 * (extent[2] + extent[3])
        ),
        segments=4,
        fontsize=6,
    )

    add_old_style_susceptibility_legend(
        axis,
        include_road=True,
    )

    divider = make_axes_locatable(
        axis
    )
    colorbar_axis = divider.append_axes(
        "right",
        size=Config.CAX_SIZE,
        pad=Config.CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(
        labelsize=Config.COLORBAR_TICK_FONTSIZE,
    )
    colorbar.set_label(
        "Susceptibility (0–1)",
        fontsize=Config.COLORBAR_LABEL_FONTSIZE,
        fontweight="bold",
    )

    figure.tight_layout()

    for output_key, output_format in (
        ("map_png", None),
        ("map_pdf", None),
        ("map_svg", "svg"),
    ):
        save_kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
            "edgecolor": "white",
            "transparent": False,
        }
        if output_key == "map_png":
            save_kwargs["dpi"] = Config.FIG_DPI
        if output_format is not None:
            save_kwargs["format"] = output_format

        figure.savefig(
            outputs[output_key],
            **save_kwargs,
        )

    log(
        f"[ENSEMBLE-VIZ] Map saved: {outputs['map_png']}"
    )

    if Config.SHOW_LSM_FIGURES:
        plt.show()

    plt.close(
        figure
    )

    return values, reference



def classify_susceptibility_probability(values: np.ndarray) -> np.ndarray:
    """Classify continuous probability into five equal-interval classes."""
    classes = np.zeros(values.shape, dtype=np.uint8)
    finite = np.isfinite(values)
    classes[finite & (values >= 0.0) & (values < 0.2)] = 1
    classes[finite & (values >= 0.2) & (values < 0.4)] = 2
    classes[finite & (values >= 0.4) & (values < 0.6)] = 3
    classes[finite & (values >= 0.6) & (values < 0.8)] = 4
    classes[finite & (values >= 0.8) & (values <= 1.0)] = 5
    return classes


def calculate_susceptibility_area_percentages(
    probability_raster_path: Path,
) -> pd.DataFrame:
    """
    Calculate mapped area percentage for each susceptibility class on the
    original projected analytical grid. Pixel fractions therefore equal area
    fractions because cell area is constant.
    """
    require_file(probability_raster_path, "ensemble susceptibility raster")

    with rasterio.open(probability_raster_path) as dataset:
        probability = clean_probability_array(
            dataset.read(1),
            dataset.nodata,
        )

    valid = probability[np.isfinite(probability)]
    classes = classify_susceptibility_probability(valid)
    total = int(valid.size)

    definitions = [
        (5, "Very High", 0.8, 1.0),
        (4, "High", 0.6, 0.8),
        (3, "Moderate", 0.4, 0.6),
        (2, "Low", 0.2, 0.4),
        (1, "Very Low", 0.0, 0.2),
    ]

    rows = []
    for class_id, label, lower, upper in definitions:
        count = int(np.sum(classes == class_id))
        percentage = 100.0 * count / total if total > 0 else np.nan
        rows.append(
            {
                "class_id": class_id,
                "class_label": label,
                "lower_probability": lower,
                "upper_probability": upper,
                "pixel_count": count,
                "area_percentage": percentage,
            }
        )

    return pd.DataFrame(rows)


def create_susceptibility_area_percentage_plot(
    area_df: pd.DataFrame,
    outputs: dict[str, Path],
) -> None:
    """Create a separate 4 x 4 bar plot of susceptibility-class area percentages."""
    labels = area_df["class_label"].tolist()
    percentages = area_df["area_percentage"].to_numpy(dtype=float)
    colours = [SUSCEPTIBILITY_CLASS_COLORS[label] for label in labels]

    figure, axis = plt.subplots(
        1,
        1,
        figsize=Config.AREA_PERCENT_FIGSIZE,
        facecolor="white",
    )
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    bars = axis.bar(
        labels,
        percentages,
        color=colours,
        edgecolor="black",
        linewidth=0.8,
    )

    upper_limit = max(5.0, float(np.nanmax(percentages)) * 1.18)
    axis.set_ylim(0.0, upper_limit)

    for bar, percentage in zip(bars, percentages):
        axis.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + upper_limit * 0.015,
            f"{percentage:.2f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
        )

    axis.set_xlabel("Susceptibility Class", fontsize=10, fontweight="bold")
    axis.set_ylabel("Area (%)", fontsize=10, fontweight="bold")
    axis.tick_params(axis="x", labelsize=8, rotation=20)
    axis.tick_params(axis="y", labelsize=8)
    axis.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.35)
    axis.set_axisbelow(True)

    for spine in axis.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("black")

    figure.tight_layout()

    for output_key, output_format in (
        ("area_percentage_png", None),
        ("area_percentage_pdf", None),
        ("area_percentage_svg", "svg"),
    ):
        kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
            "edgecolor": "white",
            "transparent": False,
        }
        if output_key == "area_percentage_png":
            kwargs["dpi"] = Config.FIG_DPI
        if output_format is not None:
            kwargs["format"] = output_format
        figure.savefig(outputs[output_key], **kwargs)

    if Config.SHOW_LSM_FIGURES:
        plt.show()
    plt.close(figure)


def write_information_support_raster(
    static_raster_path: Path,
    enhanced_raster_path: Path,
    output_path: Path,
) -> Path:
    """
    Write information-support classes:
      1 = Static-only prediction (Model A)
      2 = Static + deformation-informed prediction (Model E support present)
    """
    require_file(static_raster_path, "Model A full-domain probability raster")
    require_file(enhanced_raster_path, "Model E full-domain probability raster")

    with rasterio.open(static_raster_path) as static_ds, rasterio.open(enhanced_raster_path) as enhanced_ds:
        if not (
            static_ds.width == enhanced_ds.width
            and static_ds.height == enhanced_ds.height
            and static_ds.transform == enhanced_ds.transform
            and static_ds.crs == enhanced_ds.crs
        ):
            raise RuntimeError("Model A and Model E rasters are not aligned.")

        profile = static_ds.profile.copy()
        profile.update(driver="GTiff", count=1, dtype="uint8", nodata=0, compress="LZW", tiled=True)

        with rasterio.open(output_path, "w", **profile) as output_ds:
            for row_start in range(0, static_ds.height, Config.RASTER_BLOCK_ROWS):
                block_height = min(Config.RASTER_BLOCK_ROWS, static_ds.height - row_start)
                window = Window(0, row_start, static_ds.width, block_height)

                static_values = np.asarray(
                    static_ds.read(1, window=window, masked=True).filled(np.nan),
                    dtype=float,
                )
                enhanced_values = np.asarray(
                    enhanced_ds.read(1, window=window, masked=True).filled(np.nan),
                    dtype=float,
                )

                static_valid = np.isfinite(static_values)
                enhanced_valid = np.isfinite(enhanced_values)

                support = np.zeros(static_values.shape, dtype=np.uint8)
                support[static_valid & ~enhanced_valid] = 1
                support[enhanced_valid] = 2
                output_ds.write(support, 1, window=window)

            output_ds.set_band_description(1, "XGBoost A/E ensemble information support")
            output_ds.update_tags(
                class_1="Static-only prediction (Model A)",
                class_2="Static + deformation-informed prediction (Model E support present)",
                generated_at_utc=utc_now_iso(),
            )

    log(f"[SUPPORT-MAP] Saved: {output_path}")
    return output_path


def create_information_support_map(
    support_raster_path: Path,
    outputs: dict[str, Path],
) -> None:
    """Plot static-only versus deformation-informed support using the same map style."""
    display_path = Config.VISUALIZATION_DIR / "XGB_Ensemble_A30_E70_Information_Support_EPSG4326.tif"

    with rasterio.open(support_raster_path) as source:
        if source.crs.to_epsg() == 4326:
            display_path = support_raster_path
        else:
            dst_crs = "EPSG:4326"
            dst_transform, width, height = calculate_default_transform(
                source.crs, dst_crs, source.width, source.height, *source.bounds
            )
            profile = source.profile.copy()
            profile.update(
                crs=dst_crs,
                transform=dst_transform,
                width=width,
                height=height,
                dtype="uint8",
                nodata=0,
                compress="LZW",
            )
            with rasterio.open(display_path, "w", **profile) as destination:
                reproject(
                    source=rasterio.band(source, 1),
                    destination=rasterio.band(destination, 1),
                    src_transform=source.transform,
                    src_crs=source.crs,
                    src_nodata=0,
                    dst_transform=dst_transform,
                    dst_crs=dst_crs,
                    dst_nodata=0,
                    resampling=Resampling.nearest,
                )

    with rasterio.open(display_path) as dataset:
        support = dataset.read(1)
        b = dataset.bounds
        extent = (b.left, b.right, b.bottom, b.top)

    display_array = np.ma.masked_where(support == 0, support)
    cmap = mcolors.ListedColormap(["#bdbdbd", "#2c7fb8"])
    cmap.set_bad(color="white", alpha=1.0)
    norm = mcolors.BoundaryNorm([0.5, 1.5, 2.5], cmap.N)

    study_boundary = load_vector_overlay(Config.STUDY_SHP, "EPSG:4326")
    roads = load_vector_overlay(Config.ROAD_SHP, "EPSG:4326")

    figure, axis = plt.subplots(1, 1, figsize=Config.SUPPORT_MAP_FIGSIZE, facecolor="white")
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
        alpha=1.0,
        zorder=1,
    )

    clip_patch = build_clip_patch_from_vector_overlay(study_boundary, axis)
    if clip_patch is not None:
        image.set_clip_path(clip_patch)

    plot_vector_overlay(axis, study_boundary, colour="black", linewidth=2.0, zorder=1000, polygon_boundary_only=True)
    plot_vector_overlay(axis, roads, colour="red", linewidth=1.0, zorder=1100, polygon_boundary_only=False)

    axis.set_xlim(extent[0], extent[1])
    axis.set_ylim(extent[2], extent[3])
    axis.xaxis.set_major_formatter(FuncFormatter(lambda value, position: to_dms(value, True)))
    axis.yaxis.set_major_formatter(FuncFormatter(lambda value, position: to_dms(value, False)))
    axis.tick_params(axis="both", which="major", labelsize=Config.DMS_TICK_FONTSIZE)

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")
    for label in axis.get_yticklabels():
        label.set_rotation(90)
        label.set_va("center")

    add_north_arrow_tri(axis, size=0.08, pad=0.06)
    add_checker_scale_bar(
        axis,
        extent[0], extent[1], extent[2], extent[3],
        crs_type="wgs84",
        mid_lat_deg=0.5 * (extent[2] + extent[3]),
        segments=4,
        fontsize=Config.SCALEBAR_FONTSIZE,
    )

    handles = [
        Line2D([0], [0], marker="s", color="white", markerfacecolor="#bdbdbd", markeredgecolor="black", markersize=8, label="Static-only prediction"),
        Line2D([0], [0], marker="s", color="white", markerfacecolor="#2c7fb8", markeredgecolor="black", markersize=8, label="Static + deformation-informed"),
        Line2D([0], [0], color="red", linewidth=1.5, label="Highway/Roads"),
    ]
    axis.legend(
        handles=handles,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.60), # Changed here to move the Map Up
        fontsize=Config.LEGEND_FONTSIZE,
        title="Information Support",
        title_fontsize=Config.LEGEND_TITLE_FONTSIZE,
        frameon=True,
        fancybox=True,
        shadow=True,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0,
    )

    figure.tight_layout()
    for output_key, output_format in (
        ("support_map_png", None),
        ("support_map_pdf", None),
        ("support_map_svg", "svg"),
    ):
        kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
            "edgecolor": "white",
            "transparent": False,
        }
        if output_key == "support_map_png":
            kwargs["dpi"] = Config.FIG_DPI
        if output_format is not None:
            kwargs["format"] = output_format
        figure.savefig(outputs[output_key], **kwargs)

    if Config.SHOW_LSM_FIGURES:
        plt.show()
    plt.close(figure)

def create_complete_ensemble_lsm_visualization(
    probability_raster_path: Path,
    ensemble_oof_metrics: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Create the old-style ensemble susceptibility map and histogram."""
    outputs = ensemble_visualization_output_paths()

    values, reference = create_old_style_ensemble_lsm_map(
        probability_raster_path=probability_raster_path,
        outputs=outputs,
    )

    statistics = calculate_lsm_statistics(
        values
    )

    # Five-class area-percentage analysis is calculated on the original
    # projected ensemble raster, not on the display-reprojected copy.
    area_percentage_df = calculate_susceptibility_area_percentages(
        probability_raster_path
    )
    area_percentage_df.to_csv(
        outputs["area_percentage_csv"],
        index=False,
        float_format="%.6f",
    )
    create_susceptibility_area_percentage_plot(
        area_df=area_percentage_df,
        outputs=outputs,
    )

    # Explicit information-support map for transparency.
    support_raster_path = write_information_support_raster(
        static_raster_path=Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
        enhanced_raster_path=Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
        output_path=Config.ENSEMBLE_SUPPORT_TIF,
    )
    create_information_support_map(
        support_raster_path=support_raster_path,
        outputs=outputs,
    )

    # Reuse the established histogram function with the ensemble values.
    create_lsm_probability_histogram(
        values=values,
        statistics=statistics,
        outputs=outputs,
    )

    pd.DataFrame(
        [
            {
                "map_source": "A30_E70_fixed_ensemble",
                "static_scenario": Config.ENSEMBLE_STATIC_SCENARIO,
                "enhanced_scenario": Config.ENSEMBLE_ENHANCED_SCENARIO,
                "static_weight": (
                    1.0 - Config.ENSEMBLE_ENHANCED_WEIGHT
                ),
                "enhanced_weight": Config.ENSEMBLE_ENHANCED_WEIGHT,
                **statistics,
            }
        ]
    ).to_csv(
        outputs["statistics_csv"],
        index=False,
    )

    summary = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "XGBoost",
        "map_source": "fixed_A30_E70_ensemble",
        "ensemble_formula": (
            f"{1.0 - Config.ENSEMBLE_ENHANCED_WEIGHT:.2f} * Model A + "
            f"{Config.ENSEMBLE_ENHANCED_WEIGHT:.2f} * Model E "
            "where Model E is available; otherwise retain Model A"
        ),
        "ensemble_weight_note": (
            "Weight fixed a priori to mirror the earlier context/integrated "
            "ensemble approach; not optimized on OOF predictions."
        ),
        "probability_raster": str(
            probability_raster_path
        ),
        "oof_ensemble_metrics": (
            ensemble_oof_metrics
            if ensemble_oof_metrics is not None
            else {}
        ),
        "visualization_style": (
            "Earlier XGB ensemble LSM cartographic style with opaque white "
            "background, study-area vector clipping, enlarged DMS labels, "
            "five susceptibility classes and a separate class-area plot."
        ),
        "susceptibility_classes": area_percentage_df.to_dict(orient="records"),
        "information_support": {
            "class_1": "Static-only prediction (Model A)",
            "class_2": "Static + deformation-informed prediction (Model E support present)",
            "support_raster": str(support_raster_path),
        },
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }

    with open(
        outputs["summary_json"],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(summary),
            handle,
            indent=2,
        )

    return {
        "map_source": "fixed_A30_E70_ensemble",
        "probability_raster": str(
            probability_raster_path
        ),
        "statistics": statistics,
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }


def expected_scenario_map_path(scenario_name: str) -> Path:
    """Return the canonical raster path written by predict_scenario_raster()."""
    return (
        Config.MAP_DIR
        / f"{scenario_prefix(scenario_name)}_XGB_probability_0_1_common_AF_domain.tif"
    )


def clean_probability_array(
    array: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    """
    Convert nodata to NaN and verify that susceptibility probabilities remain
    on [0,1]. Only tiny floating-point excursions are clipped.
    """
    output = np.asarray(array, dtype="float64").copy()

    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if np.isfinite(nodata_value):
                output[output == nodata_value] = np.nan
        except (TypeError, ValueError):
            pass

    output[output == -9999.0] = np.nan
    output[output == -32768.0] = np.nan
    output[np.abs(output) > 1.0e20] = np.nan

    outside = (
        np.isfinite(output)
        & ((output < -1.0e-6) | (output > 1.0 + 1.0e-6))
    )
    if outside.any():
        raise ValueError(
            f"Susceptibility raster contains {int(outside.sum()):,} values "
            "outside [0,1]. Recheck final probability prediction."
        )

    # Floating-point safeguard only; this is not renormalization.
    output = np.where(
        np.isfinite(output),
        np.clip(output, 0.0, 1.0),
        np.nan,
    )
    return output


def read_probability_raster(
    raster_path: Path,
) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(raster_path, "XGBoost susceptibility probability raster")

    with rasterio.open(raster_path) as dataset:
        if dataset.crs is None:
            raise ValueError(
                f"Susceptibility raster has no CRS: {raster_path}"
            )

        probability = clean_probability_array(
            dataset.read(1),
            dataset.nodata,
        )

        reference = {
            "crs": dataset.crs,
            "transform": dataset.transform,
            "width": dataset.width,
            "height": dataset.height,
            "bounds": dataset.bounds,
            "epsg": dataset.crs.to_epsg(),
            "path": str(raster_path),
        }

    if not np.any(np.isfinite(probability)):
        raise RuntimeError(
            f"No valid probabilities found in susceptibility raster: {raster_path}"
        )

    return probability, reference


def to_dms(value: float, is_lon: bool = True) -> str:
    absolute = abs(value)
    degrees = int(absolute)
    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_lon
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def add_north_arrow_tri(
    axis,
    size: float = 0.09,
    pad: float = 0.07,
    label: str = "N",
    arrow_downshift: float = 0.025,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=Config.NORTH_ARROW_FONTSIZE,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base
        if candidate >= target_metres:
            return candidate

    return 10 * base


def add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    crs_type: str = "utm",
    mid_lat_deg: float | None = None,
    segments: int = 4,
    fontsize: float = 4,
    frame_lw: float = 0.4,
) -> None:
    width_data = east - west
    height_data = north - south

    if crs_type == "utm":
        width_metres = width_data
        data_units_per_metre = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = 0.5 * (south + north)

        metres_per_degree = metres_per_degree_lon(mid_lat_deg)
        width_metres = width_data * metres_per_degree
        data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = total_length_metres * data_units_per_metre
    segment_length_units = segment_length_metres * data_units_per_metre

    x_start = west + 0.5 * width_data - 0.5 * total_length_units
    y_base = south + 0.06 * height_data
    bar_height = 0.018 * height_data

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_data

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (
                    x_start + index * segment_length_units,
                    y_base,
                ),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=(
                    "black" if index % 2 == 0 else "white"
                ),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_data,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_data,
        "0",
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_data,
        end_label,
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )


def load_vector_overlay(
    vector_path: Path,
    target_crs,
):
    """Load vector data and transform it to the raster CRS."""
    if not vector_path.exists():
        log(f"[VIZ-WARN] Vector overlay not found: {vector_path}")
        return None

    target_crs_text = str(target_crs)

    if HAVE_GPD:
        try:
            data = gpd.read_file(vector_path)
            if data.crs is None:
                log(
                    f"[VIZ-WARN] {vector_path.name} has no CRS; "
                    "assuming EPSG:4326."
                )
                data = data.set_crs(epsg=4326)
            return data.to_crs(target_crs_text)
        except Exception as exc:
            log(
                f"[VIZ-WARN] GeoPandas loading failed for "
                f"{vector_path.name}: {exc}"
            )

    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(vector_path, "r") as source:
                source_crs = source.crs_wkt or source.crs
                if not source_crs:
                    source_crs = "EPSG:4326"
                    log(
                        f"[VIZ-WARN] {vector_path.name} has no CRS; "
                        "assuming EPSG:4326."
                    )

                for feature in source:
                    geometry = feature.get("geometry")
                    if geometry is None:
                        continue
                    geometries.append(
                        transform_geom(
                            source_crs,
                            target_crs_text,
                            geometry,
                        )
                    )

            return {"geometries": geometries}

        except Exception as exc:
            log(
                f"[VIZ-WARN] Fiona loading failed for "
                f"{vector_path.name}: {exc}"
            )

    return None


def plot_vector_overlay(
    axis,
    vector_data,
    colour: str,
    linewidth: float,
    zorder: int,
    polygon_boundary_only: bool = False,
) -> None:
    if vector_data is None:
        return

    if HAVE_GPD and hasattr(vector_data, "plot"):
        try:
            if polygon_boundary_only:
                vector_data.boundary.plot(
                    ax=axis,
                    color=colour,
                    linewidth=linewidth,
                    zorder=zorder,
                )
            else:
                vector_data.plot(
                    ax=axis,
                    color=colour,
                    linewidth=linewidth,
                    zorder=zorder,
                )
            return
        except Exception as exc:
            log(f"[VIZ-WARN] GeoPandas plotting failed: {exc}")

    if not isinstance(vector_data, dict):
        return

    for geometry in vector_data.get("geometries", []):
        if geometry is None:
            continue

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")
        if coordinates is None:
            continue

        if geometry_type == "LineString":
            line_groups = [coordinates]
        elif geometry_type == "MultiLineString":
            line_groups = coordinates
        elif geometry_type == "Polygon":
            line_groups = [coordinates[0]]
        elif geometry_type == "MultiPolygon":
            line_groups = [
                polygon[0]
                for polygon in coordinates
                if polygon
            ]
        else:
            continue

        for line in line_groups:
            xy = np.asarray(line, dtype="float64")
            if xy.ndim != 2 or xy.shape[1] < 2:
                continue
            axis.plot(
                xy[:, 0],
                xy[:, 1],
                color=colour,
                linewidth=linewidth,
                zorder=zorder,
            )



def build_clip_patch_from_vector_overlay(
    vector_data,
    axis,
):
    """
    Build a compound Matplotlib PathPatch from the study-area polygon.

    The raster artist is clipped to this polygon rather than relying on
    transparent pixels outside the study area. The axes and figure backgrounds
    remain opaque white, which is safer for PNG/PDF/SVG and later SVG-to-EMF
    conversion in Word.

    Supports:
      1. GeoPandas GeoDataFrame / GeoSeries input.
      2. Fiona fallback GeoJSON-style geometry dictionaries.

    Returns
    -------
    PathPatch or None
        Invisible patch suitable for image.set_clip_path().
    """
    if vector_data is None:
        return None

    all_vertices: list[np.ndarray] = []
    all_codes: list[int] = []

    def add_ring(ring_coordinates) -> None:
        coordinates = np.asarray(ring_coordinates, dtype="float64")

        if (
            coordinates.ndim != 2
            or coordinates.shape[0] < 3
            or coordinates.shape[1] < 2
        ):
            return

        coordinates = coordinates[:, :2]

        # Ensure the ring is explicitly closed.
        if not np.allclose(
            coordinates[0],
            coordinates[-1],
            equal_nan=False,
        ):
            coordinates = np.vstack(
                [coordinates, coordinates[0]]
            )

        n_vertices = len(coordinates)
        codes = (
            [MplPath.MOVETO]
            + [MplPath.LINETO] * (n_vertices - 2)
            + [MplPath.CLOSEPOLY]
        )

        all_vertices.append(coordinates)
        all_codes.extend(codes)

    def add_shapely_geometry(geometry) -> None:
        if geometry is None or getattr(geometry, "is_empty", True):
            return

        geometry_type = geometry.geom_type

        if geometry_type == "Polygon":
            add_ring(geometry.exterior.coords)
            for interior in geometry.interiors:
                add_ring(interior.coords)

        elif geometry_type == "MultiPolygon":
            for polygon in geometry.geoms:
                add_shapely_geometry(polygon)

        elif geometry_type == "GeometryCollection":
            for sub_geometry in geometry.geoms:
                add_shapely_geometry(sub_geometry)

    def add_geojson_geometry(geometry: dict[str, Any]) -> None:
        if not geometry:
            return

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")

        if coordinates is None:
            return

        if geometry_type == "Polygon":
            for ring in coordinates:
                add_ring(ring)

        elif geometry_type == "MultiPolygon":
            for polygon in coordinates:
                for ring in polygon:
                    add_ring(ring)

        elif geometry_type == "GeometryCollection":
            for sub_geometry in geometry.get("geometries", []):
                add_geojson_geometry(sub_geometry)

    # GeoPandas path.
    if HAVE_GPD and hasattr(vector_data, "geometry"):
        try:
            for geometry in vector_data.geometry:
                add_shapely_geometry(geometry)
        except Exception as exc:
            log(
                "[VIZ-WARN] Could not build GeoPandas clip geometry: "
                f"{exc}"
            )

    # Fiona / GeoJSON fallback path.
    elif isinstance(vector_data, dict):
        for geometry in vector_data.get("geometries", []):
            add_geojson_geometry(geometry)

    if not all_vertices:
        return None

    compound_path = MplPath(
        np.concatenate(all_vertices, axis=0),
        all_codes,
    )

    clip_patch = PathPatch(
        compound_path,
        facecolor="none",
        edgecolor="none",
        linewidth=0.0,
        transform=axis.transData,
        zorder=0,
    )

    # Adding the invisible patch initializes the axes data transform correctly.
    axis.add_patch(clip_patch)

    return clip_patch


def calculate_lsm_statistics(values: np.ndarray) -> dict[str, float | int]:
    return {
        "valid_pixels": int(values.size),
        "minimum": float(np.min(values)),
        "maximum": float(np.max(values)),
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "standard_deviation": float(np.std(values)),
        "p05": float(np.percentile(values, 5.0)),
        "p25": float(np.percentile(values, 25.0)),
        "p75": float(np.percentile(values, 75.0)),
        "p95": float(np.percentile(values, 95.0)),
    }


def lsm_visualization_output_paths(
    scenario_name: str,
) -> dict[str, Path]:
    prefix = scenario_prefix(scenario_name)
    stem = Config.VISUALIZATION_DIR / f"{prefix}_XGB_Landslide_Susceptibility"

    return {
        "map_png": Path(str(stem) + "_Map.png"),
        "map_pdf": Path(str(stem) + "_Map.pdf"),
        "map_svg": Path(str(stem) + "_Map.svg"),
        "histogram_png": Path(str(stem) + "_Histogram.png"),
        "histogram_pdf": Path(str(stem) + "_Histogram.pdf"),
        "histogram_svg": Path(str(stem) + "_Histogram.svg"),
        "summary_json": Path(str(stem) + "_Visualization_Summary.json"),
        "statistics_csv": Path(str(stem) + "_Statistics.csv"),
    }


def create_lsm_probability_map(
    probability: np.ndarray,
    reference: dict[str, Any],
    scenario_name: str,
    outputs: dict[str, Path],
) -> None:
    bounds = reference["bounds"]

    raster_extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    buffer_x = (
        bounds.right - bounds.left
    ) * Config.DISPLAY_BUFFER_FRACTION
    buffer_y = (
        bounds.top - bounds.bottom
    ) * Config.DISPLAY_BUFFER_FRACTION

    west = bounds.left - buffer_x
    east = bounds.right + buffer_x
    south = bounds.bottom - buffer_y
    north = bounds.top + buffer_y

    display_array = np.ma.masked_invalid(probability)

    # Use an opaque white colour for masked / nodata cells.
    # This avoids transparent raster pixels in exported figures.
    map_cmap = LSM_CMAP.copy()
    map_cmap.set_bad(
        color="white",
        alpha=1.0,
    )

    raster_crs = reference["crs"]
    crs_is_geographic = bool(
        raster_crs and raster_crs.is_geographic
    )

    x_mid = 0.5 * (bounds.left + bounds.right)
    y_mid = 0.5 * (bounds.bottom + bounds.top)

    if crs_is_geographic:
        def format_longitude(value, position):
            return to_dms(value, True)

        def format_latitude(value, position):
            return to_dms(value, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError(
                "Projected susceptibility raster requires pyproj "
                "for DMS axis labels."
            )

        to_geographic = Transformer.from_crs(
            raster_crs,
            "EPSG:4326",
            always_xy=True,
        )

        def format_longitude(value, position):
            longitude, _ = to_geographic.transform(
                value,
                y_mid,
            )
            return to_dms(longitude, True)

        def format_latitude(value, position):
            _, latitude = to_geographic.transform(
                x_mid,
                value,
            )
            return to_dms(latitude, False)

    roads = load_vector_overlay(
        Config.ROAD_SHP,
        raster_crs,
    )
    study_boundary = load_vector_overlay(
        Config.STUDY_SHP,
        raster_crs,
    )

    style = {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "axes.unicode_minus": False,
    }

    with plt.rc_context(style):
        figure, axis = plt.subplots(
            1,
            1,
            figsize=Config.MAP_FIGSIZE,
            facecolor="white",
        )
        axis.set_facecolor("white")

        # Keep both the full figure canvas and axes background explicitly
        # opaque white. This is important for Word/EMF workflows.
        figure.patch.set_facecolor("white")
        figure.patch.set_alpha(1.0)
        axis.set_facecolor("white")

        image = axis.imshow(
            display_array,
            cmap=map_cmap,
            extent=raster_extent,
            vmin=0.0,
            vmax=1.0,
            origin="upper",
            interpolation=Config.RASTER_DISPLAY_INTERPOLATION,
            alpha=1.0,
            zorder=1,
        )

        # EMF-safe background handling used in the earlier LSM visualization:
        # clip the raster artist to the study-area polygon instead of leaving
        # transparent raster cells outside the boundary. The opaque white axes
        # background therefore shows through outside the study area.
        clip_patch = build_clip_patch_from_vector_overlay(
            study_boundary,
            axis,
        )
        if clip_patch is not None:
            image.set_clip_path(clip_patch)
            log(
                "[VIZ-MAP] Raster clipped to study boundary; "
                "opaque white background retained."
            )
        else:
            log(
                "[VIZ-WARN] Study-area clip path could not be created. "
                "Masked cells will still render as opaque white."
            )

        axis.set_xlim(west, east)
        axis.set_ylim(south, north)

        # Established Step 5.3 overlays.
        plot_vector_overlay(
            axis,
            study_boundary,
            colour="black",
            linewidth=0.7,
            zorder=800,
            polygon_boundary_only=True,
        )
        plot_vector_overlay(
            axis,
            roads,
            colour="red",
            linewidth=0.7,
            zorder=850,
            polygon_boundary_only=False,
        )

        axis.xaxis.set_major_locator(
            LinearLocator(Config.LONGITUDE_DMS_TICK_COUNT)
        )
        axis.yaxis.set_major_locator(
            LinearLocator(Config.LONGITUDE_DMS_TICK_COUNT)
        )
        axis.xaxis.set_major_formatter(
            FuncFormatter(format_longitude)
        )
        axis.yaxis.set_major_formatter(
            FuncFormatter(format_latitude)
        )
        axis.tick_params(
            axis="both",
            which="major",
            labelsize=5,
        )

        for tick_label in axis.get_xticklabels():
            tick_label.set_rotation(45)
            tick_label.set_ha("right")
            tick_label.set_fontsize(5)

        for tick_label in axis.get_yticklabels():
            tick_label.set_fontsize(5)

        add_north_arrow_tri(
            axis,
            size=0.09,
            pad=0.09,
        )

        crs_type = (
            "wgs84" if crs_is_geographic else "utm"
        )

        add_checker_scale_bar(
            axis,
            west,
            east,
            south,
            north,
            crs_type=crs_type,
            mid_lat_deg=(
                y_mid if crs_is_geographic else None
            ),
            segments=4,
            fontsize=4,
        )

        legend_elements = [
            Line2D(
                [0],
                [0],
                color="red",
                linewidth=1.0,
                label="Highway",
            ),
            Line2D(
                [0],
                [0],
                color="black",
                linewidth=1.0,
                label="Study Area",
            ),
        ]

        legend = axis.legend(
            handles=legend_elements,
            loc="center",
            bbox_to_anchor=(0.82, 0.35),
            bbox_transform=axis.transAxes,
            frameon=True,
            framealpha=0.9,
            fontsize=2.5,
            labelspacing=0.3,
            borderpad=0.3,
            handlelength=1.2,
            handletextpad=0.4,
            borderaxespad=0.0,
        )
        legend.get_frame().set_edgecolor("lightgray")
        legend.get_frame().set_linewidth(0.3)

        divider = make_axes_locatable(axis)
        colorbar_axis = divider.append_axes(
            "right",
            size=Config.CAX_SIZE,
            pad=Config.CAX_PAD,
        )

        colorbar = figure.colorbar(
            image,
            cax=colorbar_axis,
        )
        colorbar.ax.tick_params(labelsize=5)
        colorbar.set_label(
            "Landslide susceptibility probability (0–1)",
            fontsize=4,
            fontweight="bold",
        )
        colorbar.set_ticks(
            np.linspace(0.0, 1.0, 5)
        )

        figure.tight_layout()

        if Config.SAVE_LSM_FIGURES:
            figure.savefig(
                outputs["map_png"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["map_pdf"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["map_svg"],
                format="svg",
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )

            log(f"[VIZ-MAP] {outputs['map_png']}")
            log(f"[VIZ-MAP] {outputs['map_pdf']}")
            log(f"[VIZ-MAP] {outputs['map_svg']}")

        if Config.SHOW_LSM_FIGURES:
            # Use Matplotlib's standard display path. This is more reliable
            # across Jupyter, Spyder, VS Code, and standard Python sessions.
            plt.show()

        plt.close(figure)


def create_lsm_probability_histogram(
    values: np.ndarray,
    statistics: dict[str, float | int],
    outputs: dict[str, Path],
) -> None:
    style = {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "axes.unicode_minus": False,
    }

    with plt.rc_context(style):
        figure, axis = plt.subplots(
            1,
            1,
            figsize=Config.HIST_FIGSIZE,
            facecolor="white",
        )

        bins = np.linspace(0.0, 1.0, 21)
        counts, edges = np.histogram(values, bins=bins)
        centres = 0.5 * (edges[:-1] + edges[1:])
        colours = LSM_CMAP(centres)

        axis.bar(
            centres,
            counts,
            width=(bins[1] - bins[0]) * 0.9,
            color=colours,
            edgecolor="black",
            linewidth=0.4,
            alpha=0.9,
        )

        axis.axvline(
            statistics["mean"],
            color="darkred",
            linestyle="--",
            linewidth=0.7,
            label=f"Mean: {statistics['mean']:.3f}",
        )
        axis.axvline(
            statistics["median"],
            color="orange",
            linestyle="--",
            linewidth=0.7,
            label=f"Median: {statistics['median']:.3f}",
        )
        axis.axvline(
            statistics["p95"],
            color="green",
            linestyle="--",
            linewidth=0.7,
            label=f"95th %ile: {statistics['p95']:.3f}",
        )

        axis.legend(
            fontsize=3,
            loc="upper right",
            framealpha=0.9,
        )
        axis.set_xlabel(
            "Landslide susceptibility probability (0–1)",
            fontsize=4,
            fontweight="bold",
        )
        axis.set_ylabel(
            "Number of Pixels",
            fontsize=4,
            fontweight="bold",
        )
        axis.tick_params(
            axis="both",
            which="major",
            labelsize=4,
        )
        axis.set_xlim(0.0, 1.0)
        axis.grid(
            True,
            alpha=0.25,
            linestyle="-",
            linewidth=0.4,
            axis="y",
        )
        axis.set_axisbelow(True)

        figure.tight_layout(pad=1.2)

        if Config.SAVE_LSM_FIGURES:
            figure.savefig(
                outputs["histogram_png"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["histogram_pdf"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["histogram_svg"],
                format="svg",
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )

            log(f"[VIZ-HIST] {outputs['histogram_png']}")
            log(f"[VIZ-HIST] {outputs['histogram_pdf']}")
            log(f"[VIZ-HIST] {outputs['histogram_svg']}")

        if Config.SHOW_LSM_FIGURES:
            # Use Matplotlib's standard display path. This is more reliable
            # across Jupyter, Spyder, VS Code, and standard Python sessions.
            plt.show()

        plt.close(figure)


def create_primary_lsm_visualization(
    raster_path: Path,
    scenario_name: str,
) -> dict[str, Any]:
    """
    Create the primary Model E susceptibility map and histogram using the
    established Step 5.3 cartographic style.
    """
    probability, reference = read_probability_raster(
        raster_path
    )

    values = probability[np.isfinite(probability)]
    statistics = calculate_lsm_statistics(values)
    outputs = lsm_visualization_output_paths(
        scenario_name
    )

    create_lsm_probability_map(
        probability=probability,
        reference=reference,
        scenario_name=scenario_name,
        outputs=outputs,
    )

    create_lsm_probability_histogram(
        values=values,
        statistics=statistics,
        outputs=outputs,
    )

    statistics_df = pd.DataFrame(
        [
            {
                "scenario": scenario_name,
                "model_label": Config.MODEL_SHORT_LABELS[scenario_name],
                "model_description": Config.MODEL_DESCRIPTIONS[scenario_name],
                **statistics,
            }
        ]
    )
    statistics_df.to_csv(
        outputs["statistics_csv"],
        index=False,
    )

    summary = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "XGBoost",
        "scenario": scenario_name,
        "model_label": Config.MODEL_SHORT_LABELS[scenario_name],
        "model_description": Config.MODEL_DESCRIPTIONS[scenario_name],
        "probability_raster": str(raster_path),
        "prediction_domain": "common_A_to_F_valid_domain",
        "validation_note": (
            "This figure is generated from the final full-data deployment model. "
            "Model performance must be reported from nested spatial OOF predictions, "
            "not from the full-data map."
        ),
        "scientific_plotting": {
            "direct_probability_raster": True,
            "probabilities_renormalized_for_visualization": False,
            "RBF_used": False,
            "IDW_used": False,
            "cubic_interpolation_used": False,
            "display_interpolation": Config.RASTER_DISPLAY_INTERPOLATION,
        },
        "statistics": statistics,
        "visual_style": {
            "reference_style": "Step 5.3 HKFI visualization",
            "map_figsize_inches": list(Config.MAP_FIGSIZE),
            "histogram_figsize_inches": list(Config.HIST_FIGSIZE),
            "display_buffer_fraction": Config.DISPLAY_BUFFER_FRACTION,
            "background": "opaque_white",
            "transparent_export": False,
            "study_area_vector_clip": True,
            "masked_raster_colour": "opaque_white",
            "DMS_axes": True,
            "north_arrow": True,
            "checkerboard_scale_bar": True,
            "highway_overlay": True,
            "study_area_overlay": True,
            "compact_legend": True,
            "side_colour_bar": True,
            "colour_ramp": "green-yellow-orange-red susceptibility ramp",
        },
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }

    with open(
        outputs["summary_json"],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(summary),
            handle,
            indent=2,
        )

    log(
        "[VIZ] Primary XGBoost landslide susceptibility visualization "
        f"completed for {Config.MODEL_SHORT_LABELS[scenario_name]}."
    )

    return {
        "scenario": scenario_name,
        "raster_path": str(raster_path),
        "statistics": statistics,
        "susceptibility_area_percentages": area_percentage_df.to_dict(orient="records"),
        "support_raster": str(support_raster_path),
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }


# =============================================================================
# SUMMARIES / COMPARISONS / FIGURES
# =============================================================================
def summarize_oof_results(
    all_oof: pd.DataFrame,
    all_fold_metrics: pd.DataFrame,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for scenario_name in Config.SCENARIO_ORDER:
        subset = all_oof.loc[all_oof["scenario"] == scenario_name].copy()
        fold_subset = all_fold_metrics.loc[
            all_fold_metrics["scenario"] == scenario_name
        ].copy()

        y = subset[Config.TARGET_COLUMN].to_numpy(dtype=int)
        probability = subset["oof_probability"].to_numpy(dtype=float)
        blocks = subset[Config.FOLD_COLUMN].to_numpy(dtype=int)

        pooled = classification_metrics(y, probability)
        ci_low, ci_high, n_valid_boot = cluster_bootstrap_auc_ci(
            y_true=y,
            y_prob=probability,
            blocks=blocks,
            n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
            seed=Config.RANDOM_STATE + Config.SCENARIO_ORDER.index(scenario_name),
            ci_level=Config.CI_LEVEL,
        )

        row = {
            "scenario": scenario_name,
            "model": Config.MODEL_SHORT_LABELS[scenario_name],
            "description": Config.MODEL_DESCRIPTIONS[scenario_name],
            "n_samples": int(len(subset)),
            "n_predictors": int(fold_subset["n_predictors"].iloc[0]),
            "oof_roc_auc": pooled["roc_auc"],
            "oof_auc_cluster_bootstrap_ci_low": ci_low,
            "oof_auc_cluster_bootstrap_ci_high": ci_high,
            "cluster_bootstrap_valid_iterations": int(n_valid_boot),
            "oof_pr_auc": pooled["pr_auc"],
            "oof_brier": pooled["brier"],
            "oof_accuracy_0p5": pooled["accuracy"],
            "oof_balanced_accuracy_0p5": pooled["balanced_accuracy"],
            "oof_precision_0p5": pooled["precision"],
            "oof_recall_0p5": pooled["recall"],
            "oof_f1_0p5": pooled["f1"],
            "fold_auc_mean": float(fold_subset["roc_auc"].mean()),
            "fold_auc_std": float(fold_subset["roc_auc"].std(ddof=1)),
            "fold_auc_min": float(fold_subset["roc_auc"].min()),
            "fold_auc_max": float(fold_subset["roc_auc"].max()),
        }
        rows.append(row)

    summary = pd.DataFrame(rows)

    baseline_auc = float(
        summary.loc[
            summary["scenario"] == "A_static_baseline",
            "oof_roc_auc",
        ].iloc[0]
    )

    summary["absolute_auc_gain_vs_A"] = summary["oof_roc_auc"] - baseline_auc
    summary["relative_auc_gain_vs_A_percent"] = np.where(
        baseline_auc != 0,
        100.0 * summary["absolute_auc_gain_vs_A"] / baseline_auc,
        np.nan,
    )

    return summary


def make_paired_auc_comparisons(all_oof: pd.DataFrame) -> pd.DataFrame:
    comparisons = [
        ("B_raw_deformation", "A_static_baseline", "B - A", "Raw DV+ added value"),
        ("C_hkfi_only", "A_static_baseline", "C - A", "HKFI added value"),
        ("D_sndr_only", "A_static_baseline", "D - A", "SNDR added value"),
        ("E_physics_guided", "A_static_baseline", "E - A", "Proposed indices total added value"),
        ("F_untransformed_sources", "A_static_baseline", "F - A", "Separate source variables added value"),
        ("E_physics_guided", "B_raw_deformation", "E - B", "Physics-guided versus raw DV+"),
        ("E_physics_guided", "F_untransformed_sources", "E - F", "Indices versus separate source variables"),
    ]

    pivot = all_oof.pivot(
        index=Config.SAMPLE_ID_COLUMN,
        columns="scenario",
        values="oof_probability",
    )

    metadata = (
        all_oof.loc[
            all_oof["scenario"] == "A_static_baseline",
            [Config.SAMPLE_ID_COLUMN, Config.TARGET_COLUMN, Config.FOLD_COLUMN],
        ]
        .drop_duplicates(Config.SAMPLE_ID_COLUMN)
        .set_index(Config.SAMPLE_ID_COLUMN)
        .loc[pivot.index]
    )

    y = metadata[Config.TARGET_COLUMN].to_numpy(dtype=int)
    blocks = metadata[Config.FOLD_COLUMN].to_numpy(dtype=int)

    rows: List[Dict[str, Any]] = []
    for comparison_index, (left, right, label, purpose) in enumerate(comparisons):
        observed, ci_low, ci_high, n_valid = paired_cluster_bootstrap_auc_difference(
            y_true=y,
            prob_left=pivot[left].to_numpy(dtype=float),
            prob_right=pivot[right].to_numpy(dtype=float),
            blocks=blocks,
            n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
            seed=Config.RANDOM_STATE + 100 + comparison_index,
            ci_level=Config.CI_LEVEL,
        )

        rows.append(
            {
                "comparison": label,
                "purpose": purpose,
                "left_scenario": left,
                "right_scenario": right,
                "auc_difference_left_minus_right": observed,
                "cluster_bootstrap_ci_low": ci_low,
                "cluster_bootstrap_ci_high": ci_high,
                "cluster_bootstrap_valid_iterations": int(n_valid),
            }
        )

    return pd.DataFrame(rows)


def plot_oof_roc_curves(all_oof: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(5.0, 5.0))

    for scenario_name in Config.SCENARIO_ORDER:
        subset = all_oof.loc[all_oof["scenario"] == scenario_name]
        y = subset[Config.TARGET_COLUMN].to_numpy(dtype=int)
        probability = subset["oof_probability"].to_numpy(dtype=float)
        fpr, tpr, _ = roc_curve(y, probability)
        auc = roc_auc_score(y, probability)
        label = f"{Config.MODEL_SHORT_LABELS[scenario_name]} (AUC={auc:.3f})"
        ax.plot(fpr, tpr, linewidth=1.4, label=label)

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.0)
    ax.set_xlabel("False Positive Rate", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("True Positive Rate", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.legend(frameon=False, fontsize=9, loc="lower right")
    ax.grid(True, alpha=0.25)
    apply_axis_text_style(ax, legend_fontsize=9)
    save_figure_triplet(
        fig,
        Config.ROC_FIG_PNG,
        Config.ROC_FIG_PDF,
        Config.ROC_FIG_SVG,
        dpi=600,
    )


def plot_fold_auc(all_fold_metrics: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(5.5, 4.0))

    x_positions = np.arange(1, len(Config.SCENARIO_ORDER) + 1)
    for x, scenario_name in zip(x_positions, Config.SCENARIO_ORDER):
        values = all_fold_metrics.loc[
            all_fold_metrics["scenario"] == scenario_name,
            "roc_auc",
        ].to_numpy(dtype=float)
        ax.scatter(np.full(values.shape, x), values, s=28)
        ax.plot(
            [x - 0.16, x + 0.16],
            [values.mean(), values.mean()],
            linewidth=1.4,
        )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [Config.MODEL_SHORT_LABELS[name] for name in Config.SCENARIO_ORDER]
    )
    ax.set_xlabel("Model configuration", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("Held-out spatial-fold AUC", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylim(0.0, 1.02)
    ax.grid(True, axis="y", alpha=0.25)
    apply_axis_text_style(ax)
    save_figure_triplet(
        fig,
        Config.FOLD_AUC_FIG_PNG,
        Config.FOLD_AUC_FIG_PDF,
        Config.FOLD_AUC_FIG_SVG,
        dpi=600,
    )


# =============================================================================
# REPORTING
# =============================================================================

def write_manuscript_performance_excel(
    summary_df: pd.DataFrame,
    comparison_df: pd.DataFrame,
    all_fold_metrics: pd.DataFrame,
    ensemble_oof_metrics: Dict[str, Any],
) -> Path:
    """
    Export manuscript-ready numerical tables to a single Excel workbook.

    Sheets
    ------
    1. A-F Performance
       Main manuscript table:
       Model | Predictor configuration | OOF ROC-AUC | 95% CI | PR-AUC |
       Balanced Accuracy | F1-score

    2. AUC Comparisons
       Reviewer-focused paired delta-AUC comparisons with cluster-bootstrap CIs.

    3. Fold AUC
       Fold-specific held-out spatial AUC values plus mean and SD.

    4. Ensemble Summary
       Fixed A30/E70 ensemble OOF performance, reported separately from A-F.
    """
    predictor_labels = {
        "A_static_baseline": "Static12",
        "B_raw_deformation": "Static12 + DV+",
        "C_hkfi_only": "Static12 + HKFI",
        "D_sndr_only": "Static12 + SNDR",
        "E_physics_guided": "Static12 + HKFI + SNDR",
        "F_untransformed_sources": "Static12 + DV+ + TWI + support-preserved SM",
    }

    # ---------------- Main A-F manuscript table ----------------
    ordered_summary = (
        summary_df.set_index("scenario")
        .loc[Config.SCENARIO_ORDER]
        .reset_index()
    )

    main_table = pd.DataFrame(
        {
            "Model": ordered_summary["model"],
            "Predictor configuration": [
                predictor_labels.get(name, Config.MODEL_DESCRIPTIONS[name])
                for name in ordered_summary["scenario"]
            ],
            "OOF ROC-AUC": ordered_summary["oof_roc_auc"],
            "95% CI": [
                (
                    f"{low:.3f}–{high:.3f}"
                    if np.isfinite(low) and np.isfinite(high)
                    else ""
                )
                for low, high in zip(
                    ordered_summary["oof_auc_cluster_bootstrap_ci_low"],
                    ordered_summary["oof_auc_cluster_bootstrap_ci_high"],
                )
            ],
            "CI lower": ordered_summary["oof_auc_cluster_bootstrap_ci_low"],
            "CI upper": ordered_summary["oof_auc_cluster_bootstrap_ci_high"],
            "PR-AUC": ordered_summary["oof_pr_auc"],
            "Balanced Accuracy": ordered_summary["oof_balanced_accuracy_0p5"],
            "F1-score": ordered_summary["oof_f1_0p5"],
            "Fold AUC mean": ordered_summary["fold_auc_mean"],
            "Fold AUC SD": ordered_summary["fold_auc_std"],
        }
    )

    # ---------------- Paired delta-AUC comparison table ----------------
    comparison_table = pd.DataFrame(
        {
            "Comparison": comparison_df["comparison"],
            "ΔAUC": comparison_df["auc_difference_left_minus_right"],
            "95% CI": [
                (
                    f"{low:+.3f}–{high:+.3f}"
                    if np.isfinite(low) and np.isfinite(high)
                    else ""
                )
                for low, high in zip(
                    comparison_df["cluster_bootstrap_ci_low"],
                    comparison_df["cluster_bootstrap_ci_high"],
                )
            ],
            "CI lower": comparison_df["cluster_bootstrap_ci_low"],
            "CI upper": comparison_df["cluster_bootstrap_ci_high"],
            "Interpretation": comparison_df["purpose"],
        }
    )

    # ---------------- Fold-specific AUC table ----------------
    fold_table = (
        all_fold_metrics.pivot(
            index="scenario",
            columns="outer_fold",
            values="roc_auc",
        )
        .reindex(Config.SCENARIO_ORDER)
        .reset_index()
    )

    fold_table.insert(
        1,
        "Model",
        [
            Config.MODEL_SHORT_LABELS[scenario_name]
            for scenario_name in fold_table["scenario"]
        ],
    )
    fold_table.insert(
        2,
        "Predictor configuration",
        [
            predictor_labels.get(
                scenario_name,
                Config.MODEL_DESCRIPTIONS[scenario_name],
            )
            for scenario_name in fold_table["scenario"]
        ],
    )

    fold_columns = [
        column
        for column in fold_table.columns
        if isinstance(column, (int, np.integer))
    ]
    fold_columns = sorted(fold_columns)

    fold_table = fold_table.rename(
        columns={
            fold: f"Fold {fold}"
            for fold in fold_columns
        }
    )

    fold_metric_columns = [
        f"Fold {fold}"
        for fold in fold_columns
    ]
    fold_table["Mean"] = fold_table[fold_metric_columns].mean(axis=1)
    fold_table["SD"] = fold_table[fold_metric_columns].std(axis=1, ddof=1)
    fold_table = fold_table.drop(columns=["scenario"])

    # ---------------- Separate operational ensemble summary ----------------
    ensemble_table = pd.DataFrame(
        [
            {
                "Mapping product": "A30/E70 fixed operational ensemble",
                "Formula": "0.30 × Model A + 0.70 × Model E where E is available; otherwise Model A",
                "OOF ROC-AUC": ensemble_oof_metrics.get("roc_auc"),
                "95% CI": (
                    f"{ensemble_oof_metrics.get('auc_cluster_bootstrap_ci_low', np.nan):.3f}"
                    "–"
                    f"{ensemble_oof_metrics.get('auc_cluster_bootstrap_ci_high', np.nan):.3f}"
                ),
                "PR-AUC": ensemble_oof_metrics.get("pr_auc"),
                "Balanced Accuracy": ensemble_oof_metrics.get("balanced_accuracy"),
                "F1-score": ensemble_oof_metrics.get("f1"),
                "ΔAUC vs Model E": ensemble_oof_metrics.get(
                    "auc_difference_ensemble_minus_model_E"
                ),
                "ΔAUC 95% CI": (
                    f"{ensemble_oof_metrics.get('auc_difference_ci_low', np.nan):+.3f}"
                    "–"
                    f"{ensemble_oof_metrics.get('auc_difference_ci_high', np.nan):+.3f}"
                ),
                "Role in manuscript": (
                    "Final corridor-wide operational mapping product; "
                    "reported separately from the A–F ablation evaluation."
                ),
            }
        ]
    )

    # ---------------- Write Excel workbook ----------------
    output_path = Config.MANUSCRIPT_TABLES_XLSX

    try:
        # xlsxwriter provides publication-ready formatting when available.
        with pd.ExcelWriter(
            output_path,
            engine="xlsxwriter",
        ) as writer:
            main_table.to_excel(
                writer,
                sheet_name="A-F Performance",
                index=False,
            )
            comparison_table.to_excel(
                writer,
                sheet_name="AUC Comparisons",
                index=False,
            )
            fold_table.to_excel(
                writer,
                sheet_name="Fold AUC",
                index=False,
            )
            ensemble_table.to_excel(
                writer,
                sheet_name="Ensemble Summary",
                index=False,
            )

            workbook = writer.book

            header_format = workbook.add_format(
                {
                    "bold": True,
                    "font_color": "white",
                    "bg_color": "#1F4E78",
                    "border": 1,
                    "align": "center",
                    "valign": "vcenter",
                    "text_wrap": True,
                }
            )
            number_format = workbook.add_format(
                {
                    "num_format": "0.000",
                    "border": 1,
                    "align": "center",
                }
            )
            text_format = workbook.add_format(
                {
                    "border": 1,
                    "valign": "top",
                    "text_wrap": True,
                }
            )
            center_text_format = workbook.add_format(
                {
                    "border": 1,
                    "align": "center",
                    "valign": "center",
                    "text_wrap": True,
                }
            )

            sheet_specs = {
                "A-F Performance": {
                    "df": main_table,
                    "widths": [10, 45, 13, 18, 12, 12, 12, 18, 12, 15, 13],
                },
                "AUC Comparisons": {
                    "df": comparison_table,
                    "widths": [14, 12, 18, 12, 12, 42],
                },
                "Fold AUC": {
                    "df": fold_table,
                    "widths": [10, 45] + [12] * (len(fold_metric_columns) + 2),
                },
                "Ensemble Summary": {
                    "df": ensemble_table,
                    "widths": [32, 70, 13, 18, 12, 18, 12, 18, 18, 60],
                },
            }

            for sheet_name, spec in sheet_specs.items():
                worksheet = writer.sheets[sheet_name]
                dataframe = spec["df"]

                # Freeze the header row.
                worksheet.freeze_panes(1, 0)

                # Reformat headers.
                for column_index, column_name in enumerate(dataframe.columns):
                    worksheet.write(
                        0,
                        column_index,
                        column_name,
                        header_format,
                    )

                # Set useful column widths.
                for column_index, width in enumerate(spec["widths"]):
                    worksheet.set_column(
                        column_index,
                        column_index,
                        width,
                    )

                # Apply borders and numeric formatting to body cells.
                for row_index in range(1, len(dataframe) + 1):
                    for column_index, column_name in enumerate(dataframe.columns):
                        value = dataframe.iloc[row_index - 1, column_index]

                        if pd.isna(value):
                            worksheet.write_blank(
                                row_index,
                                column_index,
                                None,
                                center_text_format,
                            )
                        elif isinstance(value, (float, np.floating, int, np.integer)):
                            worksheet.write_number(
                                row_index,
                                column_index,
                                float(value),
                                number_format,
                            )
                        else:
                            worksheet.write(
                                row_index,
                                column_index,
                                str(value),
                                (
                                    center_text_format
                                    if column_name in {
                                        "Model",
                                        "Comparison",
                                        "95% CI",
                                        "ΔAUC 95% CI",
                                    }
                                    else text_format
                                ),
                            )

                # Add an Excel table for convenient filtering and manuscript review.
                if len(dataframe) > 0 and len(dataframe.columns) > 0:
                    worksheet.add_table(
                        0,
                        0,
                        len(dataframe),
                        len(dataframe.columns) - 1,
                        {
                            "style": "Table Style Medium 2",
                            "columns": [
                                {"header": column_name}
                                for column_name in dataframe.columns
                            ],
                        },
                    )

    except (ImportError, ModuleNotFoundError):
        # Fallback: still create a valid workbook using the default pandas engine.
        # The numerical content remains identical, but styling may be simpler.
        with pd.ExcelWriter(output_path) as writer:
            main_table.to_excel(
                writer,
                sheet_name="A-F Performance",
                index=False,
            )
            comparison_table.to_excel(
                writer,
                sheet_name="AUC Comparisons",
                index=False,
            )
            fold_table.to_excel(
                writer,
                sheet_name="Fold AUC",
                index=False,
            )
            ensemble_table.to_excel(
                writer,
                sheet_name="Ensemble Summary",
                index=False,
            )

    log(
        "[EXCEL] Manuscript-ready performance tables saved: "
        f"{output_path}"
    )

    return output_path


def write_qa_report(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
    summary_df: pd.DataFrame,
    comparison_df: pd.DataFrame,
    final_params: Dict[str, Any],
    map_paths: Dict[str, str],
) -> None:
    with open(Config.QA_REPORT_TXT, "w", encoding="utf-8") as handle:
        handle.write("STEP 7.1 — XGBOOST MODELS A–F: NESTED SPATIAL CV QA REPORT\n")
        handle.write("=" * 88 + "\n\n")
        handle.write(f"Generated: {utc_now_iso()}\n")
        handle.write(f"Input dataset: {Config.MODEL_READY_DATASET_CSV}\n")
        handle.write(f"Rows: {len(df)}\n")
        handle.write(
            f"Class balance: positive={(df[Config.TARGET_COLUMN] == 1).sum()}, "
            f"negative={(df[Config.TARGET_COLUMN] == 0).sum()}\n"
        )
        handle.write(
            f"Frozen outer folds: {sorted(df[Config.FOLD_COLUMN].unique().tolist())}\n\n"
        )

        handle.write("VALIDATION DESIGN\n")
        handle.write("-" * 88 + "\n")
        handle.write(
            "Outer evaluation: five frozen contiguous spatial blocks; one block held out at a time.\n"
        )
        handle.write(
            "Inner tuning: four-block PredefinedSplit using only the outer-training data.\n"
        )
        handle.write(
            "No outer validation observations are used for hyperparameter selection or early stopping.\n"
        )
        handle.write(
            "All A–F models use identical samples, labels and outer folds.\n"
        )
        handle.write(
            "Each frozen outer fold contains exactly 33 positives and 33 pseudo-absences.\n"
        )
        handle.write(
            "Land cover is one-hot encoded inside each fitted pipeline; continuous XGB predictors are not scaled.\n"
        )
        handle.write(
            "Binary metrics use a fixed 0.50 probability threshold.\n\n"
        )

        handle.write("SCENARIOS\n")
        handle.write("-" * 88 + "\n")
        for scenario_name in Config.SCENARIO_ORDER:
            features = scenarios[scenario_name]["features"]
            handle.write(
                f"{Config.MODEL_SHORT_LABELS[scenario_name]}: "
                f"{Config.MODEL_DESCRIPTIONS[scenario_name]} "
                f"({len(features)} predictors)\n"
            )
        handle.write("\n")

        handle.write("NESTED OOF SUMMARY\n")
        handle.write("-" * 88 + "\n")
        handle.write(summary_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("PAIRED AUC COMPARISONS\n")
        handle.write("-" * 88 + "\n")
        handle.write(comparison_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("FINAL DEPLOYMENT MODELS\n")
        handle.write("-" * 88 + "\n")
        if final_params:
            for scenario_name in Config.SCENARIO_ORDER:
                info = final_params[scenario_name]
                handle.write(
                    f"{Config.MODEL_SHORT_LABELS[scenario_name]}: "
                    f"selection AUC={info['best_mean_spatial_cv_auc_for_model_selection_only']:.6f}; "
                    f"params={info['best_params']}\n"
                )
        else:
            handle.write(
                "Skipped in this run because RUN_FINAL_DEPLOYMENT_TUNING=False. "
                "Nested spatial OOF evaluation was completed independently.\n"
            )
        handle.write("\n")

        if map_paths:
            handle.write("RASTER MAPS\n")
            handle.write("-" * 88 + "\n")
            handle.write(
                "Maps are generated from separately tuned full-data deployment models on the common A–F valid domain.\n"
            )
            handle.write(
                "These map-model tuning scores are not reported as validation performance.\n"
            )
            for scenario_name, path in map_paths.items():
                handle.write(f"{scenario_name}: {path}\n")


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=" * 96)
    log("=== Step 7.1: XGBoost Models A–F with Nested Spatial Cross-Validation ===")
    log("=== EXPECTED RUNTIME: ~5,760 XGBoost fits (nested CV) + ~1,440 deployment fits ===")
    log("=== With n_jobs=1, expect several hours. Monitor progress via verbose output. ===")
    log(
        "[RUN-MODE] Final deployment tuning="
        f"{'ON' if Config.RUN_FINAL_DEPLOYMENT_TUNING else 'OFF'} | "
        "Raster maps="
        f"{'ON' if Config.GENERATE_RASTER_MAPS else 'OFF'} | "
        "Primary Model E visualization="
        f"{'ON' if Config.GENERATE_PRIMARY_LSM_VISUALIZATION else 'OFF'} | "
        "A30/E70 ensemble LSM="
        f"{'ON' if Config.GENERATE_AE_ENSEMBLE_LSM else 'OFF'}"
    )
    log(
        "[VIZ-OUT] Primary Landslide Susceptibility Map outputs: "
        f"{Config.VISUALIZATION_DIR}"
    )
    log("=" * 96)

    # 1. Validate frozen A–F design and primary frozen ML dataset.
    scenarios = validate_scenario_manifests()
    df = load_and_validate_model_ready_dataset(scenarios)

    all_oof_tables: List[pd.DataFrame] = []
    all_fold_tables: List[pd.DataFrame] = []
    all_inner_search_tables: List[pd.DataFrame] = []
    outer_best_params: Dict[str, Any] = {}

    # 2. Unbiased nested spatial CV for each A–F model.
    for scenario_name in Config.SCENARIO_ORDER:
        features = list(scenarios[scenario_name]["features"])

        oof_df, fold_df, params, search_df = run_nested_spatial_cv_for_scenario(
            df=df,
            scenario_name=scenario_name,
            features=features,
        )

        all_oof_tables.append(oof_df)
        all_fold_tables.append(fold_df)
        all_inner_search_tables.append(search_df)
        outer_best_params[scenario_name] = params

    all_oof = pd.concat(all_oof_tables, ignore_index=True)
    all_fold_metrics = pd.concat(all_fold_tables, ignore_index=True)
    all_inner_search = pd.concat(all_inner_search_tables, ignore_index=True)

    # Cross-scenario QA: every scenario must contain the exact same sample IDs.
    reference_ids = set(
        all_oof.loc[
            all_oof["scenario"] == Config.SCENARIO_ORDER[0],
            Config.SAMPLE_ID_COLUMN,
        ]
    )
    for scenario_name in Config.SCENARIO_ORDER[1:]:
        scenario_ids = set(
            all_oof.loc[
                all_oof["scenario"] == scenario_name,
                Config.SAMPLE_ID_COLUMN,
            ]
        )
        if scenario_ids != reference_ids:
            raise RuntimeError(
                f"OOF sample IDs differ for {scenario_name}; A–F comparison invalid."
            )

    # 3. Summaries and reviewer-focused paired comparisons.
    summary_df = summarize_oof_results(all_oof, all_fold_metrics)
    comparison_df = make_paired_auc_comparisons(all_oof)

    # Fixed A30/E70 ensemble OOF diagnostic. This mirrors the old
    # static + integrated blend but does not alter the A-F ablation results.
    ensemble_oof_df, ensemble_oof_metrics = build_fixed_ae_oof_ensemble(
        all_oof
    )

    all_oof.to_csv(Config.OOF_PREDICTIONS_CSV, index=False, float_format="%.10f")
    all_fold_metrics.to_csv(
        Config.FOLD_METRICS_CSV, index=False, float_format="%.10f"
    )
    summary_df.to_csv(
        Config.SUMMARY_METRICS_CSV, index=False, float_format="%.10f"
    )
    comparison_df.to_csv(
        Config.AUC_COMPARISONS_CSV, index=False, float_format="%.10f"
    )

    # Manuscript-ready Excel workbook with the compact A-F table recommended
    # for the revised paper, paired delta-AUC comparisons, fold-level AUCs,
    # and a separate operational ensemble summary.
    manuscript_excel_path = write_manuscript_performance_excel(
        summary_df=summary_df,
        comparison_df=comparison_df,
        all_fold_metrics=all_fold_metrics,
        ensemble_oof_metrics=ensemble_oof_metrics,
    )

    all_inner_search.to_csv(
        Config.INNER_SEARCH_CSV, index=False, float_format="%.10f"
    )

    with open(Config.OUTER_PARAMS_JSON, "w", encoding="utf-8") as handle:
        json.dump(json_safe(outer_best_params), handle, indent=2)

    plot_oof_roc_curves(all_oof)
    plot_fold_auc(all_fold_metrics)

    # 4. Optional separate final full-data models for mapping/deployment only.
    final_models: Dict[str, Pipeline] = {}
    final_params: Dict[str, Any] = {}
    full_search_tables: List[pd.DataFrame] = []

    if Config.RUN_FINAL_DEPLOYMENT_TUNING:
        log("")
        log("[FINAL-MODEL] Running full-data spatial tuning for deployment models...")
        for scenario_name in Config.SCENARIO_ORDER:
            features = list(scenarios[scenario_name]["features"])
            final_model, info, search_df = tune_and_fit_final_model(
                df=df,
                scenario_name=scenario_name,
                features=features,
            )
            final_models[scenario_name] = final_model
            final_params[scenario_name] = info
            full_search_tables.append(search_df)

        full_search_all = pd.concat(full_search_tables, ignore_index=True)
        full_search_all.to_csv(
            Config.FINAL_SEARCH_CSV, index=False, float_format="%.10f"
        )

        with open(Config.FINAL_PARAMS_JSON, "w", encoding="utf-8") as handle:
            json.dump(json_safe(final_params), handle, indent=2)
    else:
        log(
            "[FINAL-MODEL] Skipped: RUN_FINAL_DEPLOYMENT_TUNING=False. "
            "This diagnostic run will report nested spatial OOF performance only."
        )

    # 5. Optional A–F susceptibility rasters on an identical common domain.
    map_paths: Dict[str, str] = {}
    if Config.GENERATE_RASTER_MAPS:
        if not Config.RUN_FINAL_DEPLOYMENT_TUNING:
            raise RuntimeError(
                "GENERATE_RASTER_MAPS=True requires "
                "RUN_FINAL_DEPLOYMENT_TUNING=True."
            )

        log("")
        log("[MAP] Generating six XGBoost A–F probability maps...")
        for scenario_name in Config.SCENARIO_ORDER:
            features = list(scenarios[scenario_name]["features"])
            output_path = predict_scenario_raster(
                fitted_pipeline=final_models[scenario_name],
                scenario_name=scenario_name,
                features=features,
            )
            map_paths[scenario_name] = str(output_path)

    # 6. Revised old-style A/E ensemble susceptibility raster.
    ensemble_map_path: Path | None = None
    ensemble_visualization_outputs: Dict[str, Any] = {}

    if Config.GENERATE_AE_ENSEMBLE_LSM:
        if not Config.RUN_FINAL_DEPLOYMENT_TUNING:
            raise RuntimeError(
                "GENERATE_AE_ENSEMBLE_LSM=True requires "
                "RUN_FINAL_DEPLOYMENT_TUNING=True."
            )

        log("")
        log(
            "[ENSEMBLE-MAP] Generating revised static-background + "
            "physics-guided ensemble map..."
        )
        ensemble_map_path = create_revised_ae_ensemble_probability_map(
            final_models=final_models,
            scenarios=scenarios,
        )

        ensemble_visualization_outputs = (
            create_complete_ensemble_lsm_visualization(
                probability_raster_path=ensemble_map_path,
                ensemble_oof_metrics=ensemble_oof_metrics,
            )
        )

    # 7. Optional publication-style primary Model E LSM visualization.
    # Retained separately so the manuscript can still show the actual primary
    # Model E map without confusing it with the A/E deployment ensemble.
    visualization_outputs: Dict[str, Any] = {}
    if Config.GENERATE_PRIMARY_LSM_VISUALIZATION:
        primary_scenario = Config.PRIMARY_VISUALIZATION_SCENARIO

        if primary_scenario not in Config.SCENARIO_ORDER:
            raise RuntimeError(
                "PRIMARY_VISUALIZATION_SCENARIO is not a valid A-F scenario: "
                f"{primary_scenario}"
            )

        if primary_scenario in map_paths:
            primary_map_path = Path(map_paths[primary_scenario])
        else:
            primary_map_path = expected_scenario_map_path(
                primary_scenario
            )

        if not primary_map_path.exists():
            raise RuntimeError(
                "Primary susceptibility visualization requested, but the "
                f"probability raster does not exist: {primary_map_path}\n"
                "Either set RUN_FINAL_DEPLOYMENT_TUNING=True and "
                "GENERATE_RASTER_MAPS=True, or first generate the Model E raster."
            )

        visualization_outputs = create_primary_lsm_visualization(
            raster_path=primary_map_path,
            scenario_name=primary_scenario,
        )

    # 8. QA / provenance metadata.
    write_qa_report(
        df=df,
        scenarios=scenarios,
        summary_df=summary_df,
        comparison_df=comparison_df,
        final_params=final_params,
        map_paths=map_paths,
    )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "XGBoost",
        "validation_design": {
            "outer": "five frozen contiguous spatial blocks from Step 6.4",
            "inner": (
                "four-block PredefinedSplit inside each outer-training set; "
                "one remaining spatial block held out per inner fold"
            ),
            "hyperparameter_selection_metric": "ROC-AUC",
            "reported_primary_performance": "pooled nested outer OOF ROC-AUC",
            "binary_threshold": Config.BINARY_THRESHOLD,
            "confidence_interval": (
                f"{int(Config.CI_LEVEL * 100)}% cluster bootstrap resampling whole spatial blocks"
            ),
            "cluster_bootstrap_iterations": Config.N_CLUSTER_BOOTSTRAP,
        },
        "input_dataset": str(Config.MODEL_READY_DATASET_CSV),
        "scenario_manifest_step61": str(Config.SCENARIO_MANIFEST_JSON),
        "scenario_manifest_step64": str(Config.STEP64_MODEL_CONFIG_JSON),
        "parameter_grid": Config.PARAM_GRID,
        "fixed_xgb_parameters": Config.XGB_FIXED_PARAMS,
        "execution_controls": {
            "run_final_deployment_tuning": Config.RUN_FINAL_DEPLOYMENT_TUNING,
            "generate_raster_maps": Config.GENERATE_RASTER_MAPS,
            "generate_primary_lsm_visualization": (
                Config.GENERATE_PRIMARY_LSM_VISUALIZATION
            ),
            "primary_visualization_scenario": (
                Config.PRIMARY_VISUALIZATION_SCENARIO
            ),
            "generate_AE_ensemble_lsm": Config.GENERATE_AE_ENSEMBLE_LSM,
            "ensemble_static_scenario": Config.ENSEMBLE_STATIC_SCENARIO,
            "ensemble_enhanced_scenario": Config.ENSEMBLE_ENHANCED_SCENARIO,
            "ensemble_enhanced_weight": Config.ENSEMBLE_ENHANCED_WEIGHT,
        },
        "scenarios": {
            scenario_name: {
                "label": Config.MODEL_SHORT_LABELS[scenario_name],
                "description": Config.MODEL_DESCRIPTIONS[scenario_name],
                "features": list(scenarios[scenario_name]["features"]),
            }
            for scenario_name in Config.SCENARIO_ORDER
        },
        "outputs": {
            "oof_predictions": str(Config.OOF_PREDICTIONS_CSV),
            "fold_metrics": str(Config.FOLD_METRICS_CSV),
            "summary_metrics": str(Config.SUMMARY_METRICS_CSV),
            "paired_auc_comparisons": str(Config.AUC_COMPARISONS_CSV),
            "outer_best_parameters": str(Config.OUTER_PARAMS_JSON),
            "final_best_parameters": str(Config.FINAL_PARAMS_JSON),
            "nested_grid_results": str(Config.INNER_SEARCH_CSV),
            "full_data_grid_results": str(Config.FINAL_SEARCH_CSV),
            "final_model_directory": str(Config.MODEL_DIR),
            "map_directory": str(Config.MAP_DIR),
            "visualization_directory": str(Config.VISUALIZATION_DIR),
            "primary_lsm_visualization": visualization_outputs,
            "ensemble_oof_predictions": str(
                Config.ENSEMBLE_OOF_PREDICTIONS_CSV
            ),
            "ensemble_oof_metrics": str(
                Config.ENSEMBLE_OOF_METRICS_JSON
            ),
            "ensemble_probability_raster": (
                str(ensemble_map_path)
                if ensemble_map_path is not None
                else None
            ),
            "ensemble_lsm_visualization": ensemble_visualization_outputs,
            "manuscript_performance_excel": str(
                Config.MANUSCRIPT_TABLES_XLSX
            ),
            "qa_report": str(Config.QA_REPORT_TXT),
        },
    }

    with open(Config.METADATA_JSON, "w", encoding="utf-8") as handle:
        json.dump(json_safe(metadata), handle, indent=2)

    # 9. Console summary.
    log("")
    log("=" * 96)
    log("NESTED SPATIAL OOF SUMMARY")
    log("=" * 96)
    for _, row in summary_df.iterrows():
        log(
            f"{row['model']}: AUC={row['oof_roc_auc']:.4f} "
            f"[{row['oof_auc_cluster_bootstrap_ci_low']:.4f}, "
            f"{row['oof_auc_cluster_bootstrap_ci_high']:.4f}], "
            f"F1={row['oof_f1_0p5']:.4f}, "
            f"Precision={row['oof_precision_0p5']:.4f}, "
            f"Recall={row['oof_recall_0p5']:.4f}"
        )

    log("")
    log("PAIRED AUC COMPARISONS")
    for _, row in comparison_df.iterrows():
        log(
            f"{row['comparison']}: ΔAUC={row['auc_difference_left_minus_right']:+.4f} "
            f"[{row['cluster_bootstrap_ci_low']:+.4f}, "
            f"{row['cluster_bootstrap_ci_high']:+.4f}]"
        )

    log("")
    log(
        "FIXED A30/E70 ENSEMBLE OOF: "
        f"AUC={ensemble_oof_metrics['roc_auc']:.4f} "
        f"[{ensemble_oof_metrics['auc_cluster_bootstrap_ci_low']:.4f}, "
        f"{ensemble_oof_metrics['auc_cluster_bootstrap_ci_high']:.4f}], "
        f"ΔAUC vs Model E="
        f"{ensemble_oof_metrics['auc_difference_ensemble_minus_model_E']:+.4f}"
    )

    if Config.GENERATE_AE_ENSEMBLE_LSM and ensemble_map_path is not None:
        log(
            "[OUT-ENSEMBLE] Old-style full-background ensemble LSM saved to: "
            f"{Config.VISUALIZATION_DIR}"
        )

    log("")
    if Config.GENERATE_PRIMARY_LSM_VISUALIZATION:
        log(
            "[OUT-VIZ] Primary Model E Landslide Susceptibility Map and "
            f"histogram saved to: {Config.VISUALIZATION_DIR}"
        )
    log(
        "[OUT-EXCEL] Manuscript A-F numerical tables: "
        f"{Config.MANUSCRIPT_TABLES_XLSX}"
    )
    log(f"[OUT] Results saved to: {Config.OUT_DIR}")
    log("=== Step 7.1 completed successfully ===")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 7.1 failed: {exc}")
        traceback.print_exc()
        raise


# STEP 7.1A — SPATIAL LEARNING CURVES + HELD-OUT PERMUTATION IMPORTANCE

In [ ]:
# =============================================================================
# STEP 7.1A — SPATIAL LEARNING CURVES + HELD-OUT PERMUTATION IMPORTANCE
# =============================================================================
# 1. Fixed-hyperparameter spatial learning curves for Model A and Model E only.
#    - The original outer validation block remains untouched.
#    - Training fractions are sampled only from the four outer-training blocks.
#    - Every training fraction retains all four spatial blocks and both classes.
#    - Fold-specific hyperparameters already selected without the outer block are
#      reused at all training sizes. This is a fixed-hyperparameter learning-curve
#      diagnostic, not a second hyperparameter optimization experiment.
#    - The outer block remains completely unseen.
#
# 2. Held-out permutation importance for Model E.
#    - The model is trained on four blocks and importance is calculated only on
#      the untouched fifth block.
#    - Individual predictor importance is complemented by a grouped joint
#      permutation of HKFI + SNDR.
# =============================================================================
from __future__ import annotations

import gc
import os
from pathlib import Path
from typing import Any

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score

from Step_7_1X_XGB_Reviewer_Diagnostics_Common_UPDATED import (
    load_outer_best_params,
    load_step71_module,
    set_pipeline_params,
    spatial_stratified_subsample,
    write_diagnostic_provenance,
)

class DiagnosticConfig:
    BASE_SCRIPT: Path | None = None

    LEARNING_SCENARIOS = (
        "A_static_baseline",
        "E_physics_guided",
    )
    PERMUTATION_SCENARIO = "E_physics_guided"

    TRAIN_FRACTIONS = (0.25, 0.50, 0.75, 1.00)
    N_REPEATS_SUBSAMPLE = 20
    N_PERMUTATION_REPEATS = 30
    RANDOM_STATE = 42071

    FIGSIZE = (5.5, 4.2)
    FIG_DPI = 900


def find_feature(features: list[str], token: str) -> str:
    matches = [feature for feature in features if token.lower() in feature.lower()]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one feature containing '{token}', found: {matches}"
        )
    return matches[0]


def get_fold_params(
    outer_params: dict[str, Any],
    scenario_name: str,
    outer_fold: int,
) -> dict[str, Any]:
    if scenario_name not in outer_params:
        raise KeyError(f"Missing scenario in outer-parameter JSON: {scenario_name}")
    fold_info = outer_params[scenario_name].get(str(outer_fold))
    if not fold_info:
        raise KeyError(
            f"Missing outer-fold parameters for {scenario_name}, fold {outer_fold}"
        )
    params = fold_info.get("best_params")
    if not params:
        raise KeyError(
            f"Missing best_params for {scenario_name}, fold {outer_fold}"
        )
    return params


def run_spatial_learning_curves(base, df, scenarios, outer_params, out_dir: Path):
    rows: list[dict[str, Any]] = []

    for scenario_name in DiagnosticConfig.LEARNING_SCENARIOS:
        features = list(scenarios[scenario_name]["features"])

        for outer_fold in base.Config.EXPECTED_FOLDS:
            train_df = df.loc[df[base.Config.FOLD_COLUMN] != outer_fold].copy()
            valid_df = df.loc[df[base.Config.FOLD_COLUMN] == outer_fold].copy()

            best_params = get_fold_params(
                outer_params,
                scenario_name,
                outer_fold,
            )

            X_valid = valid_df[features]
            y_valid = valid_df[base.Config.TARGET_COLUMN].to_numpy(dtype=int)

            for fraction_index, fraction in enumerate(DiagnosticConfig.TRAIN_FRACTIONS):
                n_repeats = (
                    1
                    if np.isclose(fraction, 1.0)
                    else DiagnosticConfig.N_REPEATS_SUBSAMPLE
                )

                for repeat in range(n_repeats):
                    seed = (
                        DiagnosticConfig.RANDOM_STATE
                        + 10000 * base.Config.SCENARIO_ORDER.index(scenario_name)
                        + 1000 * outer_fold
                        + 100 * fraction_index
                        + repeat
                    )

                    sampled_train = spatial_stratified_subsample(
                        train_df=train_df,
                        fold_column=base.Config.FOLD_COLUMN,
                        target_column=base.Config.TARGET_COLUMN,
                        fraction=float(fraction),
                        seed=seed,
                    )

                    # Strict spatial QA: every reduced training set must retain all
                    # four outer-training blocks and both classes.
                    if sampled_train[base.Config.FOLD_COLUMN].nunique() != 4:
                        raise RuntimeError(
                            f"Learning-curve sample lost a training spatial block: "
                            f"{scenario_name}, outer fold {outer_fold}, fraction {fraction}"
                        )
                    if sampled_train[base.Config.TARGET_COLUMN].nunique() != 2:
                        raise RuntimeError(
                            "Learning-curve sample lost one response class."
                        )

                    pipeline = base.build_xgb_pipeline(features)
                    set_pipeline_params(pipeline, best_params)

                    X_train = sampled_train[features]
                    y_train = sampled_train[base.Config.TARGET_COLUMN].to_numpy(dtype=int)

                    pipeline.fit(X_train, y_train)

                    train_prob = pipeline.predict_proba(X_train)[:, 1]
                    valid_prob = pipeline.predict_proba(X_valid)[:, 1]

                    rows.append(
                        {
                            "scenario": scenario_name,
                            "model": base.Config.MODEL_SHORT_LABELS[scenario_name],
                            "outer_fold": int(outer_fold),
                            "training_fraction": float(fraction),
                            "repeat": int(repeat),
                            "n_train": int(len(sampled_train)),
                            "n_train_positive": int(y_train.sum()),
                            "n_train_negative": int(len(y_train) - y_train.sum()),
                            "training_auc": float(roc_auc_score(y_train, train_prob)),
                            "training_pr_auc": float(average_precision_score(y_train, train_prob)),
                            "heldout_spatial_auc": float(roc_auc_score(y_valid, valid_prob)),
                            "heldout_spatial_pr_auc": float(average_precision_score(y_valid, valid_prob)),
                        }
                    )

                    del pipeline, sampled_train, X_train, y_train, train_prob, valid_prob
                    gc.collect()

    raw_df = pd.DataFrame(rows)
    raw_path = out_dir / "XGB_7_1A_Spatial_Learning_Curve_Raw.csv"
    raw_df.to_csv(raw_path, index=False, float_format="%.10f")

    # First average repeated subsamples within each outer fold. The five fold
    # means are then the independent spatial units used for the summary.
    fold_summary = (
        raw_df.groupby(
            ["scenario", "model", "outer_fold", "training_fraction"],
            as_index=False,
        )
        .agg(
            n_train_mean=("n_train", "mean"),
            training_auc_mean=("training_auc", "mean"),
            training_pr_auc_mean=("training_pr_auc", "mean"),
            heldout_spatial_auc_mean=("heldout_spatial_auc", "mean"),
            heldout_spatial_pr_auc_mean=("heldout_spatial_pr_auc", "mean"),
        )
    )

    summary = (
        fold_summary.groupby(
            ["scenario", "model", "training_fraction"],
            as_index=False,
        )
        .agg(
            n_train_mean=("n_train_mean", "mean"),
            training_auc_mean=("training_auc_mean", "mean"),
            training_auc_sd=("training_auc_mean", "std"),
            training_pr_auc_mean=("training_pr_auc_mean", "mean"),
            training_pr_auc_sd=("training_pr_auc_mean", "std"),
            heldout_auc_mean=("heldout_spatial_auc_mean", "mean"),
            heldout_auc_sd=("heldout_spatial_auc_mean", "std"),
            heldout_auc_min=("heldout_spatial_auc_mean", "min"),
            heldout_auc_max=("heldout_spatial_auc_mean", "max"),
            heldout_pr_auc_mean=("heldout_spatial_pr_auc_mean", "mean"),
            heldout_pr_auc_sd=("heldout_spatial_pr_auc_mean", "std"),
            heldout_pr_auc_min=("heldout_spatial_pr_auc_mean", "min"),
            heldout_pr_auc_max=("heldout_spatial_pr_auc_mean", "max"),
            n_outer_folds=("outer_fold", "nunique"),
        )
    )

    summary_path = out_dir / "XGB_7_1A_Spatial_Learning_Curve_Summary.csv"
    summary.to_csv(summary_path, index=False, float_format="%.10f")

    fig, ax = plt.subplots(figsize=DiagnosticConfig.FIGSIZE)
    for scenario_name in DiagnosticConfig.LEARNING_SCENARIOS:
        subset = summary.loc[summary["scenario"] == scenario_name].sort_values(
            "n_train_mean"
        )
        x = subset["n_train_mean"].to_numpy(dtype=float)
        y = subset["heldout_auc_mean"].to_numpy(dtype=float)
        sd = subset["heldout_auc_sd"].fillna(0.0).to_numpy(dtype=float)
        label = base.Config.MODEL_SHORT_LABELS[scenario_name]
        ax.plot(x, y, marker="o", linewidth=1.5, label=label)
        ax.fill_between(x, y - sd, y + sd, alpha=0.15)

    ax.set_xlabel("Number of outer-training samples")
    ax.set_ylabel("Held-out spatial ROC-AUC")
    ax.set_ylim(0.5, 1.01)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig_path = out_dir / "XGB_7_1A_Spatial_Learning_Curves.png"
    fig.savefig(fig_path, dpi=DiagnosticConfig.FIG_DPI, bbox_inches="tight")
    plt.close(fig)

    return raw_df, fold_summary, summary


def run_heldout_permutation_importance(base, df, scenarios, outer_params, out_dir: Path):
    scenario_name = DiagnosticConfig.PERMUTATION_SCENARIO
    features = list(scenarios[scenario_name]["features"])
    hkfi_feature = find_feature(features, "hkfi")
    sndr_feature = find_feature(features, "sndr")

    individual_rows: list[dict[str, Any]] = []
    grouped_rows: list[dict[str, Any]] = []

    for outer_fold in base.Config.EXPECTED_FOLDS:
        train_df = df.loc[df[base.Config.FOLD_COLUMN] != outer_fold].copy()
        valid_df = df.loc[df[base.Config.FOLD_COLUMN] == outer_fold].copy()

        best_params = get_fold_params(
            outer_params,
            scenario_name,
            outer_fold,
        )

        pipeline = base.build_xgb_pipeline(features)
        set_pipeline_params(pipeline, best_params)

        X_train = train_df[features]
        y_train = train_df[base.Config.TARGET_COLUMN].to_numpy(dtype=int)
        X_valid = valid_df[features]
        y_valid = valid_df[base.Config.TARGET_COLUMN].to_numpy(dtype=int)

        pipeline.fit(X_train, y_train)
        baseline_prob = pipeline.predict_proba(X_valid)[:, 1]
        baseline_auc = float(roc_auc_score(y_valid, baseline_prob))

        result = permutation_importance(
            estimator=pipeline,
            X=X_valid,
            y=y_valid,
            scoring="roc_auc",
            n_repeats=DiagnosticConfig.N_PERMUTATION_REPEATS,
            random_state=DiagnosticConfig.RANDOM_STATE + outer_fold,
            n_jobs=1,
        )

        for feature_index, feature in enumerate(features):
            for repeat, decrease in enumerate(result.importances[feature_index]):
                individual_rows.append(
                    {
                        "outer_fold": int(outer_fold),
                        "feature": feature,
                        "repeat": int(repeat),
                        "baseline_auc": baseline_auc,
                        "auc_decrease": float(decrease),
                    }
                )

        # Jointly permute HKFI and SNDR using the same row permutation so their
        # internal relationship is preserved while their alignment with labels
        # is broken.
        rng = np.random.default_rng(DiagnosticConfig.RANDOM_STATE + 1000 + outer_fold)
        for repeat in range(DiagnosticConfig.N_PERMUTATION_REPEATS):
            permutation = rng.permutation(len(X_valid))
            X_permuted = X_valid.copy()
            for feature in (hkfi_feature, sndr_feature):
                X_permuted[feature] = X_valid.iloc[permutation][feature].to_numpy()

            permuted_prob = pipeline.predict_proba(X_permuted)[:, 1]
            permuted_auc = float(roc_auc_score(y_valid, permuted_prob))
            grouped_rows.append(
                {
                    "outer_fold": int(outer_fold),
                    "feature_group": "HKFI + SNDR (joint permutation)",
                    "repeat": int(repeat),
                    "baseline_auc": baseline_auc,
                    "permuted_auc": permuted_auc,
                    "auc_decrease": baseline_auc - permuted_auc,
                }
            )

        del pipeline, result, train_df, valid_df, X_train, X_valid
        gc.collect()

    individual = pd.DataFrame(individual_rows)
    grouped = pd.DataFrame(grouped_rows)

    individual.to_csv(
        out_dir / "XGB_7_1A_Model_E_Heldout_Permutation_Importance_Raw.csv",
        index=False,
        float_format="%.10f",
    )
    grouped.to_csv(
        out_dir / "XGB_7_1A_Model_E_Grouped_HKFI_SNDR_Permutation_Raw.csv",
        index=False,
        float_format="%.10f",
    )

    # Aggregate repeats within each outer fold first, then summarize the five
    # spatial-fold means. This avoids treating permutation repeats as independent
    # spatial validation units.
    individual_fold = (
        individual.groupby(["outer_fold", "feature"], as_index=False)
        .agg(auc_decrease_mean=("auc_decrease", "mean"))
    )
    individual_summary = (
        individual_fold.groupby("feature", as_index=False)
        .agg(
            mean_auc_decrease=("auc_decrease_mean", "mean"),
            sd_across_outer_folds=("auc_decrease_mean", "std"),
            min_across_outer_folds=("auc_decrease_mean", "min"),
            max_across_outer_folds=("auc_decrease_mean", "max"),
        )
        .sort_values("mean_auc_decrease", ascending=False)
    )

    grouped_fold = (
        grouped.groupby(["outer_fold", "feature_group"], as_index=False)
        .agg(auc_decrease_mean=("auc_decrease", "mean"))
    )
    grouped_summary = (
        grouped_fold.groupby("feature_group", as_index=False)
        .agg(
            mean_auc_decrease=("auc_decrease_mean", "mean"),
            sd_across_outer_folds=("auc_decrease_mean", "std"),
            min_across_outer_folds=("auc_decrease_mean", "min"),
            max_across_outer_folds=("auc_decrease_mean", "max"),
        )
    )

    individual_summary.to_csv(
        out_dir / "XGB_7_1A_Model_E_Heldout_Permutation_Importance_Summary.csv",
        index=False,
        float_format="%.10f",
    )
    grouped_summary.to_csv(
        out_dir / "XGB_7_1A_Model_E_Grouped_HKFI_SNDR_Permutation_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    plot_df = individual_summary.head(15).sort_values("mean_auc_decrease")
    fig, ax = plt.subplots(figsize=(6.0, 5.2))
    ax.barh(
        plot_df["feature"],
        plot_df["mean_auc_decrease"],
        xerr=plot_df["sd_across_outer_folds"].fillna(0.0),
        capsize=2,
    )
    ax.axvline(0.0, linewidth=0.8)
    ax.set_xlabel("Decrease in held-out ROC-AUC after permutation")
    ax.set_ylabel("Predictor")
    ax.grid(True, axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        out_dir / "XGB_7_1A_Model_E_Heldout_Permutation_Importance.png",
        dpi=DiagnosticConfig.FIG_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    return individual_summary, grouped_summary


def main() -> None:
    base = load_step71_module(DiagnosticConfig.BASE_SCRIPT)

    out_dir = Path(base.Config.OUT_DIR) / "Step-7.1A-Learning_and_Permutation"
    out_dir.mkdir(parents=True, exist_ok=True)
    write_diagnostic_provenance(
        out_dir,
        "Step_7_1A_XGB",
        {
            "learning_curve_type": "fixed-hyperparameter spatial learning curve",
            "learning_scenarios": list(DiagnosticConfig.LEARNING_SCENARIOS),
            "permutation_scenario": DiagnosticConfig.PERMUTATION_SCENARIO,
            "training_fractions": list(DiagnosticConfig.TRAIN_FRACTIONS),
        },
    )

    scenarios = base.validate_scenario_manifests()
    df = base.load_and_validate_model_ready_dataset(scenarios)
    outer_params = load_outer_best_params(base)

    print("[7.1A] Running spatial learning curves for Models A and E...")
    _, _, learning_summary = run_spatial_learning_curves(
        base,
        df,
        scenarios,
        outer_params,
        out_dir,
    )

    print("[7.1A] Running held-out permutation importance for Model E...")
    individual_summary, grouped_summary = run_heldout_permutation_importance(
        base,
        df,
        scenarios,
        outer_params,
        out_dir,
    )

    print("\n[7.1A] Completed successfully.")
    print(f"Outputs: {out_dir}")
    print("\nLearning-curve summary:")
    print(learning_summary.to_string(index=False))
    print("\nTop held-out permutation importance:")
    print(individual_summary.head(10).to_string(index=False))
    print("\nGrouped HKFI + SNDR importance:")
    print(grouped_summary.to_string(index=False))


if __name__ == "__main__":
    main()


# STEP 7.1B — HKFI WEIGHTING SENSITIVITY

In [ ]:
# =============================================================================
# STEP 7.1B — XGBOOST HKFI WEIGHTING SENSITIVITY
# =============================================================================
#
# Scientific purpose
# ------------------
# Propagate the pre-specified HKFI weighting scenarios generated in Step 5.3
# through the EXACT successful Step 7.1 XGBoost modelling contract.
#
# This script deliberately does NOT:
#   - create a new manually maintained JSON manifest;
#   - optimize HKFI weights using model AUC;
#   - run the 66-case simplex through ML;
#   - alter any predictor other than HKFI.
#
# Instead it reads:
#   Results/Step-5.3-HKFI/weight_sensitivity/
#       hkfi_weight_scenario_manifest.csv
#
# The authoritative Step 5.3 manifest contains:
#   scenario
#   scenario_purpose
#   weight_K
#   weight_T
#   weight_S
#   raw_raster
#   norm_raster
#   is_primary
#   use_for_downstream_spatial_CV
#
# For each named downstream scenario:
#   1. load the exact primary Step 6.4 330-row model-ready dataset;
#   2. sample the scenario-specific normalized HKFI raster at the exact same
#      330 frozen sample coordinates;
#   3. replace ONLY hkfi_0_1 in a copy of the primary dataset;
#   4. verify all other values, labels, sample IDs and folds are unchanged;
#   5. run the exact successful Step 7.1 nested spatial CV for Model E;
#   6. compare each alternative with the primary equal-weight reference using
#      paired spatial-block bootstrap delta-AUC;
#   7. optionally fit a full-data deployment Model E for each alternative and
#      compare both:
#          a) the scientific Model E probability map, and
#          b) the final A30/E70 operational ensemble map.
#
# Execution architecture
# ----------------------
# Heavy XGBoost scenarios run in separate Python processes and are checkpointed.
# This limits cumulative native XGBoost memory growth and allows safe resume.
#
# Requirements
# ------------
# Keep this script in the same folder as:
#   Step_7_1X_XGB_Reviewer_Diagnostics_Common_UPDATED.py
#
# The successful Step 7.1 outputs must remain under:
#   ./data/Results/Step-7.1-XGB-Models_A_to_F
# =============================================================================

from __future__ import annotations

import argparse
import gc
import json
import os
import re
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

# Prevent native oversubscription before importing numerical libraries.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from sklearn.model_selection import GridSearchCV

from Step_7_1X_XGB_Reviewer_Diagnostics_Common_UPDATED import (
    all_scenario_features,
    assert_oof_reproduces_primary,
    load_step71_module,
    paired_auc_difference_from_oof,
    validate_dataset_exactly_matches_primary,
    validate_equal_rows_and_folds,
    validate_only_allowed_column_changes,
    write_diagnostic_provenance,
)


# =============================================================================
# CONFIGURATION
# =============================================================================
class SensitivityConfig:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Exact successful Step 7.1 base is resolved by the shared common module.
    BASE_SCRIPT: Path | None = None

    # Authoritative Step 5.3 weight-sensitivity outputs.
    STEP53_HKFI_DIR = RESULTS_DIR / "Step-5.3-HKFI"
    STEP53_WEIGHT_DIR = STEP53_HKFI_DIR / "weight_sensitivity"
    STEP53_WEIGHT_MANIFEST_CSV = (
        STEP53_WEIGHT_DIR / "hkfi_weight_scenario_manifest.csv"
    )
    STEP53_NAMED_SENSITIVITY_CSV = (
        STEP53_WEIGHT_DIR / "hkfi_named_weight_sensitivity.csv"
    )
    STEP53_SIMPLEX_SENSITIVITY_CSV = (
        STEP53_WEIGHT_DIR / "hkfi_simplex_weight_sensitivity.csv"
    )

    # Primary scenario name used by authoritative Step 5.3.
    REFERENCE_VARIANT = "primary_equal_component"

    # Model E only: Static12 + HKFI + SNDR.
    SCENARIO_NAME = "E_physics_guided"

    # None = run every named Step 5.3 scenario explicitly flagged for downstream
    # spatial CV in hkfi_weight_scenario_manifest.csv.
    #
    # To reduce runtime, replace None with a tuple, for example:
    # SCENARIOS_TO_RUN = (
    #     "primary_equal_component",
    #     "kinematic_emphasis",
    #     "twi_emphasis",
    #     "sm_emphasis",
    #     "local_emphasis",
    #     "hydrological_emphasis",
    #     "no_kinematic_K",
    #     "no_TWI",
    #     "no_SM",
    # )
    SCENARIOS_TO_RUN: tuple[str, ...] | None = None

    # The 66-case simplex is deliberately NOT propagated through ML.
    RUN_SIMPLEX_THROUGH_ML = False

    # Nested-CV and map controls.
    RUN_NESTED_CV = True
    GENERATE_MAP_SENSITIVITY = True
    RESUME_FROM_CHECKPOINTS = True

    # Reproducibility tolerances.
    OOF_REPRODUCTION_TOLERANCE = 1.0e-6
    PRIMARY_RASTER_SAMPLE_TOLERANCE = 5.0e-6
    NON_HKFI_EQUALITY_ATOL = 1.0e-10

    # Output settings.
    RANDOM_STATE = 42171
    FIG_DPI = 300
    RASTER_BLOCK_ROWS = 256

    # Worker/process output.
    WORKER_DIRNAME = "process_isolated_workers"
    LOG_DIRNAME = "logs"


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


def normalize_text(value: Any) -> str:
    return str(value).strip()


def parse_bool(value: Any) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"1", "true", "yes", "y", "t"}:
        return True
    if text in {"0", "false", "no", "n", "f", "", "nan", "none"}:
        return False
    raise ValueError(f"Cannot interpret as boolean: {value!r}")


def sanitize_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def resolve_script_path() -> Path:
    """
    Resolve this saved Python script for process-isolated worker launches.

    Jupyter launcher use is recommended. If the complete script is pasted into
    a notebook cell, __file__ may refer to ipykernel_launcher.py, so the script
    searches the current working directory for its canonical filename.
    """
    file_value = globals().get("__file__")
    if file_value:
        candidate = Path(str(file_value)).resolve()
        if candidate.exists() and candidate.name.lower() != "ipykernel_launcher.py":
            return candidate

    candidate = (
        Path.cwd()
        / "Step_7_1B_XGB_HKFI_Weight_Sensitivity_STEP53_DIRECT_FINAL.py"
    )
    if candidate.exists():
        return candidate.resolve()

    raise RuntimeError(
        "Could not resolve the saved Step 7.1B Python script. "
        "Keep the .py file in the same folder as the launcher notebook."
    )


# =============================================================================
# STEP 5.3 MANIFEST LOADING
# =============================================================================
def load_step53_weight_manifest() -> pd.DataFrame:
    """
    Read the authoritative named-weight scenario manifest produced by Step 5.3.

    No new JSON manifest is required.
    """
    path = SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV
    require_file(path, "Step 5.3 HKFI weight scenario manifest")

    manifest = pd.read_csv(path)
    required = [
        "scenario",
        "scenario_purpose",
        "weight_K",
        "weight_T",
        "weight_S",
        "norm_raster",
        "is_primary",
        "use_for_downstream_spatial_CV",
    ]
    missing = [column for column in required if column not in manifest.columns]
    if missing:
        raise RuntimeError(
            f"Step 5.3 weight scenario manifest is missing columns: {missing}\n"
            f"Found columns: {manifest.columns.tolist()}"
        )

    manifest = manifest.copy()
    manifest["scenario"] = manifest["scenario"].map(normalize_text)
    manifest["scenario_purpose"] = manifest["scenario_purpose"].map(normalize_text)
    manifest["is_primary"] = manifest["is_primary"].map(parse_bool)
    manifest["use_for_downstream_spatial_CV"] = (
        manifest["use_for_downstream_spatial_CV"].map(parse_bool)
    )

    for column in ("weight_K", "weight_T", "weight_S"):
        manifest[column] = pd.to_numeric(manifest[column], errors="raise")

    if manifest["scenario"].duplicated().any():
        duplicates = manifest.loc[
            manifest["scenario"].duplicated(keep=False), "scenario"
        ].tolist()
        raise RuntimeError(f"Duplicated Step 5.3 scenarios found: {duplicates}")

    if SensitivityConfig.REFERENCE_VARIANT not in set(manifest["scenario"]):
        raise RuntimeError(
            "The Step 5.3 manifest does not contain the required primary "
            f"scenario '{SensitivityConfig.REFERENCE_VARIANT}'."
        )

    primary_rows = manifest.loc[manifest["is_primary"]]
    if len(primary_rows) != 1:
        raise RuntimeError(
            "Step 5.3 manifest must contain exactly one is_primary=True row. "
            f"Found {len(primary_rows)}."
        )

    primary_name = str(primary_rows.iloc[0]["scenario"])
    if primary_name != SensitivityConfig.REFERENCE_VARIANT:
        raise RuntimeError(
            "Configured primary scenario does not match Step 5.3 manifest: "
            f"configured={SensitivityConfig.REFERENCE_VARIANT}, "
            f"manifest={primary_name}"
        )

    weight_sum = (
        manifest["weight_K"]
        + manifest["weight_T"]
        + manifest["weight_S"]
    )
    if not np.allclose(weight_sum.to_numpy(dtype=float), 1.0, atol=1.0e-8):
        bad = manifest.loc[
            ~np.isclose(weight_sum, 1.0, atol=1.0e-8),
            ["scenario", "weight_K", "weight_T", "weight_S"],
        ]
        raise RuntimeError(
            "One or more Step 5.3 named scenarios do not sum to 1:\n"
            + bad.to_string(index=False)
        )

    # Resolve raster paths. Step 5.3 writes absolute paths, but relative paths
    # are also supported relative to the manifest directory.
    resolved_paths = []
    for value in manifest["norm_raster"]:
        candidate = Path(str(value))
        if not candidate.is_absolute():
            candidate = path.parent / candidate
        resolved_paths.append(candidate)
    manifest["norm_raster_path"] = resolved_paths

    flagged = manifest.loc[manifest["use_for_downstream_spatial_CV"]].copy()
    if flagged.empty:
        raise RuntimeError(
            "No Step 5.3 scenarios are flagged use_for_downstream_spatial_CV=True."
        )

    if SensitivityConfig.SCENARIOS_TO_RUN is None:
        selected_names = flagged["scenario"].tolist()
    else:
        requested = list(SensitivityConfig.SCENARIOS_TO_RUN)
        unknown = [name for name in requested if name not in set(manifest["scenario"])]
        if unknown:
            raise RuntimeError(
                f"Configured SCENARIOS_TO_RUN contains unknown scenarios: {unknown}"
            )
        not_flagged = [
            name
            for name in requested
            if not bool(
                manifest.loc[
                    manifest["scenario"] == name,
                    "use_for_downstream_spatial_CV",
                ].iloc[0]
            )
        ]
        if not_flagged:
            raise RuntimeError(
                "These requested scenarios are not flagged for downstream spatial CV "
                f"in Step 5.3: {not_flagged}"
            )
        selected_names = requested

    if SensitivityConfig.REFERENCE_VARIANT not in selected_names:
        selected_names = [SensitivityConfig.REFERENCE_VARIANT] + selected_names

    # Primary always first; preserve Step 5.3 order for the remaining scenarios.
    ordered_names = [SensitivityConfig.REFERENCE_VARIANT] + [
        name for name in selected_names
        if name != SensitivityConfig.REFERENCE_VARIANT
    ]

    selected = (
        manifest.set_index("scenario")
        .loc[ordered_names]
        .reset_index()
    )

    for _, row in selected.iterrows():
        require_file(
            Path(row["norm_raster_path"]),
            f"normalized HKFI raster for scenario {row['scenario']}",
        )

    return selected


def load_optional_step53_named_summary() -> pd.DataFrame:
    path = SensitivityConfig.STEP53_NAMED_SENSITIVITY_CSV
    if not path.exists():
        return pd.DataFrame()
    try:
        frame = pd.read_csv(path)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return pd.DataFrame()

    if "scenario" not in frame.columns:
        log(
            f"[WARN] {path.name} has no 'scenario' column; "
            "upstream HKFI metrics will not be merged."
        )
        return pd.DataFrame()

    frame = frame.copy()
    frame["scenario"] = frame["scenario"].astype(str).str.strip()
    return frame


# =============================================================================
# DATASET CONSTRUCTION FROM STEP 5.3 RASTERS
# =============================================================================
def find_hkfi_feature(features: list[str]) -> str:
    matches = [feature for feature in features if "hkfi" in feature.lower()]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one HKFI feature in Model E; found {matches}"
        )
    return matches[0]


def assert_raster_aligned_to_master(
    base,
    raster_path: Path,
) -> dict[str, Any]:
    require_file(base.Config.MASTER_STACK_TIF, "Step 6.1 master predictor stack")
    require_file(raster_path, "Step 5.3 scenario HKFI raster")

    with rasterio.open(base.Config.MASTER_STACK_TIF) as master, rasterio.open(
        raster_path
    ) as scenario:
        aligned = (
            master.width == scenario.width
            and master.height == scenario.height
            and master.transform == scenario.transform
            and master.crs == scenario.crs
        )
        if not aligned:
            raise RuntimeError(
                "Step 5.3 HKFI scenario raster is not exactly aligned with the "
                "Step 6.1 master predictor stack:\n"
                f"Scenario raster: {raster_path}"
            )

        return {
            "width": int(scenario.width),
            "height": int(scenario.height),
            "crs": str(scenario.crs),
            "transform": tuple(scenario.transform),
            "nodata": scenario.nodata,
        }


def sample_raster_at_frozen_samples(
    base,
    raster_path: Path,
    primary_df: pd.DataFrame,
) -> np.ndarray:
    """
    Sample a Step 5.3 normalized HKFI raster at the exact frozen Step 6.4
    sample coordinates.

    The Step 6.4 model-ready dataset contains x/y coordinates in EPSG:32645.
    Exact raster-grid alignment is verified before sampling.
    """
    assert_raster_aligned_to_master(base, raster_path)

    required_columns = ["x", "y"]
    missing = [column for column in required_columns if column not in primary_df.columns]
    if missing:
        raise RuntimeError(
            "Primary Step 6.4 dataset is missing coordinates required for "
            f"HKFI raster sampling: {missing}"
        )

    coordinates = list(
        zip(
            primary_df["x"].to_numpy(dtype=float),
            primary_df["y"].to_numpy(dtype=float),
        )
    )

    with rasterio.open(raster_path) as dataset:
        nodata = dataset.nodata
        sampled = []
        for value_array in dataset.sample(coordinates, indexes=1, masked=False):
            value = float(value_array[0])
            if nodata is not None and np.isfinite(float(nodata)):
                if np.isclose(value, float(nodata), rtol=0.0, atol=0.0):
                    value = np.nan
            sampled.append(value)

    values = np.asarray(sampled, dtype="float64")
    if not np.isfinite(values).all():
        bad_indices = np.flatnonzero(~np.isfinite(values))
        examples = primary_df.iloc[bad_indices[:20]][
            [base.Config.SAMPLE_ID_COLUMN, "x", "y"]
        ]
        raise RuntimeError(
            f"Scenario HKFI raster produced {len(bad_indices)} non-finite values "
            "at frozen Step 6.4 sample locations.\n"
            f"Raster: {raster_path}\n"
            f"Examples:\n{examples.to_string(index=False)}"
        )

    return values


def build_variant_dataset(
    base,
    scenarios: dict[str, dict[str, Any]],
    manifest_row: pd.Series,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Build one Step 7.1B model-ready sensitivity dataset entirely in memory.

    Primary equal-weight:
        use the exact successful Step 6.4 dataset unchanged.

    Alternative:
        copy the exact primary dataset and replace ONLY HKFI with values sampled
        from the authoritative Step 5.3 scenario normalized raster.
    """
    primary_df = base.load_and_validate_model_ready_dataset(scenarios)
    validate_dataset_exactly_matches_primary(primary_df, scenarios)

    features = list(scenarios[SensitivityConfig.SCENARIO_NAME]["features"])
    hkfi_feature = find_hkfi_feature(features)

    scenario_name = str(manifest_row["scenario"])
    raster_path = Path(manifest_row["norm_raster_path"])

    sampled_hkfi = sample_raster_at_frozen_samples(
        base=base,
        raster_path=raster_path,
        primary_df=primary_df,
    )

    primary_hkfi = primary_df[hkfi_feature].to_numpy(dtype=float)
    raster_vs_primary_max_abs = float(
        np.max(np.abs(sampled_hkfi - primary_hkfi))
    )
    raster_vs_primary_mean_abs = float(
        np.mean(np.abs(sampled_hkfi - primary_hkfi))
    )

    if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
        # The exact successful Step 6.4 dataset remains authoritative for the
        # primary equal-weight rerun. Raster sampling is only a QA cross-check.
        if (
            raster_vs_primary_max_abs
            > SensitivityConfig.PRIMARY_RASTER_SAMPLE_TOLERANCE
        ):
            raise RuntimeError(
                "The Step 5.3 primary equal-weight HKFI raster does not reproduce "
                "the HKFI values frozen in the successful Step 6.4 dataset within "
                "the configured tolerance.\n"
                f"Max absolute difference={raster_vs_primary_max_abs:.10g}\n"
                f"Tolerance={SensitivityConfig.PRIMARY_RASTER_SAMPLE_TOLERANCE:.10g}\n"
                "This indicates an upstream version mismatch between Step 5.3 and "
                "the dataset used for the successful Step 7.1 analysis."
            )
        variant_df = primary_df.copy()
    else:
        variant_df = primary_df.copy()
        variant_df[hkfi_feature] = sampled_hkfi

        # Strong one-variable-at-a-time QA.
        validate_equal_rows_and_folds(primary_df, variant_df)
        compare_columns = [
            base.Config.TARGET_COLUMN,
            base.Config.FOLD_COLUMN,
            base.Config.BLOCK_COLUMN,
            *all_scenario_features(scenarios),
        ]
        validate_only_allowed_column_changes(
            primary_df,
            variant_df,
            allowed_changed_columns=(hkfi_feature,),
            columns_to_compare=compare_columns,
            atol=SensitivityConfig.NON_HKFI_EQUALITY_ATOL,
        )

    delta = variant_df[hkfi_feature].to_numpy(dtype=float) - primary_hkfi

    qa = {
        "scenario": scenario_name,
        "scenario_purpose": str(manifest_row["scenario_purpose"]),
        "weight_K": float(manifest_row["weight_K"]),
        "weight_T": float(manifest_row["weight_T"]),
        "weight_S": float(manifest_row["weight_S"]),
        "norm_raster": str(raster_path),
        "is_primary": bool(manifest_row["is_primary"]),
        "n_samples": int(len(variant_df)),
        "hkfi_mean_absolute_change_vs_primary_dataset": float(
            np.mean(np.abs(delta))
        ),
        "hkfi_maximum_absolute_change_vs_primary_dataset": float(
            np.max(np.abs(delta))
        ),
        "primary_raster_vs_primary_dataset_mean_absolute_difference": (
            raster_vs_primary_mean_abs
        ),
        "primary_raster_vs_primary_dataset_maximum_absolute_difference": (
            raster_vs_primary_max_abs
        ),
        "only_hkfi_changed": True,
    }

    return variant_df, qa


# =============================================================================
# PERFORMANCE SUMMARIES
# =============================================================================
def summarize_variant_oof(
    base,
    oof_df: pd.DataFrame,
    manifest_row: pd.Series,
) -> dict[str, Any]:
    y = oof_df[base.Config.TARGET_COLUMN].to_numpy(dtype=int)
    p = oof_df["oof_probability"].to_numpy(dtype=float)
    blocks = oof_df[base.Config.FOLD_COLUMN].to_numpy(dtype=int)

    metrics = base.classification_metrics(y, p)

    scenario_name = str(manifest_row["scenario"])
    seed_offset = sum(
        (index + 1) * ord(character)
        for index, character in enumerate(scenario_name)
    ) % 10000

    low, high, n_valid = base.cluster_bootstrap_auc_ci(
        y_true=y,
        y_prob=p,
        blocks=blocks,
        n_bootstrap=base.Config.N_CLUSTER_BOOTSTRAP,
        seed=SensitivityConfig.RANDOM_STATE + seed_offset,
        ci_level=base.Config.CI_LEVEL,
    )

    return {
        "scenario": scenario_name,
        "scenario_purpose": str(manifest_row["scenario_purpose"]),
        "weight_K": float(manifest_row["weight_K"]),
        "weight_T": float(manifest_row["weight_T"]),
        "weight_S": float(manifest_row["weight_S"]),
        "is_primary": bool(manifest_row["is_primary"]),
        "oof_roc_auc": metrics["roc_auc"],
        "auc_ci_low": low,
        "auc_ci_high": high,
        "oof_pr_auc": metrics["pr_auc"],
        "oof_balanced_accuracy": metrics["balanced_accuracy"],
        "oof_f1": metrics["f1"],
        "oof_brier": metrics["brier"],
        "oof_precision": metrics["precision"],
        "oof_recall": metrics["recall"],
        "valid_auc_bootstrap_iterations": int(n_valid),
    }


# =============================================================================
# MAP HELPERS
# =============================================================================
def read_master_band_map(base) -> dict[str, int]:
    with rasterio.open(base.Config.MASTER_STACK_TIF) as dataset:
        descriptions = list(dataset.descriptions)

    band_map = {}
    for index, description in enumerate(descriptions, start=1):
        if description:
            band_map[str(description)] = int(index)
    return band_map


def predict_model_e_with_external_hkfi_raster(
    base,
    fitted_pipeline,
    features: list[str],
    hkfi_feature: str,
    hkfi_raster_path: Path,
    output_path: Path,
) -> Path:
    """
    Predict Model E using:
      - all non-HKFI predictors from the successful Step 6.1 master stack;
      - HKFI from the selected authoritative Step 5.3 scenario raster.

    No new full multiband stack is required.
    """
    require_file(base.Config.MASTER_STACK_TIF, "Step 6.1 master predictor stack")
    require_file(hkfi_raster_path, "Step 5.3 scenario HKFI raster")
    assert_raster_aligned_to_master(base, hkfi_raster_path)

    band_map = read_master_band_map(base)
    non_hkfi_features = [
        feature for feature in features if feature != hkfi_feature
    ]
    missing = [
        feature for feature in non_hkfi_features if feature not in band_map
    ]
    if missing:
        raise RuntimeError(
            f"Master stack is missing non-HKFI Model E predictors: {missing}"
        )

    non_hkfi_band_indexes = [
        band_map[feature] for feature in non_hkfi_features
    ]

    with rasterio.open(base.Config.MASTER_STACK_TIF) as master, rasterio.open(
        hkfi_raster_path
    ) as hkfi_ds:
        profile = master.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=base.Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        valid_pixel_count = 0

        with rasterio.open(temporary_path, "w", **profile) as output_ds:
            for row_start in range(
                0,
                master.height,
                SensitivityConfig.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    SensitivityConfig.RASTER_BLOCK_ROWS,
                    master.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=master.width,
                    height=block_height,
                )

                static_dynamic = master.read(
                    non_hkfi_band_indexes,
                    window=window,
                    masked=True,
                )
                hkfi_arr = hkfi_ds.read(
                    1,
                    window=window,
                    masked=True,
                )

                non_hkfi_values = np.asarray(
                    static_dynamic.filled(np.nan),
                    dtype="float64",
                )
                hkfi_values = np.asarray(
                    hkfi_arr.filled(np.nan),
                    dtype="float64",
                )

                valid = (
                    np.all(np.isfinite(non_hkfi_values), axis=0)
                    & np.isfinite(hkfi_values)
                )

                output = np.full(
                    (block_height, master.width),
                    base.Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                if np.any(valid):
                    data_columns: dict[str, np.ndarray] = {}
                    non_hkfi_index = 0
                    for feature in features:
                        if feature == hkfi_feature:
                            data_columns[feature] = hkfi_values[valid]
                        else:
                            data_columns[feature] = (
                                non_hkfi_values[non_hkfi_index][valid]
                            )
                            non_hkfi_index += 1

                    pixel_df = pd.DataFrame(data_columns, columns=features)
                    probabilities = (
                        fitted_pipeline.predict_proba(pixel_df)[:, 1]
                    )
                    output[valid] = probabilities.astype("float32")
                    valid_pixel_count += int(valid.sum())

                output_ds.write(output, 1, window=window)

            output_ds.set_band_description(
                1,
                "XGBoost Model E HKFI-weight sensitivity probability",
            )
            output_ds.update_tags(
                model="XGBoost",
                scenario="E_physics_guided",
                hkfi_source=str(hkfi_raster_path),
                generated_at_utc=utc_now_iso(),
                validation_note=(
                    "Map generated from a separately tuned full-data deployment "
                    "model. Validation performance comes from nested spatial OOF."
                ),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(temporary_path, output_path)

    log(
        f"[MAP] {valid_pixel_count:,} valid Model E pixels -> {output_path}"
    )
    return output_path


def fit_final_variant_model(
    base,
    df: pd.DataFrame,
    features: list[str],
    scenario_name: str,
    model_dir: Path,
) -> tuple[Any, dict[str, Any], float]:
    X = df[features]
    y = df[base.Config.TARGET_COLUMN].to_numpy(dtype=int)

    search = GridSearchCV(
        estimator=base.build_xgb_pipeline(features),
        param_grid=base.Config.PARAM_GRID,
        scoring="roc_auc",
        cv=base.make_full_data_predefined_split(df),
        refit=True,
        n_jobs=base.Config.GRIDSEARCH_N_JOBS,
        verbose=0,
        return_train_score=False,
        error_score="raise",
    )
    search.fit(X, y)

    model_path = (
        model_dir
        / f"XGB_Model_E_HKFI_{sanitize_name(scenario_name)}_final.joblib"
    )
    joblib.dump(search.best_estimator_, model_path)

    return (
        search.best_estimator_,
        json_safe(search.best_params_),
        float(search.best_score_),
    )


def clean_probability_raster(dataset) -> np.ndarray:
    array = dataset.read(1, masked=True).filled(np.nan).astype("float64")
    nodata = dataset.nodata
    if nodata is not None and np.isfinite(float(nodata)):
        array[array == float(nodata)] = np.nan
    array[(array < -1.0e-6) | (array > 1.0 + 1.0e-6)] = np.nan
    return array


def compare_probability_maps(
    reference_path: Path,
    candidate_path: Path,
) -> dict[str, Any]:
    require_file(reference_path, "reference probability raster")
    require_file(candidate_path, "candidate probability raster")

    with rasterio.open(reference_path) as ref_ds, rasterio.open(
        candidate_path
    ) as can_ds:
        if not (
            ref_ds.width == can_ds.width
            and ref_ds.height == can_ds.height
            and ref_ds.transform == can_ds.transform
            and ref_ds.crs == can_ds.crs
        ):
            raise RuntimeError(
                "HKFI sensitivity probability maps are not spatially aligned."
            )

        ref = clean_probability_raster(ref_ds)
        can = clean_probability_raster(can_ds)

    valid = np.isfinite(ref) & np.isfinite(can)
    if not np.any(valid):
        raise RuntimeError("No common valid pixels between sensitivity maps.")

    ref_v = ref[valid]
    can_v = can[valid]
    difference = can_v - ref_v

    ref_class = np.digitize(
        ref_v,
        bins=[0.2, 0.4, 0.6, 0.8],
        right=False,
    )
    can_class = np.digitize(
        can_v,
        bins=[0.2, 0.4, 0.6, 0.8],
        right=False,
    )

    return {
        "common_valid_pixels": int(valid.sum()),
        "mean_absolute_probability_difference": float(
            np.mean(np.abs(difference))
        ),
        "maximum_absolute_probability_difference": float(
            np.max(np.abs(difference))
        ),
        "rmse_probability_difference": float(
            np.sqrt(np.mean(difference ** 2))
        ),
        "pearson_probability_correlation": float(
            np.corrcoef(ref_v, can_v)[0, 1]
        ),
        "spearman_probability_correlation": float(
            pd.Series(ref_v).corr(
                pd.Series(can_v),
                method="spearman",
            )
        ),
        "pixels_changing_susceptibility_class_percent": float(
            100.0 * np.mean(ref_class != can_class)
        ),
        "reference_high_plus_very_high_area_percent": float(
            100.0 * np.mean(ref_v >= 0.6)
        ),
        "candidate_high_plus_very_high_area_percent": float(
            100.0 * np.mean(can_v >= 0.6)
        ),
        "delta_high_plus_very_high_area_percentage_points": float(
            100.0 * np.mean(can_v >= 0.6)
            - 100.0 * np.mean(ref_v >= 0.6)
        ),
    }


# =============================================================================
# OUTPUT / CHECKPOINT PATHS
# =============================================================================
def output_directories(base) -> dict[str, Path]:
    out_dir = (
        Path(base.Config.OUT_DIR)
        / "Step-7.1B-HKFI_Weight_Sensitivity_STEP53_Direct"
    )
    paths = {
        "out": out_dir,
        "datasets": out_dir / "generated_model_ready_datasets",
        "models": out_dir / "models",
        "maps": out_dir / "maps",
        "workers": out_dir / SensitivityConfig.WORKER_DIRNAME,
        "logs": (
            out_dir
            / SensitivityConfig.WORKER_DIRNAME
            / SensitivityConfig.LOG_DIRNAME
        ),
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def scenario_paths(
    base,
    scenario_name: str,
) -> dict[str, Path]:
    dirs = output_directories(base)
    safe = sanitize_name(scenario_name)
    return {
        "dataset": dirs["datasets"] / f"{safe}_model_ready.csv",
        "qa": dirs["workers"] / f"{safe}_dataset_QA.json",
        "oof": dirs["workers"] / f"{safe}_OOF.csv",
        "fold_metrics": dirs["workers"] / f"{safe}_Fold_Metrics.csv",
        "search": dirs["workers"] / f"{safe}_Inner_Search.csv",
        "params": dirs["workers"] / f"{safe}_Outer_Params.json",
        "nested_done": dirs["workers"] / f"{safe}_nested.done",
        "model_info": dirs["workers"] / f"{safe}_Map_Model_Info.json",
        "e_map": dirs["maps"] / f"XGB_Model_E_HKFI_{safe}_probability.tif",
        "ensemble_map": dirs["maps"] / f"XGB_A30_E70_HKFI_{safe}_probability.tif",
        "map_done": dirs["workers"] / f"{safe}_map.done",
    }


# =============================================================================
# PROCESS-ISOLATED WORKERS
# =============================================================================
def run_external_worker(
    worker_mode: str,
    scenario_name: str,
) -> None:
    script_path = resolve_script_path()
    base = load_step71_module(SensitivityConfig.BASE_SCRIPT)
    dirs = output_directories(base)
    log_path = (
        dirs["logs"]
        / f"{sanitize_name(scenario_name)}_{worker_mode}.log"
    )

    command = [
        sys.executable,
        str(script_path),
        "--worker-mode",
        worker_mode,
        "--scenario",
        scenario_name,
    ]

    environment = os.environ.copy()
    environment["OMP_NUM_THREADS"] = "1"
    environment["OPENBLAS_NUM_THREADS"] = "1"
    environment["MKL_NUM_THREADS"] = "1"
    environment["NUMEXPR_NUM_THREADS"] = "1"

    log(
        f"[WORKER] Starting {worker_mode}: {scenario_name}\n"
        f"[WORKER] Log: {log_path}"
    )

    with open(log_path, "w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=environment,
        )

        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_handle.write(line)
            log_handle.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Step 7.1B worker failed: mode={worker_mode}, "
            f"scenario={scenario_name}, return_code={return_code}. "
            f"See log: {log_path}"
        )


def get_manifest_row(
    manifest: pd.DataFrame,
    scenario_name: str,
) -> pd.Series:
    rows = manifest.loc[manifest["scenario"] == scenario_name]
    if len(rows) != 1:
        raise RuntimeError(
            f"Expected exactly one manifest row for {scenario_name}; found {len(rows)}"
        )
    return rows.iloc[0]


def run_nested_worker(scenario_name: str) -> None:
    base = load_step71_module(SensitivityConfig.BASE_SCRIPT)
    scenarios = base.validate_scenario_manifests()
    manifest = load_step53_weight_manifest()
    row = get_manifest_row(manifest, scenario_name)
    paths = scenario_paths(base, scenario_name)

    variant_df, qa = build_variant_dataset(
        base=base,
        scenarios=scenarios,
        manifest_row=row,
    )
    variant_df.to_csv(
        paths["dataset"],
        index=False,
        float_format="%.10f",
    )

    with open(paths["qa"], "w", encoding="utf-8") as handle:
        json.dump(json_safe(qa), handle, indent=2)

    features = list(
        scenarios[SensitivityConfig.SCENARIO_NAME]["features"]
    )

    log(
        f"[NESTED] {scenario_name}: "
        f"K={row['weight_K']:.3f}, "
        f"T={row['weight_T']:.3f}, "
        f"S={row['weight_S']:.3f}"
    )

    oof_df, fold_df, params, search_df = (
        base.run_nested_spatial_cv_for_scenario(
            df=variant_df,
            scenario_name=SensitivityConfig.SCENARIO_NAME,
            features=features,
        )
    )

    reproduction_qa = None
    if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
        reproduction_qa = assert_oof_reproduces_primary(
            oof_df,
            SensitivityConfig.SCENARIO_NAME,
            tolerance=SensitivityConfig.OOF_REPRODUCTION_TOLERANCE,
        )
        qa.update(reproduction_qa)
        with open(paths["qa"], "w", encoding="utf-8") as handle:
            json.dump(json_safe(qa), handle, indent=2)

    oof_df = oof_df.copy()
    oof_df.insert(0, "hkfi_weight_variant", scenario_name)

    fold_df = fold_df.copy()
    fold_df.insert(0, "hkfi_weight_variant", scenario_name)

    search_df = search_df.copy()
    search_df.insert(0, "hkfi_weight_variant", scenario_name)

    oof_df.to_csv(paths["oof"], index=False, float_format="%.10f")
    fold_df.to_csv(
        paths["fold_metrics"],
        index=False,
        float_format="%.10f",
    )
    search_df.to_csv(
        paths["search"],
        index=False,
        float_format="%.10f",
    )
    with open(paths["params"], "w", encoding="utf-8") as handle:
        json.dump(json_safe(params), handle, indent=2)

    paths["nested_done"].write_text(
        utc_now_iso(),
        encoding="utf-8",
    )
    log(f"[NESTED] Completed: {scenario_name}")


def run_map_worker(scenario_name: str) -> None:
    base = load_step71_module(SensitivityConfig.BASE_SCRIPT)
    scenarios = base.validate_scenario_manifests()
    manifest = load_step53_weight_manifest()
    row = get_manifest_row(manifest, scenario_name)
    paths = scenario_paths(base, scenario_name)
    dirs = output_directories(base)

    if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
        # The successful Step 7.1 maps are the authoritative primary reference.
        require_file(
            base.Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
            "successful Step 7.1 primary Model E full-domain raster",
        )
        require_file(
            base.Config.ENSEMBLE_PROBABILITY_TIF,
            "successful Step 7.1 A30/E70 ensemble raster",
        )

        info = {
            "scenario": scenario_name,
            "map_source": "successful Step 7.1 primary reference",
            "model_e_map_path": str(
                base.Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF
            ),
            "operational_ensemble_map_path": str(
                base.Config.ENSEMBLE_PROBABILITY_TIF
            ),
            "refit_performed": False,
        }
        with open(paths["model_info"], "w", encoding="utf-8") as handle:
            json.dump(info, handle, indent=2)
        paths["map_done"].write_text(utc_now_iso(), encoding="utf-8")
        return

    variant_df, qa = build_variant_dataset(
        base=base,
        scenarios=scenarios,
        manifest_row=row,
    )

    features = list(
        scenarios[SensitivityConfig.SCENARIO_NAME]["features"]
    )
    hkfi_feature = find_hkfi_feature(features)

    model, best_params, selection_auc = fit_final_variant_model(
        base=base,
        df=variant_df,
        features=features,
        scenario_name=scenario_name,
        model_dir=dirs["models"],
    )

    e_map = predict_model_e_with_external_hkfi_raster(
        base=base,
        fitted_pipeline=model,
        features=features,
        hkfi_feature=hkfi_feature,
        hkfi_raster_path=Path(row["norm_raster_path"]),
        output_path=paths["e_map"],
    )

    ensemble_map = None
    if base.Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF.exists():
        ensemble_map = base.blend_static_and_enhanced_probability_rasters(
            static_raster_path=base.Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
            enhanced_raster_path=e_map,
            output_path=paths["ensemble_map"],
            enhanced_weight=float(base.Config.ENSEMBLE_ENHANCED_WEIGHT),
        )
    else:
        log(
            "[MAP-WARN] Operational ensemble skipped because successful "
            "Step 7.1 Model A full-domain raster is missing: "
            f"{base.Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF}"
        )

    info = {
        "scenario": scenario_name,
        "scenario_purpose": str(row["scenario_purpose"]),
        "weight_K": float(row["weight_K"]),
        "weight_T": float(row["weight_T"]),
        "weight_S": float(row["weight_S"]),
        "full_data_selection_auc_for_model_selection_only": selection_auc,
        "best_params": best_params,
        "model_e_map_path": str(e_map),
        "operational_ensemble_map_path": (
            str(ensemble_map) if ensemble_map is not None else None
        ),
        "refit_performed": True,
    }
    with open(paths["model_info"], "w", encoding="utf-8") as handle:
        json.dump(json_safe(info), handle, indent=2)

    paths["map_done"].write_text(utc_now_iso(), encoding="utf-8")

    del model
    gc.collect()
    log(f"[MAP] Completed: {scenario_name}")


# =============================================================================
# CONTROLLER / AGGREGATION
# =============================================================================
def load_nested_outputs(
    base,
    manifest: pd.DataFrame,
) -> tuple[
    dict[str, pd.DataFrame],
    pd.DataFrame,
    pd.DataFrame,
]:
    all_oof: dict[str, pd.DataFrame] = {}
    summaries = []
    qa_rows = []

    for _, row in manifest.iterrows():
        scenario_name = str(row["scenario"])
        paths = scenario_paths(base, scenario_name)

        if not paths["nested_done"].exists():
            raise RuntimeError(
                f"Missing nested checkpoint for scenario: {scenario_name}"
            )

        oof_df = pd.read_csv(paths["oof"])
        all_oof[scenario_name] = oof_df
        summaries.append(
            summarize_variant_oof(
                base=base,
                oof_df=oof_df,
                manifest_row=row,
            )
        )

        with open(paths["qa"], "r", encoding="utf-8") as handle:
            qa_rows.append(json.load(handle))

    return (
        all_oof,
        pd.DataFrame(summaries),
        pd.DataFrame(qa_rows),
    )


def build_pairwise_comparisons(
    base,
    manifest: pd.DataFrame,
    all_oof: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    reference = all_oof[SensitivityConfig.REFERENCE_VARIANT]
    rows = []

    for index, row in manifest.iterrows():
        scenario_name = str(row["scenario"])
        if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
            continue

        comparison = paired_auc_difference_from_oof(
            base=base,
            left_df=all_oof[scenario_name],
            right_df=reference,
            label=f"{scenario_name} - {SensitivityConfig.REFERENCE_VARIANT}",
            seed=SensitivityConfig.RANDOM_STATE + 100 + int(index),
        )
        comparison.update(
            {
                "scenario": scenario_name,
                "scenario_purpose": str(row["scenario_purpose"]),
                "weight_K": float(row["weight_K"]),
                "weight_T": float(row["weight_T"]),
                "weight_S": float(row["weight_S"]),
            }
        )
        rows.append(comparison)

    return pd.DataFrame(rows)


def build_map_comparisons(
    base,
    manifest: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    model_rows = []
    comparison_rows = []

    reference_e_map = Path(base.Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF)
    reference_ensemble = Path(base.Config.ENSEMBLE_PROBABILITY_TIF)

    for _, row in manifest.iterrows():
        scenario_name = str(row["scenario"])
        paths = scenario_paths(base, scenario_name)

        if not paths["map_done"].exists():
            continue

        with open(paths["model_info"], "r", encoding="utf-8") as handle:
            info = json.load(handle)

        model_rows.append(info)

        if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
            continue

        e_map_text = info.get("model_e_map_path")
        if e_map_text:
            e_map = Path(e_map_text)
            comparison_rows.append(
                {
                    "map_product": "Model E scientific sensitivity map",
                    "scenario": scenario_name,
                    "scenario_purpose": str(row["scenario_purpose"]),
                    "weight_K": float(row["weight_K"]),
                    "weight_T": float(row["weight_T"]),
                    "weight_S": float(row["weight_S"]),
                    "reference_scenario": SensitivityConfig.REFERENCE_VARIANT,
                    **compare_probability_maps(reference_e_map, e_map),
                }
            )

        ensemble_text = info.get("operational_ensemble_map_path")
        if ensemble_text and reference_ensemble.exists():
            ensemble_map = Path(ensemble_text)
            comparison_rows.append(
                {
                    "map_product": "A30/E70 operational ensemble map",
                    "scenario": scenario_name,
                    "scenario_purpose": str(row["scenario_purpose"]),
                    "weight_K": float(row["weight_K"]),
                    "weight_T": float(row["weight_T"]),
                    "weight_S": float(row["weight_S"]),
                    "reference_scenario": SensitivityConfig.REFERENCE_VARIANT,
                    **compare_probability_maps(
                        reference_ensemble,
                        ensemble_map,
                    ),
                }
            )

    return pd.DataFrame(model_rows), pd.DataFrame(comparison_rows)


def merge_upstream_hkfi_metrics(
    summary_df: pd.DataFrame,
) -> pd.DataFrame:
    upstream = load_optional_step53_named_summary()
    if upstream.empty:
        return summary_df

    # Avoid duplicating weight columns if they are present upstream.
    duplicate_columns = [
        column for column in upstream.columns
        if column in summary_df.columns and column != "scenario"
    ]
    upstream = upstream.drop(columns=duplicate_columns, errors="ignore")

    return summary_df.merge(
        upstream,
        on="scenario",
        how="left",
        validate="one_to_one",
    )


def create_performance_figure(
    summary_df: pd.DataFrame,
    output_path: Path,
) -> None:
    plot_df = summary_df.copy().reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    x = np.arange(len(plot_df))
    y = plot_df["oof_roc_auc"].to_numpy(dtype=float)

    lower = y - plot_df["auc_ci_low"].to_numpy(dtype=float)
    upper = plot_df["auc_ci_high"].to_numpy(dtype=float) - y

    ax.errorbar(
        x,
        y,
        yerr=np.vstack([lower, upper]),
        fmt="o",
        capsize=3,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(
        plot_df["scenario"],
        rotation=32,
        ha="right",
    )
    ax.set_ylabel("Nested spatial OOF ROC-AUC")
    ax.set_xlabel("Pre-specified HKFI weighting scenario")
    ax.set_ylim(
        max(
            0.5,
            float(np.nanmin(plot_df["auc_ci_low"])) - 0.03,
        ),
        1.0,
    )
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=SensitivityConfig.FIG_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def write_summary_report(
    base,
    manifest: pd.DataFrame,
    summary_df: pd.DataFrame,
    comparison_df: pd.DataFrame,
    qa_df: pd.DataFrame,
    map_comparison_df: pd.DataFrame,
) -> None:
    dirs = output_directories(base)
    report_path = dirs["out"] / "Reviewer_1_Comment_7_XGB_HKFI_Weight_Sensitivity_Report.txt"

    with open(report_path, "w", encoding="utf-8") as handle:
        handle.write("STEP 7.1B — XGB HKFI WEIGHTING SENSITIVITY\n")
        handle.write("=" * 88 + "\n\n")
        handle.write(
            "Source of weighting scenarios:\n"
            f"{SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV}\n\n"
        )
        handle.write(
            "The 66-case Step 5.3 simplex is not optimized through ML. "
            "Only the pre-specified named scenarios flagged for downstream "
            "spatial CV are propagated through the exact successful Step 7.1 "
            "nested spatial-CV contract.\n\n"
        )

        handle.write("SELECTED STEP 5.3 NAMED SCENARIOS\n")
        handle.write("-" * 88 + "\n")
        handle.write(
            manifest[
                [
                    "scenario",
                    "scenario_purpose",
                    "weight_K",
                    "weight_T",
                    "weight_S",
                ]
            ].to_string(index=False)
        )
        handle.write("\n\n")

        handle.write("DATASET / REPRODUCIBILITY QA\n")
        handle.write("-" * 88 + "\n")
        handle.write(qa_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("NESTED SPATIAL OOF PERFORMANCE\n")
        handle.write("-" * 88 + "\n")
        performance_columns = [
            "scenario",
            "weight_K",
            "weight_T",
            "weight_S",
            "oof_roc_auc",
            "auc_ci_low",
            "auc_ci_high",
            "oof_pr_auc",
            "oof_balanced_accuracy",
            "oof_f1",
            "oof_brier",
        ]
        handle.write(
            summary_df[
                [column for column in performance_columns if column in summary_df.columns]
            ].to_string(index=False)
        )
        handle.write("\n\n")

        handle.write("PAIRED DELTA-AUC VERSUS PRIMARY EQUAL-WEIGHT SCENARIO\n")
        handle.write("-" * 88 + "\n")
        if comparison_df.empty:
            handle.write("No alternative scenarios evaluated.\n")
        else:
            handle.write(comparison_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("MAP SENSITIVITY\n")
        handle.write("-" * 88 + "\n")
        if map_comparison_df.empty:
            handle.write("Map sensitivity was not generated.\n")
        else:
            handle.write(map_comparison_df.to_string(index=False))
        handle.write("\n")


def run_controller() -> None:
    base = load_step71_module(SensitivityConfig.BASE_SCRIPT)
    scenarios = base.validate_scenario_manifests()
    manifest = load_step53_weight_manifest()
    dirs = output_directories(base)

    log("=" * 96)
    log("STEP 7.1B — XGB HKFI WEIGHT SENSITIVITY FROM AUTHORITATIVE STEP 5.3 OUTPUTS")
    log("=" * 96)
    log(f"[STEP5.3] Manifest: {SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV}")
    log(
        "[STEP5.3] Named scenarios selected: "
        + ", ".join(manifest["scenario"].tolist())
    )
    log(
        "[RULE] The Step 5.3 simplex is not propagated through ML and weights "
        "are not optimized by AUC."
    )

    write_diagnostic_provenance(
        dirs["out"],
        "Step_7_1B_XGB_STEP53_Direct",
        {
            "step53_weight_manifest": str(
                SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV
            ),
            "step53_named_sensitivity": str(
                SensitivityConfig.STEP53_NAMED_SENSITIVITY_CSV
            ),
            "step53_simplex_sensitivity": str(
                SensitivityConfig.STEP53_SIMPLEX_SENSITIVITY_CSV
            ),
            "selected_scenarios": manifest[
                [
                    "scenario",
                    "scenario_purpose",
                    "weight_K",
                    "weight_T",
                    "weight_S",
                    "norm_raster_path",
                ]
            ].to_dict(orient="records"),
            "model_scenario": SensitivityConfig.SCENARIO_NAME,
            "primary_reference": SensitivityConfig.REFERENCE_VARIANT,
            "simplex_propagated_through_ml": False,
            "scientific_rule": (
                "Pre-specified named weight scenarios only; "
                "no downstream weight optimization."
            ),
        },
    )

    manifest.to_csv(
        dirs["out"] / "XGB_7_1B_Selected_Step53_Weight_Scenarios.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # 1. Nested CV — one fresh process per named scenario.
    # -------------------------------------------------------------------------
    if SensitivityConfig.RUN_NESTED_CV:
        for scenario_name in manifest["scenario"]:
            paths = scenario_paths(base, scenario_name)
            if (
                SensitivityConfig.RESUME_FROM_CHECKPOINTS
                and paths["nested_done"].exists()
                and paths["oof"].exists()
            ):
                log(f"[RESUME] Nested CV already complete: {scenario_name}")
                continue
            run_external_worker("nested", scenario_name)

    all_oof, summary_df, qa_df = load_nested_outputs(
        base=base,
        manifest=manifest,
    )

    comparison_df = build_pairwise_comparisons(
        base=base,
        manifest=manifest,
        all_oof=all_oof,
    )

    summary_with_upstream = merge_upstream_hkfi_metrics(summary_df)

    summary_with_upstream.to_csv(
        dirs["out"] / "XGB_7_1B_HKFI_Weight_Sensitivity_Performance.csv",
        index=False,
        float_format="%.10f",
    )
    comparison_df.to_csv(
        dirs["out"] / "XGB_7_1B_HKFI_Weight_Sensitivity_Paired_AUC.csv",
        index=False,
        float_format="%.10f",
    )
    qa_df.to_csv(
        dirs["out"] / "XGB_7_1B_HKFI_Weight_Sensitivity_QA.csv",
        index=False,
        float_format="%.10f",
    )

    # -------------------------------------------------------------------------
    # 2. Full-domain map sensitivity — fresh process per scenario.
    # -------------------------------------------------------------------------
    if SensitivityConfig.GENERATE_MAP_SENSITIVITY:
        for scenario_name in manifest["scenario"]:
            paths = scenario_paths(base, scenario_name)
            if (
                SensitivityConfig.RESUME_FROM_CHECKPOINTS
                and paths["map_done"].exists()
                and paths["model_info"].exists()
            ):
                log(f"[RESUME] Map stage already complete: {scenario_name}")
                continue
            run_external_worker("map", scenario_name)

    map_model_df, map_comparison_df = build_map_comparisons(
        base=base,
        manifest=manifest,
    )

    map_model_df.to_csv(
        dirs["out"] / "XGB_7_1B_HKFI_Weight_Sensitivity_Map_Models.csv",
        index=False,
    )
    map_comparison_df.to_csv(
        dirs["out"] / "XGB_7_1B_HKFI_Weight_Sensitivity_Map_Comparison.csv",
        index=False,
        float_format="%.10f",
    )

    create_performance_figure(
        summary_df=summary_df,
        output_path=(
            dirs["out"]
            / "XGB_7_1B_HKFI_Weight_Sensitivity_Performance.png"
        ),
    )

    write_summary_report(
        base=base,
        manifest=manifest,
        summary_df=summary_with_upstream,
        comparison_df=comparison_df,
        qa_df=qa_df,
        map_comparison_df=map_comparison_df,
    )

    log("")
    log("=" * 96)
    log("STEP 7.1B COMPLETED SUCCESSFULLY")
    log("=" * 96)
    log(f"[OUT] {dirs['out']}")
    log("")
    log("Nested spatial OOF performance:")
    log(summary_df.to_string(index=False))
    if not comparison_df.empty:
        log("")
        log("Paired delta-AUC versus primary equal weighting:")
        log(comparison_df.to_string(index=False))


# =============================================================================
# COMMAND-LINE ENTRY POINT
# =============================================================================
def parse_command_line_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Step 7.1B XGBoost HKFI weight sensitivity using authoritative "
            "Step 5.3 named scenarios."
        )
    )
    parser.add_argument(
        "--worker-mode",
        choices=["nested", "map"],
        default=None,
    )
    parser.add_argument(
        "--scenario",
        default=None,
    )

    arguments, unknown = parser.parse_known_args()

    # Jupyter/IPython injects unrelated arguments such as -f kernel.json.
    if unknown:
        log(f"[CLI] Ignoring host-environment arguments: {unknown}")

    return arguments


if __name__ == "__main__":
    try:
        args = parse_command_line_arguments()

        if args.worker_mode is None:
            run_controller()
        else:
            if not args.scenario:
                raise RuntimeError(
                    "--scenario is required when --worker-mode is supplied."
                )

            if args.worker_mode == "nested":
                run_nested_worker(args.scenario)
            elif args.worker_mode == "map":
                run_map_worker(args.scenario)
            else:
                raise RuntimeError(
                    f"Unsupported worker mode: {args.worker_mode}"
                )

    except Exception as exc:
        print(f"[ERROR] Step 7.1B failed: {exc}", flush=True)
        traceback.print_exc()
        raise


# STEP 7.1C — PSEUDO-ABSENCE BUFFER SENSITIVITY

In [ ]:
# =============================================================================
# STEP 7.1C — PSEUDO-ABSENCE BUFFER SENSITIVITY
# =============================================================================
# The primary 100 m experiment must exactly reproduce the successful Step-7.1
# Model A and Model E results. The authoritative Step 6.4 workflow already
# validated the 100/500/1000 m Step 6.3 datasets but wrote a separate model-ready
# CSV only for the primary 100 m case. Therefore 500/1000 m are loaded directly
# from Step 6.3, assigned cv_fold = spatial_block, and revalidated here using
# the exact Step 7.1 modelling contract before any XGBoost fitting.
#
# Scientific isolation rules:
#   - The 165 positive landslide samples are identical across buffers.
#   - Positive predictor values and fold assignments are identical across buffers.
#   - Only the pseudo-absence sample set changes with exclusion distance.
#   - Each buffer is independently nested-tuned using the exact successful
#     48-combination XGB grid; hyperparameters are NOT reused across buffers.
#   - The main reviewer statistic is E-A within each buffer.
# =============================================================================

from __future__ import annotations

import gc
import json
import os
from pathlib import Path
from typing import Any

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from Step_7_1X_XGB_Reviewer_Diagnostics_Common_UPDATED import (
    all_scenario_features,
    assert_oof_reproduces_primary,
    load_step71_module,
    paired_auc_difference_from_oof,
    validate_dataset_exactly_matches_primary,
    validate_same_positive_samples_across_buffers,
    write_diagnostic_provenance,
)


class BufferConfig:
    BASE_SCRIPT: Path | None = None  # Must remain None in the updated workflow.

    # Optional override manifest. It is NOT required for the normal workflow.
    # The authoritative Step 6.4 code validates the 100/500/1000 m datasets
    # directly from Step 6.3, but only writes a separate model-ready CSV for the
    # primary 100 m experiment. Therefore:
    #   - 100 m uses the exact Step 6.4 authoritative model-ready dataset;
    #   - 500/1000 m use the already generated Step 6.3 buffer datasets;
    #   - this Step 7.1C script adds cv_fold = spatial_block in memory and applies
    #     the same strict model-ready QA contract before any model is fitted.
    MANIFEST_PATH = Path(
        "./data/Results/Step-6.4-Model_Ready_QA/"
        "buffer_sensitivity/buffer_sensitivity_manifest.json"
    )

    STEP63_DIR = Path(
        "./data/Results/Step-6.3-Site_Level_Sampling"
    )
    STEP64_DIR = Path(
        "./data/Results/Step-6.4-Model_Ready_QA"
    )

    DEFAULT_BUFFER_DATASETS = {
        100: STEP64_DIR / "model_ready_dataset_A_to_F.csv",
        500: STEP63_DIR / "common_site_level_samples_A_to_F_buffer500m.csv",
        1000: STEP63_DIR / "common_site_level_samples_A_to_F_buffer1000m.csv",
    }

    # Optional Step 6.4 QA table. When present, its PASS status is checked.
    STEP64_BUFFER_QA_CSV = STEP64_DIR / "buffer_dataset_consistency_QA.csv"

    SCENARIOS = ("A_static_baseline", "E_physics_guided")
    REFERENCE_BUFFER_M = 100
    RANDOM_STATE = 42371
    FIG_DPI = 300
    OOF_REPRODUCTION_TOLERANCE = 1.0e-6


def load_buffer_paths(path: Path) -> dict[int, Path]:
    """
    Resolve the three buffer datasets.

    Normal authoritative workflow:
      100 m  -> Step 6.4 model_ready_dataset_A_to_F.csv
      500 m  -> Step 6.3 common_site_level_samples_A_to_F_buffer500m.csv
      1000 m -> Step 6.3 common_site_level_samples_A_to_F_buffer1000m.csv

    A JSON manifest may override these paths, but no manifest is required.
    """
    if path.exists():
        with open(path, "r", encoding="utf-8") as handle:
            manifest = json.load(handle)

        buffers = manifest.get("buffers", {})
        required = {"100", "500", "1000"}
        if not required.issubset(set(buffers)):
            raise RuntimeError(
                f"Manifest must define buffers {sorted(required)}; "
                f"found {sorted(buffers)}"
            )

        paths = {
            int(buffer_text): Path(info["dataset_csv"])
            for buffer_text, info in buffers.items()
            if int(buffer_text) in {100, 500, 1000}
        }
    else:
        paths = dict(BufferConfig.DEFAULT_BUFFER_DATASETS)

    missing = [
        f"{buffer_m} m: {dataset_path}"
        for buffer_m, dataset_path in paths.items()
        if not dataset_path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Step 7.1C could not find one or more authoritative buffer datasets.\n"
            "The normal workflow expects 100 m under Step 6.4 and 500/1000 m "
            "under Step 6.3.\nMissing:\n" + "\n".join(missing)
        )

    return paths


def check_optional_step64_buffer_qa() -> None:
    """
    Check the Step 6.4 buffer QA table when it exists.

    Step 6.4 already validated the Step 6.3 100/500/1000 m datasets. The table
    is used as an additional provenance check, but Step 7.1C independently
    repeats the essential modelling QA below.
    """
    path = BufferConfig.STEP64_BUFFER_QA_CSV
    if not path.exists():
        print(
            "[7.1C-WARN] Optional Step 6.4 buffer QA table was not found: "
            f"{path}\n"
            "Step 7.1C will continue because it performs the required strict "
            "dataset checks independently."
        )
        return

    qa = pd.read_csv(path)
    if "buffer_m" not in qa.columns:
        raise RuntimeError(
            f"Step 6.4 buffer QA file has no buffer_m column: {path}"
        )

    for buffer_m in (100, 500, 1000):
        rows = qa.loc[pd.to_numeric(qa["buffer_m"], errors="coerce") == buffer_m]
        if rows.empty:
            raise RuntimeError(
                f"Step 6.4 buffer QA table has no row for {buffer_m} m."
            )

        row = rows.iloc[0]

        if "status" in qa.columns:
            status = str(row["status"]).strip().upper()
            if status not in {"PASS", "OK", "TRUE"}:
                raise RuntimeError(
                    f"Step 6.4 buffer QA status is not PASS for {buffer_m} m: "
                    f"{row['status']}"
                )

        for column in (
            "same_positive_site_ids_as_100m",
            "same_positive_block_assignments_as_100m",
        ):
            if column in qa.columns:
                value = str(row[column]).strip().lower()
                if value not in {"true", "1", "yes", "pass"}:
                    raise RuntimeError(
                        f"Step 6.4 buffer QA failed {column} for {buffer_m} m."
                    )

    print(
        "[7.1C-QA] Step 6.4 buffer_dataset_consistency_QA.csv: PASS"
    )


def load_and_validate_buffer_dataset(
    base,
    scenarios: dict[str, dict[str, Any]],
    dataset_csv: Path,
    buffer_m: int,
) -> pd.DataFrame:
    """
    Convert one buffer dataset to the exact Step 7.1 modelling contract in memory.

    Step 6.3 already contains all 17 predictor values and spatial_block.
    Step 6.4 creates cv_fold as an exact alias of spatial_block for the primary
    model-ready dataset. The same deterministic alias is applied here to the
    500 m and 1000 m sensitivity datasets.

    No samples, predictor values, labels, or spatial blocks are regenerated.
    """
    if buffer_m == BufferConfig.REFERENCE_BUFFER_M:
        # The 100 m analysis must use the exact authoritative Step 6.4 file.
        df = base.load_and_validate_model_ready_dataset(
            scenarios,
            dataset_csv,
        )
        return df

    if not dataset_csv.exists():
        raise FileNotFoundError(
            f"Missing {buffer_m} m Step 6.3 buffer dataset: {dataset_csv}"
        )

    df = pd.read_csv(dataset_csv)

    # Step 6.3 uses spatial_block. Step 7.1 uses the identical frozen alias
    # cv_fold. Add only the missing alias; never regenerate spatial blocks.
    if base.Config.BLOCK_COLUMN not in df.columns:
        if base.Config.FOLD_COLUMN not in df.columns:
            raise RuntimeError(
                f"{buffer_m} m dataset contains neither "
                f"{base.Config.BLOCK_COLUMN} nor {base.Config.FOLD_COLUMN}."
            )
        df[base.Config.BLOCK_COLUMN] = pd.to_numeric(
            df[base.Config.FOLD_COLUMN],
            errors="raise",
        ).astype(int)

    if base.Config.FOLD_COLUMN not in df.columns:
        df[base.Config.FOLD_COLUMN] = pd.to_numeric(
            df[base.Config.BLOCK_COLUMN],
            errors="raise",
        ).astype(int)

    # Exact predictor contract from the successful Step 7.1 A-F definitions.
    all_features: list[str] = []
    for scenario_name in base.Config.SCENARIO_ORDER:
        for feature in scenarios[scenario_name]["features"]:
            if feature not in all_features:
                all_features.append(feature)

    required = [
        base.Config.SAMPLE_ID_COLUMN,
        base.Config.TARGET_COLUMN,
        base.Config.FOLD_COLUMN,
        base.Config.BLOCK_COLUMN,
        *all_features,
    ]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise RuntimeError(
            f"{buffer_m} m buffer dataset is missing required modelling columns: "
            f"{missing}"
        )

    # No silent row dropping.
    if len(df) != base.Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"{buffer_m} m: expected {base.Config.EXPECTED_N_ROWS} rows, "
            f"found {len(df)}."
        )

    if df[base.Config.SAMPLE_ID_COLUMN].isna().any():
        raise RuntimeError(f"{buffer_m} m: missing sample_id values.")
    if df[base.Config.SAMPLE_ID_COLUMN].duplicated().any():
        duplicates = df.loc[
            df[base.Config.SAMPLE_ID_COLUMN].duplicated(keep=False),
            base.Config.SAMPLE_ID_COLUMN,
        ].tolist()
        raise RuntimeError(
            f"{buffer_m} m: duplicated sample_id values found: {duplicates[:20]}"
        )

    df[base.Config.TARGET_COLUMN] = pd.to_numeric(
        df[base.Config.TARGET_COLUMN],
        errors="raise",
    ).astype(int)
    df[base.Config.FOLD_COLUMN] = pd.to_numeric(
        df[base.Config.FOLD_COLUMN],
        errors="raise",
    ).astype(int)
    df[base.Config.BLOCK_COLUMN] = pd.to_numeric(
        df[base.Config.BLOCK_COLUMN],
        errors="raise",
    ).astype(int)

    expected_class_counts = {
        0: base.Config.EXPECTED_N_NEGATIVE,
        1: base.Config.EXPECTED_N_POSITIVE,
    }
    observed_class_counts = (
        df[base.Config.TARGET_COLUMN]
        .value_counts()
        .sort_index()
        .to_dict()
    )
    if observed_class_counts != expected_class_counts:
        raise RuntimeError(
            f"{buffer_m} m: unexpected class counts "
            f"{observed_class_counts}; expected {expected_class_counts}."
        )

    observed_folds = sorted(
        df[base.Config.FOLD_COLUMN].unique().tolist()
    )
    if observed_folds != base.Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"{buffer_m} m: expected frozen folds "
            f"{base.Config.EXPECTED_FOLDS}; found {observed_folds}."
        )

    if not np.array_equal(
        df[base.Config.FOLD_COLUMN].to_numpy(),
        df[base.Config.BLOCK_COLUMN].to_numpy(),
    ):
        raise RuntimeError(
            f"{buffer_m} m: cv_fold and spatial_block are not identical."
        )

    # Every fold must contain 33 positives and 33 pseudo-absences.
    block_class = pd.crosstab(
        df[base.Config.FOLD_COLUMN],
        df[base.Config.TARGET_COLUMN],
    )
    expected_per_class = (
        base.Config.EXPECTED_N_POSITIVE
        // len(base.Config.EXPECTED_FOLDS)
    )

    for fold in base.Config.EXPECTED_FOLDS:
        if fold not in block_class.index:
            raise RuntimeError(f"{buffer_m} m: missing fold {fold}.")
        for label in (0, 1):
            count = int(block_class.loc[fold, label])
            if count != expected_per_class:
                raise RuntimeError(
                    f"{buffer_m} m, fold {fold}, class {label}: "
                    f"found {count}; expected {expected_per_class}."
                )

    # Predictor completeness: exactly the same rule as the successful Step 7.1.
    predictor_values = df[all_features].apply(
        pd.to_numeric,
        errors="coerce",
    )
    finite = np.isfinite(
        predictor_values.to_numpy(dtype="float64")
    )
    if not finite.all():
        bad = np.argwhere(~finite)
        examples = [
            (
                str(df.iloc[row_i][base.Config.SAMPLE_ID_COLUMN]),
                all_features[col_i],
            )
            for row_i, col_i in bad[:20]
        ]
        raise RuntimeError(
            f"{buffer_m} m: missing/non-finite predictor values remain. "
            f"Examples: {examples}"
        )

    irs_columns = [
        column
        for column in df.columns
        if "irs" in str(column).lower()
    ]
    if irs_columns:
        raise RuntimeError(
            f"{buffer_m} m: IRS-related columns found: {irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError(
            f"{buffer_m} m: tpi_s25 found; only tpi_s5 is allowed."
        )

    if "buffer_m" in df.columns:
        observed_buffer_values = sorted(
            pd.to_numeric(
                df["buffer_m"],
                errors="coerce",
            )
            .dropna()
            .unique()
            .tolist()
        )
        # Positive rows may retain 0/100 metadata in some historical outputs,
        # so this is a warning rather than a hard failure unless contradictory
        # nonzero values are present.
        contradictory = [
            value
            for value in observed_buffer_values
            if value not in {0, 100, buffer_m}
        ]
        if contradictory:
            raise RuntimeError(
                f"{buffer_m} m: contradictory buffer_m metadata values found: "
                f"{contradictory}"
            )

    print(
        f"[7.1C-QA] {buffer_m} m model-ready conversion: PASS | "
        f"rows={len(df)}, positive={expected_class_counts[1]}, "
        f"negative={expected_class_counts[0]}, "
        f"folds={observed_folds}"
    )

    return df


def summarize_oof(base, oof_df: pd.DataFrame, buffer_m: int, scenario_name: str):
    y = oof_df[base.Config.TARGET_COLUMN].to_numpy(dtype=int)
    p = oof_df["oof_probability"].to_numpy(dtype=float)
    blocks = oof_df[base.Config.FOLD_COLUMN].to_numpy(dtype=int)
    metrics = base.classification_metrics(y, p)
    low, high, n_valid = base.cluster_bootstrap_auc_ci(
        y_true=y,
        y_prob=p,
        blocks=blocks,
        n_bootstrap=base.Config.N_CLUSTER_BOOTSTRAP,
        seed=(
            BufferConfig.RANDOM_STATE
            + buffer_m
            + base.Config.SCENARIO_ORDER.index(scenario_name)
        ),
        ci_level=base.Config.CI_LEVEL,
    )
    return {
        "buffer_m": int(buffer_m),
        "scenario": scenario_name,
        "model": base.Config.MODEL_SHORT_LABELS[scenario_name],
        "oof_roc_auc": metrics["roc_auc"],
        "auc_ci_low": low,
        "auc_ci_high": high,
        "oof_pr_auc": metrics["pr_auc"],
        "oof_balanced_accuracy": metrics["balanced_accuracy"],
        "oof_f1": metrics["f1"],
        "oof_brier": metrics["brier"],
        "valid_auc_bootstrap_iterations": n_valid,
    }


def main() -> None:
    base = load_step71_module(BufferConfig.BASE_SCRIPT)
    scenarios = base.validate_scenario_manifests()
    predictor_columns = all_scenario_features(scenarios)

    out_dir = Path(base.Config.OUT_DIR) / "Step-7.1C-Buffer_Sensitivity"
    out_dir.mkdir(parents=True, exist_ok=True)

    buffer_paths = load_buffer_paths(BufferConfig.MANIFEST_PATH)
    write_diagnostic_provenance(
        out_dir,
        "Step_7_1C_XGB",
        {
            "buffers_m": [100, 500, 1000],
            "scenarios": list(BufferConfig.SCENARIOS),
            "buffer_dataset_paths": {str(k): str(v) for k, v in buffer_paths.items()},
            "input_rule": ("100 m uses exact Step 6.4 primary dataset; 500/1000 m use Step 6.3 ""buffer datasets converted in memory with cv_fold=spatial_block and strict ""Step 7.1 model-ready QA"),
        },
    )

    datasets: dict[int, pd.DataFrame] = {}
    summary_rows: list[dict[str, Any]] = []
    comparison_rows: list[dict[str, Any]] = []
    reproduction_rows: list[dict[str, Any]] = []

    # Step 6.4 already validated the three Step 6.3 buffer datasets. Check its
    # QA table when available, then independently enforce the exact Step 7.1
    # modelling contract in this script.
    check_optional_step64_buffer_qa()

    prepared_input_dir = out_dir / "prepared_model_ready_inputs"
    prepared_input_dir.mkdir(parents=True, exist_ok=True)

    # Load and validate all datasets first.
    for buffer_m in (100, 500, 1000):
        dataset_path = Path(buffer_paths[buffer_m])
        print(
            f"[7.1C] Loading {buffer_m} m buffer source dataset: "
            f"{dataset_path}"
        )
        datasets[buffer_m] = load_and_validate_buffer_dataset(
            base=base,
            scenarios=scenarios,
            dataset_csv=dataset_path,
            buffer_m=buffer_m,
        )

        # Save the exact in-memory modelling input used by Step 7.1C so the
        # sensitivity analysis is fully auditable. This does not modify Step 6.3
        # or Step 6.4 outputs.
        prepared_path = (
            prepared_input_dir
            / f"model_ready_dataset_A_to_F_buffer_{buffer_m}m.csv"
        )
        datasets[buffer_m].to_csv(
            prepared_path,
            index=False,
            float_format="%.10f",
        )

    # The 100 m dataset must be exactly the primary successful Step-7.1 dataset.
    validate_dataset_exactly_matches_primary(datasets[100], scenarios)

    # Across buffers, positives and their predictor values must remain unchanged.
    for buffer_m in (500, 1000):
        validate_same_positive_samples_across_buffers(
            datasets[100],
            datasets[buffer_m],
            predictor_columns=predictor_columns,
            atol=1.0e-10,
        )

    # Run A and E under the exact same successful nested-CV contract.
    for buffer_m in (100, 500, 1000):
        df = datasets[buffer_m]
        per_buffer_oof: dict[str, pd.DataFrame] = {}

        for scenario_name in BufferConfig.SCENARIOS:
            features = list(scenarios[scenario_name]["features"])
            print(
                f"[7.1C] Nested spatial CV | buffer={buffer_m} m | "
                f"{base.Config.MODEL_SHORT_LABELS[scenario_name]}"
            )
            oof_df, fold_df, params, search_df = base.run_nested_spatial_cv_for_scenario(
                df=df,
                scenario_name=scenario_name,
                features=features,
            )

            # 100 m must reproduce the actual successful Step-7.1 A/E OOF results.
            if buffer_m == BufferConfig.REFERENCE_BUFFER_M:
                qa = assert_oof_reproduces_primary(
                    oof_df,
                    scenario_name,
                    tolerance=BufferConfig.OOF_REPRODUCTION_TOLERANCE,
                )
                reproduction_rows.append(
                    {
                        "buffer_m": buffer_m,
                        "scenario": scenario_name,
                        **qa,
                    }
                )

            oof_df = oof_df.copy()
            oof_df.insert(0, "buffer_m", buffer_m)
            fold_df = fold_df.copy()
            fold_df.insert(0, "buffer_m", buffer_m)
            search_df = search_df.copy()
            search_df.insert(0, "buffer_m", buffer_m)

            per_buffer_oof[scenario_name] = oof_df
            summary_rows.append(summarize_oof(base, oof_df, buffer_m, scenario_name))

            prefix = (
                f"XGB_7_1C_buffer_{buffer_m}m_"
                f"{base.Config.MODEL_SHORT_LABELS[scenario_name].replace(' ', '_')}"
            )
            oof_df.to_csv(out_dir / f"{prefix}_OOF.csv", index=False)
            fold_df.to_csv(out_dir / f"{prefix}_Fold_Metrics.csv", index=False)
            search_df.to_csv(out_dir / f"{prefix}_Inner_Search.csv", index=False)
            with open(out_dir / f"{prefix}_Outer_Params.json", "w", encoding="utf-8") as handle:
                json.dump(params, handle, indent=2)
            gc.collect()

        comparison_rows.append(
            {
                "buffer_m": buffer_m,
                **paired_auc_difference_from_oof(
                    base=base,
                    left_df=per_buffer_oof["E_physics_guided"],
                    right_df=per_buffer_oof["A_static_baseline"],
                    label=f"E - A at {buffer_m} m",
                    seed=BufferConfig.RANDOM_STATE + 10000 + buffer_m,
                ),
            }
        )

    pd.DataFrame(reproduction_rows).to_csv(
        out_dir / "XGB_7_1C_100m_Reproduction_QA.csv",
        index=False,
        float_format="%.10f",
    )

    summary_df = pd.DataFrame(summary_rows).sort_values(["buffer_m", "scenario"])
    comparison_df = pd.DataFrame(comparison_rows).sort_values("buffer_m")
    summary_df.to_csv(
        out_dir / "XGB_7_1C_A_E_Buffer_Sensitivity_Performance.csv",
        index=False,
        float_format="%.10f",
    )
    comparison_df.to_csv(
        out_dir / "XGB_7_1C_A_E_Buffer_Sensitivity_Paired_AUC.csv",
        index=False,
        float_format="%.10f",
    )

    fig, ax = plt.subplots(figsize=(5.8, 4.2))
    for scenario_name in BufferConfig.SCENARIOS:
        subset = summary_df.loc[summary_df["scenario"] == scenario_name].sort_values("buffer_m")
        x = subset["buffer_m"].to_numpy(dtype=float)
        y = subset["oof_roc_auc"].to_numpy(dtype=float)
        lower = y - subset["auc_ci_low"].to_numpy(dtype=float)
        upper = subset["auc_ci_high"].to_numpy(dtype=float) - y
        ax.errorbar(
            x, y,
            yerr=np.vstack([lower, upper]),
            marker="o",
            capsize=3,
            linewidth=1.4,
            label=base.Config.MODEL_SHORT_LABELS[scenario_name],
        )
    ax.set_xlabel("Pseudo-absence exclusion buffer (m)")
    ax.set_ylabel("Nested OOF ROC-AUC")
    ax.set_xticks([100, 500, 1000])
    ax.set_ylim(max(0.5, float(np.nanmin(summary_df["auc_ci_low"])) - 0.03), 1.0)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(
        out_dir / "XGB_7_1C_A_E_Buffer_Sensitivity.png",
        dpi=BufferConfig.FIG_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    print("\n[7.1C] Completed successfully.")
    print(f"Outputs: {out_dir}")
    print("\nPerformance summary:")
    print(summary_df.to_string(index=False))
    print("\nE - A paired AUC differences within each buffer:")
    print(comparison_df.to_string(index=False))


if __name__ == "__main__":
    main()


# STEP 7.1D — PERFORMANCE AND MAPPED-AREA UNCERTAINTY

In [ ]:
# =============================================================================
# STEP 7.1D — XGBOOST PERFORMANCE AND MAPPED-AREA UNCERTAINTY
# =============================================================================
#
# Reviewer purpose
# ----------------
# Part 1 — Performance uncertainty
#   Extend spatial-block cluster-bootstrap uncertainty from ROC-AUC to:
#       - ROC-AUC
#       - PR-AUC
#       - balanced accuracy
#       - F1
#       - Brier score
#       - precision
#       - recall
#
#   The analysis uses the already saved nested outer-fold OOF predictions from
#   the successful Step 7.1 run. No model is refitted for performance CIs.
#
#   Uncertainty is reported for:
#       - Models A-F
#       - the separately reported fixed A30/E70 operational ensemble
#
#   Direct paired spatial-block bootstrap comparisons are also added for:
#       - Model E - Model C
#       - Model E - Model D
#
#
# Part 2 — Mapped-area / model variability
#   For each of the five successful outer folds:
#       1. omit that spatial block from training;
#       2. refit Model A using that fold's already selected inner-CV parameters;
#       3. refit Model E using that fold's already selected inner-CV parameters;
#       4. predict the complete available domain for A and E;
#       5. create the fixed A30/E70 ensemble map;
#       6. calculate susceptibility-class area percentages.
#
#   The five fold-specific map realizations are summarized as:
#       - mean
#       - standard deviation
#       - minimum
#       - maximum
#
#   These statistics are described as:
#       "cross-fold mapped-area variability" or
#       "cross-fold model uncertainty"
#
#   They are NOT described as a formal 95% confidence interval because only
#   five outer-fold map realizations are available.
#
#
# Critical scientific rules
# -------------------------
# 1. The primary model remains the successful 100 m / equal-HKFI Step 7.1 model.
# 2. Step 7.1B alternative HKFI weights are NOT mixed into Step 7.1D.
# 3. Step 7.1C 500/1000 m pseudo-absence buffers are NOT mixed into Step 7.1D.
# 4. Performance uncertainty is based only on genuine nested outer-fold OOF
#    predictions already produced by successful Step 7.1.
# 5. The five map realizations reuse the exact successful fold-specific best
#    hyperparameters; there is no second hyperparameter search.
# 6. The final full-data A30/E70 map is shown alongside the cross-fold map-area
#    statistics for direct manuscript interpretation.
# 7. All five cross-fold ensemble maps must have exactly the same valid-pixel
#    footprint, and this footprint must also match the successful final ensemble.
#
#
# Technical robustness
# --------------------
# - Each outer-fold map realization runs in a separate Python process.
# - A completion checkpoint is written per fold.
# - Completed folds are reused automatically on restart.
# - Native XGBoost memory is therefore released when each worker process exits.
#
#
# Required companion file
# -----------------------
# Keep this script in the same folder as:
#     Step_7_1X_XGB_Reviewer_Diagnostics_Common_UPDATED.py
#
# Recommended execution
# ---------------------
# Run the companion launcher notebook rather than pasting this entire script
# directly into a Jupyter cell.
# =============================================================================

from __future__ import annotations

import argparse
import gc
import json
import os
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# Prevent CPU oversubscription before importing numerical libraries.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import PathPatch, Polygon, Rectangle
from matplotlib.path import Path as MplPath
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, transform_geom
from rasterio.windows import Window

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    fiona = None
    HAVE_FIONA = False

from Step_7_1X_XGB_Reviewer_Diagnostics_Common_UPDATED import (
    cluster_bootstrap_metric_intervals,
    load_outer_best_params,
    load_step71_module,
    set_pipeline_params,
    write_diagnostic_provenance,
)


# =============================================================================
# CONFIGURATION
# =============================================================================
class UncertaintyConfig:
    # Must remain None: reviewer diagnostics are frozen to the exact successful
    # Step 7.1 contract through the shared common module.
    BASE_SCRIPT: Path | None = None

    # Performance uncertainty.
    N_BOOTSTRAP = 5000
    CI_LEVEL = 0.95
    RANDOM_STATE = 42571

    # Map uncertainty.
    KEEP_INTERMEDIATE_A_E_RASTERS = False
    RESUME_FROM_CHECKPOINTS = True
    RASTER_BLOCK_ROWS = 256

    # Figures.
    FIG_DPI = 300
    GENERATE_PIXELWISE_SD_FIGURE = True

    # Quantitative SD thresholds reported in the summary table.
    SD_THRESHOLDS = (0.02, 0.05, 0.10)


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def resolve_script_path() -> Path:
    """
    Resolve the saved Python script used for process-isolated fold workers.

    The launcher notebook is the recommended entry point. If the code is pasted
    into Jupyter, __file__ may refer to ipykernel_launcher.py; in that case the
    canonical saved .py file is searched in the current working directory.
    """
    file_value = globals().get("__file__")
    if file_value:
        candidate = Path(str(file_value)).resolve()
        if (
            candidate.exists()
            and candidate.name.lower() != "ipykernel_launcher.py"
        ):
            return candidate

    candidate = (
        Path.cwd()
        / "Step_7_1D_XGB_Performance_and_Mapped_Area_Uncertainty_PROCESS_ISOLATED_FINAL.py"
    )
    if candidate.exists():
        return candidate.resolve()

    raise RuntimeError(
        "Could not resolve the saved Step 7.1D Python script. Keep the .py "
        "file and launcher notebook in the same folder and run the launcher."
    )


# =============================================================================
# OUTPUT PATHS
# =============================================================================
def output_paths(base) -> dict[str, Path]:
    out_dir = (
        Path(base.Config.OUT_DIR)
        / "Step-7.1D-Performance_and_Map_Uncertainty"
    )
    paths = {
        "out": out_dir,
        "fold_dir": out_dir / "cross_fold_ensemble_rasters",
        "worker_dir": out_dir / "process_isolated_workers",
        "log_dir": out_dir / "process_isolated_workers" / "logs",
        "performance": (
            out_dir / "XGB_7_1D_Performance_Cluster_Bootstrap_Intervals.csv"
        ),
        "comparisons": (
            out_dir / "XGB_7_1D_Additional_Paired_AUC_E_C_E_D.csv"
        ),
        "area_raw": (
            out_dir / "XGB_7_1D_A30_E70_CrossFold_Area_Percentages_Raw.csv"
        ),
        "area_summary": (
            out_dir / "XGB_7_1D_A30_E70_CrossFold_Area_Uncertainty_Summary.csv"
        ),
        "mean_raster": (
            out_dir / "XGB_7_1D_A30_E70_CrossFold_Mean_Probability.tif"
        ),
        "sd_raster": (
            out_dir / "XGB_7_1D_A30_E70_CrossFold_SD_Probability.tif"
        ),
        "domain_qa": (
            out_dir / "XGB_7_1D_CrossFold_Raster_Domain_QA.csv"
        ),
        "sd_summary": (
            out_dir / "XGB_7_1D_Pixelwise_SD_Summary.csv"
        ),
        "area_figure": (
            out_dir / "XGB_7_1D_A30_E70_CrossFold_Area_Uncertainty.png"
        ),
        "sd_figure": (
            out_dir / "XGB_7_1D_A30_E70_CrossFold_SD_Probability.png"
        ),
        "report": (
            out_dir / "Step_7_1D_XGB_Uncertainty_Report.txt"
        ),
    }

    for directory in (
        paths["out"],
        paths["fold_dir"],
        paths["worker_dir"],
        paths["log_dir"],
    ):
        directory.mkdir(parents=True, exist_ok=True)

    return paths


def fold_paths(base, outer_fold: int) -> dict[str, Path]:
    paths = output_paths(base)
    fold_dir = paths["fold_dir"]
    worker_dir = paths["worker_dir"]

    return {
        "model_a_raster": (
            fold_dir / f"outer_fold_{outer_fold}_Model_A_probability.tif"
        ),
        "model_e_raster": (
            fold_dir / f"outer_fold_{outer_fold}_Model_E_probability.tif"
        ),
        "ensemble_raster": (
            fold_dir / f"outer_fold_{outer_fold}_A30_E70_probability.tif"
        ),
        "area_csv": (
            worker_dir / f"outer_fold_{outer_fold}_Area_Percentages.csv"
        ),
        "metadata_json": (
            worker_dir / f"outer_fold_{outer_fold}_Metadata.json"
        ),
        "done": (
            worker_dir / f"outer_fold_{outer_fold}.done"
        ),
        "log": (
            paths["log_dir"] / f"outer_fold_{outer_fold}.log"
        ),
    }


# =============================================================================
# PERFORMANCE UNCERTAINTY
# =============================================================================
def _append_metric_intervals(
    rows: list[dict[str, Any]],
    base,
    label: str,
    scenario: str,
    y_true: np.ndarray,
    y_prob: np.ndarray,
    blocks: np.ndarray,
    seed: int,
    product_type: str,
) -> None:
    intervals = cluster_bootstrap_metric_intervals(
        y_true=y_true,
        y_prob=y_prob,
        blocks=blocks,
        n_bootstrap=UncertaintyConfig.N_BOOTSTRAP,
        ci_level=UncertaintyConfig.CI_LEVEL,
        seed=seed,
        threshold=base.Config.BINARY_THRESHOLD,
        metrics=(
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "f1",
            "brier",
            "precision",
            "recall",
        ),
    )

    for _, metric_row in intervals.iterrows():
        rows.append(
            {
                "product_type": product_type,
                "scenario": scenario,
                "model_or_product": label,
                "metric": metric_row["metric"],
                "estimate": metric_row["estimate"],
                "ci_low": metric_row["ci_low"],
                "ci_high": metric_row["ci_high"],
                "valid_bootstrap_iterations": metric_row[
                    "valid_bootstrap_iterations"
                ],
                "binary_threshold": (
                    base.Config.BINARY_THRESHOLD
                    if metric_row["metric"]
                    in {
                        "balanced_accuracy",
                        "f1",
                        "precision",
                        "recall",
                    }
                    else np.nan
                ),
            }
        )


def build_performance_uncertainty(
    base,
    out_dir: Path,
) -> pd.DataFrame:
    """
    Calculate cluster-bootstrap intervals from already saved genuine OOF
    predictions. No model fitting occurs in this function.
    """
    require_file(
        base.Config.OOF_PREDICTIONS_CSV,
        "successful Step 7.1 A-F OOF predictions",
    )

    all_oof = pd.read_csv(base.Config.OOF_PREDICTIONS_CSV)
    rows: list[dict[str, Any]] = []

    for scenario_index, scenario_name in enumerate(
        base.Config.SCENARIO_ORDER
    ):
        subset = all_oof.loc[
            all_oof["scenario"] == scenario_name
        ].copy()

        if len(subset) != base.Config.EXPECTED_N_ROWS:
            raise RuntimeError(
                f"Unexpected OOF row count for {scenario_name}: "
                f"{len(subset)}; expected {base.Config.EXPECTED_N_ROWS}."
            )

        if subset[base.Config.SAMPLE_ID_COLUMN].duplicated().any():
            raise RuntimeError(
                f"Duplicated OOF sample IDs found for {scenario_name}."
            )

        _append_metric_intervals(
            rows=rows,
            base=base,
            label=base.Config.MODEL_SHORT_LABELS[scenario_name],
            scenario=scenario_name,
            y_true=subset[
                base.Config.TARGET_COLUMN
            ].to_numpy(dtype=int),
            y_prob=subset[
                "oof_probability"
            ].to_numpy(dtype=float),
            blocks=subset[
                base.Config.FOLD_COLUMN
            ].to_numpy(dtype=int),
            seed=(
                UncertaintyConfig.RANDOM_STATE
                + scenario_index
            ),
            product_type="A-F model",
        )

    # Separately reported fixed operational ensemble.
    require_file(
        base.Config.ENSEMBLE_OOF_PREDICTIONS_CSV,
        "successful Step 7.1 A30/E70 ensemble OOF predictions",
    )

    ensemble = pd.read_csv(
        base.Config.ENSEMBLE_OOF_PREDICTIONS_CSV
    )

    required = {
        base.Config.SAMPLE_ID_COLUMN,
        base.Config.TARGET_COLUMN,
        base.Config.FOLD_COLUMN,
        "oof_probability_ensemble",
    }
    missing = required - set(ensemble.columns)
    if missing:
        raise RuntimeError(
            "Ensemble OOF file is missing columns: "
            f"{sorted(missing)}"
        )

    if len(ensemble) != base.Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            "Unexpected A30/E70 ensemble OOF row count: "
            f"{len(ensemble)}; expected {base.Config.EXPECTED_N_ROWS}."
        )

    if ensemble[base.Config.SAMPLE_ID_COLUMN].duplicated().any():
        raise RuntimeError(
            "Duplicated sample IDs found in A30/E70 ensemble OOF file."
        )

    _append_metric_intervals(
        rows=rows,
        base=base,
        label="A30/E70 fixed operational ensemble",
        scenario="A30_E70_ensemble",
        y_true=ensemble[
            base.Config.TARGET_COLUMN
        ].to_numpy(dtype=int),
        y_prob=ensemble[
            "oof_probability_ensemble"
        ].to_numpy(dtype=float),
        blocks=ensemble[
            base.Config.FOLD_COLUMN
        ].to_numpy(dtype=int),
        seed=UncertaintyConfig.RANDOM_STATE + 100,
        product_type="operational ensemble",
    )

    result = pd.DataFrame(rows)
    result.to_csv(
        out_dir
        / "XGB_7_1D_Performance_Cluster_Bootstrap_Intervals.csv",
        index=False,
        float_format="%.10f",
    )

    return result


def add_missing_E_C_and_E_D_comparisons(
    base,
    out_dir: Path,
) -> pd.DataFrame:
    """
    Add direct paired spatial-block bootstrap comparisons not included in the
    original Step 7.1 comparison table.
    """
    require_file(
        base.Config.OOF_PREDICTIONS_CSV,
        "successful Step 7.1 A-F OOF predictions",
    )

    all_oof = pd.read_csv(
        base.Config.OOF_PREDICTIONS_CSV
    )

    pivot = all_oof.pivot(
        index=base.Config.SAMPLE_ID_COLUMN,
        columns="scenario",
        values="oof_probability",
    )

    metadata = (
        all_oof.loc[
            all_oof["scenario"] == "A_static_baseline",
            [
                base.Config.SAMPLE_ID_COLUMN,
                base.Config.TARGET_COLUMN,
                base.Config.FOLD_COLUMN,
            ],
        ]
        .drop_duplicates(
            base.Config.SAMPLE_ID_COLUMN
        )
        .set_index(
            base.Config.SAMPLE_ID_COLUMN
        )
        .loc[pivot.index]
    )

    y = metadata[
        base.Config.TARGET_COLUMN
    ].to_numpy(dtype=int)
    blocks = metadata[
        base.Config.FOLD_COLUMN
    ].to_numpy(dtype=int)

    comparisons = [
        (
            "E_physics_guided",
            "C_hkfi_only",
            "E - C",
            "Combined HKFI + SNDR versus HKFI alone",
        ),
        (
            "E_physics_guided",
            "D_sndr_only",
            "E - D",
            "Combined HKFI + SNDR versus SNDR alone",
        ),
    ]

    rows: list[dict[str, Any]] = []

    for comparison_index, (
        left,
        right,
        label,
        purpose,
    ) in enumerate(comparisons):
        observed, low, high, n_valid = (
            base.paired_cluster_bootstrap_auc_difference(
                y_true=y,
                prob_left=pivot[
                    left
                ].to_numpy(dtype=float),
                prob_right=pivot[
                    right
                ].to_numpy(dtype=float),
                blocks=blocks,
                n_bootstrap=UncertaintyConfig.N_BOOTSTRAP,
                seed=(
                    UncertaintyConfig.RANDOM_STATE
                    + 1000
                    + comparison_index
                ),
                ci_level=UncertaintyConfig.CI_LEVEL,
            )
        )

        rows.append(
            {
                "comparison": label,
                "purpose": purpose,
                "left_scenario": left,
                "right_scenario": right,
                "auc_difference_left_minus_right": observed,
                "ci_low": low,
                "ci_high": high,
                "valid_bootstrap_iterations": n_valid,
            }
        )

    result = pd.DataFrame(rows)

    result.to_csv(
        out_dir
        / "XGB_7_1D_Additional_Paired_AUC_E_C_E_D.csv",
        index=False,
        float_format="%.10f",
    )

    return result


# =============================================================================
# FOLD-SPECIFIC MODEL FITTING
# =============================================================================
def get_fold_params(
    outer_params: dict[str, Any],
    scenario_name: str,
    outer_fold: int,
) -> dict[str, Any]:
    scenario = outer_params.get(scenario_name)

    if scenario is None:
        raise KeyError(
            "Missing scenario in successful outer-parameter JSON: "
            f"{scenario_name}"
        )

    fold_info = scenario.get(str(outer_fold))

    if fold_info is None:
        raise KeyError(
            f"Missing fold {outer_fold} parameters for "
            f"{scenario_name}"
        )

    params = fold_info.get("best_params")

    if params is None:
        raise KeyError(
            f"Missing best_params for {scenario_name}, "
            f"outer fold {outer_fold}"
        )

    return params


def fit_outer_fold_model(
    base,
    df: pd.DataFrame,
    scenarios: dict[str, dict[str, Any]],
    outer_params: dict[str, Any],
    scenario_name: str,
    outer_fold: int,
):
    """
    Refit one fold-specific deployment realization using the exact best
    hyperparameters selected inside the corresponding successful Step 7.1
    outer-training set.

    No hyperparameter tuning is repeated here.
    """
    features = list(
        scenarios[scenario_name]["features"]
    )

    train_df = df.loc[
        df[base.Config.FOLD_COLUMN] != outer_fold
    ].copy()

    if train_df[
        base.Config.FOLD_COLUMN
    ].nunique() != 4:
        raise RuntimeError(
            f"Outer fold {outer_fold}: expected four training blocks."
        )

    params = get_fold_params(
        outer_params=outer_params,
        scenario_name=scenario_name,
        outer_fold=outer_fold,
    )

    pipeline = base.build_xgb_pipeline(features)
    set_pipeline_params(
        pipeline,
        params,
    )

    X_train = train_df[features]
    y_train = train_df[
        base.Config.TARGET_COLUMN
    ].to_numpy(dtype=int)

    pipeline.fit(
        X_train,
        y_train,
    )

    return pipeline, features, params


# =============================================================================
# PROCESS-ISOLATED OUTER-FOLD MAP WORKER
# =============================================================================
def run_outer_fold_map_worker(
    outer_fold: int,
) -> None:
    base = load_step71_module(
        UncertaintyConfig.BASE_SCRIPT
    )
    scenarios = base.validate_scenario_manifests()
    df = base.load_and_validate_model_ready_dataset(
        scenarios
    )
    outer_params = load_outer_best_params(base)

    paths = fold_paths(
        base,
        outer_fold,
    )

    log(
        f"[FOLD {outer_fold}] Fitting Model A on the four "
        "remaining spatial blocks..."
    )
    model_a, features_a, params_a = fit_outer_fold_model(
        base=base,
        df=df,
        scenarios=scenarios,
        outer_params=outer_params,
        scenario_name="A_static_baseline",
        outer_fold=outer_fold,
    )

    log(
        f"[FOLD {outer_fold}] Fitting Model E on the four "
        "remaining spatial blocks..."
    )
    model_e, features_e, params_e = fit_outer_fold_model(
        base=base,
        df=df,
        scenarios=scenarios,
        outer_params=outer_params,
        scenario_name="E_physics_guided",
        outer_fold=outer_fold,
    )

    log(
        f"[FOLD {outer_fold}] Predicting full available "
        "Model A domain..."
    )
    base.predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=model_a,
        scenario_name="A_static_baseline",
        features=features_a,
        output_path=paths["model_a_raster"],
    )

    log(
        f"[FOLD {outer_fold}] Predicting full available "
        "Model E domain..."
    )
    base.predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=model_e,
        scenario_name="E_physics_guided",
        features=features_e,
        output_path=paths["model_e_raster"],
    )

    log(
        f"[FOLD {outer_fold}] Creating fixed A30/E70 ensemble..."
    )
    base.blend_static_and_enhanced_probability_rasters(
        static_raster_path=paths["model_a_raster"],
        enhanced_raster_path=paths["model_e_raster"],
        output_path=paths["ensemble_raster"],
        enhanced_weight=float(
            base.Config.ENSEMBLE_ENHANCED_WEIGHT
        ),
    )

    area_df = (
        base.calculate_susceptibility_area_percentages(
            paths["ensemble_raster"]
        )
    )
    area_df.insert(
        0,
        "outer_fold_omitted_from_training",
        int(outer_fold),
    )
    area_df.to_csv(
        paths["area_csv"],
        index=False,
        float_format="%.10f",
    )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "outer_fold_omitted_from_training": int(outer_fold),
        "training_blocks": [
            int(value)
            for value in sorted(
                df.loc[
                    df[base.Config.FOLD_COLUMN]
                    != outer_fold,
                    base.Config.FOLD_COLUMN,
                ]
                .unique()
                .tolist()
            )
        ],
        "model_A_best_params": json_safe(params_a),
        "model_E_best_params": json_safe(params_e),
        "ensemble_enhanced_weight": float(
            base.Config.ENSEMBLE_ENHANCED_WEIGHT
        ),
        "model_A_raster": str(
            paths["model_a_raster"]
        ),
        "model_E_raster": str(
            paths["model_e_raster"]
        ),
        "ensemble_raster": str(
            paths["ensemble_raster"]
        ),
        "area_csv": str(
            paths["area_csv"]
        ),
        "interpretation": (
            "One cross-fold model realization. This is used to "
            "characterize model/map variability, not as a formal "
            "independent validation map."
        ),
    }

    with open(
        paths["metadata_json"],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(metadata),
            handle,
            indent=2,
        )

    # Delete large intermediate A/E rasters only after successful ensemble
    # creation and area calculation.
    if not UncertaintyConfig.KEEP_INTERMEDIATE_A_E_RASTERS:
        for intermediate_path in (
            paths["model_a_raster"],
            paths["model_e_raster"],
        ):
            if intermediate_path.exists():
                intermediate_path.unlink()

    paths["done"].write_text(
        utc_now_iso(),
        encoding="utf-8",
    )

    del model_a
    del model_e
    gc.collect()

    log(
        f"[FOLD {outer_fold}] Completed successfully."
    )


def run_external_fold_worker(
    outer_fold: int,
) -> None:
    """
    Launch one outer-fold map realization in a fresh OS process so native
    XGBoost memory is fully reclaimed when the fold completes.
    """
    script_path = resolve_script_path()
    base = load_step71_module(
        UncertaintyConfig.BASE_SCRIPT
    )
    paths = fold_paths(
        base,
        outer_fold,
    )

    command = [
        sys.executable,
        str(script_path),
        "--worker-fold",
        str(outer_fold),
    ]

    environment = os.environ.copy()
    environment["OMP_NUM_THREADS"] = "1"
    environment["OPENBLAS_NUM_THREADS"] = "1"
    environment["MKL_NUM_THREADS"] = "1"
    environment["VECLIB_MAXIMUM_THREADS"] = "1"
    environment["NUMEXPR_NUM_THREADS"] = "1"

    log(
        f"[PROCESS] Starting outer fold {outer_fold} "
        f"in a fresh Python process."
    )
    log(
        f"[PROCESS] Worker log: {paths['log']}"
    )

    with open(
        paths["log"],
        "w",
        encoding="utf-8",
    ) as log_handle:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=str(script_path.parent),
            env=environment,
        )

        assert process.stdout is not None

        for line in process.stdout:
            print(
                line,
                end="",
                flush=True,
            )
            log_handle.write(line)
            log_handle.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Outer-fold worker {outer_fold} failed with "
            f"return code {return_code}. See:\n{paths['log']}"
        )


# =============================================================================
# RASTER DOMAIN QA
# =============================================================================
def _raster_valid_mask(
    dataset,
    window: Window,
) -> np.ndarray:
    array = (
        dataset.read(
            1,
            window=window,
            masked=True,
        )
        .filled(np.nan)
        .astype("float64")
    )
    return np.isfinite(array)


def validate_cross_fold_raster_domains(
    fold_rasters: list[Path],
    final_full_data_raster: Path,
    output_csv: Path,
) -> pd.DataFrame:
    """
    Verify that:
      - all five fold ensemble rasters are aligned;
      - all five have exactly identical valid-pixel masks;
      - their valid mask matches the successful final full-data ensemble map.

    This ensures mapped-area differences reflect probability/model differences,
    not changes in the prediction footprint.
    """
    if not fold_rasters:
        raise ValueError(
            "No fold rasters supplied for domain QA."
        )

    for path in fold_rasters:
        require_file(
            path,
            "cross-fold ensemble probability raster",
        )

    require_file(
        final_full_data_raster,
        "successful final full-data ensemble raster",
    )

    datasets = [
        rasterio.open(path)
        for path in fold_rasters
    ]
    final_ds = rasterio.open(
        final_full_data_raster
    )

    try:
        reference = datasets[0]

        for index, dataset in enumerate(
            datasets[1:],
            start=2,
        ):
            aligned = (
                dataset.width == reference.width
                and dataset.height == reference.height
                and dataset.transform == reference.transform
                and dataset.crs == reference.crs
            )
            if not aligned:
                raise RuntimeError(
                    "Cross-fold ensemble raster alignment mismatch: "
                    f"fold raster index {index}."
                )

        final_aligned = (
            final_ds.width == reference.width
            and final_ds.height == reference.height
            and final_ds.transform == reference.transform
            and final_ds.crs == reference.crs
        )
        if not final_aligned:
            raise RuntimeError(
                "Successful final full-data ensemble raster is not aligned "
                "with the cross-fold ensemble rasters."
            )

        valid_counts = np.zeros(
            len(datasets),
            dtype=np.int64,
        )
        mismatch_vs_reference = np.zeros(
            len(datasets),
            dtype=np.int64,
        )
        mismatch_vs_final = np.zeros(
            len(datasets),
            dtype=np.int64,
        )
        final_valid_count = 0

        for row_start in range(
            0,
            reference.height,
            UncertaintyConfig.RASTER_BLOCK_ROWS,
        ):
            height = min(
                UncertaintyConfig.RASTER_BLOCK_ROWS,
                reference.height - row_start,
            )
            window = Window(
                0,
                row_start,
                reference.width,
                height,
            )

            masks = [
                _raster_valid_mask(
                    dataset,
                    window,
                )
                for dataset in datasets
            ]
            final_mask = _raster_valid_mask(
                final_ds,
                window,
            )

            reference_mask = masks[0]
            final_valid_count += int(
                final_mask.sum()
            )

            for index, mask in enumerate(masks):
                valid_counts[index] += int(
                    mask.sum()
                )
                mismatch_vs_reference[index] += int(
                    np.count_nonzero(
                        mask != reference_mask
                    )
                )
                mismatch_vs_final[index] += int(
                    np.count_nonzero(
                        mask != final_mask
                    )
                )

        rows: list[dict[str, Any]] = []

        for index, (
            raster_path,
            valid_count,
            mismatch_reference,
            mismatch_final,
        ) in enumerate(
            zip(
                fold_rasters,
                valid_counts,
                mismatch_vs_reference,
                mismatch_vs_final,
            ),
            start=1,
        ):
            rows.append(
                {
                    "raster": str(raster_path),
                    "fold_raster_sequence": index,
                    "valid_pixel_count": int(valid_count),
                    "mismatched_pixels_vs_fold1_mask": int(
                        mismatch_reference
                    ),
                    "mismatched_pixels_vs_final_full_data_mask": int(
                        mismatch_final
                    ),
                    "identical_to_fold1_mask": bool(
                        mismatch_reference == 0
                    ),
                    "identical_to_final_full_data_mask": bool(
                        mismatch_final == 0
                    ),
                }
            )

        rows.append(
            {
                "raster": str(
                    final_full_data_raster
                ),
                "fold_raster_sequence": 0,
                "valid_pixel_count": int(
                    final_valid_count
                ),
                "mismatched_pixels_vs_fold1_mask": 0,
                "mismatched_pixels_vs_final_full_data_mask": 0,
                "identical_to_fold1_mask": bool(
                    mismatch_vs_final[0] == 0
                ),
                "identical_to_final_full_data_mask": True,
            }
        )

        qa_df = pd.DataFrame(rows)

        qa_df.to_csv(
            output_csv,
            index=False,
        )

        if np.any(
            mismatch_vs_reference != 0
        ):
            raise RuntimeError(
                "Cross-fold ensemble rasters do not have identical "
                "valid-pixel masks. See raster-domain QA CSV:\n"
                f"{output_csv}"
            )

        if np.any(
            mismatch_vs_final != 0
        ):
            raise RuntimeError(
                "Cross-fold ensemble valid-pixel masks do not match "
                "the successful final full-data ensemble mask. See:\n"
                f"{output_csv}"
            )

        log(
            "[RASTER-QA] All five cross-fold ensemble rasters and the "
            "final full-data ensemble have identical valid domains: PASS"
        )

        return qa_df

    finally:
        for dataset in datasets:
            dataset.close()
        final_ds.close()


# =============================================================================
# CROSS-FOLD RASTER AGGREGATION
# =============================================================================
def aggregate_fold_rasters(
    fold_rasters: list[Path],
    mean_output: Path,
    sd_output: Path,
) -> None:
    """
    Calculate pixelwise cross-fold mean and sample SD.

    Domain equality is checked separately before this function is called.
    Since masks are identical, the implementation avoids all-NaN operations
    and the previous invalid np.errstate(degrees=...) bug.
    """
    if not fold_rasters:
        raise ValueError(
            "No fold rasters supplied for aggregation."
        )

    datasets = [
        rasterio.open(path)
        for path in fold_rasters
    ]

    try:
        reference = datasets[0]

        profile = reference.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=-9999.0,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        with rasterio.open(
            mean_output,
            "w",
            **profile,
        ) as mean_ds, rasterio.open(
            sd_output,
            "w",
            **profile,
        ) as sd_ds:

            for row_start in range(
                0,
                reference.height,
                UncertaintyConfig.RASTER_BLOCK_ROWS,
            ):
                height = min(
                    UncertaintyConfig.RASTER_BLOCK_ROWS,
                    reference.height - row_start,
                )
                window = Window(
                    0,
                    row_start,
                    reference.width,
                    height,
                )

                arrays = [
                    dataset.read(
                        1,
                        window=window,
                        masked=True,
                    )
                    .filled(np.nan)
                    .astype("float64")
                    for dataset in datasets
                ]

                stack = np.stack(
                    arrays,
                    axis=0,
                )

                reference_valid = np.isfinite(
                    stack[0]
                )

                # Defensive second-level mask QA.
                for fold_index in range(
                    1,
                    stack.shape[0],
                ):
                    candidate_valid = np.isfinite(
                        stack[fold_index]
                    )
                    if not np.array_equal(
                        reference_valid,
                        candidate_valid,
                    ):
                        raise RuntimeError(
                            "Cross-fold ensemble rasters changed valid "
                            "domain during aggregation."
                        )

                mean_output_array = np.full(
                    reference_valid.shape,
                    -9999.0,
                    dtype="float32",
                )
                sd_output_array = np.full(
                    reference_valid.shape,
                    -9999.0,
                    dtype="float32",
                )

                if np.any(
                    reference_valid
                ):
                    valid_values = stack[
                        :,
                        reference_valid,
                    ]

                    # Correct np.errstate usage: only valid NumPy error keys.
                    with np.errstate(
                        invalid="ignore",
                        divide="ignore",
                    ):
                        mean_values = np.mean(
                            valid_values,
                            axis=0,
                        )
                        sd_values = np.std(
                            valid_values,
                            axis=0,
                            ddof=1,
                        )

                    mean_output_array[
                        reference_valid
                    ] = mean_values.astype(
                        "float32"
                    )
                    sd_output_array[
                        reference_valid
                    ] = sd_values.astype(
                        "float32"
                    )

                mean_ds.write(
                    mean_output_array,
                    1,
                    window=window,
                )
                sd_ds.write(
                    sd_output_array,
                    1,
                    window=window,
                )

            mean_ds.set_band_description(
                1,
                "Cross-fold mean A30/E70 susceptibility probability",
            )
            mean_ds.update_tags(
                uncertainty_type="cross-fold model variability",
                number_of_outer_fold_models=len(
                    fold_rasters
                ),
                generated_at_utc=utc_now_iso(),
            )

            sd_ds.set_band_description(
                1,
                "Cross-fold SD of A30/E70 susceptibility probability",
            )
            sd_ds.update_tags(
                uncertainty_type="cross-fold model variability",
                number_of_outer_fold_models=len(
                    fold_rasters
                ),
                ddof=1,
                generated_at_utc=utc_now_iso(),
                interpretation=(
                    "Sample standard deviation across five outer-fold "
                    "model realizations; not a formal 95% confidence interval."
                ),
            )

    finally:
        for dataset in datasets:
            dataset.close()


# =============================================================================
# AREA UNCERTAINTY
# =============================================================================
def _final_full_data_area_rows(
    base,
) -> pd.DataFrame:
    require_file(
        base.Config.ENSEMBLE_PROBABILITY_TIF,
        "successful final Step 7.1 A30/E70 ensemble map",
    )

    final_area = (
        base.calculate_susceptibility_area_percentages(
            base.Config.ENSEMBLE_PROBABILITY_TIF
        )[
            [
                "class_id",
                "class_label",
                "area_percentage",
            ]
        ]
        .copy()
    )

    high_plus_very_high = float(
        final_area.loc[
            final_area["class_label"].isin(
                [
                    "High",
                    "Very High",
                ]
            ),
            "area_percentage",
        ].sum()
    )

    final_area = pd.concat(
        [
            final_area,
            pd.DataFrame(
                [
                    {
                        "class_id": 99,
                        "class_label": (
                            "High + Very High"
                        ),
                        "area_percentage": (
                            high_plus_very_high
                        ),
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    return final_area.rename(
        columns={
            "area_percentage": (
                "final_full_data_area_percentage"
            )
        }
    )


def collect_fold_area_results(
    base,
) -> tuple[pd.DataFrame, list[Path]]:
    """
    Load the five completed worker outputs.
    """
    area_frames: list[pd.DataFrame] = []
    ensemble_rasters: list[Path] = []

    for outer_fold in base.Config.EXPECTED_FOLDS:
        paths = fold_paths(
            base,
            outer_fold,
        )

        require_file(
            paths["done"],
            f"outer-fold {outer_fold} completion checkpoint",
        )
        require_file(
            paths["area_csv"],
            f"outer-fold {outer_fold} area-percentage CSV",
        )
        require_file(
            paths["ensemble_raster"],
            f"outer-fold {outer_fold} ensemble raster",
        )

        area_df = pd.read_csv(
            paths["area_csv"]
        )

        required = {
            "outer_fold_omitted_from_training",
            "class_id",
            "class_label",
            "area_percentage",
        }
        missing = required - set(
            area_df.columns
        )

        if missing:
            raise RuntimeError(
                f"Fold {outer_fold} area CSV is missing columns: "
                f"{sorted(missing)}"
            )

        area_frames.append(
            area_df
        )
        ensemble_rasters.append(
            paths["ensemble_raster"]
        )

    return (
        pd.concat(
            area_frames,
            ignore_index=True,
        ),
        ensemble_rasters,
    )


def summarize_mapped_area_uncertainty(
    base,
    area_raw: pd.DataFrame,
) -> pd.DataFrame:
    area_summary = (
        area_raw.groupby(
            [
                "class_id",
                "class_label",
            ],
            as_index=False,
        )
        .agg(
            mean_area_percentage=(
                "area_percentage",
                "mean",
            ),
            sd_area_percentage=(
                "area_percentage",
                "std",
            ),
            minimum_area_percentage=(
                "area_percentage",
                "min",
            ),
            maximum_area_percentage=(
                "area_percentage",
                "max",
            ),
            n_spatial_fold_models=(
                "outer_fold_omitted_from_training",
                "nunique",
            ),
        )
        .sort_values(
            "class_id",
            ascending=False,
        )
    )

    high_combined = (
        area_raw.loc[
            area_raw["class_label"].isin(
                [
                    "High",
                    "Very High",
                ]
            )
        ]
        .groupby(
            "outer_fold_omitted_from_training",
            as_index=False,
        )[
            "area_percentage"
        ]
        .sum()
    )

    area_summary = pd.concat(
        [
            area_summary,
            pd.DataFrame(
                [
                    {
                        "class_id": 99,
                        "class_label": (
                            "High + Very High"
                        ),
                        "mean_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].mean()
                        ),
                        "sd_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].std(
                                ddof=1
                            )
                        ),
                        "minimum_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].min()
                        ),
                        "maximum_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].max()
                        ),
                        "n_spatial_fold_models": int(
                            len(
                                high_combined
                            )
                        ),
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    final_area = _final_full_data_area_rows(
        base
    )

    area_summary = area_summary.merge(
        final_area,
        on=[
            "class_id",
            "class_label",
        ],
        how="left",
        validate="one_to_one",
    )

    area_summary[
        "final_minus_crossfold_mean_percentage_points"
    ] = (
        area_summary[
            "final_full_data_area_percentage"
        ]
        - area_summary[
            "mean_area_percentage"
        ]
    )

    area_summary[
        "crossfold_range_percentage_points"
    ] = (
        area_summary[
            "maximum_area_percentage"
        ]
        - area_summary[
            "minimum_area_percentage"
        ]
    )

    return area_summary


# =============================================================================
# PIXELWISE SD QUANTIFICATION
# =============================================================================
def summarize_pixelwise_sd(
    sd_raster_path: Path,
    output_csv: Path,
) -> pd.DataFrame:
    require_file(
        sd_raster_path,
        "cross-fold SD probability raster",
    )

    chunks: list[np.ndarray] = []

    with rasterio.open(
        sd_raster_path
    ) as dataset:
        for row_start in range(
            0,
            dataset.height,
            UncertaintyConfig.RASTER_BLOCK_ROWS,
        ):
            height = min(
                UncertaintyConfig.RASTER_BLOCK_ROWS,
                dataset.height - row_start,
            )
            window = Window(
                0,
                row_start,
                dataset.width,
                height,
            )

            values = (
                dataset.read(
                    1,
                    window=window,
                    masked=True,
                )
                .compressed()
                .astype("float64")
            )

            if values.size:
                chunks.append(
                    values
                )

    if not chunks:
        raise RuntimeError(
            "No valid values found in cross-fold SD raster."
        )

    values = np.concatenate(
        chunks
    )

    summary = {
        "valid_pixels": int(
            values.size
        ),
        "mean_pixelwise_sd": float(
            np.mean(
                values
            )
        ),
        "median_pixelwise_sd": float(
            np.median(
                values
            )
        ),
        "p75_pixelwise_sd": float(
            np.percentile(
                values,
                75.0,
            )
        ),
        "p90_pixelwise_sd": float(
            np.percentile(
                values,
                90.0,
            )
        ),
        "p95_pixelwise_sd": float(
            np.percentile(
                values,
                95.0,
            )
        ),
        "p99_pixelwise_sd": float(
            np.percentile(
                values,
                99.0,
            )
        ),
        "maximum_pixelwise_sd": float(
            np.max(
                values
            )
        ),
    }

    for threshold in (
        UncertaintyConfig.SD_THRESHOLDS
    ):
        key = (
            "pixels_with_sd_greater_than_"
            + str(threshold).replace(
                ".",
                "p",
            )
            + "_percent"
        )
        summary[key] = float(
            100.0
            * np.mean(
                values
                > float(
                    threshold
                )
            )
        )

    result = pd.DataFrame(
        [
            summary
        ]
    )

    result.to_csv(
        output_csv,
        index=False,
        float_format="%.10f",
    )

    return result


# =============================================================================
# FIGURES
# =============================================================================
def create_area_uncertainty_figure(
    area_summary: pd.DataFrame,
    output_path: Path,
) -> None:
    plot_df = area_summary.loc[
        area_summary["class_id"].isin(
            [
                1,
                2,
                3,
                4,
                5,
            ]
        )
    ].copy()

    # Display conventional Very Low -> Very High order.
    plot_df = plot_df.sort_values(
        "class_id",
        ascending=True,
    )

    figure, axis = plt.subplots(
        figsize=(
            6.2,
            4.3,
        )
    )

    x = np.arange(
        len(
            plot_df
        )
    )

    means = plot_df[
        "mean_area_percentage"
    ].to_numpy(
        dtype=float
    )

    sd = (
        plot_df[
            "sd_area_percentage"
        ]
        .fillna(
            0.0
        )
        .to_numpy(
            dtype=float
        )
    )

    final = plot_df[
        "final_full_data_area_percentage"
    ].to_numpy(
        dtype=float
    )

    axis.bar(
        x,
        means,
        yerr=sd,
        capsize=3,
        edgecolor="black",
        linewidth=0.7,
        label="Cross-fold mean ± SD",
    )

    axis.scatter(
        x,
        final,
        marker="D",
        zorder=5,
        label="Final full-data map",
    )

    axis.set_xticks(
        x
    )
    axis.set_xticklabels(
        plot_df[
            "class_label"
        ],
        rotation=20,
        ha="right",
    )
    axis.set_ylabel(
        "Mapped area (%)"
    )
    axis.grid(
        True,
        axis="y",
        alpha=0.25,
    )
    axis.legend(
        frameon=False,
        fontsize=8,
    )

    figure.tight_layout()
    figure.savefig(
        output_path,
        dpi=UncertaintyConfig.FIG_DPI,
        bbox_inches="tight",
    )
    plt.close(
        figure
    )


# -----------------------------------------------------------------------------
# CARTOGRAPHIC HELPERS FOR THE PIXELWISE SD FIGURE ONLY
# -----------------------------------------------------------------------------
# These constants reproduce the established Step 7.1 XGB ensemble-map style.
# They are deliberately local to the visualization layer and do not alter the
# Step 7.1D analytical/model configuration.
SD_MAP_PROJECT_BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
SD_MAP_STUDY_SHP = SD_MAP_PROJECT_BASE / "Study_Area.shp"
SD_MAP_ROAD_SHP = SD_MAP_PROJECT_BASE / "NTD_Road.shp"

SD_MAP_FIGSIZE = (4, 6)
SD_MAP_DMS_TICK_FONTSIZE = 7
SD_MAP_LEGEND_FONTSIZE = 7
SD_MAP_LEGEND_TITLE_FONTSIZE = 8
SD_MAP_COLORBAR_TICK_FONTSIZE = 8
SD_MAP_COLORBAR_LABEL_FONTSIZE = 9
SD_MAP_NORTH_ARROW_FONTSIZE = 8
SD_MAP_CAX_SIZE = "5%"
SD_MAP_CAX_PAD = 0.01
SD_MAP_DPI = 900

SD_MAP_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "xgb_crossfold_sd",
    [
        "#006400",
        "#32CD32",
        "#FFFF00",
        "#FFA500",
        "#FF4500",
        "#8B0000",
    ],
    N=256,
)


def _sd_to_dms(value: float, is_lon: bool = True) -> str:
    absolute = abs(value)
    degrees = int(absolute)
    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_lon
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def _sd_metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def _sd_nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base_value = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base_value
        if candidate >= target_metres:
            return candidate

    return 10 * base_value


def _sd_add_north_arrow(
    axis,
    size: float = 0.08,
    pad: float = 0.06,
    label: str = "N",
    arrow_downshift: float = 0.025,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=SD_MAP_NORTH_ARROW_FONTSIZE,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def _sd_add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    mid_lat_deg: float,
    segments: int = 4,
    fontsize: float = 6,
    frame_lw: float = 0.4,
) -> None:
    width_data = east - west
    height_data = north - south

    metres_per_degree = _sd_metres_per_degree_lon(mid_lat_deg)
    width_metres = width_data * metres_per_degree
    data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = _sd_nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = total_length_metres * data_units_per_metre
    segment_length_units = segment_length_metres * data_units_per_metre

    x_start = west + 0.5 * width_data - 0.5 * total_length_units
    y_base = south + 0.06 * height_data
    bar_height = 0.018 * height_data

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_data

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (x_start + index * segment_length_units, y_base),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=("black" if index % 2 == 0 else "white"),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_data,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_data,
        "0",
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_data,
        end_label,
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )


def _sd_reproject_to_epsg4326(
    source_path: Path,
    output_path: Path,
) -> Path:
    """Create an EPSG:4326 display copy; the analytical SD raster is unchanged."""
    require_file(
        source_path,
        "cross-fold SD probability raster for visualization",
    )

    with rasterio.open(source_path) as source:
        if source.crs is None:
            raise RuntimeError(
                f"Cross-fold SD raster has no CRS: {source_path}"
            )

        if source.crs.to_epsg() == 4326:
            return source_path

        destination_crs = "EPSG:4326"
        destination_transform, width, height = calculate_default_transform(
            source.crs,
            destination_crs,
            source.width,
            source.height,
            *source.bounds,
        )

        output_nodata = (
            float(source.nodata)
            if source.nodata is not None
            else -9999.0
        )

        profile = source.profile.copy()
        profile.update(
            driver="GTiff",
            crs=destination_crs,
            transform=destination_transform,
            width=width,
            height=height,
            count=1,
            dtype="float32",
            nodata=output_nodata,
            compress="LZW",
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as destination:
            reproject(
                source=rasterio.band(source, 1),
                destination=rasterio.band(destination, 1),
                src_transform=source.transform,
                src_crs=source.crs,
                src_nodata=source.nodata,
                dst_transform=destination_transform,
                dst_crs=destination_crs,
                dst_nodata=output_nodata,
                resampling=Resampling.bilinear,
            )

            destination.set_band_description(
                1,
                "Cross-fold SD of A30/E70 susceptibility probability - EPSG:4326 visualization copy",
            )

    if output_path.exists():
        output_path.unlink()

    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[SD-VIZ] EPSG:4326 display raster created: "
        f"{output_path}"
    )

    return output_path


def _sd_load_vector_overlay(
    vector_path: Path,
    target_crs: str,
) -> dict[str, list[dict[str, Any]]]:
    """Load a vector with Fiona and transform geometries to the display CRS."""
    require_file(
        vector_path,
        "cartographic vector overlay",
    )

    if not HAVE_FIONA:
        raise ImportError(
            "Fiona is required for the Step 7.1-style SD map overlays. "
            "The successful Step 7.1 workflow also uses Fiona-compatible shapefiles."
        )

    geometries: list[dict[str, Any]] = []

    with fiona.open(vector_path, "r") as source:
        source_crs = source.crs_wkt or source.crs
        if not source_crs:
            raise RuntimeError(
                f"Vector layer has no CRS: {vector_path}"
            )

        for feature in source:
            geometry = feature.get("geometry")
            if geometry is None:
                continue

            geometries.append(
                transform_geom(
                    source_crs,
                    target_crs,
                    geometry,
                    precision=-1,
                )
            )

    return {"geometries": geometries}


def _sd_plot_vector_overlay(
    axis,
    vector_data: dict[str, list[dict[str, Any]]],
    colour: str,
    linewidth: float,
    zorder: int,
) -> None:
    for geometry in vector_data.get("geometries", []):
        if not geometry:
            continue

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")
        if coordinates is None:
            continue

        if geometry_type == "LineString":
            line_groups = [coordinates]
        elif geometry_type == "MultiLineString":
            line_groups = list(coordinates)
        elif geometry_type == "Polygon":
            line_groups = [coordinates[0]] if coordinates else []
        elif geometry_type == "MultiPolygon":
            line_groups = [
                polygon[0]
                for polygon in coordinates
                if polygon
            ]
        else:
            continue

        for line in line_groups:
            xy = np.asarray(line, dtype="float64")
            if xy.ndim != 2 or xy.shape[1] < 2:
                continue

            axis.plot(
                xy[:, 0],
                xy[:, 1],
                color=colour,
                linewidth=linewidth,
                zorder=zorder,
            )


def _sd_build_clip_patch(
    study_vector: dict[str, list[dict[str, Any]]],
    axis,
):
    """Build the same opaque-background polygon clip used by the Step 7.1 map."""
    all_vertices: list[np.ndarray] = []
    all_codes: list[int] = []

    def add_ring(ring_coordinates) -> None:
        coordinates = np.asarray(
            ring_coordinates,
            dtype="float64",
        )

        if (
            coordinates.ndim != 2
            or coordinates.shape[0] < 3
            or coordinates.shape[1] < 2
        ):
            return

        coordinates = coordinates[:, :2]

        if not np.allclose(
            coordinates[0],
            coordinates[-1],
            equal_nan=False,
        ):
            coordinates = np.vstack(
                [coordinates, coordinates[0]]
            )

        n_vertices = len(coordinates)
        codes = (
            [MplPath.MOVETO]
            + [MplPath.LINETO] * (n_vertices - 2)
            + [MplPath.CLOSEPOLY]
        )

        all_vertices.append(coordinates)
        all_codes.extend(codes)

    for geometry in study_vector.get("geometries", []):
        if not geometry:
            continue

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")

        if geometry_type == "Polygon" and coordinates is not None:
            for ring in coordinates:
                add_ring(ring)

        elif geometry_type == "MultiPolygon" and coordinates is not None:
            for polygon in coordinates:
                for ring in polygon:
                    add_ring(ring)

    if not all_vertices:
        return None

    compound_path = MplPath(
        np.concatenate(all_vertices, axis=0),
        all_codes,
    )

    clip_patch = PathPatch(
        compound_path,
        facecolor="none",
        edgecolor="none",
        linewidth=0.0,
        transform=axis.transData,
        zorder=0,
    )

    axis.add_patch(clip_patch)
    return clip_patch


def create_pixelwise_sd_figure(
    sd_raster_path: Path,
    output_path: Path,
) -> None:
    """
    Plot the existing pixelwise cross-fold SD raster with the established
    XGB_Susceptibility_Map_Ensemble_A30_E70 cartographic style.

    This function changes visualization only. It does not change the SD raster,
    the five cross-fold realizations, the aggregation, ddof=1, domain QA, or any
    uncertainty statistic.
    """
    if not UncertaintyConfig.GENERATE_PIXELWISE_SD_FIGURE:
        return

    require_file(
        sd_raster_path,
        "cross-fold SD probability raster",
    )
    require_file(
        SD_MAP_STUDY_SHP,
        "study-area shapefile used by the Step 7.1 map style",
    )
    require_file(
        SD_MAP_ROAD_SHP,
        "road shapefile used by the Step 7.1 map style",
    )

    display_raster_path = output_path.with_name(
        output_path.stem + "_EPSG4326_Display.tif"
    )

    display_raster_path = _sd_reproject_to_epsg4326(
        source_path=sd_raster_path,
        output_path=display_raster_path,
    )

    with rasterio.open(display_raster_path) as dataset:
        sd = (
            dataset.read(1, masked=True)
            .filled(np.nan)
            .astype("float64")
        )
        bounds = dataset.bounds

    valid_sd = sd[np.isfinite(sd)]
    if valid_sd.size == 0:
        raise RuntimeError(
            "No valid values found in the cross-fold SD raster for plotting."
        )

    extent = (
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    )

    study_boundary = _sd_load_vector_overlay(
        SD_MAP_STUDY_SHP,
        "EPSG:4326",
    )
    roads = _sd_load_vector_overlay(
        SD_MAP_ROAD_SHP,
        "EPSG:4326",
    )

    display_array = np.ma.masked_invalid(sd)

    map_cmap = SD_MAP_CMAP.copy()
    map_cmap.set_bad(
        color="white",
        alpha=1.0,
    )

    sd_vmax = float(np.max(valid_sd))
    if not np.isfinite(sd_vmax) or sd_vmax <= 0.0:
        sd_vmax = 1.0

    figure, axis = plt.subplots(
        1,
        1,
        figsize=SD_MAP_FIGSIZE,
        facecolor="white",
    )
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        vmin=0.0,
        vmax=sd_vmax,
        cmap=map_cmap,
        interpolation="nearest",
        alpha=1.0,
        zorder=1,
    )

    clip_patch = _sd_build_clip_patch(
        study_boundary,
        axis,
    )
    if clip_patch is not None:
        image.set_clip_path(clip_patch)

    _sd_plot_vector_overlay(
        axis,
        study_boundary,
        colour="black",
        linewidth=2.0,
        zorder=1000,
    )
    _sd_plot_vector_overlay(
        axis,
        roads,
        colour="red",
        linewidth=1.0,
        zorder=1100,
    )

    # Exact raster extent, matching the successful Step 7.1 ensemble map.
    axis.set_xlim(extent[0], extent[1])
    axis.set_ylim(extent[2], extent[3])

    axis.xaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: _sd_to_dms(value, True)
        )
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: _sd_to_dms(value, False)
        )
    )
    axis.tick_params(
        axis="both",
        which="major",
        labelsize=SD_MAP_DMS_TICK_FONTSIZE,
    )

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")

    for label in axis.get_yticklabels():
        label.set_rotation(90)
        label.set_va("center")

    _sd_add_north_arrow(
        axis,
        size=0.08,
        pad=0.06,
    )

    _sd_add_checker_scale_bar(
        axis,
        extent[0],
        extent[1],
        extent[2],
        extent[3],
        mid_lat_deg=0.5 * (extent[2] + extent[3]),
        segments=4,
        fontsize=6,
    )

    legend_handles = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=1.5,
            label="Highway/Roads",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=2.0,
            label="Study Area Boundary",
        ),
    ]

    legend = axis.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.50),
        fontsize=SD_MAP_LEGEND_FONTSIZE,
        title="Legend",
        title_fontsize=SD_MAP_LEGEND_TITLE_FONTSIZE,
        frameon=True,
        fancybox=True,
        shadow=True,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0,
    )
    legend.get_title().set_fontweight("bold")
    legend.set_zorder(2000)

    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes(
        "right",
        size=SD_MAP_CAX_SIZE,
        pad=SD_MAP_CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(
        labelsize=SD_MAP_COLORBAR_TICK_FONTSIZE,
    )
    colorbar.set_label(
        "Cross-fold probability SD",
        fontsize=SD_MAP_COLORBAR_LABEL_FONTSIZE,
        fontweight="bold",
    )

    figure.tight_layout()

    pdf_path = output_path.with_suffix(".pdf")
    svg_path = output_path.with_suffix(".svg")

    figure.savefig(
        output_path,
        dpi=SD_MAP_DPI,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False,
    )
    figure.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False,
    )
    figure.savefig(
        svg_path,
        format="svg",
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False,
    )

    plt.close(figure)

    log(f"[SD-VIZ] PNG saved: {output_path}")
    log(f"[SD-VIZ] PDF saved: {pdf_path}")
    log(f"[SD-VIZ] SVG saved: {svg_path}")


# =============================================================================
# REPORT
# =============================================================================
def write_uncertainty_report(
    base,
    paths: dict[str, Path],
    performance_intervals: pd.DataFrame,
    additional_comparisons: pd.DataFrame,
    area_summary: pd.DataFrame,
    domain_qa: pd.DataFrame,
    sd_summary: pd.DataFrame,
) -> None:
    with open(
        paths["report"],
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(
            "STEP 7.1D — XGBOOST PERFORMANCE AND MAPPED-AREA UNCERTAINTY\n"
        )
        handle.write(
            "=" * 92
            + "\n\n"
        )

        handle.write(
            "Generated: "
            + utc_now_iso()
            + "\n\n"
        )

        handle.write(
            "SCIENTIFIC SCOPE\n"
        )
        handle.write(
            "-" * 92
            + "\n"
        )
        handle.write(
            "Primary model: successful Step 7.1 equal-HKFI, 100 m pseudo-absence "
            "buffer XGBoost framework.\n"
        )
        handle.write(
            "Step 7.1B alternative HKFI weights and Step 7.1C alternative "
            "pseudo-absence buffers are treated as separate sensitivity analyses "
            "and are not pooled into this uncertainty analysis.\n\n"
        )

        handle.write(
            "PERFORMANCE UNCERTAINTY\n"
        )
        handle.write(
            "-" * 92
            + "\n"
        )
        handle.write(
            "Intervals are spatial-block cluster-bootstrap intervals generated "
            "from genuine nested outer-fold OOF predictions.\n\n"
        )
        handle.write(
            performance_intervals.to_string(
                index=False
            )
        )
        handle.write(
            "\n\n"
        )

        handle.write(
            "ADDITIONAL PAIRED AUC COMPARISONS\n"
        )
        handle.write(
            "-" * 92
            + "\n"
        )
        handle.write(
            additional_comparisons.to_string(
                index=False
            )
        )
        handle.write(
            "\n\n"
        )

        handle.write(
            "CROSS-FOLD MAPPED-AREA VARIABILITY\n"
        )
        handle.write(
            "-" * 92
            + "\n"
        )
        handle.write(
            "The five map realizations are summarized by mean, SD, minimum and "
            "maximum. These are not formal 95% confidence intervals.\n\n"
        )
        handle.write(
            area_summary.to_string(
                index=False
            )
        )
        handle.write(
            "\n\n"
        )

        handle.write(
            "RASTER VALID-DOMAIN QA\n"
        )
        handle.write(
            "-" * 92
            + "\n"
        )
        handle.write(
            domain_qa.to_string(
                index=False
            )
        )
        handle.write(
            "\n\n"
        )

        handle.write(
            "PIXELWISE CROSS-FOLD SD SUMMARY\n"
        )
        handle.write(
            "-" * 92
            + "\n"
        )
        handle.write(
            sd_summary.to_string(
                index=False
            )
        )
        handle.write(
            "\n"
        )


# =============================================================================
# CONTROLLER
# =============================================================================
def run_controller() -> None:
    base = load_step71_module(
        UncertaintyConfig.BASE_SCRIPT
    )
    paths = output_paths(
        base
    )

    # Validate final map availability before expensive fold work.
    require_file(
        base.Config.ENSEMBLE_PROBABILITY_TIF,
        "successful final Step 7.1 A30/E70 ensemble probability raster",
    )

    write_diagnostic_provenance(
        paths["out"],
        "Step_7_1D_XGB",
        {
            "performance_uncertainty": (
                "cluster bootstrap resampling whole spatial blocks"
            ),
            "performance_inputs": (
                "successful Step 7.1 nested OOF predictions"
            ),
            "mapped_area_uncertainty": (
                "five process-isolated outer-fold A30/E70 map realizations"
            ),
            "map_model_parameters": (
                "exact successful Step 7.1 fold-specific best parameters; "
                "no retuning"
            ),
            "mapped_area_reporting": (
                "cross-fold mean, SD, min-max; not a formal 95% CI"
            ),
            "final_map_reference": str(
                base.Config.ENSEMBLE_PROBABILITY_TIF
            ),
            "valid_domain_rule": (
                "all five fold maps must have identical masks and match "
                "the final full-data ensemble mask"
            ),
            "step_7_1B_included": False,
            "step_7_1C_alternative_buffers_included": False,
        },
    )

    log(
        "=" * 96
    )
    log(
        "STEP 7.1D — PERFORMANCE AND MAPPED-AREA UNCERTAINTY"
    )
    log(
        "=" * 96
    )

    # -------------------------------------------------------------------------
    # 1. Performance uncertainty — no model fitting.
    # -------------------------------------------------------------------------
    log(
        "[1/6] Calculating cluster-bootstrap uncertainty "
        "for Models A-F and the A30/E70 ensemble..."
    )
    performance_intervals = (
        build_performance_uncertainty(
            base,
            paths["out"],
        )
    )

    # -------------------------------------------------------------------------
    # 2. Additional direct paired AUC comparisons.
    # -------------------------------------------------------------------------
    log(
        "[2/6] Calculating direct E-C and E-D paired AUC comparisons..."
    )
    additional_comparisons = (
        add_missing_E_C_and_E_D_comparisons(
            base,
            paths["out"],
        )
    )

    # -------------------------------------------------------------------------
    # 3. Five process-isolated cross-fold ensemble maps.
    # -------------------------------------------------------------------------
    log(
        "[3/6] Building/resuming five process-isolated "
        "cross-fold A30/E70 map realizations..."
    )

    for outer_fold in base.Config.EXPECTED_FOLDS:
        fold_output = fold_paths(
            base,
            outer_fold,
        )

        complete = (
            fold_output["done"].exists()
            and fold_output["ensemble_raster"].exists()
            and fold_output["area_csv"].exists()
            and fold_output["metadata_json"].exists()
        )

        if (
            UncertaintyConfig.RESUME_FROM_CHECKPOINTS
            and complete
        ):
            log(
                f"[RESUME] Outer fold {outer_fold} already complete."
            )
            continue

        run_external_fold_worker(
            outer_fold
        )

    # -------------------------------------------------------------------------
    # 4. Collect areas and validate raster domains.
    # -------------------------------------------------------------------------
    log(
        "[4/6] Collecting fold-area results and checking "
        "identical raster support domains..."
    )

    area_raw, ensemble_rasters = (
        collect_fold_area_results(
            base
        )
    )

    area_raw.to_csv(
        paths["area_raw"],
        index=False,
        float_format="%.10f",
    )

    domain_qa = (
        validate_cross_fold_raster_domains(
            fold_rasters=ensemble_rasters,
            final_full_data_raster=(
                base.Config.ENSEMBLE_PROBABILITY_TIF
            ),
            output_csv=paths["domain_qa"],
        )
    )

    # -------------------------------------------------------------------------
    # 5. Area statistics and pixelwise mean/SD.
    # -------------------------------------------------------------------------
    log(
        "[5/6] Summarizing mapped-area variability and "
        "aggregating pixelwise mean/SD rasters..."
    )

    area_summary = (
        summarize_mapped_area_uncertainty(
            base=base,
            area_raw=area_raw,
        )
    )

    area_summary.to_csv(
        paths["area_summary"],
        index=False,
        float_format="%.10f",
    )

    aggregate_fold_rasters(
        fold_rasters=ensemble_rasters,
        mean_output=paths["mean_raster"],
        sd_output=paths["sd_raster"],
    )

    sd_summary = (
        summarize_pixelwise_sd(
            sd_raster_path=paths["sd_raster"],
            output_csv=paths["sd_summary"],
        )
    )

    # -------------------------------------------------------------------------
    # 6. Figures and final report.
    # -------------------------------------------------------------------------
    log(
        "[6/6] Creating figures and final uncertainty report..."
    )

    create_area_uncertainty_figure(
        area_summary=area_summary,
        output_path=paths["area_figure"],
    )

    create_pixelwise_sd_figure(
        sd_raster_path=paths["sd_raster"],
        output_path=paths["sd_figure"],
    )

    write_uncertainty_report(
        base=base,
        paths=paths,
        performance_intervals=performance_intervals,
        additional_comparisons=additional_comparisons,
        area_summary=area_summary,
        domain_qa=domain_qa,
        sd_summary=sd_summary,
    )

    log(
        ""
    )
    log(
        "=" * 96
    )
    log(
        "STEP 7.1D COMPLETED SUCCESSFULLY"
    )
    log(
        "=" * 96
    )
    log(
        f"[OUT] {paths['out']}"
    )

    log(
        ""
    )
    log(
        "Additional paired AUC comparisons:"
    )
    print(
        additional_comparisons.to_string(
            index=False
        )
    )

    log(
        ""
    )
    log(
        "Cross-fold mapped-area uncertainty:"
    )
    print(
        area_summary.to_string(
            index=False
        )
    )

    log(
        ""
    )
    log(
        "Pixelwise SD summary:"
    )
    print(
        sd_summary.to_string(
            index=False
        )
    )


# =============================================================================
# COMMAND-LINE ENTRY POINT
# =============================================================================
def parse_command_line_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Step 7.1D XGBoost performance and mapped-area uncertainty."
        )
    )

    parser.add_argument(
        "--worker-fold",
        type=int,
        default=None,
        help=(
            "Internal process-isolated worker mode. "
            "Valid values are 1-5."
        ),
    )

    arguments, unknown = (
        parser.parse_known_args()
    )

    # Jupyter/IPython can inject arguments such as -f kernel.json.
    if unknown:
        log(
            "[CLI] Ignoring host-environment arguments: "
            f"{unknown}"
        )

    return arguments


if __name__ == "__main__":
    try:
        args = (
            parse_command_line_arguments()
        )

        if args.worker_fold is None:
            run_controller()
        else:
            base_for_validation = (
                load_step71_module(
                    UncertaintyConfig.BASE_SCRIPT
                )
            )

            if args.worker_fold not in (
                base_for_validation.Config.EXPECTED_FOLDS
            ):
                raise ValueError(
                    "--worker-fold must be one of "
                    f"{base_for_validation.Config.EXPECTED_FOLDS}; "
                    f"received {args.worker_fold}."
                )

            run_outer_fold_map_worker(
                int(
                    args.worker_fold
                )
            )

    except Exception as exc:
        print(
            f"[ERROR] Step 7.1D failed: {exc}",
            flush=True,
        )
        traceback.print_exc()
        raise


# Step 7.2 — Random Forest Models A–F with Nested Spatial Cross-Validation

In [ ]:
# =============================================================================
# Step 7.2 — Random Forest Models A–F with Nested Spatial Cross-Validation
# =============================================================================
#
# Purpose
# -------
# Train and evaluate Random Forest for all six frozen A–F predictor configurations
# using the primary frozen Step 6.4 ML dataset and EXACT frozen five spatial folds.
#
# Design
# ------
# Outer CV (unbiased evaluation):
#   - One frozen spatial block is held out.
#   - The remaining four blocks are used for all model selection.
#
# Inner CV (hyperparameter tuning):
#   - Each of the four remaining spatial blocks serves once as the inner
#     validation fold via PredefinedSplit.
#   - Grid-search AUC is computed only inside the outer-training data.
#
# After nested-CV evaluation:
#   - OOF predictions are pooled across the five outer folds.
#   - A separate full-data spatial grid search can be enabled later to select
#     final deployment/map models. These full-data models are NOT used for
#     validation statistics.
#
# Important scientific rules
# --------------------------
# 1. Never regenerate spatial folds. Use Step 6.4 cv_fold exactly.
# 2. Models A–F use identical rows, labels and outer folds.
# 3. No global imputation, scaling, downsampling or train/test split.
# 4. Land cover is treated as categorical and one-hot encoded inside each fit.
# 5. Random Forest continuous predictors are not standardized.
# 6. IRS is never used as an ML predictor.
# 7. Outer validation data are never used for tuning or model selection.
# 8. Binary metrics use the fixed threshold 0.50. AUC remains primary.
# 9. Cluster-bootstrap confidence intervals resample the five spatial blocks,
#    preserving the spatial grouping structure better than sample-wise bootstrap.
# 10. Final susceptibility visualization uses the direct Model E deployment
#     probability raster; it never substitutes for nested spatial OOF validation.
#
# Primary reviewer comparisons supported
# --------------------------------------
# B - A : added value of raw DV+
# C - A : added value of HKFI alone
# D - A : added value of SNDR alone
# E - A : total added value of the proposed physics-guided indices
# F - A : added value of supplying DV+, TWI and SM as separate source variables
# E - B : physics-guided indices versus raw deformation alone
# E - F : transformed indices versus supplying source variables separately
# =============================================================================

from __future__ import annotations

import os
# =============================================================================
# RESOURCE CONTROL
# =============================================================================
# Random Forest itself and GridSearchCV are both kept single-threaded here.
# This avoids nested CPU parallelism, makes runs more reproducible, and follows
# the same resource-safe principle used in the successful Step 7.2 workflow.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


import json
import os
import re
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Sequence, Tuple

import joblib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Polygon, Rectangle
from matplotlib.ticker import FuncFormatter, LinearLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling,
    transform_geom,
)
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Optional cartographic/vector libraries.
try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False


# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Step 6.1 frozen predictor definitions and raster stack.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"
    MASTER_STACK_TIF = (
        STEP61_DIR
        / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
    )
    COMMON_AF_MASK_TIF = (
        STEP61_DIR / "common_A_to_F_valid_mask_32645_12p5m_byte.tif"
    )

    # Step 6.4 frozen modelling contract.
    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    MODEL_READY_DATASET_CSV = STEP64_DIR / "model_ready_dataset_A_to_F.csv"
    STEP64_MODEL_CONFIG_JSON = STEP64_DIR / "model_configuration_manifest.json"

    # Outputs.
    OUT_DIR = RESULTS_DIR / "Step-7.2-RF-Models_A_to_F"
    MODEL_DIR = OUT_DIR / "final_models"
    MAP_DIR = OUT_DIR / "susceptibility_maps"
    SEARCH_DIR = OUT_DIR / "grid_search_details"

    OOF_PREDICTIONS_CSV = OUT_DIR / "RF_A_to_F_OOF_predictions.csv"
    FOLD_METRICS_CSV = OUT_DIR / "RF_A_to_F_outer_fold_metrics.csv"
    SUMMARY_METRICS_CSV = OUT_DIR / "RF_A_to_F_summary_metrics.csv"
    AUC_COMPARISONS_CSV = OUT_DIR / "RF_A_to_F_paired_AUC_comparisons.csv"
    OUTER_PARAMS_JSON = OUT_DIR / "RF_A_to_F_outer_fold_best_parameters.json"
    FINAL_PARAMS_JSON = OUT_DIR / "RF_A_to_F_final_full_data_best_parameters.json"
    INNER_SEARCH_CSV = SEARCH_DIR / "RF_A_to_F_nested_inner_grid_results.csv"
    FINAL_SEARCH_CSV = SEARCH_DIR / "RF_A_to_F_full_data_grid_results.csv"
    ROC_FIG_PNG = OUT_DIR / "RF_A_to_F_OOF_ROC_Curves.png"
    ROC_FIG_PDF = OUT_DIR / "RF_A_to_F_OOF_ROC_Curves.pdf"
    ROC_FIG_SVG = OUT_DIR / "RF_A_to_F_OOF_ROC_Curves.svg"
    FOLD_AUC_FIG_PNG = OUT_DIR / "RF_A_to_F_Fold_AUC.png"
    FOLD_AUC_FIG_PDF = OUT_DIR / "RF_A_to_F_Fold_AUC.pdf"
    FOLD_AUC_FIG_SVG = OUT_DIR / "RF_A_to_F_Fold_AUC.svg"
    QA_REPORT_TXT = OUT_DIR / "Step_7_2_RF_A_to_F_QA_Report.txt"
    METADATA_JSON = OUT_DIR / "Step_7_2_RF_A_to_F_Metadata.json"

    # Manuscript-ready Excel workbook containing the compact A-F performance
    # table, paired AUC comparisons, fold-level AUCs and ensemble summary.
    MANUSCRIPT_TABLES_XLSX = (
        OUT_DIR / "RF_A_to_F_Manuscript_Performance_Tables.xlsx"
    )

    # Cartographic overlays used by the established Step 5.3 map style.
    STUDY_SHP = BASE / "Study_Area.shp"
    ROAD_SHP = BASE / "NTD_Road.shp"
    VISUALIZATION_DIR = MAP_DIR / "visualization"

    # Primary A-F model retained for scientific interpretation.
    PRIMARY_VISUALIZATION_SCENARIO = "E_physics_guided"

    # Old-style static + integrated ensemble analogue for the revised A-F design.
    #
    # Old workflow:
    #   final = 0.30 * context/static + 0.70 * integrated
    #   where integrated support exists; otherwise retain static background.
    #
    # Revised workflow:
    #   final = 0.30 * Model A + 0.70 * Model E
    #   where Model E predictors are available; otherwise retain Model A.
    #
    # The 0.70 weight is fixed a priori to reproduce the earlier ensemble logic.
    # It is NOT tuned on the OOF predictions.
    GENERATE_AE_ENSEMBLE_LSM = True
    ENSEMBLE_STATIC_SCENARIO = "A_static_baseline"
    ENSEMBLE_ENHANCED_SCENARIO = "E_physics_guided"
    ENSEMBLE_ENHANCED_WEIGHT = 0.70

    ENSEMBLE_STATIC_FULL_DOMAIN_TIF = (
        MAP_DIR / "A-static-baseline_RF_probability_full_available_domain.tif"
    )
    ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF = (
        MAP_DIR / "E-physics-guided_RF_probability_full_available_domain.tif"
    )
    ENSEMBLE_PROBABILITY_TIF = (
        MAP_DIR / "RF_Ensemble_A30_E70_probability_full_study_domain.tif"
    )
    ENSEMBLE_SUPPORT_TIF = (
        MAP_DIR / "RF_Ensemble_A30_E70_information_support.tif"
    )
    ENSEMBLE_OOF_PREDICTIONS_CSV = (
        OUT_DIR / "RF_Ensemble_A30_E70_OOF_predictions.csv"
    )
    ENSEMBLE_OOF_METRICS_JSON = (
        OUT_DIR / "RF_Ensemble_A30_E70_OOF_metrics.json"
    )

    # Step 5.3-style map/histogram settings.
    # Publication-size figures.
    # The two requested maps are explicitly controlled independently so that
    # later changes to other figures cannot accidentally alter their dimensions.
    ENSEMBLE_MAP_FIGSIZE = (4, 6)
    SUPPORT_MAP_FIGSIZE = (4, 6)

    # Other visualization products.
    MAP_FIGSIZE = (4, 4)
    HIST_FIGSIZE = (4, 4)
    AREA_PERCENT_FIGSIZE = (4, 4)

    # Enlarged cartographic text for the 4 x 4 inch figure.
    DMS_TICK_FONTSIZE = 7  # Reduced from 9 to prevent latitude label overlap
    LEGEND_FONTSIZE = 7
    LEGEND_TITLE_FONTSIZE = 8
    COLORBAR_TICK_FONTSIZE = 8
    COLORBAR_LABEL_FONTSIZE = 9
    NORTH_ARROW_FONTSIZE = 8  # Slightly reduced for balance
    SCALEBAR_FONTSIZE = 7

    CAX_SIZE = "5%"
    CAX_PAD = 0.01
    FIG_DPI = 900
    DISPLAY_BUFFER_FRACTION = 0.00
    LONGITUDE_DMS_TICK_COUNT = 4
    RASTER_DISPLAY_INTERPOLATION = "nearest"
    SHOW_LSM_FIGURES = True
    SAVE_LSM_FIGURES = True

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"
    CATEGORICAL_FEATURE_ALIASES = {"landcover", "landuse", "land_use"}

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    MODEL_DESCRIPTIONS = {
        "A_static_baseline": "Static baseline",
        "B_raw_deformation": "Static + DV+",
        "C_hkfi_only": "Static + HKFI",
        "D_sndr_only": "Static + SNDR",
        "E_physics_guided": "Static + HKFI + SNDR",
        "F_untransformed_sources": "Static + DV+ + TWI + SM",
    }

    # Random-Forest-specific 48-combination grid.
    #
    # The modelling principle is identical to Step 7.1:
    #   - hyperparameters are selected only inside the four outer-training blocks;
    #   - the untouched outer block is used once for unbiased evaluation;
    #   - the same grid is applied to Models A-F.
    #
    # Grid size:
    #   2 n_estimators
    # x 3 max_depth
    # x 2 min_samples_leaf
    # x 2 max_features
    # x 2 criterion
    # = 48 combinations.
    PARAM_GRID = {
        "model__n_estimators": [200, 500],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 2],
        "model__max_features": ["sqrt", 0.5],
        "model__criterion": ["gini", "entropy"],
    }

    # Fixed Random Forest settings shared by A-F.
    #
    # The dataset is already balanced 1:1, therefore class_weight is left as
    # None rather than imposing an additional weighting scheme.
    RF_FIXED_PARAMS = {
        "bootstrap": True,
        "class_weight": None,
        "random_state": 42,
        "n_jobs": 1,
    }

    GRIDSEARCH_N_JOBS = 1  # Avoid nested Random Forest / GridSearchCV parallelism
    GRIDSEARCH_VERBOSE = 2
    RANDOM_STATE = 42
    BINARY_THRESHOLD = 0.50

    # Spatial-block cluster bootstrap.
    N_CLUSTER_BOOTSTRAP = 5000
    CI_LEVEL = 0.95

    # Complete execution controls.
    #
    # True  -> after nested spatial-CV evaluation, tune and fit the final
    #          full-data deployment models for A-F.
    # False -> stop after leakage-safe nested spatial OOF evaluation.
    RUN_FINAL_DEPLOYMENT_TUNING = True

    # True  -> generate A-F full-domain probability rasters from the final
    #          deployment models. Requires RUN_FINAL_DEPLOYMENT_TUNING=True.
    GENERATE_RASTER_MAPS = True

    # True  -> create the publication-style primary Model E landslide
    #          susceptibility map and probability histogram.
    GENERATE_PRIMARY_LSM_VISUALIZATION = True
    RASTER_BLOCK_ROWS = 512
    OUTPUT_NODATA = -9999.0

    # Keep nested-CV fold estimators only if needed for later diagnostics.
    SAVE_OUTER_FOLD_MODELS = False


# Create output folders once the script is imported/run.
for _directory in (
    Config.OUT_DIR,
    Config.MODEL_DIR,
    Config.MAP_DIR,
    Config.SEARCH_DIR,
    Config.VISUALIZATION_DIR,
):
    _directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


# =============================================================================
# STATISTICAL FIGURE STYLE HELPERS
# =============================================================================
# Kept consistent with the revised Step 7.1 plotting style.
STAT_AXIS_LABEL_FONTSIZE = 11
STAT_TICK_LABEL_FONTSIZE = 10
STAT_LEGEND_FONTSIZE = 9
STAT_TITLE_FONTSIZE = 11
STAT_FIG_DPI = 600


def apply_statistical_axis_style(
    ax: plt.Axes,
    legend_fontsize: float = STAT_LEGEND_FONTSIZE,
) -> None:
    """Apply bold axis labels/ticks and clearer in-figure text."""
    ax.xaxis.label.set_fontsize(STAT_AXIS_LABEL_FONTSIZE)
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontsize(STAT_AXIS_LABEL_FONTSIZE)
    ax.yaxis.label.set_fontweight("bold")

    ax.title.set_fontsize(STAT_TITLE_FONTSIZE)
    ax.title.set_fontweight("bold")

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=STAT_TICK_LABEL_FONTSIZE,
        width=0.8,
        length=3.5,
    )

    # Make all numeric/category tick labels bold.
    for tick_label in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        tick_label.set_fontsize(STAT_TICK_LABEL_FONTSIZE)
        tick_label.set_fontweight("bold")

    legend = ax.get_legend()
    if legend is not None:
        for label in legend.get_texts():
            label.set_fontsize(legend_fontsize)


def save_statistical_figure_triplet(
    fig: plt.Figure,
    png_path: Path,
    pdf_path: Path,
    svg_path: Path,
    dpi: int = STAT_FIG_DPI,
) -> None:
    """Save a statistical figure in PNG, PDF, and SVG formats."""
    fig.tight_layout()
    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        svg_path,
        format="svg",
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json(path: Path) -> Dict[str, Any]:
    require_file(path, "JSON file")
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


def scenario_prefix(scenario_name: str) -> str:
    return scenario_name.replace("_", "-")


def normalize_feature_name(name: str) -> str:
    """Normalize a feature name for robust alias matching."""
    return re.sub(r"[^a-z0-9_]+", "_", str(name).strip().lower()).strip("_")


def is_categorical_feature(name: str) -> bool:
    """Identify categorical predictors using conservative land-cover aliases."""
    normalized = normalize_feature_name(name)
    compact = normalized.replace("_", "")
    aliases = {alias.lower() for alias in Config.CATEGORICAL_FEATURE_ALIASES}
    compact_aliases = {alias.replace("_", "") for alias in aliases}
    return normalized in aliases or compact in compact_aliases


# =============================================================================
# DATA / MANIFEST VALIDATION
# =============================================================================
def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """Cross-check Step 6.1 and frozen Step 6.4 A–F definitions."""
    step61 = load_json(Config.SCENARIO_MANIFEST_JSON).get("scenarios", {})
    step64 = load_json(Config.STEP64_MODEL_CONFIG_JSON).get("scenarios", {})

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                f"Step 6.1 scenario manifest is missing {scenario_name}."
            )
        if scenario_name not in step64:
            raise RuntimeError(
                f"Step 6.4 frozen model manifest is missing {scenario_name}."
            )

        features_61 = list(step61[scenario_name].get("features", []))
        features_64 = list(step64[scenario_name].get("features", []))

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any("irs" in feature.lower() for feature in features_61):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; IRS must not be an ML predictor."
            )

    log("[QA] Step 6.1 and Step 6.4 A–F scenario definitions match: PASS")
    return step64


def load_and_validate_model_ready_dataset(
    scenarios: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    """Load the primary frozen 330-row Step 6.4 ML dataset with strict QA."""
    require_file(
        Config.MODEL_READY_DATASET_CSV,
        "Step 6.4 primary frozen model-ready ML dataset",
    )

    df = pd.read_csv(Config.MODEL_READY_DATASET_CSV)

    required_metadata = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ]

    all_features: List[str] = []
    for scenario_name in Config.SCENARIO_ORDER:
        for feature in scenarios[scenario_name]["features"]:
            if feature not in all_features:
                all_features.append(feature)

    missing = [
        column
        for column in required_metadata + all_features
        if column not in df.columns
    ]
    if missing:
        raise RuntimeError(f"Model-ready dataset is missing columns: {missing}")

    if len(df) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Expected {Config.EXPECTED_N_ROWS} rows; found {len(df)}. "
            "No silent row dropping is allowed."
        )

    if df[Config.SAMPLE_ID_COLUMN].isna().any():
        raise RuntimeError("Missing sample_id values found.")
    if df[Config.SAMPLE_ID_COLUMN].duplicated().any():
        raise RuntimeError("Duplicated sample_id values found.")

    df[Config.TARGET_COLUMN] = pd.to_numeric(
        df[Config.TARGET_COLUMN], errors="raise"
    ).astype(int)
    df[Config.FOLD_COLUMN] = pd.to_numeric(
        df[Config.FOLD_COLUMN], errors="raise"
    ).astype(int)
    df[Config.BLOCK_COLUMN] = pd.to_numeric(
        df[Config.BLOCK_COLUMN], errors="raise"
    ).astype(int)

    class_counts = df[Config.TARGET_COLUMN].value_counts().sort_index().to_dict()
    expected_counts = {
        0: Config.EXPECTED_N_NEGATIVE,
        1: Config.EXPECTED_N_POSITIVE,
    }
    if class_counts != expected_counts:
        raise RuntimeError(
            f"Unexpected class counts: {class_counts}; expected {expected_counts}."
        )

    folds = sorted(df[Config.FOLD_COLUMN].unique().tolist())
    if folds != Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected frozen folds {Config.EXPECTED_FOLDS}; found {folds}."
        )

    if not np.array_equal(
        df[Config.FOLD_COLUMN].to_numpy(),
        df[Config.BLOCK_COLUMN].to_numpy(),
    ):
        raise RuntimeError("cv_fold and spatial_block are not identical.")

    # Every frozen block must contain exactly 33 positives and 33 pseudo-absences.
    block_class = pd.crosstab(
        df[Config.FOLD_COLUMN],
        df[Config.TARGET_COLUMN],
    )
    for required_class in (0, 1):
        if required_class not in block_class.columns:
            raise RuntimeError(
                f"Frozen fold table is missing class {required_class}."
            )

    expected_per_class_per_fold = Config.EXPECTED_N_POSITIVE // len(Config.EXPECTED_FOLDS)
    if expected_per_class_per_fold * len(Config.EXPECTED_FOLDS) != Config.EXPECTED_N_POSITIVE:
        raise RuntimeError(
            "Expected positive count is not divisible evenly across the frozen folds."
        )

    for fold in Config.EXPECTED_FOLDS:
        if fold not in block_class.index:
            raise RuntimeError(f"Missing fold {fold}.")
        n0 = int(block_class.loc[fold, 0])
        n1 = int(block_class.loc[fold, 1])
        if n0 != expected_per_class_per_fold or n1 != expected_per_class_per_fold:
            raise RuntimeError(
                f"Fold {fold} violates the frozen Step 6.4 contract: "
                f"negative={n0}, positive={n1}; expected "
                f"{expected_per_class_per_fold}+{expected_per_class_per_fold}=66."
            )

    log(
        "[QA] Frozen fold balance: each fold contains "
        f"{expected_per_class_per_fold} positives + "
        f"{expected_per_class_per_fold} pseudo-absences: PASS"
    )

    # Step 6.4 should already guarantee complete predictors; verify again.
    predictor_values = df[all_features].apply(pd.to_numeric, errors="coerce")
    if not np.isfinite(predictor_values.to_numpy(dtype="float64")).all():
        bad = np.argwhere(
            ~np.isfinite(predictor_values.to_numpy(dtype="float64"))
        )
        examples = [
            (
                str(df.iloc[row_i][Config.SAMPLE_ID_COLUMN]),
                all_features[col_i],
            )
            for row_i, col_i in bad[:20]
        ]
        raise RuntimeError(
            "Missing/non-finite predictors remain in the model-ready dataset. "
            f"Examples: {examples}"
        )

    irs_columns = [
        column for column in df.columns if "irs" in str(column).lower()
    ]
    if irs_columns:
        raise RuntimeError(
            f"IRS-related columns found in ML dataset: {irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError("tpi_s25 found; only tpi_s5 is allowed.")

    log(
        "[QA] Model-ready dataset: "
        f"rows={len(df)}, positive={class_counts[1]}, negative={class_counts[0]}, "
        f"folds={folds}: PASS"
    )
    return df


# =============================================================================
# MODEL BUILDING / NESTED SPATIAL CV
# =============================================================================
def build_rf_pipeline(features: Sequence[str]) -> Pipeline:
    """Create fold-safe preprocessing plus Random Forest pipeline."""
    categorical = [
        feature for feature in features if is_categorical_feature(feature)
    ]
    numeric = [
        feature for feature in features if not is_categorical_feature(feature)
    ]

    transformers = []
    if numeric:
        transformers.append(("numeric", "passthrough", numeric))
    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = RandomForestClassifier(**Config.RF_FIXED_PARAMS)

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", classifier),
        ]
    )


def make_inner_predefined_split(outer_train_df: pd.DataFrame) -> PredefinedSplit:
    """
    Build a four-fold spatial inner CV from the four blocks remaining after one
    outer block has been held out.
    """
    blocks = sorted(outer_train_df[Config.FOLD_COLUMN].unique().tolist())
    if len(blocks) != 4:
        raise RuntimeError(
            f"Nested CV expected four inner blocks; found {blocks}."
        )

    block_to_test_fold = {
        block: inner_index for inner_index, block in enumerate(blocks)
    }
    test_fold = outer_train_df[Config.FOLD_COLUMN].map(block_to_test_fold).to_numpy()
    return PredefinedSplit(test_fold=test_fold)


def make_full_data_predefined_split(df: pd.DataFrame) -> PredefinedSplit:
    """Use the frozen five blocks for final deployment-model tuning."""
    test_fold = df[Config.FOLD_COLUMN].to_numpy(dtype=int) - 1
    return PredefinedSplit(test_fold=test_fold)


def classification_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = Config.BINARY_THRESHOLD,
) -> Dict[str, Any]:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def cluster_bootstrap_auc_ci(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> Tuple[float, float, int]:
    """Cluster bootstrap AUC by resampling whole spatial blocks."""
    rng = np.random.default_rng(seed)
    unique_blocks = np.unique(blocks)
    values: List[float] = []

    indices_by_block = {
        block: np.flatnonzero(blocks == block) for block in unique_blocks
    }

    for i in range(n_bootstrap):
        if (i + 1) % 1000 == 0:
            log(f"  Bootstrap progress: {i + 1}/{n_bootstrap}")
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(unique_blocks),
            replace=True,
        )
        sampled_indices = np.concatenate(
            [indices_by_block[block] for block in sampled_blocks]
        )
        y_b = y_true[sampled_indices]
        p_b = y_prob[sampled_indices]

        if np.unique(y_b).size < 2:
            continue

        values.append(float(roc_auc_score(y_b, p_b)))

    if not values:
        return float("nan"), float("nan"), 0

    alpha = (1.0 - ci_level) / 2.0
    low, high = np.quantile(values, [alpha, 1.0 - alpha])
    return float(low), float(high), len(values)


def paired_cluster_bootstrap_auc_difference(
    y_true: np.ndarray,
    prob_left: np.ndarray,
    prob_right: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> Tuple[float, float, float, int]:
    """
    Paired spatial-block bootstrap for AUC(left) - AUC(right), preserving the
    identical samples/folds used by all A–F models.
    """
    observed = float(
        roc_auc_score(y_true, prob_left) - roc_auc_score(y_true, prob_right)
    )

    rng = np.random.default_rng(seed)
    unique_blocks = np.unique(blocks)
    indices_by_block = {
        block: np.flatnonzero(blocks == block) for block in unique_blocks
    }
    differences: List[float] = []

    for i in range(n_bootstrap):
        if (i + 1) % 1000 == 0:
            log(f"  Bootstrap progress: {i + 1}/{n_bootstrap}")
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(unique_blocks),
            replace=True,
        )
        sampled_indices = np.concatenate(
            [indices_by_block[block] for block in sampled_blocks]
        )
        y_b = y_true[sampled_indices]

        if np.unique(y_b).size < 2:
            continue

        difference = (
            roc_auc_score(y_b, prob_left[sampled_indices])
            - roc_auc_score(y_b, prob_right[sampled_indices])
        )
        differences.append(float(difference))

    if not differences:
        return observed, float("nan"), float("nan"), 0

    alpha = (1.0 - ci_level) / 2.0
    low, high = np.quantile(differences, [alpha, 1.0 - alpha])
    return observed, float(low), float(high), len(differences)


def normalize_grid_results(
    cv_results: Dict[str, Any],
    scenario_name: str,
    search_stage: str,
    outer_fold: int | None,
) -> pd.DataFrame:
    """Convert GridSearchCV.cv_results_ to a compact audit table."""
    frame = pd.DataFrame(cv_results)

    keep = [
        column
        for column in frame.columns
        if column.startswith("param_")
        or column
        in {
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "mean_fit_time",
            "std_fit_time",
        }
    ]

    frame = frame[keep].copy()
    frame.insert(0, "scenario", scenario_name)
    frame.insert(1, "model", Config.MODEL_SHORT_LABELS[scenario_name])
    frame.insert(2, "search_stage", search_stage)
    frame.insert(3, "outer_fold", outer_fold)
    return frame


def run_nested_spatial_cv_for_scenario(
    df: pd.DataFrame,
    scenario_name: str,
    features: Sequence[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any], pd.DataFrame]:
    """Run five outer spatial folds with four-block inner spatial grid search."""
    log("")
    log("=" * 96)
    log(
        f"[NESTED-CV] {Config.MODEL_SHORT_LABELS[scenario_name]} — "
        f"{Config.MODEL_DESCRIPTIONS[scenario_name]} ({len(features)} predictors)"
    )
    log("=" * 96)

    oof_probability = np.full(len(df), np.nan, dtype="float64")
    oof_predicted = np.full(len(df), -1, dtype="int16")

    fold_rows: List[Dict[str, Any]] = []
    best_params: Dict[str, Any] = {}
    search_tables: List[pd.DataFrame] = []

    for outer_fold in Config.EXPECTED_FOLDS:
        outer_train_mask = df[Config.FOLD_COLUMN].to_numpy() != outer_fold
        outer_valid_mask = df[Config.FOLD_COLUMN].to_numpy() == outer_fold

        train_df = df.loc[outer_train_mask].copy()
        valid_df = df.loc[outer_valid_mask].copy()

        if train_df[Config.FOLD_COLUMN].nunique() != 4:
            raise RuntimeError(
                f"Outer fold {outer_fold}: expected four training blocks."
            )
        if valid_df[Config.FOLD_COLUMN].nunique() != 1:
            raise RuntimeError(
                f"Outer fold {outer_fold}: validation must contain one block only."
            )

        X_train = train_df[list(features)]
        y_train = train_df[Config.TARGET_COLUMN].to_numpy(dtype=int)
        X_valid = valid_df[list(features)]
        y_valid = valid_df[Config.TARGET_COLUMN].to_numpy(dtype=int)

        inner_cv = make_inner_predefined_split(train_df)
        pipeline = build_rf_pipeline(features)

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=Config.PARAM_GRID,
            scoring="roc_auc",
            cv=inner_cv,
            refit=True,
            n_jobs=Config.GRIDSEARCH_N_JOBS,
            verbose=Config.GRIDSEARCH_VERBOSE,
            return_train_score=False,
            error_score="raise",
        )

        log(
            f"[OUTER {outer_fold}] Tuning on blocks "
            f"{sorted(train_df[Config.FOLD_COLUMN].unique().tolist())}; "
            f"holding out block {outer_fold}."
        )
        search.fit(X_train, y_train)

        probability = search.best_estimator_.predict_proba(X_valid)[:, 1]
        predicted = (probability >= Config.BINARY_THRESHOLD).astype(int)

        valid_indices = np.flatnonzero(outer_valid_mask)
        oof_probability[valid_indices] = probability
        oof_predicted[valid_indices] = predicted

        metrics = classification_metrics(y_valid, probability)

        row = {
            "scenario": scenario_name,
            "model": Config.MODEL_SHORT_LABELS[scenario_name],
            "description": Config.MODEL_DESCRIPTIONS[scenario_name],
            "outer_fold": int(outer_fold),
            "n_predictors": int(len(features)),
            "n_train": int(len(train_df)),
            "n_validation": int(len(valid_df)),
            "n_train_positive": int(y_train.sum()),
            "n_train_negative": int(len(y_train) - y_train.sum()),
            "n_validation_positive": int(y_valid.sum()),
            "n_validation_negative": int(len(y_valid) - y_valid.sum()),
            "inner_best_mean_auc": float(search.best_score_),
            **metrics,
        }
        fold_rows.append(row)

        best_params[str(outer_fold)] = {
            "inner_best_mean_auc": float(search.best_score_),
            "best_params": json_safe(search.best_params_),
        }

        search_tables.append(
            normalize_grid_results(
                search.cv_results_,
                scenario_name=scenario_name,
                search_stage="nested_inner",
                outer_fold=outer_fold,
            )
        )

        if Config.SAVE_OUTER_FOLD_MODELS:
            fold_model_path = (
                Config.MODEL_DIR
                / f"{scenario_prefix(scenario_name)}_outer-fold-{outer_fold}.joblib"
            )
            joblib.dump(search.best_estimator_, fold_model_path)

        log(
            f"[OUTER {outer_fold}] inner best AUC={search.best_score_:.4f}; "
            f"held-out AUC={metrics['roc_auc']:.4f}; "
            f"F1={metrics['f1']:.4f}"
        )

    if not np.isfinite(oof_probability).all():
        missing_count = int((~np.isfinite(oof_probability)).sum())
        raise RuntimeError(
            f"{scenario_name}: {missing_count} samples did not receive OOF predictions."
        )
    if (oof_predicted < 0).any():
        raise RuntimeError(f"{scenario_name}: incomplete OOF class predictions.")

    oof_df = df[
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
        ]
    ].copy()
    oof_df.insert(0, "scenario", scenario_name)
    oof_df.insert(1, "model", Config.MODEL_SHORT_LABELS[scenario_name])
    oof_df.insert(2, "description", Config.MODEL_DESCRIPTIONS[scenario_name])
    oof_df["oof_probability"] = oof_probability
    oof_df["oof_predicted_class_0p5"] = oof_predicted

    fold_df = pd.DataFrame(fold_rows)
    search_df = pd.concat(search_tables, ignore_index=True)

    return oof_df, fold_df, best_params, search_df




def build_fixed_ae_oof_ensemble(
    all_oof: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Build and evaluate the fixed Model A / Model E OOF ensemble.

    This applies the same fixed A30/E70 operational blending principle used in Step 7.1:
        P_ensemble = 0.30 * P_static + 0.70 * P_enhanced

    Because the weight is fixed before inspecting the OOF ensemble result,
    the resulting OOF metric can be reported as a separate, pre-specified
    ensemble diagnostic. It does not replace the primary A-F ablation results.
    """
    static_name = Config.ENSEMBLE_STATIC_SCENARIO
    enhanced_name = Config.ENSEMBLE_ENHANCED_SCENARIO
    enhanced_weight = float(Config.ENSEMBLE_ENHANCED_WEIGHT)
    static_weight = 1.0 - enhanced_weight

    static_oof = all_oof.loc[
        all_oof["scenario"] == static_name,
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
            "oof_probability",
        ],
    ].copy()
    static_oof = static_oof.rename(
        columns={"oof_probability": "probability_static_A"}
    )

    enhanced_oof = all_oof.loc[
        all_oof["scenario"] == enhanced_name,
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
            "oof_probability",
        ],
    ].copy()
    enhanced_oof = enhanced_oof.rename(
        columns={"oof_probability": "probability_enhanced_E"}
    )

    merged = static_oof.merge(
        enhanced_oof,
        on=Config.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=("_A", "_E"),
        validate="one_to_one",
    )

    expected_rows = Config.EXPECTED_N_ROWS
    if len(merged) != expected_rows:
        raise RuntimeError(
            f"A/E ensemble OOF merge returned {len(merged)} rows; "
            f"expected {expected_rows}."
        )

    for column_name in (
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ):
        left = merged[f"{column_name}_A"].to_numpy()
        right = merged[f"{column_name}_E"].to_numpy()
        if not np.array_equal(left, right):
            raise RuntimeError(
                f"A/E ensemble OOF mismatch in {column_name}."
            )

    merged[Config.TARGET_COLUMN] = merged[
        f"{Config.TARGET_COLUMN}_A"
    ].astype(int)
    merged[Config.FOLD_COLUMN] = merged[
        f"{Config.FOLD_COLUMN}_A"
    ].astype(int)
    merged[Config.BLOCK_COLUMN] = merged[
        f"{Config.BLOCK_COLUMN}_A"
    ].astype(int)

    merged["oof_probability_ensemble"] = (
        static_weight * merged["probability_static_A"].to_numpy(dtype=float)
        + enhanced_weight * merged["probability_enhanced_E"].to_numpy(dtype=float)
    )

    merged["oof_predicted_class_0p5"] = (
        merged["oof_probability_ensemble"].to_numpy(dtype=float)
        >= Config.BINARY_THRESHOLD
    ).astype(int)

    y_true = merged[Config.TARGET_COLUMN].to_numpy(dtype=int)
    y_prob = merged["oof_probability_ensemble"].to_numpy(dtype=float)
    blocks = merged[Config.BLOCK_COLUMN].to_numpy(dtype=int)

    metrics = classification_metrics(
        y_true,
        y_prob,
        threshold=Config.BINARY_THRESHOLD,
    )
    ci_low, ci_high, n_valid_bootstrap = cluster_bootstrap_auc_ci(
        y_true=y_true,
        y_prob=y_prob,
        blocks=blocks,
        n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
        seed=Config.RANDOM_STATE + 9001,
        ci_level=Config.CI_LEVEL,
    )

    # Paired comparison against the primary Model E OOF prediction.
    observed_difference, diff_low, diff_high, n_valid_diff = (
        paired_cluster_bootstrap_auc_difference(
            y_true=y_true,
            prob_left=y_prob,
            prob_right=merged["probability_enhanced_E"].to_numpy(dtype=float),
            blocks=blocks,
            n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
            seed=Config.RANDOM_STATE + 9002,
            ci_level=Config.CI_LEVEL,
        )
    )

    metrics_summary = {
        "ensemble_name": "RF A30-E70 fixed ensemble",
        "formula": (
            f"{static_weight:.2f} * Model A + "
            f"{enhanced_weight:.2f} * Model E"
        ),
        "weight_selection": (
            "Fixed a priori to mirror the previous static/integrated "
            "Random Forest ensemble; not optimized on OOF predictions."
        ),
        **metrics,
        "auc_cluster_bootstrap_ci_low": float(ci_low),
        "auc_cluster_bootstrap_ci_high": float(ci_high),
        "auc_cluster_bootstrap_valid_iterations": int(n_valid_bootstrap),
        "auc_difference_ensemble_minus_model_E": float(observed_difference),
        "auc_difference_ci_low": float(diff_low),
        "auc_difference_ci_high": float(diff_high),
        "auc_difference_valid_iterations": int(n_valid_diff),
    }

    output_columns = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
        "probability_static_A",
        "probability_enhanced_E",
        "oof_probability_ensemble",
        "oof_predicted_class_0p5",
    ]
    output_df = merged[output_columns].copy()

    output_df.to_csv(
        Config.ENSEMBLE_OOF_PREDICTIONS_CSV,
        index=False,
        float_format="%.10f",
    )
    with open(
        Config.ENSEMBLE_OOF_METRICS_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(metrics_summary),
            handle,
            indent=2,
        )

    log(
        "[ENSEMBLE-OOF] A30/E70 fixed ensemble: "
        f"AUC={metrics_summary['roc_auc']:.4f} "
        f"[{ci_low:.4f}, {ci_high:.4f}], "
        f"ΔAUC vs Model E={observed_difference:+.4f} "
        f"[{diff_low:+.4f}, {diff_high:+.4f}]"
    )

    if observed_difference < 0:
        log(
            "[ENSEMBLE-WARN] The fixed A30/E70 ensemble has lower pooled "
            "OOF AUC than Model E. The ensemble map can still be generated "
            "for comparison, but Model E should remain the primary scientific "
            "map unless there is another justified reason to prefer the ensemble."
        )

    return output_df, metrics_summary


# =============================================================================
# FINAL FULL-DATA MODEL FOR MAPPING / DEPLOYMENT
# =============================================================================
def tune_and_fit_final_model(
    df: pd.DataFrame,
    scenario_name: str,
    features: Sequence[str],
) -> Tuple[Pipeline, Dict[str, Any], pd.DataFrame]:
    """
    Tune on all 330 samples using the frozen five spatial folds, then refit on
    all samples. This model is for raster prediction only, not performance claims.
    """
    X = df[list(features)]
    y = df[Config.TARGET_COLUMN].to_numpy(dtype=int)

    pipeline = build_rf_pipeline(features)
    spatial_cv = make_full_data_predefined_split(df)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=Config.PARAM_GRID,
        scoring="roc_auc",
        cv=spatial_cv,
        refit=True,
        n_jobs=Config.GRIDSEARCH_N_JOBS,
        verbose=Config.GRIDSEARCH_VERBOSE,
        return_train_score=False,
        error_score="raise",
    )

    log(
        f"[FINAL-MODEL] {Config.MODEL_SHORT_LABELS[scenario_name]}: "
        "full-data spatial grid search for deployment model."
    )
    search.fit(X, y)

    model_path = Config.MODEL_DIR / f"{scenario_prefix(scenario_name)}_RF.joblib"
    joblib.dump(search.best_estimator_, model_path)

    info = {
        "best_mean_spatial_cv_auc_for_model_selection_only": float(
            search.best_score_
        ),
        "best_params": json_safe(search.best_params_),
        "model_path": str(model_path),
        "note": (
            "This full-data tuning score is not used as the reported validation "
            "performance; reported performance comes only from nested outer OOF predictions."
        ),
    }

    search_df = normalize_grid_results(
        search.cv_results_,
        scenario_name=scenario_name,
        search_stage="full_data_deployment_tuning",
        outer_fold=None,
    )

    return search.best_estimator_, info, search_df


# =============================================================================
# RASTER PREDICTION
# =============================================================================
def read_stack_band_map(stack_path: Path) -> Dict[str, int]:
    require_file(stack_path, "Step 6.1 master predictor stack")
    with rasterio.open(stack_path) as dataset:
        descriptions = list(dataset.descriptions)

    band_map: Dict[str, int] = {}
    for band_index, description in enumerate(descriptions, start=1):
        if description:
            band_map[str(description)] = band_index

    return band_map


def predict_scenario_raster(
    fitted_pipeline: Pipeline,
    scenario_name: str,
    features: Sequence[str],
) -> Path:
    """
    Predict one susceptibility raster on the common A–F valid domain so all six
    maps have an identical comparison footprint.
    """
    require_file(Config.MASTER_STACK_TIF, "Step 6.1 master predictor stack")
    require_file(Config.COMMON_AF_MASK_TIF, "common A–F valid mask")

    band_map = read_stack_band_map(Config.MASTER_STACK_TIF)
    missing_bands = [feature for feature in features if feature not in band_map]
    if missing_bands:
        raise RuntimeError(
            f"Master stack is missing bands required by {scenario_name}: {missing_bands}"
        )

    band_indexes = [band_map[feature] for feature in features]
    output_path = (
        Config.MAP_DIR
        / f"{scenario_prefix(scenario_name)}_RF_probability_0_1_common_AF_domain.tif"
    )

    with rasterio.open(Config.MASTER_STACK_TIF) as stack_ds, rasterio.open(
        Config.COMMON_AF_MASK_TIF
    ) as mask_ds:
        if (
            stack_ds.width != mask_ds.width
            or stack_ds.height != mask_ds.height
            or stack_ds.transform != mask_ds.transform
            or stack_ds.crs != mask_ds.crs
        ):
            raise RuntimeError("Master stack and common A–F mask are not aligned.")

        profile = stack_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(temporary_path, "w", **profile) as output_ds:
            for row_start in range(0, stack_ds.height, Config.RASTER_BLOCK_ROWS):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    stack_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=stack_ds.width,
                    height=block_height,
                )

                data = stack_ds.read(band_indexes, window=window, masked=True)
                common_mask = mask_ds.read(1, window=window, masked=True)

                output = np.full(
                    (block_height, stack_ds.width),
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                values = np.asarray(data.filled(np.nan), dtype="float64")
                valid = (
                    np.asarray(common_mask.filled(0)) > 0
                ) & np.all(np.isfinite(values), axis=0)

                if np.any(valid):
                    pixel_matrix = values[:, valid].T
                    pixel_df = pd.DataFrame(pixel_matrix, columns=list(features))
                    probabilities = fitted_pipeline.predict_proba(pixel_df)[:, 1]
                    output[valid] = probabilities.astype("float32")

                output_ds.write(output, 1, window=window)

            output_ds.set_band_description(
                1,
                f"{Config.MODEL_SHORT_LABELS[scenario_name]} RF susceptibility probability",
            )
            output_ds.update_tags(
                scenario=scenario_name,
                model="Random Forest",
                prediction_domain="common_A_to_F_valid_domain",
                validation_note=(
                    "Raster generated from final full-data deployment model; "
                    "validation metrics come from nested spatial OOF predictions."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(temporary_path, output_path)
    log(f"[MAP] Saved: {output_path}")
    return output_path






def predict_scenario_raster_on_own_feature_domain(
    fitted_pipeline: Pipeline,
    scenario_name: str,
    features: Sequence[str],
    output_path: Path,
) -> Path:
    """
    Predict a scenario wherever all predictors required by that scenario are
    finite in the study-domain master stack.

    Unlike the A-F comparison maps, this function does NOT restrict prediction
    to the common A-F mask. This is required to reproduce the earlier ensemble
    logic, where the static model supplies the broad susceptibility background
    and the enhanced model is blended only where its additional predictors exist.
    """
    require_file(
        Config.MASTER_STACK_TIF,
        "Step 6.1 master predictor stack",
    )

    band_map = read_stack_band_map(Config.MASTER_STACK_TIF)
    missing_bands = [
        feature
        for feature in features
        if feature not in band_map
    ]
    if missing_bands:
        raise RuntimeError(
            f"Master stack is missing bands required by "
            f"{scenario_name}: {missing_bands}"
        )

    band_indexes = [
        band_map[feature]
        for feature in features
    ]

    with rasterio.open(Config.MASTER_STACK_TIF) as stack_ds:
        profile = stack_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        valid_pixel_count = 0

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_ds:
            for row_start in range(
                0,
                stack_ds.height,
                Config.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    stack_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=stack_ds.width,
                    height=block_height,
                )

                data = stack_ds.read(
                    band_indexes,
                    window=window,
                    masked=True,
                )

                output = np.full(
                    (block_height, stack_ds.width),
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                values = np.asarray(
                    data.filled(np.nan),
                    dtype="float64",
                )
                valid = np.all(
                    np.isfinite(values),
                    axis=0,
                )

                if np.any(valid):
                    pixel_matrix = values[:, valid].T
                    pixel_df = pd.DataFrame(
                        pixel_matrix,
                        columns=list(features),
                    )
                    probabilities = (
                        fitted_pipeline.predict_proba(pixel_df)[:, 1]
                    )
                    output[valid] = probabilities.astype("float32")
                    valid_pixel_count += int(valid.sum())

                output_ds.write(
                    output,
                    1,
                    window=window,
                )

            output_ds.set_band_description(
                1,
                (
                    f"{Config.MODEL_SHORT_LABELS[scenario_name]} "
                    "RF susceptibility probability"
                ),
            )
            output_ds.update_tags(
                scenario=scenario_name,
                model="Random Forest",
                prediction_domain="scenario_specific_valid_feature_domain",
                validation_note=(
                    "Raster generated from the final full-data deployment model. "
                    "Reported validation metrics come from nested spatial OOF "
                    "predictions, not from this raster."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        f"[MAP-FULL-DOMAIN] {Config.MODEL_SHORT_LABELS[scenario_name]}: "
        f"{valid_pixel_count:,} valid pixels -> {output_path}"
    )

    return output_path


def blend_static_and_enhanced_probability_rasters(
    static_raster_path: Path,
    enhanced_raster_path: Path,
    output_path: Path,
    enhanced_weight: float,
) -> Path:
    """
    Apply the same fixed A30/E70 operational ensemble logic to the Random Forest A-F framework.

    Where both models are valid:
        ensemble = (1-w) * Model A + w * Model E

    Where only Model A is valid:
        retain Model A as the susceptibility background.

    Where only Model E is valid:
        retain Model E.

    Outside both valid domains:
        nodata.

    This is intentionally different from averaging all A-F configurations.
    Models B-F are ablation experiments and should not be indiscriminately
    averaged into the final susceptibility map.
    """
    if not 0.0 <= enhanced_weight <= 1.0:
        raise ValueError(
            "ENSEMBLE_ENHANCED_WEIGHT must lie between 0 and 1."
        )

    static_weight = 1.0 - enhanced_weight

    require_file(
        static_raster_path,
        "Model A full-domain probability raster",
    )
    require_file(
        enhanced_raster_path,
        "Model E full-domain probability raster",
    )

    with rasterio.open(static_raster_path) as static_ds, rasterio.open(
        enhanced_raster_path
    ) as enhanced_ds:
        aligned = (
            static_ds.width == enhanced_ds.width
            and static_ds.height == enhanced_ds.height
            and static_ds.transform == enhanced_ds.transform
            and static_ds.crs == enhanced_ds.crs
        )
        if not aligned:
            raise RuntimeError(
                "Model A and Model E full-domain rasters are not aligned."
            )

        profile = static_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        n_static_background = 0
        n_blended = 0
        n_enhanced_only = 0

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_ds:
            for row_start in range(
                0,
                static_ds.height,
                Config.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    static_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=static_ds.width,
                    height=block_height,
                )

                static_arr = static_ds.read(
                    1,
                    window=window,
                    masked=True,
                )
                enhanced_arr = enhanced_ds.read(
                    1,
                    window=window,
                    masked=True,
                )

                static_values = np.asarray(
                    static_arr.filled(np.nan),
                    dtype="float64",
                )
                enhanced_values = np.asarray(
                    enhanced_arr.filled(np.nan),
                    dtype="float64",
                )

                static_valid = np.isfinite(static_values)
                enhanced_valid = np.isfinite(enhanced_values)

                both = static_valid & enhanced_valid
                static_only = static_valid & ~enhanced_valid
                enhanced_only = enhanced_valid & ~static_valid

                output = np.full(
                    static_values.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                output[static_only] = static_values[
                    static_only
                ].astype("float32")

                output[enhanced_only] = enhanced_values[
                    enhanced_only
                ].astype("float32")

                output[both] = (
                    static_weight * static_values[both]
                    + enhanced_weight * enhanced_values[both]
                ).astype("float32")

                n_static_background += int(static_only.sum())
                n_blended += int(both.sum())
                n_enhanced_only += int(enhanced_only.sum())

                output_ds.write(
                    output,
                    1,
                    window=window,
                )

            output_ds.set_band_description(
                1,
                "Random Forest A30-E70 ensemble susceptibility probability",
            )
            output_ds.update_tags(
                model="Random Forest",
                ensemble="Model_A_static_plus_Model_E_physics_guided",
                static_weight=f"{static_weight:.6f}",
                enhanced_weight=f"{enhanced_weight:.6f}",
                blend_rule=(
                    "weighted blend where both are valid; "
                    "retain static background elsewhere"
                ),
                validation_note=(
                    "The fixed A/E ensemble is separately evaluated from nested "
                    "OOF probabilities. A-F ablation metrics remain the primary "
                    "model-comparison results."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[ENSEMBLE-MAP] Saved A30/E70 ensemble: "
        f"blended={n_blended:,}, "
        f"static-background={n_static_background:,}, "
        f"enhanced-only={n_enhanced_only:,} -> "
        f"{output_path}"
    )

    return output_path


def create_revised_ae_ensemble_probability_map(
    final_models: dict[str, Pipeline],
    scenarios: dict[str, dict[str, Any]],
) -> Path:
    """
    Generate the revised equivalent of the previous RF blended ensemble map.

    Model A supplies the broad static susceptibility background.
    Model E contributes the deformation-informed physics-guided prediction
    wherever HKFI and SNDR are available.
    """
    static_scenario = Config.ENSEMBLE_STATIC_SCENARIO
    enhanced_scenario = Config.ENSEMBLE_ENHANCED_SCENARIO

    for scenario_name in (
        static_scenario,
        enhanced_scenario,
    ):
        if scenario_name not in final_models:
            raise RuntimeError(
                f"Final deployment model unavailable for {scenario_name}."
            )
        if scenario_name not in scenarios:
            raise RuntimeError(
                f"Scenario definition unavailable for {scenario_name}."
            )

    predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=final_models[static_scenario],
        scenario_name=static_scenario,
        features=list(scenarios[static_scenario]["features"]),
        output_path=Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
    )

    predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=final_models[enhanced_scenario],
        scenario_name=enhanced_scenario,
        features=list(scenarios[enhanced_scenario]["features"]),
        output_path=Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
    )

    return blend_static_and_enhanced_probability_rasters(
        static_raster_path=Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
        enhanced_raster_path=Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
        output_path=Config.ENSEMBLE_PROBABILITY_TIF,
        enhanced_weight=float(Config.ENSEMBLE_ENHANCED_WEIGHT),
    )


# =============================================================================
# STEP 5.3-STYLE LANDSLIDE SUSCEPTIBILITY VISUALIZATION
# =============================================================================

LSM_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "xgb_landslide_susceptibility",
    [
        "#006400",
        "#32CD32",
        "#FFFF00",
        "#FFA500",
        "#FF4500",
        "#8B0000",
    ],
    N=256,
)



SUSCEPTIBILITY_CLASSES = {
    "Very Low": (0.0, 0.2),
    "Low": (0.2, 0.4),
    "Moderate": (0.4, 0.6),
    "High": (0.6, 0.8),
    "Very High": (0.8, 1.0),
}

SUSCEPTIBILITY_CLASS_COLORS = {
    "Very Low": "#006400",
    "Low": "#32CD32",
    "Moderate": "#FFFF00",
    "High": "#FFA500",
    "Very High": "#8B0000",
}


def add_old_style_susceptibility_legend(
    axis,
    include_road: bool = True,
) -> None:
    """Reproduce the earlier LSM legend layout."""
    legend_elements: list[Line2D] = []

    if include_road:
        legend_elements.append(
            Line2D(
                [0],
                [0],
                color="red",
                linewidth=1.5,
                label="Highway/Roads",
            )
        )

    for class_name in (
        "Very High",
        "High",
        "Moderate",
        "Low",
        "Very Low",
    ):
        legend_elements.append(
            Line2D(
                [0],
                [0],
                marker="s",
                color="white",
                markerfacecolor=(
                    SUSCEPTIBILITY_CLASS_COLORS[class_name]
                ),
                markersize=7,
                markeredgecolor="black",
                markeredgewidth=0.5,
                label=class_name,
            )
        )

    legend = axis.legend(
        handles=legend_elements,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.50), # # Changed here to move the Map Up
        fontsize=Config.LEGEND_FONTSIZE,
        title="Legend",
        title_fontsize=Config.LEGEND_TITLE_FONTSIZE,
        frameon=True,
        fancybox=True,
        shadow=True,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0,
    )
    legend.get_title().set_fontweight("bold")
    legend.set_zorder(2000)


def reproject_probability_raster_to_epsg4326(
    source_path: Path,
    output_path: Path,
) -> Path:
    """
    Reproject only for cartographic display, matching the earlier LSM workflow.

    The source analytical probability raster is never modified.
    Bilinear resampling is used only for display reprojection of the continuous
    probability surface.
    """
    require_file(
        source_path,
        "susceptibility probability raster for visualization",
    )

    with rasterio.open(source_path) as source:
        if source.crs is None:
            raise RuntimeError(
                f"Probability raster has no CRS: {source_path}"
            )

        if source.crs.to_epsg() == 4326:
            return source_path

        destination_crs = "EPSG:4326"
        destination_transform, width, height = (
            calculate_default_transform(
                source.crs,
                destination_crs,
                source.width,
                source.height,
                *source.bounds,
            )
        )

        profile = source.profile.copy()
        profile.update(
            crs=destination_crs,
            transform=destination_transform,
            width=width,
            height=height,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as destination:
            reproject(
                source=rasterio.band(source, 1),
                destination=rasterio.band(destination, 1),
                src_transform=source.transform,
                src_crs=source.crs,
                src_nodata=source.nodata,
                dst_transform=destination_transform,
                dst_crs=destination_crs,
                dst_nodata=Config.OUTPUT_NODATA,
                resampling=Resampling.bilinear,
            )

            destination.set_band_description(
                1,
                "Random Forest ensemble susceptibility probability — EPSG:4326 visualization copy",
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[VIZ-REPROJECT] Display raster reprojected to EPSG:4326: "
        f"{output_path}"
    )
    return output_path


def ensemble_visualization_output_paths() -> dict[str, Path]:
    stem = (
        Config.VISUALIZATION_DIR
        / "RF_Susceptibility_Map_Ensemble_A30_E70"
    )
    return {
        "map_png": Path(str(stem) + ".png"),
        "map_pdf": Path(str(stem) + ".pdf"),
        "map_svg": Path(str(stem) + ".svg"),
        "histogram_png": Path(
            str(Config.VISUALIZATION_DIR / "RF_Ensemble_Histogram_A30_E70")
            + ".png"
        ),
        "histogram_pdf": Path(
            str(Config.VISUALIZATION_DIR / "RF_Ensemble_Histogram_A30_E70")
            + ".pdf"
        ),
        "histogram_svg": Path(
            str(Config.VISUALIZATION_DIR / "RF_Ensemble_Histogram_A30_E70")
            + ".svg"
        ),
        "summary_json": (
            Config.VISUALIZATION_DIR
            / "RF_Ensemble_A30_E70_Visualization_Summary.json"
        ),
        "statistics_csv": (
            Config.VISUALIZATION_DIR
            / "RF_Ensemble_A30_E70_Statistics.csv"
        ),
        "area_percentage_png": (
            Config.VISUALIZATION_DIR
            / "RF_Ensemble_A30_E70_Area_Percentage.png"
        ),
        "area_percentage_pdf": (
            Config.VISUALIZATION_DIR
            / "RF_Ensemble_A30_E70_Area_Percentage.pdf"
        ),
        "area_percentage_svg": (
            Config.VISUALIZATION_DIR
            / "RF_Ensemble_A30_E70_Area_Percentage.svg"
        ),
        "area_percentage_csv": (
            Config.VISUALIZATION_DIR
            / "RF_Ensemble_A30_E70_Area_Percentage.csv"
        ),
        "support_map_png": (
            Config.VISUALIZATION_DIR
            / "RF_Ensemble_A30_E70_Information_Support_Map.png"
        ),
        "support_map_pdf": (
            Config.VISUALIZATION_DIR
            / "RF_Ensemble_A30_E70_Information_Support_Map.pdf"
        ),
        "support_map_svg": (
            Config.VISUALIZATION_DIR
            / "RF_Ensemble_A30_E70_Information_Support_Map.svg"
        ),
        "display_raster_epsg4326": (
            Config.VISUALIZATION_DIR
            / "RF_Ensemble_A30_E70_Susceptibility_EPSG4326.tif"
        ),
    }


def create_old_style_ensemble_lsm_map(
    probability_raster_path: Path,
    outputs: dict[str, Path],
) -> tuple[np.ndarray, dict[str, Any]]:
    """
    Draw the ensemble LSM using the earlier successful RF map approach.

    Key elements reproduced:
      - static susceptibility background retained across the broader study domain;
      - EPSG:4326 display raster;
      - exact raster extent (no added map buffer);
      - white opaque map background;
      - vector clip path to the study boundary;
      - thick black study boundary;
      - red highway;
      - north arrow;
      - checker scale bar;
      - susceptibility-class legend plus continuous colour bar.
    """
    display_raster_path = reproject_probability_raster_to_epsg4326(
        source_path=probability_raster_path,
        output_path=outputs["display_raster_epsg4326"],
    )

    probability, reference = read_probability_raster(
        display_raster_path
    )
    values = probability[np.isfinite(probability)]

    bounds = reference["bounds"]
    extent = (
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    )

    display_array = np.ma.masked_invalid(
        probability
    )

    # Opaque white for any nodata cells that remain.
    map_cmap = LSM_CMAP.copy()
    map_cmap.set_bad(
        color="white",
        alpha=1.0,
    )

    study_boundary = load_vector_overlay(
        Config.STUDY_SHP,
        "EPSG:4326",
    )
    roads = load_vector_overlay(
        Config.ROAD_SHP,
        "EPSG:4326",
    )

    figure, axis = plt.subplots(
        1,
        1,
        figsize=Config.ENSEMBLE_MAP_FIGSIZE,
        facecolor="white",
    )
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        vmin=0.0,
        vmax=1.0,
        cmap=map_cmap,
        interpolation="nearest",
        alpha=1.0,
        zorder=1,
    )

    # The raster is clipped instead of relying on alpha transparency.
    clip_patch = build_clip_patch_from_vector_overlay(
        study_boundary,
        axis,
    )
    if clip_patch is not None:
        image.set_clip_path(
            clip_patch
        )
        log(
            "[ENSEMBLE-VIZ] Raster clipped to study boundary; "
            "opaque white background retained."
        )

    # Match the visual hierarchy of the earlier map.
    plot_vector_overlay(
        axis,
        study_boundary,
        colour="black",
        linewidth=2.0,
        zorder=1000,
        polygon_boundary_only=True,
    )
    plot_vector_overlay(
        axis,
        roads,
        colour="red",
        linewidth=1.0,
        zorder=1100,
        polygon_boundary_only=False,
    )

    axis.set_xlim(
        extent[0],
        extent[1],
    )
    axis.set_ylim(
        extent[2],
        extent[3],
    )

    axis.xaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: to_dms(
                value,
                True,
            )
        )
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: to_dms(
                value,
                False,
            )
        )
    )
    axis.tick_params(
        axis="both",
        which="major",
        labelsize=Config.DMS_TICK_FONTSIZE,
    )

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")

    for label in axis.get_yticklabels():
        label.set_rotation(90)
        label.set_va("center")

    add_north_arrow_tri(
        axis,
        size=0.08,
        pad=0.06,
    )

    add_checker_scale_bar(
        axis,
        extent[0],
        extent[1],
        extent[2],
        extent[3],
        crs_type="wgs84",
        mid_lat_deg=(
            0.5 * (extent[2] + extent[3])
        ),
        segments=4,
        fontsize=6,
    )

    add_old_style_susceptibility_legend(
        axis,
        include_road=True,
    )

    divider = make_axes_locatable(
        axis
    )
    colorbar_axis = divider.append_axes(
        "right",
        size=Config.CAX_SIZE,
        pad=Config.CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(
        labelsize=Config.COLORBAR_TICK_FONTSIZE,
    )
    colorbar.set_label(
        "Susceptibility (0–1)",
        fontsize=Config.COLORBAR_LABEL_FONTSIZE,
        fontweight="bold",
    )

    figure.tight_layout()

    for output_key, output_format in (
        ("map_png", None),
        ("map_pdf", None),
        ("map_svg", "svg"),
    ):
        save_kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
            "edgecolor": "white",
            "transparent": False,
        }
        if output_key == "map_png":
            save_kwargs["dpi"] = Config.FIG_DPI
        if output_format is not None:
            save_kwargs["format"] = output_format

        figure.savefig(
            outputs[output_key],
            **save_kwargs,
        )

    log(
        f"[ENSEMBLE-VIZ] Map saved: {outputs['map_png']}"
    )

    if Config.SHOW_LSM_FIGURES:
        plt.show()

    plt.close(
        figure
    )

    return values, reference



def classify_susceptibility_probability(values: np.ndarray) -> np.ndarray:
    """Classify continuous probability into five equal-interval classes."""
    classes = np.zeros(values.shape, dtype=np.uint8)
    finite = np.isfinite(values)
    classes[finite & (values >= 0.0) & (values < 0.2)] = 1
    classes[finite & (values >= 0.2) & (values < 0.4)] = 2
    classes[finite & (values >= 0.4) & (values < 0.6)] = 3
    classes[finite & (values >= 0.6) & (values < 0.8)] = 4
    classes[finite & (values >= 0.8) & (values <= 1.0)] = 5
    return classes


def calculate_susceptibility_area_percentages(
    probability_raster_path: Path,
) -> pd.DataFrame:
    """
    Calculate mapped area percentage for each susceptibility class on the
    original projected analytical grid. Pixel fractions therefore equal area
    fractions because cell area is constant.
    """
    require_file(probability_raster_path, "ensemble susceptibility raster")

    with rasterio.open(probability_raster_path) as dataset:
        probability = clean_probability_array(
            dataset.read(1),
            dataset.nodata,
        )

    valid = probability[np.isfinite(probability)]
    classes = classify_susceptibility_probability(valid)
    total = int(valid.size)

    definitions = [
        (5, "Very High", 0.8, 1.0),
        (4, "High", 0.6, 0.8),
        (3, "Moderate", 0.4, 0.6),
        (2, "Low", 0.2, 0.4),
        (1, "Very Low", 0.0, 0.2),
    ]

    rows = []
    for class_id, label, lower, upper in definitions:
        count = int(np.sum(classes == class_id))
        percentage = 100.0 * count / total if total > 0 else np.nan
        rows.append(
            {
                "class_id": class_id,
                "class_label": label,
                "lower_probability": lower,
                "upper_probability": upper,
                "pixel_count": count,
                "area_percentage": percentage,
            }
        )

    return pd.DataFrame(rows)


def create_susceptibility_area_percentage_plot(
    area_df: pd.DataFrame,
    outputs: dict[str, Path],
) -> None:
    """Create a separate 4 x 4 bar plot of susceptibility-class area percentages."""
    labels = area_df["class_label"].tolist()
    percentages = area_df["area_percentage"].to_numpy(dtype=float)
    colours = [SUSCEPTIBILITY_CLASS_COLORS[label] for label in labels]

    figure, axis = plt.subplots(
        1,
        1,
        figsize=Config.AREA_PERCENT_FIGSIZE,
        facecolor="white",
    )
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    bars = axis.bar(
        labels,
        percentages,
        color=colours,
        edgecolor="black",
        linewidth=0.8,
    )

    upper_limit = max(5.0, float(np.nanmax(percentages)) * 1.18)
    axis.set_ylim(0.0, upper_limit)

    for bar, percentage in zip(bars, percentages):
        axis.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + upper_limit * 0.015,
            f"{percentage:.2f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
        )

    axis.set_xlabel("Susceptibility Class", fontsize=10, fontweight="bold")
    axis.set_ylabel("Area (%)", fontsize=10, fontweight="bold")
    axis.tick_params(axis="x", labelsize=8, rotation=20)
    axis.tick_params(axis="y", labelsize=8)
    axis.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.35)
    axis.set_axisbelow(True)

    for spine in axis.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("black")

    figure.tight_layout()

    for output_key, output_format in (
        ("area_percentage_png", None),
        ("area_percentage_pdf", None),
        ("area_percentage_svg", "svg"),
    ):
        kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
            "edgecolor": "white",
            "transparent": False,
        }
        if output_key == "area_percentage_png":
            kwargs["dpi"] = Config.FIG_DPI
        if output_format is not None:
            kwargs["format"] = output_format
        figure.savefig(outputs[output_key], **kwargs)

    if Config.SHOW_LSM_FIGURES:
        plt.show()
    plt.close(figure)


def write_information_support_raster(
    static_raster_path: Path,
    enhanced_raster_path: Path,
    output_path: Path,
) -> Path:
    """
    Write information-support classes:
      1 = Static-only prediction (Model A)
      2 = Static + deformation-informed prediction (Model E support present)
    """
    require_file(static_raster_path, "Model A full-domain probability raster")
    require_file(enhanced_raster_path, "Model E full-domain probability raster")

    with rasterio.open(static_raster_path) as static_ds, rasterio.open(enhanced_raster_path) as enhanced_ds:
        if not (
            static_ds.width == enhanced_ds.width
            and static_ds.height == enhanced_ds.height
            and static_ds.transform == enhanced_ds.transform
            and static_ds.crs == enhanced_ds.crs
        ):
            raise RuntimeError("Model A and Model E rasters are not aligned.")

        profile = static_ds.profile.copy()
        profile.update(driver="GTiff", count=1, dtype="uint8", nodata=0, compress="LZW", tiled=True)

        with rasterio.open(output_path, "w", **profile) as output_ds:
            for row_start in range(0, static_ds.height, Config.RASTER_BLOCK_ROWS):
                block_height = min(Config.RASTER_BLOCK_ROWS, static_ds.height - row_start)
                window = Window(0, row_start, static_ds.width, block_height)

                static_values = np.asarray(
                    static_ds.read(1, window=window, masked=True).filled(np.nan),
                    dtype=float,
                )
                enhanced_values = np.asarray(
                    enhanced_ds.read(1, window=window, masked=True).filled(np.nan),
                    dtype=float,
                )

                static_valid = np.isfinite(static_values)
                enhanced_valid = np.isfinite(enhanced_values)

                support = np.zeros(static_values.shape, dtype=np.uint8)
                support[static_valid & ~enhanced_valid] = 1
                support[enhanced_valid] = 2
                output_ds.write(support, 1, window=window)

            output_ds.set_band_description(1, "Random Forest A/E ensemble information support")
            output_ds.update_tags(
                class_1="Static-only prediction (Model A)",
                class_2="Static + deformation-informed prediction (Model E support present)",
                generated_at_utc=utc_now_iso(),
            )

    log(f"[SUPPORT-MAP] Saved: {output_path}")
    return output_path


def create_information_support_map(
    support_raster_path: Path,
    outputs: dict[str, Path],
) -> None:
    """Plot static-only versus deformation-informed support using the same map style."""
    display_path = Config.VISUALIZATION_DIR / "RF_Ensemble_A30_E70_Information_Support_EPSG4326.tif"

    with rasterio.open(support_raster_path) as source:
        if source.crs.to_epsg() == 4326:
            display_path = support_raster_path
        else:
            dst_crs = "EPSG:4326"
            dst_transform, width, height = calculate_default_transform(
                source.crs, dst_crs, source.width, source.height, *source.bounds
            )
            profile = source.profile.copy()
            profile.update(
                crs=dst_crs,
                transform=dst_transform,
                width=width,
                height=height,
                dtype="uint8",
                nodata=0,
                compress="LZW",
            )
            with rasterio.open(display_path, "w", **profile) as destination:
                reproject(
                    source=rasterio.band(source, 1),
                    destination=rasterio.band(destination, 1),
                    src_transform=source.transform,
                    src_crs=source.crs,
                    src_nodata=0,
                    dst_transform=dst_transform,
                    dst_crs=dst_crs,
                    dst_nodata=0,
                    resampling=Resampling.nearest,
                )

    with rasterio.open(display_path) as dataset:
        support = dataset.read(1)
        b = dataset.bounds
        extent = (b.left, b.right, b.bottom, b.top)

    display_array = np.ma.masked_where(support == 0, support)
    cmap = mcolors.ListedColormap(["#bdbdbd", "#2c7fb8"])
    cmap.set_bad(color="white", alpha=1.0)
    norm = mcolors.BoundaryNorm([0.5, 1.5, 2.5], cmap.N)

    study_boundary = load_vector_overlay(Config.STUDY_SHP, "EPSG:4326")
    roads = load_vector_overlay(Config.ROAD_SHP, "EPSG:4326")

    figure, axis = plt.subplots(1, 1, figsize=Config.SUPPORT_MAP_FIGSIZE, facecolor="white")
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
        alpha=1.0,
        zorder=1,
    )

    clip_patch = build_clip_patch_from_vector_overlay(study_boundary, axis)
    if clip_patch is not None:
        image.set_clip_path(clip_patch)

    plot_vector_overlay(axis, study_boundary, colour="black", linewidth=2.0, zorder=1000, polygon_boundary_only=True)
    plot_vector_overlay(axis, roads, colour="red", linewidth=1.0, zorder=1100, polygon_boundary_only=False)

    axis.set_xlim(extent[0], extent[1])
    axis.set_ylim(extent[2], extent[3])
    axis.xaxis.set_major_formatter(FuncFormatter(lambda value, position: to_dms(value, True)))
    axis.yaxis.set_major_formatter(FuncFormatter(lambda value, position: to_dms(value, False)))
    axis.tick_params(axis="both", which="major", labelsize=Config.DMS_TICK_FONTSIZE)

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")
    for label in axis.get_yticklabels():
        label.set_rotation(90)
        label.set_va("center")

    add_north_arrow_tri(axis, size=0.08, pad=0.06)
    add_checker_scale_bar(
        axis,
        extent[0], extent[1], extent[2], extent[3],
        crs_type="wgs84",
        mid_lat_deg=0.5 * (extent[2] + extent[3]),
        segments=4,
        fontsize=Config.SCALEBAR_FONTSIZE,
    )

    handles = [
        Line2D([0], [0], marker="s", color="white", markerfacecolor="#bdbdbd", markeredgecolor="black", markersize=8, label="Static-only prediction"),
        Line2D([0], [0], marker="s", color="white", markerfacecolor="#2c7fb8", markeredgecolor="black", markersize=8, label="Static + deformation-informed"),
        Line2D([0], [0], color="red", linewidth=1.5, label="Highway/Roads"),
    ]
    axis.legend(
        handles=handles,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.60), # Changed here to move the Map Up
        fontsize=Config.LEGEND_FONTSIZE,
        title="Information Support",
        title_fontsize=Config.LEGEND_TITLE_FONTSIZE,
        frameon=True,
        fancybox=True,
        shadow=True,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0,
    )

    figure.tight_layout()
    for output_key, output_format in (
        ("support_map_png", None),
        ("support_map_pdf", None),
        ("support_map_svg", "svg"),
    ):
        kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
            "edgecolor": "white",
            "transparent": False,
        }
        if output_key == "support_map_png":
            kwargs["dpi"] = Config.FIG_DPI
        if output_format is not None:
            kwargs["format"] = output_format
        figure.savefig(outputs[output_key], **kwargs)

    if Config.SHOW_LSM_FIGURES:
        plt.show()
    plt.close(figure)

def create_complete_ensemble_lsm_visualization(
    probability_raster_path: Path,
    ensemble_oof_metrics: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Create the old-style ensemble susceptibility map and histogram."""
    outputs = ensemble_visualization_output_paths()

    values, reference = create_old_style_ensemble_lsm_map(
        probability_raster_path=probability_raster_path,
        outputs=outputs,
    )

    statistics = calculate_lsm_statistics(
        values
    )

    # Five-class area-percentage analysis is calculated on the original
    # projected ensemble raster, not on the display-reprojected copy.
    area_percentage_df = calculate_susceptibility_area_percentages(
        probability_raster_path
    )
    area_percentage_df.to_csv(
        outputs["area_percentage_csv"],
        index=False,
        float_format="%.6f",
    )
    create_susceptibility_area_percentage_plot(
        area_df=area_percentage_df,
        outputs=outputs,
    )

    # Explicit information-support map for transparency.
    support_raster_path = write_information_support_raster(
        static_raster_path=Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
        enhanced_raster_path=Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
        output_path=Config.ENSEMBLE_SUPPORT_TIF,
    )
    create_information_support_map(
        support_raster_path=support_raster_path,
        outputs=outputs,
    )

    # Reuse the established histogram function with the ensemble values.
    create_lsm_probability_histogram(
        values=values,
        statistics=statistics,
        outputs=outputs,
    )

    pd.DataFrame(
        [
            {
                "map_source": "A30_E70_fixed_ensemble",
                "static_scenario": Config.ENSEMBLE_STATIC_SCENARIO,
                "enhanced_scenario": Config.ENSEMBLE_ENHANCED_SCENARIO,
                "static_weight": (
                    1.0 - Config.ENSEMBLE_ENHANCED_WEIGHT
                ),
                "enhanced_weight": Config.ENSEMBLE_ENHANCED_WEIGHT,
                **statistics,
            }
        ]
    ).to_csv(
        outputs["statistics_csv"],
        index=False,
    )

    summary = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "Random Forest",
        "map_source": "fixed_A30_E70_ensemble",
        "ensemble_formula": (
            f"{1.0 - Config.ENSEMBLE_ENHANCED_WEIGHT:.2f} * Model A + "
            f"{Config.ENSEMBLE_ENHANCED_WEIGHT:.2f} * Model E "
            "where Model E is available; otherwise retain Model A"
        ),
        "ensemble_weight_note": (
            "Weight fixed a priori to mirror the earlier context/integrated "
            "ensemble approach; not optimized on OOF predictions."
        ),
        "probability_raster": str(
            probability_raster_path
        ),
        "oof_ensemble_metrics": (
            ensemble_oof_metrics
            if ensemble_oof_metrics is not None
            else {}
        ),
        "visualization_style": (
            "Earlier RF ensemble LSM cartographic style with opaque white "
            "background, study-area vector clipping, enlarged DMS labels, "
            "five susceptibility classes and a separate class-area plot."
        ),
        "susceptibility_classes": area_percentage_df.to_dict(orient="records"),
        "information_support": {
            "class_1": "Static-only prediction (Model A)",
            "class_2": "Static + deformation-informed prediction (Model E support present)",
            "support_raster": str(support_raster_path),
        },
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }

    with open(
        outputs["summary_json"],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(summary),
            handle,
            indent=2,
        )

    return {
        "map_source": "fixed_A30_E70_ensemble",
        "probability_raster": str(
            probability_raster_path
        ),
        "statistics": statistics,
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }


def expected_scenario_map_path(scenario_name: str) -> Path:
    """Return the canonical raster path written by predict_scenario_raster()."""
    return (
        Config.MAP_DIR
        / f"{scenario_prefix(scenario_name)}_RF_probability_0_1_common_AF_domain.tif"
    )


def clean_probability_array(
    array: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    """
    Convert nodata to NaN and verify that susceptibility probabilities remain
    on [0,1]. Only tiny floating-point excursions are clipped.
    """
    output = np.asarray(array, dtype="float64").copy()

    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if np.isfinite(nodata_value):
                output[output == nodata_value] = np.nan
        except (TypeError, ValueError):
            pass

    output[output == -9999.0] = np.nan
    output[output == -32768.0] = np.nan
    output[np.abs(output) > 1.0e20] = np.nan

    outside = (
        np.isfinite(output)
        & ((output < -1.0e-6) | (output > 1.0 + 1.0e-6))
    )
    if outside.any():
        raise ValueError(
            f"Susceptibility raster contains {int(outside.sum()):,} values "
            "outside [0,1]. Recheck final probability prediction."
        )

    # Floating-point safeguard only; this is not renormalization.
    output = np.where(
        np.isfinite(output),
        np.clip(output, 0.0, 1.0),
        np.nan,
    )
    return output


def read_probability_raster(
    raster_path: Path,
) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(raster_path, "Random Forest susceptibility probability raster")

    with rasterio.open(raster_path) as dataset:
        if dataset.crs is None:
            raise ValueError(
                f"Susceptibility raster has no CRS: {raster_path}"
            )

        probability = clean_probability_array(
            dataset.read(1),
            dataset.nodata,
        )

        reference = {
            "crs": dataset.crs,
            "transform": dataset.transform,
            "width": dataset.width,
            "height": dataset.height,
            "bounds": dataset.bounds,
            "epsg": dataset.crs.to_epsg(),
            "path": str(raster_path),
        }

    if not np.any(np.isfinite(probability)):
        raise RuntimeError(
            f"No valid probabilities found in susceptibility raster: {raster_path}"
        )

    return probability, reference


def to_dms(value: float, is_lon: bool = True) -> str:
    absolute = abs(value)
    degrees = int(absolute)
    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_lon
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def add_north_arrow_tri(
    axis,
    size: float = 0.09,
    pad: float = 0.07,
    label: str = "N",
    arrow_downshift: float = 0.025,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=Config.NORTH_ARROW_FONTSIZE,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base
        if candidate >= target_metres:
            return candidate

    return 10 * base


def add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    crs_type: str = "utm",
    mid_lat_deg: float | None = None,
    segments: int = 4,
    fontsize: float = 4,
    frame_lw: float = 0.4,
) -> None:
    width_data = east - west
    height_data = north - south

    if crs_type == "utm":
        width_metres = width_data
        data_units_per_metre = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = 0.5 * (south + north)

        metres_per_degree = metres_per_degree_lon(mid_lat_deg)
        width_metres = width_data * metres_per_degree
        data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = total_length_metres * data_units_per_metre
    segment_length_units = segment_length_metres * data_units_per_metre

    x_start = west + 0.5 * width_data - 0.5 * total_length_units
    y_base = south + 0.06 * height_data
    bar_height = 0.018 * height_data

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_data

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (
                    x_start + index * segment_length_units,
                    y_base,
                ),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=(
                    "black" if index % 2 == 0 else "white"
                ),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_data,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_data,
        "0",
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_data,
        end_label,
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )


def load_vector_overlay(
    vector_path: Path,
    target_crs,
):
    """Load vector data and transform it to the raster CRS."""
    if not vector_path.exists():
        log(f"[VIZ-WARN] Vector overlay not found: {vector_path}")
        return None

    target_crs_text = str(target_crs)

    if HAVE_GPD:
        try:
            data = gpd.read_file(vector_path)
            if data.crs is None:
                log(
                    f"[VIZ-WARN] {vector_path.name} has no CRS; "
                    "assuming EPSG:4326."
                )
                data = data.set_crs(epsg=4326)
            return data.to_crs(target_crs_text)
        except Exception as exc:
            log(
                f"[VIZ-WARN] GeoPandas loading failed for "
                f"{vector_path.name}: {exc}"
            )

    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(vector_path, "r") as source:
                source_crs = source.crs_wkt or source.crs
                if not source_crs:
                    source_crs = "EPSG:4326"
                    log(
                        f"[VIZ-WARN] {vector_path.name} has no CRS; "
                        "assuming EPSG:4326."
                    )

                for feature in source:
                    geometry = feature.get("geometry")
                    if geometry is None:
                        continue
                    geometries.append(
                        transform_geom(
                            source_crs,
                            target_crs_text,
                            geometry,
                        )
                    )

            return {"geometries": geometries}

        except Exception as exc:
            log(
                f"[VIZ-WARN] Fiona loading failed for "
                f"{vector_path.name}: {exc}"
            )

    return None


def plot_vector_overlay(
    axis,
    vector_data,
    colour: str,
    linewidth: float,
    zorder: int,
    polygon_boundary_only: bool = False,
) -> None:
    if vector_data is None:
        return

    if HAVE_GPD and hasattr(vector_data, "plot"):
        try:
            if polygon_boundary_only:
                vector_data.boundary.plot(
                    ax=axis,
                    color=colour,
                    linewidth=linewidth,
                    zorder=zorder,
                )
            else:
                vector_data.plot(
                    ax=axis,
                    color=colour,
                    linewidth=linewidth,
                    zorder=zorder,
                )
            return
        except Exception as exc:
            log(f"[VIZ-WARN] GeoPandas plotting failed: {exc}")

    if not isinstance(vector_data, dict):
        return

    for geometry in vector_data.get("geometries", []):
        if geometry is None:
            continue

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")
        if coordinates is None:
            continue

        if geometry_type == "LineString":
            line_groups = [coordinates]
        elif geometry_type == "MultiLineString":
            line_groups = coordinates
        elif geometry_type == "Polygon":
            line_groups = [coordinates[0]]
        elif geometry_type == "MultiPolygon":
            line_groups = [
                polygon[0]
                for polygon in coordinates
                if polygon
            ]
        else:
            continue

        for line in line_groups:
            xy = np.asarray(line, dtype="float64")
            if xy.ndim != 2 or xy.shape[1] < 2:
                continue
            axis.plot(
                xy[:, 0],
                xy[:, 1],
                color=colour,
                linewidth=linewidth,
                zorder=zorder,
            )



def build_clip_patch_from_vector_overlay(
    vector_data,
    axis,
):
    """
    Build a compound Matplotlib PathPatch from the study-area polygon.

    The raster artist is clipped to this polygon rather than relying on
    transparent pixels outside the study area. The axes and figure backgrounds
    remain opaque white, which is safer for PNG/PDF/SVG and later SVG-to-EMF
    conversion in Word.

    Supports:
      1. GeoPandas GeoDataFrame / GeoSeries input.
      2. Fiona fallback GeoJSON-style geometry dictionaries.

    Returns
    -------
    PathPatch or None
        Invisible patch suitable for image.set_clip_path().
    """
    if vector_data is None:
        return None

    all_vertices: list[np.ndarray] = []
    all_codes: list[int] = []

    def add_ring(ring_coordinates) -> None:
        coordinates = np.asarray(ring_coordinates, dtype="float64")

        if (
            coordinates.ndim != 2
            or coordinates.shape[0] < 3
            or coordinates.shape[1] < 2
        ):
            return

        coordinates = coordinates[:, :2]

        # Ensure the ring is explicitly closed.
        if not np.allclose(
            coordinates[0],
            coordinates[-1],
            equal_nan=False,
        ):
            coordinates = np.vstack(
                [coordinates, coordinates[0]]
            )

        n_vertices = len(coordinates)
        codes = (
            [MplPath.MOVETO]
            + [MplPath.LINETO] * (n_vertices - 2)
            + [MplPath.CLOSEPOLY]
        )

        all_vertices.append(coordinates)
        all_codes.extend(codes)

    def add_shapely_geometry(geometry) -> None:
        if geometry is None or getattr(geometry, "is_empty", True):
            return

        geometry_type = geometry.geom_type

        if geometry_type == "Polygon":
            add_ring(geometry.exterior.coords)
            for interior in geometry.interiors:
                add_ring(interior.coords)

        elif geometry_type == "MultiPolygon":
            for polygon in geometry.geoms:
                add_shapely_geometry(polygon)

        elif geometry_type == "GeometryCollection":
            for sub_geometry in geometry.geoms:
                add_shapely_geometry(sub_geometry)

    def add_geojson_geometry(geometry: dict[str, Any]) -> None:
        if not geometry:
            return

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")

        if coordinates is None:
            return

        if geometry_type == "Polygon":
            for ring in coordinates:
                add_ring(ring)

        elif geometry_type == "MultiPolygon":
            for polygon in coordinates:
                for ring in polygon:
                    add_ring(ring)

        elif geometry_type == "GeometryCollection":
            for sub_geometry in geometry.get("geometries", []):
                add_geojson_geometry(sub_geometry)

    # GeoPandas path.
    if HAVE_GPD and hasattr(vector_data, "geometry"):
        try:
            for geometry in vector_data.geometry:
                add_shapely_geometry(geometry)
        except Exception as exc:
            log(
                "[VIZ-WARN] Could not build GeoPandas clip geometry: "
                f"{exc}"
            )

    # Fiona / GeoJSON fallback path.
    elif isinstance(vector_data, dict):
        for geometry in vector_data.get("geometries", []):
            add_geojson_geometry(geometry)

    if not all_vertices:
        return None

    compound_path = MplPath(
        np.concatenate(all_vertices, axis=0),
        all_codes,
    )

    clip_patch = PathPatch(
        compound_path,
        facecolor="none",
        edgecolor="none",
        linewidth=0.0,
        transform=axis.transData,
        zorder=0,
    )

    # Adding the invisible patch initializes the axes data transform correctly.
    axis.add_patch(clip_patch)

    return clip_patch


def calculate_lsm_statistics(values: np.ndarray) -> dict[str, float | int]:
    return {
        "valid_pixels": int(values.size),
        "minimum": float(np.min(values)),
        "maximum": float(np.max(values)),
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "standard_deviation": float(np.std(values)),
        "p05": float(np.percentile(values, 5.0)),
        "p25": float(np.percentile(values, 25.0)),
        "p75": float(np.percentile(values, 75.0)),
        "p95": float(np.percentile(values, 95.0)),
    }


def lsm_visualization_output_paths(
    scenario_name: str,
) -> dict[str, Path]:
    prefix = scenario_prefix(scenario_name)
    stem = Config.VISUALIZATION_DIR / f"{prefix}_RF_Landslide_Susceptibility"

    return {
        "map_png": Path(str(stem) + "_Map.png"),
        "map_pdf": Path(str(stem) + "_Map.pdf"),
        "map_svg": Path(str(stem) + "_Map.svg"),
        "histogram_png": Path(str(stem) + "_Histogram.png"),
        "histogram_pdf": Path(str(stem) + "_Histogram.pdf"),
        "histogram_svg": Path(str(stem) + "_Histogram.svg"),
        "summary_json": Path(str(stem) + "_Visualization_Summary.json"),
        "statistics_csv": Path(str(stem) + "_Statistics.csv"),
    }


def create_lsm_probability_map(
    probability: np.ndarray,
    reference: dict[str, Any],
    scenario_name: str,
    outputs: dict[str, Path],
) -> None:
    bounds = reference["bounds"]

    raster_extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    buffer_x = (
        bounds.right - bounds.left
    ) * Config.DISPLAY_BUFFER_FRACTION
    buffer_y = (
        bounds.top - bounds.bottom
    ) * Config.DISPLAY_BUFFER_FRACTION

    west = bounds.left - buffer_x
    east = bounds.right + buffer_x
    south = bounds.bottom - buffer_y
    north = bounds.top + buffer_y

    display_array = np.ma.masked_invalid(probability)

    # Use an opaque white colour for masked / nodata cells.
    # This avoids transparent raster pixels in exported figures.
    map_cmap = LSM_CMAP.copy()
    map_cmap.set_bad(
        color="white",
        alpha=1.0,
    )

    raster_crs = reference["crs"]
    crs_is_geographic = bool(
        raster_crs and raster_crs.is_geographic
    )

    x_mid = 0.5 * (bounds.left + bounds.right)
    y_mid = 0.5 * (bounds.bottom + bounds.top)

    if crs_is_geographic:
        def format_longitude(value, position):
            return to_dms(value, True)

        def format_latitude(value, position):
            return to_dms(value, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError(
                "Projected susceptibility raster requires pyproj "
                "for DMS axis labels."
            )

        to_geographic = Transformer.from_crs(
            raster_crs,
            "EPSG:4326",
            always_xy=True,
        )

        def format_longitude(value, position):
            longitude, _ = to_geographic.transform(
                value,
                y_mid,
            )
            return to_dms(longitude, True)

        def format_latitude(value, position):
            _, latitude = to_geographic.transform(
                x_mid,
                value,
            )
            return to_dms(latitude, False)

    roads = load_vector_overlay(
        Config.ROAD_SHP,
        raster_crs,
    )
    study_boundary = load_vector_overlay(
        Config.STUDY_SHP,
        raster_crs,
    )

    style = {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "axes.unicode_minus": False,
    }

    with plt.rc_context(style):
        figure, axis = plt.subplots(
            1,
            1,
            figsize=Config.MAP_FIGSIZE,
            facecolor="white",
        )
        axis.set_facecolor("white")

        # Keep both the full figure canvas and axes background explicitly
        # opaque white. This is important for Word/EMF workflows.
        figure.patch.set_facecolor("white")
        figure.patch.set_alpha(1.0)
        axis.set_facecolor("white")

        image = axis.imshow(
            display_array,
            cmap=map_cmap,
            extent=raster_extent,
            vmin=0.0,
            vmax=1.0,
            origin="upper",
            interpolation=Config.RASTER_DISPLAY_INTERPOLATION,
            alpha=1.0,
            zorder=1,
        )

        # EMF-safe background handling used in the earlier LSM visualization:
        # clip the raster artist to the study-area polygon instead of leaving
        # transparent raster cells outside the boundary. The opaque white axes
        # background therefore shows through outside the study area.
        clip_patch = build_clip_patch_from_vector_overlay(
            study_boundary,
            axis,
        )
        if clip_patch is not None:
            image.set_clip_path(clip_patch)
            log(
                "[VIZ-MAP] Raster clipped to study boundary; "
                "opaque white background retained."
            )
        else:
            log(
                "[VIZ-WARN] Study-area clip path could not be created. "
                "Masked cells will still render as opaque white."
            )

        axis.set_xlim(west, east)
        axis.set_ylim(south, north)

        # Established Step 5.3 overlays.
        plot_vector_overlay(
            axis,
            study_boundary,
            colour="black",
            linewidth=0.7,
            zorder=800,
            polygon_boundary_only=True,
        )
        plot_vector_overlay(
            axis,
            roads,
            colour="red",
            linewidth=0.7,
            zorder=850,
            polygon_boundary_only=False,
        )

        axis.xaxis.set_major_locator(
            LinearLocator(Config.LONGITUDE_DMS_TICK_COUNT)
        )
        axis.yaxis.set_major_locator(
            LinearLocator(Config.LONGITUDE_DMS_TICK_COUNT)
        )
        axis.xaxis.set_major_formatter(
            FuncFormatter(format_longitude)
        )
        axis.yaxis.set_major_formatter(
            FuncFormatter(format_latitude)
        )
        axis.tick_params(
            axis="both",
            which="major",
            labelsize=5,
        )

        for tick_label in axis.get_xticklabels():
            tick_label.set_rotation(45)
            tick_label.set_ha("right")
            tick_label.set_fontsize(5)

        for tick_label in axis.get_yticklabels():
            tick_label.set_fontsize(5)

        add_north_arrow_tri(
            axis,
            size=0.09,
            pad=0.09,
        )

        crs_type = (
            "wgs84" if crs_is_geographic else "utm"
        )

        add_checker_scale_bar(
            axis,
            west,
            east,
            south,
            north,
            crs_type=crs_type,
            mid_lat_deg=(
                y_mid if crs_is_geographic else None
            ),
            segments=4,
            fontsize=4,
        )

        legend_elements = [
            Line2D(
                [0],
                [0],
                color="red",
                linewidth=1.0,
                label="Highway",
            ),
            Line2D(
                [0],
                [0],
                color="black",
                linewidth=1.0,
                label="Study Area",
            ),
        ]

        legend = axis.legend(
            handles=legend_elements,
            loc="center",
            bbox_to_anchor=(0.82, 0.35),
            bbox_transform=axis.transAxes,
            frameon=True,
            framealpha=0.9,
            fontsize=2.5,
            labelspacing=0.3,
            borderpad=0.3,
            handlelength=1.2,
            handletextpad=0.4,
            borderaxespad=0.0,
        )
        legend.get_frame().set_edgecolor("lightgray")
        legend.get_frame().set_linewidth(0.3)

        divider = make_axes_locatable(axis)
        colorbar_axis = divider.append_axes(
            "right",
            size=Config.CAX_SIZE,
            pad=Config.CAX_PAD,
        )

        colorbar = figure.colorbar(
            image,
            cax=colorbar_axis,
        )
        colorbar.ax.tick_params(labelsize=5)
        colorbar.set_label(
            "Landslide susceptibility probability (0–1)",
            fontsize=4,
            fontweight="bold",
        )
        colorbar.set_ticks(
            np.linspace(0.0, 1.0, 5)
        )

        figure.tight_layout()

        if Config.SAVE_LSM_FIGURES:
            figure.savefig(
                outputs["map_png"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["map_pdf"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["map_svg"],
                format="svg",
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )

            log(f"[VIZ-MAP] {outputs['map_png']}")
            log(f"[VIZ-MAP] {outputs['map_pdf']}")
            log(f"[VIZ-MAP] {outputs['map_svg']}")

        if Config.SHOW_LSM_FIGURES:
            # Use Matplotlib's standard display path. This is more reliable
            # across Jupyter, Spyder, VS Code, and standard Python sessions.
            plt.show()

        plt.close(figure)


def create_lsm_probability_histogram(
    values: np.ndarray,
    statistics: dict[str, float | int],
    outputs: dict[str, Path],
) -> None:
    style = {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "axes.unicode_minus": False,
    }

    with plt.rc_context(style):
        figure, axis = plt.subplots(
            1,
            1,
            figsize=Config.HIST_FIGSIZE,
            facecolor="white",
        )

        bins = np.linspace(0.0, 1.0, 21)
        counts, edges = np.histogram(values, bins=bins)
        centres = 0.5 * (edges[:-1] + edges[1:])
        colours = LSM_CMAP(centres)

        axis.bar(
            centres,
            counts,
            width=(bins[1] - bins[0]) * 0.9,
            color=colours,
            edgecolor="black",
            linewidth=0.4,
            alpha=0.9,
        )

        axis.axvline(
            statistics["mean"],
            color="darkred",
            linestyle="--",
            linewidth=0.7,
            label=f"Mean: {statistics['mean']:.3f}",
        )
        axis.axvline(
            statistics["median"],
            color="orange",
            linestyle="--",
            linewidth=0.7,
            label=f"Median: {statistics['median']:.3f}",
        )
        axis.axvline(
            statistics["p95"],
            color="green",
            linestyle="--",
            linewidth=0.7,
            label=f"95th %ile: {statistics['p95']:.3f}",
        )

        axis.legend(
            fontsize=3,
            loc="upper right",
            framealpha=0.9,
        )
        axis.set_xlabel(
            "Landslide susceptibility probability (0–1)",
            fontsize=4,
            fontweight="bold",
        )
        axis.set_ylabel(
            "Number of Pixels",
            fontsize=4,
            fontweight="bold",
        )
        axis.tick_params(
            axis="both",
            which="major",
            labelsize=4,
        )
        axis.set_xlim(0.0, 1.0)
        axis.grid(
            True,
            alpha=0.25,
            linestyle="-",
            linewidth=0.4,
            axis="y",
        )
        axis.set_axisbelow(True)

        figure.tight_layout(pad=1.2)

        if Config.SAVE_LSM_FIGURES:
            figure.savefig(
                outputs["histogram_png"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["histogram_pdf"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["histogram_svg"],
                format="svg",
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )

            log(f"[VIZ-HIST] {outputs['histogram_png']}")
            log(f"[VIZ-HIST] {outputs['histogram_pdf']}")
            log(f"[VIZ-HIST] {outputs['histogram_svg']}")

        if Config.SHOW_LSM_FIGURES:
            # Use Matplotlib's standard display path. This is more reliable
            # across Jupyter, Spyder, VS Code, and standard Python sessions.
            plt.show()

        plt.close(figure)


def create_primary_lsm_visualization(
    raster_path: Path,
    scenario_name: str,
) -> dict[str, Any]:
    """
    Create the primary Model E susceptibility map and histogram using the
    established Step 5.3 cartographic style.
    """
    probability, reference = read_probability_raster(
        raster_path
    )

    values = probability[np.isfinite(probability)]
    statistics = calculate_lsm_statistics(values)
    outputs = lsm_visualization_output_paths(
        scenario_name
    )

    create_lsm_probability_map(
        probability=probability,
        reference=reference,
        scenario_name=scenario_name,
        outputs=outputs,
    )

    create_lsm_probability_histogram(
        values=values,
        statistics=statistics,
        outputs=outputs,
    )

    statistics_df = pd.DataFrame(
        [
            {
                "scenario": scenario_name,
                "model_label": Config.MODEL_SHORT_LABELS[scenario_name],
                "model_description": Config.MODEL_DESCRIPTIONS[scenario_name],
                **statistics,
            }
        ]
    )
    statistics_df.to_csv(
        outputs["statistics_csv"],
        index=False,
    )

    summary = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "Random Forest",
        "scenario": scenario_name,
        "model_label": Config.MODEL_SHORT_LABELS[scenario_name],
        "model_description": Config.MODEL_DESCRIPTIONS[scenario_name],
        "probability_raster": str(raster_path),
        "prediction_domain": "common_A_to_F_valid_domain",
        "validation_note": (
            "This figure is generated from the final full-data deployment model. "
            "Model performance must be reported from nested spatial OOF predictions, "
            "not from the full-data map."
        ),
        "scientific_plotting": {
            "direct_probability_raster": True,
            "probabilities_renormalized_for_visualization": False,
            "RBF_used": False,
            "IDW_used": False,
            "cubic_interpolation_used": False,
            "display_interpolation": Config.RASTER_DISPLAY_INTERPOLATION,
        },
        "statistics": statistics,
        "visual_style": {
            "reference_style": "Step 5.3 HKFI visualization",
            "map_figsize_inches": list(Config.MAP_FIGSIZE),
            "histogram_figsize_inches": list(Config.HIST_FIGSIZE),
            "display_buffer_fraction": Config.DISPLAY_BUFFER_FRACTION,
            "background": "opaque_white",
            "transparent_export": False,
            "study_area_vector_clip": True,
            "masked_raster_colour": "opaque_white",
            "DMS_axes": True,
            "north_arrow": True,
            "checkerboard_scale_bar": True,
            "highway_overlay": True,
            "study_area_overlay": True,
            "compact_legend": True,
            "side_colour_bar": True,
            "colour_ramp": "green-yellow-orange-red susceptibility ramp",
        },
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }

    with open(
        outputs["summary_json"],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(summary),
            handle,
            indent=2,
        )

    log(
        "[VIZ] Primary Random Forest landslide susceptibility visualization "
        f"completed for {Config.MODEL_SHORT_LABELS[scenario_name]}."
    )

    # This is the direct Model E visualization on the common A-F domain.
    # Area-percentage and information-support products belong to the separate
    # A30/E70 ensemble visualization workflow and are therefore not referenced
    # here. Keeping the return object limited to values created inside this
    # function prevents accidental cross-scope NameError failures.
    return {
        "scenario": scenario_name,
        "raster_path": str(raster_path),
        "statistics": statistics,
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }


# =============================================================================
# SUMMARIES / COMPARISONS / FIGURES
# =============================================================================
def summarize_oof_results(
    all_oof: pd.DataFrame,
    all_fold_metrics: pd.DataFrame,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for scenario_name in Config.SCENARIO_ORDER:
        subset = all_oof.loc[all_oof["scenario"] == scenario_name].copy()
        fold_subset = all_fold_metrics.loc[
            all_fold_metrics["scenario"] == scenario_name
        ].copy()

        y = subset[Config.TARGET_COLUMN].to_numpy(dtype=int)
        probability = subset["oof_probability"].to_numpy(dtype=float)
        blocks = subset[Config.FOLD_COLUMN].to_numpy(dtype=int)

        pooled = classification_metrics(y, probability)
        ci_low, ci_high, n_valid_boot = cluster_bootstrap_auc_ci(
            y_true=y,
            y_prob=probability,
            blocks=blocks,
            n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
            seed=Config.RANDOM_STATE + Config.SCENARIO_ORDER.index(scenario_name),
            ci_level=Config.CI_LEVEL,
        )

        row = {
            "scenario": scenario_name,
            "model": Config.MODEL_SHORT_LABELS[scenario_name],
            "description": Config.MODEL_DESCRIPTIONS[scenario_name],
            "n_samples": int(len(subset)),
            "n_predictors": int(fold_subset["n_predictors"].iloc[0]),
            "oof_roc_auc": pooled["roc_auc"],
            "oof_auc_cluster_bootstrap_ci_low": ci_low,
            "oof_auc_cluster_bootstrap_ci_high": ci_high,
            "cluster_bootstrap_valid_iterations": int(n_valid_boot),
            "oof_pr_auc": pooled["pr_auc"],
            "oof_brier": pooled["brier"],
            "oof_accuracy_0p5": pooled["accuracy"],
            "oof_balanced_accuracy_0p5": pooled["balanced_accuracy"],
            "oof_precision_0p5": pooled["precision"],
            "oof_recall_0p5": pooled["recall"],
            "oof_f1_0p5": pooled["f1"],
            "fold_auc_mean": float(fold_subset["roc_auc"].mean()),
            "fold_auc_std": float(fold_subset["roc_auc"].std(ddof=1)),
            "fold_auc_min": float(fold_subset["roc_auc"].min()),
            "fold_auc_max": float(fold_subset["roc_auc"].max()),
        }
        rows.append(row)

    summary = pd.DataFrame(rows)

    baseline_auc = float(
        summary.loc[
            summary["scenario"] == "A_static_baseline",
            "oof_roc_auc",
        ].iloc[0]
    )

    summary["absolute_auc_gain_vs_A"] = summary["oof_roc_auc"] - baseline_auc
    summary["relative_auc_gain_vs_A_percent"] = np.where(
        baseline_auc != 0,
        100.0 * summary["absolute_auc_gain_vs_A"] / baseline_auc,
        np.nan,
    )

    return summary


def make_paired_auc_comparisons(all_oof: pd.DataFrame) -> pd.DataFrame:
    comparisons = [
        ("B_raw_deformation", "A_static_baseline", "B - A", "Raw DV+ added value"),
        ("C_hkfi_only", "A_static_baseline", "C - A", "HKFI added value"),
        ("D_sndr_only", "A_static_baseline", "D - A", "SNDR added value"),
        ("E_physics_guided", "A_static_baseline", "E - A", "Proposed indices total added value"),
        ("F_untransformed_sources", "A_static_baseline", "F - A", "Separate source variables added value"),
        ("E_physics_guided", "B_raw_deformation", "E - B", "Physics-guided versus raw DV+"),
        ("E_physics_guided", "F_untransformed_sources", "E - F", "Indices versus separate source variables"),
    ]

    pivot = all_oof.pivot(
        index=Config.SAMPLE_ID_COLUMN,
        columns="scenario",
        values="oof_probability",
    )

    metadata = (
        all_oof.loc[
            all_oof["scenario"] == "A_static_baseline",
            [Config.SAMPLE_ID_COLUMN, Config.TARGET_COLUMN, Config.FOLD_COLUMN],
        ]
        .drop_duplicates(Config.SAMPLE_ID_COLUMN)
        .set_index(Config.SAMPLE_ID_COLUMN)
        .loc[pivot.index]
    )

    y = metadata[Config.TARGET_COLUMN].to_numpy(dtype=int)
    blocks = metadata[Config.FOLD_COLUMN].to_numpy(dtype=int)

    rows: List[Dict[str, Any]] = []
    for comparison_index, (left, right, label, purpose) in enumerate(comparisons):
        observed, ci_low, ci_high, n_valid = paired_cluster_bootstrap_auc_difference(
            y_true=y,
            prob_left=pivot[left].to_numpy(dtype=float),
            prob_right=pivot[right].to_numpy(dtype=float),
            blocks=blocks,
            n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
            seed=Config.RANDOM_STATE + 100 + comparison_index,
            ci_level=Config.CI_LEVEL,
        )

        rows.append(
            {
                "comparison": label,
                "purpose": purpose,
                "left_scenario": left,
                "right_scenario": right,
                "auc_difference_left_minus_right": observed,
                "cluster_bootstrap_ci_low": ci_low,
                "cluster_bootstrap_ci_high": ci_high,
                "cluster_bootstrap_valid_iterations": int(n_valid),
            }
        )

    return pd.DataFrame(rows)


def plot_oof_roc_curves(all_oof: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(5.0, 5.0))

    for scenario_name in Config.SCENARIO_ORDER:
        subset = all_oof.loc[all_oof["scenario"] == scenario_name]
        y = subset[Config.TARGET_COLUMN].to_numpy(dtype=int)
        probability = subset["oof_probability"].to_numpy(dtype=float)
        fpr, tpr, _ = roc_curve(y, probability)
        auc = roc_auc_score(y, probability)
        label = f"{Config.MODEL_SHORT_LABELS[scenario_name]} (AUC={auc:.3f})"
        ax.plot(fpr, tpr, linewidth=1.4, label=label)

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.0)
    ax.set_xlabel(
        "False Positive Rate",
        fontsize=STAT_AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )
    ax.set_ylabel(
        "True Positive Rate",
        fontsize=STAT_AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

    # Slightly larger legend text than the original figure.
    ax.legend(
        frameon=False,
        fontsize=STAT_LEGEND_FONTSIZE,
        loc="lower right",
    )
    ax.grid(True, alpha=0.25)

    apply_statistical_axis_style(
        ax,
        legend_fontsize=STAT_LEGEND_FONTSIZE,
    )

    save_statistical_figure_triplet(
        fig,
        Config.ROC_FIG_PNG,
        Config.ROC_FIG_PDF,
        Config.ROC_FIG_SVG,
        dpi=STAT_FIG_DPI,
    )


def plot_fold_auc(all_fold_metrics: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(5.5, 4.0))

    x_positions = np.arange(1, len(Config.SCENARIO_ORDER) + 1)
    for x, scenario_name in zip(x_positions, Config.SCENARIO_ORDER):
        values = all_fold_metrics.loc[
            all_fold_metrics["scenario"] == scenario_name,
            "roc_auc",
        ].to_numpy(dtype=float)
        ax.scatter(np.full(values.shape, x), values, s=28)
        ax.plot(
            [x - 0.16, x + 0.16],
            [values.mean(), values.mean()],
            linewidth=1.4,
        )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [Config.MODEL_SHORT_LABELS[name] for name in Config.SCENARIO_ORDER]
    )
    ax.set_xlabel(
        "Model configuration",
        fontsize=STAT_AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )
    ax.set_ylabel(
        "Held-out spatial-fold AUC",
        fontsize=STAT_AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )
    ax.set_ylim(0.0, 1.02)
    ax.grid(True, axis="y", alpha=0.25)

    apply_statistical_axis_style(ax)

    save_statistical_figure_triplet(
        fig,
        Config.FOLD_AUC_FIG_PNG,
        Config.FOLD_AUC_FIG_PDF,
        Config.FOLD_AUC_FIG_SVG,
        dpi=STAT_FIG_DPI,
    )


# =============================================================================
# REPORTING
# =============================================================================

def write_manuscript_performance_excel(
    summary_df: pd.DataFrame,
    comparison_df: pd.DataFrame,
    all_fold_metrics: pd.DataFrame,
    ensemble_oof_metrics: Dict[str, Any],
) -> Path:
    """
    Export manuscript-ready numerical tables to a single Excel workbook.

    Sheets
    ------
    1. A-F Performance
       Main manuscript table:
       Model | Predictor configuration | OOF ROC-AUC | 95% CI | PR-AUC |
       Balanced Accuracy | F1-score

    2. AUC Comparisons
       Reviewer-focused paired delta-AUC comparisons with cluster-bootstrap CIs.

    3. Fold AUC
       Fold-specific held-out spatial AUC values plus mean and SD.

    4. Ensemble Summary
       Fixed A30/E70 ensemble OOF performance, reported separately from A-F.
    """
    predictor_labels = {
        "A_static_baseline": "Static12",
        "B_raw_deformation": "Static12 + DV+",
        "C_hkfi_only": "Static12 + HKFI",
        "D_sndr_only": "Static12 + SNDR",
        "E_physics_guided": "Static12 + HKFI + SNDR",
        "F_untransformed_sources": "Static12 + DV+ + TWI + support-preserved SM",
    }

    # ---------------- Main A-F manuscript table ----------------
    ordered_summary = (
        summary_df.set_index("scenario")
        .loc[Config.SCENARIO_ORDER]
        .reset_index()
    )

    main_table = pd.DataFrame(
        {
            "Model": ordered_summary["model"],
            "Predictor configuration": [
                predictor_labels.get(name, Config.MODEL_DESCRIPTIONS[name])
                for name in ordered_summary["scenario"]
            ],
            "OOF ROC-AUC": ordered_summary["oof_roc_auc"],
            "95% CI": [
                (
                    f"{low:.3f}–{high:.3f}"
                    if np.isfinite(low) and np.isfinite(high)
                    else ""
                )
                for low, high in zip(
                    ordered_summary["oof_auc_cluster_bootstrap_ci_low"],
                    ordered_summary["oof_auc_cluster_bootstrap_ci_high"],
                )
            ],
            "CI lower": ordered_summary["oof_auc_cluster_bootstrap_ci_low"],
            "CI upper": ordered_summary["oof_auc_cluster_bootstrap_ci_high"],
            "PR-AUC": ordered_summary["oof_pr_auc"],
            "Balanced Accuracy": ordered_summary["oof_balanced_accuracy_0p5"],
            "F1-score": ordered_summary["oof_f1_0p5"],
            "Fold AUC mean": ordered_summary["fold_auc_mean"],
            "Fold AUC SD": ordered_summary["fold_auc_std"],
        }
    )

    # ---------------- Paired delta-AUC comparison table ----------------
    comparison_table = pd.DataFrame(
        {
            "Comparison": comparison_df["comparison"],
            "ΔAUC": comparison_df["auc_difference_left_minus_right"],
            "95% CI": [
                (
                    f"{low:+.3f}–{high:+.3f}"
                    if np.isfinite(low) and np.isfinite(high)
                    else ""
                )
                for low, high in zip(
                    comparison_df["cluster_bootstrap_ci_low"],
                    comparison_df["cluster_bootstrap_ci_high"],
                )
            ],
            "CI lower": comparison_df["cluster_bootstrap_ci_low"],
            "CI upper": comparison_df["cluster_bootstrap_ci_high"],
            "Interpretation": comparison_df["purpose"],
        }
    )

    # ---------------- Fold-specific AUC table ----------------
    fold_table = (
        all_fold_metrics.pivot(
            index="scenario",
            columns="outer_fold",
            values="roc_auc",
        )
        .reindex(Config.SCENARIO_ORDER)
        .reset_index()
    )

    fold_table.insert(
        1,
        "Model",
        [
            Config.MODEL_SHORT_LABELS[scenario_name]
            for scenario_name in fold_table["scenario"]
        ],
    )
    fold_table.insert(
        2,
        "Predictor configuration",
        [
            predictor_labels.get(
                scenario_name,
                Config.MODEL_DESCRIPTIONS[scenario_name],
            )
            for scenario_name in fold_table["scenario"]
        ],
    )

    fold_columns = [
        column
        for column in fold_table.columns
        if isinstance(column, (int, np.integer))
    ]
    fold_columns = sorted(fold_columns)

    fold_table = fold_table.rename(
        columns={
            fold: f"Fold {fold}"
            for fold in fold_columns
        }
    )

    fold_metric_columns = [
        f"Fold {fold}"
        for fold in fold_columns
    ]
    fold_table["Mean"] = fold_table[fold_metric_columns].mean(axis=1)
    fold_table["SD"] = fold_table[fold_metric_columns].std(axis=1, ddof=1)
    fold_table = fold_table.drop(columns=["scenario"])

    # ---------------- Separate operational ensemble summary ----------------
    ensemble_table = pd.DataFrame(
        [
            {
                "Mapping product": "A30/E70 fixed operational ensemble",
                "Formula": "0.30 × Model A + 0.70 × Model E where E is available; otherwise Model A",
                "OOF ROC-AUC": ensemble_oof_metrics.get("roc_auc"),
                "95% CI": (
                    f"{ensemble_oof_metrics.get('auc_cluster_bootstrap_ci_low', np.nan):.3f}"
                    "–"
                    f"{ensemble_oof_metrics.get('auc_cluster_bootstrap_ci_high', np.nan):.3f}"
                ),
                "PR-AUC": ensemble_oof_metrics.get("pr_auc"),
                "Balanced Accuracy": ensemble_oof_metrics.get("balanced_accuracy"),
                "F1-score": ensemble_oof_metrics.get("f1"),
                "ΔAUC vs Model E": ensemble_oof_metrics.get(
                    "auc_difference_ensemble_minus_model_E"
                ),
                "ΔAUC 95% CI": (
                    f"{ensemble_oof_metrics.get('auc_difference_ci_low', np.nan):+.3f}"
                    "–"
                    f"{ensemble_oof_metrics.get('auc_difference_ci_high', np.nan):+.3f}"
                ),
                "Role in manuscript": (
                    "Final corridor-wide operational mapping product; "
                    "reported separately from the A–F ablation evaluation."
                ),
            }
        ]
    )

    # ---------------- Write Excel workbook ----------------
    output_path = Config.MANUSCRIPT_TABLES_XLSX

    try:
        # xlsxwriter provides publication-ready formatting when available.
        with pd.ExcelWriter(
            output_path,
            engine="xlsxwriter",
        ) as writer:
            main_table.to_excel(
                writer,
                sheet_name="A-F Performance",
                index=False,
            )
            comparison_table.to_excel(
                writer,
                sheet_name="AUC Comparisons",
                index=False,
            )
            fold_table.to_excel(
                writer,
                sheet_name="Fold AUC",
                index=False,
            )
            ensemble_table.to_excel(
                writer,
                sheet_name="Ensemble Summary",
                index=False,
            )

            workbook = writer.book

            header_format = workbook.add_format(
                {
                    "bold": True,
                    "font_color": "white",
                    "bg_color": "#1F4E78",
                    "border": 1,
                    "align": "center",
                    "valign": "vcenter",
                    "text_wrap": True,
                }
            )
            number_format = workbook.add_format(
                {
                    "num_format": "0.000",
                    "border": 1,
                    "align": "center",
                }
            )
            text_format = workbook.add_format(
                {
                    "border": 1,
                    "valign": "top",
                    "text_wrap": True,
                }
            )
            center_text_format = workbook.add_format(
                {
                    "border": 1,
                    "align": "center",
                    "valign": "center",
                    "text_wrap": True,
                }
            )

            sheet_specs = {
                "A-F Performance": {
                    "df": main_table,
                    "widths": [10, 45, 13, 18, 12, 12, 12, 18, 12, 15, 13],
                },
                "AUC Comparisons": {
                    "df": comparison_table,
                    "widths": [14, 12, 18, 12, 12, 42],
                },
                "Fold AUC": {
                    "df": fold_table,
                    "widths": [10, 45] + [12] * (len(fold_metric_columns) + 2),
                },
                "Ensemble Summary": {
                    "df": ensemble_table,
                    "widths": [32, 70, 13, 18, 12, 18, 12, 18, 18, 60],
                },
            }

            for sheet_name, spec in sheet_specs.items():
                worksheet = writer.sheets[sheet_name]
                dataframe = spec["df"]

                # Freeze the header row.
                worksheet.freeze_panes(1, 0)

                # Reformat headers.
                for column_index, column_name in enumerate(dataframe.columns):
                    worksheet.write(
                        0,
                        column_index,
                        column_name,
                        header_format,
                    )

                # Set useful column widths.
                for column_index, width in enumerate(spec["widths"]):
                    worksheet.set_column(
                        column_index,
                        column_index,
                        width,
                    )

                # Apply borders and numeric formatting to body cells.
                for row_index in range(1, len(dataframe) + 1):
                    for column_index, column_name in enumerate(dataframe.columns):
                        value = dataframe.iloc[row_index - 1, column_index]

                        if pd.isna(value):
                            worksheet.write_blank(
                                row_index,
                                column_index,
                                None,
                                center_text_format,
                            )
                        elif isinstance(value, (float, np.floating, int, np.integer)):
                            worksheet.write_number(
                                row_index,
                                column_index,
                                float(value),
                                number_format,
                            )
                        else:
                            worksheet.write(
                                row_index,
                                column_index,
                                str(value),
                                (
                                    center_text_format
                                    if column_name in {
                                        "Model",
                                        "Comparison",
                                        "95% CI",
                                        "ΔAUC 95% CI",
                                    }
                                    else text_format
                                ),
                            )

                # Add an Excel table for convenient filtering and manuscript review.
                if len(dataframe) > 0 and len(dataframe.columns) > 0:
                    worksheet.add_table(
                        0,
                        0,
                        len(dataframe),
                        len(dataframe.columns) - 1,
                        {
                            "style": "Table Style Medium 2",
                            "columns": [
                                {"header": column_name}
                                for column_name in dataframe.columns
                            ],
                        },
                    )

    except (ImportError, ModuleNotFoundError):
        # Fallback: still create a valid workbook using the default pandas engine.
        # The numerical content remains identical, but styling may be simpler.
        with pd.ExcelWriter(output_path) as writer:
            main_table.to_excel(
                writer,
                sheet_name="A-F Performance",
                index=False,
            )
            comparison_table.to_excel(
                writer,
                sheet_name="AUC Comparisons",
                index=False,
            )
            fold_table.to_excel(
                writer,
                sheet_name="Fold AUC",
                index=False,
            )
            ensemble_table.to_excel(
                writer,
                sheet_name="Ensemble Summary",
                index=False,
            )

    log(
        "[EXCEL] Manuscript-ready performance tables saved: "
        f"{output_path}"
    )

    return output_path


def write_qa_report(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
    summary_df: pd.DataFrame,
    comparison_df: pd.DataFrame,
    final_params: Dict[str, Any],
    map_paths: Dict[str, str],
) -> None:
    with open(Config.QA_REPORT_TXT, "w", encoding="utf-8") as handle:
        handle.write("STEP 7.2 — RANDOM FOREST MODELS A–F: NESTED SPATIAL CV QA REPORT\n")
        handle.write("=" * 88 + "\n\n")
        handle.write(f"Generated: {utc_now_iso()}\n")
        handle.write(f"Input dataset: {Config.MODEL_READY_DATASET_CSV}\n")
        handle.write(f"Rows: {len(df)}\n")
        handle.write(
            f"Class balance: positive={(df[Config.TARGET_COLUMN] == 1).sum()}, "
            f"negative={(df[Config.TARGET_COLUMN] == 0).sum()}\n"
        )
        handle.write(
            f"Frozen outer folds: {sorted(df[Config.FOLD_COLUMN].unique().tolist())}\n\n"
        )

        handle.write("VALIDATION DESIGN\n")
        handle.write("-" * 88 + "\n")
        handle.write(
            "Outer evaluation: five frozen contiguous spatial blocks; one block held out at a time.\n"
        )
        handle.write(
            "Inner tuning: four-block PredefinedSplit using only the outer-training data.\n"
        )
        handle.write(
            "No outer validation observations are used for hyperparameter selection or model selection.\n"
        )
        handle.write(
            "All A–F models use identical samples, labels and outer folds.\n"
        )
        handle.write(
            "Each frozen outer fold contains exactly 33 positives and 33 pseudo-absences.\n"
        )
        handle.write(
            "Land cover is one-hot encoded inside each fitted pipeline; continuous RF predictors are not scaled.\n"
        )
        handle.write(
            "Binary metrics use a fixed 0.50 probability threshold.\n\n"
        )

        handle.write("SCENARIOS\n")
        handle.write("-" * 88 + "\n")
        for scenario_name in Config.SCENARIO_ORDER:
            features = scenarios[scenario_name]["features"]
            handle.write(
                f"{Config.MODEL_SHORT_LABELS[scenario_name]}: "
                f"{Config.MODEL_DESCRIPTIONS[scenario_name]} "
                f"({len(features)} predictors)\n"
            )
        handle.write("\n")

        handle.write("NESTED OOF SUMMARY\n")
        handle.write("-" * 88 + "\n")
        handle.write(summary_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("PAIRED AUC COMPARISONS\n")
        handle.write("-" * 88 + "\n")
        handle.write(comparison_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("FINAL DEPLOYMENT MODELS\n")
        handle.write("-" * 88 + "\n")
        if final_params:
            for scenario_name in Config.SCENARIO_ORDER:
                info = final_params[scenario_name]
                handle.write(
                    f"{Config.MODEL_SHORT_LABELS[scenario_name]}: "
                    f"selection AUC={info['best_mean_spatial_cv_auc_for_model_selection_only']:.6f}; "
                    f"params={info['best_params']}\n"
                )
        else:
            handle.write(
                "Skipped in this run because RUN_FINAL_DEPLOYMENT_TUNING=False. "
                "Nested spatial OOF evaluation was completed independently.\n"
            )
        handle.write("\n")

        if map_paths:
            handle.write("RASTER MAPS\n")
            handle.write("-" * 88 + "\n")
            handle.write(
                "Maps are generated from separately tuned full-data deployment models on the common A–F valid domain.\n"
            )
            handle.write(
                "These map-model tuning scores are not reported as validation performance.\n"
            )
            for scenario_name, path in map_paths.items():
                handle.write(f"{scenario_name}: {path}\n")


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=" * 96)
    log("=== Step 7.2: Random Forest Models A–F with Nested Spatial Cross-Validation ===")
    log("=== EXPECTED RUNTIME: ~5,760 Random Forest fits (nested CV) + ~1,440 deployment fits ===")
    log("=== With n_jobs=1, expect several hours. Monitor progress via verbose output. ===")
    log(
        "[RUN-MODE] Final deployment tuning="
        f"{'ON' if Config.RUN_FINAL_DEPLOYMENT_TUNING else 'OFF'} | "
        "Raster maps="
        f"{'ON' if Config.GENERATE_RASTER_MAPS else 'OFF'} | "
        "Primary Model E visualization="
        f"{'ON' if Config.GENERATE_PRIMARY_LSM_VISUALIZATION else 'OFF'} | "
        "A30/E70 ensemble LSM="
        f"{'ON' if Config.GENERATE_AE_ENSEMBLE_LSM else 'OFF'}"
    )
    log(
        "[VIZ-OUT] Primary Landslide Susceptibility Map outputs: "
        f"{Config.VISUALIZATION_DIR}"
    )
    log("=" * 96)

    # 1. Validate frozen A–F design and primary frozen ML dataset.
    scenarios = validate_scenario_manifests()
    df = load_and_validate_model_ready_dataset(scenarios)

    all_oof_tables: List[pd.DataFrame] = []
    all_fold_tables: List[pd.DataFrame] = []
    all_inner_search_tables: List[pd.DataFrame] = []
    outer_best_params: Dict[str, Any] = {}

    # 2. Unbiased nested spatial CV for each A–F model.
    for scenario_name in Config.SCENARIO_ORDER:
        features = list(scenarios[scenario_name]["features"])

        oof_df, fold_df, params, search_df = run_nested_spatial_cv_for_scenario(
            df=df,
            scenario_name=scenario_name,
            features=features,
        )

        all_oof_tables.append(oof_df)
        all_fold_tables.append(fold_df)
        all_inner_search_tables.append(search_df)
        outer_best_params[scenario_name] = params

    all_oof = pd.concat(all_oof_tables, ignore_index=True)
    all_fold_metrics = pd.concat(all_fold_tables, ignore_index=True)
    all_inner_search = pd.concat(all_inner_search_tables, ignore_index=True)

    # Cross-scenario QA: every scenario must contain the exact same sample IDs.
    reference_ids = set(
        all_oof.loc[
            all_oof["scenario"] == Config.SCENARIO_ORDER[0],
            Config.SAMPLE_ID_COLUMN,
        ]
    )
    for scenario_name in Config.SCENARIO_ORDER[1:]:
        scenario_ids = set(
            all_oof.loc[
                all_oof["scenario"] == scenario_name,
                Config.SAMPLE_ID_COLUMN,
            ]
        )
        if scenario_ids != reference_ids:
            raise RuntimeError(
                f"OOF sample IDs differ for {scenario_name}; A–F comparison invalid."
            )

    # 3. Summaries and reviewer-focused paired comparisons.
    summary_df = summarize_oof_results(all_oof, all_fold_metrics)
    comparison_df = make_paired_auc_comparisons(all_oof)

    # Fixed A30/E70 ensemble OOF diagnostic. This mirrors the old
    # static + integrated blend but does not alter the A-F ablation results.
    ensemble_oof_df, ensemble_oof_metrics = build_fixed_ae_oof_ensemble(
        all_oof
    )

    all_oof.to_csv(Config.OOF_PREDICTIONS_CSV, index=False, float_format="%.10f")
    all_fold_metrics.to_csv(
        Config.FOLD_METRICS_CSV, index=False, float_format="%.10f"
    )
    summary_df.to_csv(
        Config.SUMMARY_METRICS_CSV, index=False, float_format="%.10f"
    )
    comparison_df.to_csv(
        Config.AUC_COMPARISONS_CSV, index=False, float_format="%.10f"
    )

    # Manuscript-ready Excel workbook with the compact A-F table recommended
    # for the revised paper, paired delta-AUC comparisons, fold-level AUCs,
    # and a separate operational ensemble summary.
    manuscript_excel_path = write_manuscript_performance_excel(
        summary_df=summary_df,
        comparison_df=comparison_df,
        all_fold_metrics=all_fold_metrics,
        ensemble_oof_metrics=ensemble_oof_metrics,
    )

    all_inner_search.to_csv(
        Config.INNER_SEARCH_CSV, index=False, float_format="%.10f"
    )

    with open(Config.OUTER_PARAMS_JSON, "w", encoding="utf-8") as handle:
        json.dump(json_safe(outer_best_params), handle, indent=2)

    plot_oof_roc_curves(all_oof)
    plot_fold_auc(all_fold_metrics)

    # 4. Optional separate final full-data models for mapping/deployment only.
    final_models: Dict[str, Pipeline] = {}
    final_params: Dict[str, Any] = {}
    full_search_tables: List[pd.DataFrame] = []

    if Config.RUN_FINAL_DEPLOYMENT_TUNING:
        log("")
        log("[FINAL-MODEL] Running full-data spatial tuning for deployment models...")
        for scenario_name in Config.SCENARIO_ORDER:
            features = list(scenarios[scenario_name]["features"])
            final_model, info, search_df = tune_and_fit_final_model(
                df=df,
                scenario_name=scenario_name,
                features=features,
            )
            final_models[scenario_name] = final_model
            final_params[scenario_name] = info
            full_search_tables.append(search_df)

        full_search_all = pd.concat(full_search_tables, ignore_index=True)
        full_search_all.to_csv(
            Config.FINAL_SEARCH_CSV, index=False, float_format="%.10f"
        )

        with open(Config.FINAL_PARAMS_JSON, "w", encoding="utf-8") as handle:
            json.dump(json_safe(final_params), handle, indent=2)
    else:
        log(
            "[FINAL-MODEL] Skipped: RUN_FINAL_DEPLOYMENT_TUNING=False. "
            "This diagnostic run will report nested spatial OOF performance only."
        )

    # 5. Optional A–F susceptibility rasters on an identical common domain.
    map_paths: Dict[str, str] = {}
    if Config.GENERATE_RASTER_MAPS:
        if not Config.RUN_FINAL_DEPLOYMENT_TUNING:
            raise RuntimeError(
                "GENERATE_RASTER_MAPS=True requires "
                "RUN_FINAL_DEPLOYMENT_TUNING=True."
            )

        log("")
        log("[MAP] Generating six Random Forest A–F probability maps...")
        for scenario_name in Config.SCENARIO_ORDER:
            features = list(scenarios[scenario_name]["features"])
            output_path = predict_scenario_raster(
                fitted_pipeline=final_models[scenario_name],
                scenario_name=scenario_name,
                features=features,
            )
            map_paths[scenario_name] = str(output_path)

    # 6. Revised old-style A/E ensemble susceptibility raster.
    ensemble_map_path: Path | None = None
    ensemble_visualization_outputs: Dict[str, Any] = {}

    if Config.GENERATE_AE_ENSEMBLE_LSM:
        if not Config.RUN_FINAL_DEPLOYMENT_TUNING:
            raise RuntimeError(
                "GENERATE_AE_ENSEMBLE_LSM=True requires "
                "RUN_FINAL_DEPLOYMENT_TUNING=True."
            )

        log("")
        log(
            "[ENSEMBLE-MAP] Generating revised static-background + "
            "physics-guided ensemble map..."
        )
        ensemble_map_path = create_revised_ae_ensemble_probability_map(
            final_models=final_models,
            scenarios=scenarios,
        )

        ensemble_visualization_outputs = (
            create_complete_ensemble_lsm_visualization(
                probability_raster_path=ensemble_map_path,
                ensemble_oof_metrics=ensemble_oof_metrics,
            )
        )

    # 7. Optional publication-style primary Model E LSM visualization.
    # Retained separately so the manuscript can still show the actual primary
    # Model E map without confusing it with the A/E deployment ensemble.
    visualization_outputs: Dict[str, Any] = {}
    if Config.GENERATE_PRIMARY_LSM_VISUALIZATION:
        primary_scenario = Config.PRIMARY_VISUALIZATION_SCENARIO

        if primary_scenario not in Config.SCENARIO_ORDER:
            raise RuntimeError(
                "PRIMARY_VISUALIZATION_SCENARIO is not a valid A-F scenario: "
                f"{primary_scenario}"
            )

        if primary_scenario in map_paths:
            primary_map_path = Path(map_paths[primary_scenario])
        else:
            primary_map_path = expected_scenario_map_path(
                primary_scenario
            )

        if not primary_map_path.exists():
            raise RuntimeError(
                "Primary susceptibility visualization requested, but the "
                f"probability raster does not exist: {primary_map_path}\n"
                "Either set RUN_FINAL_DEPLOYMENT_TUNING=True and "
                "GENERATE_RASTER_MAPS=True, or first generate the Model E raster."
            )

        visualization_outputs = create_primary_lsm_visualization(
            raster_path=primary_map_path,
            scenario_name=primary_scenario,
        )

    # 8. QA / provenance metadata.
    write_qa_report(
        df=df,
        scenarios=scenarios,
        summary_df=summary_df,
        comparison_df=comparison_df,
        final_params=final_params,
        map_paths=map_paths,
    )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "Random Forest",
        "validation_design": {
            "outer": "five frozen contiguous spatial blocks from Step 6.4",
            "inner": (
                "four-block PredefinedSplit inside each outer-training set; "
                "one remaining spatial block held out per inner fold"
            ),
            "hyperparameter_selection_metric": "ROC-AUC",
            "reported_primary_performance": "pooled nested outer OOF ROC-AUC",
            "binary_threshold": Config.BINARY_THRESHOLD,
            "confidence_interval": (
                f"{int(Config.CI_LEVEL * 100)}% cluster bootstrap resampling whole spatial blocks"
            ),
            "cluster_bootstrap_iterations": Config.N_CLUSTER_BOOTSTRAP,
        },
        "input_dataset": str(Config.MODEL_READY_DATASET_CSV),
        "scenario_manifest_step61": str(Config.SCENARIO_MANIFEST_JSON),
        "scenario_manifest_step64": str(Config.STEP64_MODEL_CONFIG_JSON),
        "parameter_grid": Config.PARAM_GRID,
        "fixed_rf_parameters": Config.RF_FIXED_PARAMS,
        "execution_controls": {
            "run_final_deployment_tuning": Config.RUN_FINAL_DEPLOYMENT_TUNING,
            "generate_raster_maps": Config.GENERATE_RASTER_MAPS,
            "generate_primary_lsm_visualization": (
                Config.GENERATE_PRIMARY_LSM_VISUALIZATION
            ),
            "primary_visualization_scenario": (
                Config.PRIMARY_VISUALIZATION_SCENARIO
            ),
            "generate_AE_ensemble_lsm": Config.GENERATE_AE_ENSEMBLE_LSM,
            "ensemble_static_scenario": Config.ENSEMBLE_STATIC_SCENARIO,
            "ensemble_enhanced_scenario": Config.ENSEMBLE_ENHANCED_SCENARIO,
            "ensemble_enhanced_weight": Config.ENSEMBLE_ENHANCED_WEIGHT,
        },
        "scenarios": {
            scenario_name: {
                "label": Config.MODEL_SHORT_LABELS[scenario_name],
                "description": Config.MODEL_DESCRIPTIONS[scenario_name],
                "features": list(scenarios[scenario_name]["features"]),
            }
            for scenario_name in Config.SCENARIO_ORDER
        },
        "outputs": {
            "oof_predictions": str(Config.OOF_PREDICTIONS_CSV),
            "fold_metrics": str(Config.FOLD_METRICS_CSV),
            "summary_metrics": str(Config.SUMMARY_METRICS_CSV),
            "paired_auc_comparisons": str(Config.AUC_COMPARISONS_CSV),
            "outer_best_parameters": str(Config.OUTER_PARAMS_JSON),
            "final_best_parameters": str(Config.FINAL_PARAMS_JSON),
            "nested_grid_results": str(Config.INNER_SEARCH_CSV),
            "full_data_grid_results": str(Config.FINAL_SEARCH_CSV),
            "final_model_directory": str(Config.MODEL_DIR),
            "map_directory": str(Config.MAP_DIR),
            "visualization_directory": str(Config.VISUALIZATION_DIR),
            "primary_lsm_visualization": visualization_outputs,
            "ensemble_oof_predictions": str(
                Config.ENSEMBLE_OOF_PREDICTIONS_CSV
            ),
            "ensemble_oof_metrics": str(
                Config.ENSEMBLE_OOF_METRICS_JSON
            ),
            "ensemble_probability_raster": (
                str(ensemble_map_path)
                if ensemble_map_path is not None
                else None
            ),
            "ensemble_lsm_visualization": ensemble_visualization_outputs,
            "manuscript_performance_excel": str(
                Config.MANUSCRIPT_TABLES_XLSX
            ),
            "qa_report": str(Config.QA_REPORT_TXT),
        },
    }

    with open(Config.METADATA_JSON, "w", encoding="utf-8") as handle:
        json.dump(json_safe(metadata), handle, indent=2)

    # 9. Console summary.
    log("")
    log("=" * 96)
    log("NESTED SPATIAL OOF SUMMARY")
    log("=" * 96)
    for _, row in summary_df.iterrows():
        log(
            f"{row['model']}: AUC={row['oof_roc_auc']:.4f} "
            f"[{row['oof_auc_cluster_bootstrap_ci_low']:.4f}, "
            f"{row['oof_auc_cluster_bootstrap_ci_high']:.4f}], "
            f"F1={row['oof_f1_0p5']:.4f}, "
            f"Precision={row['oof_precision_0p5']:.4f}, "
            f"Recall={row['oof_recall_0p5']:.4f}"
        )

    log("")
    log("PAIRED AUC COMPARISONS")
    for _, row in comparison_df.iterrows():
        log(
            f"{row['comparison']}: ΔAUC={row['auc_difference_left_minus_right']:+.4f} "
            f"[{row['cluster_bootstrap_ci_low']:+.4f}, "
            f"{row['cluster_bootstrap_ci_high']:+.4f}]"
        )

    log("")
    log(
        "FIXED A30/E70 ENSEMBLE OOF: "
        f"AUC={ensemble_oof_metrics['roc_auc']:.4f} "
        f"[{ensemble_oof_metrics['auc_cluster_bootstrap_ci_low']:.4f}, "
        f"{ensemble_oof_metrics['auc_cluster_bootstrap_ci_high']:.4f}], "
        f"ΔAUC vs Model E="
        f"{ensemble_oof_metrics['auc_difference_ensemble_minus_model_E']:+.4f}"
    )

    if Config.GENERATE_AE_ENSEMBLE_LSM and ensemble_map_path is not None:
        log(
            "[OUT-ENSEMBLE] Old-style full-background ensemble LSM saved to: "
            f"{Config.VISUALIZATION_DIR}"
        )

    log("")
    if Config.GENERATE_PRIMARY_LSM_VISUALIZATION:
        log(
            "[OUT-VIZ] Primary Model E Landslide Susceptibility Map and "
            f"histogram saved to: {Config.VISUALIZATION_DIR}"
        )
    log(
        "[OUT-EXCEL] Manuscript A-F numerical tables: "
        f"{Config.MANUSCRIPT_TABLES_XLSX}"
    )
    log(f"[OUT] Results saved to: {Config.OUT_DIR}")
    log("=== Step 7.2 completed successfully ===")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 7.2 failed: {exc}")
        traceback.print_exc()
        raise


# STEP 7.2A — RANDOM FOREST SPATIAL LEARNING CURVES

In [ ]:
# =============================================================================
# STEP 7.2A — RANDOM FOREST SPATIAL LEARNING CURVES
#              + HELD-OUT PERMUTATION IMPORTANCE
# =============================================================================
#
# Scientific purpose
# ------------------
# This script is the Random Forest equivalent of Step 7.1A (XGBoost).
# It does NOT change the primary Step 7.2 nested spatial-CV evaluation.
#
# It adds two reviewer-oriented robustness diagnostics using the EXACT successful
# Step 7.2 Random Forest modelling contract:
#
# 1. Fixed-hyperparameter spatial learning curves for Model A and Model E only.
#    - The original outer validation block remains untouched.
#    - Training fractions are sampled only from the four outer-training blocks.
#    - Every reduced training set retains all four spatial blocks and both classes.
#    - The exact fold-specific hyperparameters selected during successful Step 7.2
#      are reused at all training sizes.
#    - No second hyperparameter search is performed.
#    - The held-out outer block remains completely unseen during fitting.
#
# 2. Held-out permutation importance for Model E.
#    - The RF model is fitted on four spatial blocks.
#    - Importance is calculated only on the untouched fifth block.
#    - Individual predictor importance is complemented by a grouped joint
#      permutation of HKFI + SNDR.
#
# Primary provenance
# ------------------
# This script reads the successful Step 7.2 outputs directly from:
#
#   ./data/Results/Step-7.2-RF-Models_A_to_F/
#
# Required Step 7.2 outputs:
#   RF_A_to_F_outer_fold_best_parameters.json
#   RF_A_to_F_OOF_predictions.csv
#
# Required upstream inputs:
#   Step 6.1 model_ablation_scenarios.json
#   Step 6.4 model_configuration_manifest.json
#   Step 6.4 model_ready_dataset_A_to_F.csv
#
# Scientific interpretation
# -------------------------
# Learning curves are a sample-size robustness diagnostic. Their uncertainty
# bands describe variability across the five held-out spatial folds; they are
# not paired confidence intervals for Model E - Model A.
#
# Permutation importance is conditional on the fitted Model E predictor set.
# Near-zero individual importance does not prove that a predictor has no value,
# particularly when predictors are correlated or redundant. The A-F ablation
# analysis remains the stronger retraining-based test of incremental information.
# =============================================================================

from __future__ import annotations

import gc
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Sequence

# Resource-safe execution. The successful Step 7.2 RF workflow also keeps the
# estimator single-threaded to avoid nested parallelism.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


# =============================================================================
# CONFIGURATION — FROZEN TO THE SUCCESSFUL STEP 7.2 RF EXPERIMENT
# =============================================================================
class DiagnosticConfig:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Frozen Step 6.1 and Step 6.4 modelling definitions.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"

    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    MODEL_READY_DATASET_CSV = STEP64_DIR / "model_ready_dataset_A_to_F.csv"
    STEP64_MODEL_CONFIG_JSON = STEP64_DIR / "model_configuration_manifest.json"

    # Successful Step 7.2 Random Forest outputs.
    STEP72_OUT_DIR = RESULTS_DIR / "Step-7.2-RF-Models_A_to_F"
    OUTER_PARAMS_JSON = (
        STEP72_OUT_DIR / "RF_A_to_F_outer_fold_best_parameters.json"
    )
    PRIMARY_OOF_CSV = (
        STEP72_OUT_DIR / "RF_A_to_F_OOF_predictions.csv"
    )
    STEP72_METADATA_JSON = (
        STEP72_OUT_DIR / "Step_7_2_RF_A_to_F_Metadata.json"
    )

    # Step 7.2A outputs.
    OUT_DIR = STEP72_OUT_DIR / "Step-7.2A-Learning_and_Permutation"

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    # Same two learning-curve models used in Step 7.1A.
    LEARNING_SCENARIOS = (
        "A_static_baseline",
        "E_physics_guided",
    )

    # Same permutation-importance target used in Step 7.1A.
    PERMUTATION_SCENARIO = "E_physics_guided"

    # Same training fractions/repeats used in Step 7.1A.
    TRAIN_FRACTIONS = (0.25, 0.50, 0.75, 1.00)
    N_REPEATS_SUBSAMPLE = 20
    N_PERMUTATION_REPEATS = 30

    # Random state intentionally differs from XGB Step 7.1A while remaining fixed.
    RANDOM_STATE = 52071

    # The RF estimator itself is reconstructed using the exact fixed settings
    # used in Step 7.2. Fold-specific tuned parameters are then applied from the
    # saved Step 7.2 outer-parameter JSON.
    RF_FIXED_PARAMS = {
        "bootstrap": True,
        "class_weight": None,
        "random_state": 42,
        "n_jobs": 1,
    }

    CATEGORICAL_FEATURE_ALIASES = {
        "landcover",
        "landuse",
        "land_use",
    }

    FIGSIZE = (5.5, 4.2)
    FIG_DPI = 900

    # Full-training reproduction check against successful Step 7.2 OOF results.
    OOF_REPRODUCTION_TOLERANCE = 1.0e-10


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(
        f"[{datetime.now():%H:%M:%S}] {message}",
        flush=True,
    )


def utc_now_iso() -> str:
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def require_file(
    path: Path,
    description: str,
) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}"
        )


def load_json(
    path: Path,
) -> Dict[str, Any]:
    require_file(
        path,
        "JSON file",
    )
    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def json_safe(
    value: Any,
) -> Any:
    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()
    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()
    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }
    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe(item)
            for item in value
        ]
    return value


def normalize_feature_name(
    name: str,
) -> str:
    import re

    return re.sub(
        r"[^a-z0-9_]+",
        "_",
        str(name).strip().lower(),
    ).strip(
        "_"
    )


def is_categorical_feature(
    name: str,
) -> bool:
    normalized = normalize_feature_name(
        name
    )
    compact = normalized.replace(
        "_",
        "",
    )

    aliases = {
        alias.lower()
        for alias in DiagnosticConfig.CATEGORICAL_FEATURE_ALIASES
    }
    compact_aliases = {
        alias.replace(
            "_",
            "",
        )
        for alias in aliases
    }

    return (
        normalized in aliases
        or compact in compact_aliases
    )


# =============================================================================
# STEP 6.1 / STEP 6.4 PROVENANCE AND DATASET VALIDATION
# =============================================================================
def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """
    Cross-check the Step 6.1 predictor definitions against the frozen Step 6.4
    model configuration exactly as the successful Step 7.2 RF workflow does.
    """
    step61 = load_json(
        DiagnosticConfig.SCENARIO_MANIFEST_JSON
    ).get(
        "scenarios",
        {},
    )

    step64 = load_json(
        DiagnosticConfig.STEP64_MODEL_CONFIG_JSON
    ).get(
        "scenarios",
        {},
    )

    for scenario_name in DiagnosticConfig.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                "Step 6.1 scenario manifest is missing "
                f"{scenario_name}."
            )

        if scenario_name not in step64:
            raise RuntimeError(
                "Step 6.4 frozen model manifest is missing "
                f"{scenario_name}."
            )

        features_61 = list(
            step61[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        features_64 = list(
            step64[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any(
            "irs" in feature.lower()
            for feature in features_61
        ):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; "
                "IRS must not be an ML predictor."
            )

    log(
        "[QA] Step 6.1 and Step 6.4 A-F scenario definitions match: PASS"
    )

    return step64


def all_scenario_features(
    scenarios: Dict[str, Dict[str, Any]],
) -> list[str]:
    features: list[str] = []

    for scenario_name in DiagnosticConfig.SCENARIO_ORDER:
        for feature in scenarios[
            scenario_name
        ][
            "features"
        ]:
            if feature not in features:
                features.append(
                    feature
                )

    return features


def load_and_validate_model_ready_dataset(
    scenarios: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    """
    Load the exact frozen Step 6.4 330-row dataset used by successful Step 7.2.
    """
    require_file(
        DiagnosticConfig.MODEL_READY_DATASET_CSV,
        "Step 6.4 primary frozen model-ready dataset",
    )

    df = pd.read_csv(
        DiagnosticConfig.MODEL_READY_DATASET_CSV
    )

    predictor_columns = all_scenario_features(
        scenarios
    )

    required = [
        DiagnosticConfig.SAMPLE_ID_COLUMN,
        DiagnosticConfig.TARGET_COLUMN,
        DiagnosticConfig.FOLD_COLUMN,
        DiagnosticConfig.BLOCK_COLUMN,
        *predictor_columns,
    ]

    missing = [
        column
        for column in required
        if column not in df.columns
    ]

    if missing:
        raise RuntimeError(
            "Model-ready dataset is missing columns: "
            f"{missing}"
        )

    if len(
        df
    ) != DiagnosticConfig.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Expected {DiagnosticConfig.EXPECTED_N_ROWS} rows; "
            f"found {len(df)}."
        )

    if df[
        DiagnosticConfig.SAMPLE_ID_COLUMN
    ].isna().any():
        raise RuntimeError(
            "Missing sample_id values found."
        )

    if df[
        DiagnosticConfig.SAMPLE_ID_COLUMN
    ].duplicated().any():
        raise RuntimeError(
            "Duplicated sample_id values found."
        )

    for column in (
        DiagnosticConfig.TARGET_COLUMN,
        DiagnosticConfig.FOLD_COLUMN,
        DiagnosticConfig.BLOCK_COLUMN,
    ):
        df[
            column
        ] = pd.to_numeric(
            df[
                column
            ],
            errors="raise",
        ).astype(
            int
        )

    class_counts = (
        df[
            DiagnosticConfig.TARGET_COLUMN
        ]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    expected_class_counts = {
        0: DiagnosticConfig.EXPECTED_N_NEGATIVE,
        1: DiagnosticConfig.EXPECTED_N_POSITIVE,
    }

    if class_counts != expected_class_counts:
        raise RuntimeError(
            f"Unexpected class counts: {class_counts}; "
            f"expected {expected_class_counts}."
        )

    folds = sorted(
        df[
            DiagnosticConfig.FOLD_COLUMN
        ].unique().tolist()
    )

    if folds != DiagnosticConfig.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected frozen folds "
            f"{DiagnosticConfig.EXPECTED_FOLDS}; "
            f"found {folds}."
        )

    if not np.array_equal(
        df[
            DiagnosticConfig.FOLD_COLUMN
        ].to_numpy(),
        df[
            DiagnosticConfig.BLOCK_COLUMN
        ].to_numpy(),
    ):
        raise RuntimeError(
            "cv_fold and spatial_block are not identical."
        )

    block_class = pd.crosstab(
        df[
            DiagnosticConfig.FOLD_COLUMN
        ],
        df[
            DiagnosticConfig.TARGET_COLUMN
        ],
    )

    expected_per_class = (
        DiagnosticConfig.EXPECTED_N_POSITIVE
        // len(
            DiagnosticConfig.EXPECTED_FOLDS
        )
    )

    for fold in DiagnosticConfig.EXPECTED_FOLDS:
        if fold not in block_class.index:
            raise RuntimeError(
                f"Missing fold {fold}."
            )

        for label in (
            0,
            1,
        ):
            count = int(
                block_class.loc[
                    fold,
                    label,
                ]
            )

            if count != expected_per_class:
                raise RuntimeError(
                    f"Fold {fold}, class {label}: "
                    f"found {count}; expected {expected_per_class}."
                )

    predictor_values = df[
        predictor_columns
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    finite = np.isfinite(
        predictor_values.to_numpy(
            dtype="float64"
        )
    )

    if not finite.all():
        bad = np.argwhere(
            ~finite
        )

        examples = [
            (
                str(
                    df.iloc[
                        row_index
                    ][
                        DiagnosticConfig.SAMPLE_ID_COLUMN
                    ]
                ),
                predictor_columns[
                    column_index
                ],
            )
            for row_index, column_index in bad[
                :20
            ]
        ]

        raise RuntimeError(
            "Missing/non-finite predictors remain. "
            f"Examples: {examples}"
        )

    irs_columns = [
        column
        for column in df.columns
        if "irs" in str(
            column
        ).lower()
    ]

    if irs_columns:
        raise RuntimeError(
            "IRS-related columns found in ML dataset: "
            f"{irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError(
            "tpi_s25 found; only tpi_s5 is allowed."
        )

    log(
        "[QA] Frozen Step 6.4 RF modelling dataset: "
        f"rows={len(df)}, "
        f"positive={class_counts[1]}, "
        f"negative={class_counts[0]}, "
        f"folds={folds}: PASS"
    )

    return df


# =============================================================================
# EXACT STEP 7.2 RANDOM FOREST PIPELINE
# =============================================================================
def build_rf_pipeline(
    features: Sequence[str],
) -> Pipeline:
    """
    Reconstruct the exact fold-safe preprocessing + Random Forest pipeline used
    in Step 7.2.
    """
    categorical = [
        feature
        for feature in features
        if is_categorical_feature(
            feature
        )
    ]

    numeric = [
        feature
        for feature in features
        if not is_categorical_feature(
            feature
        )
    ]

    transformers = []

    if numeric:
        transformers.append(
            (
                "numeric",
                "passthrough",
                numeric,
            )
        )

    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = RandomForestClassifier(
        **DiagnosticConfig.RF_FIXED_PARAMS
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                classifier,
            ),
        ]
    )


def set_pipeline_params(
    pipeline: Pipeline,
    params: Dict[str, Any],
) -> Pipeline:
    """
    Apply the exact fold-specific Step 7.2 GridSearchCV best parameters.
    """
    if not isinstance(
        params,
        dict,
    ):
        raise TypeError(
            "Fold-specific best parameters must be a dictionary."
        )

    pipeline.set_params(
        **params
    )

    return pipeline


# =============================================================================
# SUCCESSFUL STEP 7.2 OUTPUT LOADING
# =============================================================================
def load_outer_best_params() -> Dict[str, Any]:
    require_file(
        DiagnosticConfig.OUTER_PARAMS_JSON,
        "successful Step 7.2 RF outer-fold best-parameter JSON",
    )

    with open(
        DiagnosticConfig.OUTER_PARAMS_JSON,
        "r",
        encoding="utf-8",
    ) as handle:
        outer_params = json.load(
            handle
        )

    for scenario_name in DiagnosticConfig.SCENARIO_ORDER:
        if scenario_name not in outer_params:
            raise RuntimeError(
                "Step 7.2 outer-parameter JSON is missing scenario: "
                f"{scenario_name}"
            )

        for outer_fold in DiagnosticConfig.EXPECTED_FOLDS:
            fold_info = outer_params[
                scenario_name
            ].get(
                str(
                    outer_fold
                )
            )

            if fold_info is None:
                raise RuntimeError(
                    f"Missing Step 7.2 outer-fold parameter record: "
                    f"{scenario_name}, fold {outer_fold}"
                )

            if "best_params" not in fold_info:
                raise RuntimeError(
                    f"Missing best_params: "
                    f"{scenario_name}, fold {outer_fold}"
                )

    log(
        "[QA] Successful Step 7.2 RF outer-fold best parameters loaded: PASS"
    )

    return outer_params


def get_fold_params(
    outer_params: Dict[str, Any],
    scenario_name: str,
    outer_fold: int,
) -> Dict[str, Any]:
    if scenario_name not in outer_params:
        raise KeyError(
            "Missing scenario in Step 7.2 outer-parameter JSON: "
            f"{scenario_name}"
        )

    fold_info = outer_params[
        scenario_name
    ].get(
        str(
            outer_fold
        )
    )

    if not fold_info:
        raise KeyError(
            "Missing outer-fold parameters for "
            f"{scenario_name}, fold {outer_fold}"
        )

    params = fold_info.get(
        "best_params"
    )

    if not params:
        raise KeyError(
            "Missing best_params for "
            f"{scenario_name}, fold {outer_fold}"
        )

    return params


def load_primary_step72_oof() -> pd.DataFrame:
    require_file(
        DiagnosticConfig.PRIMARY_OOF_CSV,
        "successful Step 7.2 RF A-F OOF predictions",
    )

    oof = pd.read_csv(
        DiagnosticConfig.PRIMARY_OOF_CSV
    )

    required = {
        "scenario",
        DiagnosticConfig.SAMPLE_ID_COLUMN,
        DiagnosticConfig.TARGET_COLUMN,
        DiagnosticConfig.FOLD_COLUMN,
        "oof_probability",
    }

    missing = required - set(
        oof.columns
    )

    if missing:
        raise RuntimeError(
            "Step 7.2 OOF file is missing columns: "
            f"{sorted(missing)}"
        )

    for scenario_name in DiagnosticConfig.LEARNING_SCENARIOS:
        subset = oof.loc[
            oof[
                "scenario"
            ]
            == scenario_name
        ]

        if len(
            subset
        ) != DiagnosticConfig.EXPECTED_N_ROWS:
            raise RuntimeError(
                f"Unexpected Step 7.2 OOF row count for {scenario_name}: "
                f"{len(subset)}"
            )

    return oof


# =============================================================================
# PROVENANCE
# =============================================================================
def write_diagnostic_provenance(
    out_dir: Path,
) -> None:
    metadata = {
        "generated_at_utc": utc_now_iso(),
        "diagnostic_step": "Step 7.2A",
        "algorithm": "Random Forest",
        "scientific_role": (
            "Sample-size robustness and held-out permutation-importance "
            "diagnostics; does not replace primary Step 7.2 nested spatial CV."
        ),
        "primary_step72_output_directory": str(
            DiagnosticConfig.STEP72_OUT_DIR
        ),
        "outer_fold_best_parameters": str(
            DiagnosticConfig.OUTER_PARAMS_JSON
        ),
        "primary_oof_predictions": str(
            DiagnosticConfig.PRIMARY_OOF_CSV
        ),
        "model_ready_dataset": str(
            DiagnosticConfig.MODEL_READY_DATASET_CSV
        ),
        "learning_curve_type": (
            "fixed-hyperparameter spatial learning curve"
        ),
        "learning_scenarios": list(
            DiagnosticConfig.LEARNING_SCENARIOS
        ),
        "permutation_scenario": (
            DiagnosticConfig.PERMUTATION_SCENARIO
        ),
        "training_fractions": list(
            DiagnosticConfig.TRAIN_FRACTIONS
        ),
        "subsample_repeats": int(
            DiagnosticConfig.N_REPEATS_SUBSAMPLE
        ),
        "permutation_repeats": int(
            DiagnosticConfig.N_PERMUTATION_REPEATS
        ),
        "rf_fixed_parameters": json_safe(
            DiagnosticConfig.RF_FIXED_PARAMS
        ),
        "full_training_reproduction_check": (
            "At training_fraction=1.0, held-out probabilities for Models A and E "
            "must reproduce the successful Step 7.2 OOF predictions."
        ),
    }

    with open(
        out_dir
        / "Step_7_2A_RF_Diagnostic_Provenance.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            metadata,
            handle,
            indent=2,
        )


# =============================================================================
# SPATIAL STRATIFIED SUBSAMPLING
# =============================================================================
def spatial_stratified_subsample(
    train_df: pd.DataFrame,
    fold_column: str,
    target_column: str,
    fraction: float,
    seed: int,
) -> pd.DataFrame:
    """
    Sample the requested fraction independently within every
    spatial-block x class stratum.

    With 33 observations per class in each spatial block, the Step 7.1A/7.2A
    fractions produce approximately:
        0.25 -> 64 samples
        0.50 -> 128 samples
        0.75 -> 200 samples
        1.00 -> 264 samples

    This preserves all four outer-training blocks and both classes.
    """
    if not 0.0 < float(
        fraction
    ) <= 1.0:
        raise ValueError(
            "Training fraction must lie in (0, 1]."
        )

    if np.isclose(
        fraction,
        1.0,
    ):
        return train_df.copy()

    rng = np.random.default_rng(
        seed
    )

    selected_indices: list[Any] = []

    grouped = train_df.groupby(
        [
            fold_column,
            target_column,
        ],
        sort=True,
    )

    for (
        _fold,
        _label,
    ), group in grouped:
        n_group = len(
            group
        )

        n_select = max(
            1,
            int(
                round(
                    n_group
                    * float(
                        fraction
                    )
                )
            ),
        )

        n_select = min(
            n_select,
            n_group,
        )

        group_indices = group.index.to_numpy()

        chosen = rng.choice(
            group_indices,
            size=n_select,
            replace=False,
        )

        selected_indices.extend(
            chosen.tolist()
        )

    sampled = (
        train_df.loc[
            selected_indices
        ]
        .copy()
        .sort_index()
    )

    return sampled


# =============================================================================
# FEATURE HELPERS
# =============================================================================
def find_feature(
    features: list[str],
    token: str,
) -> str:
    matches = [
        feature
        for feature in features
        if token.lower()
        in feature.lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Expected exactly one feature containing '{token}', "
            f"found: {matches}"
        )

    return matches[
        0
    ]


# =============================================================================
# FULL-TRAINING REPRODUCTION QA
# =============================================================================
def compare_full_training_to_primary_oof(
    full_training_predictions: pd.DataFrame,
    primary_oof: pd.DataFrame,
    out_dir: Path,
) -> pd.DataFrame:
    """
    At training_fraction=1.0, each fold-specific RF model is reconstructed from
    the exact Step 7.2 saved best parameters and trained on the same four blocks.
    Its held-out probabilities should therefore reproduce the successful Step 7.2
    OOF probabilities for Models A and E.
    """
    rows: list[dict[str, Any]] = []

    for scenario_name in DiagnosticConfig.LEARNING_SCENARIOS:
        diagnostic = full_training_predictions.loc[
            full_training_predictions[
                "scenario"
            ]
            == scenario_name
        ].copy()

        reference = primary_oof.loc[
            primary_oof[
                "scenario"
            ]
            == scenario_name,
            [
                DiagnosticConfig.SAMPLE_ID_COLUMN,
                "oof_probability",
            ],
        ].copy()

        reference = reference.rename(
            columns={
                "oof_probability": (
                    "primary_step72_oof_probability"
                )
            }
        )

        merged = diagnostic.merge(
            reference,
            on=DiagnosticConfig.SAMPLE_ID_COLUMN,
            how="inner",
            validate="one_to_one",
        )

        if len(
            merged
        ) != DiagnosticConfig.EXPECTED_N_ROWS:
            raise RuntimeError(
                "Full-training reproduction merge returned "
                f"{len(merged)} rows for {scenario_name}; "
                f"expected {DiagnosticConfig.EXPECTED_N_ROWS}."
            )

        difference = (
            merged[
                "diagnostic_probability"
            ].to_numpy(
                dtype=float
            )
            - merged[
                "primary_step72_oof_probability"
            ].to_numpy(
                dtype=float
            )
        )

        max_abs = float(
            np.max(
                np.abs(
                    difference
                )
            )
        )

        mean_abs = float(
            np.mean(
                np.abs(
                    difference
                )
            )
        )

        passed = bool(
            max_abs
            <= DiagnosticConfig.OOF_REPRODUCTION_TOLERANCE
        )

        rows.append(
            {
                "scenario": scenario_name,
                "model": DiagnosticConfig.MODEL_SHORT_LABELS[
                    scenario_name
                ],
                "n_samples_compared": int(
                    len(
                        merged
                    )
                ),
                "maximum_absolute_probability_difference": max_abs,
                "mean_absolute_probability_difference": mean_abs,
                "tolerance": float(
                    DiagnosticConfig.OOF_REPRODUCTION_TOLERANCE
                ),
                "reproduction_pass": passed,
            }
        )

        if not passed:
            raise RuntimeError(
                "Step 7.2A full-training reproduction check failed for "
                f"{scenario_name}.\n"
                f"Maximum absolute probability difference = {max_abs:.12g}\n"
                f"Tolerance = "
                f"{DiagnosticConfig.OOF_REPRODUCTION_TOLERANCE:.12g}\n"
                "This indicates that the Step 7.2A RF pipeline or saved "
                "fold-specific parameters do not exactly match the successful "
                "Step 7.2 experiment."
            )

    qa_df = pd.DataFrame(
        rows
    )

    qa_df.to_csv(
        out_dir
        / "RF_7_2A_Full_Training_Reproduction_QA.csv",
        index=False,
        float_format="%.12g",
    )

    log(
        "[QA] Step 7.2A full-training A/E held-out probabilities reproduce "
        "successful Step 7.2 OOF predictions: PASS"
    )

    return qa_df


# =============================================================================
# 1. SPATIAL LEARNING CURVES
# =============================================================================
def run_spatial_learning_curves(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
    outer_params: Dict[str, Any],
    primary_oof: pd.DataFrame,
    out_dir: Path,
):
    rows: list[dict[str, Any]] = []
    full_training_oof_rows: list[dict[str, Any]] = []

    for scenario_name in DiagnosticConfig.LEARNING_SCENARIOS:
        features = list(
            scenarios[
                scenario_name
            ][
                "features"
            ]
        )

        for outer_fold in DiagnosticConfig.EXPECTED_FOLDS:
            train_df = df.loc[
                df[
                    DiagnosticConfig.FOLD_COLUMN
                ]
                != outer_fold
            ].copy()

            valid_df = df.loc[
                df[
                    DiagnosticConfig.FOLD_COLUMN
                ]
                == outer_fold
            ].copy()

            if train_df[
                DiagnosticConfig.FOLD_COLUMN
            ].nunique() != 4:
                raise RuntimeError(
                    f"{scenario_name}, outer fold {outer_fold}: "
                    "expected four outer-training spatial blocks."
                )

            if valid_df[
                DiagnosticConfig.FOLD_COLUMN
            ].nunique() != 1:
                raise RuntimeError(
                    f"{scenario_name}, outer fold {outer_fold}: "
                    "validation data must contain exactly one spatial block."
                )

            best_params = get_fold_params(
                outer_params=outer_params,
                scenario_name=scenario_name,
                outer_fold=outer_fold,
            )

            X_valid = valid_df[
                features
            ]
            y_valid = valid_df[
                DiagnosticConfig.TARGET_COLUMN
            ].to_numpy(
                dtype=int
            )

            for (
                fraction_index,
                fraction,
            ) in enumerate(
                DiagnosticConfig.TRAIN_FRACTIONS
            ):
                n_repeats = (
                    1
                    if np.isclose(
                        fraction,
                        1.0,
                    )
                    else DiagnosticConfig.N_REPEATS_SUBSAMPLE
                )

                for repeat in range(
                    n_repeats
                ):
                    seed = (
                        DiagnosticConfig.RANDOM_STATE
                        + 10000
                        * DiagnosticConfig.SCENARIO_ORDER.index(
                            scenario_name
                        )
                        + 1000
                        * outer_fold
                        + 100
                        * fraction_index
                        + repeat
                    )

                    sampled_train = spatial_stratified_subsample(
                        train_df=train_df,
                        fold_column=DiagnosticConfig.FOLD_COLUMN,
                        target_column=DiagnosticConfig.TARGET_COLUMN,
                        fraction=float(
                            fraction
                        ),
                        seed=seed,
                    )

                    # Strict spatial QA.
                    if sampled_train[
                        DiagnosticConfig.FOLD_COLUMN
                    ].nunique() != 4:
                        raise RuntimeError(
                            "Learning-curve sample lost a training "
                            "spatial block: "
                            f"{scenario_name}, outer fold {outer_fold}, "
                            f"fraction {fraction}"
                        )

                    if sampled_train[
                        DiagnosticConfig.TARGET_COLUMN
                    ].nunique() != 2:
                        raise RuntimeError(
                            "Learning-curve sample lost one response class."
                        )

                    block_class = pd.crosstab(
                        sampled_train[
                            DiagnosticConfig.FOLD_COLUMN
                        ],
                        sampled_train[
                            DiagnosticConfig.TARGET_COLUMN
                        ],
                    )

                    if set(
                        block_class.columns
                    ) != {
                        0,
                        1,
                    }:
                        raise RuntimeError(
                            "Learning-curve subsample does not contain "
                            "both classes in every retained block."
                        )

                    pipeline = build_rf_pipeline(
                        features
                    )

                    set_pipeline_params(
                        pipeline,
                        best_params,
                    )

                    X_train = sampled_train[
                        features
                    ]

                    y_train = sampled_train[
                        DiagnosticConfig.TARGET_COLUMN
                    ].to_numpy(
                        dtype=int
                    )

                    pipeline.fit(
                        X_train,
                        y_train,
                    )

                    train_prob = (
                        pipeline.predict_proba(
                            X_train
                        )[
                            :,
                            1,
                        ]
                    )

                    valid_prob = (
                        pipeline.predict_proba(
                            X_valid
                        )[
                            :,
                            1,
                        ]
                    )

                    rows.append(
                        {
                            "algorithm": "Random Forest",
                            "scenario": scenario_name,
                            "model": DiagnosticConfig.MODEL_SHORT_LABELS[
                                scenario_name
                            ],
                            "outer_fold": int(
                                outer_fold
                            ),
                            "training_fraction": float(
                                fraction
                            ),
                            "repeat": int(
                                repeat
                            ),
                            "n_train": int(
                                len(
                                    sampled_train
                                )
                            ),
                            "n_train_positive": int(
                                y_train.sum()
                            ),
                            "n_train_negative": int(
                                len(
                                    y_train
                                )
                                - y_train.sum()
                            ),
                            "training_auc": float(
                                roc_auc_score(
                                    y_train,
                                    train_prob,
                                )
                            ),
                            "training_pr_auc": float(
                                average_precision_score(
                                    y_train,
                                    train_prob,
                                )
                            ),
                            "heldout_spatial_auc": float(
                                roc_auc_score(
                                    y_valid,
                                    valid_prob,
                                )
                            ),
                            "heldout_spatial_pr_auc": float(
                                average_precision_score(
                                    y_valid,
                                    valid_prob,
                                )
                            ),
                        }
                    )

                    # Save exact full-training held-out probabilities for
                    # reproducibility QA against successful Step 7.2.
                    if np.isclose(
                        fraction,
                        1.0,
                    ):
                        for (
                            sample_id,
                            probability,
                        ) in zip(
                            valid_df[
                                DiagnosticConfig.SAMPLE_ID_COLUMN
                            ].tolist(),
                            valid_prob.tolist(),
                        ):
                            full_training_oof_rows.append(
                                {
                                    "scenario": scenario_name,
                                    "outer_fold": int(
                                        outer_fold
                                    ),
                                    DiagnosticConfig.SAMPLE_ID_COLUMN: (
                                        sample_id
                                    ),
                                    "diagnostic_probability": float(
                                        probability
                                    ),
                                }
                            )

                    del (
                        pipeline,
                        sampled_train,
                        X_train,
                        y_train,
                        train_prob,
                        valid_prob,
                    )
                    gc.collect()

    raw_df = pd.DataFrame(
        rows
    )

    raw_df.to_csv(
        out_dir
        / "RF_7_2A_Spatial_Learning_Curve_Raw.csv",
        index=False,
        float_format="%.10f",
    )

    # Average repeated random subsamples within each held-out spatial fold first.
    # The five fold means are then the independent spatial units summarized.
    fold_summary = (
        raw_df.groupby(
            [
                "scenario",
                "model",
                "outer_fold",
                "training_fraction",
            ],
            as_index=False,
        )
        .agg(
            n_train_mean=(
                "n_train",
                "mean",
            ),
            training_auc_mean=(
                "training_auc",
                "mean",
            ),
            training_pr_auc_mean=(
                "training_pr_auc",
                "mean",
            ),
            heldout_spatial_auc_mean=(
                "heldout_spatial_auc",
                "mean",
            ),
            heldout_spatial_pr_auc_mean=(
                "heldout_spatial_pr_auc",
                "mean",
            ),
        )
    )

    fold_summary.to_csv(
        out_dir
        / "RF_7_2A_Spatial_Learning_Curve_Fold_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    summary = (
        fold_summary.groupby(
            [
                "scenario",
                "model",
                "training_fraction",
            ],
            as_index=False,
        )
        .agg(
            n_train_mean=(
                "n_train_mean",
                "mean",
            ),
            training_auc_mean=(
                "training_auc_mean",
                "mean",
            ),
            training_auc_sd=(
                "training_auc_mean",
                "std",
            ),
            training_pr_auc_mean=(
                "training_pr_auc_mean",
                "mean",
            ),
            training_pr_auc_sd=(
                "training_pr_auc_mean",
                "std",
            ),
            heldout_auc_mean=(
                "heldout_spatial_auc_mean",
                "mean",
            ),
            heldout_auc_sd=(
                "heldout_spatial_auc_mean",
                "std",
            ),
            heldout_auc_min=(
                "heldout_spatial_auc_mean",
                "min",
            ),
            heldout_auc_max=(
                "heldout_spatial_auc_mean",
                "max",
            ),
            heldout_pr_auc_mean=(
                "heldout_spatial_pr_auc_mean",
                "mean",
            ),
            heldout_pr_auc_sd=(
                "heldout_spatial_pr_auc_mean",
                "std",
            ),
            heldout_pr_auc_min=(
                "heldout_spatial_pr_auc_mean",
                "min",
            ),
            heldout_pr_auc_max=(
                "heldout_spatial_pr_auc_mean",
                "max",
            ),
            n_outer_folds=(
                "outer_fold",
                "nunique",
            ),
        )
    )

    summary.to_csv(
        out_dir
        / "RF_7_2A_Spatial_Learning_Curve_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    # -------------------------------------------------------------------------
    # Figure — same scientific visualization as Step 7.1A.
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=DiagnosticConfig.FIGSIZE
    )

    for scenario_name in DiagnosticConfig.LEARNING_SCENARIOS:
        subset = summary.loc[
            summary[
                "scenario"
            ]
            == scenario_name
        ].sort_values(
            "n_train_mean"
        )

        x = subset[
            "n_train_mean"
        ].to_numpy(
            dtype=float
        )

        y = subset[
            "heldout_auc_mean"
        ].to_numpy(
            dtype=float
        )

        sd = subset[
            "heldout_auc_sd"
        ].fillna(
            0.0
        ).to_numpy(
            dtype=float
        )

        label = DiagnosticConfig.MODEL_SHORT_LABELS[
            scenario_name
        ]

        ax.plot(
            x,
            y,
            marker="o",
            linewidth=1.5,
            label=label,
        )

        ax.fill_between(
            x,
            y - sd,
            y + sd,
            alpha=0.15,
        )

    ax.set_xlabel(
        "Number of outer-training samples"
    )
    ax.set_ylabel(
        "Held-out spatial ROC-AUC"
    )
    ax.set_ylim(
        0.5,
        1.01,
    )
    ax.grid(
        True,
        alpha=0.25,
    )
    ax.legend(
        frameon=False
    )
    fig.tight_layout()

    fig.savefig(
        out_dir
        / "RF_7_2A_Spatial_Learning_Curves.png",
        dpi=DiagnosticConfig.FIG_DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

    # -------------------------------------------------------------------------
    # Exact full-training reproduction QA.
    # -------------------------------------------------------------------------
    full_training_predictions = pd.DataFrame(
        full_training_oof_rows
    )

    full_training_predictions.to_csv(
        out_dir
        / "RF_7_2A_Full_Training_Heldout_Probabilities.csv",
        index=False,
        float_format="%.12g",
    )

    reproduction_qa = compare_full_training_to_primary_oof(
        full_training_predictions=full_training_predictions,
        primary_oof=primary_oof,
        out_dir=out_dir,
    )

    return (
        raw_df,
        fold_summary,
        summary,
        reproduction_qa,
    )


# =============================================================================
# 2. HELD-OUT PERMUTATION IMPORTANCE
# =============================================================================
def run_heldout_permutation_importance(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
    outer_params: Dict[str, Any],
    out_dir: Path,
):
    scenario_name = (
        DiagnosticConfig.PERMUTATION_SCENARIO
    )

    features = list(
        scenarios[
            scenario_name
        ][
            "features"
        ]
    )

    hkfi_feature = find_feature(
        features,
        "hkfi",
    )

    sndr_feature = find_feature(
        features,
        "sndr",
    )

    individual_rows: list[dict[str, Any]] = []
    grouped_rows: list[dict[str, Any]] = []

    for outer_fold in DiagnosticConfig.EXPECTED_FOLDS:
        train_df = df.loc[
            df[
                DiagnosticConfig.FOLD_COLUMN
            ]
            != outer_fold
        ].copy()

        valid_df = df.loc[
            df[
                DiagnosticConfig.FOLD_COLUMN
            ]
            == outer_fold
        ].copy()

        best_params = get_fold_params(
            outer_params=outer_params,
            scenario_name=scenario_name,
            outer_fold=outer_fold,
        )

        pipeline = build_rf_pipeline(
            features
        )

        set_pipeline_params(
            pipeline,
            best_params,
        )

        X_train = train_df[
            features
        ]

        y_train = train_df[
            DiagnosticConfig.TARGET_COLUMN
        ].to_numpy(
            dtype=int
        )

        X_valid = valid_df[
            features
        ]

        y_valid = valid_df[
            DiagnosticConfig.TARGET_COLUMN
        ].to_numpy(
            dtype=int
        )

        pipeline.fit(
            X_train,
            y_train,
        )

        baseline_prob = (
            pipeline.predict_proba(
                X_valid
            )[
                :,
                1,
            ]
        )

        baseline_auc = float(
            roc_auc_score(
                y_valid,
                baseline_prob,
            )
        )

        # ---------------------------------------------------------------------
        # Individual held-out permutation importance.
        #
        # sklearn.inspection.permutation_importance receives the full pipeline,
        # so original input columns are permuted before fold-fitted preprocessing.
        # ---------------------------------------------------------------------
        result = permutation_importance(
            estimator=pipeline,
            X=X_valid,
            y=y_valid,
            scoring="roc_auc",
            n_repeats=(
                DiagnosticConfig.N_PERMUTATION_REPEATS
            ),
            random_state=(
                DiagnosticConfig.RANDOM_STATE
                + outer_fold
            ),
            n_jobs=1,
        )

        for (
            feature_index,
            feature,
        ) in enumerate(
            features
        ):
            for (
                repeat,
                decrease,
            ) in enumerate(
                result.importances[
                    feature_index
                ]
            ):
                individual_rows.append(
                    {
                        "algorithm": "Random Forest",
                        "outer_fold": int(
                            outer_fold
                        ),
                        "feature": feature,
                        "repeat": int(
                            repeat
                        ),
                        "baseline_auc": baseline_auc,
                        "auc_decrease": float(
                            decrease
                        ),
                    }
                )

        # ---------------------------------------------------------------------
        # Joint HKFI + SNDR permutation.
        #
        # The same row permutation is applied to both variables, preserving
        # their internal pairing while breaking their alignment with the
        # held-out response and the remaining predictors.
        # ---------------------------------------------------------------------
        rng = np.random.default_rng(
            DiagnosticConfig.RANDOM_STATE
            + 1000
            + outer_fold
        )

        for repeat in range(
            DiagnosticConfig.N_PERMUTATION_REPEATS
        ):
            permutation = rng.permutation(
                len(
                    X_valid
                )
            )

            X_permuted = X_valid.copy()

            for feature in (
                hkfi_feature,
                sndr_feature,
            ):
                X_permuted[
                    feature
                ] = (
                    X_valid.iloc[
                        permutation
                    ][
                        feature
                    ]
                    .to_numpy()
                )

            permuted_prob = (
                pipeline.predict_proba(
                    X_permuted
                )[
                    :,
                    1,
                ]
            )

            permuted_auc = float(
                roc_auc_score(
                    y_valid,
                    permuted_prob,
                )
            )

            grouped_rows.append(
                {
                    "algorithm": "Random Forest",
                    "outer_fold": int(
                        outer_fold
                    ),
                    "feature_group": (
                        "HKFI + SNDR (joint permutation)"
                    ),
                    "repeat": int(
                        repeat
                    ),
                    "baseline_auc": baseline_auc,
                    "permuted_auc": permuted_auc,
                    "auc_decrease": (
                        baseline_auc
                        - permuted_auc
                    ),
                }
            )

        del (
            pipeline,
            result,
            train_df,
            valid_df,
            X_train,
            X_valid,
            baseline_prob,
        )
        gc.collect()

    individual = pd.DataFrame(
        individual_rows
    )

    grouped = pd.DataFrame(
        grouped_rows
    )

    individual.to_csv(
        out_dir
        / "RF_7_2A_Model_E_Heldout_Permutation_Importance_Raw.csv",
        index=False,
        float_format="%.10f",
    )

    grouped.to_csv(
        out_dir
        / "RF_7_2A_Model_E_Grouped_HKFI_SNDR_Permutation_Raw.csv",
        index=False,
        float_format="%.10f",
    )

    # Average permutation repeats within each outer fold first.
    individual_fold = (
        individual.groupby(
            [
                "outer_fold",
                "feature",
            ],
            as_index=False,
        )
        .agg(
            auc_decrease_mean=(
                "auc_decrease",
                "mean",
            )
        )
    )

    individual_fold.to_csv(
        out_dir
        / "RF_7_2A_Model_E_Heldout_Permutation_Importance_Fold_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    individual_summary = (
        individual_fold.groupby(
            "feature",
            as_index=False,
        )
        .agg(
            mean_auc_decrease=(
                "auc_decrease_mean",
                "mean",
            ),
            sd_across_outer_folds=(
                "auc_decrease_mean",
                "std",
            ),
            min_across_outer_folds=(
                "auc_decrease_mean",
                "min",
            ),
            max_across_outer_folds=(
                "auc_decrease_mean",
                "max",
            ),
        )
        .sort_values(
            "mean_auc_decrease",
            ascending=False,
        )
    )

    grouped_fold = (
        grouped.groupby(
            [
                "outer_fold",
                "feature_group",
            ],
            as_index=False,
        )
        .agg(
            auc_decrease_mean=(
                "auc_decrease",
                "mean",
            )
        )
    )

    grouped_fold.to_csv(
        out_dir
        / "RF_7_2A_Model_E_Grouped_HKFI_SNDR_Permutation_Fold_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    grouped_summary = (
        grouped_fold.groupby(
            "feature_group",
            as_index=False,
        )
        .agg(
            mean_auc_decrease=(
                "auc_decrease_mean",
                "mean",
            ),
            sd_across_outer_folds=(
                "auc_decrease_mean",
                "std",
            ),
            min_across_outer_folds=(
                "auc_decrease_mean",
                "min",
            ),
            max_across_outer_folds=(
                "auc_decrease_mean",
                "max",
            ),
        )
    )

    individual_summary.to_csv(
        out_dir
        / "RF_7_2A_Model_E_Heldout_Permutation_Importance_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    grouped_summary.to_csv(
        out_dir
        / "RF_7_2A_Model_E_Grouped_HKFI_SNDR_Permutation_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    # -------------------------------------------------------------------------
    # Figure — same as Step 7.1A, now using RF held-out permutation results.
    # -------------------------------------------------------------------------
    plot_df = (
        individual_summary.head(
            15
        )
        .sort_values(
            "mean_auc_decrease"
        )
    )

    fig, ax = plt.subplots(
        figsize=(
            6.0,
            5.2,
        )
    )

    ax.barh(
        plot_df[
            "feature"
        ],
        plot_df[
            "mean_auc_decrease"
        ],
        xerr=plot_df[
            "sd_across_outer_folds"
        ].fillna(
            0.0
        ),
        capsize=2,
    )

    ax.axvline(
        0.0,
        linewidth=0.8,
    )

    ax.set_xlabel(
        "Decrease in held-out ROC-AUC after permutation"
    )

    ax.set_ylabel(
        "Predictor"
    )

    ax.grid(
        True,
        axis="x",
        alpha=0.25,
    )

    fig.tight_layout()

    fig.savefig(
        out_dir
        / "RF_7_2A_Model_E_Heldout_Permutation_Importance.png",
        dpi=DiagnosticConfig.FIG_DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

    return (
        individual_summary,
        grouped_summary,
    )


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    DiagnosticConfig.OUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    log(
        "=" * 96
    )
    log(
        "STEP 7.2A — RANDOM FOREST SPATIAL LEARNING CURVES "
        "+ HELD-OUT PERMUTATION IMPORTANCE"
    )
    log(
        "=" * 96
    )

    write_diagnostic_provenance(
        DiagnosticConfig.OUT_DIR
    )

    # 1. Validate frozen predictor definitions and dataset.
    scenarios = validate_scenario_manifests()

    df = load_and_validate_model_ready_dataset(
        scenarios
    )

    # 2. Load exact successful Step 7.2 RF fold-specific hyperparameters.
    outer_params = load_outer_best_params()

    # 3. Load successful Step 7.2 RF OOF predictions for reproduction QA.
    primary_oof = load_primary_step72_oof()

    # 4. Spatial learning curves.
    log(
        "[7.2A] Running RF spatial learning curves for Models A and E..."
    )

    (
        _,
        _,
        learning_summary,
        reproduction_qa,
    ) = run_spatial_learning_curves(
        df=df,
        scenarios=scenarios,
        outer_params=outer_params,
        primary_oof=primary_oof,
        out_dir=DiagnosticConfig.OUT_DIR,
    )

    # 5. Held-out permutation importance.
    log(
        "[7.2A] Running RF held-out permutation importance for Model E..."
    )

    (
        individual_summary,
        grouped_summary,
    ) = run_heldout_permutation_importance(
        df=df,
        scenarios=scenarios,
        outer_params=outer_params,
        out_dir=DiagnosticConfig.OUT_DIR,
    )

    log(
        ""
    )
    log(
        "=" * 96
    )
    log(
        "STEP 7.2A COMPLETED SUCCESSFULLY"
    )
    log(
        "=" * 96
    )
    log(
        f"[OUT] {DiagnosticConfig.OUT_DIR}"
    )

    print(
        "\nFull-training reproduction QA:"
    )
    print(
        reproduction_qa.to_string(
            index=False
        )
    )

    print(
        "\nLearning-curve summary:"
    )
    print(
        learning_summary.to_string(
            index=False
        )
    )

    print(
        "\nTop held-out permutation importance:"
    )
    print(
        individual_summary.head(
            10
        ).to_string(
            index=False
        )
    )

    print(
        "\nGrouped HKFI + SNDR importance:"
    )
    print(
        grouped_summary.to_string(
            index=False
        )
    )


if __name__ == "__main__":
    main()


# STEP 7.2B — RANDOM FOREST HKFI WEIGHTING SENSITIVITY

In [ ]:
# =============================================================================
# STEP 7.2B — RANDOM FOREST HKFI WEIGHTING SENSITIVITY
# =============================================================================
#
# Targeted Random Forest cross-algorithm counterpart to Step 7.1B.
#
# Step 7.1B already completed the full XGBoost HKFI weighting sensitivity and
# map-sensitivity analysis. Step 7.2B therefore propagates a focused subset of
# authoritative, pre-specified Step 5.3 named HKFI weighting scenarios through
# the exact successful Step 7.2 Random Forest Model E nested spatial-CV contract
# to test whether the weighting-robustness conclusion is algorithm-dependent.
#
# Scientific rules
# ----------------
# 1. The exact frozen Step 6.4 330-row dataset and five spatial folds are used.
# 2. Only HKFI changes between sensitivity scenarios.
# 3. The exact successful Step 7.2 48-combination RF grid is used.
# 4. The primary equal-component scenario must reproduce successful Step 7.2
#    Model E OOF predictions.
# 5. The 66-case Step 5.3 simplex is not propagated through ML.
# 6. HKFI weights are not selected post hoc by maximizing validation AUC.
# 7. Full-domain RF map sensitivity is deliberately not repeated because the
#    complete map-sensitivity experiment was already performed for the primary
#    XGBoost framework in Step 7.1B.
#
# Execution
# ---------
# Heavy scenarios run in separate Python processes and are checkpointed.
# Save this file as Step_7_2B_RF_HKFI_Weight_Sensitivity.py and run the saved
# .py file. Pasting the complete script into a Jupyter cell is not recommended
# because worker processes need a real script path.
# =============================================================================

from __future__ import annotations

import argparse
import gc
import json
import os
import re
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Sequence, Tuple

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Step 6.1 frozen predictor definitions and raster stack.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"
    MASTER_STACK_TIF = (
        STEP61_DIR
        / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
    )
    COMMON_AF_MASK_TIF = (
        STEP61_DIR / "common_A_to_F_valid_mask_32645_12p5m_byte.tif"
    )

    # Step 6.4 frozen modelling contract.
    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    MODEL_READY_DATASET_CSV = STEP64_DIR / "model_ready_dataset_A_to_F.csv"
    STEP64_MODEL_CONFIG_JSON = STEP64_DIR / "model_configuration_manifest.json"

    # Outputs.
    OUT_DIR = RESULTS_DIR / "Step-7.2-RF-Models_A_to_F"
    MODEL_DIR = OUT_DIR / "final_models"
    MAP_DIR = OUT_DIR / "susceptibility_maps"
    SEARCH_DIR = OUT_DIR / "grid_search_details"

    OOF_PREDICTIONS_CSV = OUT_DIR / "RF_A_to_F_OOF_predictions.csv"
    FOLD_METRICS_CSV = OUT_DIR / "RF_A_to_F_outer_fold_metrics.csv"
    SUMMARY_METRICS_CSV = OUT_DIR / "RF_A_to_F_summary_metrics.csv"
    AUC_COMPARISONS_CSV = OUT_DIR / "RF_A_to_F_paired_AUC_comparisons.csv"
    OUTER_PARAMS_JSON = OUT_DIR / "RF_A_to_F_outer_fold_best_parameters.json"
    FINAL_PARAMS_JSON = OUT_DIR / "RF_A_to_F_final_full_data_best_parameters.json"
    INNER_SEARCH_CSV = SEARCH_DIR / "RF_A_to_F_nested_inner_grid_results.csv"
    FINAL_SEARCH_CSV = SEARCH_DIR / "RF_A_to_F_full_data_grid_results.csv"
    ROC_FIG_PNG = OUT_DIR / "RF_A_to_F_OOF_ROC_Curves.png"
    ROC_FIG_PDF = OUT_DIR / "RF_A_to_F_OOF_ROC_Curves.pdf"
    FOLD_AUC_FIG_PNG = OUT_DIR / "RF_A_to_F_Fold_AUC.png"
    FOLD_AUC_FIG_PDF = OUT_DIR / "RF_A_to_F_Fold_AUC.pdf"
    QA_REPORT_TXT = OUT_DIR / "Step_7_2_RF_A_to_F_QA_Report.txt"
    METADATA_JSON = OUT_DIR / "Step_7_2_RF_A_to_F_Metadata.json"

    # Manuscript-ready Excel workbook containing the compact A-F performance
    # table, paired AUC comparisons, fold-level AUCs and ensemble summary.
    MANUSCRIPT_TABLES_XLSX = (
        OUT_DIR / "RF_A_to_F_Manuscript_Performance_Tables.xlsx"
    )

    # Cartographic overlays used by the established Step 5.3 map style.
    STUDY_SHP = BASE / "Study_Area.shp"
    ROAD_SHP = BASE / "NTD_Road.shp"
    VISUALIZATION_DIR = MAP_DIR / "visualization"

    # Primary A-F model retained for scientific interpretation.
    PRIMARY_VISUALIZATION_SCENARIO = "E_physics_guided"

    # Old-style static + integrated ensemble analogue for the revised A-F design.
    #
    # Old workflow:
    #   final = 0.30 * context/static + 0.70 * integrated
    #   where integrated support exists; otherwise retain static background.
    #
    # Revised workflow:
    #   final = 0.30 * Model A + 0.70 * Model E
    #   where Model E predictors are available; otherwise retain Model A.
    #
    # The 0.70 weight is fixed a priori to reproduce the earlier ensemble logic.
    # It is NOT tuned on the OOF predictions.
    GENERATE_AE_ENSEMBLE_LSM = True
    ENSEMBLE_STATIC_SCENARIO = "A_static_baseline"
    ENSEMBLE_ENHANCED_SCENARIO = "E_physics_guided"
    ENSEMBLE_ENHANCED_WEIGHT = 0.70

    ENSEMBLE_STATIC_FULL_DOMAIN_TIF = (
        MAP_DIR / "A-static-baseline_RF_probability_full_available_domain.tif"
    )
    ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF = (
        MAP_DIR / "E-physics-guided_RF_probability_full_available_domain.tif"
    )
    ENSEMBLE_PROBABILITY_TIF = (
        MAP_DIR / "RF_Ensemble_A30_E70_probability_full_study_domain.tif"
    )
    ENSEMBLE_SUPPORT_TIF = (
        MAP_DIR / "RF_Ensemble_A30_E70_information_support.tif"
    )
    ENSEMBLE_OOF_PREDICTIONS_CSV = (
        OUT_DIR / "RF_Ensemble_A30_E70_OOF_predictions.csv"
    )
    ENSEMBLE_OOF_METRICS_JSON = (
        OUT_DIR / "RF_Ensemble_A30_E70_OOF_metrics.json"
    )

    # Step 5.3-style map/histogram settings.
    # Publication-size figures.
    # The two requested maps are explicitly controlled independently so that
    # later changes to other figures cannot accidentally alter their dimensions.
    ENSEMBLE_MAP_FIGSIZE = (4, 6)
    SUPPORT_MAP_FIGSIZE = (4, 6)

    # Other visualization products.
    MAP_FIGSIZE = (4, 4)
    HIST_FIGSIZE = (4, 4)
    AREA_PERCENT_FIGSIZE = (4, 4)

    # Enlarged cartographic text for the 4 x 4 inch figure.
    DMS_TICK_FONTSIZE = 7  # Reduced from 9 to prevent latitude label overlap
    LEGEND_FONTSIZE = 7
    LEGEND_TITLE_FONTSIZE = 8
    COLORBAR_TICK_FONTSIZE = 8
    COLORBAR_LABEL_FONTSIZE = 9
    NORTH_ARROW_FONTSIZE = 8  # Slightly reduced for balance
    SCALEBAR_FONTSIZE = 7

    CAX_SIZE = "5%"
    CAX_PAD = 0.01
    FIG_DPI = 900
    DISPLAY_BUFFER_FRACTION = 0.00
    LONGITUDE_DMS_TICK_COUNT = 4
    RASTER_DISPLAY_INTERPOLATION = "nearest"
    SHOW_LSM_FIGURES = True
    SAVE_LSM_FIGURES = True

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"
    CATEGORICAL_FEATURE_ALIASES = {"landcover", "landuse", "land_use"}

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    MODEL_DESCRIPTIONS = {
        "A_static_baseline": "Static baseline",
        "B_raw_deformation": "Static + DV+",
        "C_hkfi_only": "Static + HKFI",
        "D_sndr_only": "Static + SNDR",
        "E_physics_guided": "Static + HKFI + SNDR",
        "F_untransformed_sources": "Static + DV+ + TWI + SM",
    }

    # Random-Forest-specific 48-combination grid.
    #
    # The modelling principle is identical to Step 7.1:
    #   - hyperparameters are selected only inside the four outer-training blocks;
    #   - the untouched outer block is used once for unbiased evaluation;
    #   - the same grid is applied to Models A-F.
    #
    # Grid size:
    #   2 n_estimators
    # x 3 max_depth
    # x 2 min_samples_leaf
    # x 2 max_features
    # x 2 criterion
    # = 48 combinations.
    PARAM_GRID = {
        "model__n_estimators": [200, 500],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 2],
        "model__max_features": ["sqrt", 0.5],
        "model__criterion": ["gini", "entropy"],
    }

    # Fixed Random Forest settings shared by A-F.
    #
    # The dataset is already balanced 1:1, therefore class_weight is left as
    # None rather than imposing an additional weighting scheme.
    RF_FIXED_PARAMS = {
        "bootstrap": True,
        "class_weight": None,
        "random_state": 42,
        "n_jobs": 1,
    }

    GRIDSEARCH_N_JOBS = 1  # Avoid nested Random Forest / GridSearchCV parallelism
    GRIDSEARCH_VERBOSE = 2
    RANDOM_STATE = 42
    BINARY_THRESHOLD = 0.50

    # Spatial-block cluster bootstrap.
    N_CLUSTER_BOOTSTRAP = 5000
    CI_LEVEL = 0.95

    # Complete execution controls.
    #
    # True  -> after nested spatial-CV evaluation, tune and fit the final
    #          full-data deployment models for A-F.
    # False -> stop after leakage-safe nested spatial OOF evaluation.
    RUN_FINAL_DEPLOYMENT_TUNING = True

    # True  -> generate A-F full-domain probability rasters from the final
    #          deployment models. Requires RUN_FINAL_DEPLOYMENT_TUNING=True.
    GENERATE_RASTER_MAPS = True

    # True  -> create the publication-style primary Model E landslide
    #          susceptibility map and probability histogram.
    GENERATE_PRIMARY_LSM_VISUALIZATION = True
    RASTER_BLOCK_ROWS = 512
    OUTPUT_NODATA = -9999.0

    # Keep nested-CV fold estimators only if needed for later diagnostics.
    SAVE_OUTER_FOLD_MODELS = False


def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json(path: Path) -> Dict[str, Any]:
    require_file(path, "JSON file")
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value



def scenario_prefix(scenario_name: str) -> str:
    return str(scenario_name).replace("_", "-")

def normalize_feature_name(name: str) -> str:
    """Normalize a feature name for robust alias matching."""
    return re.sub(r"[^a-z0-9_]+", "_", str(name).strip().lower()).strip("_")


def is_categorical_feature(name: str) -> bool:
    """Identify categorical predictors using conservative land-cover aliases."""
    normalized = normalize_feature_name(name)
    compact = normalized.replace("_", "")
    aliases = {alias.lower() for alias in Config.CATEGORICAL_FEATURE_ALIASES}
    compact_aliases = {alias.replace("_", "") for alias in aliases}
    return normalized in aliases or compact in compact_aliases


def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """Cross-check Step 6.1 and frozen Step 6.4 A–F definitions."""
    step61 = load_json(Config.SCENARIO_MANIFEST_JSON).get("scenarios", {})
    step64 = load_json(Config.STEP64_MODEL_CONFIG_JSON).get("scenarios", {})

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                f"Step 6.1 scenario manifest is missing {scenario_name}."
            )
        if scenario_name not in step64:
            raise RuntimeError(
                f"Step 6.4 frozen model manifest is missing {scenario_name}."
            )

        features_61 = list(step61[scenario_name].get("features", []))
        features_64 = list(step64[scenario_name].get("features", []))

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any("irs" in feature.lower() for feature in features_61):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; IRS must not be an ML predictor."
            )

    log("[QA] Step 6.1 and Step 6.4 A–F scenario definitions match: PASS")
    return step64


def load_and_validate_model_ready_dataset(
    scenarios: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    """Load the primary frozen 330-row Step 6.4 ML dataset with strict QA."""
    require_file(
        Config.MODEL_READY_DATASET_CSV,
        "Step 6.4 primary frozen model-ready ML dataset",
    )

    df = pd.read_csv(Config.MODEL_READY_DATASET_CSV)

    required_metadata = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ]

    all_features: List[str] = []
    for scenario_name in Config.SCENARIO_ORDER:
        for feature in scenarios[scenario_name]["features"]:
            if feature not in all_features:
                all_features.append(feature)

    missing = [
        column
        for column in required_metadata + all_features
        if column not in df.columns
    ]
    if missing:
        raise RuntimeError(f"Model-ready dataset is missing columns: {missing}")

    if len(df) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Expected {Config.EXPECTED_N_ROWS} rows; found {len(df)}. "
            "No silent row dropping is allowed."
        )

    if df[Config.SAMPLE_ID_COLUMN].isna().any():
        raise RuntimeError("Missing sample_id values found.")
    if df[Config.SAMPLE_ID_COLUMN].duplicated().any():
        raise RuntimeError("Duplicated sample_id values found.")

    df[Config.TARGET_COLUMN] = pd.to_numeric(
        df[Config.TARGET_COLUMN], errors="raise"
    ).astype(int)
    df[Config.FOLD_COLUMN] = pd.to_numeric(
        df[Config.FOLD_COLUMN], errors="raise"
    ).astype(int)
    df[Config.BLOCK_COLUMN] = pd.to_numeric(
        df[Config.BLOCK_COLUMN], errors="raise"
    ).astype(int)

    class_counts = df[Config.TARGET_COLUMN].value_counts().sort_index().to_dict()
    expected_counts = {
        0: Config.EXPECTED_N_NEGATIVE,
        1: Config.EXPECTED_N_POSITIVE,
    }
    if class_counts != expected_counts:
        raise RuntimeError(
            f"Unexpected class counts: {class_counts}; expected {expected_counts}."
        )

    folds = sorted(df[Config.FOLD_COLUMN].unique().tolist())
    if folds != Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected frozen folds {Config.EXPECTED_FOLDS}; found {folds}."
        )

    if not np.array_equal(
        df[Config.FOLD_COLUMN].to_numpy(),
        df[Config.BLOCK_COLUMN].to_numpy(),
    ):
        raise RuntimeError("cv_fold and spatial_block are not identical.")

    # Every frozen block must contain exactly 33 positives and 33 pseudo-absences.
    block_class = pd.crosstab(
        df[Config.FOLD_COLUMN],
        df[Config.TARGET_COLUMN],
    )
    for required_class in (0, 1):
        if required_class not in block_class.columns:
            raise RuntimeError(
                f"Frozen fold table is missing class {required_class}."
            )

    expected_per_class_per_fold = Config.EXPECTED_N_POSITIVE // len(Config.EXPECTED_FOLDS)
    if expected_per_class_per_fold * len(Config.EXPECTED_FOLDS) != Config.EXPECTED_N_POSITIVE:
        raise RuntimeError(
            "Expected positive count is not divisible evenly across the frozen folds."
        )

    for fold in Config.EXPECTED_FOLDS:
        if fold not in block_class.index:
            raise RuntimeError(f"Missing fold {fold}.")
        n0 = int(block_class.loc[fold, 0])
        n1 = int(block_class.loc[fold, 1])
        if n0 != expected_per_class_per_fold or n1 != expected_per_class_per_fold:
            raise RuntimeError(
                f"Fold {fold} violates the frozen Step 6.4 contract: "
                f"negative={n0}, positive={n1}; expected "
                f"{expected_per_class_per_fold}+{expected_per_class_per_fold}=66."
            )

    log(
        "[QA] Frozen fold balance: each fold contains "
        f"{expected_per_class_per_fold} positives + "
        f"{expected_per_class_per_fold} pseudo-absences: PASS"
    )

    # Step 6.4 should already guarantee complete predictors; verify again.
    predictor_values = df[all_features].apply(pd.to_numeric, errors="coerce")
    if not np.isfinite(predictor_values.to_numpy(dtype="float64")).all():
        bad = np.argwhere(
            ~np.isfinite(predictor_values.to_numpy(dtype="float64"))
        )
        examples = [
            (
                str(df.iloc[row_i][Config.SAMPLE_ID_COLUMN]),
                all_features[col_i],
            )
            for row_i, col_i in bad[:20]
        ]
        raise RuntimeError(
            "Missing/non-finite predictors remain in the model-ready dataset. "
            f"Examples: {examples}"
        )

    irs_columns = [
        column for column in df.columns if "irs" in str(column).lower()
    ]
    if irs_columns:
        raise RuntimeError(
            f"IRS-related columns found in ML dataset: {irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError("tpi_s25 found; only tpi_s5 is allowed.")

    log(
        "[QA] Model-ready dataset: "
        f"rows={len(df)}, positive={class_counts[1]}, negative={class_counts[0]}, "
        f"folds={folds}: PASS"
    )
    return df


def build_rf_pipeline(features: Sequence[str]) -> Pipeline:
    """Create fold-safe preprocessing plus Random Forest pipeline."""
    categorical = [
        feature for feature in features if is_categorical_feature(feature)
    ]
    numeric = [
        feature for feature in features if not is_categorical_feature(feature)
    ]

    transformers = []
    if numeric:
        transformers.append(("numeric", "passthrough", numeric))
    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = RandomForestClassifier(**Config.RF_FIXED_PARAMS)

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", classifier),
        ]
    )


def make_inner_predefined_split(outer_train_df: pd.DataFrame) -> PredefinedSplit:
    """
    Build a four-fold spatial inner CV from the four blocks remaining after one
    outer block has been held out.
    """
    blocks = sorted(outer_train_df[Config.FOLD_COLUMN].unique().tolist())
    if len(blocks) != 4:
        raise RuntimeError(
            f"Nested CV expected four inner blocks; found {blocks}."
        )

    block_to_test_fold = {
        block: inner_index for inner_index, block in enumerate(blocks)
    }
    test_fold = outer_train_df[Config.FOLD_COLUMN].map(block_to_test_fold).to_numpy()
    return PredefinedSplit(test_fold=test_fold)


def make_full_data_predefined_split(df: pd.DataFrame) -> PredefinedSplit:
    """Use the frozen five blocks for final deployment-model tuning."""
    test_fold = df[Config.FOLD_COLUMN].to_numpy(dtype=int) - 1
    return PredefinedSplit(test_fold=test_fold)


def classification_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = Config.BINARY_THRESHOLD,
) -> Dict[str, Any]:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def cluster_bootstrap_auc_ci(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> Tuple[float, float, int]:
    """Cluster bootstrap AUC by resampling whole spatial blocks."""
    rng = np.random.default_rng(seed)
    unique_blocks = np.unique(blocks)
    values: List[float] = []

    indices_by_block = {
        block: np.flatnonzero(blocks == block) for block in unique_blocks
    }

    for i in range(n_bootstrap):
        if (i + 1) % 1000 == 0:
            log(f"  Bootstrap progress: {i + 1}/{n_bootstrap}")
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(unique_blocks),
            replace=True,
        )
        sampled_indices = np.concatenate(
            [indices_by_block[block] for block in sampled_blocks]
        )
        y_b = y_true[sampled_indices]
        p_b = y_prob[sampled_indices]

        if np.unique(y_b).size < 2:
            continue

        values.append(float(roc_auc_score(y_b, p_b)))

    if not values:
        return float("nan"), float("nan"), 0

    alpha = (1.0 - ci_level) / 2.0
    low, high = np.quantile(values, [alpha, 1.0 - alpha])
    return float(low), float(high), len(values)


def paired_cluster_bootstrap_auc_difference(
    y_true: np.ndarray,
    prob_left: np.ndarray,
    prob_right: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> Tuple[float, float, float, int]:
    """
    Paired spatial-block bootstrap for AUC(left) - AUC(right), preserving the
    identical samples/folds used by all A–F models.
    """
    observed = float(
        roc_auc_score(y_true, prob_left) - roc_auc_score(y_true, prob_right)
    )

    rng = np.random.default_rng(seed)
    unique_blocks = np.unique(blocks)
    indices_by_block = {
        block: np.flatnonzero(blocks == block) for block in unique_blocks
    }
    differences: List[float] = []

    for i in range(n_bootstrap):
        if (i + 1) % 1000 == 0:
            log(f"  Bootstrap progress: {i + 1}/{n_bootstrap}")
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(unique_blocks),
            replace=True,
        )
        sampled_indices = np.concatenate(
            [indices_by_block[block] for block in sampled_blocks]
        )
        y_b = y_true[sampled_indices]

        if np.unique(y_b).size < 2:
            continue

        difference = (
            roc_auc_score(y_b, prob_left[sampled_indices])
            - roc_auc_score(y_b, prob_right[sampled_indices])
        )
        differences.append(float(difference))

    if not differences:
        return observed, float("nan"), float("nan"), 0

    alpha = (1.0 - ci_level) / 2.0
    low, high = np.quantile(differences, [alpha, 1.0 - alpha])
    return observed, float(low), float(high), len(differences)


def normalize_grid_results(
    cv_results: Dict[str, Any],
    scenario_name: str,
    search_stage: str,
    outer_fold: int | None,
) -> pd.DataFrame:
    """Convert GridSearchCV.cv_results_ to a compact audit table."""
    frame = pd.DataFrame(cv_results)

    keep = [
        column
        for column in frame.columns
        if column.startswith("param_")
        or column
        in {
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "mean_fit_time",
            "std_fit_time",
        }
    ]

    frame = frame[keep].copy()
    frame.insert(0, "scenario", scenario_name)
    frame.insert(1, "model", Config.MODEL_SHORT_LABELS[scenario_name])
    frame.insert(2, "search_stage", search_stage)
    frame.insert(3, "outer_fold", outer_fold)
    return frame


def run_nested_spatial_cv_for_scenario(
    df: pd.DataFrame,
    scenario_name: str,
    features: Sequence[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any], pd.DataFrame]:
    """Run five outer spatial folds with four-block inner spatial grid search."""
    log("")
    log("=" * 96)
    log(
        f"[NESTED-CV] {Config.MODEL_SHORT_LABELS[scenario_name]} — "
        f"{Config.MODEL_DESCRIPTIONS[scenario_name]} ({len(features)} predictors)"
    )
    log("=" * 96)

    oof_probability = np.full(len(df), np.nan, dtype="float64")
    oof_predicted = np.full(len(df), -1, dtype="int16")

    fold_rows: List[Dict[str, Any]] = []
    best_params: Dict[str, Any] = {}
    search_tables: List[pd.DataFrame] = []

    for outer_fold in Config.EXPECTED_FOLDS:
        outer_train_mask = df[Config.FOLD_COLUMN].to_numpy() != outer_fold
        outer_valid_mask = df[Config.FOLD_COLUMN].to_numpy() == outer_fold

        train_df = df.loc[outer_train_mask].copy()
        valid_df = df.loc[outer_valid_mask].copy()

        if train_df[Config.FOLD_COLUMN].nunique() != 4:
            raise RuntimeError(
                f"Outer fold {outer_fold}: expected four training blocks."
            )
        if valid_df[Config.FOLD_COLUMN].nunique() != 1:
            raise RuntimeError(
                f"Outer fold {outer_fold}: validation must contain one block only."
            )

        X_train = train_df[list(features)]
        y_train = train_df[Config.TARGET_COLUMN].to_numpy(dtype=int)
        X_valid = valid_df[list(features)]
        y_valid = valid_df[Config.TARGET_COLUMN].to_numpy(dtype=int)

        inner_cv = make_inner_predefined_split(train_df)
        pipeline = build_rf_pipeline(features)

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=Config.PARAM_GRID,
            scoring="roc_auc",
            cv=inner_cv,
            refit=True,
            n_jobs=Config.GRIDSEARCH_N_JOBS,
            verbose=Config.GRIDSEARCH_VERBOSE,
            return_train_score=False,
            error_score="raise",
        )

        log(
            f"[OUTER {outer_fold}] Tuning on blocks "
            f"{sorted(train_df[Config.FOLD_COLUMN].unique().tolist())}; "
            f"holding out block {outer_fold}."
        )
        search.fit(X_train, y_train)

        probability = search.best_estimator_.predict_proba(X_valid)[:, 1]
        predicted = (probability >= Config.BINARY_THRESHOLD).astype(int)

        valid_indices = np.flatnonzero(outer_valid_mask)
        oof_probability[valid_indices] = probability
        oof_predicted[valid_indices] = predicted

        metrics = classification_metrics(y_valid, probability)

        row = {
            "scenario": scenario_name,
            "model": Config.MODEL_SHORT_LABELS[scenario_name],
            "description": Config.MODEL_DESCRIPTIONS[scenario_name],
            "outer_fold": int(outer_fold),
            "n_predictors": int(len(features)),
            "n_train": int(len(train_df)),
            "n_validation": int(len(valid_df)),
            "n_train_positive": int(y_train.sum()),
            "n_train_negative": int(len(y_train) - y_train.sum()),
            "n_validation_positive": int(y_valid.sum()),
            "n_validation_negative": int(len(y_valid) - y_valid.sum()),
            "inner_best_mean_auc": float(search.best_score_),
            **metrics,
        }
        fold_rows.append(row)

        best_params[str(outer_fold)] = {
            "inner_best_mean_auc": float(search.best_score_),
            "best_params": json_safe(search.best_params_),
        }

        search_tables.append(
            normalize_grid_results(
                search.cv_results_,
                scenario_name=scenario_name,
                search_stage="nested_inner",
                outer_fold=outer_fold,
            )
        )

        if Config.SAVE_OUTER_FOLD_MODELS:
            fold_model_path = (
                Config.MODEL_DIR
                / f"{scenario_prefix(scenario_name)}_outer-fold-{outer_fold}.joblib"
            )
            joblib.dump(search.best_estimator_, fold_model_path)

        log(
            f"[OUTER {outer_fold}] inner best AUC={search.best_score_:.4f}; "
            f"held-out AUC={metrics['roc_auc']:.4f}; "
            f"F1={metrics['f1']:.4f}"
        )

    if not np.isfinite(oof_probability).all():
        missing_count = int((~np.isfinite(oof_probability)).sum())
        raise RuntimeError(
            f"{scenario_name}: {missing_count} samples did not receive OOF predictions."
        )
    if (oof_predicted < 0).any():
        raise RuntimeError(f"{scenario_name}: incomplete OOF class predictions.")

    oof_df = df[
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
        ]
    ].copy()
    oof_df.insert(0, "scenario", scenario_name)
    oof_df.insert(1, "model", Config.MODEL_SHORT_LABELS[scenario_name])
    oof_df.insert(2, "description", Config.MODEL_DESCRIPTIONS[scenario_name])
    oof_df["oof_probability"] = oof_probability
    oof_df["oof_predicted_class_0p5"] = oof_predicted

    fold_df = pd.DataFrame(fold_rows)
    search_df = pd.concat(search_tables, ignore_index=True)

    return oof_df, fold_df, best_params, search_df


def read_stack_band_map(stack_path: Path) -> Dict[str, int]:
    require_file(stack_path, "Step 6.1 master predictor stack")
    with rasterio.open(stack_path) as dataset:
        descriptions = list(dataset.descriptions)

    band_map: Dict[str, int] = {}
    for band_index, description in enumerate(descriptions, start=1):
        if description:
            band_map[str(description)] = band_index

    return band_map


def blend_static_and_enhanced_probability_rasters(
    static_raster_path: Path,
    enhanced_raster_path: Path,
    output_path: Path,
    enhanced_weight: float,
) -> Path:
    """
    Apply the same fixed A30/E70 operational ensemble logic to the Random Forest A-F framework.

    Where both models are valid:
        ensemble = (1-w) * Model A + w * Model E

    Where only Model A is valid:
        retain Model A as the susceptibility background.

    Where only Model E is valid:
        retain Model E.

    Outside both valid domains:
        nodata.

    This is intentionally different from averaging all A-F configurations.
    Models B-F are ablation experiments and should not be indiscriminately
    averaged into the final susceptibility map.
    """
    if not 0.0 <= enhanced_weight <= 1.0:
        raise ValueError(
            "ENSEMBLE_ENHANCED_WEIGHT must lie between 0 and 1."
        )

    static_weight = 1.0 - enhanced_weight

    require_file(
        static_raster_path,
        "Model A full-domain probability raster",
    )
    require_file(
        enhanced_raster_path,
        "Model E full-domain probability raster",
    )

    with rasterio.open(static_raster_path) as static_ds, rasterio.open(
        enhanced_raster_path
    ) as enhanced_ds:
        aligned = (
            static_ds.width == enhanced_ds.width
            and static_ds.height == enhanced_ds.height
            and static_ds.transform == enhanced_ds.transform
            and static_ds.crs == enhanced_ds.crs
        )
        if not aligned:
            raise RuntimeError(
                "Model A and Model E full-domain rasters are not aligned."
            )

        profile = static_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        n_static_background = 0
        n_blended = 0
        n_enhanced_only = 0

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_ds:
            for row_start in range(
                0,
                static_ds.height,
                Config.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    static_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=static_ds.width,
                    height=block_height,
                )

                static_arr = static_ds.read(
                    1,
                    window=window,
                    masked=True,
                )
                enhanced_arr = enhanced_ds.read(
                    1,
                    window=window,
                    masked=True,
                )

                static_values = np.asarray(
                    static_arr.filled(np.nan),
                    dtype="float64",
                )
                enhanced_values = np.asarray(
                    enhanced_arr.filled(np.nan),
                    dtype="float64",
                )

                static_valid = np.isfinite(static_values)
                enhanced_valid = np.isfinite(enhanced_values)

                both = static_valid & enhanced_valid
                static_only = static_valid & ~enhanced_valid
                enhanced_only = enhanced_valid & ~static_valid

                output = np.full(
                    static_values.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                output[static_only] = static_values[
                    static_only
                ].astype("float32")

                output[enhanced_only] = enhanced_values[
                    enhanced_only
                ].astype("float32")

                output[both] = (
                    static_weight * static_values[both]
                    + enhanced_weight * enhanced_values[both]
                ).astype("float32")

                n_static_background += int(static_only.sum())
                n_blended += int(both.sum())
                n_enhanced_only += int(enhanced_only.sum())

                output_ds.write(
                    output,
                    1,
                    window=window,
                )

            output_ds.set_band_description(
                1,
                "Random Forest A30-E70 ensemble susceptibility probability",
            )
            output_ds.update_tags(
                model="Random Forest",
                ensemble="Model_A_static_plus_Model_E_physics_guided",
                static_weight=f"{static_weight:.6f}",
                enhanced_weight=f"{enhanced_weight:.6f}",
                blend_rule=(
                    "weighted blend where both are valid; "
                    "retain static background elsewhere"
                ),
                validation_note=(
                    "The fixed A/E ensemble is separately evaluated from nested "
                    "OOF probabilities. A-F ablation metrics remain the primary "
                    "model-comparison results."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[ENSEMBLE-MAP] Saved A30/E70 ensemble: "
        f"blended={n_blended:,}, "
        f"static-background={n_static_background:,}, "
        f"enhanced-only={n_enhanced_only:,} -> "
        f"{output_path}"
    )

    return output_path


# =============================================================================
# STEP 7.2B ADAPTER / QA HELPERS
# =============================================================================

class _Step72RFAdapter:
    """Expose the embedded successful Step 7.2 RF contract using the same
    attribute style used by the original Step 7.2B diagnostic code."""
    Config = Config

    validate_scenario_manifests = staticmethod(validate_scenario_manifests)
    load_and_validate_model_ready_dataset = staticmethod(
        load_and_validate_model_ready_dataset
    )
    run_nested_spatial_cv_for_scenario = staticmethod(
        run_nested_spatial_cv_for_scenario
    )
    classification_metrics = staticmethod(classification_metrics)
    cluster_bootstrap_auc_ci = staticmethod(cluster_bootstrap_auc_ci)
    paired_cluster_bootstrap_auc_difference = staticmethod(
        paired_cluster_bootstrap_auc_difference
    )
    build_rf_pipeline = staticmethod(build_rf_pipeline)
    make_full_data_predefined_split = staticmethod(
        make_full_data_predefined_split
    )
    blend_static_and_enhanced_probability_rasters = staticmethod(
        blend_static_and_enhanced_probability_rasters
    )


def load_step72_module(explicit_path: Path | None = None):
    """Return the embedded, frozen Step 7.2 Random Forest modelling contract."""
    if explicit_path is not None:
        raise RuntimeError(
            "Step 7.2B is self-contained. BASE_SCRIPT must remain None so the "
            "embedded successful Step 7.2 RF contract is used."
        )
    return _Step72RFAdapter


def all_scenario_features(
    scenarios: Dict[str, Dict[str, Any]],
) -> list[str]:
    features: list[str] = []
    for scenario_name in Config.SCENARIO_ORDER:
        for feature in scenarios[scenario_name]["features"]:
            if feature not in features:
                features.append(feature)
    return features


def validate_dataset_exactly_matches_primary(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
) -> None:
    """Verify exact row identity and predictor values against the frozen
    Step 6.4 primary dataset."""
    reference = load_and_validate_model_ready_dataset(scenarios)

    if len(df) != len(reference):
        raise RuntimeError("Dataset row count differs from the primary Step 6.4 dataset.")

    if not np.array_equal(
        df[Config.SAMPLE_ID_COLUMN].astype(str).to_numpy(),
        reference[Config.SAMPLE_ID_COLUMN].astype(str).to_numpy(),
    ):
        raise RuntimeError("Sample IDs/order differ from the primary Step 6.4 dataset.")

    columns = [
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
        *all_scenario_features(scenarios),
    ]

    for column in columns:
        left = df[column]
        right = reference[column]

        if (
            pd.api.types.is_numeric_dtype(left)
            and pd.api.types.is_numeric_dtype(right)
        ):
            same = np.allclose(
                left.to_numpy(dtype=float),
                right.to_numpy(dtype=float),
                rtol=0.0,
                atol=1.0e-12,
                equal_nan=True,
            )
        else:
            same = np.array_equal(
                left.astype(str).to_numpy(),
                right.astype(str).to_numpy(),
            )

        if not same:
            raise RuntimeError(
                f"Dataset does not exactly match the primary Step 6.4 dataset "
                f"in column: {column}"
            )


def validate_equal_rows_and_folds(
    reference_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
) -> None:
    if len(reference_df) != len(candidate_df):
        raise RuntimeError("HKFI sensitivity dataset changed row count.")

    for column in (
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ):
        if not np.array_equal(
            reference_df[column].astype(str).to_numpy(),
            candidate_df[column].astype(str).to_numpy(),
        ):
            raise RuntimeError(
                f"HKFI sensitivity dataset changed frozen rows/folds in {column}."
            )


def validate_only_allowed_column_changes(
    reference_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
    allowed_changed_columns: Sequence[str],
    columns_to_compare: Sequence[str],
    atol: float,
) -> None:
    allowed = set(allowed_changed_columns)

    for column in columns_to_compare:
        if column in allowed:
            continue

        left = reference_df[column]
        right = candidate_df[column]

        if (
            pd.api.types.is_numeric_dtype(left)
            and pd.api.types.is_numeric_dtype(right)
        ):
            same = np.allclose(
                left.to_numpy(dtype=float),
                right.to_numpy(dtype=float),
                rtol=0.0,
                atol=atol,
                equal_nan=True,
            )
        else:
            same = np.array_equal(
                left.astype(str).to_numpy(),
                right.astype(str).to_numpy(),
            )

        if not same:
            raise RuntimeError(
                f"Unexpected non-HKFI change detected in column: {column}"
            )


def assert_oof_reproduces_primary(
    oof_df: pd.DataFrame,
    scenario_name: str,
    tolerance: float,
) -> Dict[str, Any]:
    """The primary equal-component rerun must reproduce successful Step 7.2
    Model E OOF probabilities."""
    require_file(
        Config.OOF_PREDICTIONS_CSV,
        "successful Step 7.2 RF A-F OOF predictions",
    )

    reference_all = pd.read_csv(Config.OOF_PREDICTIONS_CSV)
    reference = reference_all.loc[
        reference_all["scenario"] == scenario_name,
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            "oof_probability",
        ],
    ].copy()

    if len(reference) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Successful Step 7.2 OOF file contains {len(reference)} rows for "
            f"{scenario_name}; expected {Config.EXPECTED_N_ROWS}."
        )

    candidate = oof_df[
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            "oof_probability",
        ]
    ].copy()

    merged = candidate.merge(
        reference,
        on=Config.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=("_candidate", "_reference"),
        validate="one_to_one",
    )

    if len(merged) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Primary OOF reproduction merge returned {len(merged)} rows; "
            f"expected {Config.EXPECTED_N_ROWS}."
        )

    for column in (Config.TARGET_COLUMN, Config.FOLD_COLUMN):
        if not np.array_equal(
            merged[f"{column}_candidate"].to_numpy(),
            merged[f"{column}_reference"].to_numpy(),
        ):
            raise RuntimeError(
                f"Primary OOF reproduction metadata mismatch in {column}."
            )

    difference = (
        merged["oof_probability_candidate"].to_numpy(dtype=float)
        - merged["oof_probability_reference"].to_numpy(dtype=float)
    )

    max_abs = float(np.max(np.abs(difference)))
    mean_abs = float(np.mean(np.abs(difference)))
    passed = bool(max_abs <= tolerance)

    if not passed:
        raise RuntimeError(
            "Primary equal-component Step 7.2B RF rerun did not reproduce the "
            "successful Step 7.2 Model E OOF predictions.\n"
            f"Maximum absolute probability difference={max_abs:.12g}\n"
            f"Tolerance={tolerance:.12g}"
        )

    return {
        "oof_reproduction_pass": passed,
        "oof_reproduction_maximum_absolute_probability_difference": max_abs,
        "oof_reproduction_mean_absolute_probability_difference": mean_abs,
        "oof_reproduction_tolerance": float(tolerance),
    }


def paired_auc_difference_from_oof(
    base,
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    label: str,
    seed: int,
) -> Dict[str, Any]:
    left = left_df[
        [
            base.Config.SAMPLE_ID_COLUMN,
            base.Config.TARGET_COLUMN,
            base.Config.FOLD_COLUMN,
            "oof_probability",
        ]
    ].rename(columns={"oof_probability": "probability_left"})

    right = right_df[
        [
            base.Config.SAMPLE_ID_COLUMN,
            base.Config.TARGET_COLUMN,
            base.Config.FOLD_COLUMN,
            "oof_probability",
        ]
    ].rename(columns={"oof_probability": "probability_right"})

    merged = left.merge(
        right,
        on=base.Config.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=("_left", "_right"),
        validate="one_to_one",
    )

    if len(merged) != base.Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Paired comparison '{label}' returned {len(merged)} rows; "
            f"expected {base.Config.EXPECTED_N_ROWS}."
        )

    for column in (base.Config.TARGET_COLUMN, base.Config.FOLD_COLUMN):
        if not np.array_equal(
            merged[f"{column}_left"].to_numpy(),
            merged[f"{column}_right"].to_numpy(),
        ):
            raise RuntimeError(f"Paired OOF mismatch in {column}: {label}")

    y_true = merged[
        f"{base.Config.TARGET_COLUMN}_left"
    ].to_numpy(dtype=int)
    blocks = merged[
        f"{base.Config.FOLD_COLUMN}_left"
    ].to_numpy(dtype=int)

    observed, low, high, n_valid = (
        base.paired_cluster_bootstrap_auc_difference(
            y_true=y_true,
            prob_left=merged["probability_left"].to_numpy(dtype=float),
            prob_right=merged["probability_right"].to_numpy(dtype=float),
            blocks=blocks,
            n_bootstrap=base.Config.N_CLUSTER_BOOTSTRAP,
            seed=seed,
            ci_level=base.Config.CI_LEVEL,
        )
    )

    return {
        "comparison": label,
        "auc_difference_left_minus_right": observed,
        "ci_low": low,
        "ci_high": high,
        "valid_bootstrap_iterations": int(n_valid),
    }


def write_diagnostic_provenance(
    out_dir: Path,
    diagnostic_name: str,
    details: Dict[str, Any],
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "diagnostic_name": diagnostic_name,
        "algorithm": "Random Forest",
        "successful_step72_output_directory": str(Config.OUT_DIR),
        "successful_step72_oof": str(Config.OOF_PREDICTIONS_CSV),
        "rf_parameter_grid": json_safe(Config.PARAM_GRID),
        "rf_fixed_parameters": json_safe(Config.RF_FIXED_PARAMS),
        **details,
    }

    with open(
        out_dir / f"{diagnostic_name}_Provenance.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(json_safe(metadata), handle, indent=2)


# =============================================================================
# CONFIGURATION
# =============================================================================
class SensitivityConfig:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # The RF Step 7.2 contract is embedded directly in this self-contained file.
    BASE_SCRIPT: Path | None = None

    # Authoritative Step 5.3 weight-sensitivity outputs.
    STEP53_HKFI_DIR = RESULTS_DIR / "Step-5.3-HKFI"
    STEP53_WEIGHT_DIR = STEP53_HKFI_DIR / "weight_sensitivity"
    STEP53_WEIGHT_MANIFEST_CSV = (
        STEP53_WEIGHT_DIR / "hkfi_weight_scenario_manifest.csv"
    )
    STEP53_NAMED_SENSITIVITY_CSV = (
        STEP53_WEIGHT_DIR / "hkfi_named_weight_sensitivity.csv"
    )
    STEP53_SIMPLEX_SENSITIVITY_CSV = (
        STEP53_WEIGHT_DIR / "hkfi_simplex_weight_sensitivity.csv"
    )

    # Primary scenario name used by authoritative Step 5.3.
    REFERENCE_VARIANT = "primary_equal_component"

    # Model E only: Static12 + HKFI + SNDR.
    SCENARIO_NAME = "E_physics_guided"

    # Targeted cross-algorithm robustness design.
    #
    # Step 7.1B already completed the full XGBoost HKFI weighting sensitivity,
    # including all named weighting scenarios and full-domain map sensitivity.
    # Step 7.2B therefore uses a focused Random Forest replication to test whether
    # the main weighting-robustness conclusion is algorithm-dependent.
    #
    # Four pre-specified scenarios are retained:
    #   1. primary_equal_component -> a priori reference
    #   2. kinematic_emphasis      -> stronger kinematic contribution
    #   3. twi_emphasis            -> stronger TWI contribution
    #   4. sm_emphasis             -> stronger soil-moisture contribution
    #
    # No RF-optimal weight is selected post hoc from these results.
    SCENARIOS_TO_RUN: tuple[str, ...] | None = (
        "primary_equal_component",
        "kinematic_emphasis",
        "twi_emphasis",
        "sm_emphasis",
    )

    # The 66-case simplex is deliberately NOT propagated through ML.
    RUN_SIMPLEX_THROUGH_ML = False

    # Nested-CV and map controls.
    RUN_NESTED_CV = True

    # Performance-only cross-algorithm robustness analysis.
    # Full HKFI map sensitivity was already completed in Step 7.1B using the
    # primary XGBoost framework and is not duplicated here.
    GENERATE_MAP_SENSITIVITY = False

    RESUME_FROM_CHECKPOINTS = True

    # Reproducibility tolerances.
    OOF_REPRODUCTION_TOLERANCE = 1.0e-8
    PRIMARY_RASTER_SAMPLE_TOLERANCE = 5.0e-6
    NON_HKFI_EQUALITY_ATOL = 1.0e-10

    # Output settings.
    RANDOM_STATE = 42171
    FIG_DPI = 300
    RASTER_BLOCK_ROWS = 256

    # Worker/process output.
    WORKER_DIRNAME = "process_isolated_workers"
    LOG_DIRNAME = "logs"


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


def normalize_text(value: Any) -> str:
    return str(value).strip()


def parse_bool(value: Any) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"1", "true", "yes", "y", "t"}:
        return True
    if text in {"0", "false", "no", "n", "f", "", "nan", "none"}:
        return False
    raise ValueError(f"Cannot interpret as boolean: {value!r}")


def sanitize_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def resolve_script_path() -> Path:
    """
    Resolve this saved Python script for process-isolated worker launches.

    Jupyter launcher use is recommended. If the complete script is pasted into
    a notebook cell, __file__ may refer to ipykernel_launcher.py, so the script
    searches the current working directory for its canonical filename.
    """
    file_value = globals().get("__file__")
    if file_value:
        candidate = Path(str(file_value)).resolve()
        if candidate.exists() and candidate.name.lower() != "ipykernel_launcher.py":
            return candidate

    candidate = (
        Path.cwd()
        / "Step_7_2B_RF_HKFI_Weight_Sensitivity.py"
    )
    if candidate.exists():
        return candidate.resolve()

    raise RuntimeError(
        "Could not resolve the saved Step 7.2B Python script. "
        "Keep the .py file in the same folder as the launcher notebook."
    )


# =============================================================================
# STEP 5.3 MANIFEST LOADING
# =============================================================================
def load_step53_weight_manifest() -> pd.DataFrame:
    """
    Read the authoritative named-weight scenario manifest produced by Step 5.3.

    No new JSON manifest is required.
    """
    path = SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV
    require_file(path, "Step 5.3 HKFI weight scenario manifest")

    manifest = pd.read_csv(path)
    required = [
        "scenario",
        "scenario_purpose",
        "weight_K",
        "weight_T",
        "weight_S",
        "norm_raster",
        "is_primary",
        "use_for_downstream_spatial_CV",
    ]
    missing = [column for column in required if column not in manifest.columns]
    if missing:
        raise RuntimeError(
            f"Step 5.3 weight scenario manifest is missing columns: {missing}\n"
            f"Found columns: {manifest.columns.tolist()}"
        )

    manifest = manifest.copy()
    manifest["scenario"] = manifest["scenario"].map(normalize_text)
    manifest["scenario_purpose"] = manifest["scenario_purpose"].map(normalize_text)
    manifest["is_primary"] = manifest["is_primary"].map(parse_bool)
    manifest["use_for_downstream_spatial_CV"] = (
        manifest["use_for_downstream_spatial_CV"].map(parse_bool)
    )

    for column in ("weight_K", "weight_T", "weight_S"):
        manifest[column] = pd.to_numeric(manifest[column], errors="raise")

    if manifest["scenario"].duplicated().any():
        duplicates = manifest.loc[
            manifest["scenario"].duplicated(keep=False), "scenario"
        ].tolist()
        raise RuntimeError(f"Duplicated Step 5.3 scenarios found: {duplicates}")

    if SensitivityConfig.REFERENCE_VARIANT not in set(manifest["scenario"]):
        raise RuntimeError(
            "The Step 5.3 manifest does not contain the required primary "
            f"scenario '{SensitivityConfig.REFERENCE_VARIANT}'."
        )

    primary_rows = manifest.loc[manifest["is_primary"]]
    if len(primary_rows) != 1:
        raise RuntimeError(
            "Step 5.3 manifest must contain exactly one is_primary=True row. "
            f"Found {len(primary_rows)}."
        )

    primary_name = str(primary_rows.iloc[0]["scenario"])
    if primary_name != SensitivityConfig.REFERENCE_VARIANT:
        raise RuntimeError(
            "Configured primary scenario does not match Step 5.3 manifest: "
            f"configured={SensitivityConfig.REFERENCE_VARIANT}, "
            f"manifest={primary_name}"
        )

    weight_sum = (
        manifest["weight_K"]
        + manifest["weight_T"]
        + manifest["weight_S"]
    )
    if not np.allclose(weight_sum.to_numpy(dtype=float), 1.0, atol=1.0e-8):
        bad = manifest.loc[
            ~np.isclose(weight_sum, 1.0, atol=1.0e-8),
            ["scenario", "weight_K", "weight_T", "weight_S"],
        ]
        raise RuntimeError(
            "One or more Step 5.3 named scenarios do not sum to 1:\n"
            + bad.to_string(index=False)
        )

    # Resolve raster paths. Step 5.3 writes absolute paths, but relative paths
    # are also supported relative to the manifest directory.
    resolved_paths = []
    for value in manifest["norm_raster"]:
        candidate = Path(str(value))
        if not candidate.is_absolute():
            candidate = path.parent / candidate
        resolved_paths.append(candidate)
    manifest["norm_raster_path"] = resolved_paths

    flagged = manifest.loc[manifest["use_for_downstream_spatial_CV"]].copy()
    if flagged.empty:
        raise RuntimeError(
            "No Step 5.3 scenarios are flagged use_for_downstream_spatial_CV=True."
        )

    if SensitivityConfig.SCENARIOS_TO_RUN is None:
        selected_names = flagged["scenario"].tolist()
    else:
        requested = list(SensitivityConfig.SCENARIOS_TO_RUN)
        unknown = [name for name in requested if name not in set(manifest["scenario"])]
        if unknown:
            raise RuntimeError(
                f"Configured SCENARIOS_TO_RUN contains unknown scenarios: {unknown}"
            )
        not_flagged = [
            name
            for name in requested
            if not bool(
                manifest.loc[
                    manifest["scenario"] == name,
                    "use_for_downstream_spatial_CV",
                ].iloc[0]
            )
        ]
        if not_flagged:
            raise RuntimeError(
                "These requested scenarios are not flagged for downstream spatial CV "
                f"in Step 5.3: {not_flagged}"
            )
        selected_names = requested

    if SensitivityConfig.REFERENCE_VARIANT not in selected_names:
        selected_names = [SensitivityConfig.REFERENCE_VARIANT] + selected_names

    # Primary always first; preserve Step 5.3 order for the remaining scenarios.
    ordered_names = [SensitivityConfig.REFERENCE_VARIANT] + [
        name for name in selected_names
        if name != SensitivityConfig.REFERENCE_VARIANT
    ]

    selected = (
        manifest.set_index("scenario")
        .loc[ordered_names]
        .reset_index()
    )

    for _, row in selected.iterrows():
        require_file(
            Path(row["norm_raster_path"]),
            f"normalized HKFI raster for scenario {row['scenario']}",
        )

    return selected


def load_optional_step53_named_summary() -> pd.DataFrame:
    path = SensitivityConfig.STEP53_NAMED_SENSITIVITY_CSV
    if not path.exists():
        return pd.DataFrame()
    try:
        frame = pd.read_csv(path)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return pd.DataFrame()

    if "scenario" not in frame.columns:
        log(
            f"[WARN] {path.name} has no 'scenario' column; "
            "upstream HKFI metrics will not be merged."
        )
        return pd.DataFrame()

    frame = frame.copy()
    frame["scenario"] = frame["scenario"].astype(str).str.strip()
    return frame


# =============================================================================
# DATASET CONSTRUCTION FROM STEP 5.3 RASTERS
# =============================================================================
def find_hkfi_feature(features: list[str]) -> str:
    matches = [feature for feature in features if "hkfi" in feature.lower()]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one HKFI feature in Model E; found {matches}"
        )
    return matches[0]


def assert_raster_aligned_to_master(
    base,
    raster_path: Path,
) -> dict[str, Any]:
    require_file(base.Config.MASTER_STACK_TIF, "Step 6.1 master predictor stack")
    require_file(raster_path, "Step 5.3 scenario HKFI raster")

    with rasterio.open(base.Config.MASTER_STACK_TIF) as master, rasterio.open(
        raster_path
    ) as scenario:
        aligned = (
            master.width == scenario.width
            and master.height == scenario.height
            and master.transform == scenario.transform
            and master.crs == scenario.crs
        )
        if not aligned:
            raise RuntimeError(
                "Step 5.3 HKFI scenario raster is not exactly aligned with the "
                "Step 6.1 master predictor stack:\n"
                f"Scenario raster: {raster_path}"
            )

        return {
            "width": int(scenario.width),
            "height": int(scenario.height),
            "crs": str(scenario.crs),
            "transform": tuple(scenario.transform),
            "nodata": scenario.nodata,
        }


def sample_raster_at_frozen_samples(
    base,
    raster_path: Path,
    primary_df: pd.DataFrame,
) -> np.ndarray:
    """
    Sample a Step 5.3 normalized HKFI raster at the exact frozen Step 6.4
    sample coordinates.

    The Step 6.4 model-ready dataset contains x/y coordinates in EPSG:32645.
    Exact raster-grid alignment is verified before sampling.
    """
    assert_raster_aligned_to_master(base, raster_path)

    required_columns = ["x", "y"]
    missing = [column for column in required_columns if column not in primary_df.columns]
    if missing:
        raise RuntimeError(
            "Primary Step 6.4 dataset is missing coordinates required for "
            f"HKFI raster sampling: {missing}"
        )

    coordinates = list(
        zip(
            primary_df["x"].to_numpy(dtype=float),
            primary_df["y"].to_numpy(dtype=float),
        )
    )

    with rasterio.open(raster_path) as dataset:
        nodata = dataset.nodata
        sampled = []
        for value_array in dataset.sample(coordinates, indexes=1, masked=False):
            value = float(value_array[0])
            if nodata is not None and np.isfinite(float(nodata)):
                if np.isclose(value, float(nodata), rtol=0.0, atol=0.0):
                    value = np.nan
            sampled.append(value)

    values = np.asarray(sampled, dtype="float64")
    if not np.isfinite(values).all():
        bad_indices = np.flatnonzero(~np.isfinite(values))
        examples = primary_df.iloc[bad_indices[:20]][
            [base.Config.SAMPLE_ID_COLUMN, "x", "y"]
        ]
        raise RuntimeError(
            f"Scenario HKFI raster produced {len(bad_indices)} non-finite values "
            "at frozen Step 6.4 sample locations.\n"
            f"Raster: {raster_path}\n"
            f"Examples:\n{examples.to_string(index=False)}"
        )

    return values


def build_variant_dataset(
    base,
    scenarios: dict[str, dict[str, Any]],
    manifest_row: pd.Series,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Build one Step 7.2B model-ready sensitivity dataset entirely in memory.

    Primary equal-weight:
        use the exact successful Step 6.4 dataset unchanged.

    Alternative:
        copy the exact primary dataset and replace ONLY HKFI with values sampled
        from the authoritative Step 5.3 scenario normalized raster.
    """
    primary_df = base.load_and_validate_model_ready_dataset(scenarios)
    validate_dataset_exactly_matches_primary(primary_df, scenarios)

    features = list(scenarios[SensitivityConfig.SCENARIO_NAME]["features"])
    hkfi_feature = find_hkfi_feature(features)

    scenario_name = str(manifest_row["scenario"])
    raster_path = Path(manifest_row["norm_raster_path"])

    sampled_hkfi = sample_raster_at_frozen_samples(
        base=base,
        raster_path=raster_path,
        primary_df=primary_df,
    )

    primary_hkfi = primary_df[hkfi_feature].to_numpy(dtype=float)
    raster_vs_primary_max_abs = float(
        np.max(np.abs(sampled_hkfi - primary_hkfi))
    )
    raster_vs_primary_mean_abs = float(
        np.mean(np.abs(sampled_hkfi - primary_hkfi))
    )

    if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
        # The exact successful Step 6.4 dataset remains authoritative for the
        # primary equal-weight rerun. Raster sampling is only a QA cross-check.
        if (
            raster_vs_primary_max_abs
            > SensitivityConfig.PRIMARY_RASTER_SAMPLE_TOLERANCE
        ):
            raise RuntimeError(
                "The Step 5.3 primary equal-weight HKFI raster does not reproduce "
                "the HKFI values frozen in the successful Step 6.4 dataset within "
                "the configured tolerance.\n"
                f"Max absolute difference={raster_vs_primary_max_abs:.10g}\n"
                f"Tolerance={SensitivityConfig.PRIMARY_RASTER_SAMPLE_TOLERANCE:.10g}\n"
                "This indicates an upstream version mismatch between Step 5.3 and "
                "the dataset used for the successful Step 7.2 analysis."
            )
        variant_df = primary_df.copy()
    else:
        variant_df = primary_df.copy()
        variant_df[hkfi_feature] = sampled_hkfi

        # Strong one-variable-at-a-time QA.
        validate_equal_rows_and_folds(primary_df, variant_df)
        compare_columns = [
            base.Config.TARGET_COLUMN,
            base.Config.FOLD_COLUMN,
            base.Config.BLOCK_COLUMN,
            *all_scenario_features(scenarios),
        ]
        validate_only_allowed_column_changes(
            primary_df,
            variant_df,
            allowed_changed_columns=(hkfi_feature,),
            columns_to_compare=compare_columns,
            atol=SensitivityConfig.NON_HKFI_EQUALITY_ATOL,
        )

    delta = variant_df[hkfi_feature].to_numpy(dtype=float) - primary_hkfi

    qa = {
        "scenario": scenario_name,
        "scenario_purpose": str(manifest_row["scenario_purpose"]),
        "weight_K": float(manifest_row["weight_K"]),
        "weight_T": float(manifest_row["weight_T"]),
        "weight_S": float(manifest_row["weight_S"]),
        "norm_raster": str(raster_path),
        "is_primary": bool(manifest_row["is_primary"]),
        "n_samples": int(len(variant_df)),
        "hkfi_mean_absolute_change_vs_primary_dataset": float(
            np.mean(np.abs(delta))
        ),
        "hkfi_maximum_absolute_change_vs_primary_dataset": float(
            np.max(np.abs(delta))
        ),
        "primary_raster_vs_primary_dataset_mean_absolute_difference": (
            raster_vs_primary_mean_abs
        ),
        "primary_raster_vs_primary_dataset_maximum_absolute_difference": (
            raster_vs_primary_max_abs
        ),
        "only_hkfi_changed": True,
    }

    return variant_df, qa


# =============================================================================
# PERFORMANCE SUMMARIES
# =============================================================================
def summarize_variant_oof(
    base,
    oof_df: pd.DataFrame,
    manifest_row: pd.Series,
) -> dict[str, Any]:
    y = oof_df[base.Config.TARGET_COLUMN].to_numpy(dtype=int)
    p = oof_df["oof_probability"].to_numpy(dtype=float)
    blocks = oof_df[base.Config.FOLD_COLUMN].to_numpy(dtype=int)

    metrics = base.classification_metrics(y, p)

    scenario_name = str(manifest_row["scenario"])
    seed_offset = sum(
        (index + 1) * ord(character)
        for index, character in enumerate(scenario_name)
    ) % 10000

    low, high, n_valid = base.cluster_bootstrap_auc_ci(
        y_true=y,
        y_prob=p,
        blocks=blocks,
        n_bootstrap=base.Config.N_CLUSTER_BOOTSTRAP,
        seed=SensitivityConfig.RANDOM_STATE + seed_offset,
        ci_level=base.Config.CI_LEVEL,
    )

    return {
        "scenario": scenario_name,
        "scenario_purpose": str(manifest_row["scenario_purpose"]),
        "weight_K": float(manifest_row["weight_K"]),
        "weight_T": float(manifest_row["weight_T"]),
        "weight_S": float(manifest_row["weight_S"]),
        "is_primary": bool(manifest_row["is_primary"]),
        "oof_roc_auc": metrics["roc_auc"],
        "auc_ci_low": low,
        "auc_ci_high": high,
        "oof_pr_auc": metrics["pr_auc"],
        "oof_balanced_accuracy": metrics["balanced_accuracy"],
        "oof_f1": metrics["f1"],
        "oof_brier": metrics["brier"],
        "oof_precision": metrics["precision"],
        "oof_recall": metrics["recall"],
        "valid_auc_bootstrap_iterations": int(n_valid),
    }


# =============================================================================
# MAP HELPERS
# =============================================================================
def read_master_band_map(base) -> dict[str, int]:
    with rasterio.open(base.Config.MASTER_STACK_TIF) as dataset:
        descriptions = list(dataset.descriptions)

    band_map = {}
    for index, description in enumerate(descriptions, start=1):
        if description:
            band_map[str(description)] = int(index)
    return band_map


def predict_model_e_with_external_hkfi_raster(
    base,
    fitted_pipeline,
    features: list[str],
    hkfi_feature: str,
    hkfi_raster_path: Path,
    output_path: Path,
) -> Path:
    """
    Predict Model E using:
      - all non-HKFI predictors from the successful Step 6.1 master stack;
      - HKFI from the selected authoritative Step 5.3 scenario raster.

    No new full multiband stack is required.
    """
    require_file(base.Config.MASTER_STACK_TIF, "Step 6.1 master predictor stack")
    require_file(hkfi_raster_path, "Step 5.3 scenario HKFI raster")
    assert_raster_aligned_to_master(base, hkfi_raster_path)

    band_map = read_master_band_map(base)
    non_hkfi_features = [
        feature for feature in features if feature != hkfi_feature
    ]
    missing = [
        feature for feature in non_hkfi_features if feature not in band_map
    ]
    if missing:
        raise RuntimeError(
            f"Master stack is missing non-HKFI Model E predictors: {missing}"
        )

    non_hkfi_band_indexes = [
        band_map[feature] for feature in non_hkfi_features
    ]

    with rasterio.open(base.Config.MASTER_STACK_TIF) as master, rasterio.open(
        hkfi_raster_path
    ) as hkfi_ds:
        profile = master.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=base.Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        valid_pixel_count = 0

        with rasterio.open(temporary_path, "w", **profile) as output_ds:
            for row_start in range(
                0,
                master.height,
                SensitivityConfig.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    SensitivityConfig.RASTER_BLOCK_ROWS,
                    master.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=master.width,
                    height=block_height,
                )

                static_dynamic = master.read(
                    non_hkfi_band_indexes,
                    window=window,
                    masked=True,
                )
                hkfi_arr = hkfi_ds.read(
                    1,
                    window=window,
                    masked=True,
                )

                non_hkfi_values = np.asarray(
                    static_dynamic.filled(np.nan),
                    dtype="float64",
                )
                hkfi_values = np.asarray(
                    hkfi_arr.filled(np.nan),
                    dtype="float64",
                )

                valid = (
                    np.all(np.isfinite(non_hkfi_values), axis=0)
                    & np.isfinite(hkfi_values)
                )

                output = np.full(
                    (block_height, master.width),
                    base.Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                if np.any(valid):
                    data_columns: dict[str, np.ndarray] = {}
                    non_hkfi_index = 0
                    for feature in features:
                        if feature == hkfi_feature:
                            data_columns[feature] = hkfi_values[valid]
                        else:
                            data_columns[feature] = (
                                non_hkfi_values[non_hkfi_index][valid]
                            )
                            non_hkfi_index += 1

                    pixel_df = pd.DataFrame(data_columns, columns=features)
                    probabilities = (
                        fitted_pipeline.predict_proba(pixel_df)[:, 1]
                    )
                    output[valid] = probabilities.astype("float32")
                    valid_pixel_count += int(valid.sum())

                output_ds.write(output, 1, window=window)

            output_ds.set_band_description(
                1,
                "Random Forest Model E HKFI-weight sensitivity probability",
            )
            output_ds.update_tags(
                model="Random Forest",
                scenario="E_physics_guided",
                hkfi_source=str(hkfi_raster_path),
                generated_at_utc=utc_now_iso(),
                validation_note=(
                    "Map generated from a separately tuned full-data deployment "
                    "model. Validation performance comes from nested spatial OOF."
                ),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(temporary_path, output_path)

    log(
        f"[MAP] {valid_pixel_count:,} valid Model E pixels -> {output_path}"
    )
    return output_path


def fit_final_variant_model(
    base,
    df: pd.DataFrame,
    features: list[str],
    scenario_name: str,
    model_dir: Path,
) -> tuple[Any, dict[str, Any], float]:
    X = df[features]
    y = df[base.Config.TARGET_COLUMN].to_numpy(dtype=int)

    search = GridSearchCV(
        estimator=base.build_rf_pipeline(features),
        param_grid=base.Config.PARAM_GRID,
        scoring="roc_auc",
        cv=base.make_full_data_predefined_split(df),
        refit=True,
        n_jobs=base.Config.GRIDSEARCH_N_JOBS,
        verbose=0,
        return_train_score=False,
        error_score="raise",
    )
    search.fit(X, y)

    model_path = (
        model_dir
        / f"RF_Model_E_HKFI_{sanitize_name(scenario_name)}_final.joblib"
    )
    joblib.dump(search.best_estimator_, model_path)

    return (
        search.best_estimator_,
        json_safe(search.best_params_),
        float(search.best_score_),
    )


def clean_probability_raster(dataset) -> np.ndarray:
    array = dataset.read(1, masked=True).filled(np.nan).astype("float64")
    nodata = dataset.nodata
    if nodata is not None and np.isfinite(float(nodata)):
        array[array == float(nodata)] = np.nan
    array[(array < -1.0e-6) | (array > 1.0 + 1.0e-6)] = np.nan
    return array


def compare_probability_maps(
    reference_path: Path,
    candidate_path: Path,
) -> dict[str, Any]:
    require_file(reference_path, "reference probability raster")
    require_file(candidate_path, "candidate probability raster")

    with rasterio.open(reference_path) as ref_ds, rasterio.open(
        candidate_path
    ) as can_ds:
        if not (
            ref_ds.width == can_ds.width
            and ref_ds.height == can_ds.height
            and ref_ds.transform == can_ds.transform
            and ref_ds.crs == can_ds.crs
        ):
            raise RuntimeError(
                "HKFI sensitivity probability maps are not spatially aligned."
            )

        ref = clean_probability_raster(ref_ds)
        can = clean_probability_raster(can_ds)

    valid = np.isfinite(ref) & np.isfinite(can)
    if not np.any(valid):
        raise RuntimeError("No common valid pixels between sensitivity maps.")

    ref_v = ref[valid]
    can_v = can[valid]
    difference = can_v - ref_v

    ref_class = np.digitize(
        ref_v,
        bins=[0.2, 0.4, 0.6, 0.8],
        right=False,
    )
    can_class = np.digitize(
        can_v,
        bins=[0.2, 0.4, 0.6, 0.8],
        right=False,
    )

    return {
        "common_valid_pixels": int(valid.sum()),
        "mean_absolute_probability_difference": float(
            np.mean(np.abs(difference))
        ),
        "maximum_absolute_probability_difference": float(
            np.max(np.abs(difference))
        ),
        "rmse_probability_difference": float(
            np.sqrt(np.mean(difference ** 2))
        ),
        "pearson_probability_correlation": float(
            np.corrcoef(ref_v, can_v)[0, 1]
        ),
        "spearman_probability_correlation": float(
            pd.Series(ref_v).corr(
                pd.Series(can_v),
                method="spearman",
            )
        ),
        "pixels_changing_susceptibility_class_percent": float(
            100.0 * np.mean(ref_class != can_class)
        ),
        "reference_high_plus_very_high_area_percent": float(
            100.0 * np.mean(ref_v >= 0.6)
        ),
        "candidate_high_plus_very_high_area_percent": float(
            100.0 * np.mean(can_v >= 0.6)
        ),
        "delta_high_plus_very_high_area_percentage_points": float(
            100.0 * np.mean(can_v >= 0.6)
            - 100.0 * np.mean(ref_v >= 0.6)
        ),
    }


# =============================================================================
# OUTPUT / CHECKPOINT PATHS
# =============================================================================
def output_directories(base) -> dict[str, Path]:
    out_dir = (
        Path(base.Config.OUT_DIR)
        / "Step-7.2B-HKFI_Weight_Sensitivity_STEP53_Direct"
    )
    paths = {
        "out": out_dir,
        "datasets": out_dir / "generated_model_ready_datasets",
        "models": out_dir / "models",
        "maps": out_dir / "maps",
        "workers": out_dir / SensitivityConfig.WORKER_DIRNAME,
        "logs": (
            out_dir
            / SensitivityConfig.WORKER_DIRNAME
            / SensitivityConfig.LOG_DIRNAME
        ),
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def scenario_paths(
    base,
    scenario_name: str,
) -> dict[str, Path]:
    dirs = output_directories(base)
    safe = sanitize_name(scenario_name)
    return {
        "dataset": dirs["datasets"] / f"{safe}_model_ready.csv",
        "qa": dirs["workers"] / f"{safe}_dataset_QA.json",
        "oof": dirs["workers"] / f"{safe}_OOF.csv",
        "fold_metrics": dirs["workers"] / f"{safe}_Fold_Metrics.csv",
        "search": dirs["workers"] / f"{safe}_Inner_Search.csv",
        "params": dirs["workers"] / f"{safe}_Outer_Params.json",
        "nested_done": dirs["workers"] / f"{safe}_nested.done",
        "model_info": dirs["workers"] / f"{safe}_Map_Model_Info.json",
        "e_map": dirs["maps"] / f"RF_Model_E_HKFI_{safe}_probability.tif",
        "ensemble_map": dirs["maps"] / f"RF_A30_E70_HKFI_{safe}_probability.tif",
        "map_done": dirs["workers"] / f"{safe}_map.done",
    }


# =============================================================================
# PROCESS-ISOLATED WORKERS
# =============================================================================
def run_external_worker(
    worker_mode: str,
    scenario_name: str,
) -> None:
    script_path = resolve_script_path()
    base = load_step72_module(SensitivityConfig.BASE_SCRIPT)
    dirs = output_directories(base)
    log_path = (
        dirs["logs"]
        / f"{sanitize_name(scenario_name)}_{worker_mode}.log"
    )

    command = [
        sys.executable,
        str(script_path),
        "--worker-mode",
        worker_mode,
        "--scenario",
        scenario_name,
    ]

    environment = os.environ.copy()
    environment["OMP_NUM_THREADS"] = "1"
    environment["OPENBLAS_NUM_THREADS"] = "1"
    environment["MKL_NUM_THREADS"] = "1"
    environment["NUMEXPR_NUM_THREADS"] = "1"

    log(
        f"[WORKER] Starting {worker_mode}: {scenario_name}\n"
        f"[WORKER] Log: {log_path}"
    )

    with open(log_path, "w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=environment,
        )

        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_handle.write(line)
            log_handle.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Step 7.2B RF worker failed: mode={worker_mode}, "
            f"scenario={scenario_name}, return_code={return_code}. "
            f"See log: {log_path}"
        )


def get_manifest_row(
    manifest: pd.DataFrame,
    scenario_name: str,
) -> pd.Series:
    rows = manifest.loc[manifest["scenario"] == scenario_name]
    if len(rows) != 1:
        raise RuntimeError(
            f"Expected exactly one manifest row for {scenario_name}; found {len(rows)}"
        )
    return rows.iloc[0]


def run_nested_worker(scenario_name: str) -> None:
    base = load_step72_module(SensitivityConfig.BASE_SCRIPT)
    scenarios = base.validate_scenario_manifests()
    manifest = load_step53_weight_manifest()
    row = get_manifest_row(manifest, scenario_name)
    paths = scenario_paths(base, scenario_name)

    variant_df, qa = build_variant_dataset(
        base=base,
        scenarios=scenarios,
        manifest_row=row,
    )
    variant_df.to_csv(
        paths["dataset"],
        index=False,
        float_format="%.10f",
    )

    with open(paths["qa"], "w", encoding="utf-8") as handle:
        json.dump(json_safe(qa), handle, indent=2)

    features = list(
        scenarios[SensitivityConfig.SCENARIO_NAME]["features"]
    )

    log(
        f"[NESTED] {scenario_name}: "
        f"K={row['weight_K']:.3f}, "
        f"T={row['weight_T']:.3f}, "
        f"S={row['weight_S']:.3f}"
    )

    oof_df, fold_df, params, search_df = (
        base.run_nested_spatial_cv_for_scenario(
            df=variant_df,
            scenario_name=SensitivityConfig.SCENARIO_NAME,
            features=features,
        )
    )

    reproduction_qa = None
    if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
        reproduction_qa = assert_oof_reproduces_primary(
            oof_df,
            SensitivityConfig.SCENARIO_NAME,
            tolerance=SensitivityConfig.OOF_REPRODUCTION_TOLERANCE,
        )
        qa.update(reproduction_qa)
        with open(paths["qa"], "w", encoding="utf-8") as handle:
            json.dump(json_safe(qa), handle, indent=2)

    oof_df = oof_df.copy()
    oof_df.insert(0, "hkfi_weight_variant", scenario_name)

    fold_df = fold_df.copy()
    fold_df.insert(0, "hkfi_weight_variant", scenario_name)

    search_df = search_df.copy()
    search_df.insert(0, "hkfi_weight_variant", scenario_name)

    oof_df.to_csv(paths["oof"], index=False, float_format="%.10f")
    fold_df.to_csv(
        paths["fold_metrics"],
        index=False,
        float_format="%.10f",
    )
    search_df.to_csv(
        paths["search"],
        index=False,
        float_format="%.10f",
    )
    with open(paths["params"], "w", encoding="utf-8") as handle:
        json.dump(json_safe(params), handle, indent=2)

    paths["nested_done"].write_text(
        utc_now_iso(),
        encoding="utf-8",
    )
    log(f"[NESTED] Completed: {scenario_name}")


def run_map_worker(scenario_name: str) -> None:
    base = load_step72_module(SensitivityConfig.BASE_SCRIPT)
    scenarios = base.validate_scenario_manifests()
    manifest = load_step53_weight_manifest()
    row = get_manifest_row(manifest, scenario_name)
    paths = scenario_paths(base, scenario_name)
    dirs = output_directories(base)

    if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
        # The successful Step 7.2 maps are the authoritative primary reference.
        require_file(
            base.Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
            "successful Step 7.2 primary Model E full-domain raster",
        )
        require_file(
            base.Config.ENSEMBLE_PROBABILITY_TIF,
            "successful Step 7.2 A30/E70 ensemble raster",
        )

        info = {
            "scenario": scenario_name,
            "map_source": "successful Step 7.2 primary reference",
            "model_e_map_path": str(
                base.Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF
            ),
            "operational_ensemble_map_path": str(
                base.Config.ENSEMBLE_PROBABILITY_TIF
            ),
            "refit_performed": False,
        }
        with open(paths["model_info"], "w", encoding="utf-8") as handle:
            json.dump(info, handle, indent=2)
        paths["map_done"].write_text(utc_now_iso(), encoding="utf-8")
        return

    variant_df, qa = build_variant_dataset(
        base=base,
        scenarios=scenarios,
        manifest_row=row,
    )

    features = list(
        scenarios[SensitivityConfig.SCENARIO_NAME]["features"]
    )
    hkfi_feature = find_hkfi_feature(features)

    model, best_params, selection_auc = fit_final_variant_model(
        base=base,
        df=variant_df,
        features=features,
        scenario_name=scenario_name,
        model_dir=dirs["models"],
    )

    e_map = predict_model_e_with_external_hkfi_raster(
        base=base,
        fitted_pipeline=model,
        features=features,
        hkfi_feature=hkfi_feature,
        hkfi_raster_path=Path(row["norm_raster_path"]),
        output_path=paths["e_map"],
    )

    ensemble_map = None
    if base.Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF.exists():
        ensemble_map = base.blend_static_and_enhanced_probability_rasters(
            static_raster_path=base.Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
            enhanced_raster_path=e_map,
            output_path=paths["ensemble_map"],
            enhanced_weight=float(base.Config.ENSEMBLE_ENHANCED_WEIGHT),
        )
    else:
        log(
            "[MAP-WARN] Operational ensemble skipped because successful "
            "Step 7.2 Model A full-domain raster is missing: "
            f"{base.Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF}"
        )

    info = {
        "scenario": scenario_name,
        "scenario_purpose": str(row["scenario_purpose"]),
        "weight_K": float(row["weight_K"]),
        "weight_T": float(row["weight_T"]),
        "weight_S": float(row["weight_S"]),
        "full_data_selection_auc_for_model_selection_only": selection_auc,
        "best_params": best_params,
        "model_e_map_path": str(e_map),
        "operational_ensemble_map_path": (
            str(ensemble_map) if ensemble_map is not None else None
        ),
        "refit_performed": True,
    }
    with open(paths["model_info"], "w", encoding="utf-8") as handle:
        json.dump(json_safe(info), handle, indent=2)

    paths["map_done"].write_text(utc_now_iso(), encoding="utf-8")

    del model
    gc.collect()
    log(f"[MAP] Completed: {scenario_name}")


# =============================================================================
# CONTROLLER / AGGREGATION
# =============================================================================
def load_nested_outputs(
    base,
    manifest: pd.DataFrame,
) -> tuple[
    dict[str, pd.DataFrame],
    pd.DataFrame,
    pd.DataFrame,
]:
    all_oof: dict[str, pd.DataFrame] = {}
    summaries = []
    qa_rows = []

    for _, row in manifest.iterrows():
        scenario_name = str(row["scenario"])
        paths = scenario_paths(base, scenario_name)

        if not paths["nested_done"].exists():
            raise RuntimeError(
                f"Missing nested checkpoint for scenario: {scenario_name}"
            )

        oof_df = pd.read_csv(paths["oof"])
        all_oof[scenario_name] = oof_df
        summaries.append(
            summarize_variant_oof(
                base=base,
                oof_df=oof_df,
                manifest_row=row,
            )
        )

        with open(paths["qa"], "r", encoding="utf-8") as handle:
            qa_rows.append(json.load(handle))

    return (
        all_oof,
        pd.DataFrame(summaries),
        pd.DataFrame(qa_rows),
    )


def build_pairwise_comparisons(
    base,
    manifest: pd.DataFrame,
    all_oof: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    reference = all_oof[SensitivityConfig.REFERENCE_VARIANT]
    rows = []

    for index, row in manifest.iterrows():
        scenario_name = str(row["scenario"])
        if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
            continue

        comparison = paired_auc_difference_from_oof(
            base=base,
            left_df=all_oof[scenario_name],
            right_df=reference,
            label=f"{scenario_name} - {SensitivityConfig.REFERENCE_VARIANT}",
            seed=SensitivityConfig.RANDOM_STATE + 100 + int(index),
        )
        comparison.update(
            {
                "scenario": scenario_name,
                "scenario_purpose": str(row["scenario_purpose"]),
                "weight_K": float(row["weight_K"]),
                "weight_T": float(row["weight_T"]),
                "weight_S": float(row["weight_S"]),
            }
        )
        rows.append(comparison)

    return pd.DataFrame(rows)


def build_map_comparisons(
    base,
    manifest: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    model_rows = []
    comparison_rows = []

    reference_e_map = Path(base.Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF)
    reference_ensemble = Path(base.Config.ENSEMBLE_PROBABILITY_TIF)

    for _, row in manifest.iterrows():
        scenario_name = str(row["scenario"])
        paths = scenario_paths(base, scenario_name)

        if not paths["map_done"].exists():
            continue

        with open(paths["model_info"], "r", encoding="utf-8") as handle:
            info = json.load(handle)

        model_rows.append(info)

        if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
            continue

        e_map_text = info.get("model_e_map_path")
        if e_map_text:
            e_map = Path(e_map_text)
            comparison_rows.append(
                {
                    "map_product": "Model E scientific sensitivity map",
                    "scenario": scenario_name,
                    "scenario_purpose": str(row["scenario_purpose"]),
                    "weight_K": float(row["weight_K"]),
                    "weight_T": float(row["weight_T"]),
                    "weight_S": float(row["weight_S"]),
                    "reference_scenario": SensitivityConfig.REFERENCE_VARIANT,
                    **compare_probability_maps(reference_e_map, e_map),
                }
            )

        ensemble_text = info.get("operational_ensemble_map_path")
        if ensemble_text and reference_ensemble.exists():
            ensemble_map = Path(ensemble_text)
            comparison_rows.append(
                {
                    "map_product": "A30/E70 operational ensemble map",
                    "scenario": scenario_name,
                    "scenario_purpose": str(row["scenario_purpose"]),
                    "weight_K": float(row["weight_K"]),
                    "weight_T": float(row["weight_T"]),
                    "weight_S": float(row["weight_S"]),
                    "reference_scenario": SensitivityConfig.REFERENCE_VARIANT,
                    **compare_probability_maps(
                        reference_ensemble,
                        ensemble_map,
                    ),
                }
            )

    return pd.DataFrame(model_rows), pd.DataFrame(comparison_rows)


def merge_upstream_hkfi_metrics(
    summary_df: pd.DataFrame,
) -> pd.DataFrame:
    upstream = load_optional_step53_named_summary()
    if upstream.empty:
        return summary_df

    # Avoid duplicating weight columns if they are present upstream.
    duplicate_columns = [
        column for column in upstream.columns
        if column in summary_df.columns and column != "scenario"
    ]
    upstream = upstream.drop(columns=duplicate_columns, errors="ignore")

    return summary_df.merge(
        upstream,
        on="scenario",
        how="left",
        validate="one_to_one",
    )


def create_performance_figure(
    summary_df: pd.DataFrame,
    output_path: Path,
) -> None:
    plot_df = summary_df.copy().reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    x = np.arange(len(plot_df))
    y = plot_df["oof_roc_auc"].to_numpy(dtype=float)

    lower = y - plot_df["auc_ci_low"].to_numpy(dtype=float)
    upper = plot_df["auc_ci_high"].to_numpy(dtype=float) - y

    ax.errorbar(
        x,
        y,
        yerr=np.vstack([lower, upper]),
        fmt="o",
        capsize=3,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(
        plot_df["scenario"],
        rotation=32,
        ha="right",
    )
    ax.set_ylabel("Nested spatial OOF ROC-AUC")
    ax.set_xlabel("Pre-specified HKFI weighting scenario")
    ax.set_ylim(
        max(
            0.5,
            float(np.nanmin(plot_df["auc_ci_low"])) - 0.03,
        ),
        1.0,
    )
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=SensitivityConfig.FIG_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def write_summary_report(
    base,
    manifest: pd.DataFrame,
    summary_df: pd.DataFrame,
    comparison_df: pd.DataFrame,
    qa_df: pd.DataFrame,
    map_comparison_df: pd.DataFrame,
) -> None:
    dirs = output_directories(base)
    report_path = dirs["out"] / "Reviewer_1_Comment_7_RF_HKFI_Weight_Sensitivity_Report.txt"

    with open(report_path, "w", encoding="utf-8") as handle:
        handle.write("STEP 7.2B — RF HKFI WEIGHTING SENSITIVITY\n")
        handle.write("=" * 88 + "\n\n")
        handle.write(
            "Source of weighting scenarios:\n"
            f"{SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV}\n\n"
        )
        handle.write(
            "Step 7.2B is a targeted cross-algorithm robustness analysis. "
            "The primary equal-component formulation and three pre-specified "
            "emphasis scenarios are propagated through the exact successful "
            "Step 7.2 RF nested spatial-CV contract. The Step 5.3 simplex is "
            "not optimized through RF, weights are not selected post hoc by "
            "AUC, and full-domain map sensitivity is not repeated because it "
            "was already completed for the primary XGBoost framework in "
            "Step 7.1B.\n\n"
        )

        handle.write("SELECTED STEP 5.3 NAMED SCENARIOS\n")
        handle.write("-" * 88 + "\n")
        handle.write(
            manifest[
                [
                    "scenario",
                    "scenario_purpose",
                    "weight_K",
                    "weight_T",
                    "weight_S",
                ]
            ].to_string(index=False)
        )
        handle.write("\n\n")

        handle.write("DATASET / REPRODUCIBILITY QA\n")
        handle.write("-" * 88 + "\n")
        handle.write(qa_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("NESTED SPATIAL OOF PERFORMANCE\n")
        handle.write("-" * 88 + "\n")
        performance_columns = [
            "scenario",
            "weight_K",
            "weight_T",
            "weight_S",
            "oof_roc_auc",
            "auc_ci_low",
            "auc_ci_high",
            "oof_pr_auc",
            "oof_balanced_accuracy",
            "oof_f1",
            "oof_brier",
        ]
        handle.write(
            summary_df[
                [column for column in performance_columns if column in summary_df.columns]
            ].to_string(index=False)
        )
        handle.write("\n\n")

        handle.write("PAIRED DELTA-AUC VERSUS PRIMARY EQUAL-WEIGHT SCENARIO\n")
        handle.write("-" * 88 + "\n")
        if comparison_df.empty:
            handle.write("No alternative scenarios evaluated.\n")
        else:
            handle.write(comparison_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("MAP SENSITIVITY SCOPE\n")
        handle.write("-" * 88 + "\n")
        if map_comparison_df.empty:
            handle.write(
                "Not repeated for Random Forest by design. The full HKFI "
                "map-sensitivity experiment was already completed in Step 7.1B "
                "for the primary XGBoost framework.\n"
            )
        else:
            handle.write(map_comparison_df.to_string(index=False))

        handle.write("\n\n")
        handle.write("INTERPRETATION RULE\n")
        handle.write("-" * 88 + "\n")
        handle.write(
            "Step 7.2B is not used to identify an RF-optimal HKFI weighting. "
            "Its purpose is to test whether the XGBoost conclusion that plausible "
            "HKFI weighting changes do not materially alter discrimination is "
            "also supported under Random Forest.\n"
        )


def run_controller() -> None:
    base = load_step72_module(SensitivityConfig.BASE_SCRIPT)
    scenarios = base.validate_scenario_manifests()
    manifest = load_step53_weight_manifest()

    expected_cross_algorithm_scenarios = [
        "primary_equal_component",
        "kinematic_emphasis",
        "twi_emphasis",
        "sm_emphasis",
    ]

    observed_cross_algorithm_scenarios = manifest["scenario"].tolist()

    if observed_cross_algorithm_scenarios != expected_cross_algorithm_scenarios:
        raise RuntimeError(
            "Step 7.2B targeted cross-algorithm scenario set mismatch.\n"
            f"Expected: {expected_cross_algorithm_scenarios}\n"
            f"Found: {observed_cross_algorithm_scenarios}"
        )

    dirs = output_directories(base)

    log("=" * 96)
    log("STEP 7.2B — TARGETED RF CROSS-ALGORITHM HKFI WEIGHT ROBUSTNESS")
    log("=" * 96)
    log(f"[STEP5.3] Manifest: {SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV}")
    log(
        "[STEP5.3] Named scenarios selected: "
        + ", ".join(manifest["scenario"].tolist())
    )
    log(
        "[RULE] Selected pre-specified HKFI scenarios only; the Step 5.3 simplex "
        "is not propagated through RF, weights are not optimized by AUC, and "
        "full-domain map sensitivity is not repeated."
    )

    write_diagnostic_provenance(
        dirs["out"],
        "Step_7_2B_RF_STEP53_Direct",
        {
            "step53_weight_manifest": str(
                SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV
            ),
            "step53_named_sensitivity": str(
                SensitivityConfig.STEP53_NAMED_SENSITIVITY_CSV
            ),
            "step53_simplex_sensitivity": str(
                SensitivityConfig.STEP53_SIMPLEX_SENSITIVITY_CSV
            ),
            "selected_scenarios": manifest[
                [
                    "scenario",
                    "scenario_purpose",
                    "weight_K",
                    "weight_T",
                    "weight_S",
                    "norm_raster_path",
                ]
            ].to_dict(orient="records"),
            "model_scenario": SensitivityConfig.SCENARIO_NAME,
            "primary_reference": SensitivityConfig.REFERENCE_VARIANT,
            "simplex_propagated_through_ml": False,
            "map_sensitivity_generated": False,
            "cross_algorithm_scope": (
                "Performance robustness only. Full HKFI map sensitivity was "
                "already evaluated in Step 7.1B using the primary XGBoost "
                "framework."
            ),
            "scientific_rule": (
                "Selected pre-specified weight scenarios only; no downstream "
                "weight optimization and no duplicate RF map-sensitivity stage."
            ),
        },
    )

    manifest.to_csv(
        dirs["out"] / "RF_7_2B_Selected_Step53_Weight_Scenarios.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # 1. Nested CV — one fresh process per named scenario.
    # -------------------------------------------------------------------------
    if SensitivityConfig.RUN_NESTED_CV:
        for scenario_name in manifest["scenario"]:
            paths = scenario_paths(base, scenario_name)
            if (
                SensitivityConfig.RESUME_FROM_CHECKPOINTS
                and paths["nested_done"].exists()
                and paths["oof"].exists()
            ):
                log(f"[RESUME] Nested CV already complete: {scenario_name}")
                continue
            run_external_worker("nested", scenario_name)

    all_oof, summary_df, qa_df = load_nested_outputs(
        base=base,
        manifest=manifest,
    )

    comparison_df = build_pairwise_comparisons(
        base=base,
        manifest=manifest,
        all_oof=all_oof,
    )

    summary_with_upstream = merge_upstream_hkfi_metrics(summary_df)

    summary_with_upstream.to_csv(
        dirs["out"] / "RF_7_2B_HKFI_Weight_Sensitivity_Performance.csv",
        index=False,
        float_format="%.10f",
    )
    comparison_df.to_csv(
        dirs["out"] / "RF_7_2B_HKFI_Weight_Sensitivity_Paired_AUC.csv",
        index=False,
        float_format="%.10f",
    )
    qa_df.to_csv(
        dirs["out"] / "RF_7_2B_HKFI_Weight_Sensitivity_QA.csv",
        index=False,
        float_format="%.10f",
    )

    # -------------------------------------------------------------------------
    # 2. Full-domain RF map sensitivity.
    #
    # Deliberately disabled for this targeted cross-algorithm analysis because
    # Step 7.1B already completed the full XGBoost map-sensitivity experiment.
    # -------------------------------------------------------------------------
    if SensitivityConfig.GENERATE_MAP_SENSITIVITY:
        for scenario_name in manifest["scenario"]:
            paths = scenario_paths(base, scenario_name)

            if (
                SensitivityConfig.RESUME_FROM_CHECKPOINTS
                and paths["map_done"].exists()
                and paths["model_info"].exists()
            ):
                log(f"[RESUME] Map stage already complete: {scenario_name}")
                continue

            run_external_worker("map", scenario_name)

        map_model_df, map_comparison_df = build_map_comparisons(
            base=base,
            manifest=manifest,
        )

        map_model_df.to_csv(
            dirs["out"] / "RF_7_2B_HKFI_Weight_Sensitivity_Map_Models.csv",
            index=False,
        )

        map_comparison_df.to_csv(
            dirs["out"] / "RF_7_2B_HKFI_Weight_Sensitivity_Map_Comparison.csv",
            index=False,
            float_format="%.10f",
        )

    else:
        map_model_df = pd.DataFrame()
        map_comparison_df = pd.DataFrame()

        log(
            "[MAP] Skipped by design: Step 7.2B is a performance-only "
            "cross-algorithm robustness analysis."
        )

    create_performance_figure(
        summary_df=summary_df,
        output_path=(
            dirs["out"]
            / "RF_7_2B_HKFI_Weight_Sensitivity_Performance.png"
        ),
    )

    write_summary_report(
        base=base,
        manifest=manifest,
        summary_df=summary_with_upstream,
        comparison_df=comparison_df,
        qa_df=qa_df,
        map_comparison_df=map_comparison_df,
    )

    log("")
    log("=" * 96)
    log("STEP 7.2B COMPLETED SUCCESSFULLY")
    log("=" * 96)
    log(f"[OUT] {dirs['out']}")
    log("")
    log("Nested spatial OOF performance:")
    log(summary_df.to_string(index=False))
    if not comparison_df.empty:
        log("")
        log("Paired delta-AUC versus primary equal weighting:")
        log(comparison_df.to_string(index=False))


# =============================================================================
# COMMAND-LINE ENTRY POINT
# =============================================================================
def parse_command_line_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Step 7.2B Random Forest HKFI weight sensitivity using authoritative "
            "Step 5.3 named scenarios."
        )
    )
    parser.add_argument(
        "--worker-mode",
        choices=["nested", "map"],
        default=None,
    )
    parser.add_argument(
        "--scenario",
        default=None,
    )

    arguments, unknown = parser.parse_known_args()

    # Jupyter/IPython injects unrelated arguments such as -f kernel.json.
    if unknown:
        log(f"[CLI] Ignoring host-environment arguments: {unknown}")

    return arguments


if __name__ == "__main__":
    try:
        args = parse_command_line_arguments()

        if args.worker_mode is None:
            run_controller()
        else:
            if not args.scenario:
                raise RuntimeError(
                    "--scenario is required when --worker-mode is supplied."
                )

            if args.worker_mode == "nested":
                run_nested_worker(args.scenario)
            elif args.worker_mode == "map":
                run_map_worker(args.scenario)
            else:
                raise RuntimeError(
                    f"Unsupported worker mode: {args.worker_mode}"
                )

    except Exception as exc:
        print(f"[ERROR] Step 7.2B failed: {exc}", flush=True)
        traceback.print_exc()
        raise


# STEP 7.2C — RANDOM FOREST PSEUDO-ABSENCE BUFFER SENSITIVITY

In [ ]:
# =============================================================================
# STEP 7.2C — RANDOM FOREST PSEUDO-ABSENCE BUFFER SENSITIVITY
# =============================================================================
#
# Scientific purpose
# ------------------
# This script is the Random Forest counterpart to Step 7.1C (XGBoost).
# It evaluates whether the primary conclusion that the physics-guided Model E
# improves over the static Model A is sensitive to pseudo-absence exclusion
# distance when the machine-learning algorithm is Random Forest.
#
# The experimental design exactly mirrors Step 7.1C:
#
#   Buffers:
#       100 m   -> primary Step 6.4 model-ready dataset
#       500 m   -> Step 6.3 buffer-specific dataset
#       1000 m  -> Step 6.3 buffer-specific dataset
#
#   Models:
#       Model A -> static baseline
#       Model E -> static + HKFI + SNDR
#
# Scientific isolation rules
# --------------------------
# 1. The same 165 positive landslide samples must be present at all buffers.
# 2. Positive predictor values and spatial-fold assignments must be identical.
# 3. Only the pseudo-absence sample set changes with exclusion distance.
# 4. Each buffer/model combination is independently nested-tuned using the exact
#    successful Step 7.2 Random Forest 48-combination hyperparameter grid.
# 5. Hyperparameters are NOT transferred between buffers.
# 6. The primary reviewer statistic is paired E - A ROC-AUC within each buffer.
# 7. The 100 m Model A and Model E reruns must reproduce the successful Step 7.2
#    RF OOF predictions.
#
# Plotting rule
# -------------
# The final plot intentionally reproduces the exact Step 7.1C plotting style:
#   - figure size = (5.8, 4.2)
#   - Model A and Model E shown as connected error-bar lines
#   - x-axis buffers = 100, 500, 1000 m
#   - y-axis = Nested OOF ROC-AUC
#   - asymmetric 95% spatial-block bootstrap error bars
#   - marker = "o", capsize = 3, linewidth = 1.4
#   - same axis labels, grid and legend style
#
# Output directory
# ----------------
# ./data/Results/
#     Step-7.2-RF-Models_A_to_F/
#     Step-7.2C-Buffer_Sensitivity/
#
# =============================================================================

from __future__ import annotations

import gc
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Sequence

# Resource-safe execution.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


# =============================================================================
# CONFIGURATION — EXACT SUCCESSFUL STEP 7.2 RF CONTRACT
# =============================================================================
class BufferConfig:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Step 6.1 / Step 6.4 predictor definitions.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = (
        STEP61_DIR / "model_ablation_scenarios.json"
    )

    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    STEP64_MODEL_CONFIG_JSON = (
        STEP64_DIR / "model_configuration_manifest.json"
    )

    # Step 6.3 buffer-specific datasets.
    STEP63_DIR = (
        RESULTS_DIR / "Step-6.3-Site_Level_Sampling"
    )

    # Normal authoritative workflow:
    #   100 m  -> exact Step 6.4 model-ready dataset
    #   500 m  -> Step 6.3 buffer dataset
    #   1000 m -> Step 6.3 buffer dataset
    DEFAULT_BUFFER_DATASETS = {
        100: (
            STEP64_DIR
            / "model_ready_dataset_A_to_F.csv"
        ),
        500: (
            STEP63_DIR
            / "common_site_level_samples_A_to_F_buffer500m.csv"
        ),
        1000: (
            STEP63_DIR
            / "common_site_level_samples_A_to_F_buffer1000m.csv"
        ),
    }

    # Optional override manifest. Not required for the normal workflow.
    MANIFEST_PATH = (
        STEP64_DIR
        / "buffer_sensitivity"
        / "buffer_sensitivity_manifest.json"
    )

    # Optional Step 6.4 QA record.
    STEP64_BUFFER_QA_CSV = (
        STEP64_DIR
        / "buffer_dataset_consistency_QA.csv"
    )

    # Successful Step 7.2 RF outputs.
    STEP72_OUT_DIR = (
        RESULTS_DIR / "Step-7.2-RF-Models_A_to_F"
    )
    PRIMARY_STEP72_OOF_CSV = (
        STEP72_OUT_DIR / "RF_A_to_F_OOF_predictions.csv"
    )

    # Step 7.2C output.
    OUT_DIR = (
        STEP72_OUT_DIR
        / "Step-7.2C-Buffer_Sensitivity"
    )

    # Models evaluated exactly as in Step 7.1C.
    SCENARIOS = (
        "A_static_baseline",
        "E_physics_guided",
    )

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    # Exact successful Step 7.2 RF 48-combination grid.
    PARAM_GRID = {
        "model__n_estimators": [200, 500],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 2],
        "model__max_features": ["sqrt", 0.5],
        "model__criterion": ["gini", "entropy"],
    }

    RF_FIXED_PARAMS = {
        "bootstrap": True,
        "class_weight": None,
        "random_state": 42,
        "n_jobs": 1,
    }

    GRIDSEARCH_N_JOBS = 1
    GRIDSEARCH_VERBOSE = 2

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    CATEGORICAL_FEATURE_ALIASES = {
        "landcover",
        "landuse",
        "land_use",
    }

    BINARY_THRESHOLD = 0.50

    # Spatial-block bootstrap.
    N_CLUSTER_BOOTSTRAP = 5000
    CI_LEVEL = 0.95

    REFERENCE_BUFFER_M = 100
    RANDOM_STATE = 52371

    # Step 7.2 OOF probabilities are saved to finite decimal precision.
    OOF_REPRODUCTION_TOLERANCE = 1.0e-8

    # Positive rows should be identical across buffer experiments. The 100 m
    # Step 6.4 CSV and Step 6.3 CSVs can differ only through harmless decimal
    # serialization. This tolerance is intentionally strict.
    POSITIVE_PREDICTOR_EQUALITY_ATOL = 5.0e-8

    FIG_DPI = 300


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(
        f"[{datetime.now():%H:%M:%S}] {message}",
        flush=True,
    )


def utc_now_iso() -> str:
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def require_file(
    path: Path,
    description: str,
) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}"
        )


def load_json(
    path: Path,
) -> Dict[str, Any]:
    require_file(
        path,
        "JSON file",
    )

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def json_safe(
    value: Any,
) -> Any:
    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    return value


def normalize_feature_name(
    name: str,
) -> str:
    return re.sub(
        r"[^a-z0-9_]+",
        "_",
        str(
            name
        ).strip().lower(),
    ).strip(
        "_"
    )


def is_categorical_feature(
    name: str,
) -> bool:
    normalized = normalize_feature_name(
        name
    )

    compact = normalized.replace(
        "_",
        "",
    )

    aliases = {
        alias.lower()
        for alias
        in BufferConfig.CATEGORICAL_FEATURE_ALIASES
    }

    compact_aliases = {
        alias.replace(
            "_",
            "",
        )
        for alias
        in aliases
    }

    return (
        normalized in aliases
        or compact in compact_aliases
    )


# =============================================================================
# STEP 6.1 / STEP 6.4 SCENARIO CONTRACT
# =============================================================================
def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """
    Verify that Step 6.1 and Step 6.4 define Models A-F identically.
    """
    step61 = load_json(
        BufferConfig.SCENARIO_MANIFEST_JSON
    ).get(
        "scenarios",
        {},
    )

    step64 = load_json(
        BufferConfig.STEP64_MODEL_CONFIG_JSON
    ).get(
        "scenarios",
        {},
    )

    for scenario_name in BufferConfig.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                "Step 6.1 scenario manifest is missing "
                f"{scenario_name}."
            )

        if scenario_name not in step64:
            raise RuntimeError(
                "Step 6.4 model configuration is missing "
                f"{scenario_name}."
            )

        features_61 = list(
            step61[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        features_64 = list(
            step64[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any(
            "irs" in feature.lower()
            for feature in features_61
        ):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; "
                "IRS must not be an ML predictor."
            )

    log(
        "[QA] Step 6.1 and Step 6.4 A-F scenario definitions match: PASS"
    )

    return step64


def all_scenario_features(
    scenarios: Dict[str, Dict[str, Any]],
) -> list[str]:
    features: list[str] = []

    for scenario_name in BufferConfig.SCENARIO_ORDER:
        for feature in scenarios[
            scenario_name
        ][
            "features"
        ]:
            if feature not in features:
                features.append(
                    feature
                )

    return features


# =============================================================================
# BUFFER INPUT RESOLUTION
# =============================================================================
def load_buffer_paths(
    path: Path,
) -> Dict[int, Path]:
    """
    Resolve 100, 500 and 1000 m datasets.

    Normal workflow:
      100 m  -> Step 6.4 model_ready_dataset_A_to_F.csv
      500 m  -> Step 6.3 common_site_level_samples_A_to_F_buffer500m.csv
      1000 m -> Step 6.3 common_site_level_samples_A_to_F_buffer1000m.csv

    A JSON override manifest is supported but not required.
    """
    if path.exists():
        with open(
            path,
            "r",
            encoding="utf-8",
        ) as handle:
            manifest = json.load(
                handle
            )

        buffers = manifest.get(
            "buffers",
            {},
        )

        required = {
            "100",
            "500",
            "1000",
        }

        if not required.issubset(
            set(
                buffers
            )
        ):
            raise RuntimeError(
                "Buffer manifest must define 100, 500 and 1000 m. "
                f"Found: {sorted(buffers)}"
            )

        paths = {
            int(
                buffer_text
            ): Path(
                info[
                    "dataset_csv"
                ]
            )
            for (
                buffer_text,
                info,
            )
            in buffers.items()
            if int(
                buffer_text
            )
            in {
                100,
                500,
                1000,
            }
        }

    else:
        paths = dict(
            BufferConfig.DEFAULT_BUFFER_DATASETS
        )

    missing = [
        f"{buffer_m} m: {dataset_path}"
        for (
            buffer_m,
            dataset_path,
        )
        in paths.items()
        if not dataset_path.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Step 7.2C could not find one or more authoritative buffer "
            "datasets.\n"
            "The normal workflow expects 100 m under Step 6.4 and 500/1000 m "
            "under Step 6.3.\nMissing:\n"
            + "\n".join(
                missing
            )
        )

    return paths


def check_optional_step64_buffer_qa() -> None:
    """
    Read the Step 6.4 buffer QA table when available.
    Step 7.2C independently repeats all essential modelling checks.
    """
    path = (
        BufferConfig.STEP64_BUFFER_QA_CSV
    )

    if not path.exists():
        log(
            "[7.2C-WARN] Optional Step 6.4 buffer QA table was not found: "
            f"{path}"
        )
        return

    qa = pd.read_csv(
        path
    )

    if "buffer_m" not in qa.columns:
        raise RuntimeError(
            "Step 6.4 buffer QA file has no buffer_m column: "
            f"{path}"
        )

    for buffer_m in (
        100,
        500,
        1000,
    ):
        rows = qa.loc[
            pd.to_numeric(
                qa[
                    "buffer_m"
                ],
                errors="coerce",
            )
            == buffer_m
        ]

        if rows.empty:
            raise RuntimeError(
                "Step 6.4 buffer QA table has no row for "
                f"{buffer_m} m."
            )

        row = rows.iloc[
            0
        ]

        if "status" in qa.columns:
            status = str(
                row[
                    "status"
                ]
            ).strip().upper()

            if status not in {
                "PASS",
                "OK",
                "TRUE",
            }:
                raise RuntimeError(
                    "Step 6.4 buffer QA status is not PASS for "
                    f"{buffer_m} m: {row['status']}"
                )

        for column in (
            "same_positive_site_ids_as_100m",
            "same_positive_block_assignments_as_100m",
        ):
            if column in qa.columns:
                value = str(
                    row[
                        column
                    ]
                ).strip().lower()

                if value not in {
                    "true",
                    "1",
                    "yes",
                    "pass",
                }:
                    raise RuntimeError(
                        "Step 6.4 buffer QA failed "
                        f"{column} for {buffer_m} m."
                    )

    log(
        "[7.2C-QA] Step 6.4 buffer_dataset_consistency_QA.csv: PASS"
    )


# =============================================================================
# BUFFER DATASET VALIDATION
# =============================================================================
def load_and_validate_buffer_dataset(
    scenarios: Dict[str, Dict[str, Any]],
    dataset_csv: Path,
    buffer_m: int,
) -> pd.DataFrame:
    """
    Convert one buffer dataset to the exact Step 7.2 RF modelling contract.

    For 100 m the exact Step 6.4 model-ready dataset is used.
    For 500/1000 m, cv_fold is deterministically set equal to spatial_block
    when the alias is absent.
    """
    require_file(
        dataset_csv,
        f"{buffer_m} m buffer dataset",
    )

    df = pd.read_csv(
        dataset_csv
    )

    # Step 6.3 contains spatial_block. Step 7.2 uses the identical cv_fold alias.
    if (
        BufferConfig.BLOCK_COLUMN
        not in df.columns
    ):
        if (
            BufferConfig.FOLD_COLUMN
            not in df.columns
        ):
            raise RuntimeError(
                f"{buffer_m} m dataset contains neither "
                f"{BufferConfig.BLOCK_COLUMN} nor "
                f"{BufferConfig.FOLD_COLUMN}."
            )

        df[
            BufferConfig.BLOCK_COLUMN
        ] = pd.to_numeric(
            df[
                BufferConfig.FOLD_COLUMN
            ],
            errors="raise",
        ).astype(
            int
        )

    if (
        BufferConfig.FOLD_COLUMN
        not in df.columns
    ):
        df[
            BufferConfig.FOLD_COLUMN
        ] = pd.to_numeric(
            df[
                BufferConfig.BLOCK_COLUMN
            ],
            errors="raise",
        ).astype(
            int
        )

    all_features = all_scenario_features(
        scenarios
    )

    required = [
        BufferConfig.SAMPLE_ID_COLUMN,
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
        BufferConfig.BLOCK_COLUMN,
        *all_features,
    ]

    missing = [
        column
        for column in required
        if column not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{buffer_m} m dataset is missing required modelling columns: "
            f"{missing}"
        )

    if len(
        df
    ) != BufferConfig.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"{buffer_m} m: expected "
            f"{BufferConfig.EXPECTED_N_ROWS} rows; found {len(df)}."
        )

    if df[
        BufferConfig.SAMPLE_ID_COLUMN
    ].isna().any():
        raise RuntimeError(
            f"{buffer_m} m: missing sample_id values."
        )

    if df[
        BufferConfig.SAMPLE_ID_COLUMN
    ].duplicated().any():
        duplicates = df.loc[
            df[
                BufferConfig.SAMPLE_ID_COLUMN
            ].duplicated(
                keep=False
            ),
            BufferConfig.SAMPLE_ID_COLUMN,
        ].tolist()

        raise RuntimeError(
            f"{buffer_m} m: duplicated sample_id values found: "
            f"{duplicates[:20]}"
        )

    for column in (
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
        BufferConfig.BLOCK_COLUMN,
    ):
        df[
            column
        ] = pd.to_numeric(
            df[
                column
            ],
            errors="raise",
        ).astype(
            int
        )

    observed_class_counts = (
        df[
            BufferConfig.TARGET_COLUMN
        ]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    expected_class_counts = {
        0: BufferConfig.EXPECTED_N_NEGATIVE,
        1: BufferConfig.EXPECTED_N_POSITIVE,
    }

    if (
        observed_class_counts
        != expected_class_counts
    ):
        raise RuntimeError(
            f"{buffer_m} m: unexpected class counts "
            f"{observed_class_counts}; expected {expected_class_counts}."
        )

    observed_folds = sorted(
        df[
            BufferConfig.FOLD_COLUMN
        ].unique().tolist()
    )

    if (
        observed_folds
        != BufferConfig.EXPECTED_FOLDS
    ):
        raise RuntimeError(
            f"{buffer_m} m: expected frozen folds "
            f"{BufferConfig.EXPECTED_FOLDS}; found {observed_folds}."
        )

    if not np.array_equal(
        df[
            BufferConfig.FOLD_COLUMN
        ].to_numpy(),
        df[
            BufferConfig.BLOCK_COLUMN
        ].to_numpy(),
    ):
        raise RuntimeError(
            f"{buffer_m} m: cv_fold and spatial_block are not identical."
        )

    block_class = pd.crosstab(
        df[
            BufferConfig.FOLD_COLUMN
        ],
        df[
            BufferConfig.TARGET_COLUMN
        ],
    )

    expected_per_class = (
        BufferConfig.EXPECTED_N_POSITIVE
        // len(
            BufferConfig.EXPECTED_FOLDS
        )
    )

    for fold in BufferConfig.EXPECTED_FOLDS:
        if fold not in block_class.index:
            raise RuntimeError(
                f"{buffer_m} m: missing fold {fold}."
            )

        for label in (
            0,
            1,
        ):
            if label not in block_class.columns:
                raise RuntimeError(
                    f"{buffer_m} m, fold {fold}: missing class {label}."
                )

            count = int(
                block_class.loc[
                    fold,
                    label,
                ]
            )

            if (
                count
                != expected_per_class
            ):
                raise RuntimeError(
                    f"{buffer_m} m, fold {fold}, class {label}: "
                    f"found {count}; expected {expected_per_class}."
                )

    predictor_values = df[
        all_features
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    finite = np.isfinite(
        predictor_values.to_numpy(
            dtype="float64"
        )
    )

    if not finite.all():
        bad = np.argwhere(
            ~finite
        )

        examples = [
            (
                str(
                    df.iloc[
                        row_index
                    ][
                        BufferConfig.SAMPLE_ID_COLUMN
                    ]
                ),
                all_features[
                    column_index
                ],
            )
            for (
                row_index,
                column_index,
            )
            in bad[
                :20
            ]
        ]

        raise RuntimeError(
            f"{buffer_m} m: missing/non-finite predictor values remain. "
            f"Examples: {examples}"
        )

    irs_columns = [
        column
        for column in df.columns
        if "irs" in str(
            column
        ).lower()
    ]

    if irs_columns:
        raise RuntimeError(
            f"{buffer_m} m: IRS-related columns found: "
            f"{irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError(
            f"{buffer_m} m: tpi_s25 found; only tpi_s5 is allowed."
        )

    if "buffer_m" in df.columns:
        observed_buffer_values = sorted(
            pd.to_numeric(
                df[
                    "buffer_m"
                ],
                errors="coerce",
            )
            .dropna()
            .unique()
            .tolist()
        )

        contradictory = [
            value
            for value
            in observed_buffer_values
            if value not in {
                0,
                100,
                buffer_m,
            }
        ]

        if contradictory:
            raise RuntimeError(
                f"{buffer_m} m: contradictory buffer_m metadata values: "
                f"{contradictory}"
            )

    log(
        f"[7.2C-QA] {buffer_m} m model-ready conversion: PASS | "
        f"rows={len(df)}, positive={BufferConfig.EXPECTED_N_POSITIVE}, "
        f"negative={BufferConfig.EXPECTED_N_NEGATIVE}, "
        f"folds={observed_folds}"
    )

    return df


def validate_primary_100m_dataset(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
) -> None:
    """
    Verify that the in-memory 100 m dataset exactly matches the authoritative
    Step 6.4 model-ready CSV.
    """
    reference_path = (
        BufferConfig.DEFAULT_BUFFER_DATASETS[
            100
        ]
    )

    require_file(
        reference_path,
        "authoritative Step 6.4 100 m model-ready dataset",
    )

    reference = pd.read_csv(
        reference_path
    )

    if len(
        df
    ) != len(
        reference
    ):
        raise RuntimeError(
            "100 m dataset row count does not match the authoritative "
            "Step 6.4 dataset."
        )

    if not np.array_equal(
        df[
            BufferConfig.SAMPLE_ID_COLUMN
        ].astype(
            str
        ).to_numpy(),
        reference[
            BufferConfig.SAMPLE_ID_COLUMN
        ].astype(
            str
        ).to_numpy(),
    ):
        raise RuntimeError(
            "100 m sample IDs/order do not match the authoritative "
            "Step 6.4 dataset."
        )

    compare_columns = [
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
        BufferConfig.BLOCK_COLUMN,
        *all_scenario_features(
            scenarios
        ),
    ]

    for column in compare_columns:
        if column not in reference.columns:
            raise RuntimeError(
                "Authoritative 100 m dataset is missing column: "
                f"{column}"
            )

        left = df[
            column
        ]

        right = reference[
            column
        ]

        if (
            pd.api.types.is_numeric_dtype(
                left
            )
            and pd.api.types.is_numeric_dtype(
                right
            )
        ):
            equal = np.allclose(
                left.to_numpy(
                    dtype=float
                ),
                right.to_numpy(
                    dtype=float
                ),
                rtol=0.0,
                atol=1.0e-12,
                equal_nan=True,
            )

        else:
            equal = np.array_equal(
                left.astype(
                    str
                ).to_numpy(),
                right.astype(
                    str
                ).to_numpy(),
            )

        if not equal:
            raise RuntimeError(
                "100 m dataset differs from authoritative Step 6.4 in "
                f"column: {column}"
            )

    log(
        "[7.2C-QA] 100 m dataset exactly matches Step 6.4 primary input: PASS"
    )


def validate_same_positive_samples_across_buffers(
    reference_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
    predictor_columns: Sequence[str],
    candidate_buffer_m: int,
) -> None:
    """
    Verify that positive landslide samples are identical across buffer choices.

    Pseudo-absence rows are expected to change. Positive sample IDs, labels,
    spatial blocks and predictor values must remain fixed.
    """
    reference_positive = (
        reference_df.loc[
            reference_df[
                BufferConfig.TARGET_COLUMN
            ]
            == 1
        ]
        .copy()
        .sort_values(
            BufferConfig.SAMPLE_ID_COLUMN
        )
        .reset_index(
            drop=True
        )
    )

    candidate_positive = (
        candidate_df.loc[
            candidate_df[
                BufferConfig.TARGET_COLUMN
            ]
            == 1
        ]
        .copy()
        .sort_values(
            BufferConfig.SAMPLE_ID_COLUMN
        )
        .reset_index(
            drop=True
        )
    )

    if (
        len(
            reference_positive
        )
        != BufferConfig.EXPECTED_N_POSITIVE
        or len(
            candidate_positive
        )
        != BufferConfig.EXPECTED_N_POSITIVE
    ):
        raise RuntimeError(
            f"{candidate_buffer_m} m: unexpected positive sample count."
        )

    if not np.array_equal(
        reference_positive[
            BufferConfig.SAMPLE_ID_COLUMN
        ].astype(
            str
        ).to_numpy(),
        candidate_positive[
            BufferConfig.SAMPLE_ID_COLUMN
        ].astype(
            str
        ).to_numpy(),
    ):
        raise RuntimeError(
            f"{candidate_buffer_m} m: positive sample IDs differ from 100 m."
        )

    for column in (
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
        BufferConfig.BLOCK_COLUMN,
    ):
        if not np.array_equal(
            reference_positive[
                column
            ].to_numpy(),
            candidate_positive[
                column
            ].to_numpy(),
        ):
            raise RuntimeError(
                f"{candidate_buffer_m} m: positive rows differ in {column}."
            )

    mismatched_predictors: list[
        tuple[
            str,
            float,
        ]
    ] = []

    for column in predictor_columns:
        reference_values = reference_positive[
            column
        ].to_numpy(
            dtype=float
        )

        candidate_values = candidate_positive[
            column
        ].to_numpy(
            dtype=float
        )

        max_absolute_difference = float(
            np.max(
                np.abs(
                    reference_values
                    - candidate_values
                )
            )
        )

        if (
            max_absolute_difference
            > BufferConfig.POSITIVE_PREDICTOR_EQUALITY_ATOL
        ):
            mismatched_predictors.append(
                (
                    column,
                    max_absolute_difference,
                )
            )

    if mismatched_predictors:
        details = "\n".join(
            (
                f"  {column}: "
                f"max_abs_difference={difference:.12g}"
            )
            for (
                column,
                difference,
            )
            in mismatched_predictors
        )

        raise RuntimeError(
            f"{candidate_buffer_m} m: positive predictor values differ "
            "from the authoritative 100 m positives beyond tolerance "
            f"{BufferConfig.POSITIVE_PREDICTOR_EQUALITY_ATOL:.3g}.\n"
            f"{details}"
        )

    log(
        f"[7.2C-QA] {candidate_buffer_m} m positives match 100 m "
        "sample IDs, folds and predictors: PASS"
    )


# =============================================================================
# RANDOM FOREST PIPELINE
# =============================================================================
def build_rf_pipeline(
    features: Sequence[str],
) -> Pipeline:
    """
    Exact fold-safe preprocessing + Random Forest pipeline used in Step 7.2.
    """
    categorical = [
        feature
        for feature in features
        if is_categorical_feature(
            feature
        )
    ]

    numeric = [
        feature
        for feature in features
        if not is_categorical_feature(
            feature
        )
    ]

    transformers = []

    if numeric:
        transformers.append(
            (
                "numeric",
                "passthrough",
                numeric,
            )
        )

    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = RandomForestClassifier(
        **BufferConfig.RF_FIXED_PARAMS
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                classifier,
            ),
        ]
    )


def make_inner_predefined_split(
    outer_train_df: pd.DataFrame,
) -> PredefinedSplit:
    """
    Four-fold spatial inner CV using the four blocks remaining after one outer
    block is held out.
    """
    blocks = sorted(
        outer_train_df[
            BufferConfig.FOLD_COLUMN
        ].unique().tolist()
    )

    if len(
        blocks
    ) != 4:
        raise RuntimeError(
            "Nested CV expected four inner spatial blocks; "
            f"found {blocks}."
        )

    block_to_test_fold = {
        block: index
        for (
            index,
            block,
        )
        in enumerate(
            blocks
        )
    }

    test_fold = (
        outer_train_df[
            BufferConfig.FOLD_COLUMN
        ]
        .map(
            block_to_test_fold
        )
        .to_numpy()
    )

    return PredefinedSplit(
        test_fold=test_fold
    )


def classification_metrics(
    y_true: np.ndarray,
    y_probability: np.ndarray,
    threshold: float = (
        BufferConfig.BINARY_THRESHOLD
    ),
) -> Dict[str, Any]:
    predicted = (
        y_probability
        >= threshold
    ).astype(
        int
    )

    (
        tn,
        fp,
        fn,
        tp,
    ) = confusion_matrix(
        y_true,
        predicted,
        labels=[
            0,
            1,
        ],
    ).ravel()

    return {
        "roc_auc": float(
            roc_auc_score(
                y_true,
                y_probability,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                y_probability,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                y_probability,
            )
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                predicted,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predicted,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "threshold": float(
            threshold
        ),
        "tn": int(
            tn
        ),
        "fp": int(
            fp
        ),
        "fn": int(
            fn
        ),
        "tp": int(
            tp
        ),
    }


def normalize_grid_results(
    cv_results: Dict[str, Any],
    scenario_name: str,
    outer_fold: int,
) -> pd.DataFrame:
    frame = pd.DataFrame(
        cv_results
    )

    keep = [
        column
        for column in frame.columns
        if (
            column.startswith(
                "param_"
            )
            or column
            in {
                "mean_test_score",
                "std_test_score",
                "rank_test_score",
                "mean_fit_time",
                "std_fit_time",
            }
        )
    ]

    frame = frame[
        keep
    ].copy()

    frame.insert(
        0,
        "scenario",
        scenario_name,
    )

    frame.insert(
        1,
        "model",
        BufferConfig.MODEL_SHORT_LABELS[
            scenario_name
        ],
    )

    frame.insert(
        2,
        "search_stage",
        "nested_inner",
    )

    frame.insert(
        3,
        "outer_fold",
        int(
            outer_fold
        ),
    )

    return frame


def run_nested_spatial_cv_for_scenario(
    df: pd.DataFrame,
    scenario_name: str,
    features: Sequence[str],
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    Dict[str, Any],
    pd.DataFrame,
]:
    """
    Exact successful Step 7.2 nested RF contract:
      five outer spatial folds;
      four-block inner GridSearchCV;
      48 RF hyperparameter combinations.
    """
    oof_probability = np.full(
        len(
            df
        ),
        np.nan,
        dtype="float64",
    )

    oof_predicted = np.full(
        len(
            df
        ),
        -1,
        dtype="int16",
    )

    fold_rows: list[
        Dict[str, Any]
    ] = []

    best_params: Dict[
        str,
        Any,
    ] = {}

    search_tables: list[
        pd.DataFrame
    ] = []

    for outer_fold in BufferConfig.EXPECTED_FOLDS:
        outer_train_mask = (
            df[
                BufferConfig.FOLD_COLUMN
            ].to_numpy()
            != outer_fold
        )

        outer_valid_mask = (
            df[
                BufferConfig.FOLD_COLUMN
            ].to_numpy()
            == outer_fold
        )

        train_df = df.loc[
            outer_train_mask
        ].copy()

        valid_df = df.loc[
            outer_valid_mask
        ].copy()

        if train_df[
            BufferConfig.FOLD_COLUMN
        ].nunique() != 4:
            raise RuntimeError(
                f"Outer fold {outer_fold}: "
                "expected four training blocks."
            )

        if valid_df[
            BufferConfig.FOLD_COLUMN
        ].nunique() != 1:
            raise RuntimeError(
                f"Outer fold {outer_fold}: "
                "validation must contain exactly one block."
            )

        X_train = train_df[
            list(
                features
            )
        ]

        y_train = train_df[
            BufferConfig.TARGET_COLUMN
        ].to_numpy(
            dtype=int
        )

        X_valid = valid_df[
            list(
                features
            )
        ]

        y_valid = valid_df[
            BufferConfig.TARGET_COLUMN
        ].to_numpy(
            dtype=int
        )

        inner_cv = make_inner_predefined_split(
            train_df
        )

        search = GridSearchCV(
            estimator=build_rf_pipeline(
                features
            ),
            param_grid=BufferConfig.PARAM_GRID,
            scoring="roc_auc",
            cv=inner_cv,
            refit=True,
            n_jobs=BufferConfig.GRIDSEARCH_N_JOBS,
            verbose=BufferConfig.GRIDSEARCH_VERBOSE,
            return_train_score=False,
            error_score="raise",
        )

        log(
            f"[OUTER {outer_fold}] RF tuning | "
            f"{BufferConfig.MODEL_SHORT_LABELS[scenario_name]} | "
            f"training blocks="
            f"{sorted(train_df[BufferConfig.FOLD_COLUMN].unique().tolist())} | "
            f"held-out block={outer_fold}"
        )

        search.fit(
            X_train,
            y_train,
        )

        probability = (
            search.best_estimator_.predict_proba(
                X_valid
            )[
                :,
                1,
            ]
        )

        predicted = (
            probability
            >= BufferConfig.BINARY_THRESHOLD
        ).astype(
            int
        )

        valid_indices = np.flatnonzero(
            outer_valid_mask
        )

        oof_probability[
            valid_indices
        ] = probability

        oof_predicted[
            valid_indices
        ] = predicted

        metrics = classification_metrics(
            y_valid,
            probability,
        )

        fold_rows.append(
            {
                "scenario": scenario_name,
                "model": BufferConfig.MODEL_SHORT_LABELS[
                    scenario_name
                ],
                "outer_fold": int(
                    outer_fold
                ),
                "n_predictors": int(
                    len(
                        features
                    )
                ),
                "n_train": int(
                    len(
                        train_df
                    )
                ),
                "n_validation": int(
                    len(
                        valid_df
                    )
                ),
                "inner_best_mean_auc": float(
                    search.best_score_
                ),
                **metrics,
            }
        )

        best_params[
            str(
                outer_fold
            )
        ] = {
            "inner_best_mean_auc": float(
                search.best_score_
            ),
            "best_params": json_safe(
                search.best_params_
            ),
        }

        search_tables.append(
            normalize_grid_results(
                search.cv_results_,
                scenario_name=scenario_name,
                outer_fold=outer_fold,
            )
        )

        log(
            f"[OUTER {outer_fold}] "
            f"inner best AUC={search.best_score_:.4f}; "
            f"held-out AUC={metrics['roc_auc']:.4f}; "
            f"F1={metrics['f1']:.4f}"
        )

        del search
        gc.collect()

    if not np.isfinite(
        oof_probability
    ).all():
        raise RuntimeError(
            f"{scenario_name}: incomplete OOF probability predictions."
        )

    if (
        oof_predicted
        < 0
    ).any():
        raise RuntimeError(
            f"{scenario_name}: incomplete OOF class predictions."
        )

    oof_df = df[
        [
            BufferConfig.SAMPLE_ID_COLUMN,
            BufferConfig.TARGET_COLUMN,
            BufferConfig.FOLD_COLUMN,
            BufferConfig.BLOCK_COLUMN,
        ]
    ].copy()

    oof_df.insert(
        0,
        "scenario",
        scenario_name,
    )

    oof_df.insert(
        1,
        "model",
        BufferConfig.MODEL_SHORT_LABELS[
            scenario_name
        ],
    )

    oof_df[
        "oof_probability"
    ] = oof_probability

    oof_df[
        "oof_predicted_class_0p5"
    ] = oof_predicted

    return (
        oof_df,
        pd.DataFrame(
            fold_rows
        ),
        best_params,
        pd.concat(
            search_tables,
            ignore_index=True,
        ),
    )


# =============================================================================
# SPATIAL-BLOCK BOOTSTRAP
# =============================================================================
def cluster_bootstrap_auc_ci(
    y_true: np.ndarray,
    y_probability: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> tuple[
    float,
    float,
    int,
]:
    rng = np.random.default_rng(
        seed
    )

    unique_blocks = np.unique(
        blocks
    )

    indices_by_block = {
        block: np.flatnonzero(
            blocks
            == block
        )
        for block
        in unique_blocks
    }

    values: list[
        float
    ] = []

    for _ in range(
        n_bootstrap
    ):
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(
                unique_blocks
            ),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                indices_by_block[
                    block
                ]
                for block
                in sampled_blocks
            ]
        )

        y_bootstrap = y_true[
            sampled_indices
        ]

        if np.unique(
            y_bootstrap
        ).size < 2:
            continue

        values.append(
            float(
                roc_auc_score(
                    y_bootstrap,
                    y_probability[
                        sampled_indices
                    ],
                )
            )
        )

    if not values:
        return (
            float(
                "nan"
            ),
            float(
                "nan"
            ),
            0,
        )

    alpha = (
        1.0
        - ci_level
    ) / 2.0

    low, high = np.quantile(
        values,
        [
            alpha,
            1.0
            - alpha,
        ],
    )

    return (
        float(
            low
        ),
        float(
            high
        ),
        len(
            values
        ),
    )


def paired_cluster_bootstrap_auc_difference(
    y_true: np.ndarray,
    probability_left: np.ndarray,
    probability_right: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> tuple[
    float,
    float,
    float,
    int,
]:
    observed = float(
        roc_auc_score(
            y_true,
            probability_left,
        )
        - roc_auc_score(
            y_true,
            probability_right,
        )
    )

    rng = np.random.default_rng(
        seed
    )

    unique_blocks = np.unique(
        blocks
    )

    indices_by_block = {
        block: np.flatnonzero(
            blocks
            == block
        )
        for block
        in unique_blocks
    }

    differences: list[
        float
    ] = []

    for _ in range(
        n_bootstrap
    ):
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(
                unique_blocks
            ),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                indices_by_block[
                    block
                ]
                for block
                in sampled_blocks
            ]
        )

        y_bootstrap = y_true[
            sampled_indices
        ]

        if np.unique(
            y_bootstrap
        ).size < 2:
            continue

        differences.append(
            float(
                roc_auc_score(
                    y_bootstrap,
                    probability_left[
                        sampled_indices
                    ],
                )
                - roc_auc_score(
                    y_bootstrap,
                    probability_right[
                        sampled_indices
                    ],
                )
            )
        )

    if not differences:
        return (
            observed,
            float(
                "nan"
            ),
            float(
                "nan"
            ),
            0,
        )

    alpha = (
        1.0
        - ci_level
    ) / 2.0

    low, high = np.quantile(
        differences,
        [
            alpha,
            1.0
            - alpha,
        ],
    )

    return (
        observed,
        float(
            low
        ),
        float(
            high
        ),
        len(
            differences
        ),
    )


# =============================================================================
# SUCCESSFUL STEP 7.2 OOF REPRODUCTION
# =============================================================================
def load_primary_step72_oof(
    scenario_name: str,
) -> pd.DataFrame:
    require_file(
        BufferConfig.PRIMARY_STEP72_OOF_CSV,
        "successful Step 7.2 RF A-F OOF predictions",
    )

    all_oof = pd.read_csv(
        BufferConfig.PRIMARY_STEP72_OOF_CSV
    )

    required = {
        "scenario",
        BufferConfig.SAMPLE_ID_COLUMN,
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
        "oof_probability",
    }

    missing = (
        required
        - set(
            all_oof.columns
        )
    )

    if missing:
        raise RuntimeError(
            "Successful Step 7.2 OOF file is missing columns: "
            f"{sorted(missing)}"
        )

    subset = all_oof.loc[
        all_oof[
            "scenario"
        ]
        == scenario_name
    ].copy()

    if len(
        subset
    ) != BufferConfig.EXPECTED_N_ROWS:
        raise RuntimeError(
            "Successful Step 7.2 OOF row count for "
            f"{scenario_name} is {len(subset)}; expected "
            f"{BufferConfig.EXPECTED_N_ROWS}."
        )

    return subset


def assert_oof_reproduces_primary(
    candidate_oof: pd.DataFrame,
    scenario_name: str,
) -> Dict[str, Any]:
    reference = load_primary_step72_oof(
        scenario_name
    )

    candidate = candidate_oof[
        [
            BufferConfig.SAMPLE_ID_COLUMN,
            BufferConfig.TARGET_COLUMN,
            BufferConfig.FOLD_COLUMN,
            "oof_probability",
        ]
    ].copy()

    candidate = candidate.rename(
        columns={
            "oof_probability": (
                "candidate_probability"
            )
        }
    )

    reference = reference[
        [
            BufferConfig.SAMPLE_ID_COLUMN,
            BufferConfig.TARGET_COLUMN,
            BufferConfig.FOLD_COLUMN,
            "oof_probability",
        ]
    ].copy()

    reference = reference.rename(
        columns={
            "oof_probability": (
                "reference_probability"
            )
        }
    )

    merged = candidate.merge(
        reference,
        on=BufferConfig.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=(
            "_candidate",
            "_reference",
        ),
        validate="one_to_one",
    )

    if len(
        merged
    ) != BufferConfig.EXPECTED_N_ROWS:
        raise RuntimeError(
            "Step 7.2C 100 m reproduction merge returned "
            f"{len(merged)} rows for {scenario_name}."
        )

    for column in (
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
    ):
        if not np.array_equal(
            merged[
                f"{column}_candidate"
            ].to_numpy(),
            merged[
                f"{column}_reference"
            ].to_numpy(),
        ):
            raise RuntimeError(
                "Step 7.2C 100 m reproduction metadata mismatch in "
                f"{column} for {scenario_name}."
            )

    difference = (
        merged[
            "candidate_probability"
        ].to_numpy(
            dtype=float
        )
        - merged[
            "reference_probability"
        ].to_numpy(
            dtype=float
        )
    )

    maximum_absolute_difference = float(
        np.max(
            np.abs(
                difference
            )
        )
    )

    mean_absolute_difference = float(
        np.mean(
            np.abs(
                difference
            )
        )
    )

    passed = bool(
        maximum_absolute_difference
        <= BufferConfig.OOF_REPRODUCTION_TOLERANCE
    )

    if not passed:
        raise RuntimeError(
            "Step 7.2C 100 m rerun did not reproduce the successful "
            f"Step 7.2 {scenario_name} RF OOF probabilities.\n"
            f"Maximum absolute difference = "
            f"{maximum_absolute_difference:.12g}\n"
            f"Tolerance = "
            f"{BufferConfig.OOF_REPRODUCTION_TOLERANCE:.12g}"
        )

    return {
        "oof_reproduction_pass": passed,
        "oof_reproduction_maximum_absolute_probability_difference": (
            maximum_absolute_difference
        ),
        "oof_reproduction_mean_absolute_probability_difference": (
            mean_absolute_difference
        ),
        "oof_reproduction_tolerance": float(
            BufferConfig.OOF_REPRODUCTION_TOLERANCE
        ),
    }


# =============================================================================
# PERFORMANCE SUMMARIES / PAIRED E-A COMPARISON
# =============================================================================
def summarize_oof(
    oof_df: pd.DataFrame,
    buffer_m: int,
    scenario_name: str,
) -> Dict[str, Any]:
    y = oof_df[
        BufferConfig.TARGET_COLUMN
    ].to_numpy(
        dtype=int
    )

    probability = oof_df[
        "oof_probability"
    ].to_numpy(
        dtype=float
    )

    blocks = oof_df[
        BufferConfig.FOLD_COLUMN
    ].to_numpy(
        dtype=int
    )

    metrics = classification_metrics(
        y,
        probability,
    )

    (
        ci_low,
        ci_high,
        n_valid,
    ) = cluster_bootstrap_auc_ci(
        y_true=y,
        y_probability=probability,
        blocks=blocks,
        n_bootstrap=BufferConfig.N_CLUSTER_BOOTSTRAP,
        seed=(
            BufferConfig.RANDOM_STATE
            + buffer_m
            + BufferConfig.SCENARIO_ORDER.index(
                scenario_name
            )
        ),
        ci_level=BufferConfig.CI_LEVEL,
    )

    return {
        "buffer_m": int(
            buffer_m
        ),
        "scenario": scenario_name,
        "model": BufferConfig.MODEL_SHORT_LABELS[
            scenario_name
        ],
        "oof_roc_auc": metrics[
            "roc_auc"
        ],
        "auc_ci_low": ci_low,
        "auc_ci_high": ci_high,
        "oof_pr_auc": metrics[
            "pr_auc"
        ],
        "oof_balanced_accuracy": metrics[
            "balanced_accuracy"
        ],
        "oof_f1": metrics[
            "f1"
        ],
        "oof_brier": metrics[
            "brier"
        ],
        "valid_auc_bootstrap_iterations": int(
            n_valid
        ),
    }


def paired_auc_difference_from_oof(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    label: str,
    seed: int,
) -> Dict[str, Any]:
    left = left_df[
        [
            BufferConfig.SAMPLE_ID_COLUMN,
            BufferConfig.TARGET_COLUMN,
            BufferConfig.FOLD_COLUMN,
            "oof_probability",
        ]
    ].copy()

    left = left.rename(
        columns={
            "oof_probability": (
                "probability_left"
            )
        }
    )

    right = right_df[
        [
            BufferConfig.SAMPLE_ID_COLUMN,
            BufferConfig.TARGET_COLUMN,
            BufferConfig.FOLD_COLUMN,
            "oof_probability",
        ]
    ].copy()

    right = right.rename(
        columns={
            "oof_probability": (
                "probability_right"
            )
        }
    )

    merged = left.merge(
        right,
        on=BufferConfig.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=(
            "_left",
            "_right",
        ),
        validate="one_to_one",
    )

    if len(
        merged
    ) != BufferConfig.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Paired comparison {label} returned "
            f"{len(merged)} rows; expected "
            f"{BufferConfig.EXPECTED_N_ROWS}."
        )

    for column in (
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
    ):
        if not np.array_equal(
            merged[
                f"{column}_left"
            ].to_numpy(),
            merged[
                f"{column}_right"
            ].to_numpy(),
        ):
            raise RuntimeError(
                f"Paired comparison metadata mismatch in {column}: "
                f"{label}"
            )

    y = merged[
        f"{BufferConfig.TARGET_COLUMN}_left"
    ].to_numpy(
        dtype=int
    )

    blocks = merged[
        f"{BufferConfig.FOLD_COLUMN}_left"
    ].to_numpy(
        dtype=int
    )

    (
        observed,
        ci_low,
        ci_high,
        n_valid,
    ) = paired_cluster_bootstrap_auc_difference(
        y_true=y,
        probability_left=merged[
            "probability_left"
        ].to_numpy(
            dtype=float
        ),
        probability_right=merged[
            "probability_right"
        ].to_numpy(
            dtype=float
        ),
        blocks=blocks,
        n_bootstrap=BufferConfig.N_CLUSTER_BOOTSTRAP,
        seed=seed,
        ci_level=BufferConfig.CI_LEVEL,
    )

    return {
        "comparison": label,
        "auc_difference_left_minus_right": observed,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "valid_bootstrap_iterations": int(
            n_valid
        ),
    }


# =============================================================================
# PROVENANCE
# =============================================================================
def write_diagnostic_provenance(
    buffer_paths: Dict[int, Path],
) -> None:
    BufferConfig.OUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "diagnostic_step": "Step 7.2C",
        "algorithm": "Random Forest",
        "scientific_purpose": (
            "Cross-algorithm pseudo-absence exclusion-buffer sensitivity for "
            "Models A and E using the exact successful Step 7.2 RF nested "
            "spatial-CV contract."
        ),
        "buffers_m": [
            100,
            500,
            1000,
        ],
        "scenarios": list(
            BufferConfig.SCENARIOS
        ),
        "buffer_dataset_paths": {
            str(
                key
            ): str(
                value
            )
            for (
                key,
                value,
            )
            in buffer_paths.items()
        },
        "input_rule": (
            "100 m uses the exact Step 6.4 primary dataset; 500/1000 m use "
            "Step 6.3 buffer datasets converted in memory with "
            "cv_fold=spatial_block and strict Step 7.2 model-ready QA."
        ),
        "primary_step72_oof_reference": str(
            BufferConfig.PRIMARY_STEP72_OOF_CSV
        ),
        "rf_parameter_grid": json_safe(
            BufferConfig.PARAM_GRID
        ),
        "rf_fixed_parameters": json_safe(
            BufferConfig.RF_FIXED_PARAMS
        ),
        "positive_predictor_equality_tolerance": float(
            BufferConfig.POSITIVE_PREDICTOR_EQUALITY_ATOL
        ),
        "plot_style": (
            "Exact Step 7.1C figure geometry and Matplotlib commands, with RF "
            "results and RF-prefixed output filename."
        ),
    }

    with open(
        BufferConfig.OUT_DIR
        / "Step_7_2C_RF_Buffer_Sensitivity_Provenance.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(
                metadata
            ),
            handle,
            indent=2,
        )


# =============================================================================
# EXACT STEP 7.1C-STYLE PLOT
# =============================================================================
def create_buffer_sensitivity_plot(
    summary_df: pd.DataFrame,
) -> Path:
    """
    Exact same plotting style used in Step 7.1C, now using RF results.
    """
    figure_path = (
        BufferConfig.OUT_DIR
        / "RF_7_2C_A_E_Buffer_Sensitivity.png"
    )

    fig, ax = plt.subplots(
        figsize=(
            5.8,
            4.2,
        )
    )

    for scenario_name in BufferConfig.SCENARIOS:
        subset = summary_df.loc[
            summary_df[
                "scenario"
            ]
            == scenario_name
        ].sort_values(
            "buffer_m"
        )

        x = subset[
            "buffer_m"
        ].to_numpy(
            dtype=float
        )

        y = subset[
            "oof_roc_auc"
        ].to_numpy(
            dtype=float
        )

        lower = (
            y
            - subset[
                "auc_ci_low"
            ].to_numpy(
                dtype=float
            )
        )

        upper = (
            subset[
                "auc_ci_high"
            ].to_numpy(
                dtype=float
            )
            - y
        )

        ax.errorbar(
            x,
            y,
            yerr=np.vstack(
                [
                    lower,
                    upper,
                ]
            ),
            marker="o",
            capsize=3,
            linewidth=1.4,
            label=BufferConfig.MODEL_SHORT_LABELS[
                scenario_name
            ],
        )

    ax.set_xlabel(
        "Pseudo-absence exclusion buffer (m)"
    )

    ax.set_ylabel(
        "Nested OOF ROC-AUC"
    )

    ax.set_xticks(
        [
            100,
            500,
            1000,
        ]
    )

    ax.set_ylim(
        max(
            0.5,
            float(
                np.nanmin(
                    summary_df[
                        "auc_ci_low"
                    ]
                )
            )
            - 0.03,
        ),
        1.0,
    )

    ax.grid(
        True,
        alpha=0.25,
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    fig.savefig(
        figure_path,
        dpi=BufferConfig.FIG_DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

    return figure_path


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log(
        "="
        * 96
    )

    log(
        "STEP 7.2C — RANDOM FOREST PSEUDO-ABSENCE BUFFER SENSITIVITY"
    )

    log(
        "="
        * 96
    )

    scenarios = validate_scenario_manifests()

    predictor_columns = all_scenario_features(
        scenarios
    )

    buffer_paths = load_buffer_paths(
        BufferConfig.MANIFEST_PATH
    )

    BufferConfig.OUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    write_diagnostic_provenance(
        buffer_paths
    )

    check_optional_step64_buffer_qa()

    datasets: Dict[
        int,
        pd.DataFrame,
    ] = {}

    summary_rows: list[
        Dict[str, Any]
    ] = []

    comparison_rows: list[
        Dict[str, Any]
    ] = []

    reproduction_rows: list[
        Dict[str, Any]
    ] = []

    prepared_input_dir = (
        BufferConfig.OUT_DIR
        / "prepared_model_ready_inputs"
    )

    prepared_input_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # -------------------------------------------------------------------------
    # 1. Load, convert and validate all buffer datasets before any model fitting.
    # -------------------------------------------------------------------------
    for buffer_m in (
        100,
        500,
        1000,
    ):
        dataset_path = Path(
            buffer_paths[
                buffer_m
            ]
        )

        log(
            f"[7.2C] Loading {buffer_m} m buffer source dataset: "
            f"{dataset_path}"
        )

        datasets[
            buffer_m
        ] = load_and_validate_buffer_dataset(
            scenarios=scenarios,
            dataset_csv=dataset_path,
            buffer_m=buffer_m,
        )

        prepared_path = (
            prepared_input_dir
            / (
                "model_ready_dataset_A_to_F_buffer_"
                f"{buffer_m}m.csv"
            )
        )

        datasets[
            buffer_m
        ].to_csv(
            prepared_path,
            index=False,
            float_format="%.10f",
        )

    # The 100 m dataset must be exactly the authoritative primary dataset.
    validate_primary_100m_dataset(
        datasets[
            100
        ],
        scenarios,
    )

    # Positive landslide samples and predictors must remain fixed across buffers.
    for buffer_m in (
        500,
        1000,
    ):
        validate_same_positive_samples_across_buffers(
            reference_df=datasets[
                100
            ],
            candidate_df=datasets[
                buffer_m
            ],
            predictor_columns=predictor_columns,
            candidate_buffer_m=buffer_m,
        )

    # -------------------------------------------------------------------------
    # 2. Run independent nested RF tuning for Models A and E at each buffer.
    # -------------------------------------------------------------------------
    for buffer_m in (
        100,
        500,
        1000,
    ):
        df = datasets[
            buffer_m
        ]

        per_buffer_oof: Dict[
            str,
            pd.DataFrame,
        ] = {}

        for scenario_name in BufferConfig.SCENARIOS:
            features = list(
                scenarios[
                    scenario_name
                ][
                    "features"
                ]
            )

            log(
                f"[7.2C] Nested spatial RF CV | "
                f"buffer={buffer_m} m | "
                f"{BufferConfig.MODEL_SHORT_LABELS[scenario_name]}"
            )

            (
                oof_df,
                fold_df,
                params,
                search_df,
            ) = run_nested_spatial_cv_for_scenario(
                df=df,
                scenario_name=scenario_name,
                features=features,
            )

            # The 100 m experiment must reproduce successful Step 7.2 RF A/E.
            if (
                buffer_m
                == BufferConfig.REFERENCE_BUFFER_M
            ):
                qa = assert_oof_reproduces_primary(
                    candidate_oof=oof_df,
                    scenario_name=scenario_name,
                )

                reproduction_rows.append(
                    {
                        "buffer_m": int(
                            buffer_m
                        ),
                        "scenario": scenario_name,
                        "model": BufferConfig.MODEL_SHORT_LABELS[
                            scenario_name
                        ],
                        **qa,
                    }
                )

            oof_df = oof_df.copy()

            oof_df.insert(
                0,
                "buffer_m",
                int(
                    buffer_m
                ),
            )

            fold_df = fold_df.copy()

            fold_df.insert(
                0,
                "buffer_m",
                int(
                    buffer_m
                ),
            )

            search_df = search_df.copy()

            search_df.insert(
                0,
                "buffer_m",
                int(
                    buffer_m
                ),
            )

            per_buffer_oof[
                scenario_name
            ] = oof_df

            summary_rows.append(
                summarize_oof(
                    oof_df=oof_df,
                    buffer_m=buffer_m,
                    scenario_name=scenario_name,
                )
            )

            prefix = (
                f"RF_7_2C_buffer_{buffer_m}m_"
                f"{BufferConfig.MODEL_SHORT_LABELS[scenario_name].replace(' ', '_')}"
            )

            oof_df.to_csv(
                BufferConfig.OUT_DIR
                / f"{prefix}_OOF.csv",
                index=False,
                float_format="%.10f",
            )

            fold_df.to_csv(
                BufferConfig.OUT_DIR
                / f"{prefix}_Fold_Metrics.csv",
                index=False,
                float_format="%.10f",
            )

            search_df.to_csv(
                BufferConfig.OUT_DIR
                / f"{prefix}_Inner_Search.csv",
                index=False,
                float_format="%.10f",
            )

            with open(
                BufferConfig.OUT_DIR
                / f"{prefix}_Outer_Params.json",
                "w",
                encoding="utf-8",
            ) as handle:
                json.dump(
                    json_safe(
                        params
                    ),
                    handle,
                    indent=2,
                )

            gc.collect()

        # Main reviewer comparison: Model E - Model A within the same buffer.
        comparison_rows.append(
            {
                "buffer_m": int(
                    buffer_m
                ),
                **paired_auc_difference_from_oof(
                    left_df=per_buffer_oof[
                        "E_physics_guided"
                    ],
                    right_df=per_buffer_oof[
                        "A_static_baseline"
                    ],
                    label=(
                        f"E - A at {buffer_m} m"
                    ),
                    seed=(
                        BufferConfig.RANDOM_STATE
                        + 10000
                        + buffer_m
                    ),
                ),
            }
        )

    # -------------------------------------------------------------------------
    # 3. Save QA, performance and paired-comparison tables.
    # -------------------------------------------------------------------------
    reproduction_df = pd.DataFrame(
        reproduction_rows
    )

    reproduction_df.to_csv(
        BufferConfig.OUT_DIR
        / "RF_7_2C_100m_Reproduction_QA.csv",
        index=False,
        float_format="%.10f",
    )

    summary_df = (
        pd.DataFrame(
            summary_rows
        )
        .sort_values(
            [
                "buffer_m",
                "scenario",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    comparison_df = (
        pd.DataFrame(
            comparison_rows
        )
        .sort_values(
            "buffer_m"
        )
        .reset_index(
            drop=True
        )
    )

    summary_df.to_csv(
        BufferConfig.OUT_DIR
        / "RF_7_2C_A_E_Buffer_Sensitivity_Performance.csv",
        index=False,
        float_format="%.10f",
    )

    comparison_df.to_csv(
        BufferConfig.OUT_DIR
        / "RF_7_2C_A_E_Buffer_Sensitivity_Paired_AUC.csv",
        index=False,
        float_format="%.10f",
    )

    # -------------------------------------------------------------------------
    # 4. Reproduce the exact Step 7.1C plot style using RF results.
    # -------------------------------------------------------------------------
    figure_path = create_buffer_sensitivity_plot(
        summary_df
    )

    log(
        ""
    )

    log(
        "="
        * 96
    )

    log(
        "STEP 7.2C COMPLETED SUCCESSFULLY"
    )

    log(
        "="
        * 96
    )

    log(
        f"[OUT] {BufferConfig.OUT_DIR}"
    )

    log(
        f"[FIGURE] {figure_path}"
    )

    print(
        "\n100 m Step 7.2 reproduction QA:"
    )

    print(
        reproduction_df.to_string(
            index=False
        )
    )

    print(
        "\nPerformance summary:"
    )

    print(
        summary_df.to_string(
            index=False
        )
    )

    print(
        "\nE - A paired AUC differences within each buffer:"
    )

    print(
        comparison_df.to_string(
            index=False
        )
    )


if __name__ == "__main__":
    main()


# STEP 7.2D — RANDOM FOREST PERFORMANCE AND MAPPED-AREA UNCERTAINTY

In [ ]:
# =============================================================================
# STEP 7.2D — RANDOM FOREST PERFORMANCE AND MAPPED-AREA UNCERTAINTY
# =============================================================================
#
# Scientific purpose
# ------------------
# This script is the Random Forest counterpart to Step 7.1D (XGBoost) and
# reproduces the same uncertainty-analysis architecture using the exact
# successful Step 7.2 Random Forest modelling contract.
#
# Part 1 — Performance uncertainty
# --------------------------------
# Extend spatial-block cluster-bootstrap uncertainty from ROC-AUC to:
#     - ROC-AUC
#     - PR-AUC
#     - balanced accuracy
#     - F1
#     - Brier score
#     - precision
#     - recall
#
# The analysis uses the already saved genuine nested outer-fold OOF predictions
# from successful Step 7.2. No model is refitted for performance confidence
# intervals.
#
# Uncertainty is reported for:
#     - Models A-F
#     - the separately reported fixed A30/E70 RF operational ensemble
#
# Direct paired spatial-block bootstrap comparisons are also added for:
#     - Model E - Model C
#     - Model E - Model D
#
#
# Part 2 — Mapped-area / model variability
# ----------------------------------------
# For each of the five successful outer folds:
#     1. omit that spatial block from training;
#     2. refit RF Model A using that fold's already selected Step 7.2
#        inner-CV hyperparameters;
#     3. refit RF Model E using that fold's already selected Step 7.2
#        inner-CV hyperparameters;
#     4. predict the complete available domain for A and E;
#     5. create the fixed RF A30/E70 ensemble map;
#     6. calculate susceptibility-class area percentages.
#
# The five fold-specific map realizations are summarized as:
#     - mean
#     - standard deviation
#     - minimum
#     - maximum
#
# These statistics are described as:
#     "cross-fold mapped-area variability"
#     or
#     "cross-fold model uncertainty"
#
# They are NOT formal 95% confidence intervals because only five outer-fold map
# realizations are available.
#
#
# Critical scientific rules
# -------------------------
# 1. The primary RF model remains the successful Step 7.2 100 m / equal-HKFI
#    framework.
# 2. Step 7.2B alternative HKFI weights are NOT mixed into Step 7.2D.
# 3. Step 7.2C 500/1000 m pseudo-absence buffers are NOT mixed into Step 7.2D.
# 4. Performance uncertainty is based only on genuine Step 7.2 nested OOF
#    predictions.
# 5. The five map realizations reuse the exact successful Step 7.2 fold-specific
#    best parameters; there is no second hyperparameter search.
# 6. The final full-data RF A30/E70 map is shown alongside the cross-fold
#    mapped-area statistics.
# 7. All five cross-fold ensemble maps must have exactly the same valid-pixel
#    footprint, and this footprint must match the successful final RF ensemble.
#
#
# Technical robustness
# --------------------
# - Each outer-fold map realization runs in a separate Python process.
# - A completion checkpoint is written per fold.
# - Completed folds are reused automatically on restart.
# - Random Forest and numerical-library thread counts are constrained to avoid
#   nested CPU oversubscription.
#
#
# Exact successful Step 7.2 RF contract
# -------------------------------------
# Hyperparameter grid:
#     n_estimators      = [200, 500]
#     max_depth         = [None, 10, 20]
#     min_samples_leaf  = [1, 2]
#     max_features      = ["sqrt", 0.5]
#     criterion         = ["gini", "entropy"]
#
# Fixed RF parameters:
#     bootstrap    = True
#     class_weight = None
#     random_state = 42
#     n_jobs        = 1
#
#
# Recommended execution
# ---------------------
# Save and run this file as:
#
#     Step_7_2D_RF_Performance_and_Mapped_Area_Uncertainty.py
#
# Running the saved .py file is recommended because process-isolated map workers
# relaunch this same script.
# =============================================================================

from __future__ import annotations

import argparse
import gc
import json
import os
import re
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Sequence

# Prevent nested CPU oversubscription before importing numerical libraries.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import PathPatch, Polygon, Rectangle
from matplotlib.path import Path as MplPath
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, transform_geom
from rasterio.windows import Window

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    fiona = None
    HAVE_FIONA = False
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


# =============================================================================
# CONFIGURATION — FROZEN TO SUCCESSFUL STEP 7.2
# =============================================================================
class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Step 6.1 frozen predictor definitions and master raster stack.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = (
        STEP61_DIR / "model_ablation_scenarios.json"
    )
    MASTER_STACK_TIF = (
        STEP61_DIR
        / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
    )

    # Step 6.4 frozen modelling contract.
    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    MODEL_READY_DATASET_CSV = (
        STEP64_DIR / "model_ready_dataset_A_to_F.csv"
    )
    STEP64_MODEL_CONFIG_JSON = (
        STEP64_DIR / "model_configuration_manifest.json"
    )

    # Successful Step 7.2 RF outputs.
    OUT_DIR = RESULTS_DIR / "Step-7.2-RF-Models_A_to_F"
    MAP_DIR = OUT_DIR / "susceptibility_maps"

    OOF_PREDICTIONS_CSV = (
        OUT_DIR / "RF_A_to_F_OOF_predictions.csv"
    )
    OUTER_PARAMS_JSON = (
        OUT_DIR / "RF_A_to_F_outer_fold_best_parameters.json"
    )
    ENSEMBLE_OOF_PREDICTIONS_CSV = (
        OUT_DIR / "RF_Ensemble_A30_E70_OOF_predictions.csv"
    )
    ENSEMBLE_PROBABILITY_TIF = (
        MAP_DIR / "RF_Ensemble_A30_E70_probability_full_study_domain.tif"
    )

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    CATEGORICAL_FEATURE_ALIASES = {
        "landcover",
        "landuse",
        "land_use",
    }

    # Exact successful Step 7.2 fixed RF settings.
    RF_FIXED_PARAMS = {
        "bootstrap": True,
        "class_weight": None,
        "random_state": 42,
        "n_jobs": 1,
    }

    BINARY_THRESHOLD = 0.50
    ENSEMBLE_ENHANCED_WEIGHT = 0.70
    OUTPUT_NODATA = -9999.0


class UncertaintyConfig:
    # Performance uncertainty.
    N_BOOTSTRAP = 5000
    CI_LEVEL = 0.95
    RANDOM_STATE = 52571

    # Map uncertainty.
    KEEP_INTERMEDIATE_A_E_RASTERS = False
    RESUME_FROM_CHECKPOINTS = True
    RASTER_BLOCK_ROWS = 256

    # Figures.
    FIG_DPI = 300
    GENERATE_PIXELWISE_SD_FIGURE = True

    # Descriptive thresholds for the pixelwise cross-fold SD summary.
    SD_THRESHOLDS = (
        0.02,
        0.05,
        0.10,
    )


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(
    message: str,
) -> None:
    print(
        f"[{datetime.now():%H:%M:%S}] {message}",
        flush=True,
    )


def utc_now_iso() -> str:
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def json_safe(
    value: Any,
) -> Any:
    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    return value


def require_file(
    path: Path,
    description: str,
) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}"
        )


def load_json(
    path: Path,
) -> Dict[str, Any]:
    require_file(
        path,
        "JSON file",
    )

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def normalize_feature_name(
    name: str,
) -> str:
    return re.sub(
        r"[^a-z0-9_]+",
        "_",
        str(
            name
        ).strip().lower(),
    ).strip(
        "_"
    )


def is_categorical_feature(
    name: str,
) -> bool:
    normalized = normalize_feature_name(
        name
    )

    compact = normalized.replace(
        "_",
        "",
    )

    aliases = {
        alias.lower()
        for alias in Config.CATEGORICAL_FEATURE_ALIASES
    }

    compact_aliases = {
        alias.replace(
            "_",
            "",
        )
        for alias in aliases
    }

    return (
        normalized in aliases
        or compact in compact_aliases
    )


def resolve_script_path() -> Path:
    """
    Resolve the saved Python script used for process-isolated fold workers.
    """
    file_value = globals().get(
        "__file__"
    )

    if file_value:
        candidate = Path(
            str(
                file_value
            )
        ).resolve()

        if (
            candidate.exists()
            and candidate.name.lower()
            != "ipykernel_launcher.py"
        ):
            return candidate

    candidate = (
        Path.cwd()
        / "Step_7_2D_RF_Performance_and_Mapped_Area_Uncertainty.py"
    )

    if candidate.exists():
        return candidate.resolve()

    raise RuntimeError(
        "Could not resolve the saved Step 7.2D Python script. "
        "Save this program as "
        "Step_7_2D_RF_Performance_and_Mapped_Area_Uncertainty.py "
        "and run the saved .py file."
    )


# =============================================================================
# STEP 6.1 / STEP 6.4 CONTRACT
# =============================================================================
def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """
    Verify that Step 6.1 and Step 6.4 define Models A-F identically.
    """
    step61 = load_json(
        Config.SCENARIO_MANIFEST_JSON
    ).get(
        "scenarios",
        {},
    )

    step64 = load_json(
        Config.STEP64_MODEL_CONFIG_JSON
    ).get(
        "scenarios",
        {},
    )

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                "Step 6.1 scenario manifest is missing "
                f"{scenario_name}."
            )

        if scenario_name not in step64:
            raise RuntimeError(
                "Step 6.4 model configuration is missing "
                f"{scenario_name}."
            )

        features_61 = list(
            step61[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        features_64 = list(
            step64[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any(
            "irs" in feature.lower()
            for feature in features_61
        ):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; "
                "IRS must not be an ML predictor."
            )

    return step64


def all_scenario_features(
    scenarios: Dict[str, Dict[str, Any]],
) -> list[str]:
    features: list[str] = []

    for scenario_name in Config.SCENARIO_ORDER:
        for feature in scenarios[
            scenario_name
        ][
            "features"
        ]:
            if feature not in features:
                features.append(
                    feature
                )

    return features


def load_and_validate_model_ready_dataset(
    scenarios: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    """
    Load the exact frozen Step 6.4 dataset used by successful Step 7.2.
    """
    require_file(
        Config.MODEL_READY_DATASET_CSV,
        "Step 6.4 frozen model-ready dataset",
    )

    df = pd.read_csv(
        Config.MODEL_READY_DATASET_CSV
    )

    features = all_scenario_features(
        scenarios
    )

    required = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
        *features,
    ]

    missing = [
        column
        for column in required
        if column not in df.columns
    ]

    if missing:
        raise RuntimeError(
            "Model-ready dataset is missing columns: "
            f"{missing}"
        )

    if len(
        df
    ) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Expected {Config.EXPECTED_N_ROWS} rows; "
            f"found {len(df)}."
        )

    if df[
        Config.SAMPLE_ID_COLUMN
    ].isna().any():
        raise RuntimeError(
            "Missing sample_id values found."
        )

    if df[
        Config.SAMPLE_ID_COLUMN
    ].duplicated().any():
        raise RuntimeError(
            "Duplicated sample_id values found."
        )

    for column in (
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ):
        df[
            column
        ] = pd.to_numeric(
            df[
                column
            ],
            errors="raise",
        ).astype(
            int
        )

    class_counts = (
        df[
            Config.TARGET_COLUMN
        ]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    expected_class_counts = {
        0: Config.EXPECTED_N_NEGATIVE,
        1: Config.EXPECTED_N_POSITIVE,
    }

    if class_counts != expected_class_counts:
        raise RuntimeError(
            f"Unexpected class counts: {class_counts}; "
            f"expected {expected_class_counts}."
        )

    folds = sorted(
        df[
            Config.FOLD_COLUMN
        ].unique().tolist()
    )

    if folds != Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected folds {Config.EXPECTED_FOLDS}; "
            f"found {folds}."
        )

    if not np.array_equal(
        df[
            Config.FOLD_COLUMN
        ].to_numpy(),
        df[
            Config.BLOCK_COLUMN
        ].to_numpy(),
    ):
        raise RuntimeError(
            "cv_fold and spatial_block are not identical."
        )

    predictor_values = df[
        features
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    if not np.isfinite(
        predictor_values.to_numpy(
            dtype="float64"
        )
    ).all():
        raise RuntimeError(
            "Missing/non-finite predictor values remain in Step 6.4 dataset."
        )

    return df


# =============================================================================
# EXACT STEP 7.2 RANDOM FOREST PIPELINE
# =============================================================================
def build_rf_pipeline(
    features: Sequence[str],
) -> Pipeline:
    """
    Exact fold-safe preprocessing + Random Forest pipeline used in Step 7.2.
    """
    categorical = [
        feature
        for feature in features
        if is_categorical_feature(
            feature
        )
    ]

    numeric = [
        feature
        for feature in features
        if not is_categorical_feature(
            feature
        )
    ]

    transformers = []

    if numeric:
        transformers.append(
            (
                "numeric",
                "passthrough",
                numeric,
            )
        )

    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = RandomForestClassifier(
        **Config.RF_FIXED_PARAMS
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                classifier,
            ),
        ]
    )


def set_pipeline_params(
    pipeline: Pipeline,
    params: Dict[str, Any],
) -> Pipeline:
    pipeline.set_params(
        **params
    )

    return pipeline


def load_outer_best_params() -> Dict[str, Any]:
    require_file(
        Config.OUTER_PARAMS_JSON,
        "successful Step 7.2 RF outer-fold best-parameter JSON",
    )

    with open(
        Config.OUTER_PARAMS_JSON,
        "r",
        encoding="utf-8",
    ) as handle:
        params = json.load(
            handle
        )

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in params:
            raise RuntimeError(
                "Outer-parameter JSON is missing scenario: "
                f"{scenario_name}"
            )

        for outer_fold in Config.EXPECTED_FOLDS:
            fold_info = params[
                scenario_name
            ].get(
                str(
                    outer_fold
                )
            )

            if (
                fold_info is None
                or "best_params"
                not in fold_info
            ):
                raise RuntimeError(
                    "Outer-parameter JSON is missing best_params for "
                    f"{scenario_name}, fold {outer_fold}."
                )

    return params


def get_fold_params(
    outer_params: Dict[str, Any],
    scenario_name: str,
    outer_fold: int,
) -> Dict[str, Any]:
    scenario = outer_params.get(
        scenario_name
    )

    if scenario is None:
        raise KeyError(
            "Missing scenario in successful outer-parameter JSON: "
            f"{scenario_name}"
        )

    fold_info = scenario.get(
        str(
            outer_fold
        )
    )

    if fold_info is None:
        raise KeyError(
            f"Missing fold {outer_fold} parameters for "
            f"{scenario_name}"
        )

    params = fold_info.get(
        "best_params"
    )

    if params is None:
        raise KeyError(
            f"Missing best_params for {scenario_name}, "
            f"outer fold {outer_fold}"
        )

    return params


# =============================================================================
# PERFORMANCE METRICS AND SPATIAL-BLOCK BOOTSTRAP
# =============================================================================
def metric_value(
    metric_name: str,
    y_true: np.ndarray,
    y_probability: np.ndarray,
    threshold: float,
) -> float:
    predicted = (
        y_probability
        >= threshold
    ).astype(
        int
    )

    if metric_name == "roc_auc":
        return float(
            roc_auc_score(
                y_true,
                y_probability,
            )
        )

    if metric_name == "pr_auc":
        return float(
            average_precision_score(
                y_true,
                y_probability,
            )
        )

    if metric_name == "balanced_accuracy":
        return float(
            balanced_accuracy_score(
                y_true,
                predicted,
            )
        )

    if metric_name == "f1":
        return float(
            f1_score(
                y_true,
                predicted,
                zero_division=0,
            )
        )

    if metric_name == "brier":
        return float(
            brier_score_loss(
                y_true,
                y_probability,
            )
        )

    if metric_name == "precision":
        return float(
            precision_score(
                y_true,
                predicted,
                zero_division=0,
            )
        )

    if metric_name == "recall":
        return float(
            recall_score(
                y_true,
                predicted,
                zero_division=0,
            )
        )

    raise ValueError(
        f"Unsupported metric: {metric_name}"
    )


def cluster_bootstrap_metric_intervals(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    ci_level: float,
    seed: int,
    threshold: float,
    metrics: Sequence[str],
) -> pd.DataFrame:
    """
    Resample whole spatial blocks with replacement and calculate uncertainty for
    the requested metrics.
    """
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float,
    )

    blocks = np.asarray(
        blocks,
    )

    unique_blocks = np.unique(
        blocks
    )

    indices_by_block = {
        block: np.flatnonzero(
            blocks
            == block
        )
        for block in unique_blocks
    }

    observed = {
        metric: metric_value(
            metric,
            y_true,
            y_prob,
            threshold,
        )
        for metric in metrics
    }

    bootstrap_values = {
        metric: []
        for metric in metrics
    }

    rng = np.random.default_rng(
        seed
    )

    for _ in range(
        n_bootstrap
    ):
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(
                unique_blocks
            ),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                indices_by_block[
                    block
                ]
                for block in sampled_blocks
            ]
        )

        y_boot = y_true[
            sampled_indices
        ]

        p_boot = y_prob[
            sampled_indices
        ]

        if np.unique(
            y_boot
        ).size < 2:
            continue

        for metric in metrics:
            bootstrap_values[
                metric
            ].append(
                metric_value(
                    metric,
                    y_boot,
                    p_boot,
                    threshold,
                )
            )

    alpha = (
        1.0
        - ci_level
    ) / 2.0

    rows: list[
        Dict[str, Any]
    ] = []

    for metric in metrics:
        values = np.asarray(
            bootstrap_values[
                metric
            ],
            dtype=float,
        )

        if values.size:
            low, high = np.quantile(
                values,
                [
                    alpha,
                    1.0
                    - alpha,
                ],
            )
        else:
            low = np.nan
            high = np.nan

        rows.append(
            {
                "metric": metric,
                "estimate": observed[
                    metric
                ],
                "ci_low": float(
                    low
                ),
                "ci_high": float(
                    high
                ),
                "valid_bootstrap_iterations": int(
                    values.size
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


def paired_cluster_bootstrap_auc_difference(
    y_true: np.ndarray,
    probability_left: np.ndarray,
    probability_right: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> tuple[
    float,
    float,
    float,
    int,
]:
    observed = float(
        roc_auc_score(
            y_true,
            probability_left,
        )
        - roc_auc_score(
            y_true,
            probability_right,
        )
    )

    unique_blocks = np.unique(
        blocks
    )

    indices_by_block = {
        block: np.flatnonzero(
            blocks
            == block
        )
        for block in unique_blocks
    }

    rng = np.random.default_rng(
        seed
    )

    differences: list[
        float
    ] = []

    for _ in range(
        n_bootstrap
    ):
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(
                unique_blocks
            ),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                indices_by_block[
                    block
                ]
                for block in sampled_blocks
            ]
        )

        y_boot = y_true[
            sampled_indices
        ]

        if np.unique(
            y_boot
        ).size < 2:
            continue

        differences.append(
            float(
                roc_auc_score(
                    y_boot,
                    probability_left[
                        sampled_indices
                    ],
                )
                - roc_auc_score(
                    y_boot,
                    probability_right[
                        sampled_indices
                    ],
                )
            )
        )

    if not differences:
        return (
            observed,
            np.nan,
            np.nan,
            0,
        )

    alpha = (
        1.0
        - ci_level
    ) / 2.0

    low, high = np.quantile(
        differences,
        [
            alpha,
            1.0
            - alpha,
        ],
    )

    return (
        observed,
        float(
            low
        ),
        float(
            high
        ),
        len(
            differences
        ),
    )


# =============================================================================
# RASTER HELPERS — SAME STEP 7.2 RF MAPPING PRINCIPLE
# =============================================================================
def read_stack_band_map(
    stack_path: Path,
) -> Dict[str, int]:
    require_file(
        stack_path,
        "Step 6.1 master predictor stack",
    )

    with rasterio.open(
        stack_path
    ) as dataset:
        descriptions = list(
            dataset.descriptions
        )

    band_map = {
        str(
            description
        ): int(
            index
        )
        for (
            index,
            description,
        )
        in enumerate(
            descriptions,
            start=1,
        )
        if description
    }

    if not band_map:
        raise RuntimeError(
            "Master predictor stack has no band descriptions."
        )

    return band_map


def predict_scenario_raster_on_own_feature_domain(
    fitted_pipeline: Pipeline,
    scenario_name: str,
    features: Sequence[str],
    output_path: Path,
) -> Path:
    """
    Predict a scenario wherever all predictors required by that scenario are
    finite in the Step 6.1 study-domain master stack.
    """
    band_map = read_stack_band_map(
        Config.MASTER_STACK_TIF
    )

    missing_bands = [
        feature
        for feature in features
        if feature not in band_map
    ]

    if missing_bands:
        raise RuntimeError(
            f"Master stack is missing bands required by "
            f"{scenario_name}: {missing_bands}"
        )

    band_indexes = [
        band_map[
            feature
        ]
        for feature in features
    ]

    with rasterio.open(
        Config.MASTER_STACK_TIF
    ) as stack_dataset:
        profile = stack_dataset.profile.copy()

        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(
            str(
                output_path
            )
            + ".part"
        )

        if temporary_path.exists():
            temporary_path.unlink()

        valid_pixel_count = 0

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_dataset:
            for row_start in range(
                0,
                stack_dataset.height,
                UncertaintyConfig.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    UncertaintyConfig.RASTER_BLOCK_ROWS,
                    stack_dataset.height
                    - row_start,
                )

                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=stack_dataset.width,
                    height=block_height,
                )

                stack_values = np.asarray(
                    stack_dataset.read(
                        band_indexes,
                        window=window,
                        masked=True,
                    ).filled(
                        np.nan
                    ),
                    dtype="float64",
                )

                valid = np.all(
                    np.isfinite(
                        stack_values
                    ),
                    axis=0,
                )

                output = np.full(
                    (
                        block_height,
                        stack_dataset.width,
                    ),
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                if np.any(
                    valid
                ):
                    data = {
                        feature: stack_values[
                            feature_index
                        ][
                            valid
                        ]
                        for (
                            feature_index,
                            feature,
                        )
                        in enumerate(
                            features
                        )
                    }

                    pixel_df = pd.DataFrame(
                        data,
                        columns=list(
                            features
                        ),
                    )

                    probability = (
                        fitted_pipeline.predict_proba(
                            pixel_df
                        )[
                            :,
                            1,
                        ]
                    )

                    output[
                        valid
                    ] = probability.astype(
                        "float32"
                    )

                    valid_pixel_count += int(
                        valid.sum()
                    )

                output_dataset.write(
                    output,
                    1,
                    window=window,
                )

            output_dataset.set_band_description(
                1,
                (
                    "Random Forest "
                    f"{Config.MODEL_SHORT_LABELS[scenario_name]} "
                    "susceptibility probability"
                ),
            )

            output_dataset.update_tags(
                algorithm="Random Forest",
                scenario=scenario_name,
                generated_at_utc=utc_now_iso(),
                validation_note=(
                    "Cross-fold model realization for map variability; "
                    "not a formal independent validation map."
                ),
            )

    if output_path.exists():
        output_path.unlink()

    os.replace(
        temporary_path,
        output_path,
    )

    log(
        f"[MAP] {scenario_name}: {valid_pixel_count:,} valid pixels -> "
        f"{output_path}"
    )

    return output_path


def blend_static_and_enhanced_probability_rasters(
    static_raster_path: Path,
    enhanced_raster_path: Path,
    output_path: Path,
    enhanced_weight: float,
) -> Path:
    """
    Apply the exact fixed A30/E70 operational blending principle used in Step 7.2.
    """
    if not (
        0.0
        <= enhanced_weight
        <= 1.0
    ):
        raise ValueError(
            "Enhanced weight must lie between 0 and 1."
        )

    static_weight = (
        1.0
        - enhanced_weight
    )

    require_file(
        static_raster_path,
        "Model A full-domain probability raster",
    )

    require_file(
        enhanced_raster_path,
        "Model E full-domain probability raster",
    )

    with rasterio.open(
        static_raster_path
    ) as static_dataset, rasterio.open(
        enhanced_raster_path
    ) as enhanced_dataset:
        aligned = (
            static_dataset.width
            == enhanced_dataset.width
            and static_dataset.height
            == enhanced_dataset.height
            and static_dataset.transform
            == enhanced_dataset.transform
            and static_dataset.crs
            == enhanced_dataset.crs
        )

        if not aligned:
            raise RuntimeError(
                "Model A and Model E full-domain rasters are not aligned."
            )

        profile = static_dataset.profile.copy()

        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(
            str(
                output_path
            )
            + ".part"
        )

        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_dataset:
            for row_start in range(
                0,
                static_dataset.height,
                UncertaintyConfig.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    UncertaintyConfig.RASTER_BLOCK_ROWS,
                    static_dataset.height
                    - row_start,
                )

                window = Window(
                    0,
                    row_start,
                    static_dataset.width,
                    block_height,
                )

                static_values = np.asarray(
                    static_dataset.read(
                        1,
                        window=window,
                        masked=True,
                    ).filled(
                        np.nan
                    ),
                    dtype="float64",
                )

                enhanced_values = np.asarray(
                    enhanced_dataset.read(
                        1,
                        window=window,
                        masked=True,
                    ).filled(
                        np.nan
                    ),
                    dtype="float64",
                )

                static_valid = np.isfinite(
                    static_values
                )

                enhanced_valid = np.isfinite(
                    enhanced_values
                )

                both_valid = (
                    static_valid
                    & enhanced_valid
                )

                static_only = (
                    static_valid
                    & ~enhanced_valid
                )

                enhanced_only = (
                    enhanced_valid
                    & ~static_valid
                )

                output = np.full(
                    static_values.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                output[
                    static_only
                ] = static_values[
                    static_only
                ].astype(
                    "float32"
                )

                output[
                    enhanced_only
                ] = enhanced_values[
                    enhanced_only
                ].astype(
                    "float32"
                )

                output[
                    both_valid
                ] = (
                    static_weight
                    * static_values[
                        both_valid
                    ]
                    + enhanced_weight
                    * enhanced_values[
                        both_valid
                    ]
                ).astype(
                    "float32"
                )

                output_dataset.write(
                    output,
                    1,
                    window=window,
                )

            output_dataset.set_band_description(
                1,
                "Random Forest A30/E70 ensemble susceptibility probability",
            )

            output_dataset.update_tags(
                algorithm="Random Forest",
                static_weight=f"{static_weight:.6f}",
                enhanced_weight=f"{enhanced_weight:.6f}",
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()

    os.replace(
        temporary_path,
        output_path,
    )

    return output_path


def clean_probability_array(
    array: np.ndarray,
    nodata: float | None,
) -> np.ndarray:
    result = np.asarray(
        array,
        dtype="float64",
    ).copy()

    if (
        nodata is not None
        and np.isfinite(
            float(
                nodata
            )
        )
    ):
        result[
            result
            == float(
                nodata
            )
        ] = np.nan

    result[
        (
            result
            < -1.0e-6
        )
        | (
            result
            > 1.0
            + 1.0e-6
        )
    ] = np.nan

    return result


def classify_susceptibility_probability(
    probability: np.ndarray,
) -> np.ndarray:
    """
    Same five fixed probability classes used in Step 7.2:
      1 Very Low : [0.0, 0.2)
      2 Low      : [0.2, 0.4)
      3 Moderate : [0.4, 0.6)
      4 High     : [0.6, 0.8)
      5 Very High: [0.8, 1.0]
    """
    classes = np.digitize(
        probability,
        bins=[
            0.2,
            0.4,
            0.6,
            0.8,
        ],
        right=False,
    )

    return (
        classes
        + 1
    )


def calculate_susceptibility_area_percentages(
    probability_raster_path: Path,
) -> pd.DataFrame:
    require_file(
        probability_raster_path,
        "ensemble susceptibility raster",
    )

    with rasterio.open(
        probability_raster_path
    ) as dataset:
        probability = clean_probability_array(
            dataset.read(
                1
            ),
            dataset.nodata,
        )

    valid = probability[
        np.isfinite(
            probability
        )
    ]

    if valid.size == 0:
        raise RuntimeError(
            "No valid susceptibility probabilities found."
        )

    classes = classify_susceptibility_probability(
        valid
    )

    total = int(
        valid.size
    )

    definitions = [
        (
            5,
            "Very High",
            0.8,
            1.0,
        ),
        (
            4,
            "High",
            0.6,
            0.8,
        ),
        (
            3,
            "Moderate",
            0.4,
            0.6,
        ),
        (
            2,
            "Low",
            0.2,
            0.4,
        ),
        (
            1,
            "Very Low",
            0.0,
            0.2,
        ),
    ]

    rows = []

    for (
        class_id,
        label,
        lower,
        upper,
    ) in definitions:
        count = int(
            np.sum(
                classes
                == class_id
            )
        )

        percentage = (
            100.0
            * count
            / total
        )

        rows.append(
            {
                "class_id": int(
                    class_id
                ),
                "class_label": label,
                "lower_probability": float(
                    lower
                ),
                "upper_probability": float(
                    upper
                ),
                "pixel_count": int(
                    count
                ),
                "area_percentage": float(
                    percentage
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


# =============================================================================
# PROVENANCE
# =============================================================================
def write_diagnostic_provenance(
    out_dir: Path,
) -> None:
    metadata = {
        "generated_at_utc": utc_now_iso(),
        "diagnostic_step": "Step 7.2D",
        "algorithm": "Random Forest",
        "performance_uncertainty": (
            "spatial-block cluster bootstrap resampling whole outer spatial blocks"
        ),
        "performance_inputs": str(
            Config.OOF_PREDICTIONS_CSV
        ),
        "ensemble_performance_inputs": str(
            Config.ENSEMBLE_OOF_PREDICTIONS_CSV
        ),
        "mapped_area_uncertainty": (
            "five process-isolated outer-fold RF A30/E70 map realizations"
        ),
        "map_model_parameters": (
            "exact successful Step 7.2 fold-specific best parameters; no retuning"
        ),
        "mapped_area_reporting": (
            "cross-fold mean, SD, minimum and maximum; not a formal 95% CI"
        ),
        "final_map_reference": str(
            Config.ENSEMBLE_PROBABILITY_TIF
        ),
        "valid_domain_rule": (
            "all five cross-fold ensemble maps must have identical valid masks "
            "and must match the successful final full-data RF ensemble mask"
        ),
        "step_7_2B_alternative_hkfi_weights_included": False,
        "step_7_2C_alternative_buffers_included": False,
        "rf_fixed_parameters": json_safe(
            Config.RF_FIXED_PARAMS
        ),
    }

    with open(
        out_dir
        / "Step_7_2D_RF_Diagnostic_Provenance.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            metadata,
            handle,
            indent=2,
        )


# =============================================================================
# OUTPUT PATHS
# =============================================================================
def output_paths() -> Dict[str, Path]:
    out_dir = (
        Config.OUT_DIR
        / "Step-7.2D-Performance_and_Map_Uncertainty"
    )

    paths = {
        "out": out_dir,
        "fold_dir": (
            out_dir
            / "cross_fold_ensemble_rasters"
        ),
        "worker_dir": (
            out_dir
            / "process_isolated_workers"
        ),
        "log_dir": (
            out_dir
            / "process_isolated_workers"
            / "logs"
        ),
        "performance": (
            out_dir
            / "RF_7_2D_Performance_Cluster_Bootstrap_Intervals.csv"
        ),
        "comparisons": (
            out_dir
            / "RF_7_2D_Additional_Paired_AUC_E_C_E_D.csv"
        ),
        "area_raw": (
            out_dir
            / "RF_7_2D_A30_E70_CrossFold_Area_Percentages_Raw.csv"
        ),
        "area_summary": (
            out_dir
            / "RF_7_2D_A30_E70_CrossFold_Area_Uncertainty_Summary.csv"
        ),
        "mean_raster": (
            out_dir
            / "RF_7_2D_A30_E70_CrossFold_Mean_Probability.tif"
        ),
        "sd_raster": (
            out_dir
            / "RF_7_2D_A30_E70_CrossFold_SD_Probability.tif"
        ),
        "domain_qa": (
            out_dir
            / "RF_7_2D_CrossFold_Raster_Domain_QA.csv"
        ),
        "sd_summary": (
            out_dir
            / "RF_7_2D_Pixelwise_SD_Summary.csv"
        ),
        "area_figure": (
            out_dir
            / "RF_7_2D_A30_E70_CrossFold_Area_Uncertainty.png"
        ),
        "sd_figure": (
            out_dir
            / "RF_7_2D_A30_E70_CrossFold_SD_Probability.png"
        ),
        "report": (
            out_dir
            / "Step_7_2D_RF_Uncertainty_Report.txt"
        ),
    }

    for directory in (
        paths[
            "out"
        ],
        paths[
            "fold_dir"
        ],
        paths[
            "worker_dir"
        ],
        paths[
            "log_dir"
        ],
    ):
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    return paths


def fold_paths(
    outer_fold: int,
) -> Dict[str, Path]:
    paths = output_paths()

    fold_dir = paths[
        "fold_dir"
    ]

    worker_dir = paths[
        "worker_dir"
    ]

    return {
        "model_a_raster": (
            fold_dir
            / f"outer_fold_{outer_fold}_Model_A_probability.tif"
        ),
        "model_e_raster": (
            fold_dir
            / f"outer_fold_{outer_fold}_Model_E_probability.tif"
        ),
        "ensemble_raster": (
            fold_dir
            / f"outer_fold_{outer_fold}_A30_E70_probability.tif"
        ),
        "area_csv": (
            worker_dir
            / f"outer_fold_{outer_fold}_Area_Percentages.csv"
        ),
        "metadata_json": (
            worker_dir
            / f"outer_fold_{outer_fold}_Metadata.json"
        ),
        "done": (
            worker_dir
            / f"outer_fold_{outer_fold}.done"
        ),
        "log": (
            paths[
                "log_dir"
            ]
            / f"outer_fold_{outer_fold}.log"
        ),
    }


# =============================================================================
# PERFORMANCE UNCERTAINTY
# =============================================================================
def append_metric_intervals(
    rows: list[Dict[str, Any]],
    label: str,
    scenario: str,
    y_true: np.ndarray,
    y_probability: np.ndarray,
    blocks: np.ndarray,
    seed: int,
    product_type: str,
) -> None:
    intervals = cluster_bootstrap_metric_intervals(
        y_true=y_true,
        y_prob=y_probability,
        blocks=blocks,
        n_bootstrap=UncertaintyConfig.N_BOOTSTRAP,
        ci_level=UncertaintyConfig.CI_LEVEL,
        seed=seed,
        threshold=Config.BINARY_THRESHOLD,
        metrics=(
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "f1",
            "brier",
            "precision",
            "recall",
        ),
    )

    for _, metric_row in intervals.iterrows():
        rows.append(
            {
                "product_type": product_type,
                "scenario": scenario,
                "model_or_product": label,
                "metric": metric_row[
                    "metric"
                ],
                "estimate": metric_row[
                    "estimate"
                ],
                "ci_low": metric_row[
                    "ci_low"
                ],
                "ci_high": metric_row[
                    "ci_high"
                ],
                "valid_bootstrap_iterations": metric_row[
                    "valid_bootstrap_iterations"
                ],
                "binary_threshold": (
                    Config.BINARY_THRESHOLD
                    if metric_row[
                        "metric"
                    ]
                    in {
                        "balanced_accuracy",
                        "f1",
                        "precision",
                        "recall",
                    }
                    else np.nan
                ),
            }
        )


def build_performance_uncertainty(
    out_dir: Path,
) -> pd.DataFrame:
    """
    Calculate cluster-bootstrap intervals from already saved genuine Step 7.2
    OOF predictions. No model fitting occurs here.
    """
    require_file(
        Config.OOF_PREDICTIONS_CSV,
        "successful Step 7.2 RF A-F OOF predictions",
    )

    all_oof = pd.read_csv(
        Config.OOF_PREDICTIONS_CSV
    )

    rows: list[
        Dict[str, Any]
    ] = []

    for (
        scenario_index,
        scenario_name,
    ) in enumerate(
        Config.SCENARIO_ORDER
    ):
        subset = all_oof.loc[
            all_oof[
                "scenario"
            ]
            == scenario_name
        ].copy()

        if len(
            subset
        ) != Config.EXPECTED_N_ROWS:
            raise RuntimeError(
                f"Unexpected OOF row count for {scenario_name}: "
                f"{len(subset)}; expected "
                f"{Config.EXPECTED_N_ROWS}."
            )

        if subset[
            Config.SAMPLE_ID_COLUMN
        ].duplicated().any():
            raise RuntimeError(
                "Duplicated OOF sample IDs found for "
                f"{scenario_name}."
            )

        append_metric_intervals(
            rows=rows,
            label=Config.MODEL_SHORT_LABELS[
                scenario_name
            ],
            scenario=scenario_name,
            y_true=subset[
                Config.TARGET_COLUMN
            ].to_numpy(
                dtype=int
            ),
            y_probability=subset[
                "oof_probability"
            ].to_numpy(
                dtype=float
            ),
            blocks=subset[
                Config.FOLD_COLUMN
            ].to_numpy(
                dtype=int
            ),
            seed=(
                UncertaintyConfig.RANDOM_STATE
                + scenario_index
            ),
            product_type="A-F model",
        )

    # Separately reported fixed operational ensemble.
    require_file(
        Config.ENSEMBLE_OOF_PREDICTIONS_CSV,
        "successful Step 7.2 RF A30/E70 ensemble OOF predictions",
    )

    ensemble = pd.read_csv(
        Config.ENSEMBLE_OOF_PREDICTIONS_CSV
    )

    required = {
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        "oof_probability_ensemble",
    }

    missing = (
        required
        - set(
            ensemble.columns
        )
    )

    if missing:
        raise RuntimeError(
            "RF ensemble OOF file is missing columns: "
            f"{sorted(missing)}"
        )

    if len(
        ensemble
    ) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            "Unexpected RF A30/E70 ensemble OOF row count: "
            f"{len(ensemble)}; expected "
            f"{Config.EXPECTED_N_ROWS}."
        )

    if ensemble[
        Config.SAMPLE_ID_COLUMN
    ].duplicated().any():
        raise RuntimeError(
            "Duplicated sample IDs found in RF A30/E70 ensemble OOF file."
        )

    append_metric_intervals(
        rows=rows,
        label=(
            "A30/E70 fixed operational ensemble"
        ),
        scenario="A30_E70_ensemble",
        y_true=ensemble[
            Config.TARGET_COLUMN
        ].to_numpy(
            dtype=int
        ),
        y_probability=ensemble[
            "oof_probability_ensemble"
        ].to_numpy(
            dtype=float
        ),
        blocks=ensemble[
            Config.FOLD_COLUMN
        ].to_numpy(
            dtype=int
        ),
        seed=(
            UncertaintyConfig.RANDOM_STATE
            + 100
        ),
        product_type="operational ensemble",
    )

    result = pd.DataFrame(
        rows
    )

    result.to_csv(
        out_dir
        / "RF_7_2D_Performance_Cluster_Bootstrap_Intervals.csv",
        index=False,
        float_format="%.10f",
    )

    return result


def add_missing_E_C_and_E_D_comparisons(
    out_dir: Path,
) -> pd.DataFrame:
    """
    Add the same direct paired E-C and E-D spatial-block bootstrap comparisons
    used in Step 7.1D, now using successful Step 7.2 RF OOF predictions.
    """
    require_file(
        Config.OOF_PREDICTIONS_CSV,
        "successful Step 7.2 RF A-F OOF predictions",
    )

    all_oof = pd.read_csv(
        Config.OOF_PREDICTIONS_CSV
    )

    pivot = all_oof.pivot(
        index=Config.SAMPLE_ID_COLUMN,
        columns="scenario",
        values="oof_probability",
    )

    metadata = (
        all_oof.loc[
            all_oof[
                "scenario"
            ]
            == "A_static_baseline",
            [
                Config.SAMPLE_ID_COLUMN,
                Config.TARGET_COLUMN,
                Config.FOLD_COLUMN,
            ],
        ]
        .drop_duplicates(
            Config.SAMPLE_ID_COLUMN
        )
        .set_index(
            Config.SAMPLE_ID_COLUMN
        )
        .loc[
            pivot.index
        ]
    )

    y = metadata[
        Config.TARGET_COLUMN
    ].to_numpy(
        dtype=int
    )

    blocks = metadata[
        Config.FOLD_COLUMN
    ].to_numpy(
        dtype=int
    )

    comparisons = [
        (
            "E_physics_guided",
            "C_hkfi_only",
            "E - C",
            "Combined HKFI + SNDR versus HKFI alone",
        ),
        (
            "E_physics_guided",
            "D_sndr_only",
            "E - D",
            "Combined HKFI + SNDR versus SNDR alone",
        ),
    ]

    rows: list[
        Dict[str, Any]
    ] = []

    for (
        comparison_index,
        (
            left,
            right,
            label,
            purpose,
        ),
    ) in enumerate(
        comparisons
    ):
        (
            observed,
            low,
            high,
            n_valid,
        ) = paired_cluster_bootstrap_auc_difference(
            y_true=y,
            probability_left=pivot[
                left
            ].to_numpy(
                dtype=float
            ),
            probability_right=pivot[
                right
            ].to_numpy(
                dtype=float
            ),
            blocks=blocks,
            n_bootstrap=UncertaintyConfig.N_BOOTSTRAP,
            seed=(
                UncertaintyConfig.RANDOM_STATE
                + 1000
                + comparison_index
            ),
            ci_level=UncertaintyConfig.CI_LEVEL,
        )

        rows.append(
            {
                "comparison": label,
                "purpose": purpose,
                "left_scenario": left,
                "right_scenario": right,
                "auc_difference_left_minus_right": observed,
                "ci_low": low,
                "ci_high": high,
                "valid_bootstrap_iterations": int(
                    n_valid
                ),
            }
        )

    result = pd.DataFrame(
        rows
    )

    result.to_csv(
        out_dir
        / "RF_7_2D_Additional_Paired_AUC_E_C_E_D.csv",
        index=False,
        float_format="%.10f",
    )

    return result


# =============================================================================
# FOLD-SPECIFIC RF MODEL FITTING
# =============================================================================
def fit_outer_fold_model(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
    outer_params: Dict[str, Any],
    scenario_name: str,
    outer_fold: int,
):
    """
    Refit one fold-specific RF map realization using the exact best parameters
    selected inside the corresponding successful Step 7.2 outer-training set.

    No hyperparameter tuning is repeated here.
    """
    features = list(
        scenarios[
            scenario_name
        ][
            "features"
        ]
    )

    train_df = df.loc[
        df[
            Config.FOLD_COLUMN
        ]
        != outer_fold
    ].copy()

    if train_df[
        Config.FOLD_COLUMN
    ].nunique() != 4:
        raise RuntimeError(
            f"Outer fold {outer_fold}: expected four training blocks."
        )

    params = get_fold_params(
        outer_params=outer_params,
        scenario_name=scenario_name,
        outer_fold=outer_fold,
    )

    pipeline = build_rf_pipeline(
        features
    )

    set_pipeline_params(
        pipeline,
        params,
    )

    X_train = train_df[
        features
    ]

    y_train = train_df[
        Config.TARGET_COLUMN
    ].to_numpy(
        dtype=int
    )

    pipeline.fit(
        X_train,
        y_train,
    )

    return (
        pipeline,
        features,
        params,
    )


# =============================================================================
# PROCESS-ISOLATED OUTER-FOLD MAP WORKER
# =============================================================================
def run_outer_fold_map_worker(
    outer_fold: int,
) -> None:
    scenarios = validate_scenario_manifests()

    df = load_and_validate_model_ready_dataset(
        scenarios
    )

    outer_params = load_outer_best_params()

    paths = fold_paths(
        outer_fold
    )

    log(
        f"[FOLD {outer_fold}] Fitting RF Model A on the four "
        "remaining spatial blocks..."
    )

    (
        model_a,
        features_a,
        params_a,
    ) = fit_outer_fold_model(
        df=df,
        scenarios=scenarios,
        outer_params=outer_params,
        scenario_name="A_static_baseline",
        outer_fold=outer_fold,
    )

    log(
        f"[FOLD {outer_fold}] Fitting RF Model E on the four "
        "remaining spatial blocks..."
    )

    (
        model_e,
        features_e,
        params_e,
    ) = fit_outer_fold_model(
        df=df,
        scenarios=scenarios,
        outer_params=outer_params,
        scenario_name="E_physics_guided",
        outer_fold=outer_fold,
    )

    log(
        f"[FOLD {outer_fold}] Predicting full available RF Model A domain..."
    )

    predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=model_a,
        scenario_name="A_static_baseline",
        features=features_a,
        output_path=paths[
            "model_a_raster"
        ],
    )

    log(
        f"[FOLD {outer_fold}] Predicting full available RF Model E domain..."
    )

    predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=model_e,
        scenario_name="E_physics_guided",
        features=features_e,
        output_path=paths[
            "model_e_raster"
        ],
    )

    log(
        f"[FOLD {outer_fold}] Creating fixed RF A30/E70 ensemble..."
    )

    blend_static_and_enhanced_probability_rasters(
        static_raster_path=paths[
            "model_a_raster"
        ],
        enhanced_raster_path=paths[
            "model_e_raster"
        ],
        output_path=paths[
            "ensemble_raster"
        ],
        enhanced_weight=float(
            Config.ENSEMBLE_ENHANCED_WEIGHT
        ),
    )

    area_df = calculate_susceptibility_area_percentages(
        paths[
            "ensemble_raster"
        ]
    )

    area_df.insert(
        0,
        "outer_fold_omitted_from_training",
        int(
            outer_fold
        ),
    )

    area_df.to_csv(
        paths[
            "area_csv"
        ],
        index=False,
        float_format="%.10f",
    )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "Random Forest",
        "outer_fold_omitted_from_training": int(
            outer_fold
        ),
        "training_blocks": [
            int(
                value
            )
            for value
            in sorted(
                df.loc[
                    df[
                        Config.FOLD_COLUMN
                    ]
                    != outer_fold,
                    Config.FOLD_COLUMN,
                ]
                .unique()
                .tolist()
            )
        ],
        "model_A_best_params": json_safe(
            params_a
        ),
        "model_E_best_params": json_safe(
            params_e
        ),
        "ensemble_enhanced_weight": float(
            Config.ENSEMBLE_ENHANCED_WEIGHT
        ),
        "model_A_raster": str(
            paths[
                "model_a_raster"
            ]
        ),
        "model_E_raster": str(
            paths[
                "model_e_raster"
            ]
        ),
        "ensemble_raster": str(
            paths[
                "ensemble_raster"
            ]
        ),
        "area_csv": str(
            paths[
                "area_csv"
            ]
        ),
        "interpretation": (
            "One Random Forest cross-fold model realization used to characterize "
            "map/model variability; not a formal independent validation map."
        ),
    }

    with open(
        paths[
            "metadata_json"
        ],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(
                metadata
            ),
            handle,
            indent=2,
        )

    # Delete large intermediate A/E rasters only after successful ensemble
    # creation and mapped-area calculation.
    if not UncertaintyConfig.KEEP_INTERMEDIATE_A_E_RASTERS:
        for intermediate_path in (
            paths[
                "model_a_raster"
            ],
            paths[
                "model_e_raster"
            ],
        ):
            if intermediate_path.exists():
                intermediate_path.unlink()

    paths[
        "done"
    ].write_text(
        utc_now_iso(),
        encoding="utf-8",
    )

    del model_a
    del model_e
    gc.collect()

    log(
        f"[FOLD {outer_fold}] Completed successfully."
    )


def run_external_fold_worker(
    outer_fold: int,
) -> None:
    """
    Launch one outer-fold RF map realization in a fresh operating-system process.
    """
    script_path = resolve_script_path()

    paths = fold_paths(
        outer_fold
    )

    command = [
        sys.executable,
        str(
            script_path
        ),
        "--worker-fold",
        str(
            outer_fold
        ),
    ]

    environment = os.environ.copy()

    environment[
        "OMP_NUM_THREADS"
    ] = "1"

    environment[
        "OPENBLAS_NUM_THREADS"
    ] = "1"

    environment[
        "MKL_NUM_THREADS"
    ] = "1"

    environment[
        "VECLIB_MAXIMUM_THREADS"
    ] = "1"

    environment[
        "NUMEXPR_NUM_THREADS"
    ] = "1"

    log(
        f"[PROCESS] Starting RF outer fold {outer_fold} "
        "in a fresh Python process."
    )

    log(
        f"[PROCESS] Worker log: "
        f"{paths['log']}"
    )

    with open(
        paths[
            "log"
        ],
        "w",
        encoding="utf-8",
    ) as log_handle:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=str(
                script_path.parent
            ),
            env=environment,
        )

        assert (
            process.stdout
            is not None
        )

        for line in process.stdout:
            print(
                line,
                end="",
                flush=True,
            )

            log_handle.write(
                line
            )

            log_handle.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"RF outer-fold worker {outer_fold} failed with "
            f"return code {return_code}. See:\n"
            f"{paths['log']}"
        )


# =============================================================================
# RASTER VALID-DOMAIN QA
# =============================================================================
def raster_valid_mask(
    dataset,
    window: Window,
) -> np.ndarray:
    array = (
        dataset.read(
            1,
            window=window,
            masked=True,
        )
        .filled(
            np.nan
        )
        .astype(
            "float64"
        )
    )

    return np.isfinite(
        array
    )


def validate_cross_fold_raster_domains(
    fold_rasters: list[Path],
    final_full_data_raster: Path,
    output_csv: Path,
) -> pd.DataFrame:
    """
    Verify that all five RF fold ensemble maps:
      - are aligned;
      - have identical valid-pixel masks;
      - match the successful final full-data RF ensemble valid mask.
    """
    if not fold_rasters:
        raise ValueError(
            "No fold rasters supplied for domain QA."
        )

    for path in fold_rasters:
        require_file(
            path,
            "cross-fold RF ensemble probability raster",
        )

    require_file(
        final_full_data_raster,
        "successful final full-data RF ensemble raster",
    )

    datasets = [
        rasterio.open(
            path
        )
        for path in fold_rasters
    ]

    final_dataset = rasterio.open(
        final_full_data_raster
    )

    try:
        reference = datasets[
            0
        ]

        for (
            index,
            dataset,
        ) in enumerate(
            datasets[
                1:
            ],
            start=2,
        ):
            aligned = (
                dataset.width
                == reference.width
                and dataset.height
                == reference.height
                and dataset.transform
                == reference.transform
                and dataset.crs
                == reference.crs
            )

            if not aligned:
                raise RuntimeError(
                    "Cross-fold RF ensemble raster alignment mismatch: "
                    f"fold raster sequence {index}."
                )

        final_aligned = (
            final_dataset.width
            == reference.width
            and final_dataset.height
            == reference.height
            and final_dataset.transform
            == reference.transform
            and final_dataset.crs
            == reference.crs
        )

        if not final_aligned:
            raise RuntimeError(
                "Successful final full-data RF ensemble raster is not aligned "
                "with the cross-fold ensemble rasters."
            )

        valid_counts = np.zeros(
            len(
                datasets
            ),
            dtype=np.int64,
        )

        mismatch_vs_reference = np.zeros(
            len(
                datasets
            ),
            dtype=np.int64,
        )

        mismatch_vs_final = np.zeros(
            len(
                datasets
            ),
            dtype=np.int64,
        )

        final_valid_count = 0

        for row_start in range(
            0,
            reference.height,
            UncertaintyConfig.RASTER_BLOCK_ROWS,
        ):
            height = min(
                UncertaintyConfig.RASTER_BLOCK_ROWS,
                reference.height
                - row_start,
            )

            window = Window(
                0,
                row_start,
                reference.width,
                height,
            )

            masks = [
                raster_valid_mask(
                    dataset,
                    window,
                )
                for dataset in datasets
            ]

            final_mask = raster_valid_mask(
                final_dataset,
                window,
            )

            reference_mask = masks[
                0
            ]

            final_valid_count += int(
                final_mask.sum()
            )

            for (
                index,
                mask,
            ) in enumerate(
                masks
            ):
                valid_counts[
                    index
                ] += int(
                    mask.sum()
                )

                mismatch_vs_reference[
                    index
                ] += int(
                    np.count_nonzero(
                        mask
                        != reference_mask
                    )
                )

                mismatch_vs_final[
                    index
                ] += int(
                    np.count_nonzero(
                        mask
                        != final_mask
                    )
                )

        rows: list[
            Dict[str, Any]
        ] = []

        for (
            index,
            (
                raster_path,
                valid_count,
                mismatch_reference,
                mismatch_final,
            ),
        ) in enumerate(
            zip(
                fold_rasters,
                valid_counts,
                mismatch_vs_reference,
                mismatch_vs_final,
            ),
            start=1,
        ):
            rows.append(
                {
                    "raster": str(
                        raster_path
                    ),
                    "fold_raster_sequence": int(
                        index
                    ),
                    "valid_pixel_count": int(
                        valid_count
                    ),
                    "mismatched_pixels_vs_fold1_mask": int(
                        mismatch_reference
                    ),
                    "mismatched_pixels_vs_final_full_data_mask": int(
                        mismatch_final
                    ),
                    "identical_to_fold1_mask": bool(
                        mismatch_reference
                        == 0
                    ),
                    "identical_to_final_full_data_mask": bool(
                        mismatch_final
                        == 0
                    ),
                }
            )

        rows.append(
            {
                "raster": str(
                    final_full_data_raster
                ),
                "fold_raster_sequence": 0,
                "valid_pixel_count": int(
                    final_valid_count
                ),
                "mismatched_pixels_vs_fold1_mask": int(
                    mismatch_vs_final[
                        0
                    ]
                ),
                "mismatched_pixels_vs_final_full_data_mask": 0,
                "identical_to_fold1_mask": bool(
                    mismatch_vs_final[
                        0
                    ]
                    == 0
                ),
                "identical_to_final_full_data_mask": True,
            }
        )

        qa_df = pd.DataFrame(
            rows
        )

        qa_df.to_csv(
            output_csv,
            index=False,
        )

        if np.any(
            mismatch_vs_reference
            != 0
        ):
            raise RuntimeError(
                "Cross-fold RF ensemble rasters do not have identical "
                "valid-pixel masks. See:\n"
                f"{output_csv}"
            )

        if np.any(
            mismatch_vs_final
            != 0
        ):
            raise RuntimeError(
                "Cross-fold RF ensemble valid-pixel masks do not match the "
                "successful final full-data RF ensemble mask. See:\n"
                f"{output_csv}"
            )

        log(
            "[RASTER-QA] All five RF cross-fold ensemble rasters and the "
            "final full-data RF ensemble have identical valid domains: PASS"
        )

        return qa_df

    finally:
        for dataset in datasets:
            dataset.close()

        final_dataset.close()


# =============================================================================
# CROSS-FOLD RASTER AGGREGATION
# =============================================================================
def aggregate_fold_rasters(
    fold_rasters: list[Path],
    mean_output: Path,
    sd_output: Path,
) -> None:
    """
    Calculate pixelwise cross-fold mean and sample SD using the same method as
    Step 7.1D.
    """
    if not fold_rasters:
        raise ValueError(
            "No fold rasters supplied for aggregation."
        )

    datasets = [
        rasterio.open(
            path
        )
        for path in fold_rasters
    ]

    try:
        reference = datasets[
            0
        ]

        profile = reference.profile.copy()

        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        with rasterio.open(
            mean_output,
            "w",
            **profile,
        ) as mean_dataset, rasterio.open(
            sd_output,
            "w",
            **profile,
        ) as sd_dataset:
            for row_start in range(
                0,
                reference.height,
                UncertaintyConfig.RASTER_BLOCK_ROWS,
            ):
                height = min(
                    UncertaintyConfig.RASTER_BLOCK_ROWS,
                    reference.height
                    - row_start,
                )

                window = Window(
                    0,
                    row_start,
                    reference.width,
                    height,
                )

                arrays = [
                    dataset.read(
                        1,
                        window=window,
                        masked=True,
                    )
                    .filled(
                        np.nan
                    )
                    .astype(
                        "float64"
                    )
                    for dataset in datasets
                ]

                stack = np.stack(
                    arrays,
                    axis=0,
                )

                reference_valid = np.isfinite(
                    stack[
                        0
                    ]
                )

                for fold_index in range(
                    1,
                    stack.shape[
                        0
                    ],
                ):
                    candidate_valid = np.isfinite(
                        stack[
                            fold_index
                        ]
                    )

                    if not np.array_equal(
                        reference_valid,
                        candidate_valid,
                    ):
                        raise RuntimeError(
                            "Cross-fold RF ensemble rasters changed valid "
                            "domain during aggregation."
                        )

                mean_output_array = np.full(
                    reference_valid.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                sd_output_array = np.full(
                    reference_valid.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                if np.any(
                    reference_valid
                ):
                    valid_values = stack[
                        :,
                        reference_valid,
                    ]

                    with np.errstate(
                        invalid="ignore",
                        divide="ignore",
                    ):
                        mean_values = np.mean(
                            valid_values,
                            axis=0,
                        )

                        sd_values = np.std(
                            valid_values,
                            axis=0,
                            ddof=1,
                        )

                    mean_output_array[
                        reference_valid
                    ] = mean_values.astype(
                        "float32"
                    )

                    sd_output_array[
                        reference_valid
                    ] = sd_values.astype(
                        "float32"
                    )

                mean_dataset.write(
                    mean_output_array,
                    1,
                    window=window,
                )

                sd_dataset.write(
                    sd_output_array,
                    1,
                    window=window,
                )

            mean_dataset.set_band_description(
                1,
                "RF cross-fold mean A30/E70 susceptibility probability",
            )

            mean_dataset.update_tags(
                algorithm="Random Forest",
                uncertainty_type=(
                    "cross-fold model variability"
                ),
                number_of_outer_fold_models=len(
                    fold_rasters
                ),
                generated_at_utc=utc_now_iso(),
            )

            sd_dataset.set_band_description(
                1,
                "RF cross-fold SD of A30/E70 susceptibility probability",
            )

            sd_dataset.update_tags(
                algorithm="Random Forest",
                uncertainty_type=(
                    "cross-fold model variability"
                ),
                number_of_outer_fold_models=len(
                    fold_rasters
                ),
                ddof=1,
                generated_at_utc=utc_now_iso(),
                interpretation=(
                    "Sample standard deviation across five RF outer-fold model "
                    "realizations; not a formal 95% confidence interval."
                ),
            )

    finally:
        for dataset in datasets:
            dataset.close()


# =============================================================================
# AREA UNCERTAINTY
# =============================================================================
def final_full_data_area_rows() -> pd.DataFrame:
    require_file(
        Config.ENSEMBLE_PROBABILITY_TIF,
        "successful final Step 7.2 RF A30/E70 ensemble map",
    )

    final_area = (
        calculate_susceptibility_area_percentages(
            Config.ENSEMBLE_PROBABILITY_TIF
        )[
            [
                "class_id",
                "class_label",
                "area_percentage",
            ]
        ]
        .copy()
    )

    high_plus_very_high = float(
        final_area.loc[
            final_area[
                "class_label"
            ].isin(
                [
                    "High",
                    "Very High",
                ]
            ),
            "area_percentage",
        ].sum()
    )

    final_area = pd.concat(
        [
            final_area,
            pd.DataFrame(
                [
                    {
                        "class_id": 99,
                        "class_label": (
                            "High + Very High"
                        ),
                        "area_percentage": (
                            high_plus_very_high
                        ),
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    return final_area.rename(
        columns={
            "area_percentage": (
                "final_full_data_area_percentage"
            )
        }
    )


def collect_fold_area_results() -> tuple[
    pd.DataFrame,
    list[Path],
]:
    area_frames: list[
        pd.DataFrame
    ] = []

    ensemble_rasters: list[
        Path
    ] = []

    for outer_fold in Config.EXPECTED_FOLDS:
        paths = fold_paths(
            outer_fold
        )

        require_file(
            paths[
                "done"
            ],
            f"RF outer-fold {outer_fold} completion checkpoint",
        )

        require_file(
            paths[
                "area_csv"
            ],
            f"RF outer-fold {outer_fold} area-percentage CSV",
        )

        require_file(
            paths[
                "ensemble_raster"
            ],
            f"RF outer-fold {outer_fold} ensemble raster",
        )

        area_df = pd.read_csv(
            paths[
                "area_csv"
            ]
        )

        required = {
            "outer_fold_omitted_from_training",
            "class_id",
            "class_label",
            "area_percentage",
        }

        missing = (
            required
            - set(
                area_df.columns
            )
        )

        if missing:
            raise RuntimeError(
                f"RF fold {outer_fold} area CSV is missing columns: "
                f"{sorted(missing)}"
            )

        area_frames.append(
            area_df
        )

        ensemble_rasters.append(
            paths[
                "ensemble_raster"
            ]
        )

    return (
        pd.concat(
            area_frames,
            ignore_index=True,
        ),
        ensemble_rasters,
    )


def summarize_mapped_area_uncertainty(
    area_raw: pd.DataFrame,
) -> pd.DataFrame:
    area_summary = (
        area_raw.groupby(
            [
                "class_id",
                "class_label",
            ],
            as_index=False,
        )
        .agg(
            mean_area_percentage=(
                "area_percentage",
                "mean",
            ),
            sd_area_percentage=(
                "area_percentage",
                "std",
            ),
            minimum_area_percentage=(
                "area_percentage",
                "min",
            ),
            maximum_area_percentage=(
                "area_percentage",
                "max",
            ),
            n_spatial_fold_models=(
                "outer_fold_omitted_from_training",
                "nunique",
            ),
        )
        .sort_values(
            "class_id",
            ascending=False,
        )
    )

    high_combined = (
        area_raw.loc[
            area_raw[
                "class_label"
            ].isin(
                [
                    "High",
                    "Very High",
                ]
            )
        ]
        .groupby(
            "outer_fold_omitted_from_training",
            as_index=False,
        )[
            "area_percentage"
        ]
        .sum()
    )

    area_summary = pd.concat(
        [
            area_summary,
            pd.DataFrame(
                [
                    {
                        "class_id": 99,
                        "class_label": (
                            "High + Very High"
                        ),
                        "mean_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].mean()
                        ),
                        "sd_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].std(
                                ddof=1
                            )
                        ),
                        "minimum_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].min()
                        ),
                        "maximum_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].max()
                        ),
                        "n_spatial_fold_models": int(
                            len(
                                high_combined
                            )
                        ),
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    final_area = final_full_data_area_rows()

    area_summary = area_summary.merge(
        final_area,
        on=[
            "class_id",
            "class_label",
        ],
        how="left",
        validate="one_to_one",
    )

    area_summary[
        "final_minus_crossfold_mean_percentage_points"
    ] = (
        area_summary[
            "final_full_data_area_percentage"
        ]
        - area_summary[
            "mean_area_percentage"
        ]
    )

    area_summary[
        "crossfold_range_percentage_points"
    ] = (
        area_summary[
            "maximum_area_percentage"
        ]
        - area_summary[
            "minimum_area_percentage"
        ]
    )

    return area_summary


# =============================================================================
# PIXELWISE SD QUANTIFICATION
# =============================================================================
def summarize_pixelwise_sd(
    sd_raster_path: Path,
    output_csv: Path,
) -> pd.DataFrame:
    require_file(
        sd_raster_path,
        "RF cross-fold SD probability raster",
    )

    chunks: list[
        np.ndarray
    ] = []

    with rasterio.open(
        sd_raster_path
    ) as dataset:
        for row_start in range(
            0,
            dataset.height,
            UncertaintyConfig.RASTER_BLOCK_ROWS,
        ):
            height = min(
                UncertaintyConfig.RASTER_BLOCK_ROWS,
                dataset.height
                - row_start,
            )

            window = Window(
                0,
                row_start,
                dataset.width,
                height,
            )

            values = (
                dataset.read(
                    1,
                    window=window,
                    masked=True,
                )
                .compressed()
                .astype(
                    "float64"
                )
            )

            if values.size:
                chunks.append(
                    values
                )

    if not chunks:
        raise RuntimeError(
            "No valid values found in RF cross-fold SD raster."
        )

    values = np.concatenate(
        chunks
    )

    summary = {
        "algorithm": "Random Forest",
        "valid_pixels": int(
            values.size
        ),
        "mean_pixelwise_sd": float(
            np.mean(
                values
            )
        ),
        "median_pixelwise_sd": float(
            np.median(
                values
            )
        ),
        "p75_pixelwise_sd": float(
            np.percentile(
                values,
                75.0,
            )
        ),
        "p90_pixelwise_sd": float(
            np.percentile(
                values,
                90.0,
            )
        ),
        "p95_pixelwise_sd": float(
            np.percentile(
                values,
                95.0,
            )
        ),
        "p99_pixelwise_sd": float(
            np.percentile(
                values,
                99.0,
            )
        ),
        "maximum_pixelwise_sd": float(
            np.max(
                values
            )
        ),
    }

    for threshold in UncertaintyConfig.SD_THRESHOLDS:
        key = (
            "pixels_with_sd_greater_than_"
            + str(
                threshold
            ).replace(
                ".",
                "p",
            )
            + "_percent"
        )

        summary[
            key
        ] = float(
            100.0
            * np.mean(
                values
                > float(
                    threshold
                )
            )
        )

    result = pd.DataFrame(
        [
            summary
        ]
    )

    result.to_csv(
        output_csv,
        index=False,
        float_format="%.10f",
    )

    return result


# =============================================================================
# FIGURES — SAME STYLE AS STEP 7.1D
# =============================================================================
def create_area_uncertainty_figure(
    area_summary: pd.DataFrame,
    output_path: Path,
) -> None:
    plot_df = area_summary.loc[
        area_summary[
            "class_id"
        ].isin(
            [
                1,
                2,
                3,
                4,
                5,
            ]
        )
    ].copy()

    plot_df = plot_df.sort_values(
        "class_id",
        ascending=True,
    )

    figure, axis = plt.subplots(
        figsize=(
            6.2,
            4.3,
        )
    )

    x = np.arange(
        len(
            plot_df
        )
    )

    means = plot_df[
        "mean_area_percentage"
    ].to_numpy(
        dtype=float
    )

    standard_deviation = (
        plot_df[
            "sd_area_percentage"
        ]
        .fillna(
            0.0
        )
        .to_numpy(
            dtype=float
        )
    )

    final = plot_df[
        "final_full_data_area_percentage"
    ].to_numpy(
        dtype=float
    )

    axis.bar(
        x,
        means,
        yerr=standard_deviation,
        capsize=3,
        edgecolor="black",
        linewidth=0.7,
        label="Cross-fold mean ± SD",
    )

    axis.scatter(
        x,
        final,
        marker="D",
        zorder=5,
        label="Final full-data map",
    )

    axis.set_xticks(
        x
    )

    axis.set_xticklabels(
        plot_df[
            "class_label"
        ],
        rotation=20,
        ha="right",
    )

    axis.set_ylabel(
        "Mapped area (%)"
    )

    axis.grid(
        True,
        axis="y",
        alpha=0.25,
    )

    axis.legend(
        frameon=False,
        fontsize=8,
    )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=UncertaintyConfig.FIG_DPI,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


# -----------------------------------------------------------------------------
# CARTOGRAPHIC HELPERS FOR THE PIXELWISE SD FIGURE ONLY
# -----------------------------------------------------------------------------
# These constants reproduce the established Step 7.1/7.2 ensemble-map style.
# They are deliberately local to the visualization layer and do not alter the
# Step 7.2D analytical/model configuration.
SD_MAP_PROJECT_BASE = Config.BASE
SD_MAP_STUDY_SHP = SD_MAP_PROJECT_BASE / "Study_Area.shp"
SD_MAP_ROAD_SHP = SD_MAP_PROJECT_BASE / "NTD_Road.shp"

SD_MAP_FIGSIZE = (4, 6)
SD_MAP_DMS_TICK_FONTSIZE = 7
SD_MAP_LEGEND_FONTSIZE = 7
SD_MAP_LEGEND_TITLE_FONTSIZE = 8
SD_MAP_COLORBAR_TICK_FONTSIZE = 8
SD_MAP_COLORBAR_LABEL_FONTSIZE = 9
SD_MAP_NORTH_ARROW_FONTSIZE = 8
SD_MAP_CAX_SIZE = "5%"
SD_MAP_CAX_PAD = 0.01
SD_MAP_DPI = 900

SD_MAP_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "rf_crossfold_sd",
    [
        "#006400",
        "#32CD32",
        "#FFFF00",
        "#FFA500",
        "#FF4500",
        "#8B0000",
    ],
    N=256,
)


def _sd_to_dms(value: float, is_lon: bool = True) -> str:
    absolute = abs(value)
    degrees = int(absolute)
    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_lon
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def _sd_metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def _sd_nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base_value = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base_value
        if candidate >= target_metres:
            return candidate

    return 10 * base_value


def _sd_add_north_arrow(
    axis,
    size: float = 0.08,
    pad: float = 0.06,
    label: str = "N",
    arrow_downshift: float = 0.025,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=SD_MAP_NORTH_ARROW_FONTSIZE,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def _sd_add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    mid_lat_deg: float,
    segments: int = 4,
    fontsize: float = 6,
    frame_lw: float = 0.4,
) -> None:
    width_data = east - west
    height_data = north - south

    metres_per_degree = _sd_metres_per_degree_lon(mid_lat_deg)
    width_metres = width_data * metres_per_degree
    data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = _sd_nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = total_length_metres * data_units_per_metre
    segment_length_units = segment_length_metres * data_units_per_metre

    x_start = west + 0.5 * width_data - 0.5 * total_length_units
    y_base = south + 0.06 * height_data
    bar_height = 0.018 * height_data

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_data

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (x_start + index * segment_length_units, y_base),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=("black" if index % 2 == 0 else "white"),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_data,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_data,
        "0",
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_data,
        end_label,
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )


def _sd_reproject_to_epsg4326(
    source_path: Path,
    output_path: Path,
) -> Path:
    """Create an EPSG:4326 display copy; the analytical SD raster is unchanged."""
    require_file(
        source_path,
        "cross-fold SD probability raster for visualization",
    )

    with rasterio.open(source_path) as source:
        if source.crs is None:
            raise RuntimeError(
                f"Cross-fold SD raster has no CRS: {source_path}"
            )

        if source.crs.to_epsg() == 4326:
            return source_path

        destination_crs = "EPSG:4326"
        destination_transform, width, height = calculate_default_transform(
            source.crs,
            destination_crs,
            source.width,
            source.height,
            *source.bounds,
        )

        output_nodata = (
            float(source.nodata)
            if source.nodata is not None
            else -9999.0
        )

        profile = source.profile.copy()
        profile.update(
            driver="GTiff",
            crs=destination_crs,
            transform=destination_transform,
            width=width,
            height=height,
            count=1,
            dtype="float32",
            nodata=output_nodata,
            compress="LZW",
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as destination:
            reproject(
                source=rasterio.band(source, 1),
                destination=rasterio.band(destination, 1),
                src_transform=source.transform,
                src_crs=source.crs,
                src_nodata=source.nodata,
                dst_transform=destination_transform,
                dst_crs=destination_crs,
                dst_nodata=output_nodata,
                resampling=Resampling.bilinear,
            )

            destination.set_band_description(
                1,
                "RF cross-fold SD of A30/E70 susceptibility probability - EPSG:4326 visualization copy",
            )

    if output_path.exists():
        output_path.unlink()

    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[SD-VIZ] EPSG:4326 display raster created: "
        f"{output_path}"
    )

    return output_path


def _sd_load_vector_overlay(
    vector_path: Path,
    target_crs: str,
) -> dict[str, list[dict[str, Any]]]:
    """Load a vector with Fiona and transform geometries to the display CRS."""
    require_file(
        vector_path,
        "cartographic vector overlay",
    )

    if not HAVE_FIONA:
        raise ImportError(
            "Fiona is required for the Step 7.1-style SD map overlays. "
            "The successful Step 7.1/7.2 workflow also uses Fiona-compatible shapefiles."
        )

    geometries: list[dict[str, Any]] = []

    with fiona.open(vector_path, "r") as source:
        source_crs = source.crs_wkt or source.crs
        if not source_crs:
            raise RuntimeError(
                f"Vector layer has no CRS: {vector_path}"
            )

        for feature in source:
            geometry = feature.get("geometry")
            if geometry is None:
                continue

            geometries.append(
                transform_geom(
                    source_crs,
                    target_crs,
                    geometry,
                    precision=-1,
                )
            )

    return {"geometries": geometries}


def _sd_plot_vector_overlay(
    axis,
    vector_data: dict[str, list[dict[str, Any]]],
    colour: str,
    linewidth: float,
    zorder: int,
) -> None:
    for geometry in vector_data.get("geometries", []):
        if not geometry:
            continue

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")
        if coordinates is None:
            continue

        if geometry_type == "LineString":
            line_groups = [coordinates]
        elif geometry_type == "MultiLineString":
            line_groups = list(coordinates)
        elif geometry_type == "Polygon":
            line_groups = [coordinates[0]] if coordinates else []
        elif geometry_type == "MultiPolygon":
            line_groups = [
                polygon[0]
                for polygon in coordinates
                if polygon
            ]
        else:
            continue

        for line in line_groups:
            xy = np.asarray(line, dtype="float64")
            if xy.ndim != 2 or xy.shape[1] < 2:
                continue

            axis.plot(
                xy[:, 0],
                xy[:, 1],
                color=colour,
                linewidth=linewidth,
                zorder=zorder,
            )


def _sd_build_clip_patch(
    study_vector: dict[str, list[dict[str, Any]]],
    axis,
):
    """Build the same opaque-background polygon clip used by the Step 7.1 map."""
    all_vertices: list[np.ndarray] = []
    all_codes: list[int] = []

    def add_ring(ring_coordinates) -> None:
        coordinates = np.asarray(
            ring_coordinates,
            dtype="float64",
        )

        if (
            coordinates.ndim != 2
            or coordinates.shape[0] < 3
            or coordinates.shape[1] < 2
        ):
            return

        coordinates = coordinates[:, :2]

        if not np.allclose(
            coordinates[0],
            coordinates[-1],
            equal_nan=False,
        ):
            coordinates = np.vstack(
                [coordinates, coordinates[0]]
            )

        n_vertices = len(coordinates)
        codes = (
            [MplPath.MOVETO]
            + [MplPath.LINETO] * (n_vertices - 2)
            + [MplPath.CLOSEPOLY]
        )

        all_vertices.append(coordinates)
        all_codes.extend(codes)

    for geometry in study_vector.get("geometries", []):
        if not geometry:
            continue

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")

        if geometry_type == "Polygon" and coordinates is not None:
            for ring in coordinates:
                add_ring(ring)

        elif geometry_type == "MultiPolygon" and coordinates is not None:
            for polygon in coordinates:
                for ring in polygon:
                    add_ring(ring)

    if not all_vertices:
        return None

    compound_path = MplPath(
        np.concatenate(all_vertices, axis=0),
        all_codes,
    )

    clip_patch = PathPatch(
        compound_path,
        facecolor="none",
        edgecolor="none",
        linewidth=0.0,
        transform=axis.transData,
        zorder=0,
    )

    axis.add_patch(clip_patch)
    return clip_patch


def create_pixelwise_sd_figure(
    sd_raster_path: Path,
    output_path: Path,
) -> None:
    """
    Plot the existing RF pixelwise cross-fold SD raster with the exact
    cartographic style already used for the corrected Step 7.1D SD map.

    This function changes visualization only. It does not change the RF SD raster,
    the five RF cross-fold realizations, the aggregation, ddof=1, domain QA, or
    any uncertainty statistic.
    """
    if not UncertaintyConfig.GENERATE_PIXELWISE_SD_FIGURE:
        return

    require_file(
        sd_raster_path,
        "cross-fold SD probability raster",
    )
    require_file(
        SD_MAP_STUDY_SHP,
        "study-area shapefile used by the Step 7.1 map style",
    )
    require_file(
        SD_MAP_ROAD_SHP,
        "road shapefile used by the Step 7.1 map style",
    )

    display_raster_path = output_path.with_name(
        output_path.stem + "_EPSG4326_Display.tif"
    )

    display_raster_path = _sd_reproject_to_epsg4326(
        source_path=sd_raster_path,
        output_path=display_raster_path,
    )

    with rasterio.open(display_raster_path) as dataset:
        sd = (
            dataset.read(1, masked=True)
            .filled(np.nan)
            .astype("float64")
        )
        bounds = dataset.bounds

    valid_sd = sd[np.isfinite(sd)]
    if valid_sd.size == 0:
        raise RuntimeError(
            "No valid values found in the cross-fold SD raster for plotting."
        )

    extent = (
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    )

    study_boundary = _sd_load_vector_overlay(
        SD_MAP_STUDY_SHP,
        "EPSG:4326",
    )
    roads = _sd_load_vector_overlay(
        SD_MAP_ROAD_SHP,
        "EPSG:4326",
    )

    display_array = np.ma.masked_invalid(sd)

    map_cmap = SD_MAP_CMAP.copy()
    map_cmap.set_bad(
        color="white",
        alpha=1.0,
    )

    sd_vmax = float(np.max(valid_sd))
    if not np.isfinite(sd_vmax) or sd_vmax <= 0.0:
        sd_vmax = 1.0

    figure, axis = plt.subplots(
        1,
        1,
        figsize=SD_MAP_FIGSIZE,
        facecolor="white",
    )
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        vmin=0.0,
        vmax=sd_vmax,
        cmap=map_cmap,
        interpolation="nearest",
        alpha=1.0,
        zorder=1,
    )

    clip_patch = _sd_build_clip_patch(
        study_boundary,
        axis,
    )
    if clip_patch is not None:
        image.set_clip_path(clip_patch)

    _sd_plot_vector_overlay(
        axis,
        study_boundary,
        colour="black",
        linewidth=2.0,
        zorder=1000,
    )
    _sd_plot_vector_overlay(
        axis,
        roads,
        colour="red",
        linewidth=1.0,
        zorder=1100,
    )

    # Exact raster extent, matching the successful Step 7.1 ensemble map.
    axis.set_xlim(extent[0], extent[1])
    axis.set_ylim(extent[2], extent[3])

    axis.xaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: _sd_to_dms(value, True)
        )
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: _sd_to_dms(value, False)
        )
    )
    axis.tick_params(
        axis="both",
        which="major",
        labelsize=SD_MAP_DMS_TICK_FONTSIZE,
    )

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")

    for label in axis.get_yticklabels():
        label.set_rotation(90)
        label.set_va("center")

    _sd_add_north_arrow(
        axis,
        size=0.08,
        pad=0.06,
    )

    _sd_add_checker_scale_bar(
        axis,
        extent[0],
        extent[1],
        extent[2],
        extent[3],
        mid_lat_deg=0.5 * (extent[2] + extent[3]),
        segments=4,
        fontsize=6,
    )

    legend_handles = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=1.5,
            label="Highway/Roads",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=2.0,
            label="Study Area Boundary",
        ),
    ]

    legend = axis.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.50),
        fontsize=SD_MAP_LEGEND_FONTSIZE,
        title="Legend",
        title_fontsize=SD_MAP_LEGEND_TITLE_FONTSIZE,
        frameon=True,
        fancybox=True,
        shadow=True,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0,
    )
    legend.get_title().set_fontweight("bold")
    legend.set_zorder(2000)

    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes(
        "right",
        size=SD_MAP_CAX_SIZE,
        pad=SD_MAP_CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(
        labelsize=SD_MAP_COLORBAR_TICK_FONTSIZE,
    )
    colorbar.set_label(
        "Cross-fold probability SD",
        fontsize=SD_MAP_COLORBAR_LABEL_FONTSIZE,
        fontweight="bold",
    )

    figure.tight_layout()

    pdf_path = output_path.with_suffix(".pdf")
    svg_path = output_path.with_suffix(".svg")

    figure.savefig(
        output_path,
        dpi=SD_MAP_DPI,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False,
    )
    figure.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False,
    )
    figure.savefig(
        svg_path,
        format="svg",
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False,
    )

    plt.close(figure)

    log(f"[SD-VIZ] PNG saved: {output_path}")
    log(f"[SD-VIZ] PDF saved: {pdf_path}")
    log(f"[SD-VIZ] SVG saved: {svg_path}")


# =============================================================================
# REPORT
# =============================================================================
def write_uncertainty_report(
    paths: Dict[str, Path],
    performance_intervals: pd.DataFrame,
    additional_comparisons: pd.DataFrame,
    area_summary: pd.DataFrame,
    domain_qa: pd.DataFrame,
    sd_summary: pd.DataFrame,
) -> None:
    with open(
        paths[
            "report"
        ],
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(
            "STEP 7.2D — RANDOM FOREST PERFORMANCE AND "
            "MAPPED-AREA UNCERTAINTY\n"
        )

        handle.write(
            "="
            * 92
            + "\n\n"
        )

        handle.write(
            "Generated: "
            + utc_now_iso()
            + "\n\n"
        )

        handle.write(
            "SCIENTIFIC SCOPE\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            "Primary model framework: successful Step 7.2 equal-HKFI, 100 m "
            "pseudo-absence buffer Random Forest experiment.\n"
        )

        handle.write(
            "Step 7.2B alternative HKFI weights and Step 7.2C alternative "
            "pseudo-absence buffers are treated as separate sensitivity "
            "analyses and are not pooled into this uncertainty analysis.\n\n"
        )

        handle.write(
            "PERFORMANCE UNCERTAINTY\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            "Intervals are spatial-block cluster-bootstrap intervals generated "
            "from genuine successful Step 7.2 nested outer-fold RF OOF "
            "predictions.\n\n"
        )

        handle.write(
            performance_intervals.to_string(
                index=False
            )
        )

        handle.write(
            "\n\n"
        )

        handle.write(
            "ADDITIONAL PAIRED AUC COMPARISONS\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            additional_comparisons.to_string(
                index=False
            )
        )

        handle.write(
            "\n\n"
        )

        handle.write(
            "CROSS-FOLD MAPPED-AREA VARIABILITY\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            "The five RF map realizations are summarized by mean, SD, minimum "
            "and maximum. These are not formal 95% confidence intervals.\n\n"
        )

        handle.write(
            area_summary.to_string(
                index=False
            )
        )

        handle.write(
            "\n\n"
        )

        handle.write(
            "RASTER VALID-DOMAIN QA\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            domain_qa.to_string(
                index=False
            )
        )

        handle.write(
            "\n\n"
        )

        handle.write(
            "PIXELWISE CROSS-FOLD SD SUMMARY\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            sd_summary.to_string(
                index=False
            )
        )

        handle.write(
            "\n"
        )


# =============================================================================
# CONTROLLER
# =============================================================================
def run_controller() -> None:
    paths = output_paths()

    # Validate the primary Step 7.2 data/model contract before analysis.
    scenarios = validate_scenario_manifests()

    load_and_validate_model_ready_dataset(
        scenarios
    )

    load_outer_best_params()

    # Validate final RF ensemble map before expensive cross-fold map work.
    require_file(
        Config.ENSEMBLE_PROBABILITY_TIF,
        "successful final Step 7.2 RF A30/E70 ensemble probability raster",
    )

    write_diagnostic_provenance(
        paths[
            "out"
        ]
    )

    log(
        "="
        * 96
    )

    log(
        "STEP 7.2D — RANDOM FOREST PERFORMANCE AND MAPPED-AREA UNCERTAINTY"
    )

    log(
        "="
        * 96
    )

    # -------------------------------------------------------------------------
    # 1. Performance uncertainty — no RF model fitting.
    # -------------------------------------------------------------------------
    log(
        "[1/6] Calculating RF cluster-bootstrap uncertainty "
        "for Models A-F and the A30/E70 ensemble..."
    )

    performance_intervals = build_performance_uncertainty(
        paths[
            "out"
        ]
    )

    # -------------------------------------------------------------------------
    # 2. Additional direct paired AUC comparisons.
    # -------------------------------------------------------------------------
    log(
        "[2/6] Calculating direct RF E-C and E-D paired AUC comparisons..."
    )

    additional_comparisons = add_missing_E_C_and_E_D_comparisons(
        paths[
            "out"
        ]
    )

    # -------------------------------------------------------------------------
    # 3. Five process-isolated cross-fold RF ensemble maps.
    # -------------------------------------------------------------------------
    log(
        "[3/6] Building/resuming five process-isolated "
        "RF cross-fold A30/E70 map realizations..."
    )

    for outer_fold in Config.EXPECTED_FOLDS:
        fold_output = fold_paths(
            outer_fold
        )

        complete = (
            fold_output[
                "done"
            ].exists()
            and fold_output[
                "ensemble_raster"
            ].exists()
            and fold_output[
                "area_csv"
            ].exists()
            and fold_output[
                "metadata_json"
            ].exists()
        )

        if (
            UncertaintyConfig.RESUME_FROM_CHECKPOINTS
            and complete
        ):
            log(
                f"[RESUME] RF outer fold {outer_fold} already complete."
            )

            continue

        run_external_fold_worker(
            outer_fold
        )

    # -------------------------------------------------------------------------
    # 4. Collect mapped areas and validate raster domains.
    # -------------------------------------------------------------------------
    log(
        "[4/6] Collecting RF fold-area results and checking "
        "identical raster support domains..."
    )

    (
        area_raw,
        ensemble_rasters,
    ) = collect_fold_area_results()

    area_raw.to_csv(
        paths[
            "area_raw"
        ],
        index=False,
        float_format="%.10f",
    )

    domain_qa = validate_cross_fold_raster_domains(
        fold_rasters=ensemble_rasters,
        final_full_data_raster=(
            Config.ENSEMBLE_PROBABILITY_TIF
        ),
        output_csv=paths[
            "domain_qa"
        ],
    )

    # -------------------------------------------------------------------------
    # 5. Area statistics and pixelwise mean/SD.
    # -------------------------------------------------------------------------
    log(
        "[5/6] Summarizing RF mapped-area variability and "
        "aggregating pixelwise mean/SD rasters..."
    )

    area_summary = summarize_mapped_area_uncertainty(
        area_raw=area_raw
    )

    area_summary.to_csv(
        paths[
            "area_summary"
        ],
        index=False,
        float_format="%.10f",
    )

    aggregate_fold_rasters(
        fold_rasters=ensemble_rasters,
        mean_output=paths[
            "mean_raster"
        ],
        sd_output=paths[
            "sd_raster"
        ],
    )

    sd_summary = summarize_pixelwise_sd(
        sd_raster_path=paths[
            "sd_raster"
        ],
        output_csv=paths[
            "sd_summary"
        ],
    )

    # -------------------------------------------------------------------------
    # 6. Figures and report — same Step 7.1D style.
    # -------------------------------------------------------------------------
    log(
        "[6/6] Creating RF uncertainty figures and final report..."
    )

    create_area_uncertainty_figure(
        area_summary=area_summary,
        output_path=paths[
            "area_figure"
        ],
    )

    create_pixelwise_sd_figure(
        sd_raster_path=paths[
            "sd_raster"
        ],
        output_path=paths[
            "sd_figure"
        ],
    )

    write_uncertainty_report(
        paths=paths,
        performance_intervals=performance_intervals,
        additional_comparisons=additional_comparisons,
        area_summary=area_summary,
        domain_qa=domain_qa,
        sd_summary=sd_summary,
    )

    log(
        ""
    )

    log(
        "="
        * 96
    )

    log(
        "STEP 7.2D COMPLETED SUCCESSFULLY"
    )

    log(
        "="
        * 96
    )

    log(
        f"[OUT] "
        f"{paths['out']}"
    )

    print(
        "\nAdditional paired RF AUC comparisons:"
    )

    print(
        additional_comparisons.to_string(
            index=False
        )
    )

    print(
        "\nRF cross-fold mapped-area uncertainty:"
    )

    print(
        area_summary.to_string(
            index=False
        )
    )

    print(
        "\nRF pixelwise SD summary:"
    )

    print(
        sd_summary.to_string(
            index=False
        )
    )


# =============================================================================
# COMMAND-LINE ENTRY POINT
# =============================================================================
def parse_command_line_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Step 7.2D Random Forest performance and mapped-area uncertainty."
        )
    )

    parser.add_argument(
        "--worker-fold",
        type=int,
        default=None,
        help=(
            "Internal process-isolated RF worker mode. "
            "Valid values are 1-5."
        ),
    )

    arguments, unknown = parser.parse_known_args()

    if unknown:
        log(
            "[CLI] Ignoring host-environment arguments: "
            f"{unknown}"
        )

    return arguments


if __name__ == "__main__":
    try:
        args = parse_command_line_arguments()

        if args.worker_fold is None:
            run_controller()

        else:
            if (
                args.worker_fold
                not in Config.EXPECTED_FOLDS
            ):
                raise ValueError(
                    "--worker-fold must be one of "
                    f"{Config.EXPECTED_FOLDS}; "
                    f"received {args.worker_fold}."
                )

            run_outer_fold_map_worker(
                int(
                    args.worker_fold
                )
            )

    except Exception as exc:
        print(
            f"[ERROR] Step 7.2D failed: {exc}",
            flush=True,
        )

        traceback.print_exc()

        raise


# Step 7.3 — Logistic Regression (LR) Models A–F with Nested Spatial Cross-Validation

In [ ]:
# =============================================================================
# Step 7.3 — Logistic Regression Models A–F with Nested Spatial Cross-Validation
# =============================================================================
#
# Purpose
# -------
# Train and evaluate Logistic Regression for all six frozen A–F predictor configurations
# using the primary frozen Step 6.4 ML dataset and EXACT frozen five spatial folds.
#
# Design
# ------
# Outer CV (unbiased evaluation):
#   - One frozen spatial block is held out.
#   - The remaining four blocks are used for all model selection.
#
# Inner CV (hyperparameter tuning):
#   - Each of the four remaining spatial blocks serves once as the inner
#     validation fold via PredefinedSplit.
#   - Grid-search AUC is computed only inside the outer-training data.
#
# After nested-CV evaluation:
#   - OOF predictions are pooled across the five outer folds.
#   - A separate full-data spatial grid search can be enabled later to select
#     final deployment/map models. These full-data models are NOT used for
#     validation statistics.
#
# Important scientific rules
# --------------------------
# 1. Never regenerate spatial folds. Use Step 6.4 cv_fold exactly.
# 2. Models A–F use identical rows, labels and outer folds.
# 3. No global imputation, scaling, downsampling or train/test split.
# 4. Land cover is treated as categorical and one-hot encoded inside each fit.
# 5. Logistic Regression numeric predictors are standardized inside each training fit only.
# 6. IRS is never used as an ML predictor.
# 7. Outer validation data are never used for tuning or model selection.
# 8. Binary metrics use the fixed threshold 0.50. AUC remains primary.
# 9. Cluster-bootstrap confidence intervals resample the five spatial blocks,
#    preserving the spatial grouping structure better than sample-wise bootstrap.
# 10. Final susceptibility visualization uses the direct Model E deployment
#     probability raster; it never substitutes for nested spatial OOF validation.
#
# Primary reviewer comparisons supported
# --------------------------------------
# B - A : added value of raw DV+
# C - A : added value of HKFI alone
# D - A : added value of SNDR alone
# E - A : total added value of the proposed physics-guided indices
# F - A : added value of supplying DV+, TWI and SM as separate source variables
# E - B : physics-guided indices versus raw deformation alone
# E - F : transformed indices versus supplying source variables separately
# =============================================================================

from __future__ import annotations

import os
# =============================================================================
# RESOURCE CONTROL
# =============================================================================
# Logistic Regression itself and GridSearchCV are both kept single-threaded here.
# This avoids nested CPU parallelism, makes runs more reproducible, and follows
# the same resource-safe principle used in the successful Step 7.3 workflow.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


import json
import os
import re
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Sequence, Tuple

import joblib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Polygon, Rectangle
from matplotlib.ticker import FuncFormatter, LinearLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling,
    transform_geom,
)
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Optional cartographic/vector libraries.
try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    HAVE_FIONA = False

try:
    from IPython.display import display
    HAVE_IPYTHON_DISPLAY = True
except Exception:
    HAVE_IPYTHON_DISPLAY = False


# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Step 6.1 frozen predictor definitions and raster stack.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"
    MASTER_STACK_TIF = (
        STEP61_DIR
        / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
    )
    COMMON_AF_MASK_TIF = (
        STEP61_DIR / "common_A_to_F_valid_mask_32645_12p5m_byte.tif"
    )

    # Step 6.4 frozen modelling contract.
    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    MODEL_READY_DATASET_CSV = STEP64_DIR / "model_ready_dataset_A_to_F.csv"
    STEP64_MODEL_CONFIG_JSON = STEP64_DIR / "model_configuration_manifest.json"

    # Outputs.
    OUT_DIR = RESULTS_DIR / "Step-7.3-LR-Models_A_to_F"
    MODEL_DIR = OUT_DIR / "final_models"
    MAP_DIR = OUT_DIR / "susceptibility_maps"
    SEARCH_DIR = OUT_DIR / "grid_search_details"

    OOF_PREDICTIONS_CSV = OUT_DIR / "LR_A_to_F_OOF_predictions.csv"
    FOLD_METRICS_CSV = OUT_DIR / "LR_A_to_F_outer_fold_metrics.csv"
    SUMMARY_METRICS_CSV = OUT_DIR / "LR_A_to_F_summary_metrics.csv"
    AUC_COMPARISONS_CSV = OUT_DIR / "LR_A_to_F_paired_AUC_comparisons.csv"
    OUTER_PARAMS_JSON = OUT_DIR / "LR_A_to_F_outer_fold_best_parameters.json"
    FINAL_PARAMS_JSON = OUT_DIR / "LR_A_to_F_final_full_data_best_parameters.json"
    INNER_SEARCH_CSV = SEARCH_DIR / "LR_A_to_F_nested_inner_grid_results.csv"
    FINAL_SEARCH_CSV = SEARCH_DIR / "LR_A_to_F_full_data_grid_results.csv"
    ROC_FIG_PNG = OUT_DIR / "LR_A_to_F_OOF_ROC_Curves.png"
    ROC_FIG_PDF = OUT_DIR / "LR_A_to_F_OOF_ROC_Curves.pdf"
    ROC_FIG_SVG = OUT_DIR / "LR_A_to_F_OOF_ROC_Curves.svg"
    FOLD_AUC_FIG_PNG = OUT_DIR / "LR_A_to_F_Fold_AUC.png"
    FOLD_AUC_FIG_PDF = OUT_DIR / "LR_A_to_F_Fold_AUC.pdf"
    FOLD_AUC_FIG_SVG = OUT_DIR / "LR_A_to_F_Fold_AUC.svg"
    QA_REPORT_TXT = OUT_DIR / "Step_7_3_LR_A_to_F_QA_Report.txt"
    METADATA_JSON = OUT_DIR / "Step_7_3_LR_A_to_F_Metadata.json"

    # Manuscript-ready Excel workbook containing the compact A-F performance
    # table, paired AUC comparisons, fold-level AUCs and ensemble summary.
    MANUSCRIPT_TABLES_XLSX = (
        OUT_DIR / "LR_A_to_F_Manuscript_Performance_Tables.xlsx"
    )

    # Cartographic overlays used by the established Step 5.3 map style.
    STUDY_SHP = BASE / "Study_Area.shp"
    ROAD_SHP = BASE / "NTD_Road.shp"
    VISUALIZATION_DIR = MAP_DIR / "visualization"

    # Primary A-F model retained for scientific interpretation.
    PRIMARY_VISUALIZATION_SCENARIO = "E_physics_guided"

    # Old-style static + integrated ensemble analogue for the revised A-F design.
    #
    # Old workflow:
    #   final = 0.30 * context/static + 0.70 * integrated
    #   where integrated support exists; otherwise retain static background.
    #
    # Revised workflow:
    #   final = 0.30 * Model A + 0.70 * Model E
    #   where Model E predictors are available; otherwise retain Model A.
    #
    # The 0.70 weight is fixed a priori to reproduce the earlier ensemble logic.
    # It is NOT tuned on the OOF predictions.
    GENERATE_AE_ENSEMBLE_LSM = True
    ENSEMBLE_STATIC_SCENARIO = "A_static_baseline"
    ENSEMBLE_ENHANCED_SCENARIO = "E_physics_guided"
    ENSEMBLE_ENHANCED_WEIGHT = 0.70

    ENSEMBLE_STATIC_FULL_DOMAIN_TIF = (
        MAP_DIR / "A-static-baseline_LR_probability_full_available_domain.tif"
    )
    ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF = (
        MAP_DIR / "E-physics-guided_LR_probability_full_available_domain.tif"
    )
    ENSEMBLE_PROBABILITY_TIF = (
        MAP_DIR / "LR_Ensemble_A30_E70_probability_full_study_domain.tif"
    )
    ENSEMBLE_SUPPORT_TIF = (
        MAP_DIR / "LR_Ensemble_A30_E70_information_support.tif"
    )
    ENSEMBLE_OOF_PREDICTIONS_CSV = (
        OUT_DIR / "LR_Ensemble_A30_E70_OOF_predictions.csv"
    )
    ENSEMBLE_OOF_METRICS_JSON = (
        OUT_DIR / "LR_Ensemble_A30_E70_OOF_metrics.json"
    )

    # Step 5.3-style map/histogram settings.
    # Publication-size figures.
    # The two requested maps are explicitly controlled independently so that
    # later changes to other figures cannot accidentally alter their dimensions.
    ENSEMBLE_MAP_FIGSIZE = (4, 6)
    SUPPORT_MAP_FIGSIZE = (4, 6)

    # Other visualization products.
    MAP_FIGSIZE = (4, 4)
    HIST_FIGSIZE = (4, 4)
    AREA_PERCENT_FIGSIZE = (4, 4)

    # Enlarged cartographic text for the 4 x 4 inch figure.
    DMS_TICK_FONTSIZE = 7  # Reduced from 9 to prevent latitude label overlap
    LEGEND_FONTSIZE = 7
    LEGEND_TITLE_FONTSIZE = 8
    COLORBAR_TICK_FONTSIZE = 8
    COLORBAR_LABEL_FONTSIZE = 9
    NORTH_ARROW_FONTSIZE = 8  # Slightly reduced for balance
    SCALEBAR_FONTSIZE = 7

    CAX_SIZE = "5%"
    CAX_PAD = 0.01
    FIG_DPI = 900
    DISPLAY_BUFFER_FRACTION = 0.00
    LONGITUDE_DMS_TICK_COUNT = 4
    RASTER_DISPLAY_INTERPOLATION = "nearest"
    SHOW_LSM_FIGURES = True
    SAVE_LSM_FIGURES = True

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"
    CATEGORICAL_FEATURE_ALIASES = {"landcover", "landuse", "land_use"}

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    MODEL_DESCRIPTIONS = {
        "A_static_baseline": "Static baseline",
        "B_raw_deformation": "Static + DV+",
        "C_hkfi_only": "Static + HKFI",
        "D_sndr_only": "Static + SNDR",
        "E_physics_guided": "Static + HKFI + SNDR",
        "F_untransformed_sources": "Static + DV+ + TWI + SM",
    }

    # Logistic-Regression-specific regularization grid.
    #
    # Manuscript Section 3.4.2 contract:
    #   - liblinear solver
    #   - L2 / Ridge regularization
    #   - balanced class weights
    #
    # Only C (inverse regularization strength) is tuned. Hyperparameter
    # selection remains entirely inside the outer-training data.
    PARAM_GRID = {
        "model__C": [
            0.01,
            0.05,
            0.10,
            0.50,
            1.00,
            5.00,
            10.00,
            100.00,
        ],
    }

    # Fixed Logistic Regression settings shared by Models A-F.
    LR_FIXED_PARAMS = {
        "penalty": "l2",
        "solver": "liblinear",
        "class_weight": "balanced",
        "fit_intercept": True,
        "max_iter": 10000,
        "tol": 1.0e-6,
        "random_state": 42,
    }

    GRIDSEARCH_N_JOBS = 1
    GRIDSEARCH_VERBOSE = 2
    RANDOM_STATE = 42
    BINARY_THRESHOLD = 0.50

    # Spatial-block cluster bootstrap.
    N_CLUSTER_BOOTSTRAP = 5000
    CI_LEVEL = 0.95

    # Complete execution controls.
    #
    # True  -> after nested spatial-CV evaluation, tune and fit the final
    #          full-data deployment models for A-F.
    # False -> stop after leakage-safe nested spatial OOF evaluation.
    RUN_FINAL_DEPLOYMENT_TUNING = True

    # True  -> generate A-F full-domain probability rasters from the final
    #          deployment models. Requires RUN_FINAL_DEPLOYMENT_TUNING=True.
    GENERATE_RASTER_MAPS = True

    # True  -> create the publication-style primary Model E landslide
    #          susceptibility map and probability histogram.
    GENERATE_PRIMARY_LSM_VISUALIZATION = True
    RASTER_BLOCK_ROWS = 512
    OUTPUT_NODATA = -9999.0

    # Keep nested-CV fold estimators only if needed for later diagnostics.
    SAVE_OUTER_FOLD_MODELS = False


# Create output folders once the script is imported/run.
for _directory in (
    Config.OUT_DIR,
    Config.MODEL_DIR,
    Config.MAP_DIR,
    Config.SEARCH_DIR,
    Config.VISUALIZATION_DIR,
):
    _directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


# =============================================================================
# FIGURE STYLE HELPERS
# =============================================================================
AXIS_LABEL_FONTSIZE = 11
TICK_LABEL_FONTSIZE = 10
LEGEND_FONTSIZE = 9
TITLE_FONTSIZE = 11

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.labelsize": AXIS_LABEL_FONTSIZE,
    "axes.labelweight": "bold",
    "axes.titlesize": TITLE_FONTSIZE,
    "axes.titleweight": "bold",
    "xtick.labelsize": TICK_LABEL_FONTSIZE,
    "ytick.labelsize": TICK_LABEL_FONTSIZE,
    "legend.fontsize": LEGEND_FONTSIZE,
    "axes.unicode_minus": False,
})


def apply_axis_text_style(
    ax: plt.Axes,
    legend_fontsize: float = LEGEND_FONTSIZE,
) -> None:
    """Apply consistent bold axis labels and tick labels to statistical plots."""
    ax.xaxis.label.set_fontsize(AXIS_LABEL_FONTSIZE)
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontsize(AXIS_LABEL_FONTSIZE)
    ax.yaxis.label.set_fontweight("bold")

    ax.title.set_fontsize(TITLE_FONTSIZE)
    ax.title.set_fontweight("bold")

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=TICK_LABEL_FONTSIZE,
        width=0.8,
        length=3.5,
    )

    for tick_label in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        tick_label.set_fontsize(TICK_LABEL_FONTSIZE)
        tick_label.set_fontweight("bold")

    ax.xaxis.get_offset_text().set_fontsize(TICK_LABEL_FONTSIZE)
    ax.xaxis.get_offset_text().set_fontweight("bold")
    ax.yaxis.get_offset_text().set_fontsize(TICK_LABEL_FONTSIZE)
    ax.yaxis.get_offset_text().set_fontweight("bold")

    legend = ax.get_legend()
    if legend is not None:
        for label in legend.get_texts():
            label.set_fontsize(legend_fontsize)


def save_figure_triplet(
    fig: plt.Figure,
    png_path: Path,
    pdf_path: Path,
    svg_path: Path,
    dpi: int = 600,
) -> None:
    """Save one figure as PNG, PDF and SVG."""
    fig.tight_layout()
    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        svg_path,
        format="svg",
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json(path: Path) -> Dict[str, Any]:
    require_file(path, "JSON file")
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


def scenario_prefix(scenario_name: str) -> str:
    return scenario_name.replace("_", "-")


def normalize_feature_name(name: str) -> str:
    """Normalize a feature name for robust alias matching."""
    return re.sub(r"[^a-z0-9_]+", "_", str(name).strip().lower()).strip("_")


def is_categorical_feature(name: str) -> bool:
    """Identify categorical predictors using conservative land-cover aliases."""
    normalized = normalize_feature_name(name)
    compact = normalized.replace("_", "")
    aliases = {alias.lower() for alias in Config.CATEGORICAL_FEATURE_ALIASES}
    compact_aliases = {alias.replace("_", "") for alias in aliases}
    return normalized in aliases or compact in compact_aliases


# =============================================================================
# DATA / MANIFEST VALIDATION
# =============================================================================
def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """Cross-check Step 6.1 and frozen Step 6.4 A–F definitions."""
    step61 = load_json(Config.SCENARIO_MANIFEST_JSON).get("scenarios", {})
    step64 = load_json(Config.STEP64_MODEL_CONFIG_JSON).get("scenarios", {})

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                f"Step 6.1 scenario manifest is missing {scenario_name}."
            )
        if scenario_name not in step64:
            raise RuntimeError(
                f"Step 6.4 frozen model manifest is missing {scenario_name}."
            )

        features_61 = list(step61[scenario_name].get("features", []))
        features_64 = list(step64[scenario_name].get("features", []))

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any("irs" in feature.lower() for feature in features_61):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; IRS must not be an ML predictor."
            )

    log("[QA] Step 6.1 and Step 6.4 A–F scenario definitions match: PASS")
    return step64


def load_and_validate_model_ready_dataset(
    scenarios: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    """Load the primary frozen 330-row Step 6.4 ML dataset with strict QA."""
    require_file(
        Config.MODEL_READY_DATASET_CSV,
        "Step 6.4 primary frozen model-ready ML dataset",
    )

    df = pd.read_csv(Config.MODEL_READY_DATASET_CSV)

    required_metadata = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ]

    all_features: List[str] = []
    for scenario_name in Config.SCENARIO_ORDER:
        for feature in scenarios[scenario_name]["features"]:
            if feature not in all_features:
                all_features.append(feature)

    missing = [
        column
        for column in required_metadata + all_features
        if column not in df.columns
    ]
    if missing:
        raise RuntimeError(f"Model-ready dataset is missing columns: {missing}")

    if len(df) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Expected {Config.EXPECTED_N_ROWS} rows; found {len(df)}. "
            "No silent row dropping is allowed."
        )

    if df[Config.SAMPLE_ID_COLUMN].isna().any():
        raise RuntimeError("Missing sample_id values found.")
    if df[Config.SAMPLE_ID_COLUMN].duplicated().any():
        raise RuntimeError("Duplicated sample_id values found.")

    df[Config.TARGET_COLUMN] = pd.to_numeric(
        df[Config.TARGET_COLUMN], errors="raise"
    ).astype(int)
    df[Config.FOLD_COLUMN] = pd.to_numeric(
        df[Config.FOLD_COLUMN], errors="raise"
    ).astype(int)
    df[Config.BLOCK_COLUMN] = pd.to_numeric(
        df[Config.BLOCK_COLUMN], errors="raise"
    ).astype(int)

    class_counts = df[Config.TARGET_COLUMN].value_counts().sort_index().to_dict()
    expected_counts = {
        0: Config.EXPECTED_N_NEGATIVE,
        1: Config.EXPECTED_N_POSITIVE,
    }
    if class_counts != expected_counts:
        raise RuntimeError(
            f"Unexpected class counts: {class_counts}; expected {expected_counts}."
        )

    folds = sorted(df[Config.FOLD_COLUMN].unique().tolist())
    if folds != Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected frozen folds {Config.EXPECTED_FOLDS}; found {folds}."
        )

    if not np.array_equal(
        df[Config.FOLD_COLUMN].to_numpy(),
        df[Config.BLOCK_COLUMN].to_numpy(),
    ):
        raise RuntimeError("cv_fold and spatial_block are not identical.")

    # Every frozen block must contain exactly 33 positives and 33 pseudo-absences.
    block_class = pd.crosstab(
        df[Config.FOLD_COLUMN],
        df[Config.TARGET_COLUMN],
    )
    for required_class in (0, 1):
        if required_class not in block_class.columns:
            raise RuntimeError(
                f"Frozen fold table is missing class {required_class}."
            )

    expected_per_class_per_fold = Config.EXPECTED_N_POSITIVE // len(Config.EXPECTED_FOLDS)
    if expected_per_class_per_fold * len(Config.EXPECTED_FOLDS) != Config.EXPECTED_N_POSITIVE:
        raise RuntimeError(
            "Expected positive count is not divisible evenly across the frozen folds."
        )

    for fold in Config.EXPECTED_FOLDS:
        if fold not in block_class.index:
            raise RuntimeError(f"Missing fold {fold}.")
        n0 = int(block_class.loc[fold, 0])
        n1 = int(block_class.loc[fold, 1])
        if n0 != expected_per_class_per_fold or n1 != expected_per_class_per_fold:
            raise RuntimeError(
                f"Fold {fold} violates the frozen Step 6.4 contract: "
                f"negative={n0}, positive={n1}; expected "
                f"{expected_per_class_per_fold}+{expected_per_class_per_fold}=66."
            )

    log(
        "[QA] Frozen fold balance: each fold contains "
        f"{expected_per_class_per_fold} positives + "
        f"{expected_per_class_per_fold} pseudo-absences: PASS"
    )

    # Step 6.4 should already guarantee complete predictors; verify again.
    predictor_values = df[all_features].apply(pd.to_numeric, errors="coerce")
    if not np.isfinite(predictor_values.to_numpy(dtype="float64")).all():
        bad = np.argwhere(
            ~np.isfinite(predictor_values.to_numpy(dtype="float64"))
        )
        examples = [
            (
                str(df.iloc[row_i][Config.SAMPLE_ID_COLUMN]),
                all_features[col_i],
            )
            for row_i, col_i in bad[:20]
        ]
        raise RuntimeError(
            "Missing/non-finite predictors remain in the model-ready dataset. "
            f"Examples: {examples}"
        )

    irs_columns = [
        column for column in df.columns if "irs" in str(column).lower()
    ]
    if irs_columns:
        raise RuntimeError(
            f"IRS-related columns found in ML dataset: {irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError("tpi_s25 found; only tpi_s5 is allowed.")

    log(
        "[QA] Model-ready dataset: "
        f"rows={len(df)}, positive={class_counts[1]}, negative={class_counts[0]}, "
        f"folds={folds}: PASS"
    )
    return df


# =============================================================================
# MODEL BUILDING / NESTED SPATIAL CV
# =============================================================================
def build_lr_pipeline(features: Sequence[str]) -> Pipeline:
    """
    Create the fold-safe preprocessing + Logistic Regression pipeline.

    Numeric predictors are standardized using StandardScaler fitted only on the
    current training partition. Land cover is one-hot encoded inside the same
    pipeline. Thus all data-dependent preprocessing is repeated within every
    inner-CV fit, outer-fold refit, and final deployment fit without leakage.
    """
    categorical = [
        feature
        for feature in features
        if is_categorical_feature(feature)
    ]

    numeric = [
        feature
        for feature in features
        if not is_categorical_feature(feature)
    ]

    transformers = []

    if numeric:
        transformers.append(
            (
                "numeric",
                StandardScaler(),
                numeric,
            )
        )

    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                ),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = LogisticRegression(
        **Config.LR_FIXED_PARAMS
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                classifier,
            ),
        ]
    )


def make_inner_predefined_split(outer_train_df: pd.DataFrame) -> PredefinedSplit:
    """
    Build a four-fold spatial inner CV from the four blocks remaining after one
    outer block has been held out.
    """
    blocks = sorted(outer_train_df[Config.FOLD_COLUMN].unique().tolist())
    if len(blocks) != 4:
        raise RuntimeError(
            f"Nested CV expected four inner blocks; found {blocks}."
        )

    block_to_test_fold = {
        block: inner_index for inner_index, block in enumerate(blocks)
    }
    test_fold = outer_train_df[Config.FOLD_COLUMN].map(block_to_test_fold).to_numpy()
    return PredefinedSplit(test_fold=test_fold)


def make_full_data_predefined_split(df: pd.DataFrame) -> PredefinedSplit:
    """Use the frozen five blocks for final deployment-model tuning."""
    test_fold = df[Config.FOLD_COLUMN].to_numpy(dtype=int) - 1
    return PredefinedSplit(test_fold=test_fold)


def classification_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = Config.BINARY_THRESHOLD,
) -> Dict[str, Any]:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def cluster_bootstrap_auc_ci(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> Tuple[float, float, int]:
    """Cluster bootstrap AUC by resampling whole spatial blocks."""
    rng = np.random.default_rng(seed)
    unique_blocks = np.unique(blocks)
    values: List[float] = []

    indices_by_block = {
        block: np.flatnonzero(blocks == block) for block in unique_blocks
    }

    for i in range(n_bootstrap):
        if (i + 1) % 1000 == 0:
            log(f"  Bootstrap progress: {i + 1}/{n_bootstrap}")
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(unique_blocks),
            replace=True,
        )
        sampled_indices = np.concatenate(
            [indices_by_block[block] for block in sampled_blocks]
        )
        y_b = y_true[sampled_indices]
        p_b = y_prob[sampled_indices]

        if np.unique(y_b).size < 2:
            continue

        values.append(float(roc_auc_score(y_b, p_b)))

    if not values:
        return float("nan"), float("nan"), 0

    alpha = (1.0 - ci_level) / 2.0
    low, high = np.quantile(values, [alpha, 1.0 - alpha])
    return float(low), float(high), len(values)


def paired_cluster_bootstrap_auc_difference(
    y_true: np.ndarray,
    prob_left: np.ndarray,
    prob_right: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> Tuple[float, float, float, int]:
    """
    Paired spatial-block bootstrap for AUC(left) - AUC(right), preserving the
    identical samples/folds used by all A–F models.
    """
    observed = float(
        roc_auc_score(y_true, prob_left) - roc_auc_score(y_true, prob_right)
    )

    rng = np.random.default_rng(seed)
    unique_blocks = np.unique(blocks)
    indices_by_block = {
        block: np.flatnonzero(blocks == block) for block in unique_blocks
    }
    differences: List[float] = []

    for i in range(n_bootstrap):
        if (i + 1) % 1000 == 0:
            log(f"  Bootstrap progress: {i + 1}/{n_bootstrap}")
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(unique_blocks),
            replace=True,
        )
        sampled_indices = np.concatenate(
            [indices_by_block[block] for block in sampled_blocks]
        )
        y_b = y_true[sampled_indices]

        if np.unique(y_b).size < 2:
            continue

        difference = (
            roc_auc_score(y_b, prob_left[sampled_indices])
            - roc_auc_score(y_b, prob_right[sampled_indices])
        )
        differences.append(float(difference))

    if not differences:
        return observed, float("nan"), float("nan"), 0

    alpha = (1.0 - ci_level) / 2.0
    low, high = np.quantile(differences, [alpha, 1.0 - alpha])
    return observed, float(low), float(high), len(differences)


def normalize_grid_results(
    cv_results: Dict[str, Any],
    scenario_name: str,
    search_stage: str,
    outer_fold: int | None,
) -> pd.DataFrame:
    """Convert GridSearchCV.cv_results_ to a compact audit table."""
    frame = pd.DataFrame(cv_results)

    keep = [
        column
        for column in frame.columns
        if column.startswith("param_")
        or column
        in {
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "mean_fit_time",
            "std_fit_time",
        }
    ]

    frame = frame[keep].copy()
    frame.insert(0, "scenario", scenario_name)
    frame.insert(1, "model", Config.MODEL_SHORT_LABELS[scenario_name])
    frame.insert(2, "search_stage", search_stage)
    frame.insert(3, "outer_fold", outer_fold)
    return frame


def run_nested_spatial_cv_for_scenario(
    df: pd.DataFrame,
    scenario_name: str,
    features: Sequence[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any], pd.DataFrame]:
    """Run five outer spatial folds with four-block inner spatial grid search."""
    log("")
    log("=" * 96)
    log(
        f"[NESTED-CV] {Config.MODEL_SHORT_LABELS[scenario_name]} — "
        f"{Config.MODEL_DESCRIPTIONS[scenario_name]} ({len(features)} predictors)"
    )
    log("=" * 96)

    oof_probability = np.full(len(df), np.nan, dtype="float64")
    oof_predicted = np.full(len(df), -1, dtype="int16")

    fold_rows: List[Dict[str, Any]] = []
    best_params: Dict[str, Any] = {}
    search_tables: List[pd.DataFrame] = []

    for outer_fold in Config.EXPECTED_FOLDS:
        outer_train_mask = df[Config.FOLD_COLUMN].to_numpy() != outer_fold
        outer_valid_mask = df[Config.FOLD_COLUMN].to_numpy() == outer_fold

        train_df = df.loc[outer_train_mask].copy()
        valid_df = df.loc[outer_valid_mask].copy()

        if train_df[Config.FOLD_COLUMN].nunique() != 4:
            raise RuntimeError(
                f"Outer fold {outer_fold}: expected four training blocks."
            )
        if valid_df[Config.FOLD_COLUMN].nunique() != 1:
            raise RuntimeError(
                f"Outer fold {outer_fold}: validation must contain one block only."
            )

        X_train = train_df[list(features)]
        y_train = train_df[Config.TARGET_COLUMN].to_numpy(dtype=int)
        X_valid = valid_df[list(features)]
        y_valid = valid_df[Config.TARGET_COLUMN].to_numpy(dtype=int)

        inner_cv = make_inner_predefined_split(train_df)
        pipeline = build_lr_pipeline(features)

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=Config.PARAM_GRID,
            scoring="roc_auc",
            cv=inner_cv,
            refit=True,
            n_jobs=Config.GRIDSEARCH_N_JOBS,
            verbose=Config.GRIDSEARCH_VERBOSE,
            return_train_score=False,
            error_score="raise",
        )

        log(
            f"[OUTER {outer_fold}] Tuning on blocks "
            f"{sorted(train_df[Config.FOLD_COLUMN].unique().tolist())}; "
            f"holding out block {outer_fold}."
        )
        search.fit(X_train, y_train)

        probability = search.best_estimator_.predict_proba(X_valid)[:, 1]
        predicted = (probability >= Config.BINARY_THRESHOLD).astype(int)

        valid_indices = np.flatnonzero(outer_valid_mask)
        oof_probability[valid_indices] = probability
        oof_predicted[valid_indices] = predicted

        metrics = classification_metrics(y_valid, probability)

        row = {
            "scenario": scenario_name,
            "model": Config.MODEL_SHORT_LABELS[scenario_name],
            "description": Config.MODEL_DESCRIPTIONS[scenario_name],
            "outer_fold": int(outer_fold),
            "n_predictors": int(len(features)),
            "n_train": int(len(train_df)),
            "n_validation": int(len(valid_df)),
            "n_train_positive": int(y_train.sum()),
            "n_train_negative": int(len(y_train) - y_train.sum()),
            "n_validation_positive": int(y_valid.sum()),
            "n_validation_negative": int(len(y_valid) - y_valid.sum()),
            "inner_best_mean_auc": float(search.best_score_),
            **metrics,
        }
        fold_rows.append(row)

        best_params[str(outer_fold)] = {
            "inner_best_mean_auc": float(search.best_score_),
            "best_params": json_safe(search.best_params_),
        }

        search_tables.append(
            normalize_grid_results(
                search.cv_results_,
                scenario_name=scenario_name,
                search_stage="nested_inner",
                outer_fold=outer_fold,
            )
        )

        if Config.SAVE_OUTER_FOLD_MODELS:
            fold_model_path = (
                Config.MODEL_DIR
                / f"{scenario_prefix(scenario_name)}_outer-fold-{outer_fold}.joblib"
            )
            joblib.dump(search.best_estimator_, fold_model_path)

        log(
            f"[OUTER {outer_fold}] inner best AUC={search.best_score_:.4f}; "
            f"held-out AUC={metrics['roc_auc']:.4f}; "
            f"F1={metrics['f1']:.4f}"
        )

    if not np.isfinite(oof_probability).all():
        missing_count = int((~np.isfinite(oof_probability)).sum())
        raise RuntimeError(
            f"{scenario_name}: {missing_count} samples did not receive OOF predictions."
        )
    if (oof_predicted < 0).any():
        raise RuntimeError(f"{scenario_name}: incomplete OOF class predictions.")

    oof_df = df[
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
        ]
    ].copy()
    oof_df.insert(0, "scenario", scenario_name)
    oof_df.insert(1, "model", Config.MODEL_SHORT_LABELS[scenario_name])
    oof_df.insert(2, "description", Config.MODEL_DESCRIPTIONS[scenario_name])
    oof_df["oof_probability"] = oof_probability
    oof_df["oof_predicted_class_0p5"] = oof_predicted

    fold_df = pd.DataFrame(fold_rows)
    search_df = pd.concat(search_tables, ignore_index=True)

    return oof_df, fold_df, best_params, search_df




def build_fixed_ae_oof_ensemble(
    all_oof: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Build and evaluate the fixed Model A / Model E OOF ensemble.

    This applies the same fixed A30/E70 operational blending principle used in Step 7.1:
        P_ensemble = 0.30 * P_static + 0.70 * P_enhanced

    Because the weight is fixed before inspecting the OOF ensemble result,
    the resulting OOF metric can be reported as a separate, pre-specified
    ensemble diagnostic. It does not replace the primary A-F ablation results.
    """
    static_name = Config.ENSEMBLE_STATIC_SCENARIO
    enhanced_name = Config.ENSEMBLE_ENHANCED_SCENARIO
    enhanced_weight = float(Config.ENSEMBLE_ENHANCED_WEIGHT)
    static_weight = 1.0 - enhanced_weight

    static_oof = all_oof.loc[
        all_oof["scenario"] == static_name,
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
            "oof_probability",
        ],
    ].copy()
    static_oof = static_oof.rename(
        columns={"oof_probability": "probability_static_A"}
    )

    enhanced_oof = all_oof.loc[
        all_oof["scenario"] == enhanced_name,
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
            "oof_probability",
        ],
    ].copy()
    enhanced_oof = enhanced_oof.rename(
        columns={"oof_probability": "probability_enhanced_E"}
    )

    merged = static_oof.merge(
        enhanced_oof,
        on=Config.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=("_A", "_E"),
        validate="one_to_one",
    )

    expected_rows = Config.EXPECTED_N_ROWS
    if len(merged) != expected_rows:
        raise RuntimeError(
            f"A/E ensemble OOF merge returned {len(merged)} rows; "
            f"expected {expected_rows}."
        )

    for column_name in (
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ):
        left = merged[f"{column_name}_A"].to_numpy()
        right = merged[f"{column_name}_E"].to_numpy()
        if not np.array_equal(left, right):
            raise RuntimeError(
                f"A/E ensemble OOF mismatch in {column_name}."
            )

    merged[Config.TARGET_COLUMN] = merged[
        f"{Config.TARGET_COLUMN}_A"
    ].astype(int)
    merged[Config.FOLD_COLUMN] = merged[
        f"{Config.FOLD_COLUMN}_A"
    ].astype(int)
    merged[Config.BLOCK_COLUMN] = merged[
        f"{Config.BLOCK_COLUMN}_A"
    ].astype(int)

    merged["oof_probability_ensemble"] = (
        static_weight * merged["probability_static_A"].to_numpy(dtype=float)
        + enhanced_weight * merged["probability_enhanced_E"].to_numpy(dtype=float)
    )

    merged["oof_predicted_class_0p5"] = (
        merged["oof_probability_ensemble"].to_numpy(dtype=float)
        >= Config.BINARY_THRESHOLD
    ).astype(int)

    y_true = merged[Config.TARGET_COLUMN].to_numpy(dtype=int)
    y_prob = merged["oof_probability_ensemble"].to_numpy(dtype=float)
    blocks = merged[Config.BLOCK_COLUMN].to_numpy(dtype=int)

    metrics = classification_metrics(
        y_true,
        y_prob,
        threshold=Config.BINARY_THRESHOLD,
    )
    ci_low, ci_high, n_valid_bootstrap = cluster_bootstrap_auc_ci(
        y_true=y_true,
        y_prob=y_prob,
        blocks=blocks,
        n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
        seed=Config.RANDOM_STATE + 9001,
        ci_level=Config.CI_LEVEL,
    )

    # Paired comparison against the primary Model E OOF prediction.
    observed_difference, diff_low, diff_high, n_valid_diff = (
        paired_cluster_bootstrap_auc_difference(
            y_true=y_true,
            prob_left=y_prob,
            prob_right=merged["probability_enhanced_E"].to_numpy(dtype=float),
            blocks=blocks,
            n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
            seed=Config.RANDOM_STATE + 9002,
            ci_level=Config.CI_LEVEL,
        )
    )

    metrics_summary = {
        "ensemble_name": "LR A30-E70 fixed ensemble",
        "formula": (
            f"{static_weight:.2f} * Model A + "
            f"{enhanced_weight:.2f} * Model E"
        ),
        "weight_selection": (
            "Fixed a priori to mirror the previous static/integrated "
            "Logistic Regression ensemble; not optimized on OOF predictions."
        ),
        **metrics,
        "auc_cluster_bootstrap_ci_low": float(ci_low),
        "auc_cluster_bootstrap_ci_high": float(ci_high),
        "auc_cluster_bootstrap_valid_iterations": int(n_valid_bootstrap),
        "auc_difference_ensemble_minus_model_E": float(observed_difference),
        "auc_difference_ci_low": float(diff_low),
        "auc_difference_ci_high": float(diff_high),
        "auc_difference_valid_iterations": int(n_valid_diff),
    }

    output_columns = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
        "probability_static_A",
        "probability_enhanced_E",
        "oof_probability_ensemble",
        "oof_predicted_class_0p5",
    ]
    output_df = merged[output_columns].copy()

    output_df.to_csv(
        Config.ENSEMBLE_OOF_PREDICTIONS_CSV,
        index=False,
        float_format="%.10f",
    )
    with open(
        Config.ENSEMBLE_OOF_METRICS_JSON,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(metrics_summary),
            handle,
            indent=2,
        )

    log(
        "[ENSEMBLE-OOF] A30/E70 fixed ensemble: "
        f"AUC={metrics_summary['roc_auc']:.4f} "
        f"[{ci_low:.4f}, {ci_high:.4f}], "
        f"ΔAUC vs Model E={observed_difference:+.4f} "
        f"[{diff_low:+.4f}, {diff_high:+.4f}]"
    )

    if observed_difference < 0:
        log(
            "[ENSEMBLE-WARN] The fixed A30/E70 ensemble has lower pooled "
            "OOF AUC than Model E. The ensemble map can still be generated "
            "for comparison, but Model E should remain the primary scientific "
            "map unless there is another justified reason to prefer the ensemble."
        )

    return output_df, metrics_summary


# =============================================================================
# FINAL FULL-DATA MODEL FOR MAPPING / DEPLOYMENT
# =============================================================================
def tune_and_fit_final_model(
    df: pd.DataFrame,
    scenario_name: str,
    features: Sequence[str],
) -> Tuple[Pipeline, Dict[str, Any], pd.DataFrame]:
    """
    Tune on all 330 samples using the frozen five spatial folds, then refit on
    all samples. This model is for raster prediction only, not performance claims.
    """
    X = df[list(features)]
    y = df[Config.TARGET_COLUMN].to_numpy(dtype=int)

    pipeline = build_lr_pipeline(features)
    spatial_cv = make_full_data_predefined_split(df)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=Config.PARAM_GRID,
        scoring="roc_auc",
        cv=spatial_cv,
        refit=True,
        n_jobs=Config.GRIDSEARCH_N_JOBS,
        verbose=Config.GRIDSEARCH_VERBOSE,
        return_train_score=False,
        error_score="raise",
    )

    log(
        f"[FINAL-MODEL] {Config.MODEL_SHORT_LABELS[scenario_name]}: "
        "full-data spatial grid search for deployment model."
    )
    search.fit(X, y)

    model_path = Config.MODEL_DIR / f"{scenario_prefix(scenario_name)}_LR.joblib"
    joblib.dump(search.best_estimator_, model_path)

    info = {
        "best_mean_spatial_cv_auc_for_model_selection_only": float(
            search.best_score_
        ),
        "best_params": json_safe(search.best_params_),
        "model_path": str(model_path),
        "note": (
            "This full-data tuning score is not used as the reported validation "
            "performance; reported performance comes only from nested outer OOF predictions."
        ),
    }

    search_df = normalize_grid_results(
        search.cv_results_,
        scenario_name=scenario_name,
        search_stage="full_data_deployment_tuning",
        outer_fold=None,
    )

    return search.best_estimator_, info, search_df


# =============================================================================
# RASTER PREDICTION
# =============================================================================
def read_stack_band_map(stack_path: Path) -> Dict[str, int]:
    require_file(stack_path, "Step 6.1 master predictor stack")
    with rasterio.open(stack_path) as dataset:
        descriptions = list(dataset.descriptions)

    band_map: Dict[str, int] = {}
    for band_index, description in enumerate(descriptions, start=1):
        if description:
            band_map[str(description)] = band_index

    return band_map


def predict_scenario_raster(
    fitted_pipeline: Pipeline,
    scenario_name: str,
    features: Sequence[str],
) -> Path:
    """
    Predict one susceptibility raster on the common A–F valid domain so all six
    maps have an identical comparison footprint.
    """
    require_file(Config.MASTER_STACK_TIF, "Step 6.1 master predictor stack")
    require_file(Config.COMMON_AF_MASK_TIF, "common A–F valid mask")

    band_map = read_stack_band_map(Config.MASTER_STACK_TIF)
    missing_bands = [feature for feature in features if feature not in band_map]
    if missing_bands:
        raise RuntimeError(
            f"Master stack is missing bands required by {scenario_name}: {missing_bands}"
        )

    band_indexes = [band_map[feature] for feature in features]
    output_path = (
        Config.MAP_DIR
        / f"{scenario_prefix(scenario_name)}_LR_probability_0_1_common_AF_domain.tif"
    )

    with rasterio.open(Config.MASTER_STACK_TIF) as stack_ds, rasterio.open(
        Config.COMMON_AF_MASK_TIF
    ) as mask_ds:
        if (
            stack_ds.width != mask_ds.width
            or stack_ds.height != mask_ds.height
            or stack_ds.transform != mask_ds.transform
            or stack_ds.crs != mask_ds.crs
        ):
            raise RuntimeError("Master stack and common A–F mask are not aligned.")

        profile = stack_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(temporary_path, "w", **profile) as output_ds:
            for row_start in range(0, stack_ds.height, Config.RASTER_BLOCK_ROWS):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    stack_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=stack_ds.width,
                    height=block_height,
                )

                data = stack_ds.read(band_indexes, window=window, masked=True)
                common_mask = mask_ds.read(1, window=window, masked=True)

                output = np.full(
                    (block_height, stack_ds.width),
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                values = np.asarray(data.filled(np.nan), dtype="float64")
                valid = (
                    np.asarray(common_mask.filled(0)) > 0
                ) & np.all(np.isfinite(values), axis=0)

                if np.any(valid):
                    pixel_matrix = values[:, valid].T
                    pixel_df = pd.DataFrame(pixel_matrix, columns=list(features))
                    probabilities = fitted_pipeline.predict_proba(pixel_df)[:, 1]
                    output[valid] = probabilities.astype("float32")

                output_ds.write(output, 1, window=window)

            output_ds.set_band_description(
                1,
                f"{Config.MODEL_SHORT_LABELS[scenario_name]} LR susceptibility probability",
            )
            output_ds.update_tags(
                scenario=scenario_name,
                model="Logistic Regression",
                prediction_domain="common_A_to_F_valid_domain",
                validation_note=(
                    "Raster generated from final full-data deployment model; "
                    "validation metrics come from nested spatial OOF predictions."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(temporary_path, output_path)
    log(f"[MAP] Saved: {output_path}")
    return output_path






def predict_scenario_raster_on_own_feature_domain(
    fitted_pipeline: Pipeline,
    scenario_name: str,
    features: Sequence[str],
    output_path: Path,
) -> Path:
    """
    Predict a scenario wherever all predictors required by that scenario are
    finite in the study-domain master stack.

    Unlike the A-F comparison maps, this function does NOT restrict prediction
    to the common A-F mask. This is required to reproduce the earlier ensemble
    logic, where the static model supplies the broad susceptibility background
    and the enhanced model is blended only where its additional predictors exist.
    """
    require_file(
        Config.MASTER_STACK_TIF,
        "Step 6.1 master predictor stack",
    )

    band_map = read_stack_band_map(Config.MASTER_STACK_TIF)
    missing_bands = [
        feature
        for feature in features
        if feature not in band_map
    ]
    if missing_bands:
        raise RuntimeError(
            f"Master stack is missing bands required by "
            f"{scenario_name}: {missing_bands}"
        )

    band_indexes = [
        band_map[feature]
        for feature in features
    ]

    with rasterio.open(Config.MASTER_STACK_TIF) as stack_ds:
        profile = stack_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        valid_pixel_count = 0

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_ds:
            for row_start in range(
                0,
                stack_ds.height,
                Config.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    stack_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=stack_ds.width,
                    height=block_height,
                )

                data = stack_ds.read(
                    band_indexes,
                    window=window,
                    masked=True,
                )

                output = np.full(
                    (block_height, stack_ds.width),
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                values = np.asarray(
                    data.filled(np.nan),
                    dtype="float64",
                )
                valid = np.all(
                    np.isfinite(values),
                    axis=0,
                )

                if np.any(valid):
                    pixel_matrix = values[:, valid].T
                    pixel_df = pd.DataFrame(
                        pixel_matrix,
                        columns=list(features),
                    )
                    probabilities = (
                        fitted_pipeline.predict_proba(pixel_df)[:, 1]
                    )
                    output[valid] = probabilities.astype("float32")
                    valid_pixel_count += int(valid.sum())

                output_ds.write(
                    output,
                    1,
                    window=window,
                )

            output_ds.set_band_description(
                1,
                (
                    f"{Config.MODEL_SHORT_LABELS[scenario_name]} "
                    "LR susceptibility probability"
                ),
            )
            output_ds.update_tags(
                scenario=scenario_name,
                model="Logistic Regression",
                prediction_domain="scenario_specific_valid_feature_domain",
                validation_note=(
                    "Raster generated from the final full-data deployment model. "
                    "Reported validation metrics come from nested spatial OOF "
                    "predictions, not from this raster."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        f"[MAP-FULL-DOMAIN] {Config.MODEL_SHORT_LABELS[scenario_name]}: "
        f"{valid_pixel_count:,} valid pixels -> {output_path}"
    )

    return output_path


def blend_static_and_enhanced_probability_rasters(
    static_raster_path: Path,
    enhanced_raster_path: Path,
    output_path: Path,
    enhanced_weight: float,
) -> Path:
    """
    Apply the same fixed A30/E70 operational ensemble logic to the Logistic Regression A-F framework.

    Where both models are valid:
        ensemble = (1-w) * Model A + w * Model E

    Where only Model A is valid:
        retain Model A as the susceptibility background.

    Where only Model E is valid:
        retain Model E.

    Outside both valid domains:
        nodata.

    This is intentionally different from averaging all A-F configurations.
    Models B-F are ablation experiments and should not be indiscriminately
    averaged into the final susceptibility map.
    """
    if not 0.0 <= enhanced_weight <= 1.0:
        raise ValueError(
            "ENSEMBLE_ENHANCED_WEIGHT must lie between 0 and 1."
        )

    static_weight = 1.0 - enhanced_weight

    require_file(
        static_raster_path,
        "Model A full-domain probability raster",
    )
    require_file(
        enhanced_raster_path,
        "Model E full-domain probability raster",
    )

    with rasterio.open(static_raster_path) as static_ds, rasterio.open(
        enhanced_raster_path
    ) as enhanced_ds:
        aligned = (
            static_ds.width == enhanced_ds.width
            and static_ds.height == enhanced_ds.height
            and static_ds.transform == enhanced_ds.transform
            and static_ds.crs == enhanced_ds.crs
        )
        if not aligned:
            raise RuntimeError(
                "Model A and Model E full-domain rasters are not aligned."
            )

        profile = static_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        n_static_background = 0
        n_blended = 0
        n_enhanced_only = 0

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_ds:
            for row_start in range(
                0,
                static_ds.height,
                Config.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    static_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=static_ds.width,
                    height=block_height,
                )

                static_arr = static_ds.read(
                    1,
                    window=window,
                    masked=True,
                )
                enhanced_arr = enhanced_ds.read(
                    1,
                    window=window,
                    masked=True,
                )

                static_values = np.asarray(
                    static_arr.filled(np.nan),
                    dtype="float64",
                )
                enhanced_values = np.asarray(
                    enhanced_arr.filled(np.nan),
                    dtype="float64",
                )

                static_valid = np.isfinite(static_values)
                enhanced_valid = np.isfinite(enhanced_values)

                both = static_valid & enhanced_valid
                static_only = static_valid & ~enhanced_valid
                enhanced_only = enhanced_valid & ~static_valid

                output = np.full(
                    static_values.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                output[static_only] = static_values[
                    static_only
                ].astype("float32")

                output[enhanced_only] = enhanced_values[
                    enhanced_only
                ].astype("float32")

                output[both] = (
                    static_weight * static_values[both]
                    + enhanced_weight * enhanced_values[both]
                ).astype("float32")

                n_static_background += int(static_only.sum())
                n_blended += int(both.sum())
                n_enhanced_only += int(enhanced_only.sum())

                output_ds.write(
                    output,
                    1,
                    window=window,
                )

            output_ds.set_band_description(
                1,
                "Logistic Regression A30-E70 ensemble susceptibility probability",
            )
            output_ds.update_tags(
                model="Logistic Regression",
                ensemble="Model_A_static_plus_Model_E_physics_guided",
                static_weight=f"{static_weight:.6f}",
                enhanced_weight=f"{enhanced_weight:.6f}",
                blend_rule=(
                    "weighted blend where both are valid; "
                    "retain static background elsewhere"
                ),
                validation_note=(
                    "The fixed A/E ensemble is separately evaluated from nested "
                    "OOF probabilities. A-F ablation metrics remain the primary "
                    "model-comparison results."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[ENSEMBLE-MAP] Saved A30/E70 ensemble: "
        f"blended={n_blended:,}, "
        f"static-background={n_static_background:,}, "
        f"enhanced-only={n_enhanced_only:,} -> "
        f"{output_path}"
    )

    return output_path


def create_revised_ae_ensemble_probability_map(
    final_models: dict[str, Pipeline],
    scenarios: dict[str, dict[str, Any]],
) -> Path:
    """
    Generate the revised equivalent of the previous LR blended ensemble map.

    Model A supplies the broad static susceptibility background.
    Model E contributes the deformation-informed physics-guided prediction
    wherever HKFI and SNDR are available.
    """
    static_scenario = Config.ENSEMBLE_STATIC_SCENARIO
    enhanced_scenario = Config.ENSEMBLE_ENHANCED_SCENARIO

    for scenario_name in (
        static_scenario,
        enhanced_scenario,
    ):
        if scenario_name not in final_models:
            raise RuntimeError(
                f"Final deployment model unavailable for {scenario_name}."
            )
        if scenario_name not in scenarios:
            raise RuntimeError(
                f"Scenario definition unavailable for {scenario_name}."
            )

    predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=final_models[static_scenario],
        scenario_name=static_scenario,
        features=list(scenarios[static_scenario]["features"]),
        output_path=Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
    )

    predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=final_models[enhanced_scenario],
        scenario_name=enhanced_scenario,
        features=list(scenarios[enhanced_scenario]["features"]),
        output_path=Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
    )

    return blend_static_and_enhanced_probability_rasters(
        static_raster_path=Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
        enhanced_raster_path=Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
        output_path=Config.ENSEMBLE_PROBABILITY_TIF,
        enhanced_weight=float(Config.ENSEMBLE_ENHANCED_WEIGHT),
    )


# =============================================================================
# STEP 5.3-STYLE LANDSLIDE SUSCEPTIBILITY VISUALIZATION
# =============================================================================

LSM_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "lr_landslide_susceptibility",
    [
        "#006400",
        "#32CD32",
        "#FFFF00",
        "#FFA500",
        "#FF4500",
        "#8B0000",
    ],
    N=256,
)



SUSCEPTIBILITY_CLASSES = {
    "Very Low": (0.0, 0.2),
    "Low": (0.2, 0.4),
    "Moderate": (0.4, 0.6),
    "High": (0.6, 0.8),
    "Very High": (0.8, 1.0),
}

SUSCEPTIBILITY_CLASS_COLORS = {
    "Very Low": "#006400",
    "Low": "#32CD32",
    "Moderate": "#FFFF00",
    "High": "#FFA500",
    "Very High": "#8B0000",
}


def add_old_style_susceptibility_legend(
    axis,
    include_road: bool = True,
) -> None:
    """Reproduce the earlier LSM legend layout."""
    legend_elements: list[Line2D] = []

    if include_road:
        legend_elements.append(
            Line2D(
                [0],
                [0],
                color="red",
                linewidth=1.5,
                label="Highway/Roads",
            )
        )

    for class_name in (
        "Very High",
        "High",
        "Moderate",
        "Low",
        "Very Low",
    ):
        legend_elements.append(
            Line2D(
                [0],
                [0],
                marker="s",
                color="white",
                markerfacecolor=(
                    SUSCEPTIBILITY_CLASS_COLORS[class_name]
                ),
                markersize=7,
                markeredgecolor="black",
                markeredgewidth=0.5,
                label=class_name,
            )
        )

    legend = axis.legend(
        handles=legend_elements,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.50), # # Changed here to move the Map Up
        fontsize=Config.LEGEND_FONTSIZE,
        title="Legend",
        title_fontsize=Config.LEGEND_TITLE_FONTSIZE,
        frameon=True,
        fancybox=True,
        shadow=True,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0,
    )
    legend.get_title().set_fontweight("bold")
    legend.set_zorder(2000)


def reproject_probability_raster_to_epsg4326(
    source_path: Path,
    output_path: Path,
) -> Path:
    """
    Reproject only for cartographic display, matching the earlier LSM workflow.

    The source analytical probability raster is never modified.
    Bilinear resampling is used only for display reprojection of the continuous
    probability surface.
    """
    require_file(
        source_path,
        "susceptibility probability raster for visualization",
    )

    with rasterio.open(source_path) as source:
        if source.crs is None:
            raise RuntimeError(
                f"Probability raster has no CRS: {source_path}"
            )

        if source.crs.to_epsg() == 4326:
            return source_path

        destination_crs = "EPSG:4326"
        destination_transform, width, height = (
            calculate_default_transform(
                source.crs,
                destination_crs,
                source.width,
                source.height,
                *source.bounds,
            )
        )

        profile = source.profile.copy()
        profile.update(
            crs=destination_crs,
            transform=destination_transform,
            width=width,
            height=height,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as destination:
            reproject(
                source=rasterio.band(source, 1),
                destination=rasterio.band(destination, 1),
                src_transform=source.transform,
                src_crs=source.crs,
                src_nodata=source.nodata,
                dst_transform=destination_transform,
                dst_crs=destination_crs,
                dst_nodata=Config.OUTPUT_NODATA,
                resampling=Resampling.bilinear,
            )

            destination.set_band_description(
                1,
                "Logistic Regression ensemble susceptibility probability — EPSG:4326 visualization copy",
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[VIZ-REPROJECT] Display raster reprojected to EPSG:4326: "
        f"{output_path}"
    )
    return output_path


def ensemble_visualization_output_paths() -> dict[str, Path]:
    stem = (
        Config.VISUALIZATION_DIR
        / "LR_Susceptibility_Map_Ensemble_A30_E70"
    )
    return {
        "map_png": Path(str(stem) + ".png"),
        "map_pdf": Path(str(stem) + ".pdf"),
        "map_svg": Path(str(stem) + ".svg"),
        "histogram_png": Path(
            str(Config.VISUALIZATION_DIR / "LR_Ensemble_Histogram_A30_E70")
            + ".png"
        ),
        "histogram_pdf": Path(
            str(Config.VISUALIZATION_DIR / "LR_Ensemble_Histogram_A30_E70")
            + ".pdf"
        ),
        "histogram_svg": Path(
            str(Config.VISUALIZATION_DIR / "LR_Ensemble_Histogram_A30_E70")
            + ".svg"
        ),
        "summary_json": (
            Config.VISUALIZATION_DIR
            / "LR_Ensemble_A30_E70_Visualization_Summary.json"
        ),
        "statistics_csv": (
            Config.VISUALIZATION_DIR
            / "LR_Ensemble_A30_E70_Statistics.csv"
        ),
        "area_percentage_png": (
            Config.VISUALIZATION_DIR
            / "LR_Ensemble_A30_E70_Area_Percentage.png"
        ),
        "area_percentage_pdf": (
            Config.VISUALIZATION_DIR
            / "LR_Ensemble_A30_E70_Area_Percentage.pdf"
        ),
        "area_percentage_svg": (
            Config.VISUALIZATION_DIR
            / "LR_Ensemble_A30_E70_Area_Percentage.svg"
        ),
        "area_percentage_csv": (
            Config.VISUALIZATION_DIR
            / "LR_Ensemble_A30_E70_Area_Percentage.csv"
        ),
        "support_map_png": (
            Config.VISUALIZATION_DIR
            / "LR_Ensemble_A30_E70_Information_Support_Map.png"
        ),
        "support_map_pdf": (
            Config.VISUALIZATION_DIR
            / "LR_Ensemble_A30_E70_Information_Support_Map.pdf"
        ),
        "support_map_svg": (
            Config.VISUALIZATION_DIR
            / "LR_Ensemble_A30_E70_Information_Support_Map.svg"
        ),
        "display_raster_epsg4326": (
            Config.VISUALIZATION_DIR
            / "LR_Ensemble_A30_E70_Susceptibility_EPSG4326.tif"
        ),
    }


def create_old_style_ensemble_lsm_map(
    probability_raster_path: Path,
    outputs: dict[str, Path],
) -> tuple[np.ndarray, dict[str, Any]]:
    """
    Draw the ensemble LSM using the earlier successful LR map approach.

    Key elements reproduced:
      - static susceptibility background retained across the broader study domain;
      - EPSG:4326 display raster;
      - exact raster extent (no added map buffer);
      - white opaque map background;
      - vector clip path to the study boundary;
      - thick black study boundary;
      - red highway;
      - north arrow;
      - checker scale bar;
      - susceptibility-class legend plus continuous colour bar.
    """
    display_raster_path = reproject_probability_raster_to_epsg4326(
        source_path=probability_raster_path,
        output_path=outputs["display_raster_epsg4326"],
    )

    probability, reference = read_probability_raster(
        display_raster_path
    )
    values = probability[np.isfinite(probability)]

    bounds = reference["bounds"]
    extent = (
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    )

    display_array = np.ma.masked_invalid(
        probability
    )

    # Opaque white for any nodata cells that remain.
    map_cmap = LSM_CMAP.copy()
    map_cmap.set_bad(
        color="white",
        alpha=1.0,
    )

    study_boundary = load_vector_overlay(
        Config.STUDY_SHP,
        "EPSG:4326",
    )
    roads = load_vector_overlay(
        Config.ROAD_SHP,
        "EPSG:4326",
    )

    figure, axis = plt.subplots(
        1,
        1,
        figsize=Config.ENSEMBLE_MAP_FIGSIZE,
        facecolor="white",
    )
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        vmin=0.0,
        vmax=1.0,
        cmap=map_cmap,
        interpolation="nearest",
        alpha=1.0,
        zorder=1,
    )

    # The raster is clipped instead of relying on alpha transparency.
    clip_patch = build_clip_patch_from_vector_overlay(
        study_boundary,
        axis,
    )
    if clip_patch is not None:
        image.set_clip_path(
            clip_patch
        )
        log(
            "[ENSEMBLE-VIZ] Raster clipped to study boundary; "
            "opaque white background retained."
        )

    # Match the visual hierarchy of the earlier map.
    plot_vector_overlay(
        axis,
        study_boundary,
        colour="black",
        linewidth=2.0,
        zorder=1000,
        polygon_boundary_only=True,
    )
    plot_vector_overlay(
        axis,
        roads,
        colour="red",
        linewidth=1.0,
        zorder=1100,
        polygon_boundary_only=False,
    )

    axis.set_xlim(
        extent[0],
        extent[1],
    )
    axis.set_ylim(
        extent[2],
        extent[3],
    )

    axis.xaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: to_dms(
                value,
                True,
            )
        )
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: to_dms(
                value,
                False,
            )
        )
    )
    axis.tick_params(
        axis="both",
        which="major",
        labelsize=Config.DMS_TICK_FONTSIZE,
    )

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")

    for label in axis.get_yticklabels():
        label.set_rotation(90)
        label.set_va("center")

    add_north_arrow_tri(
        axis,
        size=0.08,
        pad=0.06,
    )

    add_checker_scale_bar(
        axis,
        extent[0],
        extent[1],
        extent[2],
        extent[3],
        crs_type="wgs84",
        mid_lat_deg=(
            0.5 * (extent[2] + extent[3])
        ),
        segments=4,
        fontsize=6,
    )

    add_old_style_susceptibility_legend(
        axis,
        include_road=True,
    )

    divider = make_axes_locatable(
        axis
    )
    colorbar_axis = divider.append_axes(
        "right",
        size=Config.CAX_SIZE,
        pad=Config.CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(
        labelsize=Config.COLORBAR_TICK_FONTSIZE,
    )
    colorbar.set_label(
        "Susceptibility (0–1)",
        fontsize=Config.COLORBAR_LABEL_FONTSIZE,
        fontweight="bold",
    )

    figure.tight_layout()

    for output_key, output_format in (
        ("map_png", None),
        ("map_pdf", None),
        ("map_svg", "svg"),
    ):
        save_kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
            "edgecolor": "white",
            "transparent": False,
        }
        if output_key == "map_png":
            save_kwargs["dpi"] = Config.FIG_DPI
        if output_format is not None:
            save_kwargs["format"] = output_format

        figure.savefig(
            outputs[output_key],
            **save_kwargs,
        )

    log(
        f"[ENSEMBLE-VIZ] Map saved: {outputs['map_png']}"
    )

    if Config.SHOW_LSM_FIGURES:
        plt.show()

    plt.close(
        figure
    )

    return values, reference



def classify_susceptibility_probability(values: np.ndarray) -> np.ndarray:
    """Classify continuous probability into five equal-interval classes."""
    classes = np.zeros(values.shape, dtype=np.uint8)
    finite = np.isfinite(values)
    classes[finite & (values >= 0.0) & (values < 0.2)] = 1
    classes[finite & (values >= 0.2) & (values < 0.4)] = 2
    classes[finite & (values >= 0.4) & (values < 0.6)] = 3
    classes[finite & (values >= 0.6) & (values < 0.8)] = 4
    classes[finite & (values >= 0.8) & (values <= 1.0)] = 5
    return classes


def calculate_susceptibility_area_percentages(
    probability_raster_path: Path,
) -> pd.DataFrame:
    """
    Calculate mapped area percentage for each susceptibility class on the
    original projected analytical grid. Pixel fractions therefore equal area
    fractions because cell area is constant.
    """
    require_file(probability_raster_path, "ensemble susceptibility raster")

    with rasterio.open(probability_raster_path) as dataset:
        probability = clean_probability_array(
            dataset.read(1),
            dataset.nodata,
        )

    valid = probability[np.isfinite(probability)]
    classes = classify_susceptibility_probability(valid)
    total = int(valid.size)

    definitions = [
        (5, "Very High", 0.8, 1.0),
        (4, "High", 0.6, 0.8),
        (3, "Moderate", 0.4, 0.6),
        (2, "Low", 0.2, 0.4),
        (1, "Very Low", 0.0, 0.2),
    ]

    rows = []
    for class_id, label, lower, upper in definitions:
        count = int(np.sum(classes == class_id))
        percentage = 100.0 * count / total if total > 0 else np.nan
        rows.append(
            {
                "class_id": class_id,
                "class_label": label,
                "lower_probability": lower,
                "upper_probability": upper,
                "pixel_count": count,
                "area_percentage": percentage,
            }
        )

    return pd.DataFrame(rows)


def create_susceptibility_area_percentage_plot(
    area_df: pd.DataFrame,
    outputs: dict[str, Path],
) -> None:
    """Create a separate 4 x 4 bar plot of susceptibility-class area percentages."""
    labels = area_df["class_label"].tolist()
    percentages = area_df["area_percentage"].to_numpy(dtype=float)
    colours = [SUSCEPTIBILITY_CLASS_COLORS[label] for label in labels]

    figure, axis = plt.subplots(
        1,
        1,
        figsize=Config.AREA_PERCENT_FIGSIZE,
        facecolor="white",
    )
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    bars = axis.bar(
        labels,
        percentages,
        color=colours,
        edgecolor="black",
        linewidth=0.8,
    )

    upper_limit = max(5.0, float(np.nanmax(percentages)) * 1.18)
    axis.set_ylim(0.0, upper_limit)

    for bar, percentage in zip(bars, percentages):
        axis.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + upper_limit * 0.015,
            f"{percentage:.2f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
        )

    axis.set_xlabel("Susceptibility Class", fontsize=10, fontweight="bold")
    axis.set_ylabel("Area (%)", fontsize=10, fontweight="bold")
    axis.tick_params(axis="x", labelsize=8, rotation=20)
    axis.tick_params(axis="y", labelsize=8)
    axis.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.35)
    axis.set_axisbelow(True)

    for spine in axis.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("black")

    figure.tight_layout()

    for output_key, output_format in (
        ("area_percentage_png", None),
        ("area_percentage_pdf", None),
        ("area_percentage_svg", "svg"),
    ):
        kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
            "edgecolor": "white",
            "transparent": False,
        }
        if output_key == "area_percentage_png":
            kwargs["dpi"] = Config.FIG_DPI
        if output_format is not None:
            kwargs["format"] = output_format
        figure.savefig(outputs[output_key], **kwargs)

    if Config.SHOW_LSM_FIGURES:
        plt.show()
    plt.close(figure)


def write_information_support_raster(
    static_raster_path: Path,
    enhanced_raster_path: Path,
    output_path: Path,
) -> Path:
    """
    Write information-support classes:
      1 = Static-only prediction (Model A)
      2 = Static + deformation-informed prediction (Model E support present)
    """
    require_file(static_raster_path, "Model A full-domain probability raster")
    require_file(enhanced_raster_path, "Model E full-domain probability raster")

    with rasterio.open(static_raster_path) as static_ds, rasterio.open(enhanced_raster_path) as enhanced_ds:
        if not (
            static_ds.width == enhanced_ds.width
            and static_ds.height == enhanced_ds.height
            and static_ds.transform == enhanced_ds.transform
            and static_ds.crs == enhanced_ds.crs
        ):
            raise RuntimeError("Model A and Model E rasters are not aligned.")

        profile = static_ds.profile.copy()
        profile.update(driver="GTiff", count=1, dtype="uint8", nodata=0, compress="LZW", tiled=True)

        with rasterio.open(output_path, "w", **profile) as output_ds:
            for row_start in range(0, static_ds.height, Config.RASTER_BLOCK_ROWS):
                block_height = min(Config.RASTER_BLOCK_ROWS, static_ds.height - row_start)
                window = Window(0, row_start, static_ds.width, block_height)

                static_values = np.asarray(
                    static_ds.read(1, window=window, masked=True).filled(np.nan),
                    dtype=float,
                )
                enhanced_values = np.asarray(
                    enhanced_ds.read(1, window=window, masked=True).filled(np.nan),
                    dtype=float,
                )

                static_valid = np.isfinite(static_values)
                enhanced_valid = np.isfinite(enhanced_values)

                support = np.zeros(static_values.shape, dtype=np.uint8)
                support[static_valid & ~enhanced_valid] = 1
                support[enhanced_valid] = 2
                output_ds.write(support, 1, window=window)

            output_ds.set_band_description(1, "Logistic Regression A/E ensemble information support")
            output_ds.update_tags(
                class_1="Static-only prediction (Model A)",
                class_2="Static + deformation-informed prediction (Model E support present)",
                generated_at_utc=utc_now_iso(),
            )

    log(f"[SUPPORT-MAP] Saved: {output_path}")
    return output_path


def create_information_support_map(
    support_raster_path: Path,
    outputs: dict[str, Path],
) -> None:
    """Plot static-only versus deformation-informed support using the same map style."""
    display_path = Config.VISUALIZATION_DIR / "LR_Ensemble_A30_E70_Information_Support_EPSG4326.tif"

    with rasterio.open(support_raster_path) as source:
        if source.crs.to_epsg() == 4326:
            display_path = support_raster_path
        else:
            dst_crs = "EPSG:4326"
            dst_transform, width, height = calculate_default_transform(
                source.crs, dst_crs, source.width, source.height, *source.bounds
            )
            profile = source.profile.copy()
            profile.update(
                crs=dst_crs,
                transform=dst_transform,
                width=width,
                height=height,
                dtype="uint8",
                nodata=0,
                compress="LZW",
            )
            with rasterio.open(display_path, "w", **profile) as destination:
                reproject(
                    source=rasterio.band(source, 1),
                    destination=rasterio.band(destination, 1),
                    src_transform=source.transform,
                    src_crs=source.crs,
                    src_nodata=0,
                    dst_transform=dst_transform,
                    dst_crs=dst_crs,
                    dst_nodata=0,
                    resampling=Resampling.nearest,
                )

    with rasterio.open(display_path) as dataset:
        support = dataset.read(1)
        b = dataset.bounds
        extent = (b.left, b.right, b.bottom, b.top)

    display_array = np.ma.masked_where(support == 0, support)
    cmap = mcolors.ListedColormap(["#bdbdbd", "#2c7fb8"])
    cmap.set_bad(color="white", alpha=1.0)
    norm = mcolors.BoundaryNorm([0.5, 1.5, 2.5], cmap.N)

    study_boundary = load_vector_overlay(Config.STUDY_SHP, "EPSG:4326")
    roads = load_vector_overlay(Config.ROAD_SHP, "EPSG:4326")

    figure, axis = plt.subplots(1, 1, figsize=Config.SUPPORT_MAP_FIGSIZE, facecolor="white")
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
        alpha=1.0,
        zorder=1,
    )

    clip_patch = build_clip_patch_from_vector_overlay(study_boundary, axis)
    if clip_patch is not None:
        image.set_clip_path(clip_patch)

    plot_vector_overlay(axis, study_boundary, colour="black", linewidth=2.0, zorder=1000, polygon_boundary_only=True)
    plot_vector_overlay(axis, roads, colour="red", linewidth=1.0, zorder=1100, polygon_boundary_only=False)

    axis.set_xlim(extent[0], extent[1])
    axis.set_ylim(extent[2], extent[3])
    axis.xaxis.set_major_formatter(FuncFormatter(lambda value, position: to_dms(value, True)))
    axis.yaxis.set_major_formatter(FuncFormatter(lambda value, position: to_dms(value, False)))
    axis.tick_params(axis="both", which="major", labelsize=Config.DMS_TICK_FONTSIZE)

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")
    for label in axis.get_yticklabels():
        label.set_rotation(90)
        label.set_va("center")

    add_north_arrow_tri(axis, size=0.08, pad=0.06)
    add_checker_scale_bar(
        axis,
        extent[0], extent[1], extent[2], extent[3],
        crs_type="wgs84",
        mid_lat_deg=0.5 * (extent[2] + extent[3]),
        segments=4,
        fontsize=Config.SCALEBAR_FONTSIZE,
    )

    handles = [
        Line2D([0], [0], marker="s", color="white", markerfacecolor="#bdbdbd", markeredgecolor="black", markersize=8, label="Static-only prediction"),
        Line2D([0], [0], marker="s", color="white", markerfacecolor="#2c7fb8", markeredgecolor="black", markersize=8, label="Static + deformation-informed"),
        Line2D([0], [0], color="red", linewidth=1.5, label="Highway/Roads"),
    ]
    axis.legend(
        handles=handles,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.60), # Changed here to move the Map Up
        fontsize=Config.LEGEND_FONTSIZE,
        title="Information Support",
        title_fontsize=Config.LEGEND_TITLE_FONTSIZE,
        frameon=True,
        fancybox=True,
        shadow=True,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0,
    )

    figure.tight_layout()
    for output_key, output_format in (
        ("support_map_png", None),
        ("support_map_pdf", None),
        ("support_map_svg", "svg"),
    ):
        kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
            "edgecolor": "white",
            "transparent": False,
        }
        if output_key == "support_map_png":
            kwargs["dpi"] = Config.FIG_DPI
        if output_format is not None:
            kwargs["format"] = output_format
        figure.savefig(outputs[output_key], **kwargs)

    if Config.SHOW_LSM_FIGURES:
        plt.show()
    plt.close(figure)

def create_complete_ensemble_lsm_visualization(
    probability_raster_path: Path,
    ensemble_oof_metrics: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Create the old-style ensemble susceptibility map and histogram."""
    outputs = ensemble_visualization_output_paths()

    values, reference = create_old_style_ensemble_lsm_map(
        probability_raster_path=probability_raster_path,
        outputs=outputs,
    )

    statistics = calculate_lsm_statistics(
        values
    )

    # Five-class area-percentage analysis is calculated on the original
    # projected ensemble raster, not on the display-reprojected copy.
    area_percentage_df = calculate_susceptibility_area_percentages(
        probability_raster_path
    )
    area_percentage_df.to_csv(
        outputs["area_percentage_csv"],
        index=False,
        float_format="%.6f",
    )
    create_susceptibility_area_percentage_plot(
        area_df=area_percentage_df,
        outputs=outputs,
    )

    # Explicit information-support map for transparency.
    support_raster_path = write_information_support_raster(
        static_raster_path=Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
        enhanced_raster_path=Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
        output_path=Config.ENSEMBLE_SUPPORT_TIF,
    )
    create_information_support_map(
        support_raster_path=support_raster_path,
        outputs=outputs,
    )

    # Reuse the established histogram function with the ensemble values.
    create_lsm_probability_histogram(
        values=values,
        statistics=statistics,
        outputs=outputs,
    )

    pd.DataFrame(
        [
            {
                "map_source": "A30_E70_fixed_ensemble",
                "static_scenario": Config.ENSEMBLE_STATIC_SCENARIO,
                "enhanced_scenario": Config.ENSEMBLE_ENHANCED_SCENARIO,
                "static_weight": (
                    1.0 - Config.ENSEMBLE_ENHANCED_WEIGHT
                ),
                "enhanced_weight": Config.ENSEMBLE_ENHANCED_WEIGHT,
                **statistics,
            }
        ]
    ).to_csv(
        outputs["statistics_csv"],
        index=False,
    )

    summary = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "Logistic Regression",
        "map_source": "fixed_A30_E70_ensemble",
        "ensemble_formula": (
            f"{1.0 - Config.ENSEMBLE_ENHANCED_WEIGHT:.2f} * Model A + "
            f"{Config.ENSEMBLE_ENHANCED_WEIGHT:.2f} * Model E "
            "where Model E is available; otherwise retain Model A"
        ),
        "ensemble_weight_note": (
            "Weight fixed a priori to mirror the earlier context/integrated "
            "ensemble approach; not optimized on OOF predictions."
        ),
        "probability_raster": str(
            probability_raster_path
        ),
        "oof_ensemble_metrics": (
            ensemble_oof_metrics
            if ensemble_oof_metrics is not None
            else {}
        ),
        "visualization_style": (
            "Earlier LR ensemble LSM cartographic style with opaque white "
            "background, study-area vector clipping, enlarged DMS labels, "
            "five susceptibility classes and a separate class-area plot."
        ),
        "susceptibility_classes": area_percentage_df.to_dict(orient="records"),
        "information_support": {
            "class_1": "Static-only prediction (Model A)",
            "class_2": "Static + deformation-informed prediction (Model E support present)",
            "support_raster": str(support_raster_path),
        },
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }

    with open(
        outputs["summary_json"],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(summary),
            handle,
            indent=2,
        )

    return {
        "map_source": "fixed_A30_E70_ensemble",
        "probability_raster": str(
            probability_raster_path
        ),
        "statistics": statistics,
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }


def expected_scenario_map_path(scenario_name: str) -> Path:
    """Return the canonical raster path written by predict_scenario_raster()."""
    return (
        Config.MAP_DIR
        / f"{scenario_prefix(scenario_name)}_LR_probability_0_1_common_AF_domain.tif"
    )


def clean_probability_array(
    array: np.ndarray,
    nodata: float | int | None,
) -> np.ndarray:
    """
    Convert nodata to NaN and verify that susceptibility probabilities remain
    on [0,1]. Only tiny floating-point excursions are clipped.
    """
    output = np.asarray(array, dtype="float64").copy()

    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if np.isfinite(nodata_value):
                output[output == nodata_value] = np.nan
        except (TypeError, ValueError):
            pass

    output[output == -9999.0] = np.nan
    output[output == -32768.0] = np.nan
    output[np.abs(output) > 1.0e20] = np.nan

    outside = (
        np.isfinite(output)
        & ((output < -1.0e-6) | (output > 1.0 + 1.0e-6))
    )
    if outside.any():
        raise ValueError(
            f"Susceptibility raster contains {int(outside.sum()):,} values "
            "outside [0,1]. Recheck final probability prediction."
        )

    # Floating-point safeguard only; this is not renormalization.
    output = np.where(
        np.isfinite(output),
        np.clip(output, 0.0, 1.0),
        np.nan,
    )
    return output


def read_probability_raster(
    raster_path: Path,
) -> tuple[np.ndarray, dict[str, Any]]:
    require_file(raster_path, "Logistic Regression susceptibility probability raster")

    with rasterio.open(raster_path) as dataset:
        if dataset.crs is None:
            raise ValueError(
                f"Susceptibility raster has no CRS: {raster_path}"
            )

        probability = clean_probability_array(
            dataset.read(1),
            dataset.nodata,
        )

        reference = {
            "crs": dataset.crs,
            "transform": dataset.transform,
            "width": dataset.width,
            "height": dataset.height,
            "bounds": dataset.bounds,
            "epsg": dataset.crs.to_epsg(),
            "path": str(raster_path),
        }

    if not np.any(np.isfinite(probability)):
        raise RuntimeError(
            f"No valid probabilities found in susceptibility raster: {raster_path}"
        )

    return probability, reference


def to_dms(value: float, is_lon: bool = True) -> str:
    absolute = abs(value)
    degrees = int(absolute)
    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_lon
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def add_north_arrow_tri(
    axis,
    size: float = 0.09,
    pad: float = 0.07,
    label: str = "N",
    arrow_downshift: float = 0.025,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=Config.NORTH_ARROW_FONTSIZE,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base
        if candidate >= target_metres:
            return candidate

    return 10 * base


def add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    crs_type: str = "utm",
    mid_lat_deg: float | None = None,
    segments: int = 4,
    fontsize: float = 4,
    frame_lw: float = 0.4,
) -> None:
    width_data = east - west
    height_data = north - south

    if crs_type == "utm":
        width_metres = width_data
        data_units_per_metre = 1.0
    else:
        if mid_lat_deg is None:
            mid_lat_deg = 0.5 * (south + north)

        metres_per_degree = metres_per_degree_lon(mid_lat_deg)
        width_metres = width_data * metres_per_degree
        data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = total_length_metres * data_units_per_metre
    segment_length_units = segment_length_metres * data_units_per_metre

    x_start = west + 0.5 * width_data - 0.5 * total_length_units
    y_base = south + 0.06 * height_data
    bar_height = 0.018 * height_data

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_data

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (
                    x_start + index * segment_length_units,
                    y_base,
                ),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=(
                    "black" if index % 2 == 0 else "white"
                ),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_data,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_data,
        "0",
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_data,
        end_label,
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )


def load_vector_overlay(
    vector_path: Path,
    target_crs,
):
    """Load vector data and transform it to the raster CRS."""
    if not vector_path.exists():
        log(f"[VIZ-WARN] Vector overlay not found: {vector_path}")
        return None

    target_crs_text = str(target_crs)

    if HAVE_GPD:
        try:
            data = gpd.read_file(vector_path)
            if data.crs is None:
                log(
                    f"[VIZ-WARN] {vector_path.name} has no CRS; "
                    "assuming EPSG:4326."
                )
                data = data.set_crs(epsg=4326)
            return data.to_crs(target_crs_text)
        except Exception as exc:
            log(
                f"[VIZ-WARN] GeoPandas loading failed for "
                f"{vector_path.name}: {exc}"
            )

    if HAVE_FIONA:
        try:
            geometries = []
            with fiona.open(vector_path, "r") as source:
                source_crs = source.crs_wkt or source.crs
                if not source_crs:
                    source_crs = "EPSG:4326"
                    log(
                        f"[VIZ-WARN] {vector_path.name} has no CRS; "
                        "assuming EPSG:4326."
                    )

                for feature in source:
                    geometry = feature.get("geometry")
                    if geometry is None:
                        continue
                    geometries.append(
                        transform_geom(
                            source_crs,
                            target_crs_text,
                            geometry,
                        )
                    )

            return {"geometries": geometries}

        except Exception as exc:
            log(
                f"[VIZ-WARN] Fiona loading failed for "
                f"{vector_path.name}: {exc}"
            )

    return None


def plot_vector_overlay(
    axis,
    vector_data,
    colour: str,
    linewidth: float,
    zorder: int,
    polygon_boundary_only: bool = False,
) -> None:
    if vector_data is None:
        return

    if HAVE_GPD and hasattr(vector_data, "plot"):
        try:
            if polygon_boundary_only:
                vector_data.boundary.plot(
                    ax=axis,
                    color=colour,
                    linewidth=linewidth,
                    zorder=zorder,
                )
            else:
                vector_data.plot(
                    ax=axis,
                    color=colour,
                    linewidth=linewidth,
                    zorder=zorder,
                )
            return
        except Exception as exc:
            log(f"[VIZ-WARN] GeoPandas plotting failed: {exc}")

    if not isinstance(vector_data, dict):
        return

    for geometry in vector_data.get("geometries", []):
        if geometry is None:
            continue

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")
        if coordinates is None:
            continue

        if geometry_type == "LineString":
            line_groups = [coordinates]
        elif geometry_type == "MultiLineString":
            line_groups = coordinates
        elif geometry_type == "Polygon":
            line_groups = [coordinates[0]]
        elif geometry_type == "MultiPolygon":
            line_groups = [
                polygon[0]
                for polygon in coordinates
                if polygon
            ]
        else:
            continue

        for line in line_groups:
            xy = np.asarray(line, dtype="float64")
            if xy.ndim != 2 or xy.shape[1] < 2:
                continue
            axis.plot(
                xy[:, 0],
                xy[:, 1],
                color=colour,
                linewidth=linewidth,
                zorder=zorder,
            )



def build_clip_patch_from_vector_overlay(
    vector_data,
    axis,
):
    """
    Build a compound Matplotlib PathPatch from the study-area polygon.

    The raster artist is clipped to this polygon rather than relying on
    transparent pixels outside the study area. The axes and figure backgrounds
    remain opaque white, which is safer for PNG/PDF/SVG and later SVG-to-EMF
    conversion in Word.

    Supports:
      1. GeoPandas GeoDataFrame / GeoSeries input.
      2. Fiona fallback GeoJSON-style geometry dictionaries.

    Returns
    -------
    PathPatch or None
        Invisible patch suitable for image.set_clip_path().
    """
    if vector_data is None:
        return None

    all_vertices: list[np.ndarray] = []
    all_codes: list[int] = []

    def add_ring(ring_coordinates) -> None:
        coordinates = np.asarray(ring_coordinates, dtype="float64")

        if (
            coordinates.ndim != 2
            or coordinates.shape[0] < 3
            or coordinates.shape[1] < 2
        ):
            return

        coordinates = coordinates[:, :2]

        # Ensure the ring is explicitly closed.
        if not np.allclose(
            coordinates[0],
            coordinates[-1],
            equal_nan=False,
        ):
            coordinates = np.vstack(
                [coordinates, coordinates[0]]
            )

        n_vertices = len(coordinates)
        codes = (
            [MplPath.MOVETO]
            + [MplPath.LINETO] * (n_vertices - 2)
            + [MplPath.CLOSEPOLY]
        )

        all_vertices.append(coordinates)
        all_codes.extend(codes)

    def add_shapely_geometry(geometry) -> None:
        if geometry is None or getattr(geometry, "is_empty", True):
            return

        geometry_type = geometry.geom_type

        if geometry_type == "Polygon":
            add_ring(geometry.exterior.coords)
            for interior in geometry.interiors:
                add_ring(interior.coords)

        elif geometry_type == "MultiPolygon":
            for polygon in geometry.geoms:
                add_shapely_geometry(polygon)

        elif geometry_type == "GeometryCollection":
            for sub_geometry in geometry.geoms:
                add_shapely_geometry(sub_geometry)

    def add_geojson_geometry(geometry: dict[str, Any]) -> None:
        if not geometry:
            return

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")

        if coordinates is None:
            return

        if geometry_type == "Polygon":
            for ring in coordinates:
                add_ring(ring)

        elif geometry_type == "MultiPolygon":
            for polygon in coordinates:
                for ring in polygon:
                    add_ring(ring)

        elif geometry_type == "GeometryCollection":
            for sub_geometry in geometry.get("geometries", []):
                add_geojson_geometry(sub_geometry)

    # GeoPandas path.
    if HAVE_GPD and hasattr(vector_data, "geometry"):
        try:
            for geometry in vector_data.geometry:
                add_shapely_geometry(geometry)
        except Exception as exc:
            log(
                "[VIZ-WARN] Could not build GeoPandas clip geometry: "
                f"{exc}"
            )

    # Fiona / GeoJSON fallback path.
    elif isinstance(vector_data, dict):
        for geometry in vector_data.get("geometries", []):
            add_geojson_geometry(geometry)

    if not all_vertices:
        return None

    compound_path = MplPath(
        np.concatenate(all_vertices, axis=0),
        all_codes,
    )

    clip_patch = PathPatch(
        compound_path,
        facecolor="none",
        edgecolor="none",
        linewidth=0.0,
        transform=axis.transData,
        zorder=0,
    )

    # Adding the invisible patch initializes the axes data transform correctly.
    axis.add_patch(clip_patch)

    return clip_patch


def calculate_lsm_statistics(values: np.ndarray) -> dict[str, float | int]:
    return {
        "valid_pixels": int(values.size),
        "minimum": float(np.min(values)),
        "maximum": float(np.max(values)),
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "standard_deviation": float(np.std(values)),
        "p05": float(np.percentile(values, 5.0)),
        "p25": float(np.percentile(values, 25.0)),
        "p75": float(np.percentile(values, 75.0)),
        "p95": float(np.percentile(values, 95.0)),
    }


def lsm_visualization_output_paths(
    scenario_name: str,
) -> dict[str, Path]:
    prefix = scenario_prefix(scenario_name)
    stem = Config.VISUALIZATION_DIR / f"{prefix}_LR_Landslide_Susceptibility"

    return {
        "map_png": Path(str(stem) + "_Map.png"),
        "map_pdf": Path(str(stem) + "_Map.pdf"),
        "map_svg": Path(str(stem) + "_Map.svg"),
        "histogram_png": Path(str(stem) + "_Histogram.png"),
        "histogram_pdf": Path(str(stem) + "_Histogram.pdf"),
        "histogram_svg": Path(str(stem) + "_Histogram.svg"),
        "summary_json": Path(str(stem) + "_Visualization_Summary.json"),
        "statistics_csv": Path(str(stem) + "_Statistics.csv"),
    }


def create_lsm_probability_map(
    probability: np.ndarray,
    reference: dict[str, Any],
    scenario_name: str,
    outputs: dict[str, Path],
) -> None:
    bounds = reference["bounds"]

    raster_extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    buffer_x = (
        bounds.right - bounds.left
    ) * Config.DISPLAY_BUFFER_FRACTION
    buffer_y = (
        bounds.top - bounds.bottom
    ) * Config.DISPLAY_BUFFER_FRACTION

    west = bounds.left - buffer_x
    east = bounds.right + buffer_x
    south = bounds.bottom - buffer_y
    north = bounds.top + buffer_y

    display_array = np.ma.masked_invalid(probability)

    # Use an opaque white colour for masked / nodata cells.
    # This avoids transparent raster pixels in exported figures.
    map_cmap = LSM_CMAP.copy()
    map_cmap.set_bad(
        color="white",
        alpha=1.0,
    )

    raster_crs = reference["crs"]
    crs_is_geographic = bool(
        raster_crs and raster_crs.is_geographic
    )

    x_mid = 0.5 * (bounds.left + bounds.right)
    y_mid = 0.5 * (bounds.bottom + bounds.top)

    if crs_is_geographic:
        def format_longitude(value, position):
            return to_dms(value, True)

        def format_latitude(value, position):
            return to_dms(value, False)
    else:
        if not HAVE_PYPROJ:
            raise RuntimeError(
                "Projected susceptibility raster requires pyproj "
                "for DMS axis labels."
            )

        to_geographic = Transformer.from_crs(
            raster_crs,
            "EPSG:4326",
            always_xy=True,
        )

        def format_longitude(value, position):
            longitude, _ = to_geographic.transform(
                value,
                y_mid,
            )
            return to_dms(longitude, True)

        def format_latitude(value, position):
            _, latitude = to_geographic.transform(
                x_mid,
                value,
            )
            return to_dms(latitude, False)

    roads = load_vector_overlay(
        Config.ROAD_SHP,
        raster_crs,
    )
    study_boundary = load_vector_overlay(
        Config.STUDY_SHP,
        raster_crs,
    )

    style = {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "axes.unicode_minus": False,
    }

    with plt.rc_context(style):
        figure, axis = plt.subplots(
            1,
            1,
            figsize=Config.MAP_FIGSIZE,
            facecolor="white",
        )
        axis.set_facecolor("white")

        # Keep both the full figure canvas and axes background explicitly
        # opaque white. This is important for Word/EMF workflows.
        figure.patch.set_facecolor("white")
        figure.patch.set_alpha(1.0)
        axis.set_facecolor("white")

        image = axis.imshow(
            display_array,
            cmap=map_cmap,
            extent=raster_extent,
            vmin=0.0,
            vmax=1.0,
            origin="upper",
            interpolation=Config.RASTER_DISPLAY_INTERPOLATION,
            alpha=1.0,
            zorder=1,
        )

        # EMF-safe background handling used in the earlier LSM visualization:
        # clip the raster artist to the study-area polygon instead of leaving
        # transparent raster cells outside the boundary. The opaque white axes
        # background therefore shows through outside the study area.
        clip_patch = build_clip_patch_from_vector_overlay(
            study_boundary,
            axis,
        )
        if clip_patch is not None:
            image.set_clip_path(clip_patch)
            log(
                "[VIZ-MAP] Raster clipped to study boundary; "
                "opaque white background retained."
            )
        else:
            log(
                "[VIZ-WARN] Study-area clip path could not be created. "
                "Masked cells will still render as opaque white."
            )

        axis.set_xlim(west, east)
        axis.set_ylim(south, north)

        # Established Step 5.3 overlays.
        plot_vector_overlay(
            axis,
            study_boundary,
            colour="black",
            linewidth=0.7,
            zorder=800,
            polygon_boundary_only=True,
        )
        plot_vector_overlay(
            axis,
            roads,
            colour="red",
            linewidth=0.7,
            zorder=850,
            polygon_boundary_only=False,
        )

        axis.xaxis.set_major_locator(
            LinearLocator(Config.LONGITUDE_DMS_TICK_COUNT)
        )
        axis.yaxis.set_major_locator(
            LinearLocator(Config.LONGITUDE_DMS_TICK_COUNT)
        )
        axis.xaxis.set_major_formatter(
            FuncFormatter(format_longitude)
        )
        axis.yaxis.set_major_formatter(
            FuncFormatter(format_latitude)
        )
        axis.tick_params(
            axis="both",
            which="major",
            labelsize=5,
        )

        for tick_label in axis.get_xticklabels():
            tick_label.set_rotation(45)
            tick_label.set_ha("right")
            tick_label.set_fontsize(5)

        for tick_label in axis.get_yticklabels():
            tick_label.set_fontsize(5)

        add_north_arrow_tri(
            axis,
            size=0.09,
            pad=0.09,
        )

        crs_type = (
            "wgs84" if crs_is_geographic else "utm"
        )

        add_checker_scale_bar(
            axis,
            west,
            east,
            south,
            north,
            crs_type=crs_type,
            mid_lat_deg=(
                y_mid if crs_is_geographic else None
            ),
            segments=4,
            fontsize=4,
        )

        legend_elements = [
            Line2D(
                [0],
                [0],
                color="red",
                linewidth=1.0,
                label="Highway",
            ),
            Line2D(
                [0],
                [0],
                color="black",
                linewidth=1.0,
                label="Study Area",
            ),
        ]

        legend = axis.legend(
            handles=legend_elements,
            loc="center",
            bbox_to_anchor=(0.82, 0.35),
            bbox_transform=axis.transAxes,
            frameon=True,
            framealpha=0.9,
            fontsize=2.5,
            labelspacing=0.3,
            borderpad=0.3,
            handlelength=1.2,
            handletextpad=0.4,
            borderaxespad=0.0,
        )
        legend.get_frame().set_edgecolor("lightgray")
        legend.get_frame().set_linewidth(0.3)

        divider = make_axes_locatable(axis)
        colorbar_axis = divider.append_axes(
            "right",
            size=Config.CAX_SIZE,
            pad=Config.CAX_PAD,
        )

        colorbar = figure.colorbar(
            image,
            cax=colorbar_axis,
        )
        colorbar.ax.tick_params(labelsize=5)
        colorbar.set_label(
            "Landslide susceptibility probability (0–1)",
            fontsize=4,
            fontweight="bold",
        )
        colorbar.set_ticks(
            np.linspace(0.0, 1.0, 5)
        )

        figure.tight_layout()

        if Config.SAVE_LSM_FIGURES:
            figure.savefig(
                outputs["map_png"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["map_pdf"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["map_svg"],
                format="svg",
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )

            log(f"[VIZ-MAP] {outputs['map_png']}")
            log(f"[VIZ-MAP] {outputs['map_pdf']}")
            log(f"[VIZ-MAP] {outputs['map_svg']}")

        if Config.SHOW_LSM_FIGURES:
            # Use Matplotlib's standard display path. This is more reliable
            # across Jupyter, Spyder, VS Code, and standard Python sessions.
            plt.show()

        plt.close(figure)


def create_lsm_probability_histogram(
    values: np.ndarray,
    statistics: dict[str, float | int],
    outputs: dict[str, Path],
) -> None:
    style = {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "axes.unicode_minus": False,
    }

    with plt.rc_context(style):
        figure, axis = plt.subplots(
            1,
            1,
            figsize=Config.HIST_FIGSIZE,
            facecolor="white",
        )

        bins = np.linspace(0.0, 1.0, 21)
        counts, edges = np.histogram(values, bins=bins)
        centres = 0.5 * (edges[:-1] + edges[1:])
        colours = LSM_CMAP(centres)

        axis.bar(
            centres,
            counts,
            width=(bins[1] - bins[0]) * 0.9,
            color=colours,
            edgecolor="black",
            linewidth=0.4,
            alpha=0.9,
        )

        axis.axvline(
            statistics["mean"],
            color="darkred",
            linestyle="--",
            linewidth=0.7,
            label=f"Mean: {statistics['mean']:.3f}",
        )
        axis.axvline(
            statistics["median"],
            color="orange",
            linestyle="--",
            linewidth=0.7,
            label=f"Median: {statistics['median']:.3f}",
        )
        axis.axvline(
            statistics["p95"],
            color="green",
            linestyle="--",
            linewidth=0.7,
            label=f"95th %ile: {statistics['p95']:.3f}",
        )

        axis.legend(
            fontsize=3,
            loc="upper right",
            framealpha=0.9,
        )
        axis.set_xlabel(
            "Landslide susceptibility probability (0–1)",
            fontsize=4,
            fontweight="bold",
        )
        axis.set_ylabel(
            "Number of Pixels",
            fontsize=4,
            fontweight="bold",
        )
        axis.tick_params(
            axis="both",
            which="major",
            labelsize=4,
        )
        axis.set_xlim(0.0, 1.0)
        axis.grid(
            True,
            alpha=0.25,
            linestyle="-",
            linewidth=0.4,
            axis="y",
        )
        axis.set_axisbelow(True)

        figure.tight_layout(pad=1.2)

        if Config.SAVE_LSM_FIGURES:
            figure.savefig(
                outputs["histogram_png"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["histogram_pdf"],
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )
            figure.savefig(
                outputs["histogram_svg"],
                format="svg",
                dpi=Config.FIG_DPI,
                bbox_inches="tight",
                facecolor="white",
                edgecolor="white",
                transparent=False,
            )

            log(f"[VIZ-HIST] {outputs['histogram_png']}")
            log(f"[VIZ-HIST] {outputs['histogram_pdf']}")
            log(f"[VIZ-HIST] {outputs['histogram_svg']}")

        if Config.SHOW_LSM_FIGURES:
            # Use Matplotlib's standard display path. This is more reliable
            # across Jupyter, Spyder, VS Code, and standard Python sessions.
            plt.show()

        plt.close(figure)


def create_primary_lsm_visualization(
    raster_path: Path,
    scenario_name: str,
) -> dict[str, Any]:
    """
    Create the primary Model E susceptibility map and histogram using the
    established Step 5.3 cartographic style.
    """
    probability, reference = read_probability_raster(
        raster_path
    )

    values = probability[np.isfinite(probability)]
    statistics = calculate_lsm_statistics(values)
    outputs = lsm_visualization_output_paths(
        scenario_name
    )

    create_lsm_probability_map(
        probability=probability,
        reference=reference,
        scenario_name=scenario_name,
        outputs=outputs,
    )

    create_lsm_probability_histogram(
        values=values,
        statistics=statistics,
        outputs=outputs,
    )

    statistics_df = pd.DataFrame(
        [
            {
                "scenario": scenario_name,
                "model_label": Config.MODEL_SHORT_LABELS[scenario_name],
                "model_description": Config.MODEL_DESCRIPTIONS[scenario_name],
                **statistics,
            }
        ]
    )
    statistics_df.to_csv(
        outputs["statistics_csv"],
        index=False,
    )

    summary = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "Logistic Regression",
        "scenario": scenario_name,
        "model_label": Config.MODEL_SHORT_LABELS[scenario_name],
        "model_description": Config.MODEL_DESCRIPTIONS[scenario_name],
        "probability_raster": str(raster_path),
        "prediction_domain": "common_A_to_F_valid_domain",
        "validation_note": (
            "This figure is generated from the final full-data deployment model. "
            "Model performance must be reported from nested spatial OOF predictions, "
            "not from the full-data map."
        ),
        "scientific_plotting": {
            "direct_probability_raster": True,
            "probabilities_renormalized_for_visualization": False,
            "RBF_used": False,
            "IDW_used": False,
            "cubic_interpolation_used": False,
            "display_interpolation": Config.RASTER_DISPLAY_INTERPOLATION,
        },
        "statistics": statistics,
        "visual_style": {
            "reference_style": "Step 5.3 HKFI visualization",
            "map_figsize_inches": list(Config.MAP_FIGSIZE),
            "histogram_figsize_inches": list(Config.HIST_FIGSIZE),
            "display_buffer_fraction": Config.DISPLAY_BUFFER_FRACTION,
            "background": "opaque_white",
            "transparent_export": False,
            "study_area_vector_clip": True,
            "masked_raster_colour": "opaque_white",
            "DMS_axes": True,
            "north_arrow": True,
            "checkerboard_scale_bar": True,
            "highway_overlay": True,
            "study_area_overlay": True,
            "compact_legend": True,
            "side_colour_bar": True,
            "colour_ramp": "green-yellow-orange-red susceptibility ramp",
        },
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }

    with open(
        outputs["summary_json"],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(summary),
            handle,
            indent=2,
        )

    log(
        "[VIZ] Primary Logistic Regression landslide susceptibility visualization "
        f"completed for {Config.MODEL_SHORT_LABELS[scenario_name]}."
    )

    # This is the direct Model E visualization on the common A-F domain.
    # Area-percentage and information-support products belong to the separate
    # A30/E70 ensemble visualization workflow and are therefore not referenced
    # here. Keeping the return object limited to values created inside this
    # function prevents accidental cross-scope NameError failures.
    return {
        "scenario": scenario_name,
        "raster_path": str(raster_path),
        "statistics": statistics,
        "outputs": {
            key: str(value)
            for key, value in outputs.items()
        },
    }


# =============================================================================
# SUMMARIES / COMPARISONS / FIGURES
# =============================================================================
def summarize_oof_results(
    all_oof: pd.DataFrame,
    all_fold_metrics: pd.DataFrame,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for scenario_name in Config.SCENARIO_ORDER:
        subset = all_oof.loc[all_oof["scenario"] == scenario_name].copy()
        fold_subset = all_fold_metrics.loc[
            all_fold_metrics["scenario"] == scenario_name
        ].copy()

        y = subset[Config.TARGET_COLUMN].to_numpy(dtype=int)
        probability = subset["oof_probability"].to_numpy(dtype=float)
        blocks = subset[Config.FOLD_COLUMN].to_numpy(dtype=int)

        pooled = classification_metrics(y, probability)
        ci_low, ci_high, n_valid_boot = cluster_bootstrap_auc_ci(
            y_true=y,
            y_prob=probability,
            blocks=blocks,
            n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
            seed=Config.RANDOM_STATE + Config.SCENARIO_ORDER.index(scenario_name),
            ci_level=Config.CI_LEVEL,
        )

        row = {
            "scenario": scenario_name,
            "model": Config.MODEL_SHORT_LABELS[scenario_name],
            "description": Config.MODEL_DESCRIPTIONS[scenario_name],
            "n_samples": int(len(subset)),
            "n_predictors": int(fold_subset["n_predictors"].iloc[0]),
            "oof_roc_auc": pooled["roc_auc"],
            "oof_auc_cluster_bootstrap_ci_low": ci_low,
            "oof_auc_cluster_bootstrap_ci_high": ci_high,
            "cluster_bootstrap_valid_iterations": int(n_valid_boot),
            "oof_pr_auc": pooled["pr_auc"],
            "oof_brier": pooled["brier"],
            "oof_accuracy_0p5": pooled["accuracy"],
            "oof_balanced_accuracy_0p5": pooled["balanced_accuracy"],
            "oof_precision_0p5": pooled["precision"],
            "oof_recall_0p5": pooled["recall"],
            "oof_f1_0p5": pooled["f1"],
            "fold_auc_mean": float(fold_subset["roc_auc"].mean()),
            "fold_auc_std": float(fold_subset["roc_auc"].std(ddof=1)),
            "fold_auc_min": float(fold_subset["roc_auc"].min()),
            "fold_auc_max": float(fold_subset["roc_auc"].max()),
        }
        rows.append(row)

    summary = pd.DataFrame(rows)

    baseline_auc = float(
        summary.loc[
            summary["scenario"] == "A_static_baseline",
            "oof_roc_auc",
        ].iloc[0]
    )

    summary["absolute_auc_gain_vs_A"] = summary["oof_roc_auc"] - baseline_auc
    summary["relative_auc_gain_vs_A_percent"] = np.where(
        baseline_auc != 0,
        100.0 * summary["absolute_auc_gain_vs_A"] / baseline_auc,
        np.nan,
    )

    return summary


def make_paired_auc_comparisons(all_oof: pd.DataFrame) -> pd.DataFrame:
    comparisons = [
        ("B_raw_deformation", "A_static_baseline", "B - A", "Raw DV+ added value"),
        ("C_hkfi_only", "A_static_baseline", "C - A", "HKFI added value"),
        ("D_sndr_only", "A_static_baseline", "D - A", "SNDR added value"),
        ("E_physics_guided", "A_static_baseline", "E - A", "Proposed indices total added value"),
        ("F_untransformed_sources", "A_static_baseline", "F - A", "Separate source variables added value"),
        ("E_physics_guided", "B_raw_deformation", "E - B", "Physics-guided versus raw DV+"),
        ("E_physics_guided", "F_untransformed_sources", "E - F", "Indices versus separate source variables"),
    ]

    pivot = all_oof.pivot(
        index=Config.SAMPLE_ID_COLUMN,
        columns="scenario",
        values="oof_probability",
    )

    metadata = (
        all_oof.loc[
            all_oof["scenario"] == "A_static_baseline",
            [Config.SAMPLE_ID_COLUMN, Config.TARGET_COLUMN, Config.FOLD_COLUMN],
        ]
        .drop_duplicates(Config.SAMPLE_ID_COLUMN)
        .set_index(Config.SAMPLE_ID_COLUMN)
        .loc[pivot.index]
    )

    y = metadata[Config.TARGET_COLUMN].to_numpy(dtype=int)
    blocks = metadata[Config.FOLD_COLUMN].to_numpy(dtype=int)

    rows: List[Dict[str, Any]] = []
    for comparison_index, (left, right, label, purpose) in enumerate(comparisons):
        observed, ci_low, ci_high, n_valid = paired_cluster_bootstrap_auc_difference(
            y_true=y,
            prob_left=pivot[left].to_numpy(dtype=float),
            prob_right=pivot[right].to_numpy(dtype=float),
            blocks=blocks,
            n_bootstrap=Config.N_CLUSTER_BOOTSTRAP,
            seed=Config.RANDOM_STATE + 100 + comparison_index,
            ci_level=Config.CI_LEVEL,
        )

        rows.append(
            {
                "comparison": label,
                "purpose": purpose,
                "left_scenario": left,
                "right_scenario": right,
                "auc_difference_left_minus_right": observed,
                "cluster_bootstrap_ci_low": ci_low,
                "cluster_bootstrap_ci_high": ci_high,
                "cluster_bootstrap_valid_iterations": int(n_valid),
            }
        )

    return pd.DataFrame(rows)


def plot_oof_roc_curves(all_oof: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(5.0, 5.0))

    for scenario_name in Config.SCENARIO_ORDER:
        subset = all_oof.loc[all_oof["scenario"] == scenario_name]
        y = subset[Config.TARGET_COLUMN].to_numpy(dtype=int)
        probability = subset["oof_probability"].to_numpy(dtype=float)
        fpr, tpr, _ = roc_curve(y, probability)
        auc = roc_auc_score(y, probability)
        label = f"{Config.MODEL_SHORT_LABELS[scenario_name]} (AUC={auc:.3f})"
        ax.plot(fpr, tpr, linewidth=1.4, label=label)

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.0)

    ax.set_xlabel(
        "False Positive Rate",
        fontsize=AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )
    ax.set_ylabel(
        "True Positive Rate",
        fontsize=AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

    ax.legend(
        frameon=False,
        fontsize=LEGEND_FONTSIZE,
        loc="lower right",
    )
    ax.grid(True, alpha=0.25)

    apply_axis_text_style(
        ax,
        legend_fontsize=LEGEND_FONTSIZE,
    )

    save_figure_triplet(
        fig,
        Config.ROC_FIG_PNG,
        Config.ROC_FIG_PDF,
        Config.ROC_FIG_SVG,
        dpi=600,
    )


def plot_fold_auc(all_fold_metrics: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(5.5, 4.0))

    x_positions = np.arange(1, len(Config.SCENARIO_ORDER) + 1)
    for x, scenario_name in zip(x_positions, Config.SCENARIO_ORDER):
        values = all_fold_metrics.loc[
            all_fold_metrics["scenario"] == scenario_name,
            "roc_auc",
        ].to_numpy(dtype=float)
        ax.scatter(np.full(values.shape, x), values, s=28)
        ax.plot(
            [x - 0.16, x + 0.16],
            [values.mean(), values.mean()],
            linewidth=1.4,
        )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [Config.MODEL_SHORT_LABELS[name] for name in Config.SCENARIO_ORDER]
    )

    ax.set_xlabel(
        "Model configuration",
        fontsize=AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )
    ax.set_ylabel(
        "Held-out spatial-fold AUC",
        fontsize=AXIS_LABEL_FONTSIZE,
        fontweight="bold",
    )

    ax.set_ylim(0.0, 1.02)
    ax.grid(True, axis="y", alpha=0.25)

    apply_axis_text_style(ax)

    save_figure_triplet(
        fig,
        Config.FOLD_AUC_FIG_PNG,
        Config.FOLD_AUC_FIG_PDF,
        Config.FOLD_AUC_FIG_SVG,
        dpi=600,
    )


# =============================================================================
# REPORTING
# =============================================================================

def write_manuscript_performance_excel(
    summary_df: pd.DataFrame,
    comparison_df: pd.DataFrame,
    all_fold_metrics: pd.DataFrame,
    ensemble_oof_metrics: Dict[str, Any],
) -> Path:
    """
    Export manuscript-ready numerical tables to a single Excel workbook.

    Sheets
    ------
    1. A-F Performance
       Main manuscript table:
       Model | Predictor configuration | OOF ROC-AUC | 95% CI | PR-AUC |
       Balanced Accuracy | F1-score

    2. AUC Comparisons
       Reviewer-focused paired delta-AUC comparisons with cluster-bootstrap CIs.

    3. Fold AUC
       Fold-specific held-out spatial AUC values plus mean and SD.

    4. Ensemble Summary
       Fixed A30/E70 ensemble OOF performance, reported separately from A-F.
    """
    predictor_labels = {
        "A_static_baseline": "Static12",
        "B_raw_deformation": "Static12 + DV+",
        "C_hkfi_only": "Static12 + HKFI",
        "D_sndr_only": "Static12 + SNDR",
        "E_physics_guided": "Static12 + HKFI + SNDR",
        "F_untransformed_sources": "Static12 + DV+ + TWI + support-preserved SM",
    }

    # ---------------- Main A-F manuscript table ----------------
    ordered_summary = (
        summary_df.set_index("scenario")
        .loc[Config.SCENARIO_ORDER]
        .reset_index()
    )

    main_table = pd.DataFrame(
        {
            "Model": ordered_summary["model"],
            "Predictor configuration": [
                predictor_labels.get(name, Config.MODEL_DESCRIPTIONS[name])
                for name in ordered_summary["scenario"]
            ],
            "OOF ROC-AUC": ordered_summary["oof_roc_auc"],
            "95% CI": [
                (
                    f"{low:.3f}–{high:.3f}"
                    if np.isfinite(low) and np.isfinite(high)
                    else ""
                )
                for low, high in zip(
                    ordered_summary["oof_auc_cluster_bootstrap_ci_low"],
                    ordered_summary["oof_auc_cluster_bootstrap_ci_high"],
                )
            ],
            "CI lower": ordered_summary["oof_auc_cluster_bootstrap_ci_low"],
            "CI upper": ordered_summary["oof_auc_cluster_bootstrap_ci_high"],
            "PR-AUC": ordered_summary["oof_pr_auc"],
            "Balanced Accuracy": ordered_summary["oof_balanced_accuracy_0p5"],
            "F1-score": ordered_summary["oof_f1_0p5"],
            "Fold AUC mean": ordered_summary["fold_auc_mean"],
            "Fold AUC SD": ordered_summary["fold_auc_std"],
        }
    )

    # ---------------- Paired delta-AUC comparison table ----------------
    comparison_table = pd.DataFrame(
        {
            "Comparison": comparison_df["comparison"],
            "ΔAUC": comparison_df["auc_difference_left_minus_right"],
            "95% CI": [
                (
                    f"{low:+.3f}–{high:+.3f}"
                    if np.isfinite(low) and np.isfinite(high)
                    else ""
                )
                for low, high in zip(
                    comparison_df["cluster_bootstrap_ci_low"],
                    comparison_df["cluster_bootstrap_ci_high"],
                )
            ],
            "CI lower": comparison_df["cluster_bootstrap_ci_low"],
            "CI upper": comparison_df["cluster_bootstrap_ci_high"],
            "Interpretation": comparison_df["purpose"],
        }
    )

    # ---------------- Fold-specific AUC table ----------------
    fold_table = (
        all_fold_metrics.pivot(
            index="scenario",
            columns="outer_fold",
            values="roc_auc",
        )
        .reindex(Config.SCENARIO_ORDER)
        .reset_index()
    )

    fold_table.insert(
        1,
        "Model",
        [
            Config.MODEL_SHORT_LABELS[scenario_name]
            for scenario_name in fold_table["scenario"]
        ],
    )
    fold_table.insert(
        2,
        "Predictor configuration",
        [
            predictor_labels.get(
                scenario_name,
                Config.MODEL_DESCRIPTIONS[scenario_name],
            )
            for scenario_name in fold_table["scenario"]
        ],
    )

    fold_columns = [
        column
        for column in fold_table.columns
        if isinstance(column, (int, np.integer))
    ]
    fold_columns = sorted(fold_columns)

    fold_table = fold_table.rename(
        columns={
            fold: f"Fold {fold}"
            for fold in fold_columns
        }
    )

    fold_metric_columns = [
        f"Fold {fold}"
        for fold in fold_columns
    ]
    fold_table["Mean"] = fold_table[fold_metric_columns].mean(axis=1)
    fold_table["SD"] = fold_table[fold_metric_columns].std(axis=1, ddof=1)
    fold_table = fold_table.drop(columns=["scenario"])

    # ---------------- Separate operational ensemble summary ----------------
    ensemble_table = pd.DataFrame(
        [
            {
                "Mapping product": "A30/E70 fixed operational ensemble",
                "Formula": "0.30 × Model A + 0.70 × Model E where E is available; otherwise Model A",
                "OOF ROC-AUC": ensemble_oof_metrics.get("roc_auc"),
                "95% CI": (
                    f"{ensemble_oof_metrics.get('auc_cluster_bootstrap_ci_low', np.nan):.3f}"
                    "–"
                    f"{ensemble_oof_metrics.get('auc_cluster_bootstrap_ci_high', np.nan):.3f}"
                ),
                "PR-AUC": ensemble_oof_metrics.get("pr_auc"),
                "Balanced Accuracy": ensemble_oof_metrics.get("balanced_accuracy"),
                "F1-score": ensemble_oof_metrics.get("f1"),
                "ΔAUC vs Model E": ensemble_oof_metrics.get(
                    "auc_difference_ensemble_minus_model_E"
                ),
                "ΔAUC 95% CI": (
                    f"{ensemble_oof_metrics.get('auc_difference_ci_low', np.nan):+.3f}"
                    "–"
                    f"{ensemble_oof_metrics.get('auc_difference_ci_high', np.nan):+.3f}"
                ),
                "Role in manuscript": (
                    "Final corridor-wide operational mapping product; "
                    "reported separately from the A–F ablation evaluation."
                ),
            }
        ]
    )

    # ---------------- Write Excel workbook ----------------
    output_path = Config.MANUSCRIPT_TABLES_XLSX

    try:
        # xlsxwriter provides publication-ready formatting when available.
        with pd.ExcelWriter(
            output_path,
            engine="xlsxwriter",
        ) as writer:
            main_table.to_excel(
                writer,
                sheet_name="A-F Performance",
                index=False,
            )
            comparison_table.to_excel(
                writer,
                sheet_name="AUC Comparisons",
                index=False,
            )
            fold_table.to_excel(
                writer,
                sheet_name="Fold AUC",
                index=False,
            )
            ensemble_table.to_excel(
                writer,
                sheet_name="Ensemble Summary",
                index=False,
            )

            workbook = writer.book

            header_format = workbook.add_format(
                {
                    "bold": True,
                    "font_color": "white",
                    "bg_color": "#1F4E78",
                    "border": 1,
                    "align": "center",
                    "valign": "vcenter",
                    "text_wrap": True,
                }
            )
            number_format = workbook.add_format(
                {
                    "num_format": "0.000",
                    "border": 1,
                    "align": "center",
                }
            )
            text_format = workbook.add_format(
                {
                    "border": 1,
                    "valign": "top",
                    "text_wrap": True,
                }
            )
            center_text_format = workbook.add_format(
                {
                    "border": 1,
                    "align": "center",
                    "valign": "center",
                    "text_wrap": True,
                }
            )

            sheet_specs = {
                "A-F Performance": {
                    "df": main_table,
                    "widths": [10, 45, 13, 18, 12, 12, 12, 18, 12, 15, 13],
                },
                "AUC Comparisons": {
                    "df": comparison_table,
                    "widths": [14, 12, 18, 12, 12, 42],
                },
                "Fold AUC": {
                    "df": fold_table,
                    "widths": [10, 45] + [12] * (len(fold_metric_columns) + 2),
                },
                "Ensemble Summary": {
                    "df": ensemble_table,
                    "widths": [32, 70, 13, 18, 12, 18, 12, 18, 18, 60],
                },
            }

            for sheet_name, spec in sheet_specs.items():
                worksheet = writer.sheets[sheet_name]
                dataframe = spec["df"]

                # Freeze the header row.
                worksheet.freeze_panes(1, 0)

                # Reformat headers.
                for column_index, column_name in enumerate(dataframe.columns):
                    worksheet.write(
                        0,
                        column_index,
                        column_name,
                        header_format,
                    )

                # Set useful column widths.
                for column_index, width in enumerate(spec["widths"]):
                    worksheet.set_column(
                        column_index,
                        column_index,
                        width,
                    )

                # Apply borders and numeric formatting to body cells.
                for row_index in range(1, len(dataframe) + 1):
                    for column_index, column_name in enumerate(dataframe.columns):
                        value = dataframe.iloc[row_index - 1, column_index]

                        if pd.isna(value):
                            worksheet.write_blank(
                                row_index,
                                column_index,
                                None,
                                center_text_format,
                            )
                        elif isinstance(value, (float, np.floating, int, np.integer)):
                            worksheet.write_number(
                                row_index,
                                column_index,
                                float(value),
                                number_format,
                            )
                        else:
                            worksheet.write(
                                row_index,
                                column_index,
                                str(value),
                                (
                                    center_text_format
                                    if column_name in {
                                        "Model",
                                        "Comparison",
                                        "95% CI",
                                        "ΔAUC 95% CI",
                                    }
                                    else text_format
                                ),
                            )

                # Add an Excel table for convenient filtering and manuscript review.
                if len(dataframe) > 0 and len(dataframe.columns) > 0:
                    worksheet.add_table(
                        0,
                        0,
                        len(dataframe),
                        len(dataframe.columns) - 1,
                        {
                            "style": "Table Style Medium 2",
                            "columns": [
                                {"header": column_name}
                                for column_name in dataframe.columns
                            ],
                        },
                    )

    except (ImportError, ModuleNotFoundError):
        # Fallback: still create a valid workbook using the default pandas engine.
        # The numerical content remains identical, but styling may be simpler.
        with pd.ExcelWriter(output_path) as writer:
            main_table.to_excel(
                writer,
                sheet_name="A-F Performance",
                index=False,
            )
            comparison_table.to_excel(
                writer,
                sheet_name="AUC Comparisons",
                index=False,
            )
            fold_table.to_excel(
                writer,
                sheet_name="Fold AUC",
                index=False,
            )
            ensemble_table.to_excel(
                writer,
                sheet_name="Ensemble Summary",
                index=False,
            )

    log(
        "[EXCEL] Manuscript-ready performance tables saved: "
        f"{output_path}"
    )

    return output_path


def write_qa_report(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
    summary_df: pd.DataFrame,
    comparison_df: pd.DataFrame,
    final_params: Dict[str, Any],
    map_paths: Dict[str, str],
) -> None:
    with open(Config.QA_REPORT_TXT, "w", encoding="utf-8") as handle:
        handle.write("STEP 7.2 — RANDOM FOREST MODELS A–F: NESTED SPATIAL CV QA REPORT\n")
        handle.write("=" * 88 + "\n\n")
        handle.write(f"Generated: {utc_now_iso()}\n")
        handle.write(f"Input dataset: {Config.MODEL_READY_DATASET_CSV}\n")
        handle.write(f"Rows: {len(df)}\n")
        handle.write(
            f"Class balance: positive={(df[Config.TARGET_COLUMN] == 1).sum()}, "
            f"negative={(df[Config.TARGET_COLUMN] == 0).sum()}\n"
        )
        handle.write(
            f"Frozen outer folds: {sorted(df[Config.FOLD_COLUMN].unique().tolist())}\n\n"
        )

        handle.write("VALIDATION DESIGN\n")
        handle.write("-" * 88 + "\n")
        handle.write(
            "Outer evaluation: five frozen contiguous spatial blocks; one block held out at a time.\n"
        )
        handle.write(
            "Inner tuning: four-block PredefinedSplit using only the outer-training data.\n"
        )
        handle.write(
            "No outer validation observations are used for hyperparameter selection or model selection.\n"
        )
        handle.write(
            "All A–F models use identical samples, labels and outer folds.\n"
        )
        handle.write(
            "Each frozen outer fold contains exactly 33 positives and 33 pseudo-absences.\n"
        )
        handle.write(
            "Land cover is one-hot encoded inside each fitted pipeline; continuous LR predictors are not scaled.\n"
        )
        handle.write(
            "Binary metrics use a fixed 0.50 probability threshold.\n\n"
        )

        handle.write("SCENARIOS\n")
        handle.write("-" * 88 + "\n")
        for scenario_name in Config.SCENARIO_ORDER:
            features = scenarios[scenario_name]["features"]
            handle.write(
                f"{Config.MODEL_SHORT_LABELS[scenario_name]}: "
                f"{Config.MODEL_DESCRIPTIONS[scenario_name]} "
                f"({len(features)} predictors)\n"
            )
        handle.write("\n")

        handle.write("NESTED OOF SUMMARY\n")
        handle.write("-" * 88 + "\n")
        handle.write(summary_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("PAIRED AUC COMPARISONS\n")
        handle.write("-" * 88 + "\n")
        handle.write(comparison_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("FINAL DEPLOYMENT MODELS\n")
        handle.write("-" * 88 + "\n")
        if final_params:
            for scenario_name in Config.SCENARIO_ORDER:
                info = final_params[scenario_name]
                handle.write(
                    f"{Config.MODEL_SHORT_LABELS[scenario_name]}: "
                    f"selection AUC={info['best_mean_spatial_cv_auc_for_model_selection_only']:.6f}; "
                    f"params={info['best_params']}\n"
                )
        else:
            handle.write(
                "Skipped in this run because RUN_FINAL_DEPLOYMENT_TUNING=False. "
                "Nested spatial OOF evaluation was completed independently.\n"
            )
        handle.write("\n")

        if map_paths:
            handle.write("RASTER MAPS\n")
            handle.write("-" * 88 + "\n")
            handle.write(
                "Maps are generated from separately tuned full-data deployment models on the common A–F valid domain.\n"
            )
            handle.write(
                "These map-model tuning scores are not reported as validation performance.\n"
            )
            for scenario_name, path in map_paths.items():
                handle.write(f"{scenario_name}: {path}\n")


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log("=" * 96)
    log("=== Step 7.3: Logistic Regression Models A–F with Nested Spatial Cross-Validation ===")
    log("=== GRID SIZE: 8 C values; ~960 nested-CV candidate fits + ~240 deployment candidate fits, plus refits ===")
    log("=== Logistic Regression is substantially lighter than XGB/RF; monitor progress via verbose output. ===")
    log(
        "[RUN-MODE] Final deployment tuning="
        f"{'ON' if Config.RUN_FINAL_DEPLOYMENT_TUNING else 'OFF'} | "
        "Raster maps="
        f"{'ON' if Config.GENERATE_RASTER_MAPS else 'OFF'} | "
        "Primary Model E visualization="
        f"{'ON' if Config.GENERATE_PRIMARY_LSM_VISUALIZATION else 'OFF'} | "
        "A30/E70 ensemble LSM="
        f"{'ON' if Config.GENERATE_AE_ENSEMBLE_LSM else 'OFF'}"
    )
    log(
        "[VIZ-OUT] Primary Landslide Susceptibility Map outputs: "
        f"{Config.VISUALIZATION_DIR}"
    )
    log("=" * 96)

    # 1. Validate frozen A–F design and primary frozen ML dataset.
    scenarios = validate_scenario_manifests()
    df = load_and_validate_model_ready_dataset(scenarios)

    all_oof_tables: List[pd.DataFrame] = []
    all_fold_tables: List[pd.DataFrame] = []
    all_inner_search_tables: List[pd.DataFrame] = []
    outer_best_params: Dict[str, Any] = {}

    # 2. Unbiased nested spatial CV for each A–F model.
    for scenario_name in Config.SCENARIO_ORDER:
        features = list(scenarios[scenario_name]["features"])

        oof_df, fold_df, params, search_df = run_nested_spatial_cv_for_scenario(
            df=df,
            scenario_name=scenario_name,
            features=features,
        )

        all_oof_tables.append(oof_df)
        all_fold_tables.append(fold_df)
        all_inner_search_tables.append(search_df)
        outer_best_params[scenario_name] = params

    all_oof = pd.concat(all_oof_tables, ignore_index=True)
    all_fold_metrics = pd.concat(all_fold_tables, ignore_index=True)
    all_inner_search = pd.concat(all_inner_search_tables, ignore_index=True)

    # Cross-scenario QA: every scenario must contain the exact same sample IDs.
    reference_ids = set(
        all_oof.loc[
            all_oof["scenario"] == Config.SCENARIO_ORDER[0],
            Config.SAMPLE_ID_COLUMN,
        ]
    )
    for scenario_name in Config.SCENARIO_ORDER[1:]:
        scenario_ids = set(
            all_oof.loc[
                all_oof["scenario"] == scenario_name,
                Config.SAMPLE_ID_COLUMN,
            ]
        )
        if scenario_ids != reference_ids:
            raise RuntimeError(
                f"OOF sample IDs differ for {scenario_name}; A–F comparison invalid."
            )

    # 3. Summaries and reviewer-focused paired comparisons.
    summary_df = summarize_oof_results(all_oof, all_fold_metrics)
    comparison_df = make_paired_auc_comparisons(all_oof)

    # Fixed A30/E70 ensemble OOF diagnostic. This mirrors the old
    # static + integrated blend but does not alter the A-F ablation results.
    ensemble_oof_df, ensemble_oof_metrics = build_fixed_ae_oof_ensemble(
        all_oof
    )

    all_oof.to_csv(Config.OOF_PREDICTIONS_CSV, index=False, float_format="%.10f")
    all_fold_metrics.to_csv(
        Config.FOLD_METRICS_CSV, index=False, float_format="%.10f"
    )
    summary_df.to_csv(
        Config.SUMMARY_METRICS_CSV, index=False, float_format="%.10f"
    )
    comparison_df.to_csv(
        Config.AUC_COMPARISONS_CSV, index=False, float_format="%.10f"
    )

    # Manuscript-ready Excel workbook with the compact A-F table recommended
    # for the revised paper, paired delta-AUC comparisons, fold-level AUCs,
    # and a separate operational ensemble summary.
    manuscript_excel_path = write_manuscript_performance_excel(
        summary_df=summary_df,
        comparison_df=comparison_df,
        all_fold_metrics=all_fold_metrics,
        ensemble_oof_metrics=ensemble_oof_metrics,
    )

    all_inner_search.to_csv(
        Config.INNER_SEARCH_CSV, index=False, float_format="%.10f"
    )

    with open(Config.OUTER_PARAMS_JSON, "w", encoding="utf-8") as handle:
        json.dump(json_safe(outer_best_params), handle, indent=2)

    plot_oof_roc_curves(all_oof)
    plot_fold_auc(all_fold_metrics)

    # 4. Optional separate final full-data models for mapping/deployment only.
    final_models: Dict[str, Pipeline] = {}
    final_params: Dict[str, Any] = {}
    full_search_tables: List[pd.DataFrame] = []

    if Config.RUN_FINAL_DEPLOYMENT_TUNING:
        log("")
        log("[FINAL-MODEL] Running full-data spatial tuning for deployment models...")
        for scenario_name in Config.SCENARIO_ORDER:
            features = list(scenarios[scenario_name]["features"])
            final_model, info, search_df = tune_and_fit_final_model(
                df=df,
                scenario_name=scenario_name,
                features=features,
            )
            final_models[scenario_name] = final_model
            final_params[scenario_name] = info
            full_search_tables.append(search_df)

        full_search_all = pd.concat(full_search_tables, ignore_index=True)
        full_search_all.to_csv(
            Config.FINAL_SEARCH_CSV, index=False, float_format="%.10f"
        )

        with open(Config.FINAL_PARAMS_JSON, "w", encoding="utf-8") as handle:
            json.dump(json_safe(final_params), handle, indent=2)
    else:
        log(
            "[FINAL-MODEL] Skipped: RUN_FINAL_DEPLOYMENT_TUNING=False. "
            "This diagnostic run will report nested spatial OOF performance only."
        )

    # 5. Optional A–F susceptibility rasters on an identical common domain.
    map_paths: Dict[str, str] = {}
    if Config.GENERATE_RASTER_MAPS:
        if not Config.RUN_FINAL_DEPLOYMENT_TUNING:
            raise RuntimeError(
                "GENERATE_RASTER_MAPS=True requires "
                "RUN_FINAL_DEPLOYMENT_TUNING=True."
            )

        log("")
        log("[MAP] Generating six Logistic Regression A–F probability maps...")
        for scenario_name in Config.SCENARIO_ORDER:
            features = list(scenarios[scenario_name]["features"])
            output_path = predict_scenario_raster(
                fitted_pipeline=final_models[scenario_name],
                scenario_name=scenario_name,
                features=features,
            )
            map_paths[scenario_name] = str(output_path)

    # 6. Revised old-style A/E ensemble susceptibility raster.
    ensemble_map_path: Path | None = None
    ensemble_visualization_outputs: Dict[str, Any] = {}

    if Config.GENERATE_AE_ENSEMBLE_LSM:
        if not Config.RUN_FINAL_DEPLOYMENT_TUNING:
            raise RuntimeError(
                "GENERATE_AE_ENSEMBLE_LSM=True requires "
                "RUN_FINAL_DEPLOYMENT_TUNING=True."
            )

        log("")
        log(
            "[ENSEMBLE-MAP] Generating revised static-background + "
            "physics-guided ensemble map..."
        )
        ensemble_map_path = create_revised_ae_ensemble_probability_map(
            final_models=final_models,
            scenarios=scenarios,
        )

        ensemble_visualization_outputs = (
            create_complete_ensemble_lsm_visualization(
                probability_raster_path=ensemble_map_path,
                ensemble_oof_metrics=ensemble_oof_metrics,
            )
        )

    # 7. Optional publication-style primary Model E LSM visualization.
    # Retained separately so the manuscript can still show the actual primary
    # Model E map without confusing it with the A/E deployment ensemble.
    visualization_outputs: Dict[str, Any] = {}
    if Config.GENERATE_PRIMARY_LSM_VISUALIZATION:
        primary_scenario = Config.PRIMARY_VISUALIZATION_SCENARIO

        if primary_scenario not in Config.SCENARIO_ORDER:
            raise RuntimeError(
                "PRIMARY_VISUALIZATION_SCENARIO is not a valid A-F scenario: "
                f"{primary_scenario}"
            )

        if primary_scenario in map_paths:
            primary_map_path = Path(map_paths[primary_scenario])
        else:
            primary_map_path = expected_scenario_map_path(
                primary_scenario
            )

        if not primary_map_path.exists():
            raise RuntimeError(
                "Primary susceptibility visualization requested, but the "
                f"probability raster does not exist: {primary_map_path}\n"
                "Either set RUN_FINAL_DEPLOYMENT_TUNING=True and "
                "GENERATE_RASTER_MAPS=True, or first generate the Model E raster."
            )

        visualization_outputs = create_primary_lsm_visualization(
            raster_path=primary_map_path,
            scenario_name=primary_scenario,
        )

    # 8. QA / provenance metadata.
    write_qa_report(
        df=df,
        scenarios=scenarios,
        summary_df=summary_df,
        comparison_df=comparison_df,
        final_params=final_params,
        map_paths=map_paths,
    )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "Logistic Regression",
        "validation_design": {
            "outer": "five frozen contiguous spatial blocks from Step 6.4",
            "inner": (
                "four-block PredefinedSplit inside each outer-training set; "
                "one remaining spatial block held out per inner fold"
            ),
            "hyperparameter_selection_metric": "ROC-AUC",
            "reported_primary_performance": "pooled nested outer OOF ROC-AUC",
            "binary_threshold": Config.BINARY_THRESHOLD,
            "confidence_interval": (
                f"{int(Config.CI_LEVEL * 100)}% cluster bootstrap resampling whole spatial blocks"
            ),
            "cluster_bootstrap_iterations": Config.N_CLUSTER_BOOTSTRAP,
        },
        "input_dataset": str(Config.MODEL_READY_DATASET_CSV),
        "scenario_manifest_step61": str(Config.SCENARIO_MANIFEST_JSON),
        "scenario_manifest_step64": str(Config.STEP64_MODEL_CONFIG_JSON),
        "parameter_grid": Config.PARAM_GRID,
        "fixed_lr_parameters": Config.LR_FIXED_PARAMS,
        "lr_methodological_contract": {
            "solver": "liblinear",
            "regularization": "L2 / Ridge",
            "class_weight": "balanced",
            "numeric_preprocessing": (
                "StandardScaler fitted inside each training fold only"
            ),
            "categorical_preprocessing": (
                "OneHotEncoder(handle_unknown='ignore') fitted inside each "
                "training fold only"
            ),
            "tuned_hyperparameter": "C (inverse regularization strength)",
        },
        "execution_controls": {
            "run_final_deployment_tuning": Config.RUN_FINAL_DEPLOYMENT_TUNING,
            "generate_raster_maps": Config.GENERATE_RASTER_MAPS,
            "generate_primary_lsm_visualization": (
                Config.GENERATE_PRIMARY_LSM_VISUALIZATION
            ),
            "primary_visualization_scenario": (
                Config.PRIMARY_VISUALIZATION_SCENARIO
            ),
            "generate_AE_ensemble_lsm": Config.GENERATE_AE_ENSEMBLE_LSM,
            "ensemble_static_scenario": Config.ENSEMBLE_STATIC_SCENARIO,
            "ensemble_enhanced_scenario": Config.ENSEMBLE_ENHANCED_SCENARIO,
            "ensemble_enhanced_weight": Config.ENSEMBLE_ENHANCED_WEIGHT,
        },
        "scenarios": {
            scenario_name: {
                "label": Config.MODEL_SHORT_LABELS[scenario_name],
                "description": Config.MODEL_DESCRIPTIONS[scenario_name],
                "features": list(scenarios[scenario_name]["features"]),
            }
            for scenario_name in Config.SCENARIO_ORDER
        },
        "outputs": {
            "oof_predictions": str(Config.OOF_PREDICTIONS_CSV),
            "fold_metrics": str(Config.FOLD_METRICS_CSV),
            "summary_metrics": str(Config.SUMMARY_METRICS_CSV),
            "paired_auc_comparisons": str(Config.AUC_COMPARISONS_CSV),
            "outer_best_parameters": str(Config.OUTER_PARAMS_JSON),
            "final_best_parameters": str(Config.FINAL_PARAMS_JSON),
            "nested_grid_results": str(Config.INNER_SEARCH_CSV),
            "full_data_grid_results": str(Config.FINAL_SEARCH_CSV),
            "final_model_directory": str(Config.MODEL_DIR),
            "map_directory": str(Config.MAP_DIR),
            "visualization_directory": str(Config.VISUALIZATION_DIR),
            "primary_lsm_visualization": visualization_outputs,
            "ensemble_oof_predictions": str(
                Config.ENSEMBLE_OOF_PREDICTIONS_CSV
            ),
            "ensemble_oof_metrics": str(
                Config.ENSEMBLE_OOF_METRICS_JSON
            ),
            "ensemble_probability_raster": (
                str(ensemble_map_path)
                if ensemble_map_path is not None
                else None
            ),
            "ensemble_lsm_visualization": ensemble_visualization_outputs,
            "manuscript_performance_excel": str(
                Config.MANUSCRIPT_TABLES_XLSX
            ),
            "qa_report": str(Config.QA_REPORT_TXT),
        },
    }

    with open(Config.METADATA_JSON, "w", encoding="utf-8") as handle:
        json.dump(json_safe(metadata), handle, indent=2)

    # 9. Console summary.
    log("")
    log("=" * 96)
    log("NESTED SPATIAL OOF SUMMARY")
    log("=" * 96)
    for _, row in summary_df.iterrows():
        log(
            f"{row['model']}: AUC={row['oof_roc_auc']:.4f} "
            f"[{row['oof_auc_cluster_bootstrap_ci_low']:.4f}, "
            f"{row['oof_auc_cluster_bootstrap_ci_high']:.4f}], "
            f"F1={row['oof_f1_0p5']:.4f}, "
            f"Precision={row['oof_precision_0p5']:.4f}, "
            f"Recall={row['oof_recall_0p5']:.4f}"
        )

    log("")
    log("PAIRED AUC COMPARISONS")
    for _, row in comparison_df.iterrows():
        log(
            f"{row['comparison']}: ΔAUC={row['auc_difference_left_minus_right']:+.4f} "
            f"[{row['cluster_bootstrap_ci_low']:+.4f}, "
            f"{row['cluster_bootstrap_ci_high']:+.4f}]"
        )

    log("")
    log(
        "FIXED A30/E70 ENSEMBLE OOF: "
        f"AUC={ensemble_oof_metrics['roc_auc']:.4f} "
        f"[{ensemble_oof_metrics['auc_cluster_bootstrap_ci_low']:.4f}, "
        f"{ensemble_oof_metrics['auc_cluster_bootstrap_ci_high']:.4f}], "
        f"ΔAUC vs Model E="
        f"{ensemble_oof_metrics['auc_difference_ensemble_minus_model_E']:+.4f}"
    )

    if Config.GENERATE_AE_ENSEMBLE_LSM and ensemble_map_path is not None:
        log(
            "[OUT-ENSEMBLE] Old-style full-background ensemble LSM saved to: "
            f"{Config.VISUALIZATION_DIR}"
        )

    log("")
    if Config.GENERATE_PRIMARY_LSM_VISUALIZATION:
        log(
            "[OUT-VIZ] Primary Model E Landslide Susceptibility Map and "
            f"histogram saved to: {Config.VISUALIZATION_DIR}"
        )
    log(
        "[OUT-EXCEL] Manuscript A-F numerical tables: "
        f"{Config.MANUSCRIPT_TABLES_XLSX}"
    )
    log(f"[OUT] Results saved to: {Config.OUT_DIR}")
    log("=== Step 7.3 completed successfully ===")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 7.3 failed: {exc}")
        traceback.print_exc()
        raise


# STEP 7.3A — LOGISTIC REGRESSION SPATIAL LEARNING CURVES

In [ ]:
# =============================================================================
# STEP 7.3A — LOGISTIC REGRESSION SPATIAL LEARNING CURVES
#              + HELD-OUT PERMUTATION IMPORTANCE
# =============================================================================
#
# Scientific purpose
# ------------------
# This script is the Logistic Regression counterpart to Step 7.1A (XGBoost) and Step 7.2A (Random Forest).
# It does NOT change the primary Step 7.3 nested spatial-CV evaluation.
#
# It adds two reviewer-oriented robustness diagnostics using the EXACT successful
# Step 7.3 Logistic Regression modelling contract:
#
# 1. Fixed-hyperparameter spatial learning curves for Model A and Model E only.
#    - The original outer validation block remains untouched.
#    - Training fractions are sampled only from the four outer-training blocks.
#    - Every reduced training set retains all four spatial blocks and both classes.
#    - The exact fold-specific hyperparameters selected during successful Step 7.3
#      are reused at all training sizes.
#    - No second hyperparameter search is performed.
#    - The held-out outer block remains completely unseen during fitting.
#
# 2. Held-out permutation importance for Model E.
#    - The LR model is fitted on four spatial blocks.
#    - Importance is calculated only on the untouched fifth block.
#    - Individual predictor importance is complemented by a grouped joint
#      permutation of HKFI + SNDR.
#
# Primary provenance
# ------------------
# This script reads the successful Step 7.3 outputs directly from:
#
#   ./data/Results/Step-7.3-LR-Models_A_to_F/
#
# Required Step 7.3 outputs:
#   LR_A_to_F_outer_fold_best_parameters.json
#   LR_A_to_F_OOF_predictions.csv
#
# Required upstream inputs:
#   Step 6.1 model_ablation_scenarios.json
#   Step 6.4 model_configuration_manifest.json
#   Step 6.4 model_ready_dataset_A_to_F.csv
#
# Scientific interpretation
# -------------------------
# Learning curves are a sample-size robustness diagnostic. Their uncertainty
# bands describe variability across the five held-out spatial folds; they are
# not paired confidence intervals for Model E - Model A.
#
# Permutation importance is conditional on the fitted Model E predictor set.
# Near-zero individual importance does not prove that a predictor has no value,
# particularly when predictors are correlated or redundant. The A-F ablation
# analysis remains the stronger retraining-based test of incremental information.
#
# For Logistic Regression, coefficients and held-out permutation importance
# answer different questions. Step 7.3A intentionally preserves the same held-
# out permutation framework used for XGB and RF so that algorithm robustness can
# be compared on a common diagnostic basis.
# =============================================================================

from __future__ import annotations

import gc
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Sequence

# Resource-safe execution. The successful Step 7.3 LR workflow also keeps the
# estimator single-threaded to avoid nested parallelism.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# =============================================================================
# CONFIGURATION — FROZEN TO THE SUCCESSFUL STEP 7.2 LR EXPERIMENT
# =============================================================================
class DiagnosticConfig:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Frozen Step 6.1 and Step 6.4 modelling definitions.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"

    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    MODEL_READY_DATASET_CSV = STEP64_DIR / "model_ready_dataset_A_to_F.csv"
    STEP64_MODEL_CONFIG_JSON = STEP64_DIR / "model_configuration_manifest.json"

    # Successful Step 7.3 Logistic Regression outputs.
    STEP73_OUT_DIR = RESULTS_DIR / "Step-7.3-LR-Models_A_to_F"
    OUTER_PARAMS_JSON = (
        STEP73_OUT_DIR / "LR_A_to_F_outer_fold_best_parameters.json"
    )
    PRIMARY_OOF_CSV = (
        STEP73_OUT_DIR / "LR_A_to_F_OOF_predictions.csv"
    )
    STEP73_METADATA_JSON = (
        STEP73_OUT_DIR / "Step_7_3_LR_A_to_F_Metadata.json"
    )

    # Step 7.3A outputs.
    OUT_DIR = STEP73_OUT_DIR / "Step-7.3A-Learning_and_Permutation"

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    # Same two learning-curve models used in Step 7.1A.
    LEARNING_SCENARIOS = (
        "A_static_baseline",
        "E_physics_guided",
    )

    # Same permutation-importance target used in Step 7.1A.
    PERMUTATION_SCENARIO = "E_physics_guided"

    # Same training fractions/repeats used in Step 7.1A.
    TRAIN_FRACTIONS = (0.25, 0.50, 0.75, 1.00)
    N_REPEATS_SUBSAMPLE = 20
    N_PERMUTATION_REPEATS = 30

    # Random state intentionally differs from XGB Step 7.1A while remaining fixed.
    RANDOM_STATE = 62071

    # The Logistic Regression estimator is reconstructed using the exact fixed
    # settings used in successful Step 7.3. The fold-specific tuned C value is
    # then applied from the saved Step 7.3 outer-parameter JSON.
    LR_FIXED_PARAMS = {
        "penalty": "l2",
        "solver": "liblinear",
        "class_weight": "balanced",
        "fit_intercept": True,
        "max_iter": 10000,
        "tol": 1.0e-6,
        "random_state": 42,
    }

    CATEGORICAL_FEATURE_ALIASES = {
        "landcover",
        "landuse",
        "land_use",
    }

    FIGSIZE = (5.5, 4.2)
    FIG_DPI = 900

    # Full-training reproduction check against successful Step 7.3 OOF results.
    OOF_REPRODUCTION_TOLERANCE = 1.0e-10


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(
        f"[{datetime.now():%H:%M:%S}] {message}",
        flush=True,
    )


def utc_now_iso() -> str:
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def require_file(
    path: Path,
    description: str,
) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}"
        )


def load_json(
    path: Path,
) -> Dict[str, Any]:
    require_file(
        path,
        "JSON file",
    )
    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def json_safe(
    value: Any,
) -> Any:
    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()
    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()
    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }
    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe(item)
            for item in value
        ]
    return value


def normalize_feature_name(
    name: str,
) -> str:
    import re

    return re.sub(
        r"[^a-z0-9_]+",
        "_",
        str(name).strip().lower(),
    ).strip(
        "_"
    )


def is_categorical_feature(
    name: str,
) -> bool:
    normalized = normalize_feature_name(
        name
    )
    compact = normalized.replace(
        "_",
        "",
    )

    aliases = {
        alias.lower()
        for alias in DiagnosticConfig.CATEGORICAL_FEATURE_ALIASES
    }
    compact_aliases = {
        alias.replace(
            "_",
            "",
        )
        for alias in aliases
    }

    return (
        normalized in aliases
        or compact in compact_aliases
    )


# =============================================================================
# STEP 6.1 / STEP 6.4 PROVENANCE AND DATASET VALIDATION
# =============================================================================
def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """
    Cross-check the Step 6.1 predictor definitions against the frozen Step 6.4
    model configuration exactly as the successful Step 7.3 LR workflow does.
    """
    step61 = load_json(
        DiagnosticConfig.SCENARIO_MANIFEST_JSON
    ).get(
        "scenarios",
        {},
    )

    step64 = load_json(
        DiagnosticConfig.STEP64_MODEL_CONFIG_JSON
    ).get(
        "scenarios",
        {},
    )

    for scenario_name in DiagnosticConfig.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                "Step 6.1 scenario manifest is missing "
                f"{scenario_name}."
            )

        if scenario_name not in step64:
            raise RuntimeError(
                "Step 6.4 frozen model manifest is missing "
                f"{scenario_name}."
            )

        features_61 = list(
            step61[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        features_64 = list(
            step64[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any(
            "irs" in feature.lower()
            for feature in features_61
        ):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; "
                "IRS must not be an ML predictor."
            )

    log(
        "[QA] Step 6.1 and Step 6.4 A-F scenario definitions match: PASS"
    )

    return step64


def all_scenario_features(
    scenarios: Dict[str, Dict[str, Any]],
) -> list[str]:
    features: list[str] = []

    for scenario_name in DiagnosticConfig.SCENARIO_ORDER:
        for feature in scenarios[
            scenario_name
        ][
            "features"
        ]:
            if feature not in features:
                features.append(
                    feature
                )

    return features


def load_and_validate_model_ready_dataset(
    scenarios: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    """
    Load the exact frozen Step 6.4 330-row dataset used by successful Step 7.3.
    """
    require_file(
        DiagnosticConfig.MODEL_READY_DATASET_CSV,
        "Step 6.4 primary frozen model-ready dataset",
    )

    df = pd.read_csv(
        DiagnosticConfig.MODEL_READY_DATASET_CSV
    )

    predictor_columns = all_scenario_features(
        scenarios
    )

    required = [
        DiagnosticConfig.SAMPLE_ID_COLUMN,
        DiagnosticConfig.TARGET_COLUMN,
        DiagnosticConfig.FOLD_COLUMN,
        DiagnosticConfig.BLOCK_COLUMN,
        *predictor_columns,
    ]

    missing = [
        column
        for column in required
        if column not in df.columns
    ]

    if missing:
        raise RuntimeError(
            "Model-ready dataset is missing columns: "
            f"{missing}"
        )

    if len(
        df
    ) != DiagnosticConfig.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Expected {DiagnosticConfig.EXPECTED_N_ROWS} rows; "
            f"found {len(df)}."
        )

    if df[
        DiagnosticConfig.SAMPLE_ID_COLUMN
    ].isna().any():
        raise RuntimeError(
            "Missing sample_id values found."
        )

    if df[
        DiagnosticConfig.SAMPLE_ID_COLUMN
    ].duplicated().any():
        raise RuntimeError(
            "Duplicated sample_id values found."
        )

    for column in (
        DiagnosticConfig.TARGET_COLUMN,
        DiagnosticConfig.FOLD_COLUMN,
        DiagnosticConfig.BLOCK_COLUMN,
    ):
        df[
            column
        ] = pd.to_numeric(
            df[
                column
            ],
            errors="raise",
        ).astype(
            int
        )

    class_counts = (
        df[
            DiagnosticConfig.TARGET_COLUMN
        ]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    expected_class_counts = {
        0: DiagnosticConfig.EXPECTED_N_NEGATIVE,
        1: DiagnosticConfig.EXPECTED_N_POSITIVE,
    }

    if class_counts != expected_class_counts:
        raise RuntimeError(
            f"Unexpected class counts: {class_counts}; "
            f"expected {expected_class_counts}."
        )

    folds = sorted(
        df[
            DiagnosticConfig.FOLD_COLUMN
        ].unique().tolist()
    )

    if folds != DiagnosticConfig.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected frozen folds "
            f"{DiagnosticConfig.EXPECTED_FOLDS}; "
            f"found {folds}."
        )

    if not np.array_equal(
        df[
            DiagnosticConfig.FOLD_COLUMN
        ].to_numpy(),
        df[
            DiagnosticConfig.BLOCK_COLUMN
        ].to_numpy(),
    ):
        raise RuntimeError(
            "cv_fold and spatial_block are not identical."
        )

    block_class = pd.crosstab(
        df[
            DiagnosticConfig.FOLD_COLUMN
        ],
        df[
            DiagnosticConfig.TARGET_COLUMN
        ],
    )

    expected_per_class = (
        DiagnosticConfig.EXPECTED_N_POSITIVE
        // len(
            DiagnosticConfig.EXPECTED_FOLDS
        )
    )

    for fold in DiagnosticConfig.EXPECTED_FOLDS:
        if fold not in block_class.index:
            raise RuntimeError(
                f"Missing fold {fold}."
            )

        for label in (
            0,
            1,
        ):
            count = int(
                block_class.loc[
                    fold,
                    label,
                ]
            )

            if count != expected_per_class:
                raise RuntimeError(
                    f"Fold {fold}, class {label}: "
                    f"found {count}; expected {expected_per_class}."
                )

    predictor_values = df[
        predictor_columns
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    finite = np.isfinite(
        predictor_values.to_numpy(
            dtype="float64"
        )
    )

    if not finite.all():
        bad = np.argwhere(
            ~finite
        )

        examples = [
            (
                str(
                    df.iloc[
                        row_index
                    ][
                        DiagnosticConfig.SAMPLE_ID_COLUMN
                    ]
                ),
                predictor_columns[
                    column_index
                ],
            )
            for row_index, column_index in bad[
                :20
            ]
        ]

        raise RuntimeError(
            "Missing/non-finite predictors remain. "
            f"Examples: {examples}"
        )

    irs_columns = [
        column
        for column in df.columns
        if "irs" in str(
            column
        ).lower()
    ]

    if irs_columns:
        raise RuntimeError(
            "IRS-related columns found in ML dataset: "
            f"{irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError(
            "tpi_s25 found; only tpi_s5 is allowed."
        )

    log(
        "[QA] Frozen Step 6.4 LR modelling dataset: "
        f"rows={len(df)}, "
        f"positive={class_counts[1]}, "
        f"negative={class_counts[0]}, "
        f"folds={folds}: PASS"
    )

    return df


# =============================================================================
# EXACT STEP 7.2 LOGISTIC REGRESSION PIPELINE
# =============================================================================
def build_lr_pipeline(
    features: Sequence[str],
) -> Pipeline:
    """
    Reconstruct the exact fold-safe preprocessing + Logistic Regression pipeline
    used in successful Step 7.3.

    Numeric predictors are standardized with StandardScaler fitted only on the
    current training partition. Land cover is one-hot encoded inside the same
    pipeline. Thus preprocessing remains leakage-safe during every learning-
    curve fit and held-out permutation-importance fit.
    """
    categorical = [
        feature
        for feature in features
        if is_categorical_feature(
            feature
        )
    ]

    numeric = [
        feature
        for feature in features
        if not is_categorical_feature(
            feature
        )
    ]

    transformers = []

    if numeric:
        transformers.append(
            (
                "numeric",
                StandardScaler(),
                numeric,
            )
        )

    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = LogisticRegression(
        **DiagnosticConfig.LR_FIXED_PARAMS
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                classifier,
            ),
        ]
    )


def set_pipeline_params(
    pipeline: Pipeline,
    params: Dict[str, Any],
) -> Pipeline:
    """
    Apply the exact fold-specific Step 7.3 GridSearchCV best parameters.
    """
    if not isinstance(
        params,
        dict,
    ):
        raise TypeError(
            "Fold-specific best parameters must be a dictionary."
        )

    pipeline.set_params(
        **params
    )

    return pipeline


# =============================================================================
# SUCCESSFUL STEP 7.2 OUTPUT LOADING
# =============================================================================
def load_outer_best_params() -> Dict[str, Any]:
    require_file(
        DiagnosticConfig.OUTER_PARAMS_JSON,
        "successful Step 7.3 LR outer-fold best-parameter JSON",
    )

    with open(
        DiagnosticConfig.OUTER_PARAMS_JSON,
        "r",
        encoding="utf-8",
    ) as handle:
        outer_params = json.load(
            handle
        )

    for scenario_name in DiagnosticConfig.SCENARIO_ORDER:
        if scenario_name not in outer_params:
            raise RuntimeError(
                "Step 7.3 outer-parameter JSON is missing scenario: "
                f"{scenario_name}"
            )

        for outer_fold in DiagnosticConfig.EXPECTED_FOLDS:
            fold_info = outer_params[
                scenario_name
            ].get(
                str(
                    outer_fold
                )
            )

            if fold_info is None:
                raise RuntimeError(
                    f"Missing Step 7.3 outer-fold parameter record: "
                    f"{scenario_name}, fold {outer_fold}"
                )

            if "best_params" not in fold_info:
                raise RuntimeError(
                    f"Missing best_params: "
                    f"{scenario_name}, fold {outer_fold}"
                )

    log(
        "[QA] Successful Step 7.3 LR outer-fold best parameters loaded: PASS"
    )

    return outer_params


def get_fold_params(
    outer_params: Dict[str, Any],
    scenario_name: str,
    outer_fold: int,
) -> Dict[str, Any]:
    if scenario_name not in outer_params:
        raise KeyError(
            "Missing scenario in Step 7.3 outer-parameter JSON: "
            f"{scenario_name}"
        )

    fold_info = outer_params[
        scenario_name
    ].get(
        str(
            outer_fold
        )
    )

    if not fold_info:
        raise KeyError(
            "Missing outer-fold parameters for "
            f"{scenario_name}, fold {outer_fold}"
        )

    params = fold_info.get(
        "best_params"
    )

    if not params:
        raise KeyError(
            "Missing best_params for "
            f"{scenario_name}, fold {outer_fold}"
        )

    return params


def load_primary_step73_oof() -> pd.DataFrame:
    require_file(
        DiagnosticConfig.PRIMARY_OOF_CSV,
        "successful Step 7.3 LR A-F OOF predictions",
    )

    oof = pd.read_csv(
        DiagnosticConfig.PRIMARY_OOF_CSV
    )

    required = {
        "scenario",
        DiagnosticConfig.SAMPLE_ID_COLUMN,
        DiagnosticConfig.TARGET_COLUMN,
        DiagnosticConfig.FOLD_COLUMN,
        "oof_probability",
    }

    missing = required - set(
        oof.columns
    )

    if missing:
        raise RuntimeError(
            "Step 7.3 OOF file is missing columns: "
            f"{sorted(missing)}"
        )

    for scenario_name in DiagnosticConfig.LEARNING_SCENARIOS:
        subset = oof.loc[
            oof[
                "scenario"
            ]
            == scenario_name
        ]

        if len(
            subset
        ) != DiagnosticConfig.EXPECTED_N_ROWS:
            raise RuntimeError(
                f"Unexpected Step 7.3 OOF row count for {scenario_name}: "
                f"{len(subset)}"
            )

    return oof


# =============================================================================
# PROVENANCE
# =============================================================================
def write_diagnostic_provenance(
    out_dir: Path,
) -> None:
    metadata = {
        "generated_at_utc": utc_now_iso(),
        "diagnostic_step": "Step 7.3A",
        "algorithm": "Logistic Regression",
        "scientific_role": (
            "Sample-size robustness and held-out permutation-importance "
            "diagnostics; does not replace primary Step 7.3 nested spatial CV."
        ),
        "primary_step73_output_directory": str(
            DiagnosticConfig.STEP73_OUT_DIR
        ),
        "outer_fold_best_parameters": str(
            DiagnosticConfig.OUTER_PARAMS_JSON
        ),
        "primary_oof_predictions": str(
            DiagnosticConfig.PRIMARY_OOF_CSV
        ),
        "model_ready_dataset": str(
            DiagnosticConfig.MODEL_READY_DATASET_CSV
        ),
        "learning_curve_type": (
            "fixed-hyperparameter spatial learning curve"
        ),
        "learning_scenarios": list(
            DiagnosticConfig.LEARNING_SCENARIOS
        ),
        "permutation_scenario": (
            DiagnosticConfig.PERMUTATION_SCENARIO
        ),
        "training_fractions": list(
            DiagnosticConfig.TRAIN_FRACTIONS
        ),
        "subsample_repeats": int(
            DiagnosticConfig.N_REPEATS_SUBSAMPLE
        ),
        "permutation_repeats": int(
            DiagnosticConfig.N_PERMUTATION_REPEATS
        ),
        "lr_fixed_parameters": json_safe(
            DiagnosticConfig.LR_FIXED_PARAMS
        ),
        "lr_preprocessing": {
            "numeric": (
                "StandardScaler fitted only on each current training partition"
            ),
            "categorical": (
                "OneHotEncoder(handle_unknown='ignore') fitted inside pipeline"
            ),
        },
        "full_training_reproduction_check": (
            "At training_fraction=1.0, held-out probabilities for Models A and E "
            "must reproduce the successful Step 7.3 OOF predictions."
        ),
    }

    with open(
        out_dir
        / "Step_7_3A_LR_Diagnostic_Provenance.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            metadata,
            handle,
            indent=2,
        )


# =============================================================================
# SPATIAL STRATIFIED SUBSAMPLING
# =============================================================================
def spatial_stratified_subsample(
    train_df: pd.DataFrame,
    fold_column: str,
    target_column: str,
    fraction: float,
    seed: int,
) -> pd.DataFrame:
    """
    Sample the requested fraction independently within every
    spatial-block x class stratum.

    With 33 observations per class in each spatial block, the Step 7.1A/7.3A
    fractions produce approximately:
        0.25 -> 64 samples
        0.50 -> 128 samples
        0.75 -> 200 samples
        1.00 -> 264 samples

    This preserves all four outer-training blocks and both classes.
    """
    if not 0.0 < float(
        fraction
    ) <= 1.0:
        raise ValueError(
            "Training fraction must lie in (0, 1]."
        )

    if np.isclose(
        fraction,
        1.0,
    ):
        return train_df.copy()

    rng = np.random.default_rng(
        seed
    )

    selected_indices: list[Any] = []

    grouped = train_df.groupby(
        [
            fold_column,
            target_column,
        ],
        sort=True,
    )

    for (
        _fold,
        _label,
    ), group in grouped:
        n_group = len(
            group
        )

        n_select = max(
            1,
            int(
                round(
                    n_group
                    * float(
                        fraction
                    )
                )
            ),
        )

        n_select = min(
            n_select,
            n_group,
        )

        group_indices = group.index.to_numpy()

        chosen = rng.choice(
            group_indices,
            size=n_select,
            replace=False,
        )

        selected_indices.extend(
            chosen.tolist()
        )

    sampled = (
        train_df.loc[
            selected_indices
        ]
        .copy()
        .sort_index()
    )

    return sampled


# =============================================================================
# FEATURE HELPERS
# =============================================================================
def find_feature(
    features: list[str],
    token: str,
) -> str:
    matches = [
        feature
        for feature in features
        if token.lower()
        in feature.lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Expected exactly one feature containing '{token}', "
            f"found: {matches}"
        )

    return matches[
        0
    ]


# =============================================================================
# FULL-TRAINING REPRODUCTION QA
# =============================================================================
def compare_full_training_to_primary_oof(
    full_training_predictions: pd.DataFrame,
    primary_oof: pd.DataFrame,
    out_dir: Path,
) -> pd.DataFrame:
    """
    At training_fraction=1.0, each fold-specific LR model is reconstructed from
    the exact Step 7.3 saved best parameters and trained on the same four blocks.
    Its held-out probabilities should therefore reproduce the successful Step 7.3
    OOF probabilities for Models A and E.
    """
    rows: list[dict[str, Any]] = []

    for scenario_name in DiagnosticConfig.LEARNING_SCENARIOS:
        diagnostic = full_training_predictions.loc[
            full_training_predictions[
                "scenario"
            ]
            == scenario_name
        ].copy()

        reference = primary_oof.loc[
            primary_oof[
                "scenario"
            ]
            == scenario_name,
            [
                DiagnosticConfig.SAMPLE_ID_COLUMN,
                "oof_probability",
            ],
        ].copy()

        reference = reference.rename(
            columns={
                "oof_probability": (
                    "primary_step73_oof_probability"
                )
            }
        )

        merged = diagnostic.merge(
            reference,
            on=DiagnosticConfig.SAMPLE_ID_COLUMN,
            how="inner",
            validate="one_to_one",
        )

        if len(
            merged
        ) != DiagnosticConfig.EXPECTED_N_ROWS:
            raise RuntimeError(
                "Full-training reproduction merge returned "
                f"{len(merged)} rows for {scenario_name}; "
                f"expected {DiagnosticConfig.EXPECTED_N_ROWS}."
            )

        difference = (
            merged[
                "diagnostic_probability"
            ].to_numpy(
                dtype=float
            )
            - merged[
                "primary_step73_oof_probability"
            ].to_numpy(
                dtype=float
            )
        )

        max_abs = float(
            np.max(
                np.abs(
                    difference
                )
            )
        )

        mean_abs = float(
            np.mean(
                np.abs(
                    difference
                )
            )
        )

        passed = bool(
            max_abs
            <= DiagnosticConfig.OOF_REPRODUCTION_TOLERANCE
        )

        rows.append(
            {
                "scenario": scenario_name,
                "model": DiagnosticConfig.MODEL_SHORT_LABELS[
                    scenario_name
                ],
                "n_samples_compared": int(
                    len(
                        merged
                    )
                ),
                "maximum_absolute_probability_difference": max_abs,
                "mean_absolute_probability_difference": mean_abs,
                "tolerance": float(
                    DiagnosticConfig.OOF_REPRODUCTION_TOLERANCE
                ),
                "reproduction_pass": passed,
            }
        )

        if not passed:
            raise RuntimeError(
                "Step 7.3A full-training reproduction check failed for "
                f"{scenario_name}.\n"
                f"Maximum absolute probability difference = {max_abs:.12g}\n"
                f"Tolerance = "
                f"{DiagnosticConfig.OOF_REPRODUCTION_TOLERANCE:.12g}\n"
                "This indicates that the Step 7.3A LR pipeline or saved "
                "fold-specific parameters do not exactly match the successful "
                "Step 7.3 experiment."
            )

    qa_df = pd.DataFrame(
        rows
    )

    qa_df.to_csv(
        out_dir
        / "LR_7_3A_Full_Training_Reproduction_QA.csv",
        index=False,
        float_format="%.12g",
    )

    log(
        "[QA] Step 7.3A full-training A/E held-out probabilities reproduce "
        "successful Step 7.3 OOF predictions: PASS"
    )

    return qa_df


# =============================================================================
# 1. SPATIAL LEARNING CURVES
# =============================================================================
def run_spatial_learning_curves(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
    outer_params: Dict[str, Any],
    primary_oof: pd.DataFrame,
    out_dir: Path,
):
    rows: list[dict[str, Any]] = []
    full_training_oof_rows: list[dict[str, Any]] = []

    for scenario_name in DiagnosticConfig.LEARNING_SCENARIOS:
        features = list(
            scenarios[
                scenario_name
            ][
                "features"
            ]
        )

        for outer_fold in DiagnosticConfig.EXPECTED_FOLDS:
            train_df = df.loc[
                df[
                    DiagnosticConfig.FOLD_COLUMN
                ]
                != outer_fold
            ].copy()

            valid_df = df.loc[
                df[
                    DiagnosticConfig.FOLD_COLUMN
                ]
                == outer_fold
            ].copy()

            if train_df[
                DiagnosticConfig.FOLD_COLUMN
            ].nunique() != 4:
                raise RuntimeError(
                    f"{scenario_name}, outer fold {outer_fold}: "
                    "expected four outer-training spatial blocks."
                )

            if valid_df[
                DiagnosticConfig.FOLD_COLUMN
            ].nunique() != 1:
                raise RuntimeError(
                    f"{scenario_name}, outer fold {outer_fold}: "
                    "validation data must contain exactly one spatial block."
                )

            best_params = get_fold_params(
                outer_params=outer_params,
                scenario_name=scenario_name,
                outer_fold=outer_fold,
            )

            X_valid = valid_df[
                features
            ]
            y_valid = valid_df[
                DiagnosticConfig.TARGET_COLUMN
            ].to_numpy(
                dtype=int
            )

            for (
                fraction_index,
                fraction,
            ) in enumerate(
                DiagnosticConfig.TRAIN_FRACTIONS
            ):
                n_repeats = (
                    1
                    if np.isclose(
                        fraction,
                        1.0,
                    )
                    else DiagnosticConfig.N_REPEATS_SUBSAMPLE
                )

                for repeat in range(
                    n_repeats
                ):
                    seed = (
                        DiagnosticConfig.RANDOM_STATE
                        + 10000
                        * DiagnosticConfig.SCENARIO_ORDER.index(
                            scenario_name
                        )
                        + 1000
                        * outer_fold
                        + 100
                        * fraction_index
                        + repeat
                    )

                    sampled_train = spatial_stratified_subsample(
                        train_df=train_df,
                        fold_column=DiagnosticConfig.FOLD_COLUMN,
                        target_column=DiagnosticConfig.TARGET_COLUMN,
                        fraction=float(
                            fraction
                        ),
                        seed=seed,
                    )

                    # Strict spatial QA.
                    if sampled_train[
                        DiagnosticConfig.FOLD_COLUMN
                    ].nunique() != 4:
                        raise RuntimeError(
                            "Learning-curve sample lost a training "
                            "spatial block: "
                            f"{scenario_name}, outer fold {outer_fold}, "
                            f"fraction {fraction}"
                        )

                    if sampled_train[
                        DiagnosticConfig.TARGET_COLUMN
                    ].nunique() != 2:
                        raise RuntimeError(
                            "Learning-curve sample lost one response class."
                        )

                    block_class = pd.crosstab(
                        sampled_train[
                            DiagnosticConfig.FOLD_COLUMN
                        ],
                        sampled_train[
                            DiagnosticConfig.TARGET_COLUMN
                        ],
                    )

                    if set(
                        block_class.columns
                    ) != {
                        0,
                        1,
                    }:
                        raise RuntimeError(
                            "Learning-curve subsample does not contain "
                            "both classes in every retained block."
                        )

                    pipeline = build_lr_pipeline(
                        features
                    )

                    set_pipeline_params(
                        pipeline,
                        best_params,
                    )

                    X_train = sampled_train[
                        features
                    ]

                    y_train = sampled_train[
                        DiagnosticConfig.TARGET_COLUMN
                    ].to_numpy(
                        dtype=int
                    )

                    pipeline.fit(
                        X_train,
                        y_train,
                    )

                    train_prob = (
                        pipeline.predict_proba(
                            X_train
                        )[
                            :,
                            1,
                        ]
                    )

                    valid_prob = (
                        pipeline.predict_proba(
                            X_valid
                        )[
                            :,
                            1,
                        ]
                    )

                    rows.append(
                        {
                            "algorithm": "Logistic Regression",
                            "scenario": scenario_name,
                            "model": DiagnosticConfig.MODEL_SHORT_LABELS[
                                scenario_name
                            ],
                            "outer_fold": int(
                                outer_fold
                            ),
                            "training_fraction": float(
                                fraction
                            ),
                            "repeat": int(
                                repeat
                            ),
                            "n_train": int(
                                len(
                                    sampled_train
                                )
                            ),
                            "n_train_positive": int(
                                y_train.sum()
                            ),
                            "n_train_negative": int(
                                len(
                                    y_train
                                )
                                - y_train.sum()
                            ),
                            "training_auc": float(
                                roc_auc_score(
                                    y_train,
                                    train_prob,
                                )
                            ),
                            "training_pr_auc": float(
                                average_precision_score(
                                    y_train,
                                    train_prob,
                                )
                            ),
                            "heldout_spatial_auc": float(
                                roc_auc_score(
                                    y_valid,
                                    valid_prob,
                                )
                            ),
                            "heldout_spatial_pr_auc": float(
                                average_precision_score(
                                    y_valid,
                                    valid_prob,
                                )
                            ),
                        }
                    )

                    # Save exact full-training held-out probabilities for
                    # reproducibility QA against successful Step 7.3.
                    if np.isclose(
                        fraction,
                        1.0,
                    ):
                        for (
                            sample_id,
                            probability,
                        ) in zip(
                            valid_df[
                                DiagnosticConfig.SAMPLE_ID_COLUMN
                            ].tolist(),
                            valid_prob.tolist(),
                        ):
                            full_training_oof_rows.append(
                                {
                                    "scenario": scenario_name,
                                    "outer_fold": int(
                                        outer_fold
                                    ),
                                    DiagnosticConfig.SAMPLE_ID_COLUMN: (
                                        sample_id
                                    ),
                                    "diagnostic_probability": float(
                                        probability
                                    ),
                                }
                            )

                    del (
                        pipeline,
                        sampled_train,
                        X_train,
                        y_train,
                        train_prob,
                        valid_prob,
                    )
                    gc.collect()

    raw_df = pd.DataFrame(
        rows
    )

    raw_df.to_csv(
        out_dir
        / "LR_7_3A_Spatial_Learning_Curve_Raw.csv",
        index=False,
        float_format="%.10f",
    )

    # Average repeated random subsamples within each held-out spatial fold first.
    # The five fold means are then the independent spatial units summarized.
    fold_summary = (
        raw_df.groupby(
            [
                "scenario",
                "model",
                "outer_fold",
                "training_fraction",
            ],
            as_index=False,
        )
        .agg(
            n_train_mean=(
                "n_train",
                "mean",
            ),
            training_auc_mean=(
                "training_auc",
                "mean",
            ),
            training_pr_auc_mean=(
                "training_pr_auc",
                "mean",
            ),
            heldout_spatial_auc_mean=(
                "heldout_spatial_auc",
                "mean",
            ),
            heldout_spatial_pr_auc_mean=(
                "heldout_spatial_pr_auc",
                "mean",
            ),
        )
    )

    fold_summary.to_csv(
        out_dir
        / "LR_7_3A_Spatial_Learning_Curve_Fold_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    summary = (
        fold_summary.groupby(
            [
                "scenario",
                "model",
                "training_fraction",
            ],
            as_index=False,
        )
        .agg(
            n_train_mean=(
                "n_train_mean",
                "mean",
            ),
            training_auc_mean=(
                "training_auc_mean",
                "mean",
            ),
            training_auc_sd=(
                "training_auc_mean",
                "std",
            ),
            training_pr_auc_mean=(
                "training_pr_auc_mean",
                "mean",
            ),
            training_pr_auc_sd=(
                "training_pr_auc_mean",
                "std",
            ),
            heldout_auc_mean=(
                "heldout_spatial_auc_mean",
                "mean",
            ),
            heldout_auc_sd=(
                "heldout_spatial_auc_mean",
                "std",
            ),
            heldout_auc_min=(
                "heldout_spatial_auc_mean",
                "min",
            ),
            heldout_auc_max=(
                "heldout_spatial_auc_mean",
                "max",
            ),
            heldout_pr_auc_mean=(
                "heldout_spatial_pr_auc_mean",
                "mean",
            ),
            heldout_pr_auc_sd=(
                "heldout_spatial_pr_auc_mean",
                "std",
            ),
            heldout_pr_auc_min=(
                "heldout_spatial_pr_auc_mean",
                "min",
            ),
            heldout_pr_auc_max=(
                "heldout_spatial_pr_auc_mean",
                "max",
            ),
            n_outer_folds=(
                "outer_fold",
                "nunique",
            ),
        )
    )

    summary.to_csv(
        out_dir
        / "LR_7_3A_Spatial_Learning_Curve_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    # -------------------------------------------------------------------------
    # Figure — same scientific visualization as Step 7.1A.
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=DiagnosticConfig.FIGSIZE
    )

    for scenario_name in DiagnosticConfig.LEARNING_SCENARIOS:
        subset = summary.loc[
            summary[
                "scenario"
            ]
            == scenario_name
        ].sort_values(
            "n_train_mean"
        )

        x = subset[
            "n_train_mean"
        ].to_numpy(
            dtype=float
        )

        y = subset[
            "heldout_auc_mean"
        ].to_numpy(
            dtype=float
        )

        sd = subset[
            "heldout_auc_sd"
        ].fillna(
            0.0
        ).to_numpy(
            dtype=float
        )

        label = DiagnosticConfig.MODEL_SHORT_LABELS[
            scenario_name
        ]

        ax.plot(
            x,
            y,
            marker="o",
            linewidth=1.5,
            label=label,
        )

        ax.fill_between(
            x,
            y - sd,
            y + sd,
            alpha=0.15,
        )

    ax.set_xlabel(
        "Number of outer-training samples"
    )
    ax.set_ylabel(
        "Held-out spatial ROC-AUC"
    )
    ax.set_ylim(
        0.5,
        1.01,
    )
    ax.grid(
        True,
        alpha=0.25,
    )
    ax.legend(
        frameon=False
    )
    fig.tight_layout()

    fig.savefig(
        out_dir
        / "LR_7_3A_Spatial_Learning_Curves.png",
        dpi=DiagnosticConfig.FIG_DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

    # -------------------------------------------------------------------------
    # Exact full-training reproduction QA.
    # -------------------------------------------------------------------------
    full_training_predictions = pd.DataFrame(
        full_training_oof_rows
    )

    full_training_predictions.to_csv(
        out_dir
        / "LR_7_3A_Full_Training_Heldout_Probabilities.csv",
        index=False,
        float_format="%.12g",
    )

    reproduction_qa = compare_full_training_to_primary_oof(
        full_training_predictions=full_training_predictions,
        primary_oof=primary_oof,
        out_dir=out_dir,
    )

    return (
        raw_df,
        fold_summary,
        summary,
        reproduction_qa,
    )


# =============================================================================
# 2. HELD-OUT PERMUTATION IMPORTANCE
# =============================================================================
def run_heldout_permutation_importance(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
    outer_params: Dict[str, Any],
    out_dir: Path,
):
    scenario_name = (
        DiagnosticConfig.PERMUTATION_SCENARIO
    )

    features = list(
        scenarios[
            scenario_name
        ][
            "features"
        ]
    )

    hkfi_feature = find_feature(
        features,
        "hkfi",
    )

    sndr_feature = find_feature(
        features,
        "sndr",
    )

    individual_rows: list[dict[str, Any]] = []
    grouped_rows: list[dict[str, Any]] = []

    for outer_fold in DiagnosticConfig.EXPECTED_FOLDS:
        train_df = df.loc[
            df[
                DiagnosticConfig.FOLD_COLUMN
            ]
            != outer_fold
        ].copy()

        valid_df = df.loc[
            df[
                DiagnosticConfig.FOLD_COLUMN
            ]
            == outer_fold
        ].copy()

        best_params = get_fold_params(
            outer_params=outer_params,
            scenario_name=scenario_name,
            outer_fold=outer_fold,
        )

        pipeline = build_lr_pipeline(
            features
        )

        set_pipeline_params(
            pipeline,
            best_params,
        )

        X_train = train_df[
            features
        ]

        y_train = train_df[
            DiagnosticConfig.TARGET_COLUMN
        ].to_numpy(
            dtype=int
        )

        X_valid = valid_df[
            features
        ]

        y_valid = valid_df[
            DiagnosticConfig.TARGET_COLUMN
        ].to_numpy(
            dtype=int
        )

        pipeline.fit(
            X_train,
            y_train,
        )

        baseline_prob = (
            pipeline.predict_proba(
                X_valid
            )[
                :,
                1,
            ]
        )

        baseline_auc = float(
            roc_auc_score(
                y_valid,
                baseline_prob,
            )
        )

        # ---------------------------------------------------------------------
        # Individual held-out permutation importance.
        #
        # sklearn.inspection.permutation_importance receives the full pipeline,
        # so original input columns are permuted before fold-fitted preprocessing.
        # ---------------------------------------------------------------------
        result = permutation_importance(
            estimator=pipeline,
            X=X_valid,
            y=y_valid,
            scoring="roc_auc",
            n_repeats=(
                DiagnosticConfig.N_PERMUTATION_REPEATS
            ),
            random_state=(
                DiagnosticConfig.RANDOM_STATE
                + outer_fold
            ),
            n_jobs=1,
        )

        for (
            feature_index,
            feature,
        ) in enumerate(
            features
        ):
            for (
                repeat,
                decrease,
            ) in enumerate(
                result.importances[
                    feature_index
                ]
            ):
                individual_rows.append(
                    {
                        "algorithm": "Logistic Regression",
                        "outer_fold": int(
                            outer_fold
                        ),
                        "feature": feature,
                        "repeat": int(
                            repeat
                        ),
                        "baseline_auc": baseline_auc,
                        "auc_decrease": float(
                            decrease
                        ),
                    }
                )

        # ---------------------------------------------------------------------
        # Joint HKFI + SNDR permutation.
        #
        # The same row permutation is applied to both variables, preserving
        # their internal pairing while breaking their alignment with the
        # held-out response and the remaining predictors.
        # ---------------------------------------------------------------------
        rng = np.random.default_rng(
            DiagnosticConfig.RANDOM_STATE
            + 1000
            + outer_fold
        )

        for repeat in range(
            DiagnosticConfig.N_PERMUTATION_REPEATS
        ):
            permutation = rng.permutation(
                len(
                    X_valid
                )
            )

            X_permuted = X_valid.copy()

            for feature in (
                hkfi_feature,
                sndr_feature,
            ):
                X_permuted[
                    feature
                ] = (
                    X_valid.iloc[
                        permutation
                    ][
                        feature
                    ]
                    .to_numpy()
                )

            permuted_prob = (
                pipeline.predict_proba(
                    X_permuted
                )[
                    :,
                    1,
                ]
            )

            permuted_auc = float(
                roc_auc_score(
                    y_valid,
                    permuted_prob,
                )
            )

            grouped_rows.append(
                {
                    "algorithm": "Logistic Regression",
                    "outer_fold": int(
                        outer_fold
                    ),
                    "feature_group": (
                        "HKFI + SNDR (joint permutation)"
                    ),
                    "repeat": int(
                        repeat
                    ),
                    "baseline_auc": baseline_auc,
                    "permuted_auc": permuted_auc,
                    "auc_decrease": (
                        baseline_auc
                        - permuted_auc
                    ),
                }
            )

        del (
            pipeline,
            result,
            train_df,
            valid_df,
            X_train,
            X_valid,
            baseline_prob,
        )
        gc.collect()

    individual = pd.DataFrame(
        individual_rows
    )

    grouped = pd.DataFrame(
        grouped_rows
    )

    individual.to_csv(
        out_dir
        / "LR_7_3A_Model_E_Heldout_Permutation_Importance_Raw.csv",
        index=False,
        float_format="%.10f",
    )

    grouped.to_csv(
        out_dir
        / "LR_7_3A_Model_E_Grouped_HKFI_SNDR_Permutation_Raw.csv",
        index=False,
        float_format="%.10f",
    )

    # Average permutation repeats within each outer fold first.
    individual_fold = (
        individual.groupby(
            [
                "outer_fold",
                "feature",
            ],
            as_index=False,
        )
        .agg(
            auc_decrease_mean=(
                "auc_decrease",
                "mean",
            )
        )
    )

    individual_fold.to_csv(
        out_dir
        / "LR_7_3A_Model_E_Heldout_Permutation_Importance_Fold_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    individual_summary = (
        individual_fold.groupby(
            "feature",
            as_index=False,
        )
        .agg(
            mean_auc_decrease=(
                "auc_decrease_mean",
                "mean",
            ),
            sd_across_outer_folds=(
                "auc_decrease_mean",
                "std",
            ),
            min_across_outer_folds=(
                "auc_decrease_mean",
                "min",
            ),
            max_across_outer_folds=(
                "auc_decrease_mean",
                "max",
            ),
        )
        .sort_values(
            "mean_auc_decrease",
            ascending=False,
        )
    )

    grouped_fold = (
        grouped.groupby(
            [
                "outer_fold",
                "feature_group",
            ],
            as_index=False,
        )
        .agg(
            auc_decrease_mean=(
                "auc_decrease",
                "mean",
            )
        )
    )

    grouped_fold.to_csv(
        out_dir
        / "LR_7_3A_Model_E_Grouped_HKFI_SNDR_Permutation_Fold_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    grouped_summary = (
        grouped_fold.groupby(
            "feature_group",
            as_index=False,
        )
        .agg(
            mean_auc_decrease=(
                "auc_decrease_mean",
                "mean",
            ),
            sd_across_outer_folds=(
                "auc_decrease_mean",
                "std",
            ),
            min_across_outer_folds=(
                "auc_decrease_mean",
                "min",
            ),
            max_across_outer_folds=(
                "auc_decrease_mean",
                "max",
            ),
        )
    )

    individual_summary.to_csv(
        out_dir
        / "LR_7_3A_Model_E_Heldout_Permutation_Importance_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    grouped_summary.to_csv(
        out_dir
        / "LR_7_3A_Model_E_Grouped_HKFI_SNDR_Permutation_Summary.csv",
        index=False,
        float_format="%.10f",
    )

    # -------------------------------------------------------------------------
    # Figure — same as Step 7.1A, now using LR held-out permutation results.
    # -------------------------------------------------------------------------
    plot_df = (
        individual_summary.head(
            15
        )
        .sort_values(
            "mean_auc_decrease"
        )
    )

    fig, ax = plt.subplots(
        figsize=(
            6.0,
            5.2,
        )
    )

    ax.barh(
        plot_df[
            "feature"
        ],
        plot_df[
            "mean_auc_decrease"
        ],
        xerr=plot_df[
            "sd_across_outer_folds"
        ].fillna(
            0.0
        ),
        capsize=2,
    )

    ax.axvline(
        0.0,
        linewidth=0.8,
    )

    ax.set_xlabel(
        "Decrease in held-out ROC-AUC after permutation"
    )

    ax.set_ylabel(
        "Predictor"
    )

    ax.grid(
        True,
        axis="x",
        alpha=0.25,
    )

    fig.tight_layout()

    fig.savefig(
        out_dir
        / "LR_7_3A_Model_E_Heldout_Permutation_Importance.png",
        dpi=DiagnosticConfig.FIG_DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

    return (
        individual_summary,
        grouped_summary,
    )


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    DiagnosticConfig.OUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    log(
        "=" * 96
    )
    log(
        "STEP 7.3A — LOGISTIC REGRESSION SPATIAL LEARNING CURVES "
        "+ HELD-OUT PERMUTATION IMPORTANCE"
    )
    log(
        "=" * 96
    )

    write_diagnostic_provenance(
        DiagnosticConfig.OUT_DIR
    )

    # 1. Validate frozen predictor definitions and dataset.
    scenarios = validate_scenario_manifests()

    df = load_and_validate_model_ready_dataset(
        scenarios
    )

    # 2. Load exact successful Step 7.3 LR fold-specific hyperparameters.
    outer_params = load_outer_best_params()

    # 3. Load successful Step 7.3 LR OOF predictions for reproduction QA.
    primary_oof = load_primary_step73_oof()

    # 4. Spatial learning curves.
    log(
        "[7.3A] Running LR spatial learning curves for Models A and E..."
    )

    (
        _,
        _,
        learning_summary,
        reproduction_qa,
    ) = run_spatial_learning_curves(
        df=df,
        scenarios=scenarios,
        outer_params=outer_params,
        primary_oof=primary_oof,
        out_dir=DiagnosticConfig.OUT_DIR,
    )

    # 5. Held-out permutation importance.
    log(
        "[7.3A] Running LR held-out permutation importance for Model E..."
    )

    (
        individual_summary,
        grouped_summary,
    ) = run_heldout_permutation_importance(
        df=df,
        scenarios=scenarios,
        outer_params=outer_params,
        out_dir=DiagnosticConfig.OUT_DIR,
    )

    log(
        ""
    )
    log(
        "=" * 96
    )
    log(
        "STEP 7.3A COMPLETED SUCCESSFULLY"
    )
    log(
        "=" * 96
    )
    log(
        f"[OUT] {DiagnosticConfig.OUT_DIR}"
    )

    print(
        "\nFull-training reproduction QA:"
    )
    print(
        reproduction_qa.to_string(
            index=False
        )
    )

    print(
        "\nLearning-curve summary:"
    )
    print(
        learning_summary.to_string(
            index=False
        )
    )

    print(
        "\nTop held-out permutation importance:"
    )
    print(
        individual_summary.head(
            10
        ).to_string(
            index=False
        )
    )

    print(
        "\nGrouped HKFI + SNDR importance:"
    )
    print(
        grouped_summary.to_string(
            index=False
        )
    )


if __name__ == "__main__":
    main()


# STEP 7.3B — LOGISTIC REGRESSION HKFI WEIGHTING SENSITIVITY

In [ ]:
# =============================================================================
# STEP 7.3B — LOGISTIC REGRESSION HKFI WEIGHTING SENSITIVITY
# =============================================================================
#
# Targeted Logistic Regression cross-algorithm counterpart to Steps 7.1B and 7.2B.
#
# Step 7.1B completed the full XGBoost HKFI weighting sensitivity and map-
# sensitivity analysis. Step 7.2B then tested the main conclusion under Random
# Forest using a focused four-scenario design. Step 7.3B applies that same
# focused design to the exact successful Step 7.3 Logistic Regression Model E
# nested spatial-CV contract to test whether the weighting-robustness conclusion
# is also supported by the linear regularized model.
#
# Scientific rules
# ----------------
# 1. The exact frozen Step 6.4 330-row dataset and five spatial folds are used.
# 2. Only HKFI changes between sensitivity scenarios.
# 3. The exact successful Step 7.3 LR regularization grid and fold-safe preprocessing are used.
# 4. The primary equal-component scenario must reproduce successful Step 7.3
#    Model E OOF predictions.
# 5. The 66-case Step 5.3 simplex is not propagated through ML.
# 6. HKFI weights are not selected post hoc by maximizing validation AUC.
# 7. Full-domain LR map sensitivity is deliberately not repeated because the
#    complete map-sensitivity experiment was already performed for the primary
#    XGBoost framework in Step 7.1B.
#
# Execution
# ---------
# Heavy scenarios run in separate Python processes and are checkpointed.
# Save this file as Step_7_3B_LR_HKFI_Weight_Sensitivity.py and run the saved
# .py file. Pasting the complete script into a Jupyter cell is not recommended
# because worker processes need a real script path.
# =============================================================================

from __future__ import annotations

import argparse
import gc
import json
import os
import re
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Sequence, Tuple

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Step 6.1 frozen predictor definitions and raster stack.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = STEP61_DIR / "model_ablation_scenarios.json"
    MASTER_STACK_TIF = (
        STEP61_DIR
        / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
    )
    COMMON_AF_MASK_TIF = (
        STEP61_DIR / "common_A_to_F_valid_mask_32645_12p5m_byte.tif"
    )

    # Step 6.4 frozen modelling contract.
    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    MODEL_READY_DATASET_CSV = STEP64_DIR / "model_ready_dataset_A_to_F.csv"
    STEP64_MODEL_CONFIG_JSON = STEP64_DIR / "model_configuration_manifest.json"

    # Outputs.
    OUT_DIR = RESULTS_DIR / "Step-7.3-LR-Models_A_to_F"
    MODEL_DIR = OUT_DIR / "final_models"
    MAP_DIR = OUT_DIR / "susceptibility_maps"
    SEARCH_DIR = OUT_DIR / "grid_search_details"

    OOF_PREDICTIONS_CSV = OUT_DIR / "LR_A_to_F_OOF_predictions.csv"
    FOLD_METRICS_CSV = OUT_DIR / "LR_A_to_F_outer_fold_metrics.csv"
    SUMMARY_METRICS_CSV = OUT_DIR / "LR_A_to_F_summary_metrics.csv"
    AUC_COMPARISONS_CSV = OUT_DIR / "LR_A_to_F_paired_AUC_comparisons.csv"
    OUTER_PARAMS_JSON = OUT_DIR / "LR_A_to_F_outer_fold_best_parameters.json"
    FINAL_PARAMS_JSON = OUT_DIR / "LR_A_to_F_final_full_data_best_parameters.json"
    INNER_SEARCH_CSV = SEARCH_DIR / "LR_A_to_F_nested_inner_grid_results.csv"
    FINAL_SEARCH_CSV = SEARCH_DIR / "LR_A_to_F_full_data_grid_results.csv"
    ROC_FIG_PNG = OUT_DIR / "LR_A_to_F_OOF_ROC_Curves.png"
    ROC_FIG_PDF = OUT_DIR / "LR_A_to_F_OOF_ROC_Curves.pdf"
    FOLD_AUC_FIG_PNG = OUT_DIR / "LR_A_to_F_Fold_AUC.png"
    FOLD_AUC_FIG_PDF = OUT_DIR / "LR_A_to_F_Fold_AUC.pdf"
    QA_REPORT_TXT = OUT_DIR / "Step_7_3_LR_A_to_F_QA_Report.txt"
    METADATA_JSON = OUT_DIR / "Step_7_3_LR_A_to_F_Metadata.json"

    # Manuscript-ready Excel workbook containing the compact A-F performance
    # table, paired AUC comparisons, fold-level AUCs and ensemble summary.
    MANUSCRIPT_TABLES_XLSX = (
        OUT_DIR / "LR_A_to_F_Manuscript_Performance_Tables.xlsx"
    )

    # Cartographic overlays used by the established Step 5.3 map style.
    STUDY_SHP = BASE / "Study_Area.shp"
    ROAD_SHP = BASE / "NTD_Road.shp"
    VISUALIZATION_DIR = MAP_DIR / "visualization"

    # Primary A-F model retained for scientific interpretation.
    PRIMARY_VISUALIZATION_SCENARIO = "E_physics_guided"

    # Old-style static + integrated ensemble analogue for the revised A-F design.
    #
    # Old workflow:
    #   final = 0.30 * context/static + 0.70 * integrated
    #   where integrated support exists; otherwise retain static background.
    #
    # Revised workflow:
    #   final = 0.30 * Model A + 0.70 * Model E
    #   where Model E predictors are available; otherwise retain Model A.
    #
    # The 0.70 weight is fixed a priori to reproduce the earlier ensemble logic.
    # It is NOT tuned on the OOF predictions.
    GENERATE_AE_ENSEMBLE_LSM = True
    ENSEMBLE_STATIC_SCENARIO = "A_static_baseline"
    ENSEMBLE_ENHANCED_SCENARIO = "E_physics_guided"
    ENSEMBLE_ENHANCED_WEIGHT = 0.70

    ENSEMBLE_STATIC_FULL_DOMAIN_TIF = (
        MAP_DIR / "A-static-baseline_LR_probability_full_available_domain.tif"
    )
    ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF = (
        MAP_DIR / "E-physics-guided_LR_probability_full_available_domain.tif"
    )
    ENSEMBLE_PROBABILITY_TIF = (
        MAP_DIR / "LR_Ensemble_A30_E70_probability_full_study_domain.tif"
    )
    ENSEMBLE_SUPPORT_TIF = (
        MAP_DIR / "LR_Ensemble_A30_E70_information_support.tif"
    )
    ENSEMBLE_OOF_PREDICTIONS_CSV = (
        OUT_DIR / "LR_Ensemble_A30_E70_OOF_predictions.csv"
    )
    ENSEMBLE_OOF_METRICS_JSON = (
        OUT_DIR / "LR_Ensemble_A30_E70_OOF_metrics.json"
    )

    # Step 5.3-style map/histogram settings.
    # Publication-size figures.
    # The two requested maps are explicitly controlled independently so that
    # later changes to other figures cannot accidentally alter their dimensions.
    ENSEMBLE_MAP_FIGSIZE = (4, 6)
    SUPPORT_MAP_FIGSIZE = (4, 6)

    # Other visualization products.
    MAP_FIGSIZE = (4, 4)
    HIST_FIGSIZE = (4, 4)
    AREA_PERCENT_FIGSIZE = (4, 4)

    # Enlarged cartographic text for the 4 x 4 inch figure.
    DMS_TICK_FONTSIZE = 7  # Reduced from 9 to prevent latitude label overlap
    LEGEND_FONTSIZE = 7
    LEGEND_TITLE_FONTSIZE = 8
    COLORBAR_TICK_FONTSIZE = 8
    COLORBAR_LABEL_FONTSIZE = 9
    NORTH_ARROW_FONTSIZE = 8  # Slightly reduced for balance
    SCALEBAR_FONTSIZE = 7

    CAX_SIZE = "5%"
    CAX_PAD = 0.01
    FIG_DPI = 900
    DISPLAY_BUFFER_FRACTION = 0.00
    LONGITUDE_DMS_TICK_COUNT = 4
    RASTER_DISPLAY_INTERPOLATION = "nearest"
    SHOW_LSM_FIGURES = True
    SAVE_LSM_FIGURES = True

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"
    CATEGORICAL_FEATURE_ALIASES = {"landcover", "landuse", "land_use"}

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    MODEL_DESCRIPTIONS = {
        "A_static_baseline": "Static baseline",
        "B_raw_deformation": "Static + DV+",
        "C_hkfi_only": "Static + HKFI",
        "D_sndr_only": "Static + SNDR",
        "E_physics_guided": "Static + HKFI + SNDR",
        "F_untransformed_sources": "Static + DV+ + TWI + SM",
    }

    # Logistic-Regression-specific regularization grid.
    #
    # Exact successful Step 7.3 methodological contract:
    #   - solver = liblinear
    #   - penalty = L2 / Ridge
    #   - class_weight = balanced
    #
    # Only C, the inverse regularization strength, is tuned. Hyperparameter
    # selection occurs only inside the four outer-training spatial blocks.
    PARAM_GRID = {
        "model__C": [
            0.01,
            0.05,
            0.10,
            0.50,
            1.00,
            5.00,
            10.00,
            100.00,
        ],
    }

    # Fixed Logistic Regression settings shared by Models A-F.
    LR_FIXED_PARAMS = {
        "penalty": "l2",
        "solver": "liblinear",
        "class_weight": "balanced",
        "fit_intercept": True,
        "max_iter": 10000,
        "tol": 1.0e-6,
        "random_state": 42,
    }

    GRIDSEARCH_N_JOBS = 1
    GRIDSEARCH_VERBOSE = 2
    RANDOM_STATE = 42
    BINARY_THRESHOLD = 0.50

    # Spatial-block cluster bootstrap.
    N_CLUSTER_BOOTSTRAP = 5000
    CI_LEVEL = 0.95

    # Complete execution controls.
    #
    # True  -> after nested spatial-CV evaluation, tune and fit the final
    #          full-data deployment models for A-F.
    # False -> stop after leakage-safe nested spatial OOF evaluation.
    RUN_FINAL_DEPLOYMENT_TUNING = True

    # True  -> generate A-F full-domain probability rasters from the final
    #          deployment models. Requires RUN_FINAL_DEPLOYMENT_TUNING=True.
    GENERATE_RASTER_MAPS = True

    # True  -> create the publication-style primary Model E landslide
    #          susceptibility map and probability histogram.
    GENERATE_PRIMARY_LSM_VISUALIZATION = True
    RASTER_BLOCK_ROWS = 512
    OUTPUT_NODATA = -9999.0

    # Keep nested-CV fold estimators only if needed for later diagnostics.
    SAVE_OUTER_FOLD_MODELS = False


def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def load_json(path: Path) -> Dict[str, Any]:
    require_file(path, "JSON file")
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value



def scenario_prefix(scenario_name: str) -> str:
    return str(scenario_name).replace("_", "-")

def normalize_feature_name(name: str) -> str:
    """Normalize a feature name for robust alias matching."""
    return re.sub(r"[^a-z0-9_]+", "_", str(name).strip().lower()).strip("_")


def is_categorical_feature(name: str) -> bool:
    """Identify categorical predictors using conservative land-cover aliases."""
    normalized = normalize_feature_name(name)
    compact = normalized.replace("_", "")
    aliases = {alias.lower() for alias in Config.CATEGORICAL_FEATURE_ALIASES}
    compact_aliases = {alias.replace("_", "") for alias in aliases}
    return normalized in aliases or compact in compact_aliases


def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """Cross-check Step 6.1 and frozen Step 6.4 A–F definitions."""
    step61 = load_json(Config.SCENARIO_MANIFEST_JSON).get("scenarios", {})
    step64 = load_json(Config.STEP64_MODEL_CONFIG_JSON).get("scenarios", {})

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                f"Step 6.1 scenario manifest is missing {scenario_name}."
            )
        if scenario_name not in step64:
            raise RuntimeError(
                f"Step 6.4 frozen model manifest is missing {scenario_name}."
            )

        features_61 = list(step61[scenario_name].get("features", []))
        features_64 = list(step64[scenario_name].get("features", []))

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any("irs" in feature.lower() for feature in features_61):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; IRS must not be an ML predictor."
            )

    log("[QA] Step 6.1 and Step 6.4 A–F scenario definitions match: PASS")
    return step64


def load_and_validate_model_ready_dataset(
    scenarios: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    """Load the primary frozen 330-row Step 6.4 ML dataset with strict QA."""
    require_file(
        Config.MODEL_READY_DATASET_CSV,
        "Step 6.4 primary frozen model-ready ML dataset",
    )

    df = pd.read_csv(Config.MODEL_READY_DATASET_CSV)

    required_metadata = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ]

    all_features: List[str] = []
    for scenario_name in Config.SCENARIO_ORDER:
        for feature in scenarios[scenario_name]["features"]:
            if feature not in all_features:
                all_features.append(feature)

    missing = [
        column
        for column in required_metadata + all_features
        if column not in df.columns
    ]
    if missing:
        raise RuntimeError(f"Model-ready dataset is missing columns: {missing}")

    if len(df) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Expected {Config.EXPECTED_N_ROWS} rows; found {len(df)}. "
            "No silent row dropping is allowed."
        )

    if df[Config.SAMPLE_ID_COLUMN].isna().any():
        raise RuntimeError("Missing sample_id values found.")
    if df[Config.SAMPLE_ID_COLUMN].duplicated().any():
        raise RuntimeError("Duplicated sample_id values found.")

    df[Config.TARGET_COLUMN] = pd.to_numeric(
        df[Config.TARGET_COLUMN], errors="raise"
    ).astype(int)
    df[Config.FOLD_COLUMN] = pd.to_numeric(
        df[Config.FOLD_COLUMN], errors="raise"
    ).astype(int)
    df[Config.BLOCK_COLUMN] = pd.to_numeric(
        df[Config.BLOCK_COLUMN], errors="raise"
    ).astype(int)

    class_counts = df[Config.TARGET_COLUMN].value_counts().sort_index().to_dict()
    expected_counts = {
        0: Config.EXPECTED_N_NEGATIVE,
        1: Config.EXPECTED_N_POSITIVE,
    }
    if class_counts != expected_counts:
        raise RuntimeError(
            f"Unexpected class counts: {class_counts}; expected {expected_counts}."
        )

    folds = sorted(df[Config.FOLD_COLUMN].unique().tolist())
    if folds != Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected frozen folds {Config.EXPECTED_FOLDS}; found {folds}."
        )

    if not np.array_equal(
        df[Config.FOLD_COLUMN].to_numpy(),
        df[Config.BLOCK_COLUMN].to_numpy(),
    ):
        raise RuntimeError("cv_fold and spatial_block are not identical.")

    # Every frozen block must contain exactly 33 positives and 33 pseudo-absences.
    block_class = pd.crosstab(
        df[Config.FOLD_COLUMN],
        df[Config.TARGET_COLUMN],
    )
    for required_class in (0, 1):
        if required_class not in block_class.columns:
            raise RuntimeError(
                f"Frozen fold table is missing class {required_class}."
            )

    expected_per_class_per_fold = Config.EXPECTED_N_POSITIVE // len(Config.EXPECTED_FOLDS)
    if expected_per_class_per_fold * len(Config.EXPECTED_FOLDS) != Config.EXPECTED_N_POSITIVE:
        raise RuntimeError(
            "Expected positive count is not divisible evenly across the frozen folds."
        )

    for fold in Config.EXPECTED_FOLDS:
        if fold not in block_class.index:
            raise RuntimeError(f"Missing fold {fold}.")
        n0 = int(block_class.loc[fold, 0])
        n1 = int(block_class.loc[fold, 1])
        if n0 != expected_per_class_per_fold or n1 != expected_per_class_per_fold:
            raise RuntimeError(
                f"Fold {fold} violates the frozen Step 6.4 contract: "
                f"negative={n0}, positive={n1}; expected "
                f"{expected_per_class_per_fold}+{expected_per_class_per_fold}=66."
            )

    log(
        "[QA] Frozen fold balance: each fold contains "
        f"{expected_per_class_per_fold} positives + "
        f"{expected_per_class_per_fold} pseudo-absences: PASS"
    )

    # Step 6.4 should already guarantee complete predictors; verify again.
    predictor_values = df[all_features].apply(pd.to_numeric, errors="coerce")
    if not np.isfinite(predictor_values.to_numpy(dtype="float64")).all():
        bad = np.argwhere(
            ~np.isfinite(predictor_values.to_numpy(dtype="float64"))
        )
        examples = [
            (
                str(df.iloc[row_i][Config.SAMPLE_ID_COLUMN]),
                all_features[col_i],
            )
            for row_i, col_i in bad[:20]
        ]
        raise RuntimeError(
            "Missing/non-finite predictors remain in the model-ready dataset. "
            f"Examples: {examples}"
        )

    irs_columns = [
        column for column in df.columns if "irs" in str(column).lower()
    ]
    if irs_columns:
        raise RuntimeError(
            f"IRS-related columns found in ML dataset: {irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError("tpi_s25 found; only tpi_s5 is allowed.")

    log(
        "[QA] Model-ready dataset: "
        f"rows={len(df)}, positive={class_counts[1]}, negative={class_counts[0]}, "
        f"folds={folds}: PASS"
    )
    return df


def build_lr_pipeline(features: Sequence[str]) -> Pipeline:
    """
    Create the exact fold-safe Step 7.3 Logistic Regression pipeline.

    Numeric predictors are standardized using StandardScaler fitted only on the
    current training partition. Land cover is one-hot encoded inside the same
    pipeline. Consequently, all preprocessing remains leakage-safe during every
    inner-CV fit, outer-fold refit, and full-data deployment-model fit.
    """
    categorical = [
        feature
        for feature in features
        if is_categorical_feature(
            feature
        )
    ]

    numeric = [
        feature
        for feature in features
        if not is_categorical_feature(
            feature
        )
    ]

    transformers = []

    if numeric:
        transformers.append(
            (
                "numeric",
                StandardScaler(),
                numeric,
            )
        )

    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = LogisticRegression(
        **Config.LR_FIXED_PARAMS
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                classifier,
            ),
        ]
    )


def make_inner_predefined_split(outer_train_df: pd.DataFrame) -> PredefinedSplit:
    """
    Build a four-fold spatial inner CV from the four blocks remaining after one
    outer block has been held out.
    """
    blocks = sorted(outer_train_df[Config.FOLD_COLUMN].unique().tolist())
    if len(blocks) != 4:
        raise RuntimeError(
            f"Nested CV expected four inner blocks; found {blocks}."
        )

    block_to_test_fold = {
        block: inner_index for inner_index, block in enumerate(blocks)
    }
    test_fold = outer_train_df[Config.FOLD_COLUMN].map(block_to_test_fold).to_numpy()
    return PredefinedSplit(test_fold=test_fold)


def make_full_data_predefined_split(df: pd.DataFrame) -> PredefinedSplit:
    """Use the frozen five blocks for final deployment-model tuning."""
    test_fold = df[Config.FOLD_COLUMN].to_numpy(dtype=int) - 1
    return PredefinedSplit(test_fold=test_fold)


def classification_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = Config.BINARY_THRESHOLD,
) -> Dict[str, Any]:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def cluster_bootstrap_auc_ci(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> Tuple[float, float, int]:
    """Cluster bootstrap AUC by resampling whole spatial blocks."""
    rng = np.random.default_rng(seed)
    unique_blocks = np.unique(blocks)
    values: List[float] = []

    indices_by_block = {
        block: np.flatnonzero(blocks == block) for block in unique_blocks
    }

    for i in range(n_bootstrap):
        if (i + 1) % 1000 == 0:
            log(f"  Bootstrap progress: {i + 1}/{n_bootstrap}")
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(unique_blocks),
            replace=True,
        )
        sampled_indices = np.concatenate(
            [indices_by_block[block] for block in sampled_blocks]
        )
        y_b = y_true[sampled_indices]
        p_b = y_prob[sampled_indices]

        if np.unique(y_b).size < 2:
            continue

        values.append(float(roc_auc_score(y_b, p_b)))

    if not values:
        return float("nan"), float("nan"), 0

    alpha = (1.0 - ci_level) / 2.0
    low, high = np.quantile(values, [alpha, 1.0 - alpha])
    return float(low), float(high), len(values)


def paired_cluster_bootstrap_auc_difference(
    y_true: np.ndarray,
    prob_left: np.ndarray,
    prob_right: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> Tuple[float, float, float, int]:
    """
    Paired spatial-block bootstrap for AUC(left) - AUC(right), preserving the
    identical samples/folds used by all A–F models.
    """
    observed = float(
        roc_auc_score(y_true, prob_left) - roc_auc_score(y_true, prob_right)
    )

    rng = np.random.default_rng(seed)
    unique_blocks = np.unique(blocks)
    indices_by_block = {
        block: np.flatnonzero(blocks == block) for block in unique_blocks
    }
    differences: List[float] = []

    for i in range(n_bootstrap):
        if (i + 1) % 1000 == 0:
            log(f"  Bootstrap progress: {i + 1}/{n_bootstrap}")
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(unique_blocks),
            replace=True,
        )
        sampled_indices = np.concatenate(
            [indices_by_block[block] for block in sampled_blocks]
        )
        y_b = y_true[sampled_indices]

        if np.unique(y_b).size < 2:
            continue

        difference = (
            roc_auc_score(y_b, prob_left[sampled_indices])
            - roc_auc_score(y_b, prob_right[sampled_indices])
        )
        differences.append(float(difference))

    if not differences:
        return observed, float("nan"), float("nan"), 0

    alpha = (1.0 - ci_level) / 2.0
    low, high = np.quantile(differences, [alpha, 1.0 - alpha])
    return observed, float(low), float(high), len(differences)


def normalize_grid_results(
    cv_results: Dict[str, Any],
    scenario_name: str,
    search_stage: str,
    outer_fold: int | None,
) -> pd.DataFrame:
    """Convert GridSearchCV.cv_results_ to a compact audit table."""
    frame = pd.DataFrame(cv_results)

    keep = [
        column
        for column in frame.columns
        if column.startswith("param_")
        or column
        in {
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "mean_fit_time",
            "std_fit_time",
        }
    ]

    frame = frame[keep].copy()
    frame.insert(0, "scenario", scenario_name)
    frame.insert(1, "model", Config.MODEL_SHORT_LABELS[scenario_name])
    frame.insert(2, "search_stage", search_stage)
    frame.insert(3, "outer_fold", outer_fold)
    return frame


def run_nested_spatial_cv_for_scenario(
    df: pd.DataFrame,
    scenario_name: str,
    features: Sequence[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any], pd.DataFrame]:
    """Run five outer spatial folds with four-block inner spatial grid search."""
    log("")
    log("=" * 96)
    log(
        f"[NESTED-CV] {Config.MODEL_SHORT_LABELS[scenario_name]} — "
        f"{Config.MODEL_DESCRIPTIONS[scenario_name]} ({len(features)} predictors)"
    )
    log("=" * 96)

    oof_probability = np.full(len(df), np.nan, dtype="float64")
    oof_predicted = np.full(len(df), -1, dtype="int16")

    fold_rows: List[Dict[str, Any]] = []
    best_params: Dict[str, Any] = {}
    search_tables: List[pd.DataFrame] = []

    for outer_fold in Config.EXPECTED_FOLDS:
        outer_train_mask = df[Config.FOLD_COLUMN].to_numpy() != outer_fold
        outer_valid_mask = df[Config.FOLD_COLUMN].to_numpy() == outer_fold

        train_df = df.loc[outer_train_mask].copy()
        valid_df = df.loc[outer_valid_mask].copy()

        if train_df[Config.FOLD_COLUMN].nunique() != 4:
            raise RuntimeError(
                f"Outer fold {outer_fold}: expected four training blocks."
            )
        if valid_df[Config.FOLD_COLUMN].nunique() != 1:
            raise RuntimeError(
                f"Outer fold {outer_fold}: validation must contain one block only."
            )

        X_train = train_df[list(features)]
        y_train = train_df[Config.TARGET_COLUMN].to_numpy(dtype=int)
        X_valid = valid_df[list(features)]
        y_valid = valid_df[Config.TARGET_COLUMN].to_numpy(dtype=int)

        inner_cv = make_inner_predefined_split(train_df)
        pipeline = build_lr_pipeline(features)

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=Config.PARAM_GRID,
            scoring="roc_auc",
            cv=inner_cv,
            refit=True,
            n_jobs=Config.GRIDSEARCH_N_JOBS,
            verbose=Config.GRIDSEARCH_VERBOSE,
            return_train_score=False,
            error_score="raise",
        )

        log(
            f"[OUTER {outer_fold}] Tuning on blocks "
            f"{sorted(train_df[Config.FOLD_COLUMN].unique().tolist())}; "
            f"holding out block {outer_fold}."
        )
        search.fit(X_train, y_train)

        probability = search.best_estimator_.predict_proba(X_valid)[:, 1]
        predicted = (probability >= Config.BINARY_THRESHOLD).astype(int)

        valid_indices = np.flatnonzero(outer_valid_mask)
        oof_probability[valid_indices] = probability
        oof_predicted[valid_indices] = predicted

        metrics = classification_metrics(y_valid, probability)

        row = {
            "scenario": scenario_name,
            "model": Config.MODEL_SHORT_LABELS[scenario_name],
            "description": Config.MODEL_DESCRIPTIONS[scenario_name],
            "outer_fold": int(outer_fold),
            "n_predictors": int(len(features)),
            "n_train": int(len(train_df)),
            "n_validation": int(len(valid_df)),
            "n_train_positive": int(y_train.sum()),
            "n_train_negative": int(len(y_train) - y_train.sum()),
            "n_validation_positive": int(y_valid.sum()),
            "n_validation_negative": int(len(y_valid) - y_valid.sum()),
            "inner_best_mean_auc": float(search.best_score_),
            **metrics,
        }
        fold_rows.append(row)

        best_params[str(outer_fold)] = {
            "inner_best_mean_auc": float(search.best_score_),
            "best_params": json_safe(search.best_params_),
        }

        search_tables.append(
            normalize_grid_results(
                search.cv_results_,
                scenario_name=scenario_name,
                search_stage="nested_inner",
                outer_fold=outer_fold,
            )
        )

        if Config.SAVE_OUTER_FOLD_MODELS:
            fold_model_path = (
                Config.MODEL_DIR
                / f"{scenario_prefix(scenario_name)}_outer-fold-{outer_fold}.joblib"
            )
            joblib.dump(search.best_estimator_, fold_model_path)

        log(
            f"[OUTER {outer_fold}] inner best AUC={search.best_score_:.4f}; "
            f"held-out AUC={metrics['roc_auc']:.4f}; "
            f"F1={metrics['f1']:.4f}"
        )

    if not np.isfinite(oof_probability).all():
        missing_count = int((~np.isfinite(oof_probability)).sum())
        raise RuntimeError(
            f"{scenario_name}: {missing_count} samples did not receive OOF predictions."
        )
    if (oof_predicted < 0).any():
        raise RuntimeError(f"{scenario_name}: incomplete OOF class predictions.")

    oof_df = df[
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            Config.BLOCK_COLUMN,
        ]
    ].copy()
    oof_df.insert(0, "scenario", scenario_name)
    oof_df.insert(1, "model", Config.MODEL_SHORT_LABELS[scenario_name])
    oof_df.insert(2, "description", Config.MODEL_DESCRIPTIONS[scenario_name])
    oof_df["oof_probability"] = oof_probability
    oof_df["oof_predicted_class_0p5"] = oof_predicted

    fold_df = pd.DataFrame(fold_rows)
    search_df = pd.concat(search_tables, ignore_index=True)

    return oof_df, fold_df, best_params, search_df


def read_stack_band_map(stack_path: Path) -> Dict[str, int]:
    require_file(stack_path, "Step 6.1 master predictor stack")
    with rasterio.open(stack_path) as dataset:
        descriptions = list(dataset.descriptions)

    band_map: Dict[str, int] = {}
    for band_index, description in enumerate(descriptions, start=1):
        if description:
            band_map[str(description)] = band_index

    return band_map


def blend_static_and_enhanced_probability_rasters(
    static_raster_path: Path,
    enhanced_raster_path: Path,
    output_path: Path,
    enhanced_weight: float,
) -> Path:
    """
    Apply the same fixed A30/E70 operational ensemble logic to the Logistic Regression A-F framework.

    Where both models are valid:
        ensemble = (1-w) * Model A + w * Model E

    Where only Model A is valid:
        retain Model A as the susceptibility background.

    Where only Model E is valid:
        retain Model E.

    Outside both valid domains:
        nodata.

    This is intentionally different from averaging all A-F configurations.
    Models B-F are ablation experiments and should not be indiscriminately
    averaged into the final susceptibility map.
    """
    if not 0.0 <= enhanced_weight <= 1.0:
        raise ValueError(
            "ENSEMBLE_ENHANCED_WEIGHT must lie between 0 and 1."
        )

    static_weight = 1.0 - enhanced_weight

    require_file(
        static_raster_path,
        "Model A full-domain probability raster",
    )
    require_file(
        enhanced_raster_path,
        "Model E full-domain probability raster",
    )

    with rasterio.open(static_raster_path) as static_ds, rasterio.open(
        enhanced_raster_path
    ) as enhanced_ds:
        aligned = (
            static_ds.width == enhanced_ds.width
            and static_ds.height == enhanced_ds.height
            and static_ds.transform == enhanced_ds.transform
            and static_ds.crs == enhanced_ds.crs
        )
        if not aligned:
            raise RuntimeError(
                "Model A and Model E full-domain rasters are not aligned."
            )

        profile = static_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        n_static_background = 0
        n_blended = 0
        n_enhanced_only = 0

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_ds:
            for row_start in range(
                0,
                static_ds.height,
                Config.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    Config.RASTER_BLOCK_ROWS,
                    static_ds.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=static_ds.width,
                    height=block_height,
                )

                static_arr = static_ds.read(
                    1,
                    window=window,
                    masked=True,
                )
                enhanced_arr = enhanced_ds.read(
                    1,
                    window=window,
                    masked=True,
                )

                static_values = np.asarray(
                    static_arr.filled(np.nan),
                    dtype="float64",
                )
                enhanced_values = np.asarray(
                    enhanced_arr.filled(np.nan),
                    dtype="float64",
                )

                static_valid = np.isfinite(static_values)
                enhanced_valid = np.isfinite(enhanced_values)

                both = static_valid & enhanced_valid
                static_only = static_valid & ~enhanced_valid
                enhanced_only = enhanced_valid & ~static_valid

                output = np.full(
                    static_values.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                output[static_only] = static_values[
                    static_only
                ].astype("float32")

                output[enhanced_only] = enhanced_values[
                    enhanced_only
                ].astype("float32")

                output[both] = (
                    static_weight * static_values[both]
                    + enhanced_weight * enhanced_values[both]
                ).astype("float32")

                n_static_background += int(static_only.sum())
                n_blended += int(both.sum())
                n_enhanced_only += int(enhanced_only.sum())

                output_ds.write(
                    output,
                    1,
                    window=window,
                )

            output_ds.set_band_description(
                1,
                "Logistic Regression A30-E70 ensemble susceptibility probability",
            )
            output_ds.update_tags(
                model="Logistic Regression",
                ensemble="Model_A_static_plus_Model_E_physics_guided",
                static_weight=f"{static_weight:.6f}",
                enhanced_weight=f"{enhanced_weight:.6f}",
                blend_rule=(
                    "weighted blend where both are valid; "
                    "retain static background elsewhere"
                ),
                validation_note=(
                    "The fixed A/E ensemble is separately evaluated from nested "
                    "OOF probabilities. A-F ablation metrics remain the primary "
                    "model-comparison results."
                ),
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[ENSEMBLE-MAP] Saved A30/E70 ensemble: "
        f"blended={n_blended:,}, "
        f"static-background={n_static_background:,}, "
        f"enhanced-only={n_enhanced_only:,} -> "
        f"{output_path}"
    )

    return output_path


# =============================================================================
# STEP 7.3B ADAPTER / QA HELPERS
# =============================================================================

class _Step73LRAdapter:
    """Expose the embedded successful Step 7.3 Logistic Regression contract
    using the same attribute style used by the Step 7.3B sensitivity code."""
    Config = Config

    validate_scenario_manifests = staticmethod(validate_scenario_manifests)
    load_and_validate_model_ready_dataset = staticmethod(
        load_and_validate_model_ready_dataset
    )
    run_nested_spatial_cv_for_scenario = staticmethod(
        run_nested_spatial_cv_for_scenario
    )
    classification_metrics = staticmethod(classification_metrics)
    cluster_bootstrap_auc_ci = staticmethod(cluster_bootstrap_auc_ci)
    paired_cluster_bootstrap_auc_difference = staticmethod(
        paired_cluster_bootstrap_auc_difference
    )
    build_lr_pipeline = staticmethod(build_lr_pipeline)
    make_full_data_predefined_split = staticmethod(
        make_full_data_predefined_split
    )
    blend_static_and_enhanced_probability_rasters = staticmethod(
        blend_static_and_enhanced_probability_rasters
    )


def load_step73_module(explicit_path: Path | None = None):
    """Return the embedded, frozen Step 7.3 Logistic Regression modelling contract."""
    if explicit_path is not None:
        raise RuntimeError(
            "Step 7.3B is self-contained. BASE_SCRIPT must remain None so the "
            "embedded successful Step 7.3 LR contract is used."
        )
    return _Step73LRAdapter


def all_scenario_features(
    scenarios: Dict[str, Dict[str, Any]],
) -> list[str]:
    features: list[str] = []
    for scenario_name in Config.SCENARIO_ORDER:
        for feature in scenarios[scenario_name]["features"]:
            if feature not in features:
                features.append(feature)
    return features


def validate_dataset_exactly_matches_primary(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
) -> None:
    """Verify exact row identity and predictor values against the frozen
    Step 6.4 primary dataset."""
    reference = load_and_validate_model_ready_dataset(scenarios)

    if len(df) != len(reference):
        raise RuntimeError("Dataset row count differs from the primary Step 6.4 dataset.")

    if not np.array_equal(
        df[Config.SAMPLE_ID_COLUMN].astype(str).to_numpy(),
        reference[Config.SAMPLE_ID_COLUMN].astype(str).to_numpy(),
    ):
        raise RuntimeError("Sample IDs/order differ from the primary Step 6.4 dataset.")

    columns = [
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
        *all_scenario_features(scenarios),
    ]

    for column in columns:
        left = df[column]
        right = reference[column]

        if (
            pd.api.types.is_numeric_dtype(left)
            and pd.api.types.is_numeric_dtype(right)
        ):
            same = np.allclose(
                left.to_numpy(dtype=float),
                right.to_numpy(dtype=float),
                rtol=0.0,
                atol=1.0e-12,
                equal_nan=True,
            )
        else:
            same = np.array_equal(
                left.astype(str).to_numpy(),
                right.astype(str).to_numpy(),
            )

        if not same:
            raise RuntimeError(
                f"Dataset does not exactly match the primary Step 6.4 dataset "
                f"in column: {column}"
            )


def validate_equal_rows_and_folds(
    reference_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
) -> None:
    if len(reference_df) != len(candidate_df):
        raise RuntimeError("HKFI sensitivity dataset changed row count.")

    for column in (
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ):
        if not np.array_equal(
            reference_df[column].astype(str).to_numpy(),
            candidate_df[column].astype(str).to_numpy(),
        ):
            raise RuntimeError(
                f"HKFI sensitivity dataset changed frozen rows/folds in {column}."
            )


def validate_only_allowed_column_changes(
    reference_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
    allowed_changed_columns: Sequence[str],
    columns_to_compare: Sequence[str],
    atol: float,
) -> None:
    allowed = set(allowed_changed_columns)

    for column in columns_to_compare:
        if column in allowed:
            continue

        left = reference_df[column]
        right = candidate_df[column]

        if (
            pd.api.types.is_numeric_dtype(left)
            and pd.api.types.is_numeric_dtype(right)
        ):
            same = np.allclose(
                left.to_numpy(dtype=float),
                right.to_numpy(dtype=float),
                rtol=0.0,
                atol=atol,
                equal_nan=True,
            )
        else:
            same = np.array_equal(
                left.astype(str).to_numpy(),
                right.astype(str).to_numpy(),
            )

        if not same:
            raise RuntimeError(
                f"Unexpected non-HKFI change detected in column: {column}"
            )


def assert_oof_reproduces_primary(
    oof_df: pd.DataFrame,
    scenario_name: str,
    tolerance: float,
) -> Dict[str, Any]:
    """The primary equal-component rerun must reproduce successful Step 7.3
    Model E OOF probabilities."""
    require_file(
        Config.OOF_PREDICTIONS_CSV,
        "successful Step 7.3 LR A-F OOF predictions",
    )

    reference_all = pd.read_csv(Config.OOF_PREDICTIONS_CSV)
    reference = reference_all.loc[
        reference_all["scenario"] == scenario_name,
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            "oof_probability",
        ],
    ].copy()

    if len(reference) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Successful Step 7.3 OOF file contains {len(reference)} rows for "
            f"{scenario_name}; expected {Config.EXPECTED_N_ROWS}."
        )

    candidate = oof_df[
        [
            Config.SAMPLE_ID_COLUMN,
            Config.TARGET_COLUMN,
            Config.FOLD_COLUMN,
            "oof_probability",
        ]
    ].copy()

    merged = candidate.merge(
        reference,
        on=Config.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=("_candidate", "_reference"),
        validate="one_to_one",
    )

    if len(merged) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Primary OOF reproduction merge returned {len(merged)} rows; "
            f"expected {Config.EXPECTED_N_ROWS}."
        )

    for column in (Config.TARGET_COLUMN, Config.FOLD_COLUMN):
        if not np.array_equal(
            merged[f"{column}_candidate"].to_numpy(),
            merged[f"{column}_reference"].to_numpy(),
        ):
            raise RuntimeError(
                f"Primary OOF reproduction metadata mismatch in {column}."
            )

    difference = (
        merged["oof_probability_candidate"].to_numpy(dtype=float)
        - merged["oof_probability_reference"].to_numpy(dtype=float)
    )

    max_abs = float(np.max(np.abs(difference)))
    mean_abs = float(np.mean(np.abs(difference)))
    passed = bool(max_abs <= tolerance)

    if not passed:
        raise RuntimeError(
            "Primary equal-component Step 7.3B LR rerun did not reproduce the "
            "successful Step 7.3 Model E OOF predictions.\n"
            f"Maximum absolute probability difference={max_abs:.12g}\n"
            f"Tolerance={tolerance:.12g}"
        )

    return {
        "oof_reproduction_pass": passed,
        "oof_reproduction_maximum_absolute_probability_difference": max_abs,
        "oof_reproduction_mean_absolute_probability_difference": mean_abs,
        "oof_reproduction_tolerance": float(tolerance),
    }


def paired_auc_difference_from_oof(
    base,
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    label: str,
    seed: int,
) -> Dict[str, Any]:
    left = left_df[
        [
            base.Config.SAMPLE_ID_COLUMN,
            base.Config.TARGET_COLUMN,
            base.Config.FOLD_COLUMN,
            "oof_probability",
        ]
    ].rename(columns={"oof_probability": "probability_left"})

    right = right_df[
        [
            base.Config.SAMPLE_ID_COLUMN,
            base.Config.TARGET_COLUMN,
            base.Config.FOLD_COLUMN,
            "oof_probability",
        ]
    ].rename(columns={"oof_probability": "probability_right"})

    merged = left.merge(
        right,
        on=base.Config.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=("_left", "_right"),
        validate="one_to_one",
    )

    if len(merged) != base.Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Paired comparison '{label}' returned {len(merged)} rows; "
            f"expected {base.Config.EXPECTED_N_ROWS}."
        )

    for column in (base.Config.TARGET_COLUMN, base.Config.FOLD_COLUMN):
        if not np.array_equal(
            merged[f"{column}_left"].to_numpy(),
            merged[f"{column}_right"].to_numpy(),
        ):
            raise RuntimeError(f"Paired OOF mismatch in {column}: {label}")

    y_true = merged[
        f"{base.Config.TARGET_COLUMN}_left"
    ].to_numpy(dtype=int)
    blocks = merged[
        f"{base.Config.FOLD_COLUMN}_left"
    ].to_numpy(dtype=int)

    observed, low, high, n_valid = (
        base.paired_cluster_bootstrap_auc_difference(
            y_true=y_true,
            prob_left=merged["probability_left"].to_numpy(dtype=float),
            prob_right=merged["probability_right"].to_numpy(dtype=float),
            blocks=blocks,
            n_bootstrap=base.Config.N_CLUSTER_BOOTSTRAP,
            seed=seed,
            ci_level=base.Config.CI_LEVEL,
        )
    )

    return {
        "comparison": label,
        "auc_difference_left_minus_right": observed,
        "ci_low": low,
        "ci_high": high,
        "valid_bootstrap_iterations": int(n_valid),
    }


def write_diagnostic_provenance(
    out_dir: Path,
    diagnostic_name: str,
    details: Dict[str, Any],
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "diagnostic_name": diagnostic_name,
        "algorithm": "Logistic Regression",
        "successful_step73_output_directory": str(Config.OUT_DIR),
        "successful_step73_oof": str(Config.OOF_PREDICTIONS_CSV),
        "rf_parameter_grid": json_safe(Config.PARAM_GRID),
        "rf_fixed_parameters": json_safe(Config.LR_FIXED_PARAMS),
        **details,
    }

    with open(
        out_dir / f"{diagnostic_name}_Provenance.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(json_safe(metadata), handle, indent=2)


# =============================================================================
# CONFIGURATION
# =============================================================================
class SensitivityConfig:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # The LR Step 7.3 contract is embedded directly in this self-contained file.
    BASE_SCRIPT: Path | None = None

    # Authoritative Step 5.3 weight-sensitivity outputs.
    STEP53_HKFI_DIR = RESULTS_DIR / "Step-5.3-HKFI"
    STEP53_WEIGHT_DIR = STEP53_HKFI_DIR / "weight_sensitivity"
    STEP53_WEIGHT_MANIFEST_CSV = (
        STEP53_WEIGHT_DIR / "hkfi_weight_scenario_manifest.csv"
    )
    STEP53_NAMED_SENSITIVITY_CSV = (
        STEP53_WEIGHT_DIR / "hkfi_named_weight_sensitivity.csv"
    )
    STEP53_SIMPLEX_SENSITIVITY_CSV = (
        STEP53_WEIGHT_DIR / "hkfi_simplex_weight_sensitivity.csv"
    )

    # Primary scenario name used by authoritative Step 5.3.
    REFERENCE_VARIANT = "primary_equal_component"

    # Model E only: Static12 + HKFI + SNDR.
    SCENARIO_NAME = "E_physics_guided"

    # Targeted cross-algorithm robustness design.
    #
    # Step 7.1B already completed the full XGBoost HKFI weighting sensitivity,
    # including all named scenarios and full-domain map sensitivity. Step 7.2B
    # then tested the focused design under Random Forest. Step 7.3B applies the
    # same focused design under Logistic Regression.
    #
    # Four pre-specified scenarios are retained:
    #   1. primary_equal_component -> a priori reference
    #   2. kinematic_emphasis      -> stronger kinematic contribution
    #   3. twi_emphasis            -> stronger TWI contribution
    #   4. sm_emphasis             -> stronger soil-moisture contribution
    #
    # No LR-optimal HKFI weighting is selected post hoc from these results.
    SCENARIOS_TO_RUN: tuple[str, ...] | None = (
        "primary_equal_component",
        "kinematic_emphasis",
        "twi_emphasis",
        "sm_emphasis",
    )

    # The 66-case simplex is deliberately NOT propagated through ML.
    RUN_SIMPLEX_THROUGH_ML = False

    # Nested-CV and map controls.
    RUN_NESTED_CV = True

    # Performance-only cross-algorithm robustness analysis.
    # Full HKFI map sensitivity was already completed in Step 7.1B using the
    # primary XGBoost framework and is not duplicated here.
    GENERATE_MAP_SENSITIVITY = False

    RESUME_FROM_CHECKPOINTS = True

    # Reproducibility tolerances.
    OOF_REPRODUCTION_TOLERANCE = 1.0e-8
    PRIMARY_RASTER_SAMPLE_TOLERANCE = 5.0e-6
    NON_HKFI_EQUALITY_ATOL = 1.0e-10

    # Output settings.
    RANDOM_STATE = 42171
    FIG_DPI = 300
    RASTER_BLOCK_ROWS = 256

    # Worker/process output.
    WORKER_DIRNAME = "process_isolated_workers"
    LOG_DIRNAME = "logs"


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


def normalize_text(value: Any) -> str:
    return str(value).strip()


def parse_bool(value: Any) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"1", "true", "yes", "y", "t"}:
        return True
    if text in {"0", "false", "no", "n", "f", "", "nan", "none"}:
        return False
    raise ValueError(f"Cannot interpret as boolean: {value!r}")


def sanitize_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")


def resolve_script_path() -> Path:
    """
    Resolve this saved Python script for process-isolated worker launches.

    Jupyter launcher use is recommended. If the complete script is pasted into
    a notebook cell, __file__ may refer to ipykernel_launcher.py, so the script
    searches the current working directory for its canonical filename.
    """
    file_value = globals().get("__file__")
    if file_value:
        candidate = Path(str(file_value)).resolve()
        if candidate.exists() and candidate.name.lower() != "ipykernel_launcher.py":
            return candidate

    candidate = (
        Path.cwd()
        / "Step_7_3B_LR_HKFI_Weight_Sensitivity.py"
    )
    if candidate.exists():
        return candidate.resolve()

    raise RuntimeError(
        "Could not resolve the saved Step 7.3B Python script. "
        "Keep the .py file in the same folder as the launcher notebook."
    )


# =============================================================================
# STEP 5.3 MANIFEST LOADING
# =============================================================================
def load_step53_weight_manifest() -> pd.DataFrame:
    """
    Read the authoritative named-weight scenario manifest produced by Step 5.3.

    No new JSON manifest is required.
    """
    path = SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV
    require_file(path, "Step 5.3 HKFI weight scenario manifest")

    manifest = pd.read_csv(path)
    required = [
        "scenario",
        "scenario_purpose",
        "weight_K",
        "weight_T",
        "weight_S",
        "norm_raster",
        "is_primary",
        "use_for_downstream_spatial_CV",
    ]
    missing = [column for column in required if column not in manifest.columns]
    if missing:
        raise RuntimeError(
            f"Step 5.3 weight scenario manifest is missing columns: {missing}\n"
            f"Found columns: {manifest.columns.tolist()}"
        )

    manifest = manifest.copy()
    manifest["scenario"] = manifest["scenario"].map(normalize_text)
    manifest["scenario_purpose"] = manifest["scenario_purpose"].map(normalize_text)
    manifest["is_primary"] = manifest["is_primary"].map(parse_bool)
    manifest["use_for_downstream_spatial_CV"] = (
        manifest["use_for_downstream_spatial_CV"].map(parse_bool)
    )

    for column in ("weight_K", "weight_T", "weight_S"):
        manifest[column] = pd.to_numeric(manifest[column], errors="raise")

    if manifest["scenario"].duplicated().any():
        duplicates = manifest.loc[
            manifest["scenario"].duplicated(keep=False), "scenario"
        ].tolist()
        raise RuntimeError(f"Duplicated Step 5.3 scenarios found: {duplicates}")

    if SensitivityConfig.REFERENCE_VARIANT not in set(manifest["scenario"]):
        raise RuntimeError(
            "The Step 5.3 manifest does not contain the required primary "
            f"scenario '{SensitivityConfig.REFERENCE_VARIANT}'."
        )

    primary_rows = manifest.loc[manifest["is_primary"]]
    if len(primary_rows) != 1:
        raise RuntimeError(
            "Step 5.3 manifest must contain exactly one is_primary=True row. "
            f"Found {len(primary_rows)}."
        )

    primary_name = str(primary_rows.iloc[0]["scenario"])
    if primary_name != SensitivityConfig.REFERENCE_VARIANT:
        raise RuntimeError(
            "Configured primary scenario does not match Step 5.3 manifest: "
            f"configured={SensitivityConfig.REFERENCE_VARIANT}, "
            f"manifest={primary_name}"
        )

    weight_sum = (
        manifest["weight_K"]
        + manifest["weight_T"]
        + manifest["weight_S"]
    )
    if not np.allclose(weight_sum.to_numpy(dtype=float), 1.0, atol=1.0e-8):
        bad = manifest.loc[
            ~np.isclose(weight_sum, 1.0, atol=1.0e-8),
            ["scenario", "weight_K", "weight_T", "weight_S"],
        ]
        raise RuntimeError(
            "One or more Step 5.3 named scenarios do not sum to 1:\n"
            + bad.to_string(index=False)
        )

    # Resolve raster paths. Step 5.3 writes absolute paths, but relative paths
    # are also supported relative to the manifest directory.
    resolved_paths = []
    for value in manifest["norm_raster"]:
        candidate = Path(str(value))
        if not candidate.is_absolute():
            candidate = path.parent / candidate
        resolved_paths.append(candidate)
    manifest["norm_raster_path"] = resolved_paths

    flagged = manifest.loc[manifest["use_for_downstream_spatial_CV"]].copy()
    if flagged.empty:
        raise RuntimeError(
            "No Step 5.3 scenarios are flagged use_for_downstream_spatial_CV=True."
        )

    if SensitivityConfig.SCENARIOS_TO_RUN is None:
        selected_names = flagged["scenario"].tolist()
    else:
        requested = list(SensitivityConfig.SCENARIOS_TO_RUN)
        unknown = [name for name in requested if name not in set(manifest["scenario"])]
        if unknown:
            raise RuntimeError(
                f"Configured SCENARIOS_TO_RUN contains unknown scenarios: {unknown}"
            )
        not_flagged = [
            name
            for name in requested
            if not bool(
                manifest.loc[
                    manifest["scenario"] == name,
                    "use_for_downstream_spatial_CV",
                ].iloc[0]
            )
        ]
        if not_flagged:
            raise RuntimeError(
                "These requested scenarios are not flagged for downstream spatial CV "
                f"in Step 5.3: {not_flagged}"
            )
        selected_names = requested

    if SensitivityConfig.REFERENCE_VARIANT not in selected_names:
        selected_names = [SensitivityConfig.REFERENCE_VARIANT] + selected_names

    # Primary always first; preserve Step 5.3 order for the remaining scenarios.
    ordered_names = [SensitivityConfig.REFERENCE_VARIANT] + [
        name for name in selected_names
        if name != SensitivityConfig.REFERENCE_VARIANT
    ]

    selected = (
        manifest.set_index("scenario")
        .loc[ordered_names]
        .reset_index()
    )

    for _, row in selected.iterrows():
        require_file(
            Path(row["norm_raster_path"]),
            f"normalized HKFI raster for scenario {row['scenario']}",
        )

    return selected


def load_optional_step53_named_summary() -> pd.DataFrame:
    path = SensitivityConfig.STEP53_NAMED_SENSITIVITY_CSV
    if not path.exists():
        return pd.DataFrame()
    try:
        frame = pd.read_csv(path)
    except Exception as exc:
        log(f"[WARN] Could not read {path.name}: {exc}")
        return pd.DataFrame()

    if "scenario" not in frame.columns:
        log(
            f"[WARN] {path.name} has no 'scenario' column; "
            "upstream HKFI metrics will not be merged."
        )
        return pd.DataFrame()

    frame = frame.copy()
    frame["scenario"] = frame["scenario"].astype(str).str.strip()
    return frame


# =============================================================================
# DATASET CONSTRUCTION FROM STEP 5.3 RASTERS
# =============================================================================
def find_hkfi_feature(features: list[str]) -> str:
    matches = [feature for feature in features if "hkfi" in feature.lower()]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one HKFI feature in Model E; found {matches}"
        )
    return matches[0]


def assert_raster_aligned_to_master(
    base,
    raster_path: Path,
) -> dict[str, Any]:
    require_file(base.Config.MASTER_STACK_TIF, "Step 6.1 master predictor stack")
    require_file(raster_path, "Step 5.3 scenario HKFI raster")

    with rasterio.open(base.Config.MASTER_STACK_TIF) as master, rasterio.open(
        raster_path
    ) as scenario:
        aligned = (
            master.width == scenario.width
            and master.height == scenario.height
            and master.transform == scenario.transform
            and master.crs == scenario.crs
        )
        if not aligned:
            raise RuntimeError(
                "Step 5.3 HKFI scenario raster is not exactly aligned with the "
                "Step 6.1 master predictor stack:\n"
                f"Scenario raster: {raster_path}"
            )

        return {
            "width": int(scenario.width),
            "height": int(scenario.height),
            "crs": str(scenario.crs),
            "transform": tuple(scenario.transform),
            "nodata": scenario.nodata,
        }


def sample_raster_at_frozen_samples(
    base,
    raster_path: Path,
    primary_df: pd.DataFrame,
) -> np.ndarray:
    """
    Sample a Step 5.3 normalized HKFI raster at the exact frozen Step 6.4
    sample coordinates.

    The Step 6.4 model-ready dataset contains x/y coordinates in EPSG:32645.
    Exact raster-grid alignment is verified before sampling.
    """
    assert_raster_aligned_to_master(base, raster_path)

    required_columns = ["x", "y"]
    missing = [column for column in required_columns if column not in primary_df.columns]
    if missing:
        raise RuntimeError(
            "Primary Step 6.4 dataset is missing coordinates required for "
            f"HKFI raster sampling: {missing}"
        )

    coordinates = list(
        zip(
            primary_df["x"].to_numpy(dtype=float),
            primary_df["y"].to_numpy(dtype=float),
        )
    )

    with rasterio.open(raster_path) as dataset:
        nodata = dataset.nodata
        sampled = []
        for value_array in dataset.sample(coordinates, indexes=1, masked=False):
            value = float(value_array[0])
            if nodata is not None and np.isfinite(float(nodata)):
                if np.isclose(value, float(nodata), rtol=0.0, atol=0.0):
                    value = np.nan
            sampled.append(value)

    values = np.asarray(sampled, dtype="float64")
    if not np.isfinite(values).all():
        bad_indices = np.flatnonzero(~np.isfinite(values))
        examples = primary_df.iloc[bad_indices[:20]][
            [base.Config.SAMPLE_ID_COLUMN, "x", "y"]
        ]
        raise RuntimeError(
            f"Scenario HKFI raster produced {len(bad_indices)} non-finite values "
            "at frozen Step 6.4 sample locations.\n"
            f"Raster: {raster_path}\n"
            f"Examples:\n{examples.to_string(index=False)}"
        )

    return values


def build_variant_dataset(
    base,
    scenarios: dict[str, dict[str, Any]],
    manifest_row: pd.Series,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Build one Step 7.3B model-ready sensitivity dataset entirely in memory.

    Primary equal-weight:
        use the exact successful Step 6.4 dataset unchanged.

    Alternative:
        copy the exact primary dataset and replace ONLY HKFI with values sampled
        from the authoritative Step 5.3 scenario normalized raster.
    """
    primary_df = base.load_and_validate_model_ready_dataset(scenarios)
    validate_dataset_exactly_matches_primary(primary_df, scenarios)

    features = list(scenarios[SensitivityConfig.SCENARIO_NAME]["features"])
    hkfi_feature = find_hkfi_feature(features)

    scenario_name = str(manifest_row["scenario"])
    raster_path = Path(manifest_row["norm_raster_path"])

    sampled_hkfi = sample_raster_at_frozen_samples(
        base=base,
        raster_path=raster_path,
        primary_df=primary_df,
    )

    primary_hkfi = primary_df[hkfi_feature].to_numpy(dtype=float)
    raster_vs_primary_max_abs = float(
        np.max(np.abs(sampled_hkfi - primary_hkfi))
    )
    raster_vs_primary_mean_abs = float(
        np.mean(np.abs(sampled_hkfi - primary_hkfi))
    )

    if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
        # The exact successful Step 6.4 dataset remains authoritative for the
        # primary equal-weight rerun. Raster sampling is only a QA cross-check.
        if (
            raster_vs_primary_max_abs
            > SensitivityConfig.PRIMARY_RASTER_SAMPLE_TOLERANCE
        ):
            raise RuntimeError(
                "The Step 5.3 primary equal-weight HKFI raster does not reproduce "
                "the HKFI values frozen in the successful Step 6.4 dataset within "
                "the configured tolerance.\n"
                f"Max absolute difference={raster_vs_primary_max_abs:.10g}\n"
                f"Tolerance={SensitivityConfig.PRIMARY_RASTER_SAMPLE_TOLERANCE:.10g}\n"
                "This indicates an upstream version mismatch between Step 5.3 and "
                "the dataset used for the successful Step 7.3 analysis."
            )
        variant_df = primary_df.copy()
    else:
        variant_df = primary_df.copy()
        variant_df[hkfi_feature] = sampled_hkfi

        # Strong one-variable-at-a-time QA.
        validate_equal_rows_and_folds(primary_df, variant_df)
        compare_columns = [
            base.Config.TARGET_COLUMN,
            base.Config.FOLD_COLUMN,
            base.Config.BLOCK_COLUMN,
            *all_scenario_features(scenarios),
        ]
        validate_only_allowed_column_changes(
            primary_df,
            variant_df,
            allowed_changed_columns=(hkfi_feature,),
            columns_to_compare=compare_columns,
            atol=SensitivityConfig.NON_HKFI_EQUALITY_ATOL,
        )

    delta = variant_df[hkfi_feature].to_numpy(dtype=float) - primary_hkfi

    qa = {
        "scenario": scenario_name,
        "scenario_purpose": str(manifest_row["scenario_purpose"]),
        "weight_K": float(manifest_row["weight_K"]),
        "weight_T": float(manifest_row["weight_T"]),
        "weight_S": float(manifest_row["weight_S"]),
        "norm_raster": str(raster_path),
        "is_primary": bool(manifest_row["is_primary"]),
        "n_samples": int(len(variant_df)),
        "hkfi_mean_absolute_change_vs_primary_dataset": float(
            np.mean(np.abs(delta))
        ),
        "hkfi_maximum_absolute_change_vs_primary_dataset": float(
            np.max(np.abs(delta))
        ),
        "primary_raster_vs_primary_dataset_mean_absolute_difference": (
            raster_vs_primary_mean_abs
        ),
        "primary_raster_vs_primary_dataset_maximum_absolute_difference": (
            raster_vs_primary_max_abs
        ),
        "only_hkfi_changed": True,
    }

    return variant_df, qa


# =============================================================================
# PELRORMANCE SUMMARIES
# =============================================================================
def summarize_variant_oof(
    base,
    oof_df: pd.DataFrame,
    manifest_row: pd.Series,
) -> dict[str, Any]:
    y = oof_df[base.Config.TARGET_COLUMN].to_numpy(dtype=int)
    p = oof_df["oof_probability"].to_numpy(dtype=float)
    blocks = oof_df[base.Config.FOLD_COLUMN].to_numpy(dtype=int)

    metrics = base.classification_metrics(y, p)

    scenario_name = str(manifest_row["scenario"])
    seed_offset = sum(
        (index + 1) * ord(character)
        for index, character in enumerate(scenario_name)
    ) % 10000

    low, high, n_valid = base.cluster_bootstrap_auc_ci(
        y_true=y,
        y_prob=p,
        blocks=blocks,
        n_bootstrap=base.Config.N_CLUSTER_BOOTSTRAP,
        seed=SensitivityConfig.RANDOM_STATE + seed_offset,
        ci_level=base.Config.CI_LEVEL,
    )

    return {
        "scenario": scenario_name,
        "scenario_purpose": str(manifest_row["scenario_purpose"]),
        "weight_K": float(manifest_row["weight_K"]),
        "weight_T": float(manifest_row["weight_T"]),
        "weight_S": float(manifest_row["weight_S"]),
        "is_primary": bool(manifest_row["is_primary"]),
        "oof_roc_auc": metrics["roc_auc"],
        "auc_ci_low": low,
        "auc_ci_high": high,
        "oof_pr_auc": metrics["pr_auc"],
        "oof_balanced_accuracy": metrics["balanced_accuracy"],
        "oof_f1": metrics["f1"],
        "oof_brier": metrics["brier"],
        "oof_precision": metrics["precision"],
        "oof_recall": metrics["recall"],
        "valid_auc_bootstrap_iterations": int(n_valid),
    }


# =============================================================================
# MAP HELPERS
# =============================================================================
def read_master_band_map(base) -> dict[str, int]:
    with rasterio.open(base.Config.MASTER_STACK_TIF) as dataset:
        descriptions = list(dataset.descriptions)

    band_map = {}
    for index, description in enumerate(descriptions, start=1):
        if description:
            band_map[str(description)] = int(index)
    return band_map


def predict_model_e_with_external_hkfi_raster(
    base,
    fitted_pipeline,
    features: list[str],
    hkfi_feature: str,
    hkfi_raster_path: Path,
    output_path: Path,
) -> Path:
    """
    Predict Model E using:
      - all non-HKFI predictors from the successful Step 6.1 master stack;
      - HKFI from the selected authoritative Step 5.3 scenario raster.

    No new full multiband stack is required.
    """
    require_file(base.Config.MASTER_STACK_TIF, "Step 6.1 master predictor stack")
    require_file(hkfi_raster_path, "Step 5.3 scenario HKFI raster")
    assert_raster_aligned_to_master(base, hkfi_raster_path)

    band_map = read_master_band_map(base)
    non_hkfi_features = [
        feature for feature in features if feature != hkfi_feature
    ]
    missing = [
        feature for feature in non_hkfi_features if feature not in band_map
    ]
    if missing:
        raise RuntimeError(
            f"Master stack is missing non-HKFI Model E predictors: {missing}"
        )

    non_hkfi_band_indexes = [
        band_map[feature] for feature in non_hkfi_features
    ]

    with rasterio.open(base.Config.MASTER_STACK_TIF) as master, rasterio.open(
        hkfi_raster_path
    ) as hkfi_ds:
        profile = master.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=base.Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        valid_pixel_count = 0

        with rasterio.open(temporary_path, "w", **profile) as output_ds:
            for row_start in range(
                0,
                master.height,
                SensitivityConfig.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    SensitivityConfig.RASTER_BLOCK_ROWS,
                    master.height - row_start,
                )
                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=master.width,
                    height=block_height,
                )

                static_dynamic = master.read(
                    non_hkfi_band_indexes,
                    window=window,
                    masked=True,
                )
                hkfi_arr = hkfi_ds.read(
                    1,
                    window=window,
                    masked=True,
                )

                non_hkfi_values = np.asarray(
                    static_dynamic.filled(np.nan),
                    dtype="float64",
                )
                hkfi_values = np.asarray(
                    hkfi_arr.filled(np.nan),
                    dtype="float64",
                )

                valid = (
                    np.all(np.isfinite(non_hkfi_values), axis=0)
                    & np.isfinite(hkfi_values)
                )

                output = np.full(
                    (block_height, master.width),
                    base.Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                if np.any(valid):
                    data_columns: dict[str, np.ndarray] = {}
                    non_hkfi_index = 0
                    for feature in features:
                        if feature == hkfi_feature:
                            data_columns[feature] = hkfi_values[valid]
                        else:
                            data_columns[feature] = (
                                non_hkfi_values[non_hkfi_index][valid]
                            )
                            non_hkfi_index += 1

                    pixel_df = pd.DataFrame(data_columns, columns=features)
                    probabilities = (
                        fitted_pipeline.predict_proba(pixel_df)[:, 1]
                    )
                    output[valid] = probabilities.astype("float32")
                    valid_pixel_count += int(valid.sum())

                output_ds.write(output, 1, window=window)

            output_ds.set_band_description(
                1,
                "Logistic Regression Model E HKFI-weight sensitivity probability",
            )
            output_ds.update_tags(
                model="Logistic Regression",
                scenario="E_physics_guided",
                hkfi_source=str(hkfi_raster_path),
                generated_at_utc=utc_now_iso(),
                validation_note=(
                    "Map generated from a separately tuned full-data deployment "
                    "model. Validation performance comes from nested spatial OOF."
                ),
            )

    if output_path.exists():
        output_path.unlink()
    os.replace(temporary_path, output_path)

    log(
        f"[MAP] {valid_pixel_count:,} valid Model E pixels -> {output_path}"
    )
    return output_path


def fit_final_variant_model(
    base,
    df: pd.DataFrame,
    features: list[str],
    scenario_name: str,
    model_dir: Path,
) -> tuple[Any, dict[str, Any], float]:
    X = df[features]
    y = df[base.Config.TARGET_COLUMN].to_numpy(dtype=int)

    search = GridSearchCV(
        estimator=base.build_lr_pipeline(features),
        param_grid=base.Config.PARAM_GRID,
        scoring="roc_auc",
        cv=base.make_full_data_predefined_split(df),
        refit=True,
        n_jobs=base.Config.GRIDSEARCH_N_JOBS,
        verbose=0,
        return_train_score=False,
        error_score="raise",
    )
    search.fit(X, y)

    model_path = (
        model_dir
        / f"LR_Model_E_HKFI_{sanitize_name(scenario_name)}_final.joblib"
    )
    joblib.dump(search.best_estimator_, model_path)

    return (
        search.best_estimator_,
        json_safe(search.best_params_),
        float(search.best_score_),
    )


def clean_probability_raster(dataset) -> np.ndarray:
    array = dataset.read(1, masked=True).filled(np.nan).astype("float64")
    nodata = dataset.nodata
    if nodata is not None and np.isfinite(float(nodata)):
        array[array == float(nodata)] = np.nan
    array[(array < -1.0e-6) | (array > 1.0 + 1.0e-6)] = np.nan
    return array


def compare_probability_maps(
    reference_path: Path,
    candidate_path: Path,
) -> dict[str, Any]:
    require_file(reference_path, "reference probability raster")
    require_file(candidate_path, "candidate probability raster")

    with rasterio.open(reference_path) as ref_ds, rasterio.open(
        candidate_path
    ) as can_ds:
        if not (
            ref_ds.width == can_ds.width
            and ref_ds.height == can_ds.height
            and ref_ds.transform == can_ds.transform
            and ref_ds.crs == can_ds.crs
        ):
            raise RuntimeError(
                "HKFI sensitivity probability maps are not spatially aligned."
            )

        ref = clean_probability_raster(ref_ds)
        can = clean_probability_raster(can_ds)

    valid = np.isfinite(ref) & np.isfinite(can)
    if not np.any(valid):
        raise RuntimeError("No common valid pixels between sensitivity maps.")

    ref_v = ref[valid]
    can_v = can[valid]
    difference = can_v - ref_v

    ref_class = np.digitize(
        ref_v,
        bins=[0.2, 0.4, 0.6, 0.8],
        right=False,
    )
    can_class = np.digitize(
        can_v,
        bins=[0.2, 0.4, 0.6, 0.8],
        right=False,
    )

    return {
        "common_valid_pixels": int(valid.sum()),
        "mean_absolute_probability_difference": float(
            np.mean(np.abs(difference))
        ),
        "maximum_absolute_probability_difference": float(
            np.max(np.abs(difference))
        ),
        "rmse_probability_difference": float(
            np.sqrt(np.mean(difference ** 2))
        ),
        "pearson_probability_correlation": float(
            np.corrcoef(ref_v, can_v)[0, 1]
        ),
        "spearman_probability_correlation": float(
            pd.Series(ref_v).corr(
                pd.Series(can_v),
                method="spearman",
            )
        ),
        "pixels_changing_susceptibility_class_percent": float(
            100.0 * np.mean(ref_class != can_class)
        ),
        "reference_high_plus_very_high_area_percent": float(
            100.0 * np.mean(ref_v >= 0.6)
        ),
        "candidate_high_plus_very_high_area_percent": float(
            100.0 * np.mean(can_v >= 0.6)
        ),
        "delta_high_plus_very_high_area_percentage_points": float(
            100.0 * np.mean(can_v >= 0.6)
            - 100.0 * np.mean(ref_v >= 0.6)
        ),
    }


# =============================================================================
# OUTPUT / CHECKPOINT PATHS
# =============================================================================
def output_directories(base) -> dict[str, Path]:
    out_dir = (
        Path(base.Config.OUT_DIR)
        / "Step-7.3B-HKFI_Weight_Sensitivity_STEP53_Direct"
    )
    paths = {
        "out": out_dir,
        "datasets": out_dir / "generated_model_ready_datasets",
        "models": out_dir / "models",
        "maps": out_dir / "maps",
        "workers": out_dir / SensitivityConfig.WORKER_DIRNAME,
        "logs": (
            out_dir
            / SensitivityConfig.WORKER_DIRNAME
            / SensitivityConfig.LOG_DIRNAME
        ),
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def scenario_paths(
    base,
    scenario_name: str,
) -> dict[str, Path]:
    dirs = output_directories(base)
    safe = sanitize_name(scenario_name)
    return {
        "dataset": dirs["datasets"] / f"{safe}_model_ready.csv",
        "qa": dirs["workers"] / f"{safe}_dataset_QA.json",
        "oof": dirs["workers"] / f"{safe}_OOF.csv",
        "fold_metrics": dirs["workers"] / f"{safe}_Fold_Metrics.csv",
        "search": dirs["workers"] / f"{safe}_Inner_Search.csv",
        "params": dirs["workers"] / f"{safe}_Outer_Params.json",
        "nested_done": dirs["workers"] / f"{safe}_nested.done",
        "model_info": dirs["workers"] / f"{safe}_Map_Model_Info.json",
        "e_map": dirs["maps"] / f"LR_Model_E_HKFI_{safe}_probability.tif",
        "ensemble_map": dirs["maps"] / f"LR_A30_E70_HKFI_{safe}_probability.tif",
        "map_done": dirs["workers"] / f"{safe}_map.done",
    }


# =============================================================================
# PROCESS-ISOLATED WORKERS
# =============================================================================
def run_external_worker(
    worker_mode: str,
    scenario_name: str,
) -> None:
    script_path = resolve_script_path()
    base = load_step73_module(SensitivityConfig.BASE_SCRIPT)
    dirs = output_directories(base)
    log_path = (
        dirs["logs"]
        / f"{sanitize_name(scenario_name)}_{worker_mode}.log"
    )

    command = [
        sys.executable,
        str(script_path),
        "--worker-mode",
        worker_mode,
        "--scenario",
        scenario_name,
    ]

    environment = os.environ.copy()
    environment["OMP_NUM_THREADS"] = "1"
    environment["OPENBLAS_NUM_THREADS"] = "1"
    environment["MKL_NUM_THREADS"] = "1"
    environment["NUMEXPR_NUM_THREADS"] = "1"

    log(
        f"[WORKER] Starting {worker_mode}: {scenario_name}\n"
        f"[WORKER] Log: {log_path}"
    )

    with open(log_path, "w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=environment,
        )

        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_handle.write(line)
            log_handle.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Step 7.3B LR worker failed: mode={worker_mode}, "
            f"scenario={scenario_name}, return_code={return_code}. "
            f"See log: {log_path}"
        )


def get_manifest_row(
    manifest: pd.DataFrame,
    scenario_name: str,
) -> pd.Series:
    rows = manifest.loc[manifest["scenario"] == scenario_name]
    if len(rows) != 1:
        raise RuntimeError(
            f"Expected exactly one manifest row for {scenario_name}; found {len(rows)}"
        )
    return rows.iloc[0]


def run_nested_worker(scenario_name: str) -> None:
    base = load_step73_module(SensitivityConfig.BASE_SCRIPT)
    scenarios = base.validate_scenario_manifests()
    manifest = load_step53_weight_manifest()
    row = get_manifest_row(manifest, scenario_name)
    paths = scenario_paths(base, scenario_name)

    variant_df, qa = build_variant_dataset(
        base=base,
        scenarios=scenarios,
        manifest_row=row,
    )
    variant_df.to_csv(
        paths["dataset"],
        index=False,
        float_format="%.10f",
    )

    with open(paths["qa"], "w", encoding="utf-8") as handle:
        json.dump(json_safe(qa), handle, indent=2)

    features = list(
        scenarios[SensitivityConfig.SCENARIO_NAME]["features"]
    )

    log(
        f"[NESTED] {scenario_name}: "
        f"K={row['weight_K']:.3f}, "
        f"T={row['weight_T']:.3f}, "
        f"S={row['weight_S']:.3f}"
    )

    oof_df, fold_df, params, search_df = (
        base.run_nested_spatial_cv_for_scenario(
            df=variant_df,
            scenario_name=SensitivityConfig.SCENARIO_NAME,
            features=features,
        )
    )

    reproduction_qa = None
    if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
        reproduction_qa = assert_oof_reproduces_primary(
            oof_df,
            SensitivityConfig.SCENARIO_NAME,
            tolerance=SensitivityConfig.OOF_REPRODUCTION_TOLERANCE,
        )
        qa.update(reproduction_qa)
        with open(paths["qa"], "w", encoding="utf-8") as handle:
            json.dump(json_safe(qa), handle, indent=2)

    oof_df = oof_df.copy()
    oof_df.insert(0, "hkfi_weight_variant", scenario_name)

    fold_df = fold_df.copy()
    fold_df.insert(0, "hkfi_weight_variant", scenario_name)

    search_df = search_df.copy()
    search_df.insert(0, "hkfi_weight_variant", scenario_name)

    oof_df.to_csv(paths["oof"], index=False, float_format="%.10f")
    fold_df.to_csv(
        paths["fold_metrics"],
        index=False,
        float_format="%.10f",
    )
    search_df.to_csv(
        paths["search"],
        index=False,
        float_format="%.10f",
    )
    with open(paths["params"], "w", encoding="utf-8") as handle:
        json.dump(json_safe(params), handle, indent=2)

    paths["nested_done"].write_text(
        utc_now_iso(),
        encoding="utf-8",
    )
    log(f"[NESTED] Completed: {scenario_name}")


def run_map_worker(scenario_name: str) -> None:
    base = load_step73_module(SensitivityConfig.BASE_SCRIPT)
    scenarios = base.validate_scenario_manifests()
    manifest = load_step53_weight_manifest()
    row = get_manifest_row(manifest, scenario_name)
    paths = scenario_paths(base, scenario_name)
    dirs = output_directories(base)

    if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
        # The successful Step 7.3 maps are the authoritative primary reference.
        require_file(
            base.Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF,
            "successful Step 7.3 primary Model E full-domain raster",
        )
        require_file(
            base.Config.ENSEMBLE_PROBABILITY_TIF,
            "successful Step 7.3 A30/E70 ensemble raster",
        )

        info = {
            "scenario": scenario_name,
            "map_source": "successful Step 7.3 primary reference",
            "model_e_map_path": str(
                base.Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF
            ),
            "operational_ensemble_map_path": str(
                base.Config.ENSEMBLE_PROBABILITY_TIF
            ),
            "refit_performed": False,
        }
        with open(paths["model_info"], "w", encoding="utf-8") as handle:
            json.dump(info, handle, indent=2)
        paths["map_done"].write_text(utc_now_iso(), encoding="utf-8")
        return

    variant_df, qa = build_variant_dataset(
        base=base,
        scenarios=scenarios,
        manifest_row=row,
    )

    features = list(
        scenarios[SensitivityConfig.SCENARIO_NAME]["features"]
    )
    hkfi_feature = find_hkfi_feature(features)

    model, best_params, selection_auc = fit_final_variant_model(
        base=base,
        df=variant_df,
        features=features,
        scenario_name=scenario_name,
        model_dir=dirs["models"],
    )

    e_map = predict_model_e_with_external_hkfi_raster(
        base=base,
        fitted_pipeline=model,
        features=features,
        hkfi_feature=hkfi_feature,
        hkfi_raster_path=Path(row["norm_raster_path"]),
        output_path=paths["e_map"],
    )

    ensemble_map = None
    if base.Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF.exists():
        ensemble_map = base.blend_static_and_enhanced_probability_rasters(
            static_raster_path=base.Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF,
            enhanced_raster_path=e_map,
            output_path=paths["ensemble_map"],
            enhanced_weight=float(base.Config.ENSEMBLE_ENHANCED_WEIGHT),
        )
    else:
        log(
            "[MAP-WARN] Operational ensemble skipped because successful "
            "Step 7.3 Model A full-domain raster is missing: "
            f"{base.Config.ENSEMBLE_STATIC_FULL_DOMAIN_TIF}"
        )

    info = {
        "scenario": scenario_name,
        "scenario_purpose": str(row["scenario_purpose"]),
        "weight_K": float(row["weight_K"]),
        "weight_T": float(row["weight_T"]),
        "weight_S": float(row["weight_S"]),
        "full_data_selection_auc_for_model_selection_only": selection_auc,
        "best_params": best_params,
        "model_e_map_path": str(e_map),
        "operational_ensemble_map_path": (
            str(ensemble_map) if ensemble_map is not None else None
        ),
        "refit_performed": True,
    }
    with open(paths["model_info"], "w", encoding="utf-8") as handle:
        json.dump(json_safe(info), handle, indent=2)

    paths["map_done"].write_text(utc_now_iso(), encoding="utf-8")

    del model
    gc.collect()
    log(f"[MAP] Completed: {scenario_name}")


# =============================================================================
# CONTROLLER / AGGREGATION
# =============================================================================
def load_nested_outputs(
    base,
    manifest: pd.DataFrame,
) -> tuple[
    dict[str, pd.DataFrame],
    pd.DataFrame,
    pd.DataFrame,
]:
    all_oof: dict[str, pd.DataFrame] = {}
    summaries = []
    qa_rows = []

    for _, row in manifest.iterrows():
        scenario_name = str(row["scenario"])
        paths = scenario_paths(base, scenario_name)

        if not paths["nested_done"].exists():
            raise RuntimeError(
                f"Missing nested checkpoint for scenario: {scenario_name}"
            )

        oof_df = pd.read_csv(paths["oof"])
        all_oof[scenario_name] = oof_df
        summaries.append(
            summarize_variant_oof(
                base=base,
                oof_df=oof_df,
                manifest_row=row,
            )
        )

        with open(paths["qa"], "r", encoding="utf-8") as handle:
            qa_rows.append(json.load(handle))

    return (
        all_oof,
        pd.DataFrame(summaries),
        pd.DataFrame(qa_rows),
    )


def build_pairwise_comparisons(
    base,
    manifest: pd.DataFrame,
    all_oof: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    reference = all_oof[SensitivityConfig.REFERENCE_VARIANT]
    rows = []

    for index, row in manifest.iterrows():
        scenario_name = str(row["scenario"])
        if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
            continue

        comparison = paired_auc_difference_from_oof(
            base=base,
            left_df=all_oof[scenario_name],
            right_df=reference,
            label=f"{scenario_name} - {SensitivityConfig.REFERENCE_VARIANT}",
            seed=SensitivityConfig.RANDOM_STATE + 100 + int(index),
        )
        comparison.update(
            {
                "scenario": scenario_name,
                "scenario_purpose": str(row["scenario_purpose"]),
                "weight_K": float(row["weight_K"]),
                "weight_T": float(row["weight_T"]),
                "weight_S": float(row["weight_S"]),
            }
        )
        rows.append(comparison)

    return pd.DataFrame(rows)


def build_map_comparisons(
    base,
    manifest: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    model_rows = []
    comparison_rows = []

    reference_e_map = Path(base.Config.ENSEMBLE_ENHANCED_FULL_DOMAIN_TIF)
    reference_ensemble = Path(base.Config.ENSEMBLE_PROBABILITY_TIF)

    for _, row in manifest.iterrows():
        scenario_name = str(row["scenario"])
        paths = scenario_paths(base, scenario_name)

        if not paths["map_done"].exists():
            continue

        with open(paths["model_info"], "r", encoding="utf-8") as handle:
            info = json.load(handle)

        model_rows.append(info)

        if scenario_name == SensitivityConfig.REFERENCE_VARIANT:
            continue

        e_map_text = info.get("model_e_map_path")
        if e_map_text:
            e_map = Path(e_map_text)
            comparison_rows.append(
                {
                    "map_product": "Model E scientific sensitivity map",
                    "scenario": scenario_name,
                    "scenario_purpose": str(row["scenario_purpose"]),
                    "weight_K": float(row["weight_K"]),
                    "weight_T": float(row["weight_T"]),
                    "weight_S": float(row["weight_S"]),
                    "reference_scenario": SensitivityConfig.REFERENCE_VARIANT,
                    **compare_probability_maps(reference_e_map, e_map),
                }
            )

        ensemble_text = info.get("operational_ensemble_map_path")
        if ensemble_text and reference_ensemble.exists():
            ensemble_map = Path(ensemble_text)
            comparison_rows.append(
                {
                    "map_product": "A30/E70 operational ensemble map",
                    "scenario": scenario_name,
                    "scenario_purpose": str(row["scenario_purpose"]),
                    "weight_K": float(row["weight_K"]),
                    "weight_T": float(row["weight_T"]),
                    "weight_S": float(row["weight_S"]),
                    "reference_scenario": SensitivityConfig.REFERENCE_VARIANT,
                    **compare_probability_maps(
                        reference_ensemble,
                        ensemble_map,
                    ),
                }
            )

    return pd.DataFrame(model_rows), pd.DataFrame(comparison_rows)


def merge_upstream_hkfi_metrics(
    summary_df: pd.DataFrame,
) -> pd.DataFrame:
    upstream = load_optional_step53_named_summary()
    if upstream.empty:
        return summary_df

    # Avoid duplicating weight columns if they are present upstream.
    duplicate_columns = [
        column for column in upstream.columns
        if column in summary_df.columns and column != "scenario"
    ]
    upstream = upstream.drop(columns=duplicate_columns, errors="ignore")

    return summary_df.merge(
        upstream,
        on="scenario",
        how="left",
        validate="one_to_one",
    )


def create_performance_figure(
    summary_df: pd.DataFrame,
    output_path: Path,
) -> None:
    plot_df = summary_df.copy().reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    x = np.arange(len(plot_df))
    y = plot_df["oof_roc_auc"].to_numpy(dtype=float)

    lower = y - plot_df["auc_ci_low"].to_numpy(dtype=float)
    upper = plot_df["auc_ci_high"].to_numpy(dtype=float) - y

    ax.errorbar(
        x,
        y,
        yerr=np.vstack([lower, upper]),
        fmt="o",
        capsize=3,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(
        plot_df["scenario"],
        rotation=32,
        ha="right",
    )
    ax.set_ylabel("Nested spatial OOF ROC-AUC")
    ax.set_xlabel("Pre-specified HKFI weighting scenario")
    ax.set_ylim(
        max(
            0.5,
            float(np.nanmin(plot_df["auc_ci_low"])) - 0.03,
        ),
        1.0,
    )
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=SensitivityConfig.FIG_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def write_summary_report(
    base,
    manifest: pd.DataFrame,
    summary_df: pd.DataFrame,
    comparison_df: pd.DataFrame,
    qa_df: pd.DataFrame,
    map_comparison_df: pd.DataFrame,
) -> None:
    dirs = output_directories(base)
    report_path = dirs["out"] / "Reviewer_1_Comment_7_LR_HKFI_Weight_Sensitivity_Report.txt"

    with open(report_path, "w", encoding="utf-8") as handle:
        handle.write("STEP 7.3B — LR HKFI WEIGHTING SENSITIVITY\n")
        handle.write("=" * 88 + "\n\n")
        handle.write(
            "Source of weighting scenarios:\n"
            f"{SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV}\n\n"
        )
        handle.write(
            "Step 7.3B is a targeted cross-algorithm robustness analysis. "
            "The primary equal-component formulation and three pre-specified "
            "emphasis scenarios are propagated through the exact successful "
            "Step 7.3 LR nested spatial-CV contract. The Step 5.3 simplex is "
            "not optimized through LR, weights are not selected post hoc by "
            "AUC, and full-domain map sensitivity is not repeated because it "
            "was already completed for the primary XGBoost framework in "
            "Step 7.1B.\n\n"
        )

        handle.write("SELECTED STEP 5.3 NAMED SCENARIOS\n")
        handle.write("-" * 88 + "\n")
        handle.write(
            manifest[
                [
                    "scenario",
                    "scenario_purpose",
                    "weight_K",
                    "weight_T",
                    "weight_S",
                ]
            ].to_string(index=False)
        )
        handle.write("\n\n")

        handle.write("DATASET / REPRODUCIBILITY QA\n")
        handle.write("-" * 88 + "\n")
        handle.write(qa_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("NESTED SPATIAL OOF PELRORMANCE\n")
        handle.write("-" * 88 + "\n")
        performance_columns = [
            "scenario",
            "weight_K",
            "weight_T",
            "weight_S",
            "oof_roc_auc",
            "auc_ci_low",
            "auc_ci_high",
            "oof_pr_auc",
            "oof_balanced_accuracy",
            "oof_f1",
            "oof_brier",
        ]
        handle.write(
            summary_df[
                [column for column in performance_columns if column in summary_df.columns]
            ].to_string(index=False)
        )
        handle.write("\n\n")

        handle.write("PAIRED DELTA-AUC VERSUS PRIMARY EQUAL-WEIGHT SCENARIO\n")
        handle.write("-" * 88 + "\n")
        if comparison_df.empty:
            handle.write("No alternative scenarios evaluated.\n")
        else:
            handle.write(comparison_df.to_string(index=False))
        handle.write("\n\n")

        handle.write("MAP SENSITIVITY SCOPE\n")
        handle.write("-" * 88 + "\n")
        if map_comparison_df.empty:
            handle.write(
                "Not repeated for Logistic Regression by design. The full HKFI "
                "map-sensitivity experiment was already completed in Step 7.1B "
                "for the primary XGBoost framework.\n"
            )
        else:
            handle.write(map_comparison_df.to_string(index=False))

        handle.write("\n\n")
        handle.write("INTERPRETATION RULE\n")
        handle.write("-" * 88 + "\n")
        handle.write(
            "Step 7.3B is not used to identify an LR-optimal HKFI weighting. "
            "Its purpose is to test whether the XGBoost weighting-robustness "
            "conclusion, already examined under Random Forest in Step 7.2B, is "
            "also supported under Logistic Regression.\n"
        )


def run_controller() -> None:
    base = load_step73_module(SensitivityConfig.BASE_SCRIPT)
    scenarios = base.validate_scenario_manifests()
    manifest = load_step53_weight_manifest()

    expected_cross_algorithm_scenarios = [
        "primary_equal_component",
        "kinematic_emphasis",
        "twi_emphasis",
        "sm_emphasis",
    ]

    observed_cross_algorithm_scenarios = manifest["scenario"].tolist()

    if observed_cross_algorithm_scenarios != expected_cross_algorithm_scenarios:
        raise RuntimeError(
            "Step 7.3B targeted cross-algorithm scenario set mismatch.\n"
            f"Expected: {expected_cross_algorithm_scenarios}\n"
            f"Found: {observed_cross_algorithm_scenarios}"
        )

    dirs = output_directories(base)

    log("=" * 96)
    log("STEP 7.3B — TARGETED LR CROSS-ALGORITHM HKFI WEIGHT ROBUSTNESS")
    log("=" * 96)
    log(f"[STEP5.3] Manifest: {SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV}")
    log(
        "[STEP5.3] Named scenarios selected: "
        + ", ".join(manifest["scenario"].tolist())
    )
    log(
        "[RULE] Selected pre-specified HKFI scenarios only; the Step 5.3 simplex "
        "is not propagated through LR, weights are not optimized by AUC, and "
        "full-domain map sensitivity is not repeated."
    )

    write_diagnostic_provenance(
        dirs["out"],
        "Step_7_3B_LR_STEP53_Direct",
        {
            "step53_weight_manifest": str(
                SensitivityConfig.STEP53_WEIGHT_MANIFEST_CSV
            ),
            "step53_named_sensitivity": str(
                SensitivityConfig.STEP53_NAMED_SENSITIVITY_CSV
            ),
            "step53_simplex_sensitivity": str(
                SensitivityConfig.STEP53_SIMPLEX_SENSITIVITY_CSV
            ),
            "selected_scenarios": manifest[
                [
                    "scenario",
                    "scenario_purpose",
                    "weight_K",
                    "weight_T",
                    "weight_S",
                    "norm_raster_path",
                ]
            ].to_dict(orient="records"),
            "model_scenario": SensitivityConfig.SCENARIO_NAME,
            "primary_reference": SensitivityConfig.REFERENCE_VARIANT,
            "simplex_propagated_through_ml": False,
            "map_sensitivity_generated": False,
            "cross_algorithm_scope": (
                "Performance robustness only. Full HKFI map sensitivity was "
                "already evaluated in Step 7.1B using the primary XGBoost "
                "framework; Step 7.3B extends the focused replication to LR."
            ),
            "scientific_rule": (
                "Selected pre-specified weight scenarios only; no downstream "
                "weight optimization and no duplicate LR map-sensitivity stage."
            ),
        },
    )

    manifest.to_csv(
        dirs["out"] / "LR_7_3B_Selected_Step53_Weight_Scenarios.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # 1. Nested CV — one fresh process per named scenario.
    # -------------------------------------------------------------------------
    if SensitivityConfig.RUN_NESTED_CV:
        for scenario_name in manifest["scenario"]:
            paths = scenario_paths(base, scenario_name)
            if (
                SensitivityConfig.RESUME_FROM_CHECKPOINTS
                and paths["nested_done"].exists()
                and paths["oof"].exists()
            ):
                log(f"[RESUME] Nested CV already complete: {scenario_name}")
                continue
            run_external_worker("nested", scenario_name)

    all_oof, summary_df, qa_df = load_nested_outputs(
        base=base,
        manifest=manifest,
    )

    comparison_df = build_pairwise_comparisons(
        base=base,
        manifest=manifest,
        all_oof=all_oof,
    )

    summary_with_upstream = merge_upstream_hkfi_metrics(summary_df)

    summary_with_upstream.to_csv(
        dirs["out"] / "LR_7_3B_HKFI_Weight_Sensitivity_Performance.csv",
        index=False,
        float_format="%.10f",
    )
    comparison_df.to_csv(
        dirs["out"] / "LR_7_3B_HKFI_Weight_Sensitivity_Paired_AUC.csv",
        index=False,
        float_format="%.10f",
    )
    qa_df.to_csv(
        dirs["out"] / "LR_7_3B_HKFI_Weight_Sensitivity_QA.csv",
        index=False,
        float_format="%.10f",
    )

    # -------------------------------------------------------------------------
    # 2. Full-domain LR map sensitivity.
    #
    # Deliberately disabled for this targeted cross-algorithm analysis because
    # Step 7.1B already completed the full XGBoost map-sensitivity experiment.
    # -------------------------------------------------------------------------
    if SensitivityConfig.GENERATE_MAP_SENSITIVITY:
        for scenario_name in manifest["scenario"]:
            paths = scenario_paths(base, scenario_name)

            if (
                SensitivityConfig.RESUME_FROM_CHECKPOINTS
                and paths["map_done"].exists()
                and paths["model_info"].exists()
            ):
                log(f"[RESUME] Map stage already complete: {scenario_name}")
                continue

            run_external_worker("map", scenario_name)

        map_model_df, map_comparison_df = build_map_comparisons(
            base=base,
            manifest=manifest,
        )

        map_model_df.to_csv(
            dirs["out"] / "LR_7_3B_HKFI_Weight_Sensitivity_Map_Models.csv",
            index=False,
        )

        map_comparison_df.to_csv(
            dirs["out"] / "LR_7_3B_HKFI_Weight_Sensitivity_Map_Comparison.csv",
            index=False,
            float_format="%.10f",
        )

    else:
        map_model_df = pd.DataFrame()
        map_comparison_df = pd.DataFrame()

        log(
            "[MAP] Skipped by design: Step 7.3B is a performance-only "
            "cross-algorithm robustness analysis."
        )

    create_performance_figure(
        summary_df=summary_df,
        output_path=(
            dirs["out"]
            / "LR_7_3B_HKFI_Weight_Sensitivity_Performance.png"
        ),
    )

    write_summary_report(
        base=base,
        manifest=manifest,
        summary_df=summary_with_upstream,
        comparison_df=comparison_df,
        qa_df=qa_df,
        map_comparison_df=map_comparison_df,
    )

    log("")
    log("=" * 96)
    log("STEP 7.3B COMPLETED SUCCESSFULLY")
    log("=" * 96)
    log(f"[OUT] {dirs['out']}")
    log("")
    log("Nested spatial OOF performance:")
    log(summary_df.to_string(index=False))
    if not comparison_df.empty:
        log("")
        log("Paired delta-AUC versus primary equal weighting:")
        log(comparison_df.to_string(index=False))


# =============================================================================
# COMMAND-LINE ENTRY POINT
# =============================================================================
def parse_command_line_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Step 7.3B Logistic Regression HKFI weight sensitivity using authoritative "
            "Step 5.3 named scenarios."
        )
    )
    parser.add_argument(
        "--worker-mode",
        choices=["nested", "map"],
        default=None,
    )
    parser.add_argument(
        "--scenario",
        default=None,
    )

    arguments, unknown = parser.parse_known_args()

    # Jupyter/IPython injects unrelated arguments such as -f kernel.json.
    if unknown:
        log(f"[CLI] Ignoring host-environment arguments: {unknown}")

    return arguments


if __name__ == "__main__":
    try:
        args = parse_command_line_arguments()

        if args.worker_mode is None:
            run_controller()
        else:
            if not args.scenario:
                raise RuntimeError(
                    "--scenario is required when --worker-mode is supplied."
                )

            if args.worker_mode == "nested":
                run_nested_worker(args.scenario)
            elif args.worker_mode == "map":
                run_map_worker(args.scenario)
            else:
                raise RuntimeError(
                    f"Unsupported worker mode: {args.worker_mode}"
                )

    except Exception as exc:
        print(f"[ERROR] Step 7.3B failed: {exc}", flush=True)
        traceback.print_exc()
        raise


# STEP 7.3C — LOGISTIC REGRESSION PSEUDO-ABSENCE BUFFER SENSITIVITY

In [ ]:
# =============================================================================
# STEP 7.3C — LOGISTIC REGRESSION PSEUDO-ABSENCE BUFFER SENSITIVITY
# =============================================================================
#
# Scientific purpose
# ------------------
# This script is the Logistic Regression counterpart to Step 7.1C (XGBoost)
# and Step 7.2C (Random Forest). It evaluates whether the primary conclusion
# that the physics-guided Model E improves over the static Model A is sensitive
# to pseudo-absence exclusion distance under the regularized LR algorithm.
#
# The experimental design exactly mirrors Step 7.1C:
#
#   Buffers:
#       100 m   -> primary Step 6.4 model-ready dataset
#       500 m   -> Step 6.3 buffer-specific dataset
#       1000 m  -> Step 6.3 buffer-specific dataset
#
#   Models:
#       Model A -> static baseline
#       Model E -> static + HKFI + SNDR
#
# Scientific isolation rules
# --------------------------
# 1. The same 165 positive landslide samples must be present at all buffers.
# 2. Positive predictor values and spatial-fold assignments must be identical.
# 3. Only the pseudo-absence sample set changes with exclusion distance.
# 4. Each buffer/model combination is independently nested-tuned using the exact
#    successful Step 7.3 LR regularization grid and fold-safe preprocessing.
# 5. Hyperparameters are NOT transferred between buffers.
# 6. The primary reviewer statistic is paired E - A ROC-AUC within each buffer.
# 7. The 100 m Model A and Model E reruns must reproduce the successful Step 7.3
#    LR OOF predictions.
#
# Plotting rule
# -------------
# The final plot intentionally reproduces the exact Step 7.1C plotting style:
#   - figure size = (5.8, 4.2)
#   - Model A and Model E shown as connected error-bar lines
#   - x-axis buffers = 100, 500, 1000 m
#   - y-axis = Nested OOF ROC-AUC
#   - asymmetric 95% spatial-block bootstrap error bars
#   - marker = "o", capsize = 3, linewidth = 1.4
#   - same axis labels, grid and legend style
#
# Output directory
# ----------------
# ./data/Results/
#     Step-7.3-LR-Models_A_to_F/
#     Step-7.3C-Buffer_Sensitivity/
#
# =============================================================================

from __future__ import annotations

import gc
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Sequence

# Resource-safe execution.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# =============================================================================
# CONFIGURATION — EXACT SUCCESSFUL STEP 7.2 LR CONTRACT
# =============================================================================
class BufferConfig:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Step 6.1 / Step 6.4 predictor definitions.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = (
        STEP61_DIR / "model_ablation_scenarios.json"
    )

    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    STEP64_MODEL_CONFIG_JSON = (
        STEP64_DIR / "model_configuration_manifest.json"
    )

    # Step 6.3 buffer-specific datasets.
    STEP63_DIR = (
        RESULTS_DIR / "Step-6.3-Site_Level_Sampling"
    )

    # Normal authoritative workflow:
    #   100 m  -> exact Step 6.4 model-ready dataset
    #   500 m  -> Step 6.3 buffer dataset
    #   1000 m -> Step 6.3 buffer dataset
    DEFAULT_BUFFER_DATASETS = {
        100: (
            STEP64_DIR
            / "model_ready_dataset_A_to_F.csv"
        ),
        500: (
            STEP63_DIR
            / "common_site_level_samples_A_to_F_buffer500m.csv"
        ),
        1000: (
            STEP63_DIR
            / "common_site_level_samples_A_to_F_buffer1000m.csv"
        ),
    }

    # Optional override manifest. Not required for the normal workflow.
    MANIFEST_PATH = (
        STEP64_DIR
        / "buffer_sensitivity"
        / "buffer_sensitivity_manifest.json"
    )

    # Optional Step 6.4 QA record.
    STEP64_BUFFER_QA_CSV = (
        STEP64_DIR
        / "buffer_dataset_consistency_QA.csv"
    )

    # Successful Step 7.3 LR outputs.
    STEP73_OUT_DIR = (
        RESULTS_DIR / "Step-7.3-LR-Models_A_to_F"
    )
    PRIMARY_STEP73_OOF_CSV = (
        STEP73_OUT_DIR / "LR_A_to_F_OOF_predictions.csv"
    )

    # Step 7.3C output.
    OUT_DIR = (
        STEP73_OUT_DIR
        / "Step-7.3C-Buffer_Sensitivity"
    )

    # Models evaluated exactly as in Step 7.1C.
    SCENARIOS = (
        "A_static_baseline",
        "E_physics_guided",
    )

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    # Exact successful Step 7.3 Logistic Regression regularization grid.
    #
    # Scientific contract:
    #   - solver = liblinear
    #   - penalty = L2 / Ridge
    #   - class_weight = balanced
    #   - numeric predictors standardized fold-safely with StandardScaler
    #
    # Only C, the inverse regularization strength, is tuned independently
    # within each buffer/model nested spatial-CV analysis.
    PARAM_GRID = {
        "model__C": [
            0.01,
            0.05,
            0.10,
            0.50,
            1.00,
            5.00,
            10.00,
            100.00,
        ],
    }

    LR_FIXED_PARAMS = {
        "penalty": "l2",
        "solver": "liblinear",
        "class_weight": "balanced",
        "fit_intercept": True,
        "max_iter": 10000,
        "tol": 1.0e-6,
        "random_state": 42,
    }

    GRIDSEARCH_N_JOBS = 1
    GRIDSEARCH_VERBOSE = 2

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    CATEGORICAL_FEATURE_ALIASES = {
        "landcover",
        "landuse",
        "land_use",
    }

    BINARY_THRESHOLD = 0.50

    # Spatial-block bootstrap.
    N_CLUSTER_BOOTSTRAP = 5000
    CI_LEVEL = 0.95

    REFERENCE_BUFFER_M = 100
    RANDOM_STATE = 52371

    # Step 7.3 OOF probabilities are saved to finite decimal precision.
    OOF_REPRODUCTION_TOLERANCE = 1.0e-8

    # Positive rows should be identical across buffer experiments. The 100 m
    # Step 6.4 CSV and Step 6.3 CSVs can differ only through harmless decimal
    # serialization. This tolerance is intentionally strict.
    POSITIVE_PREDICTOR_EQUALITY_ATOL = 5.0e-8

    FIG_DPI = 300


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(
        f"[{datetime.now():%H:%M:%S}] {message}",
        flush=True,
    )


def utc_now_iso() -> str:
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def require_file(
    path: Path,
    description: str,
) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}"
        )


def load_json(
    path: Path,
) -> Dict[str, Any]:
    require_file(
        path,
        "JSON file",
    )

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def json_safe(
    value: Any,
) -> Any:
    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    return value


def normalize_feature_name(
    name: str,
) -> str:
    return re.sub(
        r"[^a-z0-9_]+",
        "_",
        str(
            name
        ).strip().lower(),
    ).strip(
        "_"
    )


def is_categorical_feature(
    name: str,
) -> bool:
    normalized = normalize_feature_name(
        name
    )

    compact = normalized.replace(
        "_",
        "",
    )

    aliases = {
        alias.lower()
        for alias
        in BufferConfig.CATEGORICAL_FEATURE_ALIASES
    }

    compact_aliases = {
        alias.replace(
            "_",
            "",
        )
        for alias
        in aliases
    }

    return (
        normalized in aliases
        or compact in compact_aliases
    )


# =============================================================================
# STEP 6.1 / STEP 6.4 SCENARIO CONTRACT
# =============================================================================
def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """
    Verify that Step 6.1 and Step 6.4 define Models A-F identically.
    """
    step61 = load_json(
        BufferConfig.SCENARIO_MANIFEST_JSON
    ).get(
        "scenarios",
        {},
    )

    step64 = load_json(
        BufferConfig.STEP64_MODEL_CONFIG_JSON
    ).get(
        "scenarios",
        {},
    )

    for scenario_name in BufferConfig.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                "Step 6.1 scenario manifest is missing "
                f"{scenario_name}."
            )

        if scenario_name not in step64:
            raise RuntimeError(
                "Step 6.4 model configuration is missing "
                f"{scenario_name}."
            )

        features_61 = list(
            step61[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        features_64 = list(
            step64[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any(
            "irs" in feature.lower()
            for feature in features_61
        ):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; "
                "IRS must not be an ML predictor."
            )

    log(
        "[QA] Step 6.1 and Step 6.4 A-F scenario definitions match: PASS"
    )

    return step64


def all_scenario_features(
    scenarios: Dict[str, Dict[str, Any]],
) -> list[str]:
    features: list[str] = []

    for scenario_name in BufferConfig.SCENARIO_ORDER:
        for feature in scenarios[
            scenario_name
        ][
            "features"
        ]:
            if feature not in features:
                features.append(
                    feature
                )

    return features


# =============================================================================
# BUFFER INPUT RESOLUTION
# =============================================================================
def load_buffer_paths(
    path: Path,
) -> Dict[int, Path]:
    """
    Resolve 100, 500 and 1000 m datasets.

    Normal workflow:
      100 m  -> Step 6.4 model_ready_dataset_A_to_F.csv
      500 m  -> Step 6.3 common_site_level_samples_A_to_F_buffer500m.csv
      1000 m -> Step 6.3 common_site_level_samples_A_to_F_buffer1000m.csv

    A JSON override manifest is supported but not required.
    """
    if path.exists():
        with open(
            path,
            "r",
            encoding="utf-8",
        ) as handle:
            manifest = json.load(
                handle
            )

        buffers = manifest.get(
            "buffers",
            {},
        )

        required = {
            "100",
            "500",
            "1000",
        }

        if not required.issubset(
            set(
                buffers
            )
        ):
            raise RuntimeError(
                "Buffer manifest must define 100, 500 and 1000 m. "
                f"Found: {sorted(buffers)}"
            )

        paths = {
            int(
                buffer_text
            ): Path(
                info[
                    "dataset_csv"
                ]
            )
            for (
                buffer_text,
                info,
            )
            in buffers.items()
            if int(
                buffer_text
            )
            in {
                100,
                500,
                1000,
            }
        }

    else:
        paths = dict(
            BufferConfig.DEFAULT_BUFFER_DATASETS
        )

    missing = [
        f"{buffer_m} m: {dataset_path}"
        for (
            buffer_m,
            dataset_path,
        )
        in paths.items()
        if not dataset_path.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Step 7.3C could not find one or more authoritative buffer "
            "datasets.\n"
            "The normal workflow expects 100 m under Step 6.4 and 500/1000 m "
            "under Step 6.3.\nMissing:\n"
            + "\n".join(
                missing
            )
        )

    return paths


def check_optional_step64_buffer_qa() -> None:
    """
    Read the Step 6.4 buffer QA table when available.
    Step 7.3C independently repeats all essential modelling checks.
    """
    path = (
        BufferConfig.STEP64_BUFFER_QA_CSV
    )

    if not path.exists():
        log(
            "[7.3C-WARN] Optional Step 6.4 buffer QA table was not found: "
            f"{path}"
        )
        return

    qa = pd.read_csv(
        path
    )

    if "buffer_m" not in qa.columns:
        raise RuntimeError(
            "Step 6.4 buffer QA file has no buffer_m column: "
            f"{path}"
        )

    for buffer_m in (
        100,
        500,
        1000,
    ):
        rows = qa.loc[
            pd.to_numeric(
                qa[
                    "buffer_m"
                ],
                errors="coerce",
            )
            == buffer_m
        ]

        if rows.empty:
            raise RuntimeError(
                "Step 6.4 buffer QA table has no row for "
                f"{buffer_m} m."
            )

        row = rows.iloc[
            0
        ]

        if "status" in qa.columns:
            status = str(
                row[
                    "status"
                ]
            ).strip().upper()

            if status not in {
                "PASS",
                "OK",
                "TRUE",
            }:
                raise RuntimeError(
                    "Step 6.4 buffer QA status is not PASS for "
                    f"{buffer_m} m: {row['status']}"
                )

        for column in (
            "same_positive_site_ids_as_100m",
            "same_positive_block_assignments_as_100m",
        ):
            if column in qa.columns:
                value = str(
                    row[
                        column
                    ]
                ).strip().lower()

                if value not in {
                    "true",
                    "1",
                    "yes",
                    "pass",
                }:
                    raise RuntimeError(
                        "Step 6.4 buffer QA failed "
                        f"{column} for {buffer_m} m."
                    )

    log(
        "[7.3C-QA] Step 6.4 buffer_dataset_consistency_QA.csv: PASS"
    )


# =============================================================================
# BUFFER DATASET VALIDATION
# =============================================================================
def load_and_validate_buffer_dataset(
    scenarios: Dict[str, Dict[str, Any]],
    dataset_csv: Path,
    buffer_m: int,
) -> pd.DataFrame:
    """
    Convert one buffer dataset to the exact Step 7.3 LR modelling contract.

    For 100 m the exact Step 6.4 model-ready dataset is used.
    For 500/1000 m, cv_fold is deterministically set equal to spatial_block
    when the alias is absent.
    """
    require_file(
        dataset_csv,
        f"{buffer_m} m buffer dataset",
    )

    df = pd.read_csv(
        dataset_csv
    )

    # Step 6.3 contains spatial_block. Step 7.3 uses the identical cv_fold alias.
    if (
        BufferConfig.BLOCK_COLUMN
        not in df.columns
    ):
        if (
            BufferConfig.FOLD_COLUMN
            not in df.columns
        ):
            raise RuntimeError(
                f"{buffer_m} m dataset contains neither "
                f"{BufferConfig.BLOCK_COLUMN} nor "
                f"{BufferConfig.FOLD_COLUMN}."
            )

        df[
            BufferConfig.BLOCK_COLUMN
        ] = pd.to_numeric(
            df[
                BufferConfig.FOLD_COLUMN
            ],
            errors="raise",
        ).astype(
            int
        )

    if (
        BufferConfig.FOLD_COLUMN
        not in df.columns
    ):
        df[
            BufferConfig.FOLD_COLUMN
        ] = pd.to_numeric(
            df[
                BufferConfig.BLOCK_COLUMN
            ],
            errors="raise",
        ).astype(
            int
        )

    all_features = all_scenario_features(
        scenarios
    )

    required = [
        BufferConfig.SAMPLE_ID_COLUMN,
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
        BufferConfig.BLOCK_COLUMN,
        *all_features,
    ]

    missing = [
        column
        for column in required
        if column not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{buffer_m} m dataset is missing required modelling columns: "
            f"{missing}"
        )

    if len(
        df
    ) != BufferConfig.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"{buffer_m} m: expected "
            f"{BufferConfig.EXPECTED_N_ROWS} rows; found {len(df)}."
        )

    if df[
        BufferConfig.SAMPLE_ID_COLUMN
    ].isna().any():
        raise RuntimeError(
            f"{buffer_m} m: missing sample_id values."
        )

    if df[
        BufferConfig.SAMPLE_ID_COLUMN
    ].duplicated().any():
        duplicates = df.loc[
            df[
                BufferConfig.SAMPLE_ID_COLUMN
            ].duplicated(
                keep=False
            ),
            BufferConfig.SAMPLE_ID_COLUMN,
        ].tolist()

        raise RuntimeError(
            f"{buffer_m} m: duplicated sample_id values found: "
            f"{duplicates[:20]}"
        )

    for column in (
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
        BufferConfig.BLOCK_COLUMN,
    ):
        df[
            column
        ] = pd.to_numeric(
            df[
                column
            ],
            errors="raise",
        ).astype(
            int
        )

    observed_class_counts = (
        df[
            BufferConfig.TARGET_COLUMN
        ]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    expected_class_counts = {
        0: BufferConfig.EXPECTED_N_NEGATIVE,
        1: BufferConfig.EXPECTED_N_POSITIVE,
    }

    if (
        observed_class_counts
        != expected_class_counts
    ):
        raise RuntimeError(
            f"{buffer_m} m: unexpected class counts "
            f"{observed_class_counts}; expected {expected_class_counts}."
        )

    observed_folds = sorted(
        df[
            BufferConfig.FOLD_COLUMN
        ].unique().tolist()
    )

    if (
        observed_folds
        != BufferConfig.EXPECTED_FOLDS
    ):
        raise RuntimeError(
            f"{buffer_m} m: expected frozen folds "
            f"{BufferConfig.EXPECTED_FOLDS}; found {observed_folds}."
        )

    if not np.array_equal(
        df[
            BufferConfig.FOLD_COLUMN
        ].to_numpy(),
        df[
            BufferConfig.BLOCK_COLUMN
        ].to_numpy(),
    ):
        raise RuntimeError(
            f"{buffer_m} m: cv_fold and spatial_block are not identical."
        )

    block_class = pd.crosstab(
        df[
            BufferConfig.FOLD_COLUMN
        ],
        df[
            BufferConfig.TARGET_COLUMN
        ],
    )

    expected_per_class = (
        BufferConfig.EXPECTED_N_POSITIVE
        // len(
            BufferConfig.EXPECTED_FOLDS
        )
    )

    for fold in BufferConfig.EXPECTED_FOLDS:
        if fold not in block_class.index:
            raise RuntimeError(
                f"{buffer_m} m: missing fold {fold}."
            )

        for label in (
            0,
            1,
        ):
            if label not in block_class.columns:
                raise RuntimeError(
                    f"{buffer_m} m, fold {fold}: missing class {label}."
                )

            count = int(
                block_class.loc[
                    fold,
                    label,
                ]
            )

            if (
                count
                != expected_per_class
            ):
                raise RuntimeError(
                    f"{buffer_m} m, fold {fold}, class {label}: "
                    f"found {count}; expected {expected_per_class}."
                )

    predictor_values = df[
        all_features
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    finite = np.isfinite(
        predictor_values.to_numpy(
            dtype="float64"
        )
    )

    if not finite.all():
        bad = np.argwhere(
            ~finite
        )

        examples = [
            (
                str(
                    df.iloc[
                        row_index
                    ][
                        BufferConfig.SAMPLE_ID_COLUMN
                    ]
                ),
                all_features[
                    column_index
                ],
            )
            for (
                row_index,
                column_index,
            )
            in bad[
                :20
            ]
        ]

        raise RuntimeError(
            f"{buffer_m} m: missing/non-finite predictor values remain. "
            f"Examples: {examples}"
        )

    irs_columns = [
        column
        for column in df.columns
        if "irs" in str(
            column
        ).lower()
    ]

    if irs_columns:
        raise RuntimeError(
            f"{buffer_m} m: IRS-related columns found: "
            f"{irs_columns}"
        )

    if "tpi_s25" in df.columns:
        raise RuntimeError(
            f"{buffer_m} m: tpi_s25 found; only tpi_s5 is allowed."
        )

    if "buffer_m" in df.columns:
        observed_buffer_values = sorted(
            pd.to_numeric(
                df[
                    "buffer_m"
                ],
                errors="coerce",
            )
            .dropna()
            .unique()
            .tolist()
        )

        contradictory = [
            value
            for value
            in observed_buffer_values
            if value not in {
                0,
                100,
                buffer_m,
            }
        ]

        if contradictory:
            raise RuntimeError(
                f"{buffer_m} m: contradictory buffer_m metadata values: "
                f"{contradictory}"
            )

    log(
        f"[7.3C-QA] {buffer_m} m model-ready conversion: PASS | "
        f"rows={len(df)}, positive={BufferConfig.EXPECTED_N_POSITIVE}, "
        f"negative={BufferConfig.EXPECTED_N_NEGATIVE}, "
        f"folds={observed_folds}"
    )

    return df


def validate_primary_100m_dataset(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
) -> None:
    """
    Verify that the in-memory 100 m dataset exactly matches the authoritative
    Step 6.4 model-ready CSV.
    """
    reference_path = (
        BufferConfig.DEFAULT_BUFFER_DATASETS[
            100
        ]
    )

    require_file(
        reference_path,
        "authoritative Step 6.4 100 m model-ready dataset",
    )

    reference = pd.read_csv(
        reference_path
    )

    if len(
        df
    ) != len(
        reference
    ):
        raise RuntimeError(
            "100 m dataset row count does not match the authoritative "
            "Step 6.4 dataset."
        )

    if not np.array_equal(
        df[
            BufferConfig.SAMPLE_ID_COLUMN
        ].astype(
            str
        ).to_numpy(),
        reference[
            BufferConfig.SAMPLE_ID_COLUMN
        ].astype(
            str
        ).to_numpy(),
    ):
        raise RuntimeError(
            "100 m sample IDs/order do not match the authoritative "
            "Step 6.4 dataset."
        )

    compare_columns = [
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
        BufferConfig.BLOCK_COLUMN,
        *all_scenario_features(
            scenarios
        ),
    ]

    for column in compare_columns:
        if column not in reference.columns:
            raise RuntimeError(
                "Authoritative 100 m dataset is missing column: "
                f"{column}"
            )

        left = df[
            column
        ]

        right = reference[
            column
        ]

        if (
            pd.api.types.is_numeric_dtype(
                left
            )
            and pd.api.types.is_numeric_dtype(
                right
            )
        ):
            equal = np.allclose(
                left.to_numpy(
                    dtype=float
                ),
                right.to_numpy(
                    dtype=float
                ),
                rtol=0.0,
                atol=1.0e-12,
                equal_nan=True,
            )

        else:
            equal = np.array_equal(
                left.astype(
                    str
                ).to_numpy(),
                right.astype(
                    str
                ).to_numpy(),
            )

        if not equal:
            raise RuntimeError(
                "100 m dataset differs from authoritative Step 6.4 in "
                f"column: {column}"
            )

    log(
        "[7.3C-QA] 100 m dataset exactly matches Step 6.4 primary input: PASS"
    )


def validate_same_positive_samples_across_buffers(
    reference_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
    predictor_columns: Sequence[str],
    candidate_buffer_m: int,
) -> None:
    """
    Verify that positive landslide samples are identical across buffer choices.

    Pseudo-absence rows are expected to change. Positive sample IDs, labels,
    spatial blocks and predictor values must remain fixed.
    """
    reference_positive = (
        reference_df.loc[
            reference_df[
                BufferConfig.TARGET_COLUMN
            ]
            == 1
        ]
        .copy()
        .sort_values(
            BufferConfig.SAMPLE_ID_COLUMN
        )
        .reset_index(
            drop=True
        )
    )

    candidate_positive = (
        candidate_df.loc[
            candidate_df[
                BufferConfig.TARGET_COLUMN
            ]
            == 1
        ]
        .copy()
        .sort_values(
            BufferConfig.SAMPLE_ID_COLUMN
        )
        .reset_index(
            drop=True
        )
    )

    if (
        len(
            reference_positive
        )
        != BufferConfig.EXPECTED_N_POSITIVE
        or len(
            candidate_positive
        )
        != BufferConfig.EXPECTED_N_POSITIVE
    ):
        raise RuntimeError(
            f"{candidate_buffer_m} m: unexpected positive sample count."
        )

    if not np.array_equal(
        reference_positive[
            BufferConfig.SAMPLE_ID_COLUMN
        ].astype(
            str
        ).to_numpy(),
        candidate_positive[
            BufferConfig.SAMPLE_ID_COLUMN
        ].astype(
            str
        ).to_numpy(),
    ):
        raise RuntimeError(
            f"{candidate_buffer_m} m: positive sample IDs differ from 100 m."
        )

    for column in (
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
        BufferConfig.BLOCK_COLUMN,
    ):
        if not np.array_equal(
            reference_positive[
                column
            ].to_numpy(),
            candidate_positive[
                column
            ].to_numpy(),
        ):
            raise RuntimeError(
                f"{candidate_buffer_m} m: positive rows differ in {column}."
            )

    mismatched_predictors: list[
        tuple[
            str,
            float,
        ]
    ] = []

    for column in predictor_columns:
        reference_values = reference_positive[
            column
        ].to_numpy(
            dtype=float
        )

        candidate_values = candidate_positive[
            column
        ].to_numpy(
            dtype=float
        )

        max_absolute_difference = float(
            np.max(
                np.abs(
                    reference_values
                    - candidate_values
                )
            )
        )

        if (
            max_absolute_difference
            > BufferConfig.POSITIVE_PREDICTOR_EQUALITY_ATOL
        ):
            mismatched_predictors.append(
                (
                    column,
                    max_absolute_difference,
                )
            )

    if mismatched_predictors:
        details = "\n".join(
            (
                f"  {column}: "
                f"max_abs_difference={difference:.12g}"
            )
            for (
                column,
                difference,
            )
            in mismatched_predictors
        )

        raise RuntimeError(
            f"{candidate_buffer_m} m: positive predictor values differ "
            "from the authoritative 100 m positives beyond tolerance "
            f"{BufferConfig.POSITIVE_PREDICTOR_EQUALITY_ATOL:.3g}.\n"
            f"{details}"
        )

    log(
        f"[7.3C-QA] {candidate_buffer_m} m positives match 100 m "
        "sample IDs, folds and predictors: PASS"
    )


# =============================================================================
# LOGISTIC REGRESSION PIPELINE
# =============================================================================
def build_lr_pipeline(
    features: Sequence[str],
) -> Pipeline:
    """
    Exact fold-safe preprocessing + Logistic Regression pipeline used in Step 7.3.

    Numeric predictors are standardized with StandardScaler fitted only on the
    current training partition. Land cover is one-hot encoded inside the same
    pipeline. Therefore preprocessing is repeated leakage-safely within every
    inner-CV fit and outer-fold refit.
    """
    categorical = [
        feature
        for feature in features
        if is_categorical_feature(
            feature
        )
    ]

    numeric = [
        feature
        for feature in features
        if not is_categorical_feature(
            feature
        )
    ]

    transformers = []

    if numeric:
        transformers.append(
            (
                "numeric",
                StandardScaler(),
                numeric,
            )
        )

    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = LogisticRegression(
        **BufferConfig.LR_FIXED_PARAMS
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                classifier,
            ),
        ]
    )


def make_inner_predefined_split(
    outer_train_df: pd.DataFrame,
) -> PredefinedSplit:
    """
    Four-fold spatial inner CV using the four blocks remaining after one outer
    block is held out.
    """
    blocks = sorted(
        outer_train_df[
            BufferConfig.FOLD_COLUMN
        ].unique().tolist()
    )

    if len(
        blocks
    ) != 4:
        raise RuntimeError(
            "Nested CV expected four inner spatial blocks; "
            f"found {blocks}."
        )

    block_to_test_fold = {
        block: index
        for (
            index,
            block,
        )
        in enumerate(
            blocks
        )
    }

    test_fold = (
        outer_train_df[
            BufferConfig.FOLD_COLUMN
        ]
        .map(
            block_to_test_fold
        )
        .to_numpy()
    )

    return PredefinedSplit(
        test_fold=test_fold
    )


def classification_metrics(
    y_true: np.ndarray,
    y_probability: np.ndarray,
    threshold: float = (
        BufferConfig.BINARY_THRESHOLD
    ),
) -> Dict[str, Any]:
    predicted = (
        y_probability
        >= threshold
    ).astype(
        int
    )

    (
        tn,
        fp,
        fn,
        tp,
    ) = confusion_matrix(
        y_true,
        predicted,
        labels=[
            0,
            1,
        ],
    ).ravel()

    return {
        "roc_auc": float(
            roc_auc_score(
                y_true,
                y_probability,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                y_probability,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                y_probability,
            )
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                predicted,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predicted,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "threshold": float(
            threshold
        ),
        "tn": int(
            tn
        ),
        "fp": int(
            fp
        ),
        "fn": int(
            fn
        ),
        "tp": int(
            tp
        ),
    }


def normalize_grid_results(
    cv_results: Dict[str, Any],
    scenario_name: str,
    outer_fold: int,
) -> pd.DataFrame:
    frame = pd.DataFrame(
        cv_results
    )

    keep = [
        column
        for column in frame.columns
        if (
            column.startswith(
                "param_"
            )
            or column
            in {
                "mean_test_score",
                "std_test_score",
                "rank_test_score",
                "mean_fit_time",
                "std_fit_time",
            }
        )
    ]

    frame = frame[
        keep
    ].copy()

    frame.insert(
        0,
        "scenario",
        scenario_name,
    )

    frame.insert(
        1,
        "model",
        BufferConfig.MODEL_SHORT_LABELS[
            scenario_name
        ],
    )

    frame.insert(
        2,
        "search_stage",
        "nested_inner",
    )

    frame.insert(
        3,
        "outer_fold",
        int(
            outer_fold
        ),
    )

    return frame


def run_nested_spatial_cv_for_scenario(
    df: pd.DataFrame,
    scenario_name: str,
    features: Sequence[str],
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    Dict[str, Any],
    pd.DataFrame,
]:
    """
    Exact successful Step 7.3 nested LR contract:
      five outer spatial folds;
      four-block inner GridSearchCV;
      eight candidate C regularization strengths;
      fold-safe StandardScaler + OneHotEncoder preprocessing.
    """
    oof_probability = np.full(
        len(
            df
        ),
        np.nan,
        dtype="float64",
    )

    oof_predicted = np.full(
        len(
            df
        ),
        -1,
        dtype="int16",
    )

    fold_rows: list[
        Dict[str, Any]
    ] = []

    best_params: Dict[
        str,
        Any,
    ] = {}

    search_tables: list[
        pd.DataFrame
    ] = []

    for outer_fold in BufferConfig.EXPECTED_FOLDS:
        outer_train_mask = (
            df[
                BufferConfig.FOLD_COLUMN
            ].to_numpy()
            != outer_fold
        )

        outer_valid_mask = (
            df[
                BufferConfig.FOLD_COLUMN
            ].to_numpy()
            == outer_fold
        )

        train_df = df.loc[
            outer_train_mask
        ].copy()

        valid_df = df.loc[
            outer_valid_mask
        ].copy()

        if train_df[
            BufferConfig.FOLD_COLUMN
        ].nunique() != 4:
            raise RuntimeError(
                f"Outer fold {outer_fold}: "
                "expected four training blocks."
            )

        if valid_df[
            BufferConfig.FOLD_COLUMN
        ].nunique() != 1:
            raise RuntimeError(
                f"Outer fold {outer_fold}: "
                "validation must contain exactly one block."
            )

        X_train = train_df[
            list(
                features
            )
        ]

        y_train = train_df[
            BufferConfig.TARGET_COLUMN
        ].to_numpy(
            dtype=int
        )

        X_valid = valid_df[
            list(
                features
            )
        ]

        y_valid = valid_df[
            BufferConfig.TARGET_COLUMN
        ].to_numpy(
            dtype=int
        )

        inner_cv = make_inner_predefined_split(
            train_df
        )

        search = GridSearchCV(
            estimator=build_lr_pipeline(
                features
            ),
            param_grid=BufferConfig.PARAM_GRID,
            scoring="roc_auc",
            cv=inner_cv,
            refit=True,
            n_jobs=BufferConfig.GRIDSEARCH_N_JOBS,
            verbose=BufferConfig.GRIDSEARCH_VERBOSE,
            return_train_score=False,
            error_score="raise",
        )

        log(
            f"[OUTER {outer_fold}] LR tuning | "
            f"{BufferConfig.MODEL_SHORT_LABELS[scenario_name]} | "
            f"training blocks="
            f"{sorted(train_df[BufferConfig.FOLD_COLUMN].unique().tolist())} | "
            f"held-out block={outer_fold}"
        )

        search.fit(
            X_train,
            y_train,
        )

        probability = (
            search.best_estimator_.predict_proba(
                X_valid
            )[
                :,
                1,
            ]
        )

        predicted = (
            probability
            >= BufferConfig.BINARY_THRESHOLD
        ).astype(
            int
        )

        valid_indices = np.flatnonzero(
            outer_valid_mask
        )

        oof_probability[
            valid_indices
        ] = probability

        oof_predicted[
            valid_indices
        ] = predicted

        metrics = classification_metrics(
            y_valid,
            probability,
        )

        fold_rows.append(
            {
                "scenario": scenario_name,
                "model": BufferConfig.MODEL_SHORT_LABELS[
                    scenario_name
                ],
                "outer_fold": int(
                    outer_fold
                ),
                "n_predictors": int(
                    len(
                        features
                    )
                ),
                "n_train": int(
                    len(
                        train_df
                    )
                ),
                "n_validation": int(
                    len(
                        valid_df
                    )
                ),
                "inner_best_mean_auc": float(
                    search.best_score_
                ),
                **metrics,
            }
        )

        best_params[
            str(
                outer_fold
            )
        ] = {
            "inner_best_mean_auc": float(
                search.best_score_
            ),
            "best_params": json_safe(
                search.best_params_
            ),
        }

        search_tables.append(
            normalize_grid_results(
                search.cv_results_,
                scenario_name=scenario_name,
                outer_fold=outer_fold,
            )
        )

        log(
            f"[OUTER {outer_fold}] "
            f"inner best AUC={search.best_score_:.4f}; "
            f"held-out AUC={metrics['roc_auc']:.4f}; "
            f"F1={metrics['f1']:.4f}"
        )

        del search
        gc.collect()

    if not np.isfinite(
        oof_probability
    ).all():
        raise RuntimeError(
            f"{scenario_name}: incomplete OOF probability predictions."
        )

    if (
        oof_predicted
        < 0
    ).any():
        raise RuntimeError(
            f"{scenario_name}: incomplete OOF class predictions."
        )

    oof_df = df[
        [
            BufferConfig.SAMPLE_ID_COLUMN,
            BufferConfig.TARGET_COLUMN,
            BufferConfig.FOLD_COLUMN,
            BufferConfig.BLOCK_COLUMN,
        ]
    ].copy()

    oof_df.insert(
        0,
        "scenario",
        scenario_name,
    )

    oof_df.insert(
        1,
        "model",
        BufferConfig.MODEL_SHORT_LABELS[
            scenario_name
        ],
    )

    oof_df[
        "oof_probability"
    ] = oof_probability

    oof_df[
        "oof_predicted_class_0p5"
    ] = oof_predicted

    return (
        oof_df,
        pd.DataFrame(
            fold_rows
        ),
        best_params,
        pd.concat(
            search_tables,
            ignore_index=True,
        ),
    )


# =============================================================================
# SPATIAL-BLOCK BOOTSTRAP
# =============================================================================
def cluster_bootstrap_auc_ci(
    y_true: np.ndarray,
    y_probability: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> tuple[
    float,
    float,
    int,
]:
    rng = np.random.default_rng(
        seed
    )

    unique_blocks = np.unique(
        blocks
    )

    indices_by_block = {
        block: np.flatnonzero(
            blocks
            == block
        )
        for block
        in unique_blocks
    }

    values: list[
        float
    ] = []

    for _ in range(
        n_bootstrap
    ):
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(
                unique_blocks
            ),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                indices_by_block[
                    block
                ]
                for block
                in sampled_blocks
            ]
        )

        y_bootstrap = y_true[
            sampled_indices
        ]

        if np.unique(
            y_bootstrap
        ).size < 2:
            continue

        values.append(
            float(
                roc_auc_score(
                    y_bootstrap,
                    y_probability[
                        sampled_indices
                    ],
                )
            )
        )

    if not values:
        return (
            float(
                "nan"
            ),
            float(
                "nan"
            ),
            0,
        )

    alpha = (
        1.0
        - ci_level
    ) / 2.0

    low, high = np.quantile(
        values,
        [
            alpha,
            1.0
            - alpha,
        ],
    )

    return (
        float(
            low
        ),
        float(
            high
        ),
        len(
            values
        ),
    )


def paired_cluster_bootstrap_auc_difference(
    y_true: np.ndarray,
    probability_left: np.ndarray,
    probability_right: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> tuple[
    float,
    float,
    float,
    int,
]:
    observed = float(
        roc_auc_score(
            y_true,
            probability_left,
        )
        - roc_auc_score(
            y_true,
            probability_right,
        )
    )

    rng = np.random.default_rng(
        seed
    )

    unique_blocks = np.unique(
        blocks
    )

    indices_by_block = {
        block: np.flatnonzero(
            blocks
            == block
        )
        for block
        in unique_blocks
    }

    differences: list[
        float
    ] = []

    for _ in range(
        n_bootstrap
    ):
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(
                unique_blocks
            ),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                indices_by_block[
                    block
                ]
                for block
                in sampled_blocks
            ]
        )

        y_bootstrap = y_true[
            sampled_indices
        ]

        if np.unique(
            y_bootstrap
        ).size < 2:
            continue

        differences.append(
            float(
                roc_auc_score(
                    y_bootstrap,
                    probability_left[
                        sampled_indices
                    ],
                )
                - roc_auc_score(
                    y_bootstrap,
                    probability_right[
                        sampled_indices
                    ],
                )
            )
        )

    if not differences:
        return (
            observed,
            float(
                "nan"
            ),
            float(
                "nan"
            ),
            0,
        )

    alpha = (
        1.0
        - ci_level
    ) / 2.0

    low, high = np.quantile(
        differences,
        [
            alpha,
            1.0
            - alpha,
        ],
    )

    return (
        observed,
        float(
            low
        ),
        float(
            high
        ),
        len(
            differences
        ),
    )


# =============================================================================
# SUCCESSFUL STEP 7.2 OOF REPRODUCTION
# =============================================================================
def load_primary_step73_oof(
    scenario_name: str,
) -> pd.DataFrame:
    require_file(
        BufferConfig.PRIMARY_STEP73_OOF_CSV,
        "successful Step 7.3 LR A-F OOF predictions",
    )

    all_oof = pd.read_csv(
        BufferConfig.PRIMARY_STEP73_OOF_CSV
    )

    required = {
        "scenario",
        BufferConfig.SAMPLE_ID_COLUMN,
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
        "oof_probability",
    }

    missing = (
        required
        - set(
            all_oof.columns
        )
    )

    if missing:
        raise RuntimeError(
            "Successful Step 7.3 OOF file is missing columns: "
            f"{sorted(missing)}"
        )

    subset = all_oof.loc[
        all_oof[
            "scenario"
        ]
        == scenario_name
    ].copy()

    if len(
        subset
    ) != BufferConfig.EXPECTED_N_ROWS:
        raise RuntimeError(
            "Successful Step 7.3 OOF row count for "
            f"{scenario_name} is {len(subset)}; expected "
            f"{BufferConfig.EXPECTED_N_ROWS}."
        )

    return subset


def assert_oof_reproduces_primary(
    candidate_oof: pd.DataFrame,
    scenario_name: str,
) -> Dict[str, Any]:
    reference = load_primary_step73_oof(
        scenario_name
    )

    candidate = candidate_oof[
        [
            BufferConfig.SAMPLE_ID_COLUMN,
            BufferConfig.TARGET_COLUMN,
            BufferConfig.FOLD_COLUMN,
            "oof_probability",
        ]
    ].copy()

    candidate = candidate.rename(
        columns={
            "oof_probability": (
                "candidate_probability"
            )
        }
    )

    reference = reference[
        [
            BufferConfig.SAMPLE_ID_COLUMN,
            BufferConfig.TARGET_COLUMN,
            BufferConfig.FOLD_COLUMN,
            "oof_probability",
        ]
    ].copy()

    reference = reference.rename(
        columns={
            "oof_probability": (
                "reference_probability"
            )
        }
    )

    merged = candidate.merge(
        reference,
        on=BufferConfig.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=(
            "_candidate",
            "_reference",
        ),
        validate="one_to_one",
    )

    if len(
        merged
    ) != BufferConfig.EXPECTED_N_ROWS:
        raise RuntimeError(
            "Step 7.3C 100 m reproduction merge returned "
            f"{len(merged)} rows for {scenario_name}."
        )

    for column in (
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
    ):
        if not np.array_equal(
            merged[
                f"{column}_candidate"
            ].to_numpy(),
            merged[
                f"{column}_reference"
            ].to_numpy(),
        ):
            raise RuntimeError(
                "Step 7.3C 100 m reproduction metadata mismatch in "
                f"{column} for {scenario_name}."
            )

    difference = (
        merged[
            "candidate_probability"
        ].to_numpy(
            dtype=float
        )
        - merged[
            "reference_probability"
        ].to_numpy(
            dtype=float
        )
    )

    maximum_absolute_difference = float(
        np.max(
            np.abs(
                difference
            )
        )
    )

    mean_absolute_difference = float(
        np.mean(
            np.abs(
                difference
            )
        )
    )

    passed = bool(
        maximum_absolute_difference
        <= BufferConfig.OOF_REPRODUCTION_TOLERANCE
    )

    if not passed:
        raise RuntimeError(
            "Step 7.3C 100 m rerun did not reproduce the successful "
            f"Step 7.3 {scenario_name} LR OOF probabilities.\n"
            f"Maximum absolute difference = "
            f"{maximum_absolute_difference:.12g}\n"
            f"Tolerance = "
            f"{BufferConfig.OOF_REPRODUCTION_TOLERANCE:.12g}"
        )

    return {
        "oof_reproduction_pass": passed,
        "oof_reproduction_maximum_absolute_probability_difference": (
            maximum_absolute_difference
        ),
        "oof_reproduction_mean_absolute_probability_difference": (
            mean_absolute_difference
        ),
        "oof_reproduction_tolerance": float(
            BufferConfig.OOF_REPRODUCTION_TOLERANCE
        ),
    }


# =============================================================================
# PELRORMANCE SUMMARIES / PAIRED E-A COMPARISON
# =============================================================================
def summarize_oof(
    oof_df: pd.DataFrame,
    buffer_m: int,
    scenario_name: str,
) -> Dict[str, Any]:
    y = oof_df[
        BufferConfig.TARGET_COLUMN
    ].to_numpy(
        dtype=int
    )

    probability = oof_df[
        "oof_probability"
    ].to_numpy(
        dtype=float
    )

    blocks = oof_df[
        BufferConfig.FOLD_COLUMN
    ].to_numpy(
        dtype=int
    )

    metrics = classification_metrics(
        y,
        probability,
    )

    (
        ci_low,
        ci_high,
        n_valid,
    ) = cluster_bootstrap_auc_ci(
        y_true=y,
        y_probability=probability,
        blocks=blocks,
        n_bootstrap=BufferConfig.N_CLUSTER_BOOTSTRAP,
        seed=(
            BufferConfig.RANDOM_STATE
            + buffer_m
            + BufferConfig.SCENARIO_ORDER.index(
                scenario_name
            )
        ),
        ci_level=BufferConfig.CI_LEVEL,
    )

    return {
        "buffer_m": int(
            buffer_m
        ),
        "scenario": scenario_name,
        "model": BufferConfig.MODEL_SHORT_LABELS[
            scenario_name
        ],
        "oof_roc_auc": metrics[
            "roc_auc"
        ],
        "auc_ci_low": ci_low,
        "auc_ci_high": ci_high,
        "oof_pr_auc": metrics[
            "pr_auc"
        ],
        "oof_balanced_accuracy": metrics[
            "balanced_accuracy"
        ],
        "oof_f1": metrics[
            "f1"
        ],
        "oof_brier": metrics[
            "brier"
        ],
        "valid_auc_bootstrap_iterations": int(
            n_valid
        ),
    }


def paired_auc_difference_from_oof(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    label: str,
    seed: int,
) -> Dict[str, Any]:
    left = left_df[
        [
            BufferConfig.SAMPLE_ID_COLUMN,
            BufferConfig.TARGET_COLUMN,
            BufferConfig.FOLD_COLUMN,
            "oof_probability",
        ]
    ].copy()

    left = left.rename(
        columns={
            "oof_probability": (
                "probability_left"
            )
        }
    )

    right = right_df[
        [
            BufferConfig.SAMPLE_ID_COLUMN,
            BufferConfig.TARGET_COLUMN,
            BufferConfig.FOLD_COLUMN,
            "oof_probability",
        ]
    ].copy()

    right = right.rename(
        columns={
            "oof_probability": (
                "probability_right"
            )
        }
    )

    merged = left.merge(
        right,
        on=BufferConfig.SAMPLE_ID_COLUMN,
        how="inner",
        suffixes=(
            "_left",
            "_right",
        ),
        validate="one_to_one",
    )

    if len(
        merged
    ) != BufferConfig.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Paired comparison {label} returned "
            f"{len(merged)} rows; expected "
            f"{BufferConfig.EXPECTED_N_ROWS}."
        )

    for column in (
        BufferConfig.TARGET_COLUMN,
        BufferConfig.FOLD_COLUMN,
    ):
        if not np.array_equal(
            merged[
                f"{column}_left"
            ].to_numpy(),
            merged[
                f"{column}_right"
            ].to_numpy(),
        ):
            raise RuntimeError(
                f"Paired comparison metadata mismatch in {column}: "
                f"{label}"
            )

    y = merged[
        f"{BufferConfig.TARGET_COLUMN}_left"
    ].to_numpy(
        dtype=int
    )

    blocks = merged[
        f"{BufferConfig.FOLD_COLUMN}_left"
    ].to_numpy(
        dtype=int
    )

    (
        observed,
        ci_low,
        ci_high,
        n_valid,
    ) = paired_cluster_bootstrap_auc_difference(
        y_true=y,
        probability_left=merged[
            "probability_left"
        ].to_numpy(
            dtype=float
        ),
        probability_right=merged[
            "probability_right"
        ].to_numpy(
            dtype=float
        ),
        blocks=blocks,
        n_bootstrap=BufferConfig.N_CLUSTER_BOOTSTRAP,
        seed=seed,
        ci_level=BufferConfig.CI_LEVEL,
    )

    return {
        "comparison": label,
        "auc_difference_left_minus_right": observed,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "valid_bootstrap_iterations": int(
            n_valid
        ),
    }


# =============================================================================
# PROVENANCE
# =============================================================================
def write_diagnostic_provenance(
    buffer_paths: Dict[int, Path],
) -> None:
    BufferConfig.OUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "diagnostic_step": "Step 7.3C",
        "algorithm": "Logistic Regression",
        "scientific_purpose": (
            "Cross-algorithm pseudo-absence exclusion-buffer sensitivity for "
            "Models A and E using the exact successful Step 7.3 LR nested "
            "spatial-CV contract."
        ),
        "buffers_m": [
            100,
            500,
            1000,
        ],
        "scenarios": list(
            BufferConfig.SCENARIOS
        ),
        "buffer_dataset_paths": {
            str(
                key
            ): str(
                value
            )
            for (
                key,
                value,
            )
            in buffer_paths.items()
        },
        "input_rule": (
            "100 m uses the exact Step 6.4 primary dataset; 500/1000 m use "
            "Step 6.3 buffer datasets converted in memory with "
            "cv_fold=spatial_block and strict Step 7.3 model-ready QA."
        ),
        "primary_step73_oof_reference": str(
            BufferConfig.PRIMARY_STEP73_OOF_CSV
        ),
        "rf_parameter_grid": json_safe(
            BufferConfig.PARAM_GRID
        ),
        "rf_fixed_parameters": json_safe(
            BufferConfig.LR_FIXED_PARAMS
        ),
        "positive_predictor_equality_tolerance": float(
            BufferConfig.POSITIVE_PREDICTOR_EQUALITY_ATOL
        ),
        "plot_style": (
            "Exact Step 7.1C/7.2C figure geometry and Matplotlib commands, "
            "with LR results and LR-prefixed output filename."
        ),
    }

    with open(
        BufferConfig.OUT_DIR
        / "Step_7_3C_LR_Buffer_Sensitivity_Provenance.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(
                metadata
            ),
            handle,
            indent=2,
        )


# =============================================================================
# EXACT STEP 7.1C-STYLE PLOT
# =============================================================================
def create_buffer_sensitivity_plot(
    summary_df: pd.DataFrame,
) -> Path:
    """
    Exact same plotting style used in Step 7.1C, now using LR results.
    """
    figure_path = (
        BufferConfig.OUT_DIR
        / "LR_7_3C_A_E_Buffer_Sensitivity.png"
    )

    fig, ax = plt.subplots(
        figsize=(
            5.8,
            4.2,
        )
    )

    for scenario_name in BufferConfig.SCENARIOS:
        subset = summary_df.loc[
            summary_df[
                "scenario"
            ]
            == scenario_name
        ].sort_values(
            "buffer_m"
        )

        x = subset[
            "buffer_m"
        ].to_numpy(
            dtype=float
        )

        y = subset[
            "oof_roc_auc"
        ].to_numpy(
            dtype=float
        )

        lower = (
            y
            - subset[
                "auc_ci_low"
            ].to_numpy(
                dtype=float
            )
        )

        upper = (
            subset[
                "auc_ci_high"
            ].to_numpy(
                dtype=float
            )
            - y
        )

        ax.errorbar(
            x,
            y,
            yerr=np.vstack(
                [
                    lower,
                    upper,
                ]
            ),
            marker="o",
            capsize=3,
            linewidth=1.4,
            label=BufferConfig.MODEL_SHORT_LABELS[
                scenario_name
            ],
        )

    ax.set_xlabel(
        "Pseudo-absence exclusion buffer (m)"
    )

    ax.set_ylabel(
        "Nested OOF ROC-AUC"
    )

    ax.set_xticks(
        [
            100,
            500,
            1000,
        ]
    )

    ax.set_ylim(
        max(
            0.5,
            float(
                np.nanmin(
                    summary_df[
                        "auc_ci_low"
                    ]
                )
            )
            - 0.03,
        ),
        1.0,
    )

    ax.grid(
        True,
        alpha=0.25,
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    fig.savefig(
        figure_path,
        dpi=BufferConfig.FIG_DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

    return figure_path


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    log(
        "="
        * 96
    )

    log(
        "STEP 7.3C — LOGISTIC REGRESSION PSEUDO-ABSENCE BUFFER SENSITIVITY"
    )

    log(
        "="
        * 96
    )

    scenarios = validate_scenario_manifests()

    predictor_columns = all_scenario_features(
        scenarios
    )

    buffer_paths = load_buffer_paths(
        BufferConfig.MANIFEST_PATH
    )

    BufferConfig.OUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    write_diagnostic_provenance(
        buffer_paths
    )

    check_optional_step64_buffer_qa()

    datasets: Dict[
        int,
        pd.DataFrame,
    ] = {}

    summary_rows: list[
        Dict[str, Any]
    ] = []

    comparison_rows: list[
        Dict[str, Any]
    ] = []

    reproduction_rows: list[
        Dict[str, Any]
    ] = []

    prepared_input_dir = (
        BufferConfig.OUT_DIR
        / "prepared_model_ready_inputs"
    )

    prepared_input_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # -------------------------------------------------------------------------
    # 1. Load, convert and validate all buffer datasets before any model fitting.
    # -------------------------------------------------------------------------
    for buffer_m in (
        100,
        500,
        1000,
    ):
        dataset_path = Path(
            buffer_paths[
                buffer_m
            ]
        )

        log(
            f"[7.3C] Loading {buffer_m} m buffer source dataset: "
            f"{dataset_path}"
        )

        datasets[
            buffer_m
        ] = load_and_validate_buffer_dataset(
            scenarios=scenarios,
            dataset_csv=dataset_path,
            buffer_m=buffer_m,
        )

        prepared_path = (
            prepared_input_dir
            / (
                "model_ready_dataset_A_to_F_buffer_"
                f"{buffer_m}m.csv"
            )
        )

        datasets[
            buffer_m
        ].to_csv(
            prepared_path,
            index=False,
            float_format="%.10f",
        )

    # The 100 m dataset must be exactly the authoritative primary dataset.
    validate_primary_100m_dataset(
        datasets[
            100
        ],
        scenarios,
    )

    # Positive landslide samples and predictors must remain fixed across buffers.
    for buffer_m in (
        500,
        1000,
    ):
        validate_same_positive_samples_across_buffers(
            reference_df=datasets[
                100
            ],
            candidate_df=datasets[
                buffer_m
            ],
            predictor_columns=predictor_columns,
            candidate_buffer_m=buffer_m,
        )

    # -------------------------------------------------------------------------
    # 2. Run independent nested LR tuning for Models A and E at each buffer.
    # -------------------------------------------------------------------------
    for buffer_m in (
        100,
        500,
        1000,
    ):
        df = datasets[
            buffer_m
        ]

        per_buffer_oof: Dict[
            str,
            pd.DataFrame,
        ] = {}

        for scenario_name in BufferConfig.SCENARIOS:
            features = list(
                scenarios[
                    scenario_name
                ][
                    "features"
                ]
            )

            log(
                f"[7.3C] Nested spatial LR CV | "
                f"buffer={buffer_m} m | "
                f"{BufferConfig.MODEL_SHORT_LABELS[scenario_name]}"
            )

            (
                oof_df,
                fold_df,
                params,
                search_df,
            ) = run_nested_spatial_cv_for_scenario(
                df=df,
                scenario_name=scenario_name,
                features=features,
            )

            # The 100 m experiment must reproduce successful Step 7.3 LR A/E.
            if (
                buffer_m
                == BufferConfig.REFERENCE_BUFFER_M
            ):
                qa = assert_oof_reproduces_primary(
                    candidate_oof=oof_df,
                    scenario_name=scenario_name,
                )

                reproduction_rows.append(
                    {
                        "buffer_m": int(
                            buffer_m
                        ),
                        "scenario": scenario_name,
                        "model": BufferConfig.MODEL_SHORT_LABELS[
                            scenario_name
                        ],
                        **qa,
                    }
                )

            oof_df = oof_df.copy()

            oof_df.insert(
                0,
                "buffer_m",
                int(
                    buffer_m
                ),
            )

            fold_df = fold_df.copy()

            fold_df.insert(
                0,
                "buffer_m",
                int(
                    buffer_m
                ),
            )

            search_df = search_df.copy()

            search_df.insert(
                0,
                "buffer_m",
                int(
                    buffer_m
                ),
            )

            per_buffer_oof[
                scenario_name
            ] = oof_df

            summary_rows.append(
                summarize_oof(
                    oof_df=oof_df,
                    buffer_m=buffer_m,
                    scenario_name=scenario_name,
                )
            )

            prefix = (
                f"LR_7_3C_buffer_{buffer_m}m_"
                f"{BufferConfig.MODEL_SHORT_LABELS[scenario_name].replace(' ', '_')}"
            )

            oof_df.to_csv(
                BufferConfig.OUT_DIR
                / f"{prefix}_OOF.csv",
                index=False,
                float_format="%.10f",
            )

            fold_df.to_csv(
                BufferConfig.OUT_DIR
                / f"{prefix}_Fold_Metrics.csv",
                index=False,
                float_format="%.10f",
            )

            search_df.to_csv(
                BufferConfig.OUT_DIR
                / f"{prefix}_Inner_Search.csv",
                index=False,
                float_format="%.10f",
            )

            with open(
                BufferConfig.OUT_DIR
                / f"{prefix}_Outer_Params.json",
                "w",
                encoding="utf-8",
            ) as handle:
                json.dump(
                    json_safe(
                        params
                    ),
                    handle,
                    indent=2,
                )

            gc.collect()

        # Main reviewer comparison: Model E - Model A within the same buffer.
        comparison_rows.append(
            {
                "buffer_m": int(
                    buffer_m
                ),
                **paired_auc_difference_from_oof(
                    left_df=per_buffer_oof[
                        "E_physics_guided"
                    ],
                    right_df=per_buffer_oof[
                        "A_static_baseline"
                    ],
                    label=(
                        f"E - A at {buffer_m} m"
                    ),
                    seed=(
                        BufferConfig.RANDOM_STATE
                        + 10000
                        + buffer_m
                    ),
                ),
            }
        )

    # -------------------------------------------------------------------------
    # 3. Save QA, performance and paired-comparison tables.
    # -------------------------------------------------------------------------
    reproduction_df = pd.DataFrame(
        reproduction_rows
    )

    reproduction_df.to_csv(
        BufferConfig.OUT_DIR
        / "LR_7_3C_100m_Reproduction_QA.csv",
        index=False,
        float_format="%.10f",
    )

    summary_df = (
        pd.DataFrame(
            summary_rows
        )
        .sort_values(
            [
                "buffer_m",
                "scenario",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    comparison_df = (
        pd.DataFrame(
            comparison_rows
        )
        .sort_values(
            "buffer_m"
        )
        .reset_index(
            drop=True
        )
    )

    summary_df.to_csv(
        BufferConfig.OUT_DIR
        / "LR_7_3C_A_E_Buffer_Sensitivity_Performance.csv",
        index=False,
        float_format="%.10f",
    )

    comparison_df.to_csv(
        BufferConfig.OUT_DIR
        / "LR_7_3C_A_E_Buffer_Sensitivity_Paired_AUC.csv",
        index=False,
        float_format="%.10f",
    )

    # -------------------------------------------------------------------------
    # 4. Reproduce the exact Step 7.1C plot style using LR results.
    # -------------------------------------------------------------------------
    figure_path = create_buffer_sensitivity_plot(
        summary_df
    )

    log(
        ""
    )

    log(
        "="
        * 96
    )

    log(
        "STEP 7.3C COMPLETED SUCCESSFULLY"
    )

    log(
        "="
        * 96
    )

    log(
        f"[OUT] {BufferConfig.OUT_DIR}"
    )

    log(
        f"[FIGURE] {figure_path}"
    )

    print(
        "\n100 m Step 7.3 reproduction QA:"
    )

    print(
        reproduction_df.to_string(
            index=False
        )
    )

    print(
        "\nPerformance summary:"
    )

    print(
        summary_df.to_string(
            index=False
        )
    )

    print(
        "\nE - A paired AUC differences within each buffer:"
    )

    print(
        comparison_df.to_string(
            index=False
        )
    )


if __name__ == "__main__":
    main()


# STEP 7.3D — LOGISTIC REGRESSION PELRORMANCE AND MAPPED-AREA UNCERTAINTY

In [ ]:
# =============================================================================
# STEP 7.3D — LOGISTIC REGRESSION PELRORMANCE AND MAPPED-AREA UNCERTAINTY
# =============================================================================
#
# Scientific purpose
# ------------------
# This script is the Logistic Regression counterpart to Step 7.1D (XGBoost)
# and Step 7.2D (Random Forest). It reproduces the same uncertainty-analysis
# architecture using the exact successful Step 7.3 Logistic Regression
# modelling contract.
#
# Part 1 — Performance uncertainty
# --------------------------------
# Extend spatial-block cluster-bootstrap uncertainty from ROC-AUC to:
#     - ROC-AUC
#     - PR-AUC
#     - balanced accuracy
#     - F1
#     - Brier score
#     - precision
#     - recall
#
# The analysis uses the already saved genuine nested outer-fold OOF predictions
# from successful Step 7.3. No model is refitted for performance confidence
# intervals.
#
# Uncertainty is reported for:
#     - Models A-F
#     - the separately reported fixed A30/E70 LR operational ensemble
#
# Direct paired spatial-block bootstrap comparisons are also added for:
#     - Model E - Model C
#     - Model E - Model D
#
#
# Part 2 — Mapped-area / model variability
# ----------------------------------------
# For each of the five successful outer folds:
#     1. omit that spatial block from training;
#     2. refit LR Model A using that fold's already selected Step 7.3
#        inner-CV hyperparameters;
#     3. refit LR Model E using that fold's already selected Step 7.3
#        inner-CV hyperparameters;
#     4. predict the complete available domain for A and E;
#     5. create the fixed LR A30/E70 ensemble map;
#     6. calculate susceptibility-class area percentages.
#
# The five fold-specific map realizations are summarized as:
#     - mean
#     - standard deviation
#     - minimum
#     - maximum
#
# These statistics are described as:
#     "cross-fold mapped-area variability"
#     or
#     "cross-fold model uncertainty"
#
# They are NOT formal 95% confidence intervals because only five outer-fold map
# realizations are available.
#
#
# Critical scientific rules
# -------------------------
# 1. The primary LR model remains the successful Step 7.3 100 m / equal-HKFI
#    framework.
# 2. Step 7.3B alternative HKFI weights are NOT mixed into Step 7.3D.
# 3. Step 7.3C 500/1000 m pseudo-absence buffers are NOT mixed into Step 7.3D.
# 4. Performance uncertainty is based only on genuine Step 7.3 nested OOF
#    predictions.
# 5. The five map realizations reuse the exact successful Step 7.3 fold-specific
#    best parameters; there is no second hyperparameter search.
# 6. The final full-data LR A30/E70 map is shown alongside the cross-fold
#    mapped-area statistics.
# 7. All five cross-fold ensemble maps must have exactly the same valid-pixel
#    footprint, and this footprint must match the successful final LR ensemble.
#
#
# Technical robustness
# --------------------
# - Each outer-fold map realization runs in a separate Python process.
# - A completion checkpoint is written per fold.
# - Completed folds are reused automatically on restart.
# - Numerical-library thread counts are constrained to avoid CPU
#   oversubscription during process-isolated LR map generation.
#
#
# Exact successful Step 7.3 LR contract
# -------------------------------------
# Fixed model settings:
#     penalty       = "l2"
#     solver        = "liblinear"
#     class_weight  = "balanced"
#     fit_intercept = True
#     max_iter      = 10000
#     tol           = 1e-6
#     random_state  = 42
#
# Fold-safe preprocessing:
#     - StandardScaler for numeric predictors
#     - OneHotEncoder(handle_unknown="ignore") for categorical land cover
#
# The fold-specific tuned inverse regularization strength C is read directly
# from the successful Step 7.3 outer-fold best-parameter JSON. Step 7.3D does
# not repeat hyperparameter tuning.
#
#
# Recommended execution
# ---------------------
# Save and run this file as:
#
#     Step_7_3D_LR_Performance_and_Mapped_Area_Uncertainty.py
#
# Running the saved .py file is recommended because process-isolated map workers
# relaunch this same script.
# =============================================================================

from __future__ import annotations

import argparse
import gc
import json
import os
import re
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Sequence

# Prevent nested CPU oversubscription before importing numerical libraries.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import PathPatch, Polygon, Rectangle
from matplotlib.path import Path as MplPath
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, transform_geom
from rasterio.windows import Window

try:
    import fiona
    HAVE_FIONA = True
except Exception:
    fiona = None
    HAVE_FIONA = False
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# =============================================================================
# CONFIGURATION — FROZEN TO SUCCESSFUL STEP 7.2
# =============================================================================
class Config:
    BASE = Path(os.environ.get("LSM_DATA_DIR", "./data"))
    RESULTS_DIR = BASE / "Results"

    # Step 6.1 frozen predictor definitions and master raster stack.
    STEP61_DIR = RESULTS_DIR / "Step-6.1-Stacks"
    SCENARIO_MANIFEST_JSON = (
        STEP61_DIR / "model_ablation_scenarios.json"
    )
    MASTER_STACK_TIF = (
        STEP61_DIR
        / "stack_master_candidates_17band_studydomain_32645_12p5m_float32.tif"
    )

    # Step 6.4 frozen modelling contract.
    STEP64_DIR = RESULTS_DIR / "Step-6.4-Model_Ready_QA"
    MODEL_READY_DATASET_CSV = (
        STEP64_DIR / "model_ready_dataset_A_to_F.csv"
    )
    STEP64_MODEL_CONFIG_JSON = (
        STEP64_DIR / "model_configuration_manifest.json"
    )

    # Successful Step 7.3 LR outputs.
    OUT_DIR = RESULTS_DIR / "Step-7.3-LR-Models_A_to_F"
    MAP_DIR = OUT_DIR / "susceptibility_maps"

    OOF_PREDICTIONS_CSV = (
        OUT_DIR / "LR_A_to_F_OOF_predictions.csv"
    )
    OUTER_PARAMS_JSON = (
        OUT_DIR / "LR_A_to_F_outer_fold_best_parameters.json"
    )
    ENSEMBLE_OOF_PREDICTIONS_CSV = (
        OUT_DIR / "LR_Ensemble_A30_E70_OOF_predictions.csv"
    )
    ENSEMBLE_PROBABILITY_TIF = (
        MAP_DIR / "LR_Ensemble_A30_E70_probability_full_study_domain.tif"
    )

    TARGET_COLUMN = "label"
    FOLD_COLUMN = "cv_fold"
    BLOCK_COLUMN = "spatial_block"
    SAMPLE_ID_COLUMN = "sample_id"

    EXPECTED_N_ROWS = 330
    EXPECTED_N_POSITIVE = 165
    EXPECTED_N_NEGATIVE = 165
    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_SHORT_LABELS = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    CATEGORICAL_FEATURE_ALIASES = {
        "landcover",
        "landuse",
        "land_use",
    }

    # Exact successful Step 7.3 fixed Logistic Regression settings.
    #
    # The fold-specific tuned C value is loaded from the successful Step 7.3
    # outer-fold best-parameter JSON. No hyperparameter search is repeated here.
    LR_FIXED_PARAMS = {
        "penalty": "l2",
        "solver": "liblinear",
        "class_weight": "balanced",
        "fit_intercept": True,
        "max_iter": 10000,
        "tol": 1.0e-6,
        "random_state": 42,
    }

    BINARY_THRESHOLD = 0.50
    ENSEMBLE_ENHANCED_WEIGHT = 0.70
    OUTPUT_NODATA = -9999.0


class UncertaintyConfig:
    # Performance uncertainty.
    N_BOOTSTRAP = 5000
    CI_LEVEL = 0.95
    RANDOM_STATE = 62571

    # Map uncertainty.
    KEEP_INTERMEDIATE_A_E_RASTERS = False
    RESUME_FROM_CHECKPOINTS = True
    RASTER_BLOCK_ROWS = 256

    # Figures.
    FIG_DPI = 300
    GENERATE_PIXELWISE_SD_FIGURE = True

    # Descriptive thresholds for the pixelwise cross-fold SD summary.
    SD_THRESHOLDS = (
        0.02,
        0.05,
        0.10,
    )


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(
    message: str,
) -> None:
    print(
        f"[{datetime.now():%H:%M:%S}] {message}",
        flush=True,
    )


def utc_now_iso() -> str:
    return datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def json_safe(
    value: Any,
) -> Any:
    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    return value


def require_file(
    path: Path,
    description: str,
) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}"
        )


def load_json(
    path: Path,
) -> Dict[str, Any]:
    require_file(
        path,
        "JSON file",
    )

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def normalize_feature_name(
    name: str,
) -> str:
    return re.sub(
        r"[^a-z0-9_]+",
        "_",
        str(
            name
        ).strip().lower(),
    ).strip(
        "_"
    )


def is_categorical_feature(
    name: str,
) -> bool:
    normalized = normalize_feature_name(
        name
    )

    compact = normalized.replace(
        "_",
        "",
    )

    aliases = {
        alias.lower()
        for alias in Config.CATEGORICAL_FEATURE_ALIASES
    }

    compact_aliases = {
        alias.replace(
            "_",
            "",
        )
        for alias in aliases
    }

    return (
        normalized in aliases
        or compact in compact_aliases
    )


def resolve_script_path() -> Path:
    """
    Resolve the saved Python script used for process-isolated fold workers.
    """
    file_value = globals().get(
        "__file__"
    )

    if file_value:
        candidate = Path(
            str(
                file_value
            )
        ).resolve()

        if (
            candidate.exists()
            and candidate.name.lower()
            != "ipykernel_launcher.py"
        ):
            return candidate

    candidate = (
        Path.cwd()
        / "Step_7_3D_LR_Performance_and_Mapped_Area_Uncertainty.py"
    )

    if candidate.exists():
        return candidate.resolve()

    raise RuntimeError(
        "Could not resolve the saved Step 7.3D Python script. "
        "Save this program as "
        "Step_7_3D_LR_Performance_and_Mapped_Area_Uncertainty.py "
        "and run the saved .py file."
    )


# =============================================================================
# STEP 6.1 / STEP 6.4 CONTRACT
# =============================================================================
def validate_scenario_manifests() -> Dict[str, Dict[str, Any]]:
    """
    Verify that Step 6.1 and Step 6.4 define Models A-F identically.
    """
    step61 = load_json(
        Config.SCENARIO_MANIFEST_JSON
    ).get(
        "scenarios",
        {},
    )

    step64 = load_json(
        Config.STEP64_MODEL_CONFIG_JSON
    ).get(
        "scenarios",
        {},
    )

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in step61:
            raise RuntimeError(
                "Step 6.1 scenario manifest is missing "
                f"{scenario_name}."
            )

        if scenario_name not in step64:
            raise RuntimeError(
                "Step 6.4 model configuration is missing "
                f"{scenario_name}."
            )

        features_61 = list(
            step61[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        features_64 = list(
            step64[
                scenario_name
            ].get(
                "features",
                [],
            )
        )

        if features_61 != features_64:
            raise RuntimeError(
                f"Predictor mismatch for {scenario_name}.\n"
                f"Step 6.1: {features_61}\n"
                f"Step 6.4: {features_64}"
            )

        if any(
            "irs" in feature.lower()
            for feature in features_61
        ):
            raise RuntimeError(
                f"IRS is present in {scenario_name}; "
                "IRS must not be an ML predictor."
            )

    return step64


def all_scenario_features(
    scenarios: Dict[str, Dict[str, Any]],
) -> list[str]:
    features: list[str] = []

    for scenario_name in Config.SCENARIO_ORDER:
        for feature in scenarios[
            scenario_name
        ][
            "features"
        ]:
            if feature not in features:
                features.append(
                    feature
                )

    return features


def load_and_validate_model_ready_dataset(
    scenarios: Dict[str, Dict[str, Any]],
) -> pd.DataFrame:
    """
    Load the exact frozen Step 6.4 dataset used by successful Step 7.3.
    """
    require_file(
        Config.MODEL_READY_DATASET_CSV,
        "Step 6.4 frozen model-ready dataset",
    )

    df = pd.read_csv(
        Config.MODEL_READY_DATASET_CSV
    )

    features = all_scenario_features(
        scenarios
    )

    required = [
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
        *features,
    ]

    missing = [
        column
        for column in required
        if column not in df.columns
    ]

    if missing:
        raise RuntimeError(
            "Model-ready dataset is missing columns: "
            f"{missing}"
        )

    if len(
        df
    ) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            f"Expected {Config.EXPECTED_N_ROWS} rows; "
            f"found {len(df)}."
        )

    if df[
        Config.SAMPLE_ID_COLUMN
    ].isna().any():
        raise RuntimeError(
            "Missing sample_id values found."
        )

    if df[
        Config.SAMPLE_ID_COLUMN
    ].duplicated().any():
        raise RuntimeError(
            "Duplicated sample_id values found."
        )

    for column in (
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        Config.BLOCK_COLUMN,
    ):
        df[
            column
        ] = pd.to_numeric(
            df[
                column
            ],
            errors="raise",
        ).astype(
            int
        )

    class_counts = (
        df[
            Config.TARGET_COLUMN
        ]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    expected_class_counts = {
        0: Config.EXPECTED_N_NEGATIVE,
        1: Config.EXPECTED_N_POSITIVE,
    }

    if class_counts != expected_class_counts:
        raise RuntimeError(
            f"Unexpected class counts: {class_counts}; "
            f"expected {expected_class_counts}."
        )

    folds = sorted(
        df[
            Config.FOLD_COLUMN
        ].unique().tolist()
    )

    if folds != Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected folds {Config.EXPECTED_FOLDS}; "
            f"found {folds}."
        )

    if not np.array_equal(
        df[
            Config.FOLD_COLUMN
        ].to_numpy(),
        df[
            Config.BLOCK_COLUMN
        ].to_numpy(),
    ):
        raise RuntimeError(
            "cv_fold and spatial_block are not identical."
        )

    predictor_values = df[
        features
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    if not np.isfinite(
        predictor_values.to_numpy(
            dtype="float64"
        )
    ).all():
        raise RuntimeError(
            "Missing/non-finite predictor values remain in Step 6.4 dataset."
        )

    return df


# =============================================================================
# EXACT STEP 7.2 LOGISTIC REGRESSION PIPELINE
# =============================================================================
def build_lr_pipeline(
    features: Sequence[str],
) -> Pipeline:
    """
    Reconstruct the exact fold-safe preprocessing + Logistic Regression pipeline
    used in successful Step 7.3.

    Numeric predictors are standardized with StandardScaler fitted only on the
    current training partition. Land cover is one-hot encoded inside the same
    pipeline. Therefore the five cross-fold map realizations use the same
    leakage-safe preprocessing contract as the successful Step 7.3 analysis.
    """
    categorical = [
        feature
        for feature in features
        if is_categorical_feature(
            feature
        )
    ]

    numeric = [
        feature
        for feature in features
        if not is_categorical_feature(
            feature
        )
    ]

    transformers = []

    if numeric:
        transformers.append(
            (
                "numeric",
                StandardScaler(),
                numeric,
            )
        )

    if categorical:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    classifier = LogisticRegression(
        **Config.LR_FIXED_PARAMS
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                classifier,
            ),
        ]
    )


def set_pipeline_params(
    pipeline: Pipeline,
    params: Dict[str, Any],
) -> Pipeline:
    pipeline.set_params(
        **params
    )

    return pipeline


def load_outer_best_params() -> Dict[str, Any]:
    require_file(
        Config.OUTER_PARAMS_JSON,
        "successful Step 7.3 LR outer-fold best-parameter JSON",
    )

    with open(
        Config.OUTER_PARAMS_JSON,
        "r",
        encoding="utf-8",
    ) as handle:
        params = json.load(
            handle
        )

    for scenario_name in Config.SCENARIO_ORDER:
        if scenario_name not in params:
            raise RuntimeError(
                "Outer-parameter JSON is missing scenario: "
                f"{scenario_name}"
            )

        for outer_fold in Config.EXPECTED_FOLDS:
            fold_info = params[
                scenario_name
            ].get(
                str(
                    outer_fold
                )
            )

            if (
                fold_info is None
                or "best_params"
                not in fold_info
            ):
                raise RuntimeError(
                    "Outer-parameter JSON is missing best_params for "
                    f"{scenario_name}, fold {outer_fold}."
                )

    return params


def get_fold_params(
    outer_params: Dict[str, Any],
    scenario_name: str,
    outer_fold: int,
) -> Dict[str, Any]:
    scenario = outer_params.get(
        scenario_name
    )

    if scenario is None:
        raise KeyError(
            "Missing scenario in successful outer-parameter JSON: "
            f"{scenario_name}"
        )

    fold_info = scenario.get(
        str(
            outer_fold
        )
    )

    if fold_info is None:
        raise KeyError(
            f"Missing fold {outer_fold} parameters for "
            f"{scenario_name}"
        )

    params = fold_info.get(
        "best_params"
    )

    if params is None:
        raise KeyError(
            f"Missing best_params for {scenario_name}, "
            f"outer fold {outer_fold}"
        )

    return params


# =============================================================================
# PELRORMANCE METRICS AND SPATIAL-BLOCK BOOTSTRAP
# =============================================================================
def metric_value(
    metric_name: str,
    y_true: np.ndarray,
    y_probability: np.ndarray,
    threshold: float,
) -> float:
    predicted = (
        y_probability
        >= threshold
    ).astype(
        int
    )

    if metric_name == "roc_auc":
        return float(
            roc_auc_score(
                y_true,
                y_probability,
            )
        )

    if metric_name == "pr_auc":
        return float(
            average_precision_score(
                y_true,
                y_probability,
            )
        )

    if metric_name == "balanced_accuracy":
        return float(
            balanced_accuracy_score(
                y_true,
                predicted,
            )
        )

    if metric_name == "f1":
        return float(
            f1_score(
                y_true,
                predicted,
                zero_division=0,
            )
        )

    if metric_name == "brier":
        return float(
            brier_score_loss(
                y_true,
                y_probability,
            )
        )

    if metric_name == "precision":
        return float(
            precision_score(
                y_true,
                predicted,
                zero_division=0,
            )
        )

    if metric_name == "recall":
        return float(
            recall_score(
                y_true,
                predicted,
                zero_division=0,
            )
        )

    raise ValueError(
        f"Unsupported metric: {metric_name}"
    )


def cluster_bootstrap_metric_intervals(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    ci_level: float,
    seed: int,
    threshold: float,
    metrics: Sequence[str],
) -> pd.DataFrame:
    """
    Resample whole spatial blocks with replacement and calculate uncertainty for
    the requested metrics.
    """
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float,
    )

    blocks = np.asarray(
        blocks,
    )

    unique_blocks = np.unique(
        blocks
    )

    indices_by_block = {
        block: np.flatnonzero(
            blocks
            == block
        )
        for block in unique_blocks
    }

    observed = {
        metric: metric_value(
            metric,
            y_true,
            y_prob,
            threshold,
        )
        for metric in metrics
    }

    bootstrap_values = {
        metric: []
        for metric in metrics
    }

    rng = np.random.default_rng(
        seed
    )

    for _ in range(
        n_bootstrap
    ):
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(
                unique_blocks
            ),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                indices_by_block[
                    block
                ]
                for block in sampled_blocks
            ]
        )

        y_boot = y_true[
            sampled_indices
        ]

        p_boot = y_prob[
            sampled_indices
        ]

        if np.unique(
            y_boot
        ).size < 2:
            continue

        for metric in metrics:
            bootstrap_values[
                metric
            ].append(
                metric_value(
                    metric,
                    y_boot,
                    p_boot,
                    threshold,
                )
            )

    alpha = (
        1.0
        - ci_level
    ) / 2.0

    rows: list[
        Dict[str, Any]
    ] = []

    for metric in metrics:
        values = np.asarray(
            bootstrap_values[
                metric
            ],
            dtype=float,
        )

        if values.size:
            low, high = np.quantile(
                values,
                [
                    alpha,
                    1.0
                    - alpha,
                ],
            )
        else:
            low = np.nan
            high = np.nan

        rows.append(
            {
                "metric": metric,
                "estimate": observed[
                    metric
                ],
                "ci_low": float(
                    low
                ),
                "ci_high": float(
                    high
                ),
                "valid_bootstrap_iterations": int(
                    values.size
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


def paired_cluster_bootstrap_auc_difference(
    y_true: np.ndarray,
    probability_left: np.ndarray,
    probability_right: np.ndarray,
    blocks: np.ndarray,
    n_bootstrap: int,
    seed: int,
    ci_level: float,
) -> tuple[
    float,
    float,
    float,
    int,
]:
    observed = float(
        roc_auc_score(
            y_true,
            probability_left,
        )
        - roc_auc_score(
            y_true,
            probability_right,
        )
    )

    unique_blocks = np.unique(
        blocks
    )

    indices_by_block = {
        block: np.flatnonzero(
            blocks
            == block
        )
        for block in unique_blocks
    }

    rng = np.random.default_rng(
        seed
    )

    differences: list[
        float
    ] = []

    for _ in range(
        n_bootstrap
    ):
        sampled_blocks = rng.choice(
            unique_blocks,
            size=len(
                unique_blocks
            ),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                indices_by_block[
                    block
                ]
                for block in sampled_blocks
            ]
        )

        y_boot = y_true[
            sampled_indices
        ]

        if np.unique(
            y_boot
        ).size < 2:
            continue

        differences.append(
            float(
                roc_auc_score(
                    y_boot,
                    probability_left[
                        sampled_indices
                    ],
                )
                - roc_auc_score(
                    y_boot,
                    probability_right[
                        sampled_indices
                    ],
                )
            )
        )

    if not differences:
        return (
            observed,
            np.nan,
            np.nan,
            0,
        )

    alpha = (
        1.0
        - ci_level
    ) / 2.0

    low, high = np.quantile(
        differences,
        [
            alpha,
            1.0
            - alpha,
        ],
    )

    return (
        observed,
        float(
            low
        ),
        float(
            high
        ),
        len(
            differences
        ),
    )


# =============================================================================
# RASTER HELPERS — SAME STEP 7.2 LR MAPPING PRINCIPLE
# =============================================================================
def read_stack_band_map(
    stack_path: Path,
) -> Dict[str, int]:
    require_file(
        stack_path,
        "Step 6.1 master predictor stack",
    )

    with rasterio.open(
        stack_path
    ) as dataset:
        descriptions = list(
            dataset.descriptions
        )

    band_map = {
        str(
            description
        ): int(
            index
        )
        for (
            index,
            description,
        )
        in enumerate(
            descriptions,
            start=1,
        )
        if description
    }

    if not band_map:
        raise RuntimeError(
            "Master predictor stack has no band descriptions."
        )

    return band_map


def predict_scenario_raster_on_own_feature_domain(
    fitted_pipeline: Pipeline,
    scenario_name: str,
    features: Sequence[str],
    output_path: Path,
) -> Path:
    """
    Predict a scenario wherever all predictors required by that scenario are
    finite in the Step 6.1 study-domain master stack.
    """
    band_map = read_stack_band_map(
        Config.MASTER_STACK_TIF
    )

    missing_bands = [
        feature
        for feature in features
        if feature not in band_map
    ]

    if missing_bands:
        raise RuntimeError(
            f"Master stack is missing bands required by "
            f"{scenario_name}: {missing_bands}"
        )

    band_indexes = [
        band_map[
            feature
        ]
        for feature in features
    ]

    with rasterio.open(
        Config.MASTER_STACK_TIF
    ) as stack_dataset:
        profile = stack_dataset.profile.copy()

        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(
            str(
                output_path
            )
            + ".part"
        )

        if temporary_path.exists():
            temporary_path.unlink()

        valid_pixel_count = 0

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_dataset:
            for row_start in range(
                0,
                stack_dataset.height,
                UncertaintyConfig.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    UncertaintyConfig.RASTER_BLOCK_ROWS,
                    stack_dataset.height
                    - row_start,
                )

                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=stack_dataset.width,
                    height=block_height,
                )

                stack_values = np.asarray(
                    stack_dataset.read(
                        band_indexes,
                        window=window,
                        masked=True,
                    ).filled(
                        np.nan
                    ),
                    dtype="float64",
                )

                valid = np.all(
                    np.isfinite(
                        stack_values
                    ),
                    axis=0,
                )

                output = np.full(
                    (
                        block_height,
                        stack_dataset.width,
                    ),
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                if np.any(
                    valid
                ):
                    data = {
                        feature: stack_values[
                            feature_index
                        ][
                            valid
                        ]
                        for (
                            feature_index,
                            feature,
                        )
                        in enumerate(
                            features
                        )
                    }

                    pixel_df = pd.DataFrame(
                        data,
                        columns=list(
                            features
                        ),
                    )

                    probability = (
                        fitted_pipeline.predict_proba(
                            pixel_df
                        )[
                            :,
                            1,
                        ]
                    )

                    output[
                        valid
                    ] = probability.astype(
                        "float32"
                    )

                    valid_pixel_count += int(
                        valid.sum()
                    )

                output_dataset.write(
                    output,
                    1,
                    window=window,
                )

            output_dataset.set_band_description(
                1,
                (
                    "Logistic Regression "
                    f"{Config.MODEL_SHORT_LABELS[scenario_name]} "
                    "susceptibility probability"
                ),
            )

            output_dataset.update_tags(
                algorithm="Logistic Regression",
                scenario=scenario_name,
                generated_at_utc=utc_now_iso(),
                validation_note=(
                    "Cross-fold model realization for map variability; "
                    "not a formal independent validation map."
                ),
            )

    if output_path.exists():
        output_path.unlink()

    os.replace(
        temporary_path,
        output_path,
    )

    log(
        f"[MAP] {scenario_name}: {valid_pixel_count:,} valid pixels -> "
        f"{output_path}"
    )

    return output_path


def blend_static_and_enhanced_probability_rasters(
    static_raster_path: Path,
    enhanced_raster_path: Path,
    output_path: Path,
    enhanced_weight: float,
) -> Path:
    """
    Apply the exact fixed A30/E70 operational blending principle used in Step 7.3.
    """
    if not (
        0.0
        <= enhanced_weight
        <= 1.0
    ):
        raise ValueError(
            "Enhanced weight must lie between 0 and 1."
        )

    static_weight = (
        1.0
        - enhanced_weight
    )

    require_file(
        static_raster_path,
        "Model A full-domain probability raster",
    )

    require_file(
        enhanced_raster_path,
        "Model E full-domain probability raster",
    )

    with rasterio.open(
        static_raster_path
    ) as static_dataset, rasterio.open(
        enhanced_raster_path
    ) as enhanced_dataset:
        aligned = (
            static_dataset.width
            == enhanced_dataset.width
            and static_dataset.height
            == enhanced_dataset.height
            and static_dataset.transform
            == enhanced_dataset.transform
            and static_dataset.crs
            == enhanced_dataset.crs
        )

        if not aligned:
            raise RuntimeError(
                "Model A and Model E full-domain rasters are not aligned."
            )

        profile = static_dataset.profile.copy()

        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(
            str(
                output_path
            )
            + ".part"
        )

        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as output_dataset:
            for row_start in range(
                0,
                static_dataset.height,
                UncertaintyConfig.RASTER_BLOCK_ROWS,
            ):
                block_height = min(
                    UncertaintyConfig.RASTER_BLOCK_ROWS,
                    static_dataset.height
                    - row_start,
                )

                window = Window(
                    0,
                    row_start,
                    static_dataset.width,
                    block_height,
                )

                static_values = np.asarray(
                    static_dataset.read(
                        1,
                        window=window,
                        masked=True,
                    ).filled(
                        np.nan
                    ),
                    dtype="float64",
                )

                enhanced_values = np.asarray(
                    enhanced_dataset.read(
                        1,
                        window=window,
                        masked=True,
                    ).filled(
                        np.nan
                    ),
                    dtype="float64",
                )

                static_valid = np.isfinite(
                    static_values
                )

                enhanced_valid = np.isfinite(
                    enhanced_values
                )

                both_valid = (
                    static_valid
                    & enhanced_valid
                )

                static_only = (
                    static_valid
                    & ~enhanced_valid
                )

                enhanced_only = (
                    enhanced_valid
                    & ~static_valid
                )

                output = np.full(
                    static_values.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                output[
                    static_only
                ] = static_values[
                    static_only
                ].astype(
                    "float32"
                )

                output[
                    enhanced_only
                ] = enhanced_values[
                    enhanced_only
                ].astype(
                    "float32"
                )

                output[
                    both_valid
                ] = (
                    static_weight
                    * static_values[
                        both_valid
                    ]
                    + enhanced_weight
                    * enhanced_values[
                        both_valid
                    ]
                ).astype(
                    "float32"
                )

                output_dataset.write(
                    output,
                    1,
                    window=window,
                )

            output_dataset.set_band_description(
                1,
                "Logistic Regression A30/E70 ensemble susceptibility probability",
            )

            output_dataset.update_tags(
                algorithm="Logistic Regression",
                static_weight=f"{static_weight:.6f}",
                enhanced_weight=f"{enhanced_weight:.6f}",
                generated_at_utc=utc_now_iso(),
            )

    if output_path.exists():
        output_path.unlink()

    os.replace(
        temporary_path,
        output_path,
    )

    return output_path


def clean_probability_array(
    array: np.ndarray,
    nodata: float | None,
) -> np.ndarray:
    result = np.asarray(
        array,
        dtype="float64",
    ).copy()

    if (
        nodata is not None
        and np.isfinite(
            float(
                nodata
            )
        )
    ):
        result[
            result
            == float(
                nodata
            )
        ] = np.nan

    result[
        (
            result
            < -1.0e-6
        )
        | (
            result
            > 1.0
            + 1.0e-6
        )
    ] = np.nan

    return result


def classify_susceptibility_probability(
    probability: np.ndarray,
) -> np.ndarray:
    """
    Same five fixed probability classes used in Step 7.3:
      1 Very Low : [0.0, 0.2)
      2 Low      : [0.2, 0.4)
      3 Moderate : [0.4, 0.6)
      4 High     : [0.6, 0.8)
      5 Very High: [0.8, 1.0]
    """
    classes = np.digitize(
        probability,
        bins=[
            0.2,
            0.4,
            0.6,
            0.8,
        ],
        right=False,
    )

    return (
        classes
        + 1
    )


def calculate_susceptibility_area_percentages(
    probability_raster_path: Path,
) -> pd.DataFrame:
    require_file(
        probability_raster_path,
        "ensemble susceptibility raster",
    )

    with rasterio.open(
        probability_raster_path
    ) as dataset:
        probability = clean_probability_array(
            dataset.read(
                1
            ),
            dataset.nodata,
        )

    valid = probability[
        np.isfinite(
            probability
        )
    ]

    if valid.size == 0:
        raise RuntimeError(
            "No valid susceptibility probabilities found."
        )

    classes = classify_susceptibility_probability(
        valid
    )

    total = int(
        valid.size
    )

    definitions = [
        (
            5,
            "Very High",
            0.8,
            1.0,
        ),
        (
            4,
            "High",
            0.6,
            0.8,
        ),
        (
            3,
            "Moderate",
            0.4,
            0.6,
        ),
        (
            2,
            "Low",
            0.2,
            0.4,
        ),
        (
            1,
            "Very Low",
            0.0,
            0.2,
        ),
    ]

    rows = []

    for (
        class_id,
        label,
        lower,
        upper,
    ) in definitions:
        count = int(
            np.sum(
                classes
                == class_id
            )
        )

        percentage = (
            100.0
            * count
            / total
        )

        rows.append(
            {
                "class_id": int(
                    class_id
                ),
                "class_label": label,
                "lower_probability": float(
                    lower
                ),
                "upper_probability": float(
                    upper
                ),
                "pixel_count": int(
                    count
                ),
                "area_percentage": float(
                    percentage
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


# =============================================================================
# PROVENANCE
# =============================================================================
def write_diagnostic_provenance(
    out_dir: Path,
) -> None:
    metadata = {
        "generated_at_utc": utc_now_iso(),
        "diagnostic_step": "Step 7.3D",
        "algorithm": "Logistic Regression",
        "performance_uncertainty": (
            "spatial-block cluster bootstrap resampling whole outer spatial blocks"
        ),
        "performance_inputs": str(
            Config.OOF_PREDICTIONS_CSV
        ),
        "ensemble_performance_inputs": str(
            Config.ENSEMBLE_OOF_PREDICTIONS_CSV
        ),
        "mapped_area_uncertainty": (
            "five process-isolated outer-fold LR A30/E70 map realizations"
        ),
        "map_model_parameters": (
            "exact successful Step 7.3 fold-specific best parameters; no retuning"
        ),
        "mapped_area_reporting": (
            "cross-fold mean, SD, minimum and maximum; not a formal 95% CI"
        ),
        "final_map_reference": str(
            Config.ENSEMBLE_PROBABILITY_TIF
        ),
        "valid_domain_rule": (
            "all five cross-fold ensemble maps must have identical valid masks "
            "and must match the successful final full-data LR ensemble mask"
        ),
        "step_7_2B_alternative_hkfi_weights_included": False,
        "step_7_2C_alternative_buffers_included": False,
        "rf_fixed_parameters": json_safe(
            Config.LR_FIXED_PARAMS
        ),
    }

    with open(
        out_dir
        / "Step_7_3D_LR_Diagnostic_Provenance.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            metadata,
            handle,
            indent=2,
        )


# =============================================================================
# OUTPUT PATHS
# =============================================================================
def output_paths() -> Dict[str, Path]:
    out_dir = (
        Config.OUT_DIR
        / "Step-7.3D-Performance_and_Map_Uncertainty"
    )

    paths = {
        "out": out_dir,
        "fold_dir": (
            out_dir
            / "cross_fold_ensemble_rasters"
        ),
        "worker_dir": (
            out_dir
            / "process_isolated_workers"
        ),
        "log_dir": (
            out_dir
            / "process_isolated_workers"
            / "logs"
        ),
        "performance": (
            out_dir
            / "LR_7_3D_Performance_Cluster_Bootstrap_Intervals.csv"
        ),
        "comparisons": (
            out_dir
            / "LR_7_3D_Additional_Paired_AUC_E_C_E_D.csv"
        ),
        "area_raw": (
            out_dir
            / "LR_7_3D_A30_E70_CrossFold_Area_Percentages_Raw.csv"
        ),
        "area_summary": (
            out_dir
            / "LR_7_3D_A30_E70_CrossFold_Area_Uncertainty_Summary.csv"
        ),
        "mean_raster": (
            out_dir
            / "LR_7_3D_A30_E70_CrossFold_Mean_Probability.tif"
        ),
        "sd_raster": (
            out_dir
            / "LR_7_3D_A30_E70_CrossFold_SD_Probability.tif"
        ),
        "domain_qa": (
            out_dir
            / "LR_7_3D_CrossFold_Raster_Domain_QA.csv"
        ),
        "sd_summary": (
            out_dir
            / "LR_7_3D_Pixelwise_SD_Summary.csv"
        ),
        "area_figure": (
            out_dir
            / "LR_7_3D_A30_E70_CrossFold_Area_Uncertainty.png"
        ),
        "sd_figure": (
            out_dir
            / "LR_7_3D_A30_E70_CrossFold_SD_Probability.png"
        ),
        "report": (
            out_dir
            / "Step_7_3D_LR_Uncertainty_Report.txt"
        ),
    }

    for directory in (
        paths[
            "out"
        ],
        paths[
            "fold_dir"
        ],
        paths[
            "worker_dir"
        ],
        paths[
            "log_dir"
        ],
    ):
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    return paths


def fold_paths(
    outer_fold: int,
) -> Dict[str, Path]:
    paths = output_paths()

    fold_dir = paths[
        "fold_dir"
    ]

    worker_dir = paths[
        "worker_dir"
    ]

    return {
        "model_a_raster": (
            fold_dir
            / f"outer_fold_{outer_fold}_Model_A_probability.tif"
        ),
        "model_e_raster": (
            fold_dir
            / f"outer_fold_{outer_fold}_Model_E_probability.tif"
        ),
        "ensemble_raster": (
            fold_dir
            / f"outer_fold_{outer_fold}_A30_E70_probability.tif"
        ),
        "area_csv": (
            worker_dir
            / f"outer_fold_{outer_fold}_Area_Percentages.csv"
        ),
        "metadata_json": (
            worker_dir
            / f"outer_fold_{outer_fold}_Metadata.json"
        ),
        "done": (
            worker_dir
            / f"outer_fold_{outer_fold}.done"
        ),
        "log": (
            paths[
                "log_dir"
            ]
            / f"outer_fold_{outer_fold}.log"
        ),
    }


# =============================================================================
# PELRORMANCE UNCERTAINTY
# =============================================================================
def append_metric_intervals(
    rows: list[Dict[str, Any]],
    label: str,
    scenario: str,
    y_true: np.ndarray,
    y_probability: np.ndarray,
    blocks: np.ndarray,
    seed: int,
    product_type: str,
) -> None:
    intervals = cluster_bootstrap_metric_intervals(
        y_true=y_true,
        y_prob=y_probability,
        blocks=blocks,
        n_bootstrap=UncertaintyConfig.N_BOOTSTRAP,
        ci_level=UncertaintyConfig.CI_LEVEL,
        seed=seed,
        threshold=Config.BINARY_THRESHOLD,
        metrics=(
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "f1",
            "brier",
            "precision",
            "recall",
        ),
    )

    for _, metric_row in intervals.iterrows():
        rows.append(
            {
                "product_type": product_type,
                "scenario": scenario,
                "model_or_product": label,
                "metric": metric_row[
                    "metric"
                ],
                "estimate": metric_row[
                    "estimate"
                ],
                "ci_low": metric_row[
                    "ci_low"
                ],
                "ci_high": metric_row[
                    "ci_high"
                ],
                "valid_bootstrap_iterations": metric_row[
                    "valid_bootstrap_iterations"
                ],
                "binary_threshold": (
                    Config.BINARY_THRESHOLD
                    if metric_row[
                        "metric"
                    ]
                    in {
                        "balanced_accuracy",
                        "f1",
                        "precision",
                        "recall",
                    }
                    else np.nan
                ),
            }
        )


def build_performance_uncertainty(
    out_dir: Path,
) -> pd.DataFrame:
    """
    Calculate cluster-bootstrap intervals from already saved genuine Step 7.3
    OOF predictions. No model fitting occurs here.
    """
    require_file(
        Config.OOF_PREDICTIONS_CSV,
        "successful Step 7.3 LR A-F OOF predictions",
    )

    all_oof = pd.read_csv(
        Config.OOF_PREDICTIONS_CSV
    )

    rows: list[
        Dict[str, Any]
    ] = []

    for (
        scenario_index,
        scenario_name,
    ) in enumerate(
        Config.SCENARIO_ORDER
    ):
        subset = all_oof.loc[
            all_oof[
                "scenario"
            ]
            == scenario_name
        ].copy()

        if len(
            subset
        ) != Config.EXPECTED_N_ROWS:
            raise RuntimeError(
                f"Unexpected OOF row count for {scenario_name}: "
                f"{len(subset)}; expected "
                f"{Config.EXPECTED_N_ROWS}."
            )

        if subset[
            Config.SAMPLE_ID_COLUMN
        ].duplicated().any():
            raise RuntimeError(
                "Duplicated OOF sample IDs found for "
                f"{scenario_name}."
            )

        append_metric_intervals(
            rows=rows,
            label=Config.MODEL_SHORT_LABELS[
                scenario_name
            ],
            scenario=scenario_name,
            y_true=subset[
                Config.TARGET_COLUMN
            ].to_numpy(
                dtype=int
            ),
            y_probability=subset[
                "oof_probability"
            ].to_numpy(
                dtype=float
            ),
            blocks=subset[
                Config.FOLD_COLUMN
            ].to_numpy(
                dtype=int
            ),
            seed=(
                UncertaintyConfig.RANDOM_STATE
                + scenario_index
            ),
            product_type="A-F model",
        )

    # Separately reported fixed operational ensemble.
    require_file(
        Config.ENSEMBLE_OOF_PREDICTIONS_CSV,
        "successful Step 7.3 LR A30/E70 ensemble OOF predictions",
    )

    ensemble = pd.read_csv(
        Config.ENSEMBLE_OOF_PREDICTIONS_CSV
    )

    required = {
        Config.SAMPLE_ID_COLUMN,
        Config.TARGET_COLUMN,
        Config.FOLD_COLUMN,
        "oof_probability_ensemble",
    }

    missing = (
        required
        - set(
            ensemble.columns
        )
    )

    if missing:
        raise RuntimeError(
            "LR ensemble OOF file is missing columns: "
            f"{sorted(missing)}"
        )

    if len(
        ensemble
    ) != Config.EXPECTED_N_ROWS:
        raise RuntimeError(
            "Unexpected LR A30/E70 ensemble OOF row count: "
            f"{len(ensemble)}; expected "
            f"{Config.EXPECTED_N_ROWS}."
        )

    if ensemble[
        Config.SAMPLE_ID_COLUMN
    ].duplicated().any():
        raise RuntimeError(
            "Duplicated sample IDs found in LR A30/E70 ensemble OOF file."
        )

    append_metric_intervals(
        rows=rows,
        label=(
            "A30/E70 fixed operational ensemble"
        ),
        scenario="A30_E70_ensemble",
        y_true=ensemble[
            Config.TARGET_COLUMN
        ].to_numpy(
            dtype=int
        ),
        y_probability=ensemble[
            "oof_probability_ensemble"
        ].to_numpy(
            dtype=float
        ),
        blocks=ensemble[
            Config.FOLD_COLUMN
        ].to_numpy(
            dtype=int
        ),
        seed=(
            UncertaintyConfig.RANDOM_STATE
            + 100
        ),
        product_type="operational ensemble",
    )

    result = pd.DataFrame(
        rows
    )

    result.to_csv(
        out_dir
        / "LR_7_3D_Performance_Cluster_Bootstrap_Intervals.csv",
        index=False,
        float_format="%.10f",
    )

    return result


def add_missing_E_C_and_E_D_comparisons(
    out_dir: Path,
) -> pd.DataFrame:
    """
    Add the same direct paired E-C and E-D spatial-block bootstrap comparisons
    used in Step 7.1D, now using successful Step 7.3 LR OOF predictions.
    """
    require_file(
        Config.OOF_PREDICTIONS_CSV,
        "successful Step 7.3 LR A-F OOF predictions",
    )

    all_oof = pd.read_csv(
        Config.OOF_PREDICTIONS_CSV
    )

    pivot = all_oof.pivot(
        index=Config.SAMPLE_ID_COLUMN,
        columns="scenario",
        values="oof_probability",
    )

    metadata = (
        all_oof.loc[
            all_oof[
                "scenario"
            ]
            == "A_static_baseline",
            [
                Config.SAMPLE_ID_COLUMN,
                Config.TARGET_COLUMN,
                Config.FOLD_COLUMN,
            ],
        ]
        .drop_duplicates(
            Config.SAMPLE_ID_COLUMN
        )
        .set_index(
            Config.SAMPLE_ID_COLUMN
        )
        .loc[
            pivot.index
        ]
    )

    y = metadata[
        Config.TARGET_COLUMN
    ].to_numpy(
        dtype=int
    )

    blocks = metadata[
        Config.FOLD_COLUMN
    ].to_numpy(
        dtype=int
    )

    comparisons = [
        (
            "E_physics_guided",
            "C_hkfi_only",
            "E - C",
            "Combined HKFI + SNDR versus HKFI alone",
        ),
        (
            "E_physics_guided",
            "D_sndr_only",
            "E - D",
            "Combined HKFI + SNDR versus SNDR alone",
        ),
    ]

    rows: list[
        Dict[str, Any]
    ] = []

    for (
        comparison_index,
        (
            left,
            right,
            label,
            purpose,
        ),
    ) in enumerate(
        comparisons
    ):
        (
            observed,
            low,
            high,
            n_valid,
        ) = paired_cluster_bootstrap_auc_difference(
            y_true=y,
            probability_left=pivot[
                left
            ].to_numpy(
                dtype=float
            ),
            probability_right=pivot[
                right
            ].to_numpy(
                dtype=float
            ),
            blocks=blocks,
            n_bootstrap=UncertaintyConfig.N_BOOTSTRAP,
            seed=(
                UncertaintyConfig.RANDOM_STATE
                + 1000
                + comparison_index
            ),
            ci_level=UncertaintyConfig.CI_LEVEL,
        )

        rows.append(
            {
                "comparison": label,
                "purpose": purpose,
                "left_scenario": left,
                "right_scenario": right,
                "auc_difference_left_minus_right": observed,
                "ci_low": low,
                "ci_high": high,
                "valid_bootstrap_iterations": int(
                    n_valid
                ),
            }
        )

    result = pd.DataFrame(
        rows
    )

    result.to_csv(
        out_dir
        / "LR_7_3D_Additional_Paired_AUC_E_C_E_D.csv",
        index=False,
        float_format="%.10f",
    )

    return result


# =============================================================================
# FOLD-SPECIFIC LR MODEL FITTING
# =============================================================================
def fit_outer_fold_model(
    df: pd.DataFrame,
    scenarios: Dict[str, Dict[str, Any]],
    outer_params: Dict[str, Any],
    scenario_name: str,
    outer_fold: int,
):
    """
    Refit one fold-specific LR map realization using the exact best parameters
    selected inside the corresponding successful Step 7.3 outer-training set.

    No hyperparameter tuning is repeated here.
    """
    features = list(
        scenarios[
            scenario_name
        ][
            "features"
        ]
    )

    train_df = df.loc[
        df[
            Config.FOLD_COLUMN
        ]
        != outer_fold
    ].copy()

    if train_df[
        Config.FOLD_COLUMN
    ].nunique() != 4:
        raise RuntimeError(
            f"Outer fold {outer_fold}: expected four training blocks."
        )

    params = get_fold_params(
        outer_params=outer_params,
        scenario_name=scenario_name,
        outer_fold=outer_fold,
    )

    pipeline = build_lr_pipeline(
        features
    )

    set_pipeline_params(
        pipeline,
        params,
    )

    X_train = train_df[
        features
    ]

    y_train = train_df[
        Config.TARGET_COLUMN
    ].to_numpy(
        dtype=int
    )

    pipeline.fit(
        X_train,
        y_train,
    )

    return (
        pipeline,
        features,
        params,
    )


# =============================================================================
# PROCESS-ISOLATED OUTER-FOLD MAP WORKER
# =============================================================================
def run_outer_fold_map_worker(
    outer_fold: int,
) -> None:
    scenarios = validate_scenario_manifests()

    df = load_and_validate_model_ready_dataset(
        scenarios
    )

    outer_params = load_outer_best_params()

    paths = fold_paths(
        outer_fold
    )

    log(
        f"[FOLD {outer_fold}] Fitting LR Model A on the four "
        "remaining spatial blocks..."
    )

    (
        model_a,
        features_a,
        params_a,
    ) = fit_outer_fold_model(
        df=df,
        scenarios=scenarios,
        outer_params=outer_params,
        scenario_name="A_static_baseline",
        outer_fold=outer_fold,
    )

    log(
        f"[FOLD {outer_fold}] Fitting LR Model E on the four "
        "remaining spatial blocks..."
    )

    (
        model_e,
        features_e,
        params_e,
    ) = fit_outer_fold_model(
        df=df,
        scenarios=scenarios,
        outer_params=outer_params,
        scenario_name="E_physics_guided",
        outer_fold=outer_fold,
    )

    log(
        f"[FOLD {outer_fold}] Predicting full available LR Model A domain..."
    )

    predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=model_a,
        scenario_name="A_static_baseline",
        features=features_a,
        output_path=paths[
            "model_a_raster"
        ],
    )

    log(
        f"[FOLD {outer_fold}] Predicting full available LR Model E domain..."
    )

    predict_scenario_raster_on_own_feature_domain(
        fitted_pipeline=model_e,
        scenario_name="E_physics_guided",
        features=features_e,
        output_path=paths[
            "model_e_raster"
        ],
    )

    log(
        f"[FOLD {outer_fold}] Creating fixed LR A30/E70 ensemble..."
    )

    blend_static_and_enhanced_probability_rasters(
        static_raster_path=paths[
            "model_a_raster"
        ],
        enhanced_raster_path=paths[
            "model_e_raster"
        ],
        output_path=paths[
            "ensemble_raster"
        ],
        enhanced_weight=float(
            Config.ENSEMBLE_ENHANCED_WEIGHT
        ),
    )

    area_df = calculate_susceptibility_area_percentages(
        paths[
            "ensemble_raster"
        ]
    )

    area_df.insert(
        0,
        "outer_fold_omitted_from_training",
        int(
            outer_fold
        ),
    )

    area_df.to_csv(
        paths[
            "area_csv"
        ],
        index=False,
        float_format="%.10f",
    )

    metadata = {
        "generated_at_utc": utc_now_iso(),
        "algorithm": "Logistic Regression",
        "outer_fold_omitted_from_training": int(
            outer_fold
        ),
        "training_blocks": [
            int(
                value
            )
            for value
            in sorted(
                df.loc[
                    df[
                        Config.FOLD_COLUMN
                    ]
                    != outer_fold,
                    Config.FOLD_COLUMN,
                ]
                .unique()
                .tolist()
            )
        ],
        "model_A_best_params": json_safe(
            params_a
        ),
        "model_E_best_params": json_safe(
            params_e
        ),
        "ensemble_enhanced_weight": float(
            Config.ENSEMBLE_ENHANCED_WEIGHT
        ),
        "model_A_raster": str(
            paths[
                "model_a_raster"
            ]
        ),
        "model_E_raster": str(
            paths[
                "model_e_raster"
            ]
        ),
        "ensemble_raster": str(
            paths[
                "ensemble_raster"
            ]
        ),
        "area_csv": str(
            paths[
                "area_csv"
            ]
        ),
        "interpretation": (
            "One Logistic Regression cross-fold model realization used to characterize "
            "map/model variability; not a formal independent validation map."
        ),
    }

    with open(
        paths[
            "metadata_json"
        ],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            json_safe(
                metadata
            ),
            handle,
            indent=2,
        )

    # Delete large intermediate A/E rasters only after successful ensemble
    # creation and mapped-area calculation.
    if not UncertaintyConfig.KEEP_INTERMEDIATE_A_E_RASTERS:
        for intermediate_path in (
            paths[
                "model_a_raster"
            ],
            paths[
                "model_e_raster"
            ],
        ):
            if intermediate_path.exists():
                intermediate_path.unlink()

    paths[
        "done"
    ].write_text(
        utc_now_iso(),
        encoding="utf-8",
    )

    del model_a
    del model_e
    gc.collect()

    log(
        f"[FOLD {outer_fold}] Completed successfully."
    )


def run_external_fold_worker(
    outer_fold: int,
) -> None:
    """
    Launch one outer-fold LR map realization in a fresh operating-system process.
    """
    script_path = resolve_script_path()

    paths = fold_paths(
        outer_fold
    )

    command = [
        sys.executable,
        str(
            script_path
        ),
        "--worker-fold",
        str(
            outer_fold
        ),
    ]

    environment = os.environ.copy()

    environment[
        "OMP_NUM_THREADS"
    ] = "1"

    environment[
        "OPENBLAS_NUM_THREADS"
    ] = "1"

    environment[
        "MKL_NUM_THREADS"
    ] = "1"

    environment[
        "VECLIB_MAXIMUM_THREADS"
    ] = "1"

    environment[
        "NUMEXPR_NUM_THREADS"
    ] = "1"

    log(
        f"[PROCESS] Starting LR outer fold {outer_fold} "
        "in a fresh Python process."
    )

    log(
        f"[PROCESS] Worker log: "
        f"{paths['log']}"
    )

    with open(
        paths[
            "log"
        ],
        "w",
        encoding="utf-8",
    ) as log_handle:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=str(
                script_path.parent
            ),
            env=environment,
        )

        assert (
            process.stdout
            is not None
        )

        for line in process.stdout:
            print(
                line,
                end="",
                flush=True,
            )

            log_handle.write(
                line
            )

            log_handle.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"LR outer-fold worker {outer_fold} failed with "
            f"return code {return_code}. See:\n"
            f"{paths['log']}"
        )


# =============================================================================
# RASTER VALID-DOMAIN QA
# =============================================================================
def raster_valid_mask(
    dataset,
    window: Window,
) -> np.ndarray:
    array = (
        dataset.read(
            1,
            window=window,
            masked=True,
        )
        .filled(
            np.nan
        )
        .astype(
            "float64"
        )
    )

    return np.isfinite(
        array
    )


def validate_cross_fold_raster_domains(
    fold_rasters: list[Path],
    final_full_data_raster: Path,
    output_csv: Path,
) -> pd.DataFrame:
    """
    Verify that all five LR fold ensemble maps:
      - are aligned;
      - have identical valid-pixel masks;
      - match the successful final full-data LR ensemble valid mask.
    """
    if not fold_rasters:
        raise ValueError(
            "No fold rasters supplied for domain QA."
        )

    for path in fold_rasters:
        require_file(
            path,
            "cross-fold LR ensemble probability raster",
        )

    require_file(
        final_full_data_raster,
        "successful final full-data LR ensemble raster",
    )

    datasets = [
        rasterio.open(
            path
        )
        for path in fold_rasters
    ]

    final_dataset = rasterio.open(
        final_full_data_raster
    )

    try:
        reference = datasets[
            0
        ]

        for (
            index,
            dataset,
        ) in enumerate(
            datasets[
                1:
            ],
            start=2,
        ):
            aligned = (
                dataset.width
                == reference.width
                and dataset.height
                == reference.height
                and dataset.transform
                == reference.transform
                and dataset.crs
                == reference.crs
            )

            if not aligned:
                raise RuntimeError(
                    "Cross-fold LR ensemble raster alignment mismatch: "
                    f"fold raster sequence {index}."
                )

        final_aligned = (
            final_dataset.width
            == reference.width
            and final_dataset.height
            == reference.height
            and final_dataset.transform
            == reference.transform
            and final_dataset.crs
            == reference.crs
        )

        if not final_aligned:
            raise RuntimeError(
                "Successful final full-data LR ensemble raster is not aligned "
                "with the cross-fold ensemble rasters."
            )

        valid_counts = np.zeros(
            len(
                datasets
            ),
            dtype=np.int64,
        )

        mismatch_vs_reference = np.zeros(
            len(
                datasets
            ),
            dtype=np.int64,
        )

        mismatch_vs_final = np.zeros(
            len(
                datasets
            ),
            dtype=np.int64,
        )

        final_valid_count = 0

        for row_start in range(
            0,
            reference.height,
            UncertaintyConfig.RASTER_BLOCK_ROWS,
        ):
            height = min(
                UncertaintyConfig.RASTER_BLOCK_ROWS,
                reference.height
                - row_start,
            )

            window = Window(
                0,
                row_start,
                reference.width,
                height,
            )

            masks = [
                raster_valid_mask(
                    dataset,
                    window,
                )
                for dataset in datasets
            ]

            final_mask = raster_valid_mask(
                final_dataset,
                window,
            )

            reference_mask = masks[
                0
            ]

            final_valid_count += int(
                final_mask.sum()
            )

            for (
                index,
                mask,
            ) in enumerate(
                masks
            ):
                valid_counts[
                    index
                ] += int(
                    mask.sum()
                )

                mismatch_vs_reference[
                    index
                ] += int(
                    np.count_nonzero(
                        mask
                        != reference_mask
                    )
                )

                mismatch_vs_final[
                    index
                ] += int(
                    np.count_nonzero(
                        mask
                        != final_mask
                    )
                )

        rows: list[
            Dict[str, Any]
        ] = []

        for (
            index,
            (
                raster_path,
                valid_count,
                mismatch_reference,
                mismatch_final,
            ),
        ) in enumerate(
            zip(
                fold_rasters,
                valid_counts,
                mismatch_vs_reference,
                mismatch_vs_final,
            ),
            start=1,
        ):
            rows.append(
                {
                    "raster": str(
                        raster_path
                    ),
                    "fold_raster_sequence": int(
                        index
                    ),
                    "valid_pixel_count": int(
                        valid_count
                    ),
                    "mismatched_pixels_vs_fold1_mask": int(
                        mismatch_reference
                    ),
                    "mismatched_pixels_vs_final_full_data_mask": int(
                        mismatch_final
                    ),
                    "identical_to_fold1_mask": bool(
                        mismatch_reference
                        == 0
                    ),
                    "identical_to_final_full_data_mask": bool(
                        mismatch_final
                        == 0
                    ),
                }
            )

        rows.append(
            {
                "raster": str(
                    final_full_data_raster
                ),
                "fold_raster_sequence": 0,
                "valid_pixel_count": int(
                    final_valid_count
                ),
                "mismatched_pixels_vs_fold1_mask": int(
                    mismatch_vs_final[
                        0
                    ]
                ),
                "mismatched_pixels_vs_final_full_data_mask": 0,
                "identical_to_fold1_mask": bool(
                    mismatch_vs_final[
                        0
                    ]
                    == 0
                ),
                "identical_to_final_full_data_mask": True,
            }
        )

        qa_df = pd.DataFrame(
            rows
        )

        qa_df.to_csv(
            output_csv,
            index=False,
        )

        if np.any(
            mismatch_vs_reference
            != 0
        ):
            raise RuntimeError(
                "Cross-fold LR ensemble rasters do not have identical "
                "valid-pixel masks. See:\n"
                f"{output_csv}"
            )

        if np.any(
            mismatch_vs_final
            != 0
        ):
            raise RuntimeError(
                "Cross-fold LR ensemble valid-pixel masks do not match the "
                "successful final full-data LR ensemble mask. See:\n"
                f"{output_csv}"
            )

        log(
            "[RASTER-QA] All five LR cross-fold ensemble rasters and the "
            "final full-data LR ensemble have identical valid domains: PASS"
        )

        return qa_df

    finally:
        for dataset in datasets:
            dataset.close()

        final_dataset.close()


# =============================================================================
# CROSS-FOLD RASTER AGGREGATION
# =============================================================================
def aggregate_fold_rasters(
    fold_rasters: list[Path],
    mean_output: Path,
    sd_output: Path,
) -> None:
    """
    Calculate pixelwise cross-fold mean and sample SD using the same method as
    Step 7.1D.
    """
    if not fold_rasters:
        raise ValueError(
            "No fold rasters supplied for aggregation."
        )

    datasets = [
        rasterio.open(
            path
        )
        for path in fold_rasters
    ]

    try:
        reference = datasets[
            0
        ]

        profile = reference.profile.copy()

        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=Config.OUTPUT_NODATA,
            compress="LZW",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        with rasterio.open(
            mean_output,
            "w",
            **profile,
        ) as mean_dataset, rasterio.open(
            sd_output,
            "w",
            **profile,
        ) as sd_dataset:
            for row_start in range(
                0,
                reference.height,
                UncertaintyConfig.RASTER_BLOCK_ROWS,
            ):
                height = min(
                    UncertaintyConfig.RASTER_BLOCK_ROWS,
                    reference.height
                    - row_start,
                )

                window = Window(
                    0,
                    row_start,
                    reference.width,
                    height,
                )

                arrays = [
                    dataset.read(
                        1,
                        window=window,
                        masked=True,
                    )
                    .filled(
                        np.nan
                    )
                    .astype(
                        "float64"
                    )
                    for dataset in datasets
                ]

                stack = np.stack(
                    arrays,
                    axis=0,
                )

                reference_valid = np.isfinite(
                    stack[
                        0
                    ]
                )

                for fold_index in range(
                    1,
                    stack.shape[
                        0
                    ],
                ):
                    candidate_valid = np.isfinite(
                        stack[
                            fold_index
                        ]
                    )

                    if not np.array_equal(
                        reference_valid,
                        candidate_valid,
                    ):
                        raise RuntimeError(
                            "Cross-fold LR ensemble rasters changed valid "
                            "domain during aggregation."
                        )

                mean_output_array = np.full(
                    reference_valid.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                sd_output_array = np.full(
                    reference_valid.shape,
                    Config.OUTPUT_NODATA,
                    dtype="float32",
                )

                if np.any(
                    reference_valid
                ):
                    valid_values = stack[
                        :,
                        reference_valid,
                    ]

                    with np.errstate(
                        invalid="ignore",
                        divide="ignore",
                    ):
                        mean_values = np.mean(
                            valid_values,
                            axis=0,
                        )

                        sd_values = np.std(
                            valid_values,
                            axis=0,
                            ddof=1,
                        )

                    mean_output_array[
                        reference_valid
                    ] = mean_values.astype(
                        "float32"
                    )

                    sd_output_array[
                        reference_valid
                    ] = sd_values.astype(
                        "float32"
                    )

                mean_dataset.write(
                    mean_output_array,
                    1,
                    window=window,
                )

                sd_dataset.write(
                    sd_output_array,
                    1,
                    window=window,
                )

            mean_dataset.set_band_description(
                1,
                "LR cross-fold mean A30/E70 susceptibility probability",
            )

            mean_dataset.update_tags(
                algorithm="Logistic Regression",
                uncertainty_type=(
                    "cross-fold model variability"
                ),
                number_of_outer_fold_models=len(
                    fold_rasters
                ),
                generated_at_utc=utc_now_iso(),
            )

            sd_dataset.set_band_description(
                1,
                "LR cross-fold SD of A30/E70 susceptibility probability",
            )

            sd_dataset.update_tags(
                algorithm="Logistic Regression",
                uncertainty_type=(
                    "cross-fold model variability"
                ),
                number_of_outer_fold_models=len(
                    fold_rasters
                ),
                ddof=1,
                generated_at_utc=utc_now_iso(),
                interpretation=(
                    "Sample standard deviation across five LR outer-fold model "
                    "realizations; not a formal 95% confidence interval."
                ),
            )

    finally:
        for dataset in datasets:
            dataset.close()


# =============================================================================
# AREA UNCERTAINTY
# =============================================================================
def final_full_data_area_rows() -> pd.DataFrame:
    require_file(
        Config.ENSEMBLE_PROBABILITY_TIF,
        "successful final Step 7.3 LR A30/E70 ensemble map",
    )

    final_area = (
        calculate_susceptibility_area_percentages(
            Config.ENSEMBLE_PROBABILITY_TIF
        )[
            [
                "class_id",
                "class_label",
                "area_percentage",
            ]
        ]
        .copy()
    )

    high_plus_very_high = float(
        final_area.loc[
            final_area[
                "class_label"
            ].isin(
                [
                    "High",
                    "Very High",
                ]
            ),
            "area_percentage",
        ].sum()
    )

    final_area = pd.concat(
        [
            final_area,
            pd.DataFrame(
                [
                    {
                        "class_id": 99,
                        "class_label": (
                            "High + Very High"
                        ),
                        "area_percentage": (
                            high_plus_very_high
                        ),
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    return final_area.rename(
        columns={
            "area_percentage": (
                "final_full_data_area_percentage"
            )
        }
    )


def collect_fold_area_results() -> tuple[
    pd.DataFrame,
    list[Path],
]:
    area_frames: list[
        pd.DataFrame
    ] = []

    ensemble_rasters: list[
        Path
    ] = []

    for outer_fold in Config.EXPECTED_FOLDS:
        paths = fold_paths(
            outer_fold
        )

        require_file(
            paths[
                "done"
            ],
            f"LR outer-fold {outer_fold} completion checkpoint",
        )

        require_file(
            paths[
                "area_csv"
            ],
            f"LR outer-fold {outer_fold} area-percentage CSV",
        )

        require_file(
            paths[
                "ensemble_raster"
            ],
            f"LR outer-fold {outer_fold} ensemble raster",
        )

        area_df = pd.read_csv(
            paths[
                "area_csv"
            ]
        )

        required = {
            "outer_fold_omitted_from_training",
            "class_id",
            "class_label",
            "area_percentage",
        }

        missing = (
            required
            - set(
                area_df.columns
            )
        )

        if missing:
            raise RuntimeError(
                f"LR fold {outer_fold} area CSV is missing columns: "
                f"{sorted(missing)}"
            )

        area_frames.append(
            area_df
        )

        ensemble_rasters.append(
            paths[
                "ensemble_raster"
            ]
        )

    return (
        pd.concat(
            area_frames,
            ignore_index=True,
        ),
        ensemble_rasters,
    )


def summarize_mapped_area_uncertainty(
    area_raw: pd.DataFrame,
) -> pd.DataFrame:
    area_summary = (
        area_raw.groupby(
            [
                "class_id",
                "class_label",
            ],
            as_index=False,
        )
        .agg(
            mean_area_percentage=(
                "area_percentage",
                "mean",
            ),
            sd_area_percentage=(
                "area_percentage",
                "std",
            ),
            minimum_area_percentage=(
                "area_percentage",
                "min",
            ),
            maximum_area_percentage=(
                "area_percentage",
                "max",
            ),
            n_spatial_fold_models=(
                "outer_fold_omitted_from_training",
                "nunique",
            ),
        )
        .sort_values(
            "class_id",
            ascending=False,
        )
    )

    high_combined = (
        area_raw.loc[
            area_raw[
                "class_label"
            ].isin(
                [
                    "High",
                    "Very High",
                ]
            )
        ]
        .groupby(
            "outer_fold_omitted_from_training",
            as_index=False,
        )[
            "area_percentage"
        ]
        .sum()
    )

    area_summary = pd.concat(
        [
            area_summary,
            pd.DataFrame(
                [
                    {
                        "class_id": 99,
                        "class_label": (
                            "High + Very High"
                        ),
                        "mean_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].mean()
                        ),
                        "sd_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].std(
                                ddof=1
                            )
                        ),
                        "minimum_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].min()
                        ),
                        "maximum_area_percentage": float(
                            high_combined[
                                "area_percentage"
                            ].max()
                        ),
                        "n_spatial_fold_models": int(
                            len(
                                high_combined
                            )
                        ),
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    final_area = final_full_data_area_rows()

    area_summary = area_summary.merge(
        final_area,
        on=[
            "class_id",
            "class_label",
        ],
        how="left",
        validate="one_to_one",
    )

    area_summary[
        "final_minus_crossfold_mean_percentage_points"
    ] = (
        area_summary[
            "final_full_data_area_percentage"
        ]
        - area_summary[
            "mean_area_percentage"
        ]
    )

    area_summary[
        "crossfold_range_percentage_points"
    ] = (
        area_summary[
            "maximum_area_percentage"
        ]
        - area_summary[
            "minimum_area_percentage"
        ]
    )

    return area_summary


# =============================================================================
# PIXELWISE SD QUANTIFICATION
# =============================================================================
def summarize_pixelwise_sd(
    sd_raster_path: Path,
    output_csv: Path,
) -> pd.DataFrame:
    require_file(
        sd_raster_path,
        "LR cross-fold SD probability raster",
    )

    chunks: list[
        np.ndarray
    ] = []

    with rasterio.open(
        sd_raster_path
    ) as dataset:
        for row_start in range(
            0,
            dataset.height,
            UncertaintyConfig.RASTER_BLOCK_ROWS,
        ):
            height = min(
                UncertaintyConfig.RASTER_BLOCK_ROWS,
                dataset.height
                - row_start,
            )

            window = Window(
                0,
                row_start,
                dataset.width,
                height,
            )

            values = (
                dataset.read(
                    1,
                    window=window,
                    masked=True,
                )
                .compressed()
                .astype(
                    "float64"
                )
            )

            if values.size:
                chunks.append(
                    values
                )

    if not chunks:
        raise RuntimeError(
            "No valid values found in LR cross-fold SD raster."
        )

    values = np.concatenate(
        chunks
    )

    summary = {
        "algorithm": "Logistic Regression",
        "valid_pixels": int(
            values.size
        ),
        "mean_pixelwise_sd": float(
            np.mean(
                values
            )
        ),
        "median_pixelwise_sd": float(
            np.median(
                values
            )
        ),
        "p75_pixelwise_sd": float(
            np.percentile(
                values,
                75.0,
            )
        ),
        "p90_pixelwise_sd": float(
            np.percentile(
                values,
                90.0,
            )
        ),
        "p95_pixelwise_sd": float(
            np.percentile(
                values,
                95.0,
            )
        ),
        "p99_pixelwise_sd": float(
            np.percentile(
                values,
                99.0,
            )
        ),
        "maximum_pixelwise_sd": float(
            np.max(
                values
            )
        ),
    }

    for threshold in UncertaintyConfig.SD_THRESHOLDS:
        key = (
            "pixels_with_sd_greater_than_"
            + str(
                threshold
            ).replace(
                ".",
                "p",
            )
            + "_percent"
        )

        summary[
            key
        ] = float(
            100.0
            * np.mean(
                values
                > float(
                    threshold
                )
            )
        )

    result = pd.DataFrame(
        [
            summary
        ]
    )

    result.to_csv(
        output_csv,
        index=False,
        float_format="%.10f",
    )

    return result


# =============================================================================
# FIGURES — SAME STYLE AS STEP 7.1D
# =============================================================================
def create_area_uncertainty_figure(
    area_summary: pd.DataFrame,
    output_path: Path,
) -> None:
    plot_df = area_summary.loc[
        area_summary[
            "class_id"
        ].isin(
            [
                1,
                2,
                3,
                4,
                5,
            ]
        )
    ].copy()

    plot_df = plot_df.sort_values(
        "class_id",
        ascending=True,
    )

    figure, axis = plt.subplots(
        figsize=(
            6.2,
            4.3,
        )
    )

    x = np.arange(
        len(
            plot_df
        )
    )

    means = plot_df[
        "mean_area_percentage"
    ].to_numpy(
        dtype=float
    )

    standard_deviation = (
        plot_df[
            "sd_area_percentage"
        ]
        .fillna(
            0.0
        )
        .to_numpy(
            dtype=float
        )
    )

    final = plot_df[
        "final_full_data_area_percentage"
    ].to_numpy(
        dtype=float
    )

    axis.bar(
        x,
        means,
        yerr=standard_deviation,
        capsize=3,
        edgecolor="black",
        linewidth=0.7,
        label="Cross-fold mean ± SD",
    )

    axis.scatter(
        x,
        final,
        marker="D",
        zorder=5,
        label="Final full-data map",
    )

    axis.set_xticks(
        x
    )

    axis.set_xticklabels(
        plot_df[
            "class_label"
        ],
        rotation=20,
        ha="right",
    )

    axis.set_ylabel(
        "Mapped area (%)"
    )

    axis.grid(
        True,
        axis="y",
        alpha=0.25,
    )

    axis.legend(
        frameon=False,
        fontsize=8,
    )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=UncertaintyConfig.FIG_DPI,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


# -----------------------------------------------------------------------------
# CARTOGRAPHIC HELPERS FOR THE PIXELWISE SD FIGURE ONLY
# -----------------------------------------------------------------------------
# These constants reproduce the established Step 7.1/7.2/7.3 ensemble-map style.
# They are deliberately local to the visualization layer and do not alter the
# Step 7.3D analytical/model configuration.
SD_MAP_PROJECT_BASE = Config.BASE
SD_MAP_STUDY_SHP = SD_MAP_PROJECT_BASE / "Study_Area.shp"
SD_MAP_ROAD_SHP = SD_MAP_PROJECT_BASE / "NTD_Road.shp"

SD_MAP_FIGSIZE = (4, 6)
SD_MAP_DMS_TICK_FONTSIZE = 7
SD_MAP_LEGEND_FONTSIZE = 7
SD_MAP_LEGEND_TITLE_FONTSIZE = 8
SD_MAP_COLORBAR_TICK_FONTSIZE = 8
SD_MAP_COLORBAR_LABEL_FONTSIZE = 9
SD_MAP_NORTH_ARROW_FONTSIZE = 8
SD_MAP_CAX_SIZE = "5%"
SD_MAP_CAX_PAD = 0.01
SD_MAP_DPI = 900

SD_MAP_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "lr_crossfold_sd",
    [
        "#006400",
        "#32CD32",
        "#FFFF00",
        "#FFA500",
        "#FF4500",
        "#8B0000",
    ],
    N=256,
)


def _sd_to_dms(value: float, is_lon: bool = True) -> str:
    absolute = abs(value)
    degrees = int(absolute)
    minutes_float = (absolute - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0

    hemisphere = (
        ("E" if value >= 0 else "W")
        if is_lon
        else ("N" if value >= 0 else "S")
    )

    return (
        f"{degrees}°{minutes:02d}'"
        f'{seconds:04.1f}"{hemisphere}'
    )


def _sd_metres_per_degree_lon(latitude_deg: float) -> float:
    return 111320.0 * np.cos(np.radians(latitude_deg))


def _sd_nice_length_125(target_metres: float) -> float:
    target_metres = max(float(target_metres), 1.0)
    exponent = int(np.floor(np.log10(target_metres)))
    base_value = 10 ** exponent

    for multiplier in (1, 2, 5, 10):
        candidate = multiplier * base_value
        if candidate >= target_metres:
            return candidate

    return 10 * base_value


def _sd_add_north_arrow(
    axis,
    size: float = 0.08,
    pad: float = 0.06,
    label: str = "N",
    arrow_downshift: float = 0.025,
) -> None:
    label_anchor_x = pad + size * 0.36
    label_anchor_y = (1.0 - pad) + size * 0.10

    x0 = pad
    y0 = 1.0 - pad - size - arrow_downshift
    x1 = x0 + size * 0.72
    y1 = y0 + size

    apex = ((x0 + x1) / 2.0, y1)
    left_base = (x0, y0)
    right_base = (x1, y0)
    tail = ((x0 + x1) / 2.0, y0 - size * 0.09)

    axis.add_patch(
        Polygon(
            [apex, left_base, tail],
            closed=True,
            transform=axis.transAxes,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.add_patch(
        Polygon(
            [apex, tail, right_base],
            closed=True,
            transform=axis.transAxes,
            facecolor="black",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
    )
    axis.text(
        label_anchor_x,
        label_anchor_y,
        label,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=SD_MAP_NORTH_ARROW_FONTSIZE,
        fontweight="bold",
        color="black",
        zorder=21,
    )


def _sd_add_checker_scale_bar(
    axis,
    west: float,
    east: float,
    south: float,
    north: float,
    mid_lat_deg: float,
    segments: int = 4,
    fontsize: float = 6,
    frame_lw: float = 0.4,
) -> None:
    width_data = east - west
    height_data = north - south

    metres_per_degree = _sd_metres_per_degree_lon(mid_lat_deg)
    width_metres = width_data * metres_per_degree
    data_units_per_metre = 1.0 / metres_per_degree

    total_length_metres = _sd_nice_length_125(width_metres / 5.0)
    segment_length_metres = total_length_metres / float(segments)

    total_length_units = total_length_metres * data_units_per_metre
    segment_length_units = segment_length_metres * data_units_per_metre

    x_start = west + 0.5 * width_data - 0.5 * total_length_units
    y_base = south + 0.06 * height_data
    bar_height = 0.018 * height_data

    pad_x = total_length_units * 0.08
    pad_y = 0.035 * height_data

    axis.add_patch(
        Rectangle(
            (x_start - pad_x, y_base - pad_y),
            total_length_units + 2.0 * pad_x,
            bar_height + 2.0 * pad_y,
            transform=axis.transData,
            facecolor="white",
            edgecolor="none",
            alpha=0.70,
            zorder=12,
        )
    )

    for index in range(segments):
        axis.add_patch(
            Rectangle(
                (x_start + index * segment_length_units, y_base),
                segment_length_units,
                bar_height,
                transform=axis.transData,
                facecolor=("black" if index % 2 == 0 else "white"),
                edgecolor="none",
                zorder=13,
            )
        )

    axis.add_patch(
        Rectangle(
            (x_start, y_base),
            total_length_units,
            bar_height,
            transform=axis.transData,
            fill=False,
            edgecolor="black",
            linewidth=frame_lw,
            zorder=14,
        )
    )

    axis.text(
        x_start + total_length_units / 2.0,
        y_base + bar_height + 0.012 * height_data,
        "km",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )
    axis.text(
        x_start,
        y_base - 0.012 * height_data,
        "0",
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )

    end_km = total_length_metres / 1000.0
    end_label = (
        f"{int(round(end_km))}"
        if abs(end_km - int(round(end_km))) < 1.0e-6
        else f"{end_km:.1f}"
    )

    axis.text(
        x_start + total_length_units,
        y_base - 0.012 * height_data,
        end_label,
        ha="center",
        va="top",
        fontsize=fontsize,
        color="black",
        zorder=15,
    )


def _sd_reproject_to_epsg4326(
    source_path: Path,
    output_path: Path,
) -> Path:
    """Create an EPSG:4326 display copy; the analytical SD raster is unchanged."""
    require_file(
        source_path,
        "cross-fold SD probability raster for visualization",
    )

    with rasterio.open(source_path) as source:
        if source.crs is None:
            raise RuntimeError(
                f"Cross-fold SD raster has no CRS: {source_path}"
            )

        if source.crs.to_epsg() == 4326:
            return source_path

        destination_crs = "EPSG:4326"
        destination_transform, width, height = calculate_default_transform(
            source.crs,
            destination_crs,
            source.width,
            source.height,
            *source.bounds,
        )

        output_nodata = (
            float(source.nodata)
            if source.nodata is not None
            else -9999.0
        )

        profile = source.profile.copy()
        profile.update(
            driver="GTiff",
            crs=destination_crs,
            transform=destination_transform,
            width=width,
            height=height,
            count=1,
            dtype="float32",
            nodata=output_nodata,
            compress="LZW",
            BIGTIFF="IF_SAFER",
        )

        temporary_path = Path(str(output_path) + ".part")
        if temporary_path.exists():
            temporary_path.unlink()

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as destination:
            reproject(
                source=rasterio.band(source, 1),
                destination=rasterio.band(destination, 1),
                src_transform=source.transform,
                src_crs=source.crs,
                src_nodata=source.nodata,
                dst_transform=destination_transform,
                dst_crs=destination_crs,
                dst_nodata=output_nodata,
                resampling=Resampling.bilinear,
            )

            destination.set_band_description(
                1,
                "RF cross-fold SD of A30/E70 susceptibility probability - EPSG:4326 visualization copy",
            )

    if output_path.exists():
        output_path.unlink()

    os.replace(
        temporary_path,
        output_path,
    )

    log(
        "[SD-VIZ] EPSG:4326 display raster created: "
        f"{output_path}"
    )

    return output_path


def _sd_load_vector_overlay(
    vector_path: Path,
    target_crs: str,
) -> dict[str, list[dict[str, Any]]]:
    """Load a vector with Fiona and transform geometries to the display CRS."""
    require_file(
        vector_path,
        "cartographic vector overlay",
    )

    if not HAVE_FIONA:
        raise ImportError(
            "Fiona is required for the Step 7.1-style SD map overlays. "
            "The successful Step 7.1/7.2 workflow also uses Fiona-compatible shapefiles."
        )

    geometries: list[dict[str, Any]] = []

    with fiona.open(vector_path, "r") as source:
        source_crs = source.crs_wkt or source.crs
        if not source_crs:
            raise RuntimeError(
                f"Vector layer has no CRS: {vector_path}"
            )

        for feature in source:
            geometry = feature.get("geometry")
            if geometry is None:
                continue

            geometries.append(
                transform_geom(
                    source_crs,
                    target_crs,
                    geometry,
                    precision=-1,
                )
            )

    return {"geometries": geometries}


def _sd_plot_vector_overlay(
    axis,
    vector_data: dict[str, list[dict[str, Any]]],
    colour: str,
    linewidth: float,
    zorder: int,
) -> None:
    for geometry in vector_data.get("geometries", []):
        if not geometry:
            continue

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")
        if coordinates is None:
            continue

        if geometry_type == "LineString":
            line_groups = [coordinates]
        elif geometry_type == "MultiLineString":
            line_groups = list(coordinates)
        elif geometry_type == "Polygon":
            line_groups = [coordinates[0]] if coordinates else []
        elif geometry_type == "MultiPolygon":
            line_groups = [
                polygon[0]
                for polygon in coordinates
                if polygon
            ]
        else:
            continue

        for line in line_groups:
            xy = np.asarray(line, dtype="float64")
            if xy.ndim != 2 or xy.shape[1] < 2:
                continue

            axis.plot(
                xy[:, 0],
                xy[:, 1],
                color=colour,
                linewidth=linewidth,
                zorder=zorder,
            )


def _sd_build_clip_patch(
    study_vector: dict[str, list[dict[str, Any]]],
    axis,
):
    """Build the same opaque-background polygon clip used by the Step 7.1 map."""
    all_vertices: list[np.ndarray] = []
    all_codes: list[int] = []

    def add_ring(ring_coordinates) -> None:
        coordinates = np.asarray(
            ring_coordinates,
            dtype="float64",
        )

        if (
            coordinates.ndim != 2
            or coordinates.shape[0] < 3
            or coordinates.shape[1] < 2
        ):
            return

        coordinates = coordinates[:, :2]

        if not np.allclose(
            coordinates[0],
            coordinates[-1],
            equal_nan=False,
        ):
            coordinates = np.vstack(
                [coordinates, coordinates[0]]
            )

        n_vertices = len(coordinates)
        codes = (
            [MplPath.MOVETO]
            + [MplPath.LINETO] * (n_vertices - 2)
            + [MplPath.CLOSEPOLY]
        )

        all_vertices.append(coordinates)
        all_codes.extend(codes)

    for geometry in study_vector.get("geometries", []):
        if not geometry:
            continue

        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")

        if geometry_type == "Polygon" and coordinates is not None:
            for ring in coordinates:
                add_ring(ring)

        elif geometry_type == "MultiPolygon" and coordinates is not None:
            for polygon in coordinates:
                for ring in polygon:
                    add_ring(ring)

    if not all_vertices:
        return None

    compound_path = MplPath(
        np.concatenate(all_vertices, axis=0),
        all_codes,
    )

    clip_patch = PathPatch(
        compound_path,
        facecolor="none",
        edgecolor="none",
        linewidth=0.0,
        transform=axis.transData,
        zorder=0,
    )

    axis.add_patch(clip_patch)
    return clip_patch


def create_pixelwise_sd_figure(
    sd_raster_path: Path,
    output_path: Path,
) -> None:
    """
    Plot the existing LR pixelwise cross-fold SD raster with the exact
    cartographic style already used for the corrected Step 7.1D SD map.

    This function changes visualization only. It does not change the LR SD raster,
    the five LR cross-fold realizations, the aggregation, ddof=1, domain QA, or
    any uncertainty statistic.
    """
    if not UncertaintyConfig.GENERATE_PIXELWISE_SD_FIGURE:
        return

    require_file(
        sd_raster_path,
        "cross-fold SD probability raster",
    )
    require_file(
        SD_MAP_STUDY_SHP,
        "study-area shapefile used by the Step 7.1 map style",
    )
    require_file(
        SD_MAP_ROAD_SHP,
        "road shapefile used by the Step 7.1 map style",
    )

    display_raster_path = output_path.with_name(
        output_path.stem + "_EPSG4326_Display.tif"
    )

    display_raster_path = _sd_reproject_to_epsg4326(
        source_path=sd_raster_path,
        output_path=display_raster_path,
    )

    with rasterio.open(display_raster_path) as dataset:
        sd = (
            dataset.read(1, masked=True)
            .filled(np.nan)
            .astype("float64")
        )
        bounds = dataset.bounds

    valid_sd = sd[np.isfinite(sd)]
    if valid_sd.size == 0:
        raise RuntimeError(
            "No valid values found in the cross-fold SD raster for plotting."
        )

    extent = (
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    )

    study_boundary = _sd_load_vector_overlay(
        SD_MAP_STUDY_SHP,
        "EPSG:4326",
    )
    roads = _sd_load_vector_overlay(
        SD_MAP_ROAD_SHP,
        "EPSG:4326",
    )

    display_array = np.ma.masked_invalid(sd)

    map_cmap = SD_MAP_CMAP.copy()
    map_cmap.set_bad(
        color="white",
        alpha=1.0,
    )

    sd_vmax = float(np.max(valid_sd))
    if not np.isfinite(sd_vmax) or sd_vmax <= 0.0:
        sd_vmax = 1.0

    figure, axis = plt.subplots(
        1,
        1,
        figsize=SD_MAP_FIGSIZE,
        facecolor="white",
    )
    figure.patch.set_facecolor("white")
    figure.patch.set_alpha(1.0)
    axis.set_facecolor("white")

    image = axis.imshow(
        display_array,
        extent=extent,
        origin="upper",
        vmin=0.0,
        vmax=sd_vmax,
        cmap=map_cmap,
        interpolation="nearest",
        alpha=1.0,
        zorder=1,
    )

    clip_patch = _sd_build_clip_patch(
        study_boundary,
        axis,
    )
    if clip_patch is not None:
        image.set_clip_path(clip_patch)

    _sd_plot_vector_overlay(
        axis,
        study_boundary,
        colour="black",
        linewidth=2.0,
        zorder=1000,
    )
    _sd_plot_vector_overlay(
        axis,
        roads,
        colour="red",
        linewidth=1.0,
        zorder=1100,
    )

    # Exact raster extent, matching the successful Step 7.1 ensemble map.
    axis.set_xlim(extent[0], extent[1])
    axis.set_ylim(extent[2], extent[3])

    axis.xaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: _sd_to_dms(value, True)
        )
    )
    axis.yaxis.set_major_formatter(
        FuncFormatter(
            lambda value, position: _sd_to_dms(value, False)
        )
    )
    axis.tick_params(
        axis="both",
        which="major",
        labelsize=SD_MAP_DMS_TICK_FONTSIZE,
    )

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")

    for label in axis.get_yticklabels():
        label.set_rotation(90)
        label.set_va("center")

    _sd_add_north_arrow(
        axis,
        size=0.08,
        pad=0.06,
    )

    _sd_add_checker_scale_bar(
        axis,
        extent[0],
        extent[1],
        extent[2],
        extent[3],
        mid_lat_deg=0.5 * (extent[2] + extent[3]),
        segments=4,
        fontsize=6,
    )

    legend_handles = [
        Line2D(
            [0],
            [0],
            color="red",
            linewidth=1.5,
            label="Highway/Roads",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=2.0,
            label="Study Area Boundary",
        ),
    ]

    legend = axis.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.50),
        fontsize=SD_MAP_LEGEND_FONTSIZE,
        title="Legend",
        title_fontsize=SD_MAP_LEGEND_TITLE_FONTSIZE,
        frameon=True,
        fancybox=True,
        shadow=True,
        facecolor="white",
        edgecolor="black",
        framealpha=1.0,
    )
    legend.get_title().set_fontweight("bold")
    legend.set_zorder(2000)

    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes(
        "right",
        size=SD_MAP_CAX_SIZE,
        pad=SD_MAP_CAX_PAD,
    )

    colorbar = figure.colorbar(
        image,
        cax=colorbar_axis,
    )
    colorbar.ax.tick_params(
        labelsize=SD_MAP_COLORBAR_TICK_FONTSIZE,
    )
    colorbar.set_label(
        "Cross-fold probability SD",
        fontsize=SD_MAP_COLORBAR_LABEL_FONTSIZE,
        fontweight="bold",
    )

    figure.tight_layout()

    pdf_path = output_path.with_suffix(".pdf")
    svg_path = output_path.with_suffix(".svg")

    figure.savefig(
        output_path,
        dpi=SD_MAP_DPI,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False,
    )
    figure.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False,
    )
    figure.savefig(
        svg_path,
        format="svg",
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white",
        transparent=False,
    )

    plt.close(figure)

    log(f"[SD-VIZ] PNG saved: {output_path}")
    log(f"[SD-VIZ] PDF saved: {pdf_path}")
    log(f"[SD-VIZ] SVG saved: {svg_path}")


# =============================================================================
# REPORT
# =============================================================================
def write_uncertainty_report(
    paths: Dict[str, Path],
    performance_intervals: pd.DataFrame,
    additional_comparisons: pd.DataFrame,
    area_summary: pd.DataFrame,
    domain_qa: pd.DataFrame,
    sd_summary: pd.DataFrame,
) -> None:
    with open(
        paths[
            "report"
        ],
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(
            "STEP 7.3D — LOGISTIC REGRESSION PELRORMANCE AND "
            "MAPPED-AREA UNCERTAINTY\n"
        )

        handle.write(
            "="
            * 92
            + "\n\n"
        )

        handle.write(
            "Generated: "
            + utc_now_iso()
            + "\n\n"
        )

        handle.write(
            "SCIENTIFIC SCOPE\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            "Primary model framework: successful Step 7.3 equal-HKFI, 100 m "
            "pseudo-absence buffer Logistic Regression experiment.\n"
        )

        handle.write(
            "Step 7.3B alternative HKFI weights and Step 7.3C alternative "
            "pseudo-absence buffers are treated as separate sensitivity "
            "analyses and are not pooled into this uncertainty analysis.\n\n"
        )

        handle.write(
            "PELRORMANCE UNCERTAINTY\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            "Intervals are spatial-block cluster-bootstrap intervals generated "
            "from genuine successful Step 7.3 nested outer-fold LR OOF "
            "predictions.\n\n"
        )

        handle.write(
            performance_intervals.to_string(
                index=False
            )
        )

        handle.write(
            "\n\n"
        )

        handle.write(
            "ADDITIONAL PAIRED AUC COMPARISONS\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            additional_comparisons.to_string(
                index=False
            )
        )

        handle.write(
            "\n\n"
        )

        handle.write(
            "CROSS-FOLD MAPPED-AREA VARIABILITY\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            "The five LR map realizations are summarized by mean, SD, minimum "
            "and maximum. These are not formal 95% confidence intervals.\n\n"
        )

        handle.write(
            area_summary.to_string(
                index=False
            )
        )

        handle.write(
            "\n\n"
        )

        handle.write(
            "RASTER VALID-DOMAIN QA\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            domain_qa.to_string(
                index=False
            )
        )

        handle.write(
            "\n\n"
        )

        handle.write(
            "PIXELWISE CROSS-FOLD SD SUMMARY\n"
        )

        handle.write(
            "-"
            * 92
            + "\n"
        )

        handle.write(
            sd_summary.to_string(
                index=False
            )
        )

        handle.write(
            "\n"
        )


# =============================================================================
# CONTROLLER
# =============================================================================
def run_controller() -> None:
    paths = output_paths()

    # Validate the primary Step 7.3 data/model contract before analysis.
    scenarios = validate_scenario_manifests()

    load_and_validate_model_ready_dataset(
        scenarios
    )

    load_outer_best_params()

    # Validate final LR ensemble map before expensive cross-fold map work.
    require_file(
        Config.ENSEMBLE_PROBABILITY_TIF,
        "successful final Step 7.3 LR A30/E70 ensemble probability raster",
    )

    write_diagnostic_provenance(
        paths[
            "out"
        ]
    )

    log(
        "="
        * 96
    )

    log(
        "STEP 7.3D — LOGISTIC REGRESSION PELRORMANCE AND MAPPED-AREA UNCERTAINTY"
    )

    log(
        "="
        * 96
    )

    # -------------------------------------------------------------------------
    # 1. Performance uncertainty — no LR model fitting.
    # -------------------------------------------------------------------------
    log(
        "[1/6] Calculating LR cluster-bootstrap uncertainty "
        "for Models A-F and the A30/E70 ensemble..."
    )

    performance_intervals = build_performance_uncertainty(
        paths[
            "out"
        ]
    )

    # -------------------------------------------------------------------------
    # 2. Additional direct paired AUC comparisons.
    # -------------------------------------------------------------------------
    log(
        "[2/6] Calculating direct LR E-C and E-D paired AUC comparisons..."
    )

    additional_comparisons = add_missing_E_C_and_E_D_comparisons(
        paths[
            "out"
        ]
    )

    # -------------------------------------------------------------------------
    # 3. Five process-isolated cross-fold LR ensemble maps.
    # -------------------------------------------------------------------------
    log(
        "[3/6] Building/resuming five process-isolated "
        "LR cross-fold A30/E70 map realizations..."
    )

    for outer_fold in Config.EXPECTED_FOLDS:
        fold_output = fold_paths(
            outer_fold
        )

        complete = (
            fold_output[
                "done"
            ].exists()
            and fold_output[
                "ensemble_raster"
            ].exists()
            and fold_output[
                "area_csv"
            ].exists()
            and fold_output[
                "metadata_json"
            ].exists()
        )

        if (
            UncertaintyConfig.RESUME_FROM_CHECKPOINTS
            and complete
        ):
            log(
                f"[RESUME] LR outer fold {outer_fold} already complete."
            )

            continue

        run_external_fold_worker(
            outer_fold
        )

    # -------------------------------------------------------------------------
    # 4. Collect mapped areas and validate raster domains.
    # -------------------------------------------------------------------------
    log(
        "[4/6] Collecting LR fold-area results and checking "
        "identical raster support domains..."
    )

    (
        area_raw,
        ensemble_rasters,
    ) = collect_fold_area_results()

    area_raw.to_csv(
        paths[
            "area_raw"
        ],
        index=False,
        float_format="%.10f",
    )

    domain_qa = validate_cross_fold_raster_domains(
        fold_rasters=ensemble_rasters,
        final_full_data_raster=(
            Config.ENSEMBLE_PROBABILITY_TIF
        ),
        output_csv=paths[
            "domain_qa"
        ],
    )

    # -------------------------------------------------------------------------
    # 5. Area statistics and pixelwise mean/SD.
    # -------------------------------------------------------------------------
    log(
        "[5/6] Summarizing LR mapped-area variability and "
        "aggregating pixelwise mean/SD rasters..."
    )

    area_summary = summarize_mapped_area_uncertainty(
        area_raw=area_raw
    )

    area_summary.to_csv(
        paths[
            "area_summary"
        ],
        index=False,
        float_format="%.10f",
    )

    aggregate_fold_rasters(
        fold_rasters=ensemble_rasters,
        mean_output=paths[
            "mean_raster"
        ],
        sd_output=paths[
            "sd_raster"
        ],
    )

    sd_summary = summarize_pixelwise_sd(
        sd_raster_path=paths[
            "sd_raster"
        ],
        output_csv=paths[
            "sd_summary"
        ],
    )

    # -------------------------------------------------------------------------
    # 6. Figures and report — same Step 7.1D style.
    # -------------------------------------------------------------------------
    log(
        "[6/6] Creating LR uncertainty figures and final report..."
    )

    create_area_uncertainty_figure(
        area_summary=area_summary,
        output_path=paths[
            "area_figure"
        ],
    )

    create_pixelwise_sd_figure(
        sd_raster_path=paths[
            "sd_raster"
        ],
        output_path=paths[
            "sd_figure"
        ],
    )

    write_uncertainty_report(
        paths=paths,
        performance_intervals=performance_intervals,
        additional_comparisons=additional_comparisons,
        area_summary=area_summary,
        domain_qa=domain_qa,
        sd_summary=sd_summary,
    )

    log(
        ""
    )

    log(
        "="
        * 96
    )

    log(
        "STEP 7.3D COMPLETED SUCCESSFULLY"
    )

    log(
        "="
        * 96
    )

    log(
        f"[OUT] "
        f"{paths['out']}"
    )

    print(
        "\nAdditional paired LR AUC comparisons:"
    )

    print(
        additional_comparisons.to_string(
            index=False
        )
    )

    print(
        "\nLR cross-fold mapped-area uncertainty:"
    )

    print(
        area_summary.to_string(
            index=False
        )
    )

    print(
        "\nLR pixelwise SD summary:"
    )

    print(
        sd_summary.to_string(
            index=False
        )
    )


# =============================================================================
# COMMAND-LINE ENTRY POINT
# =============================================================================
def parse_command_line_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Step 7.3D Logistic Regression performance and mapped-area uncertainty."
        )
    )

    parser.add_argument(
        "--worker-fold",
        type=int,
        default=None,
        help=(
            "Internal process-isolated LR worker mode. "
            "Valid values are 1-5."
        ),
    )

    arguments, unknown = parser.parse_known_args()

    if unknown:
        log(
            "[CLI] Ignoring host-environment arguments: "
            f"{unknown}"
        )

    return arguments


if __name__ == "__main__":
    try:
        args = parse_command_line_arguments()

        if args.worker_fold is None:
            run_controller()

        else:
            if (
                args.worker_fold
                not in Config.EXPECTED_FOLDS
            ):
                raise ValueError(
                    "--worker-fold must be one of "
                    f"{Config.EXPECTED_FOLDS}; "
                    f"received {args.worker_fold}."
                )

            run_outer_fold_map_worker(
                int(
                    args.worker_fold
                )
            )

    except Exception as exc:
        print(
            f"[ERROR] Step 7.3D failed: {exc}",
            flush=True,
        )

        traceback.print_exc()

        raise


# STEP 7.4 — CROSS-ALGORITHM ROBUSTNESS AND SYNTHESIS

In [ ]:
# =============================================================================
# STEP 7.4 — CROSS-ALGORITHM ROBUSTNESS AND SYNTHESIS
# Figure revision: bold X/Y axis labels, clearer figure text,
# and PNG/PDF/SVG export for all figures, with bold axis tick labels.
# =============================================================================
#
# Purpose
# -------
# READ-ONLY synthesis of the completed outputs from:
#   Step 7.1  XGBoost
#   Step 7.2  Random Forest
#   Step 7.3  Logistic Regression
#
# This script does NOT:
#   - fit any ML model;
#   - tune any hyperparameter;
#   - regenerate any pseudo-absence sample;
#   - recompute bootstrap intervals;
#   - regenerate susceptibility maps.
#
# It reads the FINAL saved outputs from Steps 7.1, 7.2 and 7.3 and produces:
#
#   7.4.0  Input-contract QA and resolved-file manifest
#   7.4.1  A-F performance synthesis across XGB, RF and LR
#   7.4.2  Cross-algorithm ablation synthesis, including E-C and E-D
#   7.4.3A Learning-curve synthesis for Models A and E
#   7.4.3B Held-out permutation-importance synthesis for Model E
#   7.4.4  HKFI weighting robustness synthesis
#   7.4.5  Pseudo-absence buffer sensitivity synthesis
#   7.4.6  Geographic block robustness synthesis
#   7.4.7  Cross-algorithm mapped-area uncertainty synthesis
#   7.4.8  Supplementary five-fold spatial ROC synthesis for Model E
#
# Scientific emphasis
# -------------------
# The goal is NOT to identify an algorithmic winner.
# The goal is to determine whether the main scientific conclusions remain
# directionally and statistically consistent across XGB, RF and LR.
#
# Reviewer links
# --------------
# Reviewer #1 Comment #2  : learning curves and held-out permutation importance
# Reviewer #1 Comment #8  : raw deformation versus physics-guided formulation
# Reviewer #1 Comment #14 : consistent A-F evaluation with Logistic Regression
# Reviewer #1 Comment #15 : pseudo-absence buffer sensitivity
# Reviewer #1 Comment #16 : performance and mapped-area uncertainty
# Reviewer #1 Comment #17 : within-corridor geographic robustness
# Reviewer #2 Comment #4  : spatial validation, overfitting and leakage transparency
#
# Important interpretation rules
# ------------------------------
# 1. Model E is the primary full physics-guided formulation.
# 2. A30/E70 is an operational coverage product and is excluded from A-F
#    scientific ablation synthesis.
# 3. HKFI weighting sensitivity tests whether the a priori equal weighting is
#    reasonably robust. It is NOT used to select an AUC-optimal weighting.
# 4. Geographic block analysis is within-corridor robustness, not external
#    regional transferability validation.
# 5. Held-out permutation importance is conditional on the fitted Model E
#    predictor set. The retraining-based A-F ablation remains the stronger test
#    of incremental information when predictors are correlated or redundant.
# 6. Mapped-area uncertainty summarizes variability across five fold-trained
#    A30/E70 operational ensemble maps. Mean, SD, minimum and maximum are
#    descriptive cross-fold variability, not formal 95% confidence intervals.
# 7. Five-fold ROC curves are reconstructed only from genuine saved OOF
#    predictions for Model E. The pooled OOF curve is not a sixth fold and
#    does not constitute external validation.
# 8. E-C tests the incremental contribution of SNDR after HKFI is present.
#    E-D tests the incremental contribution of HKFI after SNDR is present.
#    These comparisons are read from the saved Step 7.1D/7.2D/7.3D outputs;
#    Step 7.4 does not refit models or recompute bootstrap intervals.
# =============================================================================

from __future__ import annotations

import json
import os
import re
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, Sequence

import matplotlib

# Non-interactive backend makes the script stable in Spyder, command line,
# remote sessions and headless environments.
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve


# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    # Override without editing the script by setting environment variable:
    #   LSM_BASE=./data
    # Resolve the working directory safely on Windows.
    # This keeps the original location while removing accidental duplicate
    # separators, surrounding quotes, or copied HTML-space entities.
    _BASE_TEXT = os.environ.get(
        "LSM_BASE",
        r"./data",
    )
    _BASE_TEXT = str(_BASE_TEXT).strip().strip('"').strip("'")
    _BASE_TEXT = _BASE_TEXT.replace("&#x20;", " ")

    if re.match(r"^[A-Za-z]:", _BASE_TEXT):
        _BASE_TEXT = re.sub(
            r"[\\/]+",
            lambda match: "\\",
            _BASE_TEXT,
        )

    BASE = Path(_BASE_TEXT)

    RESULTS_DIR = BASE / "Results"

    OUT_DIR = RESULTS_DIR / "Step-7.4-Cross_Algorithm_Robustness_and_Synthesis"
    QA_DIR = OUT_DIR / "QA"
    PERFORMANCE_DIR = OUT_DIR / "7.4.1_performance"
    ABLATION_DIR = OUT_DIR / "7.4.2_ablation"
    LEARNING_DIR = OUT_DIR / "7.4.3_learning_curves"
    PERMUTATION_DIR = OUT_DIR / "7.4.3B_permutation_importance"
    HKFI_DIR = OUT_DIR / "7.4.4_hkfi_weighting"
    BUFFER_DIR = OUT_DIR / "7.4.5_buffer_sensitivity"
    BLOCK_DIR = OUT_DIR / "7.4.6_geographic_blocks"
    MAP_UNCERTAINTY_DIR = OUT_DIR / "7.4.7_mapped_area_uncertainty"
    ROC_DIR = OUT_DIR / "7.4.8_supplementary_five_fold_roc"
    ROC_FIGURE_DIR = ROC_DIR / "figures"
    ROC_TABLE_DIR = ROC_DIR / "tables"
    ROC_QA_DIR = ROC_DIR / "QA"
    FIGURE_DIR = OUT_DIR / "figures"

    PRIMARY_SCENARIO = "E_physics_guided"
    PRIMARY_MODEL_LABEL = "Model E"
    EXPECTED_TOTAL_MODEL_E_ROWS = 330

    ROC_SCENARIO_COLUMN_ALIASES = [
        "scenario", "model_configuration", "configuration", "model"
    ]
    ROC_FOLD_COLUMN_ALIASES = [
        "cv_fold", "outer_fold", "spatial_block", "fold"
    ]
    ROC_TARGET_COLUMN_ALIASES = [
        "label", "target", "y_true", "class", "response"
    ]
    ROC_PROBABILITY_COLUMN_ALIASES = [
        "oof_probability", "predicted_probability", "probability",
        "y_probability", "y_prob", "pred_prob"
    ]
    ROC_SAMPLE_ID_COLUMN_ALIASES = ["sample_id", "site_id", "id"]

    SCENARIO_ORDER = [
        "A_static_baseline",
        "B_raw_deformation",
        "C_hkfi_only",
        "D_sndr_only",
        "E_physics_guided",
        "F_untransformed_sources",
    ]

    MODEL_LABELS = {
        "A_static_baseline": "A",
        "B_raw_deformation": "B",
        "C_hkfi_only": "C",
        "D_sndr_only": "D",
        "E_physics_guided": "E",
        "F_untransformed_sources": "F",
    }

    MODEL_NAMES = {
        "A_static_baseline": "Model A",
        "B_raw_deformation": "Model B",
        "C_hkfi_only": "Model C",
        "D_sndr_only": "Model D",
        "E_physics_guided": "Model E",
        "F_untransformed_sources": "Model F",
    }

    PRIMARY_METRICS = [
        "roc_auc",
        "pr_auc",
        "balanced_accuracy",
        "f1",
        "brier",
    ]

    METRIC_DISPLAY = {
        "roc_auc": "ROC_AUC",
        "pr_auc": "PR_AUC",
        "balanced_accuracy": "BALANCED_ACCURACY",
        "f1": "F1",
        "brier": "BRIER",
    }

    SUMMARY_METRIC_ALIASES = {
        "roc_auc": ["oof_roc_auc", "roc_auc"],
        "pr_auc": ["oof_pr_auc", "pr_auc"],
        "balanced_accuracy": [
            "oof_balanced_accuracy_0p5",
            "oof_balanced_accuracy",
            "balanced_accuracy",
        ],
        "f1": ["oof_f1_0p5", "oof_f1", "f1"],
        "brier": ["oof_brier", "brier", "brier_score"],
    }

    REQUIRED_ABLATIONS = [
        (
            "B - A",
            "B_raw_deformation",
            "A_static_baseline",
            "Does raw DV+ add predictive information beyond static factors?",
        ),
        (
            "C - A",
            "C_hkfi_only",
            "A_static_baseline",
            "Does HKFI add predictive information beyond static factors?",
        ),
        (
            "D - A",
            "D_sndr_only",
            "A_static_baseline",
            "Does SNDR add predictive information beyond static factors?",
        ),
        (
            "E - A",
            "E_physics_guided",
            "A_static_baseline",
            "Does the complete physics-guided formulation improve over the static baseline?",
        ),
        (
            "E - C",
            "E_physics_guided",
            "C_hkfi_only",
            "Does adding SNDR improve performance beyond HKFI alone?",
        ),
        (
            "E - D",
            "E_physics_guided",
            "D_sndr_only",
            "Does adding HKFI improve performance beyond SNDR alone?",
        ),
        (
            "E - B",
            "E_physics_guided",
            "B_raw_deformation",
            "Does the physics-guided formulation outperform raw deformation alone?",
        ),
        (
            "E - F",
            "E_physics_guided",
            "F_untransformed_sources",
            "Do HKFI and SNDR add information beyond their separate source variables?",
        ),
    ]

    HKFI_REFERENCE = "primary_equal_component"
    HKFI_TARGETED_ORDER = [
        "primary_equal_component",
        "kinematic_emphasis",
        "twi_emphasis",
        "sm_emphasis",
    ]

    HKFI_LABELS = {
        "primary_equal_component": "Equal weighting",
        "kinematic_emphasis": "Kinematic emphasis",
        "twi_emphasis": "TWI emphasis",
        "sm_emphasis": "Soil-moisture emphasis",
    }

    BUFFER_ORDER = [100, 500, 1000]

    EXPECTED_FOLDS = [1, 2, 3, 4, 5]

    # Step 7.1/7.2/7.3 summaries and Step D uncertainty estimates should be
    # numerically identical except for harmless output rounding. This tolerance
    # is used for QA only and does not terminate the synthesis.
    POINT_ESTIMATE_QA_TOLERANCE = 1.0e-6

    FIG_DPI = 900
    FIGSIZE = (6.2, 4.4)
    ROC_FIG_DPI = 900
    ROC_COMBINED_FIGSIZE = (13.2, 4.3)
    ROC_INDIVIDUAL_FIGSIZE = (5.2, 4.5)
    ROC_AUC_QA_TOLERANCE = 1.0e-8

    ALGORITHMS: Dict[str, Dict[str, Any]] = {
        "XGB": {
            "label": "XGBoost",
            "root": RESULTS_DIR / "Step-7.1-XGB-Models_A_to_F",
            "summary": "XGB_A_to_F_summary_metrics.csv",
            "paired": "XGB_A_to_F_paired_AUC_comparisons.csv",
            "additional_paired": "XGB_7_1D_Additional_Paired_AUC_E_C_E_D.csv",
            "additional_paired_dirs": ["Step-7.1D-Performance_and_Map_Uncertainty"],
            "fold_metrics": "XGB_A_to_F_outer_fold_metrics.csv",
            "oof": "XGB_A_to_F_OOF_predictions.csv",
            "learning": "XGB_7_1A_Spatial_Learning_Curve_Summary.csv",
            "learning_dirs": ["Step-7.1A-Learning_and_Permutation"],
            "permutation_individual": "XGB_7_1A_Model_E_Heldout_Permutation_Importance_Summary.csv",
            "permutation_grouped": "XGB_7_1A_Model_E_Grouped_HKFI_SNDR_Permutation_Summary.csv",
            "permutation_dirs": ["Step-7.1A-Learning_and_Permutation"],
            "hkfi_performance": "XGB_7_1B_HKFI_Weight_Sensitivity_Performance.csv",
            "hkfi_paired": "XGB_7_1B_HKFI_Weight_Sensitivity_Paired_AUC.csv",
            "hkfi_dirs": ["Step-7.1B-HKFI_Weight_Sensitivity_STEP53_Direct"],
            "buffer_performance": "XGB_7_1C_A_E_Buffer_Sensitivity_Performance.csv",
            "buffer_paired": "XGB_7_1C_A_E_Buffer_Sensitivity_Paired_AUC.csv",
            "buffer_dirs": ["Step-7.1C-Buffer_Sensitivity"],
            "uncertainty": "XGB_7_1D_Performance_Cluster_Bootstrap_Intervals.csv",
            "uncertainty_dirs": ["Step-7.1D-Performance_and_Map_Uncertainty"],
            "map_area_uncertainty": "XGB_7_1D_A30_E70_CrossFold_Area_Uncertainty_Summary.csv",
            "map_area_uncertainty_dirs": ["Step-7.1D-Performance_and_Map_Uncertainty"],
        },
        "RF": {
            "label": "Random Forest",
            "root": RESULTS_DIR / "Step-7.2-RF-Models_A_to_F",
            "summary": "RF_A_to_F_summary_metrics.csv",
            "paired": "RF_A_to_F_paired_AUC_comparisons.csv",
            "additional_paired": "RF_7_2D_Additional_Paired_AUC_E_C_E_D.csv",
            "additional_paired_dirs": ["Step-7.2D-Performance_and_Map_Uncertainty"],
            "fold_metrics": "RF_A_to_F_outer_fold_metrics.csv",
            "oof": "RF_A_to_F_OOF_predictions.csv",
            "learning": "RF_7_2A_Spatial_Learning_Curve_Summary.csv",
            "learning_dirs": ["Step-7.2A-Learning_and_Permutation"],
            "permutation_individual": "RF_7_2A_Model_E_Heldout_Permutation_Importance_Summary.csv",
            "permutation_grouped": "RF_7_2A_Model_E_Grouped_HKFI_SNDR_Permutation_Summary.csv",
            "permutation_dirs": ["Step-7.2A-Learning_and_Permutation"],
            "hkfi_performance": "RF_7_2B_HKFI_Weight_Sensitivity_Performance.csv",
            "hkfi_paired": "RF_7_2B_HKFI_Weight_Sensitivity_Paired_AUC.csv",
            "hkfi_dirs": ["Step-7.2B-HKFI_Weight_Sensitivity_STEP53_Direct"],
            "buffer_performance": "RF_7_2C_A_E_Buffer_Sensitivity_Performance.csv",
            "buffer_paired": "RF_7_2C_A_E_Buffer_Sensitivity_Paired_AUC.csv",
            "buffer_dirs": ["Step-7.2C-Buffer_Sensitivity"],
            "uncertainty": "RF_7_2D_Performance_Cluster_Bootstrap_Intervals.csv",
            "uncertainty_dirs": ["Step-7.2D-Performance_and_Map_Uncertainty"],
            "map_area_uncertainty": "RF_7_2D_A30_E70_CrossFold_Area_Uncertainty_Summary.csv",
            "map_area_uncertainty_dirs": ["Step-7.2D-Performance_and_Map_Uncertainty"],
        },
        "LR": {
            "label": "Logistic Regression",
            "root": RESULTS_DIR / "Step-7.3-LR-Models_A_to_F",
            "summary": "LR_A_to_F_summary_metrics.csv",
            "paired": "LR_A_to_F_paired_AUC_comparisons.csv",
            "additional_paired": "LR_7_3D_Additional_Paired_AUC_E_C_E_D.csv",
            "additional_paired_dirs": ["Step-7.3D-Performance_and_Map_Uncertainty"],
            "fold_metrics": "LR_A_to_F_outer_fold_metrics.csv",
            "oof": "LR_A_to_F_OOF_predictions.csv",
            "learning": "LR_7_3A_Spatial_Learning_Curve_Summary.csv",
            "learning_dirs": ["Step-7.3A-Learning_and_Permutation"],
            "permutation_individual": "LR_7_3A_Model_E_Heldout_Permutation_Importance_Summary.csv",
            "permutation_grouped": "LR_7_3A_Model_E_Grouped_HKFI_SNDR_Permutation_Summary.csv",
            "permutation_dirs": ["Step-7.3A-Learning_and_Permutation"],
            "hkfi_performance": "LR_7_3B_HKFI_Weight_Sensitivity_Performance.csv",
            "hkfi_paired": "LR_7_3B_HKFI_Weight_Sensitivity_Paired_AUC.csv",
            "hkfi_dirs": ["Step-7.3B-HKFI_Weight_Sensitivity_STEP53_Direct"],
            "buffer_performance": "LR_7_3C_A_E_Buffer_Sensitivity_Performance.csv",
            "buffer_paired": "LR_7_3C_A_E_Buffer_Sensitivity_Paired_AUC.csv",
            "buffer_dirs": ["Step-7.3C-Buffer_Sensitivity"],
            "uncertainty": "LR_7_3D_Performance_Cluster_Bootstrap_Intervals.csv",
            "uncertainty_dirs": ["Step-7.3D-Performance_and_Map_Uncertainty"],
            "map_area_uncertainty": "LR_7_3D_A30_E70_CrossFold_Area_Uncertainty_Summary.csv",
            "map_area_uncertainty_dirs": ["Step-7.3D-Performance_and_Map_Uncertainty"],
        },
    }


# =============================================================================
# FIGURE TYPOGRAPHY
# =============================================================================
AXIS_LABEL_FONTSIZE = 11
TICK_LABEL_FONTSIZE = 9
LEGEND_FONTSIZE = 9
TITLE_FONTSIZE = 11

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.labelsize": AXIS_LABEL_FONTSIZE,
    "axes.labelweight": "bold",
    "axes.titlesize": TITLE_FONTSIZE,
    "axes.titleweight": "bold",
    "xtick.labelsize": TICK_LABEL_FONTSIZE,
    "ytick.labelsize": TICK_LABEL_FONTSIZE,
    "legend.fontsize": LEGEND_FONTSIZE,
    "axes.unicode_minus": False,
})


def apply_clear_figure_style(fig: plt.Figure) -> None:
    """Apply clear, consistent typography to every axis before saving."""
    for ax in fig.axes:
        ax.xaxis.label.set_fontsize(AXIS_LABEL_FONTSIZE)
        ax.xaxis.label.set_fontweight("bold")
        ax.yaxis.label.set_fontsize(AXIS_LABEL_FONTSIZE)
        ax.yaxis.label.set_fontweight("bold")

        ax.title.set_fontsize(TITLE_FONTSIZE)
        ax.title.set_fontweight("bold")

        ax.tick_params(
            axis="both",
            which="major",
            labelsize=TICK_LABEL_FONTSIZE,
            width=0.8,
            length=3.5,
        )

        # Make all x- and y-axis tick labels bold, e.g. A-F and numeric ticks.
        for tick_label in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
            tick_label.set_fontsize(TICK_LABEL_FONTSIZE)
            tick_label.set_fontweight("bold")

        # If matplotlib adds scientific-notation offset text, keep that bold too.
        ax.xaxis.get_offset_text().set_fontsize(TICK_LABEL_FONTSIZE)
        ax.xaxis.get_offset_text().set_fontweight("bold")
        ax.yaxis.get_offset_text().set_fontsize(TICK_LABEL_FONTSIZE)
        ax.yaxis.get_offset_text().set_fontweight("bold")

        legend = ax.get_legend()
        if legend is not None:
            for label in legend.get_texts():
                label.set_fontsize(LEGEND_FONTSIZE)
            legend_title = legend.get_title()
            if legend_title is not None:
                legend_title.set_fontsize(LEGEND_FONTSIZE)


def normalize_output_path(path: Path) -> Path:
    """
    Return a Windows-safe output path.

    The previous revised script failed at fig.savefig because the path reaching
    PIL contained an invalid Windows path representation. This function keeps
    the original directory structure but normalizes duplicate separators and
    guarantees that the parent directory exists.
    """
    path_text = str(path).strip().strip('"').strip("'")
    path_text = path_text.replace("&#x20;", " ")

    if os.name == "nt" and re.match(r"^[A-Za-z]:", path_text):
        path_text = re.sub(
            r"[\\/]+",
            lambda match: "\\",
            path_text,
        )
        path_text = os.path.normpath(path_text)

    normalized = Path(path_text)
    normalized.parent.mkdir(parents=True, exist_ok=True)
    return normalized


def figure_output_paths(path: Path) -> tuple[Path, Path, Path]:
    """Create PNG, PDF and SVG paths from one original figure filename."""
    safe_path = normalize_output_path(path)
    stem = safe_path.with_suffix("")
    return (
        Path(str(stem) + ".png"),
        Path(str(stem) + ".pdf"),
        Path(str(stem) + ".svg"),
    )


# =============================================================================
# GENERAL HELPERS
# =============================================================================
def log(message: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {message}", flush=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def create_output_directories() -> None:
    for directory in [
        Config.OUT_DIR,
        Config.QA_DIR,
        Config.PERFORMANCE_DIR,
        Config.ABLATION_DIR,
        Config.LEARNING_DIR,
        Config.PERMUTATION_DIR,
        Config.HKFI_DIR,
        Config.BUFFER_DIR,
        Config.BLOCK_DIR,
        Config.MAP_UNCERTAINTY_DIR,
        Config.ROC_DIR,
        Config.ROC_FIGURE_DIR,
        Config.ROC_TABLE_DIR,
        Config.ROC_QA_DIR,
        Config.FIGURE_DIR,
    ]:
        directory.mkdir(parents=True, exist_ok=True)


def require_columns(
    dataframe: pd.DataFrame,
    columns: Sequence[str],
    description: str,
) -> None:
    missing = [column for column in columns if column not in dataframe.columns]
    if missing:
        raise RuntimeError(
            f"{description} is missing required columns: {missing}\n"
            f"Observed columns: {list(dataframe.columns)}"
        )


def first_existing_column(
    dataframe: pd.DataFrame,
    candidates: Sequence[str],
    description: str,
) -> str:
    for column in candidates:
        if column in dataframe.columns:
            return column
    raise RuntimeError(
        f"{description}: none of these columns were found: {list(candidates)}\n"
        f"Observed columns: {list(dataframe.columns)}"
    )


def resolve_file(
    root: Path,
    filename: str,
    preferred_dirs: Sequence[str] | None = None,
) -> Path:
    """Resolve one final output deterministically without silently choosing stale files."""
    preferred_dirs = list(preferred_dirs or [])

    direct = root / filename
    if direct.exists():
        return direct

    for directory in preferred_dirs:
        candidate = root / directory / filename
        if candidate.exists():
            return candidate

    matches = sorted(root.rglob(filename)) if root.exists() else []

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        # Prefer an exact expected parent directory when possible.
        for directory in preferred_dirs:
            preferred = [path for path in matches if directory in path.parts]
            if len(preferred) == 1:
                return preferred[0]

        raise RuntimeError(
            f"Ambiguous input file '{filename}'. Multiple copies were found under:\n"
            f"{root}\n" + "\n".join(str(path) for path in matches)
        )

    expected_locations = [str(root / filename)] + [
        str(root / directory / filename) for directory in preferred_dirs
    ]
    raise FileNotFoundError(
        f"Required Step 7.4 input was not found: {filename}\n"
        "Expected one of:\n" + "\n".join(expected_locations)
    )


def resolve_algorithm_paths(algorithm: str) -> Dict[str, Path]:
    spec = Config.ALGORITHMS[algorithm]
    root = Path(spec["root"])

    if not root.exists():
        raise FileNotFoundError(
            f"{algorithm} root output directory does not exist:\n{root}"
        )

    return {
        "root": root,
        "summary": resolve_file(root, spec["summary"]),
        "paired": resolve_file(root, spec["paired"]),
        "additional_paired": resolve_file(
            root,
            spec["additional_paired"],
            spec["additional_paired_dirs"],
        ),
        "fold_metrics": resolve_file(root, spec["fold_metrics"]),
        "oof": resolve_file(root, spec["oof"]),
        "learning": resolve_file(root, spec["learning"], spec["learning_dirs"]),
        "permutation_individual": resolve_file(
            root, spec["permutation_individual"], spec["permutation_dirs"]
        ),
        "permutation_grouped": resolve_file(
            root, spec["permutation_grouped"], spec["permutation_dirs"]
        ),
        "hkfi_performance": resolve_file(
            root, spec["hkfi_performance"], spec["hkfi_dirs"]
        ),
        "hkfi_paired": resolve_file(root, spec["hkfi_paired"], spec["hkfi_dirs"]),
        "buffer_performance": resolve_file(
            root, spec["buffer_performance"], spec["buffer_dirs"]
        ),
        "buffer_paired": resolve_file(
            root, spec["buffer_paired"], spec["buffer_dirs"]
        ),
        "uncertainty": resolve_file(
            root, spec["uncertainty"], spec["uncertainty_dirs"]
        ),
        "map_area_uncertainty": resolve_file(
            root,
            spec["map_area_uncertainty"],
            spec["map_area_uncertainty_dirs"],
        ),
    }


def canonical_scenario(value: Any) -> str:
    """
    Canonicalize scientific A-F model labels while safely recognizing the
    operational A30/E70 ensemble that may coexist in Step D output tables.

    Step 7.4 scientific A-F synthesis later filters to Config.SCENARIO_ORDER,
    so returning the ensemble label here prevents unnecessary failure without
    allowing it into the A-F ablation analysis.
    """
    text = str(value).strip()

    if text in Config.SCENARIO_ORDER:
        return text

    if text == "A30_E70_ensemble":
        return "A30_E70_ensemble"

    compact = (
        text.lower()
        .replace("model", "")
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace("/", "")
    )

    aliases = {
        "a": "A_static_baseline",
        "astaticbaseline": "A_static_baseline",
        "b": "B_raw_deformation",
        "brawdeformation": "B_raw_deformation",
        "c": "C_hkfi_only",
        "chkfionly": "C_hkfi_only",
        "d": "D_sndr_only",
        "dsndronly": "D_sndr_only",
        "e": "E_physics_guided",
        "ephysicsguided": "E_physics_guided",
        "f": "F_untransformed_sources",
        "funtransformedsources": "F_untransformed_sources",

        # Operational coverage product. Recognized only so mixed Step D tables
        # can be read safely; it remains excluded from the scientific A-F
        # synthesis because it is not present in Config.SCENARIO_ORDER.
        "a30e70ensemble": "A30_E70_ensemble",
        "a30e70": "A30_E70_ensemble",
    }

    if compact in aliases:
        return aliases[compact]

    raise RuntimeError(f"Could not canonicalize model/scenario value: {value}")


def canonical_metric(value: Any) -> str:
    text = (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    aliases = {
        "auc": "roc_auc",
        "roc_auc": "roc_auc",
        "pr_auc": "pr_auc",
        "average_precision": "pr_auc",
        "balanced_accuracy": "balanced_accuracy",
        "balanced_acc": "balanced_accuracy",
        "f1": "f1",
        "f1_score": "f1",
        "brier": "brier",
        "brier_score": "brier",
    }
    return aliases.get(text, text)


def normalize_comparison_label(value: Any) -> str:
    text = str(value).strip().upper().replace("−", "-").replace("–", "-")
    text = " ".join(text.split())
    for label, _, _, _ in Config.REQUIRED_ABLATIONS:
        if text.replace(" ", "") == label.upper().replace(" ", ""):
            return label
    return str(value).strip()


def comparison_from_scenarios(left: str, right: str) -> str | None:
    left = canonical_scenario(left)
    right = canonical_scenario(right)
    for label, expected_left, expected_right, _ in Config.REQUIRED_ABLATIONS:
        if left == expected_left and right == expected_right:
            return label
    return None


def interval_interpretation(estimate: float, low: float, high: float) -> str:
    if np.isfinite(low) and low > 0:
        return "Positive; 95% interval excludes zero"
    if np.isfinite(high) and high < 0:
        return "Negative; 95% interval excludes zero"
    if estimate > 0:
        return "Directionally positive; 95% interval includes zero"
    if estimate < 0:
        return "Directionally negative; 95% interval includes zero"
    return "No difference"


def save_figure(fig: plt.Figure, path: Path) -> None:
    """
    Save one Step 7.4 figure in PNG, PDF and SVG format.

    The scientific figure itself is unchanged; only typography and export
    handling are standardized.
    """
    apply_clear_figure_style(fig)
    fig.tight_layout()

    png_path, pdf_path, svg_path = figure_output_paths(path)

    fig.savefig(
        str(png_path),
        dpi=Config.FIG_DPI,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        str(pdf_path),
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        str(svg_path),
        format="svg",
        bbox_inches="tight",
        facecolor="white",
    )

    log(f"[FIGURE PNG] {png_path}")
    log(f"[FIGURE PDF] {pdf_path}")
    log(f"[FIGURE SVG] {svg_path}")

    plt.close(fig)


def format_ci(estimate: float, low: float, high: float) -> str:
    return f"{estimate:.3f} [{low:.3f}, {high:.3f}]"


# =============================================================================
# 7.4.0 — INPUT-CONTRACT QA
# =============================================================================
def build_input_manifest(paths_by_algorithm: Dict[str, Dict[str, Path]]) -> pd.DataFrame:
    rows = []
    for algorithm, paths in paths_by_algorithm.items():
        for key, path in paths.items():
            if key == "root":
                continue
            rows.append(
                {
                    "algorithm": algorithm,
                    "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                    "input_type": key,
                    "path": str(path),
                    "exists": bool(path.exists()),
                    "size_bytes": int(path.stat().st_size) if path.exists() else np.nan,
                }
            )
    return pd.DataFrame(rows)


# =============================================================================
# 7.4.1 — A-F PERFORMANCE SYNTHESIS
# =============================================================================
def load_primary_summary(algorithm: str, path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    scenario_col = first_existing_column(df, ["scenario", "model"], f"{algorithm} summary")
    df = df.copy()
    df["scenario"] = df[scenario_col].map(canonical_scenario)

    rows = []
    for scenario in Config.SCENARIO_ORDER:
        subset = df.loc[df["scenario"] == scenario]
        if len(subset) != 1:
            raise RuntimeError(
                f"{algorithm} summary: expected exactly one row for {scenario}; found {len(subset)}."
            )
        source = subset.iloc[0]
        row = {
            "algorithm": algorithm,
            "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
            "scenario": scenario,
            "configuration": Config.MODEL_LABELS[scenario],
            "model": Config.MODEL_NAMES[scenario],
        }
        for metric, aliases in Config.SUMMARY_METRIC_ALIASES.items():
            column = first_existing_column(df, aliases, f"{algorithm} summary metric {metric}")
            row[metric] = float(source[column])
        rows.append(row)

    return pd.DataFrame(rows)


def load_uncertainty(algorithm: str, path: Path) -> pd.DataFrame:
    """
    Read the Step D uncertainty table and retain only the scientific A-F models.

    Important:
    Step D may also contain the operational A30_E70_ensemble product. Filter
    product_type first when available, then canonicalize scenario names. This
    prevents the ensemble from causing an A-F canonicalization failure.
    """
    df = pd.read_csv(path)
    require_columns(
        df,
        ["scenario", "metric", "estimate", "ci_low", "ci_high"],
        f"{algorithm} Step D uncertainty table",
    )

    df = df.copy()

    # Prefer filtering by explicit product type before scenario normalization.
    if "product_type" in df.columns:
        product_type = df["product_type"].astype(str).str.strip().str.lower()
        af_mask = product_type.eq("a-f model")
        if af_mask.any():
            df = df.loc[af_mask].copy()

    # Canonicalization is now safe even when an older/mixed file still contains
    # the operational A30/E70 ensemble, because canonical_scenario recognizes it.
    df["scenario"] = df["scenario"].map(canonical_scenario)
    df["metric"] = df["metric"].map(canonical_metric)

    # Final hard filter: only A-F models and requested performance metrics enter
    # Step 7.4.1.
    df = df.loc[
        df["scenario"].isin(Config.SCENARIO_ORDER)
        & df["metric"].isin(Config.PRIMARY_METRICS)
    ].copy()

    df["algorithm"] = algorithm
    df["algorithm_label"] = Config.ALGORITHMS[algorithm]["label"]
    return df


def build_performance_synthesis(
    paths_by_algorithm: Dict[str, Dict[str, Path]],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    long_rows = []
    qa_rows = []

    for algorithm in Config.ALGORITHMS:
        summary = load_primary_summary(algorithm, paths_by_algorithm[algorithm]["summary"])
        uncertainty = load_uncertainty(algorithm, paths_by_algorithm[algorithm]["uncertainty"])

        for scenario in Config.SCENARIO_ORDER:
            summary_row = summary.loc[summary["scenario"] == scenario].iloc[0]

            for metric in Config.PRIMARY_METRICS:
                interval = uncertainty.loc[
                    (uncertainty["scenario"] == scenario)
                    & (uncertainty["metric"] == metric)
                ]

                if len(interval) != 1:
                    raise RuntimeError(
                        f"{algorithm} uncertainty: expected exactly one row for "
                        f"{scenario} / {metric}; found {len(interval)}."
                    )

                interval_row = interval.iloc[0]
                summary_estimate = float(summary_row[metric])
                step_d_estimate = float(interval_row["estimate"])
                difference = abs(summary_estimate - step_d_estimate)

                qa_rows.append(
                    {
                        "algorithm": algorithm,
                        "scenario": scenario,
                        "metric": metric,
                        "primary_summary_estimate": summary_estimate,
                        "step_D_bootstrap_table_estimate": step_d_estimate,
                        "absolute_difference": difference,
                        "tolerance": Config.POINT_ESTIMATE_QA_TOLERANCE,
                        "pass": difference <= Config.POINT_ESTIMATE_QA_TOLERANCE,
                    }
                )

                # Step D is authoritative here because it contains the requested
                # point estimate and its matching bootstrap interval.
                long_rows.append(
                    {
                        "algorithm": algorithm,
                        "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                        "scenario": scenario,
                        "configuration": Config.MODEL_LABELS[scenario],
                        "model": Config.MODEL_NAMES[scenario],
                        "metric": metric,
                        "estimate": step_d_estimate,
                        "ci_low": float(interval_row["ci_low"]),
                        "ci_high": float(interval_row["ci_high"]),
                    }
                )

    performance_long = pd.DataFrame(long_rows)
    qa = pd.DataFrame(qa_rows)

    # Numeric master table: one row per A-F configuration.
    master_rows = []
    display_rows = []

    for scenario in Config.SCENARIO_ORDER:
        numeric_row: Dict[str, Any] = {
            "Configuration": Config.MODEL_LABELS[scenario],
            "Scenario": scenario,
            "Model": Config.MODEL_NAMES[scenario],
        }
        display_row: Dict[str, Any] = {
            "Configuration": Config.MODEL_LABELS[scenario],
            "Model": Config.MODEL_NAMES[scenario],
        }

        for algorithm in Config.ALGORITHMS:
            for metric in Config.PRIMARY_METRICS:
                subset = performance_long.loc[
                    (performance_long["algorithm"] == algorithm)
                    & (performance_long["scenario"] == scenario)
                    & (performance_long["metric"] == metric)
                ]
                record = subset.iloc[0]
                prefix = f"{algorithm}_{Config.METRIC_DISPLAY[metric]}"
                numeric_row[f"{prefix}_ESTIMATE"] = float(record["estimate"])
                numeric_row[f"{prefix}_CI_LOW"] = float(record["ci_low"])
                numeric_row[f"{prefix}_CI_HIGH"] = float(record["ci_high"])
                display_row[f"{algorithm} {Config.METRIC_DISPLAY[metric]}"] = format_ci(
                    float(record["estimate"]),
                    float(record["ci_low"]),
                    float(record["ci_high"]),
                )

        master_rows.append(numeric_row)
        display_rows.append(display_row)

    performance_master = pd.DataFrame(master_rows)
    performance_display = pd.DataFrame(display_rows)

    return performance_long, performance_master, performance_display, qa


def create_performance_figure(performance_long: pd.DataFrame, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=Config.FIGSIZE)
    x = np.arange(len(Config.SCENARIO_ORDER), dtype=float)
    offsets = np.linspace(-0.18, 0.18, len(Config.ALGORITHMS))

    for offset, algorithm in zip(offsets, Config.ALGORITHMS):
        subset = (
            performance_long.loc[
                (performance_long["algorithm"] == algorithm)
                & (performance_long["metric"] == "roc_auc")
            ]
            .set_index("scenario")
            .loc[Config.SCENARIO_ORDER]
        )
        y = subset["estimate"].to_numpy(float)
        low = subset["ci_low"].to_numpy(float)
        high = subset["ci_high"].to_numpy(float)
        ax.errorbar(
            x + offset,
            y,
            yerr=np.vstack([y - low, high - y]),
            marker="o",
            linestyle="none",
            capsize=3,
            label=Config.ALGORITHMS[algorithm]["label"],
        )

    ax.set_xticks(x)
    ax.set_xticklabels([Config.MODEL_LABELS[s] for s in Config.SCENARIO_ORDER])
    ax.set_xlabel("Model configuration", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("Nested OOF ROC-AUC", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    save_figure(fig, output_path)


# =============================================================================
# 7.4.2 — CROSS-ALGORITHM ABLATION SYNTHESIS
# =============================================================================
def parse_paired_comparison_file(
    algorithm: str,
    path: Path,
) -> pd.DataFrame:
    """
    Parse one saved paired ROC-AUC comparison table.

    The principal A-F table and the additional E-C/E-D table are parsed
    independently and combined later. This function therefore does not require
    an individual file to contain every comparison.
    """
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    require_columns(
        df,
        ["auc_difference_left_minus_right"],
        f"{algorithm} paired AUC table: {path.name}",
    )

    ci_low_col = first_existing_column(
        df,
        ["cluster_bootstrap_ci_low", "ci_low"],
        f"{algorithm} paired AUC lower CI: {path.name}",
    )
    ci_high_col = first_existing_column(
        df,
        ["cluster_bootstrap_ci_high", "ci_high"],
        f"{algorithm} paired AUC upper CI: {path.name}",
    )

    required_labels = {item[0] for item in Config.REQUIRED_ABLATIONS}
    rows = []

    for _, source in df.iterrows():
        label: str | None = None

        # Preferred identification: explicit left and right scenarios.
        if (
            "left_scenario" in df.columns
            and "right_scenario" in df.columns
        ):
            try:
                label = comparison_from_scenarios(
                    source["left_scenario"],
                    source["right_scenario"],
                )
            except Exception:
                label = None

        # Fallback: standardized comparison label.
        if label is None and "comparison" in df.columns:
            candidate = normalize_comparison_label(source["comparison"])
            if candidate in required_labels:
                label = candidate

        # Ignore comparisons outside the declared Step 7.4 synthesis.
        if label is None:
            continue

        scientific_question = next(
            item[3]
            for item in Config.REQUIRED_ABLATIONS
            if item[0] == label
        )

        estimate = float(source["auc_difference_left_minus_right"])
        low = float(source[ci_low_col])
        high = float(source[ci_high_col])

        if not all(np.isfinite(value) for value in [estimate, low, high]):
            raise RuntimeError(
                f"{algorithm} / {label} contains non-finite paired AUC values "
                f"in {path}."
            )

        if low > high:
            raise RuntimeError(
                f"{algorithm} / {label} has ci_low > ci_high in {path}: "
                f"{low} > {high}."
            )

        rows.append(
            {
                "algorithm": algorithm,
                "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                "comparison": label,
                "scientific_question": scientific_question,
                "delta_auc": estimate,
                "ci_low": low,
                "ci_high": high,
                "interpretation": interval_interpretation(
                    estimate,
                    low,
                    high,
                ),
                "source_file": str(path),
            }
        )

    return pd.DataFrame(rows)


def load_all_paired_comparisons(
    algorithm: str,
    primary_path: Path,
    additional_path: Path,
) -> pd.DataFrame:
    """
    Combine the principal A-F paired comparisons with the separately saved
    E-C and E-D paired comparisons.

    No model fitting or bootstrap recomputation occurs here. If a future
    principal file already contains E-C or E-D, duplicate records are accepted
    only when their estimates and intervals agree within the QA tolerance.
    """
    frames = [
        parse_paired_comparison_file(
            algorithm=algorithm,
            path=primary_path,
        ),
        parse_paired_comparison_file(
            algorithm=algorithm,
            path=additional_path,
        ),
    ]

    nonempty_frames = [frame for frame in frames if not frame.empty]
    if not nonempty_frames:
        raise RuntimeError(
            f"{algorithm}: no recognized paired AUC comparisons were found."
        )

    result = pd.concat(
        nonempty_frames,
        ignore_index=True,
        sort=False,
    )

    duplicate_labels = result.loc[
        result.duplicated(
            ["algorithm", "comparison"],
            keep=False,
        ),
        "comparison",
    ].unique()

    for label in duplicate_labels:
        duplicate_rows = result.loc[
            result["comparison"] == label
        ].copy()

        numeric_range = (
            duplicate_rows[["delta_auc", "ci_low", "ci_high"]].max()
            - duplicate_rows[["delta_auc", "ci_low", "ci_high"]].min()
        )

        if float(numeric_range.max()) > Config.POINT_ESTIMATE_QA_TOLERANCE:
            raise RuntimeError(
                f"{algorithm}: conflicting duplicate paired comparison results "
                f"were found for {label}:\n"
                + duplicate_rows.to_string(index=False)
            )

    # Files are ordered principal first, additional second. Numerically
    # identical duplicate results therefore retain the dedicated additional
    # record and its provenance.
    result = result.drop_duplicates(
        ["algorithm", "comparison"],
        keep="last",
    ).reset_index(drop=True)

    required_labels = [item[0] for item in Config.REQUIRED_ABLATIONS]
    observed_labels = set(result["comparison"])
    missing = [
        label
        for label in required_labels
        if label not in observed_labels
    ]

    if missing:
        raise RuntimeError(
            f"{algorithm} paired AUC inputs are missing required Step 7.4 "
            f"comparisons: {missing}\nObserved comparisons: "
            f"{sorted(observed_labels)}"
        )

    comparison_order = {
        label: index
        for index, label in enumerate(required_labels)
    }
    result["_comparison_order"] = result["comparison"].map(
        comparison_order
    )
    result = (
        result.sort_values("_comparison_order")
        .drop(columns="_comparison_order")
        .reset_index(drop=True)
    )

    return result


def build_ablation_synthesis(
    paths_by_algorithm: Dict[str, Dict[str, Path]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames = [
        load_all_paired_comparisons(
            algorithm=algorithm,
            primary_path=paths_by_algorithm[algorithm]["paired"],
            additional_path=paths_by_algorithm[algorithm]["additional_paired"],
        )
        for algorithm in Config.ALGORITHMS
    ]

    long_df = pd.concat(frames, ignore_index=True)

    expected_rows = (
        len(Config.ALGORITHMS)
        * len(Config.REQUIRED_ABLATIONS)
    )
    if len(long_df) != expected_rows:
        raise RuntimeError(
            "Incomplete cross-algorithm ablation synthesis. "
            f"Expected {expected_rows} rows "
            f"({len(Config.ALGORITHMS)} algorithms x "
            f"{len(Config.REQUIRED_ABLATIONS)} comparisons), "
            f"but found {len(long_df)}."
        )

    duplicate_mask = long_df.duplicated(
        ["algorithm", "comparison"],
        keep=False,
    )
    if duplicate_mask.any():
        raise RuntimeError(
            "Duplicate algorithm-comparison rows remain after consolidation:\n"
            + long_df.loc[duplicate_mask].to_string(index=False)
        )

    rows = []
    for label, _, _, question in Config.REQUIRED_ABLATIONS:
        row: Dict[str, Any] = {
            "Comparison": label,
            "Scientific question": question,
        }

        for algorithm in Config.ALGORITHMS:
            subset = long_df.loc[
                (long_df["algorithm"] == algorithm)
                & (long_df["comparison"] == label)
            ]

            if len(subset) != 1:
                raise RuntimeError(
                    f"Expected one paired result for {algorithm} / {label}; "
                    f"found {len(subset)}."
                )

            record = subset.iloc[0]
            row[f"{algorithm}_DELTA_AUC"] = float(record["delta_auc"])
            row[f"{algorithm}_CI_LOW"] = float(record["ci_low"])
            row[f"{algorithm}_CI_HIGH"] = float(record["ci_high"])
            row[f"{algorithm}_INTERPRETATION"] = str(
                record["interpretation"]
            )

        rows.append(row)

    return long_df, pd.DataFrame(rows)


def build_incremental_index_summary(
    ablation_long: pd.DataFrame,
) -> pd.DataFrame:
    """
    Focused reviewer-facing synthesis:
      E-C = SNDR added after HKFI is already present.
      E-D = HKFI added after SNDR is already present.
    """
    target_comparisons = ["E - C", "E - D"]
    subset = ablation_long.loc[
        ablation_long["comparison"].isin(target_comparisons)
    ].copy()

    expected_rows = len(target_comparisons) * len(Config.ALGORITHMS)
    if len(subset) != expected_rows:
        raise RuntimeError(
            "Incomplete E-C/E-D incremental-index summary. "
            f"Expected {expected_rows} rows, found {len(subset)}."
        )

    subset["incremental_component"] = subset["comparison"].map(
        {
            "E - C": "SNDR added beyond HKFI",
            "E - D": "HKFI added beyond SNDR",
        }
    )
    subset["interval_excludes_zero"] = (
        (subset["ci_low"] > 0.0)
        | (subset["ci_high"] < 0.0)
    )
    subset["supported_positive_improvement"] = subset["ci_low"] > 0.0

    comparison_order = {"E - C": 0, "E - D": 1}
    algorithm_order = {
        algorithm: index
        for index, algorithm in enumerate(Config.ALGORITHMS)
    }
    subset["_comparison_order"] = subset["comparison"].map(
        comparison_order
    )
    subset["_algorithm_order"] = subset["algorithm"].map(
        algorithm_order
    )
    subset = subset.sort_values(
        ["_comparison_order", "_algorithm_order"]
    )

    return subset[
        [
            "algorithm",
            "algorithm_label",
            "comparison",
            "incremental_component",
            "delta_auc",
            "ci_low",
            "ci_high",
            "interval_excludes_zero",
            "supported_positive_improvement",
            "interpretation",
            "source_file",
        ]
    ].reset_index(drop=True)


def create_ablation_figure(
    ablation_long: pd.DataFrame,
    output_path: Path,
) -> None:
    comparisons = [item[0] for item in Config.REQUIRED_ABLATIONS]
    x = np.arange(len(comparisons), dtype=float)
    offsets = np.linspace(-0.20, 0.20, len(Config.ALGORITHMS))

    fig, ax = plt.subplots(figsize=(9.6, 4.9))

    for offset, algorithm in zip(offsets, Config.ALGORITHMS):
        subset = (
            ablation_long.loc[
                ablation_long["algorithm"] == algorithm
            ]
            .set_index("comparison")
            .loc[comparisons]
        )

        y = subset["delta_auc"].to_numpy(float)
        low = subset["ci_low"].to_numpy(float)
        high = subset["ci_high"].to_numpy(float)

        ax.errorbar(
            x + offset,
            y,
            yerr=np.vstack([y - low, high - y]),
            marker="o",
            linestyle="none",
            capsize=3,
            label=Config.ALGORITHMS[algorithm]["label"],
        )

    ax.axhline(0.0, linestyle="--", linewidth=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(comparisons, rotation=30, ha="right")
    ax.set_xlabel("Ablation comparison", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("Paired Delta ROC-AUC", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    ax.margins(x=0.03)

    save_figure(fig, output_path)

# =============================================================================
# 7.4.3 — LEARNING-CURVE SYNTHESIS
# =============================================================================
def load_learning_summary(algorithm: str, path: Path) -> pd.DataFrame:
    """Normalize Step 7.xA learning-curve summaries to one stable schema."""
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    scenario_col = first_existing_column(
        df,
        ["scenario", "model_configuration", "configuration", "model"],
        f"{algorithm} learning-curve scenario",
    )
    fraction_col = first_existing_column(
        df,
        ["training_fraction", "train_fraction", "fraction"],
        f"{algorithm} learning-curve training fraction",
    )
    n_train_col = first_existing_column(
        df,
        ["n_train_mean", "mean_n_train", "n_train"],
        f"{algorithm} learning-curve training sample size",
    )
    train_auc_col = first_existing_column(
        df,
        ["training_auc_mean", "train_auc_mean", "training_auc"],
        f"{algorithm} learning-curve training AUC",
    )
    heldout_auc_col = first_existing_column(
        df,
        [
            "heldout_auc_mean",
            "heldout_spatial_auc_mean",
            "validation_auc_mean",
            "test_auc_mean",
        ],
        f"{algorithm} learning-curve held-out AUC",
    )

    sd_candidates = [
        "heldout_auc_sd",
        "heldout_spatial_auc_sd",
        "validation_auc_sd",
        "test_auc_sd",
    ]
    heldout_sd_col = next(
        (column for column in sd_candidates if column in df.columns),
        None,
    )

    normalized = df.copy()
    normalized["scenario"] = normalized[scenario_col].map(canonical_scenario)
    normalized["training_fraction"] = pd.to_numeric(
        normalized[fraction_col], errors="raise"
    )
    normalized["n_train_mean"] = pd.to_numeric(
        normalized[n_train_col], errors="raise"
    )
    normalized["training_auc_mean"] = pd.to_numeric(
        normalized[train_auc_col], errors="raise"
    )
    normalized["heldout_auc_mean"] = pd.to_numeric(
        normalized[heldout_auc_col], errors="raise"
    )
    normalized["heldout_auc_sd"] = (
        pd.to_numeric(normalized[heldout_sd_col], errors="coerce")
        if heldout_sd_col is not None
        else 0.0
    )

    normalized = normalized.loc[
        normalized["scenario"].isin(
            ["A_static_baseline", "E_physics_guided"]
        )
    ].copy()

    if normalized.empty:
        raise RuntimeError(
            f"{algorithm} learning-curve summary contains no Model A/E rows."
        )

    normalized["algorithm"] = algorithm
    normalized["algorithm_label"] = Config.ALGORITHMS[algorithm]["label"]
    normalized["configuration"] = normalized["scenario"].map(Config.MODEL_LABELS)
    normalized["generalization_gap_auc"] = (
        normalized["training_auc_mean"] - normalized["heldout_auc_mean"]
    )

    duplicate_mask = normalized.duplicated(
        ["scenario", "training_fraction"], keep=False
    )
    if duplicate_mask.any():
        duplicates = normalized.loc[
            duplicate_mask,
            ["scenario", "training_fraction"],
        ].drop_duplicates()
        raise RuntimeError(
            f"{algorithm} learning-curve summary has duplicate scenario/fraction rows:\n"
            + duplicates.to_string(index=False)
        )

    return normalized


def build_learning_synthesis(
    paths_by_algorithm: Dict[str, Dict[str, Path]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames = [
        load_learning_summary(algorithm, paths_by_algorithm[algorithm]["learning"])
        for algorithm in Config.ALGORITHMS
    ]
    combined = pd.concat(frames, ignore_index=True)

    endpoint_rows = []
    for algorithm in Config.ALGORITHMS:
        for scenario in ["A_static_baseline", "E_physics_guided"]:
            subset = combined.loc[
                (combined["algorithm"] == algorithm)
                & (combined["scenario"] == scenario)
            ].copy()
            if subset.empty:
                raise RuntimeError(f"Missing learning-curve rows for {algorithm} / {scenario}.")
            max_fraction = float(subset["training_fraction"].max())
            endpoint = subset.loc[
                np.isclose(subset["training_fraction"].astype(float), max_fraction)
            ]
            if len(endpoint) != 1:
                raise RuntimeError(
                    f"Expected one maximum-training learning-curve row for {algorithm} / {scenario}."
                )
            source = endpoint.iloc[0]
            endpoint_rows.append(
                {
                    "algorithm": algorithm,
                    "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                    "configuration": Config.MODEL_LABELS[scenario],
                    "scenario": scenario,
                    "maximum_training_fraction": max_fraction,
                    "n_train_mean": float(source["n_train_mean"]),
                    "training_auc_mean": float(source["training_auc_mean"]),
                    "heldout_auc_mean": float(source["heldout_auc_mean"]),
                    "heldout_auc_sd": float(source["heldout_auc_sd"]),
                    "generalization_gap_auc": float(source["generalization_gap_auc"]),
                }
            )

    return combined, pd.DataFrame(endpoint_rows)


def create_learning_figure(
    learning_df: pd.DataFrame,
    scenario: str,
    output_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=Config.FIGSIZE)
    for algorithm in Config.ALGORITHMS:
        subset = learning_df.loc[
            (learning_df["algorithm"] == algorithm)
            & (learning_df["scenario"] == scenario)
        ].sort_values("n_train_mean")

        ax.errorbar(
            subset["n_train_mean"].to_numpy(float),
            subset["heldout_auc_mean"].to_numpy(float),
            yerr=subset["heldout_auc_sd"].to_numpy(float),
            marker="o",
            capsize=3,
            linewidth=1.3,
            label=Config.ALGORITHMS[algorithm]["label"],
        )

    ax.set_xlabel("Mean outer-training sample size", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("Mean held-out spatial ROC-AUC", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_title(f"Model {Config.MODEL_LABELS[scenario]} learning-curve synthesis", fontsize=TITLE_FONTSIZE, fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    save_figure(fig, output_path)



# =============================================================================
# 7.4.3B — HELD-OUT PERMUTATION-IMPORTANCE SYNTHESIS
# =============================================================================
def canonical_permutation_label(value: Any) -> str:
    """Standardize only the two physics-guided predictors; preserve other names."""
    text = str(value).strip()
    compact = re.sub(r"[^a-z0-9]+", "", text.lower())

    if compact == "hkfi" or compact.startswith("hkfi"):
        return "HKFI"
    if compact == "sndr" or compact.startswith("sndr"):
        return "SNDR"
    return text


def canonical_permutation_group(value: Any) -> str:
    """Standardize the joint HKFI + SNDR group across Step 7.1A/7.2A/7.3A."""
    text = str(value).strip()
    compact = re.sub(r"[^a-z0-9]+", "", text.lower())
    if "hkfi" in compact and "sndr" in compact:
        return "HKFI + SNDR"
    return text


def _load_permutation_summary(
    algorithm: str,
    path: Path,
    label_candidates: Sequence[str],
    label_name: str,
    canonicalizer,
) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    label_col = first_existing_column(
        df,
        label_candidates,
        f"{algorithm} permutation {label_name}",
    )
    mean_col = first_existing_column(
        df,
        ["mean_auc_decrease", "auc_decrease_mean", "mean_decrease_auc"],
        f"{algorithm} permutation mean AUC decrease",
    )
    sd_col = first_existing_column(
        df,
        ["sd_across_outer_folds", "auc_decrease_sd", "sd_auc_decrease"],
        f"{algorithm} permutation AUC-decrease SD",
    )
    min_col = first_existing_column(
        df,
        ["min_across_outer_folds", "minimum_auc_decrease", "min_auc_decrease"],
        f"{algorithm} permutation minimum AUC decrease",
    )
    max_col = first_existing_column(
        df,
        ["max_across_outer_folds", "maximum_auc_decrease", "max_auc_decrease"],
        f"{algorithm} permutation maximum AUC decrease",
    )

    result = pd.DataFrame(
        {
            label_name: df[label_col].map(canonicalizer),
            "mean_auc_decrease": pd.to_numeric(df[mean_col], errors="raise"),
            "sd_across_outer_folds": pd.to_numeric(df[sd_col], errors="coerce"),
            "min_across_outer_folds": pd.to_numeric(df[min_col], errors="raise"),
            "max_across_outer_folds": pd.to_numeric(df[max_col], errors="raise"),
        }
    )
    result["algorithm"] = algorithm
    result["algorithm_label"] = Config.ALGORITHMS[algorithm]["label"]

    duplicate_mask = result.duplicated([label_name], keep=False)
    if duplicate_mask.any():
        duplicates = sorted(
            result.loc[duplicate_mask, label_name].astype(str).unique()
        )
        raise RuntimeError(
            f"{algorithm} permutation summary has duplicate {label_name} rows "
            f"after normalization: {duplicates}"
        )

    return result


def load_permutation_individual(algorithm: str, path: Path) -> pd.DataFrame:
    return _load_permutation_summary(
        algorithm=algorithm,
        path=path,
        label_candidates=["feature", "predictor", "variable"],
        label_name="feature",
        canonicalizer=canonical_permutation_label,
    )


def load_permutation_grouped(algorithm: str, path: Path) -> pd.DataFrame:
    return _load_permutation_summary(
        algorithm=algorithm,
        path=path,
        label_candidates=["feature_group", "group", "predictor_group"],
        label_name="feature_group",
        canonicalizer=canonical_permutation_group,
    )


def _permutation_wide_table(
    long_df: pd.DataFrame,
    label_column: str,
) -> pd.DataFrame:
    rows = []

    labels = sorted(
        long_df[label_column].dropna().astype(str).unique().tolist()
    )

    for label in labels:
        row: Dict[str, Any] = {label_column: label}
        available_means = []

        for algorithm in Config.ALGORITHMS:
            subset = long_df.loc[
                (long_df["algorithm"] == algorithm)
                & (long_df[label_column].astype(str) == label)
            ]

            if len(subset) > 1:
                raise RuntimeError(
                    f"Multiple permutation rows found for {algorithm} / {label}."
                )

            if subset.empty:
                row[f"{algorithm}_mean_auc_decrease"] = np.nan
                row[f"{algorithm}_sd_across_outer_folds"] = np.nan
                row[f"{algorithm}_min_across_outer_folds"] = np.nan
                row[f"{algorithm}_max_across_outer_folds"] = np.nan
                continue

            record = subset.iloc[0]
            mean_value = float(record["mean_auc_decrease"])
            available_means.append(mean_value)

            row[f"{algorithm}_mean_auc_decrease"] = mean_value
            row[f"{algorithm}_sd_across_outer_folds"] = float(
                record["sd_across_outer_folds"]
            )
            row[f"{algorithm}_min_across_outer_folds"] = float(
                record["min_across_outer_folds"]
            )
            row[f"{algorithm}_max_across_outer_folds"] = float(
                record["max_across_outer_folds"]
            )

        row["n_algorithms_available"] = len(available_means)
        row["all_available_algorithms_positive"] = bool(
            available_means and all(value > 0.0 for value in available_means)
        )
        row["minimum_algorithm_mean_auc_decrease"] = (
            float(np.min(available_means)) if available_means else np.nan
        )
        row["maximum_algorithm_mean_auc_decrease"] = (
            float(np.max(available_means)) if available_means else np.nan
        )
        rows.append(row)

    return pd.DataFrame(rows)


def build_permutation_synthesis(
    paths_by_algorithm: Dict[str, Dict[str, Path]],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Read the already-computed held-out Model E permutation summaries from
    Steps 7.1A, 7.2A and 7.3A. No model fitting or permutation recomputation
    occurs in Step 7.4.
    """
    individual_long = pd.concat(
        [
            load_permutation_individual(
                algorithm,
                paths_by_algorithm[algorithm]["permutation_individual"],
            )
            for algorithm in Config.ALGORITHMS
        ],
        ignore_index=True,
    )

    grouped_long = pd.concat(
        [
            load_permutation_grouped(
                algorithm,
                paths_by_algorithm[algorithm]["permutation_grouped"],
            )
            for algorithm in Config.ALGORITHMS
        ],
        ignore_index=True,
    )

    individual_wide = _permutation_wide_table(
        individual_long,
        "feature",
    )
    grouped_wide = _permutation_wide_table(
        grouped_long,
        "feature_group",
    )

    primary_group = grouped_long.loc[
        grouped_long["feature_group"] == "HKFI + SNDR"
    ]

    if len(primary_group) != len(Config.ALGORITHMS):
        raise RuntimeError(
            "Expected one grouped HKFI + SNDR permutation row for each "
            f"algorithm; found {len(primary_group)} rows."
        )

    return individual_long, individual_wide, grouped_long, grouped_wide


def create_grouped_permutation_figure(
    grouped_df: pd.DataFrame,
    output_path: Path,
) -> None:
    groups = sorted(
        grouped_df["feature_group"].dropna().astype(str).unique().tolist()
    )

    if not groups:
        return

    algorithms = list(Config.ALGORITHMS)
    x = np.arange(len(groups), dtype=float)
    width = 0.8 / max(len(algorithms), 1)

    fig, ax = plt.subplots(figsize=Config.FIGSIZE)

    for index, algorithm in enumerate(algorithms):
        means = []
        errors = []

        for group in groups:
            subset = grouped_df.loc[
                (grouped_df["algorithm"] == algorithm)
                & (grouped_df["feature_group"] == group)
            ]

            if len(subset) != 1:
                raise RuntimeError(
                    f"Expected one grouped permutation row for "
                    f"{algorithm} / {group}; found {len(subset)}."
                )

            record = subset.iloc[0]
            means.append(float(record["mean_auc_decrease"]))
            errors.append(
                float(record["sd_across_outer_folds"])
                if np.isfinite(record["sd_across_outer_folds"])
                else 0.0
            )

        positions = x + (index - (len(algorithms) - 1) / 2.0) * width

        ax.bar(
            positions,
            means,
            width=width,
            yerr=errors,
            capsize=3,
            label=Config.ALGORITHMS[algorithm]["label"],
        )

    ax.axhline(0.0, linewidth=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(groups)
    ax.set_ylabel("Decrease in held-out ROC-AUC after permutation", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_xlabel("Joint predictor group", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(frameon=False)
    save_figure(fig, output_path)


# =============================================================================
# 7.4.4 — HKFI WEIGHTING ROBUSTNESS SYNTHESIS
# =============================================================================
def load_hkfi_performance(algorithm: str, path: Path) -> pd.DataFrame:
    """Normalize Step 7.xB HKFI performance outputs."""
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    scenario_col = first_existing_column(
        df,
        ["scenario", "hkfi_scenario", "hkfi_weight_variant", "variant"],
        f"{algorithm} HKFI performance scenario",
    )
    auc_col = first_existing_column(
        df,
        ["oof_roc_auc", "roc_auc", "auc"],
        f"{algorithm} HKFI ROC-AUC",
    )
    ci_low_col = first_existing_column(
        df,
        ["auc_ci_low", "cluster_bootstrap_ci_low", "ci_low"],
        f"{algorithm} HKFI ROC-AUC lower CI",
    )
    ci_high_col = first_existing_column(
        df,
        ["auc_ci_high", "cluster_bootstrap_ci_high", "ci_high"],
        f"{algorithm} HKFI ROC-AUC upper CI",
    )

    result = df.copy()
    result["hkfi_scenario"] = result[scenario_col].astype(str).str.strip()
    result["oof_roc_auc"] = pd.to_numeric(result[auc_col], errors="raise")
    result["auc_ci_low"] = pd.to_numeric(result[ci_low_col], errors="raise")
    result["auc_ci_high"] = pd.to_numeric(result[ci_high_col], errors="raise")
    result["algorithm"] = algorithm
    result["algorithm_label"] = Config.ALGORITHMS[algorithm]["label"]

    duplicate_mask = result.duplicated(["hkfi_scenario"], keep=False)
    if duplicate_mask.any():
        duplicates = sorted(
            result.loc[duplicate_mask, "hkfi_scenario"].astype(str).unique()
        )
        raise RuntimeError(
            f"{algorithm} HKFI performance has duplicate scenario rows: {duplicates}"
        )

    return result


def load_hkfi_paired(algorithm: str, path: Path) -> pd.DataFrame:
    """
    Normalize Step 7.xB paired HKFI comparisons.

    The upstream files normally contain a 'scenario' column. For older outputs
    that only contain a comparison label, the alternative scenario is inferred
    from that label. If an old table stores 'equal - alternative', the sign and
    confidence interval are reversed so the standardized result is always:
        alternative - equal
    """
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    delta_col = first_existing_column(
        df,
        ["auc_difference_left_minus_right", "delta_auc", "auc_difference"],
        f"{algorithm} HKFI paired AUC difference",
    )
    ci_low_col = first_existing_column(
        df,
        ["cluster_bootstrap_ci_low", "ci_low", "auc_ci_low"],
        f"{algorithm} HKFI paired lower CI",
    )
    ci_high_col = first_existing_column(
        df,
        ["cluster_bootstrap_ci_high", "ci_high", "auc_ci_high"],
        f"{algorithm} HKFI paired upper CI",
    )

    scenario_col = next(
        (
            column
            for column in [
                "scenario",
                "hkfi_scenario",
                "hkfi_weight_variant",
                "variant",
            ]
            if column in df.columns
        ),
        None,
    )
    comparison_col = next(
        (
            column
            for column in ["comparison", "contrast", "comparison_label"]
            if column in df.columns
        ),
        None,
    )

    rows = []
    known_scenarios = list(Config.HKFI_TARGETED_ORDER)

    for _, source in df.iterrows():
        scenario = None

        if scenario_col is not None:
            candidate = str(source[scenario_col]).strip()
            if candidate and candidate.lower() not in {"nan", "none"}:
                scenario = candidate

        comparison_text = (
            str(source[comparison_col]).strip()
            if comparison_col is not None
            else ""
        )

        if scenario is None:
            matches = [
                candidate
                for candidate in known_scenarios
                if candidate != Config.HKFI_REFERENCE
                and candidate.lower() in comparison_text.lower()
            ]
            if len(matches) == 1:
                scenario = matches[0]

        if scenario is None:
            # Retain non-targeted XGB scenarios as well when their labels are
            # explicitly encoded as "<scenario> - primary_equal_component".
            lower = comparison_text.lower()
            reference_lower = Config.HKFI_REFERENCE.lower()
            if reference_lower in lower:
                cleaned = comparison_text.replace("−", "-").replace("–", "-")
                parts = [part.strip() for part in cleaned.split("-") if part.strip()]
                non_reference = [
                    part for part in parts
                    if part.lower() != reference_lower
                ]
                if len(non_reference) == 1:
                    scenario = non_reference[0]

        if scenario is None:
            raise RuntimeError(
                f"{algorithm} HKFI paired table: could not identify the alternative "
                f"scenario from row:\n{source.to_dict()}"
            )

        estimate = float(source[delta_col])
        low = float(source[ci_low_col])
        high = float(source[ci_high_col])

        # Standardize direction to alternative - equal when the comparison label
        # makes an old reversed orientation explicit.
        reversed_orientation = False
        normalized_comparison = (
            comparison_text.replace("−", "-").replace("–", "-").lower()
        )
        if normalized_comparison:
            reference_pos = normalized_comparison.find(
                Config.HKFI_REFERENCE.lower()
            )
            scenario_pos = normalized_comparison.find(str(scenario).lower())
            reversed_orientation = (
                reference_pos >= 0
                and scenario_pos >= 0
                and reference_pos < scenario_pos
            )

        if reversed_orientation:
            estimate, low, high = -estimate, -high, -low

        rows.append(
            {
                "algorithm": algorithm,
                "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                "hkfi_scenario": str(scenario).strip(),
                "comparison": comparison_text,
                "delta_auc_alternative_minus_equal": estimate,
                "ci_low": low,
                "ci_high": high,
            }
        )

    result = pd.DataFrame(rows)

    duplicate_mask = result.duplicated(["hkfi_scenario"], keep=False)
    if duplicate_mask.any():
        duplicates = sorted(
            result.loc[duplicate_mask, "hkfi_scenario"].astype(str).unique()
        )
        raise RuntimeError(
            f"{algorithm} HKFI paired table has duplicate alternative scenarios: "
            f"{duplicates}"
        )

    return result


def build_hkfi_synthesis(
    paths_by_algorithm: Dict[str, Dict[str, Path]],
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    performance_frames = []
    paired_frames = []

    for algorithm in Config.ALGORITHMS:
        performance_frames.append(
            load_hkfi_performance(
                algorithm,
                paths_by_algorithm[algorithm]["hkfi_performance"],
            )
        )
        paired_frames.append(
            load_hkfi_paired(
                algorithm,
                paths_by_algorithm[algorithm]["hkfi_paired"],
            )
        )

    all_performance = pd.concat(performance_frames, ignore_index=True)
    all_paired = pd.concat(paired_frames, ignore_index=True)

    # Preserve the complete XGB sensitivity because Step 7.1B is the full
    # weighting experiment. RF/LR are intentionally targeted replications.
    xgb_full_performance = all_performance.loc[
        all_performance["algorithm"] == "XGB"
    ].copy()
    xgb_full_paired = all_paired.loc[
        all_paired["algorithm"] == "XGB"
    ].copy()
    if not xgb_full_paired.empty:
        xgb_full_paired["interpretation"] = xgb_full_paired.apply(
            lambda row: interval_interpretation(
                float(row["delta_auc_alternative_minus_equal"]),
                float(row["ci_low"]),
                float(row["ci_high"]),
            ),
            axis=1,
        )

    scenario_sets = {
        algorithm: set(
            all_performance.loc[
                all_performance["algorithm"] == algorithm,
                "hkfi_scenario",
            ].astype(str)
        )
        for algorithm in Config.ALGORITHMS
    }

    missing_targeted = {
        algorithm: [
            scenario
            for scenario in Config.HKFI_TARGETED_ORDER
            if scenario not in scenario_sets[algorithm]
        ]
        for algorithm in Config.ALGORITHMS
    }
    missing_targeted = {
        algorithm: missing
        for algorithm, missing in missing_targeted.items()
        if missing
    }
    if missing_targeted:
        raise RuntimeError(
            "Targeted HKFI scenarios are missing from one or more algorithms:\n"
            + json.dumps(missing_targeted, indent=2)
        )

    common_performance = all_performance.loc[
        all_performance["hkfi_scenario"].isin(Config.HKFI_TARGETED_ORDER)
    ].copy()
    common_performance["hkfi_label"] = common_performance[
        "hkfi_scenario"
    ].map(Config.HKFI_LABELS)

    common_paired = all_paired.loc[
        all_paired["hkfi_scenario"].isin(Config.HKFI_TARGETED_ORDER[1:])
    ].copy()
    common_paired["hkfi_label"] = common_paired[
        "hkfi_scenario"
    ].map(Config.HKFI_LABELS)
    common_paired["interpretation"] = common_paired.apply(
        lambda row: interval_interpretation(
            float(row["delta_auc_alternative_minus_equal"]),
            float(row["ci_low"]),
            float(row["ci_high"]),
        ),
        axis=1,
    )

    expected_paired_rows = (
        len(Config.ALGORITHMS)
        * (len(Config.HKFI_TARGETED_ORDER) - 1)
    )
    if len(common_paired) != expected_paired_rows:
        raise RuntimeError(
            "Incomplete targeted HKFI paired synthesis: expected "
            f"{expected_paired_rows} rows but found {len(common_paired)}."
        )

    consensus_rows = []
    for scenario in Config.HKFI_TARGETED_ORDER[1:]:
        subset = common_paired.loc[
            common_paired["hkfi_scenario"] == scenario
        ].copy()

        if subset["algorithm"].nunique() != len(Config.ALGORITHMS):
            raise RuntimeError(
                f"HKFI scenario {scenario} does not have one paired result "
                "for each of XGB, RF and LR."
            )

        intervals_include_zero = (
            (subset["ci_low"] <= 0.0)
            & (subset["ci_high"] >= 0.0)
        )
        all_include_zero = bool(intervals_include_zero.all())
        deltas = subset[
            "delta_auc_alternative_minus_equal"
        ].to_numpy(float)

        consensus_rows.append(
            {
                "hkfi_scenario": scenario,
                "hkfi_label": Config.HKFI_LABELS[scenario],
                "XGB_delta_auc_vs_equal": float(
                    subset.loc[
                        subset["algorithm"] == "XGB",
                        "delta_auc_alternative_minus_equal",
                    ].iloc[0]
                ),
                "RF_delta_auc_vs_equal": float(
                    subset.loc[
                        subset["algorithm"] == "RF",
                        "delta_auc_alternative_minus_equal",
                    ].iloc[0]
                ),
                "LR_delta_auc_vs_equal": float(
                    subset.loc[
                        subset["algorithm"] == "LR",
                        "delta_auc_alternative_minus_equal",
                    ].iloc[0]
                ),
                "max_absolute_delta_auc_across_algorithms": float(
                    np.max(np.abs(deltas))
                ),
                "all_95pct_intervals_include_zero": all_include_zero,
                "scientific_conclusion": (
                    "No targeted alternative shows clear superiority over equal "
                    "weighting across all algorithms; the a priori equal HKFI "
                    "weighting is reasonably robust."
                    if all_include_zero
                    else
                    "At least one algorithm-specific paired interval excludes "
                    "zero. Equal-weight robustness therefore requires qualified "
                    "interpretation rather than post-hoc weight optimization."
                ),
            }
        )

    return (
        xgb_full_performance,
        xgb_full_paired,
        common_performance,
        common_paired,
        pd.DataFrame(consensus_rows),
    )


def create_hkfi_figure(common_performance: pd.DataFrame, output_path: Path) -> None:
    order = Config.HKFI_TARGETED_ORDER
    x = np.arange(len(order), dtype=float)
    offsets = np.linspace(-0.18, 0.18, len(Config.ALGORITHMS))

    fig, ax = plt.subplots(figsize=(6.8, 4.5))
    for offset, algorithm in zip(offsets, Config.ALGORITHMS):
        subset = (
            common_performance.loc[common_performance["algorithm"] == algorithm]
            .set_index("hkfi_scenario")
            .loc[order]
        )
        y = subset["oof_roc_auc"].to_numpy(float)
        low = subset["auc_ci_low"].to_numpy(float)
        high = subset["auc_ci_high"].to_numpy(float)
        ax.errorbar(
            x + offset,
            y,
            yerr=np.vstack([y - low, high - y]),
            marker="o",
            linestyle="none",
            capsize=3,
            label=Config.ALGORITHMS[algorithm]["label"],
        )

    ax.set_xticks(x)
    ax.set_xticklabels([Config.HKFI_LABELS[s] for s in order], rotation=18, ha="right")
    ax.set_xlabel("Pre-specified HKFI weighting scenario", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("Nested OOF ROC-AUC", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    save_figure(fig, output_path)


# =============================================================================
# 7.4.5 — PSEUDO-ABSENCE BUFFER SENSITIVITY SYNTHESIS
# =============================================================================
def load_buffer_performance(algorithm: str, path: Path) -> pd.DataFrame:
    """Normalize Step 7.xC A/E buffer performance outputs."""
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    buffer_col = first_existing_column(
        df,
        ["buffer_m", "buffer", "buffer_distance_m"],
        f"{algorithm} buffer distance",
    )
    scenario_col = first_existing_column(
        df,
        ["scenario", "model_configuration", "configuration", "model"],
        f"{algorithm} buffer scenario",
    )
    auc_col = first_existing_column(
        df,
        ["oof_roc_auc", "roc_auc", "auc"],
        f"{algorithm} buffer ROC-AUC",
    )
    ci_low_col = first_existing_column(
        df,
        ["auc_ci_low", "cluster_bootstrap_ci_low", "ci_low"],
        f"{algorithm} buffer ROC-AUC lower CI",
    )
    ci_high_col = first_existing_column(
        df,
        ["auc_ci_high", "cluster_bootstrap_ci_high", "ci_high"],
        f"{algorithm} buffer ROC-AUC upper CI",
    )

    result = df.copy()
    result["buffer_m"] = pd.to_numeric(result[buffer_col], errors="raise").astype(int)
    result["scenario"] = result[scenario_col].map(canonical_scenario)
    result["oof_roc_auc"] = pd.to_numeric(result[auc_col], errors="raise")
    result["auc_ci_low"] = pd.to_numeric(result[ci_low_col], errors="raise")
    result["auc_ci_high"] = pd.to_numeric(result[ci_high_col], errors="raise")
    result["algorithm"] = algorithm
    result["algorithm_label"] = Config.ALGORITHMS[algorithm]["label"]
    return result


def load_buffer_paired(algorithm: str, path: Path) -> pd.DataFrame:
    """Normalize Step 7.xC paired contrasts to Model E - Model A."""
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    buffer_col = first_existing_column(
        df,
        ["buffer_m", "buffer", "buffer_distance_m"],
        f"{algorithm} paired buffer distance",
    )
    delta_col = first_existing_column(
        df,
        ["auc_difference_left_minus_right", "delta_auc", "auc_difference"],
        f"{algorithm} buffer paired AUC difference",
    )
    ci_low_col = first_existing_column(
        df,
        ["cluster_bootstrap_ci_low", "ci_low", "auc_ci_low"],
        f"{algorithm} buffer lower CI",
    )
    ci_high_col = first_existing_column(
        df,
        ["cluster_bootstrap_ci_high", "ci_high", "auc_ci_high"],
        f"{algorithm} buffer upper CI",
    )
    comparison_col = next(
        (
            column
            for column in ["comparison", "contrast", "comparison_label"]
            if column in df.columns
        ),
        None,
    )

    rows = []
    for _, source in df.iterrows():
        estimate = float(source[delta_col])
        low = float(source[ci_low_col])
        high = float(source[ci_high_col])
        comparison = (
            str(source[comparison_col]).strip()
            if comparison_col is not None
            else ""
        )

        normalized = comparison.replace("−", "-").replace("–", "-").upper()
        # Explicit old reversed orientation: A - E.
        reversed_orientation = bool(
            re.search(r"\bA\s*-\s*E\b", normalized)
        )
        if reversed_orientation:
            estimate, low, high = -estimate, -high, -low

        rows.append(
            {
                "algorithm": algorithm,
                "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                "buffer_m": int(float(source[buffer_col])),
                "comparison": comparison,
                "delta_auc_E_minus_A": estimate,
                "delta_auc_ci_low": low,
                "delta_auc_ci_high": high,
            }
        )

    result = pd.DataFrame(rows)
    duplicate_mask = result.duplicated(["buffer_m"], keep=False)
    if duplicate_mask.any():
        duplicates = sorted(
            result.loc[duplicate_mask, "buffer_m"].astype(int).unique()
        )
        raise RuntimeError(
            f"{algorithm} buffer paired table has duplicate rows for buffers: "
            f"{duplicates}"
        )
    return result


def build_buffer_synthesis(
    paths_by_algorithm: Dict[str, Dict[str, Path]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    performance = pd.concat(
        [
            load_buffer_performance(
                algorithm,
                paths_by_algorithm[algorithm]["buffer_performance"],
            )
            for algorithm in Config.ALGORITHMS
        ],
        ignore_index=True,
    )

    paired = pd.concat(
        [
            load_buffer_paired(
                algorithm,
                paths_by_algorithm[algorithm]["buffer_paired"],
            )
            for algorithm in Config.ALGORITHMS
        ],
        ignore_index=True,
    )

    rows = []
    for algorithm in Config.ALGORITHMS:
        for buffer_m in Config.BUFFER_ORDER:
            subset = performance.loc[
                (performance["algorithm"] == algorithm)
                & (performance["buffer_m"].astype(int) == buffer_m)
                & performance["scenario"].isin(["A_static_baseline", "E_physics_guided"])
            ]

            model_a = subset.loc[subset["scenario"] == "A_static_baseline"]
            model_e = subset.loc[subset["scenario"] == "E_physics_guided"]
            delta = paired.loc[
                (paired["algorithm"] == algorithm)
                & (paired["buffer_m"] == buffer_m)
            ]

            if len(model_a) != 1 or len(model_e) != 1 or len(delta) != 1:
                raise RuntimeError(
                    f"Incomplete buffer synthesis inputs for {algorithm} at {buffer_m} m."
                )

            a = model_a.iloc[0]
            e = model_e.iloc[0]
            d = delta.iloc[0]

            rows.append(
                {
                    "algorithm": algorithm,
                    "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                    "buffer_m": buffer_m,
                    "model_A_auc": float(a["oof_roc_auc"]),
                    "model_A_auc_ci_low": float(a["auc_ci_low"]),
                    "model_A_auc_ci_high": float(a["auc_ci_high"]),
                    "model_E_auc": float(e["oof_roc_auc"]),
                    "model_E_auc_ci_low": float(e["auc_ci_low"]),
                    "model_E_auc_ci_high": float(e["auc_ci_high"]),
                    "delta_auc_E_minus_A": float(d["delta_auc_E_minus_A"]),
                    "delta_auc_ci_low": float(d["delta_auc_ci_low"]),
                    "delta_auc_ci_high": float(d["delta_auc_ci_high"]),
                    "direction_positive": bool(float(d["delta_auc_E_minus_A"]) > 0.0),
                    "interpretation": interval_interpretation(
                        float(d["delta_auc_E_minus_A"]),
                        float(d["delta_auc_ci_low"]),
                        float(d["delta_auc_ci_high"]),
                    ),
                }
            )

    result = pd.DataFrame(rows)

    consistency_rows = []
    for buffer_m in Config.BUFFER_ORDER:
        subset = result.loc[result["buffer_m"] == buffer_m]
        consistency_rows.append(
            {
                "buffer_m": buffer_m,
                "XGB_delta_auc_E_minus_A": float(
                    subset.loc[subset["algorithm"] == "XGB", "delta_auc_E_minus_A"].iloc[0]
                ),
                "RF_delta_auc_E_minus_A": float(
                    subset.loc[subset["algorithm"] == "RF", "delta_auc_E_minus_A"].iloc[0]
                ),
                "LR_delta_auc_E_minus_A": float(
                    subset.loc[subset["algorithm"] == "LR", "delta_auc_E_minus_A"].iloc[0]
                ),
                "all_algorithms_directionally_positive": bool(
                    (subset["delta_auc_E_minus_A"] > 0.0).all()
                ),
                "all_algorithms_95pct_intervals_above_zero": bool(
                    (subset["delta_auc_ci_low"] > 0.0).all()
                ),
            }
        )

    consistency = pd.DataFrame(consistency_rows)
    return result, consistency


def create_buffer_figure(buffer_df: pd.DataFrame, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=Config.FIGSIZE)
    for algorithm in Config.ALGORITHMS:
        subset = buffer_df.loc[buffer_df["algorithm"] == algorithm].sort_values("buffer_m")
        x = subset["buffer_m"].to_numpy(float)
        y = subset["delta_auc_E_minus_A"].to_numpy(float)
        low = subset["delta_auc_ci_low"].to_numpy(float)
        high = subset["delta_auc_ci_high"].to_numpy(float)
        ax.errorbar(
            x,
            y,
            yerr=np.vstack([y - low, high - y]),
            marker="o",
            capsize=3,
            linewidth=1.3,
            label=Config.ALGORITHMS[algorithm]["label"],
        )

    ax.axhline(0.0, linestyle="--", linewidth=1.0)
    ax.set_xticks(Config.BUFFER_ORDER)
    ax.set_xlabel("Pseudo-absence exclusion buffer (m)", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("Paired ΔROC-AUC (Model E - Model A)", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    save_figure(fig, output_path)


# =============================================================================
# 7.4.6 — GEOGRAPHIC BLOCK ROBUSTNESS
# =============================================================================
def load_fold_metrics(algorithm: str, path: Path) -> pd.DataFrame:
    """Normalize the saved five outer-fold performance tables."""
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    scenario_col = first_existing_column(
        df,
        ["scenario", "model_configuration", "configuration", "model"],
        f"{algorithm} outer-fold scenario",
    )
    fold_col = first_existing_column(
        df,
        ["outer_fold", "cv_fold", "spatial_block", "fold"],
        f"{algorithm} outer-fold identifier",
    )
    auc_col = first_existing_column(
        df,
        ["roc_auc", "heldout_roc_auc", "auc"],
        f"{algorithm} outer-fold ROC-AUC",
    )

    result = df.copy()
    result["scenario"] = result[scenario_col].map(canonical_scenario)
    result["outer_fold"] = pd.to_numeric(
        result[fold_col], errors="raise"
    ).astype(int)
    result["roc_auc"] = pd.to_numeric(result[auc_col], errors="raise")
    result["algorithm"] = algorithm
    result["algorithm_label"] = Config.ALGORITHMS[algorithm]["label"]
    return result


def build_geographic_block_synthesis(
    paths_by_algorithm: Dict[str, Dict[str, Path]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Summarize the existing five outer-fold results geographically.

    The compact reviewer-facing table contains only Model E AUC by spatial block.
    The long table also retains Model A and E-A for diagnostic interpretation.
    """
    combined = pd.concat(
        [
            load_fold_metrics(
                algorithm,
                paths_by_algorithm[algorithm]["fold_metrics"],
            )
            for algorithm in Config.ALGORITHMS
        ],
        ignore_index=True,
    )

    rows = []
    for algorithm in Config.ALGORITHMS:
        for fold in Config.EXPECTED_FOLDS:
            subset = combined.loc[
                (combined["algorithm"] == algorithm)
                & (combined["outer_fold"] == fold)
                & combined["scenario"].isin(
                    ["A_static_baseline", "E_physics_guided"]
                )
            ]
            model_a = subset.loc[
                subset["scenario"] == "A_static_baseline"
            ]
            model_e = subset.loc[
                subset["scenario"] == "E_physics_guided"
            ]

            if len(model_a) != 1 or len(model_e) != 1:
                raise RuntimeError(
                    f"Expected exactly one Model A and one Model E outer-fold "
                    f"row for {algorithm}, spatial block {fold}; found "
                    f"A={len(model_a)}, E={len(model_e)}."
                )

            auc_a = float(model_a.iloc[0]["roc_auc"])
            auc_e = float(model_e.iloc[0]["roc_auc"])

            rows.append(
                {
                    "algorithm": algorithm,
                    "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                    "spatial_block": fold,
                    "model_A_auc": auc_a,
                    "model_E_auc": auc_e,
                    "delta_auc_E_minus_A": auc_e - auc_a,
                }
            )

    long_df = pd.DataFrame(rows)

    wide_rows = []
    for fold in Config.EXPECTED_FOLDS:
        row: Dict[str, Any] = {"Spatial block": fold}
        for algorithm in Config.ALGORITHMS:
            record = long_df.loc[
                (long_df["algorithm"] == algorithm)
                & (long_df["spatial_block"] == fold)
            ].iloc[0]
            row[f"{algorithm} E AUC"] = float(record["model_E_auc"])
        wide_rows.append(row)

    return long_df, pd.DataFrame(wide_rows)


def create_geographic_block_figure(block_df: pd.DataFrame, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=Config.FIGSIZE)
    for algorithm in Config.ALGORITHMS:
        subset = block_df.loc[block_df["algorithm"] == algorithm].sort_values("spatial_block")
        ax.plot(
            subset["spatial_block"].to_numpy(int),
            subset["model_E_auc"].to_numpy(float),
            marker="o",
            linewidth=1.3,
            label=Config.ALGORITHMS[algorithm]["label"],
        )

    ax.set_xticks(Config.EXPECTED_FOLDS)
    ax.set_xlabel("Held-out spatial block", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("Model E held-out ROC-AUC", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    save_figure(fig, output_path)



# =============================================================================
# 7.4.7 — CROSS-ALGORITHM MAPPED-AREA UNCERTAINTY SYNTHESIS
# =============================================================================
def canonical_susceptibility_class(value: Any) -> str:
    text = str(value).strip()
    compact = re.sub(r"[^a-z0-9]+", "", text.lower())

    aliases = {
        "verylow": "Very Low",
        "low": "Low",
        "moderate": "Moderate",
        "medium": "Moderate",
        "high": "High",
        "veryhigh": "Very High",
        "highveryhigh": "High + Very High",
        "veryhighhigh": "High + Very High",
    }

    return aliases.get(compact, text)


def load_mapped_area_uncertainty(
    algorithm: str,
    path: Path,
) -> pd.DataFrame:
    """
    Normalize the Step 7.xD A30/E70 cross-fold mapped-area summary.

    These values are descriptive cross-fold map variability across five
    fold-trained operational ensemble maps. They are not formal 95% CIs.
    """
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    class_id_col = first_existing_column(
        df,
        ["class_id", "susceptibility_class_id"],
        f"{algorithm} mapped-area class ID",
    )
    class_label_col = first_existing_column(
        df,
        ["class_label", "susceptibility_class", "class"],
        f"{algorithm} mapped-area class label",
    )
    mean_col = first_existing_column(
        df,
        ["mean_area_percentage", "mean_area_percent"],
        f"{algorithm} mapped-area mean percentage",
    )
    sd_col = first_existing_column(
        df,
        ["sd_area_percentage", "std_area_percentage", "sd_area_percent"],
        f"{algorithm} mapped-area SD percentage",
    )
    min_col = first_existing_column(
        df,
        ["minimum_area_percentage", "min_area_percentage"],
        f"{algorithm} mapped-area minimum percentage",
    )
    max_col = first_existing_column(
        df,
        ["maximum_area_percentage", "max_area_percentage"],
        f"{algorithm} mapped-area maximum percentage",
    )

    n_col = next(
        (
            column
            for column in [
                "n_spatial_fold_models",
                "n_fold_models",
                "n_realizations",
            ]
            if column in df.columns
        ),
        None,
    )
    final_area_col = next(
        (
            column
            for column in [
                "final_full_data_area_percentage",
                "final_area_percentage",
                "full_data_area_percentage",
            ]
            if column in df.columns
        ),
        None,
    )
    final_difference_col = next(
        (
            column
            for column in [
                "final_minus_crossfold_mean_percentage_points",
                "final_minus_crossfold_mean",
            ]
            if column in df.columns
        ),
        None,
    )

    result = pd.DataFrame(
        {
            "class_id": pd.to_numeric(df[class_id_col], errors="raise").astype(int),
            "class_label": df[class_label_col].map(canonical_susceptibility_class),
            "mean_area_percentage": pd.to_numeric(df[mean_col], errors="raise"),
            "sd_area_percentage": pd.to_numeric(df[sd_col], errors="coerce"),
            "minimum_area_percentage": pd.to_numeric(df[min_col], errors="raise"),
            "maximum_area_percentage": pd.to_numeric(df[max_col], errors="raise"),
        }
    )

    result["n_spatial_fold_models"] = (
        pd.to_numeric(df[n_col], errors="coerce")
        if n_col is not None
        else np.nan
    )
    result["final_full_data_area_percentage"] = (
        pd.to_numeric(df[final_area_col], errors="coerce")
        if final_area_col is not None
        else np.nan
    )
    result["final_minus_crossfold_mean_percentage_points"] = (
        pd.to_numeric(df[final_difference_col], errors="coerce")
        if final_difference_col is not None
        else (
            result["final_full_data_area_percentage"]
            - result["mean_area_percentage"]
        )
    )

    result["algorithm"] = algorithm
    result["algorithm_label"] = Config.ALGORITHMS[algorithm]["label"]
    result["map_product"] = "A30/E70 operational ensemble"

    duplicate_mask = result.duplicated(["class_id"], keep=False)
    if duplicate_mask.any():
        duplicates = sorted(
            result.loc[duplicate_mask, "class_id"].astype(int).unique().tolist()
        )
        raise RuntimeError(
            f"{algorithm} mapped-area uncertainty has duplicate class IDs: "
            f"{duplicates}"
        )

    standard_classes = set(result.loc[result["class_id"].isin([1, 2, 3, 4, 5]), "class_id"])
    if standard_classes != {1, 2, 3, 4, 5}:
        raise RuntimeError(
            f"{algorithm} mapped-area summary must contain class IDs 1-5; "
            f"found {sorted(standard_classes)}."
        )

    n_values = result["n_spatial_fold_models"].dropna()
    if not n_values.empty and not (n_values.astype(int) == len(Config.EXPECTED_FOLDS)).all():
        raise RuntimeError(
            f"{algorithm} mapped-area uncertainty is not based on exactly "
            f"{len(Config.EXPECTED_FOLDS)} spatial-fold map realizations."
        )

    return result


def _mapped_area_wide_table(long_df: pd.DataFrame) -> pd.DataFrame:
    class_records = (
        long_df[["class_id", "class_label"]]
        .drop_duplicates()
        .sort_values(["class_id", "class_label"])
    )

    rows = []

    for _, class_record in class_records.iterrows():
        class_id = int(class_record["class_id"])
        class_label = str(class_record["class_label"])
        row: Dict[str, Any] = {
            "class_id": class_id,
            "class_label": class_label,
            "map_product": "A30/E70 operational ensemble",
        }

        for algorithm in Config.ALGORITHMS:
            subset = long_df.loc[
                (long_df["algorithm"] == algorithm)
                & (long_df["class_id"] == class_id)
            ]

            if len(subset) != 1:
                raise RuntimeError(
                    f"Expected one mapped-area row for {algorithm}, "
                    f"class ID {class_id}; found {len(subset)}."
                )

            record = subset.iloc[0]
            row[f"{algorithm}_mean_area_percentage"] = float(
                record["mean_area_percentage"]
            )
            row[f"{algorithm}_sd_area_percentage"] = float(
                record["sd_area_percentage"]
            )
            row[f"{algorithm}_minimum_area_percentage"] = float(
                record["minimum_area_percentage"]
            )
            row[f"{algorithm}_maximum_area_percentage"] = float(
                record["maximum_area_percentage"]
            )
            row[f"{algorithm}_final_full_data_area_percentage"] = (
                float(record["final_full_data_area_percentage"])
                if np.isfinite(record["final_full_data_area_percentage"])
                else np.nan
            )
            row[
                f"{algorithm}_final_minus_crossfold_mean_percentage_points"
            ] = (
                float(
                    record[
                        "final_minus_crossfold_mean_percentage_points"
                    ]
                )
                if np.isfinite(
                    record[
                        "final_minus_crossfold_mean_percentage_points"
                    ]
                )
                else np.nan
            )

        rows.append(row)

    result = pd.DataFrame(rows)

    # Keep the five standard classes in numerical order and the combined
    # High + Very High row after them.
    result["_sort_key"] = result["class_id"].map(
        lambda value: value if value in {1, 2, 3, 4, 5} else 100 + value
    )
    result = result.sort_values("_sort_key").drop(columns="_sort_key")

    return result


def build_mapped_area_uncertainty_synthesis(
    paths_by_algorithm: Dict[str, Dict[str, Path]],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    long_df = pd.concat(
        [
            load_mapped_area_uncertainty(
                algorithm,
                paths_by_algorithm[algorithm]["map_area_uncertainty"],
            )
            for algorithm in Config.ALGORITHMS
        ],
        ignore_index=True,
    )

    wide_df = _mapped_area_wide_table(long_df)

    high_combined = long_df.loc[
        (long_df["class_id"] == 99)
        | (long_df["class_label"] == "High + Very High")
    ].copy()

    if len(high_combined) != len(Config.ALGORITHMS):
        raise RuntimeError(
            "Expected one combined High + Very High mapped-area uncertainty "
            f"row for each algorithm; found {len(high_combined)} rows."
        )

    high_summary = high_combined[
        [
            "algorithm",
            "algorithm_label",
            "map_product",
            "mean_area_percentage",
            "sd_area_percentage",
            "minimum_area_percentage",
            "maximum_area_percentage",
            "n_spatial_fold_models",
            "final_full_data_area_percentage",
            "final_minus_crossfold_mean_percentage_points",
        ]
    ].copy()

    return long_df, wide_df, high_summary


def create_mapped_area_uncertainty_figure(
    area_df: pd.DataFrame,
    output_path: Path,
) -> None:
    standard = area_df.loc[
        area_df["class_id"].isin([1, 2, 3, 4, 5])
    ].copy()

    class_order = (
        standard[["class_id", "class_label"]]
        .drop_duplicates()
        .sort_values("class_id")
    )
    class_ids = class_order["class_id"].astype(int).tolist()
    class_labels = class_order["class_label"].astype(str).tolist()

    algorithms = list(Config.ALGORITHMS)
    x = np.arange(len(class_ids), dtype=float)
    width = 0.8 / max(len(algorithms), 1)

    fig, ax = plt.subplots(figsize=(7.2, 4.8))

    for index, algorithm in enumerate(algorithms):
        means = []
        errors = []

        for class_id in class_ids:
            subset = standard.loc[
                (standard["algorithm"] == algorithm)
                & (standard["class_id"] == class_id)
            ]

            if len(subset) != 1:
                raise RuntimeError(
                    f"Expected one mapped-area row for {algorithm}, "
                    f"class ID {class_id}; found {len(subset)}."
                )

            record = subset.iloc[0]
            means.append(float(record["mean_area_percentage"]))
            errors.append(
                float(record["sd_area_percentage"])
                if np.isfinite(record["sd_area_percentage"])
                else 0.0
            )

        positions = x + (index - (len(algorithms) - 1) / 2.0) * width

        ax.bar(
            positions,
            means,
            width=width,
            yerr=errors,
            capsize=3,
            label=Config.ALGORITHMS[algorithm]["label"],
        )

    ax.set_xticks(x)
    ax.set_xticklabels(class_labels, rotation=20, ha="right")
    ax.set_ylabel("Cross-fold mapped area (%)", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_xlabel("Susceptibility class", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(frameon=False)
    save_figure(fig, output_path)


def create_high_very_high_uncertainty_figure(
    high_summary: pd.DataFrame,
    output_path: Path,
) -> None:
    ordered = high_summary.set_index("algorithm").loc[
        list(Config.ALGORITHMS)
    ].reset_index()

    x = np.arange(len(ordered))
    means = ordered["mean_area_percentage"].to_numpy(float)
    sd = ordered["sd_area_percentage"].fillna(0.0).to_numpy(float)

    fig, ax = plt.subplots(figsize=(5.8, 4.2))
    ax.bar(
        x,
        means,
        yerr=sd,
        capsize=3,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(ordered["algorithm_label"].astype(str))
    ax.set_ylabel("High + Very High mapped area (%)", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_xlabel("Algorithm", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.25)
    save_figure(fig, output_path)


# =============================================================================
# 7.4.8 — SUPPLEMENTARY FIVE-FOLD SPATIAL ROC SYNTHESIS FOR MODEL E
# =============================================================================
def normalize_roc_fold_value(value: Any) -> int:
    try:
        fold = int(float(value))
    except Exception as exc:
        raise RuntimeError(
            f"Could not interpret ROC fold identifier as an integer: {value}"
        ) from exc

    if fold not in Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"Unexpected ROC spatial fold identifier {fold}. "
            f"Expected one of {Config.EXPECTED_FOLDS}."
        )
    return fold


def safe_roc_auc(y_true: np.ndarray, y_prob: np.ndarray, context: str) -> float:
    if len(np.unique(y_true)) != 2:
        raise RuntimeError(f"{context} does not contain both target classes.")
    if not np.isfinite(y_prob).all():
        raise RuntimeError(f"{context} contains non-finite predicted probabilities.")
    return float(roc_auc_score(y_true, y_prob))


def save_roc_figure(
    fig: plt.Figure,
    png_path: Path,
    pdf_path: Path,
) -> None:
    """Save supplementary ROC figures in PNG, PDF and SVG format."""
    apply_clear_figure_style(fig)
    fig.tight_layout()

    png_path = normalize_output_path(png_path)
    pdf_path = normalize_output_path(pdf_path)
    svg_path = normalize_output_path(
        Path(str(png_path.with_suffix("")) + ".svg")
    )

    fig.savefig(
        str(png_path),
        dpi=Config.ROC_FIG_DPI,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        str(pdf_path),
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        str(svg_path),
        format="svg",
        bbox_inches="tight",
        facecolor="white",
    )

    log(f"[ROC PNG] {png_path}")
    log(f"[ROC PDF] {pdf_path}")
    log(f"[ROC SVG] {svg_path}")

    plt.close(fig)


def load_model_e_oof(
    algorithm: str,
    path: Path,
) -> tuple[pd.DataFrame, Dict[str, str]]:
    """Load the final saved Model E OOF predictions without refitting."""
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    scenario_col = first_existing_column(
        df,
        Config.ROC_SCENARIO_COLUMN_ALIASES,
        f"{algorithm} OOF scenario column",
    )
    fold_col = first_existing_column(
        df,
        Config.ROC_FOLD_COLUMN_ALIASES,
        f"{algorithm} OOF fold column",
    )
    target_col = first_existing_column(
        df,
        Config.ROC_TARGET_COLUMN_ALIASES,
        f"{algorithm} OOF target column",
    )
    probability_col = first_existing_column(
        df,
        Config.ROC_PROBABILITY_COLUMN_ALIASES,
        f"{algorithm} OOF probability column",
    )
    sample_id_col = next(
        (
            column
            for column in Config.ROC_SAMPLE_ID_COLUMN_ALIASES
            if column in df.columns
        ),
        None,
    )

    normalized = df.copy()
    normalized["_scenario"] = normalized[scenario_col].map(canonical_scenario)
    normalized = normalized.loc[
        normalized["_scenario"] == Config.PRIMARY_SCENARIO
    ].copy()

    if normalized.empty:
        observed = sorted(df[scenario_col].astype(str).unique().tolist())
        raise RuntimeError(
            f"{algorithm} OOF file contains no rows for {Config.PRIMARY_SCENARIO}.\n"
            f"Observed scenario values: {observed}"
        )

    normalized["outer_fold"] = normalized[fold_col].map(normalize_roc_fold_value)
    normalized["label"] = pd.to_numeric(
        normalized[target_col], errors="raise"
    ).astype(int)
    normalized["oof_probability"] = pd.to_numeric(
        normalized[probability_col], errors="raise"
    ).astype(float)

    if sample_id_col is not None:
        normalized["sample_id"] = normalized[sample_id_col].astype(str)
    else:
        normalized["sample_id"] = [
            f"{algorithm}_row_{index}" for index in range(len(normalized))
        ]

    if Config.EXPECTED_TOTAL_MODEL_E_ROWS is not None:
        if len(normalized) != Config.EXPECTED_TOTAL_MODEL_E_ROWS:
            raise RuntimeError(
                f"{algorithm} Model E OOF row count is {len(normalized)}; "
                f"expected {Config.EXPECTED_TOTAL_MODEL_E_ROWS}."
            )

    if normalized["sample_id"].duplicated().any():
        duplicated = normalized.loc[
            normalized["sample_id"].duplicated(keep=False), "sample_id"
        ].unique()
        raise RuntimeError(
            f"{algorithm} Model E OOF table contains duplicated sample IDs. "
            f"Examples: {list(duplicated[:10])}"
        )

    observed_folds = sorted(normalized["outer_fold"].unique().tolist())
    if observed_folds != Config.EXPECTED_FOLDS:
        raise RuntimeError(
            f"{algorithm} Model E OOF folds are {observed_folds}; "
            f"expected {Config.EXPECTED_FOLDS}."
        )

    if set(normalized["label"].unique()) != {0, 1}:
        raise RuntimeError(
            f"{algorithm} Model E labels must be exactly 0 and 1. "
            f"Observed: {sorted(normalized['label'].unique().tolist())}"
        )

    if not normalized["oof_probability"].between(0.0, 1.0).all():
        raise RuntimeError(
            f"{algorithm} Model E OOF probabilities are outside [0,1]."
        )

    for fold in Config.EXPECTED_FOLDS:
        subset = normalized.loc[normalized["outer_fold"] == fold]
        classes = sorted(subset["label"].unique().tolist())
        if classes != [0, 1]:
            raise RuntimeError(
                f"{algorithm} held-out fold {fold} contains classes {classes}; "
                "ROC requires both classes."
            )

    provenance = {
        "scenario_column": scenario_col,
        "fold_column": fold_col,
        "target_column": target_col,
        "probability_column": probability_col,
        "sample_id_column": (
            sample_id_col
            if sample_id_col is not None
            else "synthetic row identifier"
        ),
    }

    normalized = normalized[
        ["sample_id", "label", "outer_fold", "oof_probability"]
    ].sort_values(["outer_fold", "sample_id"]).reset_index(drop=True)
    normalized["algorithm"] = algorithm
    normalized["algorithm_label"] = Config.ALGORITHMS[algorithm]["label"]

    return normalized, provenance


def load_saved_model_e_fold_metrics(
    algorithm: str,
    path: Path,
) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [str(column).strip() for column in df.columns]

    scenario_col = first_existing_column(
        df,
        Config.ROC_SCENARIO_COLUMN_ALIASES,
        f"{algorithm} fold-metric scenario column",
    )
    fold_col = first_existing_column(
        df,
        ["outer_fold", "cv_fold", "spatial_block", "fold"],
        f"{algorithm} fold-metric fold column",
    )
    auc_col = first_existing_column(
        df,
        ["roc_auc", "heldout_roc_auc", "auc"],
        f"{algorithm} fold-metric ROC-AUC column",
    )

    result = df.copy()
    result["_scenario"] = result[scenario_col].map(canonical_scenario)
    result = result.loc[
        result["_scenario"] == Config.PRIMARY_SCENARIO
    ].copy()
    result["outer_fold"] = result[fold_col].map(normalize_roc_fold_value)
    result["saved_roc_auc"] = pd.to_numeric(
        result[auc_col], errors="raise"
    ).astype(float)
    result = result[["outer_fold", "saved_roc_auc"]].copy()

    if len(result) != len(Config.EXPECTED_FOLDS):
        raise RuntimeError(
            f"{algorithm} saved fold metrics contain {len(result)} Model E rows; "
            f"expected {len(Config.EXPECTED_FOLDS)}."
        )
    if result["outer_fold"].duplicated().any():
        raise RuntimeError(
            f"{algorithm} saved fold metrics contain duplicate Model E fold rows."
        )

    return result.sort_values("outer_fold").reset_index(drop=True)


def build_algorithm_roc_results(
    algorithm: str,
    model_e_oof: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, Dict[str, float]]:
    coordinate_frames = []
    summary_rows = []

    for fold in Config.EXPECTED_FOLDS:
        subset = model_e_oof.loc[
            model_e_oof["outer_fold"] == fold
        ].copy()
        y_true = subset["label"].to_numpy(dtype=int)
        y_prob = subset["oof_probability"].to_numpy(dtype=float)

        auc_value = safe_roc_auc(
            y_true,
            y_prob,
            context=f"{algorithm} Model E spatial fold {fold}",
        )
        fpr, tpr, thresholds = roc_curve(y_true, y_prob)

        coordinate_frames.append(
            pd.DataFrame(
                {
                    "algorithm": algorithm,
                    "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                    "curve_type": "held_out_spatial_fold",
                    "outer_fold": fold,
                    "roc_auc": auc_value,
                    "fpr": fpr,
                    "tpr": tpr,
                    "threshold": thresholds,
                }
            )
        )

        summary_rows.append(
            {
                "algorithm": algorithm,
                "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                "scenario": Config.PRIMARY_SCENARIO,
                "model": Config.PRIMARY_MODEL_LABEL,
                "outer_fold": fold,
                "n_samples": int(len(subset)),
                "n_positive": int((subset["label"] == 1).sum()),
                "n_negative": int((subset["label"] == 0).sum()),
                "roc_auc": auc_value,
            }
        )

    y_all = model_e_oof["label"].to_numpy(dtype=int)
    p_all = model_e_oof["oof_probability"].to_numpy(dtype=float)
    pooled_auc = safe_roc_auc(
        y_all,
        p_all,
        context=f"{algorithm} Model E pooled OOF predictions",
    )
    pooled_fpr, pooled_tpr, pooled_thresholds = roc_curve(y_all, p_all)

    coordinate_frames.append(
        pd.DataFrame(
            {
                "algorithm": algorithm,
                "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                "curve_type": "pooled_oof",
                "outer_fold": np.nan,
                "roc_auc": pooled_auc,
                "fpr": pooled_fpr,
                "tpr": pooled_tpr,
                "threshold": pooled_thresholds,
            }
        )
    )

    summary = pd.DataFrame(summary_rows)
    coordinates = pd.concat(coordinate_frames, ignore_index=True)
    fold_aucs = summary["roc_auc"].to_numpy(dtype=float)
    pooled_summary = {
        "pooled_oof_auc": pooled_auc,
        "mean_fold_auc": float(np.mean(fold_aucs)),
        "sd_fold_auc": float(np.std(fold_aucs, ddof=1)),
        "min_fold_auc": float(np.min(fold_aucs)),
        "max_fold_auc": float(np.max(fold_aucs)),
    }

    return coordinates, summary, pooled_summary


def build_roc_auc_qa(
    algorithm: str,
    reconstructed_summary: pd.DataFrame,
    saved_fold_metrics: pd.DataFrame,
) -> pd.DataFrame:
    merged = reconstructed_summary.merge(
        saved_fold_metrics,
        on="outer_fold",
        how="left",
        validate="one_to_one",
    )
    merged["absolute_difference"] = (
        merged["roc_auc"] - merged["saved_roc_auc"]
    ).abs()
    merged["tolerance"] = Config.ROC_AUC_QA_TOLERANCE
    merged["pass"] = (
        merged["absolute_difference"] <= Config.ROC_AUC_QA_TOLERANCE
    )
    merged["algorithm"] = algorithm
    merged["algorithm_label"] = Config.ALGORITHMS[algorithm]["label"]

    return merged[
        [
            "algorithm",
            "algorithm_label",
            "outer_fold",
            "roc_auc",
            "saved_roc_auc",
            "absolute_difference",
            "tolerance",
            "pass",
        ]
    ]


def configure_roc_axis(ax: plt.Axes, title: str) -> None:
    ax.plot([0.0, 1.0], [0.0, 1.0], linestyle=":", linewidth=1.0)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.02)
    ax.set_xlabel("False positive rate", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("True positive rate", fontsize=AXIS_LABEL_FONTSIZE, fontweight="bold")
    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight="bold")
    ax.grid(True, alpha=0.25)


def plot_algorithm_roc(
    algorithm: str,
    coordinates: pd.DataFrame,
    png_path: Path,
    pdf_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=Config.ROC_INDIVIDUAL_FIGSIZE)

    for fold in Config.EXPECTED_FOLDS:
        subset = coordinates.loc[
            (coordinates["curve_type"] == "held_out_spatial_fold")
            & (coordinates["outer_fold"] == fold)
        ].copy()
        auc_value = float(subset["roc_auc"].iloc[0])
        ax.plot(
            subset["fpr"].to_numpy(dtype=float),
            subset["tpr"].to_numpy(dtype=float),
            linewidth=1.3,
            label=f"Fold {fold} (AUC = {auc_value:.3f})",
        )

    pooled = coordinates.loc[
        coordinates["curve_type"] == "pooled_oof"
    ].copy()
    pooled_auc = float(pooled["roc_auc"].iloc[0])
    ax.plot(
        pooled["fpr"].to_numpy(dtype=float),
        pooled["tpr"].to_numpy(dtype=float),
        linewidth=2.5,
        linestyle="--",
        label=f"Pooled OOF (AUC = {pooled_auc:.3f})",
    )

    configure_roc_axis(
        ax,
        f"{Config.ALGORITHMS[algorithm]['label']} Model E",
    )
    ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE, loc="lower right")
    save_roc_figure(fig, png_path, pdf_path)


def plot_combined_three_panel_roc(
    all_coordinates: pd.DataFrame,
    png_path: Path,
    pdf_path: Path,
) -> None:
    fig, axes = plt.subplots(
        1,
        3,
        figsize=Config.ROC_COMBINED_FIGSIZE,
        sharex=True,
        sharey=True,
    )
    panel_labels = {
        "XGB": "(a) XGBoost",
        "RF": "(b) Random Forest",
        "LR": "(c) Logistic Regression",
    }

    for ax, algorithm in zip(axes, Config.ALGORITHMS):
        algorithm_data = all_coordinates.loc[
            all_coordinates["algorithm"] == algorithm
        ].copy()

        for fold in Config.EXPECTED_FOLDS:
            subset = algorithm_data.loc[
                (algorithm_data["curve_type"] == "held_out_spatial_fold")
                & (algorithm_data["outer_fold"] == fold)
            ].copy()
            auc_value = float(subset["roc_auc"].iloc[0])
            ax.plot(
                subset["fpr"].to_numpy(dtype=float),
                subset["tpr"].to_numpy(dtype=float),
                linewidth=1.2,
                label=f"Fold {fold} ({auc_value:.3f})",
            )

        pooled = algorithm_data.loc[
            algorithm_data["curve_type"] == "pooled_oof"
        ].copy()
        pooled_auc = float(pooled["roc_auc"].iloc[0])
        ax.plot(
            pooled["fpr"].to_numpy(dtype=float),
            pooled["tpr"].to_numpy(dtype=float),
            linewidth=2.4,
            linestyle="--",
            label=f"Pooled OOF ({pooled_auc:.3f})",
        )

        configure_roc_axis(ax, panel_labels[algorithm])
        ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE, loc="lower right")

    save_roc_figure(fig, png_path, pdf_path)


def build_cross_algorithm_roc_summary(
    pooled_summaries: Dict[str, Dict[str, float]],
) -> pd.DataFrame:
    rows = []
    for algorithm in Config.ALGORITHMS:
        source = pooled_summaries[algorithm]
        rows.append(
            {
                "algorithm": algorithm,
                "algorithm_label": Config.ALGORITHMS[algorithm]["label"],
                "scenario": Config.PRIMARY_SCENARIO,
                "model": Config.PRIMARY_MODEL_LABEL,
                "pooled_oof_roc_auc": source["pooled_oof_auc"],
                "mean_outer_fold_roc_auc": source["mean_fold_auc"],
                "sd_outer_fold_roc_auc": source["sd_fold_auc"],
                "minimum_outer_fold_roc_auc": source["min_fold_auc"],
                "maximum_outer_fold_roc_auc": source["max_fold_auc"],
                "auc_range": source["max_fold_auc"] - source["min_fold_auc"],
            }
        )
    return pd.DataFrame(rows)


def build_five_fold_roc_synthesis(
    paths_by_algorithm: Dict[str, Dict[str, Path]],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Dict[str, str]]]:
    """Build Step 7.4.8 entirely from saved Model E OOF predictions."""
    all_coordinates = []
    all_fold_summaries = []
    all_qa = []
    pooled_summaries: Dict[str, Dict[str, float]] = {}
    input_columns: Dict[str, Dict[str, str]] = {}

    for algorithm in Config.ALGORITHMS:
        model_e_oof, provenance = load_model_e_oof(
            algorithm,
            paths_by_algorithm[algorithm]["oof"],
        )
        input_columns[algorithm] = provenance

        coordinates, fold_summary, pooled_summary = build_algorithm_roc_results(
            algorithm,
            model_e_oof,
        )
        saved_fold_metrics = load_saved_model_e_fold_metrics(
            algorithm,
            paths_by_algorithm[algorithm]["fold_metrics"],
        )
        qa = build_roc_auc_qa(
            algorithm,
            fold_summary,
            saved_fold_metrics,
        )

        plot_algorithm_roc(
            algorithm,
            coordinates,
            Config.ROC_FIGURE_DIR
            / f"Step_7_4_8_{algorithm}_Model_E_Five_Fold_Spatial_ROC.png",
            Config.ROC_FIGURE_DIR
            / f"Step_7_4_8_{algorithm}_Model_E_Five_Fold_Spatial_ROC.pdf",
        )

        all_coordinates.append(coordinates)
        all_fold_summaries.append(fold_summary)
        all_qa.append(qa)
        pooled_summaries[algorithm] = pooled_summary

    combined_coordinates = pd.concat(all_coordinates, ignore_index=True)
    combined_fold_summary = pd.concat(all_fold_summaries, ignore_index=True)
    combined_qa = pd.concat(all_qa, ignore_index=True)
    cross_algorithm_summary = build_cross_algorithm_roc_summary(
        pooled_summaries
    )

    plot_combined_three_panel_roc(
        combined_coordinates,
        Config.ROC_FIGURE_DIR
        / "Step_7_4_8_Model_E_Three_Algorithm_Five_Fold_Spatial_ROC.png",
        Config.ROC_FIGURE_DIR
        / "Step_7_4_8_Model_E_Three_Algorithm_Five_Fold_Spatial_ROC.pdf",
    )

    return (
        combined_coordinates,
        combined_fold_summary,
        cross_algorithm_summary,
        combined_qa,
        input_columns,
    )


# =============================================================================
# REPORTING
# =============================================================================
def build_scientific_summary(
    ablation_long: pd.DataFrame,
    permutation_grouped: pd.DataFrame,
    hkfi_consensus: pd.DataFrame,
    buffer_consistency: pd.DataFrame,
    block_long: pd.DataFrame,
    mapped_area_high: pd.DataFrame,
    roc_summary: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for label, _, _, question in Config.REQUIRED_ABLATIONS:
        subset = ablation_long.loc[
            ablation_long["comparison"] == label
        ].copy()

        if len(subset) != len(Config.ALGORITHMS):
            raise RuntimeError(
                f"Scientific summary expected one {label} result for each "
                f"algorithm; found {len(subset)}."
            )

        all_point_estimates_positive = bool(
            (subset["delta_auc"] > 0.0).all()
        )
        all_intervals_above_zero = bool(
            (subset["ci_low"] > 0.0).all()
        )
        all_intervals_below_zero = bool(
            (subset["ci_high"] < 0.0).all()
        )
        all_intervals_include_zero = bool(
            (
                (subset["ci_low"] <= 0.0)
                & (subset["ci_high"] >= 0.0)
            ).all()
        )

        if all_intervals_above_zero:
            summary_text = (
                "The paired improvement is supported across XGB, RF and LR "
                "because every 95% spatial-block bootstrap interval is above zero."
            )
        elif all_intervals_below_zero:
            summary_text = (
                "The paired difference is negative across XGB, RF and LR "
                "because every 95% spatial-block bootstrap interval is below zero."
            )
        elif all_intervals_include_zero:
            summary_text = (
                "No algorithm shows a supported paired difference; all 95% "
                "spatial-block bootstrap intervals include zero."
            )
        elif all_point_estimates_positive:
            summary_text = (
                "Point estimates are positive across algorithms, but interval "
                "evidence is not consistently above zero."
            )
        else:
            summary_text = (
                "The direction and interval evidence are not consistent across "
                "all algorithms."
            )

        rows.append(
            {
                "domain": "A-F ablation",
                "test": label,
                "scientific_question": question,
                "all_algorithms_positive": all_point_estimates_positive,
                "all_algorithms_95pct_intervals_above_zero": (
                    all_intervals_above_zero
                ),
                "all_algorithms_95pct_intervals_include_zero": (
                    all_intervals_include_zero
                ),
                "summary": summary_text,
            }
        )

    grouped_primary = permutation_grouped.loc[
        permutation_grouped["feature_group"] == "HKFI + SNDR"
    ].copy()

    if len(grouped_primary) == len(Config.ALGORITHMS):
        permutation_parts = []
        for algorithm in Config.ALGORITHMS:
            record = grouped_primary.loc[
                grouped_primary["algorithm"] == algorithm
            ].iloc[0]
            permutation_parts.append(
                f"{algorithm}={float(record['mean_auc_decrease']):.3f} "
                f"(SD {float(record['sd_across_outer_folds']):.3f})"
            )

        rows.append(
            {
                "domain": "Held-out permutation importance",
                "test": "Joint HKFI + SNDR permutation in Model E",
                "scientific_question": (
                    "Does the joint physics-guided predictor pair carry "
                    "held-out predictive information across algorithms?"
                ),
                "all_algorithms_positive": bool(
                    (grouped_primary["mean_auc_decrease"] > 0.0).all()
                ),
                "all_algorithms_95pct_intervals_above_zero": np.nan,
                "summary": (
                    "Mean held-out ROC-AUC decrease after joint HKFI + SNDR "
                    "permutation: "
                    + "; ".join(permutation_parts)
                    + ". This is a conditional diagnostic; retraining-based "
                    "A-F ablation remains the stronger incremental-information test."
                ),
            }
        )

    if not hkfi_consensus.empty:
        rows.append(
            {
                "domain": "HKFI weighting",
                "test": "Equal-weight robustness",
                "scientific_question": (
                    "Does the conclusion that equal HKFI weighting is reasonably "
                    "robust persist across algorithms?"
                ),
                "all_algorithms_positive": np.nan,
                "all_algorithms_95pct_intervals_above_zero": np.nan,
                "summary": (
                    "All targeted alternative-versus-equal paired intervals "
                    "include zero across all algorithms."
                    if bool(
                        hkfi_consensus[
                            "all_95pct_intervals_include_zero"
                        ].all()
                    )
                    else "At least one targeted HKFI alternative shows an "
                    "algorithm-specific interval excluding zero."
                ),
            }
        )

    rows.append(
        {
            "domain": "Pseudo-absence buffer",
            "test": "E-A across 100/500/1000 m",
            "scientific_question": (
                "Does the Model E advantage over Model A remain directionally "
                "consistent across buffers and algorithms?"
            ),
            "all_algorithms_positive": bool(
                buffer_consistency[
                    "all_algorithms_directionally_positive"
                ].all()
            ),
            "all_algorithms_95pct_intervals_above_zero": bool(
                buffer_consistency[
                    "all_algorithms_95pct_intervals_above_zero"
                ].all()
            ),
            "summary": (
                "E-A remains directionally positive for every tested "
                "algorithm-buffer combination."
                if bool(
                    buffer_consistency[
                        "all_algorithms_directionally_positive"
                    ].all()
                )
                else "The E-A direction changes for at least one "
                "algorithm-buffer combination."
            ),
        }
    )

    e_auc = block_long["model_E_auc"].to_numpy(float)
    rows.append(
        {
            "domain": "Within-corridor geography",
            "test": "Five held-out spatial blocks",
            "scientific_question": (
                "How stable is Model E performance across the five held-out "
                "geographic blocks?"
            ),
            "all_algorithms_positive": np.nan,
            "all_algorithms_95pct_intervals_above_zero": np.nan,
            "summary": (
                f"Across all algorithm-block combinations, Model E held-out "
                f"AUC ranges from {np.min(e_auc):.3f} to {np.max(e_auc):.3f}. "
                "This is within-corridor robustness, not external regional "
                "transferability validation."
            ),
        }
    )

    if not mapped_area_high.empty:
        mapped_area_parts = []
        for algorithm in Config.ALGORITHMS:
            record = mapped_area_high.loc[
                mapped_area_high["algorithm"] == algorithm
            ].iloc[0]
            mapped_area_parts.append(
                f"{algorithm}={float(record['mean_area_percentage']):.2f}% "
                f"(SD {float(record['sd_area_percentage']):.2f}, "
                f"range {float(record['minimum_area_percentage']):.2f}-"
                f"{float(record['maximum_area_percentage']):.2f}%)"
            )

        rows.append(
            {
                "domain": "Mapped-area uncertainty",
                "test": "A30/E70 High + Very High cross-fold mapped area",
                "scientific_question": (
                    "How much do key susceptibility-area proportions vary across "
                    "the five fold-trained map realizations for each algorithm?"
                ),
                "all_algorithms_positive": np.nan,
                "all_algorithms_95pct_intervals_above_zero": np.nan,
                "summary": (
                    "High + Very High A30/E70 mapped-area variability: "
                    + "; ".join(mapped_area_parts)
                    + ". These are descriptive cross-fold statistics, not "
                    "formal 95% confidence intervals."
                ),
            }
        )

    if not roc_summary.empty:
        roc_parts = []
        for algorithm in Config.ALGORITHMS:
            record = roc_summary.loc[
                roc_summary["algorithm"] == algorithm
            ].iloc[0]
            roc_parts.append(
                f"{algorithm}: pooled OOF AUC "
                f"{float(record['pooled_oof_roc_auc']):.3f}, fold range "
                f"{float(record['minimum_outer_fold_roc_auc']):.3f}-"
                f"{float(record['maximum_outer_fold_roc_auc']):.3f}"
            )

        rows.append(
            {
                "domain": "Five-fold spatial ROC",
                "test": "Model E fold-specific and pooled OOF ROC curves",
                "scientific_question": (
                    "How does Model E discrimination vary across the five "
                    "geographically held-out folds for XGB, RF and LR?"
                ),
                "all_algorithms_positive": np.nan,
                "all_algorithms_95pct_intervals_above_zero": np.nan,
                "summary": (
                    "; ".join(roc_parts)
                    + ". Fold-specific curves visualize within-corridor spatial "
                    "heterogeneity; the pooled OOF curve is not external validation."
                ),
            }
        )

    return pd.DataFrame(rows)




def write_report(
    path: Path,
    performance_display: pd.DataFrame,
    ablation_wide: pd.DataFrame,
    incremental_index_summary: pd.DataFrame,
    learning_endpoint: pd.DataFrame,
    permutation_grouped_wide: pd.DataFrame,
    hkfi_consensus: pd.DataFrame,
    buffer_consistency: pd.DataFrame,
    block_wide: pd.DataFrame,
    mapped_area_wide: pd.DataFrame,
    mapped_area_high: pd.DataFrame,
    roc_cross_algorithm_summary: pd.DataFrame,
    roc_fold_summary: pd.DataFrame,
    roc_qa: pd.DataFrame,
    scientific_summary: pd.DataFrame,
) -> None:
    sections = [
        "STEP 7.4 — CROSS-ALGORITHM ROBUSTNESS AND SYNTHESIS",
        "=" * 96,
        f"Generated UTC: {utc_now_iso()}",
        "",
        "READ-ONLY SYNTHESIS RULE",
        "No model fitting, tuning, pseudo-absence generation, bootstrap recomputation, permutation recomputation, or map generation was performed.",
        "",
        "7.4.1 A-F PERFORMANCE MASTER TABLE",
        performance_display.to_string(index=False),
        "",
        "7.4.2 CROSS-ALGORITHM ABLATION SYNTHESIS",
        ablation_wide.to_string(index=False),
        "",
        "7.4.2A INCREMENTAL CONTRIBUTION OF THE TWO PHYSICS-GUIDED INDICES",
        incremental_index_summary.to_string(index=False),
        "",
        "E-C interpretation: SNDR added after HKFI is already included.",
        "E-D interpretation: HKFI added after SNDR is already included.",
        "A positive point estimate is not treated as supported unless the 95% spatial-block bootstrap interval is above zero.",
        "",
        "7.4.3A LEARNING-CURVE ENDPOINT SYNTHESIS",
        learning_endpoint.to_string(index=False),
        "",
        "7.4.3B HELD-OUT PERMUTATION-IMPORTANCE SYNTHESIS",
        permutation_grouped_wide.to_string(index=False),
        "",
        "Permutation interpretation rule: held-out permutation importance is conditional on the fitted Model E predictor set. Near-zero individual importance does not prove absence of predictive value when predictors are correlated or redundant; the retraining-based A-F ablation analysis remains the stronger incremental-information test.",
        "",
        "7.4.4 HKFI WEIGHTING ROBUSTNESS",
        hkfi_consensus.to_string(index=False),
        "",
        "Interpretation rule: the purpose is to test whether the a priori equal weighting remains reasonably robust, not to select the weighting with the maximum AUC.",
        "",
        "7.4.5 BUFFER SENSITIVITY",
        buffer_consistency.to_string(index=False),
        "",
        "7.4.6 WITHIN-CORRIDOR GEOGRAPHIC BLOCK ROBUSTNESS",
        block_wide.to_string(index=False),
        "",
        "Geographic interpretation rule: these five outer folds provide within-corridor geographic robustness evidence only and do not demonstrate external cross-region transferability.",
        "",
        "7.4.7 CROSS-ALGORITHM MAPPED-AREA UNCERTAINTY",
        mapped_area_wide.to_string(index=False),
        "",
        "High + Very High compact summary:",
        mapped_area_high.to_string(index=False),
        "",
        "Mapped-area interpretation rule: the values summarize cross-fold variability across five fold-trained A30/E70 operational ensemble maps. Mean, SD, minimum and maximum are descriptive variability statistics, not formal 95% confidence intervals.",
        "",
        "7.4.8 SUPPLEMENTARY FIVE-FOLD SPATIAL ROC SYNTHESIS FOR MODEL E",
        roc_cross_algorithm_summary.to_string(index=False),
        "",
        "Fold-level Model E ROC-AUC:",
        roc_fold_summary.to_string(index=False),
        "",
        "ROC reconstruction QA against saved outer-fold metrics:",
        roc_qa.to_string(index=False),
        "",
        "ROC interpretation rule: each curve is reconstructed from genuine saved OOF predictions for one geographically held-out spatial block. The pooled OOF curve combines those predictions and is not a sixth fold or external validation.",
        "",
        "STUDY-LEVEL SCIENTIFIC SUMMARY",
        scientific_summary.to_string(index=False),
        "",
    ]
    path.write_text("\n".join(sections), encoding="utf-8")

def write_metadata(
    path: Path,
    paths_by_algorithm: Dict[str, Dict[str, Path]],
) -> None:
    metadata = {
        "generated_at_utc": utc_now_iso(),
        "step": "7.4",
        "title": "Cross-Algorithm Robustness and Synthesis",
        "read_only": True,
        "model_fitting_performed": False,
        "hyperparameter_tuning_performed": False,
        "bootstrap_recomputed": False,
        "permutation_recomputed": False,
        "map_generation_performed": False,
        "fold_roc_curves_reconstructed": True,
        "roc_reconstruction_from_saved_oof_only": True,
        "primary_algorithms": ["XGB", "RF", "LR"],
        "primary_models": Config.SCENARIO_ORDER,
        "required_ablation_comparisons": [
            item[0] for item in Config.REQUIRED_ABLATIONS
        ],
        "incremental_index_comparisons": {
            "E - C": (
                "Incremental SNDR contribution beyond the static + HKFI "
                "configuration."
            ),
            "E - D": (
                "Incremental HKFI contribution beyond the static + SNDR "
                "configuration."
            ),
        },
        "hkfi_interpretation": (
            "Robustness of the a priori equal weighting; not optimization of AUC."
        ),
        "geographic_interpretation": (
            "Within-corridor held-out spatial-block robustness; not external transferability."
        ),
        "permutation_interpretation": (
            "Held-out Model E permutation importance is conditional on the fitted "
            "predictor set and complements, but does not replace, retraining-based "
            "A-F ablation."
        ),
        "mapped_area_interpretation": (
            "Cross-fold mean, SD, minimum and maximum across five fold-trained "
            "A30/E70 operational ensemble maps; descriptive variability, not a "
            "formal 95% confidence interval."
        ),
        "five_fold_roc_interpretation": (
            "Model E fold-specific ROC curves are reconstructed from genuine saved "
            "OOF predictions. The pooled OOF curve is not a sixth fold and does not "
            "constitute external validation."
        ),
        "resolved_inputs": {
            algorithm: {key: str(value) for key, value in paths.items()}
            for algorithm, paths in paths_by_algorithm.items()
        },
    }
    path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    create_output_directories()

    log("=" * 104)
    log("STEP 7.4 — CROSS-ALGORITHM ROBUSTNESS AND SYNTHESIS")
    log("READ-ONLY synthesis: no ML fitting and no bootstrap recomputation")
    log(f"Base directory: {Config.BASE}")
    log("=" * 104)

    # -------------------------------------------------------------------------
    # 7.4.0 Resolve final Step 7.1/7.2/7.3 outputs.
    # -------------------------------------------------------------------------
    paths_by_algorithm: Dict[str, Dict[str, Path]] = {}
    for algorithm in Config.ALGORITHMS:
        log(f"[7.4.0] Resolving {algorithm} final outputs...")
        paths_by_algorithm[algorithm] = resolve_algorithm_paths(algorithm)

    input_manifest = build_input_manifest(paths_by_algorithm)
    input_manifest.to_csv(
        Config.QA_DIR / "Step_7_4_Resolved_Input_Files.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # 7.4.1 A-F performance synthesis.
    # -------------------------------------------------------------------------
    log("[7.4.1] Building A-F cross-algorithm performance master table...")
    (
        performance_long,
        performance_master,
        performance_display,
        performance_qa,
    ) = build_performance_synthesis(paths_by_algorithm)

    performance_long.to_csv(
        Config.PERFORMANCE_DIR / "Step_7_4_A_to_F_Performance_Long.csv",
        index=False,
        float_format="%.10f",
    )
    performance_master.to_csv(
        Config.PERFORMANCE_DIR / "Step_7_4_A_to_F_Performance_Master_Table.csv",
        index=False,
        float_format="%.10f",
    )
    performance_display.to_csv(
        Config.PERFORMANCE_DIR / "Step_7_4_A_to_F_Performance_Master_Table_Formatted.csv",
        index=False,
    )
    performance_qa.to_csv(
        Config.QA_DIR / "Step_7_4_Performance_Point_Estimate_QA.csv",
        index=False,
        float_format="%.12f",
    )

    failed_qa = performance_qa.loc[~performance_qa["pass"]]
    if not failed_qa.empty:
        log(
            f"[QA WARNING] {len(failed_qa)} summary-versus-Step-D point estimates exceed "
            f"the QA tolerance of {Config.POINT_ESTIMATE_QA_TOLERANCE:g}. "
            "Synthesis continues using the Step D estimate paired with its bootstrap interval."
        )

    create_performance_figure(
        performance_long,
        Config.FIGURE_DIR / "Step_7_4_A_to_F_Cross_Algorithm_ROC_AUC.png",
    )

    # -------------------------------------------------------------------------
    # 7.4.2 Cross-algorithm ablation synthesis.
    # -------------------------------------------------------------------------
    log("[7.4.2] Reading saved paired AUC ablation comparisons, including E-C and E-D...")
    ablation_long, ablation_wide = build_ablation_synthesis(paths_by_algorithm)
    incremental_index_summary = build_incremental_index_summary(
        ablation_long
    )

    expected_ablation_rows = (
        len(Config.ALGORITHMS)
        * len(Config.REQUIRED_ABLATIONS)
    )
    if len(ablation_long) != expected_ablation_rows:
        raise RuntimeError(
            f"Step 7.4.2 expected {expected_ablation_rows} ablation rows "
            f"but produced {len(ablation_long)}."
        )

    expected_incremental_rows = len(Config.ALGORITHMS) * 2
    if len(incremental_index_summary) != expected_incremental_rows:
        raise RuntimeError(
            f"E-C/E-D summary expected {expected_incremental_rows} rows "
            f"but produced {len(incremental_index_summary)}."
        )

    log(
        "[7.4.2 QA] Confirmed "
        f"{len(ablation_long)} total ablation rows and "
        f"{len(incremental_index_summary)} focused E-C/E-D rows."
    )

    ablation_long.to_csv(
        Config.ABLATION_DIR / "Step_7_4_Ablation_Delta_AUC_Long.csv",
        index=False,
        float_format="%.10f",
    )
    ablation_wide.to_csv(
        Config.ABLATION_DIR / "Step_7_4_Ablation_Cross_Algorithm_Summary.csv",
        index=False,
        float_format="%.10f",
    )
    incremental_index_summary.to_csv(
        Config.ABLATION_DIR
        / "Step_7_4_E_C_E_D_Incremental_Index_Contribution.csv",
        index=False,
        float_format="%.10f",
    )
    create_ablation_figure(
        ablation_long,
        Config.FIGURE_DIR / "Step_7_4_Ablation_Paired_Delta_AUC.png",
    )

    # -------------------------------------------------------------------------
    # 7.4.3A Learning-curve synthesis.
    # -------------------------------------------------------------------------
    log("[7.4.3A] Combining XGB, RF and LR learning curves for Models A and E...")
    learning_all, learning_endpoint = build_learning_synthesis(paths_by_algorithm)
    learning_all.to_csv(
        Config.LEARNING_DIR / "Step_7_4_Learning_Curve_Synthesis.csv",
        index=False,
        float_format="%.10f",
    )
    learning_endpoint.to_csv(
        Config.LEARNING_DIR / "Step_7_4_Learning_Curve_Maximum_Training_Summary.csv",
        index=False,
        float_format="%.10f",
    )
    create_learning_figure(
        learning_all,
        "A_static_baseline",
        Config.FIGURE_DIR / "Step_7_4_Model_A_Learning_Curve_Synthesis.png",
    )
    create_learning_figure(
        learning_all,
        "E_physics_guided",
        Config.FIGURE_DIR / "Step_7_4_Model_E_Learning_Curve_Synthesis.png",
    )

    # -------------------------------------------------------------------------
    # 7.4.3B Cross-algorithm held-out permutation-importance synthesis.
    # -------------------------------------------------------------------------
    log(
        "[7.4.3B] Combining Model E held-out permutation importance "
        "from XGB, RF and LR..."
    )
    (
        permutation_individual_long,
        permutation_individual_wide,
        permutation_grouped_long,
        permutation_grouped_wide,
    ) = build_permutation_synthesis(paths_by_algorithm)

    permutation_individual_long.to_csv(
        Config.PERMUTATION_DIR
        / "Step_7_4_Model_E_Heldout_Permutation_Importance_Long.csv",
        index=False,
        float_format="%.10f",
    )
    permutation_individual_wide.to_csv(
        Config.PERMUTATION_DIR
        / "Step_7_4_Model_E_Heldout_Permutation_Importance_Cross_Algorithm.csv",
        index=False,
        float_format="%.10f",
    )
    permutation_grouped_long.to_csv(
        Config.PERMUTATION_DIR
        / "Step_7_4_Model_E_Grouped_HKFI_SNDR_Permutation_Long.csv",
        index=False,
        float_format="%.10f",
    )
    permutation_grouped_wide.to_csv(
        Config.PERMUTATION_DIR
        / "Step_7_4_Model_E_Grouped_HKFI_SNDR_Permutation_Cross_Algorithm.csv",
        index=False,
        float_format="%.10f",
    )
    create_grouped_permutation_figure(
        permutation_grouped_long,
        Config.FIGURE_DIR
        / "Step_7_4_Grouped_HKFI_SNDR_Heldout_Permutation.png",
    )

    # -------------------------------------------------------------------------
    # 7.4.4 HKFI weighting robustness synthesis.
    # -------------------------------------------------------------------------
    log("[7.4.4] Synthesizing full XGB and targeted RF/LR HKFI sensitivity...")
    (
        hkfi_xgb_full_performance,
        hkfi_xgb_full_paired,
        hkfi_common_performance,
        hkfi_common_paired,
        hkfi_consensus,
    ) = build_hkfi_synthesis(paths_by_algorithm)

    hkfi_xgb_full_performance.to_csv(
        Config.HKFI_DIR / "Step_7_4_HKFI_Full_XGB_Sensitivity_Performance.csv",
        index=False,
        float_format="%.10f",
    )
    hkfi_xgb_full_paired.to_csv(
        Config.HKFI_DIR / "Step_7_4_HKFI_Full_XGB_Sensitivity_Paired_AUC.csv",
        index=False,
        float_format="%.10f",
    )
    hkfi_common_performance.to_csv(
        Config.HKFI_DIR / "Step_7_4_HKFI_Common_Cross_Algorithm_Performance.csv",
        index=False,
        float_format="%.10f",
    )
    hkfi_common_paired.to_csv(
        Config.HKFI_DIR / "Step_7_4_HKFI_Alternative_minus_Equal_Paired_AUC.csv",
        index=False,
        float_format="%.10f",
    )
    hkfi_consensus.to_csv(
        Config.HKFI_DIR / "Step_7_4_HKFI_Equal_Weight_Robustness_Consensus.csv",
        index=False,
        float_format="%.10f",
    )
    create_hkfi_figure(
        hkfi_common_performance,
        Config.FIGURE_DIR / "Step_7_4_HKFI_Weight_Robustness.png",
    )

    # -------------------------------------------------------------------------
    # 7.4.5 Pseudo-absence buffer sensitivity synthesis.
    # -------------------------------------------------------------------------
    log("[7.4.5] Combining A/E results for 100, 500 and 1000 m buffers...")
    buffer_all, buffer_consistency = build_buffer_synthesis(paths_by_algorithm)
    buffer_all.to_csv(
        Config.BUFFER_DIR / "Step_7_4_Buffer_Sensitivity_Synthesis.csv",
        index=False,
        float_format="%.10f",
    )
    buffer_consistency.to_csv(
        Config.BUFFER_DIR / "Step_7_4_Buffer_Directional_Consistency.csv",
        index=False,
        float_format="%.10f",
    )
    create_buffer_figure(
        buffer_all,
        Config.FIGURE_DIR / "Step_7_4_Buffer_Sensitivity_E_minus_A.png",
    )

    # -------------------------------------------------------------------------
    # 7.4.6 Geographic block robustness.
    # -------------------------------------------------------------------------
    log("[7.4.6] Summarizing Model E performance across five held-out spatial blocks...")
    block_long, block_wide = build_geographic_block_synthesis(paths_by_algorithm)
    block_long.to_csv(
        Config.BLOCK_DIR / "Step_7_4_Geographic_Block_Robustness_Long.csv",
        index=False,
        float_format="%.10f",
    )
    block_wide.to_csv(
        Config.BLOCK_DIR / "Step_7_4_Geographic_Block_Model_E_AUC.csv",
        index=False,
        float_format="%.10f",
    )
    create_geographic_block_figure(
        block_long,
        Config.FIGURE_DIR / "Step_7_4_Model_E_Geographic_Block_Robustness.png",
    )

    # -------------------------------------------------------------------------
    # 7.4.7 Cross-algorithm mapped-area uncertainty synthesis.
    # -------------------------------------------------------------------------
    log(
        "[7.4.7] Combining A30/E70 cross-fold mapped-area uncertainty "
        "from XGB, RF and LR..."
    )
    (
        mapped_area_long,
        mapped_area_wide,
        mapped_area_high,
    ) = build_mapped_area_uncertainty_synthesis(paths_by_algorithm)

    mapped_area_long.to_csv(
        Config.MAP_UNCERTAINTY_DIR
        / "Step_7_4_Mapped_Area_Uncertainty_Long.csv",
        index=False,
        float_format="%.10f",
    )
    mapped_area_wide.to_csv(
        Config.MAP_UNCERTAINTY_DIR
        / "Step_7_4_Mapped_Area_Uncertainty_Cross_Algorithm.csv",
        index=False,
        float_format="%.10f",
    )
    mapped_area_high.to_csv(
        Config.MAP_UNCERTAINTY_DIR
        / "Step_7_4_High_Very_High_Mapped_Area_Uncertainty_Summary.csv",
        index=False,
        float_format="%.10f",
    )
    create_mapped_area_uncertainty_figure(
        mapped_area_long,
        Config.FIGURE_DIR
        / "Step_7_4_Cross_Algorithm_Mapped_Area_Uncertainty.png",
    )
    create_high_very_high_uncertainty_figure(
        mapped_area_high,
        Config.FIGURE_DIR
        / "Step_7_4_High_Very_High_Mapped_Area_Uncertainty.png",
    )

    # -------------------------------------------------------------------------
    # 7.4.8 Supplementary five-fold spatial ROC synthesis for Model E.
    # -------------------------------------------------------------------------
    log(
        "[7.4.8] Reconstructing Model E five-fold spatial ROC curves "
        "from final saved OOF predictions..."
    )
    (
        roc_coordinates,
        roc_fold_summary,
        roc_cross_algorithm_summary,
        roc_qa,
        roc_input_columns,
    ) = build_five_fold_roc_synthesis(paths_by_algorithm)

    roc_coordinates.to_csv(
        Config.ROC_TABLE_DIR
        / "Step_7_4_8_Model_E_Five_Fold_ROC_Coordinates.csv",
        index=False,
        float_format="%.12f",
    )
    roc_fold_summary.to_csv(
        Config.ROC_TABLE_DIR
        / "Step_7_4_8_Model_E_Five_Fold_ROC_AUC_Summary.csv",
        index=False,
        float_format="%.10f",
    )
    roc_cross_algorithm_summary.to_csv(
        Config.ROC_TABLE_DIR
        / "Step_7_4_8_Model_E_Cross_Algorithm_ROC_Summary.csv",
        index=False,
        float_format="%.10f",
    )
    roc_qa.to_csv(
        Config.ROC_QA_DIR
        / "Step_7_4_8_Reconstructed_vs_Saved_Fold_AUC_QA.csv",
        index=False,
        float_format="%.12f",
    )

    failed_roc_qa = roc_qa.loc[~roc_qa["pass"]]
    if not failed_roc_qa.empty:
        log(
            f"[QA WARNING] {len(failed_roc_qa)} reconstructed Model E fold "
            "AUC value(s) differ from the saved outer-fold metrics by more than "
            f"{Config.ROC_AUC_QA_TOLERANCE:g}."
        )

    roc_metadata = {
        "generated_at_utc": utc_now_iso(),
        "step": "7.4.8",
        "title": "Supplementary Five-Fold Spatial ROC Curves for Model E",
        "read_only": True,
        "model_fitting_performed": False,
        "scenario": Config.PRIMARY_SCENARIO,
        "model_label": Config.PRIMARY_MODEL_LABEL,
        "expected_spatial_folds": Config.EXPECTED_FOLDS,
        "normalized_input_columns": roc_input_columns,
        "interpretation": (
            "Within-corridor spatial robustness visualization reconstructed from "
            "genuine saved OOF predictions; not external transferability."
        ),
    }
    (Config.ROC_DIR / "Step_7_4_8_Five_Fold_Spatial_ROC_Metadata.json").write_text(
        json.dumps(roc_metadata, indent=2),
        encoding="utf-8",
    )

    # -------------------------------------------------------------------------
    # Study-level scientific synthesis.
    # -------------------------------------------------------------------------
    scientific_summary = build_scientific_summary(
        ablation_long=ablation_long,
        permutation_grouped=permutation_grouped_long,
        hkfi_consensus=hkfi_consensus,
        buffer_consistency=buffer_consistency,
        block_long=block_long,
        mapped_area_high=mapped_area_high,
        roc_summary=roc_cross_algorithm_summary,
    )
    scientific_summary.to_csv(
        Config.OUT_DIR / "Step_7_4_Study_Level_Scientific_Summary.csv",
        index=False,
    )

    write_report(
        Config.OUT_DIR / "Step_7_4_Cross_Algorithm_Synthesis_Report.txt",
        performance_display=performance_display,
        ablation_wide=ablation_wide,
        incremental_index_summary=incremental_index_summary,
        learning_endpoint=learning_endpoint,
        permutation_grouped_wide=permutation_grouped_wide,
        hkfi_consensus=hkfi_consensus,
        buffer_consistency=buffer_consistency,
        block_wide=block_wide,
        mapped_area_wide=mapped_area_wide,
        mapped_area_high=mapped_area_high,
        roc_cross_algorithm_summary=roc_cross_algorithm_summary,
        roc_fold_summary=roc_fold_summary,
        roc_qa=roc_qa,
        scientific_summary=scientific_summary,
    )

    write_metadata(
        Config.OUT_DIR / "Step_7_4_Cross_Algorithm_Synthesis_Metadata.json",
        paths_by_algorithm,
    )

    log("=" * 104)
    log("STEP 7.4 COMPLETED SUCCESSFULLY")
    log(f"Output directory: {Config.OUT_DIR}")
    log("=" * 104)

    print("\nA-F PERFORMANCE MASTER TABLE")
    print(performance_display.to_string(index=False))

    print("\nCROSS-ALGORITHM ABLATION SYNTHESIS")
    print(ablation_wide.to_string(index=False))

    print("\nE-C / E-D INCREMENTAL INDEX CONTRIBUTION")
    print(incremental_index_summary.to_string(index=False))

    print("\nGROUPED HKFI + SNDR HELD-OUT PERMUTATION IMPORTANCE")
    print(permutation_grouped_wide.to_string(index=False))

    print("\nHKFI EQUAL-WEIGHT ROBUSTNESS")
    print(hkfi_consensus.to_string(index=False))

    print("\nBUFFER DIRECTIONAL CONSISTENCY")
    print(buffer_consistency.to_string(index=False))

    print("\nGEOGRAPHIC BLOCK MODEL E AUC")
    print(block_wide.to_string(index=False))

    print("\nHIGH + VERY HIGH MAPPED-AREA UNCERTAINTY")
    print(mapped_area_high.to_string(index=False))

    print("\nMODEL E FIVE-FOLD SPATIAL ROC SUMMARY")
    print(roc_cross_algorithm_summary.to_string(index=False))

    print("\nMODEL E FOLD-LEVEL ROC-AUC")
    print(roc_fold_summary.to_string(index=False))


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"[ERROR] Step 7.4 failed: {exc}", flush=True)
        traceback.print_exc()
        raise
